In [1]:
# ============================================================
# STEP 00A — CELL 1
# PROJECT CONFIGURATION + ENVIRONMENT VERIFICATION
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

MIN_R_VERSION <- numeric_version("4.4.0")

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

stopifnot(
  length(PROJECT_ROOT) == 1L,
  nzchar(PROJECT_ROOT),
  length(PROJECT_ID) == 1L,
  nzchar(PROJECT_ID)
)

# ------------------------------------------------------------
# Confirm correct operating system
# ------------------------------------------------------------

if (.Platform$OS.type != "windows") {
  stop("STOP: This project is configured for Windows.")
}

# ------------------------------------------------------------
# Confirm required R version
# ------------------------------------------------------------

if (getRversion() < MIN_R_VERSION) {
  stop(
    paste0(
      "STOP: R >= ",
      as.character(MIN_R_VERSION),
      " is required. Current R = ",
      as.character(getRversion())
    )
  )
}

# ------------------------------------------------------------
# Confirm E: drive exists
# ------------------------------------------------------------

if (!dir.exists("E:/")) {
  stop(
    "STOP: E:/ drive is not available."
  )
}

# ------------------------------------------------------------
# Print environment summary
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00A — CELL 1\n")
cat("============================================================\n")
cat("R version:    ", R.version.string, "\n", sep = "")
cat("R home:       ", R.home(), "\n", sep = "")
cat("OS type:      ", .Platform$OS.type, "\n", sep = "")
cat("Project root: ", PROJECT_ROOT, "\n", sep = "")
cat("Project ID:   ", PROJECT_ID, "\n", sep = "")
cat("E:/ exists:   ", dir.exists("E:/"), "\n", sep = "")
cat("============================================================\n")
cat("CELL 1 — PASS\n")

STEP 00A — CELL 1
R version:    R version 4.5.2 (2025-10-31 ucrt)
R home:       E:/R-4.5.2
OS type:      windows
Project root: E:/SCZ_CROSSREGION_MSS_Q1
Project ID:   SCZ_CROSSREGION_MSS_Q1_2026
E:/ exists:   TRUE
CELL 1 — PASS


In [2]:
# ============================================================
# STEP 00A — CELL 2
# SAFE PROJECT BOOTSTRAP + SENTIN CREATION/VERIFICATION
# ============================================================

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

# ------------------------------------------------------------
# Helper: standardize path text
# ------------------------------------------------------------

clean_path <- function(x) {
  x <- gsub("\\\\", "/", x)
  x <- sub("/+$", "", x)
  tolower(x)
}

EXPECTED_PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"

# ------------------------------------------------------------
# Safety check: exact project root
# ------------------------------------------------------------

if (
  !identical(
    clean_path(PROJECT_ROOT),
    clean_path(EXPECTED_PROJECT_ROOT)
  )
) {
  stop(
    paste0(
      "SAFETY STOP: Unexpected PROJECT_ROOT.\n",
      "Expected: ", EXPECTED_PROJECT_ROOT, "\n",
      "Found:    ", PROJECT_ROOT
    )
  )
}

# ------------------------------------------------------------
# Create project root only if it does not exist
# ------------------------------------------------------------

if (!dir.exists(PROJECT_ROOT)) {

  created <- dir.create(
    PROJECT_ROOT,
    recursive = TRUE,
    showWarnings = FALSE
  )

  if (!created && !dir.exists(PROJECT_ROOT)) {
    stop(
      paste0(
        "FAILED to create project root:\n",
        PROJECT_ROOT
      )
    )
  }

  cat("Project root created.\n")

} else {

  cat("Project root already exists.\n")
}

# ------------------------------------------------------------
# If sentinel is absent, ensure directory is empty
# before creating it
# ------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {

  existing_items <- list.files(
    PROJECT_ROOT,
    all.files = TRUE,
    no.. = TRUE,
    recursive = FALSE
  )

  if (length(existing_items) > 0L) {

    stop(
      paste0(
        "SAFETY STOP:\n",
        "Project root exists but sentinel is missing.\n",
        "The directory is not empty.\n\n",
        "Existing items:\n- ",
        paste(existing_items, collapse = "\n- "),
        "\n\n",
        "Do not continue until this folder is manually checked."
      )
    )
  }

  writeLines(
    PROJECT_ID,
    SENTINEL_PATH,
    useBytes = TRUE
  )

  cat("New project sentinel created.\n")
}

# ------------------------------------------------------------
# Verify sentinel content
# ------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
  stop("SENTINEL VERIFICATION FAILED: .project_id does not exist.")
}

actual_project_id <- trimws(
  paste(
    readLines(
      SENTINEL_PATH,
      warn = FALSE
    ),
    collapse = "\n"
  )
)

if (!identical(actual_project_id, PROJECT_ID)) {

  stop(
    paste0(
      "SAFETY STOP: Sentinel mismatch.\n",
      "Expected: ", PROJECT_ID, "\n",
      "Found:    ", actual_project_id
    )
  )
}

cat("============================================================\n")
cat("STEP 00A — CELL 2\n")
cat("============================================================\n")
cat("Project root: ", PROJECT_ROOT, "\n", sep = "")
cat("Sentinel:     ", SENTINEL_PATH, "\n", sep = "")
cat("Project ID:   ", actual_project_id, "\n", sep = "")
cat("============================================================\n")
cat("CELL 2 — PASS\n")

Project root created.
New project sentinel created.
STEP 00A — CELL 2
Project root: E:/SCZ_CROSSREGION_MSS_Q1
Sentinel:     E:/SCZ_CROSSREGION_MSS_Q1/.project_id
Project ID:   SCZ_CROSSREGION_MSS_Q1_2026
CELL 2 — PASS


In [3]:
# ============================================================
# STEP 00A — CELL 3
# CREATE COMPLETE PROJECT FOLDER STRUCTURE
# ============================================================

# ------------------------------------------------------------
# Sentinel verification function
# ------------------------------------------------------------

assert_project_sentinel <- function() {

  if (!dir.exists(PROJECT_ROOT)) {
    stop("SAFETY STOP: Project root does not exist.")
  }

  if (!file.exists(SENTINEL_PATH)) {
    stop("SAFETY STOP: Project sentinel is missing.")
  }

  actual_id <- trimws(
    paste(
      readLines(
        SENTINEL_PATH,
        warn = FALSE
      ),
      collapse = "\n"
    )
  )

  if (!identical(actual_id, PROJECT_ID)) {
    stop(
      paste0(
        "SAFETY STOP: Sentinel mismatch.\n",
        "Expected: ", PROJECT_ID, "\n",
        "Found:    ", actual_id
      )
    )
  }

  invisible(TRUE)
}

# ------------------------------------------------------------
# Verify before writing anything
# ------------------------------------------------------------

assert_project_sentinel()

# ------------------------------------------------------------
# Required project folders
# ------------------------------------------------------------

REQUIRED_DIRS <- c(
  "00_protocol",

  "01_raw",
  "01_raw/brainseq_phase2",
  "01_raw/cmc",
  "01_raw/gse80655",

  "02_metadata",
  "03_intermediate",
  "04_processed",
  "05_splits",

  "06_models",
  "06_models/baselines",
  "06_models/shared_deviation",
  "06_models/oof",

  "07_stability",
  "07_stability/observed",
  "07_stability/permutations",
  "07_stability/sensitivity",

  "08_score_lock",

  "09_external",
  "09_external/cmc",
  "09_external/gse80655",

  "10_biology",
  "11_figures",
  "12_tables",
  "13_manuscript_support",
  "14_exports",
  "15_environment",
  "99_logs",

  "notebooks"
)

# ------------------------------------------------------------
# Create folders
# ------------------------------------------------------------

created_count <- 0L
existing_count <- 0L

for (rel_dir in REQUIRED_DIRS) {

  assert_project_sentinel()

  full_path <- file.path(
    PROJECT_ROOT,
    rel_dir
  )

  if (!dir.exists(full_path)) {

    ok <- dir.create(
      full_path,
      recursive = TRUE,
      showWarnings = FALSE
    )

    if (!ok && !dir.exists(full_path)) {
      stop(
        paste0(
          "FAILED to create directory:\n",
          full_path
        )
      )
    }

    created_count <- created_count + 1L

  } else {

    existing_count <- existing_count + 1L
  }
}

# ------------------------------------------------------------
# Validate all folders
# ------------------------------------------------------------

folder_status <- data.frame(
  folder = REQUIRED_DIRS,
  exists = dir.exists(
    file.path(
      PROJECT_ROOT,
      REQUIRED_DIRS
    )
  ),
  stringsAsFactors = FALSE
)

missing_dirs <- folder_status$folder[
  !folder_status$exists
]

if (length(missing_dirs) > 0L) {
  stop(
    paste0(
      "FOLDER STRUCTURE VALIDATION FAILED.\n",
      "Missing directories:\n- ",
      paste(
        missing_dirs,
        collapse = "\n- "
      )
    )
  )
}

cat("============================================================\n")
cat("STEP 00A — CELL 3\n")
cat("============================================================\n")
cat("Required folders: ", length(REQUIRED_DIRS), "\n", sep = "")
cat("Created now:      ", created_count, "\n", sep = "")
cat("Already existed:  ", existing_count, "\n", sep = "")
cat("Missing folders:  ", length(missing_dirs), "\n", sep = "")
cat("============================================================\n")
cat("CELL 3 — PASS\n")

STEP 00A — CELL 3
Required folders: 29
Created now:      29
Already existed:  0
Missing folders:  0
CELL 3 — PASS


In [4]:
# ============================================================
# STEP 00A — CELL 4
# CREATE TIMESTAMPED AUDIT LOG + RECORD BASIC ENVIRONMENT
# ============================================================

# ------------------------------------------------------------
# Verify sentinel before any write
# ------------------------------------------------------------

assert_project_sentinel()

# ------------------------------------------------------------
# Create run timestamp
# ------------------------------------------------------------

RUN_TIMESTAMP <- format(
  Sys.time(),
  "%Y%m%d_%H%M%S"
)

LOG_PATH <- file.path(
  PROJECT_ROOT,
  "99_logs",
  paste0(
    "00A_Project_Scaffold_and_Environment_",
    RUN_TIMESTAMP,
    ".log"
  )
)

# ------------------------------------------------------------
# Logging function
# ------------------------------------------------------------

log_line <- function(...) {

  assert_project_sentinel()

  message_text <- paste0(...)

  timestamp_text <- format(
    Sys.time(),
    "%Y-%m-%d %H:%M:%S %z"
  )

  full_line <- paste0(
    "[",
    timestamp_text,
    "] ",
    message_text
  )

  cat(
    full_line,
    "\n"
  )

  cat(
    full_line,
    "\n",
    file = LOG_PATH,
    append = TRUE
  )

  invisible(full_line)
}

# ------------------------------------------------------------
# Create fresh log file
# ------------------------------------------------------------

assert_project_sentinel()

writeLines(
  character(0),
  LOG_PATH,
  useBytes = TRUE
)

# ------------------------------------------------------------
# Record project/environment information
# ------------------------------------------------------------

log_line("STEP 00A START")
log_line("PROJECT_ROOT=", PROJECT_ROOT)
log_line("PROJECT_ID=", PROJECT_ID)
log_line("R_VERSION=", R.version.string)
log_line("R_HOME=", R.home())
log_line("R_PLATFORM=", R.version$platform)
log_line("OS_TYPE=", .Platform$OS.type)
log_line("SENTINEL_VERIFICATION=PASS")

# ------------------------------------------------------------
# Validate log file
# ------------------------------------------------------------

if (!file.exists(LOG_PATH)) {
  stop("LOG CREATION FAILED: Log file was not created.")
}

log_size <- file.info(LOG_PATH)$size

if (is.na(log_size) || log_size <= 0) {
  stop("LOG CREATION FAILED: Log file is empty.")
}

cat("============================================================\n")
cat("STEP 00A — CELL 4\n")
cat("============================================================\n")
cat("Log file: ", LOG_PATH, "\n", sep = "")
cat("Log exists: ", file.exists(LOG_PATH), "\n", sep = "")
cat("Log size: ", log_size, " bytes\n", sep = "")
cat("============================================================\n")
cat("CELL 4 — PASS\n")

[2026-09-29 21:48:33 +0600] STEP 00A START 
[2026-09-29 21:48:33 +0600] PROJECT_ROOT=E:/SCZ_CROSSREGION_MSS_Q1 
[2026-09-29 21:48:33 +0600] PROJECT_ID=SCZ_CROSSREGION_MSS_Q1_2026 
[2026-09-29 21:48:33 +0600] R_VERSION=R version 4.5.2 (2025-10-31 ucrt) 
[2026-09-29 21:48:33 +0600] R_HOME=E:/R-4.5.2 
[2026-09-29 21:48:33 +0600] R_PLATFORM=x86_64-w64-mingw32 
[2026-09-29 21:48:33 +0600] OS_TYPE=windows 
[2026-09-29 21:48:33 +0600] SENTINEL_VERIFICATION=PASS 
STEP 00A — CELL 4
Log file: E:/SCZ_CROSSREGION_MSS_Q1/99_logs/00A_Project_Scaffold_and_Environment_20260929_214833.log
Log exists: TRUE
Log size: 468 bytes
CELL 4 — PASS


In [5]:
# ============================================================
# STEP 00A — CELL 5
# ENVIRONMENT + JUPYTER + IRKERNEL AUDIT
# ============================================================

# ------------------------------------------------------------
# Verify project sentinel before proceeding
# ------------------------------------------------------------

assert_project_sentinel()

# ------------------------------------------------------------
# Detect executables and kernel-related components
# ------------------------------------------------------------

R_PATH <- file.path(
  R.home("bin"),
  "R.exe"
)

RSCRIPT_PATH <- file.path(
  R.home("bin"),
  "Rscript.exe"
)

JUPYTER_PATH <- Sys.which("jupyter")
PYTHON_PATH <- Sys.which("python")
GIT_PATH <- Sys.which("git")

IRKERNEL_AVAILABLE <- requireNamespace(
  "IRkernel",
  quietly = TRUE
)

# ------------------------------------------------------------
# Hard requirements
# ------------------------------------------------------------

if (!file.exists(R_PATH)) {
  stop(
    paste0(
      "ENVIRONMENT FAIL: R executable not found at:\n",
      R_PATH
    )
  )
}

if (!file.exists(RSCRIPT_PATH)) {
  stop(
    paste0(
      "ENVIRONMENT FAIL: Rscript executable not found at:\n",
      RSCRIPT_PATH
    )
  )
}

if (!IRKERNEL_AVAILABLE) {
  stop(
    paste0(
      "ENVIRONMENT FAIL: IRkernel is not available in R ",
      as.character(getRversion()),
      "."
    )
  )
}

if (!nzchar(JUPYTER_PATH)) {
  stop(
    paste0(
      "ENVIRONMENT FAIL: Jupyter is not visible to this R session.\n",
      "Expected your E:/Python/Scripts installation to be on PATH."
    )
  )
}

# ------------------------------------------------------------
# Log results
# ------------------------------------------------------------

log_line("R_EXECUTABLE=", R_PATH)
log_line("RSCRIPT_EXECUTABLE=", RSCRIPT_PATH)
log_line("JUPYTER_EXECUTABLE=", JUPYTER_PATH)
log_line("PYTHON_EXECUTABLE=", ifelse(nzchar(PYTHON_PATH), PYTHON_PATH, "NOT_ON_PATH"))
log_line("GIT_EXECUTABLE=", ifelse(nzchar(GIT_PATH), GIT_PATH, "NOT_ON_PATH"))
log_line("IRKERNEL_AVAILABLE=", IRKERNEL_AVAILABLE)

# ------------------------------------------------------------
# Print audit summary
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00A — CELL 5\n")
cat("============================================================\n")
cat("R executable:       ", R_PATH, "\n", sep = "")
cat("Rscript executable: ", RSCRIPT_PATH, "\n", sep = "")
cat("Jupyter executable: ", JUPYTER_PATH, "\n", sep = "")
cat("Python executable:  ",
    ifelse(nzchar(PYTHON_PATH), PYTHON_PATH, "NOT ON PATH"),
    "\n",
    sep = "")
cat("Git executable:     ",
    ifelse(nzchar(GIT_PATH), GIT_PATH, "NOT ON PATH"),
    "\n",
    sep = "")
cat("IRkernel available: ", IRKERNEL_AVAILABLE, "\n", sep = "")
cat("R version valid:    ", getRversion() >= MIN_R_VERSION, "\n", sep = "")
cat("============================================================\n")
cat("CELL 5 — PASS\n")

[2026-09-29 21:49:20 +0600] R_EXECUTABLE=E:/R-4.5.2/bin/x64/R.exe 
[2026-09-29 21:49:20 +0600] RSCRIPT_EXECUTABLE=E:/R-4.5.2/bin/x64/Rscript.exe 
[2026-09-29 21:49:20 +0600] JUPYTER_EXECUTABLE=E:\Python\Scripts\jupyter.exe 
[2026-09-29 21:49:20 +0600] PYTHON_EXECUTABLE=E:\Python\python.exe 
[2026-09-29 21:49:20 +0600] GIT_EXECUTABLE=NOT_ON_PATH 
[2026-09-29 21:49:20 +0600] IRKERNEL_AVAILABLE=TRUE 
STEP 00A — CELL 5
R executable:       E:/R-4.5.2/bin/x64/R.exe
Rscript executable: E:/R-4.5.2/bin/x64/Rscript.exe
Jupyter executable: E:\Python\Scripts\jupyter.exe
Python executable:  E:\Python\python.exe
Git executable:     NOT ON PATH
IRkernel available: TRUE
R version valid:    TRUE
CELL 5 — PASS


In [6]:
# ============================================================
# STEP 00A — CELL 6
# INITIALIZE PROJECT-SPECIFIC RENV ENVIRONMENT
# ============================================================

# ------------------------------------------------------------
# Safety verification
# ------------------------------------------------------------

assert_project_sentinel()

# ------------------------------------------------------------
# Use a fixed CRAN mirror for reproducibility
# ------------------------------------------------------------

options(
  repos = c(
    CRAN = "https://cloud.r-project.org"
  )
)

# ------------------------------------------------------------
# Install renv only if it is not already available
# ------------------------------------------------------------

if (!requireNamespace("renv", quietly = TRUE)) {

  log_line("renv not found; installing from CRAN.")

  install.packages(
    "renv",
    repos = "https://cloud.r-project.org"
  )
}

# ------------------------------------------------------------
# Verify renv installation
# ------------------------------------------------------------

if (!requireNamespace("renv", quietly = TRUE)) {
  stop(
    "RENV FAIL: Package 'renv' could not be installed or loaded."
  )
}

RENV_VERSION <- as.character(
  packageVersion("renv")
)

log_line(
  "RENV_VERSION=",
  RENV_VERSION
)

# ------------------------------------------------------------
# Define expected renv artifacts
# ------------------------------------------------------------

RENV_DIR <- file.path(
  PROJECT_ROOT,
  "renv"
)

RENV_LOCK <- file.path(
  PROJECT_ROOT,
  "renv.lock"
)

RPROFILE_PATH <- file.path(
  PROJECT_ROOT,
  ".Rprofile"
)

# ------------------------------------------------------------
# Initialize/activate the project environment
# ------------------------------------------------------------

assert_project_sentinel()

if (!dir.exists(RENV_DIR)) {

  log_line(
    "Initializing new bare renv environment."
  )

  renv::init(
    project = PROJECT_ROOT,
    bare = TRUE,
    restart = FALSE
  )

} else {

  log_line(
    "Existing renv directory detected; activating it."
  )

  renv::activate(
    project = PROJECT_ROOT
  )
}

# ------------------------------------------------------------
# Make sure renv.lock exists
# ------------------------------------------------------------

assert_project_sentinel()

if (!file.exists(RENV_LOCK)) {

  log_line(
    "Creating initial renv.lock snapshot."
  )

  renv::snapshot(
    project = PROJECT_ROOT,
    prompt = FALSE
  )
}

# ------------------------------------------------------------
# Validate expected artifacts
# ------------------------------------------------------------

RENV_DIR_OK <- dir.exists(RENV_DIR)
RENV_LOCK_OK <- file.exists(RENV_LOCK)
RPROFILE_OK <- file.exists(RPROFILE_PATH)

if (!RENV_DIR_OK) {
  stop(
    "RENV FAIL: renv/ directory was not created."
  )
}

if (!RENV_LOCK_OK) {
  stop(
    "RENV FAIL: renv.lock was not created."
  )
}

if (!RPROFILE_OK) {
  stop(
    "RENV FAIL: .Rprofile was not created."
  )
}

# ------------------------------------------------------------
# Record project library information
# ------------------------------------------------------------

PROJECT_LIBRARY <- .libPaths()[1]

log_line(
  "RENV_INITIALIZATION=PASS"
)

log_line(
  "PROJECT_LIBRARY=",
  PROJECT_LIBRARY
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00A — CELL 6\n")
cat("============================================================\n")
cat("renv version:      ", RENV_VERSION, "\n", sep = "")
cat("renv directory:    ", RENV_DIR_OK, "\n", sep = "")
cat("renv.lock exists:  ", RENV_LOCK_OK, "\n", sep = "")
cat(".Rprofile exists:  ", RPROFILE_OK, "\n", sep = "")
cat("Project library:   ", PROJECT_LIBRARY, "\n", sep = "")
cat("============================================================\n")
cat("CELL 6 — PASS\n")

[2026-09-29 21:50:26 +0600] renv not found; installing from CRAN. 

  There is a binary version available but the source version is later:
     binary source needs_compilation
renv  1.2.4  1.3.0             FALSE



installing the source package 'renv'




[2026-09-29 21:51:08 +0600] RENV_VERSION=1.3.0 
[2026-09-29 21:51:08 +0600] Initializing new bare renv environment. 
- The project is out-of-sync -- use `renv::status()` for details.
[2026-09-29 21:51:09 +0600] Creating initial renv.lock snapshot. 
The following package(s) will be updated in the lockfile:

# CRAN -----------------------------------------------------------------------
- renv   [* -> 1.3.0]

The version of R recorded in the lockfile will be updated:
- R      [* -> 4.5.2]

- Lockfile written to "E:/SCZ_CROSSREGION_MSS_Q1/renv.lock".
[2026-09-29 21:51:10 +0600] RENV_INITIALIZATION=PASS 
[2026-09-29 21:51:10 +0600] PROJECT_LIBRARY=E:/SCZ_CROSSREGION_MSS_Q1/renv/library/windows/R-4.5/x86_64-w64-mingw32 
STEP 00A — CELL 6
renv version:      1.3.0
renv directory:    TRUE
renv.lock exists:  TRUE
.Rprofile exists:  TRUE
Project library:   E:/SCZ_CROSSREGION_MSS_Q1/renv/library/windows/R-4.5/x86_64-w64-mingw32
CELL 6 — PASS


In [7]:
# ============================================================
# STEP 00A — CELL 7
# PACKAGE AVAILABILITY AUDIT
# ============================================================

# ------------------------------------------------------------
# Safety verification
# ------------------------------------------------------------

assert_project_sentinel()

# ------------------------------------------------------------
# Planned package sets
# ------------------------------------------------------------

CRAN_PACKAGES <- c(
  "glmnet",
  "Matrix",
  "data.table",
  "dplyr",
  "tidyr",
  "tibble",
  "purrr",
  "ggplot2",
  "pROC",
  "rsample",
  "yardstick",
  "boot",
  "metafor",
  "jsonlite",
  "digest",
  "arrow",
  "qs",
  "here",
  "renv",
  "patchwork",
  "msigdbr"
)

BIOC_PACKAGES <- c(
  "SummarizedExperiment",
  "edgeR",
  "limma",
  "fgsea",
  "AnnotationDbi",
  "org.Hs.eg.db"
)

AUXILIARY_PACKAGES <- c(
  "IRkernel"
)

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

check_package <- function(pkg, source_group) {

  available <- requireNamespace(
    pkg,
    quietly = TRUE
  )

  version <- if (available) {
    as.character(
      packageVersion(pkg)
    )
  } else {
    NA_character_
  }

  data.frame(
    package = pkg,
    source_group = source_group,
    available = available,
    version = version,
    stringsAsFactors = FALSE
  )
}

# ------------------------------------------------------------
# Build audit table
# ------------------------------------------------------------

PACKAGE_AUDIT <- do.call(
  rbind,
  c(
    lapply(
      CRAN_PACKAGES,
      check_package,
      source_group = "CRAN"
    ),
    lapply(
      BIOC_PACKAGES,
      check_package,
      source_group = "Bioconductor"
    ),
    lapply(
      AUXILIARY_PACKAGES,
      check_package,
      source_group = "Auxiliary"
    )
  )
)

PACKAGE_AUDIT <- PACKAGE_AUDIT[
  order(
    PACKAGE_AUDIT$source_group,
    PACKAGE_AUDIT$package
  ),
]

rownames(PACKAGE_AUDIT) <- NULL

# ------------------------------------------------------------
# Save package inventory
# ------------------------------------------------------------

PACKAGE_AUDIT_PATH <- file.path(
  PROJECT_ROOT,
  "15_environment",
  "step00a_package_availability.csv"
)

assert_project_sentinel()

write.csv(
  PACKAGE_AUDIT,
  PACKAGE_AUDIT_PATH,
  row.names = FALSE,
  na = ""
)

# ------------------------------------------------------------
# Summary counts
# ------------------------------------------------------------

N_TOTAL_PACKAGES <- nrow(PACKAGE_AUDIT)

N_AVAILABLE_PACKAGES <- sum(
  PACKAGE_AUDIT$available
)

N_MISSING_PACKAGES <- sum(
  !PACKAGE_AUDIT$available
)

MISSING_PACKAGES <- PACKAGE_AUDIT$package[
  !PACKAGE_AUDIT$available
]

# ------------------------------------------------------------
# Log
# ------------------------------------------------------------

log_line(
  "PACKAGE_AUDIT_TOTAL=",
  N_TOTAL_PACKAGES
)

log_line(
  "PACKAGE_AUDIT_AVAILABLE=",
  N_AVAILABLE_PACKAGES
)

log_line(
  "PACKAGE_AUDIT_MISSING=",
  N_MISSING_PACKAGES
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00A — CELL 7\n")
cat("============================================================\n")

print(
  PACKAGE_AUDIT,
  row.names = FALSE
)

cat("\n------------------------------------------------------------\n")
cat("Total planned packages: ", N_TOTAL_PACKAGES, "\n", sep = "")
cat("Currently available:    ", N_AVAILABLE_PACKAGES, "\n", sep = "")
cat("Currently missing:      ", N_MISSING_PACKAGES, "\n", sep = "")
cat("Audit saved to:         ", PACKAGE_AUDIT_PATH, "\n", sep = "")
cat("------------------------------------------------------------\n")

if (N_MISSING_PACKAGES > 0L) {

  cat("\nMissing packages:\n")

  for (pkg in MISSING_PACKAGES) {
    cat(" - ", pkg, "\n", sep = "")
  }

  cat(
    "\nNOTE: Missing downstream analysis packages are acceptable in Step 00A.\n"
  )
}

if (!file.exists(PACKAGE_AUDIT_PATH)) {
  stop(
    "PACKAGE AUDIT FAIL: CSV output was not created."
  )
}

cat("============================================================\n")
cat("CELL 7 — PASS\n")

[2026-09-29 21:51:58 +0600] PACKAGE_AUDIT_TOTAL=28 
[2026-09-29 21:51:58 +0600] PACKAGE_AUDIT_AVAILABLE=6 
[2026-09-29 21:51:58 +0600] PACKAGE_AUDIT_MISSING=22 
STEP 00A — CELL 7
              package source_group available version
             IRkernel    Auxiliary      TRUE   1.3.2
        AnnotationDbi Bioconductor     FALSE    <NA>
                edgeR Bioconductor     FALSE    <NA>
                fgsea Bioconductor     FALSE    <NA>
                limma Bioconductor     FALSE    <NA>
         org.Hs.eg.db Bioconductor     FALSE    <NA>
 SummarizedExperiment Bioconductor     FALSE    <NA>
                arrow         CRAN     FALSE    <NA>
                 boot         CRAN      TRUE  1.3.32
           data.table         CRAN     FALSE    <NA>
               digest         CRAN      TRUE  0.6.39
                dplyr         CRAN     FALSE    <NA>
              ggplot2         CRAN     FALSE    <NA>
               glmnet         CRAN     FALSE    <NA>
                 here     

In [8]:
# ============================================================
# STEP 00A — CELL 8
# SAVE ENVIRONMENT + REPRODUCIBILITY RECORDS
# ============================================================

# ------------------------------------------------------------
# Safety verification
# ------------------------------------------------------------

assert_project_sentinel()

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

SCAFFOLD_PATH <- file.path(
  PROJECT_ROOT,
  "15_environment",
  "step00a_scaffold_inventory.csv"
)

SESSION_INFO_PATH <- file.path(
  PROJECT_ROOT,
  "15_environment",
  "step00a_sessionInfo.txt"
)

RENV_STATUS_PATH <- file.path(
  PROJECT_ROOT,
  "15_environment",
  "step00a_renv_status.txt"
)

ENV_REPORT_PATH <- file.path(
  PROJECT_ROOT,
  "15_environment",
  "step00a_environment_report.txt"
)

JUPYTER_INFO_PATH <- file.path(
  PROJECT_ROOT,
  "15_environment",
  "step00a_jupyter_info.txt"
)

# ------------------------------------------------------------
# 1. Scaffold inventory
# ------------------------------------------------------------

SCAFFOLD_INVENTORY <- data.frame(
  relative_path = REQUIRED_DIRS,
  absolute_path = file.path(
    PROJECT_ROOT,
    REQUIRED_DIRS
  ),
  exists = dir.exists(
    file.path(
      PROJECT_ROOT,
      REQUIRED_DIRS
    )
  ),
  stringsAsFactors = FALSE
)

assert_project_sentinel()

write.csv(
  SCAFFOLD_INVENTORY,
  SCAFFOLD_PATH,
  row.names = FALSE
)

if (!all(SCAFFOLD_INVENTORY$exists)) {
  stop(
    "ENVIRONMENT RECORD FAIL: One or more required folders are missing."
  )
}

# ------------------------------------------------------------
# 2. Save sessionInfo()
# ------------------------------------------------------------

assert_project_sentinel()

writeLines(
  capture.output(
    sessionInfo()
  ),
  SESSION_INFO_PATH,
  useBytes = TRUE
)

# ------------------------------------------------------------
# 3. Save renv status
# ------------------------------------------------------------

RENV_STATUS_TEXT <- capture.output(
  renv::status(
    project = PROJECT_ROOT
  )
)

assert_project_sentinel()

writeLines(
  RENV_STATUS_TEXT,
  RENV_STATUS_PATH,
  useBytes = TRUE
)

# ------------------------------------------------------------
# 4. Capture Jupyter information
# ------------------------------------------------------------

JUPYTER_VERSION <- tryCatch(
  system2(
    command = JUPYTER_PATH,
    args = "--version",
    stdout = TRUE,
    stderr = TRUE
  ),
  error = function(e) {
    paste0(
      "Could not query Jupyter version: ",
      conditionMessage(e)
    )
  }
)

JUPYTER_KERNELS <- tryCatch(
  system2(
    command = JUPYTER_PATH,
    args = c(
      "kernelspec",
      "list"
    ),
    stdout = TRUE,
    stderr = TRUE
  ),
  error = function(e) {
    paste0(
      "Could not query Jupyter kernels: ",
      conditionMessage(e)
    )
  }
)

JUPYTER_INFO_TEXT <- c(
  "SCZ-CRST STEP 00A — JUPYTER INFORMATION",
  "",
  paste0(
    "Jupyter executable: ",
    JUPYTER_PATH
  ),
  "",
  "Jupyter version:",
  JUPYTER_VERSION,
  "",
  "Registered kernels:",
  JUPYTER_KERNELS
)

assert_project_sentinel()

writeLines(
  JUPYTER_INFO_TEXT,
  JUPYTER_INFO_PATH,
  useBytes = TRUE
)

# ------------------------------------------------------------
# 5. Create main environment report
# ------------------------------------------------------------

ENV_REPORT <- c(
  "SCZ-CRST STEP 00A — ENVIRONMENT REPORT",
  "========================================",
  "",
  paste0(
    "timestamp: ",
    format(
      Sys.time(),
      "%Y-%m-%d %H:%M:%S %z"
    )
  ),
  "",
  paste0(
    "project_root: ",
    PROJECT_ROOT
  ),
  paste0(
    "project_id: ",
    PROJECT_ID
  ),
  paste0(
    "sentinel_path: ",
    SENTINEL_PATH
  ),
  paste0(
    "sentinel_verified: ",
    identical(
      trimws(
        paste(
          readLines(
            SENTINEL_PATH,
            warn = FALSE
          ),
          collapse = "\n"
        )
      ),
      PROJECT_ID
    )
  ),
  "",
  paste0(
    "r_version: ",
    R.version.string
  ),
  paste0(
    "r_home: ",
    R.home()
  ),
  paste0(
    "r_platform: ",
    R.version$platform
  ),
  paste0(
    "r_version_requirement_ge_4.4.0: ",
    getRversion() >= MIN_R_VERSION
  ),
  "",
  paste0(
    "os_type: ",
    .Platform$OS.type
  ),
  paste0(
    "machine: ",
    Sys.info()[["machine"]]
  ),
  "",
  paste0(
    "r_executable: ",
    R_PATH
  ),
  paste0(
    "rscript_executable: ",
    RSCRIPT_PATH
  ),
  paste0(
    "jupyter_executable: ",
    JUPYTER_PATH
  ),
  paste0(
    "python_executable: ",
    ifelse(
      nzchar(PYTHON_PATH),
      PYTHON_PATH,
      "NOT_ON_PATH"
    )
  ),
  paste0(
    "git_executable: ",
    ifelse(
      nzchar(GIT_PATH),
      GIT_PATH,
      "NOT_ON_PATH"
    )
  ),
  "",
  paste0(
    "irkernel_available: ",
    IRKERNEL_AVAILABLE
  ),
  paste0(
    "renv_version: ",
    RENV_VERSION
  ),
  paste0(
    "renv_directory_exists: ",
    dir.exists(RENV_DIR)
  ),
  paste0(
    "renv_lock_exists: ",
    file.exists(RENV_LOCK)
  ),
  paste0(
    "rprofile_exists: ",
    file.exists(RPROFILE_PATH)
  ),
  paste0(
    "project_library: ",
    .libPaths()[1]
  ),
  "",
  paste0(
    "required_directories_total: ",
    nrow(SCAFFOLD_INVENTORY)
  ),
  paste0(
    "required_directories_present: ",
    sum(SCAFFOLD_INVENTORY$exists)
  ),
  paste0(
    "folder_structure_complete: ",
    all(SCAFFOLD_INVENTORY$exists)
  ),
  "",
  paste0(
    "planned_packages_total: ",
    N_TOTAL_PACKAGES
  ),
  paste0(
    "planned_packages_available: ",
    N_AVAILABLE_PACKAGES
  ),
  paste0(
    "planned_packages_missing: ",
    N_MISSING_PACKAGES
  ),
  "",
  "biological_data_loaded: FALSE",
  "previous_schizophrenia_project_used: FALSE",
  "scientific_analysis_performed: FALSE",
  "",
  "NOTE:",
  "Missing downstream analysis packages are not a Step 00A failure.",
  "They will be installed deliberately inside the locked renv environment",
  "only when required by subsequent analysis steps."
)

assert_project_sentinel()

writeLines(
  ENV_REPORT,
  ENV_REPORT_PATH,
  useBytes = TRUE
)

# ------------------------------------------------------------
# 6. Validate all saved records
# ------------------------------------------------------------

RECORD_CHECKS <- c(
  scaffold_inventory = file.exists(SCAFFOLD_PATH),
  session_info = file.exists(SESSION_INFO_PATH),
  renv_status = file.exists(RENV_STATUS_PATH),
  jupyter_info = file.exists(JUPYTER_INFO_PATH),
  environment_report = file.exists(ENV_REPORT_PATH)
)

if (!all(RECORD_CHECKS)) {

  failed <- names(RECORD_CHECKS)[
    !RECORD_CHECKS
  ]

  stop(
    paste0(
      "ENVIRONMENT RECORD FAIL.\n",
      "Missing outputs:\n- ",
      paste(
        failed,
        collapse = "\n- "
      )
    )
  )
}

# ------------------------------------------------------------
# Log completion
# ------------------------------------------------------------

log_line(
  "SCAFFOLD_INVENTORY_WRITTEN=PASS"
)

log_line(
  "SESSION_INFO_WRITTEN=PASS"
)

log_line(
  "RENV_STATUS_WRITTEN=PASS"
)

log_line(
  "JUPYTER_INFO_WRITTEN=PASS"
)

log_line(
  "ENVIRONMENT_REPORT_WRITTEN=PASS"
)

# ------------------------------------------------------------
# Display compact summary
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00A — CELL 8\n")
cat("============================================================\n")

cat(
  "Scaffold inventory: ",
  RECORD_CHECKS["scaffold_inventory"],
  "\n",
  sep = ""
)

cat(
  "Session info:       ",
  RECORD_CHECKS["session_info"],
  "\n",
  sep = ""
)

cat(
  "renv status:        ",
  RECORD_CHECKS["renv_status"],
  "\n",
  sep = ""
)

cat(
  "Jupyter info:       ",
  RECORD_CHECKS["jupyter_info"],
  "\n",
  sep = ""
)

cat(
  "Environment report:",
  " ",
  RECORD_CHECKS["environment_report"],
  "\n",
  sep = ""
)

cat("\nEnvironment report:\n")
cat(ENV_REPORT_PATH, "\n")

cat("============================================================\n")
cat("CELL 8 — PASS\n")

[2026-09-29 21:53:32 +0600] SCAFFOLD_INVENTORY_WRITTEN=PASS 
[2026-09-29 21:53:32 +0600] SESSION_INFO_WRITTEN=PASS 
[2026-09-29 21:53:32 +0600] RENV_STATUS_WRITTEN=PASS 
[2026-09-29 21:53:32 +0600] JUPYTER_INFO_WRITTEN=PASS 
[2026-09-29 21:53:32 +0600] ENVIRONMENT_REPORT_WRITTEN=PASS 
STEP 00A — CELL 8
Scaffold inventory: TRUE
Session info:       TRUE
renv status:        TRUE
Jupyter info:       TRUE
Environment report: TRUE

Environment report:
E:/SCZ_CROSSREGION_MSS_Q1/15_environment/step00a_environment_report.txt 
CELL 8 — PASS


In [9]:
# ============================================================
# STEP 00A — CELL 9
# FINAL VALIDATION + STEP 00A FREEZE
# ============================================================

# ------------------------------------------------------------
# Safety verification
# ------------------------------------------------------------

assert_project_sentinel()

# ------------------------------------------------------------
# Re-read sentinel independently
# ------------------------------------------------------------

FINAL_SENTINEL_ID <- trimws(
  paste(
    readLines(
      SENTINEL_PATH,
      warn = FALSE
    ),
    collapse = "\n"
  )
)

# ------------------------------------------------------------
# Final validation checks
# ------------------------------------------------------------

FINAL_CHECKS <- c(

  correct_project_root =
    identical(
      clean_path(PROJECT_ROOT),
      clean_path("E:/SCZ_CROSSREGION_MSS_Q1")
    ),

  sentinel_exists =
    file.exists(SENTINEL_PATH),

  sentinel_matches =
    identical(
      FINAL_SENTINEL_ID,
      PROJECT_ID
    ),

  r_version_valid =
    getRversion() >= MIN_R_VERSION,

  windows_environment =
    identical(
      .Platform$OS.type,
      "windows"
    ),

  e_drive_available =
    dir.exists("E:/"),

  all_required_directories =
    all(
      dir.exists(
        file.path(
          PROJECT_ROOT,
          REQUIRED_DIRS
        )
      )
    ),

  renv_directory =
    dir.exists(RENV_DIR),

  renv_lock =
    file.exists(RENV_LOCK),

  rprofile =
    file.exists(RPROFILE_PATH),

  project_library_active =
    grepl(
      clean_path(PROJECT_ROOT),
      clean_path(.libPaths()[1]),
      fixed = TRUE
    ),

  package_audit =
    file.exists(PACKAGE_AUDIT_PATH),

  scaffold_inventory =
    file.exists(SCAFFOLD_PATH),

  session_info =
    file.exists(SESSION_INFO_PATH),

  renv_status =
    file.exists(RENV_STATUS_PATH),

  jupyter_info =
    file.exists(JUPYTER_INFO_PATH),

  environment_report =
    file.exists(ENV_REPORT_PATH),

  log_file =
    file.exists(LOG_PATH)
)

# ------------------------------------------------------------
# Create validation table
# ------------------------------------------------------------

FINAL_VALIDATION_TABLE <- data.frame(
  check = names(FINAL_CHECKS),
  passed = as.logical(FINAL_CHECKS),
  stringsAsFactors = FALSE
)

FINAL_VALIDATION_PATH <- file.path(
  PROJECT_ROOT,
  "15_environment",
  "step00a_final_validation.csv"
)

assert_project_sentinel()

write.csv(
  FINAL_VALIDATION_TABLE,
  FINAL_VALIDATION_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# Fail loudly if anything is wrong
# ------------------------------------------------------------

if (!all(FINAL_CHECKS)) {

  FAILED_CHECKS <- names(FINAL_CHECKS)[
    !FINAL_CHECKS
  ]

  log_line(
    "STEP_00A_FINAL_STATUS=FAIL"
  )

  log_line(
    "FAILED_CHECKS=",
    paste(
      FAILED_CHECKS,
      collapse = ","
    )
  )

  print(
    FINAL_VALIDATION_TABLE,
    row.names = FALSE
  )

  stop(
    paste0(
      "STEP 00A FINAL VALIDATION FAILED.\n\n",
      "Failed checks:\n- ",
      paste(
        FAILED_CHECKS,
        collapse = "\n- "
      )
    )
  )
}

# ------------------------------------------------------------
# Write explicit PASS record
# ------------------------------------------------------------

STEP00A_PASS_PATH <- file.path(
  PROJECT_ROOT,
  "15_environment",
  "STEP_00A_PASS.txt"
)

PASS_TEXT <- c(
  "SCZ-CRST STEP 00A — PASS",
  paste0(
    "timestamp: ",
    format(
      Sys.time(),
      "%Y-%m-%d %H:%M:%S %z"
    )
  ),
  paste0(
    "project_root: ",
    PROJECT_ROOT
  ),
  paste0(
    "project_id: ",
    PROJECT_ID
  ),
  paste0(
    "R_version: ",
    R.version.string
  ),
  paste0(
    "renv_version: ",
    RENV_VERSION
  ),
  paste0(
    "required_directories: ",
    length(REQUIRED_DIRS)
  ),
  paste0(
    "planned_packages_available: ",
    N_AVAILABLE_PACKAGES,
    "/",
    N_TOTAL_PACKAGES
  ),
  "biological_data_loaded: FALSE",
  "previous_schizophrenia_project_used: FALSE",
  "scientific_analysis_performed: FALSE"
)

assert_project_sentinel()

writeLines(
  PASS_TEXT,
  STEP00A_PASS_PATH,
  useBytes = TRUE
)

# ------------------------------------------------------------
# Final log entries
# ------------------------------------------------------------

log_line(
  "FINAL_VALIDATION_FILE=",
  FINAL_VALIDATION_PATH
)

log_line(
  "STEP00A_PASS_FILE=",
  STEP00A_PASS_PATH
)

log_line(
  "STEP_00A_FINAL_STATUS=PASS"
)

# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00A — FINAL VALIDATION\n")
cat("============================================================\n")

print(
  FINAL_VALIDATION_TABLE,
  row.names = FALSE
)

cat("\n")
cat("Required directories:       ", length(REQUIRED_DIRS), "\n", sep = "")
cat("Missing directories:        0\n")
cat("Planned packages available: ", N_AVAILABLE_PACKAGES, "/", N_TOTAL_PACKAGES, "\n", sep = "")
cat("Planned packages missing:   ", N_MISSING_PACKAGES, "\n", sep = "")
cat("Biological data loaded:     FALSE\n")
cat("Previous project used:      FALSE\n")
cat("Scientific analysis done:   FALSE\n")

cat("\n============================================================\n")
cat("STEP 00A — PASS\n")
cat("============================================================\n")
cat("Project root:      ", PROJECT_ROOT, "\n", sep = "")
cat("Project ID:        ", PROJECT_ID, "\n", sep = "")
cat("R version:         ", R.version.string, "\n", sep = "")
cat("renv version:      ", RENV_VERSION, "\n", sep = "")
cat("Environment report:", " ", ENV_REPORT_PATH, "\n", sep = "")
cat("Validation table:  ", FINAL_VALIDATION_PATH, "\n", sep = "")
cat("PASS record:       ", STEP00A_PASS_PATH, "\n", sep = "")
cat("Log file:          ", LOG_PATH, "\n", sep = "")
cat("============================================================\n")

[2026-09-29 21:54:57 +0600] FINAL_VALIDATION_FILE=E:/SCZ_CROSSREGION_MSS_Q1/15_environment/step00a_final_validation.csv 
[2026-09-29 21:54:57 +0600] STEP00A_PASS_FILE=E:/SCZ_CROSSREGION_MSS_Q1/15_environment/STEP_00A_PASS.txt 
[2026-09-29 21:54:57 +0600] STEP_00A_FINAL_STATUS=PASS 
STEP 00A — FINAL VALIDATION
                    check passed
     correct_project_root   TRUE
          sentinel_exists   TRUE
         sentinel_matches   TRUE
          r_version_valid   TRUE
      windows_environment   TRUE
        e_drive_available   TRUE
 all_required_directories   TRUE
           renv_directory   TRUE
                renv_lock   TRUE
                 rprofile   TRUE
   project_library_active   TRUE
            package_audit   TRUE
       scaffold_inventory   TRUE
             session_info   TRUE
              renv_status   TRUE
             jupyter_info   TRUE
       environment_report   TRUE
                 log_file   TRUE

Required directories:       29
Missing directories:        0


In [10]:
# ============================================================
# STEP 00B — CELL 1
# VERIFIED LITERATURE + DATA-SOURCE REGISTRY
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# Step 00A must have passed
# ------------------------------------------------------------

STEP00A_PASS_PATH <- file.path(
  PROJECT_ROOT,
  "15_environment",
  "STEP_00A_PASS.txt"
)

if (!file.exists(STEP00A_PASS_PATH)) {
  stop(
    "STEP 00B STOP: STEP_00A_PASS.txt is missing."
  )
}

# ------------------------------------------------------------
# Canonical verified-source registry
# ------------------------------------------------------------

SOURCE_REGISTRY <- data.frame(

  source_id = c(
    "BSP2_PRIMARY",
    "BSP2_OFFICIAL_REPO",
    "CMC_PRIMARY",
    "GSE80655_GEO",
    "SCZ_MTL_2018",
    "SCZ_EXPR_SCORE_2023"
  ),

  category = c(
    "Discovery dataset / primary publication",
    "Discovery dataset / official code and processed resources",
    "External dataset / primary publication",
    "Secondary external dataset / official repository",
    "Prior schizophrenia transcriptomic machine learning",
    "Prior schizophrenia transcriptomic score"
  ),

  citation = c(
    paste0(
      "Collado-Torres L et al. Regional Heterogeneity in Gene Expression, ",
      "Regulation, and Coherence in the Frontal Cortex and Hippocampus ",
      "across Development and Schizophrenia. Neuron. 2019."
    ),

    "Lieber Institute. BrainSeq Phase II official analysis repository.",

    paste0(
      "Hoffman GE et al. CommonMind Consortium provides transcriptomic ",
      "and epigenomic data for Schizophrenia and Bipolar Disorder. ",
      "Scientific Data. 2019."
    ),

    "NCBI GEO Series GSE80655: RNA-sequencing of human post-mortem brain tissues.",

    paste0(
      "Schwarz E et al. Comparative Evaluation of Machine Learning ",
      "Strategies for Analyzing Big Data in Psychiatry. 2018."
    ),

    paste0(
      "Polygenic scores for psychiatric disorders in a diverse ",
      "postmortem brain tissue cohort. 2023."
    )
  ),

  identifier = c(
    "PMID:31174959; DOI:10.1016/j.neuron.2019.05.013",
    "GitHub:LieberInstitute/brainseq_phase2",
    "PMID:31551426; DOI:10.1038/s41597-019-0183-6",
    "GEO:GSE80655; BioProject:PRJNA319583",
    "PMID:30380679",
    "PMID:36694041"
  ),

  url = c(
    "https://pubmed.ncbi.nlm.nih.gov/31174959/",
    "https://github.com/LieberInstitute/brainseq_phase2",
    "https://pubmed.ncbi.nlm.nih.gov/31551426/",
    "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE80655",
    "https://pubmed.ncbi.nlm.nih.gov/30380679/",
    "https://pubmed.ncbi.nlm.nih.gov/36694041/"
  ),

  verified_fact = c(
    paste0(
      "BrainSeq Phase II contains DLPFC and hippocampus RNA-seq from ",
      "551 individuals; the study reports 900 RNA-seq samples overall ",
      "and explicitly analyzes regional schizophrenia heterogeneity. ",
      "The publication also documents partial overlap of its DLPFC ",
      "samples with the second-phase CMC NIMH-HBCC cohort."
    ),

    paste0(
      "Official BrainSeq Phase II repository documents processed ",
      "RangedSummarizedExperiment resources, gene-level expression, ",
      "SCZ case-control resources, region analyses, qSVA resources, ",
      "and public/raw-data access routes."
    ),

    paste0(
      "CMC contains DLPFC RNA-seq from four brain banks. ",
      "MSSM, Penn and Pitt form the original CMC cohort, while ",
      "NIMH-HBCC is a separate CMC_HBCC cohort."
    ),

    paste0(
      "GSE80655 is an official GEO RNA-seq series containing 281 ",
      "clinically annotated human postmortem brain tissue samples. ",
      "Processed gene-expression data are available at the series level."
    ),

    paste0(
      "Multi-task learning has already been evaluated across multiple ",
      "schizophrenia transcriptomic datasets; therefore generic MTL ",
      "cannot be claimed as the novelty of SCZ-CRST."
    ),

    paste0(
      "An aggregate schizophrenia gene-expression score has previously ",
      "been evaluated in postmortem DLPFC; therefore construction of a ",
      "transcriptomic score alone is not novel."
    )
  ),

  implication_for_SCZ_CRST = c(
    paste0(
      "BrainSeq Phase II is suitable as the two-region discovery resource. ",
      "CMC HBCC must not be treated as independent external validation."
    ),

    paste0(
      "Prefer processed gene-level BrainSeq resources over FASTQ unless ",
      "later audit demonstrates that processed data are inadequate."
    ),

    paste0(
      "Primary CMC external validation should use eligible MSSM/Penn/Pitt ",
      "samples and exclude HBCC after explicit metadata/overlap audit."
    ),

    paste0(
      "Use DLPFC as prespecified secondary replication tissue and derive ",
      "exact SCZ/CTL counts directly from GEO metadata."
    ),

    paste0(
      "Novelty must depend on the integrated donor-aware shared/deviation, ",
      "stability, locking and external-transfer framework rather than MTL."
    ),

    paste0(
      "Novelty must not be framed as merely deriving a molecular or ",
      "gene-expression score."
    )
  ),

  verification_status = rep(
    "VERIFIED_FROM_PRIMARY_OR_OFFICIAL_SOURCE",
    6
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

stopifnot(
  nrow(SOURCE_REGISTRY) == 6L,
  !anyDuplicated(SOURCE_REGISTRY$source_id),
  all(nzchar(SOURCE_REGISTRY$url)),
  all(nzchar(SOURCE_REGISTRY$verified_fact)),
  all(nzchar(SOURCE_REGISTRY$implication_for_SCZ_CRST))
)

# ------------------------------------------------------------
# Write canonical registry
# ------------------------------------------------------------

SOURCE_REGISTRY_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Verified_Source_Registry_v1.csv"
)

assert_project_sentinel()

write.csv(
  SOURCE_REGISTRY,
  SOURCE_REGISTRY_PATH,
  row.names = FALSE,
  na = ""
)

# ------------------------------------------------------------
# Verify output
# ------------------------------------------------------------

if (!file.exists(SOURCE_REGISTRY_PATH)) {
  stop(
    "STEP 00B CELL 1 FAIL: Source registry was not created."
  )
}

CHECK_REGISTRY <- read.csv(
  SOURCE_REGISTRY_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

if (nrow(CHECK_REGISTRY) != nrow(SOURCE_REGISTRY)) {
  stop(
    "STEP 00B CELL 1 FAIL: Written registry row count mismatch."
  )
}

if (!identical(
  CHECK_REGISTRY$source_id,
  SOURCE_REGISTRY$source_id
)) {
  stop(
    "STEP 00B CELL 1 FAIL: Source IDs changed after write/read."
  )
}

# ------------------------------------------------------------
# Log
# ------------------------------------------------------------

log_line(
  "STEP00B_VERIFIED_SOURCE_REGISTRY_ROWS=",
  nrow(SOURCE_REGISTRY)
)

log_line(
  "STEP00B_VERIFIED_SOURCE_REGISTRY_PATH=",
  SOURCE_REGISTRY_PATH
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00B — CELL 1\n")
cat("============================================================\n")
cat("Verified sources: ", nrow(SOURCE_REGISTRY), "\n", sep = "")
cat("BrainSeq primary source: TRUE\n")
cat("BrainSeq official repository: TRUE\n")
cat("CMC primary source: TRUE\n")
cat("GSE80655 official GEO source: TRUE\n")
cat("Prior SCZ MTL precedent recorded: TRUE\n")
cat("Prior SCZ expression-score precedent recorded: TRUE\n")
cat("Registry: ", SOURCE_REGISTRY_PATH, "\n", sep = "")
cat("============================================================\n")
cat("CELL 1 — PASS\n")

[2026-09-29 21:57:24 +0600] STEP00B_VERIFIED_SOURCE_REGISTRY_ROWS=6 
[2026-09-29 21:57:24 +0600] STEP00B_VERIFIED_SOURCE_REGISTRY_PATH=E:/SCZ_CROSSREGION_MSS_Q1/00_protocol/STEP00B_Verified_Source_Registry_v1.csv 
STEP 00B — CELL 1
Verified sources: 6
BrainSeq primary source: TRUE
BrainSeq official repository: TRUE
CMC primary source: TRUE
GSE80655 official GEO source: TRUE
Prior SCZ MTL precedent recorded: TRUE
Prior SCZ expression-score precedent recorded: TRUE
Registry: E:/SCZ_CROSSREGION_MSS_Q1/00_protocol/STEP00B_Verified_Source_Registry_v1.csv
CELL 1 — PASS


In [11]:
# ============================================================
# STEP 00B — CELL 2
# DATASET STRUCTURE + INDEPENDENCE / OVERLAP AUDIT
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# Require Cell 1 output
# ------------------------------------------------------------

SOURCE_REGISTRY_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Verified_Source_Registry_v1.csv"
)

if (!file.exists(SOURCE_REGISTRY_PATH)) {
  stop(
    "STEP 00B CELL 2 STOP: Cell 1 source registry is missing."
  )
}

SOURCE_REGISTRY_CHECK <- read.csv(
  SOURCE_REGISTRY_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

if (nrow(SOURCE_REGISTRY_CHECK) != 6L) {
  stop(
    "STEP 00B CELL 2 STOP: Source registry does not contain expected 6 rows."
  )
}

# ------------------------------------------------------------
# Prespecified dataset-role and independence audit
# ------------------------------------------------------------

DATASET_AUDIT <- data.frame(

  resource = c(
    "BrainSeq Phase II DLPFC",
    "BrainSeq Phase II HIPPO",
    "CMC MSSM",
    "CMC Penn",
    "CMC Pitt",
    "CMC HBCC",
    "GSE80655 DLPFC",
    "GSE80655 AnCg",
    "GSE80655 nAcc"
  ),

  parent_dataset = c(
    "BrainSeq Phase II",
    "BrainSeq Phase II",
    "CommonMind Consortium",
    "CommonMind Consortium",
    "CommonMind Consortium",
    "CommonMind Consortium",
    "GSE80655",
    "GSE80655",
    "GSE80655"
  ),

  brain_region = c(
    "DLPFC",
    "HIPPO",
    "DLPFC",
    "DLPFC",
    "DLPFC",
    "DLPFC",
    "DLPFC",
    "Anterior cingulate",
    "Nucleus accumbens"
  ),

  planned_role = c(
    "Discovery / development task 1",
    "Discovery / development task 2",
    "Primary external validation site",
    "Primary external validation site",
    "Primary external validation site",
    "Excluded from independent external validation",
    "Secondary external replication",
    "Optional exploratory regional generalization",
    "Optional exploratory regional generalization"
  ),

  independent_external = c(
    FALSE,
    FALSE,
    TRUE,
    TRUE,
    TRUE,
    FALSE,
    TRUE,
    FALSE,
    FALSE
  ),

  overlap_status = c(
    "DISCOVERY_RESOURCE",
    "DISCOVERY_RESOURCE",
    "No documented BrainSeq overlap identified in current source audit; must verify again from metadata/documentation before outcome analysis",
    "No documented BrainSeq overlap identified in current source audit; must verify again from metadata/documentation before outcome analysis",
    "No documented BrainSeq overlap identified in current source audit; must verify again from metadata/documentation before outcome analysis",
    "DOCUMENTED_PARTIAL_OVERLAP_WITH_BRAINSEQ_DLPFC",
    "No known overlap identified in current audit; exact sample identity must still be checked",
    "Same parent dataset as GSE80655 DLPFC; exploratory only",
    "Same parent dataset as GSE80655 DLPFC; exploratory only"
  ),

  primary_decision = c(
    "KEEP",
    "KEEP",
    "KEEP_PENDING_METADATA_AUDIT",
    "KEEP_PENDING_METADATA_AUDIT",
    "KEEP_PENDING_METADATA_AUDIT",
    "EXCLUDE",
    "KEEP_PENDING_METADATA_AUDIT",
    "EXPLORATORY_ONLY",
    "EXPLORATORY_ONLY"
  ),

  diagnosis_scope = c(
    "SCZ + eligible neurotypical controls; other psychiatric diagnoses excluded from primary analysis",
    "SCZ + eligible neurotypical controls; other psychiatric diagnoses excluded from primary analysis",
    "SCZ + eligible neurotypical controls",
    "SCZ + eligible neurotypical controls",
    "SCZ + eligible neurotypical controls",
    "Not eligible for independent external outcome analysis",
    "SCZ + eligible neurotypical controls",
    "Exploratory only",
    "Exploratory only"
  ),

  exact_sample_counts = rep(
    "TO_BE_VERIFIED_FROM_METADATA",
    9
  ),

  exact_donor_counts = rep(
    "TO_BE_VERIFIED_FROM_METADATA",
    9
  ),

  exact_overlap_check_required = c(
    TRUE,
    TRUE,
    TRUE,
    TRUE,
    TRUE,
    TRUE,
    TRUE,
    FALSE,
    FALSE
  ),

  notes = c(
    "Do not treat DLPFC as an independent dataset from HIPPO; both belong to one discovery resource and donors may contribute both regions.",
    "Do not treat HIPPO as an independent dataset from DLPFC; paired-donor structure must be explicitly audited.",
    "Retain only after later CMC metadata audit confirms site identity, diagnosis eligibility, duplicates, covariates and no prohibited overlap.",
    "Retain only after later CMC metadata audit confirms site identity, diagnosis eligibility, duplicates, covariates and no prohibited overlap.",
    "Retain only after later CMC metadata audit confirms site identity, diagnosis eligibility, duplicates, covariates and no prohibited overlap.",
    "Must not be used as independent external validation because BrainSeq Phase II has documented partial overlap with CMC/HBCC.",
    "Prespecified secondary replication tissue. Exact SCZ/CTL sample counts must be reconstructed directly from GEO metadata.",
    "May be used only after primary DLPFC replication is complete and must remain explicitly exploratory.",
    "May be used only after primary DLPFC replication is complete and must remain explicitly exploratory."
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

stopifnot(
  nrow(DATASET_AUDIT) == 9L,
  !anyDuplicated(DATASET_AUDIT$resource)
)

# HBCC must be excluded.
HBCC_ROW <- DATASET_AUDIT[
  DATASET_AUDIT$resource == "CMC HBCC",
]

if (nrow(HBCC_ROW) != 1L) {
  stop(
    "STEP 00B CELL 2 FAIL: CMC HBCC row missing or duplicated."
  )
}

if (!identical(
  HBCC_ROW$primary_decision,
  "EXCLUDE"
)) {
  stop(
    "STEP 00B CELL 2 FAIL: CMC HBCC is not marked EXCLUDE."
  )
}

if (!identical(
  HBCC_ROW$independent_external,
  FALSE
)) {
  stop(
    "STEP 00B CELL 2 FAIL: CMC HBCC incorrectly marked independent."
  )
}

# BrainSeq regions must not be treated as independent external cohorts.
BRAINSEQ_ROWS <- DATASET_AUDIT[
  DATASET_AUDIT$parent_dataset == "BrainSeq Phase II",
]

if (any(BRAINSEQ_ROWS$independent_external)) {
  stop(
    "STEP 00B CELL 2 FAIL: BrainSeq regional tasks were incorrectly marked as independent external cohorts."
  )
}

# Primary external CMC sites.
CMC_PRIMARY_SITES <- c(
  "CMC MSSM",
  "CMC Penn",
  "CMC Pitt"
)

CMC_PRIMARY_ROWS <- DATASET_AUDIT[
  DATASET_AUDIT$resource %in% CMC_PRIMARY_SITES,
]

if (
  nrow(CMC_PRIMARY_ROWS) != 3L ||
  !all(CMC_PRIMARY_ROWS$independent_external)
) {
  stop(
    "STEP 00B CELL 2 FAIL: MSSM/Penn/Pitt external-site structure is incorrect."
  )
}

# GSE80655 DLPFC must be secondary external replication.
GSE_DLPFC_ROW <- DATASET_AUDIT[
  DATASET_AUDIT$resource == "GSE80655 DLPFC",
]

if (
  nrow(GSE_DLPFC_ROW) != 1L ||
  !identical(
    GSE_DLPFC_ROW$planned_role,
    "Secondary external replication"
  )
) {
  stop(
    "STEP 00B CELL 2 FAIL: GSE80655 DLPFC role is incorrect."
  )
}

# ------------------------------------------------------------
# Save audit
# ------------------------------------------------------------

DATASET_AUDIT_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Dataset_Independence_Audit_v1.csv"
)

assert_project_sentinel()

write.csv(
  DATASET_AUDIT,
  DATASET_AUDIT_PATH,
  row.names = FALSE,
  na = ""
)

# ------------------------------------------------------------
# Create compact decision summary
# ------------------------------------------------------------

DECISION_SUMMARY <- c(
  "SCZ-CRST STEP 00B — DATASET / INDEPENDENCE DECISIONS",
  "=====================================================",
  "",
  "DISCOVERY:",
  "- BrainSeq Phase II DLPFC",
  "- BrainSeq Phase II HIPPO",
  "- These are two related regional tasks from one discovery resource.",
  "- They are NOT independent external datasets.",
  "- Paired donors must remain grouped throughout validation.",
  "",
  "PRIMARY EXTERNAL VALIDATION:",
  "- CMC MSSM",
  "- CMC Penn",
  "- CMC Pitt",
  "- Exact site/sample/donor counts remain unverified until metadata audit.",
  "",
  "EXCLUDED FROM INDEPENDENT VALIDATION:",
  "- CMC HBCC",
  "- Reason: documented partial overlap with BrainSeq Phase II DLPFC.",
  "",
  "SECONDARY EXTERNAL REPLICATION:",
  "- GSE80655 DLPFC",
  "- Exact SCZ/control counts must be reconstructed from GEO metadata.",
  "",
  "OPTIONAL EXPLORATORY ONLY:",
  "- GSE80655 anterior cingulate",
  "- GSE80655 nucleus accumbens",
  "",
  "IMPORTANT:",
  "- No sample counts are frozen in Step 00B.",
  "- No cohort is declared independent solely because accession names differ.",
  "- Final independence status requires later metadata-level overlap audit."
)

DATASET_SUMMARY_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Dataset_Independence_Summary_v1.txt"
)

assert_project_sentinel()

writeLines(
  DECISION_SUMMARY,
  DATASET_SUMMARY_PATH,
  useBytes = TRUE
)

# ------------------------------------------------------------
# Validate written files
# ------------------------------------------------------------

if (!file.exists(DATASET_AUDIT_PATH)) {
  stop(
    "STEP 00B CELL 2 FAIL: Dataset audit CSV was not created."
  )
}

if (!file.exists(DATASET_SUMMARY_PATH)) {
  stop(
    "STEP 00B CELL 2 FAIL: Dataset decision summary was not created."
  )
}

CHECK_DATASET_AUDIT <- read.csv(
  DATASET_AUDIT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

if (nrow(CHECK_DATASET_AUDIT) != 9L) {
  stop(
    "STEP 00B CELL 2 FAIL: Written dataset audit row count mismatch."
  )
}

# ------------------------------------------------------------
# Log
# ------------------------------------------------------------

log_line(
  "STEP00B_DATASET_AUDIT_ROWS=",
  nrow(DATASET_AUDIT)
)

log_line(
  "STEP00B_HBCC_DECISION=EXCLUDE"
)

log_line(
  "STEP00B_PRIMARY_CMC_SITES=MSSM,Penn,Pitt"
)

log_line(
  "STEP00B_SECONDARY_REPLICATION=GSE80655_DLPFC"
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00B — CELL 2\n")
cat("============================================================\n")
cat("Dataset/resource rows:            ", nrow(DATASET_AUDIT), "\n", sep = "")
cat("BrainSeq discovery regions:       2\n")
cat("Primary CMC external sites:       3\n")
cat("CMC HBCC excluded:                TRUE\n")
cat("GSE80655 DLPFC secondary:         TRUE\n")
cat("Exact sample counts frozen now:   FALSE\n")
cat("Metadata overlap audit required:  TRUE\n")
cat("Audit CSV: ", DATASET_AUDIT_PATH, "\n", sep = "")
cat("Summary:   ", DATASET_SUMMARY_PATH, "\n", sep = "")
cat("============================================================\n")
cat("CELL 2 — PASS\n")

[2026-09-29 21:59:19 +0600] STEP00B_DATASET_AUDIT_ROWS=9 
[2026-09-29 21:59:19 +0600] STEP00B_HBCC_DECISION=EXCLUDE 
[2026-09-29 21:59:19 +0600] STEP00B_PRIMARY_CMC_SITES=MSSM,Penn,Pitt 
[2026-09-29 21:59:19 +0600] STEP00B_SECONDARY_REPLICATION=GSE80655_DLPFC 
STEP 00B — CELL 2
Dataset/resource rows:            9
BrainSeq discovery regions:       2
Primary CMC external sites:       3
CMC HBCC excluded:                TRUE
GSE80655 DLPFC secondary:         TRUE
Exact sample counts frozen now:   FALSE
Metadata overlap audit required:  TRUE
Audit CSV: E:/SCZ_CROSSREGION_MSS_Q1/00_protocol/STEP00B_Dataset_Independence_Audit_v1.csv
Summary:   E:/SCZ_CROSSREGION_MSS_Q1/00_protocol/STEP00B_Dataset_Independence_Summary_v1.txt
CELL 2 — PASS


In [12]:
# ============================================================
# STEP 00B — CELL 3
# NOVELTY MATRIX + MATERIAL SCIENTIFIC CONFLICT AUDIT
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# Require previous Step 00B outputs
# ------------------------------------------------------------

SOURCE_REGISTRY_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Verified_Source_Registry_v1.csv"
)

DATASET_AUDIT_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Dataset_Independence_Audit_v1.csv"
)

if (!file.exists(SOURCE_REGISTRY_PATH)) {
  stop(
    "STEP 00B CELL 3 STOP: Verified source registry is missing."
  )
}

if (!file.exists(DATASET_AUDIT_PATH)) {
  stop(
    "STEP 00B CELL 3 STOP: Dataset independence audit is missing."
  )
}

# ------------------------------------------------------------
# Novelty matrix
#
# IMPORTANT:
# "Not established" means that the reviewed source did not
# establish that feature as part of its primary framework.
# It does NOT mean a universal literature-wide absence.
# ------------------------------------------------------------

NOVELTY_MATRIX <- data.frame(

  study_id = c(
    "BrainSeq_PhaseII_2019",
    "SCZ_MTL_2018",
    "HBCC_ExpressionScore_2023",
    "BrainSeq_SexMultiRegion_2024",
    "CrossDisorder_MultiRegion_2025"
  ),

  year = c(
    2019L,
    2018L,
    2023L,
    2024L,
    2025L
  ),

  study_focus = c(
    paste0(
      "Regional schizophrenia transcriptomic heterogeneity ",
      "across DLPFC and hippocampus"
    ),

    paste0(
      "Machine-learning comparison including multi-task ",
      "learning across schizophrenia transcriptomic datasets"
    ),

    paste0(
      "Psychiatric polygenic scores and aggregate ",
      "schizophrenia gene-expression score in postmortem DLPFC"
    ),

    paste0(
      "Sex-dependent schizophrenia transcriptional ",
      "associations across multiple BrainSeq brain regions"
    ),

    paste0(
      "Integrated postmortem transcriptomic signatures ",
      "across psychiatric disorders and multiple brain regions"
    )
  ),

  dataset_or_resource = c(
    "BrainSeq Phase II",
    "Five schizophrenia transcriptomic datasets",
    "HBCC / CMC-linked DLPFC subset",
    "BrainSeq Consortium",
    "Public postmortem RNA-seq datasets"
  ),

  brain_regions = c(
    "DLPFC + hippocampus",
    "Multiple transcriptomic datasets; not a paired DLPFC-HIPPO donor design",
    "DLPFC",
    "DLPFC + hippocampus + caudate",
    "Multiple postmortem brain regions"
  ),

  key_method = c(
    paste0(
      "Region-specific differential expression, eQTL/TWAS ",
      "and cross-region biological analyses"
    ),

    paste0(
      "Comparison of eight machine-learning methods, ",
      "including network-structured multi-task learning"
    ),

    paste0(
      "Aggregate schizophrenia gene-expression score ",
      "and association with schizophrenia polygenic score"
    ),

    paste0(
      "Region-aware differential expression, sex interaction ",
      "and sex-interacting eQTL analyses"
    ),

    paste0(
      "Large-scale integrative transcriptomic analysis ",
      "of shared/distinct psychiatric signatures"
    )
  ),

  external_validation_or_replication = c(
    "Not equivalent to the prespecified locked CMC + GSE80655 transfer framework",
    "Cross-dataset evaluation was performed",
    "Not an independent frozen-score multisite replication framework",
    "Not the primary objective",
    "Integrative multi-dataset analysis; not equivalent to SCZ-CRST lock-and-transfer design"
  ),

  molecular_score = c(
    "No SCZ-CRST-style frozen portable rank score",
    "No SCZ-CRST-style stability-derived portable state score",
    "YES — aggregate schizophrenia expression score precedent exists",
    "No SCZ-CRST-style frozen portable score established",
    "No SCZ-CRST-style score architecture established from reviewed source"
  ),

  donor_aware_cross_region_model = c(
    "Repeated measures were considered, but no SCZ-CRST sparse shared/deviation decomposition",
    "No paired-donor DLPFC-HIPPO decomposition established",
    "No paired-region donor model",
    "Uses BrainSeq multi-region data but not the proposed shared/deviation sparse state model",
    "No paired-donor SCZ-CRST decomposition established"
  ),

  explicit_shared_deviation_parameterization = c(
    "NO",
    "NO",
    "NO",
    "NO",
    "NO_EVIDENCE_FROM_REVIEWED_SOURCE"
  ),

  repeated_nested_grouped_validation = c(
    "NO_EVIDENCE_OF_SCZ_CRST_DESIGN",
    "Cross-validation used, but not the prespecified paired-donor repeated nested framework",
    "NO",
    "NO_EVIDENCE_OF_SCZ_CRST_DESIGN",
    "NO_EVIDENCE_OF_SCZ_CRST_DESIGN"
  ),

  stability_derived_state_freeze = c(
    "NO",
    "NO",
    "NO",
    "NO",
    "NO_EVIDENCE_FROM_REVIEWED_SOURCE"
  ),

  outcome_blind_external_technical_gate = c(
    "NO",
    "NO",
    "NO",
    "NO",
    "NO_EVIDENCE_FROM_REVIEWED_SOURCE"
  ),

  multisite_frozen_external_transfer = c(
    "NO",
    "Cross-dataset ML evaluation, but not the proposed frozen-score site-level framework",
    "NO",
    "NO",
    "NO_EVIDENCE_FROM_REVIEWED_SOURCE"
  ),

  overlap_with_SCZ_CRST = c(
    paste0(
      "Same discovery resource and same two primary regions; ",
      "strong precedent for regional heterogeneity"
    ),

    paste0(
      "Strong precedent for transcriptomic machine learning ",
      "and multi-task learning in schizophrenia"
    ),

    paste0(
      "Direct precedent showing that a schizophrenia-associated ",
      "aggregate expression score itself is not novel"
    ),

    paste0(
      "Strong precedent for BrainSeq multi-region schizophrenia ",
      "analysis beyond the original Phase II publication"
    ),

    paste0(
      "Precedent for large-scale cross-region/cross-disorder ",
      "postmortem transcriptomic integration"
    )
  ),

  remaining_gap_relevant_to_SCZ_CRST = c(
    paste0(
      "Does not establish the complete donor-grouped sparse ",
      "shared/deviation -> stability freeze -> outcome-blind ",
      "portable score -> independent multisite transfer architecture"
    ),

    paste0(
      "Does not establish paired-donor regional decomposition, ",
      "stability-derived state freezing, or outcome-blind technical transfer"
    ),

    paste0(
      "Does not establish donor-aware DLPFC-HIPPO decomposition ",
      "or frozen rank-based cross-cohort transfer"
    ),

    paste0(
      "Does not establish the proposed molecular-state transfer ",
      "and external locking architecture"
    ),

    paste0(
      "Does not establish the complete prespecified SCZ-CRST ",
      "framework based on reviewed evidence"
    )
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# Scientific conflict / risk audit
# ------------------------------------------------------------

SCIENTIFIC_CONFLICTS <- data.frame(

  issue_id = c(
    "N01",
    "N02",
    "N03",
    "N04",
    "N05",
    "D01",
    "D02",
    "D03",
    "D04"
  ),

  issue = c(
    "Multi-region schizophrenia transcriptomics is already established",
    "Multi-task learning for schizophrenia transcriptomic data already exists",
    "Schizophrenia aggregate gene-expression scores already exist",
    "External validation alone is not a novelty claim",
    "Current evidence does not justify a first-ever claim",
    "BrainSeq shows substantial DLPFC-HIPPO regional heterogeneity",
    "Paired donors can receive excess likelihood weight if rows are equally weighted",
    "CMC external sites may differ technically and biologically",
    "GSE80655 DLPFC is expected to be relatively small"
  ),

  severity = c(
    "IMPORTANT",
    "IMPORTANT",
    "IMPORTANT",
    "IMPORTANT",
    "IMPORTANT",
    "DESIGN_CHALLENGE",
    "DESIGN_CHALLENGE",
    "DESIGN_CHALLENGE",
    "POWER_LIMITATION"
  ),

  fatal_conflict = c(
    FALSE,
    FALSE,
    FALSE,
    FALSE,
    FALSE,
    FALSE,
    FALSE,
    FALSE,
    FALSE
  ),

  implication = c(
    paste0(
      "Do not claim novelty from merely analyzing DLPFC and hippocampus."
    ),

    paste0(
      "Do not claim novelty from machine learning or multi-task learning alone."
    ),

    paste0(
      "Do not claim novelty from constructing a transcriptomic score alone."
    ),

    paste0(
      "Independent validation is a strength, but is not itself sufficient ",
      "for a novelty claim."
    ),

    paste0(
      "Use cautious language such as 'integrated framework' and test ",
      "differentiation against current literature before manuscript claims."
    ),

    paste0(
      "A strong globally shared state is not guaranteed. ",
      "The framework must allow a scientifically valid weak/null shared component."
    ),

    paste0(
      "In pooled/shared-deviation models, donors contributing two regions ",
      "would otherwise contribute two sample-level likelihood terms."
    ),

    paste0(
      "Site-specific effects and heterogeneity must be reported; ",
      "do not hide a result driven by one brain bank."
    ),

    paste0(
      "Treat GSE80655 as secondary replication and emphasize effect estimates ",
      "with uncertainty rather than demanding nominal significance."
    )
  ),

  prespecified_resolution = c(
    "Novelty will be attached only to the integrated architecture if supported.",
    "MTL is precedent/comparator context, not the claimed contribution.",
    "The frozen score is one component of the framework, not the sole contribution.",
    "Focus external evidence on locked transfer, site consistency and replication.",
    "No 'first', 'first-ever', or absolute novelty language at this stage.",
    paste0(
      "Preserve shared/deviation decomposition and report negative/shared-null ",
      "findings transparently."
    ),
    paste0(
      "Before Step 00C lock, use donor-balanced observation weights for pooled ",
      "models: each donor should contribute total training weight approximately 1."
    ),
    paste0(
      "Use one frozen score, outcome-blind mapping/gate, site-specific estimates ",
      "and random-effects meta-analysis where justified."
    ),
    paste0(
      "No external retuning; retain GSE80655 as a confirmatory secondary ",
      "replication with appropriately wide confidence intervals."
    )
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

stopifnot(
  nrow(NOVELTY_MATRIX) == 5L,
  !anyDuplicated(NOVELTY_MATRIX$study_id),
  nrow(SCIENTIFIC_CONFLICTS) == 9L,
  !anyDuplicated(SCIENTIFIC_CONFLICTS$issue_id)
)

# No conflict is currently classified as fatal.
if (any(SCIENTIFIC_CONFLICTS$fatal_conflict)) {
  stop(
    "STEP 00B CELL 3 STOP: A fatal scientific conflict was identified."
  )
}

# Explicitly forbid premature novelty language.
PROHIBITED_CLAIMS <- c(
  "first-ever",
  "first ever",
  "first study",
  "first framework"
)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

NOVELTY_MATRIX_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Novelty_Matrix_v1.csv"
)

SCIENTIFIC_CONFLICTS_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Scientific_Conflict_Audit_v1.csv"
)

assert_project_sentinel()

write.csv(
  NOVELTY_MATRIX,
  NOVELTY_MATRIX_PATH,
  row.names = FALSE,
  na = ""
)

assert_project_sentinel()

write.csv(
  SCIENTIFIC_CONFLICTS,
  SCIENTIFIC_CONFLICTS_PATH,
  row.names = FALSE,
  na = ""
)

# ------------------------------------------------------------
# Create conservative novelty statement
# ------------------------------------------------------------

NOVELTY_SUMMARY <- c(
  "SCZ-CRST STEP 00B — NOVELTY INTERPRETATION v1",
  "==============================================",
  "",
  "WHAT IS ALREADY ESTABLISHED:",
  "- Multi-region schizophrenia postmortem transcriptomics.",
  "- DLPFC-versus-hippocampus schizophrenia transcriptomic heterogeneity.",
  "- Machine learning for schizophrenia transcriptomic data.",
  "- Multi-task learning across schizophrenia transcriptomic datasets.",
  "- Aggregate schizophrenia-associated gene-expression scores.",
  "- External/cross-dataset evaluation as a general concept.",
  "",
  "THEREFORE THESE ARE NOT STANDALONE NOVELTY CLAIMS.",
  "",
  "CURRENT DIFFERENTIATED SCZ-CRST HYPOTHESIS:",
  paste0(
    "The potentially differentiated contribution is the integrated combination ",
    "of donor-aware DLPFC/HIPPO shared-versus-deviation decomposition, ",
    "repeated nested grouped validation, stability-defined shared-state freezing, ",
    "a locked portable rank-based score, an outcome-blind technical eligibility ",
    "gate, non-overlapping CMC MSSM/Penn/Pitt external transfer, site-level ",
    "replication/meta-analysis, and secondary GSE80655 replication."
  ),
  "",
  "IMPORTANT:",
  paste0(
    "The literature audit performed so far does NOT justify 'first-ever', ",
    "'first study', or equivalent absolute novelty language."
  ),
  "",
  "MATERIAL SCIENTIFIC CONFLICT:",
  "No fatal conflict identified.",
  "",
  "IMPORTANT MODELING REFINEMENT FOR STEP 00C:",
  paste0(
    "Grouped CV prevents donor leakage but does not prevent paired donors from ",
    "receiving twice the sample-level likelihood contribution. ",
    "Donor-balanced observation weighting should therefore be prespecified ",
    "for pooled/shared-deviation training models."
  ),
  "",
  "NEGATIVE RESULT PRINCIPLE:",
  paste0(
    "Because prior BrainSeq work demonstrates substantial regional heterogeneity, ",
    "failure to identify a strong stable shared state must remain an acceptable ",
    "and reportable scientific outcome."
  )
)

NOVELTY_SUMMARY_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Novelty_Interpretation_v1.txt"
)

assert_project_sentinel()

writeLines(
  NOVELTY_SUMMARY,
  NOVELTY_SUMMARY_PATH,
  useBytes = TRUE
)

# ------------------------------------------------------------
# Validate files
# ------------------------------------------------------------

OUTPUT_CHECKS <- c(
  novelty_matrix = file.exists(NOVELTY_MATRIX_PATH),
  scientific_conflicts = file.exists(SCIENTIFIC_CONFLICTS_PATH),
  novelty_summary = file.exists(NOVELTY_SUMMARY_PATH)
)

if (!all(OUTPUT_CHECKS)) {

  stop(
    paste0(
      "STEP 00B CELL 3 FAIL. Missing outputs: ",
      paste(
        names(OUTPUT_CHECKS)[!OUTPUT_CHECKS],
        collapse = ", "
      )
    )
  )
}

# ------------------------------------------------------------
# Log
# ------------------------------------------------------------

log_line(
  "STEP00B_NOVELTY_MATRIX_ROWS=",
  nrow(NOVELTY_MATRIX)
)

log_line(
  "STEP00B_SCIENTIFIC_CONFLICT_ROWS=",
  nrow(SCIENTIFIC_CONFLICTS)
)

log_line(
  "STEP00B_FATAL_CONFLICT=FALSE"
)

log_line(
  "STEP00B_FIRST_EVER_CLAIM_ALLOWED=FALSE"
)

log_line(
  "STEP00B_DONOR_BALANCED_WEIGHTING_REFINEMENT=RECOMMENDED_FOR_STEP00C"
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00B — CELL 3\n")
cat("============================================================\n")
cat("Novelty-reference studies:             ", nrow(NOVELTY_MATRIX), "\n", sep = "")
cat("Scientific issues/conflicts recorded:  ", nrow(SCIENTIFIC_CONFLICTS), "\n", sep = "")
cat("Fatal scientific conflict identified:  FALSE\n")
cat("First-ever novelty claim allowed:       FALSE\n")
cat("Generic multi-region novelty allowed:   FALSE\n")
cat("Generic MTL novelty allowed:            FALSE\n")
cat("Generic score novelty allowed:          FALSE\n")
cat("Integrated framework remains testable:  TRUE\n")
cat("Donor-balanced weighting refinement:    REQUIRED FOR STEP 00C REVIEW\n")
cat("\nNovelty matrix:\n", NOVELTY_MATRIX_PATH, "\n", sep = "")
cat("Conflict audit:\n", SCIENTIFIC_CONFLICTS_PATH, "\n", sep = "")
cat("Interpretation:\n", NOVELTY_SUMMARY_PATH, "\n", sep = "")
cat("============================================================\n")
cat("CELL 3 — PASS\n")

[2026-09-29 22:01:44 +0600] STEP00B_NOVELTY_MATRIX_ROWS=5 
[2026-09-29 22:01:44 +0600] STEP00B_SCIENTIFIC_CONFLICT_ROWS=9 
[2026-09-29 22:01:44 +0600] STEP00B_FATAL_CONFLICT=FALSE 
[2026-09-29 22:01:44 +0600] STEP00B_FIRST_EVER_CLAIM_ALLOWED=FALSE 
[2026-09-29 22:01:44 +0600] STEP00B_DONOR_BALANCED_WEIGHTING_REFINEMENT=RECOMMENDED_FOR_STEP00C 
STEP 00B — CELL 3
Novelty-reference studies:             5
Scientific issues/conflicts recorded:  9
Fatal scientific conflict identified:  FALSE
First-ever novelty claim allowed:       FALSE
Generic multi-region novelty allowed:   FALSE
Generic MTL novelty allowed:            FALSE
Generic score novelty allowed:          FALSE
Integrated framework remains testable:  TRUE
Donor-balanced weighting refinement:    REQUIRED FOR STEP 00C REVIEW

Novelty matrix:
E:/SCZ_CROSSREGION_MSS_Q1/00_protocol/STEP00B_Novelty_Matrix_v1.csv
Conflict audit:
E:/SCZ_CROSSREGION_MSS_Q1/00_protocol/STEP00B_Scientific_Conflict_Audit_v1.csv
Interpretation:
E:/SCZ_CROSSREG

In [13]:
# ============================================================
# STEP 00B — CELL 4
# FINAL VALIDATION + VERSIONED LITERATURE/DATA AUDIT FREEZE
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# Required Step 00B artifacts
# ------------------------------------------------------------

STEP00B_FILES <- c(
  verified_source_registry = file.path(
    PROJECT_ROOT,
    "00_protocol",
    "STEP00B_Verified_Source_Registry_v1.csv"
  ),

  dataset_independence_audit = file.path(
    PROJECT_ROOT,
    "00_protocol",
    "STEP00B_Dataset_Independence_Audit_v1.csv"
  ),

  dataset_independence_summary = file.path(
    PROJECT_ROOT,
    "00_protocol",
    "STEP00B_Dataset_Independence_Summary_v1.txt"
  ),

  novelty_matrix = file.path(
    PROJECT_ROOT,
    "00_protocol",
    "STEP00B_Novelty_Matrix_v1.csv"
  ),

  scientific_conflict_audit = file.path(
    PROJECT_ROOT,
    "00_protocol",
    "STEP00B_Scientific_Conflict_Audit_v1.csv"
  ),

  novelty_interpretation = file.path(
    PROJECT_ROOT,
    "00_protocol",
    "STEP00B_Novelty_Interpretation_v1.txt"
  )
)

# ------------------------------------------------------------
# Check files exist
# ------------------------------------------------------------

FILE_EXISTS <- file.exists(STEP00B_FILES)

if (!all(FILE_EXISTS)) {

  missing_files <- names(STEP00B_FILES)[
    !FILE_EXISTS
  ]

  stop(
    paste0(
      "STEP 00B FINAL VALIDATION FAILED.\n",
      "Missing artifact(s):\n- ",
      paste(
        missing_files,
        collapse = "\n- "
      )
    )
  )
}

# ------------------------------------------------------------
# Re-read canonical artifacts
# ------------------------------------------------------------

SOURCE_REGISTRY_FINAL <- read.csv(
  STEP00B_FILES["verified_source_registry"],
  stringsAsFactors = FALSE,
  check.names = FALSE
)

DATASET_AUDIT_FINAL <- read.csv(
  STEP00B_FILES["dataset_independence_audit"],
  stringsAsFactors = FALSE,
  check.names = FALSE
)

NOVELTY_MATRIX_FINAL <- read.csv(
  STEP00B_FILES["novelty_matrix"],
  stringsAsFactors = FALSE,
  check.names = FALSE
)

CONFLICT_AUDIT_FINAL <- read.csv(
  STEP00B_FILES["scientific_conflict_audit"],
  stringsAsFactors = FALSE,
  check.names = FALSE
)

# ------------------------------------------------------------
# Independent scientific-design assertions
# ------------------------------------------------------------

CHECK_SOURCE_ROWS <- nrow(SOURCE_REGISTRY_FINAL) == 6L

CHECK_DATASET_ROWS <- nrow(DATASET_AUDIT_FINAL) == 9L

CHECK_NOVELTY_ROWS <- nrow(NOVELTY_MATRIX_FINAL) == 5L

CHECK_CONFLICT_ROWS <- nrow(CONFLICT_AUDIT_FINAL) == 9L

# BrainSeq must remain discovery only.
brainseq_check <- DATASET_AUDIT_FINAL[
  DATASET_AUDIT_FINAL$parent_dataset == "BrainSeq Phase II",
]

CHECK_BRAINSEQ_NOT_EXTERNAL <- (
  nrow(brainseq_check) == 2L &&
  !any(brainseq_check$independent_external)
)

# HBCC must remain excluded.
hbcc_check <- DATASET_AUDIT_FINAL[
  DATASET_AUDIT_FINAL$resource == "CMC HBCC",
]

CHECK_HBCC_EXCLUDED <- (
  nrow(hbcc_check) == 1L &&
  identical(
    hbcc_check$primary_decision,
    "EXCLUDE"
  ) &&
  !isTRUE(
    hbcc_check$independent_external
  )
)

# MSSM / Penn / Pitt must be primary external sites.
expected_cmc_sites <- c(
  "CMC MSSM",
  "CMC Penn",
  "CMC Pitt"
)

cmc_primary_check <- DATASET_AUDIT_FINAL[
  DATASET_AUDIT_FINAL$resource %in% expected_cmc_sites,
]

CHECK_CMC_PRIMARY <- (
  nrow(cmc_primary_check) == 3L &&
  all(cmc_primary_check$independent_external) &&
  setequal(
    cmc_primary_check$resource,
    expected_cmc_sites
  )
)

# GSE80655 DLPFC must remain secondary replication.
gse_check <- DATASET_AUDIT_FINAL[
  DATASET_AUDIT_FINAL$resource == "GSE80655 DLPFC",
]

CHECK_GSE_DLPFC_SECONDARY <- (
  nrow(gse_check) == 1L &&
  identical(
    gse_check$planned_role,
    "Secondary external replication"
  )
)

# No fatal scientific conflict should exist.
CHECK_NO_FATAL_CONFLICT <- !any(
  CONFLICT_AUDIT_FINAL$fatal_conflict
)

# No exact external sample counts should have been invented/frozen.
CHECK_COUNTS_NOT_PREMATURELY_FROZEN <- all(
  DATASET_AUDIT_FINAL$exact_sample_counts ==
    "TO_BE_VERIFIED_FROM_METADATA"
)

# ------------------------------------------------------------
# Final validation table
# ------------------------------------------------------------

STEP00B_FINAL_CHECKS <- c(

  all_artifacts_exist =
    all(FILE_EXISTS),

  source_registry_rows =
    CHECK_SOURCE_ROWS,

  dataset_audit_rows =
    CHECK_DATASET_ROWS,

  novelty_matrix_rows =
    CHECK_NOVELTY_ROWS,

  conflict_audit_rows =
    CHECK_CONFLICT_ROWS,

  brainseq_not_external =
    CHECK_BRAINSEQ_NOT_EXTERNAL,

  cmc_hbcc_excluded =
    CHECK_HBCC_EXCLUDED,

  cmc_mssm_penn_pitt_primary =
    CHECK_CMC_PRIMARY,

  gse80655_dlpfc_secondary =
    CHECK_GSE_DLPFC_SECONDARY,

  no_fatal_scientific_conflict =
    CHECK_NO_FATAL_CONFLICT,

  external_counts_not_invented =
    CHECK_COUNTS_NOT_PREMATURELY_FROZEN
)

STEP00B_VALIDATION_TABLE <- data.frame(
  check = names(STEP00B_FINAL_CHECKS),
  passed = as.logical(STEP00B_FINAL_CHECKS),
  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# Stop if any final check fails
# ------------------------------------------------------------

if (!all(STEP00B_FINAL_CHECKS)) {

  FAILED <- names(STEP00B_FINAL_CHECKS)[
    !STEP00B_FINAL_CHECKS
  ]

  log_line(
    "STEP_00B_FINAL_STATUS=FAIL"
  )

  log_line(
    "STEP_00B_FAILED_CHECKS=",
    paste(
      FAILED,
      collapse = ","
    )
  )

  print(
    STEP00B_VALIDATION_TABLE,
    row.names = FALSE
  )

  stop(
    paste0(
      "STEP 00B FINAL VALIDATION FAILED.\n\n",
      "Failed checks:\n- ",
      paste(
        FAILED,
        collapse = "\n- "
      )
    )
  )
}

# ------------------------------------------------------------
# SHA-256 helper
# ------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {
  stop(
    "STEP 00B STOP: digest package is required for SHA-256 hashing."
  )
}

sha256_file <- function(path) {

  digest::digest(
    object = path,
    algo = "sha256",
    file = TRUE,
    serialize = FALSE
  )
}

# ------------------------------------------------------------
# Build SHA-256 manifest
# ------------------------------------------------------------

STEP00B_HASH_MANIFEST <- data.frame(

  artifact_name = names(STEP00B_FILES),

  file_path = unname(
    STEP00B_FILES
  ),

  sha256 = vapply(
    unname(STEP00B_FILES),
    sha256_file,
    character(1)
  ),

  stringsAsFactors = FALSE
)

STEP00B_HASH_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Artifact_SHA256_v1.csv"
)

assert_project_sentinel()

write.csv(
  STEP00B_HASH_MANIFEST,
  STEP00B_HASH_PATH,
  row.names = FALSE,
  na = ""
)

# ------------------------------------------------------------
# Save final validation table
# ------------------------------------------------------------

STEP00B_VALIDATION_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP00B_Final_Validation_v1.csv"
)

assert_project_sentinel()

write.csv(
  STEP00B_VALIDATION_TABLE,
  STEP00B_VALIDATION_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# Create explicit PASS record
# ------------------------------------------------------------

STEP00B_PASS_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP_00B_PASS.txt"
)

STEP00B_PASS_TEXT <- c(

  "SCZ-CRST STEP 00B — PASS",

  paste0(
    "timestamp: ",
    format(
      Sys.time(),
      "%Y-%m-%d %H:%M:%S %z"
    )
  ),

  paste0(
    "project_root: ",
    PROJECT_ROOT
  ),

  paste0(
    "project_id: ",
    PROJECT_ID
  ),

  "literature_data_audit_version: v1",

  paste0(
    "verified_source_registry_rows: ",
    nrow(SOURCE_REGISTRY_FINAL)
  ),

  paste0(
    "dataset_resource_rows: ",
    nrow(DATASET_AUDIT_FINAL)
  ),

  paste0(
    "novelty_reference_rows: ",
    nrow(NOVELTY_MATRIX_FINAL)
  ),

  paste0(
    "scientific_conflict_rows: ",
    nrow(CONFLICT_AUDIT_FINAL)
  ),

  "fatal_scientific_conflict: FALSE",

  "cmc_hbcc_independent_validation: EXCLUDED",

  "primary_cmc_external_sites: MSSM,Penn,Pitt",

  "secondary_external_replication: GSE80655_DLPFC",

  "first_ever_claim_supported: FALSE",

  "generic_multi_region_novelty_supported: FALSE",

  "generic_mtl_novelty_supported: FALSE",

  "generic_expression_score_novelty_supported: FALSE",

  "integrated_SCZ_CRST_framework_remains_testable: TRUE",

  "exact_external_sample_counts_frozen: FALSE",

  "metadata_level_overlap_audit_still_required: TRUE",

  "donor_balanced_training_weighting_to_review_in_STEP_00C: TRUE"
)

assert_project_sentinel()

writeLines(
  STEP00B_PASS_TEXT,
  STEP00B_PASS_PATH,
  useBytes = TRUE
)

# ------------------------------------------------------------
# Validate newly written outputs
# ------------------------------------------------------------

NEW_OUTPUT_CHECKS <- c(
  hash_manifest = file.exists(STEP00B_HASH_PATH),
  validation_table = file.exists(STEP00B_VALIDATION_PATH),
  pass_record = file.exists(STEP00B_PASS_PATH)
)

if (!all(NEW_OUTPUT_CHECKS)) {

  stop(
    paste0(
      "STEP 00B FINAL WRITE FAILURE: ",
      paste(
        names(NEW_OUTPUT_CHECKS)[
          !NEW_OUTPUT_CHECKS
        ],
        collapse = ", "
      )
    )
  )
}

# ------------------------------------------------------------
# Log final status
# ------------------------------------------------------------

log_line(
  "STEP00B_HASH_MANIFEST=",
  STEP00B_HASH_PATH
)

log_line(
  "STEP00B_FINAL_VALIDATION=",
  STEP00B_VALIDATION_PATH
)

log_line(
  "STEP00B_PASS_RECORD=",
  STEP00B_PASS_PATH
)

log_line(
  "STEP_00B_FINAL_STATUS=PASS"
)

# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00B — FINAL VALIDATION\n")
cat("============================================================\n")

print(
  STEP00B_VALIDATION_TABLE,
  row.names = FALSE
)

cat("\n")
cat("Verified literature/source records: ", nrow(SOURCE_REGISTRY_FINAL), "\n", sep = "")
cat("Dataset/resource audit rows:        ", nrow(DATASET_AUDIT_FINAL), "\n", sep = "")
cat("Novelty-reference studies:          ", nrow(NOVELTY_MATRIX_FINAL), "\n", sep = "")
cat("Scientific issues recorded:         ", nrow(CONFLICT_AUDIT_FINAL), "\n", sep = "")
cat("Fatal scientific conflict:          FALSE\n")
cat("CMC HBCC excluded:                  TRUE\n")
cat("Primary CMC sites:                  MSSM / Penn / Pitt\n")
cat("Secondary replication:              GSE80655 DLPFC\n")
cat("External sample counts frozen:      FALSE\n")
cat("First-ever claim permitted:         FALSE\n")
cat("Integrated framework testable:      TRUE\n")

cat("\n============================================================\n")
cat("STEP 00B — PASS\n")
cat("============================================================\n")
cat("SHA-256 manifest: ", STEP00B_HASH_PATH, "\n", sep = "")
cat("Validation table: ", STEP00B_VALIDATION_PATH, "\n", sep = "")
cat("PASS record:      ", STEP00B_PASS_PATH, "\n", sep = "")
cat("============================================================\n")

[2026-09-29 22:03:00 +0600] STEP00B_HASH_MANIFEST=E:/SCZ_CROSSREGION_MSS_Q1/00_protocol/STEP00B_Artifact_SHA256_v1.csv 
[2026-09-29 22:03:00 +0600] STEP00B_FINAL_VALIDATION=E:/SCZ_CROSSREGION_MSS_Q1/00_protocol/STEP00B_Final_Validation_v1.csv 
[2026-09-29 22:03:00 +0600] STEP00B_PASS_RECORD=E:/SCZ_CROSSREGION_MSS_Q1/00_protocol/STEP_00B_PASS.txt 
[2026-09-29 22:03:00 +0600] STEP_00B_FINAL_STATUS=PASS 
STEP 00B — FINAL VALIDATION
                        check passed
          all_artifacts_exist   TRUE
         source_registry_rows   TRUE
           dataset_audit_rows   TRUE
          novelty_matrix_rows   TRUE
          conflict_audit_rows   TRUE
        brainseq_not_external   TRUE
            cmc_hbcc_excluded   TRUE
   cmc_mssm_penn_pitt_primary   TRUE
     gse80655_dlpfc_secondary   TRUE
 no_fatal_scientific_conflict   TRUE
 external_counts_not_invented   TRUE

Verified literature/source records: 6
Dataset/resource audit rows:        9
Novelty-reference studies:          5
Scientif

In [14]:
# ============================================================
# STEP 00C — CELL 1
# PROTOCOL PREREQUISITES + CORE DESIGN CONSTANTS
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# 1. Require successful upstream protocol steps
# ------------------------------------------------------------

STEP00A_PASS_PATH <- file.path(
  PROJECT_ROOT,
  "15_environment",
  "STEP_00A_PASS.txt"
)

STEP00B_PASS_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "STEP_00B_PASS.txt"
)

if (!file.exists(STEP00A_PASS_PATH)) {
  stop(
    "STEP 00C STOP: STEP 00A has not been formally passed."
  )
}

if (!file.exists(STEP00B_PASS_PATH)) {
  stop(
    "STEP 00C STOP: STEP 00B has not been formally passed."
  )
}

# ------------------------------------------------------------
# 2. Protocol identity
# ------------------------------------------------------------

PROTOCOL_VERSION <- "v1"

PROTOCOL_NAME <- "ANALYSIS_DECISION_LOCK_v1"

PROTOCOL_STATUS <- "PRE_MODELLING_PRE_EXTERNAL_OUTCOME_UNBLINDING"

PROJECT_SHORT_NAME <- "SCZ-CRST"

PROJECT_WORKING_TITLE <- paste0(
  "Donor-Aware Cross-Region Decomposition of a Stable and ",
  "Externally Transferable Schizophrenia Transcriptomic State"
)

PRIMARY_OBJECTIVE <- paste0(
  "Identify a reproducible schizophrenia-associated transcriptomic ",
  "state containing a component shared across DLPFC and hippocampus; ",
  "distinguish that component from regional deviations; derive a stable ",
  "discovery-only frozen molecular-state score; and test transfer to ",
  "non-overlapping external postmortem brain cohorts without external ",
  "outcome-guided tuning."
)

CLINICAL_BIOMARKER_CLAIM <- FALSE

# ------------------------------------------------------------
# 3. Discovery and external dataset hierarchy
# ------------------------------------------------------------

DISCOVERY_DATASET <- "BrainSeq Phase II total-RNA"

DISCOVERY_REGIONS <- c(
  "DLPFC",
  "HIPPO"
)

PRIMARY_EXTERNAL_DATASET <- "CommonMind Consortium DLPFC"

PRIMARY_EXTERNAL_SITES <- c(
  "MSSM",
  "Penn",
  "Pitt"
)

EXCLUDED_EXTERNAL_SITE <- "HBCC"

SECONDARY_EXTERNAL_DATASET <- "GSE80655"

SECONDARY_EXTERNAL_REGION <- "DLPFC"

# ------------------------------------------------------------
# 4. Primary discovery population rules
# ------------------------------------------------------------

PRIMARY_DIAGNOSES <- c(
  "Schizophrenia",
  "Neurotypical_Control"
)

PRIMARY_ADULT_RULE <- "age > 17 years"

OTHER_PSYCHIATRIC_DIAGNOSES_PRIMARY <- "EXCLUDE"

DONOR_PAIRING_POLICY <- paste0(
  "All samples from the same donor must remain in the same ",
  "training/validation/test partition."
)

# ------------------------------------------------------------
# 5. Core cross-validation design
# ------------------------------------------------------------

OUTER_FOLDS <- 5L
OUTER_REPEATS <- 5L
INNER_FOLDS <- 4L

TOTAL_OUTER_EVALUATIONS <- OUTER_FOLDS * OUTER_REPEATS

OUTER_SPLIT_SEED <- 20260929L
INNER_TUNING_SEED <- 20261001L
STABILITY_SEED <- 20261002L
PERMUTATION_SEED <- 20261003L

OUTER_STRATIFICATION_RULE <- paste0(
  "Approximate donor-level stratification by diagnosis x ",
  "region_availability_class, with deterministic balancing."
)

# ------------------------------------------------------------
# 6. Donor-balanced observation weighting
#
# This resolves the issue identified in Step 00B:
# paired donors must not contribute twice the total training
# likelihood of donors represented by only one region.
# ------------------------------------------------------------

DONOR_BALANCED_WEIGHTING <- TRUE

DONOR_WEIGHT_FORMULA <- paste0(
  "For pooled/shared-deviation training models, each sample weight ",
  "equals 1 divided by the number of eligible regional samples ",
  "contributed by that donor within the current training partition. ",
  "Thus a donor with DLPFC+HIPPO contributes 0.5+0.5 and a donor ",
  "with one eligible region contributes 1.0. Relative weights may ",
  "be internally rescaled by glmnet without changing this policy."
)

REGION_SPECIFIC_WEIGHT_POLICY <- paste0(
  "Region-specific models contain at most one sample per donor ",
  "within that region and therefore use equal donor/sample weights."
)

# ------------------------------------------------------------
# 7. Leakage rules
# ------------------------------------------------------------

LEAKAGE_RULES <- c(
  "No donor may appear in both training and outer-test data.",
  "No supervised gene selection may use the full discovery dataset before cross-validation.",
  "All model-dependent preprocessing must be fitted using training data only.",
  "Outer-test information must not influence tuning or feature selection.",
  "External diagnosis must not influence score construction, mapping, technical eligibility, or score weights.",
  "External model retraining or score-weight recalibration is prohibited.",
  "External classification-threshold optimization is prohibited."
)

# ------------------------------------------------------------
# 8. Core model parameterization
# ------------------------------------------------------------

PRIMARY_MODEL <- "Shared_plus_Regional_Deviation_ElasticNet_Logistic"

MODEL_PARAMETERIZATION <- paste0(
  "Augmented expression design [X, sX], where s=+1 for DLPFC ",
  "and s=-1 for HIPPO. Gene coefficients are parameterized as ",
  "beta_DLPFC = beta_shared + beta_diff and ",
  "beta_HIPPO = beta_shared - beta_diff."
)

SHARED_BLOCK_PENALTY_FACTOR <- 1

DEVIATION_PENALTY_MULTIPLIERS <- c(
  1,
  2,
  4,
  8
)

ELASTIC_NET_ALPHA_GRID <- c(
  0.10,
  0.25,
  0.50,
  0.75,
  1.00
)

LAMBDA_SELECTION_RULE <- paste0(
  "Lambda is selected exclusively within grouped inner CV from ",
  "the glmnet regularization path. No outer-test information is used."
)

TECHNICAL_COVARIATE_PENALTY <- 0

# ------------------------------------------------------------
# 9. Required comparator models
# ------------------------------------------------------------

REQUIRED_MODELS <- c(
  "A_DLPFC_only_elastic_net",
  "B_HIPPO_only_elastic_net",
  "C_Pooled_region_elastic_net_without_decomposition",
  "D_Independent_region_specific_models",
  "E_Shared_only_model",
  "F_Shared_plus_regional_deviation_model"
)

PRIMARY_MODEL_COMPARISON_QUESTION <- paste0(
  "Does explicit shared/deviation decomposition improve stability, ",
  "interpretability, or external transferability relative to simpler ",
  "models when evaluated using identical donor-grouped outer splits?"
)

# ------------------------------------------------------------
# 10. Stability framework
# ------------------------------------------------------------

STABILITY_SELECTION_PRIMARY_THRESHOLD <- 0.80
STABILITY_SIGN_PRIMARY_THRESHOLD <- 0.90

STABILITY_SELECTION_SENSITIVITY <- c(
  0.70,
  0.80,
  0.90
)

COMPLEMENTARY_PAIR_TARGET <- 100L
HALF_SAMPLE_FIT_TARGET <- 200L

STABILITY_FORMAL_FDR_CLAIM <- FALSE

# ------------------------------------------------------------
# 11. Validate constants
# ------------------------------------------------------------

stopifnot(
  OUTER_FOLDS == 5L,
  OUTER_REPEATS == 5L,
  TOTAL_OUTER_EVALUATIONS == 25L,
  INNER_FOLDS == 4L,
  DONOR_BALANCED_WEIGHTING,
  SHARED_BLOCK_PENALTY_FACTOR == 1,
  all(DEVIATION_PENALTY_MULTIPLIERS >= 1),
  all(ELASTIC_NET_ALPHA_GRID > 0),
  all(ELASTIC_NET_ALPHA_GRID <= 1),
  STABILITY_SELECTION_PRIMARY_THRESHOLD == 0.80,
  STABILITY_SIGN_PRIMARY_THRESHOLD == 0.90,
  !STABILITY_FORMAL_FDR_CLAIM,
  !CLINICAL_BIOMARKER_CLAIM
)

# ------------------------------------------------------------
# 12. Log locked core decisions
# ------------------------------------------------------------

log_line(
  "STEP00C_PROTOCOL_VERSION=",
  PROTOCOL_VERSION
)

log_line(
  "STEP00C_OUTER_CV=",
  OUTER_FOLDS,
  "fold_x",
  OUTER_REPEATS,
  "repeats"
)

log_line(
  "STEP00C_INNER_CV=",
  INNER_FOLDS,
  "fold"
)

log_line(
  "STEP00C_DONOR_BALANCED_WEIGHTING=TRUE"
)

log_line(
  "STEP00C_PRIMARY_MODEL=",
  PRIMARY_MODEL
)

log_line(
  "STEP00C_STABILITY_THRESHOLD=",
  STABILITY_SELECTION_PRIMARY_THRESHOLD
)

log_line(
  "STEP00C_SIGN_THRESHOLD=",
  STABILITY_SIGN_PRIMARY_THRESHOLD
)

# ------------------------------------------------------------
# 13. Compact output
# ------------------------------------------------------------

cat("============================================================\n")
cat("STEP 00C — CELL 1\n")
cat("============================================================\n")
cat("Protocol version:                ", PROTOCOL_VERSION, "\n", sep = "")
cat("Protocol status:                 ", PROTOCOL_STATUS, "\n", sep = "")
cat("Discovery dataset:               ", DISCOVERY_DATASET, "\n", sep = "")
cat("Discovery regions:               ", paste(DISCOVERY_REGIONS, collapse = " + "), "\n", sep = "")
cat("Outer CV:                        ", OUTER_FOLDS, " folds x ", OUTER_REPEATS, " repeats\n", sep = "")
cat("Outer evaluations:               ", TOTAL_OUTER_EVALUATIONS, "\n", sep = "")
cat("Inner CV:                        ", INNER_FOLDS, " grouped folds\n", sep = "")
cat("Donor grouping mandatory:        TRUE\n")
cat("Donor-balanced training weights: ", DONOR_BALANCED_WEIGHTING, "\n", sep = "")
cat("Primary model:                   ", PRIMARY_MODEL, "\n", sep = "")
cat("Deviation penalty multipliers:   ", paste(DEVIATION_PENALTY_MULTIPLIERS, collapse = ", "), "\n", sep = "")
cat("Elastic-net alpha grid:          ", paste(ELASTIC_NET_ALPHA_GRID, collapse = ", "), "\n", sep = "")
cat("Stable selection threshold:      ", STABILITY_SELECTION_PRIMARY_THRESHOLD, "\n", sep = "")
cat("Stable sign threshold:           ", STABILITY_SIGN_PRIMARY_THRESHOLD, "\n", sep = "")
cat("Clinical biomarker claim:        FALSE\n")
cat("Formal gene-level FDR claim:     FALSE\n")
cat("============================================================\n")
cat("CELL 1 — PASS\n")

[2026-09-29 22:05:31 +0600] STEP00C_PROTOCOL_VERSION=v1 
[2026-09-29 22:05:31 +0600] STEP00C_OUTER_CV=5fold_x5repeats 
[2026-09-29 22:05:31 +0600] STEP00C_INNER_CV=4fold 
[2026-09-29 22:05:31 +0600] STEP00C_DONOR_BALANCED_WEIGHTING=TRUE 
[2026-09-29 22:05:31 +0600] STEP00C_PRIMARY_MODEL=Shared_plus_Regional_Deviation_ElasticNet_Logistic 
[2026-09-29 22:05:31 +0600] STEP00C_STABILITY_THRESHOLD=0.8 
[2026-09-29 22:05:31 +0600] STEP00C_SIGN_THRESHOLD=0.9 
STEP 00C — CELL 1
Protocol version:                v1
Protocol status:                 PRE_MODELLING_PRE_EXTERNAL_OUTCOME_UNBLINDING
Discovery dataset:               BrainSeq Phase II total-RNA
Discovery regions:               DLPFC + HIPPO
Outer CV:                        5 folds x 5 repeats
Outer evaluations:               25
Inner CV:                        4 grouped folds
Donor grouping mandatory:        TRUE
Donor-balanced training weights: TRUE
Primary model:                   Shared_plus_Regional_Deviation_ElasticNet_Logistic
Devi

In [15]:
# ============================================================
# STEP 01A — CELL 1
# BRAINSEQ PHASE II: DOWNLOAD + RAW INVENTORY
# ============================================================

assert_project_sentinel()

BRAINSEQ_URL <- paste0(
  "https://s3.us-east-2.amazonaws.com/",
  "libd-brainseq2/rse_gene_unfiltered.Rdata"
)

RAW_DIR <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "brainseq_phase2"
)

RAW_FILE <- file.path(
  RAW_DIR,
  "rse_gene_unfiltered.Rdata"
)

PART_FILE <- paste0(
  RAW_FILE,
  ".part"
)

INVENTORY_FILE <- file.path(
  RAW_DIR,
  "brainseq_phase2_raw_inventory.csv"
)

if (!dir.exists(RAW_DIR)) {
  stop("STEP 01A FAIL: BrainSeq raw directory does not exist.")
}

if (!requireNamespace("digest", quietly = TRUE)) {
  stop("STEP 01A FAIL: digest package is required.")
}

# ------------------------------------------------------------
# Download only when canonical raw file is absent
# ------------------------------------------------------------

if (!file.exists(RAW_FILE)) {

  cat("Downloading BrainSeq Phase II gene-level dataset...\n")

  if (file.exists(PART_FILE)) {
    unlink(PART_FILE)
  }

  old_timeout <- getOption("timeout")
  options(timeout = 7200)

  download_status <- tryCatch({

    utils::download.file(
      url = BRAINSEQ_URL,
      destfile = PART_FILE,
      mode = "wb",
      method = "libcurl",
      quiet = FALSE
    )

    TRUE

  }, error = function(e) {

    message(conditionMessage(e))
    FALSE
  })

  options(timeout = old_timeout)

  if (!download_status) {
    stop("STEP 01A FAIL: Download failed.")
  }

  if (!file.exists(PART_FILE)) {
    stop("STEP 01A FAIL: Temporary download file missing.")
  }

  tmp_size <- file.info(PART_FILE)$size

  if (is.na(tmp_size) || tmp_size <= 0) {
    stop("STEP 01A FAIL: Downloaded file is empty.")
  }

  ok <- file.rename(
    PART_FILE,
    RAW_FILE
  )

  if (!ok) {
    stop("STEP 01A FAIL: Could not promote .part file to canonical raw file.")
  }

  STATUS <- "DOWNLOADED_NOW"

} else {

  STATUS <- "ALREADY_PRESENT_NOT_OVERWRITTEN"
}

# ------------------------------------------------------------
# File checks
# ------------------------------------------------------------

if (!file.exists(RAW_FILE)) {
  stop("STEP 01A FAIL: Canonical raw file missing.")
}

FILE_SIZE <- file.info(RAW_FILE)$size

if (is.na(FILE_SIZE) || FILE_SIZE <= 0) {
  stop("STEP 01A FAIL: Invalid raw-file size.")
}

cat("Calculating SHA-256...\n")

SHA256 <- digest::digest(
  RAW_FILE,
  algo = "sha256",
  file = TRUE,
  serialize = FALSE
)

if (nchar(SHA256) != 64L) {
  stop("STEP 01A FAIL: Invalid SHA-256.")
}

# ------------------------------------------------------------
# Save immutable raw inventory
# ------------------------------------------------------------

RAW_INVENTORY <- data.frame(
  dataset = "BrainSeq Phase II",
  artifact = "gene-level unfiltered processed expression object",
  filename = basename(RAW_FILE),
  local_path = RAW_FILE,
  source_url = BRAINSEQ_URL,
  download_status = STATUS,
  file_size_bytes = FILE_SIZE,
  file_size_mb = round(FILE_SIZE / 1024^2, 2),
  sha256 = SHA256,
  immutable_raw = TRUE,
  timestamp = format(
    Sys.time(),
    "%Y-%m-%d %H:%M:%S %z"
  ),
  stringsAsFactors = FALSE
)

assert_project_sentinel()

write.csv(
  RAW_INVENTORY,
  INVENTORY_FILE,
  row.names = FALSE
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

CHECK <- read.csv(
  INVENTORY_FILE,
  stringsAsFactors = FALSE
)

stopifnot(
  nrow(CHECK) == 1L,
  identical(CHECK$sha256, SHA256),
  file.exists(RAW_FILE)
)

cat("\n============================================================\n")
cat("STEP 01A — CELL 1 PASS\n")
cat("============================================================\n")
cat("Status:   ", STATUS, "\n", sep = "")
cat("File:     ", RAW_FILE, "\n", sep = "")
cat("Size:     ", round(FILE_SIZE / 1024^2, 2), " MB\n", sep = "")
cat("SHA-256:  ", SHA256, "\n", sep = "")
cat("Inventory:", " ", INVENTORY_FILE, "\n", sep = "")
cat("============================================================\n")

Calculating SHA-256...

STEP 01A — CELL 1 PASS
Status:   DOWNLOADED_NOW
File:     E:/SCZ_CROSSREGION_MSS_Q1/01_raw/brainseq_phase2/rse_gene_unfiltered.Rdata
Size:     290.6 MB
SHA-256:  434a3eb215cb339079cfd59621c198aea583ba17b13a27e19c1fb9e00b4b2ac2
Inventory: E:/SCZ_CROSSREGION_MSS_Q1/01_raw/brainseq_phase2/brainseq_phase2_raw_inventory.csv


In [16]:
# ============================================================
# STEP 01A — CELL 2
# LOAD BRAINSEQ RSE + STRUCTURE / METADATA AUDIT
# ============================================================

assert_project_sentinel()

RAW_FILE <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "brainseq_phase2",
  "rse_gene_unfiltered.Rdata"
)

INVENTORY_FILE <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "brainseq_phase2",
  "brainseq_phase2_raw_inventory.csv"
)

if (!file.exists(RAW_FILE)) {
  stop("STEP 01A CELL 2 FAIL: BrainSeq raw file is missing.")
}

# ------------------------------------------------------------
# 1. Verify raw-file integrity before loading
# ------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {
  stop("digest package is required.")
}

RAW_INVENTORY_CHECK <- read.csv(
  INVENTORY_FILE,
  stringsAsFactors = FALSE
)

CURRENT_SHA256 <- digest::digest(
  RAW_FILE,
  algo = "sha256",
  file = TRUE,
  serialize = FALSE
)

if (!identical(
  CURRENT_SHA256,
  RAW_INVENTORY_CHECK$sha256[1]
)) {
  stop(
    paste0(
      "RAW INTEGRITY FAIL:\n",
      "Current SHA-256 does not match acquisition inventory."
    )
  )
}

cat("Raw SHA-256 verification: PASS\n")

# ------------------------------------------------------------
# 2. Install minimal Bioconductor dependency if required
# ------------------------------------------------------------

if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages(
    "BiocManager",
    repos = "https://cloud.r-project.org"
  )
}

if (!requireNamespace("SummarizedExperiment", quietly = TRUE)) {

  cat("\nInstalling SummarizedExperiment...\n")

  BiocManager::install(
    "SummarizedExperiment",
    ask = FALSE,
    update = FALSE
  )
}

if (!requireNamespace("SummarizedExperiment", quietly = TRUE)) {
  stop(
    "STEP 01A CELL 2 FAIL: SummarizedExperiment could not be installed."
  )
}

suppressPackageStartupMessages(
  library(SummarizedExperiment)
)

# ------------------------------------------------------------
# 3. Load RData into isolated environment
#    (avoids polluting global workspace)
# ------------------------------------------------------------

BRAINSEQ_ENV <- new.env(
  parent = emptyenv()
)

LOADED_OBJECT_NAMES <- load(
  RAW_FILE,
  envir = BRAINSEQ_ENV
)

if (length(LOADED_OBJECT_NAMES) == 0L) {
  stop("STEP 01A CELL 2 FAIL: RData contained no objects.")
}

cat("\nObjects loaded from RData:\n")
print(LOADED_OBJECT_NAMES)

# ------------------------------------------------------------
# 4. Inspect object classes
# ------------------------------------------------------------

OBJECT_CLASS_TABLE <- data.frame(
  object_name = LOADED_OBJECT_NAMES,
  object_class = vapply(
    LOADED_OBJECT_NAMES,
    function(nm) {
      paste(
        class(
          get(
            nm,
            envir = BRAINSEQ_ENV
          )
        ),
        collapse = ";"
      )
    },
    character(1)
  ),
  stringsAsFactors = FALSE
)

cat("\nObject classes:\n")
print(
  OBJECT_CLASS_TABLE,
  row.names = FALSE
)

# ------------------------------------------------------------
# 5. Identify SummarizedExperiment/RangedSummarizedExperiment
# ------------------------------------------------------------

RSE_CANDIDATES <- LOADED_OBJECT_NAMES[
  vapply(
    LOADED_OBJECT_NAMES,
    function(nm) {

      obj <- get(
        nm,
        envir = BRAINSEQ_ENV
      )

      inherits(
        obj,
        "SummarizedExperiment"
      )

    },
    logical(1)
  )
]

if (length(RSE_CANDIDATES) == 0L) {
  stop(
    paste0(
      "STEP 01A CELL 2 FAIL:\n",
      "No SummarizedExperiment-compatible object found.\n",
      "Loaded objects: ",
      paste(LOADED_OBJECT_NAMES, collapse = ", ")
    )
  )
}

if (length(RSE_CANDIDATES) > 1L) {

  cat(
    "\nMultiple SummarizedExperiment objects detected:\n"
  )

  print(RSE_CANDIDATES)

  cat(
    "\nUsing the largest object by number of assay elements.\n"
  )

  candidate_sizes <- vapply(
    RSE_CANDIDATES,
    function(nm) {

      obj <- get(
        nm,
        envir = BRAINSEQ_ENV
      )

      prod(dim(obj))

    },
    numeric(1)
  )

  RSE_OBJECT_NAME <- RSE_CANDIDATES[
    which.max(candidate_sizes)
  ]

} else {

  RSE_OBJECT_NAME <- RSE_CANDIDATES[1]
}

BRAINSEQ_RSE <- get(
  RSE_OBJECT_NAME,
  envir = BRAINSEQ_ENV
)

# ------------------------------------------------------------
# 6. Core dimensions
# ------------------------------------------------------------

N_GENES <- nrow(BRAINSEQ_RSE)
N_SAMPLES <- ncol(BRAINSEQ_RSE)

ASSAY_NAMES <- SummarizedExperiment::assayNames(
  BRAINSEQ_RSE
)

COL_DATA <- as.data.frame(
  SummarizedExperiment::colData(
    BRAINSEQ_RSE
  ),
  stringsAsFactors = FALSE
)

ROW_DATA <- as.data.frame(
  SummarizedExperiment::rowData(
    BRAINSEQ_RSE
  ),
  stringsAsFactors = FALSE
)

N_METADATA_COLUMNS <- ncol(COL_DATA)
N_GENE_METADATA_COLUMNS <- ncol(ROW_DATA)

# ------------------------------------------------------------
# 7. Identify likely key metadata columns
# ------------------------------------------------------------

METADATA_COLUMNS <- names(COL_DATA)

KEY_PATTERN <- paste(
  c(
    "brnum",
    "donor",
    "subject",
    "sample",
    "region",
    "brain",
    "diagn",
    "disease",
    "(^|_)dx($|_)",
    "age",
    "sex",
    "gender",
    "race",
    "ances",
    "rin",
    "pmi",
    "ph",
    "batch",
    "library",
    "seq",
    "qc",
    "qsv",
    "toxic",
    "medication"
  ),
  collapse = "|"
)

KEY_METADATA_COLUMNS <- METADATA_COLUMNS[
  grepl(
    KEY_PATTERN,
    METADATA_COLUMNS,
    ignore.case = TRUE,
    perl = TRUE
  )
]

# ------------------------------------------------------------
# 8. Build compact metadata audit
# ------------------------------------------------------------

summarize_column <- function(x) {

  n_missing <- sum(is.na(x))

  values <- unique(
    as.character(
      x[!is.na(x)]
    )
  )

  n_unique <- length(values)

  preview <- if (n_unique == 0L) {

    NA_character_

  } else {

    paste(
      head(
        values,
        12
      ),
      collapse = " | "
    )
  }

  data.frame(
    n_missing = n_missing,
    n_unique = n_unique,
    example_values = preview,
    stringsAsFactors = FALSE
  )
}

if (length(KEY_METADATA_COLUMNS) > 0L) {

  KEY_METADATA_AUDIT <- do.call(
    rbind,
    lapply(
      KEY_METADATA_COLUMNS,
      function(col_name) {

        tmp <- summarize_column(
          COL_DATA[[col_name]]
        )

        data.frame(
          column = col_name,
          class = paste(
            class(
              COL_DATA[[col_name]]
            ),
            collapse = ";"
          ),
          tmp,
          stringsAsFactors = FALSE
        )
      }
    )
  )

  rownames(KEY_METADATA_AUDIT) <- NULL

} else {

  KEY_METADATA_AUDIT <- data.frame(
    column = character(0),
    class = character(0),
    n_missing = integer(0),
    n_unique = integer(0),
    example_values = character(0),
    stringsAsFactors = FALSE
  )
}

# ------------------------------------------------------------
# 9. Gene metadata audit
# ------------------------------------------------------------

GENE_METADATA_COLUMNS <- names(
  ROW_DATA
)

# ------------------------------------------------------------
# 10. Save audits
# ------------------------------------------------------------

OBJECT_AUDIT_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01a_brainseq_object_audit.csv"
)

METADATA_COLUMNS_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01a_brainseq_coldata_columns.csv"
)

KEY_METADATA_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01a_brainseq_key_metadata_audit.csv"
)

GENE_METADATA_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01a_brainseq_rowdata_columns.csv"
)

OBJECT_AUDIT <- data.frame(
  object_name = RSE_OBJECT_NAME,
  object_class = paste(
    class(BRAINSEQ_RSE),
    collapse = ";"
  ),
  n_genes = N_GENES,
  n_samples = N_SAMPLES,
  n_sample_metadata_columns = N_METADATA_COLUMNS,
  n_gene_metadata_columns = N_GENE_METADATA_COLUMNS,
  assay_names = paste(
    ASSAY_NAMES,
    collapse = " | "
  ),
  raw_sha256 = CURRENT_SHA256,
  stringsAsFactors = FALSE
)

COL_DATA_COLUMN_TABLE <- data.frame(
  column_index = seq_along(METADATA_COLUMNS),
  column_name = METADATA_COLUMNS,
  column_class = vapply(
    COL_DATA,
    function(x) {
      paste(class(x), collapse = ";")
    },
    character(1)
  ),
  stringsAsFactors = FALSE
)

ROW_DATA_COLUMN_TABLE <- data.frame(
  column_index = seq_along(GENE_METADATA_COLUMNS),
  column_name = GENE_METADATA_COLUMNS,
  stringsAsFactors = FALSE
)

assert_project_sentinel()

write.csv(
  OBJECT_AUDIT,
  OBJECT_AUDIT_PATH,
  row.names = FALSE
)

write.csv(
  COL_DATA_COLUMN_TABLE,
  METADATA_COLUMNS_PATH,
  row.names = FALSE
)

write.csv(
  KEY_METADATA_AUDIT,
  KEY_METADATA_PATH,
  row.names = FALSE
)

write.csv(
  ROW_DATA_COLUMN_TABLE,
  GENE_METADATA_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 11. Basic sanity checks
# ------------------------------------------------------------

stopifnot(
  N_GENES > 0L,
  N_SAMPLES > 0L,
  N_METADATA_COLUMNS > 0L,
  file.exists(OBJECT_AUDIT_PATH),
  file.exists(METADATA_COLUMNS_PATH),
  file.exists(KEY_METADATA_PATH),
  file.exists(GENE_METADATA_PATH)
)

# ------------------------------------------------------------
# 12. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 01A — CELL 2: BRAINSEQ OBJECT AUDIT\n")
cat("============================================================\n")

cat("Selected object:        ", RSE_OBJECT_NAME, "\n", sep = "")
cat("Class:                  ", paste(class(BRAINSEQ_RSE), collapse = " / "), "\n", sep = "")
cat("Genes/features:         ", format(N_GENES, big.mark = ","), "\n", sep = "")
cat("Samples:                ", format(N_SAMPLES, big.mark = ","), "\n", sep = "")
cat("Assays:                 ", paste(ASSAY_NAMES, collapse = ", "), "\n", sep = "")
cat("Sample metadata cols:   ", N_METADATA_COLUMNS, "\n", sep = "")
cat("Gene metadata cols:     ", N_GENE_METADATA_COLUMNS, "\n", sep = "")
cat("Key candidate cols:     ", length(KEY_METADATA_COLUMNS), "\n", sep = "")

cat("\n---------------- KEY METADATA CANDIDATES ----------------\n")

if (nrow(KEY_METADATA_AUDIT) > 0L) {

  print(
    KEY_METADATA_AUDIT,
    row.names = FALSE
  )

} else {

  cat("No likely metadata columns detected by current pattern.\n")
}

cat("\n---------------- ALL SAMPLE METADATA COLUMNS ----------------\n")

print(
  METADATA_COLUMNS
)

cat("\n---------------- GENE METADATA COLUMNS ----------------\n")

print(
  GENE_METADATA_COLUMNS
)

cat("\n============================================================\n")
cat("STEP 01A — CELL 2 PASS\n")
cat("============================================================\n")

Raw SHA-256 verification: PASS
- Querying repositories for available source packages ... Done!
The following package(s) will be installed:
- BiocManager [1.30.27]
These packages will be installed into "E:/SCZ_CROSSREGION_MSS_Q1/renv/library/windows/R-4.5/x86_64-w64-mingw32".

# Downloading packages -------------------------------------------------------
✔ BiocManager 1.30.27                      [664 kB in 0.59s]           
Successfully downloaded 1 package in 1.6 seconds.                               

# Installing packages --------------------------------------------------------
✔ BiocManager 1.30.27                      [installed binary]          

Installing SummarizedExperiment...


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31 ucrt)

Installing package(s) 'BiocVersion', 'SummarizedExperiment'

also installing the dependencies 'XVector', 'matrixStats', 'generics', 'abind', 'SparseArray', 'MatrixGenerics', 'GenomicRanges', 'Biobase', 'BiocGenerics', 'S4Vectors', 'IRanges', 'Seqinfo', 'S4Arrays', 'DelayedArray'




package 'XVector' successfully unpacked and MD5 sums checked
package 'matrixStats' successfully unpacked and MD5 sums checked
package 'generics' successfully unpacked and MD5 sums checked
package 'abind' successfully unpacked and MD5 sums checked
package 'SparseArray' successfully unpacked and MD5 sums checked
package 'MatrixGenerics' successfully unpacked and MD5 sums checked
package 'GenomicRanges' successfully unpacked and MD5 sums checked
package 'Biobase' successfully unpacked and MD5 sums checked
package 'BiocGenerics' successfully unpacked and MD5 sums checked
package 'S4Vectors' successfully unpacked and MD5 sums checked
package 'IRanges' successfully unpacked and MD5 sums checked
package 'Seqinfo' successfully unpacked and MD5 sums checked
package 'S4Arrays' successfully unpacked and MD5 sums checked
package 'DelayedArray' successfully unpacked and MD5 sums checked
package 'BiocVersion' successfully unpacked and MD5 sums checked
package 'SummarizedExperiment' successfully unpa

Warning message:
"package 'matrixStats' was built under R version 4.5.3"
Warning message:
"package 'generics' was built under R version 4.5.3"
Warning message:
"package 'S4Vectors' was built under R version 4.5.3"
Warning message:
"package 'Biobase' was built under R version 4.5.3"



Objects loaded from RData:
[1] "rse_gene"

Object classes:
 object_name               object_class
    rse_gene RangedSummarizedExperiment

STEP 01A — CELL 2: BRAINSEQ OBJECT AUDIT
Selected object:        rse_gene
Class:                  RangedSummarizedExperiment
Genes/features:         58,037
Samples:                900
Assays:                 counts, rpkm
Sample metadata cols:   54
Gene metadata cols:     11
Key candidate cols:     20

---------------- KEY METADATA CANDIDATES ----------------
         column     class n_missing n_unique
      SAMPLE_ID      list         0      900
  FQCbasicStats      list         0        1
    perBaseQual      list         0       11
     perSeqQual      list         0        1
  SeqLengthDist      list         0        1
 SeqDuplication      list         0       17
    OverrepSeqs      list         0        9
    phred100_R1      list         0       45
   phredGT30_R1      list         0      900
   phredGT35_R1      list         0      900
   

In [17]:
# ============================================================
# STEP 01B — CELL 1
# BRAINSEQ CORE COHORT + DONOR + REGION-PAIRING AUDIT
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# 1. Recover BrainSeq object if kernel state was lost
# ------------------------------------------------------------

if (!exists("BRAINSEQ_RSE")) {

  RAW_FILE <- file.path(
    PROJECT_ROOT,
    "01_raw",
    "brainseq_phase2",
    "rse_gene_unfiltered.Rdata"
  )

  if (!file.exists(RAW_FILE)) {
    stop("STEP 01B FAIL: BrainSeq raw file is missing.")
  }

  if (!requireNamespace("SummarizedExperiment", quietly = TRUE)) {
    stop("STEP 01B FAIL: SummarizedExperiment is unavailable.")
  }

  tmp_env <- new.env(parent = emptyenv())

  obj_names <- load(
    RAW_FILE,
    envir = tmp_env
  )

  rse_names <- obj_names[
    vapply(
      obj_names,
      function(x) {
        inherits(
          get(x, envir = tmp_env),
          "SummarizedExperiment"
        )
      },
      logical(1)
    )
  ]

  if (length(rse_names) != 1L) {
    stop(
      paste0(
        "STEP 01B FAIL: Expected exactly one RSE object; found ",
        length(rse_names)
      )
    )
  }

  BRAINSEQ_RSE <- get(
    rse_names[1],
    envir = tmp_env
  )
}

# ------------------------------------------------------------
# 2. Extract metadata
# ------------------------------------------------------------

CD <- as.data.frame(
  SummarizedExperiment::colData(BRAINSEQ_RSE),
  stringsAsFactors = FALSE
)

REQUIRED_META <- c(
  "RNum",
  "BrNum",
  "Region",
  "Dx",
  "Age",
  "Sex",
  "Race",
  "RIN",
  "SAMPLE_ID"
)

MISSING_META <- setdiff(
  REQUIRED_META,
  names(CD)
)

if (length(MISSING_META) > 0L) {
  stop(
    paste0(
      "STEP 01B FAIL: Missing metadata columns:\n- ",
      paste(MISSING_META, collapse = "\n- ")
    )
  )
}

# ------------------------------------------------------------
# 3. Construct clean sample-level core metadata
# ------------------------------------------------------------

RSE_COLNAME <- colnames(BRAINSEQ_RSE)

if (is.null(RSE_COLNAME)) {
  RSE_COLNAME <- paste0(
    "RSE_COLUMN_",
    seq_len(ncol(BRAINSEQ_RSE))
  )
}

SAMPLE_CORE <- data.frame(

  sample_index = seq_len(nrow(CD)),

  rse_colname = as.character(
    RSE_COLNAME
  ),

  RNum = trimws(
    as.character(CD$RNum)
  ),

  BrNum = trimws(
    as.character(CD$BrNum)
  ),

  Region = trimws(
    as.character(CD$Region)
  ),

  Dx = trimws(
    as.character(CD$Dx)
  ),

  Age = suppressWarnings(
    as.numeric(CD$Age)
  ),

  Sex = trimws(
    as.character(CD$Sex)
  ),

  Race = trimws(
    as.character(CD$Race)
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 4. Hard structural checks
# ------------------------------------------------------------

if (anyNA(SAMPLE_CORE$BrNum) ||
    any(SAMPLE_CORE$BrNum == "")) {
  stop("STEP 01B FAIL: Missing donor IDs detected.")
}

if (!setequal(
  unique(SAMPLE_CORE$Region),
  c("DLPFC", "HIPPO")
)) {
  stop(
    paste0(
      "Unexpected Region values: ",
      paste(
        sort(unique(SAMPLE_CORE$Region)),
        collapse = ", "
      )
    )
  )
}

if (!setequal(
  unique(SAMPLE_CORE$Dx),
  c("Control", "Schizo")
)) {
  stop(
    paste0(
      "Unexpected diagnosis values: ",
      paste(
        sort(unique(SAMPLE_CORE$Dx)),
        collapse = ", "
      )
    )
  )
}

# ------------------------------------------------------------
# 5. Duplicate sample identifier audit
# ------------------------------------------------------------

N_DUP_RNUM <- sum(
  duplicated(SAMPLE_CORE$RNum)
)

N_DUP_RSE_COLNAME <- sum(
  duplicated(SAMPLE_CORE$rse_colname)
)

# ------------------------------------------------------------
# 6. Helper functions
# ------------------------------------------------------------

n_unique_nonmissing <- function(x) {

  x <- x[
    !is.na(x) &
    x != ""
  ]

  length(unique(x))
}

collapse_unique <- function(x) {

  x <- unique(
    x[
      !is.na(x) &
      x != ""
    ]
  )

  if (length(x) == 0L) {
    return(NA_character_)
  }

  paste(
    sort(x),
    collapse = "|"
  )
}

# ------------------------------------------------------------
# 7. Build donor-level table
# ------------------------------------------------------------

DONOR_SPLIT <- split(
  seq_len(nrow(SAMPLE_CORE)),
  SAMPLE_CORE$BrNum
)

DONOR_SUMMARY <- do.call(
  rbind,
  lapply(
    names(DONOR_SPLIT),
    function(donor_id) {

      z <- SAMPLE_CORE[
        DONOR_SPLIT[[donor_id]],
        ,
        drop = FALSE
      ]

      has_dlpfc <- any(
        z$Region == "DLPFC"
      )

      has_hippo <- any(
        z$Region == "HIPPO"
      )

      availability <- if (
        has_dlpfc && has_hippo
      ) {
        "DLPFC_HIPPO"
      } else if (has_dlpfc) {
        "DLPFC_only"
      } else if (has_hippo) {
        "HIPPO_only"
      } else {
        "NEITHER"
      }

      data.frame(

        BrNum = donor_id,

        n_samples = nrow(z),

        diagnosis = collapse_unique(z$Dx),

        n_unique_diagnosis =
          n_unique_nonmissing(z$Dx),

        age = if (
          n_unique_nonmissing(
            as.character(z$Age)
          ) == 1L
        ) {
          unique(
            z$Age[
              !is.na(z$Age)
            ]
          )[1]
        } else {
          NA_real_
        },

        n_unique_age =
          n_unique_nonmissing(
            as.character(z$Age)
          ),

        sex = collapse_unique(z$Sex),

        n_unique_sex =
          n_unique_nonmissing(z$Sex),

        race = collapse_unique(z$Race),

        n_unique_race =
          n_unique_nonmissing(z$Race),

        has_DLPFC = has_dlpfc,

        has_HIPPO = has_hippo,

        n_DLPFC = sum(
          z$Region == "DLPFC"
        ),

        n_HIPPO = sum(
          z$Region == "HIPPO"
        ),

        region_availability_class =
          availability,

        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(DONOR_SUMMARY) <- NULL

# ------------------------------------------------------------
# 8. Donor consistency audit
# ------------------------------------------------------------

DONOR_CONSISTENCY <- data.frame(

  issue = c(
    "Diagnosis differs within donor",
    "Sex differs within donor",
    "Race differs within donor",
    "Age differs within donor",
    "More than one DLPFC column per donor",
    "More than one HIPPO column per donor"
  ),

  n_donors = c(
    sum(DONOR_SUMMARY$n_unique_diagnosis > 1L),
    sum(DONOR_SUMMARY$n_unique_sex > 1L),
    sum(DONOR_SUMMARY$n_unique_race > 1L),
    sum(DONOR_SUMMARY$n_unique_age > 1L),
    sum(DONOR_SUMMARY$n_DLPFC > 1L),
    sum(DONOR_SUMMARY$n_HIPPO > 1L)
  ),

  stringsAsFactors = FALSE
)

# Diagnosis inconsistency is fatal.
if (
  any(
    DONOR_SUMMARY$n_unique_diagnosis > 1L
  )
) {
  stop(
    "STEP 01B FAIL: At least one donor has inconsistent diagnosis labels."
  )
}

# ------------------------------------------------------------
# 9. Region × diagnosis sample counts
# ------------------------------------------------------------

REGION_DX_SAMPLE_COUNTS <- as.data.frame(
  table(
    Region = SAMPLE_CORE$Region,
    Dx = SAMPLE_CORE$Dx
  ),
  stringsAsFactors = FALSE
)

names(REGION_DX_SAMPLE_COUNTS)[3] <- "n_samples"

# ------------------------------------------------------------
# 10. Donor region-availability × diagnosis counts
# ------------------------------------------------------------

DONOR_REGION_DX_COUNTS <- as.data.frame(
  table(
    region_availability_class =
      DONOR_SUMMARY$region_availability_class,
    Dx =
      DONOR_SUMMARY$diagnosis
  ),
  stringsAsFactors = FALSE
)

names(DONOR_REGION_DX_COUNTS)[3] <- "n_donors"

# ------------------------------------------------------------
# 11. Preliminary adult (>17) candidate cohort
#
# This is PRE-QC only.
# No sample is permanently included/excluded here.
# ------------------------------------------------------------

PRE_QC_ELIGIBLE <- (
  !is.na(SAMPLE_CORE$Age) &
  SAMPLE_CORE$Age > 17 &
  SAMPLE_CORE$Dx %in% c(
    "Control",
    "Schizo"
  ) &
  SAMPLE_CORE$Region %in% c(
    "DLPFC",
    "HIPPO"
  )
)

SAMPLE_CORE$preQC_primary_candidate <- PRE_QC_ELIGIBLE

PREQC_SAMPLE <- SAMPLE_CORE[
  PRE_QC_ELIGIBLE,
  ,
  drop = FALSE
]

PREQC_REGION_DX_COUNTS <- as.data.frame(
  table(
    Region = PREQC_SAMPLE$Region,
    Dx = PREQC_SAMPLE$Dx
  ),
  stringsAsFactors = FALSE
)

names(PREQC_REGION_DX_COUNTS)[3] <- "n_samples"

PREQC_DONOR_IDS <- unique(
  PREQC_SAMPLE$BrNum
)

# ------------------------------------------------------------
# 12. Paired-donor counts
# ------------------------------------------------------------

PAIRED_DONORS <- DONOR_SUMMARY[
  DONOR_SUMMARY$region_availability_class ==
    "DLPFC_HIPPO",
  ,
  drop = FALSE
]

PAIRED_DX_COUNTS <- as.data.frame(
  table(
    Dx = PAIRED_DONORS$diagnosis
  ),
  stringsAsFactors = FALSE
)

names(PAIRED_DX_COUNTS)[2] <- "n_paired_donors"

# ------------------------------------------------------------
# 13. Audit list-valued metadata
# ------------------------------------------------------------

audit_list_column <- function(x, variable_name) {

  if (is.list(x)) {

    element_lengths <- lengths(x)

    n_unique_values <- vapply(
      x,
      function(z) {

        z <- z[!is.na(z)]

        length(
          unique(
            as.character(z)
          )
        )
      },
      integer(1)
    )

  } else {

    element_lengths <- rep(
      1L,
      length(x)
    )

    n_unique_values <- vapply(
      x,
      function(z) {
        if (is.na(z)) 0L else 1L
      },
      integer(1)
    )
  }

  data.frame(
    variable = variable_name,
    is_list_column = is.list(x),
    n_rows = length(x),
    min_element_length = min(element_lengths),
    max_element_length = max(element_lengths),
    n_rows_multiple_values =
      sum(element_lengths > 1L),
    n_rows_multiple_unique_values =
      sum(n_unique_values > 1L),
    stringsAsFactors = FALSE
  )
}

LIST_METADATA_AUDIT <- rbind(
  audit_list_column(
    CD$SAMPLE_ID,
    "SAMPLE_ID"
  ),
  audit_list_column(
    CD$RIN,
    "RIN"
  )
)

# ------------------------------------------------------------
# 14. Save canonical audit outputs
# ------------------------------------------------------------

SAMPLE_CORE_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_brainseq_sample_core_metadata.csv"
)

DONOR_SUMMARY_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_brainseq_donor_summary.csv"
)

REGION_DX_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_region_by_diagnosis_sample_counts.csv"
)

DONOR_REGION_DX_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_donor_region_availability_by_diagnosis.csv"
)

PREQC_COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_preQC_adult_region_by_diagnosis.csv"
)

CONSISTENCY_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_donor_consistency_audit.csv"
)

LIST_AUDIT_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_list_metadata_audit.csv"
)

assert_project_sentinel()

write.csv(
  SAMPLE_CORE,
  SAMPLE_CORE_PATH,
  row.names = FALSE
)

write.csv(
  DONOR_SUMMARY,
  DONOR_SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  REGION_DX_SAMPLE_COUNTS,
  REGION_DX_PATH,
  row.names = FALSE
)

write.csv(
  DONOR_REGION_DX_COUNTS,
  DONOR_REGION_DX_PATH,
  row.names = FALSE
)

write.csv(
  PREQC_REGION_DX_COUNTS,
  PREQC_COUNTS_PATH,
  row.names = FALSE
)

write.csv(
  DONOR_CONSISTENCY,
  CONSISTENCY_PATH,
  row.names = FALSE
)

write.csv(
  LIST_METADATA_AUDIT,
  LIST_AUDIT_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 15. Final sanity checks
# ------------------------------------------------------------

stopifnot(
  nrow(SAMPLE_CORE) == 900L,
  length(unique(SAMPLE_CORE$BrNum)) == 551L,
  nrow(DONOR_SUMMARY) == 551L,
  sum(REGION_DX_SAMPLE_COUNTS$n_samples) == 900L
)

# ------------------------------------------------------------
# 16. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 01B — BRAINSEQ CORE COHORT AUDIT\n")
cat("============================================================\n")

cat("Total expression columns/samples: ", nrow(SAMPLE_CORE), "\n", sep = "")
cat("Unique donors:                    ", nrow(DONOR_SUMMARY), "\n", sep = "")
cat("Duplicated RNum values:           ", N_DUP_RNUM, "\n", sep = "")
cat("Duplicated RSE column names:      ", N_DUP_RSE_COLNAME, "\n", sep = "")

cat("\n--- SAMPLE COUNTS: REGION x DIAGNOSIS ---\n")
print(
  REGION_DX_SAMPLE_COUNTS,
  row.names = FALSE
)

cat("\n--- DONORS: REGION AVAILABILITY x DIAGNOSIS ---\n")
print(
  DONOR_REGION_DX_COUNTS,
  row.names = FALSE
)

cat("\n--- PAIRED DLPFC + HIPPO DONORS ---\n")
cat("Total paired donors: ", nrow(PAIRED_DONORS), "\n", sep = "")
print(
  PAIRED_DX_COUNTS,
  row.names = FALSE
)

cat("\n--- DONOR CONSISTENCY AUDIT ---\n")
print(
  DONOR_CONSISTENCY,
  row.names = FALSE
)

cat("\n--- PRE-QC ADULT (>17) CANDIDATE COHORT ---\n")
cat("Samples: ", nrow(PREQC_SAMPLE), "\n", sep = "")
cat("Donors:  ", length(PREQC_DONOR_IDS), "\n", sep = "")

print(
  PREQC_REGION_DX_COUNTS,
  row.names = FALSE
)

cat("\n--- LIST-COLUMN / TECHNICAL-REPLICATE AUDIT ---\n")
print(
  LIST_METADATA_AUDIT,
  row.names = FALSE
)

cat("\n============================================================\n")
cat("STEP 01B — CELL 1 PASS\n")
cat("============================================================\n")


STEP 01B — BRAINSEQ CORE COHORT AUDIT
Total expression columns/samples: 900
Unique donors:                    551
Duplicated RNum values:           0
Duplicated RSE column names:      0

--- SAMPLE COUNTS: REGION x DIAGNOSIS ---
 Region      Dx n_samples
  DLPFC Control       300
  HIPPO Control       314
  DLPFC  Schizo       153
  HIPPO  Schizo       133

--- DONORS: REGION AVAILABILITY x DIAGNOSIS ---
 region_availability_class      Dx n_donors
               DLPFC_HIPPO Control      240
                DLPFC_only Control       60
                HIPPO_only Control       74
               DLPFC_HIPPO  Schizo      109
                DLPFC_only  Schizo       44
                HIPPO_only  Schizo       24

--- PAIRED DLPFC + HIPPO DONORS ---
Total paired donors: 349
      Dx n_paired_donors
 Control             240
  Schizo             109

--- DONOR CONSISTENCY AUDIT ---
                                issue n_donors
       Diagnosis differs within donor        0
             Sex di

In [18]:
# ============================================================
# STEP 01B — CELL 2
# AGE DISCREPANCY + RIN + TECHNICAL SAMPLE AUDIT
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# 1. Extract scalar RIN safely
# ------------------------------------------------------------

extract_single_numeric <- function(x) {

  if (length(x) == 0L) {
    return(NA_real_)
  }

  x_num <- suppressWarnings(
    as.numeric(x)
  )

  x_num <- x_num[
    !is.na(x_num)
  ]

  if (length(x_num) == 0L) {
    return(NA_real_)
  }

  if (length(unique(x_num)) > 1L) {
    stop(
      "Unexpected multiple distinct numeric values in a supposedly scalar field."
    )
  }

  unique(x_num)[1]
}

RIN_NUMERIC <- vapply(
  CD$RIN,
  extract_single_numeric,
  numeric(1)
)

if (length(RIN_NUMERIC) != nrow(SAMPLE_CORE)) {
  stop("RIN extraction length mismatch.")
}

SAMPLE_CORE$RIN_numeric <- RIN_NUMERIC


# ------------------------------------------------------------
# 2. Technical SAMPLE_ID multiplicity
# ------------------------------------------------------------

SAMPLE_CORE$n_technical_sample_ids <- lengths(
  CD$SAMPLE_ID
)

SAMPLE_CORE$technical_sample_ids <- vapply(
  CD$SAMPLE_ID,
  function(x) {
    paste(
      as.character(x),
      collapse = " | "
    )
  },
  character(1)
)


# ------------------------------------------------------------
# 3. Donor-level age audit
# ------------------------------------------------------------

AGE_AUDIT <- do.call(
  rbind,
  lapply(
    split(
      SAMPLE_CORE,
      SAMPLE_CORE$BrNum
    ),
    function(z) {

      ages <- z$Age[
        !is.na(z$Age)
      ]

      regions <- z$Region[
        !is.na(z$Age)
      ]

      age_dlpfc <- if (
        any(regions == "DLPFC")
      ) {
        ages[
          which(regions == "DLPFC")[1]
        ]
      } else {
        NA_real_
      }

      age_hippo <- if (
        any(regions == "HIPPO")
      ) {
        ages[
          which(regions == "HIPPO")[1]
        ]
      } else {
        NA_real_
      }

      data.frame(
        BrNum = z$BrNum[1],

        Dx = z$Dx[1],

        n_samples = nrow(z),

        has_DLPFC = any(
          z$Region == "DLPFC"
        ),

        has_HIPPO = any(
          z$Region == "HIPPO"
        ),

        age_DLPFC = age_dlpfc,

        age_HIPPO = age_hippo,

        age_min = if (
          length(ages) > 0L
        ) min(ages) else NA_real_,

        age_max = if (
          length(ages) > 0L
        ) max(ages) else NA_real_,

        age_range = if (
          length(ages) > 0L
        ) max(ages) - min(ages) else NA_real_,

        n_unique_age = length(
          unique(ages)
        ),

        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(AGE_AUDIT) <- NULL


# ------------------------------------------------------------
# 4. Focus on paired donors
# ------------------------------------------------------------

PAIRED_AGE_AUDIT <- AGE_AUDIT[
  AGE_AUDIT$has_DLPFC &
  AGE_AUDIT$has_HIPPO,
  ,
  drop = FALSE
]

PAIRED_AGE_AUDIT$age_difference_DLPFC_minus_HIPPO <-
  PAIRED_AGE_AUDIT$age_DLPFC -
  PAIRED_AGE_AUDIT$age_HIPPO


AGE_DISCREPANT <- PAIRED_AGE_AUDIT[
  !is.na(PAIRED_AGE_AUDIT$age_difference_DLPFC_minus_HIPPO) &
  PAIRED_AGE_AUDIT$age_difference_DLPFC_minus_HIPPO != 0,
  ,
  drop = FALSE
]


# ------------------------------------------------------------
# 5. Summarize magnitude of age discrepancies
# ------------------------------------------------------------

ABS_AGE_DIFF <- abs(
  AGE_DISCREPANT$age_difference_DLPFC_minus_HIPPO
)

AGE_DIFF_SUMMARY <- data.frame(
  metric = c(
    "n_paired_donors",
    "n_paired_with_age_difference",
    "min_abs_difference",
    "median_abs_difference",
    "mean_abs_difference",
    "max_abs_difference",
    "n_abs_diff_le_0.01",
    "n_abs_diff_le_0.1",
    "n_abs_diff_le_1",
    "n_abs_diff_gt_1"
  ),

  value = c(
    nrow(PAIRED_AGE_AUDIT),
    nrow(AGE_DISCREPANT),

    if (length(ABS_AGE_DIFF) > 0L)
      min(ABS_AGE_DIFF)
    else NA_real_,

    if (length(ABS_AGE_DIFF) > 0L)
      median(ABS_AGE_DIFF)
    else NA_real_,

    if (length(ABS_AGE_DIFF) > 0L)
      mean(ABS_AGE_DIFF)
    else NA_real_,

    if (length(ABS_AGE_DIFF) > 0L)
      max(ABS_AGE_DIFF)
    else NA_real_,

    sum(ABS_AGE_DIFF <= 0.01),
    sum(ABS_AGE_DIFF <= 0.1),
    sum(ABS_AGE_DIFF <= 1),
    sum(ABS_AGE_DIFF > 1)
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 6. RIN summary by region and diagnosis
# ------------------------------------------------------------

RIN_SUMMARY <- do.call(
  rbind,
  lapply(
    split(
      SAMPLE_CORE,
      interaction(
        SAMPLE_CORE$Region,
        SAMPLE_CORE$Dx,
        drop = TRUE
      )
    ),
    function(z) {

      data.frame(
        Region = z$Region[1],
        Dx = z$Dx[1],

        n = nrow(z),

        n_missing_RIN = sum(
          is.na(z$RIN_numeric)
        ),

        mean_RIN = mean(
          z$RIN_numeric,
          na.rm = TRUE
        ),

        sd_RIN = sd(
          z$RIN_numeric,
          na.rm = TRUE
        ),

        min_RIN = min(
          z$RIN_numeric,
          na.rm = TRUE
        ),

        median_RIN = median(
          z$RIN_numeric,
          na.rm = TRUE
        ),

        max_RIN = max(
          z$RIN_numeric,
          na.rm = TRUE
        ),

        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(RIN_SUMMARY) <- NULL


# ------------------------------------------------------------
# 7. Technical sample multiplicity summary
# ------------------------------------------------------------

TECH_SAMPLE_SUMMARY <- as.data.frame(
  table(
    n_technical_sample_ids =
      SAMPLE_CORE$n_technical_sample_ids
  ),
  stringsAsFactors = FALSE
)

names(TECH_SAMPLE_SUMMARY)[2] <- "n_expression_columns"


# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

AGE_AUDIT_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_age_discrepancy_audit.csv"
)

RIN_SUMMARY_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_RIN_summary.csv"
)

TECH_SAMPLE_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_technical_sample_multiplicity.csv"
)

SAMPLE_CORE_UPDATED_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01b_brainseq_sample_core_metadata_with_RIN.csv"
)

assert_project_sentinel()

write.csv(
  AGE_AUDIT,
  AGE_AUDIT_PATH,
  row.names = FALSE
)

write.csv(
  RIN_SUMMARY,
  RIN_SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  TECH_SAMPLE_SUMMARY,
  TECH_SAMPLE_PATH,
  row.names = FALSE
)

write.csv(
  SAMPLE_CORE,
  SAMPLE_CORE_UPDATED_PATH,
  row.names = FALSE
)


# ------------------------------------------------------------
# 9. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 01B — AGE / RIN / TECHNICAL SAMPLE AUDIT\n")
cat("============================================================\n")

cat("\n--- AGE DIFFERENCE SUMMARY ---\n")
print(
  AGE_DIFF_SUMMARY,
  row.names = FALSE
)

cat("\n--- LARGEST AGE DISCREPANCIES ---\n")

if (nrow(AGE_DISCREPANT) > 0L) {

  AGE_DISCREPANT_ORDERED <- AGE_DISCREPANT[
    order(
      -abs(
        AGE_DISCREPANT$age_difference_DLPFC_minus_HIPPO
      )
    ),
  ]

  print(
    head(
      AGE_DISCREPANT_ORDERED,
      25
    ),
    row.names = FALSE
  )

} else {

  cat("No paired-donor age discrepancies detected.\n")
}

cat("\n--- RIN SUMMARY ---\n")
print(
  RIN_SUMMARY,
  row.names = FALSE
)

cat("\n--- TECHNICAL SAMPLE MULTIPLICITY ---\n")
print(
  TECH_SAMPLE_SUMMARY,
  row.names = FALSE
)

cat("\nRIN missing overall: ",
    sum(is.na(SAMPLE_CORE$RIN_numeric)),
    " / ",
    nrow(SAMPLE_CORE),
    "\n",
    sep = "")

cat("Expression columns with >1 technical SAMPLE_ID: ",
    sum(SAMPLE_CORE$n_technical_sample_ids > 1L),
    "\n",
    sep = "")

cat("\n============================================================\n")
cat("STEP 01B — CELL 2 PASS\n")
cat("============================================================\n")


STEP 01B — AGE / RIN / TECHNICAL SAMPLE AUDIT

--- AGE DIFFERENCE SUMMARY ---
                       metric        value
              n_paired_donors 3.490000e+02
 n_paired_with_age_difference 1.260000e+02
           min_abs_difference 4.110000e-06
        median_abs_difference 1.623000e-03
          mean_abs_difference 1.981162e-03
           max_abs_difference 4.964000e-03
           n_abs_diff_le_0.01 1.260000e+02
            n_abs_diff_le_0.1 1.260000e+02
              n_abs_diff_le_1 1.260000e+02
              n_abs_diff_gt_1 0.000000e+00

--- LARGEST AGE DISCREPANCIES ---
  BrNum      Dx n_samples has_DLPFC has_HIPPO age_DLPFC age_HIPPO  age_min
 Br1879 Control         2      TRUE      TRUE  16.71504     16.72 16.71504
 Br2337  Schizo         2      TRUE      TRUE  33.71482     33.71 33.71000
 Br1004 Control         2      TRUE      TRUE  21.00537     21.01 21.00537
 Br2038  Schizo         2      TRUE      TRUE  44.64462     44.64 44.64000
 Br1416  Schizo         2      TRUE   

In [19]:
# ============================================================
# STEP 01C — CELL 1
# PRIMARY ADULT COHORT CONSTRUCTION + SAMPLE QC AUDIT
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# 1. Canonical donor age
# ------------------------------------------------------------

DONOR_AGE <- aggregate(
  Age ~ BrNum,
  data = SAMPLE_CORE,
  FUN = function(x) {
    mean(
      x[!is.na(x)]
    )
  }
)

names(DONOR_AGE)[2] <- "donor_age"

# ------------------------------------------------------------
# 2. Attach canonical donor age to samples
# ------------------------------------------------------------

SAMPLE_COHORT <- merge(
  SAMPLE_CORE,
  DONOR_AGE,
  by = "BrNum",
  all.x = TRUE,
  sort = FALSE
)

# restore original sample order
SAMPLE_COHORT <- SAMPLE_COHORT[
  match(
    SAMPLE_CORE$sample_index,
    SAMPLE_COHORT$sample_index
  ),
]

rownames(SAMPLE_COHORT) <- NULL

# ------------------------------------------------------------
# 3. Primary eligibility
# ------------------------------------------------------------

SAMPLE_COHORT$adult_eligible <- (
  !is.na(SAMPLE_COHORT$donor_age) &
  SAMPLE_COHORT$donor_age > 17
)

SAMPLE_COHORT$diagnosis_eligible <- (
  SAMPLE_COHORT$Dx %in% c(
    "Control",
    "Schizo"
  )
)

SAMPLE_COHORT$region_eligible <- (
  SAMPLE_COHORT$Region %in% c(
    "DLPFC",
    "HIPPO"
  )
)

SAMPLE_COHORT$primary_preQC_eligible <- (
  SAMPLE_COHORT$adult_eligible &
  SAMPLE_COHORT$diagnosis_eligible &
  SAMPLE_COHORT$region_eligible
)

# ------------------------------------------------------------
# 4. Technical QC variables
# ------------------------------------------------------------

QC_VARIABLES <- c(
  "RIN",
  "numReads",
  "numMapped",
  "numUnmapped",
  "overallMapRate",
  "concordMapRate",
  "totalMapped",
  "mitoMapped",
  "mitoRate",
  "totalAssignedGene",
  "rRNA_rate"
)

missing_qc_vars <- setdiff(
  QC_VARIABLES,
  names(CD)
)

if (length(missing_qc_vars) > 0L) {
  stop(
    paste0(
      "Missing QC variables:\n- ",
      paste(
        missing_qc_vars,
        collapse = "\n- "
      )
    )
  )
}

# ------------------------------------------------------------
# 5. Robust scalar extractor for list/scalar metadata
# ------------------------------------------------------------

extract_numeric_scalar <- function(x) {

  if (length(x) == 0L) {
    return(NA_real_)
  }

  y <- suppressWarnings(
    as.numeric(x)
  )

  y <- y[!is.na(y)]

  if (length(y) == 0L) {
    return(NA_real_)
  }

  # If multiple technical values exist,
  # use mean only for audit purposes.
  mean(y)
}

QC_MATRIX <- data.frame(
  sample_index = seq_len(nrow(CD)),
  stringsAsFactors = FALSE
)

for (v in QC_VARIABLES) {

  QC_MATRIX[[v]] <- vapply(
    CD[[v]],
    extract_numeric_scalar,
    numeric(1)
  )
}

# ------------------------------------------------------------
# 6. Merge QC into cohort table
# ------------------------------------------------------------

SAMPLE_COHORT <- merge(
  SAMPLE_COHORT,
  QC_MATRIX,
  by = "sample_index",
  all.x = TRUE,
  sort = FALSE
)

SAMPLE_COHORT <- SAMPLE_COHORT[
  order(SAMPLE_COHORT$sample_index),
]

rownames(SAMPLE_COHORT) <- NULL

# ------------------------------------------------------------
# 7. QC completeness summary
# ------------------------------------------------------------

QC_COMPLETENESS <- data.frame(

  variable = QC_VARIABLES,

  n_missing = vapply(
    QC_VARIABLES,
    function(v) {
      sum(
        is.na(
          SAMPLE_COHORT[[v]]
        )
      )
    },
    integer(1)
  ),

  pct_missing = vapply(
    QC_VARIABLES,
    function(v) {
      mean(
        is.na(
          SAMPLE_COHORT[[v]]
        )
      ) * 100
    },
    numeric(1)
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 8. QC descriptive summary
# ------------------------------------------------------------

qc_summary_one <- function(x) {

  x <- x[
    is.finite(x)
  ]

  if (length(x) == 0L) {

    return(
      c(
        n = 0,
        min = NA,
        q01 = NA,
        q05 = NA,
        median = NA,
        mean = NA,
        q95 = NA,
        q99 = NA,
        max = NA
      )
    )
  }

  c(
    n = length(x),
    min = min(x),
    q01 = unname(
      quantile(
        x,
        0.01
      )
    ),
    q05 = unname(
      quantile(
        x,
        0.05
      )
    ),
    median = median(x),
    mean = mean(x),
    q95 = unname(
      quantile(
        x,
        0.95
      )
    ),
    q99 = unname(
      quantile(
        x,
        0.99
      )
    ),
    max = max(x)
  )
}

QC_SUMMARY <- do.call(
  rbind,
  lapply(
    QC_VARIABLES,
    function(v) {

      vals <- qc_summary_one(
        SAMPLE_COHORT[[v]]
      )

      data.frame(
        variable = v,
        t(vals),
        row.names = NULL,
        check.names = FALSE
      )
    }
  )
)

rownames(QC_SUMMARY) <- NULL

# ------------------------------------------------------------
# 9. Primary pre-QC cohort only
# ------------------------------------------------------------

PRIMARY_PREQC <- SAMPLE_COHORT[
  SAMPLE_COHORT$primary_preQC_eligible,
  ,
  drop = FALSE
]

PRIMARY_DONORS <- unique(
  PRIMARY_PREQC$BrNum
)

# ------------------------------------------------------------
# 10. Region x diagnosis counts
# ------------------------------------------------------------

PRIMARY_COUNTS <- as.data.frame(
  table(
    Region = PRIMARY_PREQC$Region,
    Dx = PRIMARY_PREQC$Dx
  ),
  stringsAsFactors = FALSE
)

names(PRIMARY_COUNTS)[3] <- "n_samples"

# ------------------------------------------------------------
# 11. Donor-level availability in primary cohort
# ------------------------------------------------------------

PRIMARY_DONOR_SUMMARY <- do.call(
  rbind,
  lapply(
    split(
      PRIMARY_PREQC,
      PRIMARY_PREQC$BrNum
    ),
    function(z) {

      data.frame(
        BrNum = z$BrNum[1],
        Dx = z$Dx[1],
        donor_age = z$donor_age[1],

        has_DLPFC = any(
          z$Region == "DLPFC"
        ),

        has_HIPPO = any(
          z$Region == "HIPPO"
        ),

        n_samples = nrow(z),

        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(PRIMARY_DONOR_SUMMARY) <- NULL

PRIMARY_DONOR_SUMMARY$region_availability_class <- ifelse(
  PRIMARY_DONOR_SUMMARY$has_DLPFC &
  PRIMARY_DONOR_SUMMARY$has_HIPPO,
  "DLPFC_HIPPO",
  ifelse(
    PRIMARY_DONOR_SUMMARY$has_DLPFC,
    "DLPFC_only",
    "HIPPO_only"
  )
)

PRIMARY_DONOR_COUNTS <- as.data.frame(
  table(
    region_availability_class =
      PRIMARY_DONOR_SUMMARY$region_availability_class,
    Dx =
      PRIMARY_DONOR_SUMMARY$Dx
  ),
  stringsAsFactors = FALSE
)

names(PRIMARY_DONOR_COUNTS)[3] <- "n_donors"

# ------------------------------------------------------------
# 12. Age-exclusion audit
# ------------------------------------------------------------

AGE_EXCLUDED_DONORS <- unique(
  SAMPLE_COHORT$BrNum[
    !SAMPLE_COHORT$adult_eligible
  ]
)

AGE_EXCLUDED <- unique(
  SAMPLE_COHORT[
    SAMPLE_COHORT$BrNum %in% AGE_EXCLUDED_DONORS,
    c(
      "BrNum",
      "Dx",
      "donor_age"
    )
  ]
)

AGE_EXCLUDED <- AGE_EXCLUDED[
  order(
    AGE_EXCLUDED$donor_age
  ),
]

# ------------------------------------------------------------
# 13. Save outputs
# ------------------------------------------------------------

COHORT_SAMPLE_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01c_primary_preQC_sample_metadata.csv"
)

COHORT_DONOR_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01c_primary_preQC_donor_metadata.csv"
)

QC_COMPLETENESS_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01c_QC_completeness.csv"
)

QC_SUMMARY_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01c_QC_summary.csv"
)

AGE_EXCLUDED_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step01c_age_excluded_donors.csv"
)

assert_project_sentinel()

write.csv(
  PRIMARY_PREQC,
  COHORT_SAMPLE_PATH,
  row.names = FALSE
)

write.csv(
  PRIMARY_DONOR_SUMMARY,
  COHORT_DONOR_PATH,
  row.names = FALSE
)

write.csv(
  QC_COMPLETENESS,
  QC_COMPLETENESS_PATH,
  row.names = FALSE
)

write.csv(
  QC_SUMMARY,
  QC_SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  AGE_EXCLUDED,
  AGE_EXCLUDED_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 14. Sanity checks
# ------------------------------------------------------------

stopifnot(
  nrow(PRIMARY_PREQC) > 0L,
  nrow(PRIMARY_DONOR_SUMMARY) > 0L,
  all(
    PRIMARY_PREQC$donor_age > 17
  ),
  all(
    PRIMARY_PREQC$Dx %in% c(
      "Control",
      "Schizo"
    )
  ),
  all(
    PRIMARY_PREQC$Region %in% c(
      "DLPFC",
      "HIPPO"
    )
  )
)

# ------------------------------------------------------------
# 15. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 01C — PRIMARY PRE-QC COHORT + QC AUDIT\n")
cat("============================================================\n")

cat(
  "Primary pre-QC samples: ",
  nrow(PRIMARY_PREQC),
  "\n",
  sep = ""
)

cat(
  "Primary pre-QC donors:  ",
  nrow(PRIMARY_DONOR_SUMMARY),
  "\n",
  sep = ""
)

cat(
  "Age-excluded donors:    ",
  nrow(AGE_EXCLUDED),
  "\n",
  sep = ""
)

cat("\n--- REGION x DIAGNOSIS ---\n")

print(
  PRIMARY_COUNTS,
  row.names = FALSE
)

cat("\n--- DONOR AVAILABILITY x DIAGNOSIS ---\n")

print(
  PRIMARY_DONOR_COUNTS,
  row.names = FALSE
)

cat("\n--- QC COMPLETENESS ---\n")

print(
  QC_COMPLETENESS,
  row.names = FALSE
)

cat("\n--- QC DISTRIBUTIONS ---\n")

print(
  QC_SUMMARY,
  row.names = FALSE
)

cat("\n--- AGE-EXCLUDED DONORS ---\n")

print(
  AGE_EXCLUDED,
  row.names = FALSE
)

cat("\n============================================================\n")
cat("STEP 01C — CELL 1 PASS\n")
cat("============================================================\n")


STEP 01C — PRIMARY PRE-QC COHORT + QC AUDIT
Primary pre-QC samples: 755
Primary pre-QC donors:  454
Age-excluded donors:    97

--- REGION x DIAGNOSIS ---
 Region      Dx n_samples
  DLPFC Control       226
  HIPPO Control       243
  DLPFC  Schizo       153
  HIPPO  Schizo       133

--- DONOR AVAILABILITY x DIAGNOSIS ---
 region_availability_class      Dx n_donors
               DLPFC_HIPPO Control      192
                DLPFC_only Control       34
                HIPPO_only Control       51
               DLPFC_HIPPO  Schizo      109
                DLPFC_only  Schizo       44
                HIPPO_only  Schizo       24

--- QC COMPLETENESS ---
          variable n_missing pct_missing
               RIN         0           0
          numReads         0           0
         numMapped         0           0
       numUnmapped         0           0
    overallMapRate         0           0
    concordMapRate         0           0
       totalMapped         0           0
        mitoM

In [20]:
# ============================================================
# STEP 01C — CELL 2
# FINAL DISCOVERY COHORT LOCK
# ============================================================

assert_project_sentinel()

if (!requireNamespace("digest", quietly = TRUE)) {
  stop("digest package is required.")
}

# ------------------------------------------------------------
# 1. Final eligible samples
# ------------------------------------------------------------

FINAL_COHORT <- SAMPLE_COHORT[
  SAMPLE_COHORT$primary_preQC_eligible,
  ,
  drop = FALSE
]

FINAL_COHORT <- FINAL_COHORT[
  order(FINAL_COHORT$sample_index),
  ,
  drop = FALSE
]

rownames(FINAL_COHORT) <- NULL

# No additional arbitrary QC exclusion is applied here.
FINAL_COHORT$sample_QC_exclusion <- FALSE
FINAL_COHORT$final_discovery_eligible <- TRUE

# ------------------------------------------------------------
# 2. Hard checks
# ------------------------------------------------------------

stopifnot(
  nrow(FINAL_COHORT) == 755L,
  length(unique(FINAL_COHORT$BrNum)) == 454L,
  all(FINAL_COHORT$donor_age > 17),
  all(FINAL_COHORT$Dx %in% c("Control", "Schizo")),
  all(FINAL_COHORT$Region %in% c("DLPFC", "HIPPO")),
  !anyDuplicated(FINAL_COHORT$sample_index),
  !anyDuplicated(FINAL_COHORT$rse_colname)
)

# ------------------------------------------------------------
# 3. Final sample counts
# ------------------------------------------------------------

FINAL_REGION_DX <- as.data.frame(
  table(
    Region = FINAL_COHORT$Region,
    Dx = FINAL_COHORT$Dx
  ),
  stringsAsFactors = FALSE
)

names(FINAL_REGION_DX)[3] <- "n_samples"

# ------------------------------------------------------------
# 4. Final donor-level table
# ------------------------------------------------------------

FINAL_DONORS <- do.call(
  rbind,
  lapply(
    split(
      FINAL_COHORT,
      FINAL_COHORT$BrNum
    ),
    function(z) {

      has_dlpfc <- any(z$Region == "DLPFC")
      has_hippo <- any(z$Region == "HIPPO")

      data.frame(
        BrNum = z$BrNum[1],
        Dx = z$Dx[1],
        donor_age = z$donor_age[1],
        Sex = z$Sex[1],
        Race = z$Race[1],
        has_DLPFC = has_dlpfc,
        has_HIPPO = has_hippo,
        n_samples = nrow(z),
        region_availability_class = if (
          has_dlpfc && has_hippo
        ) {
          "DLPFC_HIPPO"
        } else if (has_dlpfc) {
          "DLPFC_only"
        } else {
          "HIPPO_only"
        },
        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(FINAL_DONORS) <- NULL

FINAL_DONOR_DX <- as.data.frame(
  table(
    region_availability_class =
      FINAL_DONORS$region_availability_class,
    Dx = FINAL_DONORS$Dx
  ),
  stringsAsFactors = FALSE
)

names(FINAL_DONOR_DX)[3] <- "n_donors"

# ------------------------------------------------------------
# 5. Diagnosis totals
# ------------------------------------------------------------

FINAL_SAMPLE_DX <- as.data.frame(
  table(
    Dx = FINAL_COHORT$Dx
  ),
  stringsAsFactors = FALSE
)

names(FINAL_SAMPLE_DX)[2] <- "n_samples"

FINAL_DONOR_DIAG <- as.data.frame(
  table(
    Dx = FINAL_DONORS$Dx
  ),
  stringsAsFactors = FALSE
)

names(FINAL_DONOR_DIAG)[2] <- "n_donors"

# ------------------------------------------------------------
# 6. Freeze sample-order mapping
# ------------------------------------------------------------

FINAL_SAMPLE_ORDER <- data.frame(
  discovery_row_index = seq_len(nrow(FINAL_COHORT)),
  sample_index_in_rse = FINAL_COHORT$sample_index,
  rse_colname = FINAL_COHORT$rse_colname,
  RNum = FINAL_COHORT$RNum,
  BrNum = FINAL_COHORT$BrNum,
  Region = FINAL_COHORT$Region,
  Dx = FINAL_COHORT$Dx,
  donor_age = FINAL_COHORT$donor_age,
  Sex = FINAL_COHORT$Sex,
  Race = FINAL_COHORT$Race,
  RIN = FINAL_COHORT$RIN,
  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 7. Save canonical cohort artifacts
# ------------------------------------------------------------

FINAL_SAMPLE_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_discovery_cohort_samples_v1.csv"
)

FINAL_DONOR_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_discovery_cohort_donors_v1.csv"
)

FINAL_ORDER_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_discovery_sample_order_v1.csv"
)

assert_project_sentinel()

write.csv(
  FINAL_COHORT,
  FINAL_SAMPLE_PATH,
  row.names = FALSE
)

write.csv(
  FINAL_DONORS,
  FINAL_DONOR_PATH,
  row.names = FALSE
)

write.csv(
  FINAL_SAMPLE_ORDER,
  FINAL_ORDER_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 8. SHA-256 hashes
# ------------------------------------------------------------

FINAL_SAMPLE_SHA <- digest::digest(
  FINAL_SAMPLE_PATH,
  algo = "sha256",
  file = TRUE,
  serialize = FALSE
)

FINAL_DONOR_SHA <- digest::digest(
  FINAL_DONOR_PATH,
  algo = "sha256",
  file = TRUE,
  serialize = FALSE
)

FINAL_ORDER_SHA <- digest::digest(
  FINAL_ORDER_PATH,
  algo = "sha256",
  file = TRUE,
  serialize = FALSE
)

COHORT_LOCK_PATH <- file.path(
  PROJECT_ROOT,
  "00_protocol",
  "BRAINSEQ_DISCOVERY_COHORT_LOCK_v1.txt"
)

COHORT_LOCK <- c(
  "SCZ-CRST BRAINSEQ DISCOVERY COHORT LOCK v1",
  paste0("samples: ", nrow(FINAL_COHORT)),
  paste0("donors: ", nrow(FINAL_DONORS)),
  "adult_rule: donor_age > 17",
  "diagnoses: Control, Schizo",
  "regions: DLPFC, HIPPO",
  "additional_sample_QC_exclusion: NONE",
  paste0("sample_manifest_sha256: ", FINAL_SAMPLE_SHA),
  paste0("donor_manifest_sha256: ", FINAL_DONOR_SHA),
  paste0("sample_order_sha256: ", FINAL_ORDER_SHA),
  paste0(
    "timestamp: ",
    format(Sys.time(), "%Y-%m-%d %H:%M:%S %z")
  )
)

assert_project_sentinel()

writeLines(
  COHORT_LOCK,
  COHORT_LOCK_PATH,
  useBytes = TRUE
)

# ------------------------------------------------------------
# 9. Final checks
# ------------------------------------------------------------

stopifnot(
  file.exists(FINAL_SAMPLE_PATH),
  file.exists(FINAL_DONOR_PATH),
  file.exists(FINAL_ORDER_PATH),
  file.exists(COHORT_LOCK_PATH)
)

cat("\n============================================================\n")
cat("STEP 01C — FINAL DISCOVERY COHORT LOCK\n")
cat("============================================================\n")

cat("Final samples: ", nrow(FINAL_COHORT), "\n", sep = "")
cat("Final donors:  ", nrow(FINAL_DONORS), "\n", sep = "")

cat("\n--- SAMPLE DIAGNOSIS TOTALS ---\n")
print(
  FINAL_SAMPLE_DX,
  row.names = FALSE
)

cat("\n--- REGION x DIAGNOSIS ---\n")
print(
  FINAL_REGION_DX,
  row.names = FALSE
)

cat("\n--- DONOR DIAGNOSIS TOTALS ---\n")
print(
  FINAL_DONOR_DIAG,
  row.names = FALSE
)

cat("\n--- DONOR AVAILABILITY x DIAGNOSIS ---\n")
print(
  FINAL_DONOR_DX,
  row.names = FALSE
)

cat("\nSample manifest SHA-256:\n", FINAL_SAMPLE_SHA, "\n", sep = "")
cat("\nDonor manifest SHA-256:\n", FINAL_DONOR_SHA, "\n", sep = "")
cat("\nSample-order SHA-256:\n", FINAL_ORDER_SHA, "\n", sep = "")

cat("\n============================================================\n")
cat("STEP 01C — CELL 2 PASS\n")
cat("============================================================\n")


STEP 01C — FINAL DISCOVERY COHORT LOCK
Final samples: 755
Final donors:  454

--- SAMPLE DIAGNOSIS TOTALS ---
      Dx n_samples
 Control       469
  Schizo       286

--- REGION x DIAGNOSIS ---
 Region      Dx n_samples
  DLPFC Control       226
  HIPPO Control       243
  DLPFC  Schizo       153
  HIPPO  Schizo       133

--- DONOR DIAGNOSIS TOTALS ---
      Dx n_donors
 Control      277
  Schizo      177

--- DONOR AVAILABILITY x DIAGNOSIS ---
 region_availability_class      Dx n_donors
               DLPFC_HIPPO Control      192
                DLPFC_only Control       34
                HIPPO_only Control       51
               DLPFC_HIPPO  Schizo      109
                DLPFC_only  Schizo       44
                HIPPO_only  Schizo       24

Sample manifest SHA-256:
2f7a9cc1aae8ef5358af6395d901716bdd2175ff12aeee3d714000387ffd3367

Donor manifest SHA-256:
edec0e1d3f39bce7b64f4ab81c0d586b9923a7c9b2255d53d5bd8ae94badae4e

Sample-order SHA-256:
ee39acffdfa1758ffc9d97a0f1964cddc90a

In [22]:
# ============================================================
# STEP 02A — CELL 1 (CORRECTED)
# GENE ANNOTATION + LOCKED-COHORT EXPRESSION MATRIX AUDIT
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# 1. Required packages
# ------------------------------------------------------------

if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages(
    "BiocManager",
    repos = "https://cloud.r-project.org"
  )
}

REQUIRED_BIOC <- c(
  "SummarizedExperiment",
  "GenomeInfoDb"
)

for (pkg in REQUIRED_BIOC) {

  if (!requireNamespace(pkg, quietly = TRUE)) {

    cat("Installing missing package:", pkg, "\n")

    BiocManager::install(
      pkg,
      ask = FALSE,
      update = FALSE
    )
  }

  if (!requireNamespace(pkg, quietly = TRUE)) {
    stop(
      paste0(
        "STEP 02A FAIL: Required package unavailable: ",
        pkg
      )
    )
  }
}

suppressPackageStartupMessages(
  library(SummarizedExperiment)
)

# ------------------------------------------------------------
# 2. Recover BrainSeq RSE if needed
# ------------------------------------------------------------

if (!exists("BRAINSEQ_RSE")) {

  RAW_FILE <- file.path(
    PROJECT_ROOT,
    "01_raw",
    "brainseq_phase2",
    "rse_gene_unfiltered.Rdata"
  )

  if (!file.exists(RAW_FILE)) {
    stop("STEP 02A FAIL: BrainSeq raw RData is missing.")
  }

  tmp_env <- new.env(
    parent = emptyenv()
  )

  loaded_names <- load(
    RAW_FILE,
    envir = tmp_env
  )

  rse_names <- loaded_names[
    vapply(
      loaded_names,
      function(x) {
        inherits(
          get(
            x,
            envir = tmp_env
          ),
          "SummarizedExperiment"
        )
      },
      logical(1)
    )
  ]

  if (length(rse_names) != 1L) {

    stop(
      paste0(
        "STEP 02A FAIL: Expected exactly one RSE object; found ",
        length(rse_names)
      )
    )
  }

  BRAINSEQ_RSE <- get(
    rse_names[1],
    envir = tmp_env
  )
}

# ------------------------------------------------------------
# 3. Load locked discovery sample order
# ------------------------------------------------------------

SAMPLE_ORDER_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_discovery_sample_order_v1.csv"
)

if (!file.exists(SAMPLE_ORDER_PATH)) {

  stop(
    "STEP 02A FAIL: Locked discovery sample-order file is missing."
  )
}

DISCOVERY_ORDER <- read.csv(
  SAMPLE_ORDER_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(DISCOVERY_ORDER) == 755L,
  !anyDuplicated(
    DISCOVERY_ORDER$discovery_row_index
  ),
  !anyDuplicated(
    DISCOVERY_ORDER$sample_index_in_rse
  )
)

# ------------------------------------------------------------
# 4. Verify locked sample order against RSE
# ------------------------------------------------------------

RSE_COLNAMES <- colnames(
  BRAINSEQ_RSE
)

if (is.null(RSE_COLNAMES)) {

  stop(
    "STEP 02A FAIL: BrainSeq RSE has no column names."
  )
}

locked_idx <- DISCOVERY_ORDER$sample_index_in_rse

if (
  any(locked_idx < 1L) ||
  any(locked_idx > ncol(BRAINSEQ_RSE))
) {

  stop(
    "STEP 02A FAIL: Locked sample index falls outside RSE dimensions."
  )
}

CURRENT_LOCKED_COLNAMES <- RSE_COLNAMES[
  locked_idx
]

if (!identical(
  CURRENT_LOCKED_COLNAMES,
  DISCOVERY_ORDER$rse_colname
)) {

  stop(
    paste0(
      "STEP 02A FAIL: Locked sample order no longer matches ",
      "BrainSeq RSE column order."
    )
  )
}

cat("Locked sample-order verification: PASS\n")

# ------------------------------------------------------------
# 5. Extract locked-cohort counts + RPKM
# ------------------------------------------------------------

COUNTS_DISCOVERY <- SummarizedExperiment::assay(
  BRAINSEQ_RSE,
  "counts"
)[
  ,
  locked_idx,
  drop = FALSE
]

RPKM_DISCOVERY <- SummarizedExperiment::assay(
  BRAINSEQ_RSE,
  "rpkm"
)[
  ,
  locked_idx,
  drop = FALSE
]

if (
  nrow(COUNTS_DISCOVERY) != 58037L ||
  ncol(COUNTS_DISCOVERY) != 755L
) {

  stop(
    paste0(
      "STEP 02A FAIL: Unexpected counts dimension: ",
      paste(
        dim(COUNTS_DISCOVERY),
        collapse = " x "
      )
    )
  )
}

if (!identical(
  dim(COUNTS_DISCOVERY),
  dim(RPKM_DISCOVERY)
)) {

  stop(
    "STEP 02A FAIL: Counts and RPKM dimensions differ."
  )
}

# ------------------------------------------------------------
# 6. Expression integrity checks
# ------------------------------------------------------------

if (anyNA(COUNTS_DISCOVERY)) {

  stop(
    "STEP 02A FAIL: NA detected in counts matrix."
  )
}

if (anyNA(RPKM_DISCOVERY)) {

  stop(
    "STEP 02A FAIL: NA detected in RPKM matrix."
  )
}

if (any(COUNTS_DISCOVERY < 0)) {

  stop(
    "STEP 02A FAIL: Negative counts detected."
  )
}

if (any(RPKM_DISCOVERY < 0)) {

  stop(
    "STEP 02A FAIL: Negative RPKM detected."
  )
}

COUNT_NONINTEGER_N <- sum(
  abs(
    COUNTS_DISCOVERY -
      round(COUNTS_DISCOVERY)
  ) > 1e-8
)

# ------------------------------------------------------------
# 7. Extract gene annotation
# ------------------------------------------------------------

RD <- as.data.frame(
  SummarizedExperiment::rowData(
    BRAINSEQ_RSE
  ),
  stringsAsFactors = FALSE
)

RR <- SummarizedExperiment::rowRanges(
  BRAINSEQ_RSE
)

required_gene_cols <- c(
  "gencodeID",
  "ensemblID",
  "gene_type",
  "Symbol",
  "EntrezID",
  "Class",
  "meanExprs",
  "passExprsCut"
)

missing_gene_cols <- setdiff(
  required_gene_cols,
  names(RD)
)

if (length(missing_gene_cols) > 0L) {

  stop(
    paste0(
      "STEP 02A FAIL: Missing gene annotation columns:\n- ",
      paste(
        missing_gene_cols,
        collapse = "\n- "
      )
    )
  )
}

# ------------------------------------------------------------
# 8. Extract chromosome safely
# ------------------------------------------------------------

CHROMOSOME_VECTOR <- as.character(
  GenomeInfoDb::seqnames(
    RR
  )
)

if (
  length(CHROMOSOME_VECTOR) != nrow(RD)
) {

  stop(
    "STEP 02A FAIL: Chromosome vector length mismatch."
  )
}

# ------------------------------------------------------------
# 9. Construct canonical annotation table
# ------------------------------------------------------------

GENE_ANNOTATION <- data.frame(

  gene_row_index =
    seq_len(nrow(RD)),

  gencodeID =
    as.character(
      RD$gencodeID
    ),

  ensemblID_original =
    as.character(
      RD$ensemblID
    ),

  gene_symbol =
    as.character(
      RD$Symbol
    ),

  gene_type =
    as.character(
      RD$gene_type
    ),

  entrez_id =
    as.character(
      RD$EntrezID
    ),

  class =
    as.character(
      RD$Class
    ),

  meanExprs_source =
    suppressWarnings(
      as.numeric(
        RD$meanExprs
      )
    ),

  passExprsCut_source =
    as.logical(
      RD$passExprsCut
    ),

  chromosome =
    CHROMOSOME_VECTOR,

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 10. Stable Ensembl IDs
# Remove version suffix only.
# Do NOT collapse duplicate mappings yet.
# ------------------------------------------------------------

GENE_ANNOTATION$ensemblID_stable <- sub(
  "\\.[0-9]+$",
  "",
  GENE_ANNOTATION$ensemblID_original
)

GENE_ANNOTATION$ensembl_missing <- (
  is.na(
    GENE_ANNOTATION$ensemblID_stable
  ) |
  GENE_ANNOTATION$ensemblID_stable == ""
)

GENE_ANNOTATION$symbol_missing <- (
  is.na(
    GENE_ANNOTATION$gene_symbol
  ) |
  GENE_ANNOTATION$gene_symbol == ""
)

# ------------------------------------------------------------
# 11. Duplicate Ensembl / symbol audit
# ------------------------------------------------------------

valid_ensembl <- GENE_ANNOTATION$ensemblID_stable[
  !GENE_ANNOTATION$ensembl_missing
]

valid_symbol <- GENE_ANNOTATION$gene_symbol[
  !GENE_ANNOTATION$symbol_missing
]

ENSEMBL_DUP_TABLE <- sort(
  table(
    valid_ensembl
  ),
  decreasing = TRUE
)

SYMBOL_DUP_TABLE <- sort(
  table(
    valid_symbol
  ),
  decreasing = TRUE
)

N_DUP_ENSEMBL_IDS <- sum(
  ENSEMBL_DUP_TABLE > 1L
)

N_ROWS_IN_DUP_ENSEMBL <- sum(
  ENSEMBL_DUP_TABLE[
    ENSEMBL_DUP_TABLE > 1L
  ]
)

N_DUP_SYMBOLS <- sum(
  SYMBOL_DUP_TABLE > 1L
)

# ------------------------------------------------------------
# 12. Gene type counts
# ------------------------------------------------------------

GENE_TYPE_COUNTS <- as.data.frame(
  sort(
    table(
      GENE_ANNOTATION$gene_type,
      useNA = "ifany"
    ),
    decreasing = TRUE
  ),
  stringsAsFactors = FALSE
)

names(GENE_TYPE_COUNTS) <- c(
  "gene_type",
  "n_genes"
)

# ------------------------------------------------------------
# 13. Chromosome counts
# ------------------------------------------------------------

CHROMOSOME_COUNTS <- as.data.frame(
  sort(
    table(
      GENE_ANNOTATION$chromosome,
      useNA = "ifany"
    ),
    decreasing = TRUE
  ),
  stringsAsFactors = FALSE
)

names(CHROMOSOME_COUNTS) <- c(
  "chromosome",
  "n_genes"
)

# ------------------------------------------------------------
# 14. Source passExprsCut audit
# ------------------------------------------------------------

PASS_EXPR_COUNTS <- as.data.frame(
  table(
    passExprsCut_source =
      GENE_ANNOTATION$passExprsCut_source,
    useNA = "ifany"
  ),
  stringsAsFactors = FALSE
)

names(PASS_EXPR_COUNTS)[2] <- "n_genes"

# ------------------------------------------------------------
# 15. Discovery-cohort expression summaries
# ------------------------------------------------------------

LIBRARY_SIZE <- colSums(
  COUNTS_DISCOVERY
)

N_EXPRESSED_PER_SAMPLE <- colSums(
  COUNTS_DISCOVERY > 0
)

GENE_ZERO_FRACTION <- rowMeans(
  COUNTS_DISCOVERY == 0
)

GENE_NONZERO_N <- rowSums(
  COUNTS_DISCOVERY > 0
)

GENE_TOTAL_COUNTS <- rowSums(
  COUNTS_DISCOVERY
)

GENE_ANNOTATION$discovery_nonzero_n <-
  GENE_NONZERO_N

GENE_ANNOTATION$discovery_nonzero_fraction <-
  GENE_NONZERO_N /
  ncol(COUNTS_DISCOVERY)

GENE_ANNOTATION$discovery_zero_fraction <-
  GENE_ZERO_FRACTION

GENE_ANNOTATION$discovery_total_counts <-
  GENE_TOTAL_COUNTS

# ------------------------------------------------------------
# 16. Sample expression audit
# ------------------------------------------------------------

SAMPLE_EXPRESSION_AUDIT <- data.frame(

  discovery_row_index =
    DISCOVERY_ORDER$discovery_row_index,

  BrNum =
    DISCOVERY_ORDER$BrNum,

  Region =
    DISCOVERY_ORDER$Region,

  Dx =
    DISCOVERY_ORDER$Dx,

  library_size =
    as.numeric(
      LIBRARY_SIZE
    ),

  n_genes_nonzero =
    as.integer(
      N_EXPRESSED_PER_SAMPLE
    ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 17. Expression summary
# ------------------------------------------------------------

EXPRESSION_SUMMARY <- data.frame(

  metric = c(
    "n_genes",
    "n_samples",
    "count_NA",
    "rpkm_NA",
    "negative_counts",
    "negative_rpkm",
    "noninteger_count_values",
    "min_library_size",
    "median_library_size",
    "max_library_size",
    "min_nonzero_genes_per_sample",
    "median_nonzero_genes_per_sample",
    "max_nonzero_genes_per_sample"
  ),

  value = c(
    nrow(COUNTS_DISCOVERY),
    ncol(COUNTS_DISCOVERY),
    sum(is.na(COUNTS_DISCOVERY)),
    sum(is.na(RPKM_DISCOVERY)),
    sum(COUNTS_DISCOVERY < 0),
    sum(RPKM_DISCOVERY < 0),
    COUNT_NONINTEGER_N,
    min(LIBRARY_SIZE),
    median(LIBRARY_SIZE),
    max(LIBRARY_SIZE),
    min(N_EXPRESSED_PER_SAMPLE),
    median(N_EXPRESSED_PER_SAMPLE),
    max(N_EXPRESSED_PER_SAMPLE)
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 18. Save audit outputs
# ------------------------------------------------------------

GENE_ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_gene_annotation_audit.csv"
)

GENE_TYPE_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_gene_type_counts.csv"
)

CHROMOSOME_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_chromosome_counts.csv"
)

PASS_EXPR_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_source_passExprsCut_counts.csv"
)

SAMPLE_EXPR_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_sample_expression_audit.csv"
)

EXPRESSION_SUMMARY_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_expression_integrity_summary.csv"
)

assert_project_sentinel()

write.csv(
  GENE_ANNOTATION,
  GENE_ANNOTATION_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  GENE_TYPE_COUNTS,
  GENE_TYPE_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  CHROMOSOME_COUNTS,
  CHROMOSOME_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  PASS_EXPR_COUNTS,
  PASS_EXPR_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  SAMPLE_EXPRESSION_AUDIT,
  SAMPLE_EXPR_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  EXPRESSION_SUMMARY,
  EXPRESSION_SUMMARY_PATH,
  row.names = FALSE,
  na = ""
)

# ------------------------------------------------------------
# 19. Save locked-cohort unfiltered count matrix
# ------------------------------------------------------------

DISCOVERY_COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "03_intermediate",
  "brainseq_discovery_counts_unfiltered_v1.rds"
)

assert_project_sentinel()

saveRDS(
  COUNTS_DISCOVERY,
  DISCOVERY_COUNTS_PATH,
  compress = "gzip"
)

# ------------------------------------------------------------
# 20. Read-back validation
# ------------------------------------------------------------

COUNTS_CHECK <- readRDS(
  DISCOVERY_COUNTS_PATH
)

if (!identical(
  dim(COUNTS_CHECK),
  c(58037L, 755L)
)) {

  stop(
    "STEP 02A FAIL: Saved count-matrix dimensions changed."
  )
}

rm(COUNTS_CHECK)

# ------------------------------------------------------------
# 21. Final output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 02A — GENE / EXPRESSION AUDIT\n")
cat("============================================================\n")

cat(
  "Genes:                       ",
  nrow(COUNTS_DISCOVERY),
  "\n",
  sep = ""
)

cat(
  "Locked discovery samples:    ",
  ncol(COUNTS_DISCOVERY),
  "\n",
  sep = ""
)

cat(
  "Count non-integer values:    ",
  COUNT_NONINTEGER_N,
  "\n",
  sep = ""
)

cat(
  "Missing stable Ensembl IDs:  ",
  sum(
    GENE_ANNOTATION$ensembl_missing
  ),
  "\n",
  sep = ""
)

cat(
  "Missing gene symbols:        ",
  sum(
    GENE_ANNOTATION$symbol_missing
  ),
  "\n",
  sep = ""
)

cat(
  "Duplicated stable Ensembl IDs: ",
  N_DUP_ENSEMBL_IDS,
  "\n",
  sep = ""
)

cat(
  "Rows in duplicate Ensembl IDs: ",
  N_ROWS_IN_DUP_ENSEMBL,
  "\n",
  sep = ""
)

cat(
  "Duplicated gene symbols:       ",
  N_DUP_SYMBOLS,
  "\n",
  sep = ""
)

cat("\n--- SOURCE passExprsCut ---\n")

print(
  PASS_EXPR_COUNTS,
  row.names = FALSE
)

cat("\n--- TOP GENE TYPES ---\n")

print(
  head(
    GENE_TYPE_COUNTS,
    20
  ),
  row.names = FALSE
)

cat("\n--- CHROMOSOMES ---\n")

print(
  CHROMOSOME_COUNTS,
  row.names = FALSE
)

cat("\n--- EXPRESSION INTEGRITY ---\n")

print(
  EXPRESSION_SUMMARY,
  row.names = FALSE
)

cat(
  "\nSaved unfiltered discovery counts:\n",
  DISCOVERY_COUNTS_PATH,
  "\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 02A — CELL 1 PASS\n")
cat("============================================================\n")

Installing missing package: GenomeInfoDb 


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31 ucrt)

Installing package(s) 'GenomeInfoDb'

also installing the dependencies 'sys', 'askpass', 'curl', 'mime', 'openssl', 'R6', 'httr', 'jsonlite', 'UCSC.utils'




package 'sys' successfully unpacked and MD5 sums checked
package 'askpass' successfully unpacked and MD5 sums checked
package 'curl' successfully unpacked and MD5 sums checked
package 'mime' successfully unpacked and MD5 sums checked
package 'openssl' successfully unpacked and MD5 sums checked
package 'R6' successfully unpacked and MD5 sums checked
package 'httr' successfully unpacked and MD5 sums checked
package 'jsonlite' successfully unpacked and MD5 sums checked
package 'UCSC.utils' successfully unpacked and MD5 sums checked
package 'GenomeInfoDb' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\ACT\AppData\Local\Temp\RtmpGOlA8P\downloaded_packages
Locked sample-order verification: PASS

STEP 02A — GENE / EXPRESSION AUDIT
Genes:                       58037
Locked discovery samples:    755
Count non-integer values:    0
Missing stable Ensembl IDs:  0
Missing gene symbols:        22495
Duplicated stable Ensembl IDs: 45
Rows in duplicate Ense

In [23]:
# ============================================================
# STEP 02A — CELL 2
# PRIMARY UNSUPERVISED GENE UNIVERSE + DUPLICATE-ID POLICY
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# 1. Recover required objects if necessary
# ------------------------------------------------------------

if (!exists("GENE_ANNOTATION")) {

  GENE_ANNOTATION_PATH <- file.path(
    PROJECT_ROOT,
    "02_metadata",
    "step02a_gene_annotation_audit.csv"
  )

  if (!file.exists(GENE_ANNOTATION_PATH)) {
    stop("STEP 02A CELL 2 FAIL: Gene annotation audit is missing.")
  }

  GENE_ANNOTATION <- read.csv(
    GENE_ANNOTATION_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
  )
}

if (!exists("COUNTS_DISCOVERY")) {

  DISCOVERY_COUNTS_PATH <- file.path(
    PROJECT_ROOT,
    "03_intermediate",
    "brainseq_discovery_counts_unfiltered_v1.rds"
  )

  if (!file.exists(DISCOVERY_COUNTS_PATH)) {
    stop("STEP 02A CELL 2 FAIL: Unfiltered discovery counts are missing.")
  }

  COUNTS_DISCOVERY <- readRDS(
    DISCOVERY_COUNTS_PATH
  )
}

# ------------------------------------------------------------
# 2. Structural validation
# ------------------------------------------------------------

stopifnot(
  nrow(GENE_ANNOTATION) == nrow(COUNTS_DISCOVERY),
  nrow(COUNTS_DISCOVERY) == 58037L,
  ncol(COUNTS_DISCOVERY) == 755L
)

# ------------------------------------------------------------
# 3. Normalize logical source expression flag
# ------------------------------------------------------------

PASS_EXPR <- GENE_ANNOTATION$passExprsCut_source

if (is.character(PASS_EXPR)) {

  PASS_EXPR <- toupper(
    trimws(PASS_EXPR)
  ) == "TRUE"
}

PASS_EXPR[is.na(PASS_EXPR)] <- FALSE

# ------------------------------------------------------------
# 4. Stable Ensembl validity
# ------------------------------------------------------------

VALID_ENSEMBL <- (
  !is.na(GENE_ANNOTATION$ensemblID_stable) &
  GENE_ANNOTATION$ensemblID_stable != ""
)

# ------------------------------------------------------------
# 5. Identify duplicated stable Ensembl IDs
#
# Primary rule:
# if one stable Ensembl ID maps to >1 BrainSeq row,
# exclude ALL rows for that ID.
#
# No silent collapse / averaging / arbitrary row selection.
# ------------------------------------------------------------

ENSEMBL_ID_COUNTS <- table(
  GENE_ANNOTATION$ensemblID_stable[
    VALID_ENSEMBL
  ]
)

DUPLICATED_ENSEMBL_IDS <- names(
  ENSEMBL_ID_COUNTS[
    ENSEMBL_ID_COUNTS > 1L
  ]
)

IS_DUPLICATED_ENSEMBL <- (
  VALID_ENSEMBL &
  GENE_ANNOTATION$ensemblID_stable %in%
    DUPLICATED_ENSEMBL_IDS
)

UNIQUE_ENSEMBL <- (
  VALID_ENSEMBL &
  !IS_DUPLICATED_ENSEMBL
)

# ------------------------------------------------------------
# 6. Autosomal primary analysis
# ------------------------------------------------------------

AUTOSOMES <- paste0(
  "chr",
  1:22
)

IS_AUTOSOMAL <- GENE_ANNOTATION$chromosome %in%
  AUTOSOMES

# ------------------------------------------------------------
# 7. Primary gene universe
# ------------------------------------------------------------

PRIMARY_GENE_KEEP <- (
  PASS_EXPR &
  VALID_ENSEMBL &
  UNIQUE_ENSEMBL &
  IS_AUTOSOMAL
)

# ------------------------------------------------------------
# 8. Reason-coded gene audit
# ------------------------------------------------------------

GENE_FILTER_AUDIT <- GENE_ANNOTATION

GENE_FILTER_AUDIT$criterion_passExprsCut <-
  PASS_EXPR

GENE_FILTER_AUDIT$criterion_valid_ensembl <-
  VALID_ENSEMBL

GENE_FILTER_AUDIT$criterion_unique_ensembl <-
  UNIQUE_ENSEMBL

GENE_FILTER_AUDIT$criterion_autosomal <-
  IS_AUTOSOMAL

GENE_FILTER_AUDIT$primary_gene_keep <-
  PRIMARY_GENE_KEEP

GENE_FILTER_AUDIT$primary_exclusion_reason <- NA_character_

GENE_FILTER_AUDIT$primary_exclusion_reason[
  !PASS_EXPR
] <- "FAIL_source_passExprsCut"

GENE_FILTER_AUDIT$primary_exclusion_reason[
  PASS_EXPR &
  !VALID_ENSEMBL
] <- "FAIL_missing_stable_ensembl"

GENE_FILTER_AUDIT$primary_exclusion_reason[
  PASS_EXPR &
  VALID_ENSEMBL &
  IS_DUPLICATED_ENSEMBL
] <- "FAIL_duplicated_stable_ensembl"

GENE_FILTER_AUDIT$primary_exclusion_reason[
  PASS_EXPR &
  VALID_ENSEMBL &
  UNIQUE_ENSEMBL &
  !IS_AUTOSOMAL
] <- "FAIL_non_autosomal"

GENE_FILTER_AUDIT$primary_exclusion_reason[
  PRIMARY_GENE_KEEP
] <- "KEEP_primary_gene_universe"

# ------------------------------------------------------------
# 9. Sequential filtering counts
# ------------------------------------------------------------

FILTER_COUNTS <- data.frame(

  stage = c(
    "All BrainSeq genes",
    "Source passExprsCut TRUE",
    "Pass + valid stable Ensembl",
    "Pass + valid + unique stable Ensembl",
    "Pass + valid + unique + autosomal"
  ),

  n_genes = c(
    nrow(GENE_FILTER_AUDIT),

    sum(
      PASS_EXPR
    ),

    sum(
      PASS_EXPR &
      VALID_ENSEMBL
    ),

    sum(
      PASS_EXPR &
      VALID_ENSEMBL &
      UNIQUE_ENSEMBL
    ),

    sum(
      PRIMARY_GENE_KEEP
    )
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 10. Exclusion-reason counts
# ------------------------------------------------------------

EXCLUSION_COUNTS <- as.data.frame(
  table(
    GENE_FILTER_AUDIT$primary_exclusion_reason,
    useNA = "ifany"
  ),
  stringsAsFactors = FALSE
)

names(EXCLUSION_COUNTS) <- c(
  "status_or_reason",
  "n_genes"
)

# ------------------------------------------------------------
# 11. Duplicate-ID detailed audit
# ------------------------------------------------------------

DUPLICATE_ENSEMBL_AUDIT <- GENE_FILTER_AUDIT[
  IS_DUPLICATED_ENSEMBL,
  c(
    "gene_row_index",
    "gencodeID",
    "ensemblID_original",
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "chromosome",
    "passExprsCut_source",
    "discovery_nonzero_fraction",
    "discovery_total_counts"
  ),
  drop = FALSE
]

DUPLICATE_ENSEMBL_AUDIT <- DUPLICATE_ENSEMBL_AUDIT[
  order(
    DUPLICATE_ENSEMBL_AUDIT$ensemblID_stable,
    DUPLICATE_ENSEMBL_AUDIT$gene_row_index
  ),
  ,
  drop = FALSE
]

# ------------------------------------------------------------
# 12. Extract primary matrices
# ------------------------------------------------------------

COUNTS_PRIMARY <- COUNTS_DISCOVERY[
  PRIMARY_GENE_KEEP,
  ,
  drop = FALSE
]

ANNOTATION_PRIMARY <- GENE_FILTER_AUDIT[
  PRIMARY_GENE_KEEP,
  ,
  drop = FALSE
]

# ------------------------------------------------------------
# 13. Stable Ensembl IDs become canonical rownames
# ------------------------------------------------------------

if (anyDuplicated(
  ANNOTATION_PRIMARY$ensemblID_stable
)) {
  stop(
    "STEP 02A CELL 2 FAIL: Duplicate stable Ensembl IDs remain."
  )
}

rownames(COUNTS_PRIMARY) <-
  ANNOTATION_PRIMARY$ensemblID_stable

# ------------------------------------------------------------
# 14. Ensure matrix is aligned with annotation
# ------------------------------------------------------------

if (!identical(
  rownames(COUNTS_PRIMARY),
  ANNOTATION_PRIMARY$ensemblID_stable
)) {
  stop(
    "STEP 02A CELL 2 FAIL: Matrix/annotation row alignment failed."
  )
}

# ------------------------------------------------------------
# 15. Gene-type composition of retained universe
# ------------------------------------------------------------

PRIMARY_GENE_TYPE_COUNTS <- as.data.frame(
  sort(
    table(
      ANNOTATION_PRIMARY$gene_type,
      useNA = "ifany"
    ),
    decreasing = TRUE
  ),
  stringsAsFactors = FALSE
)

names(PRIMARY_GENE_TYPE_COUNTS) <- c(
  "gene_type",
  "n_genes"
)

# ------------------------------------------------------------
# 16. Chromosome composition
# ------------------------------------------------------------

PRIMARY_CHROMOSOME_COUNTS <- as.data.frame(
  table(
    ANNOTATION_PRIMARY$chromosome
  ),
  stringsAsFactors = FALSE
)

names(PRIMARY_CHROMOSOME_COUNTS) <- c(
  "chromosome",
  "n_genes"
)

PRIMARY_CHROMOSOME_COUNTS$chromosome_number <-
  as.integer(
    sub(
      "^chr",
      "",
      PRIMARY_CHROMOSOME_COUNTS$chromosome
    )
  )

PRIMARY_CHROMOSOME_COUNTS <-
  PRIMARY_CHROMOSOME_COUNTS[
    order(
      PRIMARY_CHROMOSOME_COUNTS$chromosome_number
    ),
    c(
      "chromosome",
      "n_genes"
    )
  ]

rownames(PRIMARY_CHROMOSOME_COUNTS) <- NULL

# ------------------------------------------------------------
# 17. Save primary canonical artifacts
# ------------------------------------------------------------

GENE_FILTER_AUDIT_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_primary_gene_filter_audit_v1.csv"
)

FILTER_COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_primary_gene_filter_counts_v1.csv"
)

EXCLUSION_COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_primary_gene_exclusion_counts_v1.csv"
)

DUPLICATE_AUDIT_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_duplicate_ensembl_audit_v1.csv"
)

PRIMARY_ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_gene_annotation_v1.csv"
)

PRIMARY_COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

PRIMARY_GENE_TYPE_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_primary_gene_type_counts_v1.csv"
)

PRIMARY_CHROMOSOME_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02a_primary_chromosome_counts_v1.csv"
)

assert_project_sentinel()

write.csv(
  GENE_FILTER_AUDIT,
  GENE_FILTER_AUDIT_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  FILTER_COUNTS,
  FILTER_COUNTS_PATH,
  row.names = FALSE
)

write.csv(
  EXCLUSION_COUNTS,
  EXCLUSION_COUNTS_PATH,
  row.names = FALSE
)

write.csv(
  DUPLICATE_ENSEMBL_AUDIT,
  DUPLICATE_AUDIT_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  ANNOTATION_PRIMARY,
  PRIMARY_ANNOTATION_PATH,
  row.names = FALSE,
  na = ""
)

saveRDS(
  COUNTS_PRIMARY,
  PRIMARY_COUNTS_PATH,
  compress = "gzip"
)

write.csv(
  PRIMARY_GENE_TYPE_COUNTS,
  PRIMARY_GENE_TYPE_PATH,
  row.names = FALSE
)

write.csv(
  PRIMARY_CHROMOSOME_COUNTS,
  PRIMARY_CHROMOSOME_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 18. Read-back validation
# ------------------------------------------------------------

COUNTS_PRIMARY_CHECK <- readRDS(
  PRIMARY_COUNTS_PATH
)

ANNOTATION_PRIMARY_CHECK <- read.csv(
  PRIMARY_ANNOTATION_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  ncol(COUNTS_PRIMARY_CHECK) == 755L,

  nrow(COUNTS_PRIMARY_CHECK) ==
    nrow(ANNOTATION_PRIMARY_CHECK),

  !anyDuplicated(
    rownames(COUNTS_PRIMARY_CHECK)
  ),

  identical(
    rownames(COUNTS_PRIMARY_CHECK),
    ANNOTATION_PRIMARY_CHECK$ensemblID_stable
  ),

  all(
    ANNOTATION_PRIMARY_CHECK$chromosome %in%
      AUTOSOMES
  )
)

# ------------------------------------------------------------
# 19. SHA-256
# ------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {
  stop("digest package is required.")
}

PRIMARY_COUNTS_SHA <- digest::digest(
  PRIMARY_COUNTS_PATH,
  algo = "sha256",
  file = TRUE,
  serialize = FALSE
)

PRIMARY_ANNOTATION_SHA <- digest::digest(
  PRIMARY_ANNOTATION_PATH,
  algo = "sha256",
  file = TRUE,
  serialize = FALSE
)

# ------------------------------------------------------------
# 20. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 02A — PRIMARY GENE UNIVERSE\n")
cat("============================================================\n")

cat("\n--- SEQUENTIAL FILTERING ---\n")

print(
  FILTER_COUNTS,
  row.names = FALSE
)

cat("\n--- EXCLUSION / KEEP COUNTS ---\n")

print(
  EXCLUSION_COUNTS,
  row.names = FALSE
)

cat("\nDuplicated stable Ensembl IDs: ",
    length(DUPLICATED_ENSEMBL_IDS),
    "\n",
    sep = "")

cat(
  "Rows excluded due to duplicate stable Ensembl ID: ",
  nrow(DUPLICATE_ENSEMBL_AUDIT),
  "\n",
  sep = ""
)

cat(
  "\nFINAL PRIMARY GENE UNIVERSE: ",
  nrow(COUNTS_PRIMARY),
  " genes x ",
  ncol(COUNTS_PRIMARY),
  " samples\n",
  sep = ""
)

cat("\n--- RETAINED GENE TYPES: TOP 20 ---\n")

print(
  head(
    PRIMARY_GENE_TYPE_COUNTS,
    20
  ),
  row.names = FALSE
)

cat("\n--- RETAINED CHROMOSOMES ---\n")

print(
  PRIMARY_CHROMOSOME_COUNTS,
  row.names = FALSE
)

cat(
  "\nPrimary counts SHA-256:\n",
  PRIMARY_COUNTS_SHA,
  "\n",
  sep = ""
)

cat(
  "\nPrimary annotation SHA-256:\n",
  PRIMARY_ANNOTATION_SHA,
  "\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 02A — CELL 2 PASS\n")
cat("============================================================\n")


STEP 02A — PRIMARY GENE UNIVERSE

--- SEQUENTIAL FILTERING ---
                                stage n_genes
                   All BrainSeq genes   58037
             Source passExprsCut TRUE   24652
          Pass + valid stable Ensembl   24652
 Pass + valid + unique stable Ensembl   24646
    Pass + valid + unique + autosomal   23740

--- EXCLUSION / KEEP COUNTS ---
               status_or_reason n_genes
 FAIL_duplicated_stable_ensembl       6
             FAIL_non_autosomal     906
       FAIL_source_passExprsCut   33385
     KEEP_primary_gene_universe   23740

Duplicated stable Ensembl IDs: 45
Rows excluded due to duplicate stable Ensembl ID: 90

FINAL PRIMARY GENE UNIVERSE: 23740 genes x 755 samples

--- RETAINED GENE TYPES: TOP 20 ---
                          gene_type n_genes
                     protein_coding   14091
                            lincRNA    1913
               processed_pseudogene    1690
                          antisense    1590
                     sense

In [24]:
# ============================================================
# STEP 02B — CELL 1
# TMM NORMALIZATION + LOG-CPM + PCA / SAMPLE QC AUDIT
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# 1. Required packages
# ------------------------------------------------------------

if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages(
    "BiocManager",
    repos = "https://cloud.r-project.org"
  )
}

if (!requireNamespace("edgeR", quietly = TRUE)) {

  cat("Installing edgeR...\n")

  BiocManager::install(
    "edgeR",
    ask = FALSE,
    update = FALSE
  )
}

if (!requireNamespace("matrixStats", quietly = TRUE)) {
  install.packages(
    "matrixStats",
    repos = "https://cloud.r-project.org"
  )
}

if (!requireNamespace("edgeR", quietly = TRUE)) {
  stop("STEP 02B FAIL: edgeR unavailable.")
}

if (!requireNamespace("matrixStats", quietly = TRUE)) {
  stop("STEP 02B FAIL: matrixStats unavailable.")
}

# ------------------------------------------------------------
# 2. Load canonical inputs
# ------------------------------------------------------------

PRIMARY_COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

PRIMARY_ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_gene_annotation_v1.csv"
)

SAMPLE_ORDER_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_discovery_sample_order_v1.csv"
)

if (!file.exists(PRIMARY_COUNTS_PATH)) {
  stop("Primary counts file missing.")
}

if (!file.exists(PRIMARY_ANNOTATION_PATH)) {
  stop("Primary annotation file missing.")
}

if (!file.exists(SAMPLE_ORDER_PATH)) {
  stop("Locked sample-order file missing.")
}

COUNTS_PRIMARY <- readRDS(
  PRIMARY_COUNTS_PATH
)

ANNOTATION_PRIMARY <- read.csv(
  PRIMARY_ANNOTATION_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

DISCOVERY_ORDER <- read.csv(
  SAMPLE_ORDER_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

# ------------------------------------------------------------
# 3. Alignment checks
# ------------------------------------------------------------

stopifnot(
  nrow(COUNTS_PRIMARY) == 23740L,
  ncol(COUNTS_PRIMARY) == 755L,
  nrow(ANNOTATION_PRIMARY) == 23740L,
  nrow(DISCOVERY_ORDER) == 755L,
  identical(
    rownames(COUNTS_PRIMARY),
    ANNOTATION_PRIMARY$ensemblID_stable
  )
)

# ------------------------------------------------------------
# 4. Build edgeR object
# ------------------------------------------------------------

DGE <- edgeR::DGEList(
  counts = COUNTS_PRIMARY
)

DGE <- edgeR::calcNormFactors(
  DGE,
  method = "TMM"
)

NORM_FACTORS <- DGE$samples$norm.factors
RAW_LIB_SIZE <- DGE$samples$lib.size

EFFECTIVE_LIB_SIZE <- (
  RAW_LIB_SIZE *
  NORM_FACTORS
)

# ------------------------------------------------------------
# 5. Descriptive log-CPM matrix
# ------------------------------------------------------------

LOGCPM_QC <- edgeR::cpm(
  DGE,
  log = TRUE,
  prior.count = 1
)

if (
  anyNA(LOGCPM_QC) ||
  any(!is.finite(LOGCPM_QC))
) {
  stop(
    "STEP 02B FAIL: Non-finite value in log-CPM matrix."
  )
}

# ------------------------------------------------------------
# 6. Gene variance for PCA
# ------------------------------------------------------------

GENE_VARIANCE <- matrixStats::rowVars(
  LOGCPM_QC
)

if (anyNA(GENE_VARIANCE)) {
  stop(
    "STEP 02B FAIL: NA detected in gene variances."
  )
}

TOP_PCA_N <- min(
  5000L,
  length(GENE_VARIANCE)
)

TOP_PCA_INDEX <- order(
  GENE_VARIANCE,
  decreasing = TRUE
)[
  seq_len(TOP_PCA_N)
]

# ------------------------------------------------------------
# 7. PCA
# ------------------------------------------------------------

PCA_INPUT <- t(
  LOGCPM_QC[
    TOP_PCA_INDEX,
    ,
    drop = FALSE
  ]
)

PCA_FIT <- prcomp(
  PCA_INPUT,
  center = TRUE,
  scale. = TRUE,
  rank. = 20
)

PCA_VARIANCE <- (
  PCA_FIT$sdev^2 /
  sum(PCA_FIT$sdev^2)
)

N_PCS <- min(
  20L,
  ncol(PCA_FIT$x)
)

PCA_SCORES <- as.data.frame(
  PCA_FIT$x[
    ,
    seq_len(N_PCS),
    drop = FALSE
  ],
  stringsAsFactors = FALSE
)

names(PCA_SCORES) <- paste0(
  "PC",
  seq_len(N_PCS)
)

# ------------------------------------------------------------
# 8. Attach sample metadata
# ------------------------------------------------------------

SAMPLE_QC <- data.frame(

  discovery_row_index =
    DISCOVERY_ORDER$discovery_row_index,

  sample_index_in_rse =
    DISCOVERY_ORDER$sample_index_in_rse,

  rse_colname =
    DISCOVERY_ORDER$rse_colname,

  RNum =
    DISCOVERY_ORDER$RNum,

  BrNum =
    DISCOVERY_ORDER$BrNum,

  Region =
    DISCOVERY_ORDER$Region,

  Dx =
    DISCOVERY_ORDER$Dx,

  donor_age =
    DISCOVERY_ORDER$donor_age,

  Sex =
    DISCOVERY_ORDER$Sex,

  Race =
    DISCOVERY_ORDER$Race,

  RIN =
    suppressWarnings(
      as.numeric(
        DISCOVERY_ORDER$RIN
      )
    ),

  raw_library_size =
    as.numeric(
      RAW_LIB_SIZE
    ),

  TMM_norm_factor =
    as.numeric(
      NORM_FACTORS
    ),

  effective_library_size =
    as.numeric(
      EFFECTIVE_LIB_SIZE
    ),

  PCA_SCORES,

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 9. Robust multivariate PCA distance
#
# Audit only.
# No automatic sample exclusion.
# ------------------------------------------------------------

PCA_DISTANCE_N <- min(
  10L,
  N_PCS
)

PCA_DISTANCE_MATRIX <- as.matrix(
  SAMPLE_QC[
    ,
    paste0(
      "PC",
      seq_len(PCA_DISTANCE_N)
    ),
    drop = FALSE
  ]
)

PC_CENTER <- apply(
  PCA_DISTANCE_MATRIX,
  2,
  median
)

PC_SCALE <- apply(
  PCA_DISTANCE_MATRIX,
  2,
  function(x) {
    stats::mad(
      x,
      center = median(x),
      constant = 1.4826
    )
  }
)

# protect against zero MAD
PC_SCALE[
  !is.finite(PC_SCALE) |
  PC_SCALE == 0
] <- 1

PCA_ROBUST_Z <- sweep(
  PCA_DISTANCE_MATRIX,
  2,
  PC_CENTER,
  "-"
)

PCA_ROBUST_Z <- sweep(
  PCA_ROBUST_Z,
  2,
  PC_SCALE,
  "/"
)

SAMPLE_QC$pca10_robust_distance <- sqrt(
  rowSums(
    PCA_ROBUST_Z^2
  )
)

# descriptive flag only
OUTLIER_THRESHOLD <- unname(
  quantile(
    SAMPLE_QC$pca10_robust_distance,
    probs = 0.995,
    na.rm = TRUE
  )
)

SAMPLE_QC$pca_extreme_flag <- (
  SAMPLE_QC$pca10_robust_distance >
    OUTLIER_THRESHOLD
)

# ------------------------------------------------------------
# 10. Normalization-factor summary
# ------------------------------------------------------------

NORM_SUMMARY <- data.frame(

  metric = c(
    "n_samples",
    "min_TMM_factor",
    "median_TMM_factor",
    "mean_TMM_factor",
    "max_TMM_factor",
    "min_raw_library_size",
    "median_raw_library_size",
    "max_raw_library_size",
    "min_effective_library_size",
    "median_effective_library_size",
    "max_effective_library_size",
    "n_PCA_extreme_flags"
  ),

  value = c(
    nrow(SAMPLE_QC),
    min(NORM_FACTORS),
    median(NORM_FACTORS),
    mean(NORM_FACTORS),
    max(NORM_FACTORS),
    min(RAW_LIB_SIZE),
    median(RAW_LIB_SIZE),
    max(RAW_LIB_SIZE),
    min(EFFECTIVE_LIB_SIZE),
    median(EFFECTIVE_LIB_SIZE),
    max(EFFECTIVE_LIB_SIZE),
    sum(SAMPLE_QC$pca_extreme_flag)
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 11. PCA variance table
# ------------------------------------------------------------

PCA_VARIANCE_TABLE <- data.frame(

  PC = paste0(
    "PC",
    seq_len(N_PCS)
  ),

  variance_explained = PCA_VARIANCE[
    seq_len(N_PCS)
  ],

  cumulative_variance = cumsum(
    PCA_VARIANCE[
      seq_len(N_PCS)
    ]
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 12. PCA group summaries
# ------------------------------------------------------------

PCA_GROUP_SUMMARY <- do.call(
  rbind,
  lapply(
    split(
      SAMPLE_QC,
      interaction(
        SAMPLE_QC$Region,
        SAMPLE_QC$Dx,
        drop = TRUE
      )
    ),
    function(z) {

      data.frame(
        Region = z$Region[1],
        Dx = z$Dx[1],
        n = nrow(z),

        mean_PC1 = mean(z$PC1),
        mean_PC2 = mean(z$PC2),
        mean_PC3 = mean(z$PC3),

        mean_RIN = mean(
          z$RIN,
          na.rm = TRUE
        ),

        median_library_size = median(
          z$raw_library_size
        ),

        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(PCA_GROUP_SUMMARY) <- NULL

# ------------------------------------------------------------
# 13. Extreme samples table
# ------------------------------------------------------------

PCA_EXTREME_SAMPLES <- SAMPLE_QC[
  SAMPLE_QC$pca_extreme_flag,
  ,
  drop = FALSE
]

PCA_EXTREME_SAMPLES <- PCA_EXTREME_SAMPLES[
  order(
    -PCA_EXTREME_SAMPLES$pca10_robust_distance
  ),
  ,
  drop = FALSE
]

# ------------------------------------------------------------
# 14. Save outputs
# ------------------------------------------------------------

LOGCPM_PATH <- file.path(
  PROJECT_ROOT,
  "03_intermediate",
  "brainseq_primary_logCPM_TMM_QC_v1.rds"
)

SAMPLE_QC_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02b_sample_PCA_QC_v1.csv"
)

NORM_SUMMARY_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02b_TMM_normalization_summary_v1.csv"
)

PCA_VARIANCE_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02b_PCA_variance_v1.csv"
)

PCA_GROUP_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02b_PCA_group_summary_v1.csv"
)

PCA_EXTREME_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02b_PCA_extreme_samples_v1.csv"
)

TOP_PCA_GENES_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02b_top5000_variable_genes_for_PCA_v1.csv"
)

TOP_PCA_GENES <- data.frame(

  rank = seq_len(TOP_PCA_N),

  ensemblID_stable =
    rownames(LOGCPM_QC)[TOP_PCA_INDEX],

  variance =
    GENE_VARIANCE[TOP_PCA_INDEX],

  stringsAsFactors = FALSE
)

assert_project_sentinel()

saveRDS(
  LOGCPM_QC,
  LOGCPM_PATH,
  compress = "gzip"
)

write.csv(
  SAMPLE_QC,
  SAMPLE_QC_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  NORM_SUMMARY,
  NORM_SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  PCA_VARIANCE_TABLE,
  PCA_VARIANCE_PATH,
  row.names = FALSE
)

write.csv(
  PCA_GROUP_SUMMARY,
  PCA_GROUP_PATH,
  row.names = FALSE
)

write.csv(
  PCA_EXTREME_SAMPLES,
  PCA_EXTREME_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  TOP_PCA_GENES,
  TOP_PCA_GENES_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 15. Final checks
# ------------------------------------------------------------

stopifnot(
  identical(
    dim(LOGCPM_QC),
    c(23740L, 755L)
  ),

  nrow(SAMPLE_QC) == 755L,

  all(
    is.finite(
      SAMPLE_QC$TMM_norm_factor
    )
  ),

  all(
    SAMPLE_QC$TMM_norm_factor > 0
  ),

  file.exists(LOGCPM_PATH),
  file.exists(SAMPLE_QC_PATH)
)

# ------------------------------------------------------------
# 16. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 02B — NORMALIZATION + PCA QC\n")
cat("============================================================\n")

cat(
  "Genes:                         ",
  nrow(LOGCPM_QC),
  "\n",
  sep = ""
)

cat(
  "Samples:                       ",
  ncol(LOGCPM_QC),
  "\n",
  sep = ""
)

cat(
  "PCA genes used:                ",
  TOP_PCA_N,
  "\n",
  sep = ""
)

cat(
  "PCA extreme flags (audit only): ",
  sum(SAMPLE_QC$pca_extreme_flag),
  "\n",
  sep = ""
)

cat("\n--- NORMALIZATION SUMMARY ---\n")

print(
  NORM_SUMMARY,
  row.names = FALSE
)

cat("\n--- PCA VARIANCE: FIRST 10 PCs ---\n")

print(
  head(
    PCA_VARIANCE_TABLE,
    10
  ),
  row.names = FALSE
)

cat("\n--- REGION x DIAGNOSIS PCA SUMMARY ---\n")

print(
  PCA_GROUP_SUMMARY,
  row.names = FALSE
)

cat("\n--- PCA EXTREME SAMPLES ---\n")

if (nrow(PCA_EXTREME_SAMPLES) == 0L) {

  cat("None flagged.\n")

} else {

  print(
    PCA_EXTREME_SAMPLES[
      ,
      c(
        "RNum",
        "BrNum",
        "Region",
        "Dx",
        "RIN",
        "raw_library_size",
        "TMM_norm_factor",
        "PC1",
        "PC2",
        "PC3",
        "pca10_robust_distance"
      )
    ],
    row.names = FALSE
  )
}

cat(
  "\nSaved descriptive QC log-CPM:\n",
  LOGCPM_PATH,
  "\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 02B — CELL 1 PASS\n")
cat("============================================================\n")

Installing edgeR...


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31 ucrt)

Installing package(s) 'edgeR'

also installing the dependencies 'statmod', 'limma', 'locfit'




package 'statmod' successfully unpacked and MD5 sums checked
package 'limma' successfully unpacked and MD5 sums checked
package 'locfit' successfully unpacked and MD5 sums checked
package 'edgeR' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\ACT\AppData\Local\Temp\RtmpGOlA8P\downloaded_packages

STEP 02B — NORMALIZATION + PCA QC
Genes:                         23740
Samples:                       755
PCA genes used:                5000
PCA extreme flags (audit only): 4

--- NORMALIZATION SUMMARY ---
                        metric        value
                     n_samples 7.550000e+02
                min_TMM_factor 7.512412e-01
             median_TMM_factor 1.002703e+00
               mean_TMM_factor 1.001674e+00
                max_TMM_factor 1.201358e+00
          min_raw_library_size 8.666325e+06
       median_raw_library_size 2.025424e+07
          max_raw_library_size 6.132186e+07
    min_effective_library_size 8.242484e+06
 median_ef

In [25]:
# ============================================================
# STEP 02C — CELL 1
# DONOR-AWARE MODEL METADATA + PAIRING / COVARIATE AUDIT
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# 1. Load locked discovery sample order
# ------------------------------------------------------------

SAMPLE_ORDER_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_discovery_sample_order_v1.csv"
)

DISCOVERY_ORDER <- read.csv(
  SAMPLE_ORDER_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(DISCOVERY_ORDER) == 755L,
  !anyDuplicated(DISCOVERY_ORDER$discovery_row_index),
  !anyDuplicated(DISCOVERY_ORDER$sample_index_in_rse)
)

# ------------------------------------------------------------
# 2. Recover BrainSeq RSE if needed
# ------------------------------------------------------------

if (!exists("BRAINSEQ_RSE")) {

  RAW_FILE <- file.path(
    PROJECT_ROOT,
    "01_raw",
    "brainseq_phase2",
    "rse_gene_unfiltered.Rdata"
  )

  tmp_env <- new.env(parent = emptyenv())

  loaded_names <- load(
    RAW_FILE,
    envir = tmp_env
  )

  rse_names <- loaded_names[
    vapply(
      loaded_names,
      function(x) {
        inherits(
          get(x, envir = tmp_env),
          "SummarizedExperiment"
        )
      },
      logical(1)
    )
  ]

  if (length(rse_names) != 1L) {
    stop("STEP 02C FAIL: Could not uniquely recover BrainSeq RSE.")
  }

  BRAINSEQ_RSE <- get(
    rse_names[1],
    envir = tmp_env
  )
}

CD <- as.data.frame(
  SummarizedExperiment::colData(BRAINSEQ_RSE),
  stringsAsFactors = FALSE
)

IDX <- DISCOVERY_ORDER$sample_index_in_rse

# ------------------------------------------------------------
# 3. Required SNP PCs
# ------------------------------------------------------------

SNP_PC_NAMES <- paste0(
  "snpPC",
  1:10
)

missing_pcs <- setdiff(
  SNP_PC_NAMES,
  names(CD)
)

if (length(missing_pcs) > 0L) {
  stop(
    paste0(
      "Missing SNP PCs: ",
      paste(missing_pcs, collapse = ", ")
    )
  )
}

# ------------------------------------------------------------
# 4. Build canonical modeling metadata
# ------------------------------------------------------------

MODEL_META <- data.frame(

  discovery_row_index =
    DISCOVERY_ORDER$discovery_row_index,

  sample_index_in_rse =
    DISCOVERY_ORDER$sample_index_in_rse,

  rse_colname =
    DISCOVERY_ORDER$rse_colname,

  RNum =
    DISCOVERY_ORDER$RNum,

  BrNum =
    DISCOVERY_ORDER$BrNum,

  Region =
    DISCOVERY_ORDER$Region,

  Dx =
    DISCOVERY_ORDER$Dx,

  donor_age =
    as.numeric(
      DISCOVERY_ORDER$donor_age
    ),

  Sex =
    DISCOVERY_ORDER$Sex,

  Race =
    DISCOVERY_ORDER$Race,

  RIN =
    as.numeric(
      DISCOVERY_ORDER$RIN
    ),

  stringsAsFactors = FALSE
)

# SNP PCs
for (pc in SNP_PC_NAMES) {

  MODEL_META[[pc]] <- suppressWarnings(
    as.numeric(
      CD[[pc]][IDX]
    )
  )
}

# ------------------------------------------------------------
# 5. Outcome + region encoding
# ------------------------------------------------------------

MODEL_META$y_schizo <- ifelse(
  MODEL_META$Dx == "Schizo",
  1L,
  0L
)

MODEL_META$region_sign <- ifelse(
  MODEL_META$Region == "DLPFC",
  1,
  -1
)

# ------------------------------------------------------------
# 6. Donor pairing structure
# ------------------------------------------------------------

DONOR_N <- table(
  MODEL_META$BrNum
)

MODEL_META$n_regions_for_donor <- as.integer(
  DONOR_N[
    MODEL_META$BrNum
  ]
)

MODEL_META$paired_donor <- (
  MODEL_META$n_regions_for_donor == 2L
)

# hard check:
# paired donor must contain exactly one DLPFC + one HIPPO
paired_ids <- names(
  DONOR_N[
    DONOR_N == 2L
  ]
)

PAIR_CHECK <- do.call(
  rbind,
  lapply(
    paired_ids,
    function(id) {

      z <- MODEL_META[
        MODEL_META$BrNum == id,
        ,
        drop = FALSE
      ]

      data.frame(
        BrNum = id,
        n_samples = nrow(z),
        n_DLPFC = sum(z$Region == "DLPFC"),
        n_HIPPO = sum(z$Region == "HIPPO"),
        Dx = paste(unique(z$Dx), collapse = "|"),
        stringsAsFactors = FALSE
      )
    }
  )
)

if (
  any(PAIR_CHECK$n_DLPFC != 1L) ||
  any(PAIR_CHECK$n_HIPPO != 1L)
) {
  stop(
    "STEP 02C FAIL: Invalid paired-donor region structure."
  )
}

# ------------------------------------------------------------
# 7. Donor-balanced observation weights
#
# paired donor:
#   0.5 DLPFC + 0.5 HIPPO = total donor weight 1
#
# unpaired donor:
#   one sample weight = 1
# ------------------------------------------------------------

MODEL_META$donor_balance_weight <- ifelse(
  MODEL_META$paired_donor,
  0.5,
  1.0
)

DONOR_WEIGHT_SUM <- aggregate(
  donor_balance_weight ~ BrNum,
  data = MODEL_META,
  FUN = sum
)

if (
  !all(
    abs(
      DONOR_WEIGHT_SUM$donor_balance_weight - 1
    ) < 1e-12
  )
) {
  stop(
    "STEP 02C FAIL: Donor-balanced weights do not sum to 1 per donor."
  )
}

# ------------------------------------------------------------
# 8. Missingness audit
# ------------------------------------------------------------

MODEL_COVARIATES <- c(
  "donor_age",
  "Sex",
  "Race",
  "RIN",
  SNP_PC_NAMES
)

COVARIATE_MISSINGNESS <- data.frame(

  variable = MODEL_COVARIATES,

  n_missing = vapply(
    MODEL_COVARIATES,
    function(v) {
      sum(
        is.na(
          MODEL_META[[v]]
        ) |
        (
          is.character(MODEL_META[[v]]) &
          trimws(MODEL_META[[v]]) == ""
        )
      )
    },
    integer(1)
  ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 9. Continuous covariate summary by Region x Dx
# ------------------------------------------------------------

CONTINUOUS_VARS <- c(
  "donor_age",
  "RIN",
  SNP_PC_NAMES
)

CONTINUOUS_SUMMARY <- do.call(
  rbind,
  lapply(
    split(
      MODEL_META,
      interaction(
        MODEL_META$Region,
        MODEL_META$Dx,
        drop = TRUE
      )
    ),
    function(z) {

      out <- data.frame(
        Region = z$Region[1],
        Dx = z$Dx[1],
        n = nrow(z),
        stringsAsFactors = FALSE
      )

      for (v in CONTINUOUS_VARS) {

        out[[paste0(v, "_mean")]] <- mean(
          z[[v]],
          na.rm = TRUE
        )

        out[[paste0(v, "_sd")]] <- sd(
          z[[v]],
          na.rm = TRUE
        )
      }

      out
    }
  )
)

rownames(CONTINUOUS_SUMMARY) <- NULL

# ------------------------------------------------------------
# 10. Sex counts
# ------------------------------------------------------------

SEX_COUNTS <- as.data.frame(
  table(
    Region = MODEL_META$Region,
    Dx = MODEL_META$Dx,
    Sex = MODEL_META$Sex
  ),
  stringsAsFactors = FALSE
)

names(SEX_COUNTS)[4] <- "n"

# ------------------------------------------------------------
# 11. Race counts
# ------------------------------------------------------------

RACE_COUNTS <- as.data.frame(
  table(
    Region = MODEL_META$Region,
    Dx = MODEL_META$Dx,
    Race = MODEL_META$Race
  ),
  stringsAsFactors = FALSE
)

names(RACE_COUNTS)[4] <- "n"

# ------------------------------------------------------------
# 12. Donor-level pairing summary
# ------------------------------------------------------------

DONOR_LEVEL <- do.call(
  rbind,
  lapply(
    split(
      MODEL_META,
      MODEL_META$BrNum
    ),
    function(z) {

      data.frame(
        BrNum = z$BrNum[1],
        Dx = z$Dx[1],
        n_regions = nrow(z),
        paired = nrow(z) == 2L,
        has_DLPFC = any(z$Region == "DLPFC"),
        has_HIPPO = any(z$Region == "HIPPO"),
        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(DONOR_LEVEL) <- NULL

DONOR_PAIRING_COUNTS <- as.data.frame(
  table(
    Dx = DONOR_LEVEL$Dx,
    paired = DONOR_LEVEL$paired
  ),
  stringsAsFactors = FALSE
)

names(DONOR_PAIRING_COUNTS)[3] <- "n_donors"

# ------------------------------------------------------------
# 13. Save canonical modeling metadata
# ------------------------------------------------------------

MODEL_META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

PAIR_CHECK_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02c_pair_check_v1.csv"
)

MISSINGNESS_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02c_covariate_missingness_v1.csv"
)

CONTINUOUS_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02c_covariate_summary_v1.csv"
)

SEX_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02c_sex_counts_v1.csv"
)

RACE_PATH <- file.path(
  PROJECT_ROOT,
  "02_metadata",
  "step02c_race_counts_v1.csv"
)

assert_project_sentinel()

write.csv(
  MODEL_META,
  MODEL_META_PATH,
  row.names = FALSE
)

write.csv(
  PAIR_CHECK,
  PAIR_CHECK_PATH,
  row.names = FALSE
)

write.csv(
  COVARIATE_MISSINGNESS,
  MISSINGNESS_PATH,
  row.names = FALSE
)

write.csv(
  CONTINUOUS_SUMMARY,
  CONTINUOUS_PATH,
  row.names = FALSE
)

write.csv(
  SEX_COUNTS,
  SEX_PATH,
  row.names = FALSE
)

write.csv(
  RACE_COUNTS,
  RACE_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 14. Final checks
# ------------------------------------------------------------

stopifnot(
  nrow(MODEL_META) == 755L,
  length(unique(MODEL_META$BrNum)) == 454L,
  sum(MODEL_META$paired_donor) == 602L,
  length(unique(MODEL_META$BrNum[MODEL_META$paired_donor])) == 301L,
  abs(sum(MODEL_META$donor_balance_weight) - 454) < 1e-12,
  all(COVARIATE_MISSINGNESS$n_missing == 0L)
)

# ------------------------------------------------------------
# 15. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 02C — DONOR-AWARE MODEL METADATA\n")
cat("============================================================\n")

cat("Samples:                 ", nrow(MODEL_META), "\n", sep = "")
cat("Unique donors:           ", length(unique(MODEL_META$BrNum)), "\n", sep = "")
cat("Paired donors:           ", length(unique(MODEL_META$BrNum[MODEL_META$paired_donor])), "\n", sep = "")
cat("Paired samples:          ", sum(MODEL_META$paired_donor), "\n", sep = "")
cat("Unpaired donors/samples: ", sum(!MODEL_META$paired_donor), "\n", sep = "")
cat("Total balanced weight:   ", sum(MODEL_META$donor_balance_weight), "\n", sep = "")

cat("\n--- DONOR PAIRING BY DIAGNOSIS ---\n")

print(
  DONOR_PAIRING_COUNTS,
  row.names = FALSE
)

cat("\n--- COVARIATE MISSINGNESS ---\n")

print(
  COVARIATE_MISSINGNESS,
  row.names = FALSE
)

cat("\n--- CONTINUOUS COVARIATES BY REGION x DIAGNOSIS ---\n")

print(
  CONTINUOUS_SUMMARY,
  row.names = FALSE
)

cat("\n--- SEX COUNTS ---\n")

print(
  SEX_COUNTS,
  row.names = FALSE
)

cat("\n--- RACE COUNTS ---\n")

print(
  RACE_COUNTS,
  row.names = FALSE
)

cat(
  "\nSaved canonical modeling metadata:\n",
  MODEL_META_PATH,
  "\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 02C — CELL 1 PASS\n")
cat("============================================================\n")


STEP 02C — DONOR-AWARE MODEL METADATA
Samples:                 755
Unique donors:           454
Paired donors:           301
Paired samples:          602
Unpaired donors/samples: 153
Total balanced weight:   454

--- DONOR PAIRING BY DIAGNOSIS ---
      Dx paired n_donors
 Control  FALSE       85
  Schizo  FALSE       68
 Control   TRUE      192
  Schizo   TRUE      109

--- COVARIATE MISSINGNESS ---
  variable n_missing
 donor_age         0
       Sex         0
      Race         0
       RIN         0
    snpPC1         0
    snpPC2         0
    snpPC3         0
    snpPC4         0
    snpPC5         0
    snpPC6         0
    snpPC7         0
    snpPC8         0
    snpPC9         0
   snpPC10         0

--- CONTINUOUS COVARIATES BY REGION x DIAGNOSIS ---
 Region      Dx   n donor_age_mean donor_age_sd RIN_mean    RIN_sd
  DLPFC Control 226       45.01751     14.97503 7.717699 0.8711913
  HIPPO Control 243       44.56722     14.73045 7.735391 0.9668378
  DLPFC  Schizo 153       

In [ ]:
# ============================================================
# STEP 03A — CELL 1
# DONOR-BLOCKED LIMMA-VOOM DESCRIPTIVE DE
# DLPFC / HIPPO / SHARED / REGIONAL DEVIATION
# ============================================================

assert_project_sentinel()

# ------------------------------------------------------------
# 1. Required packages
# ------------------------------------------------------------

if (!requireNamespace("edgeR", quietly = TRUE)) {
  stop("STEP 03A FAIL: edgeR is unavailable.")
}

if (!requireNamespace("limma", quietly = TRUE)) {
  stop("STEP 03A FAIL: limma is unavailable.")
}

# ------------------------------------------------------------
# 2. Load canonical inputs
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_gene_annotation_v1.csv"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

COUNTS_PRIMARY <- readRDS(
  COUNTS_PATH
)

ANNOTATION_PRIMARY <- read.csv(
  ANNOTATION_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

MODEL_META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(COUNTS_PRIMARY) == 23740L,
  ncol(COUNTS_PRIMARY) == 755L,
  nrow(MODEL_META) == 755L,
  nrow(ANNOTATION_PRIMARY) == 23740L,
  identical(
    rownames(COUNTS_PRIMARY),
    ANNOTATION_PRIMARY$ensemblID_stable
  )
)

# ------------------------------------------------------------
# 3. Factors
# ------------------------------------------------------------

MODEL_META$Region <- factor(
  MODEL_META$Region,
  levels = c(
    "DLPFC",
    "HIPPO"
  )
)

MODEL_META$Dx <- factor(
  MODEL_META$Dx,
  levels = c(
    "Control",
    "Schizo"
  )
)

MODEL_META$Sex <- factor(
  MODEL_META$Sex
)

MODEL_META$Race <- factor(
  MODEL_META$Race
)

MODEL_META$group <- factor(
  paste(
    MODEL_META$Region,
    MODEL_META$Dx,
    sep = "_"
  ),
  levels = c(
    "DLPFC_Control",
    "DLPFC_Schizo",
    "HIPPO_Control",
    "HIPPO_Schizo"
  )
)

# ------------------------------------------------------------
# 4. Standardize continuous adjustment variables
# ------------------------------------------------------------

MODEL_META$age_z <- as.numeric(
  scale(
    MODEL_META$donor_age
  )
)

MODEL_META$RIN_z <- as.numeric(
  scale(
    MODEL_META$RIN
  )
)

for (i in 1:10) {

  src <- paste0(
    "snpPC",
    i
  )

  dst <- paste0(
    "snpPC",
    i,
    "_z"
  )

  MODEL_META[[dst]] <- as.numeric(
    scale(
      MODEL_META[[src]]
    )
  )
}

# ------------------------------------------------------------
# 5. Design matrix
# ------------------------------------------------------------

DESIGN <- model.matrix(
  ~ 0 +
    group +
    age_z +
    Sex +
    Race +
    RIN_z +
    snpPC1_z +
    snpPC2_z +
    snpPC3_z +
    snpPC4_z +
    snpPC5_z +
    snpPC6_z +
    snpPC7_z +
    snpPC8_z +
    snpPC9_z +
    snpPC10_z,
  data = MODEL_META
)

colnames(DESIGN) <- sub(
  "^group",
  "",
  colnames(DESIGN)
)

# ------------------------------------------------------------
# 6. Rank / estimability audit
# ------------------------------------------------------------

DESIGN_RANK <- qr(
  DESIGN
)$rank

if (DESIGN_RANK < ncol(DESIGN)) {

  cat("\nNon-estimable coefficients:\n")

  print(
    limma::nonEstimable(
      DESIGN
    )
  )

  stop(
    "STEP 03A FAIL: Design matrix is rank-deficient."
  )
}

# ------------------------------------------------------------
# 7. Contrasts
#
# shared = (DLPFC effect + HIPPO effect) / 2
#
# deviation = (DLPFC effect - HIPPO effect) / 2
#
# Therefore:
# DLPFC = shared + deviation
# HIPPO = shared - deviation
# ------------------------------------------------------------

CONTRASTS <- limma::makeContrasts(

  DLPFC_SCZ =
    DLPFC_Schizo -
    DLPFC_Control,

  HIPPO_SCZ =
    HIPPO_Schizo -
    HIPPO_Control,

  Shared_SCZ =
    0.5 * (
      DLPFC_Schizo -
      DLPFC_Control +
      HIPPO_Schizo -
      HIPPO_Control
    ),

  Regional_Deviation =
    0.5 * (
      DLPFC_Schizo -
      DLPFC_Control -
      HIPPO_Schizo +
      HIPPO_Control
    ),

  levels = DESIGN
)

# ------------------------------------------------------------
# 8. edgeR object + TMM
# ------------------------------------------------------------

DGE <- edgeR::DGEList(
  counts = COUNTS_PRIMARY
)

DGE <- edgeR::calcNormFactors(
  DGE,
  method = "TMM"
)

# ------------------------------------------------------------
# 9. First-pass voom
# ------------------------------------------------------------

VOOM_1 <- limma::voom(
  DGE,
  design = DESIGN,
  plot = FALSE
)

# Apply donor-balanced observation weighting
VOOM_1$weights <- sweep(
  VOOM_1$weights,
  2,
  MODEL_META$donor_balance_weight,
  "*"
)

# ------------------------------------------------------------
# 10. Estimate within-donor correlation
# ------------------------------------------------------------

CORFIT <- limma::duplicateCorrelation(
  VOOM_1,
  design = DESIGN,
  block = MODEL_META$BrNum
)

DONOR_CORRELATION <- CORFIT$consensus.correlation

if (
  !is.finite(
    DONOR_CORRELATION
  )
) {
  stop(
    "STEP 03A FAIL: Invalid donor correlation estimate."
  )
}

# ------------------------------------------------------------
# 11. Second-pass voom using estimated donor correlation
# ------------------------------------------------------------

VOOM_2 <- limma::voom(
  DGE,
  design = DESIGN,
  block = MODEL_META$BrNum,
  correlation = DONOR_CORRELATION,
  plot = FALSE
)

# donor-balanced observation weighting
VOOM_2$weights <- sweep(
  VOOM_2$weights,
  2,
  MODEL_META$donor_balance_weight,
  "*"
)

# ------------------------------------------------------------
# 12. Donor-blocked linear model
# ------------------------------------------------------------

FIT <- limma::lmFit(
  VOOM_2,
  design = DESIGN,
  block = MODEL_META$BrNum,
  correlation = DONOR_CORRELATION
)

FIT_CONTRAST <- limma::contrasts.fit(
  FIT,
  CONTRASTS
)

FIT_EBAYES <- limma::eBayes(
  FIT_CONTRAST,
  robust = TRUE
)

# ------------------------------------------------------------
# 13. Result extractor
# ------------------------------------------------------------

extract_result <- function(
  fit,
  coef_name,
  annotation
) {

  tt <- limma::topTable(
    fit,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH",
    sort.by = "P"
  )

  tt$ensemblID_stable <- rownames(
    tt
  )

  rownames(tt) <- NULL

  ann_keep <- annotation[
    ,
    c(
      "ensemblID_stable",
      "gene_symbol",
      "gene_type",
      "chromosome",
      "entrez_id"
    ),
    drop = FALSE
  ]

  out <- merge(
    tt,
    ann_keep,
    by = "ensemblID_stable",
    all.x = TRUE,
    sort = FALSE
  )

  # restore topTable order
  out <- out[
    match(
      tt$ensemblID_stable,
      out$ensemblID_stable
    ),
    ,
    drop = FALSE
  ]

  rownames(out) <- NULL

  out$FDR_lt_0.05 <- (
    !is.na(out$adj.P.Val) &
    out$adj.P.Val < 0.05
  )

  out$direction <- ifelse(
    out$logFC > 0,
    "Higher_in_SCZ",
    ifelse(
      out$logFC < 0,
      "Lower_in_SCZ",
      "Zero"
    )
  )

  out
}

# ------------------------------------------------------------
# 14. Extract four contrasts
# ------------------------------------------------------------

RES_DLPFC <- extract_result(
  FIT_EBAYES,
  "DLPFC_SCZ",
  ANNOTATION_PRIMARY
)

RES_HIPPO <- extract_result(
  FIT_EBAYES,
  "HIPPO_SCZ",
  ANNOTATION_PRIMARY
)

RES_SHARED <- extract_result(
  FIT_EBAYES,
  "Shared_SCZ",
  ANNOTATION_PRIMARY
)

RES_DEVIATION <- extract_result(
  FIT_EBAYES,
  "Regional_Deviation",
  ANNOTATION_PRIMARY
)

# ------------------------------------------------------------
# 15. Significance summary
# ------------------------------------------------------------

result_summary <- function(
  x,
  contrast_name
) {

  data.frame(
    contrast = contrast_name,

    n_genes = nrow(x),

    FDR_lt_0.05 = sum(
      x$adj.P.Val < 0.05,
      na.rm = TRUE
    ),

    FDR_lt_0.05_positive = sum(
      x$adj.P.Val < 0.05 &
      x$logFC > 0,
      na.rm = TRUE
    ),

    FDR_lt_0.05_negative = sum(
      x$adj.P.Val < 0.05 &
      x$logFC < 0,
      na.rm = TRUE
    ),

    P_lt_0.001 = sum(
      x$P.Value < 0.001,
      na.rm = TRUE
    ),

    stringsAsFactors = FALSE
  )
}

DE_SUMMARY <- rbind(

  result_summary(
    RES_DLPFC,
    "DLPFC_SCZ"
  ),

  result_summary(
    RES_HIPPO,
    "HIPPO_SCZ"
  ),

  result_summary(
    RES_SHARED,
    "Shared_SCZ"
  ),

  result_summary(
    RES_DEVIATION,
    "Regional_Deviation"
  )
)

# ------------------------------------------------------------
# 16. Shared-effect sign concordance
# ------------------------------------------------------------

REGION_EFFECTS <- data.frame(

  ensemblID_stable =
    RES_DLPFC$ensemblID_stable,

  DLPFC_logFC =
    RES_DLPFC$logFC[
      match(
        RES_DLPFC$ensemblID_stable,
        RES_DLPFC$ensemblID_stable
      )
    ],

  HIPPO_logFC =
    RES_HIPPO$logFC[
      match(
        RES_DLPFC$ensemblID_stable,
        RES_HIPPO$ensemblID_stable
      )
    ],

  stringsAsFactors = FALSE
)

REGION_EFFECTS$sign_concordant <- (
  sign(REGION_EFFECTS$DLPFC_logFC) ==
  sign(REGION_EFFECTS$HIPPO_logFC)
)

SIGN_CONCORDANCE_RATE <- mean(
  REGION_EFFECTS$sign_concordant,
  na.rm = TRUE
)

REGION_LOGFC_COR <- cor(
  REGION_EFFECTS$DLPFC_logFC,
  REGION_EFFECTS$HIPPO_logFC,
  use = "complete.obs",
  method = "pearson"
)

# ------------------------------------------------------------
# 17. Save outputs
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "05_differential_expression"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

DLPFC_PATH <- file.path(
  OUT_DIR,
  "step03a_DLPFC_SCZ_vs_Control_v1.csv"
)

HIPPO_PATH <- file.path(
  OUT_DIR,
  "step03a_HIPPO_SCZ_vs_Control_v1.csv"
)

SHARED_PATH <- file.path(
  OUT_DIR,
  "step03a_Shared_SCZ_effect_v1.csv"
)

DEVIATION_PATH <- file.path(
  OUT_DIR,
  "step03a_Regional_Deviation_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step03a_DE_summary_v1.csv"
)

DESIGN_PATH <- file.path(
  OUT_DIR,
  "step03a_design_matrix_v1.csv"
)

CONTRAST_PATH <- file.path(
  OUT_DIR,
  "step03a_contrast_matrix_v1.csv"
)

MODEL_AUDIT_PATH <- file.path(
  OUT_DIR,
  "step03a_model_audit_v1.txt"
)

assert_project_sentinel()

write.csv(
  RES_DLPFC,
  DLPFC_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  RES_HIPPO,
  HIPPO_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  RES_SHARED,
  SHARED_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  RES_DEVIATION,
  DEVIATION_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  DE_SUMMARY,
  SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  DESIGN,
  DESIGN_PATH,
  row.names = FALSE
)

write.csv(
  CONTRASTS,
  CONTRAST_PATH,
  row.names = TRUE
)

MODEL_AUDIT <- c(
  paste0(
    "samples=",
    nrow(MODEL_META)
  ),
  paste0(
    "donors=",
    length(
      unique(
        MODEL_META$BrNum
      )
    )
  ),
  paste0(
    "design_columns=",
    ncol(DESIGN)
  ),
  paste0(
    "design_rank=",
    DESIGN_RANK
  ),
  paste0(
    "duplicateCorrelation=",
    format(
      DONOR_CORRELATION,
      digits = 10
    )
  ),
  paste0(
    "donor_balanced_total_weight=",
    sum(
      MODEL_META$donor_balance_weight
    )
  ),
  paste0(
    "DLPFC_HIPPO_logFC_correlation=",
    format(
      REGION_LOGFC_COR,
      digits = 10
    )
  ),
  paste0(
    "DLPFC_HIPPO_sign_concordance=",
    format(
      SIGN_CONCORDANCE_RATE,
      digits = 10
    )
  )
)

writeLines(
  MODEL_AUDIT,
  MODEL_AUDIT_PATH
)

# ------------------------------------------------------------
# 18. Final validation
# ------------------------------------------------------------

stopifnot(
  nrow(RES_DLPFC) == 23740L,
  nrow(RES_HIPPO) == 23740L,
  nrow(RES_SHARED) == 23740L,
  nrow(RES_DEVIATION) == 23740L,
  all(
    is.finite(
      RES_SHARED$logFC
    )
  ),
  file.exists(DLPFC_PATH),
  file.exists(HIPPO_PATH),
  file.exists(SHARED_PATH),
  file.exists(DEVIATION_PATH)
)

# ------------------------------------------------------------
# 19. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 03A — DONOR-BLOCKED DESCRIPTIVE DE\n")
cat("============================================================\n")

cat(
  "Design columns:              ",
  ncol(DESIGN),
  "\n",
  sep = ""
)

cat(
  "Design rank:                 ",
  DESIGN_RANK,
  "\n",
  sep = ""
)

cat(
  "Estimated donor correlation: ",
  round(
    DONOR_CORRELATION,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "DLPFC-HIPPO logFC correlation: ",
  round(
    REGION_LOGFC_COR,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "DLPFC-HIPPO sign concordance:  ",
  round(
    SIGN_CONCORDANCE_RATE,
    4
  ),
  "\n",
  sep = ""
)

cat("\n--- DE SUMMARY ---\n")

print(
  DE_SUMMARY,
  row.names = FALSE
)

cat("\n--- TOP 10 SHARED SCZ GENES ---\n")

print(
  RES_SHARED[
    1:10,
    c(
      "ensemblID_stable",
      "gene_symbol",
      "gene_type",
      "logFC",
      "P.Value",
      "adj.P.Val"
    )
  ],
  row.names = FALSE
)

cat("\n--- TOP 10 REGIONAL-DEVIATION GENES ---\n")

print(
  RES_DEVIATION[
    1:10,
    c(
      "ensemblID_stable",
      "gene_symbol",
      "gene_type",
      "logFC",
      "P.Value",
      "adj.P.Val"
    )
  ],
  row.names = FALSE
)

cat("\n============================================================\n")
cat("STEP 03A — CELL 1 PASS\n")
cat("============================================================\n")

In [ ]:
# ============================================================
# STEP 03A — CELL 1 (OPTIMIZED)
# FAST DONOR-BLOCKED LIMMA-VOOM DESCRIPTIVE DE
# ============================================================

assert_project_sentinel()

cat("STEP 03A optimized analysis starting...\n")
TOTAL_START <- proc.time()[3]

# ------------------------------------------------------------
# 1. Packages
# ------------------------------------------------------------

if (!requireNamespace("edgeR", quietly = TRUE)) {
  stop("edgeR unavailable.")
}

if (!requireNamespace("limma", quietly = TRUE)) {
  stop("limma unavailable.")
}

# ------------------------------------------------------------
# 2. Reload canonical inputs from disk
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_gene_annotation_v1.csv"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

COUNTS_PRIMARY <- readRDS(
  COUNTS_PATH
)

ANNOTATION_PRIMARY <- read.csv(
  ANNOTATION_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

MODEL_META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(COUNTS_PRIMARY) == 23740L,
  ncol(COUNTS_PRIMARY) == 755L,
  nrow(MODEL_META) == 755L,
  length(unique(MODEL_META$BrNum)) == 454L
)

# ------------------------------------------------------------
# 3. Factors
# ------------------------------------------------------------

MODEL_META$Region <- factor(
  MODEL_META$Region,
  levels = c("DLPFC", "HIPPO")
)

MODEL_META$Dx <- factor(
  MODEL_META$Dx,
  levels = c("Control", "Schizo")
)

MODEL_META$Sex <- factor(
  MODEL_META$Sex
)

MODEL_META$Race <- factor(
  MODEL_META$Race
)

MODEL_META$group <- factor(
  paste(
    MODEL_META$Region,
    MODEL_META$Dx,
    sep = "_"
  ),
  levels = c(
    "DLPFC_Control",
    "DLPFC_Schizo",
    "HIPPO_Control",
    "HIPPO_Schizo"
  )
)

# ------------------------------------------------------------
# 4. Standardize continuous covariates
# ------------------------------------------------------------

MODEL_META$age_z <- as.numeric(
  scale(MODEL_META$donor_age)
)

MODEL_META$RIN_z <- as.numeric(
  scale(MODEL_META$RIN)
)

for (i in 1:10) {

  MODEL_META[[paste0("snpPC", i, "_z")]] <- as.numeric(
    scale(
      MODEL_META[[paste0("snpPC", i)]]
    )
  )
}

# ------------------------------------------------------------
# 5. Full design matrix
# ------------------------------------------------------------

DESIGN <- model.matrix(
  ~ 0 +
    group +
    age_z +
    Sex +
    Race +
    RIN_z +
    snpPC1_z +
    snpPC2_z +
    snpPC3_z +
    snpPC4_z +
    snpPC5_z +
    snpPC6_z +
    snpPC7_z +
    snpPC8_z +
    snpPC9_z +
    snpPC10_z,
  data = MODEL_META
)

colnames(DESIGN) <- sub(
  "^group",
  "",
  colnames(DESIGN)
)

if (qr(DESIGN)$rank != ncol(DESIGN)) {

  print(
    limma::nonEstimable(DESIGN)
  )

  stop(
    "Full design matrix is rank deficient."
  )
}

# ------------------------------------------------------------
# 6. Contrasts
# ------------------------------------------------------------

CONTRASTS <- limma::makeContrasts(

  DLPFC_SCZ =
    DLPFC_Schizo -
    DLPFC_Control,

  HIPPO_SCZ =
    HIPPO_Schizo -
    HIPPO_Control,

  Shared_SCZ =
    0.5 * (
      DLPFC_Schizo -
      DLPFC_Control +
      HIPPO_Schizo -
      HIPPO_Control
    ),

  Regional_Deviation =
    0.5 * (
      DLPFC_Schizo -
      DLPFC_Control -
      HIPPO_Schizo +
      HIPPO_Control
    ),

  levels = DESIGN
)

# ------------------------------------------------------------
# 7. DGE + TMM
# ------------------------------------------------------------

cat("Running TMM + initial voom...\n")

DGE <- edgeR::DGEList(
  counts = COUNTS_PRIMARY
)

DGE <- edgeR::calcNormFactors(
  DGE,
  method = "TMM"
)

VOOM_PRE <- limma::voom(
  DGE,
  design = DESIGN,
  plot = FALSE
)

cat("Initial voom: PASS\n")

# ------------------------------------------------------------
# 8. Paired samples only for correlation estimation
# ------------------------------------------------------------

PAIRED_INDEX <- which(
  MODEL_META$paired_donor
)

if (length(PAIRED_INDEX) != 602L) {
  stop(
    paste0(
      "Expected 602 paired samples; found ",
      length(PAIRED_INDEX)
    )
  )
}

META_CORR <- droplevels(
  MODEL_META[
    PAIRED_INDEX,
    ,
    drop = FALSE
  ]
)

# ------------------------------------------------------------
# 9. Correlation-estimation design
#
# Rebuild after droplevels because some Race categories
# are absent among paired donors.
# ------------------------------------------------------------

DESIGN_CORR <- model.matrix(
  ~ 0 +
    group +
    age_z +
    Sex +
    Race +
    RIN_z +
    snpPC1_z +
    snpPC2_z +
    snpPC3_z +
    snpPC4_z +
    snpPC5_z +
    snpPC6_z +
    snpPC7_z +
    snpPC8_z +
    snpPC9_z +
    snpPC10_z,
  data = META_CORR
)

colnames(DESIGN_CORR) <- sub(
  "^group",
  "",
  colnames(DESIGN_CORR)
)

if (qr(DESIGN_CORR)$rank != ncol(DESIGN_CORR)) {

  cat("\nNon-estimable columns in paired-sample design:\n")

  print(
    limma::nonEstimable(
      DESIGN_CORR
    )
  )

  stop(
    "Paired-sample correlation design is rank deficient."
  )
}

# ------------------------------------------------------------
# 10. Reproducible 1000-gene subset
#
# All genes already passed primary expression filtering.
# Outcome is NOT used for selection.
# ------------------------------------------------------------

set.seed(20260929)

CORR_GENE_N <- min(
  1000L,
  nrow(VOOM_PRE$E)
)

CORR_GENE_INDEX <- sort(
  sample(
    seq_len(nrow(VOOM_PRE$E)),
    size = CORR_GENE_N,
    replace = FALSE
  )
)

CORR_GENE_IDS <- rownames(
  VOOM_PRE$E
)[
  CORR_GENE_INDEX
]

cat(
  "Estimating donor correlation using ",
  CORR_GENE_N,
  " genes and ",
  length(PAIRED_INDEX),
  " paired samples...\n",
  sep = ""
)

# ------------------------------------------------------------
# 11. FAST duplicateCorrelation
# ------------------------------------------------------------

CORR_START <- proc.time()[3]

CORFIT <- limma::duplicateCorrelation(

  VOOM_PRE$E[
    CORR_GENE_INDEX,
    PAIRED_INDEX,
    drop = FALSE
  ],

  design = DESIGN_CORR,

  block = META_CORR$BrNum,

  weights = VOOM_PRE$weights[
    CORR_GENE_INDEX,
    PAIRED_INDEX,
    drop = FALSE
  ],

  trim = 0.15
)

CORR_SECONDS <- proc.time()[3] - CORR_START

DONOR_CORRELATION <- CORFIT$consensus.correlation

if (!is.finite(DONOR_CORRELATION)) {
  stop(
    "Invalid donor correlation estimate."
  )
}

cat(
  "Consensus donor correlation = ",
  round(DONOR_CORRELATION, 5),
  "\n",
  sep = ""
)

cat(
  "Correlation estimation time = ",
  round(CORR_SECONDS, 1),
  " seconds\n",
  sep = ""
)

# ------------------------------------------------------------
# 12. Final voom using fixed donor correlation
# ------------------------------------------------------------

cat("Running final voom on all 23,740 genes...\n")

VOOM_FINAL <- limma::voom(
  DGE,
  design = DESIGN,
  block = MODEL_META$BrNum,
  correlation = DONOR_CORRELATION,
  plot = FALSE
)

# donor-balanced weights
VOOM_FINAL$weights <- sweep(
  VOOM_FINAL$weights,
  2,
  MODEL_META$donor_balance_weight,
  "*"
)

cat("Final voom: PASS\n")

# ------------------------------------------------------------
# 13. Full-gene donor-blocked fit
# ------------------------------------------------------------

cat("Fitting all genes...\n")

FIT <- limma::lmFit(
  VOOM_FINAL,
  design = DESIGN,
  block = MODEL_META$BrNum,
  correlation = DONOR_CORRELATION
)

FIT <- limma::contrasts.fit(
  FIT,
  CONTRASTS
)

FIT <- limma::eBayes(
  FIT,
  robust = TRUE
)

cat("Full-gene model fit: PASS\n")

# ------------------------------------------------------------
# 14. Result extraction
# ------------------------------------------------------------

extract_result <- function(
  fit,
  coef_name,
  annotation
) {

  tt <- limma::topTable(
    fit,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH",
    sort.by = "P"
  )

  tt$ensemblID_stable <- rownames(tt)
  rownames(tt) <- NULL

  ann <- annotation[
    ,
    c(
      "ensemblID_stable",
      "gene_symbol",
      "gene_type",
      "chromosome",
      "entrez_id"
    ),
    drop = FALSE
  ]

  m <- match(
    tt$ensemblID_stable,
    ann$ensemblID_stable
  )

  tt$gene_symbol <- ann$gene_symbol[m]
  tt$gene_type <- ann$gene_type[m]
  tt$chromosome <- ann$chromosome[m]
  tt$entrez_id <- ann$entrez_id[m]

  tt$FDR_lt_0.05 <- (
    tt$adj.P.Val < 0.05
  )

  tt$direction <- ifelse(
    tt$logFC > 0,
    "Higher_in_SCZ",
    ifelse(
      tt$logFC < 0,
      "Lower_in_SCZ",
      "Zero"
    )
  )

  tt
}

RES_DLPFC <- extract_result(
  FIT,
  "DLPFC_SCZ",
  ANNOTATION_PRIMARY
)

RES_HIPPO <- extract_result(
  FIT,
  "HIPPO_SCZ",
  ANNOTATION_PRIMARY
)

RES_SHARED <- extract_result(
  FIT,
  "Shared_SCZ",
  ANNOTATION_PRIMARY
)

RES_DEVIATION <- extract_result(
  FIT,
  "Regional_Deviation",
  ANNOTATION_PRIMARY
)

# ------------------------------------------------------------
# 15. Summary
# ------------------------------------------------------------

summarize_de <- function(x, name) {

  data.frame(

    contrast = name,

    n_genes = nrow(x),

    FDR_lt_0.05 = sum(
      x$adj.P.Val < 0.05,
      na.rm = TRUE
    ),

    FDR_lt_0.05_positive = sum(
      x$adj.P.Val < 0.05 &
      x$logFC > 0,
      na.rm = TRUE
    ),

    FDR_lt_0.05_negative = sum(
      x$adj.P.Val < 0.05 &
      x$logFC < 0,
      na.rm = TRUE
    ),

    P_lt_0.001 = sum(
      x$P.Value < 0.001,
      na.rm = TRUE
    ),

    stringsAsFactors = FALSE
  )
}

DE_SUMMARY <- rbind(

  summarize_de(
    RES_DLPFC,
    "DLPFC_SCZ"
  ),

  summarize_de(
    RES_HIPPO,
    "HIPPO_SCZ"
  ),

  summarize_de(
    RES_SHARED,
    "Shared_SCZ"
  ),

  summarize_de(
    RES_DEVIATION,
    "Regional_Deviation"
  )
)

# ------------------------------------------------------------
# 16. Cross-region effect concordance
# ------------------------------------------------------------

DLPFC_MATCH <- match(
  ANNOTATION_PRIMARY$ensemblID_stable,
  RES_DLPFC$ensemblID_stable
)

HIPPO_MATCH <- match(
  ANNOTATION_PRIMARY$ensemblID_stable,
  RES_HIPPO$ensemblID_stable
)

DLPFC_LOGFC <- RES_DLPFC$logFC[
  DLPFC_MATCH
]

HIPPO_LOGFC <- RES_HIPPO$logFC[
  HIPPO_MATCH
]

REGION_LOGFC_COR <- cor(
  DLPFC_LOGFC,
  HIPPO_LOGFC,
  use = "complete.obs"
)

SIGN_CONCORDANCE_RATE <- mean(
  sign(DLPFC_LOGFC) ==
  sign(HIPPO_LOGFC),
  na.rm = TRUE
)

# ------------------------------------------------------------
# 17. Save outputs
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "05_differential_expression"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

DLPFC_PATH <- file.path(
  OUT_DIR,
  "step03a_DLPFC_SCZ_vs_Control_v1.csv"
)

HIPPO_PATH <- file.path(
  OUT_DIR,
  "step03a_HIPPO_SCZ_vs_Control_v1.csv"
)

SHARED_PATH <- file.path(
  OUT_DIR,
  "step03a_Shared_SCZ_effect_v1.csv"
)

DEVIATION_PATH <- file.path(
  OUT_DIR,
  "step03a_Regional_Deviation_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step03a_DE_summary_v1.csv"
)

CORR_GENE_PATH <- file.path(
  OUT_DIR,
  "step03a_correlation_estimation_genes_v1.csv"
)

MODEL_AUDIT_PATH <- file.path(
  OUT_DIR,
  "step03a_model_audit_v1.txt"
)

assert_project_sentinel()

write.csv(
  RES_DLPFC,
  DLPFC_PATH,
  row.names = FALSE
)

write.csv(
  RES_HIPPO,
  HIPPO_PATH,
  row.names = FALSE
)

write.csv(
  RES_SHARED,
  SHARED_PATH,
  row.names = FALSE
)

write.csv(
  RES_DEVIATION,
  DEVIATION_PATH,
  row.names = FALSE
)

write.csv(
  DE_SUMMARY,
  SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  data.frame(
    ensemblID_stable = CORR_GENE_IDS,
    stringsAsFactors = FALSE
  ),
  CORR_GENE_PATH,
  row.names = FALSE
)

TOTAL_SECONDS <- proc.time()[3] - TOTAL_START

MODEL_AUDIT <- c(
  "method=limma_voom_donor_blocked",
  "correlation_estimation=1000_reproducibly_sampled_genes_paired_donors_only",
  paste0(
    "correlation_gene_seed=20260929"
  ),
  paste0(
    "correlation_genes=",
    CORR_GENE_N
  ),
  paste0(
    "paired_samples_for_correlation=",
    length(PAIRED_INDEX)
  ),
  paste0(
    "consensus_donor_correlation=",
    DONOR_CORRELATION
  ),
  paste0(
    "correlation_estimation_seconds=",
    CORR_SECONDS
  ),
  paste0(
    "total_seconds=",
    TOTAL_SECONDS
  ),
  paste0(
    "DLPFC_HIPPO_logFC_correlation=",
    REGION_LOGFC_COR
  ),
  paste0(
    "DLPFC_HIPPO_sign_concordance=",
    SIGN_CONCORDANCE_RATE
  )
)

writeLines(
  MODEL_AUDIT,
  MODEL_AUDIT_PATH
)

# ------------------------------------------------------------
# 18. Validation
# ------------------------------------------------------------

stopifnot(
  nrow(RES_DLPFC) == 23740L,
  nrow(RES_HIPPO) == 23740L,
  nrow(RES_SHARED) == 23740L,
  nrow(RES_DEVIATION) == 23740L
)

# ------------------------------------------------------------
# 19. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 03A — OPTIMIZED DONOR-BLOCKED DE\n")
cat("============================================================\n")

cat(
  "Consensus donor correlation:     ",
  round(DONOR_CORRELATION, 5),
  "\n",
  sep = ""
)

cat(
  "Correlation estimation seconds:  ",
  round(CORR_SECONDS, 1),
  "\n",
  sep = ""
)

cat(
  "Total cell seconds:               ",
  round(TOTAL_SECONDS, 1),
  "\n",
  sep = ""
)

cat(
  "DLPFC-HIPPO logFC correlation:   ",
  round(REGION_LOGFC_COR, 4),
  "\n",
  sep = ""
)

cat(
  "DLPFC-HIPPO sign concordance:    ",
  round(SIGN_CONCORDANCE_RATE, 4),
  "\n",
  sep = ""
)

cat("\n--- DE SUMMARY ---\n")

print(
  DE_SUMMARY,
  row.names = FALSE
)

cat("\n--- TOP 10 SHARED SCZ GENES ---\n")

print(
  RES_SHARED[
    1:10,
    c(
      "ensemblID_stable",
      "gene_symbol",
      "gene_type",
      "logFC",
      "P.Value",
      "adj.P.Val"
    )
  ],
  row.names = FALSE
)

cat("\n--- TOP 10 REGIONAL-DEVIATION GENES ---\n")

print(
  RES_DEVIATION[
    1:10,
    c(
      "ensemblID_stable",
      "gene_symbol",
      "gene_type",
      "logFC",
      "P.Value",
      "adj.P.Val"
    )
  ],
  row.names = FALSE
)

cat("\n============================================================\n")
cat("STEP 03A — OPTIMIZED CELL PASS\n")
cat("============================================================\n")

In [2]:
# ============================================================
# STEP 03A — FAST DONOR-BLOCKED DE
# FRESH-KERNEL + RENV SAFE VERSION
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

assert_project_sentinel <- function() {

  if (!dir.exists(PROJECT_ROOT)) {
    stop("Project root missing.")
  }

  if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
  }

  observed <- trimws(
    readLines(
      SENTINEL_PATH,
      warn = FALSE
    )[1]
  )

  if (!identical(observed, PROJECT_ID)) {
    stop(
      paste0(
        "Project ID mismatch. Expected ",
        PROJECT_ID,
        "; observed ",
        observed
      )
    )
  }

  invisible(TRUE)
}

assert_project_sentinel()

TOTAL_START <- proc.time()[3]

# ------------------------------------------------------------
# 1. Activate renv
# ------------------------------------------------------------

RENV_ACTIVATE <- file.path(
  PROJECT_ROOT,
  "renv",
  "activate.R"
)

if (file.exists(RENV_ACTIVATE)) {
  source(RENV_ACTIVATE)
}

cat("Project / renv activation: PASS\n")
flush.console()

# ------------------------------------------------------------
# 2. Install required Bioconductor packages INTO current library
# ------------------------------------------------------------

if (!requireNamespace("BiocManager", quietly = TRUE)) {

  install.packages(
    "BiocManager",
    repos = "https://cloud.r-project.org"
  )
}

NEEDED_BIOC <- c(
  "limma",
  "edgeR"
)

MISSING_BIOC <- NEEDED_BIOC[
  !vapply(
    NEEDED_BIOC,
    requireNamespace,
    logical(1),
    quietly = TRUE
  )
]

if (length(MISSING_BIOC) > 0L) {

  cat(
    "Installing missing packages: ",
    paste(MISSING_BIOC, collapse = ", "),
    "\n",
    sep = ""
  )

  flush.console()

  BiocManager::install(
    MISSING_BIOC,
    ask = FALSE,
    update = FALSE
  )
}

for (pkg in NEEDED_BIOC) {

  if (!requireNamespace(pkg, quietly = TRUE)) {

    stop(
      paste0(
        "Package installation failed: ",
        pkg
      )
    )
  }
}

cat(
  "limma version: ",
  as.character(
    packageVersion("limma")
  ),
  "\n",
  sep = ""
)

cat(
  "edgeR version: ",
  as.character(
    packageVersion("edgeR")
  ),
  "\n",
  sep = ""
)

cat("Required packages: PASS\n")
flush.console()

# ------------------------------------------------------------
# 3. Canonical inputs
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_gene_annotation_v1.csv"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

if (!all(
  file.exists(
    c(
      COUNTS_PATH,
      ANNOTATION_PATH,
      META_PATH
    )
  )
)) {
  stop("One or more canonical input files are missing.")
}

cat("Loading canonical data...\n")
flush.console()

COUNTS_PRIMARY <- readRDS(
  COUNTS_PATH
)

ANNOTATION_PRIMARY <- read.csv(
  ANNOTATION_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

MODEL_META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(COUNTS_PRIMARY) == 23740L,
  ncol(COUNTS_PRIMARY) == 755L,
  nrow(ANNOTATION_PRIMARY) == 23740L,
  nrow(MODEL_META) == 755L,
  length(unique(MODEL_META$BrNum)) == 454L,
  identical(
    rownames(COUNTS_PRIMARY),
    ANNOTATION_PRIMARY$ensemblID_stable
  )
)

cat(
  "Loaded: ",
  nrow(COUNTS_PRIMARY),
  " genes x ",
  ncol(COUNTS_PRIMARY),
  " samples\n",
  sep = ""
)

flush.console()

# ------------------------------------------------------------
# 4. Factors
# ------------------------------------------------------------

MODEL_META$Region <- factor(
  MODEL_META$Region,
  levels = c(
    "DLPFC",
    "HIPPO"
  )
)

MODEL_META$Dx <- factor(
  MODEL_META$Dx,
  levels = c(
    "Control",
    "Schizo"
  )
)

MODEL_META$Sex <- factor(
  MODEL_META$Sex
)

MODEL_META$Race <- factor(
  MODEL_META$Race
)

MODEL_META$group <- factor(
  paste(
    MODEL_META$Region,
    MODEL_META$Dx,
    sep = "_"
  ),
  levels = c(
    "DLPFC_Control",
    "DLPFC_Schizo",
    "HIPPO_Control",
    "HIPPO_Schizo"
  )
)

# ------------------------------------------------------------
# 5. Standardize continuous covariates
# ------------------------------------------------------------

MODEL_META$age_z <- as.numeric(
  scale(
    MODEL_META$donor_age
  )
)

MODEL_META$RIN_z <- as.numeric(
  scale(
    MODEL_META$RIN
  )
)

for (i in 1:10) {

  src <- paste0(
    "snpPC",
    i
  )

  dst <- paste0(
    "snpPC",
    i,
    "_z"
  )

  MODEL_META[[dst]] <- as.numeric(
    scale(
      MODEL_META[[src]]
    )
  )
}

# ------------------------------------------------------------
# 6. Full design
# ------------------------------------------------------------

DESIGN <- model.matrix(
  ~ 0 +
    group +
    age_z +
    Sex +
    Race +
    RIN_z +
    snpPC1_z +
    snpPC2_z +
    snpPC3_z +
    snpPC4_z +
    snpPC5_z +
    snpPC6_z +
    snpPC7_z +
    snpPC8_z +
    snpPC9_z +
    snpPC10_z,
  data = MODEL_META
)

colnames(DESIGN) <- sub(
  "^group",
  "",
  colnames(DESIGN)
)

if (qr(DESIGN)$rank != ncol(DESIGN)) {

  print(
    limma::nonEstimable(
      DESIGN
    )
  )

  stop("Full design matrix is rank deficient.")
}

cat(
  "Design: ",
  ncol(DESIGN),
  " columns; full rank\n",
  sep = ""
)

flush.console()

# ------------------------------------------------------------
# 7. Contrasts
# ------------------------------------------------------------

CONTRASTS <- limma::makeContrasts(

  DLPFC_SCZ =
    DLPFC_Schizo -
    DLPFC_Control,

  HIPPO_SCZ =
    HIPPO_Schizo -
    HIPPO_Control,

  Shared_SCZ =
    0.5 * (
      DLPFC_Schizo -
      DLPFC_Control +
      HIPPO_Schizo -
      HIPPO_Control
    ),

  Regional_Deviation =
    0.5 * (
      DLPFC_Schizo -
      DLPFC_Control -
      HIPPO_Schizo +
      HIPPO_Control
    ),

  levels = DESIGN
)

# ------------------------------------------------------------
# 8. TMM + initial voom
# ------------------------------------------------------------

cat("Running TMM normalization...\n")
flush.console()

DGE <- edgeR::DGEList(
  counts = COUNTS_PRIMARY
)

DGE <- edgeR::calcNormFactors(
  DGE,
  method = "TMM"
)

cat("Running initial voom...\n")
flush.console()

VOOM_PRE <- limma::voom(
  DGE,
  design = DESIGN,
  plot = FALSE
)

cat("Initial voom: PASS\n")
flush.console()

# ------------------------------------------------------------
# 9. Paired donor subset
# ------------------------------------------------------------

# read.csv can occasionally recover logical columns differently,
# so enforce paired_donor explicitly if needed.

if (!is.logical(MODEL_META$paired_donor)) {

  MODEL_META$paired_donor <- toupper(
    as.character(
      MODEL_META$paired_donor
    )
  ) == "TRUE"
}

MODEL_META$donor_balance_weight <- as.numeric(
  MODEL_META$donor_balance_weight
)

PAIRED_INDEX <- which(
  MODEL_META$paired_donor
)

if (length(PAIRED_INDEX) != 602L) {
  stop(
    paste0(
      "Expected 602 paired samples; found ",
      length(PAIRED_INDEX)
    )
  )
}

META_CORR <- droplevels(
  MODEL_META[
    PAIRED_INDEX,
    ,
    drop = FALSE
  ]
)

if (
  length(
    unique(
      META_CORR$BrNum
    )
  ) != 301L
) {
  stop("Expected 301 paired donors.")
}

cat(
  "Paired donors: 301; paired samples: 602\n"
)

flush.console()

# ------------------------------------------------------------
# 10. Paired-sample correlation design
# ------------------------------------------------------------

DESIGN_CORR <- model.matrix(
  ~ 0 +
    group +
    age_z +
    Sex +
    Race +
    RIN_z +
    snpPC1_z +
    snpPC2_z +
    snpPC3_z +
    snpPC4_z +
    snpPC5_z +
    snpPC6_z +
    snpPC7_z +
    snpPC8_z +
    snpPC9_z +
    snpPC10_z,
  data = META_CORR
)

colnames(DESIGN_CORR) <- sub(
  "^group",
  "",
  colnames(DESIGN_CORR)
)

if (
  qr(DESIGN_CORR)$rank !=
  ncol(DESIGN_CORR)
) {

  print(
    limma::nonEstimable(
      DESIGN_CORR
    )
  )

  stop(
    "Paired correlation design is rank deficient."
  )
}

# ------------------------------------------------------------
# 11. Outcome-blind variance-stratified 500-gene subset
# ------------------------------------------------------------

GENE_VAR <- apply(
  VOOM_PRE$E,
  1,
  var
)

if (any(!is.finite(GENE_VAR))) {
  stop("Non-finite gene variance detected.")
}

# rank genes by variance and split into 10 equally sized bins
VAR_RANK <- rank(
  GENE_VAR,
  ties.method = "first"
)

VAR_BIN <- cut(
  VAR_RANK,
  breaks = 10,
  labels = FALSE
)

set.seed(20260929)

CORR_GENE_INDEX <- unlist(
  lapply(
    1:10,
    function(b) {

      candidates <- which(
        VAR_BIN == b
      )

      sample(
        candidates,
        size = min(
          50L,
          length(candidates)
        ),
        replace = FALSE
      )
    }
  ),
  use.names = FALSE
)

CORR_GENE_INDEX <- sort(
  unique(
    CORR_GENE_INDEX
  )
)

CORR_GENE_N <- length(
  CORR_GENE_INDEX
)

if (CORR_GENE_N != 500L) {
  stop(
    paste0(
      "Expected 500 correlation genes; obtained ",
      CORR_GENE_N
    )
  )
}

CORR_GENE_IDS <- rownames(
  VOOM_PRE$E
)[
  CORR_GENE_INDEX
]

cat(
  "Estimating donor correlation from 500 representative genes...\n"
)

flush.console()

# ------------------------------------------------------------
# 12. Fast duplicateCorrelation
# ------------------------------------------------------------

CORR_START <- proc.time()[3]

CORFIT <- limma::duplicateCorrelation(

  VOOM_PRE$E[
    CORR_GENE_INDEX,
    PAIRED_INDEX,
    drop = FALSE
  ],

  design = DESIGN_CORR,

  block = META_CORR$BrNum,

  weights = VOOM_PRE$weights[
    CORR_GENE_INDEX,
    PAIRED_INDEX,
    drop = FALSE
  ],

  trim = 0.15
)

CORR_SECONDS <- proc.time()[3] -
  CORR_START

DONOR_CORRELATION <- CORFIT$consensus.correlation

if (!is.finite(DONOR_CORRELATION)) {
  stop("Invalid donor correlation estimate.")
}

cat(
  "Consensus donor correlation: ",
  round(
    DONOR_CORRELATION,
    6
  ),
  "\n",
  sep = ""
)

cat(
  "Correlation estimation time: ",
  round(
    CORR_SECONDS,
    1
  ),
  " sec\n",
  sep = ""
)

flush.console()

# ------------------------------------------------------------
# 13. Final full-gene voom
# ------------------------------------------------------------

cat(
  "Running final voom on all 23,740 genes...\n"
)

flush.console()

VOOM_FINAL <- limma::voom(
  DGE,
  design = DESIGN,
  block = MODEL_META$BrNum,
  correlation = DONOR_CORRELATION,
  plot = FALSE
)

# Donor-balanced weights
VOOM_FINAL$weights <- sweep(
  VOOM_FINAL$weights,
  2,
  MODEL_META$donor_balance_weight,
  "*"
)

cat("Final voom: PASS\n")
flush.console()

# ------------------------------------------------------------
# 14. Final full-gene fit
# ------------------------------------------------------------

cat(
  "Fitting 23,740 genes...\n"
)

flush.console()

FIT <- limma::lmFit(
  VOOM_FINAL,
  design = DESIGN,
  block = MODEL_META$BrNum,
  correlation = DONOR_CORRELATION
)

FIT <- limma::contrasts.fit(
  FIT,
  CONTRASTS
)

FIT <- limma::eBayes(
  FIT,
  robust = TRUE
)

cat("Full-gene fit: PASS\n")
flush.console()

# ------------------------------------------------------------
# 15. Extract results
# ------------------------------------------------------------

extract_result <- function(
  fit,
  coef_name,
  annotation
) {

  tt <- limma::topTable(
    fit,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH",
    sort.by = "P"
  )

  tt$ensemblID_stable <- rownames(
    tt
  )

  rownames(tt) <- NULL

  idx <- match(
    tt$ensemblID_stable,
    annotation$ensemblID_stable
  )

  if (anyNA(idx)) {
    stop(
      paste0(
        "Annotation matching failed: ",
        coef_name
      )
    )
  }

  tt$gene_symbol <-
    annotation$gene_symbol[idx]

  tt$gene_type <-
    annotation$gene_type[idx]

  tt$chromosome <-
    annotation$chromosome[idx]

  tt$entrez_id <-
    annotation$entrez_id[idx]

  tt$FDR_lt_0.05 <-
    tt$adj.P.Val < 0.05

  tt$direction <- ifelse(
    tt$logFC > 0,
    "Higher_in_SCZ",
    ifelse(
      tt$logFC < 0,
      "Lower_in_SCZ",
      "Zero"
    )
  )

  tt
}

cat("Extracting results...\n")
flush.console()

RES_DLPFC <- extract_result(
  FIT,
  "DLPFC_SCZ",
  ANNOTATION_PRIMARY
)

RES_HIPPO <- extract_result(
  FIT,
  "HIPPO_SCZ",
  ANNOTATION_PRIMARY
)

RES_SHARED <- extract_result(
  FIT,
  "Shared_SCZ",
  ANNOTATION_PRIMARY
)

RES_DEVIATION <- extract_result(
  FIT,
  "Regional_Deviation",
  ANNOTATION_PRIMARY
)

# ------------------------------------------------------------
# 16. Summary
# ------------------------------------------------------------

summarize_de <- function(
  x,
  contrast_name
) {

  data.frame(

    contrast = contrast_name,

    n_genes = nrow(x),

    FDR_lt_0.05 = sum(
      x$adj.P.Val < 0.05,
      na.rm = TRUE
    ),

    FDR_lt_0.05_positive = sum(
      x$adj.P.Val < 0.05 &
      x$logFC > 0,
      na.rm = TRUE
    ),

    FDR_lt_0.05_negative = sum(
      x$adj.P.Val < 0.05 &
      x$logFC < 0,
      na.rm = TRUE
    ),

    P_lt_0.001 = sum(
      x$P.Value < 0.001,
      na.rm = TRUE
    ),

    stringsAsFactors = FALSE
  )
}

DE_SUMMARY <- rbind(

  summarize_de(
    RES_DLPFC,
    "DLPFC_SCZ"
  ),

  summarize_de(
    RES_HIPPO,
    "HIPPO_SCZ"
  ),

  summarize_de(
    RES_SHARED,
    "Shared_SCZ"
  ),

  summarize_de(
    RES_DEVIATION,
    "Regional_Deviation"
  )
)

# ------------------------------------------------------------
# 17. DLPFC-HIPPO effect concordance
# ------------------------------------------------------------

d_idx <- match(
  ANNOTATION_PRIMARY$ensemblID_stable,
  RES_DLPFC$ensemblID_stable
)

h_idx <- match(
  ANNOTATION_PRIMARY$ensemblID_stable,
  RES_HIPPO$ensemblID_stable
)

DLPFC_LOGFC <- RES_DLPFC$logFC[
  d_idx
]

HIPPO_LOGFC <- RES_HIPPO$logFC[
  h_idx
]

REGION_LOGFC_COR <- cor(
  DLPFC_LOGFC,
  HIPPO_LOGFC,
  use = "complete.obs",
  method = "pearson"
)

SIGN_CONCORDANCE_RATE <- mean(
  sign(DLPFC_LOGFC) ==
    sign(HIPPO_LOGFC),
  na.rm = TRUE
)

# ------------------------------------------------------------
# 18. Save outputs
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "05_differential_expression"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

DLPFC_PATH <- file.path(
  OUT_DIR,
  "step03a_DLPFC_SCZ_vs_Control_v1.csv"
)

HIPPO_PATH <- file.path(
  OUT_DIR,
  "step03a_HIPPO_SCZ_vs_Control_v1.csv"
)

SHARED_PATH <- file.path(
  OUT_DIR,
  "step03a_Shared_SCZ_effect_v1.csv"
)

DEVIATION_PATH <- file.path(
  OUT_DIR,
  "step03a_Regional_Deviation_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step03a_DE_summary_v1.csv"
)

CORR_GENE_PATH <- file.path(
  OUT_DIR,
  "step03a_correlation_estimation_genes_v1.csv"
)

MODEL_AUDIT_PATH <- file.path(
  OUT_DIR,
  "step03a_model_audit_v1.txt"
)

assert_project_sentinel()

write.csv(
  RES_DLPFC,
  DLPFC_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  RES_HIPPO,
  HIPPO_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  RES_SHARED,
  SHARED_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  RES_DEVIATION,
  DEVIATION_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  DE_SUMMARY,
  SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  data.frame(
    ensemblID_stable =
      CORR_GENE_IDS,
    variance =
      GENE_VAR[
        CORR_GENE_INDEX
      ],
    stringsAsFactors = FALSE
  ),
  CORR_GENE_PATH,
  row.names = FALSE
)

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START

MODEL_AUDIT <- c(

  "method=limma_voom_donor_blocked",

  "correlation_estimation=500_variance_stratified_outcome_blind_genes",

  "correlation_gene_seed=20260929",

  "paired_donors=301",

  "paired_samples=602",

  paste0(
    "consensus_donor_correlation=",
    DONOR_CORRELATION
  ),

  paste0(
    "correlation_seconds=",
    CORR_SECONDS
  ),

  paste0(
    "total_seconds=",
    TOTAL_SECONDS
  ),

  paste0(
    "DLPFC_HIPPO_logFC_correlation=",
    REGION_LOGFC_COR
  ),

  paste0(
    "DLPFC_HIPPO_sign_concordance=",
    SIGN_CONCORDANCE_RATE
  )
)

writeLines(
  MODEL_AUDIT,
  MODEL_AUDIT_PATH
)

# ------------------------------------------------------------
# 19. Validate
# ------------------------------------------------------------

stopifnot(
  nrow(RES_DLPFC) == 23740L,
  nrow(RES_HIPPO) == 23740L,
  nrow(RES_SHARED) == 23740L,
  nrow(RES_DEVIATION) == 23740L,
  file.exists(DLPFC_PATH),
  file.exists(HIPPO_PATH),
  file.exists(SHARED_PATH),
  file.exists(DEVIATION_PATH)
)

# ------------------------------------------------------------
# 20. Final output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 03A — FAST DONOR-BLOCKED DE\n")
cat("============================================================\n")

cat(
  "Correlation genes:             ",
  CORR_GENE_N,
  "\n",
  sep = ""
)

cat(
  "Consensus donor correlation:   ",
  round(
    DONOR_CORRELATION,
    5
  ),
  "\n",
  sep = ""
)

cat(
  "Correlation seconds:           ",
  round(
    CORR_SECONDS,
    1
  ),
  "\n",
  sep = ""
)

cat(
  "Total seconds:                 ",
  round(
    TOTAL_SECONDS,
    1
  ),
  "\n",
  sep = ""
)

cat(
  "DLPFC-HIPPO logFC correlation: ",
  round(
    REGION_LOGFC_COR,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "DLPFC-HIPPO sign concordance:  ",
  round(
    SIGN_CONCORDANCE_RATE,
    4
  ),
  "\n",
  sep = ""
)

cat("\n--- DE SUMMARY ---\n")

print(
  DE_SUMMARY,
  row.names = FALSE
)

cat("\n--- TOP 10 SHARED SCZ GENES ---\n")

print(
  RES_SHARED[
    1:10,
    c(
      "ensemblID_stable",
      "gene_symbol",
      "gene_type",
      "logFC",
      "P.Value",
      "adj.P.Val"
    )
  ],
  row.names = FALSE
)

cat("\n--- TOP 10 REGIONAL-DEVIATION GENES ---\n")

print(
  RES_DEVIATION[
    1:10,
    c(
      "ensemblID_stable",
      "gene_symbol",
      "gene_type",
      "logFC",
      "P.Value",
      "adj.P.Val"
    )
  ],
  row.names = FALSE
)

cat("\n============================================================\n")
cat("STEP 03A — CELL PASS\n")
cat("============================================================\n")

The following package(s) were loaded before renv activated this project:
- digest      [E:/R-4.5.2/library/digest]
- IRdisplay   [E:/R-4.5.2/library/IRdisplay]
- base64enc   [E:/R-4.5.2/library/base64enc]
- fastmap     [E:/R-4.5.2/library/fastmap]
- glue        [E:/R-4.5.2/library/glue]
- htmltools   [E:/R-4.5.2/library/htmltools]
- repr        [E:/R-4.5.2/library/repr]
- lifecycle   [E:/R-4.5.2/library/lifecycle]
- cli         [E:/R-4.5.2/library/cli]
- vctrs       [E:/R-4.5.2/library/vctrs]
- textshaping [E:/R-4.5.2/library/textshaping]
- pbdZMQ      [E:/R-4.5.2/library/pbdZMQ]
- systemfonts [E:/R-4.5.2/library/systemfonts]
- ragg        [E:/R-4.5.2/library/ragg]
- evaluate    [E:/R-4.5.2/library/evaluate]
- pillar      [E:/R-4.5.2/library/pillar]
- rlang       [E:/R-4.5.2/library/rlang]
- jsonlite    [E:/R-4.5.2/library/jsonlite]
- crayon      [E:/R-4.5.2/library/crayon]
- IRkernel    [E:/R-4.5.2/library/IRkernel]
- uuid        [E:/R-4.5.2/library/uuid]
These packages are not manage

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31 ucrt)

Installing package(s) 'BiocVersion', 'limma', 'edgeR'

also installing the dependencies 'statmod', 'locfit'




package 'statmod' successfully unpacked and MD5 sums checked
package 'locfit' successfully unpacked and MD5 sums checked
package 'BiocVersion' successfully unpacked and MD5 sums checked
package 'limma' successfully unpacked and MD5 sums checked
package 'edgeR' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\ACT\AppData\Local\Temp\RtmpgB17LG\downloaded_packages
limma version: 3.66.0
edgeR version: 4.8.2
Required packages: PASS
Loading canonical data...
Loaded: 23740 genes x 755 samples
Design: 20 columns; full rank
Running TMM normalization...
Running initial voom...
Initial voom: PASS
Paired donors: 301; paired samples: 602
Estimating donor correlation from 500 representative genes...
Consensus donor correlation: 0.188518
Correlation estimation time: 311.2 sec
Running final voom on all 23,740 genes...
Final voom: PASS
Fitting 23,740 genes...
Full-gene fit: PASS
Extracting results...

STEP 03A — FAST DONOR-BLOCKED DE
Correlation genes:        

In [3]:
# ============================================================
# STEP 04A — CELL 1
# DONOR-GROUPED STRATIFIED OUTER CV SPLITS
# 5 FOLDS x 5 REPEATS
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

if (!file.exists(META_PATH)) {
  stop("Model metadata file missing.")
}

MODEL_META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(MODEL_META) == 755L,
  length(unique(MODEL_META$BrNum)) == 454L
)

# ------------------------------------------------------------
# 1. Build one row per donor
# ------------------------------------------------------------

DONOR_IDS <- unique(
  MODEL_META$BrNum
)

DONOR_TABLE <- do.call(
  rbind,
  lapply(
    DONOR_IDS,
    function(id) {

      z <- MODEL_META[
        MODEL_META$BrNum == id,
        ,
        drop = FALSE
      ]

      has_dlpfc <- any(
        z$Region == "DLPFC"
      )

      has_hippo <- any(
        z$Region == "HIPPO"
      )

      availability <- if (
        has_dlpfc && has_hippo
      ) {
        "DLPFC_HIPPO"
      } else if (has_dlpfc) {
        "DLPFC_only"
      } else if (has_hippo) {
        "HIPPO_only"
      } else {
        stop(
          paste(
            "Invalid donor region structure:",
            id
          )
        )
      }

      dx_values <- unique(
        z$Dx
      )

      if (length(dx_values) != 1L) {
        stop(
          paste(
            "Diagnosis inconsistent within donor:",
            id
          )
        )
      }

      data.frame(
        BrNum = id,
        Dx = dx_values[1],
        region_availability = availability,
        n_samples = nrow(z),
        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(DONOR_TABLE) <- NULL

DONOR_TABLE$stratum <- paste(
  DONOR_TABLE$Dx,
  DONOR_TABLE$region_availability,
  sep = "__"
)

# ------------------------------------------------------------
# 2. Basic donor structure audit
# ------------------------------------------------------------

cat("\n--- DONOR STRATA ---\n")

print(
  as.data.frame(
    table(
      DONOR_TABLE$Dx,
      DONOR_TABLE$region_availability
    )
  ),
  row.names = FALSE
)

# ------------------------------------------------------------
# 3. Function: stratified grouped fold assignment
# ------------------------------------------------------------

make_outer_folds <- function(
  donor_table,
  repeat_id,
  k = 5L
) {

  set.seed(
    20260930 + repeat_id
  )

  donor_table$outer_fold <- NA_integer_

  strata <- unique(
    donor_table$stratum
  )

  for (s in strata) {

    idx <- which(
      donor_table$stratum == s
    )

    idx <- sample(
      idx,
      length(idx),
      replace = FALSE
    )

    fold_sequence <- rep(
      seq_len(k),
      length.out = length(idx)
    )

    fold_sequence <- sample(
      fold_sequence,
      length(fold_sequence),
      replace = FALSE
    )

    donor_table$outer_fold[idx] <- fold_sequence
  }

  donor_table$repeat_id <- repeat_id

  donor_table
}

# ------------------------------------------------------------
# 4. Generate 5 repeats x 5 folds
# ------------------------------------------------------------

OUTER_SPLITS <- do.call(
  rbind,
  lapply(
    1:5,
    function(r) {

      make_outer_folds(
        DONOR_TABLE,
        repeat_id = r,
        k = 5L
      )
    }
  )
)

rownames(OUTER_SPLITS) <- NULL

# ------------------------------------------------------------
# 5. Hard donor leakage checks
# ------------------------------------------------------------

for (r in 1:5) {

  tmp <- OUTER_SPLITS[
    OUTER_SPLITS$repeat_id == r,
    ,
    drop = FALSE
  ]

  if (
    nrow(tmp) != 454L ||
    length(unique(tmp$BrNum)) != 454L
  ) {
    stop(
      paste(
        "Repeat",
        r,
        "does not contain exactly 454 unique donors."
      )
    )
  }

  if (
    anyNA(tmp$outer_fold) ||
    !setequal(
      unique(tmp$outer_fold),
      1:5
    )
  ) {
    stop(
      paste(
        "Invalid fold assignment in repeat",
        r
      )
    )
  }
}

# ------------------------------------------------------------
# 6. Expand donor folds to sample level
# ------------------------------------------------------------

SAMPLE_SPLITS <- merge(
  MODEL_META[
    ,
    c(
      "discovery_row_index",
      "sample_index_in_rse",
      "RNum",
      "BrNum",
      "Region",
      "Dx"
    )
  ],
  OUTER_SPLITS[
    ,
    c(
      "BrNum",
      "repeat_id",
      "outer_fold"
    )
  ],
  by = "BrNum",
  all.x = TRUE,
  sort = FALSE
)

SAMPLE_SPLITS <- SAMPLE_SPLITS[
  order(
    SAMPLE_SPLITS$repeat_id,
    SAMPLE_SPLITS$outer_fold,
    SAMPLE_SPLITS$discovery_row_index
  ),
  ,
  drop = FALSE
]

rownames(SAMPLE_SPLITS) <- NULL

stopifnot(
  nrow(SAMPLE_SPLITS) == 755L * 5L,
  !anyNA(SAMPLE_SPLITS$outer_fold)
)

# ------------------------------------------------------------
# 7. Fold balance summary
# ------------------------------------------------------------

FOLD_SUMMARY <- do.call(
  rbind,
  lapply(
    split(
      OUTER_SPLITS,
      interaction(
        OUTER_SPLITS$repeat_id,
        OUTER_SPLITS$outer_fold,
        drop = TRUE
      )
    ),
    function(z) {

      donor_ids <- z$BrNum

      samples <- MODEL_META[
        MODEL_META$BrNum %in% donor_ids,
        ,
        drop = FALSE
      ]

      data.frame(
        repeat_id = z$repeat_id[1],
        outer_fold = z$outer_fold[1],

        n_donors = nrow(z),

        control_donors = sum(
          z$Dx == "Control"
        ),

        schizo_donors = sum(
          z$Dx == "Schizo"
        ),

        paired_donors = sum(
          z$region_availability ==
            "DLPFC_HIPPO"
        ),

        dlpfc_only_donors = sum(
          z$region_availability ==
            "DLPFC_only"
        ),

        hippo_only_donors = sum(
          z$region_availability ==
            "HIPPO_only"
        ),

        n_samples = nrow(samples),

        DLPFC_Control = sum(
          samples$Region == "DLPFC" &
          samples$Dx == "Control"
        ),

        DLPFC_Schizo = sum(
          samples$Region == "DLPFC" &
          samples$Dx == "Schizo"
        ),

        HIPPO_Control = sum(
          samples$Region == "HIPPO" &
          samples$Dx == "Control"
        ),

        HIPPO_Schizo = sum(
          samples$Region == "HIPPO" &
          samples$Dx == "Schizo"
        ),

        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(FOLD_SUMMARY) <- NULL

FOLD_SUMMARY <- FOLD_SUMMARY[
  order(
    FOLD_SUMMARY$repeat_id,
    FOLD_SUMMARY$outer_fold
  ),
  ,
  drop = FALSE
]

# ------------------------------------------------------------
# 8. Verify train/test donor separation for every split
# ------------------------------------------------------------

LEAKAGE_AUDIT <- data.frame()

for (r in 1:5) {

  tmp <- OUTER_SPLITS[
    OUTER_SPLITS$repeat_id == r,
    ,
    drop = FALSE
  ]

  for (f in 1:5) {

    test_donors <- tmp$BrNum[
      tmp$outer_fold == f
    ]

    train_donors <- tmp$BrNum[
      tmp$outer_fold != f
    ]

    overlap <- intersect(
      train_donors,
      test_donors
    )

    LEAKAGE_AUDIT <- rbind(
      LEAKAGE_AUDIT,
      data.frame(
        repeat_id = r,
        outer_fold = f,
        train_donors = length(train_donors),
        test_donors = length(test_donors),
        donor_overlap = length(overlap),
        stringsAsFactors = FALSE
      )
    )
  }
}

if (
  any(
    LEAKAGE_AUDIT$donor_overlap != 0L
  )
) {
  stop(
    "DONOR LEAKAGE DETECTED."
  )
}

# ------------------------------------------------------------
# 9. Save split artifacts
# ------------------------------------------------------------

SPLIT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "cv_splits"
)

dir.create(
  SPLIT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

DONOR_SPLIT_PATH <- file.path(
  SPLIT_DIR,
  "outer_cv_donor_splits_5fold_5repeat_v1.csv"
)

SAMPLE_SPLIT_PATH <- file.path(
  SPLIT_DIR,
  "outer_cv_sample_map_5fold_5repeat_v1.csv"
)

SUMMARY_PATH <- file.path(
  SPLIT_DIR,
  "outer_cv_fold_balance_summary_v1.csv"
)

LEAKAGE_PATH <- file.path(
  SPLIT_DIR,
  "outer_cv_leakage_audit_v1.csv"
)

write.csv(
  OUTER_SPLITS,
  DONOR_SPLIT_PATH,
  row.names = FALSE
)

write.csv(
  SAMPLE_SPLITS,
  SAMPLE_SPLIT_PATH,
  row.names = FALSE
)

write.csv(
  FOLD_SUMMARY,
  SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  LEAKAGE_AUDIT,
  LEAKAGE_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 10. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 04A — OUTER CV SPLITS\n")
cat("============================================================\n")

cat("Unique donors:      ", nrow(DONOR_TABLE), "\n", sep = "")
cat("Outer repeats:      5\n")
cat("Folds per repeat:   5\n")
cat("Total outer tests:  25\n")
cat("Donor leakage:      ZERO\n")

cat("\n--- FOLD BALANCE ---\n")

print(
  FOLD_SUMMARY,
  row.names = FALSE
)

cat("\n--- LEAKAGE AUDIT ---\n")

print(
  LEAKAGE_AUDIT,
  row.names = FALSE
)

cat("\nSaved:\n")
cat(DONOR_SPLIT_PATH, "\n")
cat(SAMPLE_SPLIT_PATH, "\n")

cat("\n============================================================\n")
cat("STEP 04A — CELL 1 PASS\n")
cat("============================================================\n")


--- DONOR STRATA ---
    Var1        Var2 Freq
 Control DLPFC_HIPPO  192
  Schizo DLPFC_HIPPO  109
 Control  DLPFC_only   34
  Schizo  DLPFC_only   44
 Control  HIPPO_only   51
  Schizo  HIPPO_only   24

STEP 04A — OUTER CV SPLITS
Unique donors:      454
Outer repeats:      5
Folds per repeat:   5
Total outer tests:  25
Donor leakage:      ZERO

--- FOLD BALANCE ---
 repeat_id outer_fold n_donors control_donors schizo_donors paired_donors
         1          1       93             57            36            61
         1          2       92             56            36            61
         1          3       91             55            36            60
         1          4       91             55            36            60
         1          5       87             54            33            59
         2          1       93             57            36            61
         2          2       92             56            36            61
         2          3       91        

In [4]:
# ============================================================
# STEP 04B — CELL 1
# ONE-OUTER-FOLD SHARED/DEVIATION ELASTIC-NET BENCHMARK
#
# Outer split: Repeat 1 / Fold 1
# Inner CV:    4-fold donor-grouped
# Genes:       ALL 23,740 primary genes
# alpha:       0.50
# deviation penalty multiplier: 4
#
# Purpose:
#   Benchmark runtime + verify full modeling pipeline before
#   launching the complete nested CV.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"

TOTAL_START <- proc.time()[3]

# ------------------------------------------------------------
# 1. Package
# ------------------------------------------------------------

if (!requireNamespace("glmnet", quietly = TRUE)) {

  cat("Installing glmnet...\n")
  flush.console()

  install.packages(
    "glmnet",
    repos = "https://cloud.r-project.org"
  )
}

if (!requireNamespace("glmnet", quietly = TRUE)) {
  stop("STEP 04B FAIL: glmnet unavailable.")
}

cat(
  "glmnet version: ",
  as.character(packageVersion("glmnet")),
  "\n",
  sep = ""
)

flush.console()

# ------------------------------------------------------------
# 2. Canonical files
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_gene_annotation_v1.csv"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

SPLIT_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "cv_splits",
  "outer_cv_donor_splits_5fold_5repeat_v1.csv"
)

required_files <- c(
  COUNTS_PATH,
  ANNOTATION_PATH,
  META_PATH,
  SPLIT_PATH
)

if (!all(file.exists(required_files))) {
  stop("STEP 04B FAIL: One or more canonical inputs are missing.")
}

# ------------------------------------------------------------
# 3. Load
# ------------------------------------------------------------

cat("Loading data...\n")
flush.console()

COUNTS <- readRDS(
  COUNTS_PATH
)

ANNOTATION <- read.csv(
  ANNOTATION_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

OUTER_SPLITS <- read.csv(
  SPLIT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(COUNTS) == 23740L,
  ncol(COUNTS) == 755L,
  nrow(META) == 755L,
  nrow(ANNOTATION) == 23740L,
  identical(
    rownames(COUNTS),
    ANNOTATION$ensemblID_stable
  )
)

cat(
  "Loaded: ",
  nrow(COUNTS),
  " genes x ",
  ncol(COUNTS),
  " samples\n",
  sep = ""
)

flush.console()

# ------------------------------------------------------------
# 4. Expression transform
#
# Per-sample log2(CPM + 1).
# No cohort-wide TMM reference is used here, avoiding test-set
# information entering normalization.
# ------------------------------------------------------------

cat("Creating leakage-safe log2(CPM + 1)...\n")
flush.console()

LIB_SIZE <- colSums(
  COUNTS
)

if (
  any(!is.finite(LIB_SIZE)) ||
  any(LIB_SIZE <= 0)
) {
  stop("Invalid library size.")
}

LOGCPM <- log2(
  sweep(
    COUNTS,
    2,
    LIB_SIZE / 1e6,
    "/"
  ) + 1
)

rm(COUNTS)
gc(verbose = FALSE)

if (
  anyNA(LOGCPM) ||
  any(!is.finite(LOGCPM))
) {
  stop("Non-finite logCPM values detected.")
}

# ------------------------------------------------------------
# 5. Outer Repeat 1 / Fold 1
# ------------------------------------------------------------

OUTER_REPEAT <- 1L
OUTER_FOLD   <- 1L

SPLIT_R1 <- OUTER_SPLITS[
  OUTER_SPLITS$repeat_id == OUTER_REPEAT,
  ,
  drop = FALSE
]

TEST_DONORS <- SPLIT_R1$BrNum[
  SPLIT_R1$outer_fold == OUTER_FOLD
]

TRAIN_DONORS <- SPLIT_R1$BrNum[
  SPLIT_R1$outer_fold != OUTER_FOLD
]

if (
  length(
    intersect(
      TRAIN_DONORS,
      TEST_DONORS
    )
  ) != 0L
) {
  stop("OUTER DONOR LEAKAGE.")
}

TRAIN_IDX <- which(
  META$BrNum %in% TRAIN_DONORS
)

TEST_IDX <- which(
  META$BrNum %in% TEST_DONORS
)

cat(
  "Outer training: ",
  length(TRAIN_DONORS),
  " donors / ",
  length(TRAIN_IDX),
  " samples\n",
  sep = ""
)

cat(
  "Outer test:     ",
  length(TEST_DONORS),
  " donors / ",
  length(TEST_IDX),
  " samples\n",
  sep = ""
)

flush.console()

# ------------------------------------------------------------
# 6. Response
# ------------------------------------------------------------

Y <- ifelse(
  META$Dx == "Schizo",
  1,
  0
)

# ------------------------------------------------------------
# 7. Region sign
#
# +1 = DLPFC
# -1 = HIPPO
# ------------------------------------------------------------

REGION_SIGN <- ifelse(
  META$Region == "DLPFC",
  1,
  -1
)

# ------------------------------------------------------------
# 8. Nuisance covariate matrix
#
# Covariates enter glmnet unpenalized.
# ------------------------------------------------------------

META$Region <- factor(
  META$Region,
  levels = c(
    "DLPFC",
    "HIPPO"
  )
)

META$Sex <- factor(
  META$Sex
)

META$Race <- factor(
  META$Race
)

COVARIATE_MATRIX <- model.matrix(
  ~ Region +
    donor_age +
    Sex +
    Race +
    RIN +
    snpPC1 +
    snpPC2 +
    snpPC3 +
    snpPC4 +
    snpPC5 +
    snpPC6 +
    snpPC7 +
    snpPC8 +
    snpPC9 +
    snpPC10,
  data = META
)

# Remove intercept because glmnet supplies its own intercept.
COVARIATE_MATRIX <- COVARIATE_MATRIX[
  ,
  colnames(COVARIATE_MATRIX) != "(Intercept)",
  drop = FALSE
]

# Remove any covariate column constant in outer training.
COV_TRAIN_SD <- apply(
  COVARIATE_MATRIX[
    TRAIN_IDX,
    ,
    drop = FALSE
  ],
  2,
  sd
)

COV_KEEP <- (
  is.finite(COV_TRAIN_SD) &
  COV_TRAIN_SD > 0
)

COVARIATE_MATRIX <- COVARIATE_MATRIX[
  ,
  COV_KEEP,
  drop = FALSE
]

cat(
  "Unpenalized covariates: ",
  ncol(COVARIATE_MATRIX),
  "\n",
  sep = ""
)

flush.console()

# ------------------------------------------------------------
# 9. Expression predictors
#
# Samples x genes
# ------------------------------------------------------------

X_EXPR <- t(
  LOGCPM
)

rm(LOGCPM)
gc(verbose = FALSE)

stopifnot(
  nrow(X_EXPR) == 755L,
  ncol(X_EXPR) == 23740L
)

# ------------------------------------------------------------
# 10. Shared + deviation feature architecture
#
# beta_DLPFC = beta_shared + beta_deviation
# beta_HIPPO = beta_shared - beta_deviation
# ------------------------------------------------------------

cat("Constructing shared/deviation feature matrix...\n")
flush.console()

X_SHARED <- X_EXPR

X_DEVIATION <- X_EXPR *
  REGION_SIGN

colnames(X_SHARED) <- paste0(
  "SHARED__",
  colnames(X_EXPR)
)

colnames(X_DEVIATION) <- paste0(
  "DEV__",
  colnames(X_EXPR)
)

X_MODEL <- cbind(
  COVARIATE_MATRIX,
  X_SHARED,
  X_DEVIATION
)

rm(
  X_EXPR,
  X_SHARED,
  X_DEVIATION,
  COVARIATE_MATRIX
)

gc(verbose = FALSE)

N_COV <- ncol(X_MODEL) -
  2L * 23740L

N_GENES <- 23740L

cat(
  "Total predictors: ",
  ncol(X_MODEL),
  "\n",
  sep = ""
)

cat(
  "  Unpenalized covariates: ",
  N_COV,
  "\n",
  sep = ""
)

cat(
  "  Shared genes:           ",
  N_GENES,
  "\n",
  sep = ""
)

cat(
  "  Deviation genes:        ",
  N_GENES,
  "\n",
  sep = ""
)

flush.console()

# ------------------------------------------------------------
# 11. Penalty factors
#
# covariates = 0
# shared     = 1
# deviation  = 4
# ------------------------------------------------------------

ALPHA_BENCHMARK <- 0.50
DEVIATION_MULTIPLIER <- 4

PENALTY_FACTOR <- c(
  rep(
    0,
    N_COV
  ),
  rep(
    1,
    N_GENES
  ),
  rep(
    DEVIATION_MULTIPLIER,
    N_GENES
  )
)

stopifnot(
  length(PENALTY_FACTOR) ==
    ncol(X_MODEL)
)

# ------------------------------------------------------------
# 12. Donor-balanced observation weights
# ------------------------------------------------------------

OBS_WEIGHT <- as.numeric(
  META$donor_balance_weight
)

if (
  any(!is.finite(OBS_WEIGHT)) ||
  any(OBS_WEIGHT <= 0)
) {
  stop("Invalid donor-balanced weights.")
}

# ------------------------------------------------------------
# 13. Build 4-fold INNER grouped CV
# ------------------------------------------------------------

TRAIN_META <- META[
  TRAIN_IDX,
  ,
  drop = FALSE
]

INNER_DONORS <- unique(
  TRAIN_META$BrNum
)

INNER_DONOR_TABLE <- do.call(
  rbind,
  lapply(
    INNER_DONORS,
    function(id) {

      z <- TRAIN_META[
        TRAIN_META$BrNum == id,
        ,
        drop = FALSE
      ]

      has_dlpfc <- any(
        z$Region == "DLPFC"
      )

      has_hippo <- any(
        z$Region == "HIPPO"
      )

      availability <- if (
        has_dlpfc && has_hippo
      ) {
        "DLPFC_HIPPO"
      } else if (has_dlpfc) {
        "DLPFC_only"
      } else {
        "HIPPO_only"
      }

      data.frame(
        BrNum = id,
        Dx = as.character(z$Dx[1]),
        region_availability = availability,
        stringsAsFactors = FALSE
      )
    }
  )
)

INNER_DONOR_TABLE$stratum <- paste(
  INNER_DONOR_TABLE$Dx,
  INNER_DONOR_TABLE$region_availability,
  sep = "__"
)

set.seed(2026093001)

INNER_DONOR_TABLE$inner_fold <- NA_integer_

for (
  s in unique(
    INNER_DONOR_TABLE$stratum
  )
) {

  idx <- which(
    INNER_DONOR_TABLE$stratum == s
  )

  idx <- sample(
    idx,
    length(idx)
  )

  fold_ids <- rep(
    1:4,
    length.out = length(idx)
  )

  fold_ids <- sample(
    fold_ids,
    length(fold_ids)
  )

  INNER_DONOR_TABLE$inner_fold[idx] <-
    fold_ids
}

INNER_FOLD_LOOKUP <- setNames(
  INNER_DONOR_TABLE$inner_fold,
  INNER_DONOR_TABLE$BrNum
)

INNER_FOLD_ID <- as.integer(
  INNER_FOLD_LOOKUP[
    TRAIN_META$BrNum
  ]
)

if (
  anyNA(INNER_FOLD_ID) ||
  !setequal(
    unique(INNER_FOLD_ID),
    1:4
  )
) {
  stop("Invalid inner folds.")
}

# ------------------------------------------------------------
# 14. Inner leakage audit
# ------------------------------------------------------------

for (f in 1:4) {

  val_donors <- unique(
    TRAIN_META$BrNum[
      INNER_FOLD_ID == f
    ]
  )

  fit_donors <- unique(
    TRAIN_META$BrNum[
      INNER_FOLD_ID != f
    ]
  )

  if (
    length(
      intersect(
        val_donors,
        fit_donors
      )
    ) != 0L
  ) {
    stop(
      paste(
        "INNER DONOR LEAKAGE in fold",
        f
      )
    )
  }
}

cat(
  "Inner donor-grouped 4-fold CV: PASS\n"
)

flush.console()

# ------------------------------------------------------------
# 15. Benchmark cv.glmnet
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STARTING cv.glmnet BENCHMARK\n")
cat("alpha = 0.50\n")
cat("deviation multiplier = 4\n")
cat("lambda path = 50\n")
cat("============================================================\n")

flush.console()

CV_START <- proc.time()[3]

CVFIT <- glmnet::cv.glmnet(

  x = X_MODEL[
    TRAIN_IDX,
    ,
    drop = FALSE
  ],

  y = Y[
    TRAIN_IDX
  ],

  family = "binomial",

  weights = OBS_WEIGHT[
    TRAIN_IDX
  ],

  foldid = INNER_FOLD_ID,

  type.measure = "auc",

  alpha = ALPHA_BENCHMARK,

  penalty.factor = PENALTY_FACTOR,

  standardize = TRUE,

  intercept = TRUE,

  nlambda = 50,

  keep = FALSE,

  parallel = FALSE
)

CV_SECONDS <- proc.time()[3] -
  CV_START

cat(
  "cv.glmnet finished in ",
  round(CV_SECONDS, 1),
  " seconds\n",
  sep = ""
)

flush.console()

# ------------------------------------------------------------
# 16. Best inner-CV lambda
# ------------------------------------------------------------

VALID_CV <- is.finite(
  CVFIT$cvm
)

if (!any(VALID_CV)) {
  stop(
    "No valid inner-CV AUC values."
  )
}

BEST_INDEX <- which.max(
  ifelse(
    VALID_CV,
    CVFIT$cvm,
    -Inf
  )
)

BEST_LAMBDA <- CVFIT$lambda[
  BEST_INDEX
]

BEST_INNER_AUC <- CVFIT$cvm[
  BEST_INDEX
]

BEST_INNER_AUC_SD <- CVFIT$cvsd[
  BEST_INDEX
]

cat(
  "Best inner AUC: ",
  round(BEST_INNER_AUC, 4),
  "\n",
  sep = ""
)

cat(
  "Best lambda:    ",
  signif(BEST_LAMBDA, 6),
  "\n",
  sep = ""
)

# ------------------------------------------------------------
# 17. Final outer-training fit
# ------------------------------------------------------------

cat("Fitting final outer-training model...\n")
flush.console()

FINAL_START <- proc.time()[3]

FINAL_FIT <- glmnet::glmnet(

  x = X_MODEL[
    TRAIN_IDX,
    ,
    drop = FALSE
  ],

  y = Y[
    TRAIN_IDX
  ],

  family = "binomial",

  weights = OBS_WEIGHT[
    TRAIN_IDX
  ],

  alpha = ALPHA_BENCHMARK,

  penalty.factor = PENALTY_FACTOR,

  standardize = TRUE,

  intercept = TRUE,

  lambda = BEST_LAMBDA
)

FINAL_SECONDS <- proc.time()[3] -
  FINAL_START

# ------------------------------------------------------------
# 18. Outer-test predictions
# ------------------------------------------------------------

TEST_PRED <- as.numeric(
  predict(
    FINAL_FIT,
    newx = X_MODEL[
      TEST_IDX,
      ,
      drop = FALSE
    ],
    type = "response",
    s = BEST_LAMBDA
  )
)

if (
  anyNA(TEST_PRED) ||
  any(!is.finite(TEST_PRED))
) {
  stop("Invalid outer-test predictions.")
}

# ------------------------------------------------------------
# 19. AUROC function
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  y <- as.integer(y)

  ok <- (
    !is.na(y) &
    is.finite(score)
  )

  y <- y[ok]
  score <- score[ok]

  n1 <- sum(y == 1L)
  n0 <- sum(y == 0L)

  if (
    n1 == 0L ||
    n0 == 0L
  ) {
    return(NA_real_)
  }

  r <- rank(
    score,
    ties.method = "average"
  )

  (
    sum(
      r[y == 1L]
    ) -
      n1 * (n1 + 1) / 2
  ) / (
    n1 * n0
  )
}

# ------------------------------------------------------------
# 20. Sample-level outer AUC
# ------------------------------------------------------------

TEST_META <- META[
  TEST_IDX,
  ,
  drop = FALSE
]

TEST_META$prediction <- TEST_PRED

SAMPLE_AUC <- auc_rank(
  Y[
    TEST_IDX
  ],
  TEST_PRED
)

# ------------------------------------------------------------
# 21. Donor-level outer prediction
#
# For paired donors, average DLPFC + HIPPO probability.
# ------------------------------------------------------------

DONOR_PRED <- aggregate(
  prediction ~ BrNum,
  data = TEST_META,
  FUN = mean
)

DONOR_Y <- aggregate(
  Y[
    TEST_IDX
  ],
  by = list(
    BrNum = TEST_META$BrNum
  ),
  FUN = function(x) {

    if (
      length(unique(x)) != 1L
    ) {
      stop(
        "Diagnosis mismatch within donor."
      )
    }

    unique(x)
  }
)

names(DONOR_Y)[2] <- "y"

DONOR_RESULT <- merge(
  DONOR_Y,
  DONOR_PRED,
  by = "BrNum",
  sort = FALSE
)

DONOR_AUC <- auc_rank(
  DONOR_RESULT$y,
  DONOR_RESULT$prediction
)

# ------------------------------------------------------------
# 22. Region-specific outer AUC
# ------------------------------------------------------------

DLPFC_TEST <- TEST_META$Region == "DLPFC"

HIPPO_TEST <- TEST_META$Region == "HIPPO"

DLPFC_AUC <- auc_rank(
  Y[
    TEST_IDX
  ][DLPFC_TEST],
  TEST_PRED[
    DLPFC_TEST
  ]
)

HIPPO_AUC <- auc_rank(
  Y[
    TEST_IDX
  ][HIPPO_TEST],
  TEST_PRED[
    HIPPO_TEST
  ]
)

# ------------------------------------------------------------
# 23. Extract coefficients
# ------------------------------------------------------------

COEF_MATRIX <- as.matrix(
  coef(
    FINAL_FIT,
    s = BEST_LAMBDA
  )
)

COEF_VECTOR <- COEF_MATRIX[
  ,
  1
]

names(COEF_VECTOR) <- rownames(
  COEF_MATRIX
)

MODEL_FEATURE_NAMES <- colnames(
  X_MODEL
)

FEATURE_COEF <- COEF_VECTOR[
  MODEL_FEATURE_NAMES
]

COV_COEF <- if (
  N_COV > 0
) {
  FEATURE_COEF[
    seq_len(N_COV)
  ]
} else {
  numeric(0)
}

SHARED_COEF <- FEATURE_COEF[
  N_COV +
    seq_len(N_GENES)
]

DEV_COEF <- FEATURE_COEF[
  N_COV +
    N_GENES +
    seq_len(N_GENES)
]

N_SHARED_NONZERO <- sum(
  abs(SHARED_COEF) > 0
)

N_DEV_NONZERO <- sum(
  abs(DEV_COEF) > 0
)

# ------------------------------------------------------------
# 24. Gene coefficient table
# ------------------------------------------------------------

GENE_COEFFICIENTS <- data.frame(

  ensemblID_stable =
    ANNOTATION$ensemblID_stable,

  gene_symbol =
    ANNOTATION$gene_symbol,

  gene_type =
    ANNOTATION$gene_type,

  beta_shared =
    as.numeric(SHARED_COEF),

  beta_deviation =
    as.numeric(DEV_COEF),

  beta_DLPFC =
    as.numeric(
      SHARED_COEF +
      DEV_COEF
    ),

  beta_HIPPO =
    as.numeric(
      SHARED_COEF -
      DEV_COEF
    ),

  stringsAsFactors = FALSE
)

GENE_COEFFICIENTS$selected_shared <-
  abs(
    GENE_COEFFICIENTS$beta_shared
  ) > 0

GENE_COEFFICIENTS$selected_deviation <-
  abs(
    GENE_COEFFICIENTS$beta_deviation
  ) > 0

# ------------------------------------------------------------
# 25. Save benchmark
# ------------------------------------------------------------

BENCH_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "benchmark_outer_r1f1"
)

dir.create(
  BENCH_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

BENCH_SUMMARY_PATH <- file.path(
  BENCH_DIR,
  "step04b_benchmark_summary_v1.csv"
)

GENE_COEF_PATH <- file.path(
  BENCH_DIR,
  "step04b_gene_coefficients_v1.csv"
)

DONOR_PRED_PATH <- file.path(
  BENCH_DIR,
  "step04b_outer_test_donor_predictions_v1.csv"
)

SAMPLE_PRED_PATH <- file.path(
  BENCH_DIR,
  "step04b_outer_test_sample_predictions_v1.csv"
)

INNER_CURVE_PATH <- file.path(
  BENCH_DIR,
  "step04b_inner_cv_lambda_curve_v1.csv"
)

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START

BENCHMARK_SUMMARY <- data.frame(

  outer_repeat = OUTER_REPEAT,

  outer_fold = OUTER_FOLD,

  train_donors =
    length(TRAIN_DONORS),

  test_donors =
    length(TEST_DONORS),

  train_samples =
    length(TRAIN_IDX),

  test_samples =
    length(TEST_IDX),

  genes =
    N_GENES,

  total_predictors =
    ncol(X_MODEL),

  alpha =
    ALPHA_BENCHMARK,

  deviation_penalty_multiplier =
    DEVIATION_MULTIPLIER,

  best_lambda =
    BEST_LAMBDA,

  best_inner_auc =
    BEST_INNER_AUC,

  best_inner_auc_sd =
    BEST_INNER_AUC_SD,

  outer_sample_auc =
    SAMPLE_AUC,

  outer_donor_auc =
    DONOR_AUC,

  outer_DLPFC_auc =
    DLPFC_AUC,

  outer_HIPPO_auc =
    HIPPO_AUC,

  n_shared_nonzero =
    N_SHARED_NONZERO,

  n_deviation_nonzero =
    N_DEV_NONZERO,

  cv_seconds =
    CV_SECONDS,

  final_fit_seconds =
    FINAL_SECONDS,

  total_seconds =
    TOTAL_SECONDS,

  stringsAsFactors = FALSE
)

INNER_CURVE <- data.frame(
  lambda = CVFIT$lambda,
  mean_inner_auc = CVFIT$cvm,
  se_inner_auc = CVFIT$cvsd,
  stringsAsFactors = FALSE
)

write.csv(
  BENCHMARK_SUMMARY,
  BENCH_SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  GENE_COEFFICIENTS,
  GENE_COEF_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  DONOR_RESULT,
  DONOR_PRED_PATH,
  row.names = FALSE
)

TEST_META$y <- Y[
  TEST_IDX
]

write.csv(
  TEST_META,
  SAMPLE_PRED_PATH,
  row.names = FALSE
)

write.csv(
  INNER_CURVE,
  INNER_CURVE_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 26. Final output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 04B — ONE-FOLD MODEL BENCHMARK\n")
cat("============================================================\n")

cat(
  "Outer split:                 Repeat 1 / Fold 1\n"
)

cat(
  "Training donors/samples:     ",
  length(TRAIN_DONORS),
  " / ",
  length(TRAIN_IDX),
  "\n",
  sep = ""
)

cat(
  "Test donors/samples:         ",
  length(TEST_DONORS),
  " / ",
  length(TEST_IDX),
  "\n",
  sep = ""
)

cat(
  "Genes:                       ",
  N_GENES,
  "\n",
  sep = ""
)

cat(
  "Total predictors:            ",
  ncol(X_MODEL),
  "\n",
  sep = ""
)

cat(
  "Best inner-CV AUC:           ",
  round(BEST_INNER_AUC, 4),
  "\n",
  sep = ""
)

cat(
  "Best lambda:                 ",
  signif(BEST_LAMBDA, 6),
  "\n",
  sep = ""
)

cat(
  "Outer donor-level AUROC:     ",
  round(DONOR_AUC, 4),
  "\n",
  sep = ""
)

cat(
  "Outer sample-level AUROC:    ",
  round(SAMPLE_AUC, 4),
  "\n",
  sep = ""
)

cat(
  "Outer DLPFC AUROC:           ",
  round(DLPFC_AUC, 4),
  "\n",
  sep = ""
)

cat(
  "Outer HIPPO AUROC:           ",
  round(HIPPO_AUC, 4),
  "\n",
  sep = ""
)

cat(
  "Nonzero shared genes:        ",
  N_SHARED_NONZERO,
  "\n",
  sep = ""
)

cat(
  "Nonzero deviation genes:     ",
  N_DEV_NONZERO,
  "\n",
  sep = ""
)

cat(
  "Inner CV runtime:            ",
  round(CV_SECONDS, 1),
  " sec\n",
  sep = ""
)

cat(
  "Final fit runtime:           ",
  round(FINAL_SECONDS, 1),
  " sec\n",
  sep = ""
)

cat(
  "Total cell runtime:          ",
  round(TOTAL_SECONDS, 1),
  " sec\n",
  sep = ""
)

cat("\n--- BENCHMARK SUMMARY ---\n")

print(
  BENCHMARK_SUMMARY,
  row.names = FALSE
)

cat("\n============================================================\n")
cat("STEP 04B — CELL 1 PASS\n")
cat("============================================================\n")

Installing glmnet...
- Querying repositories for available source packages ... Done!
The following package(s) will be installed:
- foreach   [1.5.2]
- glmnet    [5.1]
- iterators [1.0.14]
- Rcpp      [1.1.2]
- RcppEigen [0.3.4.0.2]
- shape     [1.4.6.1]
These packages will be installed into "C:/Users/ACT/renv/library/windows/R-4.5/x86_64-w64-mingw32".

# Downloading packages -------------------------------------------------------
✔ foreach 1.5.2                            [150 kB in 0.85s]           
✔ iterators 1.0.14                         [354 kB in 1.0s]            
✔ RcppEigen 0.3.4.0.2                      [2.6 MB in 1.6s]            
✔ shape 1.4.6.1                            [755 kB in 2.0s]            
✔ glmnet 5.1                               [2.8 MB in 2.2s]            
✔ Rcpp 1.1.2                               [3.0 MB in 3.7s]            
Successfully downloaded 6 packages in 7.4 seconds.                              

# Installing packages ------------------------------

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2477974,132.4,4511394,241.0,4511394,241.0
Vcells,116799720,891.2,234659584,1790.4,234473948,1788.9


Outer training: 361 donors / 601 samples
Outer test:     93 donors / 154 samples
Unpenalized covariates: 17


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2484798,132.8,4511394,241.0,4511394,241.0
Vcells,116830798,891.4,234659584,1790.4,234473948,1788.9


Constructing shared/deviation feature matrix...


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2532381,135.3,4511394,241.0,4511394,241.0
Vcells,134968248,1029.8,234659584,1790.4,234473948,1788.9


Total predictors: 47497
  Unpenalized covariates: 17
  Shared genes:           23740
  Deviation genes:        23740
Inner donor-grouped 4-fold CV: PASS

STARTING cv.glmnet BENCHMARK
alpha = 0.50
deviation multiplier = 4
lambda path = 50
cv.glmnet finished in 14.8 seconds
Best inner AUC: 0.755
Best lambda:    0.00961345
Fitting final outer-training model...

STEP 04B — ONE-FOLD MODEL BENCHMARK
Outer split:                 Repeat 1 / Fold 1
Training donors/samples:     361 / 601
Test donors/samples:         93 / 154
Genes:                       23740
Total predictors:            47497
Best inner-CV AUC:           0.755
Best lambda:                 0.00961345
Outer donor-level AUROC:     0.7724
Outer sample-level AUROC:    0.7789
Outer DLPFC AUROC:           0.7686
Outer HIPPO AUROC:           0.7904
Nonzero shared genes:        484
Nonzero deviation genes:     0
Inner CV runtime:            14.8 sec
Final fit runtime:           8.6 sec
Total cell runtime:          65.2 sec

--- BENCHMAR

In [5]:
# ============================================================
# STEP 04C — CELL 1
# FULL 20-HYPERPARAMETER GRID ON OUTER REPEAT 1 / FOLD 1
#
# alpha grid:      0.10, 0.25, 0.50, 0.75, 1.00
# deviation grid:  1, 2, 4, 8
# inner CV:        4-fold donor-grouped
# genes:           23,740
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"

TOTAL_START <- proc.time()[3]

# ------------------------------------------------------------
# 1. glmnet
# ------------------------------------------------------------

if (!requireNamespace("glmnet", quietly = TRUE)) {
  stop("glmnet unavailable.")
}

# ------------------------------------------------------------
# 2. Recover benchmark objects if needed
# ------------------------------------------------------------

need_rebuild <- !all(
  c(
    "X_MODEL",
    "Y",
    "OBS_WEIGHT",
    "TRAIN_IDX",
    "TEST_IDX",
    "INNER_FOLD_ID",
    "META",
    "ANNOTATION",
    "N_COV",
    "N_GENES"
  ) %in% ls()
)

if (need_rebuild) {

  cat("Rebuilding modeling objects from disk...\n")
  flush.console()

  COUNTS_PATH <- file.path(
    PROJECT_ROOT,
    "04_processed",
    "brainseq_primary_counts_v1.rds"
  )

  ANNOTATION_PATH <- file.path(
    PROJECT_ROOT,
    "04_processed",
    "brainseq_primary_gene_annotation_v1.csv"
  )

  META_PATH <- file.path(
    PROJECT_ROOT,
    "04_processed",
    "brainseq_model_metadata_v1.csv"
  )

  SPLIT_PATH <- file.path(
    PROJECT_ROOT,
    "06_models",
    "cv_splits",
    "outer_cv_donor_splits_5fold_5repeat_v1.csv"
  )

  COUNTS <- readRDS(COUNTS_PATH)

  ANNOTATION <- read.csv(
    ANNOTATION_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
  )

  META <- read.csv(
    META_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
  )

  OUTER_SPLITS <- read.csv(
    SPLIT_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
  )

  # ----------------------------------------
  # leakage-safe log2(CPM + 1)
  # ----------------------------------------

  LIB_SIZE <- colSums(COUNTS)

  LOGCPM <- log2(
    sweep(
      COUNTS,
      2,
      LIB_SIZE / 1e6,
      "/"
    ) + 1
  )

  rm(COUNTS)

  # ----------------------------------------
  # Outer Repeat 1 / Fold 1
  # ----------------------------------------

  OUTER_REPEAT <- 1L
  OUTER_FOLD <- 1L

  SPLIT_R1 <- OUTER_SPLITS[
    OUTER_SPLITS$repeat_id == OUTER_REPEAT,
    ,
    drop = FALSE
  ]

  TEST_DONORS <- SPLIT_R1$BrNum[
    SPLIT_R1$outer_fold == OUTER_FOLD
  ]

  TRAIN_DONORS <- SPLIT_R1$BrNum[
    SPLIT_R1$outer_fold != OUTER_FOLD
  ]

  TRAIN_IDX <- which(
    META$BrNum %in% TRAIN_DONORS
  )

  TEST_IDX <- which(
    META$BrNum %in% TEST_DONORS
  )

  # ----------------------------------------
  # Outcome
  # ----------------------------------------

  Y <- ifelse(
    META$Dx == "Schizo",
    1,
    0
  )

  REGION_SIGN <- ifelse(
    META$Region == "DLPFC",
    1,
    -1
  )

  # ----------------------------------------
  # Covariates
  # ----------------------------------------

  META$Region <- factor(
    META$Region,
    levels = c("DLPFC", "HIPPO")
  )

  META$Sex <- factor(META$Sex)
  META$Race <- factor(META$Race)

  COVARIATE_MATRIX <- model.matrix(
    ~ Region +
      donor_age +
      Sex +
      Race +
      RIN +
      snpPC1 +
      snpPC2 +
      snpPC3 +
      snpPC4 +
      snpPC5 +
      snpPC6 +
      snpPC7 +
      snpPC8 +
      snpPC9 +
      snpPC10,
    data = META
  )

  COVARIATE_MATRIX <- COVARIATE_MATRIX[
    ,
    colnames(COVARIATE_MATRIX) != "(Intercept)",
    drop = FALSE
  ]

  COV_TRAIN_SD <- apply(
    COVARIATE_MATRIX[
      TRAIN_IDX,
      ,
      drop = FALSE
    ],
    2,
    sd
  )

  COV_KEEP <- (
    is.finite(COV_TRAIN_SD) &
    COV_TRAIN_SD > 0
  )

  COVARIATE_MATRIX <- COVARIATE_MATRIX[
    ,
    COV_KEEP,
    drop = FALSE
  ]

  # ----------------------------------------
  # Shared/deviation expression matrix
  # ----------------------------------------

  X_EXPR <- t(LOGCPM)

  rm(LOGCPM)

  X_SHARED <- X_EXPR

  X_DEVIATION <- X_EXPR *
    REGION_SIGN

  colnames(X_SHARED) <- paste0(
    "SHARED__",
    colnames(X_EXPR)
  )

  colnames(X_DEVIATION) <- paste0(
    "DEV__",
    colnames(X_EXPR)
  )

  X_MODEL <- cbind(
    COVARIATE_MATRIX,
    X_SHARED,
    X_DEVIATION
  )

  N_GENES <- ncol(X_EXPR)

  N_COV <- ncol(
    COVARIATE_MATRIX
  )

  rm(
    X_EXPR,
    X_SHARED,
    X_DEVIATION,
    COVARIATE_MATRIX
  )

  gc(verbose = FALSE)

  # ----------------------------------------
  # Weights
  # ----------------------------------------

  OBS_WEIGHT <- as.numeric(
    META$donor_balance_weight
  )

  # ----------------------------------------
  # Inner 4-fold grouped CV
  # ----------------------------------------

  TRAIN_META <- META[
    TRAIN_IDX,
    ,
    drop = FALSE
  ]

  INNER_DONORS <- unique(
    TRAIN_META$BrNum
  )

  INNER_DONOR_TABLE <- do.call(
    rbind,
    lapply(
      INNER_DONORS,
      function(id) {

        z <- TRAIN_META[
          TRAIN_META$BrNum == id,
          ,
          drop = FALSE
        ]

        has_dlpfc <- any(
          z$Region == "DLPFC"
        )

        has_hippo <- any(
          z$Region == "HIPPO"
        )

        availability <- if (
          has_dlpfc &&
          has_hippo
        ) {
          "DLPFC_HIPPO"
        } else if (has_dlpfc) {
          "DLPFC_only"
        } else {
          "HIPPO_only"
        }

        data.frame(
          BrNum = id,
          Dx = as.character(z$Dx[1]),
          region_availability = availability,
          stringsAsFactors = FALSE
        )
      }
    )
  )

  INNER_DONOR_TABLE$stratum <- paste(
    INNER_DONOR_TABLE$Dx,
    INNER_DONOR_TABLE$region_availability,
    sep = "__"
  )

  set.seed(2026093001)

  INNER_DONOR_TABLE$inner_fold <- NA_integer_

  for (
    s in unique(
      INNER_DONOR_TABLE$stratum
    )
  ) {

    idx <- which(
      INNER_DONOR_TABLE$stratum == s
    )

    idx <- sample(
      idx,
      length(idx)
    )

    fold_ids <- rep(
      1:4,
      length.out = length(idx)
    )

    fold_ids <- sample(
      fold_ids,
      length(fold_ids)
    )

    INNER_DONOR_TABLE$inner_fold[idx] <-
      fold_ids
  }

  fold_lookup <- setNames(
    INNER_DONOR_TABLE$inner_fold,
    INNER_DONOR_TABLE$BrNum
  )

  INNER_FOLD_ID <- as.integer(
    fold_lookup[
      TRAIN_META$BrNum
    ]
  )

  cat("Model objects rebuilt: PASS\n")
  flush.console()
}

# ------------------------------------------------------------
# 3. Hyperparameter grid
# ------------------------------------------------------------

ALPHA_GRID <- c(
  0.10,
  0.25,
  0.50,
  0.75,
  1.00
)

DEV_PENALTY_GRID <- c(
  1,
  2,
  4,
  8
)

GRID <- expand.grid(
  alpha = ALPHA_GRID,
  deviation_penalty = DEV_PENALTY_GRID,
  KEEP.OUT.ATTRS = FALSE,
  stringsAsFactors = FALSE
)

GRID <- GRID[
  order(
    GRID$deviation_penalty,
    GRID$alpha
  ),
  ,
  drop = FALSE
]

GRID$config_id <- seq_len(
  nrow(GRID)
)

# ------------------------------------------------------------
# 4. Result container
# ------------------------------------------------------------

GRID_RESULT <- data.frame(

  config_id = GRID$config_id,

  alpha = GRID$alpha,

  deviation_penalty =
    GRID$deviation_penalty,

  best_lambda = NA_real_,

  best_inner_auc = NA_real_,

  best_inner_auc_sd = NA_real_,

  n_lambda = NA_integer_,

  runtime_seconds = NA_real_,

  stringsAsFactors = FALSE
)

CV_OBJECTS <- vector(
  "list",
  nrow(GRID)
)

# ------------------------------------------------------------
# 5. Run all 20 configurations
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 04C — STARTING 20-CONFIGURATION GRID\n")
cat("============================================================\n")

flush.console()

for (
  g in seq_len(
    nrow(GRID)
  )
) {

  alpha_value <- GRID$alpha[g]

  dev_penalty <- GRID$deviation_penalty[g]

  penalty_factor <- c(

    rep(
      0,
      N_COV
    ),

    rep(
      1,
      N_GENES
    ),

    rep(
      dev_penalty,
      N_GENES
    )
  )

  cat(
    sprintf(
      "[%02d/20] alpha=%.2f  deviation=%d ... ",
      g,
      alpha_value,
      dev_penalty
    )
  )

  flush.console()

  start_time <- proc.time()[3]

  cvfit <- glmnet::cv.glmnet(

    x = X_MODEL[
      TRAIN_IDX,
      ,
      drop = FALSE
    ],

    y = Y[
      TRAIN_IDX
    ],

    family = "binomial",

    weights = OBS_WEIGHT[
      TRAIN_IDX
    ],

    foldid = INNER_FOLD_ID,

    type.measure = "auc",

    alpha = alpha_value,

    penalty.factor =
      penalty_factor,

    standardize = TRUE,

    intercept = TRUE,

    nlambda = 50,

    keep = FALSE,

    parallel = FALSE
  )

  elapsed <- proc.time()[3] -
    start_time

  valid_auc <- is.finite(
    cvfit$cvm
  )

  if (!any(valid_auc)) {

    stop(
      paste0(
        "No valid CV AUC for config ",
        g
      )
    )
  }

  best_index <- which.max(
    ifelse(
      valid_auc,
      cvfit$cvm,
      -Inf
    )
  )

  GRID_RESULT$best_lambda[g] <-
    cvfit$lambda[
      best_index
    ]

  GRID_RESULT$best_inner_auc[g] <-
    cvfit$cvm[
      best_index
    ]

  GRID_RESULT$best_inner_auc_sd[g] <-
    cvfit$cvsd[
      best_index
    ]

  GRID_RESULT$n_lambda[g] <-
    length(
      cvfit$lambda
    )

  GRID_RESULT$runtime_seconds[g] <-
    elapsed

  CV_OBJECTS[[g]] <- cvfit

  cat(
    sprintf(
      "AUC=%.4f  lambda=%.5g  time=%.1fs\n",
      GRID_RESULT$best_inner_auc[g],
      GRID_RESULT$best_lambda[g],
      elapsed
    )
  )

  flush.console()
}

# ------------------------------------------------------------
# 6. Select best configuration
#
# Primary: highest inner AUC
# Tie-break:
#   1. lower CV SE
#   2. higher deviation penalty
#   3. lower alpha
# ------------------------------------------------------------

ORDER_INDEX <- order(
  -GRID_RESULT$best_inner_auc,
  GRID_RESULT$best_inner_auc_sd,
  -GRID_RESULT$deviation_penalty,
  GRID_RESULT$alpha
)

GRID_RESULT$selection_rank <- match(
  seq_len(
    nrow(GRID_RESULT)
  ),
  ORDER_INDEX
)

BEST_ROW <- ORDER_INDEX[1]

BEST_ALPHA <-
  GRID_RESULT$alpha[
    BEST_ROW
  ]

BEST_DEV_PENALTY <-
  GRID_RESULT$deviation_penalty[
    BEST_ROW
  ]

BEST_LAMBDA <-
  GRID_RESULT$best_lambda[
    BEST_ROW
  ]

BEST_INNER_AUC <-
  GRID_RESULT$best_inner_auc[
    BEST_ROW
  ]

BEST_INNER_SD <-
  GRID_RESULT$best_inner_auc_sd[
    BEST_ROW
  ]

# ------------------------------------------------------------
# 7. Fit selected model on full outer training set
# ------------------------------------------------------------

BEST_PENALTY_FACTOR <- c(

  rep(
    0,
    N_COV
  ),

  rep(
    1,
    N_GENES
  ),

  rep(
    BEST_DEV_PENALTY,
    N_GENES
  )
)

cat("\nSelected configuration:\n")
cat("alpha:              ", BEST_ALPHA, "\n", sep = "")
cat("deviation penalty:  ", BEST_DEV_PENALTY, "\n", sep = "")
cat("lambda:             ", signif(BEST_LAMBDA, 7), "\n", sep = "")
cat("inner AUC:          ", round(BEST_INNER_AUC, 4), "\n", sep = "")

flush.console()

FINAL_START <- proc.time()[3]

FINAL_FIT <- glmnet::glmnet(

  x = X_MODEL[
    TRAIN_IDX,
    ,
    drop = FALSE
  ],

  y = Y[
    TRAIN_IDX
  ],

  family = "binomial",

  weights = OBS_WEIGHT[
    TRAIN_IDX
  ],

  alpha = BEST_ALPHA,

  penalty.factor =
    BEST_PENALTY_FACTOR,

  standardize = TRUE,

  intercept = TRUE,

  lambda = BEST_LAMBDA
)

FINAL_SECONDS <- proc.time()[3] -
  FINAL_START

# ------------------------------------------------------------
# 8. Outer predictions
# ------------------------------------------------------------

TEST_PRED <- as.numeric(
  predict(
    FINAL_FIT,
    newx = X_MODEL[
      TEST_IDX,
      ,
      drop = FALSE
    ],
    type = "response",
    s = BEST_LAMBDA
  )
)

# ------------------------------------------------------------
# 9. AUC helper
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  y <- as.integer(y)

  ok <- (
    !is.na(y) &
    is.finite(score)
  )

  y <- y[ok]
  score <- score[ok]

  n1 <- sum(
    y == 1L
  )

  n0 <- sum(
    y == 0L
  )

  if (
    n1 == 0L ||
    n0 == 0L
  ) {
    return(NA_real_)
  }

  r <- rank(
    score,
    ties.method = "average"
  )

  (
    sum(
      r[
        y == 1L
      ]
    ) -
      n1 *
      (n1 + 1) /
      2
  ) /
    (
      n1 *
      n0
    )
}

# ------------------------------------------------------------
# 10. Sample-level performance
# ------------------------------------------------------------

TEST_META <- META[
  TEST_IDX,
  ,
  drop = FALSE
]

TEST_META$y <- Y[
  TEST_IDX
]

TEST_META$prediction <- TEST_PRED

SAMPLE_AUC <- auc_rank(
  TEST_META$y,
  TEST_META$prediction
)

# ------------------------------------------------------------
# 11. Donor-level performance
# ------------------------------------------------------------

DONOR_PRED <- aggregate(
  prediction ~ BrNum,
  data = TEST_META,
  FUN = mean
)

DONOR_Y <- aggregate(
  y ~ BrNum,
  data = TEST_META,
  FUN = function(x) {

    if (
      length(
        unique(x)
      ) != 1L
    ) {
      stop(
        "Outcome mismatch within donor."
      )
    }

    unique(x)
  }
)

DONOR_RESULT <- merge(
  DONOR_Y,
  DONOR_PRED,
  by = "BrNum",
  sort = FALSE
)

DONOR_AUC <- auc_rank(
  DONOR_RESULT$y,
  DONOR_RESULT$prediction
)

# ------------------------------------------------------------
# 12. Region-specific performance
# ------------------------------------------------------------

DLPFC_IDX <- (
  TEST_META$Region ==
    "DLPFC"
)

HIPPO_IDX <- (
  TEST_META$Region ==
    "HIPPO"
)

DLPFC_AUC <- auc_rank(
  TEST_META$y[
    DLPFC_IDX
  ],
  TEST_META$prediction[
    DLPFC_IDX
  ]
)

HIPPO_AUC <- auc_rank(
  TEST_META$y[
    HIPPO_IDX
  ],
  TEST_META$prediction[
    HIPPO_IDX
  ]
)

# ------------------------------------------------------------
# 13. Coefficient decomposition
# ------------------------------------------------------------

COEF_MATRIX <- as.matrix(
  coef(
    FINAL_FIT,
    s = BEST_LAMBDA
  )
)

COEF_VECTOR <- COEF_MATRIX[
  ,
  1
]

names(COEF_VECTOR) <- rownames(
  COEF_MATRIX
)

FEATURE_COEF <- COEF_VECTOR[
  colnames(
    X_MODEL
  )
]

SHARED_START <- N_COV + 1L

SHARED_END <- N_COV +
  N_GENES

DEV_START <- SHARED_END + 1L

DEV_END <- SHARED_END +
  N_GENES

SHARED_COEF <- FEATURE_COEF[
  SHARED_START:
  SHARED_END
]

DEV_COEF <- FEATURE_COEF[
  DEV_START:
  DEV_END
]

N_SHARED_NONZERO <- sum(
  abs(
    SHARED_COEF
  ) > 0
)

N_DEV_NONZERO <- sum(
  abs(
    DEV_COEF
  ) > 0
)

# ------------------------------------------------------------
# 14. Gene coefficient table
# ------------------------------------------------------------

GENE_COEFFICIENTS <- data.frame(

  ensemblID_stable =
    ANNOTATION$ensemblID_stable,

  gene_symbol =
    ANNOTATION$gene_symbol,

  gene_type =
    ANNOTATION$gene_type,

  beta_shared =
    as.numeric(
      SHARED_COEF
    ),

  beta_deviation =
    as.numeric(
      DEV_COEF
    ),

  beta_DLPFC =
    as.numeric(
      SHARED_COEF +
      DEV_COEF
    ),

  beta_HIPPO =
    as.numeric(
      SHARED_COEF -
      DEV_COEF
    ),

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 15. Final summary
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START

FINAL_SUMMARY <- data.frame(

  outer_repeat = 1L,

  outer_fold = 1L,

  best_alpha =
    BEST_ALPHA,

  best_deviation_penalty =
    BEST_DEV_PENALTY,

  best_lambda =
    BEST_LAMBDA,

  best_inner_auc =
    BEST_INNER_AUC,

  best_inner_auc_sd =
    BEST_INNER_SD,

  outer_sample_auc =
    SAMPLE_AUC,

  outer_donor_auc =
    DONOR_AUC,

  outer_DLPFC_auc =
    DLPFC_AUC,

  outer_HIPPO_auc =
    HIPPO_AUC,

  n_shared_nonzero =
    N_SHARED_NONZERO,

  n_deviation_nonzero =
    N_DEV_NONZERO,

  final_fit_seconds =
    FINAL_SECONDS,

  total_grid_seconds =
    sum(
      GRID_RESULT$runtime_seconds
    ),

  total_cell_seconds =
    TOTAL_SECONDS,

  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 16. Save
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "grid_outer_r1f1"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

GRID_PATH <- file.path(
  OUT_DIR,
  "step04c_hyperparameter_grid_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step04c_selected_model_summary_v1.csv"
)

COEF_PATH <- file.path(
  OUT_DIR,
  "step04c_selected_model_gene_coefficients_v1.csv"
)

SAMPLE_PRED_PATH <- file.path(
  OUT_DIR,
  "step04c_outer_test_sample_predictions_v1.csv"
)

DONOR_PRED_PATH <- file.path(
  OUT_DIR,
  "step04c_outer_test_donor_predictions_v1.csv"
)

write.csv(
  GRID_RESULT[
    order(
      GRID_RESULT$selection_rank
    ),
  ],
  GRID_PATH,
  row.names = FALSE
)

write.csv(
  FINAL_SUMMARY,
  SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  GENE_COEFFICIENTS,
  COEF_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  TEST_META,
  SAMPLE_PRED_PATH,
  row.names = FALSE
)

write.csv(
  DONOR_RESULT,
  DONOR_PRED_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 17. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 04C — FULL GRID RESULT: OUTER R1/F1\n")
cat("============================================================\n")

cat("\n--- TOP HYPERPARAMETER CONFIGURATIONS ---\n")

print(
  head(
    GRID_RESULT[
      order(
        GRID_RESULT$selection_rank
      ),
      c(
        "selection_rank",
        "alpha",
        "deviation_penalty",
        "best_lambda",
        "best_inner_auc",
        "best_inner_auc_sd",
        "runtime_seconds"
      )
    ],
    10
  ),
  row.names = FALSE
)

cat("\n--- SELECTED MODEL ---\n")

cat(
  "Alpha:                    ",
  BEST_ALPHA,
  "\n",
  sep = ""
)

cat(
  "Deviation penalty:        ",
  BEST_DEV_PENALTY,
  "\n",
  sep = ""
)

cat(
  "Lambda:                   ",
  signif(
    BEST_LAMBDA,
    7
  ),
  "\n",
  sep = ""
)

cat(
  "Inner CV AUROC:           ",
  round(
    BEST_INNER_AUC,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Outer donor AUROC:        ",
  round(
    DONOR_AUC,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Outer sample AUROC:       ",
  round(
    SAMPLE_AUC,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Outer DLPFC AUROC:        ",
  round(
    DLPFC_AUC,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Outer HIPPO AUROC:        ",
  round(
    HIPPO_AUC,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Nonzero shared genes:     ",
  N_SHARED_NONZERO,
  "\n",
  sep = ""
)

cat(
  "Nonzero deviation genes:  ",
  N_DEV_NONZERO,
  "\n",
  sep = ""
)

cat(
  "20-grid runtime:          ",
  round(
    sum(
      GRID_RESULT$runtime_seconds
    ),
    1
  ),
  " sec\n",
  sep = ""
)

cat(
  "Total cell runtime:       ",
  round(
    TOTAL_SECONDS,
    1
  ),
  " sec\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 04C — CELL 1 PASS\n")
cat("============================================================\n")


STEP 04C — STARTING 20-CONFIGURATION GRID
[01/20] alpha=0.10  deviation=1 ... AUC=0.7639  lambda=0.013202  time=32.0s
[02/20] alpha=0.25  deviation=1 ... AUC=0.7606  lambda=0.0052808  time=21.4s
[03/20] alpha=0.50  deviation=1 ... AUC=0.7565  lambda=0.0026404  time=15.7s
[04/20] alpha=0.75  deviation=1 ... AUC=0.7572  lambda=0.0021243  time=10.9s
[05/20] alpha=1.00  deviation=1 ... AUC=0.7490  lambda=0.0013202  time=12.2s
[06/20] alpha=0.10  deviation=2 ... AUC=0.7622  lambda=0.019803  time=18.8s
[07/20] alpha=0.25  deviation=2 ... AUC=0.7608  lambda=0.0087018  time=10.0s
[08/20] alpha=0.50  deviation=2 ... AUC=0.7550  lambda=0.0057681  time=12.8s
[09/20] alpha=0.75  deviation=2 ... AUC=0.7557  lambda=0.0026404  time=11.9s
[10/20] alpha=1.00  deviation=2 ... AUC=0.7473  lambda=0.0021755  time=12.2s
[11/20] alpha=0.10  deviation=4 ... AUC=0.7621  lambda=0.033005  time=18.6s
[12/20] alpha=0.25  deviation=4 ... AUC=0.7608  lambda=0.014503  time=13.9s
[13/20] alpha=0.50  deviation=4 ... A

In [7]:
# ============================================================
# STEP 04D — FULL CORRECTED CELL
# FULL NESTED DONOR-GROUPED CROSS-VALIDATION
#
# Outer CV : 5 folds x 5 repeats = 25 held-out tests
# Inner CV : 4-fold donor-grouped
#
# Hyperparameter grid:
#   alpha = 0.10, 0.25, 0.50, 0.75, 1.00
#   deviation penalty = 1, 2, 4, 8
#
# Shared / deviation model:
#   beta_DLPFC = beta_shared + beta_deviation
#   beta_HIPPO = beta_shared - beta_deviation
#
# Features:
#   23,740 primary genes
#   23,740 shared coefficients
#   23,740 region-deviation coefficients
#
# CHECKPOINT / RESUME ENABLED
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {

  stop(
    paste0(
      "Project ID mismatch. Expected ",
      PROJECT_ID,
      "; observed ",
      OBSERVED_ID
    )
  )
}

cat("Project sentinel: PASS\n")
flush.console()


# ------------------------------------------------------------
# 1. GLMNET
# ------------------------------------------------------------

if (!requireNamespace(
  "glmnet",
  quietly = TRUE
)) {

  cat("Installing glmnet...\n")
  flush.console()

  install.packages(
    "glmnet",
    repos = "https://cloud.r-project.org"
  )
}

if (!requireNamespace(
  "glmnet",
  quietly = TRUE
)) {
  stop("glmnet unavailable.")
}

cat(
  "glmnet version: ",
  as.character(
    packageVersion("glmnet")
  ),
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 2. CANONICAL INPUT PATHS
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_gene_annotation_v1.csv"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

SPLIT_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "cv_splits",
  "outer_cv_donor_splits_5fold_5repeat_v1.csv"
)

REQUIRED_FILES <- c(
  COUNTS_PATH,
  ANNOTATION_PATH,
  META_PATH,
  SPLIT_PATH
)

if (!all(
  file.exists(
    REQUIRED_FILES
  )
)) {

  stop(
    paste0(
      "Missing canonical input(s):\n",
      paste(
        REQUIRED_FILES[
          !file.exists(
            REQUIRED_FILES
          )
        ],
        collapse = "\n"
      )
    )
  )
}


# ------------------------------------------------------------
# 3. LOAD CANONICAL DATA
# ------------------------------------------------------------

cat("Loading canonical inputs...\n")
flush.console()

COUNTS <- readRDS(
  COUNTS_PATH
)

ANNOTATION <- read.csv(
  ANNOTATION_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

OUTER_SPLITS <- read.csv(
  SPLIT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(COUNTS) == 23740L,
  ncol(COUNTS) == 755L,
  nrow(ANNOTATION) == 23740L,
  nrow(META) == 755L,
  length(unique(META$BrNum)) == 454L
)

if (!identical(
  rownames(COUNTS),
  ANNOTATION$ensemblID_stable
)) {
  stop(
    "Count-matrix rows and canonical annotation are not aligned."
  )
}

N_GENES <- nrow(
  COUNTS
)

cat(
  "Loaded: ",
  N_GENES,
  " genes x ",
  ncol(COUNTS),
  " samples\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 4. SAMPLE-LOCAL EXPRESSION TRANSFORMATION
#
# log2(CPM + 1)
#
# Each sample is transformed using only its own library size.
# No test-set-dependent cross-sample normalization reference.
# ------------------------------------------------------------

cat(
  "Building log2(CPM + 1) matrix...\n"
)

flush.console()

LIB_SIZE <- colSums(
  COUNTS
)

if (
  any(!is.finite(LIB_SIZE)) ||
  any(LIB_SIZE <= 0)
) {
  stop("Invalid RNA-seq library size detected.")
}

LOGCPM <- log2(
  sweep(
    COUNTS,
    MARGIN = 2,
    STATS = LIB_SIZE / 1e6,
    FUN = "/"
  ) + 1
)

rm(
  COUNTS,
  LIB_SIZE
)

invisible(
  gc()
)

if (
  anyNA(LOGCPM) ||
  any(!is.finite(LOGCPM))
) {
  stop("Non-finite logCPM values detected.")
}

stopifnot(
  nrow(LOGCPM) == 23740L,
  ncol(LOGCPM) == 755L
)

cat("Expression transformation: PASS\n")
flush.console()


# ------------------------------------------------------------
# 5. RESPONSE + REGION SIGN
# ------------------------------------------------------------

Y <- ifelse(
  META$Dx == "Schizo",
  1L,
  0L
)

if (!setequal(
  unique(Y),
  c(0L, 1L)
)) {
  stop("Outcome encoding failed.")
}

REGION_SIGN <- ifelse(
  META$Region == "DLPFC",
  1,
  ifelse(
    META$Region == "HIPPO",
    -1,
    NA_real_
  )
)

if (anyNA(
  REGION_SIGN
)) {
  stop("Unexpected brain-region label.")
}


# ------------------------------------------------------------
# 6. FACTORS FOR COVARIATES
# ------------------------------------------------------------

META$Region <- factor(
  META$Region,
  levels = c(
    "DLPFC",
    "HIPPO"
  )
)

META$Sex <- factor(
  META$Sex
)

META$Race <- factor(
  META$Race
)


# ------------------------------------------------------------
# 7. UNPENALIZED COVARIATE MATRIX
# ------------------------------------------------------------

COVARIATE_MATRIX <- model.matrix(
  ~ Region +
    donor_age +
    Sex +
    Race +
    RIN +
    snpPC1 +
    snpPC2 +
    snpPC3 +
    snpPC4 +
    snpPC5 +
    snpPC6 +
    snpPC7 +
    snpPC8 +
    snpPC9 +
    snpPC10,
  data = META
)

# glmnet fits its own intercept
COVARIATE_MATRIX <- COVARIATE_MATRIX[
  ,
  colnames(COVARIATE_MATRIX) != "(Intercept)",
  drop = FALSE
]

if (
  anyNA(COVARIATE_MATRIX) ||
  any(!is.finite(COVARIATE_MATRIX))
) {
  stop("Invalid covariate matrix.")
}

N_COV <- ncol(
  COVARIATE_MATRIX
)

cat(
  "Unpenalized covariates: ",
  N_COV,
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 8. CORRECTED SHARED / DEVIATION MATRIX
#
# IMPORTANT BUG FIX:
#
# X_EXPR = 755 samples x 23,740 genes
#
# Therefore column names MUST have length 23,740.
# Canonical Ensembl IDs are explicitly assigned here.
# ------------------------------------------------------------

cat(
  "Building shared/deviation matrix...\n"
)

flush.console()

X_EXPR <- t(
  LOGCPM
)

rm(
  LOGCPM
)

invisible(
  gc()
)

stopifnot(
  nrow(X_EXPR) == 755L,
  ncol(X_EXPR) == N_GENES,
  length(
    ANNOTATION$ensemblID_stable
  ) == N_GENES
)

# Explicit canonical gene IDs
colnames(X_EXPR) <-
  ANNOTATION$ensemblID_stable

# Shared expression component
X_SHARED <- X_EXPR

# Region-specific deviation component
#
# DLPFC = +X
# HIPPO = -X
X_DEVIATION <- sweep(
  X_EXPR,
  MARGIN = 1,
  STATS = REGION_SIGN,
  FUN = "*"
)

stopifnot(
  dim(X_SHARED)[1] == 755L,
  dim(X_SHARED)[2] == N_GENES,
  dim(X_DEVIATION)[1] == 755L,
  dim(X_DEVIATION)[2] == N_GENES
)

colnames(X_SHARED) <- paste0(
  "SHARED__",
  ANNOTATION$ensemblID_stable
)

colnames(X_DEVIATION) <- paste0(
  "DEV__",
  ANNOTATION$ensemblID_stable
)

X_MODEL <- cbind(
  COVARIATE_MATRIX,
  X_SHARED,
  X_DEVIATION
)

rm(
  X_EXPR,
  X_SHARED,
  X_DEVIATION
)

invisible(
  gc()
)

EXPECTED_PREDICTORS <- (
  N_COV +
  2L * N_GENES
)

stopifnot(
  nrow(X_MODEL) == 755L,
  ncol(X_MODEL) ==
    EXPECTED_PREDICTORS
)

cat(
  "Total predictors: ",
  ncol(X_MODEL),
  "\n",
  sep = ""
)

cat(
  "  Covariates:      ",
  N_COV,
  "\n",
  sep = ""
)

cat(
  "  Shared genes:    ",
  N_GENES,
  "\n",
  sep = ""
)

cat(
  "  Deviation genes: ",
  N_GENES,
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 9. DONOR-BALANCED SAMPLE WEIGHTS
# ------------------------------------------------------------

OBS_WEIGHT <- as.numeric(
  META$donor_balance_weight
)

if (
  length(OBS_WEIGHT) != 755L ||
  anyNA(OBS_WEIGHT) ||
  any(!is.finite(OBS_WEIGHT)) ||
  any(OBS_WEIGHT <= 0)
) {
  stop(
    "Invalid donor-balanced observation weights."
  )
}

cat(
  "Total donor-balanced weight: ",
  sum(OBS_WEIGHT),
  "\n",
  sep = ""
)

if (
  abs(
    sum(OBS_WEIGHT) -
    454
  ) > 1e-8
) {
  stop(
    "Donor-balanced total weight should equal 454."
  )
}


# ------------------------------------------------------------
# 10. VERIFY OUTER SPLITS
# ------------------------------------------------------------

stopifnot(
  nrow(OUTER_SPLITS) ==
    454L * 5L
)

for (r in 1:5) {

  TMP <- OUTER_SPLITS[
    OUTER_SPLITS$repeat_id == r,
    ,
    drop = FALSE
  ]

  if (
    nrow(TMP) != 454L ||
    length(
      unique(
        TMP$BrNum
      )
    ) != 454L
  ) {
    stop(
      paste(
        "Invalid donor structure in repeat",
        r
      )
    )
  }

  if (!setequal(
    unique(
      TMP$outer_fold
    ),
    1:5
  )) {
    stop(
      paste(
        "Invalid folds in repeat",
        r
      )
    )
  }
}

cat(
  "Outer CV structure: PASS\n"
)

flush.console()


# ------------------------------------------------------------
# 11. VERIFY COVARIATES VARY WITHIN EVERY OUTER TRAINING SET
# ------------------------------------------------------------

for (r in 1:5) {

  SPLIT_R <- OUTER_SPLITS[
    OUTER_SPLITS$repeat_id == r,
    ,
    drop = FALSE
  ]

  for (f in 1:5) {

    TRAIN_DONORS_CHECK <- SPLIT_R$BrNum[
      SPLIT_R$outer_fold != f
    ]

    TRAIN_IDX_CHECK <- which(
      META$BrNum %in%
        TRAIN_DONORS_CHECK
    )

    COV_SD <- apply(
      COVARIATE_MATRIX[
        TRAIN_IDX_CHECK,
        ,
        drop = FALSE
      ],
      2,
      sd
    )

    BAD_COV <- (
      !is.finite(COV_SD) |
      COV_SD == 0
    )

    if (any(BAD_COV)) {

      stop(
        paste0(
          "Constant covariate in Repeat ",
          r,
          ", Fold ",
          f,
          ": ",
          paste(
            colnames(
              COVARIATE_MATRIX
            )[BAD_COV],
            collapse = ", "
          )
        )
      )
    }
  }
}

cat(
  "Outer-training covariate audit: PASS\n"
)

flush.console()


# ------------------------------------------------------------
# 12. HYPERPARAMETER GRID
# ------------------------------------------------------------

ALPHA_GRID <- c(
  0.10,
  0.25,
  0.50,
  0.75,
  1.00
)

DEV_GRID <- c(
  1,
  2,
  4,
  8
)

GRID_TEMPLATE <- expand.grid(
  alpha = ALPHA_GRID,
  deviation_penalty = DEV_GRID,
  KEEP.OUT.ATTRS = FALSE,
  stringsAsFactors = FALSE
)

GRID_TEMPLATE <- GRID_TEMPLATE[
  order(
    GRID_TEMPLATE$deviation_penalty,
    GRID_TEMPLATE$alpha
  ),
  ,
  drop = FALSE
]

rownames(
  GRID_TEMPLATE
) <- NULL

GRID_TEMPLATE$config_id <- seq_len(
  nrow(
    GRID_TEMPLATE
  )
)

stopifnot(
  nrow(GRID_TEMPLATE) == 20L
)


# ------------------------------------------------------------
# 13. AUROC HELPER
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  y <- as.integer(
    y
  )

  ok <- (
    !is.na(y) &
    is.finite(score)
  )

  y <- y[ok]
  score <- score[ok]

  n1 <- sum(
    y == 1L
  )

  n0 <- sum(
    y == 0L
  )

  if (
    n1 == 0L ||
    n0 == 0L
  ) {
    return(
      NA_real_
    )
  }

  rank_score <- rank(
    score,
    ties.method = "average"
  )

  auc <- (
    sum(
      rank_score[
        y == 1L
      ]
    ) -
      n1 *
      (
        n1 + 1
      ) /
      2
  ) /
    (
      n1 *
      n0
    )

  as.numeric(
    auc
  )
}


# ------------------------------------------------------------
# 14. INNER DONOR-GROUPED 4-FOLD CONSTRUCTOR
# ------------------------------------------------------------

make_inner_foldid <- function(
  train_meta,
  seed
) {

  DONOR_IDS <- unique(
    train_meta$BrNum
  )

  DONOR_TABLE <- do.call(
    rbind,
    lapply(
      DONOR_IDS,
      function(id) {

        z <- train_meta[
          train_meta$BrNum == id,
          ,
          drop = FALSE
        ]

        DX_VALUES <- unique(
          as.character(
            z$Dx
          )
        )

        if (
          length(
            DX_VALUES
          ) != 1L
        ) {
          stop(
            paste0(
              "Diagnosis inconsistent within donor ",
              id
            )
          )
        }

        HAS_DLPFC <- any(
          z$Region == "DLPFC"
        )

        HAS_HIPPO <- any(
          z$Region == "HIPPO"
        )

        AVAILABILITY <- if (
          HAS_DLPFC &&
          HAS_HIPPO
        ) {

          "DLPFC_HIPPO"

        } else if (
          HAS_DLPFC
        ) {

          "DLPFC_only"

        } else if (
          HAS_HIPPO
        ) {

          "HIPPO_only"

        } else {

          stop(
            paste0(
              "No valid region for donor ",
              id
            )
          )
        }

        data.frame(
          BrNum = id,
          Dx = DX_VALUES[1],
          region_availability =
            AVAILABILITY,
          stringsAsFactors = FALSE
        )
      }
    )
  )

  rownames(
    DONOR_TABLE
  ) <- NULL

  DONOR_TABLE$stratum <- paste(
    DONOR_TABLE$Dx,
    DONOR_TABLE$region_availability,
    sep = "__"
  )

  DONOR_TABLE$inner_fold <-
    NA_integer_

  set.seed(
    seed
  )

  for (
    s in unique(
      DONOR_TABLE$stratum
    )
  ) {

    IDX <- which(
      DONOR_TABLE$stratum == s
    )

    IDX <- sample(
      IDX,
      length(IDX),
      replace = FALSE
    )

    FOLD_SEQUENCE <- rep(
      1:4,
      length.out = length(IDX)
    )

    FOLD_SEQUENCE <- sample(
      FOLD_SEQUENCE,
      length(FOLD_SEQUENCE),
      replace = FALSE
    )

    DONOR_TABLE$inner_fold[
      IDX
    ] <- FOLD_SEQUENCE
  }

  LOOKUP <- setNames(
    DONOR_TABLE$inner_fold,
    DONOR_TABLE$BrNum
  )

  FOLDID <- as.integer(
    LOOKUP[
      train_meta$BrNum
    ]
  )

  if (
    anyNA(FOLDID) ||
    !setequal(
      unique(FOLDID),
      1:4
    )
  ) {
    stop(
      "Invalid inner fold assignment."
    )
  }

  # Hard leakage audit
  for (f in 1:4) {

    VALIDATION_DONORS <- unique(
      train_meta$BrNum[
        FOLDID == f
      ]
    )

    FIT_DONORS <- unique(
      train_meta$BrNum[
        FOLDID != f
      ]
    )

    OVERLAP <- intersect(
      VALIDATION_DONORS,
      FIT_DONORS
    )

    if (
      length(
        OVERLAP
      ) > 0L
    ) {
      stop(
        paste0(
          "Inner donor leakage detected in fold ",
          f
        )
      )
    }
  }

  list(
    foldid = FOLDID,
    donor_table = DONOR_TABLE
  )
}


# ------------------------------------------------------------
# 15. OUTPUT / CHECKPOINT DIRECTORY
# ------------------------------------------------------------

NESTED_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "nested_cv_5x5_v1"
)

dir.create(
  NESTED_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

cat(
  "Checkpoint directory:\n",
  NESTED_DIR,
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 16. FULL 25 OUTER TESTS
# ------------------------------------------------------------

OUTER_COUNTER <- 0L


for (r in 1:5) {

  SPLIT_R <- OUTER_SPLITS[
    OUTER_SPLITS$repeat_id == r,
    ,
    drop = FALSE
  ]


  for (f in 1:5) {

    OUTER_COUNTER <- OUTER_COUNTER + 1L

    FOLD_DIR <- file.path(
      NESTED_DIR,
      sprintf(
        "repeat_%02d_fold_%02d",
        r,
        f
      )
    )

    dir.create(
      FOLD_DIR,
      recursive = TRUE,
      showWarnings = FALSE
    )


    PASS_PATH <- file.path(
      FOLD_DIR,
      "PASS.txt"
    )

    SUMMARY_PATH <- file.path(
      FOLD_DIR,
      "fold_summary.csv"
    )

    GRID_PATH <- file.path(
      FOLD_DIR,
      "inner_grid.csv"
    )

    SAMPLE_PRED_PATH <- file.path(
      FOLD_DIR,
      "sample_predictions.csv"
    )

    DONOR_PRED_PATH <- file.path(
      FOLD_DIR,
      "donor_predictions.csv"
    )

    COEF_PATH <- file.path(
      FOLD_DIR,
      "gene_coefficients.rds"
    )

    INNER_MAP_PATH <- file.path(
      FOLD_DIR,
      "inner_donor_fold_map.csv"
    )


    # --------------------------------------------------------
    # 16A. RESUME
    # --------------------------------------------------------

    FOLD_COMPLETE <- all(
      file.exists(
        c(
          PASS_PATH,
          SUMMARY_PATH,
          GRID_PATH,
          SAMPLE_PRED_PATH,
          DONOR_PRED_PATH,
          COEF_PATH,
          INNER_MAP_PATH
        )
      )
    )

    if (
      FOLD_COMPLETE
    ) {

      cat(
        sprintf(
          "\n[%02d/25] Repeat %d Fold %d — already PASS; skipping.\n",
          OUTER_COUNTER,
          r,
          f
        )
      )

      flush.console()

      next
    }


    FOLD_START <- proc.time()[3]


    cat(
      "\n============================================================\n"
    )

    cat(
      sprintf(
        "[%02d/25] REPEAT %d / FOLD %d\n",
        OUTER_COUNTER,
        r,
        f
      )
    )

    cat(
      "============================================================\n"
    )

    flush.console()


    # --------------------------------------------------------
    # 16B. OUTER TRAIN / TEST
    # --------------------------------------------------------

    TEST_DONORS <- SPLIT_R$BrNum[
      SPLIT_R$outer_fold == f
    ]

    TRAIN_DONORS <- SPLIT_R$BrNum[
      SPLIT_R$outer_fold != f
    ]

    if (
      length(
        intersect(
          TRAIN_DONORS,
          TEST_DONORS
        )
      ) != 0L
    ) {
      stop(
        "Outer donor leakage detected."
      )
    }

    TRAIN_IDX <- which(
      META$BrNum %in%
        TRAIN_DONORS
    )

    TEST_IDX <- which(
      META$BrNum %in%
        TEST_DONORS
    )

    TRAIN_META <- META[
      TRAIN_IDX,
      ,
      drop = FALSE
    ]

    TEST_META <- META[
      TEST_IDX,
      ,
      drop = FALSE
    ]

    cat(
      "Train: ",
      length(TRAIN_DONORS),
      " donors / ",
      length(TRAIN_IDX),
      " samples | Test: ",
      length(TEST_DONORS),
      " donors / ",
      length(TEST_IDX),
      " samples\n",
      sep = ""
    )

    flush.console()


    # --------------------------------------------------------
    # 16C. INNER 4-FOLD DONOR-GROUPED CV
    # --------------------------------------------------------

    INNER_SEED <- (
      2026093000L +
      (r - 1L) * 5L +
      f
    )

    INNER_OBJECT <- make_inner_foldid(
      train_meta = TRAIN_META,
      seed = INNER_SEED
    )

    INNER_FOLD_ID <-
      INNER_OBJECT$foldid

    write.csv(
      INNER_OBJECT$donor_table,
      INNER_MAP_PATH,
      row.names = FALSE
    )

    cat(
      "Inner donor-grouped CV: PASS\n"
    )

    flush.console()


    # --------------------------------------------------------
    # 16D. RUN 20-CONFIGURATION GRID
    # --------------------------------------------------------

    GRID_RESULT <- data.frame(

      config_id =
        GRID_TEMPLATE$config_id,

      alpha =
        GRID_TEMPLATE$alpha,

      deviation_penalty =
        GRID_TEMPLATE$deviation_penalty,

      best_lambda =
        NA_real_,

      best_inner_auc =
        NA_real_,

      best_inner_auc_sd =
        NA_real_,

      n_lambda =
        NA_integer_,

      runtime_seconds =
        NA_real_,

      stringsAsFactors = FALSE
    )


    for (
      g in seq_len(
        nrow(
          GRID_TEMPLATE
        )
      )
    ) {

      ALPHA_VALUE <-
        GRID_TEMPLATE$alpha[g]

      DEV_VALUE <-
        GRID_TEMPLATE$deviation_penalty[g]


      PENALTY_FACTOR <- c(

        rep(
          0,
          N_COV
        ),

        rep(
          1,
          N_GENES
        ),

        rep(
          DEV_VALUE,
          N_GENES
        )
      )


      stopifnot(
        length(
          PENALTY_FACTOR
        ) ==
          ncol(
            X_MODEL
          )
      )


      cat(
        sprintf(
          "  [%02d/20] alpha=%.2f dev=%d ... ",
          g,
          ALPHA_VALUE,
          DEV_VALUE
        )
      )

      flush.console()


      GRID_START <- proc.time()[3]


      CVFIT <- glmnet::cv.glmnet(

        x = X_MODEL[
          TRAIN_IDX,
          ,
          drop = FALSE
        ],

        y = Y[
          TRAIN_IDX
        ],

        family = "binomial",

        weights = OBS_WEIGHT[
          TRAIN_IDX
        ],

        foldid = INNER_FOLD_ID,

        type.measure = "auc",

        alpha = ALPHA_VALUE,

        penalty.factor =
          PENALTY_FACTOR,

        standardize = TRUE,

        intercept = TRUE,

        nlambda = 50,

        keep = FALSE,

        parallel = FALSE
      )


      GRID_SECONDS <- (
        proc.time()[3] -
        GRID_START
      )


      VALID_AUC <- is.finite(
        CVFIT$cvm
      )


      if (
        !any(
          VALID_AUC
        )
      ) {

        stop(
          paste0(
            "No valid inner-CV AUC for Repeat ",
            r,
            ", Fold ",
            f,
            ", configuration ",
            g
          )
        )
      }


      BEST_INDEX <- which.max(
        ifelse(
          VALID_AUC,
          CVFIT$cvm,
          -Inf
        )
      )


      GRID_RESULT$best_lambda[g] <-
        CVFIT$lambda[
          BEST_INDEX
        ]

      GRID_RESULT$best_inner_auc[g] <-
        CVFIT$cvm[
          BEST_INDEX
        ]

      GRID_RESULT$best_inner_auc_sd[g] <-
        CVFIT$cvsd[
          BEST_INDEX
        ]

      GRID_RESULT$n_lambda[g] <-
        length(
          CVFIT$lambda
        )

      GRID_RESULT$runtime_seconds[g] <-
        GRID_SECONDS


      cat(
        sprintf(
          "AUC=%.4f lambda=%.5g time=%.1fs\n",
          GRID_RESULT$best_inner_auc[g],
          GRID_RESULT$best_lambda[g],
          GRID_SECONDS
        )
      )

      flush.console()


      rm(
        CVFIT
      )
    }


    # --------------------------------------------------------
    # 16E. SELECT BEST INNER-CV CONFIGURATION
    #
    # Primary:
    #   highest mean inner-CV AUROC
    #
    # Tie-break:
    #   lower CV uncertainty
    #   stronger deviation penalty
    #   lower alpha
    # --------------------------------------------------------

    SELECT_ORDER <- order(
      -GRID_RESULT$best_inner_auc,
      GRID_RESULT$best_inner_auc_sd,
      -GRID_RESULT$deviation_penalty,
      GRID_RESULT$alpha
    )


    GRID_RESULT$selection_rank <- match(
      seq_len(
        nrow(
          GRID_RESULT
        )
      ),
      SELECT_ORDER
    )


    BEST_ROW <- SELECT_ORDER[1]


    BEST_ALPHA <-
      GRID_RESULT$alpha[
        BEST_ROW
      ]

    BEST_DEV <-
      GRID_RESULT$deviation_penalty[
        BEST_ROW
      ]

    BEST_LAMBDA <-
      GRID_RESULT$best_lambda[
        BEST_ROW
      ]

    BEST_INNER_AUC <-
      GRID_RESULT$best_inner_auc[
        BEST_ROW
      ]

    BEST_INNER_SD <-
      GRID_RESULT$best_inner_auc_sd[
        BEST_ROW
      ]


    write.csv(
      GRID_RESULT[
        order(
          GRID_RESULT$selection_rank
        ),
        ,
        drop = FALSE
      ],
      GRID_PATH,
      row.names = FALSE
    )


    cat(
      "Selected: alpha=",
      BEST_ALPHA,
      " dev=",
      BEST_DEV,
      " lambda=",
      signif(
        BEST_LAMBDA,
        7
      ),
      " innerAUC=",
      round(
        BEST_INNER_AUC,
        4
      ),
      "\n",
      sep = ""
    )

    flush.console()


    # --------------------------------------------------------
    # 16F. FINAL MODEL ON COMPLETE OUTER TRAINING SET
    # --------------------------------------------------------

    BEST_PENALTY_FACTOR <- c(

      rep(
        0,
        N_COV
      ),

      rep(
        1,
        N_GENES
      ),

      rep(
        BEST_DEV,
        N_GENES
      )
    )


    FINAL_START <- proc.time()[3]


    FINAL_FIT <- glmnet::glmnet(

      x = X_MODEL[
        TRAIN_IDX,
        ,
        drop = FALSE
      ],

      y = Y[
        TRAIN_IDX
      ],

      family = "binomial",

      weights = OBS_WEIGHT[
        TRAIN_IDX
      ],

      alpha = BEST_ALPHA,

      penalty.factor =
        BEST_PENALTY_FACTOR,

      standardize = TRUE,

      intercept = TRUE,

      lambda = BEST_LAMBDA
    )


    FINAL_SECONDS <- (
      proc.time()[3] -
      FINAL_START
    )


    # --------------------------------------------------------
    # 16G. OUTER HELD-OUT PREDICTIONS
    # --------------------------------------------------------

    TEST_PRED <- as.numeric(
      predict(
        FINAL_FIT,
        newx = X_MODEL[
          TEST_IDX,
          ,
          drop = FALSE
        ],
        type = "response",
        s = BEST_LAMBDA
      )
    )


    if (
      length(TEST_PRED) !=
        length(TEST_IDX) ||
      anyNA(TEST_PRED) ||
      any(!is.finite(TEST_PRED))
    ) {
      stop(
        "Invalid outer-test predictions."
      )
    }


    TEST_META$y <- Y[
      TEST_IDX
    ]

    TEST_META$prediction <-
      TEST_PRED

    TEST_META$outer_repeat <- r
    TEST_META$outer_fold <- f


    # --------------------------------------------------------
    # 16H. SAMPLE-LEVEL OUTER AUROC
    # --------------------------------------------------------

    SAMPLE_AUC <- auc_rank(
      TEST_META$y,
      TEST_META$prediction
    )


    # --------------------------------------------------------
    # 16I. REGION-SPECIFIC OUTER AUROC
    # --------------------------------------------------------

    DLPFC_FLAG <- (
      TEST_META$Region ==
        "DLPFC"
    )

    HIPPO_FLAG <- (
      TEST_META$Region ==
        "HIPPO"
    )


    DLPFC_AUC <- auc_rank(
      TEST_META$y[
        DLPFC_FLAG
      ],
      TEST_META$prediction[
        DLPFC_FLAG
      ]
    )


    HIPPO_AUC <- auc_rank(
      TEST_META$y[
        HIPPO_FLAG
      ],
      TEST_META$prediction[
        HIPPO_FLAG
      ]
    )


    # --------------------------------------------------------
    # 16J. DONOR-LEVEL OUTER PREDICTIONS
    #
    # Paired donors:
    #   average DLPFC and HIPPO predicted probability
    #
    # Unpaired donors:
    #   single available prediction
    # --------------------------------------------------------

    DONOR_PRED <- aggregate(
      prediction ~ BrNum,
      data = TEST_META,
      FUN = mean
    )


    DONOR_Y <- aggregate(
      y ~ BrNum,
      data = TEST_META,
      FUN = function(x) {

        ux <- unique(
          x
        )

        if (
          length(
            ux
          ) != 1L
        ) {
          stop(
            "Outcome mismatch within donor."
          )
        }

        ux
      }
    )


    DONOR_RESULT <- merge(
      DONOR_Y,
      DONOR_PRED,
      by = "BrNum",
      sort = FALSE
    )


    DONOR_RESULT$outer_repeat <- r
    DONOR_RESULT$outer_fold <- f


    DONOR_AUC <- auc_rank(
      DONOR_RESULT$y,
      DONOR_RESULT$prediction
    )


    # --------------------------------------------------------
    # 16K. EXTRACT MODEL COEFFICIENTS
    # --------------------------------------------------------

    COEF_MATRIX <- as.matrix(
      coef(
        FINAL_FIT,
        s = BEST_LAMBDA
      )
    )


    COEF_VECTOR <- COEF_MATRIX[
      ,
      1
    ]


    names(
      COEF_VECTOR
    ) <- rownames(
      COEF_MATRIX
    )


    FEATURE_NAMES <- colnames(
      X_MODEL
    )


    FEATURE_COEF <- COEF_VECTOR[
      FEATURE_NAMES
    ]


    if (
      anyNA(
        FEATURE_COEF
      )
    ) {
      stop(
        "Coefficient extraction failed."
      )
    }


    SHARED_START <- (
      N_COV + 1L
    )

    SHARED_END <- (
      N_COV +
      N_GENES
    )

    DEV_START <- (
      SHARED_END +
      1L
    )

    DEV_END <- (
      SHARED_END +
      N_GENES
    )


    SHARED_COEF <- FEATURE_COEF[
      SHARED_START:
        SHARED_END
    ]


    DEV_COEF <- FEATURE_COEF[
      DEV_START:
        DEV_END
    ]


    stopifnot(
      length(SHARED_COEF) ==
        N_GENES,
      length(DEV_COEF) ==
        N_GENES
    )


    # --------------------------------------------------------
    # 16L. GENE-LEVEL COEFFICIENT TABLE
    # --------------------------------------------------------

    COEF_TABLE <- data.frame(

      ensemblID_stable =
        ANNOTATION$ensemblID_stable,

      gene_symbol =
        ANNOTATION$gene_symbol,

      gene_type =
        ANNOTATION$gene_type,

      beta_shared =
        as.numeric(
          SHARED_COEF
        ),

      beta_deviation =
        as.numeric(
          DEV_COEF
        ),

      beta_DLPFC =
        as.numeric(
          SHARED_COEF +
          DEV_COEF
        ),

      beta_HIPPO =
        as.numeric(
          SHARED_COEF -
          DEV_COEF
        ),

      stringsAsFactors = FALSE
    )


    COEF_TABLE$selected_shared <- (
      abs(
        COEF_TABLE$beta_shared
      ) > 0
    )


    COEF_TABLE$selected_deviation <- (
      abs(
        COEF_TABLE$beta_deviation
      ) > 0
    )


    N_SHARED_NONZERO <- sum(
      COEF_TABLE$selected_shared
    )


    N_DEV_NONZERO <- sum(
      COEF_TABLE$selected_deviation
    )


    # --------------------------------------------------------
    # 16M. SAVE THIS OUTER FOLD
    # --------------------------------------------------------

    FOLD_SECONDS <- (
      proc.time()[3] -
      FOLD_START
    )


    FOLD_SUMMARY <- data.frame(

      outer_repeat = r,

      outer_fold = f,

      train_donors =
        length(
          TRAIN_DONORS
        ),

      test_donors =
        length(
          TEST_DONORS
        ),

      train_samples =
        length(
          TRAIN_IDX
        ),

      test_samples =
        length(
          TEST_IDX
        ),

      best_alpha =
        BEST_ALPHA,

      best_deviation_penalty =
        BEST_DEV,

      best_lambda =
        BEST_LAMBDA,

      best_inner_auc =
        BEST_INNER_AUC,

      best_inner_auc_sd =
        BEST_INNER_SD,

      outer_sample_auc =
        SAMPLE_AUC,

      outer_donor_auc =
        DONOR_AUC,

      outer_DLPFC_auc =
        DLPFC_AUC,

      outer_HIPPO_auc =
        HIPPO_AUC,

      n_shared_nonzero =
        N_SHARED_NONZERO,

      n_deviation_nonzero =
        N_DEV_NONZERO,

      grid_seconds =
        sum(
          GRID_RESULT$runtime_seconds
        ),

      final_fit_seconds =
        FINAL_SECONDS,

      total_fold_seconds =
        FOLD_SECONDS,

      stringsAsFactors = FALSE
    )


    write.csv(
      FOLD_SUMMARY,
      SUMMARY_PATH,
      row.names = FALSE
    )


    write.csv(
      TEST_META,
      SAMPLE_PRED_PATH,
      row.names = FALSE
    )


    write.csv(
      DONOR_RESULT,
      DONOR_PRED_PATH,
      row.names = FALSE
    )


    saveRDS(
      COEF_TABLE,
      COEF_PATH,
      compress = "gzip"
    )


    # PASS written LAST
    writeLines(
      c(
        paste0(
          "repeat=",
          r
        ),
        paste0(
          "fold=",
          f
        ),
        paste0(
          "donor_auc=",
          DONOR_AUC
        ),
        paste0(
          "sample_auc=",
          SAMPLE_AUC
        ),
        paste0(
          "completed=",
          format(
            Sys.time(),
            "%Y-%m-%d %H:%M:%S %z"
          )
        )
      ),
      PASS_PATH
    )


    cat(
      "PASS | donor AUC=",
      round(
        DONOR_AUC,
        4
      ),
      " | sample AUC=",
      round(
        SAMPLE_AUC,
        4
      ),
      " | DLPFC=",
      round(
        DLPFC_AUC,
        4
      ),
      " | HIPPO=",
      round(
        HIPPO_AUC,
        4
      ),
      " | shared=",
      N_SHARED_NONZERO,
      " | dev=",
      N_DEV_NONZERO,
      " | ",
      round(
        FOLD_SECONDS,
        1
      ),
      " sec\n",
      sep = ""
    )

    flush.console()


    rm(
      FINAL_FIT,
      COEF_MATRIX,
      COEF_VECTOR,
      FEATURE_COEF,
      SHARED_COEF,
      DEV_COEF,
      COEF_TABLE,
      TEST_PRED,
      PENALTY_FACTOR,
      BEST_PENALTY_FACTOR
    )

    invisible(
      gc()
    )
  }
}


# ============================================================
# 17. VERIFY ALL 25 OUTER FOLDS COMPLETE
# ============================================================

SUMMARY_FILES <- list.files(
  NESTED_DIR,
  pattern = "^fold_summary\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)

PASS_FILES <- list.files(
  NESTED_DIR,
  pattern = "^PASS\\.txt$",
  recursive = TRUE,
  full.names = TRUE
)


if (
  length(SUMMARY_FILES) != 25L ||
  length(PASS_FILES) != 25L
) {

  cat(
    "\n============================================================\n"
  )

  cat(
    "PARTIAL NESTED CV SAVED SAFELY\n"
  )

  cat(
    "Completed folds: ",
    length(PASS_FILES),
    " / 25\n",
    sep = ""
  )

  cat(
    "Re-run this SAME CELL to resume.\n"
  )

  cat(
    "============================================================\n"
  )

  stop(
    "Nested CV incomplete."
  )
}


# ------------------------------------------------------------
# 18. COMBINE ALL OUTER-FOLD SUMMARIES
# ------------------------------------------------------------

ALL_FOLD_SUMMARY <- do.call(
  rbind,
  lapply(
    SUMMARY_FILES,
    function(p) {

      read.csv(
        p,
        stringsAsFactors = FALSE,
        check.names = FALSE
      )
    }
  )
)


ALL_FOLD_SUMMARY <- ALL_FOLD_SUMMARY[
  order(
    ALL_FOLD_SUMMARY$outer_repeat,
    ALL_FOLD_SUMMARY$outer_fold
  ),
  ,
  drop = FALSE
]


rownames(
  ALL_FOLD_SUMMARY
) <- NULL


if (
  nrow(
    ALL_FOLD_SUMMARY
  ) != 25L
) {
  stop(
    "Expected exactly 25 outer-fold summaries."
  )
}


# ------------------------------------------------------------
# 19. COMBINE ALL SAMPLE-LEVEL OOF PREDICTIONS
# ------------------------------------------------------------

SAMPLE_FILES <- list.files(
  NESTED_DIR,
  pattern = "^sample_predictions\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)


if (
  length(
    SAMPLE_FILES
  ) != 25L
) {
  stop(
    "Expected 25 sample-prediction files."
  )
}


ALL_SAMPLE_PRED <- do.call(
  rbind,
  lapply(
    SAMPLE_FILES,
    function(p) {

      read.csv(
        p,
        stringsAsFactors = FALSE,
        check.names = FALSE
      )
    }
  )
)


# Every discovery sample should be held out once
# in each of the five repeats.
SAMPLE_NPRED <- table(
  ALL_SAMPLE_PRED$discovery_row_index
)


if (
  length(
    SAMPLE_NPRED
  ) != 755L ||
  !all(
    SAMPLE_NPRED == 5L
  )
) {

  stop(
    "OOF sample prediction multiplicity is invalid."
  )
}


# ------------------------------------------------------------
# 20. MEAN REPEATED OOF PREDICTION PER SAMPLE
# ------------------------------------------------------------

OOF_SAMPLE_MEAN <- aggregate(
  prediction ~ discovery_row_index,
  data = ALL_SAMPLE_PRED,
  FUN = mean
)


SAMPLE_REFERENCE <- ALL_SAMPLE_PRED[
  !duplicated(
    ALL_SAMPLE_PRED$discovery_row_index
  ),
  c(
    "discovery_row_index",
    "RNum",
    "BrNum",
    "Region",
    "Dx",
    "y"
  ),
  drop = FALSE
]


OOF_SAMPLE <- merge(
  SAMPLE_REFERENCE,
  OOF_SAMPLE_MEAN,
  by = "discovery_row_index",
  sort = FALSE
)


OOF_SAMPLE <- OOF_SAMPLE[
  order(
    OOF_SAMPLE$discovery_row_index
  ),
  ,
  drop = FALSE
]


stopifnot(
  nrow(
    OOF_SAMPLE
  ) == 755L
)


OVERALL_SAMPLE_AUC <- auc_rank(
  OOF_SAMPLE$y,
  OOF_SAMPLE$prediction
)


OVERALL_DLPFC_AUC <- auc_rank(
  OOF_SAMPLE$y[
    OOF_SAMPLE$Region ==
      "DLPFC"
  ],
  OOF_SAMPLE$prediction[
    OOF_SAMPLE$Region ==
      "DLPFC"
  ]
)


OVERALL_HIPPO_AUC <- auc_rank(
  OOF_SAMPLE$y[
    OOF_SAMPLE$Region ==
      "HIPPO"
  ],
  OOF_SAMPLE$prediction[
    OOF_SAMPLE$Region ==
      "HIPPO"
  ]
)


# ------------------------------------------------------------
# 21. COMBINE DONOR-LEVEL OOF PREDICTIONS
# ------------------------------------------------------------

DONOR_FILES <- list.files(
  NESTED_DIR,
  pattern = "^donor_predictions\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)


if (
  length(
    DONOR_FILES
  ) != 25L
) {
  stop(
    "Expected 25 donor-prediction files."
  )
}


ALL_DONOR_PRED <- do.call(
  rbind,
  lapply(
    DONOR_FILES,
    function(p) {

      read.csv(
        p,
        stringsAsFactors = FALSE,
        check.names = FALSE
      )
    }
  )
)


DONOR_NPRED <- table(
  ALL_DONOR_PRED$BrNum
)


if (
  length(
    DONOR_NPRED
  ) != 454L ||
  !all(
    DONOR_NPRED == 5L
  )
) {

  stop(
    "OOF donor prediction multiplicity is invalid."
  )
}


# ------------------------------------------------------------
# 22. MEAN REPEATED OOF PREDICTION PER DONOR
# ------------------------------------------------------------

OOF_DONOR_MEAN <- aggregate(
  prediction ~ BrNum,
  data = ALL_DONOR_PRED,
  FUN = mean
)


DONOR_REFERENCE <- ALL_DONOR_PRED[
  !duplicated(
    ALL_DONOR_PRED$BrNum
  ),
  c(
    "BrNum",
    "y"
  ),
  drop = FALSE
]


OOF_DONOR <- merge(
  DONOR_REFERENCE,
  OOF_DONOR_MEAN,
  by = "BrNum",
  sort = FALSE
)


stopifnot(
  nrow(
    OOF_DONOR
  ) == 454L
)


OVERALL_DONOR_AUC <- auc_rank(
  OOF_DONOR$y,
  OOF_DONOR$prediction
)


# ------------------------------------------------------------
# 23. HYPERPARAMETER SELECTION FREQUENCY
# ------------------------------------------------------------

HYPERPARAMETER_FREQUENCY <- as.data.frame(
  table(
    alpha =
      ALL_FOLD_SUMMARY$best_alpha,
    deviation_penalty =
      ALL_FOLD_SUMMARY$best_deviation_penalty
  ),
  stringsAsFactors = FALSE
)


names(
  HYPERPARAMETER_FREQUENCY
)[3] <- "n_outer_folds"


HYPERPARAMETER_FREQUENCY <-
  HYPERPARAMETER_FREQUENCY[
    HYPERPARAMETER_FREQUENCY$n_outer_folds >
      0,
    ,
    drop = FALSE
  ]


HYPERPARAMETER_FREQUENCY <-
  HYPERPARAMETER_FREQUENCY[
    order(
      -HYPERPARAMETER_FREQUENCY$n_outer_folds,
      HYPERPARAMETER_FREQUENCY$alpha,
      HYPERPARAMETER_FREQUENCY$deviation_penalty
    ),
    ,
    drop = FALSE
  ]


rownames(
  HYPERPARAMETER_FREQUENCY
) <- NULL


# ------------------------------------------------------------
# 24. FINAL PERFORMANCE SUMMARY
# ------------------------------------------------------------

TOTAL_SECONDS <- (
  proc.time()[3] -
  TOTAL_START
)


FINAL_PERFORMANCE <- data.frame(

  metric = c(

    "Repeated_OOF_donor_AUROC",

    "Repeated_OOF_sample_AUROC",

    "Repeated_OOF_DLPFC_AUROC",

    "Repeated_OOF_HIPPO_AUROC",

    "Mean_outer_fold_donor_AUROC",

    "SD_outer_fold_donor_AUROC",

    "Min_outer_fold_donor_AUROC",

    "Max_outer_fold_donor_AUROC",

    "Mean_outer_fold_sample_AUROC",

    "SD_outer_fold_sample_AUROC",

    "Mean_outer_fold_DLPFC_AUROC",

    "Mean_outer_fold_HIPPO_AUROC",

    "Mean_selected_shared_genes",

    "Median_selected_shared_genes",

    "Min_selected_shared_genes",

    "Max_selected_shared_genes",

    "Mean_selected_deviation_genes",

    "Median_selected_deviation_genes",

    "Min_selected_deviation_genes",

    "Max_selected_deviation_genes",

    "Total_current_run_seconds"
  ),


  value = c(

    OVERALL_DONOR_AUC,

    OVERALL_SAMPLE_AUC,

    OVERALL_DLPFC_AUC,

    OVERALL_HIPPO_AUC,

    mean(
      ALL_FOLD_SUMMARY$outer_donor_auc
    ),

    sd(
      ALL_FOLD_SUMMARY$outer_donor_auc
    ),

    min(
      ALL_FOLD_SUMMARY$outer_donor_auc
    ),

    max(
      ALL_FOLD_SUMMARY$outer_donor_auc
    ),

    mean(
      ALL_FOLD_SUMMARY$outer_sample_auc
    ),

    sd(
      ALL_FOLD_SUMMARY$outer_sample_auc
    ),

    mean(
      ALL_FOLD_SUMMARY$outer_DLPFC_auc
    ),

    mean(
      ALL_FOLD_SUMMARY$outer_HIPPO_auc
    ),

    mean(
      ALL_FOLD_SUMMARY$n_shared_nonzero
    ),

    median(
      ALL_FOLD_SUMMARY$n_shared_nonzero
    ),

    min(
      ALL_FOLD_SUMMARY$n_shared_nonzero
    ),

    max(
      ALL_FOLD_SUMMARY$n_shared_nonzero
    ),

    mean(
      ALL_FOLD_SUMMARY$n_deviation_nonzero
    ),

    median(
      ALL_FOLD_SUMMARY$n_deviation_nonzero
    ),

    min(
      ALL_FOLD_SUMMARY$n_deviation_nonzero
    ),

    max(
      ALL_FOLD_SUMMARY$n_deviation_nonzero
    ),

    TOTAL_SECONDS
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 25. SAVE COMBINED FINAL ARTIFACTS
# ------------------------------------------------------------

ALL_FOLD_SUMMARY_PATH <- file.path(
  NESTED_DIR,
  "ALL_25_OUTER_FOLD_SUMMARY_v1.csv"
)

FINAL_PERFORMANCE_PATH <- file.path(
  NESTED_DIR,
  "NESTED_CV_FINAL_PERFORMANCE_v1.csv"
)

OOF_SAMPLE_PATH <- file.path(
  NESTED_DIR,
  "OOF_SAMPLE_PREDICTIONS_AVERAGED_v1.csv"
)

OOF_DONOR_PATH <- file.path(
  NESTED_DIR,
  "OOF_DONOR_PREDICTIONS_AVERAGED_v1.csv"
)

HYPERPARAMETER_PATH <- file.path(
  NESTED_DIR,
  "HYPERPARAMETER_SELECTION_FREQUENCY_v1.csv"
)


write.csv(
  ALL_FOLD_SUMMARY,
  ALL_FOLD_SUMMARY_PATH,
  row.names = FALSE
)


write.csv(
  FINAL_PERFORMANCE,
  FINAL_PERFORMANCE_PATH,
  row.names = FALSE
)


write.csv(
  OOF_SAMPLE,
  OOF_SAMPLE_PATH,
  row.names = FALSE
)


write.csv(
  OOF_DONOR,
  OOF_DONOR_PATH,
  row.names = FALSE
)


write.csv(
  HYPERPARAMETER_FREQUENCY,
  HYPERPARAMETER_PATH,
  row.names = FALSE
)


# ------------------------------------------------------------
# 26. FINAL HARD CHECKS
# ------------------------------------------------------------

stopifnot(

  nrow(
    ALL_FOLD_SUMMARY
  ) == 25L,

  nrow(
    OOF_SAMPLE
  ) == 755L,

  nrow(
    OOF_DONOR
  ) == 454L,

  is.finite(
    OVERALL_DONOR_AUC
  ),

  is.finite(
    OVERALL_SAMPLE_AUC
  ),

  is.finite(
    OVERALL_DLPFC_AUC
  ),

  is.finite(
    OVERALL_HIPPO_AUC
  ),

  file.exists(
    FINAL_PERFORMANCE_PATH
  ),

  file.exists(
    ALL_FOLD_SUMMARY_PATH
  )
)


# ------------------------------------------------------------
# 27. FINAL OUTPUT
# ------------------------------------------------------------

cat(
  "\n============================================================\n"
)

cat(
  "STEP 04D — FULL NESTED CV COMPLETE\n"
)

cat(
  "============================================================\n"
)


cat(
  "Completed outer tests:          ",
  nrow(
    ALL_FOLD_SUMMARY
  ),
  "\n",
  sep = ""
)


cat(
  "Repeated-OOF donor AUROC:       ",
  round(
    OVERALL_DONOR_AUC,
    4
  ),
  "\n",
  sep = ""
)


cat(
  "Repeated-OOF sample AUROC:      ",
  round(
    OVERALL_SAMPLE_AUC,
    4
  ),
  "\n",
  sep = ""
)


cat(
  "Repeated-OOF DLPFC AUROC:       ",
  round(
    OVERALL_DLPFC_AUC,
    4
  ),
  "\n",
  sep = ""
)


cat(
  "Repeated-OOF HIPPO AUROC:       ",
  round(
    OVERALL_HIPPO_AUC,
    4
  ),
  "\n",
  sep = ""
)


cat(
  "Fold donor AUROC mean +/- SD:   ",
  round(
    mean(
      ALL_FOLD_SUMMARY$outer_donor_auc
    ),
    4
  ),
  " +/- ",
  round(
    sd(
      ALL_FOLD_SUMMARY$outer_donor_auc
    ),
    4
  ),
  "\n",
  sep = ""
)


cat(
  "Fold donor AUROC range:         ",
  round(
    min(
      ALL_FOLD_SUMMARY$outer_donor_auc
    ),
    4
  ),
  " - ",
  round(
    max(
      ALL_FOLD_SUMMARY$outer_donor_auc
    ),
    4
  ),
  "\n",
  sep = ""
)


cat(
  "Mean selected shared genes:     ",
  round(
    mean(
      ALL_FOLD_SUMMARY$n_shared_nonzero
    ),
    1
  ),
  "\n",
  sep = ""
)


cat(
  "Median selected shared genes:   ",
  median(
    ALL_FOLD_SUMMARY$n_shared_nonzero
  ),
  "\n",
  sep = ""
)


cat(
  "Mean selected deviation genes:  ",
  round(
    mean(
      ALL_FOLD_SUMMARY$n_deviation_nonzero
    ),
    1
  ),
  "\n",
  sep = ""
)


cat(
  "Median selected deviation genes:",
  " ",
  median(
    ALL_FOLD_SUMMARY$n_deviation_nonzero
  ),
  "\n",
  sep = ""
)


cat(
  "\n--- HYPERPARAMETER SELECTION FREQUENCY ---\n"
)


print(
  HYPERPARAMETER_FREQUENCY,
  row.names = FALSE
)


cat(
  "\n--- ALL 25 OUTER FOLDS ---\n"
)


print(
  ALL_FOLD_SUMMARY[
    ,
    c(
      "outer_repeat",
      "outer_fold",
      "best_alpha",
      "best_deviation_penalty",
      "best_inner_auc",
      "outer_donor_auc",
      "outer_sample_auc",
      "outer_DLPFC_auc",
      "outer_HIPPO_auc",
      "n_shared_nonzero",
      "n_deviation_nonzero",
      "total_fold_seconds"
    ),
    drop = FALSE
  ],
  row.names = FALSE
)


cat(
  "\n--- FINAL PERFORMANCE ---\n"
)


print(
  FINAL_PERFORMANCE,
  row.names = FALSE
)


cat(
  "\nSaved final performance:\n",
  FINAL_PERFORMANCE_PATH,
  "\n",
  sep = ""
)


cat(
  "\n============================================================\n"
)

cat(
  "STEP 04D — CELL 1 PASS\n"
)

cat(
  "============================================================\n"
)

Project sentinel: PASS
glmnet version: 5.1
Loading canonical inputs...
Loaded: 23740 genes x 755 samples
Building log2(CPM + 1) matrix...
Expression transformation: PASS
Unpenalized covariates: 17
Building shared/deviation matrix...
Total predictors: 47497
  Covariates:      17
  Shared genes:    23740
  Deviation genes: 23740
Total donor-balanced weight: 454
Outer CV structure: PASS
Outer-training covariate audit: PASS
Checkpoint directory:
E:/SCZ_CROSSREGION_MSS_Q1/06_models/nested_cv_5x5_v1

[01/25] REPEAT 1 / FOLD 1
Train: 361 donors / 601 samples | Test: 93 donors / 154 samples
Inner donor-grouped CV: PASS
  [01/20] alpha=0.10 dev=1 ... AUC=0.7639 lambda=0.013202 time=22.4s
  [02/20] alpha=0.25 dev=1 ... AUC=0.7606 lambda=0.0052808 time=18.6s
  [03/20] alpha=0.50 dev=1 ... AUC=0.7565 lambda=0.0026404 time=17.0s
  [04/20] alpha=0.75 dev=1 ... AUC=0.7572 lambda=0.0021243 time=14.1s
  [05/20] alpha=1.00 dev=1 ... AUC=0.7490 lambda=0.0013202 time=12.3s
  [06/20] alpha=0.10 dev=2 ... A

In [8]:
# ============================================================
# STEP 05A — CELL 1
# OUTER-MODEL COEFFICIENT STABILITY ANALYSIS
#
# Stability criteria:
#   Shared selection frequency >= 0.80
#   Sign consistency          >= 0.90
#
# Based ONLY on the 25 outer-training fitted models.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"

STABILITY_SELECTION_THRESHOLD <- 0.80
SIGN_CONSISTENCY_THRESHOLD     <- 0.90
ZERO_TOLERANCE                 <- 1e-12

TOTAL_START <- proc.time()[3]


# ------------------------------------------------------------
# 1. Input paths
# ------------------------------------------------------------

NESTED_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "nested_cv_5x5_v1"
)

ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_gene_annotation_v1.csv"
)

if (!dir.exists(NESTED_DIR)) {
  stop("Nested-CV directory missing.")
}

if (!file.exists(ANNOTATION_PATH)) {
  stop("Canonical annotation missing.")
}


# ------------------------------------------------------------
# 2. Locate all 25 coefficient files
# ------------------------------------------------------------

COEF_FILES <- list.files(
  NESTED_DIR,
  pattern = "^gene_coefficients\\.rds$",
  recursive = TRUE,
  full.names = TRUE
)

COEF_FILES <- sort(
  COEF_FILES
)

if (length(COEF_FILES) != 25L) {

  stop(
    paste0(
      "Expected 25 gene coefficient files; found ",
      length(COEF_FILES)
    )
  )
}

cat(
  "Coefficient files found: ",
  length(COEF_FILES),
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 3. Load canonical annotation
# ------------------------------------------------------------

ANNOTATION <- read.csv(
  ANNOTATION_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(ANNOTATION) == 23740L
)

GENE_IDS <- ANNOTATION$ensemblID_stable

if (
  anyNA(GENE_IDS) ||
  anyDuplicated(GENE_IDS)
) {
  stop("Canonical stable Ensembl IDs invalid.")
}


# ------------------------------------------------------------
# 4. Containers
# ------------------------------------------------------------

N_GENES <- length(GENE_IDS)
N_MODELS <- length(COEF_FILES)

SHARED_COEF_MATRIX <- matrix(
  0,
  nrow = N_GENES,
  ncol = N_MODELS
)

DEVIATION_COEF_MATRIX <- matrix(
  0,
  nrow = N_GENES,
  ncol = N_MODELS
)

rownames(SHARED_COEF_MATRIX) <- GENE_IDS
rownames(DEVIATION_COEF_MATRIX) <- GENE_IDS

MODEL_LABELS <- character(
  N_MODELS
)


# ------------------------------------------------------------
# 5. Read and align all 25 models
# ------------------------------------------------------------

cat("Reading 25 outer-model coefficient sets...\n")
flush.console()

for (i in seq_along(COEF_FILES)) {

  path <- COEF_FILES[i]

  z <- readRDS(
    path
  )

  required_cols <- c(
    "ensemblID_stable",
    "beta_shared",
    "beta_deviation"
  )

  if (!all(
    required_cols %in%
      colnames(z)
  )) {

    stop(
      paste0(
        "Required columns missing in ",
        path
      )
    )
  }

  if (
    nrow(z) != N_GENES ||
    anyDuplicated(
      z$ensemblID_stable
    )
  ) {

    stop(
      paste0(
        "Invalid coefficient table: ",
        path
      )
    )
  }

  idx <- match(
    GENE_IDS,
    z$ensemblID_stable
  )

  if (anyNA(idx)) {

    stop(
      paste0(
        "Gene alignment failure: ",
        path
      )
    )
  }

  SHARED_COEF_MATRIX[, i] <-
    as.numeric(
      z$beta_shared[idx]
    )

  DEVIATION_COEF_MATRIX[, i] <-
    as.numeric(
      z$beta_deviation[idx]
    )

  parent_name <- basename(
    dirname(path)
  )

  MODEL_LABELS[i] <- parent_name
}

colnames(SHARED_COEF_MATRIX) <- MODEL_LABELS
colnames(DEVIATION_COEF_MATRIX) <- MODEL_LABELS

if (
  anyNA(SHARED_COEF_MATRIX) ||
  anyNA(DEVIATION_COEF_MATRIX) ||
  any(!is.finite(SHARED_COEF_MATRIX)) ||
  any(!is.finite(DEVIATION_COEF_MATRIX))
) {
  stop("Non-finite coefficients detected.")
}

cat("Coefficient alignment: PASS\n")
flush.console()


# ------------------------------------------------------------
# 6. Selection matrices
# ------------------------------------------------------------

SHARED_SELECTED <- (
  abs(
    SHARED_COEF_MATRIX
  ) > ZERO_TOLERANCE
)

DEVIATION_SELECTED <- (
  abs(
    DEVIATION_COEF_MATRIX
  ) > ZERO_TOLERANCE
)

SHARED_SELECTION_COUNT <- rowSums(
  SHARED_SELECTED
)

DEVIATION_SELECTION_COUNT <- rowSums(
  DEVIATION_SELECTED
)

SHARED_SELECTION_FREQUENCY <- (
  SHARED_SELECTION_COUNT /
  N_MODELS
)

DEVIATION_SELECTION_FREQUENCY <- (
  DEVIATION_SELECTION_COUNT /
  N_MODELS
)


# ------------------------------------------------------------
# 7. Sign-consistency helper
#
# Calculated ONLY among models where that coefficient
# was non-zero.
# ------------------------------------------------------------

calculate_sign_metrics <- function(
  coef_matrix,
  selected_matrix
) {

  n_gene <- nrow(
    coef_matrix
  )

  dominant_sign <- character(
    n_gene
  )

  sign_consistency <- rep(
    NA_real_,
    n_gene
  )

  positive_count <- integer(
    n_gene
  )

  negative_count <- integer(
    n_gene
  )

  for (g in seq_len(n_gene)) {

    selected_idx <- which(
      selected_matrix[g, ]
    )

    if (
      length(
        selected_idx
      ) == 0L
    ) {

      dominant_sign[g] <- "Not_selected"

      next
    }

    vals <- coef_matrix[
      g,
      selected_idx
    ]

    n_positive <- sum(
      vals > 0
    )

    n_negative <- sum(
      vals < 0
    )

    positive_count[g] <- n_positive
    negative_count[g] <- n_negative

    if (n_positive > n_negative) {

      dominant_sign[g] <- "Positive"

    } else if (
      n_negative > n_positive
    ) {

      dominant_sign[g] <- "Negative"

    } else {

      dominant_sign[g] <- "Tie"
    }

    sign_consistency[g] <- (
      max(
        n_positive,
        n_negative
      ) /
      length(
        selected_idx
      )
    )
  }

  list(
    dominant_sign =
      dominant_sign,

    sign_consistency =
      sign_consistency,

    positive_count =
      positive_count,

    negative_count =
      negative_count
  )
}


# ------------------------------------------------------------
# 8. Shared sign stability
# ------------------------------------------------------------

SHARED_SIGN <- calculate_sign_metrics(
  SHARED_COEF_MATRIX,
  SHARED_SELECTED
)


# ------------------------------------------------------------
# 9. Deviation sign stability
# ------------------------------------------------------------

DEVIATION_SIGN <- calculate_sign_metrics(
  DEVIATION_COEF_MATRIX,
  DEVIATION_SELECTED
)


# ------------------------------------------------------------
# 10. Coefficient summaries
# ------------------------------------------------------------

median_nonzero <- function(x) {

  x <- x[
    abs(x) >
      ZERO_TOLERANCE
  ]

  if (
    length(x) == 0L
  ) {
    return(0)
  }

  median(x)
}


mean_nonzero <- function(x) {

  x <- x[
    abs(x) >
      ZERO_TOLERANCE
  ]

  if (
    length(x) == 0L
  ) {
    return(0)
  }

  mean(x)
}


SHARED_MEDIAN_NONZERO <- apply(
  SHARED_COEF_MATRIX,
  1,
  median_nonzero
)

SHARED_MEAN_NONZERO <- apply(
  SHARED_COEF_MATRIX,
  1,
  mean_nonzero
)

SHARED_MEAN_ABS <- rowMeans(
  abs(
    SHARED_COEF_MATRIX
  )
)

SHARED_MEDIAN_ALL <- apply(
  SHARED_COEF_MATRIX,
  1,
  median
)


DEVIATION_MEDIAN_NONZERO <- apply(
  DEVIATION_COEF_MATRIX,
  1,
  median_nonzero
)

DEVIATION_MEAN_NONZERO <- apply(
  DEVIATION_COEF_MATRIX,
  1,
  mean_nonzero
)

DEVIATION_MEAN_ABS <- rowMeans(
  abs(
    DEVIATION_COEF_MATRIX
  )
)


# ------------------------------------------------------------
# 11. Build full stability table
# ------------------------------------------------------------

STABILITY_TABLE <- data.frame(

  ensemblID_stable =
    GENE_IDS,

  gene_symbol =
    ANNOTATION$gene_symbol,

  gene_type =
    ANNOTATION$gene_type,

  chromosome =
    ANNOTATION$chromosome,

  shared_selection_count =
    SHARED_SELECTION_COUNT,

  shared_selection_frequency =
    SHARED_SELECTION_FREQUENCY,

  shared_positive_count =
    SHARED_SIGN$positive_count,

  shared_negative_count =
    SHARED_SIGN$negative_count,

  shared_dominant_sign =
    SHARED_SIGN$dominant_sign,

  shared_sign_consistency =
    SHARED_SIGN$sign_consistency,

  shared_median_nonzero_beta =
    SHARED_MEDIAN_NONZERO,

  shared_mean_nonzero_beta =
    SHARED_MEAN_NONZERO,

  shared_median_all_beta =
    SHARED_MEDIAN_ALL,

  shared_mean_absolute_beta =
    SHARED_MEAN_ABS,

  deviation_selection_count =
    DEVIATION_SELECTION_COUNT,

  deviation_selection_frequency =
    DEVIATION_SELECTION_FREQUENCY,

  deviation_positive_count =
    DEVIATION_SIGN$positive_count,

  deviation_negative_count =
    DEVIATION_SIGN$negative_count,

  deviation_dominant_sign =
    DEVIATION_SIGN$dominant_sign,

  deviation_sign_consistency =
    DEVIATION_SIGN$sign_consistency,

  deviation_median_nonzero_beta =
    DEVIATION_MEDIAN_NONZERO,

  deviation_mean_nonzero_beta =
    DEVIATION_MEAN_NONZERO,

  deviation_mean_absolute_beta =
    DEVIATION_MEAN_ABS,

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 12. Apply locked stability criteria
# ------------------------------------------------------------

STABILITY_TABLE$stable_shared <- (

  STABILITY_TABLE$shared_selection_frequency >=
    STABILITY_SELECTION_THRESHOLD

  &

  !is.na(
    STABILITY_TABLE$shared_sign_consistency
  )

  &

  STABILITY_TABLE$shared_sign_consistency >=
    SIGN_CONSISTENCY_THRESHOLD
)


STABILITY_TABLE$stable_deviation <- (

  STABILITY_TABLE$deviation_selection_frequency >=
    STABILITY_SELECTION_THRESHOLD

  &

  !is.na(
    STABILITY_TABLE$deviation_sign_consistency
  )

  &

  STABILITY_TABLE$deviation_sign_consistency >=
    SIGN_CONSISTENCY_THRESHOLD
)


STABLE_SHARED <- STABILITY_TABLE[
  STABILITY_TABLE$stable_shared,
  ,
  drop = FALSE
]

STABLE_DEVIATION <- STABILITY_TABLE[
  STABILITY_TABLE$stable_deviation,
  ,
  drop = FALSE
]


# ------------------------------------------------------------
# 13. Sort stable genes
# ------------------------------------------------------------

if (nrow(STABLE_SHARED) > 0L) {

  STABLE_SHARED <- STABLE_SHARED[
    order(
      -STABLE_SHARED$shared_selection_frequency,
      -STABLE_SHARED$shared_sign_consistency,
      -STABLE_SHARED$shared_mean_absolute_beta
    ),
    ,
    drop = FALSE
  ]

  rownames(
    STABLE_SHARED
  ) <- NULL
}


if (nrow(STABLE_DEVIATION) > 0L) {

  STABLE_DEVIATION <- STABLE_DEVIATION[
    order(
      -STABLE_DEVIATION$deviation_selection_frequency,
      -STABLE_DEVIATION$deviation_sign_consistency,
      -STABLE_DEVIATION$deviation_mean_absolute_beta
    ),
    ,
    drop = FALSE
  ]

  rownames(
    STABLE_DEVIATION
  ) <- NULL
}


# ------------------------------------------------------------
# 14. Stability-frequency distributions
# ------------------------------------------------------------

SHARED_FREQ_DISTRIBUTION <- data.frame(

  selected_in_models =
    0:25,

  n_genes =
    vapply(
      0:25,
      function(k) {

        sum(
          SHARED_SELECTION_COUNT ==
            k
        )
      },
      integer(1)
    ),

  stringsAsFactors = FALSE
)


DEVIATION_FREQ_DISTRIBUTION <- data.frame(

  selected_in_models =
    0:25,

  n_genes =
    vapply(
      0:25,
      function(k) {

        sum(
          DEVIATION_SELECTION_COUNT ==
            k
        )
      },
      integer(1)
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 15. Summary
# ------------------------------------------------------------

SUMMARY_TABLE <- data.frame(

  metric = c(

    "Number_outer_models",

    "Primary_genes",

    "Shared_selection_threshold",

    "Shared_minimum_models_required",

    "Sign_consistency_threshold",

    "Genes_selected_shared_at_least_once",

    "Genes_selected_shared_20_of_25_or_more",

    "Stable_shared_genes",

    "Genes_selected_deviation_at_least_once",

    "Genes_selected_deviation_20_of_25_or_more",

    "Stable_deviation_genes",

    "Stable_shared_positive",

    "Stable_shared_negative"
  ),

  value = c(

    N_MODELS,

    N_GENES,

    STABILITY_SELECTION_THRESHOLD,

    ceiling(
      STABILITY_SELECTION_THRESHOLD *
        N_MODELS
    ),

    SIGN_CONSISTENCY_THRESHOLD,

    sum(
      SHARED_SELECTION_COUNT > 0
    ),

    sum(
      SHARED_SELECTION_COUNT >=
        ceiling(
          STABILITY_SELECTION_THRESHOLD *
            N_MODELS
        )
    ),

    nrow(
      STABLE_SHARED
    ),

    sum(
      DEVIATION_SELECTION_COUNT > 0
    ),

    sum(
      DEVIATION_SELECTION_COUNT >=
        ceiling(
          STABILITY_SELECTION_THRESHOLD *
            N_MODELS
        )
    ),

    nrow(
      STABLE_DEVIATION
    ),

    sum(
      STABLE_SHARED$shared_dominant_sign ==
        "Positive"
    ),

    sum(
      STABLE_SHARED$shared_dominant_sign ==
        "Negative"
    )
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 16. Save
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "stability_v1"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


FULL_PATH <- file.path(
  OUT_DIR,
  "step05a_all_gene_stability_v1.csv"
)

SHARED_PATH <- file.path(
  OUT_DIR,
  "step05a_stable_shared_genes_v1.csv"
)

DEVIATION_PATH <- file.path(
  OUT_DIR,
  "step05a_stable_deviation_genes_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step05a_stability_summary_v1.csv"
)

SHARED_DIST_PATH <- file.path(
  OUT_DIR,
  "step05a_shared_selection_frequency_distribution_v1.csv"
)

DEV_DIST_PATH <- file.path(
  OUT_DIR,
  "step05a_deviation_selection_frequency_distribution_v1.csv"
)

MATRIX_PATH <- file.path(
  OUT_DIR,
  "step05a_outer_model_coefficient_matrices_v1.rds"
)

LOCK_PATH <- file.path(
  OUT_DIR,
  "step05a_stability_definition_v1.txt"
)


write.csv(
  STABILITY_TABLE,
  FULL_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  STABLE_SHARED,
  SHARED_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  STABLE_DEVIATION,
  DEVIATION_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  SUMMARY_TABLE,
  SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  SHARED_FREQ_DISTRIBUTION,
  SHARED_DIST_PATH,
  row.names = FALSE
)

write.csv(
  DEVIATION_FREQ_DISTRIBUTION,
  DEV_DIST_PATH,
  row.names = FALSE
)

saveRDS(
  list(

    model_labels =
      MODEL_LABELS,

    shared_coefficients =
      SHARED_COEF_MATRIX,

    deviation_coefficients =
      DEVIATION_COEF_MATRIX,

    shared_selected =
      SHARED_SELECTED,

    deviation_selected =
      DEVIATION_SELECTED
  ),
  MATRIX_PATH,
  compress = "gzip"
)


writeLines(
  c(

    "STEP 05A stability definition",

    paste0(
      "outer_models=",
      N_MODELS
    ),

    paste0(
      "selection_frequency_threshold=",
      STABILITY_SELECTION_THRESHOLD
    ),

    paste0(
      "minimum_selection_count=",
      ceiling(
        STABILITY_SELECTION_THRESHOLD *
          N_MODELS
      )
    ),

    paste0(
      "sign_consistency_threshold=",
      SIGN_CONSISTENCY_THRESHOLD
    ),

    "sign_consistency_denominator=outer_models_where_coefficient_is_nonzero",

    paste0(
      "zero_tolerance=",
      ZERO_TOLERANCE
    ),

    "stable_shared=selection_frequency>=0.80 AND sign_consistency>=0.90",

    "stable_deviation=selection_frequency>=0.80 AND sign_consistency>=0.90"
  ),
  LOCK_PATH
)


# ------------------------------------------------------------
# 17. Final checks
# ------------------------------------------------------------

stopifnot(
  nrow(STABILITY_TABLE) == 23740L,
  ncol(SHARED_COEF_MATRIX) == 25L,
  ncol(DEVIATION_COEF_MATRIX) == 25L,
  all(
    STABLE_SHARED$shared_selection_frequency >=
      0.80
  ),
  all(
    STABLE_SHARED$shared_sign_consistency >=
      0.90
  )
)


TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START


# ------------------------------------------------------------
# 18. Output
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 05A — OUTER-MODEL STABILITY COMPLETE\n")
cat("============================================================\n")

print(
  SUMMARY_TABLE,
  row.names = FALSE
)


cat(
  "\nStable shared genes: ",
  nrow(STABLE_SHARED),
  "\n",
  sep = ""
)

cat(
  "Stable deviation genes: ",
  nrow(STABLE_DEVIATION),
  "\n",
  sep = ""
)


if (
  nrow(
    STABLE_SHARED
  ) > 0L
) {

  cat(
    "\n--- TOP 30 STABLE SHARED GENES ---\n"
  )

  print(
    head(
      STABLE_SHARED[
        ,
        c(
          "ensemblID_stable",
          "gene_symbol",
          "gene_type",
          "shared_selection_count",
          "shared_selection_frequency",
          "shared_dominant_sign",
          "shared_sign_consistency",
          "shared_median_nonzero_beta",
          "shared_mean_absolute_beta"
        ),
        drop = FALSE
      ],
      30
    ),
    row.names = FALSE
  )
}


if (
  nrow(
    STABLE_DEVIATION
  ) > 0L
) {

  cat(
    "\n--- STABLE DEVIATION GENES ---\n"
  )

  print(
    STABLE_DEVIATION[
      ,
      c(
        "ensemblID_stable",
        "gene_symbol",
        "gene_type",
        "deviation_selection_count",
        "deviation_selection_frequency",
        "deviation_dominant_sign",
        "deviation_sign_consistency",
        "deviation_median_nonzero_beta"
      ),
      drop = FALSE
    ],
    row.names = FALSE
  )
}


cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS,
    1
  ),
  " sec\n",
  sep = ""
)

cat(
  "\nSaved stable shared signature:\n",
  SHARED_PATH,
  "\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 05A — CELL 1 PASS\n")
cat("============================================================\n")

Coefficient files found: 25
Reading 25 outer-model coefficient sets...
Coefficient alignment: PASS

STEP 05A — OUTER-MODEL STABILITY COMPLETE
                                    metric   value
                       Number_outer_models    25.0
                             Primary_genes 23740.0
                Shared_selection_threshold     0.8
            Shared_minimum_models_required    20.0
                Sign_consistency_threshold     0.9
       Genes_selected_shared_at_least_once  6131.0
    Genes_selected_shared_20_of_25_or_more   403.0
                       Stable_shared_genes   403.0
    Genes_selected_deviation_at_least_once   390.0
 Genes_selected_deviation_20_of_25_or_more     0.0
                    Stable_deviation_genes     0.0
                    Stable_shared_positive   191.0
                    Stable_shared_negative   212.0

Stable shared genes: 403
Stable deviation genes: 0

--- TOP 30 STABLE SHARED GENES ---
 ensemblID_stable gene_symbol                          g

In [9]:
# ============================================================
# STEP 05B — CELL 1
# STABLE SHARED SIGNATURE AUDIT + LOCK
#
# Goals:
#   1. Audit the 403 stable shared genes
#   2. Compare predictive direction with descriptive shared DE
#   3. Characterize gene-type composition
#   4. Freeze a full stable signature
#   5. Freeze a portable direction-only signature
#   6. Flag protein-coding subset for later sensitivity analysis
#
# IMPORTANT:
#   No genes are removed here.
#   External technical overlap will be assessed later.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"

TOTAL_START <- proc.time()[3]


# ------------------------------------------------------------
# 1. Input paths
# ------------------------------------------------------------

STABLE_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "stability_v1",
  "step05a_stable_shared_genes_v1.csv"
)

SHARED_DE_PATH <- file.path(
  PROJECT_ROOT,
  "05_differential_expression",
  "step03a_Shared_SCZ_effect_v1.csv"
)

if (!file.exists(STABLE_PATH)) {
  stop("Stable shared-gene file missing.")
}

if (!file.exists(SHARED_DE_PATH)) {
  stop("Shared DE file missing.")
}


# ------------------------------------------------------------
# 2. Load
# ------------------------------------------------------------

STABLE <- read.csv(
  STABLE_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

SHARED_DE <- read.csv(
  SHARED_DE_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(STABLE) == 403L
)

if (anyDuplicated(STABLE$ensemblID_stable)) {
  stop("Duplicate stable Ensembl IDs in stable signature.")
}

if (anyDuplicated(SHARED_DE$ensemblID_stable)) {
  stop("Duplicate stable Ensembl IDs in shared DE table.")
}

cat(
  "Stable shared genes loaded: ",
  nrow(STABLE),
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 3. Match descriptive shared-DE results
# ------------------------------------------------------------

DE_MATCH <- match(
  STABLE$ensemblID_stable,
  SHARED_DE$ensemblID_stable
)

if (anyNA(DE_MATCH)) {
  stop("Some stable shared genes are missing from shared DE results.")
}

STABLE$shared_DE_logFC <-
  SHARED_DE$logFC[
    DE_MATCH
  ]

STABLE$shared_DE_P <-
  SHARED_DE$P.Value[
    DE_MATCH
  ]

STABLE$shared_DE_FDR <-
  SHARED_DE$adj.P.Val[
    DE_MATCH
  ]


# ------------------------------------------------------------
# 4. Stable predictive direction
# ------------------------------------------------------------

STABLE$direction_code <- ifelse(
  STABLE$shared_dominant_sign == "Positive",
  1L,
  ifelse(
    STABLE$shared_dominant_sign == "Negative",
    -1L,
    NA_integer_
  )
)

if (anyNA(STABLE$direction_code)) {
  stop("Unexpected dominant-sign value in stable signature.")
}

STABLE$direction_label <- ifelse(
  STABLE$direction_code == 1L,
  "Higher_predictive_score_in_SCZ",
  "Lower_predictive_score_in_SCZ"
)


# ------------------------------------------------------------
# 5. Compare predictive direction with descriptive DE
# ------------------------------------------------------------

STABLE$DE_direction_code <- ifelse(
  STABLE$shared_DE_logFC > 0,
  1L,
  ifelse(
    STABLE$shared_DE_logFC < 0,
    -1L,
    0L
  )
)

STABLE$predictive_DE_sign_concordant <- (
  STABLE$direction_code ==
    STABLE$DE_direction_code
)

DIRECTION_CONCORDANCE <- mean(
  STABLE$predictive_DE_sign_concordant,
  na.rm = TRUE
)

N_DE_FDR05 <- sum(
  STABLE$shared_DE_FDR < 0.05,
  na.rm = TRUE
)

N_DE_FDR01 <- sum(
  STABLE$shared_DE_FDR < 0.01,
  na.rm = TRUE
)

N_DE_NOMINAL <- sum(
  STABLE$shared_DE_P < 0.05,
  na.rm = TRUE
)


# ------------------------------------------------------------
# 6. Additional stability flags
# ------------------------------------------------------------

STABLE$selected_25_of_25 <- (
  STABLE$shared_selection_count == 25L
)

STABLE$selected_24_of_25_or_more <- (
  STABLE$shared_selection_count >= 24L
)

STABLE$perfect_sign_consistency <- (
  STABLE$shared_sign_consistency == 1
)

STABLE$protein_coding <- (
  STABLE$gene_type == "protein_coding"
)


# ------------------------------------------------------------
# 7. Gene-type composition
# ------------------------------------------------------------

GENE_TYPE_SUMMARY <- as.data.frame(
  table(
    gene_type = STABLE$gene_type,
    useNA = "ifany"
  ),
  stringsAsFactors = FALSE
)

names(
  GENE_TYPE_SUMMARY
)[2] <- "n_genes"

GENE_TYPE_SUMMARY$percent <- (
  100 *
  GENE_TYPE_SUMMARY$n_genes /
  nrow(STABLE)
)

GENE_TYPE_SUMMARY <- GENE_TYPE_SUMMARY[
  order(
    -GENE_TYPE_SUMMARY$n_genes,
    GENE_TYPE_SUMMARY$gene_type
  ),
  ,
  drop = FALSE
]

rownames(
  GENE_TYPE_SUMMARY
) <- NULL


# ------------------------------------------------------------
# 8. Protein-coding sensitivity subset
#
# This is NOT replacing the primary 403-gene signature.
# It is only pre-defined here as a later sensitivity analysis.
# ------------------------------------------------------------

PROTEIN_CODING_SIGNATURE <- STABLE[
  STABLE$protein_coding,
  ,
  drop = FALSE
]

PROTEIN_CODING_SIGNATURE <- PROTEIN_CODING_SIGNATURE[
  order(
    -PROTEIN_CODING_SIGNATURE$shared_selection_frequency,
    -PROTEIN_CODING_SIGNATURE$shared_sign_consistency,
    -PROTEIN_CODING_SIGNATURE$shared_mean_absolute_beta
  ),
  ,
  drop = FALSE
]

rownames(
  PROTEIN_CODING_SIGNATURE
) <- NULL


# ------------------------------------------------------------
# 9. Full locked shared signature
#
# Primary discovery-derived signature:
# all 403 stable shared genes
# ------------------------------------------------------------

LOCKED_SIGNATURE <- STABLE[
  order(
    -STABLE$shared_selection_frequency,
    -STABLE$shared_sign_consistency,
    -STABLE$shared_mean_absolute_beta
  ),
  ,
  drop = FALSE
]

rownames(
  LOCKED_SIGNATURE
) <- NULL


# ------------------------------------------------------------
# 10. Portable direction-only signature
#
# No coefficient magnitude is required for the future
# rank-based external score.
#
# External score will use:
#   +1 for positive stable direction
#   -1 for negative stable direction
# ------------------------------------------------------------

PORTABLE_SIGNATURE <- LOCKED_SIGNATURE[
  ,
  c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "chromosome",
    "direction_code",
    "direction_label",
    "shared_selection_count",
    "shared_selection_frequency",
    "shared_sign_consistency",
    "shared_median_nonzero_beta",
    "shared_mean_absolute_beta",
    "shared_DE_logFC",
    "shared_DE_FDR",
    "predictive_DE_sign_concordant",
    "protein_coding",
    "selected_25_of_25",
    "perfect_sign_consistency"
  ),
  drop = FALSE
]


# ------------------------------------------------------------
# 11. Core descriptive summaries
# ------------------------------------------------------------

N_POSITIVE <- sum(
  LOCKED_SIGNATURE$direction_code == 1L
)

N_NEGATIVE <- sum(
  LOCKED_SIGNATURE$direction_code == -1L
)

N_PC <- nrow(
  PROTEIN_CODING_SIGNATURE
)

N_25OF25 <- sum(
  LOCKED_SIGNATURE$selected_25_of_25
)

N_24PLUS <- sum(
  LOCKED_SIGNATURE$selected_24_of_25_or_more
)

N_PERFECT_SIGN <- sum(
  LOCKED_SIGNATURE$perfect_sign_consistency
)

N_BOTH_PERFECT <- sum(
  LOCKED_SIGNATURE$selected_25_of_25 &
    LOCKED_SIGNATURE$perfect_sign_consistency
)

SUMMARY_TABLE <- data.frame(

  metric = c(
    "Stable_shared_genes",
    "Positive_direction",
    "Negative_direction",
    "Protein_coding_genes",
    "Non_protein_coding_genes",
    "Selected_25_of_25",
    "Selected_24_of_25_or_more",
    "Perfect_sign_consistency",
    "Selected_25_of_25_AND_perfect_sign",
    "Shared_DE_FDR_lt_0.05",
    "Shared_DE_FDR_lt_0.01",
    "Shared_DE_nominal_P_lt_0.05",
    "Predictive_vs_DE_direction_concordance"
  ),

  value = c(
    nrow(LOCKED_SIGNATURE),
    N_POSITIVE,
    N_NEGATIVE,
    N_PC,
    nrow(LOCKED_SIGNATURE) - N_PC,
    N_25OF25,
    N_24PLUS,
    N_PERFECT_SIGN,
    N_BOTH_PERFECT,
    N_DE_FDR05,
    N_DE_FDR01,
    N_DE_NOMINAL,
    DIRECTION_CONCORDANCE
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 12. Save
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "stable_signature_v1"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

LOCKED_PATH <- file.path(
  OUT_DIR,
  "step05b_locked_stable_shared_signature_v1.csv"
)

PORTABLE_PATH <- file.path(
  OUT_DIR,
  "step05b_portable_directional_signature_v1.csv"
)

PC_PATH <- file.path(
  OUT_DIR,
  "step05b_protein_coding_sensitivity_signature_v1.csv"
)

TYPE_PATH <- file.path(
  OUT_DIR,
  "step05b_gene_type_composition_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step05b_signature_audit_summary_v1.csv"
)

DE_CONCORDANCE_PATH <- file.path(
  OUT_DIR,
  "step05b_predictive_DE_concordance_v1.csv"
)

LOCK_TEXT_PATH <- file.path(
  OUT_DIR,
  "step05b_signature_lock_v1.txt"
)

write.csv(
  LOCKED_SIGNATURE,
  LOCKED_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  PORTABLE_SIGNATURE,
  PORTABLE_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  PROTEIN_CODING_SIGNATURE,
  PC_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  GENE_TYPE_SUMMARY,
  TYPE_PATH,
  row.names = FALSE
)

write.csv(
  SUMMARY_TABLE,
  SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  LOCKED_SIGNATURE[
    ,
    c(
      "ensemblID_stable",
      "gene_symbol",
      "gene_type",
      "direction_code",
      "shared_median_nonzero_beta",
      "shared_DE_logFC",
      "shared_DE_FDR",
      "predictive_DE_sign_concordant"
    ),
    drop = FALSE
  ],
  DE_CONCORDANCE_PATH,
  row.names = FALSE,
  na = ""
)

writeLines(
  c(
    "STEP 05B stable shared signature lock",
    "",
    "Primary signature source=25 outer-training models",
    "Primary stable shared genes=403",
    "Selection frequency threshold=0.80",
    "Minimum outer-model selections=20/25",
    "Sign consistency threshold=0.90",
    "Stable deviation genes=0",
    "",
    "Primary signature retains all eligible gene types.",
    "No gene is removed on the basis of external platform availability at this stage.",
    "Protein-coding subset is sensitivity-only.",
    "",
    "Portable score direction:",
    "+1 = stable positive shared predictive coefficient",
    "-1 = stable negative shared predictive coefficient",
    "",
    "Portable external score will use direction-only rank-based scoring.",
    "External technical overlap will be checked before phenotype/outcome evaluation."
  ),
  LOCK_TEXT_PATH
)


# ------------------------------------------------------------
# 13. Hard checks
# ------------------------------------------------------------

stopifnot(
  nrow(LOCKED_SIGNATURE) == 403L,
  nrow(PORTABLE_SIGNATURE) == 403L,
  N_POSITIVE == 191L,
  N_NEGATIVE == 212L,
  all(
    abs(
      LOCKED_SIGNATURE$direction_code
    ) == 1L
  ),
  file.exists(LOCKED_PATH),
  file.exists(PORTABLE_PATH)
)


# ------------------------------------------------------------
# 14. Output
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START

cat("\n============================================================\n")
cat("STEP 05B — STABLE SHARED SIGNATURE AUDIT + LOCK\n")
cat("============================================================\n")

cat("\n--- SIGNATURE SUMMARY ---\n")

print(
  SUMMARY_TABLE,
  row.names = FALSE
)

cat("\n--- GENE-TYPE COMPOSITION ---\n")

print(
  GENE_TYPE_SUMMARY,
  row.names = FALSE
)

cat(
  "\nPrimary locked signature: ",
  nrow(LOCKED_SIGNATURE),
  " genes\n",
  sep = ""
)

cat(
  "Protein-coding sensitivity subset: ",
  nrow(PROTEIN_CODING_SIGNATURE),
  " genes\n",
  sep = ""
)

cat(
  "Predictive vs shared-DE sign concordance: ",
  round(
    DIRECTION_CONCORDANCE,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Stable genes also Shared-DE FDR<0.05: ",
  N_DE_FDR05,
  " / ",
  nrow(LOCKED_SIGNATURE),
  "\n",
  sep = ""
)

cat(
  "Selected in all 25 models: ",
  N_25OF25,
  "\n",
  sep = ""
)

cat(
  "\n--- TOP 25 PROTEIN-CODING STABLE GENES ---\n"
)

print(
  head(
    PROTEIN_CODING_SIGNATURE[
      ,
      c(
        "ensemblID_stable",
        "gene_symbol",
        "shared_selection_count",
        "shared_selection_frequency",
        "shared_dominant_sign",
        "shared_sign_consistency",
        "shared_median_nonzero_beta",
        "shared_DE_logFC",
        "shared_DE_FDR"
      ),
      drop = FALSE
    ],
    25
  ),
  row.names = FALSE
)

cat(
  "\nSaved locked signature:\n",
  LOCKED_PATH,
  "\n",
  sep = ""
)

cat(
  "\nSaved portable directional signature:\n",
  PORTABLE_PATH,
  "\n",
  sep = ""
)

cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS,
    1
  ),
  " sec\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 05B — CELL 1 PASS\n")
cat("============================================================\n")

Stable shared genes loaded: 403

STEP 05B — STABLE SHARED SIGNATURE AUDIT + LOCK

--- SIGNATURE SUMMARY ---
                                 metric       value
                    Stable_shared_genes 403.0000000
                     Positive_direction 191.0000000
                     Negative_direction 212.0000000
                   Protein_coding_genes 129.0000000
               Non_protein_coding_genes 274.0000000
                      Selected_25_of_25  98.0000000
              Selected_24_of_25_or_more 158.0000000
               Perfect_sign_consistency 403.0000000
     Selected_25_of_25_AND_perfect_sign  98.0000000
                  Shared_DE_FDR_lt_0.05 198.0000000
                  Shared_DE_FDR_lt_0.01 138.0000000
            Shared_DE_nominal_P_lt_0.05 262.0000000
 Predictive_vs_DE_direction_concordance   0.9503722

--- GENE-TYPE COMPOSITION ---
                          gene_type n_genes    percent
                     protein_coding     129 32.0099256
                       

In [10]:
# ============================================================
# STEP 06A — CELL 1
# COMMONMIND PRIMARY EXTERNAL DATA AVAILABILITY AUDIT
#
# Primary external cohort:
#   CommonMind MSSM / Penn / Pitt DLPFC
#
# Required Synapse files:
#   Expression: syn3346752
#   Metadata:   syn3346807
#
# No phenotype analysis is performed here.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"

CMC_DIR <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "commonmind_mssm_penn_pitt"
)

dir.create(
  CMC_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

cat("\n============================================================\n")
cat("STEP 06A — COMMONMIND FILE AVAILABILITY AUDIT\n")
cat("============================================================\n")

cat(
  "Expected directory:\n",
  CMC_DIR,
  "\n\n",
  sep = ""
)

# ------------------------------------------------------------
# 1. Recursively inventory candidate CommonMind files
# ------------------------------------------------------------

SEARCH_ROOTS <- unique(
  c(
    CMC_DIR,
    file.path(PROJECT_ROOT, "01_raw"),
    file.path(PROJECT_ROOT, "02_metadata"),
    file.path(PROJECT_ROOT, "03_intermediate")
  )
)

SEARCH_ROOTS <- SEARCH_ROOTS[
  dir.exists(SEARCH_ROOTS)
]

ALL_FILES <- unlist(
  lapply(
    SEARCH_ROOTS,
    function(d) {

      list.files(
        d,
        recursive = TRUE,
        full.names = TRUE,
        include.dirs = FALSE
      )
    }
  ),
  use.names = FALSE
)

ALL_FILES <- unique(
  normalizePath(
    ALL_FILES,
    winslash = "/",
    mustWork = FALSE
  )
)

# ------------------------------------------------------------
# 2. Identify likely CMC expression / metadata candidates
# ------------------------------------------------------------

FILE_NAMES <- basename(
  ALL_FILES
)

CMC_FLAG <- grepl(
  "CMC|CommonMind|MSSM|Penn|Pitt|syn3346752|syn3346807",
  FILE_NAMES,
  ignore.case = TRUE
)

CMC_FILES <- ALL_FILES[
  CMC_FLAG
]

EXPR_FLAG <- grepl(
  "geneExpressionRawUCSC|syn3346752|expression.*raw|raw.*expression",
  basename(CMC_FILES),
  ignore.case = TRUE
)

META_FLAG <- grepl(
  "metaData|metadata|syn3346807",
  basename(CMC_FILES),
  ignore.case = TRUE
)

EXPRESSION_CANDIDATES <- CMC_FILES[
  EXPR_FLAG
]

METADATA_CANDIDATES <- CMC_FILES[
  META_FLAG
]

# ------------------------------------------------------------
# 3. Print inventory
# ------------------------------------------------------------

cat("--- CMC-LIKE FILES FOUND ---\n")

if (length(CMC_FILES) == 0L) {

  cat("NONE\n")

} else {

  FILE_INFO <- file.info(
    CMC_FILES
  )

  INVENTORY <- data.frame(

    file = basename(
      CMC_FILES
    ),

    size_MB = round(
      FILE_INFO$size / 1024^2,
      2
    ),

    path = CMC_FILES,

    stringsAsFactors = FALSE
  )

  print(
    INVENTORY,
    row.names = FALSE
  )
}

cat("\n--- EXPRESSION CANDIDATES ---\n")

if (
  length(
    EXPRESSION_CANDIDATES
  ) == 0L
) {

  cat("NONE\n")

} else {

  print(
    EXPRESSION_CANDIDATES
  )
}

cat("\n--- METADATA CANDIDATES ---\n")

if (
  length(
    METADATA_CANDIDATES
  ) == 0L
) {

  cat("NONE\n")

} else {

  print(
    METADATA_CANDIDATES
  )
}

# ------------------------------------------------------------
# 4. Write acquisition manifest
# ------------------------------------------------------------

MANIFEST <- data.frame(

  role = c(
    "Primary external expression",
    "Primary external metadata"
  ),

  cohort = c(
    "CommonMind MSSM-Penn-Pitt DLPFC",
    "CommonMind MSSM-Penn-Pitt DLPFC"
  ),

  synapse_id = c(
    "syn3346752",
    "syn3346807"
  ),

  expected_file = c(
    "CMC_MSSM-Penn-Pitt_DLPFC_mRNA_IlluminaHiSeq2500_geneExpressionRawUCSC.tsv.gz",
    "CMC_MSSM-Penn-Pitt_DLPFC_mRNA-metaData_release1.csv"
  ),

  access = c(
    "Controlled",
    "Controlled"
  ),

  stringsAsFactors = FALSE
)

MANIFEST_PATH <- file.path(
  CMC_DIR,
  "commonmind_required_files_manifest_v1.csv"
)

write.csv(
  MANIFEST,
  MANIFEST_PATH,
  row.names = FALSE
)

# ------------------------------------------------------------
# 5. Final status
# ------------------------------------------------------------

EXPRESSION_FOUND <- (
  length(
    EXPRESSION_CANDIDATES
  ) > 0L
)

METADATA_FOUND <- (
  length(
    METADATA_CANDIDATES
  ) > 0L
)

cat("\n============================================================\n")

cat(
  "Expression file detected: ",
  ifelse(
    EXPRESSION_FOUND,
    "YES",
    "NO"
  ),
  "\n",
  sep = ""
)

cat(
  "Metadata file detected:   ",
  ifelse(
    METADATA_FOUND,
    "YES",
    "NO"
  ),
  "\n",
  sep = ""
)

cat(
  "Required Synapse expression ID: syn3346752\n"
)

cat(
  "Required Synapse metadata ID:   syn3346807\n"
)

cat(
  "Target folder:\n",
  CMC_DIR,
  "\n",
  sep = ""
)

cat(
  "\nManifest saved:\n",
  MANIFEST_PATH,
  "\n",
  sep = ""
)

if (
  EXPRESSION_FOUND &&
  METADATA_FOUND
) {

  cat(
    "\nSTEP 06A STATUS: FILES AVAILABLE — READY FOR TECHNICAL PORTABILITY GATE\n"
  )

} else {

  cat(
    "\nSTEP 06A STATUS: COMMONMIND CONTROLLED FILE(S) NOT YET PRESENT\n"
  )
}

cat("============================================================\n")


STEP 06A — COMMONMIND FILE AVAILABILITY AUDIT
Expected directory:
E:/SCZ_CROSSREGION_MSS_Q1/01_raw/commonmind_mssm_penn_pitt

--- CMC-LIKE FILES FOUND ---
NONE

--- EXPRESSION CANDIDATES ---
NONE

--- METADATA CANDIDATES ---
NONE

Expression file detected: NO
Metadata file detected:   NO
Required Synapse expression ID: syn3346752
Required Synapse metadata ID:   syn3346807
Target folder:
E:/SCZ_CROSSREGION_MSS_Q1/01_raw/commonmind_mssm_penn_pitt

Manifest saved:
E:/SCZ_CROSSREGION_MSS_Q1/01_raw/commonmind_mssm_penn_pitt/commonmind_required_files_manifest_v1.csv

STEP 06A STATUS: COMMONMIND CONTROLLED FILE(S) NOT YET PRESENT


In [13]:
# ============================================================
# STEP 06B — CELL 1 (CORRECTED PARSE-SAFE VERSION)
# GSE80655 PUBLIC EXTERNAL DATA ACQUISITION
# + OUTCOME-BLIND TECHNICAL PORTABILITY AUDIT
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]


# ------------------------------------------------------------
# 0. PROJECT CHECK
# ------------------------------------------------------------

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop(
    paste0(
      "Project ID mismatch. Expected ",
      PROJECT_ID,
      "; observed ",
      OBSERVED_ID
    )
  )
}

cat("Project sentinel: PASS\n")
flush.console()


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

GSE_DIR <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse80655"
)

dir.create(
  GSE_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

GSE_URL <- paste0(
  "https://ftp.ncbi.nlm.nih.gov/geo/series/",
  "GSE80nnn/GSE80655/suppl/",
  "GSE80655_GeneExpressionData_Updated_3-26-2018.txt.gz"
)

GSE_PATH <- file.path(
  GSE_DIR,
  "GSE80655_GeneExpressionData_Updated_3-26-2018.txt.gz"
)

SIGNATURE_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "stable_signature_v1",
  "step05b_portable_directional_signature_v1.csv"
)

if (!file.exists(SIGNATURE_PATH)) {
  stop("Locked portable signature missing.")
}


# ------------------------------------------------------------
# 2. DOWNLOAD GSE80655 PROCESSED MATRIX
# ------------------------------------------------------------

if (!file.exists(GSE_PATH)) {

  cat(
    "Downloading GSE80655 processed expression matrix...\n"
  )

  flush.console()

  download.file(
    url = GSE_URL,
    destfile = GSE_PATH,
    mode = "wb",
    method = "libcurl",
    quiet = FALSE
  )

} else {

  cat(
    "GSE80655 file already exists; download skipped.\n"
  )
}

if (!file.exists(GSE_PATH)) {
  stop("GSE80655 download failed.")
}

FILE_SIZE <- file.info(
  GSE_PATH
)$size

if (
  is.na(FILE_SIZE) ||
  FILE_SIZE < 1000
) {
  stop("Downloaded GSE80655 file is invalid.")
}

cat(
  "File size: ",
  round(
    FILE_SIZE / 1024^2,
    2
  ),
  " MB\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 3. SHA256
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {

  install.packages(
    "digest",
    repos = "https://cloud.r-project.org"
  )
}

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop("digest package unavailable.")
}

GSE_SHA256 <- digest::digest(
  file = GSE_PATH,
  algo = "sha256"
)

cat(
  "SHA-256: ",
  GSE_SHA256,
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 4. READ PROCESSED MATRIX
# ------------------------------------------------------------

cat(
  "Reading GSE80655 processed matrix...\n"
)

flush.console()

GSE_RAW <- read.delim(
  gzfile(GSE_PATH),
  header = TRUE,
  stringsAsFactors = FALSE,
  check.names = FALSE,
  quote = "",
  comment.char = "",
  fill = TRUE
)

if (
  nrow(GSE_RAW) == 0L ||
  ncol(GSE_RAW) < 2L
) {
  stop(
    "GSE80655 processed matrix could not be parsed."
  )
}

cat(
  "Parsed dimensions: ",
  nrow(GSE_RAW),
  " rows x ",
  ncol(GSE_RAW),
  " columns\n",
  sep = ""
)

cat("\nFirst 12 column names:\n")

print(
  head(
    colnames(GSE_RAW),
    12
  )
)

cat(
  "\nFirst 8 rows / first 6 columns:\n"
)

print(
  head(
    GSE_RAW[
      ,
      seq_len(
        min(
          6L,
          ncol(GSE_RAW)
        )
      ),
      drop = FALSE
    ],
    8
  )
)

flush.console()


# ------------------------------------------------------------
# 5. LOAD LOCKED 403-GENE SIGNATURE
# ------------------------------------------------------------

SIGNATURE <- read.csv(
  SIGNATURE_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

if (nrow(SIGNATURE) != 403L) {

  stop(
    paste0(
      "Expected 403 locked genes; found ",
      nrow(SIGNATURE)
    )
  )
}

if (
  anyDuplicated(
    SIGNATURE$ensemblID_stable
  )
) {
  stop(
    "Duplicate stable Ensembl IDs in signature."
  )
}

SIGNATURE$ensemblID_stable <- sub(
  "\\..*$",
  "",
  as.character(
    SIGNATURE$ensemblID_stable
  )
)

SIGNATURE$direction_code <- as.integer(
  SIGNATURE$direction_code
)

if (
  anyNA(
    SIGNATURE$direction_code
  ) ||
  !all(
    SIGNATURE$direction_code %in%
      c(-1L, 1L)
  )
) {
  stop(
    "Invalid direction_code values."
  )
}

if (!is.logical(
  SIGNATURE$protein_coding
)) {

  SIGNATURE$protein_coding <- (
    toupper(
      as.character(
        SIGNATURE$protein_coding
      )
    ) == "TRUE"
  )
}

if (anyNA(
  SIGNATURE$protein_coding
)) {
  stop(
    "Invalid protein_coding values."
  )
}

cat(
  "\nLocked signature loaded: ",
  nrow(SIGNATURE),
  " genes\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 6. FIND ENSEMBL-LIKE COLUMN
# ------------------------------------------------------------

ENSEMBL_COUNT <- vapply(
  seq_len(
    ncol(GSE_RAW)
  ),
  function(j) {

    x <- as.character(
      GSE_RAW[
        ,
        j,
        drop = TRUE
      ]
    )

    sum(
      grepl(
        "^ENSG[0-9]+(\\.[0-9]+)?$",
        x
      ),
      na.rm = TRUE
    )
  },
  integer(1)
)

ENSEMBL_FRACTION <- vapply(
  seq_len(
    ncol(GSE_RAW)
  ),
  function(j) {

    x <- as.character(
      GSE_RAW[
        ,
        j,
        drop = TRUE
      ]
    )

    mean(
      grepl(
        "^ENSG[0-9]+(\\.[0-9]+)?$",
        x
      ),
      na.rm = TRUE
    )
  },
  numeric(1)
)

BEST_ENSEMBL_COLUMN <- which.max(
  ENSEMBL_COUNT
)

BEST_ENSEMBL_COUNT <- ENSEMBL_COUNT[
  BEST_ENSEMBL_COLUMN
]

BEST_ENSEMBL_FRACTION <- ENSEMBL_FRACTION[
  BEST_ENSEMBL_COLUMN
]

cat(
  "\nBest Ensembl-like column: ",
  colnames(GSE_RAW)[
    BEST_ENSEMBL_COLUMN
  ],
  "\n",
  sep = ""
)

cat(
  "Ensembl-like values: ",
  BEST_ENSEMBL_COUNT,
  " / ",
  nrow(GSE_RAW),
  "\n",
  sep = ""
)

cat(
  "Ensembl-like fraction: ",
  round(
    BEST_ENSEMBL_FRACTION,
    4
  ),
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 7. FIND GENE-SYMBOL COLUMN
# ------------------------------------------------------------

SIGNATURE_SYMBOLS <- unique(
  as.character(
    SIGNATURE$gene_symbol[
      !is.na(
        SIGNATURE$gene_symbol
      ) &
      SIGNATURE$gene_symbol != ""
    ]
  )
)

SYMBOL_MATCH_COUNT <- vapply(
  seq_len(
    ncol(GSE_RAW)
  ),
  function(j) {

    x <- as.character(
      GSE_RAW[
        ,
        j,
        drop = TRUE
      ]
    )

    length(
      intersect(
        unique(x),
        SIGNATURE_SYMBOLS
      )
    )
  },
  integer(1)
)

BEST_SYMBOL_COLUMN <- which.max(
  SYMBOL_MATCH_COUNT
)

BEST_SYMBOL_COUNT <- SYMBOL_MATCH_COUNT[
  BEST_SYMBOL_COLUMN
]

cat(
  "\nBest signature-symbol column candidate: ",
  colnames(GSE_RAW)[
    BEST_SYMBOL_COLUMN
  ],
  "\n",
  sep = ""
)

cat(
  "Stable signature symbols matched: ",
  BEST_SYMBOL_COUNT,
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 8. CHOOSE IDENTIFIER MODE
#
# IMPORTANT:
# No [[ ]] syntax is used anywhere.
# ------------------------------------------------------------

if (
  BEST_ENSEMBL_COUNT >= 1000L ||
  BEST_ENSEMBL_FRACTION >= 0.50
) {

  ID_MODE <- "ENSEMBL"

  BEST_ID_COLUMN <- BEST_ENSEMBL_COLUMN

  GSE_GENE_ID_RAW <- as.character(
    GSE_RAW[
      ,
      BEST_ID_COLUMN,
      drop = TRUE
    ]
  )

  GSE_GENE_ID <- sub(
    "\\..*$",
    "",
    GSE_GENE_ID_RAW
  )

  SIGNATURE_ID <- as.character(
    SIGNATURE$ensemblID_stable
  )

} else if (
  BEST_SYMBOL_COUNT > 0L
) {

  ID_MODE <- "GENE_SYMBOL"

  BEST_ID_COLUMN <- BEST_SYMBOL_COLUMN

  GSE_GENE_ID <- as.character(
    GSE_RAW[
      ,
      BEST_ID_COLUMN,
      drop = TRUE
    ]
  )

  SIGNATURE_ID <- as.character(
    SIGNATURE$gene_symbol
  )

} else {

  stop(
    paste0(
      "Could not resolve GSE80655 gene identifier column. ",
      "Best Ensembl count = ",
      BEST_ENSEMBL_COUNT,
      "; best signature-symbol count = ",
      BEST_SYMBOL_COUNT
    )
  )
}

cat(
  "\nIdentifier mode selected: ",
  ID_MODE,
  "\n",
  sep = ""
)

cat(
  "Identifier column: ",
  colnames(GSE_RAW)[
    BEST_ID_COLUMN
  ],
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 9. EXTERNAL IDENTIFIER AUDIT
# ------------------------------------------------------------

VALID_EXTERNAL_IDS <- GSE_GENE_ID[
  !is.na(GSE_GENE_ID) &
  GSE_GENE_ID != ""
]

N_EXTERNAL_VALID <- length(
  VALID_EXTERNAL_IDS
)

N_EXTERNAL_UNIQUE <- length(
  unique(
    VALID_EXTERNAL_IDS
  )
)

N_EXTERNAL_DUPLICATED_ROWS <- sum(
  duplicated(
    VALID_EXTERNAL_IDS
  )
)

cat(
  "\nValid external gene IDs: ",
  N_EXTERNAL_VALID,
  "\n",
  sep = ""
)

cat(
  "Unique external gene IDs: ",
  N_EXTERNAL_UNIQUE,
  "\n",
  sep = ""
)

cat(
  "Duplicated external ID rows: ",
  N_EXTERNAL_DUPLICATED_ROWS,
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 10. LOCKED SIGNATURE TECHNICAL OVERLAP
# ------------------------------------------------------------

EXTERNAL_ID_SET <- unique(
  VALID_EXTERNAL_IDS
)

SIGNATURE_MEASURABLE <- (
  !is.na(
    SIGNATURE_ID
  ) &
  SIGNATURE_ID != "" &
  SIGNATURE_ID %in%
    EXTERNAL_ID_SET
)

N_OVERLAP <- sum(
  SIGNATURE_MEASURABLE
)

OVERLAP_FRACTION <- (
  N_OVERLAP /
  nrow(SIGNATURE)
)


# ------------------------------------------------------------
# 11. PROTEIN-CODING OVERLAP
# ------------------------------------------------------------

N_PC_TOTAL <- sum(
  SIGNATURE$protein_coding
)

N_PC_OVERLAP <- sum(
  SIGNATURE_MEASURABLE &
  SIGNATURE$protein_coding
)

PC_OVERLAP_FRACTION <- if (
  N_PC_TOTAL > 0L
) {

  N_PC_OVERLAP /
    N_PC_TOTAL

} else {

  NA_real_
}


# ------------------------------------------------------------
# 12. POSITIVE / NEGATIVE DIRECTION OVERLAP
# ------------------------------------------------------------

N_POS_TOTAL <- sum(
  SIGNATURE$direction_code == 1L
)

N_NEG_TOTAL <- sum(
  SIGNATURE$direction_code == -1L
)

N_POS_OVERLAP <- sum(
  SIGNATURE_MEASURABLE &
  SIGNATURE$direction_code == 1L
)

N_NEG_OVERLAP <- sum(
  SIGNATURE_MEASURABLE &
  SIGNATURE$direction_code == -1L
)

POS_OVERLAP_FRACTION <- (
  N_POS_OVERLAP /
  N_POS_TOTAL
)

NEG_OVERLAP_FRACTION <- (
  N_NEG_OVERLAP /
  N_NEG_TOTAL
)


# ------------------------------------------------------------
# 13. GENE-TYPE OVERLAP
# ------------------------------------------------------------

GENE_TYPES <- sort(
  unique(
    as.character(
      SIGNATURE$gene_type
    )
  )
)

GENE_TYPE_OVERLAP <- do.call(
  rbind,
  lapply(
    GENE_TYPES,
    function(gt) {

      idx <- (
        SIGNATURE$gene_type == gt
      )

      total <- sum(
        idx
      )

      measurable <- sum(
        idx &
        SIGNATURE_MEASURABLE
      )

      data.frame(
        gene_type = gt,
        signature_genes = total,
        measurable_genes = measurable,
        measurable_fraction = if (
          total > 0
        ) {
          measurable / total
        } else {
          NA_real_
        },
        stringsAsFactors = FALSE
      )
    }
  )
)

GENE_TYPE_OVERLAP <- GENE_TYPE_OVERLAP[
  order(
    -GENE_TYPE_OVERLAP$signature_genes,
    GENE_TYPE_OVERLAP$gene_type
  ),
  ,
  drop = FALSE
]

rownames(
  GENE_TYPE_OVERLAP
) <- NULL


# ------------------------------------------------------------
# 14. GENE-LEVEL OVERLAP TABLE
# ------------------------------------------------------------

OVERLAP_TABLE <- SIGNATURE

OVERLAP_TABLE$external_identifier_mode <-
  ID_MODE

OVERLAP_TABLE$external_identifier <-
  SIGNATURE_ID

OVERLAP_TABLE$measurable_in_GSE80655 <-
  SIGNATURE_MEASURABLE


# ------------------------------------------------------------
# 15. TECHNICAL GATE STATUS
# ------------------------------------------------------------

TECHNICAL_GATE_STATUS <- if (
  OVERLAP_FRACTION >= 0.80
) {

  "HIGH_OVERLAP"

} else if (
  OVERLAP_FRACTION >= 0.50
) {

  "MODERATE_OVERLAP"

} else {

  "LOW_OVERLAP"
}


# ------------------------------------------------------------
# 16. SUMMARY
# ------------------------------------------------------------

SUMMARY_TABLE <- data.frame(

  metric = c(
    "Processed_rows",
    "Processed_columns",
    "Identifier_mode",
    "Identifier_column",
    "External_valid_gene_ids",
    "External_unique_gene_ids",
    "External_duplicate_ID_rows",
    "Stable_signature_genes",
    "Stable_signature_overlap",
    "Stable_signature_overlap_fraction",
    "Protein_coding_signature_genes",
    "Protein_coding_overlap",
    "Protein_coding_overlap_fraction",
    "Positive_signature_genes",
    "Positive_overlap",
    "Positive_overlap_fraction",
    "Negative_signature_genes",
    "Negative_overlap",
    "Negative_overlap_fraction",
    "Technical_gate_status"
  ),

  value = c(
    as.character(
      nrow(GSE_RAW)
    ),
    as.character(
      ncol(GSE_RAW)
    ),
    ID_MODE,
    colnames(GSE_RAW)[
      BEST_ID_COLUMN
    ],
    as.character(
      N_EXTERNAL_VALID
    ),
    as.character(
      N_EXTERNAL_UNIQUE
    ),
    as.character(
      N_EXTERNAL_DUPLICATED_ROWS
    ),
    as.character(
      nrow(SIGNATURE)
    ),
    as.character(
      N_OVERLAP
    ),
    format(
      OVERLAP_FRACTION,
      digits = 8
    ),
    as.character(
      N_PC_TOTAL
    ),
    as.character(
      N_PC_OVERLAP
    ),
    format(
      PC_OVERLAP_FRACTION,
      digits = 8
    ),
    as.character(
      N_POS_TOTAL
    ),
    as.character(
      N_POS_OVERLAP
    ),
    format(
      POS_OVERLAP_FRACTION,
      digits = 8
    ),
    as.character(
      N_NEG_TOTAL
    ),
    as.character(
      N_NEG_OVERLAP
    ),
    format(
      NEG_OVERLAP_FRACTION,
      digits = 8
    ),
    TECHNICAL_GATE_STATUS
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 17. IDENTIFIER COLUMN AUDIT
# ------------------------------------------------------------

IDENTIFIER_AUDIT <- data.frame(
  column_index = seq_len(
    ncol(GSE_RAW)
  ),
  column_name = colnames(
    GSE_RAW
  ),
  ensembl_like_count =
    ENSEMBL_COUNT,
  ensembl_like_fraction =
    ENSEMBL_FRACTION,
  signature_symbol_matches =
    SYMBOL_MATCH_COUNT,
  stringsAsFactors = FALSE
)

IDENTIFIER_AUDIT <- IDENTIFIER_AUDIT[
  order(
    -IDENTIFIER_AUDIT$ensembl_like_count,
    -IDENTIFIER_AUDIT$signature_symbol_matches
  ),
  ,
  drop = FALSE
]


# ------------------------------------------------------------
# 18. SAVE OUTCOME-BLIND TECHNICAL AUDIT
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "07_external_validation",
  "gse80655"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

OVERLAP_PATH <- file.path(
  OUT_DIR,
  "step06b_GSE80655_signature_technical_overlap_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step06b_GSE80655_technical_gate_summary_v1.csv"
)

TYPE_PATH <- file.path(
  OUT_DIR,
  "step06b_GSE80655_gene_type_overlap_v1.csv"
)

IDENTIFIER_AUDIT_PATH <- file.path(
  OUT_DIR,
  "step06b_GSE80655_identifier_column_audit_v1.csv"
)

INVENTORY_PATH <- file.path(
  OUT_DIR,
  "step06b_GSE80655_raw_file_inventory_v1.csv"
)

LOCK_PATH <- file.path(
  OUT_DIR,
  "step06b_GSE80655_outcome_blind_gate_v1.txt"
)

write.csv(
  OVERLAP_TABLE,
  OVERLAP_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  SUMMARY_TABLE,
  SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  GENE_TYPE_OVERLAP,
  TYPE_PATH,
  row.names = FALSE
)

write.csv(
  IDENTIFIER_AUDIT,
  IDENTIFIER_AUDIT_PATH,
  row.names = FALSE
)

write.csv(
  data.frame(
    dataset = "GSE80655",
    source_file = basename(
      GSE_PATH
    ),
    source_url = GSE_URL,
    size_bytes = FILE_SIZE,
    sha256 = GSE_SHA256,
    stringsAsFactors = FALSE
  ),
  INVENTORY_PATH,
  row.names = FALSE
)

writeLines(
  c(
    "STEP 06B — GSE80655 outcome-blind technical gate",
    "",
    paste0(
      "source_file=",
      basename(GSE_PATH)
    ),
    paste0(
      "sha256=",
      GSE_SHA256
    ),
    paste0(
      "processed_rows=",
      nrow(GSE_RAW)
    ),
    paste0(
      "processed_columns=",
      ncol(GSE_RAW)
    ),
    paste0(
      "identifier_mode=",
      ID_MODE
    ),
    paste0(
      "identifier_column=",
      colnames(GSE_RAW)[BEST_ID_COLUMN]
    ),
    paste0(
      "locked_signature_genes=",
      nrow(SIGNATURE)
    ),
    paste0(
      "technical_overlap_genes=",
      N_OVERLAP
    ),
    paste0(
      "technical_overlap_fraction=",
      OVERLAP_FRACTION
    ),
    paste0(
      "protein_coding_overlap=",
      N_PC_OVERLAP,
      "/",
      N_PC_TOTAL
    ),
    paste0(
      "positive_overlap=",
      N_POS_OVERLAP,
      "/",
      N_POS_TOTAL
    ),
    paste0(
      "negative_overlap=",
      N_NEG_OVERLAP,
      "/",
      N_NEG_TOTAL
    ),
    paste0(
      "technical_gate_status=",
      TECHNICAL_GATE_STATUS
    ),
    "",
    "diagnosis_labels_used=FALSE",
    "clinical_outcome_evaluated=FALSE",
    "external_score_calculated=FALSE"
  ),
  LOCK_PATH
)


# ------------------------------------------------------------
# 19. HARD CHECKS
# ------------------------------------------------------------

stopifnot(
  nrow(SIGNATURE) == 403L,
  N_OVERLAP >= 0L,
  N_OVERLAP <= 403L,
  N_PC_OVERLAP <= N_PC_TOTAL,
  N_POS_OVERLAP <= N_POS_TOTAL,
  N_NEG_OVERLAP <= N_NEG_TOTAL,
  file.exists(OVERLAP_PATH),
  file.exists(SUMMARY_PATH),
  file.exists(LOCK_PATH)
)


# ------------------------------------------------------------
# 20. FINAL OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START

cat("\n============================================================\n")
cat("STEP 06B — GSE80655 OUTCOME-BLIND TECHNICAL GATE\n")
cat("============================================================\n")

cat("\n--- TECHNICAL SUMMARY ---\n")

print(
  SUMMARY_TABLE,
  row.names = FALSE
)

cat("\n--- GENE-TYPE OVERLAP ---\n")

print(
  GENE_TYPE_OVERLAP,
  row.names = FALSE
)

cat(
  "\nStable signature measurable: ",
  N_OVERLAP,
  " / ",
  nrow(SIGNATURE),
  " (",
  round(
    100 * OVERLAP_FRACTION,
    1
  ),
  "%)\n",
  sep = ""
)

cat(
  "Protein-coding measurable:   ",
  N_PC_OVERLAP,
  " / ",
  N_PC_TOTAL,
  " (",
  round(
    100 * PC_OVERLAP_FRACTION,
    1
  ),
  "%)\n",
  sep = ""
)

cat(
  "Positive-direction overlap:  ",
  N_POS_OVERLAP,
  " / ",
  N_POS_TOTAL,
  " (",
  round(
    100 * POS_OVERLAP_FRACTION,
    1
  ),
  "%)\n",
  sep = ""
)

cat(
  "Negative-direction overlap:  ",
  N_NEG_OVERLAP,
  " / ",
  N_NEG_TOTAL,
  " (",
  round(
    100 * NEG_OVERLAP_FRACTION,
    1
  ),
  "%)\n",
  sep = ""
)

cat(
  "Technical gate status:       ",
  TECHNICAL_GATE_STATUS,
  "\n",
  sep = ""
)

cat("\nOutcome labels used:          NO\n")
cat("Clinical performance tested: NO\n")
cat("External score calculated:   NO\n")

cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS,
    1
  ),
  " sec\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 06B — CELL 1 PASS\n")
cat("============================================================\n")

Project sentinel: PASS
File size: 14.27 MB
SHA-256: 56b6112048fe494a78b28d0fe123d2635ad12fb261896efa6c2885065c239d9b
Reading GSE80655 processed matrix...
Parsed dimensions: 57905 rows x 282 columns

First 12 column names:
 [1] "gene_id" "SL31501" "SL31502" "SL31503" "SL31504" "SL31505" "SL31506"
 [8] "SL31507" "SL31508" "SL32225" "SL32226" "SL32227"

First 8 rows / first 6 columns:
          gene_id SL31501 SL31502 SL31503 SL31504 SL31505
1 ENSG00000223972       0       0       0       0       0
2 ENSG00000227232      93      80     116      50     233
3 ENSG00000243485       0       0       0       0       0
4 ENSG00000237613       0       0       0       0       0
5 ENSG00000268020       1       0       0       1       0
6 ENSG00000240361       0       0       0       1       0
7 ENSG00000186092       0       0       0       0       0
8 ENSG00000238009       5       3       4       6       2

Locked signature loaded: 403 genes

Best Ensembl-like column: gene_id
Ensembl-like values: 5

In [14]:
# ============================================================
# STEP 06C — CELL 1
# GSE80655 SAMPLE METADATA RECOVERY
# + DLPFC SCHIZOPHRENIA / CONTROL COHORT LOCK
#
# Technical portability gate already completed.
#
# THIS CELL:
#   - downloads GEO family SOFT metadata
#   - parses all sample metadata
#   - maps GEO samples to the 281 expression columns
#   - restricts to DLPFC
#   - retains Schizophrenia + Control only
#   - checks adult status
#   - freezes the external validation cohort
#
# NO external score or AUROC is calculated here.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]


# ------------------------------------------------------------
# 0. PROJECT CHECK
# ------------------------------------------------------------

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {

  stop(
    paste0(
      "Project ID mismatch. Expected ",
      PROJECT_ID,
      "; observed ",
      OBSERVED_ID
    )
  )
}

cat("Project sentinel: PASS\n")
flush.console()


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

GSE_DIR <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse80655"
)

EXPR_PATH <- file.path(
  GSE_DIR,
  "GSE80655_GeneExpressionData_Updated_3-26-2018.txt.gz"
)

SOFT_URL <- paste0(
  "https://ftp.ncbi.nlm.nih.gov/geo/series/",
  "GSE80nnn/GSE80655/soft/",
  "GSE80655_family.soft.gz"
)

SOFT_PATH <- file.path(
  GSE_DIR,
  "GSE80655_family.soft.gz"
)

if (!file.exists(EXPR_PATH)) {
  stop("GSE80655 expression file missing.")
}


# ------------------------------------------------------------
# 2. DOWNLOAD GEO FAMILY SOFT
# ------------------------------------------------------------

if (!file.exists(SOFT_PATH)) {

  cat("Downloading GSE80655 GEO family metadata...\n")
  flush.console()

  download.file(
    url = SOFT_URL,
    destfile = SOFT_PATH,
    mode = "wb",
    method = "libcurl",
    quiet = FALSE
  )

} else {

  cat("GSE80655 family SOFT already exists; download skipped.\n")
}

if (!file.exists(SOFT_PATH)) {
  stop("GSE80655 family SOFT download failed.")
}

SOFT_SIZE <- file.info(
  SOFT_PATH
)$size

cat(
  "SOFT size: ",
  round(
    SOFT_SIZE / 1024^2,
    2
  ),
  " MB\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 3. SHA256
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {

  install.packages(
    "digest",
    repos = "https://cloud.r-project.org"
  )
}

SOFT_SHA256 <- digest::digest(
  file = SOFT_PATH,
  algo = "sha256"
)

EXPR_SHA256 <- digest::digest(
  file = EXPR_PATH,
  algo = "sha256"
)

cat(
  "SOFT SHA-256: ",
  SOFT_SHA256,
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 4. READ EXPRESSION HEADER ONLY
# ------------------------------------------------------------

cat("Reading expression header...\n")
flush.console()

EXPR_HEADER <- read.delim(
  gzfile(EXPR_PATH),
  header = TRUE,
  nrows = 1,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

EXPR_COLUMNS <- colnames(
  EXPR_HEADER
)

if (
  length(EXPR_COLUMNS) != 282L
) {

  stop(
    paste0(
      "Expected 282 expression columns; found ",
      length(EXPR_COLUMNS)
    )
  )
}

if (
  EXPR_COLUMNS[1] != "gene_id"
) {
  stop("First expression column is not gene_id.")
}

EXPR_SAMPLE_IDS <- EXPR_COLUMNS[-1]

if (
  length(EXPR_SAMPLE_IDS) != 281L ||
  anyDuplicated(EXPR_SAMPLE_IDS)
) {
  stop("Expression sample IDs are invalid.")
}

cat(
  "Expression samples: ",
  length(EXPR_SAMPLE_IDS),
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 5. READ GEO SOFT
# ------------------------------------------------------------

cat("Reading GEO family SOFT metadata...\n")
flush.console()

SOFT_LINES <- readLines(
  gzfile(SOFT_PATH),
  warn = FALSE
)

if (
  length(SOFT_LINES) == 0L
) {
  stop("GEO SOFT file is empty.")
}

SAMPLE_START <- grep(
  "^\\^SAMPLE = ",
  SOFT_LINES
)

if (
  length(SAMPLE_START) == 0L
) {
  stop("No SAMPLE blocks found in GEO SOFT.")
}

SAMPLE_END <- c(
  SAMPLE_START[-1] - 1L,
  length(SOFT_LINES)
)

cat(
  "GEO SAMPLE blocks found: ",
  length(SAMPLE_START),
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 6. SOFT PARSER HELPERS
# ------------------------------------------------------------

soft_value <- function(
  block,
  prefix
) {

  hit <- which(
    startsWith(
      block,
      prefix
    )
  )

  if (
    length(hit) == 0L
  ) {
    return(
      NA_character_
    )
  }

  value <- sub(
    prefix,
    "",
    block[hit[1]],
    fixed = TRUE
  )

  trimws(
    value
  )
}


characteristic_value <- function(
  characteristic_lines,
  wanted_key
) {

  if (
    length(
      characteristic_lines
    ) == 0L
  ) {
    return(
      NA_character_
    )
  }

  wanted_key <- tolower(
    trimws(
      wanted_key
    )
  )

  for (
    value in characteristic_lines
  ) {

    colon_position <- regexpr(
      ":",
      value,
      fixed = TRUE
    )[1]

    if (
      colon_position <= 0L
    ) {
      next
    }

    key <- substr(
      value,
      1L,
      colon_position - 1L
    )

    val <- substr(
      value,
      colon_position + 1L,
      nchar(value)
    )

    key <- tolower(
      trimws(key)
    )

    val <- trimws(
      val
    )

    if (
      identical(
        key,
        wanted_key
      )
    ) {
      return(
        val
      )
    }
  }

  NA_character_
}


extract_sl_id <- function(
  description,
  title
) {

  if (
    !is.na(description) &&
    grepl(
      "^SL[0-9]+$",
      description
    )
  ) {
    return(
      description
    )
  }

  if (
    is.na(title)
  ) {
    return(
      NA_character_
    )
  }

  match_position <- regexpr(
    "SL[0-9]+",
    title
  )

  if (
    match_position[1] <= 0L
  ) {
    return(
      NA_character_
    )
  }

  match_length <- attr(
    match_position,
    "match.length"
  )[1]

  substr(
    title,
    match_position[1],
    match_position[1] +
      match_length - 1L
  )
}


# ------------------------------------------------------------
# 7. PARSE ALL SAMPLE BLOCKS
# ------------------------------------------------------------

cat("Parsing sample metadata...\n")
flush.console()

SAMPLE_METADATA_LIST <- vector(
  "list",
  length(SAMPLE_START)
)


for (
  i in seq_along(
    SAMPLE_START
  )
) {

  BLOCK <- SOFT_LINES[
    SAMPLE_START[i]:
      SAMPLE_END[i]
  ]

  GEO_ACCESSION <- soft_value(
    BLOCK,
    "!Sample_geo_accession = "
  )

  TITLE <- soft_value(
    BLOCK,
    "!Sample_title = "
  )

  SOURCE_NAME <- soft_value(
    BLOCK,
    "!Sample_source_name_ch1 = "
  )

  DESCRIPTION <- soft_value(
    BLOCK,
    "!Sample_description = "
  )

  CHARACTERISTIC_LINES <- BLOCK[
    startsWith(
      BLOCK,
      "!Sample_characteristics_ch1 = "
    )
  ]

  CHARACTERISTIC_VALUES <- sub(
    "!Sample_characteristics_ch1 = ",
    "",
    CHARACTERISTIC_LINES,
    fixed = TRUE
  )

  EXPRESSION_SAMPLE_ID <- extract_sl_id(
    description = DESCRIPTION,
    title = TITLE
  )

  DONOR_ID <- if (
    !is.na(TITLE)
  ) {

    sub(
      "_.*$",
      "",
      TITLE
    )

  } else {

    NA_character_
  }

  SAMPLE_METADATA_LIST[
    i
  ] <- list(
    data.frame(

      geo_accession =
        GEO_ACCESSION,

      title =
        TITLE,

      donor_id =
        DONOR_ID,

      expression_sample_id =
        EXPRESSION_SAMPLE_ID,

      source_name =
        SOURCE_NAME,

      tissue =
        characteristic_value(
          CHARACTERISTIC_VALUES,
          "tissue"
        ),

      brain_region =
        characteristic_value(
          CHARACTERISTIC_VALUES,
          "brain region"
        ),

      age_at_death =
        characteristic_value(
          CHARACTERISTIC_VALUES,
          "age at death"
        ),

      clinical_diagnosis =
        characteristic_value(
          CHARACTERISTIC_VALUES,
          "clinical diagnosis"
        ),

      gender =
        characteristic_value(
          CHARACTERISTIC_VALUES,
          "gender"
        ),

      ethnicity =
        characteristic_value(
          CHARACTERISTIC_VALUES,
          "ethnicity"
        ),

      postmortem_interval =
        characteristic_value(
          CHARACTERISTIC_VALUES,
          "post-mortem interval"
        ),

      brain_ph =
        characteristic_value(
          CHARACTERISTIC_VALUES,
          "brain ph"
        ),

      stringsAsFactors = FALSE
    )
  )
}


GEO_META <- do.call(
  rbind,
  SAMPLE_METADATA_LIST
)

rownames(
  GEO_META
) <- NULL


cat(
  "Parsed metadata rows: ",
  nrow(GEO_META),
  "\n",
  sep = ""
)

flush.console()


# ------------------------------------------------------------
# 8. BASIC METADATA VALIDATION
# ------------------------------------------------------------

if (
  anyDuplicated(
    GEO_META$geo_accession
  )
) {
  stop("Duplicated GEO accessions detected.")
}

if (
  anyDuplicated(
    GEO_META$expression_sample_id[
      !is.na(
        GEO_META$expression_sample_id
      )
    ]
  )
) {
  stop("Duplicated expression sample IDs detected.")
}


# ------------------------------------------------------------
# 9. MAP GEO METADATA TO EXPRESSION COLUMNS
# ------------------------------------------------------------

GEO_META$present_in_expression <- (
  GEO_META$expression_sample_id %in%
    EXPR_SAMPLE_IDS
)

N_META_MAPPED <- sum(
  GEO_META$present_in_expression
)

EXPR_WITH_META <- sum(
  EXPR_SAMPLE_IDS %in%
    GEO_META$expression_sample_id
)

cat(
  "Metadata samples mapped to expression: ",
  N_META_MAPPED,
  " / ",
  nrow(GEO_META),
  "\n",
  sep = ""
)

cat(
  "Expression columns mapped to metadata: ",
  EXPR_WITH_META,
  " / ",
  length(EXPR_SAMPLE_IDS),
  "\n",
  sep = ""
)


if (
  EXPR_WITH_META != 281L
) {

  MISSING_META <- setdiff(
    EXPR_SAMPLE_IDS,
    GEO_META$expression_sample_id
  )

  cat(
    "\nExpression IDs missing metadata:\n"
  )

  print(
    MISSING_META
  )

  stop(
    "Not all expression samples mapped to GEO metadata."
  )
}

flush.console()


# ------------------------------------------------------------
# 10. CONVERT NUMERIC CLINICAL VARIABLES
# ------------------------------------------------------------

to_numeric_safe <- function(x) {

  x <- trimws(
    as.character(x)
  )

  x[
    x %in%
      c(
        "",
        "NA",
        "N/A",
        "na",
        "n/a"
      )
  ] <- NA_character_

  suppressWarnings(
    as.numeric(x)
  )
}


GEO_META$age_at_death_numeric <-
  to_numeric_safe(
    GEO_META$age_at_death
  )

GEO_META$postmortem_interval_numeric <-
  to_numeric_safe(
    GEO_META$postmortem_interval
  )

GEO_META$brain_ph_numeric <-
  to_numeric_safe(
    GEO_META$brain_ph
  )


# ------------------------------------------------------------
# 11. FULL DATASET STRUCTURE AUDIT
# ------------------------------------------------------------

cat(
  "\n--- ALL GSE80655 BRAIN REGION x DIAGNOSIS ---\n"
)

print(
  as.data.frame(
    table(
      GEO_META$brain_region,
      GEO_META$clinical_diagnosis,
      useNA = "ifany"
    )
  ),
  row.names = FALSE
)


# ------------------------------------------------------------
# 12. DEFINE EXTERNAL VALIDATION COHORT
#
# Required:
#   brain region = DLPFC
#   diagnosis = Control or Schizophrenia
#   expression column present
# ------------------------------------------------------------

REGION_NORMALIZED <- tolower(
  trimws(
    GEO_META$brain_region
  )
)

DX_NORMALIZED <- tolower(
  trimws(
    GEO_META$clinical_diagnosis
  )
)


COHORT_FLAG <- (
  REGION_NORMALIZED == "dlpfc"
  &
  DX_NORMALIZED %in%
    c(
      "control",
      "schizophrenia"
    )
  &
  GEO_META$present_in_expression
)


GSE_COHORT <- GEO_META[
  COHORT_FLAG,
  ,
  drop = FALSE
]


if (
  nrow(GSE_COHORT) == 0L
) {
  stop(
    "No DLPFC Schizophrenia/Control samples identified."
  )
}


# ------------------------------------------------------------
# 13. STANDARDIZE DIAGNOSIS
# ------------------------------------------------------------

GSE_COHORT$Dx <- ifelse(
  tolower(
    GSE_COHORT$clinical_diagnosis
  ) == "schizophrenia",
  "Schizo",
  "Control"
)

GSE_COHORT$Dx <- factor(
  GSE_COHORT$Dx,
  levels = c(
    "Control",
    "Schizo"
  )
)


# ------------------------------------------------------------
# 14. ADULT POPULATION CHECK
# ------------------------------------------------------------

GSE_COHORT$adult <- (
  is.na(
    GSE_COHORT$age_at_death_numeric
  ) |
  GSE_COHORT$age_at_death_numeric > 17
)

N_NONADULT <- sum(
  !GSE_COHORT$adult,
  na.rm = TRUE
)

if (
  N_NONADULT > 0L
) {

  print(
    GSE_COHORT[
      !GSE_COHORT$adult,
      c(
        "geo_accession",
        "expression_sample_id",
        "age_at_death_numeric",
        "Dx"
      ),
      drop = FALSE
    ]
  )

  stop(
    "Non-adult sample(s) detected in external cohort."
  )
}


# ------------------------------------------------------------
# 15. COHORT INTEGRITY
# ------------------------------------------------------------

if (
  anyNA(
    GSE_COHORT$expression_sample_id
  )
) {
  stop(
    "Missing expression ID in external cohort."
  )
}

if (
  anyDuplicated(
    GSE_COHORT$expression_sample_id
  )
) {
  stop(
    "Duplicated expression sample IDs in external cohort."
  )
}

if (
  anyDuplicated(
    GSE_COHORT$donor_id
  )
) {
  stop(
    "Duplicated donor IDs within DLPFC external cohort."
  )
}

if (
  !setequal(
    unique(
      as.character(
        GSE_COHORT$Dx
      )
    ),
    c(
      "Control",
      "Schizo"
    )
  )
) {
  stop(
    "Both Control and Schizophrenia are required."
  )
}


# ------------------------------------------------------------
# 16. PRESERVE EXPRESSION-MATRIX SAMPLE ORDER
# ------------------------------------------------------------

EXPR_ORDER_INDEX <- match(
  GSE_COHORT$expression_sample_id,
  EXPR_SAMPLE_IDS
)

if (
  anyNA(
    EXPR_ORDER_INDEX
  )
) {
  stop(
    "External cohort sample-order mapping failed."
  )
}

GSE_COHORT$expression_column_index <-
  EXPR_ORDER_INDEX + 1L


GSE_COHORT <- GSE_COHORT[
  order(
    GSE_COHORT$expression_column_index
  ),
  ,
  drop = FALSE
]

rownames(
  GSE_COHORT
) <- NULL


# ------------------------------------------------------------
# 17. COHORT SUMMARIES
# ------------------------------------------------------------

DX_COUNTS <- as.data.frame(
  table(
    GSE_COHORT$Dx
  ),
  stringsAsFactors = FALSE
)

names(
  DX_COUNTS
) <- c(
  "Dx",
  "N"
)


SEX_COUNTS <- as.data.frame(
  table(
    GSE_COHORT$Dx,
    GSE_COHORT$gender,
    useNA = "ifany"
  ),
  stringsAsFactors = FALSE
)

names(
  SEX_COUNTS
) <- c(
  "Dx",
  "gender",
  "N"
)


CONTINUOUS_SUMMARY <- do.call(
  rbind,
  lapply(
    levels(
      GSE_COHORT$Dx
    ),
    function(dx) {

      z <- GSE_COHORT[
        GSE_COHORT$Dx == dx,
        ,
        drop = FALSE
      ]

      data.frame(

        Dx = dx,

        N = nrow(z),

        age_mean = mean(
          z$age_at_death_numeric,
          na.rm = TRUE
        ),

        age_sd = sd(
          z$age_at_death_numeric,
          na.rm = TRUE
        ),

        age_min = min(
          z$age_at_death_numeric,
          na.rm = TRUE
        ),

        age_max = max(
          z$age_at_death_numeric,
          na.rm = TRUE
        ),

        PMI_mean = mean(
          z$postmortem_interval_numeric,
          na.rm = TRUE
        ),

        PMI_sd = sd(
          z$postmortem_interval_numeric,
          na.rm = TRUE
        ),

        brain_pH_mean = mean(
          z$brain_ph_numeric,
          na.rm = TRUE
        ),

        brain_pH_sd = sd(
          z$brain_ph_numeric,
          na.rm = TRUE
        ),

        stringsAsFactors = FALSE
      )
    }
  )
)


# ------------------------------------------------------------
# 18. SAMPLE-ORDER MANIFEST
# ------------------------------------------------------------

SAMPLE_ORDER <- data.frame(

  external_row_index =
    seq_len(
      nrow(GSE_COHORT)
    ),

  expression_column_index =
    GSE_COHORT$expression_column_index,

  expression_sample_id =
    GSE_COHORT$expression_sample_id,

  geo_accession =
    GSE_COHORT$geo_accession,

  donor_id =
    GSE_COHORT$donor_id,

  Dx =
    as.character(
      GSE_COHORT$Dx
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 19. SAVE LOCKED COHORT
# ------------------------------------------------------------

META_OUT_DIR <- file.path(
  PROJECT_ROOT,
  "02_metadata"
)

VALIDATION_DIR <- file.path(
  PROJECT_ROOT,
  "07_external_validation",
  "gse80655"
)

dir.create(
  META_OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

dir.create(
  VALIDATION_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


FULL_META_PATH <- file.path(
  META_OUT_DIR,
  "gse80655_full_sample_metadata_v1.csv"
)

COHORT_PATH <- file.path(
  VALIDATION_DIR,
  "step06c_GSE80655_DLPFC_SCZ_Control_cohort_v1.csv"
)

ORDER_PATH <- file.path(
  VALIDATION_DIR,
  "step06c_GSE80655_DLPFC_sample_order_v1.csv"
)

DX_COUNT_PATH <- file.path(
  VALIDATION_DIR,
  "step06c_GSE80655_DLPFC_diagnosis_counts_v1.csv"
)

SEX_COUNT_PATH <- file.path(
  VALIDATION_DIR,
  "step06c_GSE80655_DLPFC_sex_counts_v1.csv"
)

CONTINUOUS_PATH <- file.path(
  VALIDATION_DIR,
  "step06c_GSE80655_DLPFC_continuous_summary_v1.csv"
)

LOCK_PATH <- file.path(
  VALIDATION_DIR,
  "step06c_GSE80655_DLPFC_cohort_lock_v1.txt"
)


write.csv(
  GEO_META,
  FULL_META_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  GSE_COHORT,
  COHORT_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  SAMPLE_ORDER,
  ORDER_PATH,
  row.names = FALSE
)

write.csv(
  DX_COUNTS,
  DX_COUNT_PATH,
  row.names = FALSE
)

write.csv(
  SEX_COUNTS,
  SEX_COUNT_PATH,
  row.names = FALSE
)

write.csv(
  CONTINUOUS_SUMMARY,
  CONTINUOUS_PATH,
  row.names = FALSE
)


writeLines(
  c(
    "STEP 06C — GSE80655 DLPFC external cohort lock",
    "",
    "dataset=GSE80655",
    paste0(
      "expression_sha256=",
      EXPR_SHA256
    ),
    paste0(
      "soft_sha256=",
      SOFT_SHA256
    ),
    paste0(
      "all_geo_samples=",
      nrow(GEO_META)
    ),
    paste0(
      "expression_samples=",
      length(EXPR_SAMPLE_IDS)
    ),
    paste0(
      "metadata_expression_mapping=",
      EXPR_WITH_META,
      "/281"
    ),
    "brain_region=DLPFC",
    "diagnoses=Control,Schizophrenia",
    "adult_population_required=TRUE",
    paste0(
      "locked_external_samples=",
      nrow(GSE_COHORT)
    ),
    paste0(
      "controls=",
      sum(
        GSE_COHORT$Dx == "Control"
      )
    ),
    paste0(
      "schizophrenia=",
      sum(
        GSE_COHORT$Dx == "Schizo"
      )
    ),
    paste0(
      "unique_donors=",
      length(
        unique(
          GSE_COHORT$donor_id
        )
      )
    ),
    "",
    "technical_signature_gate_completed_before_phenotype_unlock=TRUE",
    "external_score_calculated=FALSE",
    "external_performance_evaluated=FALSE"
  ),
  LOCK_PATH
)


# ------------------------------------------------------------
# 20. FINAL CHECKS
# ------------------------------------------------------------

stopifnot(
  nrow(GEO_META) == 281L,
  EXPR_WITH_META == 281L,
  length(
    unique(
      GSE_COHORT$donor_id
    )
  ) ==
    nrow(GSE_COHORT),
  all(
    GSE_COHORT$brain_region == "DLPFC"
  ),
  all(
    as.character(
      GSE_COHORT$Dx
    ) %in%
      c(
        "Control",
        "Schizo"
      )
  ),
  file.exists(COHORT_PATH),
  file.exists(ORDER_PATH),
  file.exists(LOCK_PATH)
)


# ------------------------------------------------------------
# 21. OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START


cat("\n============================================================\n")
cat("STEP 06C — GSE80655 DLPFC COHORT LOCK\n")
cat("============================================================\n")


cat(
  "All GEO samples:                 ",
  nrow(GEO_META),
  "\n",
  sep = ""
)

cat(
  "Expression columns mapped:       ",
  EXPR_WITH_META,
  " / 281\n",
  sep = ""
)

cat(
  "Locked DLPFC SCZ/Control cohort: ",
  nrow(GSE_COHORT),
  "\n",
  sep = ""
)

cat(
  "Unique donors:                   ",
  length(
    unique(
      GSE_COHORT$donor_id
    )
  ),
  "\n",
  sep = ""
)


cat("\n--- DIAGNOSIS COUNTS ---\n")

print(
  DX_COUNTS,
  row.names = FALSE
)


cat("\n--- SEX COUNTS ---\n")

print(
  SEX_COUNTS,
  row.names = FALSE
)


cat("\n--- CONTINUOUS CHARACTERISTICS ---\n")

print(
  CONTINUOUS_SUMMARY,
  row.names = FALSE
)


cat(
  "\nNon-adult samples: ",
  N_NONADULT,
  "\n",
  sep = ""
)


cat(
  "\nLocked cohort saved:\n",
  COHORT_PATH,
  "\n",
  sep = ""
)

cat(
  "\nSample order saved:\n",
  ORDER_PATH,
  "\n",
  sep = ""
)

cat(
  "\nExternal score calculated: NO\n"
)

cat(
  "External AUROC evaluated:  NO\n"
)

cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS,
    1
  ),
  " sec\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 06C — CELL 1 PASS\n")
cat("============================================================\n")

Project sentinel: PASS
SOFT size: 0.02 MB
SOFT SHA-256: 93e76b15d767f0cd704aba3f11a876f86b64d3bed15678f8ae23d188fd0e4c1e
Reading expression header...
Expression samples: 281
Reading GEO family SOFT metadata...
GEO SAMPLE blocks found: 281
Parsing sample metadata...
Parsed metadata rows: 281
Metadata samples mapped to expression: 281 / 281
Expression columns mapped to metadata: 281 / 281

--- ALL GSE80655 BRAIN REGION x DIAGNOSIS ---
  Var1             Var2 Freq
  AnCg Bipolar Disorder   24
 DLPFC Bipolar Disorder   23
  nAcc Bipolar Disorder   24
  AnCg          Control   24
 DLPFC          Control   24
  nAcc          Control   22
  AnCg Major Depression   24
 DLPFC Major Depression   23
  nAcc Major Depression   22
  AnCg    Schizophrenia   24
 DLPFC    Schizophrenia   24
  nAcc    Schizophrenia   23

STEP 06C — GSE80655 DLPFC COHORT LOCK
All GEO samples:                 281
Expression columns mapped:       281 / 281
Locked DLPFC SCZ/Control cohort: 48
Unique donors:                 

In [15]:
# ============================================================
# STEP 06B-DISCOVERY — COMPLEMENTARY HALF-SAMPLE STABILITY
#
# 100 complementary donor pairs = 200 half-sample fits
#
# Fixed structural hyperparameters from nested CV:
#   alpha = 0.10
#   deviation penalty = 1
#
# Lambda:
#   selected independently inside each half-sample using
#   donor-grouped 4-fold CV.
#
# NO EXTERNAL DATA USED.
#
# Current 403-gene set remains PROVISIONAL.
# Final signature will be frozen only after:
#   1. complementary half-sample stability
#   2. donor-level permutation null
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")
flush.console()


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace(
  "glmnet",
  quietly = TRUE
)) {
  stop("glmnet package unavailable.")
}

cat(
  "glmnet version: ",
  as.character(
    packageVersion("glmnet")
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 2. SETTINGS
# ------------------------------------------------------------

N_PAIRS <- 100L
N_FITS  <- N_PAIRS * 2L

FIXED_ALPHA <- 0.10
FIXED_DEV_PENALTY <- 1

N_INNER_FOLDS <- 4L
N_LAMBDA <- 50L

SELECTION_THRESHOLD <- 0.80
SIGN_THRESHOLD <- 0.90
ZERO_TOL <- 1e-12

BASE_SEED <- 2026093000L


# ------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_gene_annotation_v1.csv"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

OUTER_STABILITY_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "stability_v1",
  "step05a_stable_shared_genes_v1.csv"
)

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "07_stability",
  "complementary_half_sample_v1"
)

FIT_DIR <- file.path(
  OUT_DIR,
  "fits"
)

dir.create(
  FIT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

REQUIRED <- c(
  COUNTS_PATH,
  ANNOTATION_PATH,
  META_PATH
)

if (!all(file.exists(REQUIRED))) {
  stop("Required canonical BrainSeq input missing.")
}


# ------------------------------------------------------------
# 4. PROTOCOL-DEVIATION LOG
# ------------------------------------------------------------

DEVIATION_LOG <- file.path(
  PROJECT_ROOT,
  "99_logs",
  "PROTOCOL_DEVIATION_GSE80655_EARLY_METADATA_v1.txt"
)

if (!file.exists(DEVIATION_LOG)) {

  writeLines(
    c(
      "Protocol deviation log",
      "",
      "Issue:",
      paste(
        "GSE80655 diagnosis metadata and diagnosis counts were",
        "inspected before final discovery molecular-state score freeze."
      ),
      "",
      "Important limitations of the deviation:",
      "No GSE80655 molecular score was calculated.",
      "No GSE80655 score-vs-diagnosis association was calculated.",
      "No GSE80655 AUROC was calculated.",
      "No external expression result was used to choose genes.",
      "No discovery hyperparameter was changed based on GSE80655.",
      "No stability threshold was changed based on GSE80655.",
      "",
      "Mitigation:",
      paste(
        "All remaining discovery stability, permutation, weighting,",
        "and score-lock decisions will use BrainSeq discovery data only."
      ),
      paste(
        "GSE80655 labels will not be consulted again until the",
        "discovery score and technical gate are completely frozen."
      )
    ),
    DEVIATION_LOG
  )
}


# ------------------------------------------------------------
# 5. LOAD DISCOVERY DATA
# ------------------------------------------------------------

cat("Loading canonical BrainSeq inputs...\n")
flush.console()

COUNTS <- readRDS(
  COUNTS_PATH
)

ANNOTATION <- read.csv(
  ANNOTATION_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(COUNTS) == 23740L,
  ncol(COUNTS) == 755L,
  nrow(ANNOTATION) == 23740L,
  nrow(META) == 755L,
  length(unique(META$BrNum)) == 454L
)

N_GENES <- nrow(COUNTS)
GENE_IDS <- ANNOTATION$ensemblID_stable

if (
  anyNA(GENE_IDS) ||
  anyDuplicated(GENE_IDS)
) {
  stop("Invalid canonical gene IDs.")
}

cat(
  "Discovery data: ",
  N_GENES,
  " genes x ",
  ncol(COUNTS),
  " samples; ",
  length(unique(META$BrNum)),
  " donors\n",
  sep = ""
)


# ------------------------------------------------------------
# 6. SAME EXPRESSION TRANSFORMATION USED IN PRIMARY MODEL
# ------------------------------------------------------------

LIB_SIZE <- colSums(
  COUNTS
)

if (
  any(!is.finite(LIB_SIZE)) ||
  any(LIB_SIZE <= 0)
) {
  stop("Invalid library sizes.")
}

LOGCPM <- log2(
  sweep(
    COUNTS,
    2,
    LIB_SIZE / 1e6,
    "/"
  ) + 1
)

rm(
  COUNTS,
  LIB_SIZE
)

invisible(gc())

X_EXPR <- t(
  LOGCPM
)

rm(LOGCPM)

colnames(X_EXPR) <- GENE_IDS

stopifnot(
  nrow(X_EXPR) == 755L,
  ncol(X_EXPR) == N_GENES
)


# ------------------------------------------------------------
# 7. OUTCOME + REGION
# ------------------------------------------------------------

Y <- ifelse(
  META$Dx == "Schizo",
  1L,
  0L
)

if (!setequal(
  unique(Y),
  c(0L, 1L)
)) {
  stop("Outcome encoding failed.")
}

REGION_SIGN <- ifelse(
  META$Region == "DLPFC",
  1,
  ifelse(
    META$Region == "HIPPO",
    -1,
    NA_real_
  )
)

if (anyNA(REGION_SIGN)) {
  stop("Unexpected region value.")
}


# ------------------------------------------------------------
# 8. COVARIATES
# ------------------------------------------------------------

META$Region <- factor(
  META$Region,
  levels = c(
    "DLPFC",
    "HIPPO"
  )
)

META$Sex <- factor(
  META$Sex
)

META$Race <- factor(
  META$Race
)

COVARIATE_MATRIX <- model.matrix(
  ~ Region +
    donor_age +
    Sex +
    Race +
    RIN +
    snpPC1 +
    snpPC2 +
    snpPC3 +
    snpPC4 +
    snpPC5 +
    snpPC6 +
    snpPC7 +
    snpPC8 +
    snpPC9 +
    snpPC10,
  data = META
)

COVARIATE_MATRIX <- COVARIATE_MATRIX[
  ,
  colnames(COVARIATE_MATRIX) != "(Intercept)",
  drop = FALSE
]

if (
  anyNA(COVARIATE_MATRIX) ||
  any(!is.finite(COVARIATE_MATRIX))
) {
  stop("Invalid covariate matrix.")
}

N_COV <- ncol(
  COVARIATE_MATRIX
)

cat(
  "Covariates: ",
  N_COV,
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 9. SHARED / DEVIATION MATRIX
# ------------------------------------------------------------

cat("Building shared/deviation model matrix...\n")
flush.console()

X_SHARED <- X_EXPR

X_DEV <- sweep(
  X_EXPR,
  1,
  REGION_SIGN,
  "*"
)

colnames(X_SHARED) <- paste0(
  "SHARED__",
  GENE_IDS
)

colnames(X_DEV) <- paste0(
  "DEV__",
  GENE_IDS
)

X_MODEL <- cbind(
  COVARIATE_MATRIX,
  X_SHARED,
  X_DEV
)

rm(
  X_EXPR,
  X_SHARED,
  X_DEV
)

invisible(gc())

stopifnot(
  ncol(X_MODEL) ==
    N_COV + 2L * N_GENES
)

cat(
  "Total predictors: ",
  ncol(X_MODEL),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 10. DONOR-BALANCED WEIGHTS
# ------------------------------------------------------------

OBS_WEIGHT <- as.numeric(
  META$donor_balance_weight
)

if (
  length(OBS_WEIGHT) != nrow(META) ||
  anyNA(OBS_WEIGHT) ||
  any(OBS_WEIGHT <= 0)
) {
  stop("Invalid donor-balanced weights.")
}

stopifnot(
  abs(
    sum(OBS_WEIGHT) - 454
  ) < 1e-8
)


# ------------------------------------------------------------
# 11. DONOR TABLE
# ------------------------------------------------------------

DONOR_IDS <- unique(
  META$BrNum
)

DONOR_TABLE <- do.call(
  rbind,
  lapply(
    DONOR_IDS,
    function(id) {

      z <- META[
        META$BrNum == id,
        ,
        drop = FALSE
      ]

      dx <- unique(
        as.character(
          z$Dx
        )
      )

      if (length(dx) != 1L) {
        stop("Diagnosis inconsistent within donor.")
      }

      has_dlpfc <- any(
        z$Region == "DLPFC"
      )

      has_hippo <- any(
        z$Region == "HIPPO"
      )

      availability <- if (
        has_dlpfc &&
        has_hippo
      ) {

        "paired"

      } else if (
        has_dlpfc
      ) {

        "DLPFC_only"

      } else {

        "HIPPO_only"
      }

      data.frame(
        BrNum = id,
        Dx = dx,
        region_availability = availability,
        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(
  DONOR_TABLE
) <- NULL

DONOR_TABLE$stratum <- paste(
  DONOR_TABLE$Dx,
  DONOR_TABLE$region_availability,
  sep = "__"
)

stopifnot(
  nrow(DONOR_TABLE) == 454L,
  anyDuplicated(DONOR_TABLE$BrNum) == 0L
)

cat("\nDonor strata:\n")

print(
  as.data.frame(
    table(
      DONOR_TABLE$stratum
    )
  ),
  row.names = FALSE
)


# ------------------------------------------------------------
# 12. CREATE / LOAD 100 COMPLEMENTARY DONOR PAIRS
# ------------------------------------------------------------

PAIR_MANIFEST_PATH <- file.path(
  OUT_DIR,
  "complementary_donor_pair_manifest_100pairs_v1.csv"
)

if (!file.exists(PAIR_MANIFEST_PATH)) {

  cat(
    "\nCreating 100 complementary donor half-sample pairs...\n"
  )

  PAIR_LIST <- vector(
    "list",
    N_PAIRS
  )

  STRATA <- sort(
    unique(
      DONOR_TABLE$stratum
    )
  )

  for (pair_id in seq_len(N_PAIRS)) {

    set.seed(
      BASE_SEED + pair_id
    )

    pair_rows <- vector(
      "list",
      length(STRATA)
    )

    for (s_idx in seq_along(STRATA)) {

      s <- STRATA[s_idx]

      donors_s <- DONOR_TABLE[
        DONOR_TABLE$stratum == s,
        ,
        drop = FALSE
      ]

      perm <- sample(
        seq_len(
          nrow(donors_s)
        ),
        nrow(donors_s),
        replace = FALSE
      )

      donors_s <- donors_s[
        perm,
        ,
        drop = FALSE
      ]

      n_s <- nrow(
        donors_s
      )

      n_A <- ceiling(
        n_s / 2
      )

      half_label <- c(
        rep(
          "A",
          n_A
        ),
        rep(
          "B",
          n_s - n_A
        )
      )

      donors_s$pair_id <- pair_id
      donors_s$half <- half_label

      pair_rows[
        s_idx
      ] <- list(
        donors_s
      )
    }

    PAIR_LIST[
      pair_id
    ] <- list(
      do.call(
        rbind,
        pair_rows
      )
    )
  }

  PAIR_MANIFEST <- do.call(
    rbind,
    PAIR_LIST
  )

  rownames(
    PAIR_MANIFEST
  ) <- NULL

  write.csv(
    PAIR_MANIFEST,
    PAIR_MANIFEST_PATH,
    row.names = FALSE
  )

} else {

  cat(
    "\nExisting complementary pair manifest found; reusing it.\n"
  )

  PAIR_MANIFEST <- read.csv(
    PAIR_MANIFEST_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
  )
}


# ------------------------------------------------------------
# 13. VERIFY COMPLEMENTARY PAIRS
# ------------------------------------------------------------

if (
  nrow(PAIR_MANIFEST) !=
    454L * N_PAIRS
) {
  stop("Pair manifest row count invalid.")
}

for (pair_id in seq_len(N_PAIRS)) {

  z <- PAIR_MANIFEST[
    PAIR_MANIFEST$pair_id ==
      pair_id,
    ,
    drop = FALSE
  ]

  if (
    nrow(z) != 454L ||
    length(unique(z$BrNum)) != 454L
  ) {
    stop(
      paste(
        "Invalid donor manifest in pair",
        pair_id
      )
    )
  }

  A <- z$BrNum[
    z$half == "A"
  ]

  B <- z$BrNum[
    z$half == "B"
  ]

  if (
    length(
      intersect(
        A,
        B
      )
    ) != 0L
  ) {
    stop(
      paste(
        "Complementarity violation in pair",
        pair_id
      )
    )
  }

  if (
    length(
      union(
        A,
        B
      )
    ) != 454L
  ) {
    stop(
      paste(
        "Incomplete donor coverage in pair",
        pair_id
      )
    )
  }
}

cat(
  "Complementary pair audit: PASS\n"
)


# ------------------------------------------------------------
# 14. INNER DONOR-GROUPED FOLD FUNCTION
# ------------------------------------------------------------

make_inner_foldid <- function(
  half_meta,
  seed
) {

  donor_ids <- unique(
    half_meta$BrNum
  )

  donor_tbl <- do.call(
    rbind,
    lapply(
      donor_ids,
      function(id) {

        z <- half_meta[
          half_meta$BrNum == id,
          ,
          drop = FALSE
        ]

        dx <- unique(
          as.character(
            z$Dx
          )
        )

        if (length(dx) != 1L) {
          stop(
            "Diagnosis inconsistent within inner-CV donor."
          )
        }

        has_d <- any(
          z$Region == "DLPFC"
        )

        has_h <- any(
          z$Region == "HIPPO"
        )

        availability <- if (
          has_d && has_h
        ) {

          "paired"

        } else if (has_d) {

          "DLPFC_only"

        } else {

          "HIPPO_only"
        }

        data.frame(
          BrNum = id,
          Dx = dx,
          availability = availability,
          stringsAsFactors = FALSE
        )
      }
    )
  )

  donor_tbl$stratum <- paste(
    donor_tbl$Dx,
    donor_tbl$availability,
    sep = "__"
  )

  donor_tbl$fold <- NA_integer_

  set.seed(seed)

  for (
    s in unique(
      donor_tbl$stratum
    )
  ) {

    idx <- which(
      donor_tbl$stratum == s
    )

    idx <- sample(
      idx,
      length(idx),
      replace = FALSE
    )

    folds <- rep(
      seq_len(N_INNER_FOLDS),
      length.out = length(idx)
    )

    folds <- sample(
      folds,
      length(folds),
      replace = FALSE
    )

    donor_tbl$fold[
      idx
    ] <- folds
  }

  lookup <- setNames(
    donor_tbl$fold,
    donor_tbl$BrNum
  )

  foldid <- as.integer(
    lookup[
      half_meta$BrNum
    ]
  )

  if (
    anyNA(foldid) ||
    !setequal(
      unique(foldid),
      seq_len(N_INNER_FOLDS)
    )
  ) {
    stop(
      "Invalid inner donor-grouped fold assignment."
    )
  }

  # donor leakage check
  for (f in seq_len(N_INNER_FOLDS)) {

    val_donors <- unique(
      half_meta$BrNum[
        foldid == f
      ]
    )

    train_donors <- unique(
      half_meta$BrNum[
        foldid != f
      ]
    )

    if (
      length(
        intersect(
          val_donors,
          train_donors
        )
      ) > 0L
    ) {
      stop(
        "Inner donor leakage detected."
      )
    }
  }

  foldid
}


# ------------------------------------------------------------
# 15. FIT 200 HALF-SAMPLE MODELS
# ------------------------------------------------------------

BASE_PENALTY_FACTOR <- c(
  rep(
    0,
    N_COV
  ),
  rep(
    1,
    N_GENES
  ),
  rep(
    FIXED_DEV_PENALTY,
    N_GENES
  )
)

stopifnot(
  length(BASE_PENALTY_FACTOR) ==
    ncol(X_MODEL)
)


FIT_COUNTER <- 0L

for (pair_id in seq_len(N_PAIRS)) {

  pair_data <- PAIR_MANIFEST[
    PAIR_MANIFEST$pair_id ==
      pair_id,
    ,
    drop = FALSE
  ]

  for (half_label in c("A", "B")) {

    FIT_COUNTER <- FIT_COUNTER + 1L

    FIT_ID <- sprintf(
      "fit_%03d_%s",
      pair_id,
      half_label
    )

    RESULT_PATH <- file.path(
      FIT_DIR,
      paste0(
        FIT_ID,
        ".rds"
      )
    )

    if (file.exists(RESULT_PATH)) {

      if (
        FIT_COUNTER %% 10L == 0L
      ) {
        cat(
          sprintf(
            "[%03d/%03d] %s already complete; skipping.\n",
            FIT_COUNTER,
            N_FITS,
            FIT_ID
          )
        )
      }

      next
    }

    FIT_START <- proc.time()[3]

    HALF_DONORS <- pair_data$BrNum[
      pair_data$half ==
        half_label
    ]

    SAMPLE_IDX <- which(
      META$BrNum %in%
        HALF_DONORS
    )

    HALF_META <- META[
      SAMPLE_IDX,
      ,
      drop = FALSE
    ]

    if (
      length(
        unique(
          HALF_META$BrNum
        )
      ) !=
        length(
          HALF_DONORS
        )
    ) {
      stop(
        paste(
          "Half-sample donor mapping failure:",
          FIT_ID
        )
      )
    }

    # ------------------------------------------
    # remove only covariates that are constant
    # inside this half-sample
    # ------------------------------------------

    COV_SD <- apply(
      COVARIATE_MATRIX[
        SAMPLE_IDX,
        ,
        drop = FALSE
      ],
      2,
      sd
    )

    COV_KEEP <- (
      is.finite(COV_SD) &
      COV_SD > 0
    )

    MODEL_COLS <- c(
      which(COV_KEEP),
      (N_COV + 1L):
        ncol(X_MODEL)
    )

    PENALTY_FACTOR <- c(
      rep(
        0,
        sum(COV_KEEP)
      ),
      rep(
        1,
        N_GENES
      ),
      rep(
        FIXED_DEV_PENALTY,
        N_GENES
      )
    )

    stopifnot(
      length(PENALTY_FACTOR) ==
        length(MODEL_COLS)
    )

    INNER_SEED <- (
      BASE_SEED +
      10000L +
      pair_id * 10L +
      ifelse(
        half_label == "A",
        1L,
        2L
      )
    )

    INNER_FOLDID <- make_inner_foldid(
      HALF_META,
      INNER_SEED
    )

    cat(
      sprintf(
        "[%03d/%03d] %s | donors=%d samples=%d ... ",
        FIT_COUNTER,
        N_FITS,
        FIT_ID,
        length(HALF_DONORS),
        length(SAMPLE_IDX)
      )
    )

    flush.console()

    CVFIT <- glmnet::cv.glmnet(
      x = X_MODEL[
        SAMPLE_IDX,
        MODEL_COLS,
        drop = FALSE
      ],
      y = Y[
        SAMPLE_IDX
      ],
      family = "binomial",
      weights = OBS_WEIGHT[
        SAMPLE_IDX
      ],
      foldid = INNER_FOLDID,
      type.measure = "auc",
      alpha = FIXED_ALPHA,
      penalty.factor =
        PENALTY_FACTOR,
      standardize = TRUE,
      intercept = TRUE,
      nlambda = N_LAMBDA,
      keep = FALSE,
      parallel = FALSE
    )

    VALID <- is.finite(
      CVFIT$cvm
    )

    if (!any(VALID)) {
      stop(
        paste(
          "No valid CV AUC:",
          FIT_ID
        )
      )
    }

    BEST_INDEX <- which.max(
      ifelse(
        VALID,
        CVFIT$cvm,
        -Inf
      )
    )

    BEST_LAMBDA <- CVFIT$lambda[
      BEST_INDEX
    ]

    BEST_AUC <- CVFIT$cvm[
      BEST_INDEX
    ]

    COEF_MATRIX <- as.matrix(
      coef(
        CVFIT$glmnet.fit,
        s = BEST_LAMBDA
      )
    )

    COEF_VECTOR <- COEF_MATRIX[
      ,
      1
    ]

    names(COEF_VECTOR) <- rownames(
      COEF_MATRIX
    )

    SHARED_NAMES <- paste0(
      "SHARED__",
      GENE_IDS
    )

    DEV_NAMES <- paste0(
      "DEV__",
      GENE_IDS
    )

    SHARED_COEF <- as.numeric(
      COEF_VECTOR[
        SHARED_NAMES
      ]
    )

    DEV_COEF <- as.numeric(
      COEF_VECTOR[
        DEV_NAMES
      ]
    )

    if (
      anyNA(SHARED_COEF) ||
      anyNA(DEV_COEF)
    ) {
      stop(
        paste(
          "Gene coefficient extraction failed:",
          FIT_ID
        )
      )
    }

    FIT_SECONDS <- (
      proc.time()[3] -
      FIT_START
    )

    RESULT <- list(
      fit_id = FIT_ID,
      pair_id = pair_id,
      half = half_label,
      n_donors = length(
        HALF_DONORS
      ),
      n_samples = length(
        SAMPLE_IDX
      ),
      n_covariates_used = sum(
        COV_KEEP
      ),
      alpha = FIXED_ALPHA,
      deviation_penalty =
        FIXED_DEV_PENALTY,
      best_lambda =
        BEST_LAMBDA,
      best_inner_auc =
        BEST_AUC,
      shared_coef =
        SHARED_COEF,
      deviation_coef =
        DEV_COEF,
      selected_shared =
        sum(
          abs(
            SHARED_COEF
          ) > ZERO_TOL
        ),
      selected_deviation =
        sum(
          abs(
            DEV_COEF
          ) > ZERO_TOL
        ),
      runtime_seconds =
        FIT_SECONDS
    )

    saveRDS(
      RESULT,
      RESULT_PATH,
      compress = "gzip"
    )

    cat(
      sprintf(
        "AUC=%.4f shared=%d dev=%d time=%.1fs\n",
        BEST_AUC,
        RESULT$selected_shared,
        RESULT$selected_deviation,
        FIT_SECONDS
      )
    )

    flush.console()

    rm(
      CVFIT,
      COEF_MATRIX,
      COEF_VECTOR,
      SHARED_COEF,
      DEV_COEF,
      RESULT
    )

    invisible(gc())
  }
}


# ------------------------------------------------------------
# 16. VERIFY ALL 200 FITS COMPLETE
# ------------------------------------------------------------

RESULT_FILES <- list.files(
  FIT_DIR,
  pattern = "^fit_[0-9]{3}_[AB]\\.rds$",
  full.names = TRUE
)

RESULT_FILES <- sort(
  RESULT_FILES
)

if (
  length(
    RESULT_FILES
  ) != N_FITS
) {

  cat(
    "\nCompleted fits: ",
    length(RESULT_FILES),
    " / ",
    N_FITS,
    "\n",
    sep = ""
  )

  cat(
    "Re-run this SAME CELL to resume safely.\n"
  )

  stop(
    "Complementary half-sample analysis incomplete."
  )
}

cat(
  "\nAll 200 half-sample fits complete.\n"
)


# ------------------------------------------------------------
# 17. AGGREGATE COEFFICIENT MATRICES
# ------------------------------------------------------------

SHARED_MATRIX <- matrix(
  0,
  nrow = N_GENES,
  ncol = N_FITS
)

DEV_MATRIX <- matrix(
  0,
  nrow = N_GENES,
  ncol = N_FITS
)

FIT_SUMMARY_LIST <- vector(
  "list",
  N_FITS
)

FIT_LABELS <- character(
  N_FITS
)

for (i in seq_along(RESULT_FILES)) {

  z <- readRDS(
    RESULT_FILES[i]
  )

  if (
    length(z$shared_coef) !=
      N_GENES ||
    length(z$deviation_coef) !=
      N_GENES
  ) {
    stop(
      paste(
        "Invalid result:",
        RESULT_FILES[i]
      )
    )
  }

  SHARED_MATRIX[
    ,
    i
  ] <- z$shared_coef

  DEV_MATRIX[
    ,
    i
  ] <- z$deviation_coef

  FIT_LABELS[i] <- z$fit_id

  FIT_SUMMARY_LIST[
    i
  ] <- list(
    data.frame(
      fit_id =
        z$fit_id,
      pair_id =
        z$pair_id,
      half =
        z$half,
      n_donors =
        z$n_donors,
      n_samples =
        z$n_samples,
      n_covariates_used =
        z$n_covariates_used,
      alpha =
        z$alpha,
      deviation_penalty =
        z$deviation_penalty,
      best_lambda =
        z$best_lambda,
      best_inner_auc =
        z$best_inner_auc,
      selected_shared =
        z$selected_shared,
      selected_deviation =
        z$selected_deviation,
      runtime_seconds =
        z$runtime_seconds,
      stringsAsFactors = FALSE
    )
  )
}

colnames(
  SHARED_MATRIX
) <- FIT_LABELS

colnames(
  DEV_MATRIX
) <- FIT_LABELS

rownames(
  SHARED_MATRIX
) <- GENE_IDS

rownames(
  DEV_MATRIX
) <- GENE_IDS

FIT_SUMMARY <- do.call(
  rbind,
  FIT_SUMMARY_LIST
)

rownames(
  FIT_SUMMARY
) <- NULL


# ------------------------------------------------------------
# 18. STABILITY METRICS
# ------------------------------------------------------------

SHARED_POS <- rowSums(
  SHARED_MATRIX > ZERO_TOL
)

SHARED_NEG <- rowSums(
  SHARED_MATRIX < -ZERO_TOL
)

SHARED_SELECTED <- (
  SHARED_POS +
  SHARED_NEG
)

SHARED_SELECTION_PROB <- (
  SHARED_SELECTED /
  N_FITS
)

SHARED_SIGN_CONSISTENCY <- ifelse(
  SHARED_SELECTED > 0,
  pmax(
    SHARED_POS,
    SHARED_NEG
  ) /
    SHARED_SELECTED,
  NA_real_
)


DEV_POS <- rowSums(
  DEV_MATRIX > ZERO_TOL
)

DEV_NEG <- rowSums(
  DEV_MATRIX < -ZERO_TOL
)

DEV_SELECTED <- (
  DEV_POS +
  DEV_NEG
)

DEV_SELECTION_PROB <- (
  DEV_SELECTED /
  N_FITS
)

DEV_SIGN_CONSISTENCY <- ifelse(
  DEV_SELECTED > 0,
  pmax(
    DEV_POS,
    DEV_NEG
  ) /
    DEV_SELECTED,
  NA_real_
)


# ------------------------------------------------------------
# 19. NONZERO COEFFICIENT SUMMARY HELPERS
# ------------------------------------------------------------

median_nonzero <- function(x) {

  x <- x[
    abs(x) > ZERO_TOL
  ]

  if (length(x) == 0L) {
    return(0)
  }

  median(x)
}


iqr_nonzero <- function(x) {

  x <- x[
    abs(x) > ZERO_TOL
  ]

  if (length(x) <= 1L) {
    return(0)
  }

  IQR(x)
}


SHARED_MEDIAN <- apply(
  SHARED_MATRIX,
  1,
  median_nonzero
)

SHARED_IQR <- apply(
  SHARED_MATRIX,
  1,
  iqr_nonzero
)

DEV_MEDIAN <- apply(
  DEV_MATRIX,
  1,
  median_nonzero
)

DEV_IQR <- apply(
  DEV_MATRIX,
  1,
  iqr_nonzero
)


# ------------------------------------------------------------
# 20. STABILITY TABLE
# ------------------------------------------------------------

STABILITY_TABLE <- data.frame(

  ensemblID_stable =
    GENE_IDS,

  gene_symbol =
    ANNOTATION$gene_symbol,

  gene_type =
    ANNOTATION$gene_type,

  shared_selection_count =
    SHARED_SELECTED,

  shared_selection_probability =
    SHARED_SELECTION_PROB,

  shared_positive_count =
    SHARED_POS,

  shared_negative_count =
    SHARED_NEG,

  shared_sign_consistency =
    SHARED_SIGN_CONSISTENCY,

  shared_median_nonzero_beta =
    SHARED_MEDIAN,

  shared_IQR_nonzero_beta =
    SHARED_IQR,

  deviation_selection_count =
    DEV_SELECTED,

  deviation_selection_probability =
    DEV_SELECTION_PROB,

  deviation_positive_count =
    DEV_POS,

  deviation_negative_count =
    DEV_NEG,

  deviation_sign_consistency =
    DEV_SIGN_CONSISTENCY,

  deviation_median_nonzero_beta =
    DEV_MEDIAN,

  deviation_IQR_nonzero_beta =
    DEV_IQR,

  stringsAsFactors = FALSE
)


STABILITY_TABLE$candidate_stable_shared <- (

  STABILITY_TABLE$shared_selection_probability >=
    SELECTION_THRESHOLD

  &

  !is.na(
    STABILITY_TABLE$shared_sign_consistency
  )

  &

  STABILITY_TABLE$shared_sign_consistency >=
    SIGN_THRESHOLD
)


STABILITY_TABLE$candidate_stable_deviation <- (

  STABILITY_TABLE$deviation_selection_probability >=
    SELECTION_THRESHOLD

  &

  !is.na(
    STABILITY_TABLE$deviation_sign_consistency
  )

  &

  STABILITY_TABLE$deviation_sign_consistency >=
    SIGN_THRESHOLD
)


# ------------------------------------------------------------
# 21. COMPARE AGAINST 25-OUTER-FIT CANDIDATE SET
# ------------------------------------------------------------

if (file.exists(
  OUTER_STABILITY_PATH
)) {

  OUTER_STABLE <- read.csv(
    OUTER_STABILITY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
  )

  OUTER_IDS <- unique(
    OUTER_STABLE$ensemblID_stable
  )

  STABILITY_TABLE$stable_in_25_outer_models <- (
    STABILITY_TABLE$ensemblID_stable %in%
      OUTER_IDS
  )

} else {

  OUTER_IDS <- character(0)

  STABILITY_TABLE$stable_in_25_outer_models <-
    FALSE
}


HALF_STABLE_IDS <- STABILITY_TABLE$ensemblID_stable[
  STABILITY_TABLE$candidate_stable_shared
]

INTERSECTION_IDS <- intersect(
  HALF_STABLE_IDS,
  OUTER_IDS
)

UNION_IDS <- union(
  HALF_STABLE_IDS,
  OUTER_IDS
)

JACCARD <- if (
  length(UNION_IDS) > 0L
) {

  length(INTERSECTION_IDS) /
    length(UNION_IDS)

} else {

  NA_real_
}


# ------------------------------------------------------------
# 22. CANDIDATE TABLES
# ------------------------------------------------------------

CANDIDATE_SHARED <- STABILITY_TABLE[
  STABILITY_TABLE$candidate_stable_shared,
  ,
  drop = FALSE
]

CANDIDATE_SHARED <- CANDIDATE_SHARED[
  order(
    -CANDIDATE_SHARED$shared_selection_probability,
    -CANDIDATE_SHARED$shared_sign_consistency,
    -abs(
      CANDIDATE_SHARED$shared_median_nonzero_beta
    )
  ),
  ,
  drop = FALSE
]

rownames(
  CANDIDATE_SHARED
) <- NULL


CANDIDATE_DEV <- STABILITY_TABLE[
  STABILITY_TABLE$candidate_stable_deviation,
  ,
  drop = FALSE
]

CANDIDATE_DEV <- CANDIDATE_DEV[
  order(
    -CANDIDATE_DEV$deviation_selection_probability,
    -CANDIDATE_DEV$deviation_sign_consistency
  ),
  ,
  drop = FALSE
]

rownames(
  CANDIDATE_DEV
) <- NULL


# ------------------------------------------------------------
# 23. SUMMARY
# ------------------------------------------------------------

SUMMARY_TABLE <- data.frame(

  metric = c(
    "Complementary_pairs",
    "Half_sample_fits",
    "Alpha_fixed",
    "Deviation_penalty_fixed",
    "Shared_selection_threshold",
    "Shared_sign_threshold",
    "Genes_selected_shared_at_least_once",
    "Candidate_stable_shared_genes",
    "Candidate_stable_deviation_genes",
    "Outer_25fit_candidate_genes",
    "Overlap_half_sample_vs_outer",
    "Union_half_sample_vs_outer",
    "Jaccard_half_sample_vs_outer",
    "Half_sample_positive_stable",
    "Half_sample_negative_stable",
    "Mean_selected_shared_per_fit",
    "Median_selected_shared_per_fit",
    "Mean_selected_deviation_per_fit",
    "Median_selected_deviation_per_fit",
    "Mean_inner_CV_AUC"
  ),

  value = c(
    N_PAIRS,
    N_FITS,
    FIXED_ALPHA,
    FIXED_DEV_PENALTY,
    SELECTION_THRESHOLD,
    SIGN_THRESHOLD,
    sum(
      SHARED_SELECTED > 0
    ),
    nrow(
      CANDIDATE_SHARED
    ),
    nrow(
      CANDIDATE_DEV
    ),
    length(
      OUTER_IDS
    ),
    length(
      INTERSECTION_IDS
    ),
    length(
      UNION_IDS
    ),
    JACCARD,
    sum(
      CANDIDATE_SHARED$shared_median_nonzero_beta > 0
    ),
    sum(
      CANDIDATE_SHARED$shared_median_nonzero_beta < 0
    ),
    mean(
      FIT_SUMMARY$selected_shared
    ),
    median(
      FIT_SUMMARY$selected_shared
    ),
    mean(
      FIT_SUMMARY$selected_deviation
    ),
    median(
      FIT_SUMMARY$selected_deviation
    ),
    mean(
      FIT_SUMMARY$best_inner_auc
    )
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 24. SAVE
# ------------------------------------------------------------

STABILITY_PATH <- file.path(
  OUT_DIR,
  "step06b_half_sample_all_gene_stability_v1.csv"
)

CANDIDATE_SHARED_PATH <- file.path(
  OUT_DIR,
  "step06b_half_sample_candidate_shared_v1.csv"
)

CANDIDATE_DEV_PATH <- file.path(
  OUT_DIR,
  "step06b_half_sample_candidate_deviation_v1.csv"
)

FIT_SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step06b_half_sample_fit_summary_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step06b_half_sample_stability_summary_v1.csv"
)

MATRIX_PATH <- file.path(
  OUT_DIR,
  "step06b_half_sample_coefficient_matrices_v1.rds"
)

LOCK_PATH <- file.path(
  OUT_DIR,
  "step06b_half_sample_analysis_definition_v1.txt"
)


write.csv(
  STABILITY_TABLE,
  STABILITY_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  CANDIDATE_SHARED,
  CANDIDATE_SHARED_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  CANDIDATE_DEV,
  CANDIDATE_DEV_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  FIT_SUMMARY,
  FIT_SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  SUMMARY_TABLE,
  SUMMARY_PATH,
  row.names = FALSE
)

saveRDS(
  list(
    gene_ids =
      GENE_IDS,
    fit_labels =
      FIT_LABELS,
    shared_coefficients =
      SHARED_MATRIX,
    deviation_coefficients =
      DEV_MATRIX
  ),
  MATRIX_PATH,
  compress = "gzip"
)

writeLines(
  c(
    "STEP 06B complementary half-sample stability",
    "",
    "pairs=100",
    "fits=200",
    "donor_level_complementarity=TRUE",
    "paired_region_samples_kept_with_donor=TRUE",
    "alpha_fixed=0.10",
    "deviation_penalty_fixed=1",
    "lambda_selected_within_each_half_sample=TRUE",
    "inner_cv=4_fold_donor_grouped",
    "selection_probability_threshold=0.80",
    "sign_consistency_threshold=0.90",
    "",
    "external_expression_used=FALSE",
    "external_score_used=FALSE",
    "external_outcome_used_for_model_decision=FALSE",
    "",
    "candidate_half_sample_stable_set_is_not_final_score_lock=TRUE"
  ),
  LOCK_PATH
)


# ------------------------------------------------------------
# 25. HARD CHECKS
# ------------------------------------------------------------

stopifnot(
  nrow(STABILITY_TABLE) == N_GENES,
  ncol(SHARED_MATRIX) == 200L,
  ncol(DEV_MATRIX) == 200L,
  all(
    CANDIDATE_SHARED$shared_selection_probability >=
      0.80
  ),
  all(
    CANDIDATE_SHARED$shared_sign_consistency >=
      0.90
  ),
  file.exists(SUMMARY_PATH),
  file.exists(CANDIDATE_SHARED_PATH)
)


# ------------------------------------------------------------
# 26. OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START

cat("\n============================================================\n")
cat("STEP 06B — COMPLEMENTARY HALF-SAMPLE STABILITY COMPLETE\n")
cat("============================================================\n")

print(
  SUMMARY_TABLE,
  row.names = FALSE
)

cat(
  "\nCandidate half-sample stable shared genes: ",
  nrow(CANDIDATE_SHARED),
  "\n",
  sep = ""
)

cat(
  "Candidate half-sample stable deviation genes: ",
  nrow(CANDIDATE_DEV),
  "\n",
  sep = ""
)

cat(
  "Overlap with current 403 outer-fit genes: ",
  length(INTERSECTION_IDS),
  " / ",
  length(OUTER_IDS),
  "\n",
  sep = ""
)

cat(
  "Jaccard vs current outer-fit set: ",
  round(
    JACCARD,
    4
  ),
  "\n",
  sep = ""
)


if (nrow(CANDIDATE_SHARED) > 0L) {

  cat(
    "\n--- TOP 30 HALF-SAMPLE STABLE SHARED GENES ---\n"
  )

  print(
    head(
      CANDIDATE_SHARED[
        ,
        c(
          "ensemblID_stable",
          "gene_symbol",
          "gene_type",
          "shared_selection_count",
          "shared_selection_probability",
          "shared_positive_count",
          "shared_negative_count",
          "shared_sign_consistency",
          "shared_median_nonzero_beta",
          "shared_IQR_nonzero_beta",
          "stable_in_25_outer_models"
        ),
        drop = FALSE
      ],
      30
    ),
    row.names = FALSE
  )
}

cat(
  "\nSaved summary:\n",
  SUMMARY_PATH,
  "\n",
  sep = ""
)

cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS / 60,
    1
  ),
  " minutes\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 06B — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
glmnet version: 5.1
Loading canonical BrainSeq inputs...
Discovery data: 23740 genes x 755 samples; 454 donors
Covariates: 17
Building shared/deviation model matrix...
Total predictors: 47497

Donor strata:
                Var1 Freq
 Control__DLPFC_only   34
 Control__HIPPO_only   51
     Control__paired  192
  Schizo__DLPFC_only   44
  Schizo__HIPPO_only   24
      Schizo__paired  109

Creating 100 complementary donor half-sample pairs...
Complementary pair audit: PASS
[001/200] fit_001_A | donors=228 samples=379 ... AUC=0.7080 shared=513 dev=18 time=9.2s
[002/200] fit_001_B | donors=226 samples=376 ... AUC=0.6881 shared=1200 dev=33 time=9.1s
[003/200] fit_002_A | donors=228 samples=379 ... AUC=0.6913 shared=305 dev=3 time=8.4s
[004/200] fit_002_B | donors=226 samples=376 ... AUC=0.7066 shared=1324 dev=70 time=8.2s
[005/200] fit_003_A | donors=228 samples=379 ... AUC=0.7344 shared=1280 dev=59 time=7.9s
[006/200] fit_003_B | donors=226 samples=376 ... AUC=0.7457 

In [16]:
# ============================================================
# STEP 06C — DONOR-LEVEL DIAGNOSIS PERMUTATION NULL
#
# Purpose:
#   - matched observed reference under one fixed donor-grouped
#     5-fold CV scheme
#   - 100 donor-level diagnosis permutations
#   - null prevalidated AUROC
#   - null number of selected shared/deviation genes
#   - null coefficient-selection stability
#   - null prevalidated score separation
#
# Model structure fixed from discovery:
#   alpha = 0.10
#   deviation penalty = 1
#
# Lambda is selected inside each observed/permuted analysis
# by donor-grouped CV.
#
# NO EXTERNAL DATA USED.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace(
  "glmnet",
  quietly = TRUE
)) {
  stop("glmnet package unavailable.")
}

cat(
  "glmnet version: ",
  as.character(
    packageVersion("glmnet")
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 2. SETTINGS
# ------------------------------------------------------------

N_PERM <- 100L

FIXED_ALPHA <- 0.10
FIXED_DEV_PENALTY <- 1

N_FOLDS <- 5L
N_LAMBDA <- 50L

ZERO_TOL <- 1e-12

STABILITY_SELECTION_THRESHOLD <- 0.80
STABILITY_SIGN_THRESHOLD <- 0.90

BASE_SEED <- 2026093100L


# ------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

ANNOTATION_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_gene_annotation_v1.csv"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

HALF_STABLE_PATH <- file.path(
  PROJECT_ROOT,
  "07_stability",
  "complementary_half_sample_v1",
  "step06b_half_sample_candidate_shared_v1.csv"
)

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "07_stability",
  "permutation_null_v1"
)

PERM_DIR <- file.path(
  OUT_DIR,
  "permutations"
)

dir.create(
  PERM_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

REQUIRED <- c(
  COUNTS_PATH,
  ANNOTATION_PATH,
  META_PATH,
  HALF_STABLE_PATH
)

if (!all(file.exists(REQUIRED))) {
  stop("Required discovery input missing.")
}


# ------------------------------------------------------------
# 4. LOAD DATA
# ------------------------------------------------------------

cat("Loading canonical discovery data...\n")

COUNTS <- readRDS(
  COUNTS_PATH
)

ANNOTATION <- read.csv(
  ANNOTATION_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

HALF_STABLE <- read.csv(
  HALF_STABLE_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(COUNTS) == 23740L,
  ncol(COUNTS) == 755L,
  nrow(ANNOTATION) == 23740L,
  nrow(META) == 755L,
  length(unique(META$BrNum)) == 454L,
  nrow(HALF_STABLE) == 24L
)

N_GENES <- nrow(
  COUNTS
)

GENE_IDS <- as.character(
  ANNOTATION$ensemblID_stable
)

if (
  anyNA(GENE_IDS) ||
  anyDuplicated(GENE_IDS)
) {
  stop("Invalid canonical gene IDs.")
}

cat(
  "Discovery matrix: ",
  N_GENES,
  " genes x ",
  ncol(COUNTS),
  " samples\n",
  sep = ""
)

cat(
  "High-confidence half-sample candidate genes: ",
  nrow(HALF_STABLE),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 5. SAME EXPRESSION TRANSFORMATION AS PRIMARY MODEL
# ------------------------------------------------------------

LIB_SIZE <- colSums(
  COUNTS
)

if (
  any(!is.finite(LIB_SIZE)) ||
  any(LIB_SIZE <= 0)
) {
  stop("Invalid library sizes.")
}

LOGCPM <- log2(
  sweep(
    COUNTS,
    2,
    LIB_SIZE / 1e6,
    "/"
  ) + 1
)

rm(
  COUNTS,
  LIB_SIZE
)

invisible(gc())

X_EXPR <- t(
  LOGCPM
)

rm(LOGCPM)

invisible(gc())

colnames(
  X_EXPR
) <- GENE_IDS


# ------------------------------------------------------------
# 6. ORIGINAL OUTCOME + REGION
# ------------------------------------------------------------

Y_ORIGINAL <- ifelse(
  META$Dx == "Schizo",
  1L,
  0L
)

if (!setequal(
  unique(Y_ORIGINAL),
  c(0L, 1L)
)) {
  stop("Diagnosis encoding failed.")
}

REGION_SIGN <- ifelse(
  META$Region == "DLPFC",
  1,
  ifelse(
    META$Region == "HIPPO",
    -1,
    NA_real_
  )
)

if (anyNA(REGION_SIGN)) {
  stop("Unexpected region value.")
}


# ------------------------------------------------------------
# 7. COVARIATE MATRIX
# ------------------------------------------------------------

META$Region <- factor(
  META$Region,
  levels = c(
    "DLPFC",
    "HIPPO"
  )
)

META$Sex <- factor(
  META$Sex
)

META$Race <- factor(
  META$Race
)

COVARIATE_MATRIX <- model.matrix(
  ~ Region +
    donor_age +
    Sex +
    Race +
    RIN +
    snpPC1 +
    snpPC2 +
    snpPC3 +
    snpPC4 +
    snpPC5 +
    snpPC6 +
    snpPC7 +
    snpPC8 +
    snpPC9 +
    snpPC10,
  data = META
)

COVARIATE_MATRIX <- COVARIATE_MATRIX[
  ,
  colnames(COVARIATE_MATRIX) != "(Intercept)",
  drop = FALSE
]

if (
  anyNA(COVARIATE_MATRIX) ||
  any(!is.finite(COVARIATE_MATRIX))
) {
  stop("Invalid covariate matrix.")
}

N_COV <- ncol(
  COVARIATE_MATRIX
)

cat(
  "Unpenalized covariates: ",
  N_COV,
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 8. SHARED / DEVIATION MODEL MATRIX
# ------------------------------------------------------------

cat("Building shared/deviation matrix...\n")

X_SHARED <- X_EXPR

X_DEV <- sweep(
  X_EXPR,
  1,
  REGION_SIGN,
  "*"
)

colnames(X_SHARED) <- paste0(
  "SHARED__",
  GENE_IDS
)

colnames(X_DEV) <- paste0(
  "DEV__",
  GENE_IDS
)

X_MODEL <- cbind(
  COVARIATE_MATRIX,
  X_SHARED,
  X_DEV
)

rm(
  X_EXPR,
  X_SHARED,
  X_DEV
)

invisible(gc())

stopifnot(
  ncol(X_MODEL) ==
    N_COV + 2L * N_GENES
)

cat(
  "Total predictors: ",
  ncol(X_MODEL),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 9. PENALTY FACTOR
# ------------------------------------------------------------

PENALTY_FACTOR <- c(
  rep(
    0,
    N_COV
  ),
  rep(
    1,
    N_GENES
  ),
  rep(
    FIXED_DEV_PENALTY,
    N_GENES
  )
)

stopifnot(
  length(PENALTY_FACTOR) ==
    ncol(X_MODEL)
)


# ------------------------------------------------------------
# 10. DONOR-BALANCED WEIGHTS
# ------------------------------------------------------------

OBS_WEIGHT <- as.numeric(
  META$donor_balance_weight
)

if (
  anyNA(OBS_WEIGHT) ||
  any(OBS_WEIGHT <= 0)
) {
  stop("Invalid observation weights.")
}

stopifnot(
  abs(
    sum(OBS_WEIGHT) - 454
  ) < 1e-8
)


# ------------------------------------------------------------
# 11. DONOR TABLE
# ------------------------------------------------------------

DONOR_IDS <- unique(
  META$BrNum
)

DONOR_TABLE <- do.call(
  rbind,
  lapply(
    DONOR_IDS,
    function(id) {

      idx <- which(
        META$BrNum == id
      )

      dx <- unique(
        Y_ORIGINAL[
          idx
        ]
      )

      if (length(dx) != 1L) {
        stop("Diagnosis inconsistent within donor.")
      }

      has_d <- any(
        META$Region[
          idx
        ] == "DLPFC"
      )

      has_h <- any(
        META$Region[
          idx
        ] == "HIPPO"
      )

      availability <- if (
        has_d && has_h
      ) {

        "paired"

      } else if (has_d) {

        "DLPFC_only"

      } else {

        "HIPPO_only"
      }

      data.frame(
        BrNum = id,
        Y_original = dx,
        region_availability = availability,
        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(
  DONOR_TABLE
) <- NULL

DONOR_TABLE$stratum <- paste(
  DONOR_TABLE$Y_original,
  DONOR_TABLE$region_availability,
  sep = "__"
)

stopifnot(
  nrow(DONOR_TABLE) == 454L,
  anyDuplicated(DONOR_TABLE$BrNum) == 0L
)


# ------------------------------------------------------------
# 12. CREATE ONE FIXED DONOR-GROUPED 5-FOLD NULL FRAME
#
# Folds are created once and then never changed.
# Every sample from the same donor remains in one fold.
# ------------------------------------------------------------

FOLD_MANIFEST_PATH <- file.path(
  OUT_DIR,
  "permutation_fixed_donor_5fold_manifest_v1.csv"
)

if (!file.exists(FOLD_MANIFEST_PATH)) {

  set.seed(
    BASE_SEED
  )

  DONOR_TABLE$fold <- NA_integer_

  STRATA <- sort(
    unique(
      DONOR_TABLE$stratum
    )
  )

  for (s in STRATA) {

    idx <- which(
      DONOR_TABLE$stratum == s
    )

    idx <- sample(
      idx,
      length(idx),
      replace = FALSE
    )

    assigned <- rep(
      seq_len(N_FOLDS),
      length.out = length(idx)
    )

    DONOR_TABLE$fold[
      idx
    ] <- assigned
  }

  write.csv(
    DONOR_TABLE,
    FOLD_MANIFEST_PATH,
    row.names = FALSE
  )

} else {

  DONOR_TABLE <- read.csv(
    FOLD_MANIFEST_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
  )
}

if (
  anyNA(DONOR_TABLE$fold) ||
  !setequal(
    unique(DONOR_TABLE$fold),
    seq_len(N_FOLDS)
  )
) {
  stop("Invalid fixed donor folds.")
}

FOLD_LOOKUP <- setNames(
  DONOR_TABLE$fold,
  DONOR_TABLE$BrNum
)

SAMPLE_FOLDID <- as.integer(
  FOLD_LOOKUP[
    as.character(
      META$BrNum
    )
  ]
)

if (anyNA(SAMPLE_FOLDID)) {
  stop("Sample-to-donor fold mapping failed.")
}

for (f in seq_len(N_FOLDS)) {

  test_donors <- unique(
    META$BrNum[
      SAMPLE_FOLDID == f
    ]
  )

  train_donors <- unique(
    META$BrNum[
      SAMPLE_FOLDID != f
    ]
  )

  if (
    length(
      intersect(
        test_donors,
        train_donors
      )
    ) > 0L
  ) {
    stop("Donor leakage detected.")
  }
}

cat("Fixed donor-grouped 5-fold frame: PASS\n")


# ------------------------------------------------------------
# 13. HELPERS
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  ok <- (
    is.finite(score) &
    !is.na(y)
  )

  y <- y[ok]
  score <- score[ok]

  n_pos <- sum(
    y == 1L
  )

  n_neg <- sum(
    y == 0L
  )

  if (
    n_pos == 0L ||
    n_neg == 0L
  ) {
    return(
      NA_real_
    )
  }

  r <- rank(
    score,
    ties.method = "average"
  )

  (
    sum(
      r[
        y == 1L
      ]
    ) -
      n_pos *
      (n_pos + 1) /
      2
  ) /
    (
      n_pos *
      n_neg
    )
}


hedges_g <- function(
  y,
  score
) {

  ok <- (
    is.finite(score) &
    !is.na(y)
  )

  y <- y[ok]
  score <- score[ok]

  x1 <- score[
    y == 1L
  ]

  x0 <- score[
    y == 0L
  ]

  n1 <- length(
    x1
  )

  n0 <- length(
    x0
  )

  if (
    n1 < 2L ||
    n0 < 2L
  ) {
    return(
      NA_real_
    )
  }

  pooled_var <- (
    (
      n1 - 1
    ) *
      var(x1) +
      (
        n0 - 1
      ) *
      var(x0)
  ) /
    (
      n1 +
      n0 -
      2
    )

  if (
    !is.finite(pooled_var) ||
    pooled_var <= 0
  ) {
    return(
      NA_real_
    )
  }

  d <- (
    mean(x1) -
      mean(x0)
  ) /
    sqrt(
      pooled_var
    )

  df <- (
    n1 +
    n0 -
    2
  )

  J <- (
    1 -
      3 /
      (
        4 * df -
        1
      )
  )

  J * d
}


donor_average <- function(
  sample_score,
  sample_y
) {

  donor_list <- split(
    seq_along(
      sample_score
    ),
    META$BrNum
  )

  out <- do.call(
    rbind,
    lapply(
      names(donor_list),
      function(id) {

        idx <- donor_list[
          id
        ]

        idx <- idx[
          1
        ]

        idx <- unlist(
          idx,
          use.names = FALSE
        )

        dy <- unique(
          sample_y[
            idx
          ]
        )

        if (
          length(dy) != 1L
        ) {
          stop(
            "Permuted diagnosis inconsistent within donor."
          )
        }

        data.frame(
          BrNum = id,
          y = dy,
          score = mean(
            sample_score[
              idx
            ],
            na.rm = TRUE
          ),
          stringsAsFactors = FALSE
        )
      }
    )
  )

  rownames(out) <- NULL

  out
}


extract_gene_coefficients <- function(
  cvfit,
  lambda_value
) {

  cm <- as.matrix(
    coef(
      cvfit$glmnet.fit,
      s = lambda_value
    )
  )

  cv <- cm[
    ,
    1
  ]

  names(cv) <- rownames(
    cm
  )

  shared_names <- paste0(
    "SHARED__",
    GENE_IDS
  )

  dev_names <- paste0(
    "DEV__",
    GENE_IDS
  )

  shared <- as.numeric(
    cv[
      shared_names
    ]
  )

  deviation <- as.numeric(
    cv[
      dev_names
    ]
  )

  if (
    anyNA(shared) ||
    anyNA(deviation)
  ) {
    stop(
      "Gene coefficient extraction failed."
    )
  }

  list(
    shared = shared,
    deviation = deviation
  )
}


run_one_cv <- function(
  y_vector,
  seed_value
) {

  set.seed(
    seed_value
  )

  fit_start <- proc.time()[3]

  CVFIT <- glmnet::cv.glmnet(
    x = X_MODEL,
    y = y_vector,
    family = "binomial",
    weights = OBS_WEIGHT,
    foldid = SAMPLE_FOLDID,
    type.measure = "auc",
    alpha = FIXED_ALPHA,
    penalty.factor =
      PENALTY_FACTOR,
    standardize = TRUE,
    intercept = TRUE,
    nlambda = N_LAMBDA,
    keep = TRUE,
    parallel = FALSE
  )

  valid <- is.finite(
    CVFIT$cvm
  )

  if (!any(valid)) {
    stop(
      "No valid CV AUC."
    )
  }

  best_index <- which.max(
    ifelse(
      valid,
      CVFIT$cvm,
      -Inf
    )
  )

  best_lambda <- CVFIT$lambda[
    best_index
  ]

  best_cv_auc <- CVFIT$cvm[
    best_index
  ]

  PREVAL <- CVFIT$fit.preval[
    ,
    best_index
  ]

  PREVAL <- as.numeric(
    PREVAL
  )

  if (
    length(PREVAL) !=
      length(y_vector)
  ) {
    stop(
      "Prevalidated prediction length mismatch."
    )
  }

  if (
    any(!is.finite(PREVAL))
  ) {
    stop(
      "Non-finite prevalidated predictions."
    )
  }

  sample_auc <- auc_rank(
    y_vector,
    PREVAL
  )

  donor_data <- donor_average(
    PREVAL,
    y_vector
  )

  donor_auc <- auc_rank(
    donor_data$y,
    donor_data$score
  )

  donor_g <- hedges_g(
    donor_data$y,
    donor_data$score
  )

  dlpfc_idx <- (
    META$Region == "DLPFC"
  )

  hippo_idx <- (
    META$Region == "HIPPO"
  )

  dlpfc_auc <- auc_rank(
    y_vector[
      dlpfc_idx
    ],
    PREVAL[
      dlpfc_idx
    ]
  )

  hippo_auc <- auc_rank(
    y_vector[
      hippo_idx
    ],
    PREVAL[
      hippo_idx
    ]
  )

  betas <- extract_gene_coefficients(
    CVFIT,
    best_lambda
  )

  runtime <- (
    proc.time()[3] -
      fit_start
  )

  list(
    best_lambda =
      best_lambda,

    best_cv_auc =
      best_cv_auc,

    sample_auc =
      sample_auc,

    donor_auc =
      donor_auc,

    DLPFC_auc =
      dlpfc_auc,

    HIPPO_auc =
      hippo_auc,

    donor_hedges_g =
      donor_g,

    shared_coef =
      betas$shared,

    deviation_coef =
      betas$deviation,

    n_shared_nonzero =
      sum(
        abs(
          betas$shared
        ) > ZERO_TOL
      ),

    n_deviation_nonzero =
      sum(
        abs(
          betas$deviation
        ) > ZERO_TOL
      ),

    runtime_seconds =
      runtime
  )
}


# ------------------------------------------------------------
# 14. MATCHED OBSERVED REFERENCE
# ------------------------------------------------------------

OBSERVED_PATH <- file.path(
  OUT_DIR,
  "observed_reference_same_5fold_v1.rds"
)

if (!file.exists(
  OBSERVED_PATH
)) {

  cat(
    "\nRunning matched observed-label reference...\n"
  )

  OBSERVED_RESULT <- run_one_cv(
    Y_ORIGINAL,
    BASE_SEED + 5000L
  )

  saveRDS(
    OBSERVED_RESULT,
    OBSERVED_PATH,
    compress = "gzip"
  )

} else {

  cat(
    "\nObserved reference already exists; loading checkpoint.\n"
  )

  OBSERVED_RESULT <- readRDS(
    OBSERVED_PATH
  )
}

cat(
  "Observed matched donor AUROC: ",
  round(
    OBSERVED_RESULT$donor_auc,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Observed matched sample AUROC: ",
  round(
    OBSERVED_RESULT$sample_auc,
    4
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 15. DONOR-LEVEL LABEL VECTOR
# ------------------------------------------------------------

ORIGINAL_DONOR_Y <- DONOR_TABLE$Y_original

N_CASE_DONORS <- sum(
  ORIGINAL_DONOR_Y == 1L
)

N_CONTROL_DONORS <- sum(
  ORIGINAL_DONOR_Y == 0L
)

stopifnot(
  N_CASE_DONORS == 177L,
  N_CONTROL_DONORS == 277L
)


# ------------------------------------------------------------
# 16. 100 DONOR-LEVEL PERMUTATIONS
# ------------------------------------------------------------

for (
  perm_id in seq_len(
    N_PERM
  )
) {

  RESULT_PATH <- file.path(
    PERM_DIR,
    sprintf(
      "perm_%03d.rds",
      perm_id
    )
  )

  if (file.exists(
    RESULT_PATH
  )) {

    if (
      perm_id %% 10L == 0L
    ) {
      cat(
        sprintf(
          "[%03d/%03d] already complete; skipping.\n",
          perm_id,
          N_PERM
        )
      )
    }

    next
  }

  set.seed(
    BASE_SEED +
      10000L +
      perm_id
  )

  PERM_DONOR_Y <- sample(
    ORIGINAL_DONOR_Y,
    length(
      ORIGINAL_DONOR_Y
    ),
    replace = FALSE
  )

  stopifnot(
    sum(
      PERM_DONOR_Y == 1L
    ) == N_CASE_DONORS,
    sum(
      PERM_DONOR_Y == 0L
    ) == N_CONTROL_DONORS
  )

  PERM_LOOKUP <- setNames(
    PERM_DONOR_Y,
    DONOR_TABLE$BrNum
  )

  Y_PERM <- as.integer(
    PERM_LOOKUP[
      as.character(
        META$BrNum
      )
    ]
  )

  if (anyNA(
    Y_PERM
  )) {
    stop(
      "Permuted sample-label mapping failed."
    )
  }

  for (
    donor_id in unique(
      META$BrNum
    )
  ) {

    dy <- unique(
      Y_PERM[
        META$BrNum ==
          donor_id
      ]
    )

    if (
      length(dy) != 1L
    ) {
      stop(
        "Paired donor received inconsistent permutation labels."
      )
    }
  }

  cat(
    sprintf(
      "[%03d/%03d] permutation ... ",
      perm_id,
      N_PERM
    )
  )

  flush.console()

  z <- run_one_cv(
    Y_PERM,
    BASE_SEED +
      20000L +
      perm_id
  )

  RESULT <- list(
    permutation_id =
      perm_id,

    donor_auc =
      z$donor_auc,

    sample_auc =
      z$sample_auc,

    DLPFC_auc =
      z$DLPFC_auc,

    HIPPO_auc =
      z$HIPPO_auc,

    donor_hedges_g =
      z$donor_hedges_g,

    best_cv_auc =
      z$best_cv_auc,

    best_lambda =
      z$best_lambda,

    n_shared_nonzero =
      z$n_shared_nonzero,

    n_deviation_nonzero =
      z$n_deviation_nonzero,

    shared_coef =
      z$shared_coef,

    deviation_coef =
      z$deviation_coef,

    runtime_seconds =
      z$runtime_seconds
  )

  saveRDS(
    RESULT,
    RESULT_PATH,
    compress = "gzip"
  )

  cat(
    sprintf(
      "donorAUC=%.4f shared=%d dev=%d time=%.1fs\n",
      RESULT$donor_auc,
      RESULT$n_shared_nonzero,
      RESULT$n_deviation_nonzero,
      RESULT$runtime_seconds
    )
  )

  rm(
    z,
    RESULT,
    Y_PERM,
    PERM_DONOR_Y,
    PERM_LOOKUP
  )

  invisible(gc())
}


# ------------------------------------------------------------
# 17. VERIFY COMPLETION
# ------------------------------------------------------------

PERM_FILES <- list.files(
  PERM_DIR,
  pattern = "^perm_[0-9]{3}\\.rds$",
  full.names = TRUE
)

PERM_FILES <- sort(
  PERM_FILES
)

if (
  length(
    PERM_FILES
  ) != N_PERM
) {

  cat(
    "\nCompleted permutations: ",
    length(PERM_FILES),
    " / ",
    N_PERM,
    "\n",
    sep = ""
  )

  cat(
    "Re-run the SAME cell to resume safely.\n"
  )

  stop(
    "Permutation analysis incomplete."
  )
}

cat(
  "\nAll ",
  N_PERM,
  " donor-level permutations complete.\n",
  sep = ""
)


# ------------------------------------------------------------
# 18. AGGREGATE NULL RESULTS
# ------------------------------------------------------------

NULL_SHARED <- matrix(
  0,
  nrow = N_GENES,
  ncol = N_PERM
)

NULL_DEV <- matrix(
  0,
  nrow = N_GENES,
  ncol = N_PERM
)

rownames(
  NULL_SHARED
) <- GENE_IDS

rownames(
  NULL_DEV
) <- GENE_IDS

PERM_SUMMARY_LIST <- vector(
  "list",
  N_PERM
)

for (
  i in seq_along(
    PERM_FILES
  )
) {

  z <- readRDS(
    PERM_FILES[i]
  )

  NULL_SHARED[
    ,
    i
  ] <- z$shared_coef

  NULL_DEV[
    ,
    i
  ] <- z$deviation_coef

  PERM_SUMMARY_LIST[
    i
  ] <- list(
    data.frame(
      permutation_id =
        z$permutation_id,
      donor_auc =
        z$donor_auc,
      sample_auc =
        z$sample_auc,
      DLPFC_auc =
        z$DLPFC_auc,
      HIPPO_auc =
        z$HIPPO_auc,
      donor_hedges_g =
        z$donor_hedges_g,
      best_cv_auc =
        z$best_cv_auc,
      best_lambda =
        z$best_lambda,
      n_shared_nonzero =
        z$n_shared_nonzero,
      n_deviation_nonzero =
        z$n_deviation_nonzero,
      runtime_seconds =
        z$runtime_seconds,
      stringsAsFactors = FALSE
    )
  )
}

PERM_SUMMARY <- do.call(
  rbind,
  PERM_SUMMARY_LIST
)

rownames(
  PERM_SUMMARY
) <- NULL


# ------------------------------------------------------------
# 19. NULL GENE STABILITY
# ------------------------------------------------------------

NULL_SHARED_POS <- rowSums(
  NULL_SHARED > ZERO_TOL
)

NULL_SHARED_NEG <- rowSums(
  NULL_SHARED < -ZERO_TOL
)

NULL_SHARED_SELECTED <- (
  NULL_SHARED_POS +
  NULL_SHARED_NEG
)

NULL_SHARED_SELECTION_PROB <- (
  NULL_SHARED_SELECTED /
  N_PERM
)

NULL_SHARED_SIGN_CONSISTENCY <- ifelse(
  NULL_SHARED_SELECTED > 0L,
  pmax(
    NULL_SHARED_POS,
    NULL_SHARED_NEG
  ) /
    NULL_SHARED_SELECTED,
  NA_real_
)


NULL_DEV_POS <- rowSums(
  NULL_DEV > ZERO_TOL
)

NULL_DEV_NEG <- rowSums(
  NULL_DEV < -ZERO_TOL
)

NULL_DEV_SELECTED <- (
  NULL_DEV_POS +
  NULL_DEV_NEG
)

NULL_DEV_SELECTION_PROB <- (
  NULL_DEV_SELECTED /
  N_PERM
)

NULL_DEV_SIGN_CONSISTENCY <- ifelse(
  NULL_DEV_SELECTED > 0L,
  pmax(
    NULL_DEV_POS,
    NULL_DEV_NEG
  ) /
    NULL_DEV_SELECTED,
  NA_real_
)


NULL_GENE_TABLE <- data.frame(

  ensemblID_stable =
    GENE_IDS,

  gene_symbol =
    ANNOTATION$gene_symbol,

  gene_type =
    ANNOTATION$gene_type,

  null_shared_selection_count =
    NULL_SHARED_SELECTED,

  null_shared_selection_probability =
    NULL_SHARED_SELECTION_PROB,

  null_shared_positive_count =
    NULL_SHARED_POS,

  null_shared_negative_count =
    NULL_SHARED_NEG,

  null_shared_sign_consistency =
    NULL_SHARED_SIGN_CONSISTENCY,

  null_deviation_selection_count =
    NULL_DEV_SELECTED,

  null_deviation_selection_probability =
    NULL_DEV_SELECTION_PROB,

  null_deviation_sign_consistency =
    NULL_DEV_SIGN_CONSISTENCY,

  stringsAsFactors = FALSE
)


NULL_GENE_TABLE$null_stable_like_shared <- (

  NULL_GENE_TABLE$null_shared_selection_probability >=
    STABILITY_SELECTION_THRESHOLD

  &

  !is.na(
    NULL_GENE_TABLE$null_shared_sign_consistency
  )

  &

  NULL_GENE_TABLE$null_shared_sign_consistency >=
    STABILITY_SIGN_THRESHOLD
)


NULL_GENE_TABLE$null_stable_like_deviation <- (

  NULL_GENE_TABLE$null_deviation_selection_probability >=
    STABILITY_SELECTION_THRESHOLD

  &

  !is.na(
    NULL_GENE_TABLE$null_deviation_sign_consistency
  )

  &

  NULL_GENE_TABLE$null_deviation_sign_consistency >=
    STABILITY_SIGN_THRESHOLD
)


# ------------------------------------------------------------
# 20. NULL BEHAVIOR OF THE 24 HALF-SAMPLE CANDIDATES
# ------------------------------------------------------------

CANDIDATE_INDEX <- match(
  HALF_STABLE$ensemblID_stable,
  GENE_IDS
)

if (anyNA(
  CANDIDATE_INDEX
)) {
  stop(
    "24-gene candidate set could not be aligned."
  )
}

CANDIDATE_NULL_TABLE <- NULL_GENE_TABLE[
  CANDIDATE_INDEX,
  ,
  drop = FALSE
]

CANDIDATE_NULL_TABLE$observed_half_sample_selection_probability <-
  HALF_STABLE$shared_selection_probability

CANDIDATE_NULL_TABLE$observed_half_sample_sign_consistency <-
  HALF_STABLE$shared_sign_consistency

CANDIDATE_NULL_TABLE$observed_half_sample_median_beta <-
  HALF_STABLE$shared_median_nonzero_beta


# ------------------------------------------------------------
# 21. EMPIRICAL P FOR OBSERVED MATCHED AUROC
# ------------------------------------------------------------

EMPIRICAL_P_DONOR_AUC <- (
  1 +
    sum(
      PERM_SUMMARY$donor_auc >=
        OBSERVED_RESULT$donor_auc,
      na.rm = TRUE
    )
) /
  (
    N_PERM +
    1
  )


EMPIRICAL_P_SAMPLE_AUC <- (
  1 +
    sum(
      PERM_SUMMARY$sample_auc >=
        OBSERVED_RESULT$sample_auc,
      na.rm = TRUE
    )
) /
  (
    N_PERM +
    1
  )


OBSERVED_ABS_G <- abs(
  OBSERVED_RESULT$donor_hedges_g
)

EMPIRICAL_P_ABS_G <- (
  1 +
    sum(
      abs(
        PERM_SUMMARY$donor_hedges_g
      ) >=
        OBSERVED_ABS_G,
      na.rm = TRUE
    )
) /
  (
    N_PERM +
    1
  )


# ------------------------------------------------------------
# 22. SUMMARY
# ------------------------------------------------------------

NULL_STABLE_SHARED_N <- sum(
  NULL_GENE_TABLE$null_stable_like_shared
)

NULL_STABLE_DEV_N <- sum(
  NULL_GENE_TABLE$null_stable_like_deviation
)

MAX_NULL_SHARED_PROB <- max(
  NULL_GENE_TABLE$null_shared_selection_probability,
  na.rm = TRUE
)

MAX_NULL_DEV_PROB <- max(
  NULL_GENE_TABLE$null_deviation_selection_probability,
  na.rm = TRUE
)

MAX_CANDIDATE24_NULL_PROB <- max(
  CANDIDATE_NULL_TABLE$null_shared_selection_probability,
  na.rm = TRUE
)

MEDIAN_CANDIDATE24_NULL_PROB <- median(
  CANDIDATE_NULL_TABLE$null_shared_selection_probability,
  na.rm = TRUE
)

SUMMARY_TABLE <- data.frame(

  metric = c(

    "Permutations",

    "Observed_reference_donor_AUROC",

    "Null_donor_AUROC_mean",

    "Null_donor_AUROC_SD",

    "Null_donor_AUROC_min",

    "Null_donor_AUROC_max",

    "Empirical_P_observed_donor_AUROC",

    "Observed_reference_sample_AUROC",

    "Null_sample_AUROC_mean",

    "Empirical_P_observed_sample_AUROC",

    "Observed_reference_donor_Hedges_g",

    "Null_abs_Hedges_g_mean",

    "Empirical_P_observed_abs_Hedges_g",

    "Mean_null_selected_shared_genes",

    "Median_null_selected_shared_genes",

    "Mean_null_selected_deviation_genes",

    "Median_null_selected_deviation_genes",

    "Max_null_shared_selection_probability",

    "Max_null_deviation_selection_probability",

    "Null_stable_like_shared_genes",

    "Null_stable_like_deviation_genes",

    "Candidate24_max_null_selection_probability",

    "Candidate24_median_null_selection_probability"
  ),

  value = c(

    N_PERM,

    OBSERVED_RESULT$donor_auc,

    mean(
      PERM_SUMMARY$donor_auc,
      na.rm = TRUE
    ),

    sd(
      PERM_SUMMARY$donor_auc,
      na.rm = TRUE
    ),

    min(
      PERM_SUMMARY$donor_auc,
      na.rm = TRUE
    ),

    max(
      PERM_SUMMARY$donor_auc,
      na.rm = TRUE
    ),

    EMPIRICAL_P_DONOR_AUC,

    OBSERVED_RESULT$sample_auc,

    mean(
      PERM_SUMMARY$sample_auc,
      na.rm = TRUE
    ),

    EMPIRICAL_P_SAMPLE_AUC,

    OBSERVED_RESULT$donor_hedges_g,

    mean(
      abs(
        PERM_SUMMARY$donor_hedges_g
      ),
      na.rm = TRUE
    ),

    EMPIRICAL_P_ABS_G,

    mean(
      PERM_SUMMARY$n_shared_nonzero
    ),

    median(
      PERM_SUMMARY$n_shared_nonzero
    ),

    mean(
      PERM_SUMMARY$n_deviation_nonzero
    ),

    median(
      PERM_SUMMARY$n_deviation_nonzero
    ),

    MAX_NULL_SHARED_PROB,

    MAX_NULL_DEV_PROB,

    NULL_STABLE_SHARED_N,

    NULL_STABLE_DEV_N,

    MAX_CANDIDATE24_NULL_PROB,

    MEDIAN_CANDIDATE24_NULL_PROB
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 23. SAVE
# ------------------------------------------------------------

PERM_SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step06c_permutation_run_summary_v1.csv"
)

NULL_GENE_PATH <- file.path(
  OUT_DIR,
  "step06c_null_gene_stability_v1.csv"
)

CANDIDATE24_PATH <- file.path(
  OUT_DIR,
  "step06c_candidate24_null_behavior_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step06c_permutation_null_summary_v1.csv"
)

NULL_MATRIX_PATH <- file.path(
  OUT_DIR,
  "step06c_null_coefficient_matrices_v1.rds"
)

LOCK_PATH <- file.path(
  OUT_DIR,
  "step06c_permutation_definition_v1.txt"
)


write.csv(
  PERM_SUMMARY,
  PERM_SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  NULL_GENE_TABLE,
  NULL_GENE_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  CANDIDATE_NULL_TABLE,
  CANDIDATE24_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  SUMMARY_TABLE,
  SUMMARY_PATH,
  row.names = FALSE
)

saveRDS(
  list(
    gene_ids =
      GENE_IDS,
    null_shared_coefficients =
      NULL_SHARED,
    null_deviation_coefficients =
      NULL_DEV
  ),
  NULL_MATRIX_PATH,
  compress = "gzip"
)

writeLines(
  c(
    "STEP 06C donor-level permutation null",
    "",
    "permutations=100",
    "diagnosis_permuted_at_donor_level=TRUE",
    "paired_region_samples_receive_same_permuted_label=TRUE",
    "fixed_donor_grouped_cv_folds=5",
    "alpha_fixed=0.10",
    "deviation_penalty_fixed=1",
    "lambda_selected_by_cv_within_each_permutation=TRUE",
    "external_data_used=FALSE",
    "",
    paste0(
      "observed_reference_donor_auc=",
      OBSERVED_RESULT$donor_auc
    ),
    paste0(
      "empirical_p_donor_auc=",
      EMPIRICAL_P_DONOR_AUC
    )
  ),
  LOCK_PATH
)


# ------------------------------------------------------------
# 24. HARD CHECKS
# ------------------------------------------------------------

stopifnot(
  nrow(PERM_SUMMARY) == 100L,
  nrow(NULL_GENE_TABLE) == 23740L,
  nrow(CANDIDATE_NULL_TABLE) == 24L,
  EMPIRICAL_P_DONOR_AUC >=
    1 / 101,
  EMPIRICAL_P_DONOR_AUC <=
    1,
  file.exists(SUMMARY_PATH),
  file.exists(NULL_GENE_PATH)
)


# ------------------------------------------------------------
# 25. OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START

cat("\n============================================================\n")
cat("STEP 06C — DONOR-LEVEL PERMUTATION NULL COMPLETE\n")
cat("============================================================\n")

print(
  SUMMARY_TABLE,
  row.names = FALSE
)

cat(
  "\nObserved matched donor AUROC: ",
  round(
    OBSERVED_RESULT$donor_auc,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Null donor AUROC: mean ",
  round(
    mean(
      PERM_SUMMARY$donor_auc
    ),
    4
  ),
  " +/- ",
  round(
    sd(
      PERM_SUMMARY$donor_auc
    ),
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Null donor AUROC range: ",
  round(
    min(
      PERM_SUMMARY$donor_auc
    ),
    4
  ),
  " - ",
  round(
    max(
      PERM_SUMMARY$donor_auc
    ),
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Empirical P (donor AUROC): ",
  format(
    EMPIRICAL_P_DONOR_AUC,
    digits = 4
  ),
  "\n",
  sep = ""
)

cat(
  "\nNull stable-like shared genes: ",
  NULL_STABLE_SHARED_N,
  "\n",
  sep = ""
)

cat(
  "Null stable-like deviation genes: ",
  NULL_STABLE_DEV_N,
  "\n",
  sep = ""
)

cat(
  "Max null shared selection probability: ",
  round(
    MAX_NULL_SHARED_PROB,
    3
  ),
  "\n",
  sep = ""
)

cat(
  "\n24-gene core: max null selection probability = ",
  round(
    MAX_CANDIDATE24_NULL_PROB,
    3
  ),
  "\n",
  sep = ""
)

cat(
  "24-gene core: median null selection probability = ",
  round(
    MEDIAN_CANDIDATE24_NULL_PROB,
    3
  ),
  "\n",
  sep = ""
)

cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS / 60,
    1
  ),
  " minutes\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 06C — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
glmnet version: 5.1
Loading canonical discovery data...
Discovery matrix: 23740 genes x 755 samples
High-confidence half-sample candidate genes: 24
Unpenalized covariates: 17
Building shared/deviation matrix...
Total predictors: 47497
Fixed donor-grouped 5-fold frame: PASS

Running matched observed-label reference...
Observed matched donor AUROC: 0.7698
Observed matched sample AUROC: 0.7625
[001/100] permutation ... donorAUC=0.5633 shared=3 dev=0 time=18.9s
[002/100] permutation ... donorAUC=0.5052 shared=1252 dev=73 time=14.9s
[003/100] permutation ... donorAUC=0.5093 shared=72 dev=4 time=22.6s
[004/100] permutation ... donorAUC=0.4966 shared=915 dev=42 time=19.1s
[005/100] permutation ... donorAUC=0.4955 shared=1650 dev=106 time=27.3s
[006/100] permutation ... donorAUC=0.4995 shared=0 dev=0 time=17.8s
[007/100] permutation ... donorAUC=0.5419 shared=821 dev=28 time=15.5s
[008/100] permutation ... donorAUC=0.5161 shared=2088 dev=137 time=23.3s
[009/100] permutat

In [17]:
# ============================================================
# STEP 06D — FINAL STABLE SHARED-STATE FREEZE
#
# Final rule:
#
#   1. Stable in 25 repeated outer models
#      selection frequency >= 0.80
#      sign consistency >= 0.90
#
#   2. Stable in 200 complementary half-sample fits
#      selection probability >= 0.80
#      sign consistency >= 0.90
#
#   3. Coefficient direction agrees between the two analyses
#
# Permutation results are used as NULL VALIDATION only.
# They do not add/remove genes.
#
# NO EXTERNAL EXPRESSION OR OUTCOME DATA USED.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUT PATHS
# ------------------------------------------------------------

OUTER_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "stability_v1",
  "step05a_stable_shared_genes_v1.csv"
)

HALF_PATH <- file.path(
  PROJECT_ROOT,
  "07_stability",
  "complementary_half_sample_v1",
  "step06b_half_sample_candidate_shared_v1.csv"
)

NULL_CANDIDATE_PATH <- file.path(
  PROJECT_ROOT,
  "07_stability",
  "permutation_null_v1",
  "step06c_candidate24_null_behavior_v1.csv"
)

NULL_SUMMARY_PATH <- file.path(
  PROJECT_ROOT,
  "07_stability",
  "permutation_null_v1",
  "step06c_permutation_null_summary_v1.csv"
)

REQUIRED <- c(
  OUTER_PATH,
  HALF_PATH,
  NULL_CANDIDATE_PATH,
  NULL_SUMMARY_PATH
)

if (!all(file.exists(REQUIRED))) {

  cat(
    "Missing files:\n"
  )

  print(
    REQUIRED[
      !file.exists(REQUIRED)
    ]
  )

  stop(
    "Required stability artifact missing."
  )
}


# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

OUTER <- read.csv(
  OUTER_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

HALF <- read.csv(
  HALF_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

NULL24 <- read.csv(
  NULL_CANDIDATE_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

NULL_SUMMARY <- read.csv(
  NULL_SUMMARY_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


cat(
  "Outer-CV stable genes loaded: ",
  nrow(OUTER),
  "\n",
  sep = ""
)

cat(
  "Half-sample stable genes loaded: ",
  nrow(HALF),
  "\n",
  sep = ""
)

cat(
  "Permutation candidate rows loaded: ",
  nrow(NULL24),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 3. REQUIRED COLUMN CHECK
# ------------------------------------------------------------

OUTER_REQUIRED <- c(
  "ensemblID_stable",
  "gene_symbol",
  "gene_type",
  "shared_selection_count",
  "shared_selection_frequency",
  "shared_sign_consistency",
  "shared_median_nonzero_beta"
)

HALF_REQUIRED <- c(
  "ensemblID_stable",
  "gene_symbol",
  "gene_type",
  "shared_selection_count",
  "shared_selection_probability",
  "shared_sign_consistency",
  "shared_median_nonzero_beta"
)

NULL_REQUIRED <- c(
  "ensemblID_stable",
  "null_shared_selection_count",
  "null_shared_selection_probability",
  "null_shared_sign_consistency"
)


MISSING_OUTER <- setdiff(
  OUTER_REQUIRED,
  names(OUTER)
)

MISSING_HALF <- setdiff(
  HALF_REQUIRED,
  names(HALF)
)

MISSING_NULL <- setdiff(
  NULL_REQUIRED,
  names(NULL24)
)


if (length(MISSING_OUTER) > 0L) {

  cat(
    "Missing OUTER columns:\n"
  )

  print(MISSING_OUTER)

  stop(
    "Outer stability schema mismatch."
  )
}


if (length(MISSING_HALF) > 0L) {

  cat(
    "Missing HALF columns:\n"
  )

  print(MISSING_HALF)

  stop(
    "Half-sample stability schema mismatch."
  )
}


if (length(MISSING_NULL) > 0L) {

  cat(
    "Missing NULL columns:\n"
  )

  print(MISSING_NULL)

  stop(
    "Permutation-null schema mismatch."
  )
}


# ------------------------------------------------------------
# 4. BASIC INTEGRITY
# ------------------------------------------------------------

if (
  anyDuplicated(
    OUTER$ensemblID_stable
  )
) {
  stop(
    "Duplicated genes in outer stability file."
  )
}

if (
  anyDuplicated(
    HALF$ensemblID_stable
  )
) {
  stop(
    "Duplicated genes in half-sample stability file."
  )
}

if (
  anyDuplicated(
    NULL24$ensemblID_stable
  )
) {
  stop(
    "Duplicated genes in permutation candidate file."
  )
}


stopifnot(
  nrow(OUTER) == 403L,
  nrow(HALF) == 24L,
  nrow(NULL24) == 24L
)


# ------------------------------------------------------------
# 5. VERIFY STABILITY THRESHOLDS
# ------------------------------------------------------------

if (
  any(
    OUTER$shared_selection_frequency <
      0.80
  )
) {
  stop(
    "Outer stable file contains gene below 0.80 selection threshold."
  )
}

if (
  any(
    OUTER$shared_sign_consistency <
      0.90
  )
) {
  stop(
    "Outer stable file contains gene below 0.90 sign threshold."
  )
}


if (
  any(
    HALF$shared_selection_probability <
      0.80
  )
) {
  stop(
    "Half-sample stable file contains gene below 0.80 threshold."
  )
}

if (
  any(
    HALF$shared_sign_consistency <
      0.90
  )
) {
  stop(
    "Half-sample stable file contains gene below 0.90 sign threshold."
  )
}


# ------------------------------------------------------------
# 6. OUTER × HALF INTERSECTION
# ------------------------------------------------------------

INTERSECTION_IDS <- intersect(
  OUTER$ensemblID_stable,
  HALF$ensemblID_stable
)

cat(
  "\nOuter / half-sample intersection: ",
  length(INTERSECTION_IDS),
  "\n",
  sep = ""
)


if (
  length(INTERSECTION_IDS) !=
    nrow(HALF)
) {

  stop(
    paste0(
      "Not every half-sample stable gene is contained in ",
      "the outer-CV stable set."
    )
  )
}


# ------------------------------------------------------------
# 7. ALIGN TABLES
# ------------------------------------------------------------

OUTER_IDX <- match(
  INTERSECTION_IDS,
  OUTER$ensemblID_stable
)

HALF_IDX <- match(
  INTERSECTION_IDS,
  HALF$ensemblID_stable
)

NULL_IDX <- match(
  INTERSECTION_IDS,
  NULL24$ensemblID_stable
)


if (
  anyNA(OUTER_IDX) ||
  anyNA(HALF_IDX) ||
  anyNA(NULL_IDX)
) {
  stop(
    "Gene alignment failed."
  )
}


OUTER_CORE <- OUTER[
  OUTER_IDX,
  ,
  drop = FALSE
]

HALF_CORE <- HALF[
  HALF_IDX,
  ,
  drop = FALSE
]

NULL_CORE <- NULL24[
  NULL_IDX,
  ,
  drop = FALSE
]


# ------------------------------------------------------------
# 8. DETERMINE DIRECTIONS
# ------------------------------------------------------------

OUTER_SIGN <- sign(
  OUTER_CORE$shared_median_nonzero_beta
)

HALF_SIGN <- sign(
  HALF_CORE$shared_median_nonzero_beta
)


if (
  any(
    OUTER_SIGN == 0
  ) ||
  any(
    HALF_SIGN == 0
  )
) {
  stop(
    "Zero median coefficient found in final candidate core."
  )
}


SIGN_AGREEMENT <- (
  OUTER_SIGN ==
    HALF_SIGN
)


cat(
  "Direction agreement: ",
  sum(SIGN_AGREEMENT),
  " / ",
  length(SIGN_AGREEMENT),
  "\n",
  sep = ""
)


if (!all(
  SIGN_AGREEMENT
)) {

  BAD_IDS <- INTERSECTION_IDS[
    !SIGN_AGREEMENT
  ]

  cat(
    "\nGenes with outer/half sign disagreement:\n"
  )

  print(
    BAD_IDS
  )

  stop(
    paste0(
      "Final freeze stopped because direction disagreement ",
      "was detected."
    )
  )
}


# ------------------------------------------------------------
# 9. BUILD FINAL STABLE SHARED STATE
# ------------------------------------------------------------

FINAL <- data.frame(

  ensemblID_stable =
    INTERSECTION_IDS,

  gene_symbol =
    HALF_CORE$gene_symbol,

  gene_type =
    HALF_CORE$gene_type,

  direction_code =
    HALF_SIGN,

  direction =
    ifelse(
      HALF_SIGN > 0,
      "Positive",
      "Negative"
    ),

  outer_selection_count =
    OUTER_CORE$shared_selection_count,

  outer_selection_frequency =
    OUTER_CORE$shared_selection_frequency,

  outer_sign_consistency =
    OUTER_CORE$shared_sign_consistency,

  outer_median_nonzero_beta =
    OUTER_CORE$shared_median_nonzero_beta,

  half_sample_selection_count =
    HALF_CORE$shared_selection_count,

  half_sample_selection_probability =
    HALF_CORE$shared_selection_probability,

  half_sample_sign_consistency =
    HALF_CORE$shared_sign_consistency,

  half_sample_median_nonzero_beta =
    HALF_CORE$shared_median_nonzero_beta,

  null_selection_count =
    NULL_CORE$null_shared_selection_count,

  null_selection_probability =
    NULL_CORE$null_shared_selection_probability,

  null_sign_consistency =
    NULL_CORE$null_shared_sign_consistency,

  outer_half_direction_agreement =
    SIGN_AGREEMENT,

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 10. ADD ROBUSTNESS DESCRIPTORS
#
# These are descriptive only.
# They are NOT new gene-selection criteria.
# ------------------------------------------------------------

FINAL$minimum_observed_selection_probability <- pmin(
  FINAL$outer_selection_frequency,
  FINAL$half_sample_selection_probability
)


FINAL$minimum_observed_sign_consistency <- pmin(
  FINAL$outer_sign_consistency,
  FINAL$half_sample_sign_consistency
)


FINAL$null_selection_probability_gap <- (
  FINAL$half_sample_selection_probability -
    FINAL$null_selection_probability
)


# ------------------------------------------------------------
# 11. ORDER FINAL CORE
# ------------------------------------------------------------

FINAL <- FINAL[
  order(
    -FINAL$half_sample_selection_probability,
    -FINAL$outer_selection_frequency,
    -FINAL$minimum_observed_sign_consistency,
    -abs(
      FINAL$half_sample_median_nonzero_beta
    )
  ),
  ,
  drop = FALSE
]

rownames(FINAL) <- NULL


# ------------------------------------------------------------
# 12. FINAL FREEZE CHECKS
# ------------------------------------------------------------

stopifnot(
  nrow(FINAL) == 24L,
  all(
    FINAL$outer_selection_frequency >=
      0.80
  ),
  all(
    FINAL$outer_sign_consistency >=
      0.90
  ),
  all(
    FINAL$half_sample_selection_probability >=
      0.80
  ),
  all(
    FINAL$half_sample_sign_consistency >=
      0.90
  ),
  all(
    FINAL$outer_half_direction_agreement
  )
)


N_POS <- sum(
  FINAL$direction_code == 1L
)

N_NEG <- sum(
  FINAL$direction_code == -1L
)

N_PC <- sum(
  FINAL$gene_type ==
    "protein_coding"
)


# ------------------------------------------------------------
# 13. PERMUTATION GLOBAL SUMMARY
# ------------------------------------------------------------

get_summary_value <- function(
  metric_name
) {

  idx <- which(
    NULL_SUMMARY$metric ==
      metric_name
  )

  if (length(idx) != 1L) {
    return(
      NA_real_
    )
  }

  as.numeric(
    NULL_SUMMARY$value[
      idx
    ]
  )
}


OBS_AUC <- get_summary_value(
  "Observed_reference_donor_AUROC"
)

NULL_AUC_MEAN <- get_summary_value(
  "Null_donor_AUROC_mean"
)

NULL_AUC_MAX <- get_summary_value(
  "Null_donor_AUROC_max"
)

EMPIRICAL_P <- get_summary_value(
  "Empirical_P_observed_donor_AUROC"
)

NULL_STABLE_SHARED <- get_summary_value(
  "Null_stable_like_shared_genes"
)

MAX_NULL_STABILITY <- get_summary_value(
  "Max_null_shared_selection_probability"
)


# ------------------------------------------------------------
# 14. SUMMARY TABLE
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(

    "Outer_CV_candidate_shared_genes",

    "Half_sample_candidate_shared_genes",

    "Final_stable_shared_genes",

    "Final_positive_direction",

    "Final_negative_direction",

    "Final_protein_coding",

    "Final_non_protein_coding",

    "Outer_half_direction_agreement",

    "Minimum_half_sample_selection_probability",

    "Minimum_half_sample_sign_consistency",

    "Maximum_candidate_null_selection_probability",

    "Observed_reference_donor_AUROC",

    "Null_donor_AUROC_mean",

    "Null_donor_AUROC_max",

    "Permutation_empirical_P",

    "Null_stable_like_shared_genes",

    "Maximum_null_shared_selection_probability",

    "Stable_deviation_genes"
  ),

  value = c(

    nrow(OUTER),

    nrow(HALF),

    nrow(FINAL),

    N_POS,

    N_NEG,

    N_PC,

    nrow(FINAL) - N_PC,

    sum(
      FINAL$outer_half_direction_agreement
    ),

    min(
      FINAL$half_sample_selection_probability
    ),

    min(
      FINAL$half_sample_sign_consistency
    ),

    max(
      FINAL$null_selection_probability,
      na.rm = TRUE
    ),

    OBS_AUC,

    NULL_AUC_MEAN,

    NULL_AUC_MAX,

    EMPIRICAL_P,

    NULL_STABLE_SHARED,

    MAX_NULL_STABILITY,

    0
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 15. OUTPUT PATHS
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "07_stability",
  "final_stable_state_v1"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


FINAL_PATH <- file.path(
  OUT_DIR,
  "step06d_FINAL_stable_shared_state_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step06d_final_stable_state_summary_v1.csv"
)

EVIDENCE_PATH <- file.path(
  OUT_DIR,
  "step06d_final_gene_stability_evidence_v1.csv"
)

LOCK_PATH <- file.path(
  OUT_DIR,
  "step06d_FINAL_STABLE_STATE_LOCK_v1.txt"
)

SUPERSESSION_PATH <- file.path(
  OUT_DIR,
  "step06d_supersession_notice_v1.txt"
)

SHA_PATH <- file.path(
  OUT_DIR,
  "step06d_FINAL_stable_shared_state_v1.sha256"
)


# ------------------------------------------------------------
# 16. SAVE
# ------------------------------------------------------------

write.csv(
  FINAL,
  FINAL_PATH,
  row.names = FALSE,
  na = ""
)

write.csv(
  SUMMARY,
  SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  FINAL,
  EVIDENCE_PATH,
  row.names = FALSE,
  na = ""
)


# ------------------------------------------------------------
# 17. SHA256
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {

  install.packages(
    "digest",
    repos = "https://cloud.r-project.org"
  )
}

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop(
    "digest package unavailable."
  )
}


FINAL_SHA256 <- digest::digest(
  file = FINAL_PATH,
  algo = "sha256"
)

writeLines(
  FINAL_SHA256,
  SHA_PATH
)


# ------------------------------------------------------------
# 18. LOCK DOCUMENT
# ------------------------------------------------------------

writeLines(
  c(

    "STEP 06D — FINAL STABLE SHARED-STATE FREEZE",

    "",

    paste0(
      "final_gene_count=",
      nrow(FINAL)
    ),

    paste0(
      "positive_direction=",
      N_POS
    ),

    paste0(
      "negative_direction=",
      N_NEG
    ),

    paste0(
      "protein_coding=",
      N_PC
    ),

    "",

    "FINAL_SELECTION_RULE:",

    paste(
      "Gene must satisfy the prespecified observed outer-CV",
      "stability criterion AND the prespecified complementary",
      "half-sample stability criterion."
    ),

    "outer_selection_frequency>=0.80",

    "outer_sign_consistency>=0.90",

    "half_sample_selection_probability>=0.80",

    "half_sample_sign_consistency>=0.90",

    "outer_half_coefficient_direction_agreement=TRUE",

    "",

    "PERMUTATION_ROLE:",

    paste(
      "Donor-level permutation is used as null validation,",
      "not as a post hoc gene-selection criterion."
    ),

    paste0(
      "observed_reference_donor_AUROC=",
      OBS_AUC
    ),

    paste0(
      "null_donor_AUROC_mean=",
      NULL_AUC_MEAN
    ),

    paste0(
      "null_donor_AUROC_max=",
      NULL_AUC_MAX
    ),

    paste0(
      "empirical_P=",
      EMPIRICAL_P
    ),

    paste0(
      "null_stable_like_shared_genes=",
      NULL_STABLE_SHARED
    ),

    paste0(
      "max_null_shared_selection_probability=",
      MAX_NULL_STABILITY
    ),

    "",

    paste0(
      "final_csv_sha256=",
      FINAL_SHA256
    ),

    "",

    "external_expression_used_for_selection=FALSE",

    "external_outcome_used_for_selection=FALSE",

    "final_weight_construction_not_yet_performed=TRUE",

    "rank_score_not_yet_calculated=TRUE"
  ),
  LOCK_PATH
)


# ------------------------------------------------------------
# 19. SUPERSESSION NOTICE
# ------------------------------------------------------------

writeLines(
  c(

    "SUPERSESSION NOTICE",

    "",

    paste(
      "The earlier 403-gene observed outer-CV stable set",
      "remains a valid observed-stability artifact but is NOT",
      "the final frozen molecular-state signature."
    ),

    "",

    paste(
      "The final Step 06D stable shared state is the intersection",
      "of the observed outer-CV stable set and the independent",
      "complementary half-sample stable set, with direction agreement."
    ),

    "",

    paste0(
      "final_gene_count=",
      nrow(FINAL)
    ),

    paste0(
      "final_artifact=",
      FINAL_PATH
    ),

    paste0(
      "sha256=",
      FINAL_SHA256
    )
  ),
  SUPERSESSION_PATH
)


# ------------------------------------------------------------
# 20. FINAL HARD CHECK
# ------------------------------------------------------------

stopifnot(
  file.exists(FINAL_PATH),
  file.exists(SUMMARY_PATH),
  file.exists(LOCK_PATH),
  file.exists(SHA_PATH),
  nrow(FINAL) == 24L,
  length(
    unique(
      FINAL$ensemblID_stable
    )
  ) == 24L,
  all(
    FINAL$direction_code %in%
      c(-1L, 1L)
  )
)


# ------------------------------------------------------------
# 21. OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START


cat("\n============================================================\n")
cat("STEP 06D — FINAL STABLE SHARED-STATE FREEZE\n")
cat("============================================================\n")


print(
  SUMMARY,
  row.names = FALSE
)


cat(
  "\n--- FINAL 24-GENE STABLE SHARED STATE ---\n"
)


print(
  FINAL[
    ,
    c(
      "ensemblID_stable",
      "gene_symbol",
      "gene_type",
      "direction",
      "outer_selection_frequency",
      "outer_sign_consistency",
      "half_sample_selection_probability",
      "half_sample_sign_consistency",
      "null_selection_probability",
      "half_sample_median_nonzero_beta"
    ),
    drop = FALSE
  ],
  row.names = FALSE
)


cat(
  "\nFinal stable shared genes: ",
  nrow(FINAL),
  "\n",
  sep = ""
)

cat(
  "Positive: ",
  N_POS,
  "\n",
  sep = ""
)

cat(
  "Negative: ",
  N_NEG,
  "\n",
  sep = ""
)

cat(
  "Protein coding: ",
  N_PC,
  "\n",
  sep = ""
)

cat(
  "Stable deviation genes: 0\n"
)

cat(
  "\nFinal artifact:\n",
  FINAL_PATH,
  "\n",
  sep = ""
)

cat(
  "\nSHA-256:\n",
  FINAL_SHA256,
  "\n",
  sep = ""
)

cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS,
    1
  ),
  " sec\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 06D — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Outer-CV stable genes loaded: 403
Half-sample stable genes loaded: 24
Permutation candidate rows loaded: 24

Outer / half-sample intersection: 24
Direction agreement: 24 / 24

STEP 06D — FINAL STABLE SHARED-STATE FREEZE
                                       metric        value
              Outer_CV_candidate_shared_genes 403.00000000
           Half_sample_candidate_shared_genes  24.00000000
                    Final_stable_shared_genes  24.00000000
                     Final_positive_direction   9.00000000
                     Final_negative_direction  15.00000000
                         Final_protein_coding  12.00000000
                     Final_non_protein_coding  12.00000000
               Outer_half_direction_agreement  24.00000000
    Minimum_half_sample_selection_probability   0.80000000
         Minimum_half_sample_sign_consistency   1.00000000
 Maximum_candidate_null_selection_probability   0.22000000
               Observed_reference_donor_AUROC   0

In [18]:
# ============================================================
# STEP 07A — FROZEN MOLECULAR-STATE WEIGHT CONSTRUCTION
#
# Input:
#   Final 24-gene shared-state core from STEP 06D
#
# Primary raw weight:
#   median non-zero shared coefficient across the
#   25 repeated outer-CV discovery models
#
# Stability determines inclusion only.
# Selection probability is NOT multiplied into weight magnitude.
#
# NO EXTERNAL DATA USED.
# NO SCORE OR AUROC CALCULATED HERE.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUT
# ------------------------------------------------------------

FINAL_STATE_PATH <- file.path(
  PROJECT_ROOT,
  "07_stability",
  "final_stable_state_v1",
  "step06d_FINAL_stable_shared_state_v1.csv"
)

if (!file.exists(FINAL_STATE_PATH)) {
  stop(
    paste0(
      "Final stable-state artifact missing:\n",
      FINAL_STATE_PATH
    )
  )
}


# ------------------------------------------------------------
# 2. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "08_score_lock"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


WEIGHT_PATH <- file.path(
  OUT_DIR,
  "SCZ_MSS_v1_gene_weights.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step07a_weight_construction_summary_v1.csv"
)

DEFINITION_PATH <- file.path(
  OUT_DIR,
  "step07a_weight_construction_definition_v1.txt"
)

WEIGHT_SHA_PATH <- file.path(
  OUT_DIR,
  "SCZ_MSS_v1_gene_weights.sha256"
)


# ------------------------------------------------------------
# 3. LOAD FINAL STABLE STATE
# ------------------------------------------------------------

FINAL <- read.csv(
  FINAL_STATE_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

cat(
  "Final stable-state genes loaded: ",
  nrow(FINAL),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 4. REQUIRED COLUMN CHECK
# ------------------------------------------------------------

REQUIRED_COLUMNS <- c(
  "ensemblID_stable",
  "gene_symbol",
  "gene_type",
  "direction_code",
  "direction",
  "outer_selection_count",
  "outer_selection_frequency",
  "outer_sign_consistency",
  "outer_median_nonzero_beta",
  "half_sample_selection_count",
  "half_sample_selection_probability",
  "half_sample_sign_consistency",
  "half_sample_median_nonzero_beta",
  "null_selection_count",
  "null_selection_probability"
)

MISSING_COLUMNS <- setdiff(
  REQUIRED_COLUMNS,
  names(FINAL)
)

if (length(MISSING_COLUMNS) > 0L) {

  cat(
    "Missing columns:\n"
  )

  print(
    MISSING_COLUMNS
  )

  stop(
    "Final stable-state schema mismatch."
  )
}


# ------------------------------------------------------------
# 5. BASIC INTEGRITY
# ------------------------------------------------------------

if (nrow(FINAL) != 24L) {
  stop(
    paste0(
      "Expected 24 final genes; found ",
      nrow(FINAL)
    )
  )
}

if (
  anyNA(
    FINAL$ensemblID_stable
  ) ||
  anyDuplicated(
    FINAL$ensemblID_stable
  )
) {
  stop(
    "Invalid final Ensembl IDs."
  )
}

if (
  anyNA(
    FINAL$direction_code
  ) ||
  !all(
    FINAL$direction_code %in%
      c(-1L, 1L)
  )
) {
  stop(
    "Invalid direction codes."
  )
}

if (
  anyNA(
    FINAL$outer_median_nonzero_beta
  ) ||
  any(
    FINAL$outer_median_nonzero_beta == 0
  )
) {
  stop(
    "Invalid outer median coefficients."
  )
}

if (
  any(
    sign(
      FINAL$outer_median_nonzero_beta
    ) !=
      FINAL$direction_code
  )
) {
  stop(
    "Outer coefficient direction does not match frozen direction."
  )
}

if (
  any(
    sign(
      FINAL$half_sample_median_nonzero_beta
    ) !=
      FINAL$direction_code
  )
) {
  stop(
    "Half-sample coefficient direction does not match frozen direction."
  )
}


# ------------------------------------------------------------
# 6. DEFINE PRIMARY DISCOVERY WEIGHTS
#
# Stability controls WHETHER a gene enters the score.
#
# Magnitude:
#   median outer-CV non-zero beta.
#
# We do NOT multiply coefficient magnitude by
# selection probability.
# ------------------------------------------------------------

RAW_WEIGHT <- as.numeric(
  FINAL$outer_median_nonzero_beta
)

ABS_WEIGHT <- abs(
  RAW_WEIGHT
)

TOTAL_ABS_WEIGHT <- sum(
  ABS_WEIGHT
)

if (
  !is.finite(TOTAL_ABS_WEIGHT) ||
  TOTAL_ABS_WEIGHT <= 0
) {
  stop(
    "Invalid total absolute weight."
  )
}


NORMALIZED_WEIGHT <- (
  RAW_WEIGHT /
    TOTAL_ABS_WEIGHT
)

NORMALIZED_ABS_WEIGHT <- abs(
  NORMALIZED_WEIGHT
)


# ------------------------------------------------------------
# 7. HARD NORMALIZATION CHECK
# ------------------------------------------------------------

if (
  abs(
    sum(
      NORMALIZED_ABS_WEIGHT
    ) - 1
  ) > 1e-12
) {
  stop(
    "Normalized absolute weights do not sum to 1."
  )
}

if (
  any(
    sign(
      NORMALIZED_WEIGHT
    ) !=
      FINAL$direction_code
  )
) {
  stop(
    "Normalized weight direction mismatch."
  )
}


# ------------------------------------------------------------
# 8. BUILD FROZEN WEIGHT TABLE
# ------------------------------------------------------------

WEIGHTS <- data.frame(

  gene_index =
    seq_len(
      nrow(FINAL)
    ),

  ensemblID_stable =
    FINAL$ensemblID_stable,

  gene_symbol =
    FINAL$gene_symbol,

  gene_type =
    FINAL$gene_type,

  protein_coding =
    FINAL$gene_type ==
      "protein_coding",

  direction_code =
    FINAL$direction_code,

  direction =
    FINAL$direction,

  discovery_median_coefficient =
    RAW_WEIGHT,

  absolute_weight =
    ABS_WEIGHT,

  final_normalized_weight =
    NORMALIZED_WEIGHT,

  normalized_absolute_weight =
    NORMALIZED_ABS_WEIGHT,

  outer_selection_count =
    FINAL$outer_selection_count,

  outer_selection_frequency =
    FINAL$outer_selection_frequency,

  outer_sign_consistency =
    FINAL$outer_sign_consistency,

  half_sample_selection_count =
    FINAL$half_sample_selection_count,

  half_sample_selection_probability =
    FINAL$half_sample_selection_probability,

  half_sample_sign_consistency =
    FINAL$half_sample_sign_consistency,

  half_sample_median_nonzero_beta =
    FINAL$half_sample_median_nonzero_beta,

  null_selection_count =
    FINAL$null_selection_count,

  null_selection_probability =
    FINAL$null_selection_probability,

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 9. ABSOLUTE-WEIGHT RANK
# ------------------------------------------------------------

WEIGHTS$absolute_weight_rank <- rank(
  -WEIGHTS$absolute_weight,
  ties.method = "min"
)


# ------------------------------------------------------------
# 10. INTERNAL COEFFICIENT AGREEMENT
#
# Descriptive only.
# Does not modify weights.
# ------------------------------------------------------------

PEARSON_OUTER_HALF <- cor(
  FINAL$outer_median_nonzero_beta,
  FINAL$half_sample_median_nonzero_beta,
  method = "pearson"
)

SPEARMAN_OUTER_HALF <- cor(
  FINAL$outer_median_nonzero_beta,
  FINAL$half_sample_median_nonzero_beta,
  method = "spearman"
)


# ------------------------------------------------------------
# 11. DIRECTION / WEIGHT SUMMARY
# ------------------------------------------------------------

N_POS <- sum(
  WEIGHTS$direction_code == 1L
)

N_NEG <- sum(
  WEIGHTS$direction_code == -1L
)

N_PC <- sum(
  WEIGHTS$protein_coding
)

POS_ABS_WEIGHT <- sum(
  WEIGHTS$normalized_absolute_weight[
    WEIGHTS$direction_code == 1L
  ]
)

NEG_ABS_WEIGHT <- sum(
  WEIGHTS$normalized_absolute_weight[
    WEIGHTS$direction_code == -1L
  ]
)


# ------------------------------------------------------------
# 12. FINAL SORT
#
# Preserve the final-state stability ordering in gene_index,
# but store absolute weight rank separately.
# ------------------------------------------------------------

stopifnot(
  nrow(WEIGHTS) == 24L,
  N_POS == 9L,
  N_NEG == 15L,
  N_PC == 12L,
  all(
    sign(
      WEIGHTS$final_normalized_weight
    ) ==
      WEIGHTS$direction_code
  ),
  abs(
    sum(
      abs(
        WEIGHTS$final_normalized_weight
      )
    ) - 1
  ) < 1e-12
)


# ------------------------------------------------------------
# 13. SAVE WEIGHT TABLE
# ------------------------------------------------------------

write.csv(
  WEIGHTS,
  WEIGHT_PATH,
  row.names = FALSE,
  na = ""
)


# ------------------------------------------------------------
# 14. SHA256
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {

  install.packages(
    "digest",
    repos = "https://cloud.r-project.org"
  )
}

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop(
    "digest package unavailable."
  )
}

WEIGHT_SHA256 <- digest::digest(
  file = WEIGHT_PATH,
  algo = "sha256"
)

FINAL_STATE_SHA256 <- digest::digest(
  file = FINAL_STATE_PATH,
  algo = "sha256"
)

writeLines(
  WEIGHT_SHA256,
  WEIGHT_SHA_PATH
)


# ------------------------------------------------------------
# 15. SUMMARY TABLE
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(
    "Frozen_score_genes",
    "Positive_direction_genes",
    "Negative_direction_genes",
    "Protein_coding_genes",
    "Non_protein_coding_genes",
    "Total_absolute_raw_weight",
    "Normalized_absolute_weight_sum",
    "Positive_direction_absolute_weight_fraction",
    "Negative_direction_absolute_weight_fraction",
    "Outer_vs_half_beta_Pearson",
    "Outer_vs_half_beta_Spearman",
    "Minimum_outer_selection_frequency",
    "Minimum_half_sample_selection_probability",
    "Minimum_sign_consistency",
    "Maximum_null_selection_probability"
  ),

  value = c(
    nrow(WEIGHTS),
    N_POS,
    N_NEG,
    N_PC,
    nrow(WEIGHTS) - N_PC,
    TOTAL_ABS_WEIGHT,
    sum(
      WEIGHTS$normalized_absolute_weight
    ),
    POS_ABS_WEIGHT,
    NEG_ABS_WEIGHT,
    PEARSON_OUTER_HALF,
    SPEARMAN_OUTER_HALF,
    min(
      WEIGHTS$outer_selection_frequency
    ),
    min(
      WEIGHTS$half_sample_selection_probability
    ),
    min(
      WEIGHTS$half_sample_sign_consistency
    ),
    max(
      WEIGHTS$null_selection_probability
    )
  ),

  stringsAsFactors = FALSE
)

write.csv(
  SUMMARY,
  SUMMARY_PATH,
  row.names = FALSE
)


# ------------------------------------------------------------
# 16. WEIGHT-CONSTRUCTION DEFINITION
# ------------------------------------------------------------

writeLines(
  c(

    "STEP 07A — FROZEN MOLECULAR-STATE WEIGHT CONSTRUCTION",

    "",

    "Score name: SCZ-MSS v1",

    paste0(
      "gene_count=",
      nrow(WEIGHTS)
    ),

    "",

    "Gene inclusion:",
    paste(
      "Only genes in the final Step 06D stable shared-state",
      "intersection are eligible."
    ),

    "",

    "Primary raw signed weight:",
    paste(
      "Median non-zero shared coefficient across the",
      "25 repeated outer-CV discovery models."
    ),

    "",

    "Role of complementary half-sample analysis:",
    paste(
      "Used to confirm stability and direction;",
      "not used to rescale primary weight magnitude."
    ),

    "",

    "Role of selection probability:",
    paste(
      "Used as a stability inclusion criterion and reported",
      "as evidence; not multiplied into coefficient magnitude."
    ),

    "",

    "Weight normalization:",

    "absolute_weight_g = abs(discovery_median_coefficient_g)",

    paste(
      "final_normalized_weight_g =",
      "discovery_median_coefficient_g /",
      "sum(abs(discovery_median_coefficient))"
    ),

    paste0(
      "sum_absolute_normalized_weights=",
      sum(
        abs(
          WEIGHTS$final_normalized_weight
        )
      )
    ),

    "",

    "External missing-gene renormalization:",
    paste(
      "To be defined exactly in STEP 07B/07D.",
      "No external cohort is used in the present step."
    ),

    "",

    paste0(
      "final_stable_state_sha256=",
      FINAL_STATE_SHA256
    ),

    paste0(
      "gene_weight_table_sha256=",
      WEIGHT_SHA256
    ),

    "",

    "external_expression_used=FALSE",

    "external_diagnosis_used=FALSE",

    "external_performance_used=FALSE",

    "rank_score_calculated=FALSE",

    "weight_magnitudes_frozen_from_discovery=TRUE"
  ),
  DEFINITION_PATH
)


# ------------------------------------------------------------
# 17. FINAL VALIDATION
# ------------------------------------------------------------

WEIGHTS_CHECK <- read.csv(
  WEIGHT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(WEIGHTS_CHECK) == 24L,
  anyDuplicated(
    WEIGHTS_CHECK$ensemblID_stable
  ) == 0L,
  abs(
    sum(
      abs(
        WEIGHTS_CHECK$final_normalized_weight
      )
    ) - 1
  ) < 1e-10,
  all(
    sign(
      WEIGHTS_CHECK$final_normalized_weight
    ) ==
      WEIGHTS_CHECK$direction_code
  ),
  file.exists(
    WEIGHT_SHA_PATH
  )
)


# ------------------------------------------------------------
# 18. OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START

cat("\n============================================================\n")
cat("STEP 07A — FROZEN MOLECULAR-STATE WEIGHTS\n")
cat("============================================================\n")

print(
  SUMMARY,
  row.names = FALSE
)

cat(
  "\n--- FROZEN 24-GENE WEIGHT TABLE ---\n"
)

PRINT_TABLE <- WEIGHTS[
  order(
    WEIGHTS$absolute_weight_rank
  ),
  c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "direction",
    "discovery_median_coefficient",
    "absolute_weight",
    "final_normalized_weight",
    "absolute_weight_rank",
    "outer_selection_frequency",
    "half_sample_selection_probability",
    "null_selection_probability"
  ),
  drop = FALSE
]

print(
  PRINT_TABLE,
  row.names = FALSE
)

cat(
  "\nWeight table:\n",
  WEIGHT_PATH,
  "\n",
  sep = ""
)

cat(
  "\nWeight SHA-256:\n",
  WEIGHT_SHA256,
  "\n",
  sep = ""
)

cat(
  "\nOuter vs half-sample beta Pearson: ",
  round(
    PEARSON_OUTER_HALF,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Outer vs half-sample beta Spearman: ",
  round(
    SPEARMAN_OUTER_HALF,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS,
    1
  ),
  " sec\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 07A — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Final stable-state genes loaded: 24

STEP 07A — FROZEN MOLECULAR-STATE WEIGHTS
                                      metric      value
                          Frozen_score_genes 24.0000000
                    Positive_direction_genes  9.0000000
                    Negative_direction_genes 15.0000000
                        Protein_coding_genes 12.0000000
                    Non_protein_coding_genes 12.0000000
                   Total_absolute_raw_weight  8.0428444
              Normalized_absolute_weight_sum  1.0000000
 Positive_direction_absolute_weight_fraction  0.3869186
 Negative_direction_absolute_weight_fraction  0.6130814
                  Outer_vs_half_beta_Pearson  0.9937847
                 Outer_vs_half_beta_Spearman  0.9939130
           Minimum_outer_selection_frequency  0.9600000
   Minimum_half_sample_selection_probability  0.8000000
                    Minimum_sign_consistency  1.0000000
          Maximum_null_selection_probability  0.2200000

-

In [19]:
# ============================================================
# STEP 07B — SCZ-MSS v1 RANK-SCORE FORMULATION
# + DISCOVERY INTERNAL CHARACTERIZATION
#
# Primary portable score:
#
# For sample i with m available frozen genes:
#
# centered_rank_ig =
#   ( rank_ig - (m + 1)/2 ) / (m - 1)
#
# using average ranks for ties.
#
# Therefore centered ranks are bounded approximately
# between -0.5 and +0.5.
#
# SCZ_MSS_i =
#   sum_g( w_g * centered_rank_ig )
#   /
#   sum_g( abs(w_g) )
#
# where w_g are FROZEN Step 07A normalized weights.
#
# IMPORTANT:
# Internal score-vs-diagnosis statistics below are
# POST-SELECTION DESCRIPTIVE CHARACTERIZATION.
#
# They are NOT an unbiased internal validation estimate.
#
# Primary unbiased internal predictive evidence remains
# repeated nested donor-grouped OOF performance.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUT PATHS
# ------------------------------------------------------------

WEIGHT_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_gene_weights.csv"
)

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

REQUIRED <- c(
  WEIGHT_PATH,
  COUNTS_PATH,
  META_PATH
)

if (!all(file.exists(REQUIRED))) {

  print(
    REQUIRED[
      !file.exists(REQUIRED)
    ]
  )

  stop(
    "Required canonical input missing."
  )
}


# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

WEIGHTS <- read.csv(
  WEIGHT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

COUNTS <- readRDS(
  COUNTS_PATH
)

META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


stopifnot(
  nrow(WEIGHTS) == 24L,
  nrow(COUNTS) == 23740L,
  ncol(COUNTS) == 755L,
  nrow(META) == 755L,
  length(unique(META$BrNum)) == 454L
)


# ------------------------------------------------------------
# 3. WEIGHT INTEGRITY
# ------------------------------------------------------------

if (
  anyDuplicated(
    WEIGHTS$ensemblID_stable
  )
) {
  stop(
    "Duplicated frozen score genes."
  )
}

if (
  anyNA(
    WEIGHTS$final_normalized_weight
  )
) {
  stop(
    "Missing frozen weights."
  )
}

if (
  abs(
    sum(
      abs(
        WEIGHTS$final_normalized_weight
      )
    ) - 1
  ) > 1e-10
) {
  stop(
    "Frozen weights do not sum to unit absolute weight."
  )
}


# ------------------------------------------------------------
# 4. ALIGN 24 SCORE GENES
# ------------------------------------------------------------

COUNT_GENE_IDS <- rownames(
  COUNTS
)

if (
  is.null(COUNT_GENE_IDS)
) {
  stop(
    "Counts matrix has no gene rownames."
  )
}

GENE_INDEX <- match(
  WEIGHTS$ensemblID_stable,
  COUNT_GENE_IDS
)

if (anyNA(
  GENE_INDEX
)) {

  MISSING <- WEIGHTS$ensemblID_stable[
    is.na(GENE_INDEX)
  ]

  cat(
    "Missing frozen genes:\n"
  )

  print(MISSING)

  stop(
    "Frozen score genes not fully present in BrainSeq counts."
  )
}


SCORE_COUNTS <- COUNTS[
  GENE_INDEX,
  ,
  drop = FALSE
]

rownames(
  SCORE_COUNTS
) <- WEIGHTS$ensemblID_stable


# ------------------------------------------------------------
# 5. EXPRESSION TRANSFORMATION
#
# Ranking is invariant to the monotonic log2(CPM+1)
# transformation within each sample.
# ------------------------------------------------------------

LIB_SIZE <- colSums(
  COUNTS
)

if (
  any(
    !is.finite(LIB_SIZE)
  ) ||
  any(
    LIB_SIZE <= 0
  )
) {
  stop(
    "Invalid library sizes."
  )
}


SCORE_LOGCPM <- log2(
  sweep(
    SCORE_COUNTS,
    2,
    LIB_SIZE / 1e6,
    "/"
  ) + 1
)


rm(
  COUNTS,
  SCORE_COUNTS
)

invisible(gc())


# ------------------------------------------------------------
# 6. FROZEN RANK-SCORE FUNCTION
# ------------------------------------------------------------

rank_score_one_sample <- function(
  expression_vector,
  frozen_weights
) {

  ok <- (
    is.finite(
      expression_vector
    ) &
    is.finite(
      frozen_weights
    )
  )

  m <- sum(
    ok
  )

  if (
    m < 2L
  ) {
    return(
      NA_real_
    )
  }

  x <- expression_vector[
    ok
  ]

  w <- frozen_weights[
    ok
  ]

  r <- rank(
    x,
    ties.method = "average"
  )

  centered_rank <- (
    r -
      (
        m + 1
      ) / 2
  ) /
    (
      m - 1
    )

  denominator <- sum(
    abs(
      w
    )
  )

  if (
    !is.finite(
      denominator
    ) ||
    denominator <= 0
  ) {
    return(
      NA_real_
    )
  }

  sum(
    w *
      centered_rank
  ) /
    denominator
}


# ------------------------------------------------------------
# 7. CALCULATE SCZ-MSS v1
# ------------------------------------------------------------

FROZEN_W <- WEIGHTS$final_normalized_weight

SCZ_MSS <- apply(
  SCORE_LOGCPM,
  2,
  rank_score_one_sample,
  frozen_weights = FROZEN_W
)


if (
  length(SCZ_MSS) !=
    nrow(META)
) {
  stop(
    "Score length mismatch."
  )
}

if (
  any(
    !is.finite(SCZ_MSS)
  )
) {
  stop(
    "Non-finite SCZ-MSS score detected."
  )
}


# ------------------------------------------------------------
# 8. SAMPLE-LEVEL SCORE TABLE
# ------------------------------------------------------------

SAMPLE_SCORE <- data.frame(

  discovery_row_index =
    META$discovery_row_index,

  sample_id =
    colnames(
      SCORE_LOGCPM
    ),

  donor_id =
    META$BrNum,

  region =
    as.character(
      META$Region
    ),

  diagnosis =
    as.character(
      META$Dx
    ),

  SCZ_MSS_v1 =
    as.numeric(
      SCZ_MSS
    ),

  n_score_genes_available =
    24L,

  available_absolute_weight =
    1,

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 9. DONOR-LEVEL SCORE
#
# Paired donor:
#   mean of DLPFC + HIPPO scores
#
# Unpaired donor:
#   single region score
# ------------------------------------------------------------

DONOR_SPLIT <- split(
  seq_len(
    nrow(SAMPLE_SCORE)
  ),
  SAMPLE_SCORE$donor_id
)


DONOR_SCORE_LIST <- lapply(
  names(DONOR_SPLIT),
  function(id) {

    idx <- DONOR_SPLIT[
      id
    ]

    idx <- unlist(
      idx,
      use.names = FALSE
    )

    dx <- unique(
      SAMPLE_SCORE$diagnosis[
        idx
      ]
    )

    if (
      length(dx) != 1L
    ) {
      stop(
        "Diagnosis inconsistent within donor."
      )
    }

    regions <- unique(
      SAMPLE_SCORE$region[
        idx
      ]
    )

    data.frame(

      donor_id =
        id,

      diagnosis =
        dx,

      n_regions =
        length(
          idx
        ),

      paired =
        length(idx) == 2L,

      regions =
        paste(
          sort(regions),
          collapse = "+"
        ),

      SCZ_MSS_v1 =
        mean(
          SAMPLE_SCORE$SCZ_MSS_v1[
            idx
          ]
        ),

      stringsAsFactors = FALSE
    )
  }
)


DONOR_SCORE <- do.call(
  rbind,
  DONOR_SCORE_LIST
)

rownames(
  DONOR_SCORE
) <- NULL


stopifnot(
  nrow(DONOR_SCORE) == 454L
)


# ------------------------------------------------------------
# 10. HELPERS
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  y <- as.integer(
    y
  )

  ok <- (
    !is.na(y) &
    is.finite(score)
  )

  y <- y[
    ok
  ]

  score <- score[
    ok
  ]

  n1 <- sum(
    y == 1L
  )

  n0 <- sum(
    y == 0L
  )

  if (
    n1 == 0L ||
    n0 == 0L
  ) {
    return(
      NA_real_
    )
  }

  ranks <- rank(
    score,
    ties.method = "average"
  )

  (
    sum(
      ranks[
        y == 1L
      ]
    ) -
      n1 *
      (
        n1 + 1
      ) /
      2
  ) /
    (
      n1 *
      n0
    )
}


hedges_g <- function(
  control_scores,
  case_scores
) {

  n0 <- length(
    control_scores
  )

  n1 <- length(
    case_scores
  )

  if (
    n0 < 2L ||
    n1 < 2L
  ) {
    return(
      NA_real_
    )
  }

  pooled_sd <- sqrt(
    (
      (
        n0 - 1
      ) *
        var(
          control_scores
        ) +
        (
          n1 - 1
        ) *
        var(
          case_scores
        )
    ) /
      (
        n0 +
        n1 -
        2
      )
  )

  d <- (
    mean(
      case_scores
    ) -
      mean(
        control_scores
      )
  ) /
    pooled_sd

  df <- (
    n0 +
    n1 -
    2
  )

  J <- (
    1 -
      3 /
      (
        4 *
          df -
        1
      )
  )

  J *
    d
}


# ------------------------------------------------------------
# 11. DESCRIPTIVE DISCOVERY ASSOCIATION
#
# POST-SELECTION.
# NOT UNBIASED VALIDATION.
# ------------------------------------------------------------

DONOR_Y <- ifelse(
  DONOR_SCORE$diagnosis ==
    "Schizo",
  1L,
  0L
)


DONOR_AUC <- auc_rank(
  DONOR_Y,
  DONOR_SCORE$SCZ_MSS_v1
)


DONOR_CONTROL <- DONOR_SCORE$SCZ_MSS_v1[
  DONOR_Y == 0L
]

DONOR_SCZ <- DONOR_SCORE$SCZ_MSS_v1[
  DONOR_Y == 1L
]


DONOR_HEDGES_G <- hedges_g(
  DONOR_CONTROL,
  DONOR_SCZ
)


SAMPLE_Y <- ifelse(
  SAMPLE_SCORE$diagnosis ==
    "Schizo",
  1L,
  0L
)


SAMPLE_AUC <- auc_rank(
  SAMPLE_Y,
  SAMPLE_SCORE$SCZ_MSS_v1
)


DLPFC_IDX <- (
  SAMPLE_SCORE$region ==
    "DLPFC"
)

HIPPO_IDX <- (
  SAMPLE_SCORE$region ==
    "HIPPO"
)


DLPFC_AUC <- auc_rank(
  SAMPLE_Y[
    DLPFC_IDX
  ],
  SAMPLE_SCORE$SCZ_MSS_v1[
    DLPFC_IDX
  ]
)


HIPPO_AUC <- auc_rank(
  SAMPLE_Y[
    HIPPO_IDX
  ],
  SAMPLE_SCORE$SCZ_MSS_v1[
    HIPPO_IDX
  ]
)


# ------------------------------------------------------------
# 12. SCORE DISTRIBUTION SUMMARY
# ------------------------------------------------------------

SCORE_SUMMARY <- do.call(
  rbind,
  lapply(
    c(
      "Control",
      "Schizo"
    ),
    function(dx) {

      z <- DONOR_SCORE$SCZ_MSS_v1[
        DONOR_SCORE$diagnosis ==
          dx
      ]

      data.frame(

        diagnosis =
          dx,

        n_donors =
          length(z),

        mean_score =
          mean(z),

        sd_score =
          sd(z),

        median_score =
          median(z),

        Q1 =
          unname(
            quantile(
              z,
              0.25
            )
          ),

        Q3 =
          unname(
            quantile(
              z,
              0.75
            )
          ),

        stringsAsFactors = FALSE
      )
    }
  )
)


# ------------------------------------------------------------
# 13. FORMULA / RANGE AUDIT
# ------------------------------------------------------------

M <- nrow(
  WEIGHTS
)

THEORETICAL_MIN_CENTERED_RANK <- (
  1 -
    (
      M + 1
    ) / 2
) /
  (
    M - 1
  )

THEORETICAL_MAX_CENTERED_RANK <- (
  M -
    (
      M + 1
    ) / 2
) /
  (
    M - 1
  )


stopifnot(
  abs(
    THEORETICAL_MIN_CENTERED_RANK +
      0.5
  ) < 1e-12,
  abs(
    THEORETICAL_MAX_CENTERED_RANK -
      0.5
  ) < 1e-12
)


# ------------------------------------------------------------
# 14. OUTPUT PATHS
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "08_score_lock"
)

SAMPLE_SCORE_PATH <- file.path(
  OUT_DIR,
  "step07b_BrainSeq_SCZ_MSS_v1_sample_scores.csv"
)

DONOR_SCORE_PATH <- file.path(
  OUT_DIR,
  "step07b_BrainSeq_SCZ_MSS_v1_donor_scores.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step07b_BrainSeq_SCZ_MSS_v1_descriptive_summary.csv"
)

DIAGNOSIS_SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step07b_BrainSeq_SCZ_MSS_v1_diagnosis_summary.csv"
)

FORMULA_PATH <- file.path(
  OUT_DIR,
  "SCZ_MSS_v1_formula.md"
)


# ------------------------------------------------------------
# 15. SAVE SCORES
# ------------------------------------------------------------

write.csv(
  SAMPLE_SCORE,
  SAMPLE_SCORE_PATH,
  row.names = FALSE
)

write.csv(
  DONOR_SCORE,
  DONOR_SCORE_PATH,
  row.names = FALSE
)

write.csv(
  SCORE_SUMMARY,
  DIAGNOSIS_SUMMARY_PATH,
  row.names = FALSE
)


# ------------------------------------------------------------
# 16. DESCRIPTIVE SUMMARY
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(

    "Score_genes",

    "Discovery_samples",

    "Discovery_donors",

    "Control_donors",

    "Schizophrenia_donors",

    "Centered_rank_min",

    "Centered_rank_max",

    "Observed_score_min",

    "Observed_score_max",

    "Observed_score_mean",

    "Observed_score_SD",

    "Postselection_descriptive_donor_AUROC",

    "Postselection_descriptive_sample_AUROC",

    "Postselection_descriptive_DLPFC_AUROC",

    "Postselection_descriptive_HIPPO_AUROC",

    "Postselection_descriptive_donor_Hedges_g"
  ),

  value = c(

    24,

    nrow(
      SAMPLE_SCORE
    ),

    nrow(
      DONOR_SCORE
    ),

    sum(
      DONOR_Y == 0L
    ),

    sum(
      DONOR_Y == 1L
    ),

    THEORETICAL_MIN_CENTERED_RANK,

    THEORETICAL_MAX_CENTERED_RANK,

    min(
      SAMPLE_SCORE$SCZ_MSS_v1
    ),

    max(
      SAMPLE_SCORE$SCZ_MSS_v1
    ),

    mean(
      SAMPLE_SCORE$SCZ_MSS_v1
    ),

    sd(
      SAMPLE_SCORE$SCZ_MSS_v1
    ),

    DONOR_AUC,

    SAMPLE_AUC,

    DLPFC_AUC,

    HIPPO_AUC,

    DONOR_HEDGES_G
  ),

  stringsAsFactors = FALSE
)


write.csv(
  SUMMARY,
  SUMMARY_PATH,
  row.names = FALSE
)


# ------------------------------------------------------------
# 17. FORMULA LOCK DOCUMENT
# ------------------------------------------------------------

FORMULA_TEXT <- c(

  "# SCZ-MSS v1 Portable Rank Score",

  "",

  "## Frozen score-gene set",

  "24 stable shared-state genes.",

  "",

  "## Frozen gene weights",

  paste(
    "Signed discovery weights are taken from",
    "SCZ_MSS_v1_gene_weights.csv."
  ),

  paste(
    "They are derived from the median non-zero",
    "shared coefficient across the 25 repeated",
    "outer-CV discovery models."
  ),

  "",

  "## Within-sample ranking",

  paste(
    "For each sample, rank all technically available",
    "frozen score genes by expression."
  ),

  "Ties use average ranks.",

  "",

  "For m available score genes:",

  "",

  "centered_rank_ig = (rank_ig - (m + 1)/2) / (m - 1)",

  "",

  paste(
    "For complete data this maps the lowest and highest",
    "possible ranks exactly to -0.5 and +0.5."
  ),

  "",

  "## Molecular-state score",

  "",

  "SCZ_MSS_i = sum_g(w_g * centered_rank_ig) / sum_g(abs(w_g))",

  "",

  paste(
    "The denominator includes only technically available",
    "frozen genes in that sample/cohort."
  ),

  paste(
    "No external diagnosis information may be used for",
    "gene mapping, rank construction, weight modification,",
    "missing-gene handling, or score normalization."
  ),

  "",

  "## Direction",

  paste(
    "Higher SCZ-MSS values represent movement in the",
    "frozen schizophrenia-associated discovery direction."
  ),

  "",

  "## Internal interpretation",

  paste(
    "Discovery score-vs-diagnosis statistics calculated",
    "after final feature freezing are descriptive and",
    "must not be presented as unbiased internal validation."
  ),

  paste(
    "Primary unbiased internal predictive performance",
    "comes from repeated nested donor-grouped out-of-fold",
    "predictions generated before final score construction."
  )
)


writeLines(
  FORMULA_TEXT,
  FORMULA_PATH
)


# ------------------------------------------------------------
# 18. SHA256
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {

  install.packages(
    "digest",
    repos = "https://cloud.r-project.org"
  )
}

FORMULA_SHA256 <- digest::digest(
  file = FORMULA_PATH,
  algo = "sha256"
)

WEIGHT_SHA256 <- digest::digest(
  file = WEIGHT_PATH,
  algo = "sha256"
)


# ------------------------------------------------------------
# 19. HARD CHECKS
# ------------------------------------------------------------

stopifnot(

  nrow(SAMPLE_SCORE) == 755L,

  nrow(DONOR_SCORE) == 454L,

  all(
    SAMPLE_SCORE$n_score_genes_available ==
      24L
  ),

  all(
    abs(
      SAMPLE_SCORE$available_absolute_weight -
        1
    ) < 1e-12
  ),

  file.exists(
    FORMULA_PATH
  ),

  is.finite(
    DONOR_AUC
  ),

  is.finite(
    DONOR_HEDGES_G
  )
)


# ------------------------------------------------------------
# 20. OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START


cat("\n============================================================\n")
cat("STEP 07B — SCZ-MSS v1 RANK SCORE\n")
cat("============================================================\n")


cat(
  "Frozen genes: ",
  nrow(WEIGHTS),
  "\n",
  sep = ""
)

cat(
  "Centered-rank range: ",
  THEORETICAL_MIN_CENTERED_RANK,
  " to ",
  THEORETICAL_MAX_CENTERED_RANK,
  "\n",
  sep = ""
)


cat(
  "\n--- DONOR SCORE BY DIAGNOSIS ---\n"
)

print(
  SCORE_SUMMARY,
  row.names = FALSE
)


cat(
  "\n--- INTERNAL DESCRIPTIVE METRICS ---\n"
)

print(
  SUMMARY,
  row.names = FALSE
)


cat(
  "\nIMPORTANT:\n"
)

cat(
  "The AUC/Hedges' g values above are POST-SELECTION DESCRIPTIVE statistics.\n"
)

cat(
  "They are NOT the primary unbiased internal validation result.\n"
)

cat(
  "Primary unbiased internal donor AUROC remains the repeated nested-CV OOF estimate (~0.802).\n"
)


cat(
  "\nWeight SHA-256:\n",
  WEIGHT_SHA256,
  "\n",
  sep = ""
)

cat(
  "\nFormula SHA-256:\n",
  FORMULA_SHA256,
  "\n",
  sep = ""
)


cat(
  "\nFormula saved:\n",
  FORMULA_PATH,
  "\n",
  sep = ""
)


cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS,
    1
  ),
  " sec\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 07B — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS

STEP 07B — SCZ-MSS v1 RANK SCORE
Frozen genes: 24
Centered-rank range: -0.5 to 0.5

--- DONOR SCORE BY DIAGNOSIS ---
 diagnosis n_donors mean_score   sd_score median_score          Q1         Q3
   Control      277 0.02003712 0.01925783   0.01999572 0.007681576 0.03338676
    Schizo      177 0.03815974 0.01916628   0.03949891 0.027319467 0.04999719

--- INTERNAL DESCRIPTIVE METRICS ---
                                   metric        value
                              Score_genes  24.00000000
                        Discovery_samples 755.00000000
                         Discovery_donors 454.00000000
                           Control_donors 277.00000000
                     Schizophrenia_donors 177.00000000
                        Centered_rank_min  -0.50000000
                        Centered_rank_max   0.50000000
                       Observed_score_min  -0.04277336
                       Observed_score_max   0.08136876
                      Observed_score_

In [20]:
# ============================================================
# STEP 07C — PAIRED-DONOR CROSS-REGION STATE ANALYSIS
#
# BrainSeq donors with BOTH DLPFC and HIPPO.
#
# Evaluates:
#   1. within-donor DLPFC-HIPPO SCZ-MSS correlation
#   2. correlation separately in Control and SCZ
#   3. disease-direction effect in each region
#   4. within-donor regional difference:
#          Delta = DLPFC - HIPPO
#   5. diagnosis association with Delta
#      (= score-level diagnosis x region interaction)
#   6. paired-average molecular-state separation
#   7. individual SCZ cross-region directional concordance
#
# IMPORTANT:
# All diagnosis-associated score results here are
# discovery/post-selection DESCRIPTIVE analyses.
#
# NO EXTERNAL DATA USED.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUT
# ------------------------------------------------------------

SCORE_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "step07b_BrainSeq_SCZ_MSS_v1_sample_scores.csv"
)

WEIGHT_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_gene_weights.csv"
)

FORMULA_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_formula.md"
)

if (!file.exists(SCORE_PATH)) {
  stop("Step 07B sample-score file missing.")
}

SCORE <- read.csv(
  SCORE_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


# ------------------------------------------------------------
# 2. SCHEMA CHECK
# ------------------------------------------------------------

REQUIRED_COLUMNS <- c(
  "sample_id",
  "donor_id",
  "region",
  "diagnosis",
  "SCZ_MSS_v1"
)

MISSING_COLUMNS <- setdiff(
  REQUIRED_COLUMNS,
  names(SCORE)
)

if (length(MISSING_COLUMNS) > 0L) {

  print(MISSING_COLUMNS)

  stop(
    "Step 07B score schema mismatch."
  )
}

stopifnot(
  nrow(SCORE) == 755L
)


SCORE$donor_id <- as.character(
  SCORE$donor_id
)

SCORE$diagnosis <- as.character(
  SCORE$diagnosis
)

SCORE$region <- as.character(
  SCORE$region
)


if (
  anyNA(SCORE$SCZ_MSS_v1) ||
  any(
    !is.finite(
      SCORE$SCZ_MSS_v1
    )
  )
) {
  stop(
    "Invalid SCZ-MSS values."
  )
}


# ------------------------------------------------------------
# 3. DONOR AUDIT
# ------------------------------------------------------------

DONOR_SPLIT <- split(
  seq_len(
    nrow(SCORE)
  ),
  SCORE$donor_id
)


DONOR_AUDIT <- do.call(
  rbind,
  lapply(
    names(DONOR_SPLIT),
    function(id) {

      idx <- DONOR_SPLIT[
        id
      ]

      idx <- unlist(
        idx,
        use.names = FALSE
      )

      dx <- unique(
        SCORE$diagnosis[
          idx
        ]
      )

      if (length(dx) != 1L) {
        stop(
          paste(
            "Diagnosis inconsistent within donor:",
            id
          )
        )
      }

      data.frame(

        donor_id =
          id,

        diagnosis =
          dx,

        n_samples =
          length(idx),

        has_DLPFC =
          any(
            SCORE$region[
              idx
            ] == "DLPFC"
          ),

        has_HIPPO =
          any(
            SCORE$region[
              idx
            ] == "HIPPO"
          ),

        stringsAsFactors = FALSE
      )
    }
  )
)

rownames(
  DONOR_AUDIT
) <- NULL


DONOR_AUDIT$availability <- ifelse(
  DONOR_AUDIT$has_DLPFC &
    DONOR_AUDIT$has_HIPPO,
  "paired",
  ifelse(
    DONOR_AUDIT$has_DLPFC,
    "DLPFC_only",
    "HIPPO_only"
  )
)


cat(
  "\n--- DONOR AVAILABILITY ---\n"
)

print(
  as.data.frame(
    table(
      DONOR_AUDIT$diagnosis,
      DONOR_AUDIT$availability
    )
  ),
  row.names = FALSE
)


# ------------------------------------------------------------
# 4. IDENTIFY TRUE PAIRED DONORS
# ------------------------------------------------------------

PAIRED_IDS <- DONOR_AUDIT$donor_id[
  DONOR_AUDIT$availability ==
    "paired"
]


if (
  length(PAIRED_IDS) != 301L
) {
  stop(
    paste0(
      "Expected 301 paired donors; found ",
      length(PAIRED_IDS)
    )
  )
}


DLPFC <- SCORE[
  SCORE$donor_id %in%
    PAIRED_IDS &
  SCORE$region ==
    "DLPFC",
  c(
    "donor_id",
    "diagnosis",
    "SCZ_MSS_v1"
  ),
  drop = FALSE
]


HIPPO <- SCORE[
  SCORE$donor_id %in%
    PAIRED_IDS &
  SCORE$region ==
    "HIPPO",
  c(
    "donor_id",
    "diagnosis",
    "SCZ_MSS_v1"
  ),
  drop = FALSE
]


names(DLPFC)[
  names(DLPFC) ==
    "SCZ_MSS_v1"
] <- "DLPFC_score"

names(HIPPO)[
  names(HIPPO) ==
    "SCZ_MSS_v1"
] <- "HIPPO_score"


PAIRED <- merge(
  DLPFC,
  HIPPO,
  by = c(
    "donor_id",
    "diagnosis"
  ),
  all = FALSE,
  sort = FALSE
)


if (
  nrow(PAIRED) != 301L ||
  anyDuplicated(
    PAIRED$donor_id
  )
) {
  stop(
    "Paired donor construction failed."
  )
}


# ------------------------------------------------------------
# 5. DERIVED PAIRED-STATE VARIABLES
# ------------------------------------------------------------

PAIRED$paired_average_score <- (
  PAIRED$DLPFC_score +
    PAIRED$HIPPO_score
) / 2


PAIRED$regional_delta <- (
  PAIRED$DLPFC_score -
    PAIRED$HIPPO_score
)


PAIRED$absolute_regional_delta <- abs(
  PAIRED$regional_delta
)


# ------------------------------------------------------------
# 6. HELPERS
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  y <- as.integer(y)

  n1 <- sum(
    y == 1L
  )

  n0 <- sum(
    y == 0L
  )

  if (
    n1 == 0L ||
    n0 == 0L
  ) {
    return(
      NA_real_
    )
  }

  r <- rank(
    score,
    ties.method = "average"
  )

  (
    sum(
      r[
        y == 1L
      ]
    ) -
      n1 *
      (
        n1 + 1
      ) / 2
  ) /
    (
      n1 *
      n0
    )
}


hedges_g <- function(
  control,
  scz
) {

  n0 <- length(control)
  n1 <- length(scz)

  pooled_var <- (
    (
      n0 - 1
    ) *
      var(control) +
      (
        n1 - 1
      ) *
      var(scz)
  ) /
    (
      n0 +
      n1 -
      2
    )

  if (
    !is.finite(pooled_var) ||
    pooled_var <= 0
  ) {
    return(
      NA_real_
    )
  }

  d <- (
    mean(scz) -
      mean(control)
  ) /
    sqrt(
      pooled_var
    )

  df <- (
    n0 +
    n1 -
    2
  )

  J <- (
    1 -
      3 /
      (
        4 *
          df -
        1
      )
  )

  J * d
}


mean_ci <- function(x) {

  n <- length(x)

  m <- mean(x)

  se <- sd(x) /
    sqrt(n)

  crit <- qt(
    0.975,
    df = n - 1
  )

  c(
    mean = m,
    lower = m - crit * se,
    upper = m + crit * se
  )
}


# ------------------------------------------------------------
# 7. CROSS-REGION CORRELATION
# ------------------------------------------------------------

COR_PEARSON_ALL <- cor(
  PAIRED$DLPFC_score,
  PAIRED$HIPPO_score,
  method = "pearson"
)

COR_SPEARMAN_ALL <- cor(
  PAIRED$DLPFC_score,
  PAIRED$HIPPO_score,
  method = "spearman"
)


CONTROL_IDX <- (
  PAIRED$diagnosis ==
    "Control"
)

SCZ_IDX <- (
  PAIRED$diagnosis ==
    "Schizo"
)


COR_PEARSON_CONTROL <- cor(
  PAIRED$DLPFC_score[
    CONTROL_IDX
  ],
  PAIRED$HIPPO_score[
    CONTROL_IDX
  ],
  method = "pearson"
)

COR_PEARSON_SCZ <- cor(
  PAIRED$DLPFC_score[
    SCZ_IDX
  ],
  PAIRED$HIPPO_score[
    SCZ_IDX
  ],
  method = "pearson"
)


COR_SPEARMAN_CONTROL <- cor(
  PAIRED$DLPFC_score[
    CONTROL_IDX
  ],
  PAIRED$HIPPO_score[
    CONTROL_IDX
  ],
  method = "spearman"
)

COR_SPEARMAN_SCZ <- cor(
  PAIRED$DLPFC_score[
    SCZ_IDX
  ],
  PAIRED$HIPPO_score[
    SCZ_IDX
  ],
  method = "spearman"
)


# ------------------------------------------------------------
# 8. REGION-SPECIFIC DISEASE EFFECTS
# ------------------------------------------------------------

DX_Y <- ifelse(
  PAIRED$diagnosis ==
    "Schizo",
  1L,
  0L
)


region_effect <- function(
  score
) {

  control <- score[
    DX_Y == 0L
  ]

  scz <- score[
    DX_Y == 1L
  ]

  data.frame(

    control_N =
      length(control),

    schizophrenia_N =
      length(scz),

    control_mean =
      mean(control),

    schizophrenia_mean =
      mean(scz),

    mean_difference_SCZ_minus_Control =
      mean(scz) -
      mean(control),

    hedges_g =
      hedges_g(
        control,
        scz
      ),

    descriptive_AUROC =
      auc_rank(
        DX_Y,
        score
      ),

    stringsAsFactors = FALSE
  )
}


DLPFC_EFFECT <- region_effect(
  PAIRED$DLPFC_score
)

DLPFC_EFFECT$region <- "DLPFC"


HIPPO_EFFECT <- region_effect(
  PAIRED$HIPPO_score
)

HIPPO_EFFECT$region <- "HIPPO"


REGION_EFFECTS <- rbind(
  DLPFC_EFFECT,
  HIPPO_EFFECT
)

REGION_EFFECTS <- REGION_EFFECTS[
  ,
  c(
    "region",
    setdiff(
      names(REGION_EFFECTS),
      "region"
    )
  ),
  drop = FALSE
]


REGION_DISEASE_DIRECTION_CONCORDANT <- (
  sign(
    REGION_EFFECTS$mean_difference_SCZ_minus_Control[
      REGION_EFFECTS$region ==
        "DLPFC"
    ]
  ) ==
  sign(
    REGION_EFFECTS$mean_difference_SCZ_minus_Control[
      REGION_EFFECTS$region ==
        "HIPPO"
    ]
  )
)


# ------------------------------------------------------------
# 9. WITHIN-DONOR REGIONAL DELTA
# ------------------------------------------------------------

DELTA_ALL_CI <- mean_ci(
  PAIRED$regional_delta
)

DELTA_CONTROL_CI <- mean_ci(
  PAIRED$regional_delta[
    CONTROL_IDX
  ]
)

DELTA_SCZ_CI <- mean_ci(
  PAIRED$regional_delta[
    SCZ_IDX
  ]
)


DELTA_CONTROL <- PAIRED$regional_delta[
  CONTROL_IDX
]

DELTA_SCZ <- PAIRED$regional_delta[
  SCZ_IDX
]


# ------------------------------------------------------------
# 10. DIAGNOSIS DIFFERENCE IN REGIONAL DELTA
#
# Equivalent to a score-level diagnosis × region contrast
# in paired donors.
# ------------------------------------------------------------

N0 <- length(
  DELTA_CONTROL
)

N1 <- length(
  DELTA_SCZ
)

V0 <- var(
  DELTA_CONTROL
)

V1 <- var(
  DELTA_SCZ
)

DELTA_DX_DIFFERENCE <- (
  mean(
    DELTA_SCZ
  ) -
    mean(
      DELTA_CONTROL
    )
)


DELTA_DX_SE <- sqrt(
  V1 / N1 +
    V0 / N0
)


DELTA_DX_DF <- (
  V1 / N1 +
    V0 / N0
)^2 /
  (
    (
      V1 / N1
    )^2 /
      (
        N1 - 1
      ) +
    (
      V0 / N0
    )^2 /
      (
        N0 - 1
      )
  )


DELTA_DX_T <- (
  DELTA_DX_DIFFERENCE /
    DELTA_DX_SE
)


DELTA_DX_P <- 2 *
  pt(
    -abs(
      DELTA_DX_T
    ),
    df =
      DELTA_DX_DF
  )


DELTA_DX_CRIT <- qt(
  0.975,
  df =
    DELTA_DX_DF
)


DELTA_DX_CI_LOW <- (
  DELTA_DX_DIFFERENCE -
    DELTA_DX_CRIT *
    DELTA_DX_SE
)

DELTA_DX_CI_HIGH <- (
  DELTA_DX_DIFFERENCE +
    DELTA_DX_CRIT *
    DELTA_DX_SE
)


# ------------------------------------------------------------
# 11. PAIRED-AVERAGE STATE
# ------------------------------------------------------------

AVG_CONTROL <- PAIRED$paired_average_score[
  CONTROL_IDX
]

AVG_SCZ <- PAIRED$paired_average_score[
  SCZ_IDX
]


PAIRED_AVG_G <- hedges_g(
  AVG_CONTROL,
  AVG_SCZ
)

PAIRED_AVG_AUC <- auc_rank(
  DX_Y,
  PAIRED$paired_average_score
)


# ------------------------------------------------------------
# 12. INDIVIDUAL CROSS-REGION DIRECTIONAL CONCORDANCE
#
# Region-specific control means define zero reference.
#
# This is DESCRIPTIVE only.
# ------------------------------------------------------------

CONTROL_DLPFC_MEAN <- mean(
  PAIRED$DLPFC_score[
    CONTROL_IDX
  ]
)

CONTROL_HIPPO_MEAN <- mean(
  PAIRED$HIPPO_score[
    CONTROL_IDX
  ]
)


PAIRED$DLPFC_control_centered <- (
  PAIRED$DLPFC_score -
    CONTROL_DLPFC_MEAN
)

PAIRED$HIPPO_control_centered <- (
  PAIRED$HIPPO_score -
    CONTROL_HIPPO_MEAN
)


PAIRED$centered_direction_concordant <- (
  (
    PAIRED$DLPFC_control_centered > 0 &
    PAIRED$HIPPO_control_centered > 0
  )
  |
  (
    PAIRED$DLPFC_control_centered < 0 &
    PAIRED$HIPPO_control_centered < 0
  )
)


PAIRED$both_regions_above_control_mean <- (
  PAIRED$DLPFC_control_centered > 0 &
  PAIRED$HIPPO_control_centered > 0
)


SCZ_DIRECTION_CONCORDANCE <- mean(
  PAIRED$centered_direction_concordant[
    SCZ_IDX
  ]
)

SCZ_BOTH_ABOVE_CONTROL <- mean(
  PAIRED$both_regions_above_control_mean[
    SCZ_IDX
  ]
)


# ------------------------------------------------------------
# 13. DIAGNOSIS-SPECIFIC DELTA SUMMARY
# ------------------------------------------------------------

DELTA_BY_DIAGNOSIS <- data.frame(

  diagnosis =
    c(
      "Control",
      "Schizo"
    ),

  N =
    c(
      sum(
        CONTROL_IDX
      ),
      sum(
        SCZ_IDX
      )
    ),

  mean_DLPFC =
    c(
      mean(
        PAIRED$DLPFC_score[
          CONTROL_IDX
        ]
      ),
      mean(
        PAIRED$DLPFC_score[
          SCZ_IDX
        ]
      )
    ),

  mean_HIPPO =
    c(
      mean(
        PAIRED$HIPPO_score[
          CONTROL_IDX
        ]
      ),
      mean(
        PAIRED$HIPPO_score[
          SCZ_IDX
        ]
      )
    ),

  mean_DLPFC_minus_HIPPO =
    c(
      DELTA_CONTROL_CI[
        "mean"
      ],
      DELTA_SCZ_CI[
        "mean"
      ]
    ),

  delta_CI_lower =
    c(
      DELTA_CONTROL_CI[
        "lower"
      ],
      DELTA_SCZ_CI[
        "lower"
      ]
    ),

  delta_CI_upper =
    c(
      DELTA_CONTROL_CI[
        "upper"
      ],
      DELTA_SCZ_CI[
        "upper"
      ]
    ),

  mean_absolute_regional_delta =
    c(
      mean(
        PAIRED$absolute_regional_delta[
          CONTROL_IDX
        ]
      ),
      mean(
        PAIRED$absolute_regional_delta[
          SCZ_IDX
        ]
      )
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 14. MASTER SUMMARY
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(

    "Paired_donors",

    "Paired_Control_donors",

    "Paired_Schizophrenia_donors",

    "DLPFC_HIPPO_Pearson_all",

    "DLPFC_HIPPO_Spearman_all",

    "DLPFC_HIPPO_Pearson_Control",

    "DLPFC_HIPPO_Pearson_Schizophrenia",

    "DLPFC_HIPPO_Spearman_Control",

    "DLPFC_HIPPO_Spearman_Schizophrenia",

    "DLPFC_disease_mean_difference",

    "HIPPO_disease_mean_difference",

    "DLPFC_disease_Hedges_g",

    "HIPPO_disease_Hedges_g",

    "DLPFC_disease_descriptive_AUROC",

    "HIPPO_disease_descriptive_AUROC",

    "Region_disease_direction_concordant",

    "Overall_mean_DLPFC_minus_HIPPO",

    "Overall_delta_CI_lower",

    "Overall_delta_CI_upper",

    "SCZ_minus_Control_difference_in_regional_delta",

    "Regional_delta_interaction_CI_lower",

    "Regional_delta_interaction_CI_upper",

    "Regional_delta_interaction_P",

    "Paired_average_disease_Hedges_g",

    "Paired_average_descriptive_AUROC",

    "SCZ_within_donor_centered_direction_concordance",

    "SCZ_both_regions_above_control_mean"
  ),

  value = c(

    nrow(PAIRED),

    sum(
      CONTROL_IDX
    ),

    sum(
      SCZ_IDX
    ),

    COR_PEARSON_ALL,

    COR_SPEARMAN_ALL,

    COR_PEARSON_CONTROL,

    COR_PEARSON_SCZ,

    COR_SPEARMAN_CONTROL,

    COR_SPEARMAN_SCZ,

    REGION_EFFECTS$mean_difference_SCZ_minus_Control[
      REGION_EFFECTS$region ==
        "DLPFC"
    ],

    REGION_EFFECTS$mean_difference_SCZ_minus_Control[
      REGION_EFFECTS$region ==
        "HIPPO"
    ],

    REGION_EFFECTS$hedges_g[
      REGION_EFFECTS$region ==
        "DLPFC"
    ],

    REGION_EFFECTS$hedges_g[
      REGION_EFFECTS$region ==
        "HIPPO"
    ],

    REGION_EFFECTS$descriptive_AUROC[
      REGION_EFFECTS$region ==
        "DLPFC"
    ],

    REGION_EFFECTS$descriptive_AUROC[
      REGION_EFFECTS$region ==
        "HIPPO"
    ],

    as.numeric(
      REGION_DISEASE_DIRECTION_CONCORDANT
    ),

    DELTA_ALL_CI[
      "mean"
    ],

    DELTA_ALL_CI[
      "lower"
    ],

    DELTA_ALL_CI[
      "upper"
    ],

    DELTA_DX_DIFFERENCE,

    DELTA_DX_CI_LOW,

    DELTA_DX_CI_HIGH,

    DELTA_DX_P,

    PAIRED_AVG_G,

    PAIRED_AVG_AUC,

    SCZ_DIRECTION_CONCORDANCE,

    SCZ_BOTH_ABOVE_CONTROL
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 15. SAVE
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "08_score_lock"
)


PAIRED_PATH <- file.path(
  OUT_DIR,
  "step07c_paired_donor_SCZ_MSS_v1_scores.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "step07c_paired_donor_summary_v1.csv"
)

REGION_EFFECT_PATH <- file.path(
  OUT_DIR,
  "step07c_paired_region_disease_effects_v1.csv"
)

DELTA_PATH <- file.path(
  OUT_DIR,
  "step07c_paired_regional_delta_by_diagnosis_v1.csv"
)

AUDIT_PATH <- file.path(
  OUT_DIR,
  "step07c_donor_availability_audit_v1.csv"
)

DEFINITION_PATH <- file.path(
  OUT_DIR,
  "step07c_paired_donor_analysis_definition_v1.txt"
)


write.csv(
  PAIRED,
  PAIRED_PATH,
  row.names = FALSE
)

write.csv(
  SUMMARY,
  SUMMARY_PATH,
  row.names = FALSE
)

write.csv(
  REGION_EFFECTS,
  REGION_EFFECT_PATH,
  row.names = FALSE
)

write.csv(
  DELTA_BY_DIAGNOSIS,
  DELTA_PATH,
  row.names = FALSE
)

write.csv(
  DONOR_AUDIT,
  AUDIT_PATH,
  row.names = FALSE
)


# ------------------------------------------------------------
# 16. TRACEABILITY HASHES
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {

  install.packages(
    "digest",
    repos =
      "https://cloud.r-project.org"
  )
}


WEIGHT_SHA <- if (
  file.exists(
    WEIGHT_PATH
  )
) {

  digest::digest(
    file =
      WEIGHT_PATH,
    algo =
      "sha256"
  )

} else {

  NA_character_
}


FORMULA_SHA <- if (
  file.exists(
    FORMULA_PATH
  )
) {

  digest::digest(
    file =
      FORMULA_PATH,
    algo =
      "sha256"
  )

} else {

  NA_character_
}


# ------------------------------------------------------------
# 17. ANALYSIS DEFINITION
# ------------------------------------------------------------

writeLines(
  c(

    "STEP 07C — PAIRED-DONOR CROSS-REGION STATE ANALYSIS",

    "",

    paste0(
      "paired_donors=",
      nrow(PAIRED)
    ),

    paste0(
      "paired_controls=",
      sum(
        CONTROL_IDX
      )
    ),

    paste0(
      "paired_schizophrenia=",
      sum(
        SCZ_IDX
      )
    ),

    "",

    "Primary paired-state quantities:",

    "1. Pearson and Spearman DLPFC-HIPPO score correlation",

    "2. Region-specific SCZ-Control score shift",

    "3. Within-donor regional delta = DLPFC - HIPPO",

    paste(
      "4. Diagnosis difference in regional delta,",
      "interpreted as a score-level diagnosis-by-region contrast"
    ),

    "5. Paired-average shared-state score",

    paste(
      "6. Individual SCZ directional concordance after",
      "region-specific control-mean centering"
    ),

    "",

    paste0(
      "weight_sha256=",
      WEIGHT_SHA
    ),

    paste0(
      "formula_sha256=",
      FORMULA_SHA
    ),

    "",

    "analysis_is_discovery_postselection_descriptive=TRUE",

    "external_data_used=FALSE",

    "external_outcome_used=FALSE"
  ),
  DEFINITION_PATH
)


# ------------------------------------------------------------
# 18. HARD CHECKS
# ------------------------------------------------------------

stopifnot(

  nrow(PAIRED) == 301L,

  length(
    unique(
      PAIRED$donor_id
    )
  ) == 301L,

  all(
    is.finite(
      PAIRED$DLPFC_score
    )
  ),

  all(
    is.finite(
      PAIRED$HIPPO_score
    )
  ),

  is.finite(
    COR_PEARSON_ALL
  ),

  is.finite(
    DELTA_DX_P
  ),

  file.exists(
    SUMMARY_PATH
  )
)


# ------------------------------------------------------------
# 19. OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START


cat("\n============================================================\n")
cat("STEP 07C — PAIRED-DONOR CROSS-REGION STATE ANALYSIS\n")
cat("============================================================\n")


cat(
  "Paired donors: ",
  nrow(PAIRED),
  "\n",
  sep = ""
)

cat(
  "Controls:      ",
  sum(CONTROL_IDX),
  "\n",
  sep = ""
)

cat(
  "Schizophrenia: ",
  sum(SCZ_IDX),
  "\n",
  sep = ""
)


cat(
  "\n--- CROSS-REGION CORRELATION ---\n"
)

cat(
  "Pearson overall:  ",
  round(
    COR_PEARSON_ALL,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Spearman overall: ",
  round(
    COR_SPEARMAN_ALL,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Pearson Control:  ",
  round(
    COR_PEARSON_CONTROL,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Pearson SCZ:      ",
  round(
    COR_PEARSON_SCZ,
    4
  ),
  "\n",
  sep = ""
)


cat(
  "\n--- REGION-SPECIFIC DISEASE EFFECT ---\n"
)

print(
  REGION_EFFECTS,
  row.names = FALSE
)


cat(
  "\n--- REGIONAL DELTA BY DIAGNOSIS ---\n"
)

print(
  DELTA_BY_DIAGNOSIS,
  row.names = FALSE
)


cat(
  "\nSCZ-Control difference in regional delta: ",
  round(
    DELTA_DX_DIFFERENCE,
    6
  ),
  "\n",
  sep = ""
)

cat(
  "95% CI: [",
  round(
    DELTA_DX_CI_LOW,
    6
  ),
  ", ",
  round(
    DELTA_DX_CI_HIGH,
    6
  ),
  "]\n",
  sep = ""
)

cat(
  "Interaction P: ",
  format(
    DELTA_DX_P,
    digits = 5
  ),
  "\n",
  sep = ""
)


cat(
  "\nPaired-average Hedges' g: ",
  round(
    PAIRED_AVG_G,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Paired-average descriptive AUROC: ",
  round(
    PAIRED_AVG_AUC,
    4
  ),
  "\n",
  sep = ""
)


cat(
  "\nSCZ centered-direction concordance: ",
  round(
    100 *
      SCZ_DIRECTION_CONCORDANCE,
    1
  ),
  "%\n",
  sep = ""
)

cat(
  "SCZ both regions above regional control mean: ",
  round(
    100 *
      SCZ_BOTH_ABOVE_CONTROL,
    1
  ),
  "%\n",
  sep = ""
)


cat(
  "\nIMPORTANT: paired diagnosis effects are post-selection/descriptive.\n"
)


cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS,
    1
  ),
  " sec\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 07C — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS

--- DONOR AVAILABILITY ---
    Var1       Var2 Freq
 Control DLPFC_only   34
  Schizo DLPFC_only   44
 Control HIPPO_only   51
  Schizo HIPPO_only   24
 Control     paired  192
  Schizo     paired  109

STEP 07C — PAIRED-DONOR CROSS-REGION STATE ANALYSIS
Paired donors: 301
Controls:      192
Schizophrenia: 109

--- CROSS-REGION CORRELATION ---
Pearson overall:  0.2074
Spearman overall: 0.2341
Pearson Control:  0.1003
Pearson SCZ:      0.1473

--- REGION-SPECIFIC DISEASE EFFECT ---
 region control_N schizophrenia_N control_mean schizophrenia_mean
  DLPFC       192             109   0.01757794         0.03514395
  HIPPO       192             109   0.02462552         0.03873057
 mean_difference_SCZ_minus_Control  hedges_g descriptive_AUROC
                        0.01756601 0.8366325         0.7237672
                        0.01410505 0.5946706         0.6689125

--- REGIONAL DELTA BY DIAGNOSIS ---
 diagnosis   N mean_DLPFC mean_HIPPO mean_DLPFC_minus_HIPPO delta_

In [23]:
# ============================================================
# STEP 07D.1 — REPAIR / VERIFY SCZ-MSS v1 SCORE LOCK
#
# Fixes the integrity-check bug caused by vector names
# retained by vapply().
#
# DOES NOT modify:
#   - frozen genes
#   - weights
#   - formula
#   - mapping policy
#   - technical gate
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID   <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop("digest package unavailable.")
}

if (!requireNamespace(
  "jsonlite",
  quietly = TRUE
)) {
  stop("jsonlite package unavailable.")
}


# ------------------------------------------------------------
# 2. SCORE-LOCK PATHS
# ------------------------------------------------------------

SCORE_DIR <- file.path(
  PROJECT_ROOT,
  "08_score_lock"
)

WEIGHT_PATH <- file.path(
  SCORE_DIR,
  "SCZ_MSS_v1_gene_weights.csv"
)

FORMULA_PATH <- file.path(
  SCORE_DIR,
  "SCZ_MSS_v1_formula.md"
)

FINAL_STATE_PATH <- file.path(
  PROJECT_ROOT,
  "07_stability",
  "final_stable_state_v1",
  "step06d_FINAL_stable_shared_state_v1.csv"
)

MAPPING_PATH <- file.path(
  SCORE_DIR,
  "SCZ_MSS_v1_mapping_policy.json"
)

GATE_PATH <- file.path(
  SCORE_DIR,
  "SCZ_MSS_v1_technical_gate.json"
)

PAIRED_SUMMARY_PATH <- file.path(
  SCORE_DIR,
  "step07c_paired_donor_summary_v1.csv"
)

MANIFEST_PATH <- file.path(
  SCORE_DIR,
  "SCZ_MSS_v1_lock_manifest.csv"
)

MASTER_SHA_PATH <- file.path(
  SCORE_DIR,
  "SCZ_MSS_v1.sha256"
)

LOCK_SUMMARY_PATH <- file.path(
  SCORE_DIR,
  "SCZ_MSS_v1_LOCK_SUMMARY.txt"
)


REQUIRED <- c(
  WEIGHT_PATH,
  FORMULA_PATH,
  FINAL_STATE_PATH,
  MAPPING_PATH,
  GATE_PATH,
  PAIRED_SUMMARY_PATH,
  MANIFEST_PATH,
  MASTER_SHA_PATH,
  LOCK_SUMMARY_PATH
)

if (!all(file.exists(REQUIRED))) {

  cat("Missing files:\n")

  print(
    REQUIRED[
      !file.exists(REQUIRED)
    ]
  )

  stop(
    "STEP 07D artifacts incomplete."
  )
}


# ------------------------------------------------------------
# 3. VERIFY FROZEN WEIGHT TABLE
# ------------------------------------------------------------

WEIGHTS <- read.csv(
  WEIGHT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(WEIGHTS) == 24L,
  anyDuplicated(
    WEIGHTS$ensemblID_stable
  ) == 0L,
  abs(
    sum(
      abs(
        WEIGHTS$final_normalized_weight
      )
    ) - 1
  ) < 1e-10,
  all(
    sign(
      WEIGHTS$final_normalized_weight
    ) ==
      WEIGHTS$direction_code
  )
)

cat("Frozen 24-gene weight table: PASS\n")


# ------------------------------------------------------------
# 4. VERIFY JSON LOCK FILES
# ------------------------------------------------------------

MAPPING_CHECK <- jsonlite::fromJSON(
  MAPPING_PATH,
  simplifyVector = FALSE
)

GATE_CHECK <- jsonlite::fromJSON(
  GATE_PATH,
  simplifyVector = FALSE
)

if (
  is.null(MAPPING_CHECK$score_name) ||
  is.null(GATE_CHECK$score_name)
) {
  stop(
    "Mapping/gate JSON read-back failed."
  )
}

cat("Mapping policy JSON: PASS\n")
cat("Technical gate JSON: PASS\n")


# ------------------------------------------------------------
# 5. READ LOCK MANIFEST
# ------------------------------------------------------------

MANIFEST_CHECK <- read.csv(
  MANIFEST_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

REQUIRED_MANIFEST_COLUMNS <- c(
  "artifact",
  "path",
  "sha256"
)

if (!all(
  REQUIRED_MANIFEST_COLUMNS %in%
    names(MANIFEST_CHECK)
)) {
  stop(
    "Lock manifest schema invalid."
  )
}

if (
  nrow(MANIFEST_CHECK) != 6L
) {
  stop(
    paste0(
      "Expected 6 canonical lock artifacts; found ",
      nrow(MANIFEST_CHECK)
    )
  )
}


# ------------------------------------------------------------
# 6. RECOMPUTE SHA256 FOR EACH CANONICAL ARTIFACT
# ------------------------------------------------------------

CURRENT_SHA <- vapply(
  MANIFEST_CHECK$path,
  function(path) {

    if (!file.exists(path)) {
      return(NA_character_)
    }

    digest::digest(
      file = path,
      algo = "sha256"
    )
  },
  character(1)
)


# ------------------------------------------------------------
# 7. CORRECTED INTEGRITY COMPARISON
#
# IMPORTANT FIX:
#   vapply() preserves names.
#   CSV column has no names attribute.
#
# Compare VALUES, not vector attributes.
# ------------------------------------------------------------

EXPECTED_SHA <- as.character(
  MANIFEST_CHECK$sha256
)

ACTUAL_SHA <- unname(
  as.character(
    CURRENT_SHA
  )
)

MATCHES <- (
  !is.na(ACTUAL_SHA) &
  ACTUAL_SHA ==
    EXPECTED_SHA
)


VERIFY_TABLE <- data.frame(

  artifact =
    MANIFEST_CHECK$artifact,

  expected_sha256 =
    EXPECTED_SHA,

  actual_sha256 =
    ACTUAL_SHA,

  match =
    MATCHES,

  stringsAsFactors = FALSE
)


cat(
  "\n--- CANONICAL ARTIFACT SHA VERIFICATION ---\n"
)

print(
  VERIFY_TABLE[
    ,
    c(
      "artifact",
      "match"
    )
  ],
  row.names = FALSE
)


if (!all(MATCHES)) {

  cat(
    "\nTRUE SHA mismatch detected:\n"
  )

  print(
    VERIFY_TABLE[
      !VERIFY_TABLE$match,
      ,
      drop = FALSE
    ],
    row.names = FALSE
  )

  stop(
    "Canonical score-lock artifact genuinely changed."
  )
}


cat(
  "\nAll canonical artifact SHA-256 values: PASS\n"
)


# ------------------------------------------------------------
# 8. RECONSTRUCT MASTER SHA
# ------------------------------------------------------------

RECONSTRUCTED_TEXT <- paste(
  paste0(
    MANIFEST_CHECK$artifact,
    "=",
    EXPECTED_SHA
  ),
  collapse = "\n"
)


RECONSTRUCTED_MASTER_SHA <- digest::digest(
  RECONSTRUCTED_TEXT,
  algo = "sha256",
  serialize = FALSE
)


STORED_MASTER_SHA <- trimws(
  readLines(
    MASTER_SHA_PATH,
    warn = FALSE
  )[1]
)


cat(
  "\nStored master SHA:\n",
  STORED_MASTER_SHA,
  "\n",
  sep = ""
)

cat(
  "\nReconstructed master SHA:\n",
  RECONSTRUCTED_MASTER_SHA,
  "\n",
  sep = ""
)


if (!identical(
  STORED_MASTER_SHA,
  RECONSTRUCTED_MASTER_SHA
)) {
  stop(
    "MASTER SHA mismatch."
  )
}


# ------------------------------------------------------------
# 9. VERIFY KEY GATE VALUES
# ------------------------------------------------------------

GENE_THRESHOLD <- as.numeric(
  GATE_CHECK$required_criteria$
    mapped_gene_count_fraction$
    threshold
)

ABS_THRESHOLD <- as.numeric(
  GATE_CHECK$required_criteria$
    total_absolute_weight_coverage$
    threshold
)

POS_THRESHOLD <- as.numeric(
  GATE_CHECK$required_criteria$
    positive_absolute_weight_coverage$
    threshold
)

NEG_THRESHOLD <- as.numeric(
  GATE_CHECK$required_criteria$
    negative_absolute_weight_coverage$
    threshold
)


stopifnot(
  abs(
    GENE_THRESHOLD - 0.80
  ) < 1e-12,
  abs(
    ABS_THRESHOLD - 0.80
  ) < 1e-12,
  abs(
    POS_THRESHOLD - 0.70
  ) < 1e-12,
  abs(
    NEG_THRESHOLD - 0.70
  ) < 1e-12
)


# ------------------------------------------------------------
# 10. FINAL OUTPUT
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 07D — SCZ-MSS v1 FORMAL SCORE LOCK VERIFIED\n")
cat("============================================================\n")

cat(
  "Frozen genes: 24\n"
)

cat(
  "Canonical lock artifacts verified: ",
  sum(MATCHES),
  " / ",
  length(MATCHES),
  "\n",
  sep = ""
)

cat(
  "\nTechnical gate:\n"
)

cat(
  "  Gene-count coverage:      >= 80%\n"
)

cat(
  "  Absolute-weight coverage: >= 80%\n"
)

cat(
  "  Positive-weight coverage: >= 70%\n"
)

cat(
  "  Negative-weight coverage: >= 70%\n"
)

cat(
  "\nMASTER SCZ-MSS v1 SHA-256:\n",
  STORED_MASTER_SHA,
  "\n",
  sep = ""
)

cat(
  "\nScore definition frozen: YES\n"
)

cat(
  "External score-based outcome evaluation: NOT YET\n"
)

cat(
  "Reason: prespecified baseline/ablation block remains outstanding.\n"
)

cat("\n============================================================\n")
cat("STEP 07D — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Frozen 24-gene weight table: PASS
Mapping policy JSON: PASS
Technical gate JSON: PASS

--- CANONICAL ARTIFACT SHA VERIFICATION ---
             artifact match
         gene_weights  TRUE
   rank_score_formula  TRUE
   final_stable_state  TRUE
       mapping_policy  TRUE
       technical_gate  TRUE
 paired_donor_summary  TRUE

All canonical artifact SHA-256 values: PASS

Stored master SHA:
b8a57217a92b7ceedb44f6e028e8a2be9e4d107ae66ac6c2e958c56e6a5236ff

Reconstructed master SHA:
b8a57217a92b7ceedb44f6e028e8a2be9e4d107ae66ac6c2e958c56e6a5236ff

STEP 07D — SCZ-MSS v1 FORMAL SCORE LOCK VERIFIED
Frozen genes: 24
Canonical lock artifacts verified: 6 / 6

Technical gate:
  Gene-count coverage:      >= 80%
  Absolute-weight coverage: >= 80%
  Positive-weight coverage: >= 70%
  Negative-weight coverage: >= 70%

MASTER SCZ-MSS v1 SHA-256:
b8a57217a92b7ceedb44f6e028e8a2be9e4d107ae66ac6c2e958c56e6a5236ff

Score definition frozen: YES
External score-based outcome evaluation:

In [28]:
# ============================================================
# STEP 05A.1 — BASELINE MODEL A
# DLPFC-ONLY ELASTIC-NET
#
# VERIFIED SPLIT SCHEMA:
#   donor      = BrNum
#   outer fold = outer_fold
#   repeat     = repeat_id
#
# Uses EXACT locked 5-fold x 5-repeat donor assignments.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(OBSERVED_ID, PROJECT_ID)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("glmnet", quietly = TRUE)) {
  stop("glmnet package unavailable.")
}

cat(
  "glmnet version: ",
  as.character(packageVersion("glmnet")),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 2. INPUT PATHS
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

SPLIT_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "cv_splits",
  "outer_cv_donor_splits_5fold_5repeat_v1.csv"
)

REQUIRED_PATHS <- c(
  COUNTS_PATH,
  META_PATH,
  SPLIT_PATH
)

if (!all(file.exists(REQUIRED_PATHS))) {

  print(
    REQUIRED_PATHS[
      !file.exists(REQUIRED_PATHS)
    ]
  )

  stop("Required input missing.")
}


# ------------------------------------------------------------
# 3. SETTINGS
# ------------------------------------------------------------

ALPHA_GRID <- c(
  0.10,
  0.25,
  0.50,
  0.75,
  1.00
)

INNER_K <- 4L
N_LAMBDA <- 50L
BASE_SEED <- 2026100100L
ZERO_TOL <- 1e-12


# ------------------------------------------------------------
# 4. LOAD DATA
# ------------------------------------------------------------

COUNTS <- readRDS(
  COUNTS_PATH
)

META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

SPLITS <- read.csv(
  SPLIT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(COUNTS) == 23740L,
  ncol(COUNTS) == 755L,
  nrow(META) == 755L
)

cat(
  "Counts: ",
  nrow(COUNTS),
  " genes x ",
  ncol(COUNTS),
  " samples\n",
  sep = ""
)

cat(
  "Metadata: ",
  nrow(META),
  " samples\n",
  sep = ""
)


# ------------------------------------------------------------
# 5. EXACT LOCKED SPLIT SCHEMA
# ------------------------------------------------------------

REQUIRED_SPLIT_COLUMNS <- c(
  "BrNum",
  "outer_fold",
  "repeat_id"
)

MISSING_SPLIT_COLUMNS <- setdiff(
  REQUIRED_SPLIT_COLUMNS,
  names(SPLITS)
)

if (length(MISSING_SPLIT_COLUMNS) > 0L) {

  cat("Actual split columns:\n")
  print(names(SPLITS))

  cat("Missing required columns:\n")
  print(MISSING_SPLIT_COLUMNS)

  stop("Locked split schema mismatch.")
}

cat(
  "Locked split schema: PASS\n"
)

cat(
  "  donor column:  BrNum\n",
  "  repeat column: repeat_id\n",
  "  fold column:   outer_fold\n",
  sep = ""
)


# ------------------------------------------------------------
# 6. NORMALIZE LOCKED SPLITS
# ------------------------------------------------------------

LOCKED_SPLITS <- data.frame(

  donor_id = as.character(
    SPLITS[
      ,
      "BrNum",
      drop = TRUE
    ]
  ),

  repeat_id = as.integer(
    SPLITS[
      ,
      "repeat_id",
      drop = TRUE
    ]
  ),

  fold_id = as.integer(
    SPLITS[
      ,
      "outer_fold",
      drop = TRUE
    ]
  ),

  stringsAsFactors = FALSE
)


if (
  anyNA(LOCKED_SPLITS$donor_id) ||
  anyNA(LOCKED_SPLITS$repeat_id) ||
  anyNA(LOCKED_SPLITS$fold_id)
) {
  stop("NA detected in locked split assignments.")
}


# Each donor should occur exactly once in each repeat
DUP_CHECK <- duplicated(
  LOCKED_SPLITS[
    ,
    c(
      "donor_id",
      "repeat_id"
    ),
    drop = FALSE
  ]
)

if (any(DUP_CHECK)) {
  stop(
    "A donor has more than one fold assignment within a repeat."
  )
}


# ------------------------------------------------------------
# 7. VERIFY 5 x 5 OUTER DESIGN
# ------------------------------------------------------------

EXPECTED_REPEATS <- 1:5
EXPECTED_FOLDS <- 1:5

if (!identical(
  sort(unique(LOCKED_SPLITS$repeat_id)),
  EXPECTED_REPEATS
)) {
  stop("Expected repeat_id values 1:5.")
}

if (!identical(
  sort(unique(LOCKED_SPLITS$fold_id)),
  EXPECTED_FOLDS
)) {
  stop("Expected outer_fold values 1:5.")
}


COMBOS <- expand.grid(
  repeat_id = EXPECTED_REPEATS,
  fold_id = EXPECTED_FOLDS,
  KEEP.OUT.ATTRS = FALSE,
  stringsAsFactors = FALSE
)

COMBOS <- COMBOS[
  order(
    COMBOS$repeat_id,
    COMBOS$fold_id
  ),
  ,
  drop = FALSE
]

rownames(COMBOS) <- NULL


cat(
  "Locked outer evaluations: ",
  nrow(COMBOS),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 8. VERIFY DONOR UNIVERSE
# ------------------------------------------------------------

DISCOVERY_DONORS <- unique(
  as.character(
    META$BrNum
  )
)

SPLIT_DONORS <- unique(
  LOCKED_SPLITS$donor_id
)

if (
  length(DISCOVERY_DONORS) != 454L ||
  length(SPLIT_DONORS) != 454L
) {
  stop(
    paste0(
      "Expected 454 donors; metadata=",
      length(DISCOVERY_DONORS),
      ", split=",
      length(SPLIT_DONORS)
    )
  )
}


MISSING_DONORS <- setdiff(
  DISCOVERY_DONORS,
  SPLIT_DONORS
)

EXTRA_DONORS <- setdiff(
  SPLIT_DONORS,
  DISCOVERY_DONORS
)

if (
  length(MISSING_DONORS) > 0L ||
  length(EXTRA_DONORS) > 0L
) {
  stop(
    "Metadata and split donor universes differ."
  )
}


# Every donor should have 5 assignments
DONOR_ASSIGNMENT_COUNT <- table(
  LOCKED_SPLITS$donor_id
)

if (!all(
  DONOR_ASSIGNMENT_COUNT == 5L
)) {
  stop(
    "Each donor must have exactly 5 outer-fold assignments."
  )
}


cat(
  "Locked donor universe: PASS (454 donors)\n"
)


# ------------------------------------------------------------
# 9. DLPFC-ONLY SUBSET
# ------------------------------------------------------------

DLPFC_INDEX <- which(
  META$Region == "DLPFC"
)

META_D <- META[
  DLPFC_INDEX,
  ,
  drop = FALSE
]

COUNTS_D <- COUNTS[
  ,
  DLPFC_INDEX,
  drop = FALSE
]


DLPFC_DONORS <- as.character(
  META_D$BrNum
)

if (anyDuplicated(DLPFC_DONORS)) {
  stop(
    "DLPFC subset unexpectedly contains duplicate donors."
  )
}


stopifnot(
  nrow(META_D) == 379L,
  ncol(COUNTS_D) == 379L,
  length(unique(DLPFC_DONORS)) == 379L
)


cat(
  "DLPFC samples/donors: ",
  nrow(META_D),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 10. OUTCOME
# ------------------------------------------------------------

Y <- ifelse(
  META_D$Dx == "Schizo",
  1L,
  ifelse(
    META_D$Dx == "Control",
    0L,
    NA_integer_
  )
)

if (anyNA(Y)) {
  stop(
    "Unexpected diagnosis in DLPFC subset."
  )
}

stopifnot(
  sum(Y == 0L) == 226L,
  sum(Y == 1L) == 153L
)


cat(
  "Controls: ",
  sum(Y == 0L),
  "\n",
  sep = ""
)

cat(
  "Schizophrenia: ",
  sum(Y == 1L),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 11. PER-SAMPLE LOG2(CPM + 1)
# ------------------------------------------------------------

LIB_SIZE <- colSums(
  COUNTS_D
)

if (
  any(!is.finite(LIB_SIZE)) ||
  any(LIB_SIZE <= 0)
) {
  stop(
    "Invalid library sizes."
  )
}


LOGCPM_D <- log2(
  sweep(
    COUNTS_D,
    2,
    LIB_SIZE / 1e6,
    "/"
  ) + 1
)


X_EXPR <- t(
  LOGCPM_D
)


if (
  anyNA(X_EXPR) ||
  any(!is.finite(X_EXPR))
) {
  stop(
    "Invalid expression matrix."
  )
}


rm(
  COUNTS,
  COUNTS_D,
  LOGCPM_D,
  LIB_SIZE
)

invisible(gc())


# ------------------------------------------------------------
# 12. COVARIATE MATRIX
# ------------------------------------------------------------

META_D$Sex <- factor(
  META_D$Sex
)

META_D$Race <- factor(
  META_D$Race
)


COVARIATE_MATRIX <- model.matrix(

  ~ donor_age +
    Sex +
    Race +
    RIN +
    snpPC1 +
    snpPC2 +
    snpPC3 +
    snpPC4 +
    snpPC5 +
    snpPC6 +
    snpPC7 +
    snpPC8 +
    snpPC9 +
    snpPC10,

  data = META_D
)


COVARIATE_MATRIX <- COVARIATE_MATRIX[
  ,
  colnames(COVARIATE_MATRIX) != "(Intercept)",
  drop = FALSE
]


if (
  anyNA(COVARIATE_MATRIX) ||
  any(!is.finite(COVARIATE_MATRIX))
) {
  stop(
    "Invalid covariate matrix."
  )
}


N_COV <- ncol(
  COVARIATE_MATRIX
)

N_GENES <- ncol(
  X_EXPR
)


X_MODEL <- cbind(
  COVARIATE_MATRIX,
  X_EXPR
)


PENALTY_FACTOR <- c(
  rep(
    0,
    N_COV
  ),
  rep(
    1,
    N_GENES
  )
)


stopifnot(
  length(PENALTY_FACTOR) ==
    ncol(X_MODEL)
)


cat(
  "Unpenalized covariates: ",
  N_COV,
  "\n",
  sep = ""
)

cat(
  "Penalized genes: ",
  N_GENES,
  "\n",
  sep = ""
)

cat(
  "Total predictors: ",
  ncol(X_MODEL),
  "\n",
  sep = ""
)


rm(
  X_EXPR,
  COVARIATE_MATRIX
)

invisible(gc())


# ------------------------------------------------------------
# 13. AUC HELPER
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  valid <- (
    !is.na(y) &
    is.finite(score)
  )

  y <- as.integer(
    y[valid]
  )

  score <- score[
    valid
  ]


  n_case <- sum(
    y == 1L
  )

  n_control <- sum(
    y == 0L
  )


  if (
    n_case == 0L ||
    n_control == 0L
  ) {
    return(NA_real_)
  }


  ranks <- rank(
    score,
    ties.method = "average"
  )


  (
    sum(
      ranks[
        y == 1L
      ]
    ) -
      n_case *
      (
        n_case + 1
      ) / 2
  ) /
    (
      n_case *
      n_control
    )
}


# ------------------------------------------------------------
# 14. INNER CV FOLD HELPER
#
# DLPFC-only has one sample per donor.
# Stratify by diagnosis.
# ------------------------------------------------------------

make_inner_foldid <- function(
  train_index,
  seed_value
) {

  set.seed(
    seed_value
  )


  y_train <- Y[
    train_index
  ]


  foldid <- rep(
    NA_integer_,
    length(train_index)
  )


  for (
    class_value in c(
      0L,
      1L
    )
  ) {

    class_index <- which(
      y_train == class_value
    )


    shuffled <- sample(
      class_index,
      length(class_index),
      replace = FALSE
    )


    foldid[
      shuffled
    ] <- rep(
      seq_len(INNER_K),
      length.out =
        length(shuffled)
    )
  }


  if (anyNA(foldid)) {
    stop(
      "Inner fold construction failed."
    )
  }


  for (
    inner_fold in seq_len(INNER_K)
  ) {

    classes_here <- unique(
      y_train[
        foldid == inner_fold
      ]
    )


    if (
      !all(
        c(
          0L,
          1L
        ) %in%
          classes_here
      )
    ) {
      stop(
        paste0(
          "Inner fold ",
          inner_fold,
          " lacks one diagnosis class."
        )
      )
    }
  }


  foldid
}


# ------------------------------------------------------------
# 15. OUTPUT DIRECTORIES
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_A_DLPFC_only_v1"
)

FIT_DIR <- file.path(
  OUT_DIR,
  "outer_fits"
)

dir.create(
  FIT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


# ------------------------------------------------------------
# 16. RUN 25 LOCKED OUTER EVALUATIONS
# ------------------------------------------------------------

for (
  eval_index in seq_len(
    nrow(COMBOS)
  )
) {

  current_repeat <- COMBOS$repeat_id[
    eval_index
  ]

  current_fold <- COMBOS$fold_id[
    eval_index
  ]


  CHECKPOINT_PATH <- file.path(
    FIT_DIR,
    sprintf(
      "repeat_%02d_fold_%02d.rds",
      current_repeat,
      current_fold
    )
  )


  if (file.exists(
    CHECKPOINT_PATH
  )) {

    cat(
      sprintf(
        "[%02d/25] R%d F%d already complete; skipping.\n",
        eval_index,
        current_repeat,
        current_fold
      )
    )

    next
  }


  TEST_DONORS_LOCKED <- LOCKED_SPLITS$donor_id[
    LOCKED_SPLITS$repeat_id ==
      current_repeat &
    LOCKED_SPLITS$fold_id ==
      current_fold
  ]


  TEST_INDEX <- which(
    as.character(
      META_D$BrNum
    ) %in%
      TEST_DONORS_LOCKED
  )


  TRAIN_INDEX <- setdiff(
    seq_len(
      nrow(META_D)
    ),
    TEST_INDEX
  )


  if (
    length(TEST_INDEX) == 0L ||
    length(TRAIN_INDEX) == 0L
  ) {
    stop(
      paste0(
        "Empty outer partition at R",
        current_repeat,
        " F",
        current_fold
      )
    )
  }


  TRAIN_DONORS <- as.character(
    META_D$BrNum[
      TRAIN_INDEX
    ]
  )

  TEST_DONORS <- as.character(
    META_D$BrNum[
      TEST_INDEX
    ]
  )


  if (
    length(
      intersect(
        TRAIN_DONORS,
        TEST_DONORS
      )
    ) > 0L
  ) {
    stop(
      "Outer donor leakage detected."
    )
  }


  if (
    length(
      unique(
        Y[
          TEST_INDEX
        ]
      )
    ) != 2L
  ) {
    stop(
      paste0(
        "Outer test fold lacks one class at R",
        current_repeat,
        " F",
        current_fold
      )
    )
  }


  INNER_FOLDID <- make_inner_foldid(
    train_index =
      TRAIN_INDEX,
    seed_value =
      BASE_SEED +
      current_repeat * 100L +
      current_fold
  )


  BEST_MODEL <- NULL


  # ----------------------------------------------------------
  # INNER ALPHA/LAMBDA TUNING
  # ----------------------------------------------------------

  for (
    current_alpha in ALPHA_GRID
  ) {

    CV_FIT <- glmnet::cv.glmnet(

      x =
        X_MODEL[
          TRAIN_INDEX,
          ,
          drop = FALSE
        ],

      y =
        Y[
          TRAIN_INDEX
        ],

      family =
        "binomial",

      weights =
        rep(
          1,
          length(TRAIN_INDEX)
        ),

      foldid =
        INNER_FOLDID,

      type.measure =
        "auc",

      alpha =
        current_alpha,

      penalty.factor =
        PENALTY_FACTOR,

      standardize =
        TRUE,

      intercept =
        TRUE,

      nlambda =
        N_LAMBDA,

      parallel =
        FALSE
    )


    VALID_INDEX <- which(
      is.finite(
        CV_FIT$cvm
      )
    )


    if (
      length(VALID_INDEX) == 0L
    ) {

      rm(CV_FIT)
      invisible(gc())

      next
    }


    BEST_LAMBDA_INDEX <- VALID_INDEX[
      which.max(
        CV_FIT$cvm[
          VALID_INDEX
        ]
      )
    ]


    CANDIDATE <- list(

      alpha =
        current_alpha,

      lambda =
        CV_FIT$lambda[
          BEST_LAMBDA_INDEX
        ],

      inner_auc =
        CV_FIT$cvm[
          BEST_LAMBDA_INDEX
        ],

      inner_se =
        CV_FIT$cvsd[
          BEST_LAMBDA_INDEX
        ]
    )


    if (is.null(
      BEST_MODEL
    )) {

      BEST_MODEL <- CANDIDATE

    } else {

      better_auc <- (
        CANDIDATE$inner_auc >
          BEST_MODEL$inner_auc +
          1e-12
      )


      tied_auc <- (
        abs(
          CANDIDATE$inner_auc -
            BEST_MODEL$inner_auc
        ) <=
          1e-12
      )


      better_se <- (
        CANDIDATE$inner_se <
          BEST_MODEL$inner_se -
          1e-12
      )


      tied_se <- (
        abs(
          CANDIDATE$inner_se -
            BEST_MODEL$inner_se
        ) <=
          1e-12
      )


      lower_alpha <- (
        CANDIDATE$alpha <
          BEST_MODEL$alpha
      )


      if (
        better_auc ||
        (
          tied_auc &&
          better_se
        ) ||
        (
          tied_auc &&
          tied_se &&
          lower_alpha
        )
      ) {
        BEST_MODEL <- CANDIDATE
      }
    }


    rm(CV_FIT)
    invisible(gc())
  }


  if (is.null(
    BEST_MODEL
  )) {
    stop(
      paste0(
        "No valid inner model at R",
        current_repeat,
        " F",
        current_fold
      )
    )
  }


  # ----------------------------------------------------------
  # FINAL MODEL FOR THIS OUTER FOLD
  # ----------------------------------------------------------

  FINAL_FIT <- glmnet::glmnet(

    x =
      X_MODEL[
        TRAIN_INDEX,
        ,
        drop = FALSE
      ],

    y =
      Y[
        TRAIN_INDEX
      ],

    family =
      "binomial",

    weights =
      rep(
        1,
        length(TRAIN_INDEX)
      ),

    alpha =
      BEST_MODEL$alpha,

    lambda =
      BEST_MODEL$lambda,

    penalty.factor =
      PENALTY_FACTOR,

    standardize =
      TRUE,

    intercept =
      TRUE
  )


  TEST_PROB <- as.numeric(
    predict(
      FINAL_FIT,
      newx =
        X_MODEL[
          TEST_INDEX,
          ,
          drop = FALSE
        ],
      type =
        "response"
    )
  )


  if (
    length(TEST_PROB) !=
      length(TEST_INDEX) ||
    any(!is.finite(TEST_PROB))
  ) {
    stop(
      "Invalid outer predictions."
    )
  }


  OUTER_AUC <- auc_rank(
    Y[
      TEST_INDEX
    ],
    TEST_PROB
  )


  # ----------------------------------------------------------
  # NONZERO GENE COUNT
  # ----------------------------------------------------------

  COEF_MATRIX <- as.matrix(
    coef(
      FINAL_FIT
    )
  )


  COEF_VECTOR <- as.numeric(
    COEF_MATRIX[
      ,
      1
    ]
  )


  # coefficient vector:
  # 1 = intercept
  # 2...(N_COV+1) = covariates
  # (N_COV+2)...end = genes

  GENE_START <- N_COV + 2L


  GENE_COEFS <- COEF_VECTOR[
    GENE_START:
      length(COEF_VECTOR)
  ]


  N_NONZERO_GENES <- sum(
    abs(
      GENE_COEFS
    ) >
      ZERO_TOL
  )


  # ----------------------------------------------------------
  # SAVE CHECKPOINT
  # ----------------------------------------------------------

  RESULT_OBJECT <- list(

    model =
      "A_DLPFC_only",

    repeat_id =
      current_repeat,

    fold_id =
      current_fold,

    train_n =
      length(TRAIN_INDEX),

    test_n =
      length(TEST_INDEX),

    train_donors =
      length(
        unique(
          TRAIN_DONORS
        )
      ),

    test_donors =
      length(
        unique(
          TEST_DONORS
        )
      ),

    alpha =
      BEST_MODEL$alpha,

    lambda =
      BEST_MODEL$lambda,

    inner_auc =
      BEST_MODEL$inner_auc,

    inner_se =
      BEST_MODEL$inner_se,

    outer_auc =
      OUTER_AUC,

    n_gene_nonzero =
      N_NONZERO_GENES,

    test_donor_ids =
      TEST_DONORS,

    test_y =
      Y[
        TEST_INDEX
      ],

    test_probability =
      TEST_PROB
  )


  saveRDS(
    RESULT_OBJECT,
    CHECKPOINT_PATH,
    compress = "gzip"
  )


  cat(
    sprintf(
      paste0(
        "[%02d/25] R%d F%d | ",
        "train=%d test=%d | ",
        "alpha=%.2f inner=%.4f outer=%.4f genes=%d\n"
      ),
      eval_index,
      current_repeat,
      current_fold,
      length(TRAIN_INDEX),
      length(TEST_INDEX),
      BEST_MODEL$alpha,
      BEST_MODEL$inner_auc,
      OUTER_AUC,
      N_NONZERO_GENES
    )
  )


  rm(
    FINAL_FIT,
    COEF_MATRIX,
    COEF_VECTOR,
    GENE_COEFS,
    RESULT_OBJECT
  )

  invisible(gc())
}


# ------------------------------------------------------------
# 17. LOAD COMPLETED FITS
# ------------------------------------------------------------

FIT_FILES <- sort(
  list.files(
    FIT_DIR,
    pattern =
      "^repeat_[0-9]+_fold_[0-9]+\\.rds$",
    full.names =
      TRUE
  )
)


if (
  length(FIT_FILES) != 25L
) {
  stop(
    paste0(
      "Expected 25 fit files; found ",
      length(FIT_FILES)
    )
  )
}


RESULT_LIST <- lapply(
  FIT_FILES,
  readRDS
)


# ------------------------------------------------------------
# 18. PER-FOLD TABLE
# ------------------------------------------------------------

PER_FOLD <- do.call(
  rbind,
  lapply(
    RESULT_LIST,
    function(z) {

      data.frame(

        model =
          z$model,

        repeat_id =
          z$repeat_id,

        fold_id =
          z$fold_id,

        train_n =
          z$train_n,

        test_n =
          z$test_n,

        train_donors =
          z$train_donors,

        test_donors =
          z$test_donors,

        alpha =
          z$alpha,

        lambda =
          z$lambda,

        inner_auc =
          z$inner_auc,

        inner_se =
          z$inner_se,

        outer_auc =
          z$outer_auc,

        n_gene_nonzero =
          z$n_gene_nonzero,

        stringsAsFactors =
          FALSE
      )
    }
  )
)


PER_FOLD <- PER_FOLD[
  order(
    PER_FOLD$repeat_id,
    PER_FOLD$fold_id
  ),
  ,
  drop = FALSE
]

rownames(PER_FOLD) <- NULL


# ------------------------------------------------------------
# 19. RAW REPEATED OOF PREDICTIONS
# ------------------------------------------------------------

OOF <- do.call(
  rbind,
  lapply(
    RESULT_LIST,
    function(z) {

      data.frame(

        model =
          z$model,

        repeat_id =
          z$repeat_id,

        fold_id =
          z$fold_id,

        donor_id =
          z$test_donor_ids,

        y =
          z$test_y,

        probability =
          z$test_probability,

        stringsAsFactors =
          FALSE
      )
    }
  )
)


# ------------------------------------------------------------
# 20. VERIFY OOF COVERAGE
# ------------------------------------------------------------

OOF_COUNTS <- table(
  OOF$donor_id
)

if (
  length(OOF_COUNTS) != 379L
) {
  stop(
    paste0(
      "Expected 379 DLPFC donors in OOF; found ",
      length(OOF_COUNTS)
    )
  )
}

if (
  !all(
    OOF_COUNTS == 5L
  )
) {
  stop(
    "Each DLPFC donor must have exactly 5 OOF predictions."
  )
}


# ------------------------------------------------------------
# 21. REPEATED-OOF DONOR AVERAGE
# ------------------------------------------------------------

DONOR_IDS <- sort(
  unique(
    OOF$donor_id
  )
)


DONOR_OOF <- do.call(
  rbind,
  lapply(
    DONOR_IDS,
    function(current_donor) {

      donor_rows <- OOF[
        OOF$donor_id ==
          current_donor,
        ,
        drop = FALSE
      ]


      donor_y <- unique(
        donor_rows$y
      )


      if (
        length(donor_y) != 1L
      ) {
        stop(
          paste0(
            "Diagnosis mismatch for donor ",
            current_donor
          )
        )
      }


      data.frame(

        donor_id =
          current_donor,

        y =
          donor_y,

        repeated_predictions =
          nrow(donor_rows),

        mean_oof_probability =
          mean(
            donor_rows$probability
          ),

        stringsAsFactors =
          FALSE
      )
    }
  )
)


rownames(DONOR_OOF) <- NULL


stopifnot(
  nrow(DONOR_OOF) == 379L,
  all(
    DONOR_OOF$repeated_predictions ==
      5L
  )
)


REPEATED_OOF_AUC <- auc_rank(
  DONOR_OOF$y,
  DONOR_OOF$mean_oof_probability
)


# ------------------------------------------------------------
# 22. SUMMARY
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(

    "Outer_evaluations",

    "DLPFC_donors",

    "Control_donors",

    "Schizophrenia_donors",

    "Repeated_OOF_donor_AUROC",

    "Mean_outer_fold_AUROC",

    "SD_outer_fold_AUROC",

    "Min_outer_fold_AUROC",

    "Max_outer_fold_AUROC",

    "Mean_inner_AUROC",

    "Median_inner_AUROC",

    "Mean_selected_genes",

    "Median_selected_genes"
  ),

  value = c(

    25,

    nrow(DONOR_OOF),

    sum(
      DONOR_OOF$y == 0L
    ),

    sum(
      DONOR_OOF$y == 1L
    ),

    REPEATED_OOF_AUC,

    mean(
      PER_FOLD$outer_auc
    ),

    sd(
      PER_FOLD$outer_auc
    ),

    min(
      PER_FOLD$outer_auc
    ),

    max(
      PER_FOLD$outer_auc
    ),

    mean(
      PER_FOLD$inner_auc
    ),

    median(
      PER_FOLD$inner_auc
    ),

    mean(
      PER_FOLD$n_gene_nonzero
    ),

    median(
      PER_FOLD$n_gene_nonzero
    )
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 23. SAVE FINAL OUTPUTS
# ------------------------------------------------------------

PER_FOLD_PATH <- file.path(
  OUT_DIR,
  "model_A_DLPFC_only_per_fold_v1.csv"
)

OOF_PATH <- file.path(
  OUT_DIR,
  "model_A_DLPFC_only_OOF_predictions_v1.csv"
)

DONOR_OOF_PATH <- file.path(
  OUT_DIR,
  "model_A_DLPFC_only_repeated_OOF_donor_predictions_v1.csv"
)

SUMMARY_PATH <- file.path(
  OUT_DIR,
  "model_A_DLPFC_only_summary_v1.csv"
)


write.csv(
  PER_FOLD,
  PER_FOLD_PATH,
  row.names = FALSE
)

write.csv(
  OOF,
  OOF_PATH,
  row.names = FALSE
)

write.csv(
  DONOR_OOF,
  DONOR_OOF_PATH,
  row.names = FALSE
)

write.csv(
  SUMMARY,
  SUMMARY_PATH,
  row.names = FALSE
)


# ------------------------------------------------------------
# 24. HARD CHECKS
# ------------------------------------------------------------

stopifnot(

  nrow(PER_FOLD) == 25L,

  nrow(DONOR_OOF) == 379L,

  is.finite(
    REPEATED_OOF_AUC
  ),

  REPEATED_OOF_AUC >= 0,

  REPEATED_OOF_AUC <= 1,

  all(
    PER_FOLD$outer_auc >= 0
  ),

  all(
    PER_FOLD$outer_auc <= 1
  ),

  file.exists(
    SUMMARY_PATH
  )
)


# ------------------------------------------------------------
# 25. OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START


cat("\n============================================================\n")
cat("STEP 05A.1 — MODEL A DLPFC-ONLY COMPLETE\n")
cat("============================================================\n")


print(
  SUMMARY,
  row.names = FALSE
)


cat(
  "\nAlpha selections:\n"
)

print(
  table(
    PER_FOLD$alpha
  )
)


cat(
  "\nRepeated OOF donor AUROC: ",
  round(
    REPEATED_OOF_AUC,
    4
  ),
  "\n",
  sep = ""
)


cat(
  "Mean fold AUROC: ",
  round(
    mean(
      PER_FOLD$outer_auc
    ),
    4
  ),
  " +/- ",
  round(
    sd(
      PER_FOLD$outer_auc
    ),
    4
  ),
  "\n",
  sep = ""
)


cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS / 60,
    1
  ),
  " minutes\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 05A.1 — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
glmnet version: 5.1
Counts: 23740 genes x 755 samples
Metadata: 755 samples
Locked split schema: PASS
  donor column:  BrNum
  repeat column: repeat_id
  fold column:   outer_fold
Locked outer evaluations: 25
Locked donor universe: PASS (454 donors)
DLPFC samples/donors: 379
Controls: 226
Schizophrenia: 153
Unpenalized covariates: 16
Penalized genes: 23740
Total predictors: 23756
[01/25] R1 F1 | train=302 test=77 | alpha=0.10 inner=0.7277 outer=0.8001 genes=726
[02/25] R1 F2 | train=302 test=77 | alpha=0.10 inner=0.7619 outer=0.7966 genes=1210
[03/25] R1 F3 | train=303 test=76 | alpha=0.10 inner=0.7641 outer=0.8086 genes=1236
[04/25] R1 F4 | train=303 test=76 | alpha=0.10 inner=0.7303 outer=0.7642 genes=1185
[05/25] R1 F5 | train=306 test=73 | alpha=0.10 inner=0.7598 outer=0.7571 genes=1223
[06/25] R2 F1 | train=302 test=77 | alpha=0.10 inner=0.7401 outer=0.8163 genes=1262
[07/25] R2 F2 | train=302 test=77 | alpha=0.10 inner=0.7724 outer=0.7391 genes=1218
[08/25]

In [29]:
# ============================================================
# STEP 05A.2 — BASELINE MODEL B
# HIPPO-ONLY ELASTIC-NET
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]

# ------------------------------------------------------------
# 0. SENTINEL
# ------------------------------------------------------------

SENTINEL_PATH <- file.path(PROJECT_ROOT, ".project_id")

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(SENTINEL_PATH, warn = FALSE)[1]
)

if (!identical(OBSERVED_ID, PROJECT_ID)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")

if (!requireNamespace("glmnet", quietly = TRUE)) {
  stop("glmnet package unavailable.")
}

cat(
  "glmnet version: ",
  as.character(packageVersion("glmnet")),
  "\n",
  sep = ""
)

# ------------------------------------------------------------
# 1. INPUTS
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

SPLIT_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "cv_splits",
  "outer_cv_donor_splits_5fold_5repeat_v1.csv"
)

if (!all(file.exists(c(
  COUNTS_PATH,
  META_PATH,
  SPLIT_PATH
)))) {
  stop("Required input missing.")
}

# ------------------------------------------------------------
# 2. SETTINGS
# ------------------------------------------------------------

ALPHA_GRID <- c(
  0.10,
  0.25,
  0.50,
  0.75,
  1.00
)

INNER_K <- 4L
N_LAMBDA <- 50L
BASE_SEED <- 2026100200L
ZERO_TOL <- 1e-12

# ------------------------------------------------------------
# 3. LOAD
# ------------------------------------------------------------

COUNTS <- readRDS(COUNTS_PATH)

META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

SPLITS <- read.csv(
  SPLIT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(COUNTS) == 23740L,
  ncol(COUNTS) == 755L,
  nrow(META) == 755L
)

cat(
  "Counts: ",
  nrow(COUNTS),
  " genes x ",
  ncol(COUNTS),
  " samples\n",
  sep = ""
)

# ------------------------------------------------------------
# 4. EXACT SPLIT SCHEMA
# ------------------------------------------------------------

REQUIRED_SPLIT_COLUMNS <- c(
  "BrNum",
  "outer_fold",
  "repeat_id"
)

if (!all(
  REQUIRED_SPLIT_COLUMNS %in%
    names(SPLITS)
)) {
  print(names(SPLITS))
  stop("Locked split schema mismatch.")
}

LOCKED_SPLITS <- data.frame(
  donor_id = as.character(
    SPLITS[, "BrNum", drop = TRUE]
  ),
  repeat_id = as.integer(
    SPLITS[, "repeat_id", drop = TRUE]
  ),
  fold_id = as.integer(
    SPLITS[, "outer_fold", drop = TRUE]
  ),
  stringsAsFactors = FALSE
)

if (
  anyNA(LOCKED_SPLITS$donor_id) ||
  anyNA(LOCKED_SPLITS$repeat_id) ||
  anyNA(LOCKED_SPLITS$fold_id)
) {
  stop("NA in locked split assignments.")
}

if (any(
  duplicated(
    LOCKED_SPLITS[
      ,
      c("donor_id", "repeat_id"),
      drop = FALSE
    ]
  )
)) {
  stop("Duplicate donor assignment within repeat.")
}

stopifnot(
  identical(
    sort(unique(LOCKED_SPLITS$repeat_id)),
    1:5
  ),
  identical(
    sort(unique(LOCKED_SPLITS$fold_id)),
    1:5
  )
)

COMBOS <- expand.grid(
  repeat_id = 1:5,
  fold_id = 1:5,
  KEEP.OUT.ATTRS = FALSE,
  stringsAsFactors = FALSE
)

COMBOS <- COMBOS[
  order(
    COMBOS$repeat_id,
    COMBOS$fold_id
  ),
  ,
  drop = FALSE
]

rownames(COMBOS) <- NULL

cat("Locked split schema: PASS\n")
cat("Locked outer evaluations: 25\n")

# ------------------------------------------------------------
# 5. HIPPO SUBSET
# ------------------------------------------------------------

HIPPO_INDEX <- which(
  META$Region == "HIPPO"
)

META_H <- META[
  HIPPO_INDEX,
  ,
  drop = FALSE
]

COUNTS_H <- COUNTS[
  ,
  HIPPO_INDEX,
  drop = FALSE
]

HIPPO_DONORS <- as.character(
  META_H$BrNum
)

if (anyDuplicated(HIPPO_DONORS)) {
  stop("HIPPO subset contains duplicate donors.")
}

stopifnot(
  nrow(META_H) == 376L,
  ncol(COUNTS_H) == 376L,
  length(unique(HIPPO_DONORS)) == 376L
)

cat(
  "HIPPO samples/donors: ",
  nrow(META_H),
  "\n",
  sep = ""
)

# ------------------------------------------------------------
# 6. OUTCOME
# ------------------------------------------------------------

Y <- ifelse(
  META_H$Dx == "Schizo",
  1L,
  ifelse(
    META_H$Dx == "Control",
    0L,
    NA_integer_
  )
)

if (anyNA(Y)) {
  stop("Unexpected diagnosis.")
}

stopifnot(
  sum(Y == 0L) == 243L,
  sum(Y == 1L) == 133L
)

cat(
  "Controls: ",
  sum(Y == 0L),
  "\n",
  sep = ""
)

cat(
  "Schizophrenia: ",
  sum(Y == 1L),
  "\n",
  sep = ""
)

# ------------------------------------------------------------
# 7. LOG2(CPM+1)
# ------------------------------------------------------------

LIB_SIZE <- colSums(
  COUNTS_H
)

if (
  any(!is.finite(LIB_SIZE)) ||
  any(LIB_SIZE <= 0)
) {
  stop("Invalid library sizes.")
}

LOGCPM_H <- log2(
  sweep(
    COUNTS_H,
    2,
    LIB_SIZE / 1e6,
    "/"
  ) + 1
)

X_EXPR <- t(
  LOGCPM_H
)

if (
  anyNA(X_EXPR) ||
  any(!is.finite(X_EXPR))
) {
  stop("Invalid transformed expression.")
}

rm(
  COUNTS,
  COUNTS_H,
  LOGCPM_H,
  LIB_SIZE
)

invisible(gc())

# ------------------------------------------------------------
# 8. COVARIATES
# ------------------------------------------------------------

META_H$Sex <- factor(
  META_H$Sex
)

META_H$Race <- factor(
  META_H$Race
)

COVARIATE_MATRIX <- model.matrix(
  ~ donor_age +
    Sex +
    Race +
    RIN +
    snpPC1 +
    snpPC2 +
    snpPC3 +
    snpPC4 +
    snpPC5 +
    snpPC6 +
    snpPC7 +
    snpPC8 +
    snpPC9 +
    snpPC10,
  data = META_H
)

COVARIATE_MATRIX <- COVARIATE_MATRIX[
  ,
  colnames(COVARIATE_MATRIX) != "(Intercept)",
  drop = FALSE
]

if (
  anyNA(COVARIATE_MATRIX) ||
  any(!is.finite(COVARIATE_MATRIX))
) {
  stop("Invalid covariate matrix.")
}

N_COV <- ncol(
  COVARIATE_MATRIX
)

N_GENES <- ncol(
  X_EXPR
)

X_MODEL <- cbind(
  COVARIATE_MATRIX,
  X_EXPR
)

PENALTY_FACTOR <- c(
  rep(0, N_COV),
  rep(1, N_GENES)
)

stopifnot(
  length(PENALTY_FACTOR) ==
    ncol(X_MODEL)
)

cat(
  "Unpenalized covariates: ",
  N_COV,
  "\n",
  sep = ""
)

cat(
  "Penalized genes: ",
  N_GENES,
  "\n",
  sep = ""
)

rm(
  X_EXPR,
  COVARIATE_MATRIX
)

invisible(gc())

# ------------------------------------------------------------
# 9. HELPERS
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  ok <- (
    !is.na(y) &
    is.finite(score)
  )

  y <- as.integer(y[ok])
  score <- score[ok]

  n1 <- sum(y == 1L)
  n0 <- sum(y == 0L)

  if (
    n1 == 0L ||
    n0 == 0L
  ) {
    return(NA_real_)
  }

  r <- rank(
    score,
    ties.method = "average"
  )

  (
    sum(r[y == 1L]) -
      n1 * (n1 + 1) / 2
  ) /
    (n1 * n0)
}


make_inner_foldid <- function(
  train_index,
  seed_value
) {

  set.seed(seed_value)

  y_train <- Y[
    train_index
  ]

  foldid <- rep(
    NA_integer_,
    length(train_index)
  )

  for (
    class_value in c(0L, 1L)
  ) {

    idx <- which(
      y_train == class_value
    )

    idx <- sample(
      idx,
      length(idx),
      replace = FALSE
    )

    foldid[idx] <- rep(
      1:INNER_K,
      length.out = length(idx)
    )
  }

  if (anyNA(foldid)) {
    stop("Inner fold construction failed.")
  }

  for (
    inner_fold in 1:INNER_K
  ) {

    classes_here <- unique(
      y_train[
        foldid == inner_fold
      ]
    )

    if (!all(
      c(0L, 1L) %in%
        classes_here
    )) {
      stop(
        "Inner fold lacks one class."
      )
    }
  }

  foldid
}

# ------------------------------------------------------------
# 10. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_B_HIPPO_only_v1"
)

FIT_DIR <- file.path(
  OUT_DIR,
  "outer_fits"
)

dir.create(
  FIT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

# ------------------------------------------------------------
# 11. RUN 25 OUTER EVALUATIONS
# ------------------------------------------------------------

for (
  eval_index in seq_len(
    nrow(COMBOS)
  )
) {

  current_repeat <- COMBOS$repeat_id[
    eval_index
  ]

  current_fold <- COMBOS$fold_id[
    eval_index
  ]

  CHECKPOINT_PATH <- file.path(
    FIT_DIR,
    sprintf(
      "repeat_%02d_fold_%02d.rds",
      current_repeat,
      current_fold
    )
  )

  if (file.exists(
    CHECKPOINT_PATH
  )) {

    cat(
      sprintf(
        "[%02d/25] R%d F%d already complete; skipping.\n",
        eval_index,
        current_repeat,
        current_fold
      )
    )

    next
  }

  TEST_DONORS_LOCKED <- LOCKED_SPLITS$donor_id[
    LOCKED_SPLITS$repeat_id ==
      current_repeat &
    LOCKED_SPLITS$fold_id ==
      current_fold
  ]

  TEST_INDEX <- which(
    as.character(
      META_H$BrNum
    ) %in%
      TEST_DONORS_LOCKED
  )

  TRAIN_INDEX <- setdiff(
    seq_len(nrow(META_H)),
    TEST_INDEX
  )

  TRAIN_DONORS <- as.character(
    META_H$BrNum[
      TRAIN_INDEX
    ]
  )

  TEST_DONORS <- as.character(
    META_H$BrNum[
      TEST_INDEX
    ]
  )

  if (
    length(
      intersect(
        TRAIN_DONORS,
        TEST_DONORS
      )
    ) > 0L
  ) {
    stop("Outer donor leakage.")
  }

  if (
    length(
      unique(
        Y[
          TEST_INDEX
        ]
      )
    ) != 2L
  ) {
    stop("Outer test fold lacks a class.")
  }

  INNER_FOLDID <- make_inner_foldid(
    TRAIN_INDEX,
    BASE_SEED +
      current_repeat * 100L +
      current_fold
  )

  BEST_MODEL <- NULL

  for (
    current_alpha in ALPHA_GRID
  ) {

    CV_FIT <- glmnet::cv.glmnet(
      x = X_MODEL[
        TRAIN_INDEX,
        ,
        drop = FALSE
      ],
      y = Y[
        TRAIN_INDEX
      ],
      family = "binomial",
      weights = rep(
        1,
        length(TRAIN_INDEX)
      ),
      foldid = INNER_FOLDID,
      type.measure = "auc",
      alpha = current_alpha,
      penalty.factor = PENALTY_FACTOR,
      standardize = TRUE,
      intercept = TRUE,
      nlambda = N_LAMBDA,
      parallel = FALSE
    )

    VALID_INDEX <- which(
      is.finite(
        CV_FIT$cvm
      )
    )

    if (
      length(VALID_INDEX) == 0L
    ) {
      rm(CV_FIT)
      invisible(gc())
      next
    }

    BEST_LAMBDA_INDEX <- VALID_INDEX[
      which.max(
        CV_FIT$cvm[
          VALID_INDEX
        ]
      )
    ]

    CANDIDATE <- list(
      alpha =
        current_alpha,
      lambda =
        CV_FIT$lambda[
          BEST_LAMBDA_INDEX
        ],
      inner_auc =
        CV_FIT$cvm[
          BEST_LAMBDA_INDEX
        ],
      inner_se =
        CV_FIT$cvsd[
          BEST_LAMBDA_INDEX
        ]
    )

    if (is.null(
      BEST_MODEL
    )) {

      BEST_MODEL <- CANDIDATE

    } else {

      better_auc <- (
        CANDIDATE$inner_auc >
          BEST_MODEL$inner_auc +
          1e-12
      )

      tied_auc <- (
        abs(
          CANDIDATE$inner_auc -
            BEST_MODEL$inner_auc
        ) <= 1e-12
      )

      better_se <- (
        CANDIDATE$inner_se <
          BEST_MODEL$inner_se -
          1e-12
      )

      tied_se <- (
        abs(
          CANDIDATE$inner_se -
            BEST_MODEL$inner_se
        ) <= 1e-12
      )

      lower_alpha <- (
        CANDIDATE$alpha <
          BEST_MODEL$alpha
      )

      if (
        better_auc ||
        (
          tied_auc &&
          better_se
        ) ||
        (
          tied_auc &&
          tied_se &&
          lower_alpha
        )
      ) {
        BEST_MODEL <- CANDIDATE
      }
    }

    rm(CV_FIT)
    invisible(gc())
  }

  if (is.null(
    BEST_MODEL
  )) {
    stop("No valid inner model.")
  }

  FINAL_FIT <- glmnet::glmnet(
    x = X_MODEL[
      TRAIN_INDEX,
      ,
      drop = FALSE
    ],
    y = Y[
      TRAIN_INDEX
    ],
    family = "binomial",
    weights = rep(
      1,
      length(TRAIN_INDEX)
    ),
    alpha = BEST_MODEL$alpha,
    lambda = BEST_MODEL$lambda,
    penalty.factor = PENALTY_FACTOR,
    standardize = TRUE,
    intercept = TRUE
  )

  TEST_PROB <- as.numeric(
    predict(
      FINAL_FIT,
      newx = X_MODEL[
        TEST_INDEX,
        ,
        drop = FALSE
      ],
      type = "response"
    )
  )

  OUTER_AUC <- auc_rank(
    Y[
      TEST_INDEX
    ],
    TEST_PROB
  )

  COEF_VECTOR <- as.numeric(
    as.matrix(
      coef(FINAL_FIT)
    )[, 1]
  )

  GENE_START <- N_COV + 2L

  GENE_COEFS <- COEF_VECTOR[
    GENE_START:
      length(COEF_VECTOR)
  ]

  N_NONZERO_GENES <- sum(
    abs(GENE_COEFS) >
      ZERO_TOL
  )

  RESULT_OBJECT <- list(
    model =
      "B_HIPPO_only",
    repeat_id =
      current_repeat,
    fold_id =
      current_fold,
    train_n =
      length(TRAIN_INDEX),
    test_n =
      length(TEST_INDEX),
    alpha =
      BEST_MODEL$alpha,
    lambda =
      BEST_MODEL$lambda,
    inner_auc =
      BEST_MODEL$inner_auc,
    inner_se =
      BEST_MODEL$inner_se,
    outer_auc =
      OUTER_AUC,
    n_gene_nonzero =
      N_NONZERO_GENES,
    test_donor_ids =
      TEST_DONORS,
    test_y =
      Y[
        TEST_INDEX
      ],
    test_probability =
      TEST_PROB
  )

  saveRDS(
    RESULT_OBJECT,
    CHECKPOINT_PATH,
    compress = "gzip"
  )

  cat(
    sprintf(
      paste0(
        "[%02d/25] R%d F%d | ",
        "train=%d test=%d | ",
        "alpha=%.2f inner=%.4f ",
        "outer=%.4f genes=%d\n"
      ),
      eval_index,
      current_repeat,
      current_fold,
      length(TRAIN_INDEX),
      length(TEST_INDEX),
      BEST_MODEL$alpha,
      BEST_MODEL$inner_auc,
      OUTER_AUC,
      N_NONZERO_GENES
    )
  )

  rm(
    FINAL_FIT,
    COEF_VECTOR,
    GENE_COEFS,
    RESULT_OBJECT
  )

  invisible(gc())
}

# ------------------------------------------------------------
# 12. AGGREGATE
# ------------------------------------------------------------

FIT_FILES <- sort(
  list.files(
    FIT_DIR,
    pattern =
      "^repeat_[0-9]+_fold_[0-9]+\\.rds$",
    full.names = TRUE
  )
)

if (length(
  FIT_FILES
) != 25L) {
  stop(
    paste0(
      "Expected 25 fits; found ",
      length(FIT_FILES)
    )
  )
}

RESULT_LIST <- lapply(
  FIT_FILES,
  readRDS
)

PER_FOLD <- do.call(
  rbind,
  lapply(
    RESULT_LIST,
    function(z) {

      data.frame(
        model = z$model,
        repeat_id = z$repeat_id,
        fold_id = z$fold_id,
        train_n = z$train_n,
        test_n = z$test_n,
        alpha = z$alpha,
        lambda = z$lambda,
        inner_auc = z$inner_auc,
        inner_se = z$inner_se,
        outer_auc = z$outer_auc,
        n_gene_nonzero = z$n_gene_nonzero,
        stringsAsFactors = FALSE
      )
    }
  )
)

OOF <- do.call(
  rbind,
  lapply(
    RESULT_LIST,
    function(z) {

      data.frame(
        model = z$model,
        repeat_id = z$repeat_id,
        fold_id = z$fold_id,
        donor_id = z$test_donor_ids,
        y = z$test_y,
        probability = z$test_probability,
        stringsAsFactors = FALSE
      )
    }
  )
)

OOF_COUNTS <- table(
  OOF$donor_id
)

if (
  length(OOF_COUNTS) != 376L ||
  !all(
    OOF_COUNTS == 5L
  )
) {
  stop(
    "Repeated OOF donor coverage failed."
  )
}

DONOR_IDS <- sort(
  unique(
    OOF$donor_id
  )
)

DONOR_OOF <- do.call(
  rbind,
  lapply(
    DONOR_IDS,
    function(id) {

      z <- OOF[
        OOF$donor_id == id,
        ,
        drop = FALSE
      ]

      yy <- unique(
        z$y
      )

      if (length(
        yy
      ) != 1L) {
        stop(
          "Diagnosis mismatch."
        )
      }

      data.frame(
        donor_id = id,
        y = yy,
        repeated_predictions =
          nrow(z),
        mean_oof_probability =
          mean(
            z$probability
          ),
        stringsAsFactors = FALSE
      )
    }
  )
)

REPEATED_OOF_AUC <- auc_rank(
  DONOR_OOF$y,
  DONOR_OOF$mean_oof_probability
)

SUMMARY <- data.frame(
  metric = c(
    "Outer_evaluations",
    "HIPPO_donors",
    "Control_donors",
    "Schizophrenia_donors",
    "Repeated_OOF_donor_AUROC",
    "Mean_outer_fold_AUROC",
    "SD_outer_fold_AUROC",
    "Min_outer_fold_AUROC",
    "Max_outer_fold_AUROC",
    "Mean_inner_AUROC",
    "Median_inner_AUROC",
    "Mean_selected_genes",
    "Median_selected_genes"
  ),
  value = c(
    25,
    nrow(DONOR_OOF),
    sum(DONOR_OOF$y == 0L),
    sum(DONOR_OOF$y == 1L),
    REPEATED_OOF_AUC,
    mean(PER_FOLD$outer_auc),
    sd(PER_FOLD$outer_auc),
    min(PER_FOLD$outer_auc),
    max(PER_FOLD$outer_auc),
    mean(PER_FOLD$inner_auc),
    median(PER_FOLD$inner_auc),
    mean(PER_FOLD$n_gene_nonzero),
    median(PER_FOLD$n_gene_nonzero)
  ),
  stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

write.csv(
  PER_FOLD,
  file.path(
    OUT_DIR,
    "model_B_HIPPO_only_per_fold_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  OOF,
  file.path(
    OUT_DIR,
    "model_B_HIPPO_only_OOF_predictions_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  DONOR_OOF,
  file.path(
    OUT_DIR,
    "model_B_HIPPO_only_repeated_OOF_donor_predictions_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  SUMMARY,
  file.path(
    OUT_DIR,
    "model_B_HIPPO_only_summary_v1.csv"
  ),
  row.names = FALSE
)

# ------------------------------------------------------------
# 14. OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START

cat("\n============================================================\n")
cat("STEP 05A.2 — MODEL B HIPPO-ONLY COMPLETE\n")
cat("============================================================\n")

print(
  SUMMARY,
  row.names = FALSE
)

cat(
  "\nAlpha selections:\n"
)

print(
  table(
    PER_FOLD$alpha
  )
)

cat(
  "\nRepeated OOF donor AUROC: ",
  round(
    REPEATED_OOF_AUC,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Mean fold AUROC: ",
  round(
    mean(
      PER_FOLD$outer_auc
    ),
    4
  ),
  " +/- ",
  round(
    sd(
      PER_FOLD$outer_auc
    ),
    4
  ),
  "\n",
  sep = ""
)

cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS / 60,
    1
  ),
  " minutes\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 05A.2 — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
glmnet version: 5.1
Counts: 23740 genes x 755 samples
Locked split schema: PASS
Locked outer evaluations: 25
HIPPO samples/donors: 376
Controls: 243
Schizophrenia: 133
Unpenalized covariates: 14
Penalized genes: 23740
[01/25] R1 F1 | train=299 test=77 | alpha=0.10 inner=0.7813 outer=0.8222 genes=1185
[02/25] R1 F2 | train=300 test=76 | alpha=0.10 inner=0.7527 outer=0.8035 genes=1250
[03/25] R1 F3 | train=301 test=75 | alpha=0.10 inner=0.6945 outer=0.8086 genes=1080
[04/25] R1 F4 | train=301 test=75 | alpha=0.10 inner=0.8123 outer=0.7330 genes=1200
[05/25] R1 F5 | train=303 test=73 | alpha=0.10 inner=0.7543 outer=0.8142 genes=1104
[06/25] R2 F1 | train=299 test=77 | alpha=0.10 inner=0.7742 outer=0.7356 genes=673
[07/25] R2 F2 | train=300 test=76 | alpha=0.10 inner=0.7463 outer=0.8738 genes=1197
[08/25] R2 F3 | train=301 test=75 | alpha=0.10 inner=0.7688 outer=0.7855 genes=1252
[09/25] R2 F4 | train=301 test=75 | alpha=0.10 inner=0.7938 outer=0.7685 genes=757
[10/2

In [30]:
# ============================================================
# STEP 05A.3 — MODEL C / MODEL E
# POOLED-REGION ELASTIC-NET = SHARED-ONLY MODEL
#
# In this implementation these are mathematically equivalent:
#
#   pooled model:
#     Region + covariates + X * beta
#
#   shared-only decomposition:
#     Region + covariates + X * beta_shared
#     with beta_diff fixed to zero
#
# Therefore ONE model is fitted and explicitly documented
# as representing both prespecified baselines C and E.
#
# Uses:
#   all 755 samples
#   all 454 donors
#   exact locked 5 x 5 outer donor folds
#   donor-balanced observation weights
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

TOTAL_START <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace(
  "glmnet",
  quietly = TRUE
)) {
  stop("glmnet package unavailable.")
}

cat(
  "glmnet version: ",
  as.character(
    packageVersion("glmnet")
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 2. INPUTS
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_primary_counts_v1.rds"
)

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

SPLIT_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "cv_splits",
  "outer_cv_donor_splits_5fold_5repeat_v1.csv"
)

if (!all(file.exists(
  c(
    COUNTS_PATH,
    META_PATH,
    SPLIT_PATH
  )
))) {
  stop("Required input missing.")
}


# ------------------------------------------------------------
# 3. SETTINGS
# ------------------------------------------------------------

ALPHA_GRID <- c(
  0.10,
  0.25,
  0.50,
  0.75,
  1.00
)

INNER_K <- 4L
N_LAMBDA <- 50L
BASE_SEED <- 2026100300L
ZERO_TOL <- 1e-12


# ------------------------------------------------------------
# 4. LOAD
# ------------------------------------------------------------

COUNTS <- readRDS(
  COUNTS_PATH
)

META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

SPLITS <- read.csv(
  SPLIT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


stopifnot(
  nrow(COUNTS) == 23740L,
  ncol(COUNTS) == 755L,
  nrow(META) == 755L
)


cat(
  "Counts: ",
  nrow(COUNTS),
  " genes x ",
  ncol(COUNTS),
  " samples\n",
  sep = ""
)

cat(
  "Metadata: ",
  nrow(META),
  " samples\n",
  sep = ""
)


# ------------------------------------------------------------
# 5. LOCKED SPLIT SCHEMA
# ------------------------------------------------------------

REQUIRED_SPLIT_COLUMNS <- c(
  "BrNum",
  "Dx",
  "region_availability",
  "outer_fold",
  "repeat_id"
)


if (!all(
  REQUIRED_SPLIT_COLUMNS %in%
    names(SPLITS)
)) {

  print(
    names(SPLITS)
  )

  stop(
    "Locked split schema mismatch."
  )
}


LOCKED_SPLITS <- data.frame(

  donor_id =
    as.character(
      SPLITS[, "BrNum", drop = TRUE]
    ),

  diagnosis =
    as.character(
      SPLITS[, "Dx", drop = TRUE]
    ),

  availability =
    as.character(
      SPLITS[
        ,
        "region_availability",
        drop = TRUE
      ]
    ),

  repeat_id =
    as.integer(
      SPLITS[, "repeat_id", drop = TRUE]
    ),

  fold_id =
    as.integer(
      SPLITS[, "outer_fold", drop = TRUE]
    ),

  stringsAsFactors = FALSE
)


if (
  anyNA(LOCKED_SPLITS$donor_id) ||
  anyNA(LOCKED_SPLITS$repeat_id) ||
  anyNA(LOCKED_SPLITS$fold_id)
) {
  stop(
    "NA in locked split table."
  )
}


if (any(
  duplicated(
    LOCKED_SPLITS[
      ,
      c(
        "donor_id",
        "repeat_id"
      ),
      drop = FALSE
    ]
  )
)) {
  stop(
    "Duplicate donor assignment within repeat."
  )
}


COMBOS <- expand.grid(

  repeat_id = 1:5,

  fold_id = 1:5,

  KEEP.OUT.ATTRS = FALSE,

  stringsAsFactors = FALSE
)


COMBOS <- COMBOS[
  order(
    COMBOS$repeat_id,
    COMBOS$fold_id
  ),
  ,
  drop = FALSE
]


rownames(COMBOS) <- NULL


stopifnot(
  nrow(COMBOS) == 25L,
  length(
    unique(
      LOCKED_SPLITS$donor_id
    )
  ) == 454L
)


cat(
  "Locked split schema: PASS\n"
)

cat(
  "Locked outer evaluations: 25\n"
)

cat(
  "Locked donor universe: PASS (454 donors)\n"
)


# ------------------------------------------------------------
# 6. DONOR-LEVEL STRATIFICATION TABLE
# ------------------------------------------------------------

DONOR_META <- unique(
  LOCKED_SPLITS[
    ,
    c(
      "donor_id",
      "diagnosis",
      "availability"
    ),
    drop = FALSE
  ]
)


if (
  nrow(DONOR_META) != 454L ||
  anyDuplicated(
    DONOR_META$donor_id
  )
) {
  stop(
    "Donor-level metadata construction failed."
  )
}


DONOR_META$stratum <- paste(
  DONOR_META$diagnosis,
  DONOR_META$availability,
  sep = "__"
)


# ------------------------------------------------------------
# 7. OUTCOME
# ------------------------------------------------------------

Y <- ifelse(
  META$Dx == "Schizo",
  1L,
  ifelse(
    META$Dx == "Control",
    0L,
    NA_integer_
  )
)


if (anyNA(Y)) {
  stop(
    "Unexpected diagnosis."
  )
}


stopifnot(
  sum(Y == 0L) == 469L,
  sum(Y == 1L) == 286L
)


cat(
  "Controls samples: ",
  sum(Y == 0L),
  "\n",
  sep = ""
)

cat(
  "Schizophrenia samples: ",
  sum(Y == 1L),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 8. DONOR-BALANCED WEIGHTS
# ------------------------------------------------------------

if (!(
  "donor_balance_weight" %in%
    names(META)
)) {
  stop(
    "donor_balance_weight missing from metadata."
  )
}


OBS_WEIGHT <- as.numeric(
  META$donor_balance_weight
)


if (
  anyNA(OBS_WEIGHT) ||
  any(!is.finite(OBS_WEIGHT)) ||
  any(OBS_WEIGHT <= 0)
) {
  stop(
    "Invalid donor-balance weights."
  )
}


if (
  abs(
    sum(OBS_WEIGHT) -
      454
  ) > 1e-8
) {
  stop(
    "Donor-balanced weight sum is not 454."
  )
}


cat(
  "Donor-balanced total weight: ",
  sum(OBS_WEIGHT),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 9. LOG2(CPM + 1)
# ------------------------------------------------------------

LIB_SIZE <- colSums(
  COUNTS
)


if (
  any(!is.finite(LIB_SIZE)) ||
  any(LIB_SIZE <= 0)
) {
  stop(
    "Invalid library sizes."
  )
}


LOGCPM <- log2(
  sweep(
    COUNTS,
    2,
    LIB_SIZE / 1e6,
    "/"
  ) + 1
)


X_EXPR <- t(
  LOGCPM
)


if (
  anyNA(X_EXPR) ||
  any(!is.finite(X_EXPR))
) {
  stop(
    "Invalid transformed expression."
  )
}


rm(
  COUNTS,
  LOGCPM,
  LIB_SIZE
)

invisible(gc())


# ------------------------------------------------------------
# 10. COVARIATE MATRIX
#
# Region main effect included.
# No region-by-gene interactions.
# ------------------------------------------------------------

META$Region <- factor(
  META$Region
)

META$Sex <- factor(
  META$Sex
)

META$Race <- factor(
  META$Race
)


COVARIATE_MATRIX <- model.matrix(

  ~ Region +
    donor_age +
    Sex +
    Race +
    RIN +
    snpPC1 +
    snpPC2 +
    snpPC3 +
    snpPC4 +
    snpPC5 +
    snpPC6 +
    snpPC7 +
    snpPC8 +
    snpPC9 +
    snpPC10,

  data = META
)


COVARIATE_MATRIX <- COVARIATE_MATRIX[
  ,
  colnames(
    COVARIATE_MATRIX
  ) != "(Intercept)",
  drop = FALSE
]


if (
  anyNA(COVARIATE_MATRIX) ||
  any(!is.finite(
    COVARIATE_MATRIX
  ))
) {
  stop(
    "Invalid covariate matrix."
  )
}


N_COV <- ncol(
  COVARIATE_MATRIX
)

N_GENES <- ncol(
  X_EXPR
)


X_MODEL <- cbind(
  COVARIATE_MATRIX,
  X_EXPR
)


PENALTY_FACTOR <- c(
  rep(
    0,
    N_COV
  ),
  rep(
    1,
    N_GENES
  )
)


stopifnot(
  length(PENALTY_FACTOR) ==
    ncol(X_MODEL)
)


cat(
  "Unpenalized covariates: ",
  N_COV,
  "\n",
  sep = ""
)

cat(
  "Penalized genes: ",
  N_GENES,
  "\n",
  sep = ""
)

cat(
  "Total predictors: ",
  ncol(X_MODEL),
  "\n",
  sep = ""
)


rm(
  X_EXPR,
  COVARIATE_MATRIX
)

invisible(gc())


# ------------------------------------------------------------
# 11. AUC HELPER
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  ok <- (
    !is.na(y) &
    is.finite(score)
  )

  y <- as.integer(
    y[ok]
  )

  score <- score[
    ok
  ]


  n1 <- sum(
    y == 1L
  )

  n0 <- sum(
    y == 0L
  )


  if (
    n1 == 0L ||
    n0 == 0L
  ) {
    return(
      NA_real_
    )
  }


  r <- rank(
    score,
    ties.method = "average"
  )


  (
    sum(
      r[
        y == 1L
      ]
    ) -
      n1 *
      (
        n1 + 1
      ) / 2
  ) /
    (
      n1 *
      n0
    )
}


# ------------------------------------------------------------
# 12. INNER DONOR-GROUPED FOLDS
# ------------------------------------------------------------

make_inner_foldid <- function(
  train_sample_index,
  seed_value
) {

  set.seed(
    seed_value
  )


  train_donors <- unique(
    as.character(
      META$BrNum[
        train_sample_index
      ]
    )
  )


  donor_table <- DONOR_META[
    match(
      train_donors,
      DONOR_META$donor_id
    ),
    ,
    drop = FALSE
  ]


  if (
    anyNA(
      donor_table$donor_id
    )
  ) {
    stop(
      "Inner donor lookup failed."
    )
  }


  donor_table$inner_fold <- NA_integer_


  for (
    current_stratum in unique(
      donor_table$stratum
    )
  ) {

    idx <- which(
      donor_table$stratum ==
        current_stratum
    )


    idx <- sample(
      idx,
      length(idx),
      replace = FALSE
    )


    donor_table$inner_fold[
      idx
    ] <- rep(
      seq_len(INNER_K),
      length.out =
        length(idx)
    )
  }


  if (
    anyNA(
      donor_table$inner_fold
    )
  ) {
    stop(
      "Inner donor fold assignment failed."
    )
  }


  donor_to_fold <- setNames(
    donor_table$inner_fold,
    donor_table$donor_id
  )


  sample_foldid <- as.integer(
    donor_to_fold[
      as.character(
        META$BrNum[
          train_sample_index
        ]
      )
    ]
  )


  if (
    anyNA(sample_foldid)
  ) {
    stop(
      "Inner sample fold mapping failed."
    )
  }


  # paired samples from same donor must stay together
  check_df <- data.frame(
    donor_id =
      as.character(
        META$BrNum[
          train_sample_index
        ]
      ),
    fold_id =
      sample_foldid,
    stringsAsFactors = FALSE
  )


  donor_fold_count <- tapply(
    check_df$fold_id,
    check_df$donor_id,
    function(x) {
      length(unique(x))
    }
  )


  if (
    any(
      donor_fold_count != 1L
    )
  ) {
    stop(
      "Inner donor leakage detected."
    )
  }


  sample_foldid
}


# ------------------------------------------------------------
# 13. DONOR-LEVEL PREDICTION HELPER
# ------------------------------------------------------------

collapse_to_donor <- function(
  sample_index,
  probability
) {

  donor_ids <- as.character(
    META$BrNum[
      sample_index
    ]
  )


  y_values <- Y[
    sample_index
  ]


  donor_list <- split(
    seq_along(
      sample_index
    ),
    donor_ids
  )


  out <- lapply(
    names(donor_list),
    function(id) {

      local_index <- donor_list[
        id
      ]

      local_index <- unlist(
        local_index,
        use.names = FALSE
      )


      yy <- unique(
        y_values[
          local_index
        ]
      )


      if (
        length(yy) != 1L
      ) {
        stop(
          "Diagnosis mismatch within donor."
        )
      }


      data.frame(
        donor_id = id,
        y = yy,
        probability =
          mean(
            probability[
              local_index
            ]
          ),
        stringsAsFactors = FALSE
      )
    }
  )


  result <- do.call(
    rbind,
    out
  )

  rownames(result) <- NULL

  result
}


# ------------------------------------------------------------
# 14. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_CE_pooled_shared_only_v1"
)

FIT_DIR <- file.path(
  OUT_DIR,
  "outer_fits"
)


dir.create(
  FIT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


# ------------------------------------------------------------
# 15. RUN 25 OUTER EVALUATIONS
# ------------------------------------------------------------

for (
  eval_index in seq_len(
    nrow(COMBOS)
  )
) {

  current_repeat <- COMBOS$repeat_id[
    eval_index
  ]

  current_fold <- COMBOS$fold_id[
    eval_index
  ]


  CHECKPOINT_PATH <- file.path(
    FIT_DIR,
    sprintf(
      "repeat_%02d_fold_%02d.rds",
      current_repeat,
      current_fold
    )
  )


  if (file.exists(
    CHECKPOINT_PATH
  )) {

    cat(
      sprintf(
        "[%02d/25] R%d F%d already complete; skipping.\n",
        eval_index,
        current_repeat,
        current_fold
      )
    )

    next
  }


  TEST_DONORS <- LOCKED_SPLITS$donor_id[
    LOCKED_SPLITS$repeat_id ==
      current_repeat &
    LOCKED_SPLITS$fold_id ==
      current_fold
  ]


  TEST_INDEX <- which(
    as.character(
      META$BrNum
    ) %in%
      TEST_DONORS
  )


  TRAIN_INDEX <- setdiff(
    seq_len(
      nrow(META)
    ),
    TEST_INDEX
  )


  if (
    length(
      intersect(
        META$BrNum[
          TRAIN_INDEX
        ],
        META$BrNum[
          TEST_INDEX
        ]
      )
    ) > 0L
  ) {
    stop(
      "Outer donor leakage detected."
    )
  }


  INNER_FOLDID <- make_inner_foldid(
    TRAIN_INDEX,
    BASE_SEED +
      current_repeat * 100L +
      current_fold
  )


  BEST_MODEL <- NULL


  for (
    current_alpha in ALPHA_GRID
  ) {

    CV_FIT <- glmnet::cv.glmnet(

      x =
        X_MODEL[
          TRAIN_INDEX,
          ,
          drop = FALSE
        ],

      y =
        Y[
          TRAIN_INDEX
        ],

      family =
        "binomial",

      weights =
        OBS_WEIGHT[
          TRAIN_INDEX
        ],

      foldid =
        INNER_FOLDID,

      type.measure =
        "auc",

      alpha =
        current_alpha,

      penalty.factor =
        PENALTY_FACTOR,

      standardize =
        TRUE,

      intercept =
        TRUE,

      nlambda =
        N_LAMBDA,

      parallel =
        FALSE
    )


    VALID_INDEX <- which(
      is.finite(
        CV_FIT$cvm
      )
    )


    if (
      length(
        VALID_INDEX
      ) == 0L
    ) {

      rm(CV_FIT)
      invisible(gc())

      next
    }


    BEST_LAMBDA_INDEX <- VALID_INDEX[
      which.max(
        CV_FIT$cvm[
          VALID_INDEX
        ]
      )
    ]


    CANDIDATE <- list(

      alpha =
        current_alpha,

      lambda =
        CV_FIT$lambda[
          BEST_LAMBDA_INDEX
        ],

      inner_auc =
        CV_FIT$cvm[
          BEST_LAMBDA_INDEX
        ],

      inner_se =
        CV_FIT$cvsd[
          BEST_LAMBDA_INDEX
        ]
    )


    if (
      is.null(
        BEST_MODEL
      )
    ) {

      BEST_MODEL <- CANDIDATE

    } else {

      better_auc <- (
        CANDIDATE$inner_auc >
          BEST_MODEL$inner_auc +
          1e-12
      )


      tied_auc <- (
        abs(
          CANDIDATE$inner_auc -
            BEST_MODEL$inner_auc
        ) <=
          1e-12
      )


      better_se <- (
        CANDIDATE$inner_se <
          BEST_MODEL$inner_se -
          1e-12
      )


      tied_se <- (
        abs(
          CANDIDATE$inner_se -
            BEST_MODEL$inner_se
        ) <=
          1e-12
      )


      lower_alpha <- (
        CANDIDATE$alpha <
          BEST_MODEL$alpha
      )


      if (
        better_auc ||
        (
          tied_auc &&
          better_se
        ) ||
        (
          tied_auc &&
          tied_se &&
          lower_alpha
        )
      ) {
        BEST_MODEL <- CANDIDATE
      }
    }


    rm(CV_FIT)
    invisible(gc())
  }


  if (
    is.null(
      BEST_MODEL
    )
  ) {
    stop(
      "No valid inner model."
    )
  }


  FINAL_FIT <- glmnet::glmnet(

    x =
      X_MODEL[
        TRAIN_INDEX,
        ,
        drop = FALSE
      ],

    y =
      Y[
        TRAIN_INDEX
      ],

    family =
      "binomial",

    weights =
      OBS_WEIGHT[
        TRAIN_INDEX
      ],

    alpha =
      BEST_MODEL$alpha,

    lambda =
      BEST_MODEL$lambda,

    penalty.factor =
      PENALTY_FACTOR,

    standardize =
      TRUE,

    intercept =
      TRUE
  )


  TEST_PROB <- as.numeric(
    predict(
      FINAL_FIT,
      newx =
        X_MODEL[
          TEST_INDEX,
          ,
          drop = FALSE
        ],
      type =
        "response"
    )
  )


  DONOR_TEST <- collapse_to_donor(
    TEST_INDEX,
    TEST_PROB
  )


  OUTER_DONOR_AUC <- auc_rank(
    DONOR_TEST$y,
    DONOR_TEST$probability
  )


  OUTER_SAMPLE_AUC <- auc_rank(
    Y[
      TEST_INDEX
    ],
    TEST_PROB
  )


  REGION_TEST <- as.character(
    META$Region[
      TEST_INDEX
    ]
  )


  DLPFC_LOCAL <- which(
    REGION_TEST ==
      "DLPFC"
  )

  HIPPO_LOCAL <- which(
    REGION_TEST ==
      "HIPPO"
  )


  DLPFC_AUC <- auc_rank(
    Y[
      TEST_INDEX[
        DLPFC_LOCAL
      ]
    ],
    TEST_PROB[
      DLPFC_LOCAL
    ]
  )


  HIPPO_AUC <- auc_rank(
    Y[
      TEST_INDEX[
        HIPPO_LOCAL
      ]
    ],
    TEST_PROB[
      HIPPO_LOCAL
    ]
  )


  COEF_VECTOR <- as.numeric(
    as.matrix(
      coef(
        FINAL_FIT
      )
    )[, 1]
  )


  GENE_START <- N_COV + 2L


  GENE_COEFS <- COEF_VECTOR[
    GENE_START:
      length(
        COEF_VECTOR
      )
  ]


  N_NONZERO_GENES <- sum(
    abs(
      GENE_COEFS
    ) >
      ZERO_TOL
  )


  RESULT_OBJECT <- list(

    model =
      "C_pooled_equals_E_shared_only",

    repeat_id =
      current_repeat,

    fold_id =
      current_fold,

    alpha =
      BEST_MODEL$alpha,

    lambda =
      BEST_MODEL$lambda,

    inner_auc =
      BEST_MODEL$inner_auc,

    inner_se =
      BEST_MODEL$inner_se,

    outer_donor_auc =
      OUTER_DONOR_AUC,

    outer_sample_auc =
      OUTER_SAMPLE_AUC,

    outer_DLPFC_auc =
      DLPFC_AUC,

    outer_HIPPO_auc =
      HIPPO_AUC,

    n_gene_nonzero =
      N_NONZERO_GENES,

    test_sample_index =
      TEST_INDEX,

    test_sample_id =
      as.character(
        META$RNum[
          TEST_INDEX
        ]
      ),

    test_donor_id =
      as.character(
        META$BrNum[
          TEST_INDEX
        ]
      ),

    test_region =
      as.character(
        META$Region[
          TEST_INDEX
        ]
      ),

    test_y =
      Y[
        TEST_INDEX
      ],

    test_probability =
      TEST_PROB,

    donor_test =
      DONOR_TEST
  )


  saveRDS(
    RESULT_OBJECT,
    CHECKPOINT_PATH,
    compress = "gzip"
  )


  cat(
    sprintf(
      paste0(
        "[%02d/25] R%d F%d | ",
        "alpha=%.2f inner=%.4f ",
        "donor=%.4f sample=%.4f ",
        "DLPFC=%.4f HIPPO=%.4f genes=%d\n"
      ),
      eval_index,
      current_repeat,
      current_fold,
      BEST_MODEL$alpha,
      BEST_MODEL$inner_auc,
      OUTER_DONOR_AUC,
      OUTER_SAMPLE_AUC,
      DLPFC_AUC,
      HIPPO_AUC,
      N_NONZERO_GENES
    )
  )


  rm(
    FINAL_FIT,
    COEF_VECTOR,
    GENE_COEFS,
    RESULT_OBJECT
  )

  invisible(gc())
}


# ------------------------------------------------------------
# 16. AGGREGATE
# ------------------------------------------------------------

FIT_FILES <- sort(
  list.files(
    FIT_DIR,
    pattern =
      "^repeat_[0-9]+_fold_[0-9]+\\.rds$",
    full.names = TRUE
  )
)


if (
  length(FIT_FILES) != 25L
) {
  stop(
    paste0(
      "Expected 25 fits; found ",
      length(FIT_FILES)
    )
  )
}


RESULT_LIST <- lapply(
  FIT_FILES,
  readRDS
)


PER_FOLD <- do.call(
  rbind,
  lapply(
    RESULT_LIST,
    function(z) {

      data.frame(

        model =
          z$model,

        repeat_id =
          z$repeat_id,

        fold_id =
          z$fold_id,

        alpha =
          z$alpha,

        lambda =
          z$lambda,

        inner_auc =
          z$inner_auc,

        outer_donor_auc =
          z$outer_donor_auc,

        outer_sample_auc =
          z$outer_sample_auc,

        outer_DLPFC_auc =
          z$outer_DLPFC_auc,

        outer_HIPPO_auc =
          z$outer_HIPPO_auc,

        n_gene_nonzero =
          z$n_gene_nonzero,

        stringsAsFactors = FALSE
      )
    }
  )
)


# ------------------------------------------------------------
# 17. SAMPLE-LEVEL OOF
# ------------------------------------------------------------

SAMPLE_OOF <- do.call(
  rbind,
  lapply(
    RESULT_LIST,
    function(z) {

      data.frame(

        repeat_id =
          z$repeat_id,

        fold_id =
          z$fold_id,

        sample_id =
          z$test_sample_id,

        donor_id =
          z$test_donor_id,

        region =
          z$test_region,

        y =
          z$test_y,

        probability =
          z$test_probability,

        stringsAsFactors = FALSE
      )
    }
  )
)


# ------------------------------------------------------------
# 18. DONOR-LEVEL OOF
# ------------------------------------------------------------

DONOR_OOF <- do.call(
  rbind,
  lapply(
    RESULT_LIST,
    function(z) {

      data.frame(

        repeat_id =
          z$repeat_id,

        fold_id =
          z$fold_id,

        donor_id =
          z$donor_test$donor_id,

        y =
          z$donor_test$y,

        probability =
          z$donor_test$probability,

        stringsAsFactors = FALSE
      )
    }
  )
)


DONOR_COUNTS <- table(
  DONOR_OOF$donor_id
)


if (
  length(DONOR_COUNTS) != 454L ||
  !all(
    DONOR_COUNTS == 5L
  )
) {
  stop(
    "Repeated donor OOF coverage failed."
  )
}


DONOR_IDS <- sort(
  unique(
    DONOR_OOF$donor_id
  )
)


DONOR_REPEAT_AVG <- do.call(
  rbind,
  lapply(
    DONOR_IDS,
    function(id) {

      z <- DONOR_OOF[
        DONOR_OOF$donor_id == id,
        ,
        drop = FALSE
      ]


      yy <- unique(
        z$y
      )


      if (
        length(yy) != 1L
      ) {
        stop(
          "Diagnosis mismatch across repeats."
        )
      }


      data.frame(

        donor_id = id,

        y = yy,

        repeated_predictions =
          nrow(z),

        mean_oof_probability =
          mean(
            z$probability
          ),

        stringsAsFactors = FALSE
      )
    }
  )
)


REPEATED_OOF_DONOR_AUC <- auc_rank(
  DONOR_REPEAT_AVG$y,
  DONOR_REPEAT_AVG$mean_oof_probability
)


# ------------------------------------------------------------
# 19. SUMMARY
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(

    "Outer_evaluations",

    "Discovery_samples",

    "Discovery_donors",

    "Repeated_OOF_donor_AUROC",

    "Mean_outer_donor_AUROC",

    "SD_outer_donor_AUROC",

    "Mean_outer_sample_AUROC",

    "Mean_outer_DLPFC_AUROC",

    "Mean_outer_HIPPO_AUROC",

    "Mean_inner_AUROC",

    "Mean_selected_genes",

    "Median_selected_genes"
  ),

  value = c(

    25,

    755,

    454,

    REPEATED_OOF_DONOR_AUC,

    mean(
      PER_FOLD$outer_donor_auc
    ),

    sd(
      PER_FOLD$outer_donor_auc
    ),

    mean(
      PER_FOLD$outer_sample_auc
    ),

    mean(
      PER_FOLD$outer_DLPFC_auc
    ),

    mean(
      PER_FOLD$outer_HIPPO_auc
    ),

    mean(
      PER_FOLD$inner_auc
    ),

    mean(
      PER_FOLD$n_gene_nonzero
    ),

    median(
      PER_FOLD$n_gene_nonzero
    )
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 20. SAVE
# ------------------------------------------------------------

write.csv(
  PER_FOLD,
  file.path(
    OUT_DIR,
    "model_CE_per_fold_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  SAMPLE_OOF,
  file.path(
    OUT_DIR,
    "model_CE_sample_OOF_predictions_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  DONOR_OOF,
  file.path(
    OUT_DIR,
    "model_CE_donor_OOF_predictions_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  DONOR_REPEAT_AVG,
  file.path(
    OUT_DIR,
    "model_CE_repeated_OOF_donor_predictions_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  SUMMARY,
  file.path(
    OUT_DIR,
    "model_CE_summary_v1.csv"
  ),
  row.names = FALSE
)


writeLines(
  c(

    "MODEL C / MODEL E EQUIVALENCE",

    "",

    "Model C: pooled-region elastic-net without region decomposition",

    "Model E: shared-only decomposition with beta_diff fixed to zero",

    "",

    paste(
      "With the same Region main effect, covariates, donor-balanced",
      "weights, expression matrix, penalty structure, and CV splits,",
      "these formulations are mathematically identical."
    ),

    "",

    paste(
      "Therefore a single fit is reported for both models rather",
      "than generating duplicate numerical results."
    ),

    "",

    "No external data used."
  ),
  file.path(
    OUT_DIR,
    "MODEL_C_MODEL_E_EQUIVALENCE_NOTE_v1.txt"
  )
)


# ------------------------------------------------------------
# 21. FINAL CHECKS
# ------------------------------------------------------------

stopifnot(
  nrow(PER_FOLD) == 25L,
  nrow(DONOR_REPEAT_AVG) == 454L,
  is.finite(
    REPEATED_OOF_DONOR_AUC
  )
)


# ------------------------------------------------------------
# 22. OUTPUT
# ------------------------------------------------------------

TOTAL_SECONDS <- proc.time()[3] -
  TOTAL_START


cat("\n============================================================\n")
cat("STEP 05A.3 — MODEL C/E POOLED / SHARED-ONLY COMPLETE\n")
cat("============================================================\n")


print(
  SUMMARY,
  row.names = FALSE
)


cat(
  "\nAlpha selections:\n"
)

print(
  table(
    PER_FOLD$alpha
  )
)


cat(
  "\nRepeated OOF donor AUROC: ",
  round(
    REPEATED_OOF_DONOR_AUC,
    4
  ),
  "\n",
  sep = ""
)


cat(
  "Mean donor fold AUROC: ",
  round(
    mean(
      PER_FOLD$outer_donor_auc
    ),
    4
  ),
  " +/- ",
  round(
    sd(
      PER_FOLD$outer_donor_auc
    ),
    4
  ),
  "\n",
  sep = ""
)


cat(
  "\nModel C and Model E equivalence: DOCUMENTED\n"
)


cat(
  "\nRuntime: ",
  round(
    TOTAL_SECONDS / 60,
    1
  ),
  " minutes\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 05A.3 — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
glmnet version: 5.1
Counts: 23740 genes x 755 samples
Metadata: 755 samples
Locked split schema: PASS
Locked outer evaluations: 25
Locked donor universe: PASS (454 donors)
Controls samples: 469
Schizophrenia samples: 286
Donor-balanced total weight: 454
Unpenalized covariates: 17
Penalized genes: 23740
Total predictors: 23757
[01/25] R1 F1 | alpha=0.10 inner=0.7846 donor=0.7939 sample=0.7983 DLPFC=0.7917 HIPPO=0.7956 genes=1335
[02/25] R1 F2 | alpha=0.10 inner=0.7384 donor=0.7902 sample=0.7766 DLPFC=0.7749 HIPPO=0.7763 genes=1602
[03/25] R1 F3 | alpha=0.10 inner=0.7420 donor=0.8040 sample=0.8146 DLPFC=0.8115 HIPPO=0.8117 genes=1611
[04/25] R1 F4 | alpha=0.25 inner=0.7486 donor=0.7227 sample=0.7341 DLPFC=0.7470 HIPPO=0.7076 genes=848
[05/25] R1 F5 | alpha=0.10 inner=0.7475 donor=0.8053 sample=0.7987 DLPFC=0.7234 HIPPO=0.8742 genes=1537
[06/25] R2 F1 | alpha=0.10 inner=0.7574 donor=0.7831 sample=0.7620 DLPFC=0.7700 HIPPO=0.7489 genes=1593
[07/25] R2 F2 | alpha=0.10

In [31]:
# ============================================================
# STEP 05A.4 — MODEL D
# INDEPENDENT REGION-SPECIFIC MODELS
#
# Uses already-completed leakage-free OOF predictions from:
#   Model A = DLPFC-only elastic-net
#   Model B = HIPPO-only elastic-net
#
# Operational definition:
#   - paired donor: mean(DLPFC OOF prob, HIPPO OOF prob)
#   - DLPFC-only donor: DLPFC OOF prob
#   - HIPPO-only donor: HIPPO OOF prob
#
# NO additional fitting.
# NO tuning.
# NO external data.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUTS
# ------------------------------------------------------------

DLPFC_OOF_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_A_DLPFC_only_v1",
  "model_A_DLPFC_only_OOF_predictions_v1.csv"
)

HIPPO_OOF_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_B_HIPPO_only_v1",
  "model_B_HIPPO_only_OOF_predictions_v1.csv"
)

SPLIT_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "cv_splits",
  "outer_cv_donor_splits_5fold_5repeat_v1.csv"
)

REQUIRED_PATHS <- c(
  DLPFC_OOF_PATH,
  HIPPO_OOF_PATH,
  SPLIT_PATH
)

if (!all(file.exists(
  REQUIRED_PATHS
))) {

  print(
    REQUIRED_PATHS[
      !file.exists(
        REQUIRED_PATHS
      )
    ]
  )

  stop("Required Model A/B OOF file missing.")
}


# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

A <- read.csv(
  DLPFC_OOF_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

B <- read.csv(
  HIPPO_OOF_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

SPLITS <- read.csv(
  SPLIT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


REQUIRED_OOF_COLUMNS <- c(
  "repeat_id",
  "fold_id",
  "donor_id",
  "y",
  "probability"
)


if (!all(
  REQUIRED_OOF_COLUMNS %in%
    names(A)
)) {
  print(names(A))
  stop("Model A OOF schema mismatch.")
}


if (!all(
  REQUIRED_OOF_COLUMNS %in%
    names(B)
)) {
  print(names(B))
  stop("Model B OOF schema mismatch.")
}


cat(
  "DLPFC OOF rows: ",
  nrow(A),
  "\n",
  sep = ""
)

cat(
  "HIPPO OOF rows: ",
  nrow(B),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 3. HARD CHECK REGION-SPECIFIC OOF COVERAGE
# ------------------------------------------------------------

stopifnot(
  length(
    unique(
      A$donor_id
    )
  ) == 379L,

  length(
    unique(
      B$donor_id
    )
  ) == 376L
)


A_COUNT <- table(
  A$donor_id
)

B_COUNT <- table(
  B$donor_id
)


if (!all(
  A_COUNT == 5L
)) {
  stop(
    "Each DLPFC donor must have 5 OOF predictions."
  )
}


if (!all(
  B_COUNT == 5L
)) {
  stop(
    "Each HIPPO donor must have 5 OOF predictions."
  )
}


if (any(
  duplicated(
    A[
      ,
      c(
        "repeat_id",
        "donor_id"
      ),
      drop = FALSE
    ]
  )
)) {
  stop(
    "Duplicate DLPFC donor-repeat prediction."
  )
}


if (any(
  duplicated(
    B[
      ,
      c(
        "repeat_id",
        "donor_id"
      ),
      drop = FALSE
    ]
  )
)) {
  stop(
    "Duplicate HIPPO donor-repeat prediction."
  )
}


cat(
  "Region-specific OOF integrity: PASS\n"
)


# ------------------------------------------------------------
# 4. RENAME REGION-SPECIFIC FIELDS
# ------------------------------------------------------------

A2 <- data.frame(

  repeat_id =
    as.integer(
      A$repeat_id
    ),

  fold_id =
    as.integer(
      A$fold_id
    ),

  donor_id =
    as.character(
      A$donor_id
    ),

  y_DLPFC =
    as.integer(
      A$y
    ),

  probability_DLPFC =
    as.numeric(
      A$probability
    ),

  stringsAsFactors = FALSE
)


B2 <- data.frame(

  repeat_id =
    as.integer(
      B$repeat_id
    ),

  fold_id =
    as.integer(
      B$fold_id
    ),

  donor_id =
    as.character(
      B$donor_id
    ),

  y_HIPPO =
    as.integer(
      B$y
    ),

  probability_HIPPO =
    as.numeric(
      B$probability
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 5. MERGE INDEPENDENT REGIONAL PREDICTIONS
# ------------------------------------------------------------

MODEL_D <- merge(

  A2,

  B2,

  by = c(
    "repeat_id",
    "fold_id",
    "donor_id"
  ),

  all = TRUE,

  sort = FALSE
)


# ------------------------------------------------------------
# 6. OUTCOME CONSISTENCY
# ------------------------------------------------------------

BOTH_OUTCOME <- (
  !is.na(
    MODEL_D$y_DLPFC
  ) &
  !is.na(
    MODEL_D$y_HIPPO
  )
)


if (
  any(
    MODEL_D$y_DLPFC[
      BOTH_OUTCOME
    ] !=
      MODEL_D$y_HIPPO[
        BOTH_OUTCOME
      ]
  )
) {
  stop(
    "Diagnosis mismatch between DLPFC and HIPPO predictions."
  )
}


MODEL_D$y <- ifelse(

  !is.na(
    MODEL_D$y_DLPFC
  ),

  MODEL_D$y_DLPFC,

  MODEL_D$y_HIPPO
)


MODEL_D$y <- as.integer(
  MODEL_D$y
)


if (anyNA(
  MODEL_D$y
)) {
  stop(
    "Missing donor diagnosis after merge."
  )
}


# ------------------------------------------------------------
# 7. REGION AVAILABILITY
# ------------------------------------------------------------

MODEL_D$n_regions_predicted <- rowSums(
  cbind(
    !is.na(
      MODEL_D$probability_DLPFC
    ),
    !is.na(
      MODEL_D$probability_HIPPO
    )
  )
)


MODEL_D$region_prediction_type <- ifelse(

  MODEL_D$n_regions_predicted == 2L,

  "paired_DLPFC_HIPPO",

  ifelse(
    !is.na(
      MODEL_D$probability_DLPFC
    ),
    "DLPFC_only",
    "HIPPO_only"
  )
)


if (
  any(
    MODEL_D$n_regions_predicted <
      1L
  )
) {
  stop(
    "Donor without regional prediction."
  )
}


# ------------------------------------------------------------
# 8. MODEL D DONOR SCORE
#
# Equal-weight average when two regional predictions exist.
# ------------------------------------------------------------

MODEL_D$probability <- rowMeans(
  cbind(
    MODEL_D$probability_DLPFC,
    MODEL_D$probability_HIPPO
  ),
  na.rm = TRUE
)


if (
  any(
    !is.finite(
      MODEL_D$probability
    )
  )
) {
  stop(
    "Invalid Model D probability."
  )
}


# ------------------------------------------------------------
# 9. VERIFY LOCKED FOLD ASSIGNMENT
# ------------------------------------------------------------

LOCKED <- data.frame(

  donor_id =
    as.character(
      SPLITS[
        ,
        "BrNum",
        drop = TRUE
      ]
    ),

  repeat_id =
    as.integer(
      SPLITS[
        ,
        "repeat_id",
        drop = TRUE
      ]
    ),

  fold_id =
    as.integer(
      SPLITS[
        ,
        "outer_fold",
        drop = TRUE
      ]
    ),

  stringsAsFactors = FALSE
)


LOCKED <- LOCKED[
  ,
  c(
    "repeat_id",
    "fold_id",
    "donor_id"
  ),
  drop = FALSE
]


LOCKED <- LOCKED[
  order(
    LOCKED$repeat_id,
    LOCKED$fold_id,
    LOCKED$donor_id
  ),
  ,
  drop = FALSE
]


CHECK_D <- MODEL_D[
  ,
  c(
    "repeat_id",
    "fold_id",
    "donor_id"
  ),
  drop = FALSE
]


CHECK_D <- CHECK_D[
  order(
    CHECK_D$repeat_id,
    CHECK_D$fold_id,
    CHECK_D$donor_id
  ),
  ,
  drop = FALSE
]


rownames(LOCKED) <- NULL
rownames(CHECK_D) <- NULL


if (!identical(
  LOCKED,
  CHECK_D
)) {
  stop(
    "Model D OOF rows do not exactly match locked donor folds."
  )
}


cat(
  "Locked outer-fold alignment: PASS\n"
)


# ------------------------------------------------------------
# 10. EXPECTED AVAILABILITY COUNTS
# ------------------------------------------------------------

ONE_REPEAT <- MODEL_D[
  MODEL_D$repeat_id == 1L,
  ,
  drop = FALSE
]


AVAILABILITY_COUNTS <- table(
  ONE_REPEAT$region_prediction_type
)


print(
  AVAILABILITY_COUNTS
)


EXPECTED_COUNTS <- c(

  DLPFC_only = 78L,

  HIPPO_only = 75L,

  paired_DLPFC_HIPPO = 301L
)


OBSERVED_COUNTS <- AVAILABILITY_COUNTS[
  names(
    EXPECTED_COUNTS
  )
]


if (
  anyNA(
    OBSERVED_COUNTS
  ) ||
  !identical(
    as.integer(
      OBSERVED_COUNTS
    ),
    as.integer(
      EXPECTED_COUNTS
    )
  )
) {
  stop(
    "Regional availability counts differ from expected 78/75/301."
  )
}


cat(
  "Regional availability audit: PASS\n"
)


# ------------------------------------------------------------
# 11. AUC FUNCTION
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  ok <- (
    !is.na(y) &
    is.finite(score)
  )

  y <- as.integer(
    y[
      ok
    ]
  )

  score <- score[
    ok
  ]


  n_case <- sum(
    y == 1L
  )

  n_control <- sum(
    y == 0L
  )


  if (
    n_case == 0L ||
    n_control == 0L
  ) {
    return(
      NA_real_
    )
  }


  r <- rank(
    score,
    ties.method = "average"
  )


  (
    sum(
      r[
        y == 1L
      ]
    ) -
      n_case *
      (
        n_case + 1
      ) / 2
  ) /
    (
      n_case *
      n_control
    )
}


# ------------------------------------------------------------
# 12. PER-FOLD PERFORMANCE
# ------------------------------------------------------------

COMBOS <- unique(
  MODEL_D[
    ,
    c(
      "repeat_id",
      "fold_id"
    ),
    drop = FALSE
  ]
)


COMBOS <- COMBOS[
  order(
    COMBOS$repeat_id,
    COMBOS$fold_id
  ),
  ,
  drop = FALSE
]


PER_FOLD <- do.call(
  rbind,
  lapply(
    seq_len(
      nrow(COMBOS)
    ),
    function(i) {

      current_repeat <- COMBOS$repeat_id[
        i
      ]

      current_fold <- COMBOS$fold_id[
        i
      ]


      z <- MODEL_D[
        MODEL_D$repeat_id ==
          current_repeat &
        MODEL_D$fold_id ==
          current_fold,
        ,
        drop = FALSE
      ]


      data.frame(

        model =
          "D_independent_region_models",

        repeat_id =
          current_repeat,

        fold_id =
          current_fold,

        test_donors =
          nrow(z),

        paired_donors =
          sum(
            z$n_regions_predicted ==
              2L
          ),

        outer_donor_auc =
          auc_rank(
            z$y,
            z$probability
          ),

        stringsAsFactors = FALSE
      )
    }
  )
)


if (
  nrow(
    PER_FOLD
  ) != 25L
) {
  stop(
    "Expected 25 Model D outer evaluations."
  )
}


# ------------------------------------------------------------
# 13. VERIFY REPEATED OOF COVERAGE
# ------------------------------------------------------------

DONOR_COUNTS <- table(
  MODEL_D$donor_id
)


if (
  length(
    DONOR_COUNTS
  ) != 454L
) {
  stop(
    paste0(
      "Expected 454 donors; found ",
      length(
        DONOR_COUNTS
      )
    )
  )
}


if (
  !all(
    DONOR_COUNTS == 5L
  )
) {
  stop(
    "Each donor must have exactly 5 Model D OOF predictions."
  )
}


# ------------------------------------------------------------
# 14. REPEATED OOF DONOR AVERAGE
# ------------------------------------------------------------

DONOR_IDS <- sort(
  unique(
    MODEL_D$donor_id
  )
)


DONOR_REPEAT_AVG <- do.call(
  rbind,
  lapply(
    DONOR_IDS,
    function(id) {

      z <- MODEL_D[
        MODEL_D$donor_id ==
          id,
        ,
        drop = FALSE
      ]


      yy <- unique(
        z$y
      )


      if (
        length(
          yy
        ) != 1L
      ) {
        stop(
          "Diagnosis mismatch across repeats."
        )
      }


      data.frame(

        donor_id =
          id,

        y =
          yy,

        repeated_predictions =
          nrow(z),

        mean_oof_probability =
          mean(
            z$probability
          ),

        stringsAsFactors = FALSE
      )
    }
  )
)


REPEATED_OOF_AUC <- auc_rank(
  DONOR_REPEAT_AVG$y,
  DONOR_REPEAT_AVG$mean_oof_probability
)


# ------------------------------------------------------------
# 15. SUMMARY
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(

    "Outer_evaluations",

    "Discovery_donors",

    "Paired_donors",

    "DLPFC_only_donors",

    "HIPPO_only_donors",

    "Repeated_OOF_donor_AUROC",

    "Mean_outer_donor_AUROC",

    "SD_outer_donor_AUROC",

    "Min_outer_donor_AUROC",

    "Max_outer_donor_AUROC"
  ),

  value = c(

    25,

    454,

    301,

    78,

    75,

    REPEATED_OOF_AUC,

    mean(
      PER_FOLD$outer_donor_auc
    ),

    sd(
      PER_FOLD$outer_donor_auc
    ),

    min(
      PER_FOLD$outer_donor_auc
    ),

    max(
      PER_FOLD$outer_donor_auc
    )
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 16. SAVE
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_D_independent_region_models_v1"
)


dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


write.csv(
  MODEL_D,
  file.path(
    OUT_DIR,
    "model_D_independent_region_OOF_predictions_v1.csv"
  ),
  row.names = FALSE
)


write.csv(
  PER_FOLD,
  file.path(
    OUT_DIR,
    "model_D_independent_region_per_fold_v1.csv"
  ),
  row.names = FALSE
)


write.csv(
  DONOR_REPEAT_AVG,
  file.path(
    OUT_DIR,
    "model_D_independent_region_repeated_OOF_donor_predictions_v1.csv"
  ),
  row.names = FALSE
)


write.csv(
  SUMMARY,
  file.path(
    OUT_DIR,
    "model_D_independent_region_summary_v1.csv"
  ),
  row.names = FALSE
)


writeLines(
  c(

    "MODEL D — INDEPENDENT REGION-SPECIFIC MODELS",

    "",

    paste(
      "DLPFC and HIPPO elastic-net models were fitted independently",
      "using the locked donor-aware outer folds."
    ),

    "",

    paste(
      "For donors with both regions, the two leakage-free OOF",
      "probabilities were averaged with equal weight."
    ),

    "",

    paste(
      "For donors with only one region, the available regional",
      "OOF probability was used."
    ),

    "",

    "No additional model fitting or outcome-dependent tuning was performed."
  ),
  file.path(
    OUT_DIR,
    "MODEL_D_OPERATIONAL_DEFINITION_v1.txt"
  )
)


# ------------------------------------------------------------
# 17. FINAL OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME


cat("\n============================================================\n")
cat("STEP 05A.4 — MODEL D INDEPENDENT REGIONS COMPLETE\n")
cat("============================================================\n")


print(
  SUMMARY,
  row.names = FALSE
)


cat(
  "\nRepeated OOF donor AUROC: ",
  round(
    REPEATED_OOF_AUC,
    4
  ),
  "\n",
  sep = ""
)


cat(
  "Mean fold donor AUROC: ",
  round(
    mean(
      PER_FOLD$outer_donor_auc
    ),
    4
  ),
  " +/- ",
  round(
    sd(
      PER_FOLD$outer_donor_auc
    ),
    4
  ),
  "\n",
  sep = ""
)


cat(
  "\nRuntime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 05A.4 — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
DLPFC OOF rows: 1895
HIPPO OOF rows: 1880
Region-specific OOF integrity: PASS
Locked outer-fold alignment: PASS

        DLPFC_only         HIPPO_only paired_DLPFC_HIPPO 
                78                 75                301 
Regional availability audit: PASS

STEP 05A.4 — MODEL D INDEPENDENT REGIONS COMPLETE
                   metric        value
        Outer_evaluations  25.00000000
         Discovery_donors 454.00000000
            Paired_donors 301.00000000
        DLPFC_only_donors  78.00000000
        HIPPO_only_donors  75.00000000
 Repeated_OOF_donor_AUROC   0.81502784
   Mean_outer_donor_AUROC   0.79837002
     SD_outer_donor_AUROC   0.03454377
    Min_outer_donor_AUROC   0.71010101
    Max_outer_donor_AUROC   0.85331384

Repeated OOF donor AUROC: 0.815
Mean fold donor AUROC: 0.7984 +/- 0.0345

Runtime: 1 seconds

STEP 05A.4 — CELL PASS


In [32]:
PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"

PRIMARY_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "nested_cv_5x5_v1"
)

cat("Directory exists:", dir.exists(PRIMARY_DIR), "\n\n")

FILES <- list.files(
  PRIMARY_DIR,
  recursive = TRUE,
  full.names = TRUE
)

print(FILES)

cat("\n================ CSV SCHEMAS ================\n")

CSV_FILES <- FILES[
  grepl("\\.csv$", FILES, ignore.case = TRUE)
]

for (f in CSV_FILES) {

  cat("\nFILE:", f, "\n")

  x <- tryCatch(
    read.csv(
      f,
      nrows = 3,
      stringsAsFactors = FALSE,
      check.names = FALSE
    ),
    error = function(e) NULL
  )

  if (!is.null(x)) {
    cat("COLUMNS:\n")
    print(names(x))
  }
}

Directory exists: TRUE 

  [1] "E:/SCZ_CROSSREGION_MSS_Q1/06_models/nested_cv_5x5_v1/ALL_25_OUTER_FOLD_SUMMARY_v1.csv"          
  [2] "E:/SCZ_CROSSREGION_MSS_Q1/06_models/nested_cv_5x5_v1/HYPERPARAMETER_SELECTION_FREQUENCY_v1.csv" 
  [3] "E:/SCZ_CROSSREGION_MSS_Q1/06_models/nested_cv_5x5_v1/NESTED_CV_FINAL_PERFORMANCE_v1.csv"        
  [4] "E:/SCZ_CROSSREGION_MSS_Q1/06_models/nested_cv_5x5_v1/OOF_DONOR_PREDICTIONS_AVERAGED_v1.csv"     
  [5] "E:/SCZ_CROSSREGION_MSS_Q1/06_models/nested_cv_5x5_v1/OOF_SAMPLE_PREDICTIONS_AVERAGED_v1.csv"    
  [6] "E:/SCZ_CROSSREGION_MSS_Q1/06_models/nested_cv_5x5_v1/repeat_01_fold_01/donor_predictions.csv"   
  [7] "E:/SCZ_CROSSREGION_MSS_Q1/06_models/nested_cv_5x5_v1/repeat_01_fold_01/fold_summary.csv"        
  [8] "E:/SCZ_CROSSREGION_MSS_Q1/06_models/nested_cv_5x5_v1/repeat_01_fold_01/gene_coefficients.rds"   
  [9] "E:/SCZ_CROSSREGION_MSS_Q1/06_models/nested_cv_5x5_v1/repeat_01_fold_01/inner_donor_fold_map.csv"
 [10] "E:/SCZ_CROSSREGION_MSS_Q1/06_mod

In [34]:
# ============================================================
# STEP 05C — FINAL MODEL COMPARISON / ABLATION
# CORRECTED PARSE-SAFE VERSION
#
# Models:
#   A   = DLPFC-only
#   B   = HIPPO-only
#   C/E = pooled = shared-only
#   D   = independent regional models
#   F   = proposed shared + deviation
#
# Comparisons:
#   F vs C/E : all 454 donors
#   F vs D   : all 454 donors
#   F vs A   : DLPFC donors only
#   F vs B   : HIPPO donors only
#
# Uncertainty:
#   1. matched outer-fold differences
#      with repeat-block bootstrap CI
#
#   2. paired donor bootstrap CI using
#      averaged leakage-free OOF predictions
#
# NO refitting.
# NO external data.
# NO change to frozen 24-gene SCZ-MSS.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUT PATHS
# ------------------------------------------------------------

A_FOLD_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_A_DLPFC_only_v1",
  "model_A_DLPFC_only_per_fold_v1.csv"
)

A_DONOR_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_A_DLPFC_only_v1",
  "model_A_DLPFC_only_repeated_OOF_donor_predictions_v1.csv"
)


B_FOLD_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_B_HIPPO_only_v1",
  "model_B_HIPPO_only_per_fold_v1.csv"
)

B_DONOR_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_B_HIPPO_only_v1",
  "model_B_HIPPO_only_repeated_OOF_donor_predictions_v1.csv"
)


CE_FOLD_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_CE_pooled_shared_only_v1",
  "model_CE_per_fold_v1.csv"
)

CE_DONOR_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_CE_pooled_shared_only_v1",
  "model_CE_repeated_OOF_donor_predictions_v1.csv"
)


D_FOLD_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_D_independent_region_models_v1",
  "model_D_independent_region_per_fold_v1.csv"
)

D_DONOR_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_D_independent_region_models_v1",
  "model_D_independent_region_repeated_OOF_donor_predictions_v1.csv"
)


F_FOLD_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "nested_cv_5x5_v1",
  "ALL_25_OUTER_FOLD_SUMMARY_v1.csv"
)

F_DONOR_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "nested_cv_5x5_v1",
  "OOF_DONOR_PREDICTIONS_AVERAGED_v1.csv"
)

F_SAMPLE_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "nested_cv_5x5_v1",
  "OOF_SAMPLE_PREDICTIONS_AVERAGED_v1.csv"
)


REQUIRED_PATHS <- c(
  A_FOLD_PATH,
  A_DONOR_PATH,
  B_FOLD_PATH,
  B_DONOR_PATH,
  CE_FOLD_PATH,
  CE_DONOR_PATH,
  D_FOLD_PATH,
  D_DONOR_PATH,
  F_FOLD_PATH,
  F_DONOR_PATH,
  F_SAMPLE_PATH
)


MISSING_PATHS <- REQUIRED_PATHS[
  !file.exists(REQUIRED_PATHS)
]


if (length(MISSING_PATHS) > 0L) {

  cat("Missing required files:\n")

  print(MISSING_PATHS)

  stop(
    "STEP 05C required input missing."
  )
}


cat(
  "Required input files: PASS\n"
)


# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

A_FOLD <- read.csv(
  A_FOLD_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

A_DONOR <- read.csv(
  A_DONOR_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


B_FOLD <- read.csv(
  B_FOLD_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

B_DONOR <- read.csv(
  B_DONOR_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


CE_FOLD <- read.csv(
  CE_FOLD_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

CE_DONOR <- read.csv(
  CE_DONOR_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


D_FOLD <- read.csv(
  D_FOLD_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

D_DONOR <- read.csv(
  D_DONOR_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


F_FOLD_RAW <- read.csv(
  F_FOLD_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

F_DONOR_RAW <- read.csv(
  F_DONOR_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

F_SAMPLE_RAW <- read.csv(
  F_SAMPLE_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


# ------------------------------------------------------------
# 3. SCHEMA CHECKS
# ------------------------------------------------------------

if (!all(
  c(
    "repeat_id",
    "fold_id",
    "outer_auc"
  ) %in% names(A_FOLD)
)) {
  print(names(A_FOLD))
  stop("Model A fold schema mismatch.")
}


if (!all(
  c(
    "repeat_id",
    "fold_id",
    "outer_auc"
  ) %in% names(B_FOLD)
)) {
  print(names(B_FOLD))
  stop("Model B fold schema mismatch.")
}


if (!all(
  c(
    "repeat_id",
    "fold_id",
    "outer_donor_auc"
  ) %in% names(CE_FOLD)
)) {
  print(names(CE_FOLD))
  stop("Model C/E fold schema mismatch.")
}


if (!all(
  c(
    "repeat_id",
    "fold_id",
    "outer_donor_auc"
  ) %in% names(D_FOLD)
)) {
  print(names(D_FOLD))
  stop("Model D fold schema mismatch.")
}


if (!all(
  c(
    "outer_repeat",
    "outer_fold",
    "outer_donor_auc",
    "outer_DLPFC_auc",
    "outer_HIPPO_auc"
  ) %in% names(F_FOLD_RAW)
)) {
  print(names(F_FOLD_RAW))
  stop("Model F fold schema mismatch.")
}


if (!all(
  c(
    "BrNum",
    "y",
    "prediction"
  ) %in% names(F_DONOR_RAW)
)) {
  print(names(F_DONOR_RAW))
  stop("Model F donor OOF schema mismatch.")
}


if (!all(
  c(
    "BrNum",
    "Region",
    "y",
    "prediction"
  ) %in% names(F_SAMPLE_RAW)
)) {
  print(names(F_SAMPLE_RAW))
  stop("Model F sample OOF schema mismatch.")
}


cat(
  "Input schemas: PASS\n"
)


# ------------------------------------------------------------
# 4. AUC FUNCTION
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  valid <- (
    !is.na(y) &
    is.finite(score)
  )


  y <- as.integer(
    y[valid]
  )


  score <- as.numeric(
    score[valid]
  )


  n_case <- sum(
    y == 1L
  )


  n_control <- sum(
    y == 0L
  )


  if (
    n_case == 0L ||
    n_control == 0L
  ) {
    return(
      NA_real_
    )
  }


  score_rank <- rank(
    score,
    ties.method = "average"
  )


  auc_value <- (
    sum(
      score_rank[
        y == 1L
      ]
    ) -
      n_case *
      (
        n_case + 1
      ) / 2
  ) /
    (
      n_case *
      n_control
    )


  as.numeric(
    auc_value
  )
}


# ------------------------------------------------------------
# 5. NORMALIZE MODEL F FOLD TABLE
# ------------------------------------------------------------

F_FOLD <- data.frame(

  repeat_id =
    as.integer(
      F_FOLD_RAW$outer_repeat
    ),

  fold_id =
    as.integer(
      F_FOLD_RAW$outer_fold
    ),

  donor_auc =
    as.numeric(
      F_FOLD_RAW$outer_donor_auc
    ),

  DLPFC_auc =
    as.numeric(
      F_FOLD_RAW$outer_DLPFC_auc
    ),

  HIPPO_auc =
    as.numeric(
      F_FOLD_RAW$outer_HIPPO_auc
    ),

  stringsAsFactors = FALSE
)


if (
  nrow(F_FOLD) != 25L
) {
  stop(
    "Model F must contain 25 outer evaluations."
  )
}


F_FOLD_KEY <- paste(
  F_FOLD$repeat_id,
  F_FOLD$fold_id,
  sep = "_"
)


if (
  anyDuplicated(
    F_FOLD_KEY
  )
) {
  stop(
    "Duplicate Model F outer fold key."
  )
}


cat(
  "Model F 25-fold summary: PASS\n"
)


# ------------------------------------------------------------
# 6. NORMALIZE AVERAGED OOF TABLES
# ------------------------------------------------------------

A_AVG <- data.frame(

  donor_id =
    as.character(
      A_DONOR$donor_id
    ),

  y =
    as.integer(
      A_DONOR$y
    ),

  prediction =
    as.numeric(
      A_DONOR$mean_oof_probability
    ),

  stringsAsFactors = FALSE
)


B_AVG <- data.frame(

  donor_id =
    as.character(
      B_DONOR$donor_id
    ),

  y =
    as.integer(
      B_DONOR$y
    ),

  prediction =
    as.numeric(
      B_DONOR$mean_oof_probability
    ),

  stringsAsFactors = FALSE
)


CE_AVG <- data.frame(

  donor_id =
    as.character(
      CE_DONOR$donor_id
    ),

  y =
    as.integer(
      CE_DONOR$y
    ),

  prediction =
    as.numeric(
      CE_DONOR$mean_oof_probability
    ),

  stringsAsFactors = FALSE
)


D_AVG <- data.frame(

  donor_id =
    as.character(
      D_DONOR$donor_id
    ),

  y =
    as.integer(
      D_DONOR$y
    ),

  prediction =
    as.numeric(
      D_DONOR$mean_oof_probability
    ),

  stringsAsFactors = FALSE
)


F_AVG <- data.frame(

  donor_id =
    as.character(
      F_DONOR_RAW$BrNum
    ),

  y =
    as.integer(
      F_DONOR_RAW$y
    ),

  prediction =
    as.numeric(
      F_DONOR_RAW$prediction
    ),

  stringsAsFactors = FALSE
)


DLPFC_MASK <- (
  F_SAMPLE_RAW$Region ==
    "DLPFC"
)


HIPPO_MASK <- (
  F_SAMPLE_RAW$Region ==
    "HIPPO"
)


F_DLPFC_AVG <- data.frame(

  donor_id =
    as.character(
      F_SAMPLE_RAW$BrNum[
        DLPFC_MASK
      ]
    ),

  y =
    as.integer(
      F_SAMPLE_RAW$y[
        DLPFC_MASK
      ]
    ),

  prediction =
    as.numeric(
      F_SAMPLE_RAW$prediction[
        DLPFC_MASK
      ]
    ),

  stringsAsFactors = FALSE
)


F_HIPPO_AVG <- data.frame(

  donor_id =
    as.character(
      F_SAMPLE_RAW$BrNum[
        HIPPO_MASK
      ]
    ),

  y =
    as.integer(
      F_SAMPLE_RAW$y[
        HIPPO_MASK
      ]
    ),

  prediction =
    as.numeric(
      F_SAMPLE_RAW$prediction[
        HIPPO_MASK
      ]
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 7. OOF HARD CHECKS
# ------------------------------------------------------------

stopifnot(

  nrow(A_AVG) == 379L,

  nrow(B_AVG) == 376L,

  nrow(CE_AVG) == 454L,

  nrow(D_AVG) == 454L,

  nrow(F_AVG) == 454L,

  nrow(F_DLPFC_AVG) == 379L,

  nrow(F_HIPPO_AVG) == 376L
)


if (
  anyDuplicated(
    A_AVG$donor_id
  ) ||
  anyDuplicated(
    B_AVG$donor_id
  ) ||
  anyDuplicated(
    CE_AVG$donor_id
  ) ||
  anyDuplicated(
    D_AVG$donor_id
  ) ||
  anyDuplicated(
    F_AVG$donor_id
  )
) {
  stop(
    "Duplicate donor in averaged OOF table."
  )
}


cat(
  "Averaged OOF donor integrity: PASS\n"
)


# ------------------------------------------------------------
# 8. PERFORMANCE SUMMARY
# ------------------------------------------------------------

A_AUC <- auc_rank(
  A_AVG$y,
  A_AVG$prediction
)


B_AUC <- auc_rank(
  B_AVG$y,
  B_AVG$prediction
)


CE_AUC <- auc_rank(
  CE_AVG$y,
  CE_AVG$prediction
)


D_AUC <- auc_rank(
  D_AVG$y,
  D_AVG$prediction
)


F_AUC <- auc_rank(
  F_AVG$y,
  F_AVG$prediction
)


PERFORMANCE <- data.frame(

  model = c(

    "A_DLPFC_only",

    "B_HIPPO_only",

    "C_E_pooled_shared_only",

    "D_independent_regions",

    "F_shared_plus_deviation"
  ),

  evaluation_population = c(

    "DLPFC donors",

    "HIPPO donors",

    "All discovery donors",

    "All discovery donors",

    "All discovery donors"
  ),

  n_donors = c(
    379L,
    376L,
    454L,
    454L,
    454L
  ),

  repeated_OOF_AUROC = c(
    A_AUC,
    B_AUC,
    CE_AUC,
    D_AUC,
    F_AUC
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 9. HELPER — MATCH OUTER FOLDS
# ------------------------------------------------------------

make_fold_pair <- function(
  comparison_name,
  population_name,
  comparator_repeat,
  comparator_fold,
  comparator_auc,
  reference_auc
) {

  comparator_table <- data.frame(

    repeat_id =
      as.integer(
        comparator_repeat
      ),

    fold_id =
      as.integer(
        comparator_fold
      ),

    comparator_auc =
      as.numeric(
        comparator_auc
      ),

    stringsAsFactors = FALSE
  )


  reference_table <- data.frame(

    repeat_id =
      F_FOLD$repeat_id,

    fold_id =
      F_FOLD$fold_id,

    reference_auc =
      as.numeric(
        reference_auc
      ),

    stringsAsFactors = FALSE
  )


  merged_table <- merge(

    reference_table,

    comparator_table,

    by = c(
      "repeat_id",
      "fold_id"
    ),

    all = FALSE,

    sort = TRUE
  )


  if (
    nrow(
      merged_table
    ) != 25L
  ) {
    stop(
      paste0(
        "Fold alignment failed: ",
        comparison_name
      )
    )
  }


  merged_table$comparison <- comparison_name

  merged_table$population <- population_name

  merged_table$delta_auc <- (
    merged_table$reference_auc -
      merged_table$comparator_auc
  )


  merged_table <- merged_table[
    ,
    c(
      "comparison",
      "population",
      "repeat_id",
      "fold_id",
      "reference_auc",
      "comparator_auc",
      "delta_auc"
    ),
    drop = FALSE
  ]


  merged_table
}


# ------------------------------------------------------------
# 10. FOUR MATCHED OUTER-FOLD COMPARISONS
# ------------------------------------------------------------

PAIR_CE <- make_fold_pair(

  comparison_name =
    "F_minus_C_E_pooled_shared_only",

  population_name =
    "All discovery donors",

  comparator_repeat =
    CE_FOLD$repeat_id,

  comparator_fold =
    CE_FOLD$fold_id,

  comparator_auc =
    CE_FOLD$outer_donor_auc,

  reference_auc =
    F_FOLD$donor_auc
)


PAIR_D <- make_fold_pair(

  comparison_name =
    "F_minus_D_independent_regions",

  population_name =
    "All discovery donors",

  comparator_repeat =
    D_FOLD$repeat_id,

  comparator_fold =
    D_FOLD$fold_id,

  comparator_auc =
    D_FOLD$outer_donor_auc,

  reference_auc =
    F_FOLD$donor_auc
)


PAIR_A <- make_fold_pair(

  comparison_name =
    "F_DLPFC_minus_A_DLPFC_only",

  population_name =
    "DLPFC donors only",

  comparator_repeat =
    A_FOLD$repeat_id,

  comparator_fold =
    A_FOLD$fold_id,

  comparator_auc =
    A_FOLD$outer_auc,

  reference_auc =
    F_FOLD$DLPFC_auc
)


PAIR_B <- make_fold_pair(

  comparison_name =
    "F_HIPPO_minus_B_HIPPO_only",

  population_name =
    "HIPPO donors only",

  comparator_repeat =
    B_FOLD$repeat_id,

  comparator_fold =
    B_FOLD$fold_id,

  comparator_auc =
    B_FOLD$outer_auc,

  reference_auc =
    F_FOLD$HIPPO_auc
)


FOLD_PAIRS <- rbind(
  PAIR_CE,
  PAIR_D,
  PAIR_A,
  PAIR_B
)


stopifnot(
  nrow(
    FOLD_PAIRS
  ) == 100L
)


cat(
  "Matched outer-fold comparisons: PASS\n"
)


# ------------------------------------------------------------
# 11. REPEAT-BLOCK BOOTSTRAP
#
# 25 folds are not treated as independent.
# Fold differences are averaged within each repeat first.
# Then 5 repeat blocks are resampled.
# ------------------------------------------------------------

N_BOOT <- 5000L


repeat_block_summary <- function(
  pair_table,
  seed_value
) {

  repeat_summary <- aggregate(

    delta_auc ~ repeat_id,

    data =
      pair_table,

    FUN =
      mean
  )


  if (
    nrow(
      repeat_summary
    ) != 5L
  ) {
    stop(
      "Expected five repeat-level blocks."
    )
  }


  set.seed(
    seed_value
  )


  bootstrap_delta <- numeric(
    N_BOOT
  )


  for (
    bootstrap_index in seq_len(
      N_BOOT
    )
  ) {

    sampled_rows <- sample(
      seq_len(
        nrow(
          repeat_summary
        )
      ),
      size =
        nrow(
          repeat_summary
        ),
      replace =
        TRUE
    )


    bootstrap_delta[
      bootstrap_index
    ] <- mean(
      repeat_summary$delta_auc[
        sampled_rows
      ]
    )
  }


  ci_values <- quantile(
    bootstrap_delta,
    probs = c(
      0.025,
      0.975
    ),
    na.rm = TRUE,
    names = FALSE
  )


  output <- list(

    mean_fold_delta =
      mean(
        pair_table$delta_auc
      ),

    median_fold_delta =
      median(
        pair_table$delta_auc
      ),

    mean_repeat_delta =
      mean(
        repeat_summary$delta_auc
      ),

    ci_low =
      as.numeric(
        ci_values[1]
      ),

    ci_high =
      as.numeric(
        ci_values[2]
      ),

    proportion_F_higher =
      mean(
        pair_table$delta_auc >
          0
      ),

    bootstrap_delta =
      bootstrap_delta
  )


  output
}


BLOCK_CE <- repeat_block_summary(
  PAIR_CE,
  2026100501L
)

BLOCK_D <- repeat_block_summary(
  PAIR_D,
  2026100502L
)

BLOCK_A <- repeat_block_summary(
  PAIR_A,
  2026100503L
)

BLOCK_B <- repeat_block_summary(
  PAIR_B,
  2026100504L
)


# ------------------------------------------------------------
# 12. PAIRED DONOR BOOTSTRAP
# ------------------------------------------------------------

paired_donor_bootstrap <- function(
  reference_table,
  comparator_table,
  comparison_name,
  seed_value
) {

  reference_data <- data.frame(

    donor_id =
      as.character(
        reference_table$donor_id
      ),

    y_reference =
      as.integer(
        reference_table$y
      ),

    reference_prediction =
      as.numeric(
        reference_table$prediction
      ),

    stringsAsFactors = FALSE
  )


  comparator_data <- data.frame(

    donor_id =
      as.character(
        comparator_table$donor_id
      ),

    y_comparator =
      as.integer(
        comparator_table$y
      ),

    comparator_prediction =
      as.numeric(
        comparator_table$prediction
      ),

    stringsAsFactors = FALSE
  )


  matched_data <- merge(

    reference_data,

    comparator_data,

    by = "donor_id",

    all = FALSE,

    sort = TRUE
  )


  if (
    any(
      matched_data$y_reference !=
        matched_data$y_comparator
    )
  ) {
    stop(
      paste0(
        "Diagnosis mismatch in ",
        comparison_name
      )
    )
  }


  matched_data$y <- as.integer(
    matched_data$y_reference
  )


  observed_reference_auc <- auc_rank(
    matched_data$y,
    matched_data$reference_prediction
  )


  observed_comparator_auc <- auc_rank(
    matched_data$y,
    matched_data$comparator_prediction
  )


  observed_delta_auc <- (
    observed_reference_auc -
      observed_comparator_auc
  )


  control_rows <- which(
    matched_data$y == 0L
  )


  case_rows <- which(
    matched_data$y == 1L
  )


  set.seed(
    seed_value
  )


  bootstrap_delta <- numeric(
    N_BOOT
  )


  for (
    bootstrap_index in seq_len(
      N_BOOT
    )
  ) {

    sampled_control_rows <- sample(
      control_rows,
      size =
        length(
          control_rows
        ),
      replace =
        TRUE
    )


    sampled_case_rows <- sample(
      case_rows,
      size =
        length(
          case_rows
        ),
      replace =
        TRUE
    )


    sampled_rows <- c(
      sampled_control_rows,
      sampled_case_rows
    )


    reference_auc_boot <- auc_rank(
      matched_data$y[
        sampled_rows
      ],
      matched_data$reference_prediction[
        sampled_rows
      ]
    )


    comparator_auc_boot <- auc_rank(
      matched_data$y[
        sampled_rows
      ],
      matched_data$comparator_prediction[
        sampled_rows
      ]
    )


    bootstrap_delta[
      bootstrap_index
    ] <- (
      reference_auc_boot -
        comparator_auc_boot
    )
  }


  ci_values <- quantile(
    bootstrap_delta,
    probs = c(
      0.025,
      0.975
    ),
    na.rm = TRUE,
    names = FALSE
  )


  output <- list(

    comparison =
      comparison_name,

    n_donors =
      nrow(
        matched_data
      ),

    reference_auc =
      observed_reference_auc,

    comparator_auc =
      observed_comparator_auc,

    delta_auc =
      observed_delta_auc,

    ci_low =
      as.numeric(
        ci_values[1]
      ),

    ci_high =
      as.numeric(
        ci_values[2]
      ),

    bootstrap_delta =
      bootstrap_delta
  )


  output
}


# ------------------------------------------------------------
# 13. RUN FOUR PAIRED DONOR COMPARISONS
# ------------------------------------------------------------

DONOR_CE <- paired_donor_bootstrap(

  reference_table =
    F_AVG,

  comparator_table =
    CE_AVG,

  comparison_name =
    "F_minus_C_E_pooled_shared_only",

  seed_value =
    2026100601L
)


DONOR_D <- paired_donor_bootstrap(

  reference_table =
    F_AVG,

  comparator_table =
    D_AVG,

  comparison_name =
    "F_minus_D_independent_regions",

  seed_value =
    2026100602L
)


DONOR_A <- paired_donor_bootstrap(

  reference_table =
    F_DLPFC_AVG,

  comparator_table =
    A_AVG,

  comparison_name =
    "F_DLPFC_minus_A_DLPFC_only",

  seed_value =
    2026100603L
)


DONOR_B <- paired_donor_bootstrap(

  reference_table =
    F_HIPPO_AVG,

  comparator_table =
    B_AVG,

  comparison_name =
    "F_HIPPO_minus_B_HIPPO_only",

  seed_value =
    2026100604L
)


# ------------------------------------------------------------
# 14. COMPARISON ROW HELPER
#
# Avoids any [[i]] syntax entirely.
# ------------------------------------------------------------

make_comparison_row <- function(
  comparison_name,
  population_name,
  block_result,
  donor_result
) {

  data.frame(

    comparison =
      comparison_name,

    population =
      population_name,

    n_donors =
      donor_result$n_donors,

    mean_paired_fold_delta =
      block_result$mean_fold_delta,

    repeat_block_CI_low =
      block_result$ci_low,

    repeat_block_CI_high =
      block_result$ci_high,

    proportion_outer_folds_F_higher =
      block_result$proportion_F_higher,

    averaged_OOF_reference_AUROC =
      donor_result$reference_auc,

    averaged_OOF_comparator_AUROC =
      donor_result$comparator_auc,

    averaged_OOF_delta_AUROC =
      donor_result$delta_auc,

    paired_donor_bootstrap_CI_low =
      donor_result$ci_low,

    paired_donor_bootstrap_CI_high =
      donor_result$ci_high,

    stringsAsFactors = FALSE
  )
}


# ------------------------------------------------------------
# 15. FINAL COMPARISON SUMMARY
# ------------------------------------------------------------

ROW_CE <- make_comparison_row(

  comparison_name =
    "F_minus_C_E_pooled_shared_only",

  population_name =
    "All discovery donors",

  block_result =
    BLOCK_CE,

  donor_result =
    DONOR_CE
)


ROW_D <- make_comparison_row(

  comparison_name =
    "F_minus_D_independent_regions",

  population_name =
    "All discovery donors",

  block_result =
    BLOCK_D,

  donor_result =
    DONOR_D
)


ROW_A <- make_comparison_row(

  comparison_name =
    "F_DLPFC_minus_A_DLPFC_only",

  population_name =
    "DLPFC donors only",

  block_result =
    BLOCK_A,

  donor_result =
    DONOR_A
)


ROW_B <- make_comparison_row(

  comparison_name =
    "F_HIPPO_minus_B_HIPPO_only",

  population_name =
    "HIPPO donors only",

  block_result =
    BLOCK_B,

  donor_result =
    DONOR_B
)


COMPARISON_SUMMARY <- rbind(
  ROW_CE,
  ROW_D,
  ROW_A,
  ROW_B
)


# ------------------------------------------------------------
# 16. BOOTSTRAP DISTRIBUTION TABLES
# ------------------------------------------------------------

BLOCK_BOOTSTRAPS <- data.frame(

  bootstrap_id =
    seq_len(
      N_BOOT
    ),

  F_minus_CE =
    BLOCK_CE$bootstrap_delta,

  F_minus_D =
    BLOCK_D$bootstrap_delta,

  F_DLPFC_minus_A =
    BLOCK_A$bootstrap_delta,

  F_HIPPO_minus_B =
    BLOCK_B$bootstrap_delta,

  stringsAsFactors = FALSE
)


DONOR_BOOTSTRAPS <- data.frame(

  bootstrap_id =
    seq_len(
      N_BOOT
    ),

  F_minus_CE =
    DONOR_CE$bootstrap_delta,

  F_minus_D =
    DONOR_D$bootstrap_delta,

  F_DLPFC_minus_A =
    DONOR_A$bootstrap_delta,

  F_HIPPO_minus_B =
    DONOR_B$bootstrap_delta,

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 17. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "model_comparison_v1"
)


dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


# ------------------------------------------------------------
# 18. SAVE
# ------------------------------------------------------------

write.csv(
  PERFORMANCE,
  file.path(
    OUT_DIR,
    "STEP05C_model_performance_summary_v1.csv"
  ),
  row.names = FALSE
)


write.csv(
  FOLD_PAIRS,
  file.path(
    OUT_DIR,
    "STEP05C_paired_outer_fold_differences_v1.csv"
  ),
  row.names = FALSE
)


write.csv(
  COMPARISON_SUMMARY,
  file.path(
    OUT_DIR,
    "STEP05C_paired_comparison_summary_v1.csv"
  ),
  row.names = FALSE
)


write.csv(
  BLOCK_BOOTSTRAPS,
  file.path(
    OUT_DIR,
    "STEP05C_repeat_block_bootstrap_distributions_v1.csv"
  ),
  row.names = FALSE
)


write.csv(
  DONOR_BOOTSTRAPS,
  file.path(
    OUT_DIR,
    "STEP05C_paired_donor_bootstrap_distributions_v1.csv"
  ),
  row.names = FALSE
)


# ------------------------------------------------------------
# 19. NOTES
# ------------------------------------------------------------

NOTES_PATH <- file.path(
  OUT_DIR,
  "STEP05C_MODEL_COMPARISON_NOTES_v1.txt"
)


writeLines(

  c(

    "STEP 05C MODEL COMPARISON NOTES",

    "",

    paste(
      "Model C pooled-region elastic-net and",
      "Model E shared-only are mathematically",
      "equivalent under the implemented model."
    ),

    "",

    paste(
      "DLPFC-only and HIPPO-only baselines",
      "have different eligible donor populations."
    ),

    "",

    paste(
      "They are therefore compared with Model F",
      "only within the corresponding regional",
      "donor populations."
    ),

    "",

    paste(
      "All-donor comparisons F vs C/E and F vs D",
      "use the same 454 discovery donors."
    ),

    "",

    paste(
      "Outer-fold uncertainty uses repeat-block",
      "bootstrap rather than treating 25 folds",
      "as independent observations."
    ),

    "",

    paste(
      "Paired donor bootstrap uses averaged",
      "leakage-free OOF predictions and resamples",
      "diagnosis-stratified donors jointly for",
      "both compared models."
    ),

    "",

    paste(
      "Positive delta AUROC means Model F has",
      "higher discrimination."
    ),

    paste(
      "Negative delta AUROC means the comparator",
      "has higher discrimination."
    ),

    "",

    "No external data were used.",

    "Frozen SCZ-MSS v1 was not modified."
  ),

  NOTES_PATH
)


# ------------------------------------------------------------
# 20. HARD VALIDATION
# ------------------------------------------------------------

stopifnot(

  nrow(
    PERFORMANCE
  ) == 5L,

  nrow(
    FOLD_PAIRS
  ) == 100L,

  nrow(
    COMPARISON_SUMMARY
  ) == 4L,

  nrow(
    BLOCK_BOOTSTRAPS
  ) == 5000L,

  nrow(
    DONOR_BOOTSTRAPS
  ) == 5000L,

  all(
    is.finite(
      PERFORMANCE$repeated_OOF_AUROC
    )
  ),

  all(
    is.finite(
      COMPARISON_SUMMARY$mean_paired_fold_delta
    )
  ),

  all(
    is.finite(
      COMPARISON_SUMMARY$averaged_OOF_delta_AUROC
    )
  )
)


cat(
  "Final comparison integrity: PASS\n"
)


# ------------------------------------------------------------
# 21. OUTPUT
# ------------------------------------------------------------

RUNTIME_SECONDS <- (
  proc.time()[3] -
    START_TIME
)


cat("\n============================================================\n")
cat("STEP 05C — MODEL COMPARISON / ABLATION COMPLETE\n")
cat("============================================================\n")


cat(
  "\n--- REPEATED OOF PERFORMANCE ---\n"
)


print(
  PERFORMANCE,
  row.names = FALSE,
  digits = 6
)


cat(
  "\n--- PAIRED COMPARISONS ---\n"
)


print(
  COMPARISON_SUMMARY,
  row.names = FALSE,
  digits = 6
)


cat(
  "\nInterpretation:\n"
)

cat(
  "  delta > 0 : Model F higher AUROC\n"
)

cat(
  "  delta < 0 : comparator higher AUROC\n"
)


cat(
  "\nNo naive fold-level significance test used.\n"
)


cat(
  "Frozen 24-gene SCZ-MSS changed: NO\n"
)


cat(
  "External data used: NO\n"
)


cat(
  "\nRuntime: ",
  round(
    RUNTIME_SECONDS,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 05C — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Required input files: PASS
Input schemas: PASS
Model F 25-fold summary: PASS
Averaged OOF donor integrity: PASS
Matched outer-fold comparisons: PASS
Final comparison integrity: PASS

STEP 05C — MODEL COMPARISON / ABLATION COMPLETE

--- REPEATED OOF PERFORMANCE ---
                   model evaluation_population n_donors repeated_OOF_AUROC
            A_DLPFC_only          DLPFC donors      379           0.790676
            B_HIPPO_only          HIPPO donors      376           0.805192
  C_E_pooled_shared_only  All discovery donors      454           0.798793
   D_independent_regions  All discovery donors      454           0.815028
 F_shared_plus_deviation  All discovery donors      454           0.801974

--- PAIRED COMPARISONS ---
                     comparison           population n_donors
 F_minus_C_E_pooled_shared_only All discovery donors      454
  F_minus_D_independent_regions All discovery donors      454
     F_DLPFC_minus_A_DLPFC_only    DLPFC donors 

In [35]:
# ============================================================
# STEP 05C.1 — COVARIATE-ONLY SENSITIVITY
#
# Purpose:
#   Quantify discrimination obtainable WITHOUT transcriptomic
#   gene-expression predictors.
#
# Uses:
#   same 454 discovery donors
#   same locked 5 x 5 outer donor folds
#   same donor-balanced observation weights
#   same non-expression covariates as pooled/main models
#
# No gene expression.
# No tuning.
# No external data.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

# ------------------------------------------------------------
# 0. SENTINEL
# ------------------------------------------------------------

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUTS
# ------------------------------------------------------------

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

SPLIT_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "cv_splits",
  "outer_cv_donor_splits_5fold_5repeat_v1.csv"
)

F_DONOR_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "nested_cv_5x5_v1",
  "OOF_DONOR_PREDICTIONS_AVERAGED_v1.csv"
)

REQUIRED <- c(
  META_PATH,
  SPLIT_PATH,
  F_DONOR_PATH
)

if (!all(file.exists(REQUIRED))) {
  stop("Required input missing.")
}


# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

SPLITS <- read.csv(
  SPLIT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

F_DONOR <- read.csv(
  F_DONOR_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(META) == 755L
)


# ------------------------------------------------------------
# 3. OUTCOME
# ------------------------------------------------------------

Y <- ifelse(
  META$Dx == "Schizo",
  1L,
  ifelse(
    META$Dx == "Control",
    0L,
    NA_integer_
  )
)

if (anyNA(Y)) {
  stop("Unexpected diagnosis.")
}


# ------------------------------------------------------------
# 4. DONOR-BALANCED WEIGHTS
# ------------------------------------------------------------

if (!(
  "donor_balance_weight" %in%
    names(META)
)) {
  stop("donor_balance_weight missing.")
}

OBS_WEIGHT <- as.numeric(
  META$donor_balance_weight
)

stopifnot(
  abs(
    sum(OBS_WEIGHT) - 454
  ) < 1e-8
)


# ------------------------------------------------------------
# 5. COVARIATE MATRIX
# ------------------------------------------------------------

META$Region <- factor(
  META$Region
)

META$Sex <- factor(
  META$Sex
)

META$Race <- factor(
  META$Race
)

X_COV <- model.matrix(
  ~ Region +
    donor_age +
    Sex +
    Race +
    RIN +
    snpPC1 +
    snpPC2 +
    snpPC3 +
    snpPC4 +
    snpPC5 +
    snpPC6 +
    snpPC7 +
    snpPC8 +
    snpPC9 +
    snpPC10,
  data = META
)

X_COV <- X_COV[
  ,
  colnames(X_COV) != "(Intercept)",
  drop = FALSE
]

if (
  anyNA(X_COV) ||
  any(!is.finite(X_COV))
) {
  stop("Invalid covariate matrix.")
}

cat(
  "Covariate predictors: ",
  ncol(X_COV),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 6. LOCKED OUTER SPLITS
# ------------------------------------------------------------

LOCKED <- data.frame(
  donor_id = as.character(
    SPLITS[, "BrNum", drop = TRUE]
  ),
  repeat_id = as.integer(
    SPLITS[, "repeat_id", drop = TRUE]
  ),
  fold_id = as.integer(
    SPLITS[, "outer_fold", drop = TRUE]
  ),
  stringsAsFactors = FALSE
)

COMBOS <- expand.grid(
  repeat_id = 1:5,
  fold_id = 1:5,
  KEEP.OUT.ATTRS = FALSE,
  stringsAsFactors = FALSE
)

COMBOS <- COMBOS[
  order(
    COMBOS$repeat_id,
    COMBOS$fold_id
  ),
  ,
  drop = FALSE
]


# ------------------------------------------------------------
# 7. AUC
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  ok <- (
    !is.na(y) &
    is.finite(score)
  )

  y <- as.integer(y[ok])
  score <- score[ok]

  n1 <- sum(y == 1L)
  n0 <- sum(y == 0L)

  if (
    n1 == 0L ||
    n0 == 0L
  ) {
    return(NA_real_)
  }

  r <- rank(
    score,
    ties.method = "average"
  )

  (
    sum(r[y == 1L]) -
      n1 * (n1 + 1) / 2
  ) /
    (n1 * n0)
}


# ------------------------------------------------------------
# 8. STORAGE
# ------------------------------------------------------------

OOF_LIST <- vector(
  "list",
  nrow(COMBOS)
)

FOLD_RESULTS <- data.frame(
  repeat_id = integer(0),
  fold_id = integer(0),
  donor_auc = numeric(0),
  sample_auc = numeric(0),
  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 9. 25 OUTER FITS
# ------------------------------------------------------------

for (
  i in seq_len(
    nrow(COMBOS)
  )
) {

  current_repeat <- COMBOS$repeat_id[i]
  current_fold <- COMBOS$fold_id[i]

  TEST_DONORS <- LOCKED$donor_id[
    LOCKED$repeat_id == current_repeat &
      LOCKED$fold_id == current_fold
  ]

  TEST_INDEX <- which(
    as.character(META$BrNum) %in%
      TEST_DONORS
  )

  TRAIN_INDEX <- setdiff(
    seq_len(nrow(META)),
    TEST_INDEX
  )

  TRAIN_DATA <- data.frame(
    y = Y[TRAIN_INDEX],
    X_COV[
      TRAIN_INDEX,
      ,
      drop = FALSE
    ]
  )

  FIT <- glm(
    y ~ .,
    data = TRAIN_DATA,
    family = binomial(),
    weights = OBS_WEIGHT[
      TRAIN_INDEX
    ]
  )

  TEST_DATA <- data.frame(
    X_COV[
      TEST_INDEX,
      ,
      drop = FALSE
    ]
  )

  TEST_PROB <- as.numeric(
    predict(
      FIT,
      newdata = TEST_DATA,
      type = "response"
    )
  )

  if (
    any(!is.finite(TEST_PROB))
  ) {
    stop("Invalid covariate-only prediction.")
  }

  SAMPLE_TABLE <- data.frame(
    donor_id = as.character(
      META$BrNum[
        TEST_INDEX
      ]
    ),
    y = Y[
      TEST_INDEX
    ],
    probability = TEST_PROB,
    stringsAsFactors = FALSE
  )

  DONOR_IDS <- unique(
    SAMPLE_TABLE$donor_id
  )

  DONOR_TABLE <- do.call(
    rbind,
    lapply(
      DONOR_IDS,
      function(id) {

        z <- SAMPLE_TABLE[
          SAMPLE_TABLE$donor_id == id,
          ,
          drop = FALSE
        ]

        yy <- unique(z$y)

        if (length(yy) != 1L) {
          stop("Diagnosis mismatch within donor.")
        }

        data.frame(
          repeat_id = current_repeat,
          fold_id = current_fold,
          donor_id = id,
          y = yy,
          probability = mean(
            z$probability
          ),
          stringsAsFactors = FALSE
        )
      }
    )
  )

  OOF_LIST[[i]] <- DONOR_TABLE

  FOLD_RESULTS <- rbind(
    FOLD_RESULTS,
    data.frame(
      repeat_id = current_repeat,
      fold_id = current_fold,
      donor_auc = auc_rank(
        DONOR_TABLE$y,
        DONOR_TABLE$probability
      ),
      sample_auc = auc_rank(
        SAMPLE_TABLE$y,
        SAMPLE_TABLE$probability
      ),
      stringsAsFactors = FALSE
    )
  )

  cat(
    sprintf(
      "[%02d/25] R%d F%d donor_AUC=%.4f\n",
      i,
      current_repeat,
      current_fold,
      FOLD_RESULTS$donor_auc[
        nrow(FOLD_RESULTS)
      ]
    )
  )
}


# ------------------------------------------------------------
# 10. COMBINE OOF
# ------------------------------------------------------------

OOF <- do.call(
  rbind,
  OOF_LIST
)

COUNT_BY_DONOR <- table(
  OOF$donor_id
)

stopifnot(
  length(COUNT_BY_DONOR) == 454L,
  all(COUNT_BY_DONOR == 5L)
)


# ------------------------------------------------------------
# 11. AVERAGE FIVE OOF PREDICTIONS PER DONOR
# ------------------------------------------------------------

DONOR_IDS <- sort(
  unique(
    OOF$donor_id
  )
)

DONOR_AVG <- do.call(
  rbind,
  lapply(
    DONOR_IDS,
    function(id) {

      z <- OOF[
        OOF$donor_id == id,
        ,
        drop = FALSE
      ]

      yy <- unique(z$y)

      data.frame(
        donor_id = id,
        y = yy,
        mean_oof_probability = mean(
          z$probability
        ),
        stringsAsFactors = FALSE
      )
    }
  )
)

COVARIATE_ONLY_AUC <- auc_rank(
  DONOR_AVG$y,
  DONOR_AVG$mean_oof_probability
)


# ------------------------------------------------------------
# 12. PAIRED BOOTSTRAP:
# MODEL F - COVARIATE ONLY
# ------------------------------------------------------------

F_TABLE <- data.frame(
  donor_id = as.character(
    F_DONOR$BrNum
  ),
  y = as.integer(
    F_DONOR$y
  ),
  F_prediction = as.numeric(
    F_DONOR$prediction
  ),
  stringsAsFactors = FALSE
)

COV_TABLE <- data.frame(
  donor_id = DONOR_AVG$donor_id,
  y_cov = DONOR_AVG$y,
  cov_prediction =
    DONOR_AVG$mean_oof_probability,
  stringsAsFactors = FALSE
)

MATCHED <- merge(
  F_TABLE,
  COV_TABLE,
  by = "donor_id",
  all = FALSE
)

stopifnot(
  nrow(MATCHED) == 454L,
  all(
    MATCHED$y ==
      MATCHED$y_cov
  )
)

OBSERVED_F_AUC <- auc_rank(
  MATCHED$y,
  MATCHED$F_prediction
)

OBSERVED_COV_AUC <- auc_rank(
  MATCHED$y,
  MATCHED$cov_prediction
)

OBSERVED_DELTA <- (
  OBSERVED_F_AUC -
    OBSERVED_COV_AUC
)

N_BOOT <- 5000L
set.seed(2026100701L)

CONTROL_ROWS <- which(
  MATCHED$y == 0L
)

CASE_ROWS <- which(
  MATCHED$y == 1L
)

BOOT_DELTA <- numeric(
  N_BOOT
)

for (
  b in seq_len(N_BOOT)
) {

  sampled_rows <- c(
    sample(
      CONTROL_ROWS,
      length(CONTROL_ROWS),
      replace = TRUE
    ),
    sample(
      CASE_ROWS,
      length(CASE_ROWS),
      replace = TRUE
    )
  )

  BOOT_DELTA[b] <- (
    auc_rank(
      MATCHED$y[
        sampled_rows
      ],
      MATCHED$F_prediction[
        sampled_rows
      ]
    ) -
      auc_rank(
        MATCHED$y[
          sampled_rows
        ],
        MATCHED$cov_prediction[
          sampled_rows
        ]
      )
  )
}

DELTA_CI <- quantile(
  BOOT_DELTA,
  c(
    0.025,
    0.975
  ),
  names = FALSE
)


# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_covariate_only_sensitivity_v1"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

write.csv(
  FOLD_RESULTS,
  file.path(
    OUT_DIR,
    "covariate_only_per_fold_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  OOF,
  file.path(
    OUT_DIR,
    "covariate_only_OOF_predictions_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  DONOR_AVG,
  file.path(
    OUT_DIR,
    "covariate_only_repeated_OOF_donor_predictions_v1.csv"
  ),
  row.names = FALSE
)

SUMMARY <- data.frame(
  metric = c(
    "Covariate_only_repeated_OOF_AUROC",
    "Model_F_repeated_OOF_AUROC",
    "Delta_F_minus_covariates",
    "Delta_95CI_low",
    "Delta_95CI_high",
    "Mean_fold_covariate_AUROC",
    "SD_fold_covariate_AUROC"
  ),
  value = c(
    OBSERVED_COV_AUC,
    OBSERVED_F_AUC,
    OBSERVED_DELTA,
    DELTA_CI[1],
    DELTA_CI[2],
    mean(
      FOLD_RESULTS$donor_auc
    ),
    sd(
      FOLD_RESULTS$donor_auc
    )
  ),
  stringsAsFactors = FALSE
)

write.csv(
  SUMMARY,
  file.path(
    OUT_DIR,
    "covariate_only_sensitivity_summary_v1.csv"
  ),
  row.names = FALSE
)


# ------------------------------------------------------------
# 14. OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME

cat("\n============================================================\n")
cat("STEP 05C.1 — COVARIATE-ONLY SENSITIVITY COMPLETE\n")
cat("============================================================\n")

print(
  SUMMARY,
  row.names = FALSE,
  digits = 6
)

cat(
  "\nCovariate-only AUROC: ",
  round(
    OBSERVED_COV_AUC,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Model F AUROC:       ",
  round(
    OBSERVED_F_AUC,
    4
  ),
  "\n",
  sep = ""
)

cat(
  "Delta F - covariates: ",
  round(
    OBSERVED_DELTA,
    4
  ),
  "  95% CI [",
  round(
    DELTA_CI[1],
    4
  ),
  ", ",
  round(
    DELTA_CI[2],
    4
  ),
  "]\n",
  sep = ""
)

cat(
  "\nRuntime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 05C.1 — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Covariate predictors: 17


Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


[01/25] R1 F1 donor_AUC=0.6111


Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


[02/25] R1 F2 donor_AUC=0.6111


...

[23/25] R5 F3 donor_AUC=0.5803


Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


[24/25] R5 F4 donor_AUC=0.6076


Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


[25/25] R5 F5 donor_AUC=0.6268

STEP 05C.1 — COVARIATE-ONLY SENSITIVITY COMPLETE
                            metric     value
 Covariate_only_repeated_OOF_AUROC 0.6143507
        Model_F_repeated_OOF_AUROC 0.8019743
          Delta_F_minus_covariates 0.1876237
                    Delta_95CI_low 0.1406101
                   Delta_95CI_high 0.2374156
         Mean_fold_covariate_AUROC 0.6130236
           SD_fold_covariate_AUROC 0.0444617

Covariate-only AUROC: 0.6144
Model F AUROC:       0.802
Delta F - covariates: 0.1876  95% CI [0.1406, 0.2374]

Runtime: 6.85 seconds

STEP 05C.1 — CELL PASS


In [36]:
# ============================================================
# STEP 05C.1A — VERIFY FRACTIONAL-WEIGHT WARNING IS BENIGN
#
# Scaling all prior weights by the same constant does NOT
# change weighted logistic-regression coefficient estimates.
#
# Original:
#   paired sample   = 0.5
#   unpaired sample = 1.0
#
# Rescaled:
#   paired sample   = 1
#   unpaired sample = 2
#
# Relative donor weighting is unchanged.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"

META <- read.csv(
  file.path(
    PROJECT_ROOT,
    "04_processed",
    "brainseq_model_metadata_v1.csv"
  ),
  stringsAsFactors = FALSE,
  check.names = FALSE
)

ORIGINAL_WEIGHT <- as.numeric(
  META$donor_balance_weight
)

INTEGER_WEIGHT <- 2 * ORIGINAL_WEIGHT

cat(
  "Original unique weights: ",
  paste(
    sort(unique(ORIGINAL_WEIGHT)),
    collapse = ", "
  ),
  "\n",
  sep = ""
)

cat(
  "Rescaled unique weights: ",
  paste(
    sort(unique(INTEGER_WEIGHT)),
    collapse = ", "
  ),
  "\n",
  sep = ""
)

cat(
  "Original total weight: ",
  sum(ORIGINAL_WEIGHT),
  "\n",
  sep = ""
)

cat(
  "Rescaled total weight: ",
  sum(INTEGER_WEIGHT),
  "\n",
  sep = ""
)

stopifnot(
  all(
    INTEGER_WEIGHT %in% c(1, 2)
  ),
  all(
    abs(
      INTEGER_WEIGHT /
        ORIGINAL_WEIGHT -
        2
    ) < 1e-12
  )
)

cat("\nRelative weighting preserved exactly: PASS\n")
cat("Fractional-binomial warning explanation: VERIFIED\n")
cat("STEP 05C.1A — CELL PASS\n")

Original unique weights: 0.5, 1
Rescaled unique weights: 1, 2
Original total weight: 454
Rescaled total weight: 908

Relative weighting preserved exactly: PASS
Fractional-binomial warning explanation: VERIFIED
STEP 05C.1A — CELL PASS


In [37]:
# ============================================================
# STEP 05D — FORMAL OOF RESULT FREEZE
# RETROSPECTIVE BACKFILL
#
# Purpose:
#   Consolidate all validated leakage-free discovery OOF
#   predictions into canonical sample-level and donor-level
#   artifacts.
#
# Models:
#   A   = DLPFC-only
#   B   = HIPPO-only
#   C/E = pooled/shared-only
#   D   = independent region-specific
#   F   = shared + deviation
#
# IMPORTANT CHRONOLOGY NOTE:
#   STEP 05D was not formally frozen before later stability/
#   score steps. This notebook/cell is therefore a transparent
#   retrospective backfill from already-existing discovery-only
#   OOF artifacts.
#
# No refitting.
# No external data.
# Frozen SCZ-MSS v1 is NOT modified.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop("digest package unavailable.")
}

cat(
  "digest version: ",
  as.character(
    packageVersion("digest")
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 2. INPUT PATHS
# ------------------------------------------------------------

META_PATH <- file.path(
  PROJECT_ROOT,
  "04_processed",
  "brainseq_model_metadata_v1.csv"
)

SPLIT_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "cv_splits",
  "outer_cv_donor_splits_5fold_5repeat_v1.csv"
)


A_OOF_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_A_DLPFC_only_v1",
  "model_A_DLPFC_only_OOF_predictions_v1.csv"
)


B_OOF_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_B_HIPPO_only_v1",
  "model_B_HIPPO_only_OOF_predictions_v1.csv"
)


CE_SAMPLE_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_CE_pooled_shared_only_v1",
  "model_CE_sample_OOF_predictions_v1.csv"
)

CE_DONOR_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_CE_pooled_shared_only_v1",
  "model_CE_donor_OOF_predictions_v1.csv"
)


D_DONOR_PATH <- file.path(
  PROJECT_ROOT,
  "06_models",
  "baselines",
  "model_D_independent_region_models_v1",
  "model_D_independent_region_OOF_predictions_v1.csv"
)


F_BASE_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "nested_cv_5x5_v1"
)


REPEAT_VECTOR <- rep(
  1:5,
  each = 5
)

FOLD_VECTOR <- rep(
  1:5,
  times = 5
)


F_SUBDIRS <- sprintf(
  "repeat_%02d_fold_%02d",
  REPEAT_VECTOR,
  FOLD_VECTOR
)


F_SAMPLE_FILES <- file.path(
  F_BASE_DIR,
  F_SUBDIRS,
  "sample_predictions.csv"
)

F_DONOR_FILES <- file.path(
  F_BASE_DIR,
  F_SUBDIRS,
  "donor_predictions.csv"
)


REQUIRED_FILES <- c(
  META_PATH,
  SPLIT_PATH,
  A_OOF_PATH,
  B_OOF_PATH,
  CE_SAMPLE_PATH,
  CE_DONOR_PATH,
  D_DONOR_PATH,
  F_SAMPLE_FILES,
  F_DONOR_FILES
)


MISSING_FILES <- REQUIRED_FILES[
  !file.exists(
    REQUIRED_FILES
  )
]


if (
  length(
    MISSING_FILES
  ) > 0L
) {

  cat("Missing files:\n")

  print(
    MISSING_FILES
  )

  stop(
    "Required OOF source artifact missing."
  )
}


cat(
  "Required source artifacts: PASS\n"
)


# ------------------------------------------------------------
# 3. LOAD CORE METADATA
# ------------------------------------------------------------

META <- read.csv(
  META_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

SPLITS <- read.csv(
  SPLIT_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


stopifnot(
  nrow(META) == 755L,
  length(
    unique(
      as.character(
        META$BrNum
      )
    )
  ) == 454L
)


# ------------------------------------------------------------
# 4. LOCKED FOLD TABLE
# ------------------------------------------------------------

LOCKED <- data.frame(

  donor_id =
    as.character(
      SPLITS$BrNum
    ),

  repeat_id =
    as.integer(
      SPLITS$repeat_id
    ),

  fold_id =
    as.integer(
      SPLITS$outer_fold
    ),

  stringsAsFactors = FALSE
)


if (
  nrow(LOCKED) != 2270L
) {
  stop(
    "Locked split table must contain 454 x 5 = 2270 rows."
  )
}


if (any(
  duplicated(
    LOCKED[
      ,
      c(
        "donor_id",
        "repeat_id"
      ),
      drop = FALSE
    ]
  )
)) {
  stop(
    "Duplicate donor assignment within repeat."
  )
}


LOCKED_KEY <- paste(
  LOCKED$donor_id,
  LOCKED$repeat_id,
  LOCKED$fold_id,
  sep = "|"
)


cat(
  "Locked 5x5 donor split: PASS\n"
)


# ------------------------------------------------------------
# 5. LOAD MODEL A / B
# ------------------------------------------------------------

A_RAW <- read.csv(
  A_OOF_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

B_RAW <- read.csv(
  B_OOF_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


stopifnot(
  nrow(A_RAW) == 1895L,
  nrow(B_RAW) == 1880L
)


# ------------------------------------------------------------
# 6. MAP MODEL A / B DONORS TO SAMPLE IDs
# ------------------------------------------------------------

META_DLPFC <- META[
  META$Region == "DLPFC",
  ,
  drop = FALSE
]

META_HIPPO <- META[
  META$Region == "HIPPO",
  ,
  drop = FALSE
]


stopifnot(
  nrow(META_DLPFC) == 379L,
  nrow(META_HIPPO) == 376L,
  !anyDuplicated(
    as.character(
      META_DLPFC$BrNum
    )
  ),
  !anyDuplicated(
    as.character(
      META_HIPPO$BrNum
    )
  )
)


A_META_INDEX <- match(
  as.character(
    A_RAW$donor_id
  ),
  as.character(
    META_DLPFC$BrNum
  )
)


B_META_INDEX <- match(
  as.character(
    B_RAW$donor_id
  ),
  as.character(
    META_HIPPO$BrNum
  )
)


if (
  anyNA(A_META_INDEX) ||
  anyNA(B_META_INDEX)
) {
  stop(
    "A/B donor-to-sample mapping failed."
  )
}


# ------------------------------------------------------------
# 7. STANDARDIZE MODEL A SAMPLE OOF
# ------------------------------------------------------------

A_SAMPLE <- data.frame(

  model =
    "A_DLPFC_only",

  unit =
    "sample",

  repeat_id =
    as.integer(
      A_RAW$repeat_id
    ),

  fold_id =
    as.integer(
      A_RAW$fold_id
    ),

  sample_id =
    as.character(
      META_DLPFC$RNum[
        A_META_INDEX
      ]
    ),

  donor_id =
    as.character(
      A_RAW$donor_id
    ),

  region =
    "DLPFC",

  diagnosis =
    ifelse(
      as.integer(
        A_RAW$y
      ) == 1L,
      "Schizo",
      "Control"
    ),

  y =
    as.integer(
      A_RAW$y
    ),

  oof_probability =
    as.numeric(
      A_RAW$probability
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 8. STANDARDIZE MODEL B SAMPLE OOF
# ------------------------------------------------------------

B_SAMPLE <- data.frame(

  model =
    "B_HIPPO_only",

  unit =
    "sample",

  repeat_id =
    as.integer(
      B_RAW$repeat_id
    ),

  fold_id =
    as.integer(
      B_RAW$fold_id
    ),

  sample_id =
    as.character(
      META_HIPPO$RNum[
        B_META_INDEX
      ]
    ),

  donor_id =
    as.character(
      B_RAW$donor_id
    ),

  region =
    "HIPPO",

  diagnosis =
    ifelse(
      as.integer(
        B_RAW$y
      ) == 1L,
      "Schizo",
      "Control"
    ),

  y =
    as.integer(
      B_RAW$y
    ),

  oof_probability =
    as.numeric(
      B_RAW$probability
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 9. MODEL C/E SAMPLE + DONOR OOF
# ------------------------------------------------------------

CE_SAMPLE_RAW <- read.csv(
  CE_SAMPLE_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

CE_DONOR_RAW <- read.csv(
  CE_DONOR_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


stopifnot(
  nrow(CE_SAMPLE_RAW) == 3775L,
  nrow(CE_DONOR_RAW) == 2270L
)


CE_SAMPLE <- data.frame(

  model =
    "C_E_pooled_shared_only",

  unit =
    "sample",

  repeat_id =
    as.integer(
      CE_SAMPLE_RAW$repeat_id
    ),

  fold_id =
    as.integer(
      CE_SAMPLE_RAW$fold_id
    ),

  sample_id =
    as.character(
      CE_SAMPLE_RAW$sample_id
    ),

  donor_id =
    as.character(
      CE_SAMPLE_RAW$donor_id
    ),

  region =
    as.character(
      CE_SAMPLE_RAW$region
    ),

  diagnosis =
    ifelse(
      as.integer(
        CE_SAMPLE_RAW$y
      ) == 1L,
      "Schizo",
      "Control"
    ),

  y =
    as.integer(
      CE_SAMPLE_RAW$y
    ),

  oof_probability =
    as.numeric(
      CE_SAMPLE_RAW$probability
    ),

  stringsAsFactors = FALSE
)


CE_DONOR <- data.frame(

  model =
    "C_E_pooled_shared_only",

  unit =
    "donor",

  repeat_id =
    as.integer(
      CE_DONOR_RAW$repeat_id
    ),

  fold_id =
    as.integer(
      CE_DONOR_RAW$fold_id
    ),

  sample_id =
    NA_character_,

  donor_id =
    as.character(
      CE_DONOR_RAW$donor_id
    ),

  region =
    "DONOR_AGGREGATE",

  diagnosis =
    ifelse(
      as.integer(
        CE_DONOR_RAW$y
      ) == 1L,
      "Schizo",
      "Control"
    ),

  y =
    as.integer(
      CE_DONOR_RAW$y
    ),

  oof_probability =
    as.numeric(
      CE_DONOR_RAW$probability
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 10. MODEL D DONOR OOF
# ------------------------------------------------------------

D_RAW <- read.csv(
  D_DONOR_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


if (
  nrow(D_RAW) != 2270L
) {
  stop(
    "Model D donor OOF must contain 2270 rows."
  )
}


D_DONOR <- data.frame(

  model =
    "D_independent_regions",

  unit =
    "donor",

  repeat_id =
    as.integer(
      D_RAW$repeat_id
    ),

  fold_id =
    as.integer(
      D_RAW$fold_id
    ),

  sample_id =
    NA_character_,

  donor_id =
    as.character(
      D_RAW$donor_id
    ),

  region =
    "DONOR_AGGREGATE",

  diagnosis =
    ifelse(
      as.integer(
        D_RAW$y
      ) == 1L,
      "Schizo",
      "Control"
    ),

  y =
    as.integer(
      D_RAW$y
    ),

  oof_probability =
    as.numeric(
      D_RAW$probability
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 11. LOAD ALL MODEL F OUTER-TEST PREDICTIONS
# ------------------------------------------------------------

F_SAMPLE_LIST <- lapply(
  F_SAMPLE_FILES,
  function(current_file) {

    read.csv(
      current_file,
      stringsAsFactors = FALSE,
      check.names = FALSE
    )
  }
)


F_DONOR_LIST <- lapply(
  F_DONOR_FILES,
  function(current_file) {

    read.csv(
      current_file,
      stringsAsFactors = FALSE,
      check.names = FALSE
    )
  }
)


F_SAMPLE_RAW <- do.call(
  rbind,
  F_SAMPLE_LIST
)

F_DONOR_RAW <- do.call(
  rbind,
  F_DONOR_LIST
)


stopifnot(
  nrow(F_SAMPLE_RAW) == 3775L,
  nrow(F_DONOR_RAW) == 2270L
)


# ------------------------------------------------------------
# 12. STANDARDIZE MODEL F
# ------------------------------------------------------------

F_SAMPLE <- data.frame(

  model =
    "F_shared_plus_deviation",

  unit =
    "sample",

  repeat_id =
    as.integer(
      F_SAMPLE_RAW$outer_repeat
    ),

  fold_id =
    as.integer(
      F_SAMPLE_RAW$outer_fold
    ),

  sample_id =
    as.character(
      F_SAMPLE_RAW$RNum
    ),

  donor_id =
    as.character(
      F_SAMPLE_RAW$BrNum
    ),

  region =
    as.character(
      F_SAMPLE_RAW$Region
    ),

  diagnosis =
    ifelse(
      as.integer(
        F_SAMPLE_RAW$y
      ) == 1L,
      "Schizo",
      "Control"
    ),

  y =
    as.integer(
      F_SAMPLE_RAW$y
    ),

  oof_probability =
    as.numeric(
      F_SAMPLE_RAW$prediction
    ),

  stringsAsFactors = FALSE
)


F_DONOR <- data.frame(

  model =
    "F_shared_plus_deviation",

  unit =
    "donor",

  repeat_id =
    as.integer(
      F_DONOR_RAW$outer_repeat
    ),

  fold_id =
    as.integer(
      F_DONOR_RAW$outer_fold
    ),

  sample_id =
    NA_character_,

  donor_id =
    as.character(
      F_DONOR_RAW$BrNum
    ),

  region =
    "DONOR_AGGREGATE",

  diagnosis =
    ifelse(
      as.integer(
        F_DONOR_RAW$y
      ) == 1L,
      "Schizo",
      "Control"
    ),

  y =
    as.integer(
      F_DONOR_RAW$y
    ),

  oof_probability =
    as.numeric(
      F_DONOR_RAW$prediction
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 13. MODEL A/B DONOR TABLES
#
# One sample per donor in these region-specific models, so
# sample-level OOF probability equals donor-level probability.
# ------------------------------------------------------------

A_DONOR <- A_SAMPLE
A_DONOR$unit <- "donor"
A_DONOR$sample_id <- NA_character_


B_DONOR <- B_SAMPLE
B_DONOR$unit <- "donor"
B_DONOR$sample_id <- NA_character_


# ------------------------------------------------------------
# 14. CANONICAL SAMPLE-LEVEL FREEZE
#
# Model D has no separate sample-level final prediction:
# its primary output is a donor-level combination of the
# independently fitted regional models.
# ------------------------------------------------------------

SAMPLE_FREEZE <- rbind(
  A_SAMPLE,
  B_SAMPLE,
  CE_SAMPLE,
  F_SAMPLE
)


# ------------------------------------------------------------
# 15. CANONICAL DONOR-LEVEL FREEZE
# ------------------------------------------------------------

DONOR_FREEZE <- rbind(
  A_DONOR,
  B_DONOR,
  CE_DONOR,
  D_DONOR,
  F_DONOR
)


rownames(
  SAMPLE_FREEZE
) <- NULL

rownames(
  DONOR_FREEZE
) <- NULL


# ------------------------------------------------------------
# 16. BASIC NUMERIC VALIDITY
# ------------------------------------------------------------

if (
  anyNA(
    SAMPLE_FREEZE$oof_probability
  ) ||
  any(
    !is.finite(
      SAMPLE_FREEZE$oof_probability
    )
  )
) {
  stop(
    "Invalid sample-level OOF probability."
  )
}


if (
  anyNA(
    DONOR_FREEZE$oof_probability
  ) ||
  any(
    !is.finite(
      DONOR_FREEZE$oof_probability
    )
  )
) {
  stop(
    "Invalid donor-level OOF probability."
  )
}


if (
  any(
    SAMPLE_FREEZE$oof_probability < 0 |
    SAMPLE_FREEZE$oof_probability > 1
  )
) {
  stop(
    "Sample OOF probability outside [0,1]."
  )
}


if (
  any(
    DONOR_FREEZE$oof_probability < 0 |
    DONOR_FREEZE$oof_probability > 1
  )
) {
  stop(
    "Donor OOF probability outside [0,1]."
  )
}


cat(
  "OOF probability validity: PASS\n"
)


# ------------------------------------------------------------
# 17. LOCKED OUTER-FOLD ALIGNMENT
# ------------------------------------------------------------

SAMPLE_LOCK_KEY <- paste(
  SAMPLE_FREEZE$donor_id,
  SAMPLE_FREEZE$repeat_id,
  SAMPLE_FREEZE$fold_id,
  sep = "|"
)


DONOR_LOCK_KEY <- paste(
  DONOR_FREEZE$donor_id,
  DONOR_FREEZE$repeat_id,
  DONOR_FREEZE$fold_id,
  sep = "|"
)


if (!all(
  SAMPLE_LOCK_KEY %in%
    LOCKED_KEY
)) {
  stop(
    "Sample OOF row not aligned to locked outer test fold."
  )
}


if (!all(
  DONOR_LOCK_KEY %in%
    LOCKED_KEY
)) {
  stop(
    "Donor OOF row not aligned to locked outer test fold."
  )
}


cat(
  "Locked held-out fold alignment: PASS\n"
)


# ------------------------------------------------------------
# 18. DUPLICATE / COVERAGE CHECKS
# ------------------------------------------------------------

SAMPLE_UNIQUE_KEY <- paste(
  SAMPLE_FREEZE$model,
  SAMPLE_FREEZE$repeat_id,
  SAMPLE_FREEZE$sample_id,
  sep = "|"
)


if (
  anyDuplicated(
    SAMPLE_UNIQUE_KEY
  )
) {
  stop(
    "Duplicate sample prediction within model/repeat."
  )
}


DONOR_UNIQUE_KEY <- paste(
  DONOR_FREEZE$model,
  DONOR_FREEZE$repeat_id,
  DONOR_FREEZE$donor_id,
  sep = "|"
)


if (
  anyDuplicated(
    DONOR_UNIQUE_KEY
  )
) {
  stop(
    "Duplicate donor prediction within model/repeat."
  )
}


cat(
  "Duplicate OOF prediction check: PASS\n"
)


# ------------------------------------------------------------
# 19. EXPECTED ROW COUNTS
# ------------------------------------------------------------

EXPECTED_SAMPLE_ROWS <- (
  1895L +
  1880L +
  3775L +
  3775L
)


EXPECTED_DONOR_ROWS <- (
  1895L +
  1880L +
  2270L +
  2270L +
  2270L
)


stopifnot(
  nrow(SAMPLE_FREEZE) ==
    EXPECTED_SAMPLE_ROWS,
  nrow(DONOR_FREEZE) ==
    EXPECTED_DONOR_ROWS
)


cat(
  "Expected OOF row counts: PASS\n"
)


# ------------------------------------------------------------
# 20. AUC FUNCTION
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  ok <- (
    !is.na(y) &
    is.finite(score)
  )

  y <- as.integer(
    y[ok]
  )

  score <- as.numeric(
    score[ok]
  )

  n_case <- sum(
    y == 1L
  )

  n_control <- sum(
    y == 0L
  )

  if (
    n_case == 0L ||
    n_control == 0L
  ) {
    return(
      NA_real_
    )
  }

  score_rank <- rank(
    score,
    ties.method = "average"
  )

  (
    sum(
      score_rank[
        y == 1L
      ]
    ) -
      n_case *
      (
        n_case + 1
      ) / 2
  ) /
    (
      n_case *
      n_control
    )
}


# ------------------------------------------------------------
# 21. RECOMPUTE REPEATED-OOF DONOR AUROC
# ------------------------------------------------------------

MODEL_CODES <- c(
  "A_DLPFC_only",
  "B_HIPPO_only",
  "C_E_pooled_shared_only",
  "D_independent_regions",
  "F_shared_plus_deviation"
)


make_model_summary <- function(
  current_model
) {

  z <- DONOR_FREEZE[
    DONOR_FREEZE$model ==
      current_model,
    ,
    drop = FALSE
  ]


  donor_average <- aggregate(
    oof_probability ~ donor_id + y,
    data = z,
    FUN = mean
  )


  data.frame(

    model =
      current_model,

    donor_oof_rows =
      nrow(z),

    unique_donors =
      length(
        unique(
          z$donor_id
        )
      ),

    repeats_per_donor_min =
      min(
        table(
          z$donor_id
        )
      ),

    repeats_per_donor_max =
      max(
        table(
          z$donor_id
        )
      ),

    repeated_OOF_donor_AUROC =
      auc_rank(
        donor_average$y,
        donor_average$oof_probability
      ),

    stringsAsFactors = FALSE
  )
}


MODEL_SUMMARY_LIST <- lapply(
  MODEL_CODES,
  make_model_summary
)


MODEL_SUMMARY <- do.call(
  rbind,
  MODEL_SUMMARY_LIST
)


rownames(
  MODEL_SUMMARY
) <- NULL


# ------------------------------------------------------------
# 22. SAMPLE-LEVEL MODEL COVERAGE SUMMARY
# ------------------------------------------------------------

SAMPLE_MODEL_CODES <- unique(
  SAMPLE_FREEZE$model
)


make_sample_summary <- function(
  current_model
) {

  z <- SAMPLE_FREEZE[
    SAMPLE_FREEZE$model ==
      current_model,
    ,
    drop = FALSE
  ]


  data.frame(

    model =
      current_model,

    sample_oof_rows =
      nrow(z),

    unique_samples =
      length(
        unique(
          z$sample_id
        )
      ),

    unique_donors =
      length(
        unique(
          z$donor_id
        )
      ),

    stringsAsFactors = FALSE
  )
}


SAMPLE_SUMMARY_LIST <- lapply(
  SAMPLE_MODEL_CODES,
  make_sample_summary
)


SAMPLE_SUMMARY <- do.call(
  rbind,
  SAMPLE_SUMMARY_LIST
)


rownames(
  SAMPLE_SUMMARY
) <- NULL


# ------------------------------------------------------------
# 23. FINAL FIVE-REPEAT COVERAGE ASSERTIONS
# ------------------------------------------------------------

if (!all(
  MODEL_SUMMARY$repeats_per_donor_min ==
    5L
)) {
  stop(
    "At least one model/donor has fewer than five OOF repeats."
  )
}


if (!all(
  MODEL_SUMMARY$repeats_per_donor_max ==
    5L
)) {
  stop(
    "At least one model/donor has more than five OOF repeats."
  )
}


cat(
  "Five-repeat OOF coverage: PASS\n"
)


# ------------------------------------------------------------
# 24. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "06_models",
  "oof_freeze_v1"
)


dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


SAMPLE_OUT <- file.path(
  OUT_DIR,
  "STEP05D_CANONICAL_sample_OOF_predictions_v1.csv"
)


DONOR_OUT <- file.path(
  OUT_DIR,
  "STEP05D_CANONICAL_donor_OOF_predictions_v1.csv"
)


MODEL_SUMMARY_OUT <- file.path(
  OUT_DIR,
  "STEP05D_OOF_model_summary_v1.csv"
)


SAMPLE_SUMMARY_OUT <- file.path(
  OUT_DIR,
  "STEP05D_OOF_sample_coverage_summary_v1.csv"
)


NOTES_OUT <- file.path(
  OUT_DIR,
  "STEP05D_OOF_FREEZE_NOTES_v1.txt"
)


MANIFEST_OUT <- file.path(
  OUT_DIR,
  "STEP05D_OOF_FREEZE_manifest_v1.csv"
)


SHA_OUT <- file.path(
  OUT_DIR,
  "STEP05D_OOF_FREEZE_v1.sha256"
)


# ------------------------------------------------------------
# 25. WRITE CANONICAL ARTIFACTS
# ------------------------------------------------------------

write.csv(
  SAMPLE_FREEZE,
  SAMPLE_OUT,
  row.names = FALSE
)


write.csv(
  DONOR_FREEZE,
  DONOR_OUT,
  row.names = FALSE
)


write.csv(
  MODEL_SUMMARY,
  MODEL_SUMMARY_OUT,
  row.names = FALSE
)


write.csv(
  SAMPLE_SUMMARY,
  SAMPLE_SUMMARY_OUT,
  row.names = FALSE
)


# ------------------------------------------------------------
# 26. TRANSPARENT CHRONOLOGY / METHOD NOTES
# ------------------------------------------------------------

writeLines(

  c(

    "STEP 05D — FORMAL OOF RESULT FREEZE",

    "",

    "STATUS: RETROSPECTIVE BACKFILL",

    "",

    paste(
      "The formal Step 05D OOF freeze was created after later",
      "discovery stability and score-lock steps had already",
      "been completed."
    ),

    "",

    paste(
      "This chronology deviation is explicitly documented and",
      "must not be represented as if Step 05D had been frozen",
      "prospectively before Steps 06-07."
    ),

    "",

    paste(
      "All rows were reconstructed solely from previously",
      "completed leakage-free discovery OOF artifacts."
    ),

    "",

    "No model was refitted.",

    "No external cohort was used.",

    "No external diagnosis was used.",

    "The frozen SCZ-MSS v1 score was not altered.",

    "",

    paste(
      "Model D is defined at donor level by combining the",
      "independently trained DLPFC and HIPPO OOF predictions;",
      "therefore no separate Model D sample-level final",
      "prediction rows are included."
    ),

    "",

    paste(
      "Model C pooled-region elastic-net and Model E shared-only",
      "are represented by the single C_E_pooled_shared_only",
      "artifact because those implemented formulations are",
      "mathematically equivalent."
    ),

    "",

    paste(
      "Every frozen prediction is aligned to the prespecified",
      "locked donor-level outer test fold."
    ),

    "",

    paste(
      "No in-sample prediction is included in these canonical",
      "OOF freeze artifacts."
    )
  ),

  NOTES_OUT
)


# ------------------------------------------------------------
# 27. SHA-256 HASHES
# ------------------------------------------------------------

ARTIFACT_PATHS <- c(
  SAMPLE_OUT,
  DONOR_OUT,
  MODEL_SUMMARY_OUT,
  SAMPLE_SUMMARY_OUT,
  NOTES_OUT
)


ARTIFACT_NAMES <- basename(
  ARTIFACT_PATHS
)


ARTIFACT_SHA <- vapply(
  ARTIFACT_PATHS,
  function(current_path) {

    digest::digest(
      file = current_path,
      algo = "sha256"
    )
  },
  character(1)
)


MANIFEST <- data.frame(

  artifact_name =
    ARTIFACT_NAMES,

  relative_path =
    file.path(
      "06_models",
      "oof_freeze_v1",
      ARTIFACT_NAMES
    ),

  sha256 =
    unname(
      ARTIFACT_SHA
    ),

  status =
    "FROZEN",

  stringsAsFactors = FALSE
)


write.csv(
  MANIFEST,
  MANIFEST_OUT,
  row.names = FALSE
)


MANIFEST_SHA <- digest::digest(
  file = MANIFEST_OUT,
  algo = "sha256"
)


MASTER_SHA <- digest::digest(
  paste(
    c(
      unname(
        ARTIFACT_SHA
      ),
      MANIFEST_SHA
    ),
    collapse = "|"
  ),
  algo = "sha256",
  serialize = FALSE
)


SHA_LINES <- c(

  paste(
    unname(
      ARTIFACT_SHA
    ),
    ARTIFACT_NAMES
  ),

  paste(
    MANIFEST_SHA,
    basename(
      MANIFEST_OUT
    )
  ),

  paste(
    MASTER_SHA,
    "STEP05D_OOF_FREEZE_MASTER_SHA256"
  )
)


writeLines(
  SHA_LINES,
  SHA_OUT
)


# ------------------------------------------------------------
# 28. RE-READ + HASH VERIFICATION
# ------------------------------------------------------------

SAMPLE_VERIFY <- read.csv(
  SAMPLE_OUT,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

DONOR_VERIFY <- read.csv(
  DONOR_OUT,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


stopifnot(
  nrow(SAMPLE_VERIFY) ==
    nrow(SAMPLE_FREEZE),
  nrow(DONOR_VERIFY) ==
    nrow(DONOR_FREEZE)
)


CURRENT_SHA <- vapply(
  ARTIFACT_PATHS,
  function(current_path) {

    digest::digest(
      file = current_path,
      algo = "sha256"
    )
  },
  character(1)
)


if (!identical(
  unname(
    CURRENT_SHA
  ),
  unname(
    ARTIFACT_SHA
  )
)) {
  stop(
    "Post-write SHA verification failed."
  )
}


cat(
  "Post-write SHA verification: PASS\n"
)


# ------------------------------------------------------------
# 29. OUTPUT
# ------------------------------------------------------------

RUNTIME_SECONDS <- (
  proc.time()[3] -
    START_TIME
)


cat("\n============================================================\n")
cat("STEP 05D — FORMAL OOF RESULT FREEZE COMPLETE\n")
cat("============================================================\n")


cat(
  "\n--- DONOR-LEVEL MODEL SUMMARY ---\n"
)


print(
  MODEL_SUMMARY,
  row.names = FALSE,
  digits = 6
)


cat(
  "\n--- SAMPLE-LEVEL COVERAGE SUMMARY ---\n"
)


print(
  SAMPLE_SUMMARY,
  row.names = FALSE
)


cat(
  "\nCanonical sample OOF rows: ",
  nrow(
    SAMPLE_FREEZE
  ),
  "\n",
  sep = ""
)


cat(
  "Canonical donor OOF rows: ",
  nrow(
    DONOR_FREEZE
  ),
  "\n",
  sep = ""
)


cat(
  "\nMASTER SHA-256:\n",
  MASTER_SHA,
  "\n",
  sep = ""
)


cat(
  "\nChronology status: RETROSPECTIVE BACKFILL\n"
)


cat(
  "External data used: NO\n"
)


cat(
  "Frozen SCZ-MSS modified: NO\n"
)


cat(
  "Runtime: ",
  round(
    RUNTIME_SECONDS,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 05D — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
digest version: 0.6.39
Required source artifacts: PASS
Locked 5x5 donor split: PASS
OOF probability validity: PASS
Locked held-out fold alignment: PASS
Duplicate OOF prediction check: PASS
Expected OOF row counts: PASS
Five-repeat OOF coverage: PASS
Post-write SHA verification: PASS

STEP 05D — FORMAL OOF RESULT FREEZE COMPLETE

--- DONOR-LEVEL MODEL SUMMARY ---
                   model donor_oof_rows unique_donors repeats_per_donor_min
            A_DLPFC_only           1895           379                     5
            B_HIPPO_only           1880           376                     5
  C_E_pooled_shared_only           2270           454                     5
   D_independent_regions           2270           454                     5
 F_shared_plus_deviation           2270           454                     5
 repeats_per_donor_max repeated_OOF_donor_AUROC
                     5                 0.790676
                     5                 0.805192
            

In [38]:
# ============================================================
# STEP 08A — CMC LOCAL SOURCE AVAILABILITY AUDIT
#
# Goal:
#   Identify whether CommonMind MSSM/Penn/Pitt source files
#   are locally available for primary external validation.
#
# IMPORTANT:
#   - No diagnosis analysis
#   - No SCZ-MSS scoring
#   - No outcome unblinding
#   - HBCC is NOT accepted as primary independent validation
#
# If no source files are found, this step records a clean
# SOURCE_NOT_PRESENT status rather than fabricating inputs.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]


# ------------------------------------------------------------
# 0. PROJECT SENTINEL
# ------------------------------------------------------------

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. ROOTS TO AUDIT
# ------------------------------------------------------------

RAW_ROOT <- file.path(
  PROJECT_ROOT,
  "01_raw"
)

if (!dir.exists(RAW_ROOT)) {
  stop("01_raw directory missing.")
}


EXPECTED_CMC_DIR <- file.path(
  RAW_ROOT,
  "commonmind_mssm_penn_pitt"
)


cat(
  "Expected CMC directory:\n",
  EXPECTED_CMC_DIR,
  "\n\n",
  sep = ""
)


# ------------------------------------------------------------
# 2. SEARCH LOCAL RAW TREE
# ------------------------------------------------------------

ALL_FILES <- list.files(
  RAW_ROOT,
  recursive = TRUE,
  full.names = TRUE,
  include.dirs = FALSE
)


NORMALIZED_FILES <- gsub(
  "\\\\",
  "/",
  ALL_FILES
)


CMC_PATTERN <- paste(
  c(
    "commonmind",
    "common_mind",
    "cmc",
    "mssm",
    "mountsinai",
    "mount_sinai",
    "penn",
    "upenn",
    "pitt",
    "pittsburgh"
  ),
  collapse = "|"
)


CMC_INDEX <- grepl(
  CMC_PATTERN,
  tolower(
    NORMALIZED_FILES
  )
)


CMC_FILES <- ALL_FILES[
  CMC_INDEX
]


# ------------------------------------------------------------
# 3. BUILD FILE INVENTORY
# ------------------------------------------------------------

if (
  length(CMC_FILES) > 0L
) {

  FILE_INFO <- file.info(
    CMC_FILES
  )


  EXTENSION <- tolower(
    tools::file_ext(
      CMC_FILES
    )
  )


  INVENTORY <- data.frame(

    file_name =
      basename(
        CMC_FILES
      ),

    full_path =
      gsub(
        "\\\\",
        "/",
        CMC_FILES
      ),

    extension =
      EXTENSION,

    size_bytes =
      as.numeric(
        FILE_INFO$size
      ),

    size_MB =
      round(
        as.numeric(
          FILE_INFO$size
        ) / 1024^2,
        3
      ),

    modified_time =
      as.character(
        FILE_INFO$mtime
      ),

    stringsAsFactors = FALSE
  )


  INVENTORY <- INVENTORY[
    order(
      INVENTORY$full_path
    ),
    ,
    drop = FALSE
  ]


  rownames(
    INVENTORY
  ) <- NULL

} else {

  INVENTORY <- data.frame(

    file_name =
      character(0),

    full_path =
      character(0),

    extension =
      character(0),

    size_bytes =
      numeric(0),

    size_MB =
      numeric(0),

    modified_time =
      character(0),

    stringsAsFactors = FALSE
  )
}


# ------------------------------------------------------------
# 4. LIGHTWEIGHT FILE-TYPE CLASSIFICATION
# ------------------------------------------------------------

if (
  nrow(INVENTORY) > 0L
) {

  LOWER_NAME <- tolower(
    INVENTORY$file_name
  )


  INVENTORY$possible_expression_file <- grepl(
    paste(
      c(
        "expression",
        "expr",
        "fpkm",
        "tpm",
        "count",
        "rpkm",
        "gene"
      ),
      collapse = "|"
    ),
    LOWER_NAME
  )


  INVENTORY$possible_metadata_file <- grepl(
    paste(
      c(
        "meta",
        "phenotype",
        "clinical",
        "covariate",
        "sample",
        "subject"
      ),
      collapse = "|"
    ),
    LOWER_NAME
  )


  INVENTORY$mentions_HBCC <- grepl(
    "hbcc",
    LOWER_NAME
  )


  INVENTORY$mentions_MSSM <- grepl(
    "mssm|mountsinai|mount_sinai",
    LOWER_NAME
  )


  INVENTORY$mentions_Penn <- grepl(
    "penn|upenn",
    LOWER_NAME
  )


  INVENTORY$mentions_Pitt <- grepl(
    "pitt|pittsburgh",
    LOWER_NAME
  )
}


# ------------------------------------------------------------
# 5. STATUS
# ------------------------------------------------------------

EXPECTED_DIR_EXISTS <- dir.exists(
  EXPECTED_CMC_DIR
)


N_CMC_FILES <- nrow(
  INVENTORY
)


if (
  N_CMC_FILES == 0L
) {

  ACCESS_STATUS <- "SOURCE_NOT_PRESENT"

} else {

  ACCESS_STATUS <- "LOCAL_CMC_CANDIDATES_FOUND"
}


# ------------------------------------------------------------
# 6. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "cmc"
)


dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


INVENTORY_OUT <- file.path(
  OUT_DIR,
  "STEP08A_CMC_local_source_inventory_v1.csv"
)


STATUS_OUT <- file.path(
  OUT_DIR,
  "STEP08A_CMC_source_availability_v1.txt"
)


# ------------------------------------------------------------
# 7. SAVE
# ------------------------------------------------------------

write.csv(
  INVENTORY,
  INVENTORY_OUT,
  row.names = FALSE
)


STATUS_LINES <- c(

  "STEP 08A — CMC LOCAL SOURCE AVAILABILITY AUDIT",

  "",

  paste0(
    "PROJECT_ID: ",
    PROJECT_ID
  ),

  paste0(
    "EXPECTED_CMC_DIRECTORY: ",
    gsub(
      "\\\\",
      "/",
      EXPECTED_CMC_DIR
    )
  ),

  paste0(
    "EXPECTED_DIRECTORY_EXISTS: ",
    EXPECTED_DIR_EXISTS
  ),

  paste0(
    "CMC_CANDIDATE_FILES_FOUND: ",
    N_CMC_FILES
  ),

  paste0(
    "ACCESS_STATUS: ",
    ACCESS_STATUS
  ),

  "",

  "PRIMARY_EXTERNAL_TARGET: CommonMind MSSM/Penn/Pitt",

  "HBCC_PRIMARY_EXTERNAL_USE: EXCLUDED",

  "",

  "DIAGNOSIS_ANALYZED: NO",

  "SCZ_MSS_SCORE_COMPUTED: NO",

  "OUTCOME_UNBLINDED: NO",

  "EXTERNAL_MODEL_TUNING: NO"
)


writeLines(
  STATUS_LINES,
  STATUS_OUT
)


# ------------------------------------------------------------
# 8. DISPLAY
# ------------------------------------------------------------

cat(
  "Expected directory exists: ",
  EXPECTED_DIR_EXISTS,
  "\n",
  sep = ""
)


cat(
  "CMC candidate files found: ",
  N_CMC_FILES,
  "\n",
  sep = ""
)


cat(
  "ACCESS STATUS: ",
  ACCESS_STATUS,
  "\n",
  sep = ""
)


if (
  N_CMC_FILES > 0L
) {

  cat(
    "\n--- LOCAL CMC CANDIDATE FILES ---\n"
  )

  print(
    INVENTORY,
    row.names = FALSE
  )

} else {

  cat(
    "\nNo local CommonMind MSSM/Penn/Pitt source files detected.\n"
  )

  cat(
    "Primary external-validation branch remains blocked until legitimate source files are available.\n"
  )
}


# ------------------------------------------------------------
# 9. FINAL VALIDATION
# ------------------------------------------------------------

stopifnot(
  file.exists(
    INVENTORY_OUT
  ),
  file.exists(
    STATUS_OUT
  )
)


RUNTIME <- proc.time()[3] -
  START_TIME


cat(
  "\nDiagnosis analyzed: NO\n"
)

cat(
  "SCZ-MSS computed: NO\n"
)

cat(
  "External outcome unblinded: NO\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 08A — LOCAL SOURCE AVAILABILITY AUDIT COMPLETE\n")
cat("============================================================\n")

Project sentinel: PASS
Expected CMC directory:
E:/SCZ_CROSSREGION_MSS_Q1/01_raw/commonmind_mssm_penn_pitt

Expected directory exists: TRUE
CMC candidate files found: 1
ACCESS STATUS: LOCAL_CMC_CANDIDATES_FOUND

--- LOCAL CMC CANDIDATE FILES ---
                                 file_name
 commonmind_required_files_manifest_v1.csv
                                                                                            full_path
 E:/SCZ_CROSSREGION_MSS_Q1/01_raw/commonmind_mssm_penn_pitt/commonmind_required_files_manifest_v1.csv
 extension size_bytes size_MB              modified_time
       csv        368       0 2026-09-30 10:48:09.327843
 possible_expression_file possible_metadata_file mentions_HBCC mentions_MSSM
                    FALSE                  FALSE         FALSE         FALSE
 mentions_Penn mentions_Pitt
         FALSE         FALSE

Diagnosis analyzed: NO
SCZ-MSS computed: NO
External outcome unblinded: NO
Runtime: 0.62 seconds

STEP 08A — LOCAL SOURCE AVAILABILITY AUD

In [39]:
# ============================================================
# STEP 08A-PUBLIC — GSE87194 PRIMARY EXTERNAL SOURCE ACQUISITION
#
# Public replacement for inaccessible controlled-access CMC.
#
# Downloads:
#   1. processed raw gene counts
#   2. GEO family SOFT metadata
#
# No scoring.
# No outcome analysis.
# No model tuning.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

# ------------------------------------------------------------
# 0. PROJECT SENTINEL
# ------------------------------------------------------------

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop("digest package unavailable.")
}


# ------------------------------------------------------------
# 2. OUTPUT DIRECTORY
# ------------------------------------------------------------

RAW_DIR <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse87194"
)

dir.create(
  RAW_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


# ------------------------------------------------------------
# 3. PUBLIC GEO URLS
# ------------------------------------------------------------

COUNTS_URL <- paste0(
  "https://ftp.ncbi.nlm.nih.gov/geo/series/",
  "GSE87nnn/GSE87194/suppl/",
  "GSE87194_raw_counts.txt.gz"
)

SOFT_URL <- paste0(
  "https://ftp.ncbi.nlm.nih.gov/geo/series/",
  "GSE87nnn/GSE87194/soft/",
  "GSE87194_family.soft.gz"
)


COUNTS_PATH <- file.path(
  RAW_DIR,
  "GSE87194_raw_counts.txt.gz"
)

SOFT_PATH <- file.path(
  RAW_DIR,
  "GSE87194_family.soft.gz"
)


# ------------------------------------------------------------
# 4. DOWNLOAD HELPER
# ------------------------------------------------------------

download_if_needed <- function(
  url,
  destination
) {

  if (
    file.exists(destination) &&
    file.info(destination)$size > 0
  ) {

    cat(
      "Already present: ",
      basename(destination),
      "\n",
      sep = ""
    )

    return(invisible(TRUE))
  }


  cat(
    "Downloading: ",
    basename(destination),
    "\n",
    sep = ""
  )


  status <- tryCatch(

    {
      download.file(
        url = url,
        destfile = destination,
        mode = "wb",
        quiet = FALSE,
        method = "libcurl"
      )

      TRUE
    },

    error = function(e) {

      cat(
        "DOWNLOAD ERROR: ",
        conditionMessage(e),
        "\n",
        sep = ""
      )

      FALSE
    }
  )


  if (
    !status ||
    !file.exists(destination) ||
    file.info(destination)$size <= 0
  ) {
    stop(
      paste0(
        "Download failed: ",
        basename(destination)
      )
    )
  }


  invisible(TRUE)
}


# ------------------------------------------------------------
# 5. DOWNLOAD PUBLIC DATA
# ------------------------------------------------------------

download_if_needed(
  COUNTS_URL,
  COUNTS_PATH
)

download_if_needed(
  SOFT_URL,
  SOFT_PATH
)


# ------------------------------------------------------------
# 6. FILE INTEGRITY
# ------------------------------------------------------------

FILE_PATHS <- c(
  COUNTS_PATH,
  SOFT_PATH
)


FILE_INFO <- file.info(
  FILE_PATHS
)


FILE_SHA <- vapply(
  FILE_PATHS,
  function(current_file) {

    digest::digest(
      file = current_file,
      algo = "sha256"
    )
  },
  character(1)
)


INVENTORY <- data.frame(

  accession = c(
    "GSE87194",
    "GSE87194"
  ),

  role = c(
    "processed_raw_gene_counts",
    "GEO_family_metadata"
  ),

  file_name = basename(
    FILE_PATHS
  ),

  full_path = gsub(
    "\\\\",
    "/",
    FILE_PATHS
  ),

  size_bytes = as.numeric(
    FILE_INFO$size
  ),

  size_MB = round(
    as.numeric(
      FILE_INFO$size
    ) / 1024^2,
    3
  ),

  sha256 = unname(
    FILE_SHA
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 7. LIGHT CONTENT AUDIT
# ------------------------------------------------------------

COUNTS_CONNECTION <- gzfile(
  COUNTS_PATH,
  open = "rt"
)

COUNTS_HEAD <- readLines(
  COUNTS_CONNECTION,
  n = 5,
  warn = FALSE
)

close(
  COUNTS_CONNECTION
)


SOFT_CONNECTION <- gzfile(
  SOFT_PATH,
  open = "rt"
)

SOFT_HEAD <- readLines(
  SOFT_CONNECTION,
  n = 20,
  warn = FALSE
)

close(
  SOFT_CONNECTION
)


if (
  length(COUNTS_HEAD) == 0L
) {
  stop(
    "Counts file could not be read."
  )
}


if (
  length(SOFT_HEAD) == 0L
) {
  stop(
    "SOFT file could not be read."
  )
}


# ------------------------------------------------------------
# 8. SAVE INVENTORY
# ------------------------------------------------------------

INVENTORY_PATH <- file.path(
  RAW_DIR,
  "GSE87194_source_inventory_v1.csv"
)

write.csv(
  INVENTORY,
  INVENTORY_PATH,
  row.names = FALSE
)


# ------------------------------------------------------------
# 9. PROTOCOL AMENDMENT NOTE
# ------------------------------------------------------------

AMENDMENT_DIR <- file.path(
  PROJECT_ROOT,
  "00_protocol"
)

dir.create(
  AMENDMENT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


AMENDMENT_PATH <- file.path(
  AMENDMENT_DIR,
  "EXTERNAL_VALIDATION_ACCESS_AMENDMENT_v2.txt"
)


writeLines(

  c(

    "EXTERNAL VALIDATION ACCESS AMENDMENT v2",

    "",

    "Reason for amendment:",
    paste(
      "The prespecified CommonMind MSSM/Penn/Pitt primary",
      "external dataset requires controlled access and was",
      "not locally accessible."
    ),

    "",

    paste(
      "This change is access-driven and was made without",
      "using any external SCZ-MSS score/outcome result."
    ),

    "",

    "Revised public validation hierarchy:",

    "PRIMARY: GSE87194 DLPFC RNA-seq",

    "SECONDARY: GSE80655 DLPFC RNA-seq",

    paste(
      "OPTIONAL CROSS-PLATFORM REPLICATION:",
      "GSE53987 BA46/PFC microarray"
    ),

    "",

    paste(
      "Frozen discovery cohort, model coefficients,",
      "24-gene SCZ-MSS weights, mapping rules and",
      "technical-gate thresholds remain unchanged."
    ),

    "",

    "No external model retraining is permitted.",

    "No external score-weight tuning is permitted."
  ),

  AMENDMENT_PATH
)


# ------------------------------------------------------------
# 10. FINAL CHECKS
# ------------------------------------------------------------

stopifnot(
  file.exists(COUNTS_PATH),
  file.exists(SOFT_PATH),
  file.exists(INVENTORY_PATH),
  file.exists(AMENDMENT_PATH),
  all(INVENTORY$size_bytes > 0)
)


RUNTIME <- proc.time()[3] -
  START_TIME


# ------------------------------------------------------------
# 11. OUTPUT
# ------------------------------------------------------------

cat("\n============================================================\n")
cat("STEP 08A-PUBLIC — GSE87194 SOURCE ACQUISITION COMPLETE\n")
cat("============================================================\n")


print(
  INVENTORY,
  row.names = FALSE
)


cat(
  "\n--- COUNTS FILE FIRST LINES ---\n"
)

cat(
  paste(
    COUNTS_HEAD,
    collapse = "\n"
  ),
  "\n"
)


cat(
  "\nPrimary public external cohort: GSE87194\n"
)

cat(
  "Expected GEO design: 19 Control + 19 SCZ DLPFC\n"
)

cat(
  "SCZ-MSS score evaluated: NO\n"
)

cat(
  "External model tuning: NO\n"
)

cat(
  "Protocol amendment written: YES\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 08A-PUBLIC — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Downloading: GSE87194_raw_counts.txt.gz
Downloading: GSE87194_family.soft.gz

STEP 08A-PUBLIC — GSE87194 SOURCE ACQUISITION COMPLETE
 accession                      role                  file_name
  GSE87194 processed_raw_gene_counts GSE87194_raw_counts.txt.gz
  GSE87194       GEO_family_metadata    GSE87194_family.soft.gz
                                                            full_path
 E:/SCZ_CROSSREGION_MSS_Q1/01_raw/gse87194/GSE87194_raw_counts.txt.gz
    E:/SCZ_CROSSREGION_MSS_Q1/01_raw/gse87194/GSE87194_family.soft.gz
 size_bytes size_MB
    1948588   1.858
       4658   0.004
                                                           sha256
 cf12db7d70de4876a6a6a606afea9db04f31b06ca4e4c8179a7580b13729a177
 4b0dd8b3065664b1859844e340320d290e25d0d100c9223793fb4857907a65b3

--- COUNTS FILE FIRST LINES ---
	T18_C	T68_C	T47_C	T17_C	T88_C	T29_C	T20_C	T40_C	T86_C	T30_C	T7_C	T56_C	T37_C	T85_C	T26_C	T28_C	T9_C	T58_C	T46_C	T2_S	T5_S	T22_S	T25_S	T61_S	T34_S	T32_

In [41]:
# ============================================================
# STEP 08B/08C/08D — GSE87194
# INDEPENDENCE + FROZEN-GENE MAPPING + TECHNICAL GATE
# CORRECTED VERSION
#
# FIXES:
#   1. GSE87194 gene-name column is NOT forced into row.names
#      because duplicated gene identifiers exist.
#
#   2. Frozen gene-ID column correctly recognizes:
#      ensemblID_stable
#
# Mapping rule:
#   exact symbol match only
#   exactly one external occurrence required
#   no duplicate aggregation
#   no synonym rescue
#   no case conversion
#   no outcome-driven mapping
#
# NO diagnosis-outcome analysis.
# NO AUROC.
# NO score-by-diagnosis comparison.
# NO weight tuning.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. REQUIRED FILES
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse87194",
  "GSE87194_raw_counts.txt.gz"
)

SOFT_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse87194",
  "GSE87194_family.soft.gz"
)

WEIGHTS_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_gene_weights.csv"
)

MAPPING_POLICY_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_mapping_policy.json"
)

GATE_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_technical_gate.json"
)

REQUIRED <- c(
  COUNTS_PATH,
  SOFT_PATH,
  WEIGHTS_PATH,
  MAPPING_POLICY_PATH,
  GATE_PATH
)

MISSING <- REQUIRED[
  !file.exists(REQUIRED)
]

if (length(MISSING) > 0L) {

  print(MISSING)

  stop(
    "Required file missing."
  )
}

cat("Required files: PASS\n")


# ------------------------------------------------------------
# 2. PACKAGES
# ------------------------------------------------------------

if (!requireNamespace(
  "jsonlite",
  quietly = TRUE
)) {
  stop(
    "jsonlite package unavailable."
  )
}

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop(
    "digest package unavailable."
  )
}


# ------------------------------------------------------------
# 3. LOAD FROZEN WEIGHTS
# ------------------------------------------------------------

WEIGHTS <- read.csv(
  WEIGHTS_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

cat(
  "Frozen weight columns:\n"
)

print(
  names(WEIGHTS)
)


# ------------------------------------------------------------
# 4. IDENTIFY FROZEN COLUMNS
# ------------------------------------------------------------

pick_column <- function(
  candidates,
  available
) {

  hit <- candidates[
    candidates %in%
      available
  ]

  if (
    length(hit) == 0L
  ) {
    return(
      NA_character_
    )
  }

  hit[1]
}


GENE_ID_COL <- pick_column(
  c(
    "ensemblID_stable",
    "gene_id",
    "ensembl_id",
    "ensemblID",
    "stable_ensembl_id",
    "gencodeID"
  ),
  names(WEIGHTS)
)


SYMBOL_COL <- pick_column(
  c(
    "gene_symbol",
    "symbol",
    "Symbol"
  ),
  names(WEIGHTS)
)


WEIGHT_COL <- pick_column(
  c(
    "final_normalized_weight",
    "normalized_weight",
    "final_weight",
    "weight"
  ),
  names(WEIGHTS)
)


if (
  is.na(GENE_ID_COL)
) {
  stop(
    "Frozen Ensembl/stable gene-ID column not identified."
  )
}


if (
  is.na(SYMBOL_COL)
) {
  stop(
    "Frozen gene-symbol column not identified."
  )
}


if (
  is.na(WEIGHT_COL)
) {
  stop(
    "Frozen normalized-weight column not identified."
  )
}


cat(
  "Gene ID column: ",
  GENE_ID_COL,
  "\n",
  sep = ""
)

cat(
  "Symbol column: ",
  SYMBOL_COL,
  "\n",
  sep = ""
)

cat(
  "Weight column: ",
  WEIGHT_COL,
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 5. FROZEN WEIGHT INTEGRITY
# ------------------------------------------------------------

if (
  nrow(WEIGHTS) != 24L
) {
  stop(
    paste0(
      "Expected 24 frozen genes; found ",
      nrow(WEIGHTS)
    )
  )
}


FROZEN_WEIGHT <- as.numeric(
  WEIGHTS[
    ,
    WEIGHT_COL,
    drop = TRUE
  ]
)


if (
  anyNA(FROZEN_WEIGHT) ||
  any(
    !is.finite(
      FROZEN_WEIGHT
    )
  )
) {
  stop(
    "Invalid frozen weights."
  )
}


ABS_WEIGHT_SUM <- sum(
  abs(
    FROZEN_WEIGHT
  )
)


if (
  abs(
    ABS_WEIGHT_SUM - 1
  ) > 1e-6
) {
  stop(
    paste0(
      "Frozen normalized absolute weights sum to ",
      ABS_WEIGHT_SUM,
      ", expected 1."
    )
  )
}


cat(
  "Frozen 24-gene weight integrity: PASS\n"
)


# ------------------------------------------------------------
# 6. FROZEN IDENTIFIERS
# ------------------------------------------------------------

FROZEN_GENE_ID <- trimws(
  as.character(
    WEIGHTS[
      ,
      GENE_ID_COL,
      drop = TRUE
    ]
  )
)


FROZEN_SYMBOL <- trimws(
  as.character(
    WEIGHTS[
      ,
      SYMBOL_COL,
      drop = TRUE
    ]
  )
)


FROZEN_GENE_ID[
  FROZEN_GENE_ID == ""
] <- NA_character_


FROZEN_SYMBOL[
  FROZEN_SYMBOL == ""
] <- NA_character_


cat(
  "Frozen genes with Ensembl ID: ",
  sum(
    !is.na(
      FROZEN_GENE_ID
    )
  ),
  "/24\n",
  sep = ""
)


cat(
  "Frozen genes with symbol: ",
  sum(
    !is.na(
      FROZEN_SYMBOL
    )
  ),
  "/24\n",
  sep = ""
)


# ------------------------------------------------------------
# 7. LOAD FROZEN POLICIES
# ------------------------------------------------------------

MAPPING_POLICY <- jsonlite::fromJSON(
  MAPPING_POLICY_PATH,
  simplifyVector = FALSE
)


GATE_POLICY <- jsonlite::fromJSON(
  GATE_PATH,
  simplifyVector = FALSE
)


MAPPING_TEXT <- paste(
  readLines(
    MAPPING_POLICY_PATH,
    warn = FALSE
  ),
  collapse = "\n"
)


GATE_TEXT <- paste(
  readLines(
    GATE_PATH,
    warn = FALSE
  ),
  collapse = "\n"
)


MAPPING_SHA <- digest::digest(
  file = MAPPING_POLICY_PATH,
  algo = "sha256"
)


GATE_SHA <- digest::digest(
  file = GATE_PATH,
  algo = "sha256"
)


cat(
  "Frozen mapping policy SHA256: ",
  MAPPING_SHA,
  "\n",
  sep = ""
)


cat(
  "Frozen gate policy SHA256: ",
  GATE_SHA,
  "\n",
  sep = ""
)


POLICY_MENTIONS_SYMBOL <- grepl(
  "symbol",
  tolower(
    MAPPING_TEXT
  )
)


cat(
  "Mapping policy mentions gene symbol: ",
  POLICY_MENTIONS_SYMBOL,
  "\n",
  sep = ""
)


if (
  !POLICY_MENTIONS_SYMBOL
) {
  stop(
    paste(
      "Frozen mapping policy does not document",
      "gene-symbol-based transfer."
    )
  )
}


# ------------------------------------------------------------
# 8. LOAD GSE87194 COUNTS
#
# CRITICAL FIX:
# Do NOT use row.names = 1.
# Duplicate external gene identifiers are retained explicitly.
# ------------------------------------------------------------

COUNTS_DF <- read.delim(
  COUNTS_PATH,
  header = TRUE,
  check.names = FALSE,
  stringsAsFactors = FALSE,
  quote = "",
  comment.char = ""
)


if (
  ncol(COUNTS_DF) < 2L
) {
  stop(
    "GSE87194 counts table malformed."
  )
}


EXTERNAL_GENE_ID <- trimws(
  as.character(
    COUNTS_DF[
      ,
      1,
      drop = TRUE
    ]
  )
)


COUNTS_DF <- COUNTS_DF[
  ,
  -1,
  drop = FALSE
]


COUNTS_MATRIX <- as.matrix(
  COUNTS_DF
)


storage.mode(
  COUNTS_MATRIX
) <- "numeric"


cat(
  "GSE87194 matrix: ",
  nrow(COUNTS_MATRIX),
  " genes x ",
  ncol(COUNTS_MATRIX),
  " samples\n",
  sep = ""
)


if (
  length(
    EXTERNAL_GENE_ID
  ) !=
    nrow(
      COUNTS_MATRIX
    )
) {
  stop(
    "Gene-ID vector length mismatch."
  )
}


if (
  ncol(COUNTS_MATRIX) != 38L
) {
  stop(
    paste0(
      "Expected 38 samples; found ",
      ncol(COUNTS_MATRIX)
    )
  )
}


if (
  anyNA(
    COUNTS_MATRIX
  ) ||
  any(
    !is.finite(
      COUNTS_MATRIX
    )
  ) ||
  any(
    COUNTS_MATRIX < 0
  )
) {
  stop(
    "Invalid raw-count matrix."
  )
}


if (
  anyNA(
    EXTERNAL_GENE_ID
  ) ||
  any(
    EXTERNAL_GENE_ID == ""
  )
) {
  stop(
    "Missing external gene identifiers."
  )
}


cat(
  "Raw-count numeric integrity: PASS\n"
)


# ------------------------------------------------------------
# 9. EXTERNAL GENE-ID DUPLICATION AUDIT
# ------------------------------------------------------------

EXTERNAL_DUPLICATED_FLAG <- (
  duplicated(
    EXTERNAL_GENE_ID
  ) |
    duplicated(
      EXTERNAL_GENE_ID,
      fromLast = TRUE
    )
)


N_DUPLICATED_ROWS <- sum(
  EXTERNAL_DUPLICATED_FLAG
)


N_DUPLICATED_IDENTIFIERS <- sum(
  duplicated(
    unique(
      EXTERNAL_GENE_ID[
        EXTERNAL_DUPLICATED_FLAG
      ]
    )
  )
)


DUPLICATE_TABLE <- sort(
  table(
    EXTERNAL_GENE_ID
  ),
  decreasing = TRUE
)


DUPLICATE_TABLE <- DUPLICATE_TABLE[
  DUPLICATE_TABLE > 1L
]


cat(
  "External duplicated gene rows: ",
  N_DUPLICATED_ROWS,
  "\n",
  sep = ""
)


cat(
  "External identifiers occurring >1 time: ",
  length(
    DUPLICATE_TABLE
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 10. EXACT SYMBOL MAPPING
#
# Important:
# A frozen symbol is usable only if it occurs EXACTLY ONCE
# in the external matrix.
#
# Duplicated external symbols are NOT aggregated/rescued.
# ------------------------------------------------------------

EXTERNAL_FREQUENCY <- table(
  EXTERNAL_GENE_ID
)


MAP_INDEX <- match(
  FROZEN_SYMBOL,
  EXTERNAL_GENE_ID
)


MAP_COUNT <- rep(
  0L,
  length(
    FROZEN_SYMBOL
  )
)


NONMISSING_SYMBOL <- !is.na(
  FROZEN_SYMBOL
)


MAP_COUNT[
  NONMISSING_SYMBOL
] <- as.integer(
  EXTERNAL_FREQUENCY[
    FROZEN_SYMBOL[
      NONMISSING_SYMBOL
    ]
  ]
)


MAP_COUNT[
  is.na(
    MAP_COUNT
  )
] <- 0L


UNAMBIGUOUS <- (
  !is.na(
    MAP_INDEX
  ) &
    MAP_COUNT == 1L
)


# ------------------------------------------------------------
# 11. BUILD MAPPING TABLE
# ------------------------------------------------------------

MAPPING <- data.frame(

  frozen_row =
    seq_len(
      nrow(WEIGHTS)
    ),

  frozen_gene_id =
    FROZEN_GENE_ID,

  frozen_symbol =
    FROZEN_SYMBOL,

  gene_type =
    if (
      "gene_type" %in%
        names(WEIGHTS)
    ) {
      as.character(
        WEIGHTS$gene_type
      )
    } else {
      NA_character_
    },

  frozen_weight =
    FROZEN_WEIGHT,

  direction =
    ifelse(
      FROZEN_WEIGHT > 0,
      "positive",
      "negative"
    ),

  external_match_count =
    MAP_COUNT,

  external_row =
    ifelse(
      UNAMBIGUOUS,
      MAP_INDEX,
      NA_integer_
    ),

  external_gene_id =
    ifelse(
      UNAMBIGUOUS,
      EXTERNAL_GENE_ID[
        MAP_INDEX
      ],
      NA_character_
    ),

  mapped_unambiguous =
    UNAMBIGUOUS,

  mapping_status =
    ifelse(
      is.na(
        FROZEN_SYMBOL
      ),
      "NO_FROZEN_SYMBOL",
      ifelse(
        MAP_COUNT == 0L,
        "NOT_FOUND",
        ifelse(
          MAP_COUNT == 1L,
          "EXACT_UNIQUE_MATCH",
          "AMBIGUOUS_DUPLICATE_EXTERNAL_ID"
        )
      )
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 12. COVERAGE METRICS
# ------------------------------------------------------------

GENE_COUNT_COVERAGE <- mean(
  MAPPING$mapped_unambiguous
)


ABS_WEIGHT_COVERAGE <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$mapped_unambiguous
    ]
  )
)


TOTAL_POSITIVE_WEIGHT <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight > 0
    ]
  )
)


TOTAL_NEGATIVE_WEIGHT <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight < 0
    ]
  )
)


RETAINED_POSITIVE_WEIGHT <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight > 0 &
        MAPPING$mapped_unambiguous
    ]
  )
)


RETAINED_NEGATIVE_WEIGHT <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight < 0 &
        MAPPING$mapped_unambiguous
    ]
  )
)


POSITIVE_WEIGHT_COVERAGE <- (
  RETAINED_POSITIVE_WEIGHT /
    TOTAL_POSITIVE_WEIGHT
)


NEGATIVE_WEIGHT_COVERAGE <- (
  RETAINED_NEGATIVE_WEIGHT /
    TOTAL_NEGATIVE_WEIGHT
)


# ------------------------------------------------------------
# 13. FROZEN TECHNICAL-GATE THRESHOLDS
# ------------------------------------------------------------

MIN_GENE_COUNT <- 0.80
MIN_ABS_WEIGHT <- 0.80
MIN_POSITIVE_WEIGHT <- 0.70
MIN_NEGATIVE_WEIGHT <- 0.70


GATE_TEXT_NORMALIZED <- gsub(
  "\\s+",
  "",
  GATE_TEXT
)


if (
  !grepl(
    "0\\.8",
    GATE_TEXT_NORMALIZED
  ) ||
    !grepl(
      "0\\.7",
      GATE_TEXT_NORMALIZED
    )
) {
  stop(
    paste(
      "Frozen gate JSON does not visibly contain",
      "the expected 0.80 / 0.70 thresholds."
    )
  )
}


PASS_GENE_COUNT <- (
  GENE_COUNT_COVERAGE >=
    MIN_GENE_COUNT
)


PASS_ABS_WEIGHT <- (
  ABS_WEIGHT_COVERAGE >=
    MIN_ABS_WEIGHT
)


PASS_POSITIVE_WEIGHT <- (
  POSITIVE_WEIGHT_COVERAGE >=
    MIN_POSITIVE_WEIGHT
)


PASS_NEGATIVE_WEIGHT <- (
  NEGATIVE_WEIGHT_COVERAGE >=
    MIN_NEGATIVE_WEIGHT
)


# ------------------------------------------------------------
# 14. SAMPLE COMPLETENESS FOR MAPPED GENES
# ------------------------------------------------------------

MAPPED_ROWS <- MAPPING$external_row[
  MAPPING$mapped_unambiguous
]


if (
  length(
    MAPPED_ROWS
  ) > 0L
) {

  MAPPED_COUNTS <- COUNTS_MATRIX[
    MAPPED_ROWS,
    ,
    drop = FALSE
  ]


  SAMPLE_COMPLETENESS <- colMeans(
    is.finite(
      MAPPED_COUNTS
    )
  )


  MIN_SAMPLE_COMPLETENESS <- min(
    SAMPLE_COMPLETENESS
  )

} else {

  MAPPED_COUNTS <- matrix(
    numeric(0),
    nrow = 0,
    ncol = ncol(
      COUNTS_MATRIX
    )
  )


  MIN_SAMPLE_COMPLETENESS <- 0
}


PASS_SAMPLE_COMPLETENESS <- (
  MIN_SAMPLE_COMPLETENESS >= 1
)


TECHNICAL_PASS <- all(
  PASS_GENE_COUNT,
  PASS_ABS_WEIGHT,
  PASS_POSITIVE_WEIGHT,
  PASS_NEGATIVE_WEIGHT,
  PASS_SAMPLE_COMPLETENESS
)


# ------------------------------------------------------------
# 15. PHENOTYPE-VISIBILITY TRANSPARENCY
# ------------------------------------------------------------

SAMPLE_NAMES <- colnames(
  COUNTS_MATRIX
)


LABEL_PATTERN_VISIBLE <- any(
  grepl(
    "_C$|_S$",
    SAMPLE_NAMES
  )
)


cat(
  "Diagnosis-like labels visible in sample names: ",
  LABEL_PATTERN_VISIBLE,
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 16. SOURCE INDEPENDENCE RECORD
# ------------------------------------------------------------

INDEPENDENCE <- data.frame(

  item = c(
    "Discovery_resource",
    "External_resource",
    "External_accession",
    "Direct_donor_overlap_evidence_found",
    "Operational_independence_status"
  ),

  value = c(
    "BrainSeq Phase II",
    "Public GEO external cohort",
    "GSE87194",
    "NO",
    "NO_DOCUMENTED_DIRECT_OVERLAP"
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 17. TECHNICAL GATE SUMMARY
# ------------------------------------------------------------

GATE_SUMMARY <- data.frame(

  metric = c(
    "Frozen_score_gene_count",
    "Mapped_unambiguous_gene_count",
    "Unmapped_or_ambiguous_gene_count",
    "Gene_count_coverage",
    "Required_gene_count_coverage",
    "Absolute_weight_coverage",
    "Required_absolute_weight_coverage",
    "Positive_weight_coverage",
    "Required_positive_weight_coverage",
    "Negative_weight_coverage",
    "Required_negative_weight_coverage",
    "Minimum_sample_completeness",
    "Required_sample_completeness",
    "PASS_gene_count",
    "PASS_absolute_weight",
    "PASS_positive_weight",
    "PASS_negative_weight",
    "PASS_sample_completeness",
    "Technical_PASS"
  ),

  value = c(
    24,
    sum(
      MAPPING$mapped_unambiguous
    ),
    sum(
      !MAPPING$mapped_unambiguous
    ),
    GENE_COUNT_COVERAGE,
    MIN_GENE_COUNT,
    ABS_WEIGHT_COVERAGE,
    MIN_ABS_WEIGHT,
    POSITIVE_WEIGHT_COVERAGE,
    MIN_POSITIVE_WEIGHT,
    NEGATIVE_WEIGHT_COVERAGE,
    MIN_NEGATIVE_WEIGHT,
    MIN_SAMPLE_COMPLETENESS,
    1,
    as.numeric(
      PASS_GENE_COUNT
    ),
    as.numeric(
      PASS_ABS_WEIGHT
    ),
    as.numeric(
      PASS_POSITIVE_WEIGHT
    ),
    as.numeric(
      PASS_NEGATIVE_WEIGHT
    ),
    as.numeric(
      PASS_SAMPLE_COMPLETENESS
    ),
    as.numeric(
      TECHNICAL_PASS
    )
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 18. DUPLICATED-ID AUDIT TABLE
# ------------------------------------------------------------

if (
  length(
    DUPLICATE_TABLE
  ) > 0L
) {

  DUPLICATE_AUDIT <- data.frame(

    external_gene_id =
      names(
        DUPLICATE_TABLE
      ),

    occurrence_count =
      as.integer(
        DUPLICATE_TABLE
      ),

    stringsAsFactors = FALSE
  )

} else {

  DUPLICATE_AUDIT <- data.frame(

    external_gene_id =
      character(0),

    occurrence_count =
      integer(0),

    stringsAsFactors = FALSE
  )
}


# ------------------------------------------------------------
# 19. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse87194"
)


dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


MAPPING_OUT <- file.path(
  OUT_DIR,
  "STEP08C_GSE87194_frozen_gene_mapping_v1.csv"
)


GATE_OUT <- file.path(
  OUT_DIR,
  "STEP08D_GSE87194_technical_gate_v1.csv"
)


INDEPENDENCE_OUT <- file.path(
  OUT_DIR,
  "STEP08B_GSE87194_independence_audit_v1.csv"
)


DUPLICATE_OUT <- file.path(
  OUT_DIR,
  "STEP08C_GSE87194_duplicate_gene_identifier_audit_v1.csv"
)


NOTE_OUT <- file.path(
  OUT_DIR,
  "STEP08BCD_GSE87194_AUDIT_NOTES_v1.txt"
)


HASH_OUT <- file.path(
  OUT_DIR,
  "STEP08BCD_GSE87194_SHA256_v1.csv"
)


# ------------------------------------------------------------
# 20. SAVE
# ------------------------------------------------------------

write.csv(
  MAPPING,
  MAPPING_OUT,
  row.names = FALSE
)


write.csv(
  GATE_SUMMARY,
  GATE_OUT,
  row.names = FALSE
)


write.csv(
  INDEPENDENCE,
  INDEPENDENCE_OUT,
  row.names = FALSE
)


write.csv(
  DUPLICATE_AUDIT,
  DUPLICATE_OUT,
  row.names = FALSE
)


writeLines(

  c(

    "GSE87194 EXTERNAL TECHNICAL AUDIT",

    "",

    paste0(
      "TECHNICAL_PASS: ",
      TECHNICAL_PASS
    ),

    "",

    paste0(
      "EXTERNAL_MATRIX_GENES: ",
      nrow(
        COUNTS_MATRIX
      )
    ),

    paste0(
      "EXTERNAL_MATRIX_SAMPLES: ",
      ncol(
        COUNTS_MATRIX
      )
    ),

    paste0(
      "EXTERNAL_DUPLICATED_GENE_ROWS: ",
      N_DUPLICATED_ROWS
    ),

    paste0(
      "EXTERNAL_IDENTIFIERS_WITH_MULTIPLE_OCCURRENCES: ",
      length(
        DUPLICATE_TABLE
      )
    ),

    "",

    paste0(
      "MAPPED_UNAMBIGUOUS_GENES: ",
      sum(
        MAPPING$mapped_unambiguous
      ),
      "/24"
    ),

    paste0(
      "GENE_COUNT_COVERAGE: ",
      signif(
        GENE_COUNT_COVERAGE,
        6
      )
    ),

    paste0(
      "ABS_WEIGHT_COVERAGE: ",
      signif(
        ABS_WEIGHT_COVERAGE,
        6
      )
    ),

    paste0(
      "POSITIVE_WEIGHT_COVERAGE: ",
      signif(
        POSITIVE_WEIGHT_COVERAGE,
        6
      )
    ),

    paste0(
      "NEGATIVE_WEIGHT_COVERAGE: ",
      signif(
        NEGATIVE_WEIGHT_COVERAGE,
        6
      )
    ),

    "",

    paste0(
      "DIAGNOSIS_LIKE_LABELS_VISIBLE_IN_SAMPLE_NAMES: ",
      LABEL_PATTERN_VISIBLE
    ),

    "",

    paste(
      "The public matrix contains duplicated gene identifiers.",
      "They were retained explicitly and were not collapsed,",
      "summed, averaged, synonym-rescued, or otherwise modified."
    ),

    "",

    paste(
      "A frozen score gene was considered technically mappable",
      "only when its exact frozen gene symbol occurred exactly",
      "once in the external matrix."
    ),

    "",

    paste(
      "No diagnosis statistic, group comparison, AUROC,",
      "external weight tuning, or score optimization was",
      "performed in this step."
    ),

    "",

    paste(
      "Complete phenotype blinding cannot be claimed because",
      "the public matrix sample names visibly contain",
      "diagnosis-like _C and _S suffixes."
    ),

    "",

    paste(
      "No documented direct donor overlap with the BrainSeq",
      "discovery cohort was identified in this source audit."
    )
  ),

  NOTE_OUT
)


# ------------------------------------------------------------
# 21. HASH OUTPUTS
# ------------------------------------------------------------

OUTPUT_FILES <- c(
  MAPPING_OUT,
  GATE_OUT,
  INDEPENDENCE_OUT,
  DUPLICATE_OUT,
  NOTE_OUT
)


OUTPUT_SHA <- vapply(
  OUTPUT_FILES,
  function(path) {

    digest::digest(
      file = path,
      algo = "sha256"
    )
  },
  character(1)
)


HASH_TABLE <- data.frame(

  file_name =
    basename(
      OUTPUT_FILES
    ),

  sha256 =
    unname(
      OUTPUT_SHA
    ),

  stringsAsFactors = FALSE
)


write.csv(
  HASH_TABLE,
  HASH_OUT,
  row.names = FALSE
)


# ------------------------------------------------------------
# 22. FINAL VALIDATION
# ------------------------------------------------------------

stopifnot(

  nrow(MAPPING) == 24L,

  ncol(COUNTS_MATRIX) == 38L,

  file.exists(
    MAPPING_OUT
  ),

  file.exists(
    GATE_OUT
  ),

  file.exists(
    INDEPENDENCE_OUT
  ),

  file.exists(
    DUPLICATE_OUT
  ),

  file.exists(
    NOTE_OUT
  ),

  file.exists(
    HASH_OUT
  )
)


# ------------------------------------------------------------
# 23. OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME


cat("\n============================================================\n")
cat("STEP 08B/08C/08D — GSE87194 TECHNICAL AUDIT COMPLETE\n")
cat("============================================================\n")


cat(
  "\n--- FROZEN GENE MAPPING ---\n"
)


print(
  MAPPING[
    ,
    c(
      "frozen_gene_id",
      "frozen_symbol",
      "frozen_weight",
      "external_match_count",
      "mapping_status"
    ),
    drop = FALSE
  ],
  row.names = FALSE
)


cat(
  "\n--- TECHNICAL GATE ---\n"
)


print(
  GATE_SUMMARY,
  row.names = FALSE,
  digits = 6
)


cat(
  "\nTECHNICAL_PASS: ",
  TECHNICAL_PASS,
  "\n",
  sep = ""
)


cat(
  "Diagnosis outcome analyzed: NO\n"
)

cat(
  "AUROC calculated: NO\n"
)

cat(
  "External tuning performed: NO\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)


if (
  !TECHNICAL_PASS
) {

  cat(
    "\nIMPORTANT: Technical gate FAILED.\n"
  )

  cat(
    "Do NOT proceed to GSE87194 score-outcome evaluation.\n"
  )

} else {

  cat(
    "\nTechnical gate PASSED.\n"
  )

  cat(
    "GSE87194 is eligible for frozen-score outcome evaluation.\n"
  )
}


cat("\n============================================================\n")
cat("STEP 08B/08C/08D — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Required files: PASS
Frozen weight columns:
 [1] "gene_index"                        "ensemblID_stable"                 
 [3] "gene_symbol"                       "gene_type"                        
 [5] "protein_coding"                    "direction_code"                   
 [7] "direction"                         "discovery_median_coefficient"     
 [9] "absolute_weight"                   "final_normalized_weight"          
[11] "normalized_absolute_weight"        "outer_selection_count"            
[13] "outer_selection_frequency"         "outer_sign_consistency"           
[15] "half_sample_selection_count"       "half_sample_selection_probability"
[17] "half_sample_sign_consistency"      "half_sample_median_nonzero_beta"  
[19] "null_selection_count"              "null_selection_probability"       
[21] "absolute_weight_rank"             
Gene ID column: ensemblID_stable
Symbol column: gene_symbol
Weight column: final_normalized_weight
Frozen 24-gene weight i

In [43]:
# ============================================================
# STEP 09B — GSE224683 PUBLIC RNA-SEQ ACQUISITION + SCHEMA AUDIT
#
# Public DLPFC cohort:
#   15 schizophrenia
#   15 controls
#
# This cell:
#   - downloads processed Salmon estimated counts
#   - downloads processed Salmon TPM
#   - downloads GEO family SOFT
#   - records SHA256
#   - inspects table structure / identifier format
#
# NO score calculation.
# NO diagnosis comparison.
# NO AUROC.
# NO external tuning.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop("digest package unavailable.")
}


# ------------------------------------------------------------
# 2. RAW DIRECTORY
# ------------------------------------------------------------

RAW_DIR <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse224683"
)

dir.create(
  RAW_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


# ------------------------------------------------------------
# 3. GEO PUBLIC URLS
# ------------------------------------------------------------

BASE_SUPPL <- paste0(
  "https://ftp.ncbi.nlm.nih.gov/geo/series/",
  "GSE224nnn/GSE224683/suppl/"
)

COUNTS_URL <- paste0(
  BASE_SUPPL,
  "GSE224683_salmon_uci30_sczctrl_nreads.csv.gz"
)

TPM_URL <- paste0(
  BASE_SUPPL,
  "GSE224683_salmon_uci30_sczctrl_tpms.csv.gz"
)

SOFT_URL <- paste0(
  "https://ftp.ncbi.nlm.nih.gov/geo/series/",
  "GSE224nnn/GSE224683/soft/",
  "GSE224683_family.soft.gz"
)


COUNTS_PATH <- file.path(
  RAW_DIR,
  "GSE224683_salmon_uci30_sczctrl_nreads.csv.gz"
)

TPM_PATH <- file.path(
  RAW_DIR,
  "GSE224683_salmon_uci30_sczctrl_tpms.csv.gz"
)

SOFT_PATH <- file.path(
  RAW_DIR,
  "GSE224683_family.soft.gz"
)


# ------------------------------------------------------------
# 4. DOWNLOAD HELPER
# ------------------------------------------------------------

download_if_needed <- function(
  url,
  destination
) {

  if (
    file.exists(destination) &&
    file.info(destination)$size > 0
  ) {

    cat(
      "Already present: ",
      basename(destination),
      "\n",
      sep = ""
    )

    return(
      invisible(TRUE)
    )
  }


  cat(
    "Downloading: ",
    basename(destination),
    "\n",
    sep = ""
  )


  download.file(
    url = url,
    destfile = destination,
    mode = "wb",
    method = "libcurl",
    quiet = FALSE
  )


  if (
    !file.exists(destination) ||
    file.info(destination)$size <= 0
  ) {
    stop(
      paste0(
        "Download failed: ",
        basename(destination)
      )
    )
  }


  invisible(TRUE)
}


# ------------------------------------------------------------
# 5. DOWNLOAD
# ------------------------------------------------------------

download_if_needed(
  COUNTS_URL,
  COUNTS_PATH
)

download_if_needed(
  TPM_URL,
  TPM_PATH
)

download_if_needed(
  SOFT_URL,
  SOFT_PATH
)


# ------------------------------------------------------------
# 6. FILE INVENTORY + SHA256
# ------------------------------------------------------------

FILE_PATHS <- c(
  COUNTS_PATH,
  TPM_PATH,
  SOFT_PATH
)

FILE_INFO <- file.info(
  FILE_PATHS
)

FILE_SHA <- vapply(
  FILE_PATHS,
  function(path) {

    digest::digest(
      file = path,
      algo = "sha256"
    )
  },
  character(1)
)


INVENTORY <- data.frame(

  file_name =
    basename(
      FILE_PATHS
    ),

  size_MB =
    round(
      as.numeric(
        FILE_INFO$size
      ) / 1024^2,
      3
    ),

  sha256 =
    unname(
      FILE_SHA
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 7. READ ONLY FIRST FEW LINES
# ------------------------------------------------------------

read_gz_head <- function(
  path,
  n = 6L
) {

  con <- gzfile(
    path,
    open = "rt"
  )

  on.exit(
    close(con)
  )

  readLines(
    con,
    n = n,
    warn = FALSE
  )
}


COUNTS_HEAD <- read_gz_head(
  COUNTS_PATH,
  6L
)

TPM_HEAD <- read_gz_head(
  TPM_PATH,
  6L
)

SOFT_HEAD <- read_gz_head(
  SOFT_PATH,
  20L
)


# ------------------------------------------------------------
# 8. LIGHTWEIGHT TABLE SCHEMA
#
# Read only first 5 data rows.
# ------------------------------------------------------------

COUNTS_PREVIEW <- read.csv(
  COUNTS_PATH,
  nrows = 5,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

TPM_PREVIEW <- read.csv(
  TPM_PATH,
  nrows = 5,
  stringsAsFactors = FALSE,
  check.names = FALSE
)


cat(
  "Counts preview dimensions: ",
  nrow(COUNTS_PREVIEW),
  " x ",
  ncol(COUNTS_PREVIEW),
  "\n",
  sep = ""
)

cat(
  "TPM preview dimensions: ",
  nrow(TPM_PREVIEW),
  " x ",
  ncol(TPM_PREVIEW),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 9. IDENTIFIER FORMAT AUDIT
# ------------------------------------------------------------

FIRST_COL_NAME_COUNTS <- names(
  COUNTS_PREVIEW
)[1]

FIRST_COL_NAME_TPM <- names(
  TPM_PREVIEW
)[1]


FIRST_IDS_COUNTS <- as.character(
  COUNTS_PREVIEW[
    ,
    1,
    drop = TRUE
  ]
)

FIRST_IDS_TPM <- as.character(
  TPM_PREVIEW[
    ,
    1,
    drop = TRUE
  ]
)


detect_id_type <- function(x) {

  x <- as.character(x)

  if (
    all(
      grepl(
        "^ENSG[0-9]+(\\.[0-9]+)?$",
        x
      )
    )
  ) {
    return(
      "ENSEMBL_GENE"
    )
  }

  if (
    all(
      grepl(
        "^ENST[0-9]+(\\.[0-9]+)?$",
        x
      )
    )
  ) {
    return(
      "ENSEMBL_TRANSCRIPT"
    )
  }

  if (
    all(
      grepl(
        "^[A-Za-z0-9._-]+$",
        x
      )
    )
  ) {
    return(
      "OTHER_IDENTIFIER"
    )
  }

  "MIXED_OR_UNKNOWN"
}


COUNTS_ID_TYPE <- detect_id_type(
  FIRST_IDS_COUNTS
)

TPM_ID_TYPE <- detect_id_type(
  FIRST_IDS_TPM
)


# ------------------------------------------------------------
# 10. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse224683"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


INVENTORY_OUT <- file.path(
  OUT_DIR,
  "STEP09B_GSE224683_source_inventory_v1.csv"
)


AUDIT_OUT <- file.path(
  OUT_DIR,
  "STEP09B_GSE224683_schema_audit_v1.txt"
)


write.csv(
  INVENTORY,
  INVENTORY_OUT,
  row.names = FALSE
)


writeLines(

  c(

    "STEP 09B — GSE224683 SOURCE + SCHEMA AUDIT",

    "",

    paste0(
      "COUNTS_FIRST_COLUMN: ",
      FIRST_COL_NAME_COUNTS
    ),

    paste0(
      "COUNTS_IDENTIFIER_TYPE: ",
      COUNTS_ID_TYPE
    ),

    paste0(
      "TPM_FIRST_COLUMN: ",
      FIRST_COL_NAME_TPM
    ),

    paste0(
      "TPM_IDENTIFIER_TYPE: ",
      TPM_ID_TYPE
    ),

    "",

    "EXPECTED_COHORT: 15 schizophrenia + 15 control DLPFC",

    "SCZ_MSS_SCORE_CALCULATED: NO",

    "AUROC_CALCULATED: NO",

    "EXTERNAL_TUNING: NO"
  ),

  AUDIT_OUT
)


# ------------------------------------------------------------
# 11. FINAL OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME


cat("\n============================================================\n")
cat("STEP 09B — GSE224683 ACQUISITION + SCHEMA AUDIT COMPLETE\n")
cat("============================================================\n")


cat(
  "\n--- FILE INVENTORY ---\n"
)

print(
  INVENTORY,
  row.names = FALSE
)


cat(
  "\n--- COUNTS COLUMNS ---\n"
)

print(
  names(
    COUNTS_PREVIEW
  )
)


cat(
  "\nCounts first-column name: ",
  FIRST_COL_NAME_COUNTS,
  "\n",
  sep = ""
)

cat(
  "Counts identifier type: ",
  COUNTS_ID_TYPE,
  "\n",
  sep = ""
)


cat(
  "\nFirst counts identifiers:\n"
)

print(
  FIRST_IDS_COUNTS
)


cat(
  "\n--- TPM COLUMNS ---\n"
)

print(
  names(
    TPM_PREVIEW
  )
)


cat(
  "\nTPM first-column name: ",
  FIRST_COL_NAME_TPM,
  "\n",
  sep = ""
)

cat(
  "TPM identifier type: ",
  TPM_ID_TYPE,
  "\n",
  sep = ""
)


cat(
  "\nFirst TPM identifiers:\n"
)

print(
  FIRST_IDS_TPM
)


cat(
  "\nSCZ-MSS score calculated: NO\n"
)

cat(
  "Diagnosis comparison performed: NO\n"
)

cat(
  "External tuning performed: NO\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 09B — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Downloading: GSE224683_salmon_uci30_sczctrl_nreads.csv.gz
Downloading: GSE224683_salmon_uci30_sczctrl_tpms.csv.gz
Downloading: GSE224683_family.soft.gz
Counts preview dimensions: 5 x 31
TPM preview dimensions: 5 x 31

STEP 09B — GSE224683 ACQUISITION + SCHEMA AUDIT COMPLETE

--- FILE INVENTORY ---
                                    file_name size_MB
 GSE224683_salmon_uci30_sczctrl_nreads.csv.gz  14.570
   GSE224683_salmon_uci30_sczctrl_tpms.csv.gz  16.879
                     GSE224683_family.soft.gz   0.005
                                                           sha256
 f4a5adbf71d4285d5d82e0a22930bf7cf8776cd5d71a6dd948eacc65b32c7a03
 0e9660140d9ae315191f20d25ef06fb24aa79b42b3b135d3a8fc6b5d556bcf90
 2fb4bab99cee1a886a78831f24e25831fb860033fc84453696fcffd636a78caa

--- COUNTS COLUMNS ---
 [1] ""         "Sample1"  "Sample2"  "Sample3"  "Sample4"  "Sample5" 
 [7] "Sample6"  "Sample7"  "Sample8"  "Sample9"  "Sample10" "Sample11"
[13] "Sample12" "Sample13" "Samp

In [44]:
# ============================================================
# STEP 09C — GSE224683 FROZEN 24-GENE TECHNICAL GATE
#
# Uses:
#   GSE224683 transcript-level TPM
#   embedded Ensembl gene IDs
#   frozen SCZ-MSS v1 weights
#
# NO score calculation.
# NO diagnosis comparison.
# NO AUROC.
# NO tuning.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

# ------------------------------------------------------------
# 0. SENTINEL
# ------------------------------------------------------------

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUTS
# ------------------------------------------------------------

TPM_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse224683",
  "GSE224683_salmon_uci30_sczctrl_tpms.csv.gz"
)

WEIGHTS_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_gene_weights.csv"
)

MAPPING_POLICY_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_mapping_policy.json"
)

GATE_POLICY_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_technical_gate.json"
)

REQUIRED <- c(
  TPM_PATH,
  WEIGHTS_PATH,
  MAPPING_POLICY_PATH,
  GATE_POLICY_PATH
)

if (!all(file.exists(REQUIRED))) {
  stop("Required input missing.")
}

cat("Required files: PASS\n")


# ------------------------------------------------------------
# 2. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {
  stop("digest package unavailable.")
}


# ------------------------------------------------------------
# 3. LOAD FROZEN WEIGHTS
# ------------------------------------------------------------

WEIGHTS <- read.csv(
  WEIGHTS_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(WEIGHTS) == 24L,
  all(
    c(
      "ensemblID_stable",
      "gene_symbol",
      "final_normalized_weight"
    ) %in% names(WEIGHTS)
  )
)

FROZEN_ID <- trimws(
  as.character(
    WEIGHTS$ensemblID_stable
  )
)

FROZEN_SYMBOL <- trimws(
  as.character(
    WEIGHTS$gene_symbol
  )
)

FROZEN_SYMBOL[
  FROZEN_SYMBOL == ""
] <- NA_character_

FROZEN_WEIGHT <- as.numeric(
  WEIGHTS$final_normalized_weight
)

stopifnot(
  !anyNA(FROZEN_ID),
  !anyNA(FROZEN_WEIGHT),
  abs(
    sum(abs(FROZEN_WEIGHT)) - 1
  ) < 1e-6
)

cat("Frozen 24-gene score integrity: PASS\n")


# ------------------------------------------------------------
# 4. POLICY HASHES
# ------------------------------------------------------------

cat(
  "Frozen mapping policy SHA256: ",
  digest::digest(
    file = MAPPING_POLICY_PATH,
    algo = "sha256"
  ),
  "\n",
  sep = ""
)

cat(
  "Frozen gate policy SHA256: ",
  digest::digest(
    file = GATE_POLICY_PATH,
    algo = "sha256"
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 5. LOAD TPM TABLE
# ------------------------------------------------------------

TPM_DF <- read.csv(
  TPM_PATH,
  header = TRUE,
  check.names = FALSE,
  stringsAsFactors = FALSE
)

if (ncol(TPM_DF) != 31L) {
  stop(
    paste0(
      "Expected annotation + 30 samples; found ",
      ncol(TPM_DF),
      " columns."
    )
  )
}

ANNOTATION <- as.character(
  TPM_DF[, 1]
)

TPM_MATRIX <- as.matrix(
  TPM_DF[, -1, drop = FALSE]
)

storage.mode(TPM_MATRIX) <- "numeric"

if (
  anyNA(TPM_MATRIX) ||
  any(!is.finite(TPM_MATRIX)) ||
  any(TPM_MATRIX < 0)
) {
  stop("Invalid TPM matrix.")
}

cat(
  "GSE224683 TPM matrix: ",
  nrow(TPM_MATRIX),
  " transcript rows x ",
  ncol(TPM_MATRIX),
  " samples\n",
  sep = ""
)


# ------------------------------------------------------------
# 6. PARSE ANNOTATION
#
# Format:
# ENST...|ENSG...|...|transcript_name|gene_symbol|length|type|
# ------------------------------------------------------------

PARTS <- strsplit(
  ANNOTATION,
  "|",
  fixed = TRUE
)

if (any(lengths(PARTS) < 2L)) {
  stop("Malformed annotation string detected.")
}

EXTERNAL_TRANSCRIPT_RAW <- vapply(
  PARTS,
  function(x) x[1],
  character(1)
)

EXTERNAL_GENE_RAW <- vapply(
  PARTS,
  function(x) x[2],
  character(1)
)

EXTERNAL_GENE_STABLE <- sub(
  "\\.[0-9]+$",
  "",
  EXTERNAL_GENE_RAW
)

VALID_GENE <- grepl(
  "^ENSG[0-9]+$",
  EXTERNAL_GENE_STABLE
)

cat(
  "Valid parsed Ensembl gene IDs: ",
  sum(VALID_GENE),
  "/",
  length(VALID_GENE),
  "\n",
  sep = ""
)

if (mean(VALID_GENE) < 0.99) {
  stop("Unexpected annotation parsing failure.")
}


# ------------------------------------------------------------
# 7. GENE REPRESENTATION
# ------------------------------------------------------------

GENE_TRANSCRIPT_COUNT <- table(
  EXTERNAL_GENE_STABLE[
    VALID_GENE
  ]
)

cat(
  "Unique stable Ensembl genes represented: ",
  length(GENE_TRANSCRIPT_COUNT),
  "\n",
  sep = ""
)

cat(
  "Genes represented by >1 transcript: ",
  sum(
    GENE_TRANSCRIPT_COUNT > 1L
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 8. MAP FROZEN GENES
# ------------------------------------------------------------

N_TRANSCRIPTS <- as.integer(
  GENE_TRANSCRIPT_COUNT[
    FROZEN_ID
  ]
)

N_TRANSCRIPTS[
  is.na(N_TRANSCRIPTS)
] <- 0L

MAPPED <- (
  N_TRANSCRIPTS >= 1L
)

MAPPING <- data.frame(
  frozen_gene_id =
    FROZEN_ID,
  frozen_symbol =
    FROZEN_SYMBOL,
  frozen_weight =
    FROZEN_WEIGHT,
  direction =
    ifelse(
      FROZEN_WEIGHT > 0,
      "positive",
      "negative"
    ),
  external_transcript_count =
    N_TRANSCRIPTS,
  mapped_gene =
    MAPPED,
  mapping_status =
    ifelse(
      MAPPED,
      "EXPLICIT_ENSEMBL_GENE_ID_PRESENT",
      "NOT_FOUND"
    ),
  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 9. COVERAGE METRICS
# ------------------------------------------------------------

GENE_COUNT_COVERAGE <- mean(
  MAPPING$mapped_gene
)

ABS_WEIGHT_COVERAGE <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$mapped_gene
    ]
  )
)

TOTAL_POSITIVE_WEIGHT <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight > 0
    ]
  )
)

TOTAL_NEGATIVE_WEIGHT <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight < 0
    ]
  )
)

RETAINED_POSITIVE_WEIGHT <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight > 0 &
        MAPPING$mapped_gene
    ]
  )
)

RETAINED_NEGATIVE_WEIGHT <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight < 0 &
        MAPPING$mapped_gene
    ]
  )
)

POSITIVE_WEIGHT_COVERAGE <- (
  RETAINED_POSITIVE_WEIGHT /
    TOTAL_POSITIVE_WEIGHT
)

NEGATIVE_WEIGHT_COVERAGE <- (
  RETAINED_NEGATIVE_WEIGHT /
    TOTAL_NEGATIVE_WEIGHT
)


# ------------------------------------------------------------
# 10. SAMPLE COMPLETENESS
# ------------------------------------------------------------

MAPPED_IDS <- MAPPING$frozen_gene_id[
  MAPPING$mapped_gene
]

MAPPED_TRANSCRIPT_ROWS <- (
  EXTERNAL_GENE_STABLE %in%
    MAPPED_IDS
)

if (sum(MAPPED_TRANSCRIPT_ROWS) == 0L) {

  ACTUAL_SAMPLE_COMPLETENESS <- 0

} else {

  ACTUAL_SAMPLE_COMPLETENESS <- min(
    colMeans(
      is.finite(
        TPM_MATRIX[
          MAPPED_TRANSCRIPT_ROWS,
          ,
          drop = FALSE
        ]
      )
    )
  )
}


# ------------------------------------------------------------
# 11. FROZEN GATE
# ------------------------------------------------------------

REQ_GENE <- 0.80
REQ_ABS <- 0.80
REQ_POS <- 0.70
REQ_NEG <- 0.70
REQ_COMPLETE <- 1.00

PASS_GENE <- (
  GENE_COUNT_COVERAGE >= REQ_GENE
)

PASS_ABS <- (
  ABS_WEIGHT_COVERAGE >= REQ_ABS
)

PASS_POS <- (
  POSITIVE_WEIGHT_COVERAGE >= REQ_POS
)

PASS_NEG <- (
  NEGATIVE_WEIGHT_COVERAGE >= REQ_NEG
)

PASS_COMPLETE <- (
  ACTUAL_SAMPLE_COMPLETENESS >= REQ_COMPLETE
)

TECHNICAL_PASS <- all(
  PASS_GENE,
  PASS_ABS,
  PASS_POS,
  PASS_NEG,
  PASS_COMPLETE
)


# ------------------------------------------------------------
# 12. SUMMARY
# ------------------------------------------------------------

GATE_SUMMARY <- data.frame(
  metric = c(
    "Frozen_score_gene_count",
    "Mapped_gene_count",
    "Missing_gene_count",
    "Gene_count_coverage",
    "Required_gene_count_coverage",
    "Absolute_weight_coverage",
    "Required_absolute_weight_coverage",
    "Positive_weight_coverage",
    "Required_positive_weight_coverage",
    "Negative_weight_coverage",
    "Required_negative_weight_coverage",
    "Minimum_sample_completeness",
    "Required_sample_completeness",
    "Technical_PASS"
  ),
  value = c(
    24,
    sum(MAPPED),
    sum(!MAPPED),
    GENE_COUNT_COVERAGE,
    REQ_GENE,
    ABS_WEIGHT_COVERAGE,
    REQ_ABS,
    POSITIVE_WEIGHT_COVERAGE,
    REQ_POS,
    NEGATIVE_WEIGHT_COVERAGE,
    REQ_NEG,
    ACTUAL_SAMPLE_COMPLETENESS,
    REQ_COMPLETE,
    as.numeric(TECHNICAL_PASS)
  ),
  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse224683"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

MAPPING_OUT <- file.path(
  OUT_DIR,
  "STEP09C_GSE224683_frozen_gene_mapping_v1.csv"
)

GATE_OUT <- file.path(
  OUT_DIR,
  "STEP09C_GSE224683_technical_gate_v1.csv"
)

NOTE_OUT <- file.path(
  OUT_DIR,
  "STEP09C_GSE224683_TECHNICAL_GATE_NOTES_v1.txt"
)

write.csv(
  MAPPING,
  MAPPING_OUT,
  row.names = FALSE
)

write.csv(
  GATE_SUMMARY,
  GATE_OUT,
  row.names = FALSE
)

writeLines(
  c(
    "STEP 09C — GSE224683 TECHNICAL GATE",
    "",
    paste0(
      "TECHNICAL_PASS: ",
      TECHNICAL_PASS
    ),
    "",
    paste0(
      "MAPPED_GENES: ",
      sum(MAPPED),
      "/24"
    ),
    paste0(
      "GENE_COUNT_COVERAGE: ",
      signif(
        GENE_COUNT_COVERAGE,
        6
      )
    ),
    paste0(
      "ABS_WEIGHT_COVERAGE: ",
      signif(
        ABS_WEIGHT_COVERAGE,
        6
      )
    ),
    paste0(
      "POSITIVE_WEIGHT_COVERAGE: ",
      signif(
        POSITIVE_WEIGHT_COVERAGE,
        6
      )
    ),
    paste0(
      "NEGATIVE_WEIGHT_COVERAGE: ",
      signif(
        NEGATIVE_WEIGHT_COVERAGE,
        6
      )
    ),
    "",
    paste(
      "Transcript annotation provided explicit Ensembl",
      "gene identifiers; terminal version suffixes only",
      "were removed."
    ),
    "",
    paste(
      "No SCZ-MSS score, diagnosis comparison, AUROC,",
      "gene-weight tuning, threshold tuning, or outcome-driven",
      "mapping was performed."
    )
  ),
  NOTE_OUT
)


# ------------------------------------------------------------
# 14. OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME

cat("\n============================================================\n")
cat("STEP 09C — GSE224683 TECHNICAL GATE COMPLETE\n")
cat("============================================================\n")

cat(
  "\n--- FROZEN GENE MAPPING ---\n"
)

print(
  MAPPING[
    ,
    c(
      "frozen_gene_id",
      "frozen_symbol",
      "frozen_weight",
      "external_transcript_count",
      "mapping_status"
    ),
    drop = FALSE
  ],
  row.names = FALSE
)

cat(
  "\n--- TECHNICAL GATE ---\n"
)

print(
  GATE_SUMMARY,
  row.names = FALSE,
  digits = 6
)

cat(
  "\nTECHNICAL_PASS: ",
  TECHNICAL_PASS,
  "\n",
  sep = ""
)

cat(
  "SCZ-MSS score calculated: NO\n"
)

cat(
  "Diagnosis comparison performed: NO\n"
)

cat(
  "AUROC calculated: NO\n"
)

cat(
  "External tuning performed: NO\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)

if (TECHNICAL_PASS) {

  cat(
    "\nGSE224683 TECHNICAL GATE PASSED.\n"
  )

  cat(
    "Frozen-score outcome evaluation is permitted next.\n"
  )

} else {

  cat(
    "\nGSE224683 TECHNICAL GATE FAILED.\n"
  )

  cat(
    "Do NOT calculate SCZ-MSS diagnosis performance.\n"
  )
}

cat("\n============================================================\n")
cat("STEP 09C — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Required files: PASS
Frozen 24-gene score integrity: PASS
Frozen mapping policy SHA256: c5b6f47869246d0a0ed2ed8f6422c0207d21aff994f36602d53e458157e49d75
Frozen gate policy SHA256: 3eb578e2d0b3b805502c96c4fbff7319a37b4fdcd4da9d1184bc32708adfff1d
GSE224683 TPM matrix: 226608 transcript rows x 30 samples
Valid parsed Ensembl gene IDs: 226608/226608
Unique stable Ensembl genes represented: 60179
Genes represented by >1 transcript: 23841

STEP 09C — GSE224683 TECHNICAL GATE COMPLETE

--- FROZEN GENE MAPPING ---
  frozen_gene_id frozen_symbol frozen_weight external_transcript_count
 ENSG00000100092        SH3BP1  -0.040294344                        10
 ENSG00000117877        CD3EAP  -0.048996635                         4
 ENSG00000196565          HBG2   0.027853367                         3
 ENSG00000251521       IMPA1P1   0.069213048                         2
 ENSG00000260526          <NA>   0.039334477                         1
 ENSG00000166428          PLD4  -0.0061

In [45]:
# ============================================================
# STEP 09D — GSE224683 FROZEN SCZ-MSS EXTERNAL VALIDATION
#
# Technical gate already PASSED:
#   23/24 genes
#   gene coverage       = 95.83%
#   abs-weight coverage = 92.66%
#   positive coverage   = 100%
#   negative coverage   = 88.03%
#
# IMPORTANT ORDER INSIDE THIS CELL:
#   1. Load frozen weights
#   2. Build gene-level TPM
#   3. Calculate frozen rank score
#   4. ONLY THEN read diagnosis metadata from GEO SOFT
#   5. Evaluate SCZ vs Control
#
# NO retraining.
# NO weight tuning.
# NO threshold tuning.
# NO gene rescue.
# NO feature selection.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUT FILES
# ------------------------------------------------------------

TPM_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse224683",
  "GSE224683_salmon_uci30_sczctrl_tpms.csv.gz"
)

SOFT_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse224683",
  "GSE224683_family.soft.gz"
)

WEIGHTS_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_gene_weights.csv"
)

MAPPING_PATH <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse224683",
  "STEP09C_GSE224683_frozen_gene_mapping_v1.csv"
)

GATE_PATH <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse224683",
  "STEP09C_GSE224683_technical_gate_v1.csv"
)

REQUIRED <- c(
  TPM_PATH,
  SOFT_PATH,
  WEIGHTS_PATH,
  MAPPING_PATH,
  GATE_PATH
)

if (!all(file.exists(REQUIRED))) {

  print(
    REQUIRED[
      !file.exists(REQUIRED)
    ]
  )

  stop("Required file missing.")
}

cat("Required files: PASS\n")


# ------------------------------------------------------------
# 2. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop("digest package unavailable.")
}


# ------------------------------------------------------------
# 3. VERIFY PREVIOUS TECHNICAL GATE
# ------------------------------------------------------------

GATE <- read.csv(
  GATE_PATH,
  stringsAsFactors = FALSE
)

GATE_PASS_ROW <- which(
  GATE$metric == "Technical_PASS"
)

if (length(GATE_PASS_ROW) != 1L) {
  stop("Technical_PASS row not uniquely identified.")
}

TECHNICAL_PASS <- as.numeric(
  GATE$value[
    GATE_PASS_ROW
  ]
)

if (!identical(
  TECHNICAL_PASS,
  1
)) {
  stop(
    "GSE224683 technical gate did not pass. Outcome evaluation prohibited."
  )
}

cat("Previous technical gate: PASS\n")


# ------------------------------------------------------------
# 4. LOAD FROZEN WEIGHTS
# ------------------------------------------------------------

WEIGHTS <- read.csv(
  WEIGHTS_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

if (nrow(WEIGHTS) != 24L) {
  stop("Expected exactly 24 frozen genes.")
}

REQUIRED_WEIGHT_COLS <- c(
  "ensemblID_stable",
  "gene_symbol",
  "final_normalized_weight"
)

if (!all(
  REQUIRED_WEIGHT_COLS %in%
    names(WEIGHTS)
)) {
  stop("Frozen weight schema mismatch.")
}


FROZEN_ID <- trimws(
  as.character(
    WEIGHTS$ensemblID_stable
  )
)

FROZEN_SYMBOL <- trimws(
  as.character(
    WEIGHTS$gene_symbol
  )
)

FROZEN_SYMBOL[
  FROZEN_SYMBOL == ""
] <- NA_character_

FROZEN_WEIGHT <- as.numeric(
  WEIGHTS$final_normalized_weight
)


if (
  anyNA(FROZEN_ID) ||
  anyNA(FROZEN_WEIGHT) ||
  abs(
    sum(abs(FROZEN_WEIGHT)) - 1
  ) > 1e-6
) {
  stop("Frozen score integrity failure.")
}

cat("Frozen score integrity: PASS\n")


# ------------------------------------------------------------
# 5. LOAD STEP09C MAPPING
# ------------------------------------------------------------

MAPPING <- read.csv(
  MAPPING_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

if (nrow(MAPPING) != 24L) {
  stop("STEP09C mapping must contain 24 frozen genes.")
}

AVAILABLE_ID <- as.character(
  MAPPING$frozen_gene_id[
    MAPPING$mapped_gene
  ]
)

if (length(AVAILABLE_ID) != 23L) {
  stop(
    paste0(
      "Expected 23 technically available genes; found ",
      length(AVAILABLE_ID)
    )
  )
}

cat(
  "Frozen genes available for score: ",
  length(AVAILABLE_ID),
  "/24\n",
  sep = ""
)


# ------------------------------------------------------------
# 6. LOAD TPM
#
# Diagnosis metadata has NOT been read yet.
# ------------------------------------------------------------

TPM_DF <- read.csv(
  TPM_PATH,
  header = TRUE,
  check.names = FALSE,
  stringsAsFactors = FALSE
)

if (ncol(TPM_DF) != 31L) {
  stop(
    paste0(
      "Expected annotation + 30 samples; found ",
      ncol(TPM_DF),
      " columns."
    )
  )
}


ANNOTATION <- as.character(
  TPM_DF[, 1]
)

TPM_MATRIX <- as.matrix(
  TPM_DF[, -1, drop = FALSE]
)

storage.mode(
  TPM_MATRIX
) <- "numeric"


if (
  anyNA(TPM_MATRIX) ||
  any(!is.finite(TPM_MATRIX)) ||
  any(TPM_MATRIX < 0)
) {
  stop("Invalid TPM matrix.")
}


SAMPLE_NAMES <- colnames(
  TPM_MATRIX
)

if (
  length(SAMPLE_NAMES) != 30L ||
  anyDuplicated(SAMPLE_NAMES)
) {
  stop("Unexpected sample-column structure.")
}


cat(
  "TPM matrix: ",
  nrow(TPM_MATRIX),
  " transcript rows x ",
  ncol(TPM_MATRIX),
  " samples\n",
  sep = ""
)


# ------------------------------------------------------------
# 7. PARSE EMBEDDED ENSEMBL GENE IDs
# ------------------------------------------------------------

PARTS <- strsplit(
  ANNOTATION,
  "|",
  fixed = TRUE
)

if (any(lengths(PARTS) < 2L)) {
  stop("Malformed transcript annotation.")
}


EXTERNAL_GENE_RAW <- vapply(
  PARTS,
  function(x) x[2],
  character(1)
)


EXTERNAL_GENE_STABLE <- sub(
  "\\.[0-9]+$",
  "",
  EXTERNAL_GENE_RAW
)


VALID_GENE <- grepl(
  "^ENSG[0-9]+$",
  EXTERNAL_GENE_STABLE
)


if (mean(VALID_GENE) < 0.99) {
  stop("Unexpected Ensembl parsing failure.")
}


# ------------------------------------------------------------
# 8. KEEP ONLY FROZEN AVAILABLE GENES
# ------------------------------------------------------------

KEEP <- (
  VALID_GENE &
  EXTERNAL_GENE_STABLE %in%
    AVAILABLE_ID
)


if (!any(KEEP)) {
  stop("No frozen genes found in TPM matrix.")
}


FROZEN_TRANSCRIPT_TPM <- TPM_MATRIX[
  KEEP,
  ,
  drop = FALSE
]

FROZEN_TRANSCRIPT_GENE <- EXTERNAL_GENE_STABLE[
  KEEP
]


# ------------------------------------------------------------
# 9. TRANSCRIPT -> GENE TPM AGGREGATION
#
# Salmon TPMs for transcripts assigned to the same explicit
# Ensembl gene are summed to form gene-level TPM.
#
# This aggregation is independent of diagnosis.
# ------------------------------------------------------------

GENE_TPM <- rowsum(
  FROZEN_TRANSCRIPT_TPM,
  group = FROZEN_TRANSCRIPT_GENE,
  reorder = FALSE
)


# ------------------------------------------------------------
# 10. FORCE EXACT FROZEN-GENE ORDER
# ------------------------------------------------------------

AVAILABLE_ORDER <- FROZEN_ID[
  FROZEN_ID %in%
    AVAILABLE_ID
]


MISSING_AFTER_AGGREGATION <- setdiff(
  AVAILABLE_ORDER,
  rownames(GENE_TPM)
)


if (length(MISSING_AFTER_AGGREGATION) > 0L) {

  print(
    MISSING_AFTER_AGGREGATION
  )

  stop(
    "Technically mapped frozen gene missing after aggregation."
  )
}


GENE_TPM <- GENE_TPM[
  AVAILABLE_ORDER,
  ,
  drop = FALSE
]


if (nrow(GENE_TPM) != 23L) {
  stop(
    paste0(
      "Expected 23 gene-level rows; found ",
      nrow(GENE_TPM)
    )
  )
}


# ------------------------------------------------------------
# 11. MATCH FROZEN WEIGHTS
# ------------------------------------------------------------

WEIGHT_INDEX <- match(
  rownames(GENE_TPM),
  FROZEN_ID
)

AVAILABLE_WEIGHT <- FROZEN_WEIGHT[
  WEIGHT_INDEX
]

AVAILABLE_SYMBOL <- FROZEN_SYMBOL[
  WEIGHT_INDEX
]


RETAINED_ABS_WEIGHT <- sum(
  abs(
    AVAILABLE_WEIGHT
  )
)


cat(
  "Retained absolute frozen weight: ",
  round(
    RETAINED_ABS_WEIGHT,
    6
  ),
  "\n",
  sep = ""
)


if (
  abs(
    RETAINED_ABS_WEIGHT - 0.926599725
  ) > 1e-5
) {

  warning(
    paste0(
      "Retained weight differs slightly from expected Step09C value: ",
      RETAINED_ABS_WEIGHT
    )
  )
}


# ------------------------------------------------------------
# 12. AUDIT EXPRESSION AVAILABILITY
# ------------------------------------------------------------

ALL_ZERO_GENE <- rowSums(
  GENE_TPM
) == 0

cat(
  "Frozen available genes with TPM=0 in all 30 samples: ",
  sum(ALL_ZERO_GENE),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 13. FROZEN WITHIN-SAMPLE RANK SCORE
#
# For m available genes:
#
# centered_rank =
#   (rank - (m + 1)/2) / (m - 1)
#
# SCZ_MSS =
#   sum(weight * centered_rank) /
#   sum(abs(weight_available))
#
# ties.method = "average"
# ------------------------------------------------------------

M <- nrow(
  GENE_TPM
)

if (M < 2L) {
  stop("Too few genes for rank score.")
}


SCZ_MSS <- apply(
  GENE_TPM,
  2,
  function(x) {

    R <- rank(
      x,
      ties.method = "average"
    )

    CENTERED_RANK <- (
      R - (M + 1) / 2
    ) / (
      M - 1
    )

    sum(
      AVAILABLE_WEIGHT *
        CENTERED_RANK
    ) /
      RETAINED_ABS_WEIGHT
  }
)


if (
  anyNA(SCZ_MSS) ||
  any(!is.finite(SCZ_MSS))
) {
  stop("Invalid SCZ-MSS score generated.")
}


if (
  any(abs(SCZ_MSS) > 0.500001)
) {
  stop("Rank-score range integrity failure.")
}


# ------------------------------------------------------------
# 14. FREEZE SCORE BEFORE DIAGNOSIS READ
# ------------------------------------------------------------

SCORE_PRE_PHENOTYPE <- data.frame(

  sample_name =
    names(SCZ_MSS),

  SCZ_MSS =
    as.numeric(SCZ_MSS),

  stringsAsFactors = FALSE
)


SCORE_HASH_PRE_PHENOTYPE <- digest::digest(
  SCORE_PRE_PHENOTYPE,
  algo = "sha256"
)


cat(
  "Frozen external score hash BEFORE phenotype read: ",
  SCORE_HASH_PRE_PHENOTYPE,
  "\n",
  sep = ""
)

cat(
  "Diagnosis metadata loaded so far: NO\n"
)


# ============================================================
# 15. NOW LOAD GEO SOFT PHENOTYPE METADATA
# ============================================================

SOFT_CONNECTION <- gzfile(
  SOFT_PATH,
  open = "rt"
)

SOFT_LINES <- readLines(
  SOFT_CONNECTION,
  warn = FALSE
)

close(
  SOFT_CONNECTION
)


SAMPLE_STARTS <- grep(
  "^\\^SAMPLE = ",
  SOFT_LINES
)


if (length(SAMPLE_STARTS) != 30L) {
  stop(
    paste0(
      "Expected 30 GEO sample blocks; found ",
      length(SAMPLE_STARTS)
    )
  )
}


SAMPLE_ENDS <- c(
  SAMPLE_STARTS[-1] - 1L,
  length(SOFT_LINES)
)


get_soft_field <- function(
  block,
  prefix
) {

  HIT <- block[
    startsWith(
      block,
      prefix
    )
  ]

  if (length(HIT) == 0L) {
    return(NA_character_)
  }

  trimws(
    substring(
      HIT[1],
      nchar(prefix) + 1L
    )
  )
}


get_characteristic <- function(
  block,
  requested_key
) {

  CH <- block[
    startsWith(
      block,
      "!Sample_characteristics_ch1 = "
    )
  ]

  if (length(CH) == 0L) {
    return(NA_character_)
  }


  CH <- trimws(
    sub(
      "^!Sample_characteristics_ch1 = ",
      "",
      CH
    )
  )


  KEYS <- trimws(
    sub(
      ":.*$",
      "",
      CH
    )
  )


  HIT <- which(
    tolower(KEYS) ==
      tolower(requested_key)
  )


  if (length(HIT) == 0L) {
    return(NA_character_)
  }


  trimws(
    sub(
      "^[^:]+:\\s*",
      "",
      CH[
        HIT[1]
      ]
    )
  )
}


safe_numeric <- function(x) {

  suppressWarnings(
    as.numeric(
      trimws(x)
    )
  )
}


META_LIST <- vector(
  "list",
  length(SAMPLE_STARTS)
)


for (
  i in seq_along(
    SAMPLE_STARTS
  )
) {

  BLOCK <- SOFT_LINES[
    SAMPLE_STARTS[i]:
      SAMPLE_ENDS[i]
  ]


  GSM <- sub(
    "^\\^SAMPLE =\\s*",
    "",
    BLOCK[
      grep(
        "^\\^SAMPLE = ",
        BLOCK
      )[1]
    ]
  )


  TITLE <- get_soft_field(
    BLOCK,
    "!Sample_title = "
  )


  DESCRIPTION <- get_soft_field(
    BLOCK,
    "!Sample_description = "
  )


  SAMPLE_NAME <- NA_character_


  if (
    !is.na(DESCRIPTION) &&
    grepl(
      "^Sample[0-9]+$",
      DESCRIPTION
    )
  ) {

    SAMPLE_NAME <- DESCRIPTION

  } else if (
    !is.na(TITLE) &&
    grepl(
      "Sample[0-9]+$",
      TITLE
    )
  ) {

    SAMPLE_NAME <- sub(
      "^.*_(Sample[0-9]+)$",
      "\\1",
      TITLE
    )
  }


  META_LIST[[i]] <- data.frame(

    GSM =
      GSM,

    title =
      TITLE,

    sample_name =
      SAMPLE_NAME,

    diagnosis =
      get_characteristic(
        BLOCK,
        "psychiatric diagnosis"
      ),

    sex =
      get_characteristic(
        BLOCK,
        "Sex"
      ),

    age =
      safe_numeric(
        get_characteristic(
          BLOCK,
          "age (years)"
        )
      ),

    PMI =
      safe_numeric(
        get_characteristic(
          BLOCK,
          "pmi (hours)"
        )
      ),

    pH =
      safe_numeric(
        get_characteristic(
          BLOCK,
          "ph"
        )
      ),

    RIN =
      safe_numeric(
        get_characteristic(
          BLOCK,
          "rin (cerebellum)"
        )
      ),

    stringsAsFactors = FALSE
  )
}


META <- do.call(
  rbind,
  META_LIST
)


# ------------------------------------------------------------
# 16. PHENOTYPE INTEGRITY
# ------------------------------------------------------------

if (
  anyNA(META$sample_name) ||
  anyDuplicated(META$sample_name)
) {
  stop(
    "Could not uniquely map all GEO samples to Sample1-Sample30."
  )
}


MISSING_METADATA_SAMPLE <- setdiff(
  SAMPLE_NAMES,
  META$sample_name
)

EXTRA_METADATA_SAMPLE <- setdiff(
  META$sample_name,
  SAMPLE_NAMES
)


if (
  length(MISSING_METADATA_SAMPLE) > 0L ||
  length(EXTRA_METADATA_SAMPLE) > 0L
) {

  print(
    MISSING_METADATA_SAMPLE
  )

  print(
    EXTRA_METADATA_SAMPLE
  )

  stop(
    "Expression/metadata sample mapping mismatch."
  )
}


META <- META[
  match(
    SAMPLE_NAMES,
    META$sample_name
  ),
  ,
  drop = FALSE
]


DX_LOWER <- tolower(
  trimws(
    META$diagnosis
  )
)


Y <- ifelse(
  DX_LOWER == "schizophrenia",
  1L,
  ifelse(
    DX_LOWER == "control",
    0L,
    NA_integer_
  )
)


if (anyNA(Y)) {

  print(
    unique(
      META$diagnosis
    )
  )

  stop(
    "Unexpected diagnosis category detected."
  )
}


if (
  sum(Y == 1L) != 15L ||
  sum(Y == 0L) != 15L
) {

  print(
    table(
      META$diagnosis
    )
  )

  stop(
    "Expected 15 schizophrenia and 15 control samples."
  )
}


cat(
  "Phenotype mapping: PASS\n"
)

cat(
  "Diagnosis counts: Control=",
  sum(Y == 0L),
  ", Schizophrenia=",
  sum(Y == 1L),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 17. JOIN FROZEN SCORE + PHENOTYPE
# ------------------------------------------------------------

RESULTS <- META

RESULTS$y <- Y

RESULTS$SCZ_MSS <- as.numeric(
  SCORE_PRE_PHENOTYPE$SCZ_MSS[
    match(
      RESULTS$sample_name,
      SCORE_PRE_PHENOTYPE$sample_name
    )
  ]
)


if (
  anyNA(
    RESULTS$SCZ_MSS
  )
) {
  stop(
    "Score-to-phenotype join failure."
  )
}


# ------------------------------------------------------------
# 18. PRIMARY GROUP STATISTICS
# ------------------------------------------------------------

CONTROL_SCORE <- RESULTS$SCZ_MSS[
  RESULTS$y == 0L
]

SCZ_SCORE <- RESULTS$SCZ_MSS[
  RESULTS$y == 1L
]


CONTROL_MEAN <- mean(
  CONTROL_SCORE
)

SCZ_MEAN <- mean(
  SCZ_SCORE
)

CONTROL_SD <- sd(
  CONTROL_SCORE
)

SCZ_SD <- sd(
  SCZ_SCORE
)

MEAN_DIFF <- (
  SCZ_MEAN -
    CONTROL_MEAN
)


# ------------------------------------------------------------
# 19. HEDGES' g
# ------------------------------------------------------------

hedges_g <- function(
  control,
  case
) {

  N0 <- length(control)
  N1 <- length(case)

  DF <- (
    N0 + N1 - 2
  )


  POOLED_SD <- sqrt(
    (
      (N0 - 1) * var(control) +
      (N1 - 1) * var(case)
    ) /
      DF
  )


  if (
    !is.finite(POOLED_SD) ||
    POOLED_SD == 0
  ) {
    return(NA_real_)
  }


  COHEN_D <- (
    mean(case) -
      mean(control)
  ) /
    POOLED_SD


  J <- (
    1 -
      3 /
      (
        4 * DF - 1
      )
  )


  J * COHEN_D
}


HEDGES_G <- hedges_g(
  CONTROL_SCORE,
  SCZ_SCORE
)


# ------------------------------------------------------------
# 20. AUROC
#
# Positive class = schizophrenia.
# Larger frozen score = predicted schizophrenia direction.
# ------------------------------------------------------------

auc_rank <- function(
  y,
  score
) {

  N1 <- sum(
    y == 1L
  )

  N0 <- sum(
    y == 0L
  )


  R <- rank(
    score,
    ties.method = "average"
  )


  (
    sum(
      R[
        y == 1L
      ]
    ) -
      N1 * (
        N1 + 1
      ) / 2
  ) /
    (
      N1 * N0
    )
}


AUROC <- auc_rank(
  RESULTS$y,
  RESULTS$SCZ_MSS
)


# ------------------------------------------------------------
# 21. WELCH TEST + WILCOXON
# ------------------------------------------------------------

WELCH <- t.test(
  SCZ_SCORE,
  CONTROL_SCORE,
  alternative = "two.sided",
  var.equal = FALSE
)


WILCOX <- wilcox.test(
  SCZ_SCORE,
  CONTROL_SCORE,
  alternative = "two.sided",
  exact = FALSE
)


# ------------------------------------------------------------
# 22. STRATIFIED BOOTSTRAP
#
# Fixed once; no tuning.
# 10,000 replicates.
# ------------------------------------------------------------

set.seed(
  20260930
)

B <- 10000L

CONTROL_INDEX <- which(
  RESULTS$y == 0L
)

SCZ_INDEX <- which(
  RESULTS$y == 1L
)

N0 <- length(
  CONTROL_INDEX
)

N1 <- length(
  SCZ_INDEX
)


BOOT_MEAN_DIFF <- rep(
  NA_real_,
  B
)

BOOT_HEDGES_G <- rep(
  NA_real_,
  B
)

BOOT_AUROC <- rep(
  NA_real_,
  B
)


for (
  b in seq_len(B)
) {

  I0 <- sample(
    CONTROL_INDEX,
    size = N0,
    replace = TRUE
  )

  I1 <- sample(
    SCZ_INDEX,
    size = N1,
    replace = TRUE
  )


  CONTROL_B <- RESULTS$SCZ_MSS[
    I0
  ]

  SCZ_B <- RESULTS$SCZ_MSS[
    I1
  ]


  BOOT_MEAN_DIFF[b] <- (
    mean(SCZ_B) -
      mean(CONTROL_B)
  )


  BOOT_HEDGES_G[b] <- hedges_g(
    CONTROL_B,
    SCZ_B
  )


  Y_B <- c(
    rep(
      0L,
      N0
    ),
    rep(
      1L,
      N1
    )
  )


  SCORE_B <- c(
    CONTROL_B,
    SCZ_B
  )


  BOOT_AUROC[b] <- auc_rank(
    Y_B,
    SCORE_B
  )
}


BOOT_CI <- function(x) {

  as.numeric(
    quantile(
      x,
      probs = c(
        0.025,
        0.975
      ),
      na.rm = TRUE,
      names = FALSE,
      type = 7
    )
  )
}


MEAN_DIFF_CI <- BOOT_CI(
  BOOT_MEAN_DIFF
)

HEDGES_G_CI <- BOOT_CI(
  BOOT_HEDGES_G
)

AUROC_CI <- BOOT_CI(
  BOOT_AUROC
)


# ------------------------------------------------------------
# 23. SUMMARY TABLE
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(
    "N_control",
    "N_schizophrenia",
    "Frozen_genes_available",
    "Retained_absolute_weight",
    "Control_mean_SCZ_MSS",
    "Control_SD_SCZ_MSS",
    "Schizophrenia_mean_SCZ_MSS",
    "Schizophrenia_SD_SCZ_MSS",
    "Mean_difference_SCZ_minus_Control",
    "Mean_difference_bootstrap_CI_low",
    "Mean_difference_bootstrap_CI_high",
    "Hedges_g",
    "Hedges_g_bootstrap_CI_low",
    "Hedges_g_bootstrap_CI_high",
    "AUROC",
    "AUROC_bootstrap_CI_low",
    "AUROC_bootstrap_CI_high",
    "Welch_t_p",
    "Wilcoxon_p"
  ),

  value = c(
    length(CONTROL_SCORE),
    length(SCZ_SCORE),
    M,
    RETAINED_ABS_WEIGHT,
    CONTROL_MEAN,
    CONTROL_SD,
    SCZ_MEAN,
    SCZ_SD,
    MEAN_DIFF,
    MEAN_DIFF_CI[1],
    MEAN_DIFF_CI[2],
    HEDGES_G,
    HEDGES_G_CI[1],
    HEDGES_G_CI[2],
    AUROC,
    AUROC_CI[1],
    AUROC_CI[2],
    WELCH$p.value,
    WILCOX$p.value
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 24. GENE-LEVEL OUTPUT TABLE
# ------------------------------------------------------------

GENE_OUTPUT <- data.frame(

  frozen_gene_id =
    rownames(
      GENE_TPM
    ),

  frozen_symbol =
    AVAILABLE_SYMBOL,

  frozen_weight =
    AVAILABLE_WEIGHT,

  transcript_count =
    as.integer(
      table(
        FROZEN_TRANSCRIPT_GENE
      )[
        rownames(
          GENE_TPM
        )
      ]
    ),

  stringsAsFactors = FALSE,
  check.names = FALSE
)


GENE_OUTPUT <- cbind(
  GENE_OUTPUT,
  as.data.frame(
    GENE_TPM,
    check.names = FALSE
  )
)


# ------------------------------------------------------------
# 25. BOOTSTRAP OUTPUT
# ------------------------------------------------------------

BOOTSTRAP <- data.frame(

  bootstrap_id =
    seq_len(B),

  mean_difference =
    BOOT_MEAN_DIFF,

  hedges_g =
    BOOT_HEDGES_G,

  AUROC =
    BOOT_AUROC
)


# ------------------------------------------------------------
# 26. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse224683"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


SCORE_OUT <- file.path(
  OUT_DIR,
  "STEP09D_GSE224683_sample_SCZ_MSS_v1.csv"
)

SUMMARY_OUT <- file.path(
  OUT_DIR,
  "STEP09D_GSE224683_external_validation_summary_v1.csv"
)

GENE_OUT <- file.path(
  OUT_DIR,
  "STEP09D_GSE224683_gene_level_TPM_v1.csv"
)

BOOT_OUT <- file.path(
  OUT_DIR,
  "STEP09D_GSE224683_bootstrap_v1.csv"
)

META_OUT <- file.path(
  OUT_DIR,
  "STEP09D_GSE224683_GEO_metadata_v1.csv"
)

NOTE_OUT <- file.path(
  OUT_DIR,
  "STEP09D_GSE224683_VALIDATION_NOTES_v1.txt"
)

HASH_OUT <- file.path(
  OUT_DIR,
  "STEP09D_GSE224683_SHA256_v1.csv"
)


# ------------------------------------------------------------
# 27. SAVE
# ------------------------------------------------------------

write.csv(
  RESULTS,
  SCORE_OUT,
  row.names = FALSE
)

write.csv(
  SUMMARY,
  SUMMARY_OUT,
  row.names = FALSE
)

write.csv(
  GENE_OUTPUT,
  GENE_OUT,
  row.names = FALSE
)

write.csv(
  BOOTSTRAP,
  BOOT_OUT,
  row.names = FALSE
)

write.csv(
  META,
  META_OUT,
  row.names = FALSE
)


writeLines(

  c(

    "STEP 09D — GSE224683 FROZEN SCZ-MSS EXTERNAL VALIDATION",

    "",

    "Technical gate: PASS",

    "Frozen genes used: 23/24",

    paste0(
      "Retained absolute weight: ",
      signif(
        RETAINED_ABS_WEIGHT,
        8
      )
    ),

    "",

    paste0(
      "Pre-phenotype external score SHA256: ",
      SCORE_HASH_PRE_PHENOTYPE
    ),

    "",

    paste0(
      "Control mean: ",
      signif(
        CONTROL_MEAN,
        8
      )
    ),

    paste0(
      "Schizophrenia mean: ",
      signif(
        SCZ_MEAN,
        8
      )
    ),

    paste0(
      "SCZ-Control mean difference: ",
      signif(
        MEAN_DIFF,
        8
      )
    ),

    paste0(
      "Hedges g: ",
      signif(
        HEDGES_G,
        8
      )
    ),

    paste0(
      "AUROC: ",
      signif(
        AUROC,
        8
      )
    ),

    "",

    paste(
      "Gene-level TPM was constructed by summing transcript",
      "TPMs sharing the same explicit stable Ensembl gene ID."
    ),

    paste(
      "The frozen within-score-gene rank formula was then",
      "applied using the 23 genes that passed the pre-outcome",
      "technical gate."
    ),

    paste(
      "The score vector was generated and hashed before GEO",
      "diagnosis metadata were parsed within this analysis cell."
    ),

    "",

    "No retraining was performed.",
    "No coefficient tuning was performed.",
    "No weight tuning was performed.",
    "No score threshold was optimized.",
    "No missing-gene rescue was performed.",

    "",

    paste(
      "Because the external cohort contains 30 subjects,",
      "95% confidence intervals were estimated using 10,000",
      "diagnosis-stratified bootstrap replicates."
    )
  ),

  NOTE_OUT
)


# ------------------------------------------------------------
# 28. HASH OUTPUTS
# ------------------------------------------------------------

OUTPUT_FILES <- c(
  SCORE_OUT,
  SUMMARY_OUT,
  GENE_OUT,
  BOOT_OUT,
  META_OUT,
  NOTE_OUT
)


OUTPUT_SHA <- vapply(
  OUTPUT_FILES,
  function(path) {

    digest::digest(
      file = path,
      algo = "sha256"
    )
  },
  character(1)
)


HASH_TABLE <- data.frame(

  file_name =
    basename(
      OUTPUT_FILES
    ),

  sha256 =
    unname(
      OUTPUT_SHA
    ),

  stringsAsFactors = FALSE
)


write.csv(
  HASH_TABLE,
  HASH_OUT,
  row.names = FALSE
)


# ------------------------------------------------------------
# 29. FINAL OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME


cat("\n============================================================\n")
cat("STEP 09D — GSE224683 FROZEN EXTERNAL VALIDATION COMPLETE\n")
cat("============================================================\n")


cat(
  "\nDiagnosis counts:\n"
)

print(
  table(
    RESULTS$diagnosis
  )
)


cat(
  "\n--- PRIMARY EXTERNAL RESULTS ---\n"
)

cat(
  "Control SCZ-MSS:        ",
  round(CONTROL_MEAN, 6),
  " +/- ",
  round(CONTROL_SD, 6),
  "\n",
  sep = ""
)

cat(
  "Schizophrenia SCZ-MSS:  ",
  round(SCZ_MEAN, 6),
  " +/- ",
  round(SCZ_SD, 6),
  "\n",
  sep = ""
)

cat(
  "Mean difference:        ",
  round(MEAN_DIFF, 6),
  "  [",
  round(MEAN_DIFF_CI[1], 6),
  ", ",
  round(MEAN_DIFF_CI[2], 6),
  "]\n",
  sep = ""
)

cat(
  "Hedges g:               ",
  round(HEDGES_G, 4),
  "  [",
  round(HEDGES_G_CI[1], 4),
  ", ",
  round(HEDGES_G_CI[2], 4),
  "]\n",
  sep = ""
)

cat(
  "AUROC:                  ",
  round(AUROC, 4),
  "  [",
  round(AUROC_CI[1], 4),
  ", ",
  round(AUROC_CI[2], 4),
  "]\n",
  sep = ""
)

cat(
  "Welch p:                ",
  format.pval(
    WELCH$p.value,
    digits = 5
  ),
  "\n",
  sep = ""
)

cat(
  "Wilcoxon p:             ",
  format.pval(
    WILCOX$p.value,
    digits = 5
  ),
  "\n",
  sep = ""
)


cat(
  "\nFrozen genes used: ",
  M,
  "/24\n",
  sep = ""
)

cat(
  "Retained abs weight: ",
  round(
    RETAINED_ABS_WEIGHT,
    6
  ),
  "\n",
  sep = ""
)

cat(
  "All-zero frozen genes: ",
  sum(
    ALL_ZERO_GENE
  ),
  "\n",
  sep = ""
)

cat(
  "Retraining: NO\n"
)

cat(
  "External tuning: NO\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 09D — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Required files: PASS
Previous technical gate: PASS
Frozen score integrity: PASS
Frozen genes available for score: 23/24
TPM matrix: 226608 transcript rows x 30 samples
Retained absolute frozen weight: 0.9266
Frozen available genes with TPM=0 in all 30 samples: 6
Frozen external score hash BEFORE phenotype read: 2b81890fd0339de61dfcb58e3bec1f04139413712d2ae34fee80bf6695e8aef1
Diagnosis metadata loaded so far: NO
Phenotype mapping: PASS
Diagnosis counts: Control=15, Schizophrenia=15

STEP 09D — GSE224683 FROZEN EXTERNAL VALIDATION COMPLETE

Diagnosis counts:

      Control Schizophrenia 
           15            15 

--- PRIMARY EXTERNAL RESULTS ---
Control SCZ-MSS:        0.005851 +/- 0.009426
Schizophrenia SCZ-MSS:  0.005482 +/- 0.007473
Mean difference:        -0.000369  [-0.006485, 0.005459]
Hedges g:               -0.0422  [-0.7574, 0.7262]
AUROC:                  0.5067  [0.2889, 0.7244]
Welch p:                0.90635
Wilcoxon p:             0.96691

Frozen 

In [46]:
# ============================================================
# STEP 09E — GSE224683 OUTCOME-INDEPENDENT MEASURABILITY AUDIT
#
# Purpose:
#   Diagnose why a technically mapped frozen score may have
#   shown poor external transfer.
#
# This cell uses:
#   - transcript-level TPM
#   - transcript-level estimated counts
#   - frozen 24-gene weights
#
# It DOES NOT use:
#   - diagnosis
#   - group labels
#   - AUROC
#   - p-values
#   - any outcome-driven tuning
#
# Primary STEP09D result remains unchanged.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUT FILES
# ------------------------------------------------------------

TPM_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse224683",
  "GSE224683_salmon_uci30_sczctrl_tpms.csv.gz"
)

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse224683",
  "GSE224683_salmon_uci30_sczctrl_nreads.csv.gz"
)

WEIGHTS_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_gene_weights.csv"
)

REQUIRED <- c(
  TPM_PATH,
  COUNTS_PATH,
  WEIGHTS_PATH
)

if (!all(file.exists(REQUIRED))) {

  print(
    REQUIRED[
      !file.exists(REQUIRED)
    ]
  )

  stop("Required file missing.")
}

cat("Required files: PASS\n")


# ------------------------------------------------------------
# 2. LOAD FROZEN WEIGHTS
# ------------------------------------------------------------

WEIGHTS <- read.csv(
  WEIGHTS_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

REQ_COLS <- c(
  "ensemblID_stable",
  "gene_symbol",
  "gene_type",
  "protein_coding",
  "final_normalized_weight"
)

if (!all(
  REQ_COLS %in%
    names(WEIGHTS)
)) {
  stop("Frozen weight schema mismatch.")
}

if (nrow(WEIGHTS) != 24L) {
  stop("Expected exactly 24 frozen genes.")
}


FROZEN_ID <- trimws(
  as.character(
    WEIGHTS$ensemblID_stable
  )
)

FROZEN_WEIGHT <- as.numeric(
  WEIGHTS$final_normalized_weight
)


if (
  anyNA(FROZEN_ID) ||
  anyNA(FROZEN_WEIGHT)
) {
  stop("Invalid frozen score data.")
}


cat("Frozen score integrity: PASS\n")


# ------------------------------------------------------------
# 3. HELPER TO LOAD + PARSE SALMON TABLE
# ------------------------------------------------------------

load_salmon_matrix <- function(path) {

  DF <- read.csv(
    path,
    header = TRUE,
    check.names = FALSE,
    stringsAsFactors = FALSE
  )


  if (ncol(DF) != 31L) {
    stop(
      paste0(
        "Unexpected column count in ",
        basename(path)
      )
    )
  }


  ANNOTATION <- as.character(
    DF[, 1]
  )


  X <- as.matrix(
    DF[, -1, drop = FALSE]
  )

  storage.mode(X) <- "numeric"


  if (
    anyNA(X) ||
    any(!is.finite(X)) ||
    any(X < 0)
  ) {
    stop(
      paste0(
        "Invalid numeric values in ",
        basename(path)
      )
    )
  }


  PARTS <- strsplit(
    ANNOTATION,
    "|",
    fixed = TRUE
  )


  if (any(lengths(PARTS) < 2L)) {
    stop("Malformed Salmon annotation.")
  }


  GENE_RAW <- vapply(
    PARTS,
    function(x) x[2],
    character(1)
  )


  GENE_STABLE <- sub(
    "\\.[0-9]+$",
    "",
    GENE_RAW
  )


  VALID <- grepl(
    "^ENSG[0-9]+$",
    GENE_STABLE
  )


  if (mean(VALID) < 0.99) {
    stop("Unexpected Ensembl parsing failure.")
  }


  list(
    matrix = X,
    gene_id = GENE_STABLE,
    sample_names = colnames(X)
  )
}


# ------------------------------------------------------------
# 4. LOAD TPM + COUNTS
# ------------------------------------------------------------

TPM <- load_salmon_matrix(
  TPM_PATH
)

COUNTS <- load_salmon_matrix(
  COUNTS_PATH
)


if (!identical(
  TPM$sample_names,
  COUNTS$sample_names
)) {
  stop("TPM/count sample-column mismatch.")
}


if (!identical(
  TPM$gene_id,
  COUNTS$gene_id
)) {
  stop("TPM/count transcript annotation mismatch.")
}


cat(
  "Transcript rows: ",
  nrow(TPM$matrix),
  "\n",
  sep = ""
)

cat(
  "Samples: ",
  ncol(TPM$matrix),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 5. KEEP FROZEN-GENE TRANSCRIPTS
# ------------------------------------------------------------

KEEP <- TPM$gene_id %in%
  FROZEN_ID


if (!any(KEEP)) {
  stop("No frozen genes represented.")
}


TPM_FROZEN <- TPM$matrix[
  KEEP,
  ,
  drop = FALSE
]

COUNT_FROZEN <- COUNTS$matrix[
  KEEP,
  ,
  drop = FALSE
]

GENE_FROZEN <- TPM$gene_id[
  KEEP
]


# ------------------------------------------------------------
# 6. TRANSCRIPT -> GENE AGGREGATION
# ------------------------------------------------------------

GENE_TPM <- rowsum(
  TPM_FROZEN,
  group = GENE_FROZEN,
  reorder = FALSE
)

GENE_COUNTS <- rowsum(
  COUNT_FROZEN,
  group = GENE_FROZEN,
  reorder = FALSE
)


# ------------------------------------------------------------
# 7. BUILD COMPLETE 24-GENE AUDIT
# ------------------------------------------------------------

AUDIT <- data.frame(

  frozen_gene_id =
    FROZEN_ID,

  gene_symbol =
    as.character(
      WEIGHTS$gene_symbol
    ),

  gene_type =
    as.character(
      WEIGHTS$gene_type
    ),

  protein_coding =
    WEIGHTS$protein_coding,

  frozen_weight =
    FROZEN_WEIGHT,

  absolute_weight =
    abs(
      FROZEN_WEIGHT
    ),

  stringsAsFactors = FALSE
)


AUDIT$mapped_in_external <- (
  AUDIT$frozen_gene_id %in%
    rownames(
      GENE_TPM
    )
)


# ------------------------------------------------------------
# 8. TRANSCRIPT COUNTS PER FROZEN GENE
# ------------------------------------------------------------

TRANSCRIPT_TABLE <- table(
  GENE_FROZEN
)

AUDIT$transcript_count <- as.integer(
  TRANSCRIPT_TABLE[
    AUDIT$frozen_gene_id
  ]
)

AUDIT$transcript_count[
  is.na(
    AUDIT$transcript_count
  )
] <- 0L


# ------------------------------------------------------------
# 9. INITIALIZE EXPRESSION METRICS
# ------------------------------------------------------------

AUDIT$tpm_total <- 0
AUDIT$tpm_mean <- 0
AUDIT$tpm_median <- 0
AUDIT$tpm_max <- 0
AUDIT$tpm_nonzero_samples <- 0L
AUDIT$tpm_nonzero_fraction <- 0

AUDIT$count_total <- 0
AUDIT$count_mean <- 0
AUDIT$count_median <- 0
AUDIT$count_max <- 0
AUDIT$count_nonzero_samples <- 0L
AUDIT$count_nonzero_fraction <- 0


# ------------------------------------------------------------
# 10. POPULATE GENE-LEVEL METRICS
# ------------------------------------------------------------

for (
  i in seq_len(
    nrow(AUDIT)
  )
) {

  ID <- AUDIT$frozen_gene_id[i]


  if (
    ID %in%
      rownames(GENE_TPM)
  ) {

    TV <- as.numeric(
      GENE_TPM[
        ID,
        ,
        drop = TRUE
      ]
    )


    CV <- as.numeric(
      GENE_COUNTS[
        ID,
        ,
        drop = TRUE
      ]
    )


    AUDIT$tpm_total[i] <- sum(TV)

    AUDIT$tpm_mean[i] <- mean(TV)

    AUDIT$tpm_median[i] <- median(TV)

    AUDIT$tpm_max[i] <- max(TV)

    AUDIT$tpm_nonzero_samples[i] <- sum(
      TV > 0
    )

    AUDIT$tpm_nonzero_fraction[i] <- mean(
      TV > 0
    )


    AUDIT$count_total[i] <- sum(CV)

    AUDIT$count_mean[i] <- mean(CV)

    AUDIT$count_median[i] <- median(CV)

    AUDIT$count_max[i] <- max(CV)

    AUDIT$count_nonzero_samples[i] <- sum(
      CV > 0
    )

    AUDIT$count_nonzero_fraction[i] <- mean(
      CV > 0
    )
  }
}


# ------------------------------------------------------------
# 11. MEASURABILITY FLAGS
#
# Descriptive only.
# No new gate threshold is introduced.
# ------------------------------------------------------------

AUDIT$all_zero_TPM <- (
  AUDIT$mapped_in_external &
  AUDIT$tpm_nonzero_samples == 0L
)

AUDIT$all_zero_counts <- (
  AUDIT$mapped_in_external &
  AUDIT$count_nonzero_samples == 0L
)

AUDIT$nonzero_in_any_sample <- (
  AUDIT$tpm_nonzero_samples > 0L
)

AUDIT$nonzero_in_all_samples <- (
  AUDIT$tpm_nonzero_samples ==
    ncol(
      TPM$matrix
    )
)


# ------------------------------------------------------------
# 12. WEIGHT COMPOSITION
# ------------------------------------------------------------

MAPPED_WEIGHT <- sum(
  AUDIT$absolute_weight[
    AUDIT$mapped_in_external
  ]
)

NONZERO_ANY_WEIGHT <- sum(
  AUDIT$absolute_weight[
    AUDIT$nonzero_in_any_sample
  ]
)

ALL_ZERO_TPM_WEIGHT <- sum(
  AUDIT$absolute_weight[
    AUDIT$all_zero_TPM
  ]
)

ALL_ZERO_COUNT_WEIGHT <- sum(
  AUDIT$absolute_weight[
    AUDIT$all_zero_counts
  ]
)

UNMAPPED_WEIGHT <- sum(
  AUDIT$absolute_weight[
    !AUDIT$mapped_in_external
  ]
)


# ------------------------------------------------------------
# 13. CODING COMPOSITION OF ZERO GENES
# ------------------------------------------------------------

N_ZERO_TPM <- sum(
  AUDIT$all_zero_TPM
)

N_ZERO_TPM_PC <- sum(
  AUDIT$all_zero_TPM &
    AUDIT$protein_coding %in%
      c(TRUE, 1, "TRUE", "1")
)

N_ZERO_TPM_NONPC <- (
  N_ZERO_TPM -
    N_ZERO_TPM_PC
)


# ------------------------------------------------------------
# 14. SUMMARY
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(
    "Frozen_gene_count",
    "Mapped_gene_count",
    "Unmapped_gene_count",
    "Mapped_absolute_weight",
    "Unmapped_absolute_weight",
    "Genes_nonzero_in_any_sample",
    "Genes_all_zero_TPM",
    "Genes_all_zero_counts",
    "All_zero_TPM_absolute_weight",
    "All_zero_counts_absolute_weight",
    "Absolute_weight_nonzero_in_any_sample",
    "All_zero_TPM_protein_coding_count",
    "All_zero_TPM_nonprotein_coding_count"
  ),

  value = c(
    24,
    sum(
      AUDIT$mapped_in_external
    ),
    sum(
      !AUDIT$mapped_in_external
    ),
    MAPPED_WEIGHT,
    UNMAPPED_WEIGHT,
    sum(
      AUDIT$nonzero_in_any_sample
    ),
    N_ZERO_TPM,
    sum(
      AUDIT$all_zero_counts
    ),
    ALL_ZERO_TPM_WEIGHT,
    ALL_ZERO_COUNT_WEIGHT,
    NONZERO_ANY_WEIGHT,
    N_ZERO_TPM_PC,
    N_ZERO_TPM_NONPC
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 15. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse224683"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


AUDIT_OUT <- file.path(
  OUT_DIR,
  "STEP09E_GSE224683_frozen_gene_measurability_v1.csv"
)

SUMMARY_OUT <- file.path(
  OUT_DIR,
  "STEP09E_GSE224683_measurability_summary_v1.csv"
)

NOTE_OUT <- file.path(
  OUT_DIR,
  "STEP09E_GSE224683_MEASURABILITY_NOTES_v1.txt"
)


# ------------------------------------------------------------
# 16. SAVE
# ------------------------------------------------------------

write.csv(
  AUDIT,
  AUDIT_OUT,
  row.names = FALSE
)

write.csv(
  SUMMARY,
  SUMMARY_OUT,
  row.names = FALSE
)


ZERO_GENES <- AUDIT[
  AUDIT$all_zero_TPM |
    !AUDIT$mapped_in_external,
  c(
    "frozen_gene_id",
    "gene_symbol",
    "gene_type",
    "protein_coding",
    "frozen_weight",
    "absolute_weight",
    "mapped_in_external",
    "tpm_nonzero_samples",
    "count_nonzero_samples"
  ),
  drop = FALSE
]


writeLines(

  c(

    "STEP 09E — GSE224683 OUTCOME-INDEPENDENT MEASURABILITY AUDIT",

    "",

    paste0(
      "Mapped frozen genes: ",
      sum(AUDIT$mapped_in_external),
      "/24"
    ),

    paste0(
      "Mapped absolute weight: ",
      signif(
        MAPPED_WEIGHT,
        8
      )
    ),

    paste0(
      "Genes nonzero in >=1 sample: ",
      sum(
        AUDIT$nonzero_in_any_sample
      ),
      "/24"
    ),

    paste0(
      "All-zero TPM genes: ",
      N_ZERO_TPM
    ),

    paste0(
      "Absolute weight carried by all-zero TPM genes: ",
      signif(
        ALL_ZERO_TPM_WEIGHT,
        8
      )
    ),

    paste0(
      "Absolute weight represented by genes with TPM >0 in >=1 sample: ",
      signif(
        NONZERO_ANY_WEIGHT,
        8
      )
    ),

    "",

    paste(
      "This audit is descriptive and outcome-independent."
    ),

    paste(
      "No new technical-gate threshold was introduced."
    ),

    paste(
      "The primary STEP09D external validation result was",
      "not recalculated or replaced."
    ),

    paste(
      "Diagnosis metadata were not loaded in this cell."
    )
  ),

  NOTE_OUT
)


# ------------------------------------------------------------
# 17. OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME


cat("\n============================================================\n")
cat("STEP 09E — GSE224683 MEASURABILITY AUDIT COMPLETE\n")
cat("============================================================\n")


cat(
  "\n--- SUMMARY ---\n"
)

print(
  SUMMARY,
  row.names = FALSE,
  digits = 7
)


cat(
  "\n--- UNMAPPED / ALL-ZERO FROZEN GENES ---\n"
)

print(
  ZERO_GENES,
  row.names = FALSE,
  digits = 7
)


cat(
  "\nPrimary STEP09D result modified: NO\n"
)

cat(
  "Diagnosis metadata loaded: NO\n"
)

cat(
  "Outcome-based tuning: NO\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 09E — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Required files: PASS
Frozen score integrity: PASS
Transcript rows: 226608
Samples: 30

STEP 09E — GSE224683 MEASURABILITY AUDIT COMPLETE

--- SUMMARY ---
                                metric       value
                     Frozen_gene_count 24.00000000
                     Mapped_gene_count 23.00000000
                   Unmapped_gene_count  1.00000000
                Mapped_absolute_weight  0.92659973
              Unmapped_absolute_weight  0.07340027
           Genes_nonzero_in_any_sample 17.00000000
                    Genes_all_zero_TPM  6.00000000
                 Genes_all_zero_counts  6.00000000
          All_zero_TPM_absolute_weight  0.44462073
       All_zero_counts_absolute_weight  0.44462073
 Absolute_weight_nonzero_in_any_sample  0.48197900
     All_zero_TPM_protein_coding_count  0.00000000
  All_zero_TPM_nonprotein_coding_count  6.00000000

--- UNMAPPED / ALL-ZERO FROZEN GENES ---
  frozen_gene_id gene_symbol              gene_type protein_coding


In [47]:
# ============================================================
# STEP 09E — GSE224683 OUTCOME-INDEPENDENT MEASURABILITY AUDIT
#
# Purpose:
#   Diagnose why a technically mapped frozen score may have
#   shown poor external transfer.
#
# This cell uses:
#   - transcript-level TPM
#   - transcript-level estimated counts
#   - frozen 24-gene weights
#
# It DOES NOT use:
#   - diagnosis
#   - group labels
#   - AUROC
#   - p-values
#   - any outcome-driven tuning
#
# Primary STEP09D result remains unchanged.
# ============================================================


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUT FILES
# ------------------------------------------------------------

TPM_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse224683",
  "GSE224683_salmon_uci30_sczctrl_tpms.csv.gz"
)

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse224683",
  "GSE224683_salmon_uci30_sczctrl_nreads.csv.gz"
)

WEIGHTS_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_gene_weights.csv"
)

REQUIRED <- c(
  TPM_PATH,
  COUNTS_PATH,
  WEIGHTS_PATH
)

if (!all(file.exists(REQUIRED))) {

  print(
    REQUIRED[
      !file.exists(REQUIRED)
    ]
  )

  stop("Required file missing.")
}

cat("Required files: PASS\n")


# ------------------------------------------------------------
# 2. LOAD FROZEN WEIGHTS
# ------------------------------------------------------------

WEIGHTS <- read.csv(
  WEIGHTS_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

REQ_COLS <- c(
  "ensemblID_stable",
  "gene_symbol",
  "gene_type",
  "protein_coding",
  "final_normalized_weight"
)

if (!all(
  REQ_COLS %in%
    names(WEIGHTS)
)) {
  stop("Frozen weight schema mismatch.")
}

if (nrow(WEIGHTS) != 24L) {
  stop("Expected exactly 24 frozen genes.")
}


FROZEN_ID <- trimws(
  as.character(
    WEIGHTS$ensemblID_stable
  )
)

FROZEN_WEIGHT <- as.numeric(
  WEIGHTS$final_normalized_weight
)


if (
  anyNA(FROZEN_ID) ||
  anyNA(FROZEN_WEIGHT)
) {
  stop("Invalid frozen score data.")
}


cat("Frozen score integrity: PASS\n")


# ------------------------------------------------------------
# 3. HELPER TO LOAD + PARSE SALMON TABLE
# ------------------------------------------------------------

load_salmon_matrix <- function(path) {

  DF <- read.csv(
    path,
    header = TRUE,
    check.names = FALSE,
    stringsAsFactors = FALSE
  )


  if (ncol(DF) != 31L) {
    stop(
      paste0(
        "Unexpected column count in ",
        basename(path)
      )
    )
  }


  ANNOTATION <- as.character(
    DF[, 1]
  )


  X <- as.matrix(
    DF[, -1, drop = FALSE]
  )

  storage.mode(X) <- "numeric"


  if (
    anyNA(X) ||
    any(!is.finite(X)) ||
    any(X < 0)
  ) {
    stop(
      paste0(
        "Invalid numeric values in ",
        basename(path)
      )
    )
  }


  PARTS <- strsplit(
    ANNOTATION,
    "|",
    fixed = TRUE
  )


  if (any(lengths(PARTS) < 2L)) {
    stop("Malformed Salmon annotation.")
  }


  GENE_RAW <- vapply(
    PARTS,
    function(x) x[2],
    character(1)
  )


  GENE_STABLE <- sub(
    "\\.[0-9]+$",
    "",
    GENE_RAW
  )


  VALID <- grepl(
    "^ENSG[0-9]+$",
    GENE_STABLE
  )


  if (mean(VALID) < 0.99) {
    stop("Unexpected Ensembl parsing failure.")
  }


  list(
    matrix = X,
    gene_id = GENE_STABLE,
    sample_names = colnames(X)
  )
}


# ------------------------------------------------------------
# 4. LOAD TPM + COUNTS
# ------------------------------------------------------------

TPM <- load_salmon_matrix(
  TPM_PATH
)

COUNTS <- load_salmon_matrix(
  COUNTS_PATH
)


if (!identical(
  TPM$sample_names,
  COUNTS$sample_names
)) {
  stop("TPM/count sample-column mismatch.")
}


if (!identical(
  TPM$gene_id,
  COUNTS$gene_id
)) {
  stop("TPM/count transcript annotation mismatch.")
}


cat(
  "Transcript rows: ",
  nrow(TPM$matrix),
  "\n",
  sep = ""
)

cat(
  "Samples: ",
  ncol(TPM$matrix),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 5. KEEP FROZEN-GENE TRANSCRIPTS
# ------------------------------------------------------------

KEEP <- TPM$gene_id %in%
  FROZEN_ID


if (!any(KEEP)) {
  stop("No frozen genes represented.")
}


TPM_FROZEN <- TPM$matrix[
  KEEP,
  ,
  drop = FALSE
]

COUNT_FROZEN <- COUNTS$matrix[
  KEEP,
  ,
  drop = FALSE
]

GENE_FROZEN <- TPM$gene_id[
  KEEP
]


# ------------------------------------------------------------
# 6. TRANSCRIPT -> GENE AGGREGATION
# ------------------------------------------------------------

GENE_TPM <- rowsum(
  TPM_FROZEN,
  group = GENE_FROZEN,
  reorder = FALSE
)

GENE_COUNTS <- rowsum(
  COUNT_FROZEN,
  group = GENE_FROZEN,
  reorder = FALSE
)


# ------------------------------------------------------------
# 7. BUILD COMPLETE 24-GENE AUDIT
# ------------------------------------------------------------

AUDIT <- data.frame(

  frozen_gene_id =
    FROZEN_ID,

  gene_symbol =
    as.character(
      WEIGHTS$gene_symbol
    ),

  gene_type =
    as.character(
      WEIGHTS$gene_type
    ),

  protein_coding =
    WEIGHTS$protein_coding,

  frozen_weight =
    FROZEN_WEIGHT,

  absolute_weight =
    abs(
      FROZEN_WEIGHT
    ),

  stringsAsFactors = FALSE
)


AUDIT$mapped_in_external <- (
  AUDIT$frozen_gene_id %in%
    rownames(
      GENE_TPM
    )
)


# ------------------------------------------------------------
# 8. TRANSCRIPT COUNTS PER FROZEN GENE
# ------------------------------------------------------------

TRANSCRIPT_TABLE <- table(
  GENE_FROZEN
)

AUDIT$transcript_count <- as.integer(
  TRANSCRIPT_TABLE[
    AUDIT$frozen_gene_id
  ]
)

AUDIT$transcript_count[
  is.na(
    AUDIT$transcript_count
  )
] <- 0L


# ------------------------------------------------------------
# 9. INITIALIZE EXPRESSION METRICS
# ------------------------------------------------------------

AUDIT$tpm_total <- 0
AUDIT$tpm_mean <- 0
AUDIT$tpm_median <- 0
AUDIT$tpm_max <- 0
AUDIT$tpm_nonzero_samples <- 0L
AUDIT$tpm_nonzero_fraction <- 0

AUDIT$count_total <- 0
AUDIT$count_mean <- 0
AUDIT$count_median <- 0
AUDIT$count_max <- 0
AUDIT$count_nonzero_samples <- 0L
AUDIT$count_nonzero_fraction <- 0


# ------------------------------------------------------------
# 10. POPULATE GENE-LEVEL METRICS
# ------------------------------------------------------------

for (
  i in seq_len(
    nrow(AUDIT)
  )
) {

  ID <- AUDIT$frozen_gene_id[i]


  if (
    ID %in%
      rownames(GENE_TPM)
  ) {

    TV <- as.numeric(
      GENE_TPM[
        ID,
        ,
        drop = TRUE
      ]
    )


    CV <- as.numeric(
      GENE_COUNTS[
        ID,
        ,
        drop = TRUE
      ]
    )


    AUDIT$tpm_total[i] <- sum(TV)

    AUDIT$tpm_mean[i] <- mean(TV)

    AUDIT$tpm_median[i] <- median(TV)

    AUDIT$tpm_max[i] <- max(TV)

    AUDIT$tpm_nonzero_samples[i] <- sum(
      TV > 0
    )

    AUDIT$tpm_nonzero_fraction[i] <- mean(
      TV > 0
    )


    AUDIT$count_total[i] <- sum(CV)

    AUDIT$count_mean[i] <- mean(CV)

    AUDIT$count_median[i] <- median(CV)

    AUDIT$count_max[i] <- max(CV)

    AUDIT$count_nonzero_samples[i] <- sum(
      CV > 0
    )

    AUDIT$count_nonzero_fraction[i] <- mean(
      CV > 0
    )
  }
}


# ------------------------------------------------------------
# 11. MEASURABILITY FLAGS
#
# Descriptive only.
# No new gate threshold is introduced.
# ------------------------------------------------------------

AUDIT$all_zero_TPM <- (
  AUDIT$mapped_in_external &
  AUDIT$tpm_nonzero_samples == 0L
)

AUDIT$all_zero_counts <- (
  AUDIT$mapped_in_external &
  AUDIT$count_nonzero_samples == 0L
)

AUDIT$nonzero_in_any_sample <- (
  AUDIT$tpm_nonzero_samples > 0L
)

AUDIT$nonzero_in_all_samples <- (
  AUDIT$tpm_nonzero_samples ==
    ncol(
      TPM$matrix
    )
)


# ------------------------------------------------------------
# 12. WEIGHT COMPOSITION
# ------------------------------------------------------------

MAPPED_WEIGHT <- sum(
  AUDIT$absolute_weight[
    AUDIT$mapped_in_external
  ]
)

NONZERO_ANY_WEIGHT <- sum(
  AUDIT$absolute_weight[
    AUDIT$nonzero_in_any_sample
  ]
)

ALL_ZERO_TPM_WEIGHT <- sum(
  AUDIT$absolute_weight[
    AUDIT$all_zero_TPM
  ]
)

ALL_ZERO_COUNT_WEIGHT <- sum(
  AUDIT$absolute_weight[
    AUDIT$all_zero_counts
  ]
)

UNMAPPED_WEIGHT <- sum(
  AUDIT$absolute_weight[
    !AUDIT$mapped_in_external
  ]
)


# ------------------------------------------------------------
# 13. CODING COMPOSITION OF ZERO GENES
# ------------------------------------------------------------

N_ZERO_TPM <- sum(
  AUDIT$all_zero_TPM
)

N_ZERO_TPM_PC <- sum(
  AUDIT$all_zero_TPM &
    AUDIT$protein_coding %in%
      c(TRUE, 1, "TRUE", "1")
)

N_ZERO_TPM_NONPC <- (
  N_ZERO_TPM -
    N_ZERO_TPM_PC
)


# ------------------------------------------------------------
# 14. SUMMARY
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(
    "Frozen_gene_count",
    "Mapped_gene_count",
    "Unmapped_gene_count",
    "Mapped_absolute_weight",
    "Unmapped_absolute_weight",
    "Genes_nonzero_in_any_sample",
    "Genes_all_zero_TPM",
    "Genes_all_zero_counts",
    "All_zero_TPM_absolute_weight",
    "All_zero_counts_absolute_weight",
    "Absolute_weight_nonzero_in_any_sample",
    "All_zero_TPM_protein_coding_count",
    "All_zero_TPM_nonprotein_coding_count"
  ),

  value = c(
    24,
    sum(
      AUDIT$mapped_in_external
    ),
    sum(
      !AUDIT$mapped_in_external
    ),
    MAPPED_WEIGHT,
    UNMAPPED_WEIGHT,
    sum(
      AUDIT$nonzero_in_any_sample
    ),
    N_ZERO_TPM,
    sum(
      AUDIT$all_zero_counts
    ),
    ALL_ZERO_TPM_WEIGHT,
    ALL_ZERO_COUNT_WEIGHT,
    NONZERO_ANY_WEIGHT,
    N_ZERO_TPM_PC,
    N_ZERO_TPM_NONPC
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 15. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse224683"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


AUDIT_OUT <- file.path(
  OUT_DIR,
  "STEP09E_GSE224683_frozen_gene_measurability_v1.csv"
)

SUMMARY_OUT <- file.path(
  OUT_DIR,
  "STEP09E_GSE224683_measurability_summary_v1.csv"
)

NOTE_OUT <- file.path(
  OUT_DIR,
  "STEP09E_GSE224683_MEASURABILITY_NOTES_v1.txt"
)


# ------------------------------------------------------------
# 16. SAVE
# ------------------------------------------------------------

write.csv(
  AUDIT,
  AUDIT_OUT,
  row.names = FALSE
)

write.csv(
  SUMMARY,
  SUMMARY_OUT,
  row.names = FALSE
)


ZERO_GENES <- AUDIT[
  AUDIT$all_zero_TPM |
    !AUDIT$mapped_in_external,
  c(
    "frozen_gene_id",
    "gene_symbol",
    "gene_type",
    "protein_coding",
    "frozen_weight",
    "absolute_weight",
    "mapped_in_external",
    "tpm_nonzero_samples",
    "count_nonzero_samples"
  ),
  drop = FALSE
]


writeLines(

  c(

    "STEP 09E — GSE224683 OUTCOME-INDEPENDENT MEASURABILITY AUDIT",

    "",

    paste0(
      "Mapped frozen genes: ",
      sum(AUDIT$mapped_in_external),
      "/24"
    ),

    paste0(
      "Mapped absolute weight: ",
      signif(
        MAPPED_WEIGHT,
        8
      )
    ),

    paste0(
      "Genes nonzero in >=1 sample: ",
      sum(
        AUDIT$nonzero_in_any_sample
      ),
      "/24"
    ),

    paste0(
      "All-zero TPM genes: ",
      N_ZERO_TPM
    ),

    paste0(
      "Absolute weight carried by all-zero TPM genes: ",
      signif(
        ALL_ZERO_TPM_WEIGHT,
        8
      )
    ),

    paste0(
      "Absolute weight represented by genes with TPM >0 in >=1 sample: ",
      signif(
        NONZERO_ANY_WEIGHT,
        8
      )
    ),

    "",

    paste(
      "This audit is descriptive and outcome-independent."
    ),

    paste(
      "No new technical-gate threshold was introduced."
    ),

    paste(
      "The primary STEP09D external validation result was",
      "not recalculated or replaced."
    ),

    paste(
      "Diagnosis metadata were not loaded in this cell."
    )
  ),

  NOTE_OUT
)


# ------------------------------------------------------------
# 17. OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME


cat("\n============================================================\n")
cat("STEP 09E — GSE224683 MEASURABILITY AUDIT COMPLETE\n")
cat("============================================================\n")


cat(
  "\n--- SUMMARY ---\n"
)

print(
  SUMMARY,
  row.names = FALSE,
  digits = 7
)


cat(
  "\n--- UNMAPPED / ALL-ZERO FROZEN GENES ---\n"
)

print(
  ZERO_GENES,
  row.names = FALSE,
  digits = 7
)


cat(
  "\nPrimary STEP09D result modified: NO\n"
)

cat(
  "Diagnosis metadata loaded: NO\n"
)

cat(
  "Outcome-based tuning: NO\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 09E — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Required files: PASS
Frozen score integrity: PASS
Transcript rows: 226608
Samples: 30

STEP 09E — GSE224683 MEASURABILITY AUDIT COMPLETE

--- SUMMARY ---
                                metric       value
                     Frozen_gene_count 24.00000000
                     Mapped_gene_count 23.00000000
                   Unmapped_gene_count  1.00000000
                Mapped_absolute_weight  0.92659973
              Unmapped_absolute_weight  0.07340027
           Genes_nonzero_in_any_sample 17.00000000
                    Genes_all_zero_TPM  6.00000000
                 Genes_all_zero_counts  6.00000000
          All_zero_TPM_absolute_weight  0.44462073
       All_zero_counts_absolute_weight  0.44462073
 Absolute_weight_nonzero_in_any_sample  0.48197900
     All_zero_TPM_protein_coding_count  0.00000000
  All_zero_TPM_nonprotein_coding_count  6.00000000

--- UNMAPPED / ALL-ZERO FROZEN GENES ---
  frozen_gene_id gene_symbol              gene_type protein_coding


In [48]:
# ============================================================
# STEP 09F — GSE224683 MEASURABLE-ONLY POST-HOC SENSITIVITY
#
# Uses ONLY genes classified as measurable in STEP09E:
#   TPM > 0 in at least one external sample.
#
# Frozen directions/weights unchanged.
# Primary STEP09D result remains primary.
#
# NO retraining.
# NO diagnosis-based feature selection.
# NO weight re-estimation.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

SENTINEL <- file.path(PROJECT_ROOT, ".project_id")

if (!file.exists(SENTINEL)) {
  stop("Project sentinel missing.")
}

if (!identical(
  trimws(readLines(SENTINEL, warn = FALSE)[1]),
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUTS
# ------------------------------------------------------------

TPM_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse224683",
  "GSE224683_salmon_uci30_sczctrl_tpms.csv.gz"
)

WEIGHTS_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_gene_weights.csv"
)

MEASURE_PATH <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse224683",
  "STEP09E_GSE224683_frozen_gene_measurability_v1.csv"
)

PRIMARY_PATH <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse224683",
  "STEP09D_GSE224683_sample_SCZ_MSS_v1.csv"
)

REQUIRED <- c(
  TPM_PATH,
  WEIGHTS_PATH,
  MEASURE_PATH,
  PRIMARY_PATH
)

if (!all(file.exists(REQUIRED))) {
  print(REQUIRED[!file.exists(REQUIRED)])
  stop("Required file missing.")
}

cat("Required files: PASS\n")


# ------------------------------------------------------------
# 2. FROZEN WEIGHTS
# ------------------------------------------------------------

WEIGHTS <- read.csv(
  WEIGHTS_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(WEIGHTS) == 24L,
  all(c(
    "ensemblID_stable",
    "gene_symbol",
    "final_normalized_weight"
  ) %in% names(WEIGHTS))
)

FROZEN_ID <- as.character(
  WEIGHTS$ensemblID_stable
)

FROZEN_SYMBOL <- as.character(
  WEIGHTS$gene_symbol
)

FROZEN_WEIGHT <- as.numeric(
  WEIGHTS$final_normalized_weight
)

if (
  anyNA(FROZEN_ID) ||
  anyNA(FROZEN_WEIGHT)
) {
  stop("Frozen weight integrity failure.")
}

cat("Frozen score integrity: PASS\n")


# ------------------------------------------------------------
# 3. OUTCOME-INDEPENDENT MEASURABLE SUBSET
# ------------------------------------------------------------

MEASURE <- read.csv(
  MEASURE_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

REQ_MEASURE <- c(
  "frozen_gene_id",
  "mapped_in_external",
  "nonzero_in_any_sample"
)

if (!all(
  REQ_MEASURE %in%
    names(MEASURE)
)) {
  stop("STEP09E schema mismatch.")
}


MEASURABLE_RAW <- as.character(
  MEASURE$frozen_gene_id[
    MEASURE$mapped_in_external &
    MEASURE$nonzero_in_any_sample
  ]
)


MEASURABLE_ID <- FROZEN_ID[
  FROZEN_ID %in%
    MEASURABLE_RAW
]


if (length(MEASURABLE_ID) != 17L) {
  stop(
    paste0(
      "Expected 17 measurable genes; found ",
      length(MEASURABLE_ID)
    )
  )
}


IDX <- match(
  MEASURABLE_ID,
  FROZEN_ID
)

MEASURABLE_WEIGHT <- FROZEN_WEIGHT[
  IDX
]

MEASURABLE_SYMBOL <- FROZEN_SYMBOL[
  IDX
]

RETAINED_ABS_WEIGHT <- sum(
  abs(MEASURABLE_WEIGHT)
)


cat(
  "Measurable genes: ",
  length(MEASURABLE_ID),
  "/24\n",
  sep = ""
)

cat(
  "Retained frozen absolute weight: ",
  round(RETAINED_ABS_WEIGHT, 6),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 4. LOAD TPM
# ------------------------------------------------------------

TPM_DF <- read.csv(
  TPM_PATH,
  header = TRUE,
  check.names = FALSE,
  stringsAsFactors = FALSE
)

ANNOTATION <- as.character(
  TPM_DF[, 1]
)

TPM <- as.matrix(
  TPM_DF[, -1, drop = FALSE]
)

storage.mode(TPM) <- "numeric"

if (
  anyNA(TPM) ||
  any(!is.finite(TPM)) ||
  any(TPM < 0)
) {
  stop("Invalid TPM matrix.")
}

SAMPLE_NAMES <- colnames(TPM)


# ------------------------------------------------------------
# 5. PARSE ENSEMBL GENE IDs
# ------------------------------------------------------------

PARTS <- strsplit(
  ANNOTATION,
  "|",
  fixed = TRUE
)

if (any(lengths(PARTS) < 2L)) {
  stop("Malformed transcript annotation.")
}

GENE_RAW <- vapply(
  PARTS,
  function(x) x[2],
  character(1)
)

GENE_ID <- sub(
  "\\.[0-9]+$",
  "",
  GENE_RAW
)


# ------------------------------------------------------------
# 6. TRANSCRIPT -> GENE TPM
# ------------------------------------------------------------

KEEP <- GENE_ID %in%
  MEASURABLE_ID

GENE_TPM <- rowsum(
  TPM[
    KEEP,
    ,
    drop = FALSE
  ],
  group = GENE_ID[
    KEEP
  ],
  reorder = FALSE
)

if (!all(
  MEASURABLE_ID %in%
    rownames(GENE_TPM)
)) {
  stop("Measurable frozen gene missing after aggregation.")
}

GENE_TPM <- GENE_TPM[
  MEASURABLE_ID,
  ,
  drop = FALSE
]

if (nrow(GENE_TPM) != 17L) {
  stop("Expected 17 measurable gene rows.")
}

NONZERO_SAMPLES <- rowSums(
  GENE_TPM > 0
)

if (any(NONZERO_SAMPLES == 0L)) {
  stop("Unexpected all-zero gene in measurable subset.")
}

cat("17-gene measurable matrix integrity: PASS\n")


# ------------------------------------------------------------
# 7. FROZEN RANK SCORE
# ------------------------------------------------------------

M <- nrow(GENE_TPM)

SENS_SCORE <- apply(
  GENE_TPM,
  2,
  function(x) {

    r <- rank(
      x,
      ties.method = "average"
    )

    centered_rank <- (
      r - (M + 1) / 2
    ) / (
      M - 1
    )

    sum(
      MEASURABLE_WEIGHT *
      centered_rank
    ) /
      RETAINED_ABS_WEIGHT
  }
)

if (
  anyNA(SENS_SCORE) ||
  any(!is.finite(SENS_SCORE))
) {
  stop("Invalid sensitivity score.")
}


# ------------------------------------------------------------
# 8. LOAD PRIMARY STEP09D PHENOTYPE
# ------------------------------------------------------------

PRIMARY <- read.csv(
  PRIMARY_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

REQ_PRIMARY <- c(
  "sample_name",
  "diagnosis",
  "y",
  "SCZ_MSS"
)

if (!all(
  REQ_PRIMARY %in%
    names(PRIMARY)
)) {
  stop("STEP09D schema mismatch.")
}

if (!setequal(
  PRIMARY$sample_name,
  SAMPLE_NAMES
)) {
  stop("Sample mismatch.")
}

PRIMARY <- PRIMARY[
  match(
    SAMPLE_NAMES,
    PRIMARY$sample_name
  ),
  ,
  drop = FALSE
]

Y <- as.integer(
  PRIMARY$y
)

if (
  sum(Y == 0L) != 15L ||
  sum(Y == 1L) != 15L
) {
  stop("Unexpected diagnosis counts.")
}


# ------------------------------------------------------------
# 9. RESULTS TABLE
# ------------------------------------------------------------

RESULTS <- PRIMARY

RESULTS$SCZ_MSS_primary_23gene <- as.numeric(
  PRIMARY$SCZ_MSS
)

RESULTS$SCZ_MSS_measurable_17gene <- as.numeric(
  SENS_SCORE
)


CONTROL <- RESULTS$SCZ_MSS_measurable_17gene[
  Y == 0L
]

SCZ <- RESULTS$SCZ_MSS_measurable_17gene[
  Y == 1L
]


# ------------------------------------------------------------
# 10. EFFECT SIZE
# ------------------------------------------------------------

hedges_g <- function(control, case) {

  n0 <- length(control)
  n1 <- length(case)

  df <- n0 + n1 - 2

  pooled_sd <- sqrt(
    (
      (n0 - 1) * var(control) +
      (n1 - 1) * var(case)
    ) /
    df
  )

  if (
    !is.finite(pooled_sd) ||
    pooled_sd == 0
  ) {
    return(NA_real_)
  }

  d <- (
    mean(case) -
    mean(control)
  ) /
    pooled_sd

  J <- 1 -
    3 / (
      4 * df - 1
    )

  J * d
}


HEDGES_G <- hedges_g(
  CONTROL,
  SCZ
)


# ------------------------------------------------------------
# 11. AUROC
# ------------------------------------------------------------

auc_rank <- function(y, score) {

  n1 <- sum(y == 1L)
  n0 <- sum(y == 0L)

  r <- rank(
    score,
    ties.method = "average"
  )

  (
    sum(r[y == 1L]) -
    n1 * (n1 + 1) / 2
  ) /
    (n1 * n0)
}


AUROC <- auc_rank(
  Y,
  RESULTS$SCZ_MSS_measurable_17gene
)


# ------------------------------------------------------------
# 12. GROUP STATISTICS
# ------------------------------------------------------------

CONTROL_MEAN <- mean(CONTROL)
SCZ_MEAN <- mean(SCZ)

CONTROL_SD <- sd(CONTROL)
SCZ_SD <- sd(SCZ)

MEAN_DIFF <- (
  SCZ_MEAN -
  CONTROL_MEAN
)


WELCH <- t.test(
  SCZ,
  CONTROL,
  var.equal = FALSE
)

WILCOX <- wilcox.test(
  SCZ,
  CONTROL,
  exact = FALSE
)


# ------------------------------------------------------------
# 13. BOOTSTRAP 95% CIs
# ------------------------------------------------------------

set.seed(20260930)

B <- 10000L

IDX0 <- which(Y == 0L)
IDX1 <- which(Y == 1L)

BOOT_DIFF <- numeric(B)
BOOT_G <- numeric(B)
BOOT_AUC <- numeric(B)


for (b in seq_len(B)) {

  I0 <- sample(
    IDX0,
    length(IDX0),
    replace = TRUE
  )

  I1 <- sample(
    IDX1,
    length(IDX1),
    replace = TRUE
  )

  C0 <- RESULTS$SCZ_MSS_measurable_17gene[
    I0
  ]

  C1 <- RESULTS$SCZ_MSS_measurable_17gene[
    I1
  ]

  BOOT_DIFF[b] <- (
    mean(C1) -
    mean(C0)
  )

  BOOT_G[b] <- hedges_g(
    C0,
    C1
  )

  BOOT_AUC[b] <- auc_rank(
    c(
      rep(0L, length(C0)),
      rep(1L, length(C1))
    ),
    c(
      C0,
      C1
    )
  )
}


ci95 <- function(x) {

  as.numeric(
    quantile(
      x,
      probs = c(
        0.025,
        0.975
      ),
      na.rm = TRUE,
      names = FALSE
    )
  )
}


DIFF_CI <- ci95(
  BOOT_DIFF
)

G_CI <- ci95(
  BOOT_G
)

AUC_CI <- ci95(
  BOOT_AUC
)


# ------------------------------------------------------------
# 14. PRIMARY vs SENSITIVITY CORRELATION
# ------------------------------------------------------------

PEARSON <- cor(
  RESULTS$SCZ_MSS_primary_23gene,
  RESULTS$SCZ_MSS_measurable_17gene,
  method = "pearson"
)

SPEARMAN <- cor(
  RESULTS$SCZ_MSS_primary_23gene,
  RESULTS$SCZ_MSS_measurable_17gene,
  method = "spearman"
)


# ------------------------------------------------------------
# 15. SAVE
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse224683"
)

SUMMARY <- data.frame(

  metric = c(
    "Gene_count",
    "Retained_absolute_weight",
    "Control_mean",
    "Control_SD",
    "Schizophrenia_mean",
    "Schizophrenia_SD",
    "Mean_difference",
    "Mean_difference_CI_low",
    "Mean_difference_CI_high",
    "Hedges_g",
    "Hedges_g_CI_low",
    "Hedges_g_CI_high",
    "AUROC",
    "AUROC_CI_low",
    "AUROC_CI_high",
    "Welch_p",
    "Wilcoxon_p",
    "Pearson_primary_vs_sensitivity",
    "Spearman_primary_vs_sensitivity"
  ),

  value = c(
    17,
    RETAINED_ABS_WEIGHT,
    CONTROL_MEAN,
    CONTROL_SD,
    SCZ_MEAN,
    SCZ_SD,
    MEAN_DIFF,
    DIFF_CI[1],
    DIFF_CI[2],
    HEDGES_G,
    G_CI[1],
    G_CI[2],
    AUROC,
    AUC_CI[1],
    AUC_CI[2],
    WELCH$p.value,
    WILCOX$p.value,
    PEARSON,
    SPEARMAN
  ),

  stringsAsFactors = FALSE
)


GENES_OUT <- data.frame(

  frozen_gene_id =
    MEASURABLE_ID,

  frozen_symbol =
    MEASURABLE_SYMBOL,

  frozen_weight =
    MEASURABLE_WEIGHT,

  nonzero_sample_count =
    NONZERO_SAMPLES,

  stringsAsFactors = FALSE
)


write.csv(
  RESULTS,
  file.path(
    OUT_DIR,
    "STEP09F_GSE224683_measurable_only_scores_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  SUMMARY,
  file.path(
    OUT_DIR,
    "STEP09F_GSE224683_measurable_only_summary_v1.csv"
  ),
  row.names = FALSE
)

write.csv(
  GENES_OUT,
  file.path(
    OUT_DIR,
    "STEP09F_GSE224683_measurable_only_genes_v1.csv"
  ),
  row.names = FALSE
)


writeLines(
  c(
    "STEP 09F — MEASURABLE-ONLY POST-HOC SENSITIVITY",
    "",
    "STATUS: POST-HOC / EXPLORATORY",
    "",
    paste0(
      "Genes used: 17/24"
    ),
    paste0(
      "Retained absolute frozen weight: ",
      signif(RETAINED_ABS_WEIGHT, 8)
    ),
    paste0(
      "Hedges g: ",
      signif(HEDGES_G, 8)
    ),
    paste0(
      "AUROC: ",
      signif(AUROC, 8)
    ),
    "",
    paste(
      "Gene subset was defined solely by the",
      "outcome-independent STEP09E measurability audit."
    ),
    paste(
      "Frozen directions and weights were unchanged."
    ),
    paste(
      "The primary STEP09D 23-gene validation result",
      "remains the primary external result."
    )
  ),
  file.path(
    OUT_DIR,
    "STEP09F_GSE224683_POSTHOC_NOTES_v1.txt"
  )
)


# ------------------------------------------------------------
# 16. OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME

cat("\n============================================================\n")
cat("STEP 09F — MEASURABLE-ONLY SENSITIVITY COMPLETE\n")
cat("============================================================\n")

cat(
  "Genes used: 17/24\n"
)

cat(
  "Retained absolute weight: ",
  round(RETAINED_ABS_WEIGHT, 6),
  "\n",
  sep = ""
)

cat(
  "\nControl SCZ-MSS:        ",
  round(CONTROL_MEAN, 6),
  " +/- ",
  round(CONTROL_SD, 6),
  "\n",
  sep = ""
)

cat(
  "Schizophrenia SCZ-MSS:  ",
  round(SCZ_MEAN, 6),
  " +/- ",
  round(SCZ_SD, 6),
  "\n",
  sep = ""
)

cat(
  "Mean difference:        ",
  round(MEAN_DIFF, 6),
  " [",
  round(DIFF_CI[1], 6),
  ", ",
  round(DIFF_CI[2], 6),
  "]\n",
  sep = ""
)

cat(
  "Hedges g:               ",
  round(HEDGES_G, 4),
  " [",
  round(G_CI[1], 4),
  ", ",
  round(G_CI[2], 4),
  "]\n",
  sep = ""
)

cat(
  "AUROC:                  ",
  round(AUROC, 4),
  " [",
  round(AUC_CI[1], 4),
  ", ",
  round(AUC_CI[2], 4),
  "]\n",
  sep = ""
)

cat(
  "Welch p:                ",
  format.pval(
    WELCH$p.value,
    digits = 5
  ),
  "\n",
  sep = ""
)

cat(
  "Wilcoxon p:             ",
  format.pval(
    WILCOX$p.value,
    digits = 5
  ),
  "\n",
  sep = ""
)

cat(
  "\nPrimary-vs-sensitivity Pearson:  ",
  round(PEARSON, 4),
  "\n",
  sep = ""
)

cat(
  "Primary-vs-sensitivity Spearman: ",
  round(SPEARMAN, 4),
  "\n",
  sep = ""
)

cat(
  "\nPrimary STEP09D replaced: NO\n"
)

cat(
  "Status: POST-HOC EXPLORATORY\n"
)

cat(
  "Runtime: ",
  round(RUNTIME, 2),
  " seconds\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 09F — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Required files: PASS
Frozen score integrity: PASS
Measurable genes: 17/24
Retained frozen absolute weight: 0.481979
17-gene measurable matrix integrity: PASS

STEP 09F — MEASURABLE-ONLY SENSITIVITY COMPLETE
Genes used: 17/24
Retained absolute weight: 0.481979

Control SCZ-MSS:        -0.136396 +/- 0.019663
Schizophrenia SCZ-MSS:  -0.136173 +/- 0.017022
Mean difference:        0.000223 [-0.013325, 0.012535]
Hedges g:               0.0118 [-0.6458, 0.8632]
AUROC:                  0.5467 [0.3378, 0.7556]
Welch p:                0.9737
Wilcoxon p:             0.6783

Primary-vs-sensitivity Pearson:  0.7407
Primary-vs-sensitivity Spearman: 0.7046

Primary STEP09D replaced: NO
Status: POST-HOC EXPLORATORY
Runtime: 9.44 seconds

STEP 09F — CELL PASS


In [49]:
# ============================================================
# STEP 09G — GSE282552 PUBLIC SOURCE ACQUISITION + SCHEMA AUDIT
#
# Public DLPFC bulk RNA-seq:
#   11 Control
#   7 Schizophrenia
#
# This cell:
#   - downloads processed count matrix
#   - downloads GEO family SOFT
#   - records SHA256
#   - inspects count-table schema / identifier format
#
# NO SCZ-MSS score.
# NO diagnosis comparison.
# NO AUROC.
# NO external tuning.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]


# ------------------------------------------------------------
# 0. PROJECT SENTINEL
# ------------------------------------------------------------

SENTINEL_PATH <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
  stop("Project sentinel missing.")
}

OBSERVED_ID <- trimws(
  readLines(
    SENTINEL_PATH,
    warn = FALSE
  )[1]
)

if (!identical(
  OBSERVED_ID,
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop("digest package unavailable.")
}


# ------------------------------------------------------------
# 2. DIRECTORIES
# ------------------------------------------------------------

RAW_DIR <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse282552"
)

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse282552"
)

dir.create(
  RAW_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


# ------------------------------------------------------------
# 3. GEO PUBLIC FILES
# ------------------------------------------------------------

COUNTS_URL <- paste0(
  "https://ftp.ncbi.nlm.nih.gov/geo/series/",
  "GSE282nnn/GSE282552/suppl/",
  "GSE282552_filtE_sr_cnts.tsv.gz"
)

SOFT_URL <- paste0(
  "https://ftp.ncbi.nlm.nih.gov/geo/series/",
  "GSE282nnn/GSE282552/soft/",
  "GSE282552_family.soft.gz"
)

COUNTS_PATH <- file.path(
  RAW_DIR,
  "GSE282552_filtE_sr_cnts.tsv.gz"
)

SOFT_PATH <- file.path(
  RAW_DIR,
  "GSE282552_family.soft.gz"
)


# ------------------------------------------------------------
# 4. DOWNLOAD HELPER
# ------------------------------------------------------------

download_if_needed <- function(
  url,
  destination
) {

  if (
    file.exists(destination) &&
    file.info(destination)$size > 0
  ) {

    cat(
      "Already present: ",
      basename(destination),
      "\n",
      sep = ""
    )

    return(invisible(TRUE))
  }


  cat(
    "Downloading: ",
    basename(destination),
    "\n",
    sep = ""
  )


  download.file(
    url = url,
    destfile = destination,
    mode = "wb",
    method = "libcurl",
    quiet = FALSE
  )


  if (
    !file.exists(destination) ||
    file.info(destination)$size <= 0
  ) {
    stop(
      paste0(
        "Download failed: ",
        basename(destination)
      )
    )
  }

  invisible(TRUE)
}


# ------------------------------------------------------------
# 5. DOWNLOAD
# ------------------------------------------------------------

download_if_needed(
  COUNTS_URL,
  COUNTS_PATH
)

download_if_needed(
  SOFT_URL,
  SOFT_PATH
)


# ------------------------------------------------------------
# 6. FILE INVENTORY
# ------------------------------------------------------------

FILE_PATHS <- c(
  COUNTS_PATH,
  SOFT_PATH
)

FILE_INFO <- file.info(
  FILE_PATHS
)

FILE_SHA <- vapply(
  FILE_PATHS,
  function(path) {

    digest::digest(
      file = path,
      algo = "sha256"
    )
  },
  character(1)
)

INVENTORY <- data.frame(

  accession = c(
    "GSE282552",
    "GSE282552"
  ),

  role = c(
    "processed_short_read_counts",
    "GEO_family_metadata"
  ),

  file_name =
    basename(FILE_PATHS),

  size_MB =
    round(
      as.numeric(
        FILE_INFO$size
      ) / 1024^2,
      3
    ),

  sha256 =
    unname(FILE_SHA),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 7. LIGHT COUNT-TABLE PREVIEW
# ------------------------------------------------------------

COUNTS_PREVIEW <- read.delim(
  COUNTS_PATH,
  header = TRUE,
  nrows = 8,
  check.names = FALSE,
  stringsAsFactors = FALSE,
  quote = "",
  comment.char = ""
)

if (ncol(COUNTS_PREVIEW) < 2L) {
  stop("Processed count matrix appears malformed.")
}

FIRST_COL_NAME <- names(
  COUNTS_PREVIEW
)[1]

FIRST_IDS <- as.character(
  COUNTS_PREVIEW[
    ,
    1,
    drop = TRUE
  ]
)


# ------------------------------------------------------------
# 8. IDENTIFIER-TYPE AUDIT
# ------------------------------------------------------------

detect_identifier_type <- function(x) {

  x <- trimws(
    as.character(x)
  )

  if (
    all(
      grepl(
        "^ENSG[0-9]+(\\.[0-9]+)?$",
        x
      )
    )
  ) {
    return("ENSEMBL_GENE")
  }

  if (
    all(
      grepl(
        "^ENST[0-9]+(\\.[0-9]+)?$",
        x
      )
    )
  ) {
    return("ENSEMBL_TRANSCRIPT")
  }

  if (
    all(
      grepl(
        "^[0-9]+$",
        x
      )
    )
  ) {
    return("NUMERIC_IDENTIFIER")
  }

  if (
    all(
      grepl(
        "^[A-Za-z0-9._:-]+$",
        x
      )
    )
  ) {
    return("SYMBOL_OR_OTHER_IDENTIFIER")
  }

  "MIXED_OR_UNKNOWN"
}


ID_TYPE <- detect_identifier_type(
  FIRST_IDS
)


# ------------------------------------------------------------
# 9. COLUMN AUDIT
# ------------------------------------------------------------

DATA_COLUMNS <- names(
  COUNTS_PREVIEW
)[-1]

cat(
  "Processed matrix columns detected: ",
  length(DATA_COLUMNS),
  " sample/data columns\n",
  sep = ""
)


# ------------------------------------------------------------
# 10. SOFT LIGHT AUDIT
# ------------------------------------------------------------

SOFT_CONNECTION <- gzfile(
  SOFT_PATH,
  open = "rt"
)

SOFT_LINES <- readLines(
  SOFT_CONNECTION,
  n = 200,
  warn = FALSE
)

close(
  SOFT_CONNECTION
)

N_SAMPLE_BLOCKS_PREVIEW <- sum(
  grepl(
    "^\\^SAMPLE = ",
    SOFT_LINES
  )
)


# ------------------------------------------------------------
# 11. SAVE SOURCE AUDIT
# ------------------------------------------------------------

INVENTORY_OUT <- file.path(
  OUT_DIR,
  "STEP09G_GSE282552_source_inventory_v1.csv"
)

AUDIT_OUT <- file.path(
  OUT_DIR,
  "STEP09G_GSE282552_schema_audit_v1.txt"
)

write.csv(
  INVENTORY,
  INVENTORY_OUT,
  row.names = FALSE
)

writeLines(

  c(

    "STEP 09G — GSE282552 SOURCE + SCHEMA AUDIT",

    "",

    paste0(
      "COUNTS_FIRST_COLUMN: ",
      FIRST_COL_NAME
    ),

    paste0(
      "COUNTS_IDENTIFIER_TYPE: ",
      ID_TYPE
    ),

    paste0(
      "COUNT_TABLE_DATA_COLUMNS: ",
      length(DATA_COLUMNS)
    ),

    "",

    "EXPECTED_GEO_SERIES_SAMPLES: 18",

    "EXPECTED_GROUP_STRUCTURE: 11 control + 7 schizophrenia",

    "",

    "SCZ_MSS_SCORE_CALCULATED: NO",

    "DIAGNOSIS_COMPARISON_PERFORMED: NO",

    "AUROC_CALCULATED: NO",

    "EXTERNAL_TUNING: NO"
  ),

  AUDIT_OUT
)


# ------------------------------------------------------------
# 12. FINAL OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME


cat("\n============================================================\n")
cat("STEP 09G — GSE282552 ACQUISITION + SCHEMA AUDIT COMPLETE\n")
cat("============================================================\n")


cat(
  "\n--- FILE INVENTORY ---\n"
)

print(
  INVENTORY,
  row.names = FALSE
)


cat(
  "\n--- COUNT TABLE COLUMNS ---\n"
)

print(
  names(
    COUNTS_PREVIEW
  )
)


cat(
  "\nFirst-column name: ",
  FIRST_COL_NAME,
  "\n",
  sep = ""
)

cat(
  "Identifier type: ",
  ID_TYPE,
  "\n",
  sep = ""
)


cat(
  "\nFirst identifiers:\n"
)

print(
  FIRST_IDS
)


cat(
  "\nNumber of data columns excluding identifier: ",
  length(DATA_COLUMNS),
  "\n",
  sep = ""
)


cat(
  "\nSCZ-MSS score calculated: NO\n"
)

cat(
  "Diagnosis comparison performed: NO\n"
)

cat(
  "AUROC calculated: NO\n"
)

cat(
  "External tuning performed: NO\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 09G — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Downloading: GSE282552_filtE_sr_cnts.tsv.gz
Downloading: GSE282552_family.soft.gz
Processed matrix columns detected: 18 sample/data columns

STEP 09G — GSE282552 ACQUISITION + SCHEMA AUDIT COMPLETE

--- FILE INVENTORY ---
 accession                        role                      file_name size_MB
 GSE282552 processed_short_read_counts GSE282552_filtE_sr_cnts.tsv.gz   5.903
 GSE282552         GEO_family_metadata       GSE282552_family.soft.gz   0.004
                                                           sha256
 77f7e23c6a6b691cdd7900b7cd259cc36556904f484423d3a89d7331c1bf169a
 5abf57675e2a943f33001518803037b17fe0c1d63d3c756dabdd5486d53251c4

--- COUNT TABLE COLUMNS ---
 [1] "S25"     "S26"     "S27"     "S28"     "S29"     "S30"     "S31"    
 [8] "S32"     "S33"     "S34"     "S35"     "S36"     "S37"     "S38"    
[15] "S39"     "S40"     "S41"     "S42"     "isoform"

First-column name: S25
Identifier type: SYMBOL_OR_OTHER_IDENTIFIER

First identifiers:
[

In [50]:
# ============================================================
# STEP 09G.1 — GSE282552 CORRECTED IDENTIFIER SCHEMA AUDIT
#
# Correction:
#   Sample columns = S25 ... S42
#   Identifier column = isoform
#
# NO diagnosis analysis.
# NO SCZ-MSS score.
# NO AUROC.
# NO tuning.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

SENTINEL <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL)) {
  stop("Project sentinel missing.")
}

if (!identical(
  trimws(
    readLines(
      SENTINEL,
      warn = FALSE
    )[1]
  ),
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUT
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse282552",
  "GSE282552_filtE_sr_cnts.tsv.gz"
)

if (!file.exists(COUNTS_PATH)) {
  stop("GSE282552 count matrix missing.")
}

cat("Count matrix found: PASS\n")


# ------------------------------------------------------------
# 2. LOAD FULL TABLE
# ------------------------------------------------------------

DF <- read.delim(
  COUNTS_PATH,
  header = TRUE,
  check.names = FALSE,
  stringsAsFactors = FALSE,
  quote = "",
  comment.char = ""
)

cat(
  "Matrix dimensions: ",
  nrow(DF),
  " rows x ",
  ncol(DF),
  " columns\n",
  sep = ""
)


# ------------------------------------------------------------
# 3. IDENTIFY COLUMNS
# ------------------------------------------------------------

if (!("isoform" %in% names(DF))) {
  stop("Expected isoform identifier column not found.")
}

ID_COL <- "isoform"

SAMPLE_COLS <- setdiff(
  names(DF),
  ID_COL
)


if (length(SAMPLE_COLS) != 18L) {
  stop(
    paste0(
      "Expected 18 sample columns; found ",
      length(SAMPLE_COLS)
    )
  )
}


cat(
  "Identifier column: ",
  ID_COL,
  "\n",
  sep = ""
)

cat(
  "Sample columns: ",
  length(SAMPLE_COLS),
  "\n",
  sep = ""
)

print(
  SAMPLE_COLS
)


# ------------------------------------------------------------
# 4. VERIFY SAMPLE COLUMNS NUMERIC
# ------------------------------------------------------------

NUMERIC_CHECK <- vapply(
  DF[
    ,
    SAMPLE_COLS,
    drop = FALSE
  ],
  is.numeric,
  logical(1)
)

if (!all(NUMERIC_CHECK)) {

  print(
    NUMERIC_CHECK
  )

  stop("One or more sample columns are not numeric.")
}

cat("All 18 sample columns numeric: PASS\n")


# ------------------------------------------------------------
# 5. IDENTIFIER VECTOR
# ------------------------------------------------------------

IDS <- trimws(
  as.character(
    DF[[ID_COL]]
  )
)

if (
  anyNA(IDS) ||
  any(IDS == "")
) {
  stop("Missing isoform identifiers detected.")
}


cat(
  "Unique isoform identifiers: ",
  length(unique(IDS)),
  "\n",
  sep = ""
)

cat(
  "Duplicated isoform rows: ",
  sum(duplicated(IDS)),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 6. IDENTIFIER-TYPE CLASSIFICATION
# ------------------------------------------------------------

IS_ENST <- grepl(
  "^ENST[0-9]+(\\.[0-9]+)?$",
  IDS
)

IS_ENSG <- grepl(
  "^ENSG[0-9]+(\\.[0-9]+)?$",
  IDS
)

HAS_ENSG_ANYWHERE <- grepl(
  "ENSG[0-9]+",
  IDS
)

HAS_ENST_ANYWHERE <- grepl(
  "ENST[0-9]+",
  IDS
)

HAS_PIPE <- grepl(
  "|",
  IDS,
  fixed = TRUE
)


N_ENST_EXACT <- sum(IS_ENST)
N_ENSG_EXACT <- sum(IS_ENSG)
N_ENSG_EMBEDDED <- sum(
  HAS_ENSG_ANYWHERE &
  !IS_ENSG
)

N_ENST_EMBEDDED <- sum(
  HAS_ENST_ANYWHERE &
  !IS_ENST
)

N_PIPE <- sum(HAS_PIPE)


# ------------------------------------------------------------
# 7. EXTRACT EMBEDDED ENSG IF PRESENT
# ------------------------------------------------------------

extract_ensg <- function(x) {

  m <- regexpr(
    "ENSG[0-9]+(?:\\.[0-9]+)?",
    x,
    perl = TRUE
  )

  out <- rep(
    NA_character_,
    length(x)
  )

  hit <- m > 0

  out[hit] <- regmatches(
    x,
    m
  )[hit]

  out
}


ENSG_RAW <- extract_ensg(
  IDS
)

ENSG_STABLE <- sub(
  "\\.[0-9]+$",
  "",
  ENSG_RAW
)


N_ROWS_WITH_ENSG <- sum(
  !is.na(ENSG_STABLE)
)

N_UNIQUE_ENSG <- length(
  unique(
    ENSG_STABLE[
      !is.na(ENSG_STABLE)
    ]
  )
)


# ------------------------------------------------------------
# 8. CLASSIFY OVERALL IDENTIFIER FORMAT
# ------------------------------------------------------------

if (all(IS_ENST)) {

  ID_TYPE <- "ENSEMBL_TRANSCRIPT"

} else if (all(IS_ENSG)) {

  ID_TYPE <- "ENSEMBL_GENE"

} else if (
  mean(HAS_ENSG_ANYWHERE) > 0.95
) {

  ID_TYPE <- "COMPOSITE_WITH_EMBEDDED_ENSEMBL_GENE"

} else if (
  mean(HAS_ENST_ANYWHERE) > 0.95
) {

  ID_TYPE <- "COMPOSITE_WITH_EMBEDDED_ENSEMBL_TRANSCRIPT"

} else {

  ID_TYPE <- "OTHER_OR_MIXED"
}


# ------------------------------------------------------------
# 9. SAVE CORRECTED AUDIT
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse282552"
)

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)


SUMMARY <- data.frame(

  metric = c(
    "Rows",
    "Sample_columns",
    "Identifier_column",
    "Unique_isoform_identifiers",
    "Duplicated_isoform_rows",
    "Exact_ENST_rows",
    "Exact_ENSG_rows",
    "Rows_with_embedded_ENST",
    "Rows_with_embedded_ENSG",
    "Rows_with_pipe_delimiter",
    "Rows_with_extractable_ENSG",
    "Unique_extractable_stable_ENSG",
    "Overall_identifier_type"
  ),

  value = c(
    nrow(DF),
    length(SAMPLE_COLS),
    ID_COL,
    length(unique(IDS)),
    sum(duplicated(IDS)),
    N_ENST_EXACT,
    N_ENSG_EXACT,
    sum(HAS_ENST_ANYWHERE),
    sum(HAS_ENSG_ANYWHERE),
    N_PIPE,
    N_ROWS_WITH_ENSG,
    N_UNIQUE_ENSG,
    ID_TYPE
  ),

  stringsAsFactors = FALSE
)


write.csv(
  SUMMARY,
  file.path(
    OUT_DIR,
    "STEP09G1_GSE282552_corrected_schema_audit_v1.csv"
  ),
  row.names = FALSE
)


# ------------------------------------------------------------
# 10. OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME


cat("\n============================================================\n")
cat("STEP 09G.1 — CORRECTED IDENTIFIER AUDIT COMPLETE\n")
cat("============================================================\n")


cat(
  "\nOverall identifier type: ",
  ID_TYPE,
  "\n",
  sep = ""
)


cat(
  "\nFirst 15 isoform identifiers:\n"
)

print(
  head(
    IDS,
    15
  )
)


cat(
  "\nExact ENST rows: ",
  N_ENST_EXACT,
  "\n",
  sep = ""
)

cat(
  "Exact ENSG rows: ",
  N_ENSG_EXACT,
  "\n",
  sep = ""
)

cat(
  "Rows containing ENST: ",
  sum(HAS_ENST_ANYWHERE),
  "\n",
  sep = ""
)

cat(
  "Rows containing ENSG: ",
  sum(HAS_ENSG_ANYWHERE),
  "\n",
  sep = ""
)

cat(
  "Rows with extractable ENSG: ",
  N_ROWS_WITH_ENSG,
  "\n",
  sep = ""
)

cat(
  "Unique stable ENSG IDs: ",
  N_UNIQUE_ENSG,
  "\n",
  sep = ""
)


cat(
  "\nSCZ-MSS calculated: NO\n"
)

cat(
  "Diagnosis analyzed: NO\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 09G.1 — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Count matrix found: PASS
Matrix dimensions: 56163 rows x 19 columns
Identifier column: isoform
Sample columns: 18
 [1] "S25" "S26" "S27" "S28" "S29" "S30" "S31" "S32" "S33" "S34" "S35" "S36"
[13] "S37" "S38" "S39" "S40" "S41" "S42"
All 18 sample columns numeric: PASS
Unique isoform identifiers: 56163
Duplicated isoform rows: 0

STEP 09G.1 — CORRECTED IDENTIFIER AUDIT COMPLETE

Overall identifier type: OTHER_OR_MIXED

First 15 isoform identifiers:
 [1] "PB.2.7.WASH7P"   "PB.2.86.WASH7P"  "PB.5.1.WASH9P"   "PB.5.2.WASH9P"  
 [5] "PB.5.8.WASH9P"   "PB.5.26.WASH9P"  "PB.5.18.WASH9P"  "PB.5.16.WASH9P" 
 [9] "PB.5.20.WASH9P"  "PB.5.23.WASH9P"  "PB.5.109.WASH9P" "PB.5.133.WASH9P"
[13] "PB.5.129.WASH9P" "PB.22.78.NOC2L"  "PB.22.75.NOC2L" 

Exact ENST rows: 0
Exact ENSG rows: 0
Rows containing ENST: 0
Rows containing ENSG: 0
Rows with extractable ENSG: 0
Unique stable ENSG IDs: 0

SCZ-MSS calculated: NO
Diagnosis analyzed: NO
Runtime: 3.73 seconds

STEP 09G.1 — CELL PASS


In [51]:
# ============================================================
# STEP 09H — GSE282552 FROZEN-SCORE TECHNICAL GATE
#
# Dataset identifier format:
#   PB.<gene_cluster>.<isoform>.<GENE_SYMBOL>
#
# No Ensembl gene IDs are present in the processed matrix.
#
# Frozen mapping:
#   exact frozen gene-symbol suffix only
#
# NO synonym rescue.
# NO external annotation rescue.
# NO diagnosis analysis.
# NO score calculation.
# NO AUROC.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]


# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

SENTINEL <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL)) {
  stop("Project sentinel missing.")
}

if (!identical(
  trimws(
    readLines(
      SENTINEL,
      warn = FALSE
    )[1]
  ),
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. INPUTS
# ------------------------------------------------------------

COUNTS_PATH <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse282552",
  "GSE282552_filtE_sr_cnts.tsv.gz"
)

WEIGHTS_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_gene_weights.csv"
)

MAPPING_POLICY_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_mapping_policy.json"
)

GATE_POLICY_PATH <- file.path(
  PROJECT_ROOT,
  "08_score_lock",
  "SCZ_MSS_v1_technical_gate.json"
)

REQUIRED <- c(
  COUNTS_PATH,
  WEIGHTS_PATH,
  MAPPING_POLICY_PATH,
  GATE_POLICY_PATH
)

if (!all(file.exists(REQUIRED))) {
  print(REQUIRED[!file.exists(REQUIRED)])
  stop("Required input missing.")
}

cat("Required files: PASS\n")


# ------------------------------------------------------------
# 2. PACKAGE
# ------------------------------------------------------------

if (!requireNamespace(
  "digest",
  quietly = TRUE
)) {
  stop("digest package unavailable.")
}


# ------------------------------------------------------------
# 3. LOAD FROZEN SCORE
# ------------------------------------------------------------

WEIGHTS <- read.csv(
  WEIGHTS_PATH,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

stopifnot(
  nrow(WEIGHTS) == 24L,
  all(
    c(
      "ensemblID_stable",
      "gene_symbol",
      "final_normalized_weight"
    ) %in%
      names(WEIGHTS)
  )
)


FROZEN_ID <- trimws(
  as.character(
    WEIGHTS$ensemblID_stable
  )
)

FROZEN_SYMBOL <- trimws(
  as.character(
    WEIGHTS$gene_symbol
  )
)

FROZEN_SYMBOL[
  FROZEN_SYMBOL == ""
] <- NA_character_


FROZEN_WEIGHT <- as.numeric(
  WEIGHTS$final_normalized_weight
)


if (
  anyNA(FROZEN_ID) ||
  anyNA(FROZEN_WEIGHT) ||
  abs(
    sum(abs(FROZEN_WEIGHT)) - 1
  ) > 1e-6
) {
  stop("Frozen score integrity failure.")
}


cat(
  "Frozen genes with symbol: ",
  sum(!is.na(FROZEN_SYMBOL)),
  "/24\n",
  sep = ""
)

cat(
  "Maximum possible exact-symbol coverage: ",
  round(
    mean(!is.na(FROZEN_SYMBOL)),
    6
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 4. POLICY HASHES
# ------------------------------------------------------------

cat(
  "Frozen mapping policy SHA256: ",
  digest::digest(
    file = MAPPING_POLICY_PATH,
    algo = "sha256"
  ),
  "\n",
  sep = ""
)

cat(
  "Frozen gate policy SHA256: ",
  digest::digest(
    file = GATE_POLICY_PATH,
    algo = "sha256"
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 5. LOAD GSE282552 MATRIX
# ------------------------------------------------------------

DF <- read.delim(
  COUNTS_PATH,
  header = TRUE,
  check.names = FALSE,
  stringsAsFactors = FALSE,
  quote = "",
  comment.char = ""
)


if (!("isoform" %in% names(DF))) {
  stop("isoform identifier column missing.")
}


SAMPLE_COLS <- setdiff(
  names(DF),
  "isoform"
)


if (length(SAMPLE_COLS) != 18L) {
  stop(
    paste0(
      "Expected 18 sample columns; found ",
      length(SAMPLE_COLS)
    )
  )
}


X <- as.matrix(
  DF[
    ,
    SAMPLE_COLS,
    drop = FALSE
  ]
)

storage.mode(X) <- "numeric"


if (
  anyNA(X) ||
  any(!is.finite(X)) ||
  any(X < 0)
) {
  stop("Invalid expression matrix.")
}


ISOFORM_ID <- trimws(
  as.character(
    DF$isoform
  )
)


cat(
  "GSE282552 matrix: ",
  nrow(X),
  " isoforms x ",
  ncol(X),
  " samples\n",
  sep = ""
)


# ------------------------------------------------------------
# 6. CONFIRM NO ENSEMBL ID IS AVAILABLE
# ------------------------------------------------------------

HAS_ENSG <- grepl(
  "ENSG[0-9]+",
  ISOFORM_ID
)

HAS_ENST <- grepl(
  "ENST[0-9]+",
  ISOFORM_ID
)


cat(
  "Rows containing ENSG: ",
  sum(HAS_ENSG),
  "\n",
  sep = ""
)

cat(
  "Rows containing ENST: ",
  sum(HAS_ENST),
  "\n",
  sep = ""
)


if (
  any(HAS_ENSG) ||
  any(HAS_ENST)
) {
  stop(
    "Unexpected Ensembl identifiers detected; mapping strategy must be re-audited."
  )
}


# ------------------------------------------------------------
# 7. EXTRACT GENE SYMBOL FROM PB ISOFORM ID
#
# Example:
# PB.2.7.WASH7P -> WASH7P
#
# Exact final-dot suffix only.
# ------------------------------------------------------------

EXTERNAL_SYMBOL <- sub(
  "^.*\\.",
  "",
  ISOFORM_ID
)


if (
  anyNA(EXTERNAL_SYMBOL) ||
  any(EXTERNAL_SYMBOL == "")
) {
  stop("Could not extract gene-symbol suffix.")
}


SYMBOL_ISOFORM_COUNT <- table(
  EXTERNAL_SYMBOL
)


cat(
  "Unique external gene-symbol suffixes: ",
  length(SYMBOL_ISOFORM_COUNT),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 8. EXACT FROZEN SYMBOL MAPPING
# ------------------------------------------------------------

N_EXTERNAL_ISOFORMS <- rep(
  0L,
  24L
)


HAS_FROZEN_SYMBOL <- !is.na(
  FROZEN_SYMBOL
)


N_EXTERNAL_ISOFORMS[
  HAS_FROZEN_SYMBOL
] <- as.integer(
  SYMBOL_ISOFORM_COUNT[
    FROZEN_SYMBOL[
      HAS_FROZEN_SYMBOL
    ]
  ]
)


N_EXTERNAL_ISOFORMS[
  is.na(N_EXTERNAL_ISOFORMS)
] <- 0L


MAPPED <- (
  HAS_FROZEN_SYMBOL &
  N_EXTERNAL_ISOFORMS >= 1L
)


MAPPING <- data.frame(

  frozen_gene_id =
    FROZEN_ID,

  frozen_symbol =
    FROZEN_SYMBOL,

  frozen_weight =
    FROZEN_WEIGHT,

  direction =
    ifelse(
      FROZEN_WEIGHT > 0,
      "positive",
      "negative"
    ),

  external_isoform_count =
    N_EXTERNAL_ISOFORMS,

  mapped_exact_symbol =
    MAPPED,

  mapping_status =
    ifelse(
      is.na(FROZEN_SYMBOL),
      "NO_FROZEN_SYMBOL_AND_NO_EXTERNAL_ENSEMBL_ID",
      ifelse(
        MAPPED,
        "EXACT_GENE_SYMBOL_SUFFIX_PRESENT",
        "FROZEN_SYMBOL_NOT_FOUND"
      )
    ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 9. TECHNICAL COVERAGE
# ------------------------------------------------------------

GENE_COUNT_COVERAGE <- mean(
  MAPPING$mapped_exact_symbol
)


ABS_WEIGHT_COVERAGE <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$mapped_exact_symbol
    ]
  )
)


POS_TOTAL <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight > 0
    ]
  )
)


NEG_TOTAL <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight < 0
    ]
  )
)


POS_RETAINED <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight > 0 &
      MAPPING$mapped_exact_symbol
    ]
  )
)


NEG_RETAINED <- sum(
  abs(
    MAPPING$frozen_weight[
      MAPPING$frozen_weight < 0 &
      MAPPING$mapped_exact_symbol
    ]
  )
)


POS_COVERAGE <- (
  POS_RETAINED /
  POS_TOTAL
)


NEG_COVERAGE <- (
  NEG_RETAINED /
  NEG_TOTAL
)


# ------------------------------------------------------------
# 10. MEASURABILITY OF MAPPED SYMBOLS
# ------------------------------------------------------------

MAPPED_SYMBOLS <- MAPPING$frozen_symbol[
  MAPPING$mapped_exact_symbol
]


if (length(MAPPED_SYMBOLS) > 0L) {

  KEEP <- EXTERNAL_SYMBOL %in%
    MAPPED_SYMBOLS


  MAPPED_X <- X[
    KEEP,
    ,
    drop = FALSE
  ]


  ACTUAL_SAMPLE_COMPLETENESS <- min(
    colMeans(
      is.finite(
        MAPPED_X
      )
    )
  )

} else {

  ACTUAL_SAMPLE_COMPLETENESS <- 0
}


# ------------------------------------------------------------
# 11. LOCKED GATE
# ------------------------------------------------------------

REQ_GENE <- 0.80
REQ_ABS <- 0.80
REQ_POS <- 0.70
REQ_NEG <- 0.70
REQ_COMPLETE <- 1.00


PASS_GENE <- (
  GENE_COUNT_COVERAGE >=
  REQ_GENE
)

PASS_ABS <- (
  ABS_WEIGHT_COVERAGE >=
  REQ_ABS
)

PASS_POS <- (
  POS_COVERAGE >=
  REQ_POS
)

PASS_NEG <- (
  NEG_COVERAGE >=
  REQ_NEG
)

PASS_COMPLETE <- (
  ACTUAL_SAMPLE_COMPLETENESS >=
  REQ_COMPLETE
)


TECHNICAL_PASS <- all(
  PASS_GENE,
  PASS_ABS,
  PASS_POS,
  PASS_NEG,
  PASS_COMPLETE
)


# ------------------------------------------------------------
# 12. SUMMARY
# ------------------------------------------------------------

SUMMARY <- data.frame(

  metric = c(
    "Frozen_score_gene_count",
    "Frozen_genes_with_symbol",
    "Maximum_possible_symbol_mapping_coverage",
    "Mapped_gene_count",
    "Gene_count_coverage",
    "Required_gene_count_coverage",
    "Absolute_weight_coverage",
    "Required_absolute_weight_coverage",
    "Positive_weight_coverage",
    "Required_positive_weight_coverage",
    "Negative_weight_coverage",
    "Required_negative_weight_coverage",
    "Minimum_sample_completeness",
    "Technical_PASS"
  ),

  value = c(
    24,
    sum(HAS_FROZEN_SYMBOL),
    mean(HAS_FROZEN_SYMBOL),
    sum(MAPPED),
    GENE_COUNT_COVERAGE,
    REQ_GENE,
    ABS_WEIGHT_COVERAGE,
    REQ_ABS,
    POS_COVERAGE,
    REQ_POS,
    NEG_COVERAGE,
    REQ_NEG,
    ACTUAL_SAMPLE_COMPLETENESS,
    as.numeric(TECHNICAL_PASS)
  ),

  stringsAsFactors = FALSE
)


# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

OUT_DIR <- file.path(
  PROJECT_ROOT,
  "09_external_validation",
  "gse282552"
)

MAPPING_OUT <- file.path(
  OUT_DIR,
  "STEP09H_GSE282552_frozen_gene_mapping_v1.csv"
)

GATE_OUT <- file.path(
  OUT_DIR,
  "STEP09H_GSE282552_technical_gate_v1.csv"
)

NOTE_OUT <- file.path(
  OUT_DIR,
  "STEP09H_GSE282552_TECHNICAL_GATE_NOTES_v1.txt"
)


write.csv(
  MAPPING,
  MAPPING_OUT,
  row.names = FALSE
)

write.csv(
  SUMMARY,
  GATE_OUT,
  row.names = FALSE
)


writeLines(
  c(
    "STEP 09H — GSE282552 TECHNICAL GATE",
    "",
    paste0(
      "TECHNICAL_PASS: ",
      TECHNICAL_PASS
    ),
    "",
    paste0(
      "Frozen genes with frozen symbols: ",
      sum(HAS_FROZEN_SYMBOL),
      "/24"
    ),
    paste0(
      "Maximum possible exact-symbol coverage: ",
      signif(
        mean(HAS_FROZEN_SYMBOL),
        6
      )
    ),
    paste0(
      "Actually mapped genes: ",
      sum(MAPPED),
      "/24"
    ),
    paste0(
      "Absolute-weight coverage: ",
      signif(
        ABS_WEIGHT_COVERAGE,
        6
      )
    ),
    "",
    paste(
      "The processed GSE282552 matrix contains PB isoform",
      "identifiers with gene-symbol suffixes but no Ensembl",
      "gene or transcript identifiers."
    ),
    paste(
      "Only exact frozen gene symbols were therefore used."
    ),
    paste(
      "No synonym, external annotation, or post-outcome",
      "identifier rescue was performed."
    ),
    "",
    "Diagnosis analyzed: NO",
    "SCZ-MSS calculated: NO",
    "AUROC calculated: NO"
  ),
  NOTE_OUT
)


# ------------------------------------------------------------
# 14. OUTPUT
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME


cat("\n============================================================\n")
cat("STEP 09H — GSE282552 TECHNICAL GATE COMPLETE\n")
cat("============================================================\n")


print(
  MAPPING[
    ,
    c(
      "frozen_gene_id",
      "frozen_symbol",
      "frozen_weight",
      "external_isoform_count",
      "mapping_status"
    )
  ],
  row.names = FALSE
)


cat(
  "\n--- TECHNICAL GATE ---\n"
)

print(
  SUMMARY,
  row.names = FALSE,
  digits = 6
)


cat(
  "\nTECHNICAL_PASS: ",
  TECHNICAL_PASS,
  "\n",
  sep = ""
)

cat(
  "Diagnosis analyzed: NO\n"
)

cat(
  "SCZ-MSS calculated: NO\n"
)

cat(
  "Runtime: ",
  round(RUNTIME, 2),
  " seconds\n",
  sep = ""
)


cat("\n============================================================\n")
cat("STEP 09H — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Required files: PASS
Frozen genes with symbol: 18/24
Maximum possible exact-symbol coverage: 0.75
Frozen mapping policy SHA256: c5b6f47869246d0a0ed2ed8f6422c0207d21aff994f36602d53e458157e49d75
Frozen gate policy SHA256: 3eb578e2d0b3b805502c96c4fbff7319a37b4fdcd4da9d1184bc32708adfff1d
GSE282552 matrix: 56163 isoforms x 18 samples
Rows containing ENSG: 0
Rows containing ENST: 0
Unique external gene-symbol suffixes: 10596

STEP 09H — GSE282552 TECHNICAL GATE COMPLETE
  frozen_gene_id frozen_symbol frozen_weight external_isoform_count
 ENSG00000100092        SH3BP1  -0.040294344                      3
 ENSG00000117877        CD3EAP  -0.048996635                      1
 ENSG00000196565          HBG2   0.027853367                      0
 ENSG00000251521       IMPA1P1   0.069213048                      0
 ENSG00000260526          <NA>   0.039334477                      0
 ENSG00000166428          PLD4  -0.006185861                      1
 ENSG00000277869          <NA>  

In [53]:
# ============================================================
# STEP 09I.0 — RESUME GSE208338 LARGE SERIES-MATRIX DOWNLOAD
#
# The file is ~1.47 GB.
# Previous R download stopped at the default 60-second timeout.
#
# This cell:
#   1. keeps the existing partial file
#   2. resumes with Windows curl.exe using HTTP Range requests
#   3. retries transient failures
#   4. verifies final remote/local byte size
#   5. verifies gzip integrity
#
# NO expression analysis.
# NO diagnosis analysis.
# ============================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

START_TIME <- proc.time()[3]

# ------------------------------------------------------------
# 0. PROJECT
# ------------------------------------------------------------

SENTINEL <- file.path(
  PROJECT_ROOT,
  ".project_id"
)

if (!file.exists(SENTINEL)) {
  stop("Project sentinel missing.")
}

if (!identical(
  trimws(readLines(SENTINEL, warn = FALSE)[1]),
  PROJECT_ID
)) {
  stop("Project ID mismatch.")
}

cat("Project sentinel: PASS\n")


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

RAW_DIR <- file.path(
  PROJECT_ROOT,
  "01_raw",
  "gse208338"
)

dir.create(
  RAW_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

MATRIX_URL <- paste0(
  "https://ftp.ncbi.nlm.nih.gov/geo/series/",
  "GSE208nnn/GSE208338/matrix/",
  "GSE208338_series_matrix.txt.gz"
)

MATRIX_PATH <- file.path(
  RAW_DIR,
  "GSE208338_series_matrix.txt.gz"
)


# ------------------------------------------------------------
# 2. EXISTING PARTIAL DOWNLOAD
# ------------------------------------------------------------

if (file.exists(MATRIX_PATH)) {

  EXISTING_BYTES <- file.info(
    MATRIX_PATH
  )$size

} else {

  EXISTING_BYTES <- 0
}


cat(
  "Existing local bytes: ",
  format(
    EXISTING_BYTES,
    big.mark = ",",
    scientific = FALSE
  ),
  "\n",
  sep = ""
)

cat(
  "Existing local size: ",
  round(
    EXISTING_BYTES / 1024^2,
    2
  ),
  " MB\n",
  sep = ""
)


# ------------------------------------------------------------
# 3. FIND CURL
# ------------------------------------------------------------

CURL <- Sys.which(
  "curl"
)

if (!nzchar(CURL)) {

  CURL_WINDOWS <- "C:/Windows/System32/curl.exe"

  if (file.exists(CURL_WINDOWS)) {

    CURL <- CURL_WINDOWS

  } else {

    stop(
      paste(
        "curl.exe not found.",
        "Do not delete the partial file."
      )
    )
  }
}


cat(
  "curl executable: ",
  CURL,
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 4. REMOTE SIZE
#
# curl -I follows redirects and reads Content-Length.
# ------------------------------------------------------------

HEADER_FILE <- tempfile(
  fileext = ".txt"
)

HEADER_STATUS <- system2(
  CURL,
  args = c(
    "-L",
    "-s",
    "-I",
    "--retry",
    "3",
    "--retry-delay",
    "3",
    shQuote(MATRIX_URL)
  ),
  stdout = HEADER_FILE,
  stderr = HEADER_FILE
)


HEADER_LINES <- readLines(
  HEADER_FILE,
  warn = FALSE
)

unlink(
  HEADER_FILE
)


CONTENT_LENGTH_LINES <- grep(
  "^content-length:",
  HEADER_LINES,
  value = TRUE,
  ignore.case = TRUE
)


REMOTE_BYTES <- NA_real_


if (length(CONTENT_LENGTH_LINES) > 0L) {

  REMOTE_BYTES <- suppressWarnings(
    as.numeric(
      trimws(
        sub(
          "^content-length:",
          "",
          tail(
            CONTENT_LENGTH_LINES,
            1
          ),
          ignore.case = TRUE
        )
      )
    )
  )
}


cat(
  "Remote reported bytes: ",
  ifelse(
    is.na(REMOTE_BYTES),
    "not resolved",
    format(
      REMOTE_BYTES,
      big.mark = ",",
      scientific = FALSE
    )
  ),
  "\n",
  sep = ""
)


# ------------------------------------------------------------
# 5. RESUME DOWNLOAD
#
# -C -     = resume from current local byte position
# --retry  = retry temporary network failures
# --speed-* prevents indefinite stalled connection
# ------------------------------------------------------------

cat("\nResuming large GEO download...\n")
cat("This may take some time depending on connection speed.\n\n")


DOWNLOAD_STATUS <- system2(
  CURL,
  args = c(
    "-L",
    "-C",
    "-",
    "--retry",
    "10",
    "--retry-delay",
    "5",
    "--retry-all-errors",
    "--connect-timeout",
    "60",
    "--speed-time",
    "120",
    "--speed-limit",
    "1024",
    "-o",
    shQuote(
      normalizePath(
        MATRIX_PATH,
        winslash = "/",
        mustWork = FALSE
      )
    ),
    shQuote(
      MATRIX_URL
    )
  )
)


if (DOWNLOAD_STATUS != 0L) {
  stop(
    paste0(
      "curl resume failed with status ",
      DOWNLOAD_STATUS,
      ". Partial file has been preserved."
    )
  )
}


# ------------------------------------------------------------
# 6. FINAL SIZE
# ------------------------------------------------------------

if (!file.exists(MATRIX_PATH)) {
  stop("Downloaded file not found.")
}


FINAL_BYTES <- file.info(
  MATRIX_PATH
)$size


cat(
  "\nFinal local bytes: ",
  format(
    FINAL_BYTES,
    big.mark = ",",
    scientific = FALSE
  ),
  "\n",
  sep = ""
)

cat(
  "Final local size: ",
  round(
    FINAL_BYTES / 1024^3,
    3
  ),
  " GB\n",
  sep = ""
)


if (
  !is.na(REMOTE_BYTES) &&
  FINAL_BYTES != REMOTE_BYTES
) {
  stop(
    paste0(
      "Download size mismatch. Local=",
      FINAL_BYTES,
      ", remote=",
      REMOTE_BYTES,
      ". Partial file preserved."
    )
  )
}


# ------------------------------------------------------------
# 7. GZIP INTEGRITY TEST
#
# gzip -t if available.
# Otherwise attempt complete streaming decompression through R.
# ------------------------------------------------------------

GZIP <- Sys.which(
  "gzip"
)


if (nzchar(GZIP)) {

  GZIP_STATUS <- system2(
    GZIP,
    args = c(
      "-t",
      shQuote(
        normalizePath(
          MATRIX_PATH,
          winslash = "/",
          mustWork = TRUE
        )
      )
    )
  )

  if (GZIP_STATUS != 0L) {
    stop("gzip integrity test FAILED.")
  }

  GZIP_PASS <- TRUE

} else {

  CON <- gzfile(
    MATRIX_PATH,
    open = "rt"
  )

  GZIP_PASS <- tryCatch(
    {
      repeat {

        X <- readLines(
          CON,
          n = 100000,
          warn = FALSE
        )

        if (length(X) == 0L) {
          break
        }
      }

      TRUE
    },
    error = function(e) {
      FALSE
    },
    finally = {
      close(CON)
    }
  )

  if (!GZIP_PASS) {
    stop("gzip integrity test FAILED.")
  }
}


# ------------------------------------------------------------
# 8. FINAL
# ------------------------------------------------------------

RUNTIME <- proc.time()[3] -
  START_TIME


cat("\n============================================================\n")
cat("STEP 09I.0 — GSE208338 DOWNLOAD COMPLETE\n")
cat("============================================================\n")

cat(
  "Previous partial download retained/resumed: ",
  EXISTING_BYTES > 0,
  "\n",
  sep = ""
)

cat(
  "Final size: ",
  round(
    FINAL_BYTES / 1024^3,
    3
  ),
  " GB\n",
  sep = ""
)

cat(
  "Remote/local size agreement: ",
  ifelse(
    is.na(REMOTE_BYTES),
    "REMOTE SIZE NOT RESOLVED",
    FINAL_BYTES == REMOTE_BYTES
  ),
  "\n",
  sep = ""
)

cat(
  "GZIP integrity: PASS\n"
)

cat(
  "Diagnosis analyzed: NO\n"
)

cat(
  "Runtime: ",
  round(
    RUNTIME,
    2
  ),
  " seconds\n",
  sep = ""
)

cat("\n============================================================\n")
cat("STEP 09I.0 — CELL PASS\n")
cat("============================================================\n")

Project sentinel: PASS
Existing local bytes: 190,435,566
Existing local size: 181.61 MB
curl executable: C:\WINDOWS\SYSTEM32\curl.exe
Remote reported bytes: 1,469,412,320

Resuming large GEO download...
This may take some time depending on connection speed.


Final local bytes: 1,469,412,320
Final local size: 1.368 GB

STEP 09I.0 — GSE208338 DOWNLOAD COMPLETE
Previous partial download retained/resumed: TRUE
Final size: 1.368 GB
Remote/local size agreement: TRUE
GZIP integrity: PASS
Diagnosis analyzed: NO
Runtime: 561.86 seconds

STEP 09I.0 — CELL PASS


In [56]:
# ==================================================================================================
# STEP 09I.2 — GSE208338 / GPL5188 OUTCOME-BLIND FROZEN-GENE MAPPING FEASIBILITY AUDIT
# CORRECTED FULL CELL
# --------------------------------------------------------------------------------------------------
# IMPORTANT:
#   STEP09I.1 already established from the local 1.368-GB series matrix:
#       Platform              = GPL5188
#       Matrix ID column      = ID_REF
#       Sample columns        = 169
#       Matrix identifier     = NUMERIC_AFFY_PROBESET_OR_TRANSCRIPT_CLUSTER
#
#   Therefore this cell DOES NOT reopen / restream the 1.368-GB matrix.
#   It DOES NOT continue the incorrect GPL5175 download.
#   It DOES NOT access diagnosis / phenotype labels.
#   It DOES NOT calculate SCZ-MSS.
#
# PURPOSE:
#   1. Use GPL5188-compatible official Human Exon 1.0 ST Bioconductor annotation.
#   2. Audit exact frozen Ensembl mapping.
#   3. Audit exact official-symbol mapping separately.
#   4. Identify ambiguous probe-to-frozen-gene mappings.
#   5. Determine whether annotation-level theoretical coverage can satisfy
#      the frozen external technical gate.
#
# THIS IS NOT YET THE FINAL GSE208338 TECHNICAL PASS.
# Actual matrix-row presence, expression completeness, and deterministic
# probe-to-gene reduction must still be checked afterward if feasibility passes.
# ==================================================================================================

options(timeout = max(1200, getOption("timeout")))

cat(
    paste0(
        "\n",
        paste(rep("=", 110), collapse = ""), "\n",
        "STEP 09I.2 — GSE208338 / GPL5188 FROZEN-GENE MAPPING FEASIBILITY AUDIT\n",
        paste(rep("=", 110), collapse = ""), "\n"
    )
)

# --------------------------------------------------------------------------------------------------
# 09I.2A — PROJECT CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"

EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

SERIES_PATH <- file.path(
    PROJECT_ROOT,
    "01_raw",
    "gse208338",
    "GSE208338_series_matrix.txt.gz"
)

WRONG_GPL5175_PATH <- file.path(
    PROJECT_ROOT,
    "01_raw",
    "gse208338",
    "GPL5175_family.soft.gz"
)

WEIGHT_PATH <- file.path(
    PROJECT_ROOT,
    "08_score_lock",
    "SCZ_MSS_v1_gene_weights.csv"
)

MAPPING_POLICY_PATH <- file.path(
    PROJECT_ROOT,
    "08_score_lock",
    "SCZ_MSS_v1_mapping_policy.json"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "gse208338"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_GENE_MAP <- file.path(
    OUT_DIR,
    "STEP09I2_GPL5188_frozen_gene_mapping_feasibility_v1.csv"
)

OUT_PROBE_MAP <- file.path(
    OUT_DIR,
    "STEP09I2_GPL5188_probe_mapping_audit_v1.csv"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP09I2_GPL5188_mapping_feasibility_summary_v1.txt"
)

# --------------------------------------------------------------------------------------------------
# 09I.2B — PROJECT / INPUT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

observed_project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(observed_project_id) != 1L ||
    !identical(observed_project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(observed_project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

if (!file.exists(SERIES_PATH)) {
    stop(
        "GSE208338 series matrix missing: ",
        SERIES_PATH
    )
}

series_bytes <- file.info(SERIES_PATH)$size

cat("Series matrix found: PASS\n")
cat(
    "Series matrix local size: ",
    format(
        series_bytes / 1024^3,
        digits = 6
    ),
    " GB\n",
    sep = ""
)

if (!file.exists(WEIGHT_PATH)) {
    stop(
        "Frozen score weight file missing: ",
        WEIGHT_PATH
    )
}

if (!file.exists(MAPPING_POLICY_PATH)) {
    stop(
        "Frozen mapping-policy file missing: ",
        MAPPING_POLICY_PATH
    )
}

cat("Frozen weight file: PASS\n")
cat("Frozen mapping policy: PASS\n")

# --------------------------------------------------------------------------------------------------
# STEP09I.1 VERIFIED FACTS
#
# These were obtained from the actual local GSE208338 series matrix in the immediately preceding step.
# We deliberately reuse them instead of rereading a 1.368-GB gzip file.
# --------------------------------------------------------------------------------------------------

SERIES_PLATFORM_VERIFIED <- "GPL5188"
MATRIX_ID_COLUMN_VERIFIED <- "ID_REF"
N_SAMPLE_COLUMNS_VERIFIED <- 169L
MATRIX_IDENTIFIER_TYPE_VERIFIED <- "NUMERIC_AFFY_PROBESET_OR_TRANSCRIPT_CLUSTER"

cat("\nSTEP09I.1 VERIFIED MATRIX FACTS\n")
cat("Platform: ", SERIES_PLATFORM_VERIFIED, "\n", sep = "")
cat("ID column: ", MATRIX_ID_COLUMN_VERIFIED, "\n", sep = "")
cat("Sample columns: ", N_SAMPLE_COLUMNS_VERIFIED, "\n", sep = "")
cat(
    "Identifier type: ",
    MATRIX_IDENTIFIER_TYPE_VERIFIED,
    "\n",
    sep = ""
)

if (file.exists(WRONG_GPL5175_PATH)) {

    wrong_bytes <- file.info(WRONG_GPL5175_PATH)$size

    cat(
        "\nPartial incorrect GPL5175 download detected: ",
        format(
            wrong_bytes / 1024^3,
            digits = 6
        ),
        " GB\n",
        sep = ""
    )

    cat(
        "GPL5175 status: IGNORED — not used, not resumed, not deleted by this cell.\n"
    )
}

# --------------------------------------------------------------------------------------------------
# 09I.2C — INSTALL / LOAD OFFICIAL GPL5188-COMPATIBLE ANNOTATION
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nOFFICIAL PLATFORM ANNOTATION\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

if (!requireNamespace("BiocManager", quietly = TRUE)) {

    install.packages(
        "BiocManager",
        repos = "https://cloud.r-project.org"
    )
}

needed_packages <- c(
    "AnnotationDbi",
    "huex10stprobeset.db"
)

missing_packages <- needed_packages[
    !vapply(
        needed_packages,
        requireNamespace,
        quietly = TRUE,
        FUN.VALUE = logical(1)
    )
]

if (length(missing_packages) > 0L) {

    cat(
        "Installing missing Bioconductor package(s): ",
        paste(missing_packages, collapse = ", "),
        "\n",
        sep = ""
    )

    BiocManager::install(
        missing_packages,
        ask = FALSE,
        update = FALSE
    )
}

if (!requireNamespace("AnnotationDbi", quietly = TRUE)) {
    stop("AnnotationDbi installation/loading failed.")
}

if (!requireNamespace("huex10stprobeset.db", quietly = TRUE)) {
    stop(
        "huex10stprobeset.db installation/loading failed."
    )
}

DB <- huex10stprobeset.db::huex10stprobeset.db

db_keytypes <- AnnotationDbi::keytypes(DB)
db_columns <- AnnotationDbi::columns(DB)

cat(
    "Annotation package: huex10stprobeset.db\n"
)

cat(
    "Package version: ",
    as.character(
        utils::packageVersion(
            "huex10stprobeset.db"
        )
    ),
    "\n",
    sep = ""
)

cat(
    "Available relevant keytypes: ",
    paste(
        intersect(
            c(
                "PROBEID",
                "ENSEMBL",
                "SYMBOL",
                "ENTREZID"
            ),
            db_keytypes
        ),
        collapse = ", "
    ),
    "\n",
    sep = ""
)

cat(
    "Available relevant columns: ",
    paste(
        intersect(
            c(
                "PROBEID",
                "ENSEMBL",
                "SYMBOL",
                "ENTREZID"
            ),
            db_columns
        ),
        collapse = ", "
    ),
    "\n",
    sep = ""
)

required_keytypes <- c(
    "PROBEID",
    "ENSEMBL",
    "SYMBOL"
)

if (!all(required_keytypes %in% db_keytypes)) {

    stop(
        "Required annotation keytypes are unavailable.\n",
        "Required: ",
        paste(required_keytypes, collapse = ", "),
        "\nObserved keytypes include: ",
        paste(db_keytypes, collapse = ", ")
    )
}

if (!all(required_keytypes %in% db_columns)) {

    stop(
        "Required annotation columns are unavailable.\n",
        "Required: ",
        paste(required_keytypes, collapse = ", "),
        "\nObserved columns include: ",
        paste(db_columns, collapse = ", ")
    )
}

cat("Required annotation fields: PASS\n")

# --------------------------------------------------------------------------------------------------
# 09I.2D — LOAD FROZEN SCORE
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nFROZEN SCORE INPUT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

weights <- read.csv(
    WEIGHT_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(weights) != 24L) {

    stop(
        "Frozen score must contain exactly 24 rows.\n",
        "Observed rows: ",
        nrow(weights)
    )
}

cat("Frozen score rows: 24 — PASS\n")
cat(
    "Available columns: ",
    paste(names(weights), collapse = " | "),
    "\n",
    sep = ""
)

normalize_name <- function(x) {

    tolower(
        gsub(
            "[^a-z0-9]+",
            "_",
            x
        )
    )
}

normalized_names <- normalize_name(
    names(weights)
)

find_column <- function(priority_patterns) {

    for (pat in priority_patterns) {

        hit <- grep(
            pat,
            normalized_names,
            perl = TRUE
        )

        if (length(hit) > 0L) {
            return(names(weights)[hit[1]])
        }
    }

    NA_character_
}

ensembl_col <- find_column(
    c(
        "^ensembl_gene_id$",
        "^stable_ensembl_id$",
        "^ensembl_id$",
        "^ensembl$",
        "ensembl"
    )
)

symbol_col <- find_column(
    c(
        "^gene_symbol$",
        "^symbol$",
        "^hgnc_symbol$",
        "symbol"
    )
)

# Prefer explicitly normalized weight columns.
weight_col <- find_column(
    c(
        "^normalized_weight$",
        "^weight_normalized$",
        "^norm_weight$",
        "^frozen_weight$"
    )
)

# If not explicitly named, identify numeric weight/coef/beta column whose |weights| sum to ~1.
if (is.na(weight_col)) {

    numeric_columns <- names(weights)[
        vapply(
            weights,
            is.numeric,
            FUN.VALUE = logical(1)
        )
    ]

    candidate_weight_columns <- numeric_columns[
        grepl(
            "weight|coef|beta",
            normalize_name(numeric_columns)
        )
    ]

    if (length(candidate_weight_columns) == 0L) {

        stop(
            "Could not identify frozen weight column."
        )
    }

    candidate_abs_sums <- vapply(
        candidate_weight_columns,
        function(nm) {

            sum(
                abs(weights[[nm]]),
                na.rm = TRUE
            )
        },
        FUN.VALUE = numeric(1)
    )

    weight_col <- candidate_weight_columns[
        which.min(
            abs(candidate_abs_sums - 1)
        )
    ]
}

if (is.na(ensembl_col)) {

    stop(
        "Could not identify frozen Ensembl-ID column."
    )
}

if (is.na(symbol_col)) {

    stop(
        "Could not identify frozen gene-symbol column."
    )
}

frozen_weight_sum <- sum(
    abs(
        as.numeric(
            weights[[weight_col]]
        )
    ),
    na.rm = TRUE
)

if (
    !is.finite(frozen_weight_sum) ||
    abs(frozen_weight_sum - 1) > 1e-3
) {

    stop(
        "Selected frozen weight column does not have normalized absolute sum ~1.\n",
        "Column: ", weight_col, "\n",
        "Observed sum(abs(weight)): ",
        frozen_weight_sum
    )
}

cat(
    "Ensembl column: ",
    ensembl_col,
    "\n",
    sep = ""
)

cat(
    "Symbol column: ",
    symbol_col,
    "\n",
    sep = ""
)

cat(
    "Frozen normalized-weight column: ",
    weight_col,
    "\n",
    sep = ""
)

cat(
    "sum(abs(frozen weight)): ",
    sprintf("%.10f", frozen_weight_sum),
    "\n",
    sep = ""
)

frozen <- data.frame(
    frozen_row = seq_len(nrow(weights)),
    ensembl_original = as.character(
        weights[[ensembl_col]]
    ),
    symbol = as.character(
        weights[[symbol_col]]
    ),
    weight = as.numeric(
        weights[[weight_col]]
    ),
    stringsAsFactors = FALSE
)

frozen$ensembl_original <- trimws(
    frozen$ensembl_original
)

frozen$symbol <- trimws(
    frozen$symbol
)

# Stable Ensembl ID = remove version suffix only.
frozen$ensembl_stable <- sub(
    "\\.[0-9]+$",
    "",
    frozen$ensembl_original
)

bad_text <- c(
    "",
    "NA",
    "N/A",
    "NULL",
    "."
)

frozen$ensembl_stable[
    is.na(frozen$ensembl_stable) |
    frozen$ensembl_stable %in% bad_text
] <- NA_character_

frozen$symbol[
    is.na(frozen$symbol) |
    frozen$symbol %in% bad_text
] <- NA_character_

if (anyNA(frozen$weight)) {

    stop(
        "Missing numeric value detected in frozen weights."
    )
}

if (
    anyDuplicated(
        frozen$ensembl_stable[
            !is.na(frozen$ensembl_stable)
        ]
    ) > 0L
) {

    stop(
        "Duplicate stable Ensembl IDs detected in frozen score."
    )
}

# --------------------------------------------------------------------------------------------------
# 09I.2E — PRINT RELEVANT FROZEN MAPPING POLICY
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nFROZEN MAPPING POLICY — RELEVANT TEXT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

mapping_policy_text <- readLines(
    MAPPING_POLICY_PATH,
    warn = FALSE
)

relevant_policy_lines <- mapping_policy_text[
    grepl(
        paste(
            c(
                "ensembl",
                "symbol",
                "exact",
                "unique",
                "ambigu",
                "fallback",
                "synonym",
                "probe",
                "aggregate",
                "collapse"
            ),
            collapse = "|"
        ),
        mapping_policy_text,
        ignore.case = TRUE
    )
]

if (length(relevant_policy_lines) == 0L) {

    cat(
        "[No matching policy keywords were automatically extracted.]\n"
    )

} else {

    cat(
        paste(
            relevant_policy_lines,
            collapse = "\n"
        ),
        "\n"
    )
}

# --------------------------------------------------------------------------------------------------
# 09I.2F — EXACT ENSEMBL -> GPL5188 PROBE-SET MAPPING
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nEXACT STABLE-ENSEMBL MAPPING\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

ensembl_keys <- unique(
    frozen$ensembl_stable[
        !is.na(frozen$ensembl_stable)
    ]
)

ens_raw <- AnnotationDbi::select(
    DB,
    keys = ensembl_keys,
    keytype = "ENSEMBL",
    columns = c(
        "PROBEID",
        "SYMBOL",
        "ENTREZID"
    )
)

names(ens_raw)[
    names(ens_raw) == "ENSEMBL"
] <- "ensembl_stable"

ens_raw <- ens_raw[
    !is.na(ens_raw$PROBEID) &
    nzchar(as.character(ens_raw$PROBEID)),
    ,
    drop = FALSE
]

ens_map <- merge(
    frozen[
        ,
        c(
            "frozen_row",
            "ensembl_stable",
            "symbol",
            "weight"
        )
    ],
    ens_raw,
    by = "ensembl_stable",
    all = FALSE
)

if (nrow(ens_map) > 0L) {

    ens_map <- unique(
        ens_map[
            ,
            c(
                "frozen_row",
                "ensembl_stable",
                "symbol",
                "weight",
                "PROBEID"
            )
        ]
    )

    names(ens_map)[
        names(ens_map) == "PROBEID"
    ] <- "probe_id"

    ens_map$route <- "EXACT_ENSEMBL"

} else {

    ens_map <- data.frame(
        frozen_row = integer(0),
        ensembl_stable = character(0),
        symbol = character(0),
        weight = numeric(0),
        probe_id = character(0),
        route = character(0),
        stringsAsFactors = FALSE
    )
}

cat(
    "Frozen genes with >=1 exact Ensembl annotation mapping: ",
    length(unique(ens_map$frozen_row)),
    "/24\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 09I.2G — EXACT OFFICIAL SYMBOL -> GPL5188 PROBE-SET MAPPING
#
# IMPORTANT:
#   This is a separate feasibility audit.
#   No ALIAS / synonym rescue is allowed here.
#   Symbol mapping does NOT automatically override frozen Ensembl preference.
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nEXACT OFFICIAL-SYMBOL MAPPING\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

symbol_keys <- unique(
    frozen$symbol[
        !is.na(frozen$symbol)
    ]
)

sym_raw <- AnnotationDbi::select(
    DB,
    keys = symbol_keys,
    keytype = "SYMBOL",
    columns = c(
        "PROBEID",
        "ENSEMBL",
        "ENTREZID"
    )
)

names(sym_raw)[
    names(sym_raw) == "SYMBOL"
] <- "symbol"

sym_raw <- sym_raw[
    !is.na(sym_raw$PROBEID) &
    nzchar(as.character(sym_raw$PROBEID)),
    ,
    drop = FALSE
]

sym_map <- merge(
    frozen[
        ,
        c(
            "frozen_row",
            "ensembl_stable",
            "symbol",
            "weight"
        )
    ],
    sym_raw,
    by = "symbol",
    all = FALSE
)

if (nrow(sym_map) > 0L) {

    sym_map <- unique(
        sym_map[
            ,
            c(
                "frozen_row",
                "ensembl_stable",
                "symbol",
                "weight",
                "PROBEID"
            )
        ]
    )

    names(sym_map)[
        names(sym_map) == "PROBEID"
    ] <- "probe_id"

    sym_map$route <- "EXACT_OFFICIAL_SYMBOL"

} else {

    sym_map <- data.frame(
        frozen_row = integer(0),
        ensembl_stable = character(0),
        symbol = character(0),
        weight = numeric(0),
        probe_id = character(0),
        route = character(0),
        stringsAsFactors = FALSE
    )
}

cat(
    "Frozen genes with >=1 exact official-symbol annotation mapping: ",
    length(unique(sym_map$frozen_row)),
    "/24\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 09I.2H — COMBINE EXACT ROUTES FOR FEASIBILITY ONLY
#
# A probe ID mapping to >1 DIFFERENT frozen gene is ambiguous and excluded.
# This combined mapping is only a theoretical feasibility audit.
# It is NOT yet the final frozen mapping rule.
# --------------------------------------------------------------------------------------------------

all_map <- rbind(
    ens_map,
    sym_map
)

all_map <- unique(
    all_map
)

if (nrow(all_map) == 0L) {

    stop(
        "No exact Ensembl or exact official-symbol mappings found for the frozen 24-gene set."
    )
}

probe_gene_pairs <- unique(
    all_map[
        ,
        c(
            "probe_id",
            "frozen_row"
        )
    ]
)

probe_gene_counts <- aggregate(
    frozen_row ~ probe_id,
    data = probe_gene_pairs,
    FUN = function(x) {
        length(unique(x))
    }
)

names(probe_gene_counts)[2] <- "n_frozen_genes_per_probe"

all_map <- merge(
    all_map,
    probe_gene_counts,
    by = "probe_id",
    all.x = TRUE,
    sort = FALSE
)

all_map$probe_unambiguous <- (
    all_map$n_frozen_genes_per_probe == 1L
)

clean_union_map <- all_map[
    all_map$probe_unambiguous,
    ,
    drop = FALSE
]

n_ambiguous_probe_ids <- length(
    unique(
        all_map$probe_id[
            !all_map$probe_unambiguous
        ]
    )
)

# --------------------------------------------------------------------------------------------------
# 09I.2I — PER-FROZEN-GENE MAPPING AUDIT
# --------------------------------------------------------------------------------------------------

count_unique_probes <- function(map_df, frozen_row_value) {

    if (nrow(map_df) == 0L) {
        return(0L)
    }

    length(
        unique(
            map_df$probe_id[
                map_df$frozen_row == frozen_row_value
            ]
        )
    )
}

gene_audit <- frozen

gene_audit$n_exact_ensembl_probes <- vapply(
    gene_audit$frozen_row,
    function(i) {
        count_unique_probes(
            ens_map,
            i
        )
    },
    FUN.VALUE = integer(1)
)

gene_audit$n_exact_symbol_probes <- vapply(
    gene_audit$frozen_row,
    function(i) {
        count_unique_probes(
            sym_map,
            i
        )
    },
    FUN.VALUE = integer(1)
)

gene_audit$n_unambiguous_exact_union_probes <- vapply(
    gene_audit$frozen_row,
    function(i) {
        count_unique_probes(
            clean_union_map,
            i
        )
    },
    FUN.VALUE = integer(1)
)

gene_audit$ensembl_annotation_mappable <- (
    gene_audit$n_exact_ensembl_probes > 0L
)

gene_audit$symbol_annotation_mappable <- (
    gene_audit$n_exact_symbol_probes > 0L
)

gene_audit$best_case_exact_annotation_mappable <- (
    gene_audit$n_unambiguous_exact_union_probes > 0L
)

gene_audit$multiple_probe_resolution_needed <- (
    gene_audit$n_unambiguous_exact_union_probes > 1L
)

# --------------------------------------------------------------------------------------------------
# 09I.2J — FROZEN COVERAGE METRICS
# --------------------------------------------------------------------------------------------------

coverage_metrics <- function(mapped, weight) {

    mapped <- as.logical(mapped)
    weight <- as.numeric(weight)

    total_abs <- sum(
        abs(weight)
    )

    total_positive_abs <- sum(
        abs(weight[weight > 0])
    )

    total_negative_abs <- sum(
        abs(weight[weight < 0])
    )

    mapped_abs <- sum(
        abs(weight[mapped])
    )

    mapped_positive_abs <- sum(
        abs(
            weight[
                mapped &
                weight > 0
            ]
        )
    )

    mapped_negative_abs <- sum(
        abs(
            weight[
                mapped &
                weight < 0
            ]
        )
    )

    c(
        gene_count_coverage =
            sum(mapped) /
            length(mapped),

        absolute_weight_coverage =
            mapped_abs /
            total_abs,

        positive_weight_coverage =
            mapped_positive_abs /
            total_positive_abs,

        negative_weight_coverage =
            mapped_negative_abs /
            total_negative_abs
    )
}

cov_ens <- coverage_metrics(
    gene_audit$ensembl_annotation_mappable,
    gene_audit$weight
)

cov_sym <- coverage_metrics(
    gene_audit$symbol_annotation_mappable,
    gene_audit$weight
)

cov_union <- coverage_metrics(
    gene_audit$best_case_exact_annotation_mappable,
    gene_audit$weight
)

FROZEN_THRESHOLDS <- c(
    gene_count_coverage = 0.80,
    absolute_weight_coverage = 0.80,
    positive_weight_coverage = 0.70,
    negative_weight_coverage = 0.70
)

coverage_pass <- function(metrics) {

    all(
        metrics[
            names(FROZEN_THRESHOLDS)
        ] >= FROZEN_THRESHOLDS
    )
}

ens_theoretical_pass <- coverage_pass(
    cov_ens
)

sym_theoretical_pass <- coverage_pass(
    cov_sym
)

union_theoretical_pass <- coverage_pass(
    cov_union
)

# --------------------------------------------------------------------------------------------------
# 09I.2K — OUTPUT FILES
# --------------------------------------------------------------------------------------------------

write.csv(
    gene_audit,
    OUT_GENE_MAP,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    all_map[
        order(
            all_map$frozen_row,
            all_map$probe_id,
            all_map$route
        ),
        ,
        drop = FALSE
    ],
    OUT_PROBE_MAP,
    row.names = FALSE,
    quote = TRUE
)

format_metrics <- function(x) {

    paste(
        paste0(
            names(x),
            "=",
            sprintf("%.6f", x)
        ),
        collapse = "; "
    )
}

summary_lines <- c(

    "STEP 09I.2 — GSE208338 / GPL5188 FROZEN-GENE MAPPING FEASIBILITY AUDIT",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    paste0(
        "Series platform reused from STEP09I.1: ",
        SERIES_PLATFORM_VERIFIED
    ),

    paste0(
        "Matrix ID column reused from STEP09I.1: ",
        MATRIX_ID_COLUMN_VERIFIED
    ),

    paste0(
        "Sample columns reused from STEP09I.1: ",
        N_SAMPLE_COLUMNS_VERIFIED
    ),

    paste0(
        "Matrix identifier type reused from STEP09I.1: ",
        MATRIX_IDENTIFIER_TYPE_VERIFIED
    ),

    paste0(
        "Annotation package: huex10stprobeset.db ",
        as.character(
            utils::packageVersion(
                "huex10stprobeset.db"
            )
        )
    ),

    "",

    paste0(
        "Exact Ensembl-mappable genes: ",
        sum(
            gene_audit$ensembl_annotation_mappable
        ),
        "/24"
    ),

    paste0(
        "Exact official-symbol-mappable genes: ",
        sum(
            gene_audit$symbol_annotation_mappable
        ),
        "/24"
    ),

    paste0(
        "Best-case exact unambiguous-union genes: ",
        sum(
            gene_audit$best_case_exact_annotation_mappable
        ),
        "/24"
    ),

    "",

    paste0(
        "Exact Ensembl coverage: ",
        format_metrics(cov_ens)
    ),

    paste0(
        "Exact symbol coverage: ",
        format_metrics(cov_sym)
    ),

    paste0(
        "Best-case exact union coverage: ",
        format_metrics(cov_union)
    ),

    "",

    paste0(
        "Exact-Ensembl theoretical coverage PASS: ",
        ens_theoretical_pass
    ),

    paste0(
        "Exact-symbol theoretical coverage PASS: ",
        sym_theoretical_pass
    ),

    paste0(
        "Best-case exact-union theoretical coverage PASS: ",
        union_theoretical_pass
    ),

    paste0(
        "Ambiguous probe IDs excluded: ",
        n_ambiguous_probe_ids
    ),

    paste0(
        "Frozen genes requiring multi-probe resolution: ",
        sum(
            gene_audit$multiple_probe_resolution_needed
        )
    ),

    "",

    "IMPORTANT:",
    "This is annotation-level feasibility only.",
    "No phenotype or diagnosis labels were accessed.",
    "No SCZ-MSS score was computed.",
    "No outcome-driven mapping was performed.",
    "No synonym/manual rescue was performed.",
    "No technical threshold was relaxed.",
    "No probe-to-gene expression aggregation was performed.",
    "Final technical PASS requires actual GSE208338 matrix-row presence and sample-completeness verification."
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 09I.2L — CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nMAPPING FEASIBILITY RESULTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Exact Ensembl-mappable frozen genes: ",
    sum(
        gene_audit$ensembl_annotation_mappable
    ),
    "/24\n",
    sep = ""
)

cat(
    "Exact official-symbol-mappable frozen genes: ",
    sum(
        gene_audit$symbol_annotation_mappable
    ),
    "/24\n",
    sep = ""
)

cat(
    "Best-case exact unambiguous union: ",
    sum(
        gene_audit$best_case_exact_annotation_mappable
    ),
    "/24\n",
    sep = ""
)

cat(
    "\nEXACT ENSEMBL COVERAGE\n"
)

print(
    round(
        cov_ens,
        6
    )
)

cat(
    "\nEXACT OFFICIAL-SYMBOL COVERAGE\n"
)

print(
    round(
        cov_sym,
        6
    )
)

cat(
    "\nBEST-CASE EXACT UNAMBIGUOUS-UNION COVERAGE\n"
)

print(
    round(
        cov_union,
        6
    )
)

cat(
    "\nExact-Ensembl theoretical threshold PASS: ",
    ens_theoretical_pass,
    "\n",
    sep = ""
)

cat(
    "Exact-symbol theoretical threshold PASS: ",
    sym_theoretical_pass,
    "\n",
    sep = ""
)

cat(
    "Best-case exact-union theoretical threshold PASS: ",
    union_theoretical_pass,
    "\n",
    sep = ""
)

cat(
    "\nAmbiguous probe IDs excluded: ",
    n_ambiguous_probe_ids,
    "\n",
    sep = ""
)

cat(
    "Frozen genes requiring >1 probe resolution: ",
    sum(
        gene_audit$multiple_probe_resolution_needed
    ),
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nPER-GENE AUDIT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    gene_audit[
        ,
        c(
            "ensembl_stable",
            "symbol",
            "weight",
            "n_exact_ensembl_probes",
            "n_exact_symbol_probes",
            "n_unambiguous_exact_union_probes",
            "multiple_probe_resolution_needed"
        )
    ],
    row.names = FALSE
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nDECISION STATUS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

if (!union_theoretical_pass) {

    cat(
        "ANNOTATION-LEVEL FEASIBILITY: FAIL\n"
    )

    cat(
        "Even the best-case exact deterministic GPL5188 annotation mapping ",
        "does not satisfy all frozen coverage thresholds.\n",
        sep = ""
    )

    cat(
        "GSE208338 must NOT proceed to diagnosis-score evaluation under the frozen rules.\n"
    )

} else {

    cat(
        "ANNOTATION-LEVEL FEASIBILITY: PASS\n"
    )

    cat(
        "At least one exact deterministic annotation route has sufficient theoretical coverage.\n"
    )

    cat(
        "FINAL TECHNICAL PASS: NOT YET ASSIGNED\n"
    )

    cat(
        "Next required step: verify actual matrix-row presence, determine the ",
        "frozen-policy-compatible probe-to-gene reduction rule, and audit sample completeness ",
        "without accessing diagnosis labels.\n",
        sep = ""
    )
}

cat(
    "\nNo phenotype/diagnosis data were accessed in STEP09I.2.\n"
)

cat(
    "\nGene-level audit:\n",
    OUT_GENE_MAP,
    "\n",
    sep = ""
)

cat(
    "\nProbe-level audit:\n",
    OUT_PROBE_MAP,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 09I.2 COMPLETE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 09I.2 — GSE208338 / GPL5188 FROZEN-GENE MAPPING FEASIBILITY AUDIT
Project sentinel: PASS
Series matrix found: PASS
Series matrix local size: 1.3685 GB
Frozen weight file: PASS
Frozen mapping policy: PASS

STEP09I.1 VERIFIED MATRIX FACTS
Platform: GPL5188
ID column: ID_REF
Sample columns: 169
Identifier type: NUMERIC_AFFY_PROBESET_OR_TRANSCRIPT_CLUSTER

Partial incorrect GPL5175 download detected: 1.67801 GB
GPL5175 status: IGNORED — not used, not resumed, not deleted by this cell.

--------------------------------------------------------------------------------------------------------------
OFFICIAL PLATFORM ANNOTATION
--------------------------------------------------------------------------------------------------------------
Installing missing Bioconductor package(s): AnnotationDbi, huex10stprobeset.db


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31 ucrt)

Installing package(s) 'AnnotationDbi', 'huex10stprobeset.db'

also installing the dependencies 'sys', 'cli', 'glue', 'lifecycle', 'fastmap', 'askpass', 'bit', 'vctrs', 'cachem', 'curl', 'jsonlite', 'mime', 'openssl', 'R6', 'XVector', 'Seqinfo', 'crayon', 'generics', 'bit64', 'blob', 'memoise', 'pkgconfig', 'rlang', 'cpp11', 'httr', 'png', 'Biostrings', 'BiocGenerics', 'Biobase', 'IRanges', 'DBI', 'RSQLite', 'S4Vectors', 'KEGGREST', 'org.Hs.eg.db'




package 'sys' successfully unpacked and MD5 sums checked
package 'cli' successfully unpacked and MD5 sums checked
package 'glue' successfully unpacked and MD5 sums checked
package 'lifecycle' successfully unpacked and MD5 sums checked
package 'fastmap' successfully unpacked and MD5 sums checked
package 'askpass' successfully unpacked and MD5 sums checked
package 'bit' successfully unpacked and MD5 sums checked
package 'vctrs' successfully unpacked and MD5 sums checked
package 'cachem' successfully unpacked and MD5 sums checked
package 'curl' successfully unpacked and MD5 sums checked
package 'jsonlite' successfully unpacked and MD5 sums checked
package 'mime' successfully unpacked and MD5 sums checked
package 'openssl' successfully unpacked and MD5 sums checked
package 'R6' successfully unpacked and MD5 sums checked
package 'XVector' successfully unpacked and MD5 sums checked
package 'Seqinfo' successfully unpacked and MD5 sums checked
package 'crayon' successfully unpacked and MD5 sum

installing the source packages 'org.Hs.eg.db', 'huex10stprobeset.db'






Annotation package: huex10stprobeset.db
Package version: 8.8.0
Available relevant keytypes: PROBEID, ENSEMBL, SYMBOL, ENTREZID
Available relevant columns: PROBEID, ENSEMBL, SYMBOL, ENTREZID
Required annotation fields: PASS

--------------------------------------------------------------------------------------------------------------
FROZEN SCORE INPUT
--------------------------------------------------------------------------------------------------------------
Frozen score rows: 24 — PASS
Available columns: gene_index | ensemblID_stable | gene_symbol | gene_type | protein_coding | direction_code | direction | discovery_median_coefficient | absolute_weight | final_normalized_weight | normalized_absolute_weight | outer_selection_count | outer_selection_frequency | outer_sign_consistency | half_sample_selection_count | half_sample_selection_probability | half_sample_sign_consistency | half_sample_median_nonzero_beta | null_selection_count | null_selection_probability | absolute_weight_ran



'select()' returned 1:many mapping between keys and columns



Frozen genes with >=1 exact Ensembl annotation mapping: 14/24

--------------------------------------------------------------------------------------------------------------
EXACT OFFICIAL-SYMBOL MAPPING
--------------------------------------------------------------------------------------------------------------


'select()' returned 1:many mapping between keys and columns



Frozen genes with >=1 exact official-symbol annotation mapping: 13/24

MAPPING FEASIBILITY RESULTS
Exact Ensembl-mappable frozen genes: 14/24
Exact official-symbol-mappable frozen genes: 13/24
Best-case exact unambiguous union: 14/24

EXACT ENSEMBL COVERAGE
     gene_count_coverage absolute_weight_coverage positive_weight_coverage 
                0.583333                 0.381815                 0.261140 
negative_weight_coverage 
                0.457974 

EXACT OFFICIAL-SYMBOL COVERAGE
     gene_count_coverage absolute_weight_coverage positive_weight_coverage 
                0.541667                 0.332819                 0.261140 
negative_weight_coverage 
                0.378056 

BEST-CASE EXACT UNAMBIGUOUS-UNION COVERAGE
     gene_count_coverage absolute_weight_coverage positive_weight_coverage 
                0.583333                 0.381815                 0.261140 
negative_weight_coverage 
                0.457974 

Exact-Ensembl theoretical threshold PASS: FALSE
Exact

In [57]:
# ==================================================================================================
# STEP 09I.3 — GSE208338 FINAL TECHNICAL-GATE CLOSURE
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Formally close GSE208338 as TECHNICALLY NON-EVALUABLE under the frozen SCZ-MSS v1 rules.
#
# WHY NO FURTHER MATRIX SCAN IS REQUIRED
#   STEP09I.2 established an annotation-level UPPER BOUND:
#
#       Exact stable-Ensembl mapped genes          = 14/24
#       Gene-count coverage                        = 0.583333
#       Absolute-weight coverage                   = 0.381815
#       Positive-direction weight coverage         = 0.261140
#       Negative-direction weight coverage         = 0.457974
#
#   Frozen gates:
#       Gene-count coverage        >= 0.80
#       Absolute-weight coverage   >= 0.80
#       Positive coverage          >= 0.70
#       Negative coverage          >= 0.70
#
#   Therefore even the theoretical best-case mapping cannot pass the frozen gate.
#
# IMPORTANT
#   - NO phenotype / diagnosis metadata are read.
#   - NO SCZ-MSS scores are calculated.
#   - NO probe collapsing is performed.
#   - NO symbol/synonym rescue is performed.
#   - NO thresholds are relaxed.
#   - NO 1.368-GB matrix re-read is performed.
#   - The partial incorrect GPL5175 download is left untouched.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 09I.3 — GSE208338 FINAL TECHNICAL-GATE CLOSURE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 09I.3A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

STEP09I2_GENE_AUDIT <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "gse208338",
    "STEP09I2_GPL5188_frozen_gene_mapping_feasibility_v1.csv"
)

STEP09I2_PROBE_AUDIT <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "gse208338",
    "STEP09I2_GPL5188_probe_mapping_audit_v1.csv"
)

STEP09I2_SUMMARY <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "gse208338",
    "STEP09I2_GPL5188_mapping_feasibility_summary_v1.txt"
)

TECHNICAL_GATE_PATH <- file.path(
    PROJECT_ROOT,
    "08_score_lock",
    "SCZ_MSS_v1_technical_gate.json"
)

MAPPING_POLICY_PATH <- file.path(
    PROJECT_ROOT,
    "08_score_lock",
    "SCZ_MSS_v1_mapping_policy.json"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "gse208338"
)

OUT_DECISION_CSV <- file.path(
    OUT_DIR,
    "STEP09I3_GSE208338_FINAL_technical_gate_decision_v1.csv"
)

OUT_DECISION_TXT <- file.path(
    OUT_DIR,
    "STEP09I3_GSE208338_FINAL_technical_gate_decision_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP09I3_GSE208338_FINAL_technical_gate_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 09I.3B — PROJECT / INPUT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

required_files <- c(
    STEP09I2_GENE_AUDIT,
    STEP09I2_PROBE_AUDIT,
    STEP09I2_SUMMARY,
    TECHNICAL_GATE_PATH,
    MAPPING_POLICY_PATH
)

missing_files <- required_files[
    !file.exists(required_files)
]

if (length(missing_files) > 0L) {
    stop(
        "Required upstream file(s) missing:\n",
        paste(
            missing_files,
            collapse = "\n"
        )
    )
}

cat("STEP09I.2 artifacts: PASS\n")
cat("Frozen technical-gate file: PASS\n")
cat("Frozen mapping-policy file: PASS\n")

# --------------------------------------------------------------------------------------------------
# 09I.3C — LOAD STEP09I.2 GENE AUDIT
# --------------------------------------------------------------------------------------------------

audit <- read.csv(
    STEP09I2_GENE_AUDIT,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

required_columns <- c(
    "ensembl_stable",
    "symbol",
    "weight",
    "n_exact_ensembl_probes",
    "n_exact_symbol_probes",
    "n_unambiguous_exact_union_probes",
    "ensembl_annotation_mappable",
    "symbol_annotation_mappable",
    "best_case_exact_annotation_mappable",
    "multiple_probe_resolution_needed"
)

missing_columns <- setdiff(
    required_columns,
    names(audit)
)

if (length(missing_columns) > 0L) {
    stop(
        "STEP09I.2 audit is missing expected column(s): ",
        paste(
            missing_columns,
            collapse = ", "
        )
    )
}

if (nrow(audit) != 24L) {
    stop(
        "Expected 24 frozen genes in STEP09I.2 audit; observed ",
        nrow(audit)
    )
}

cat("Frozen genes in audit: 24 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 09I.3D — NORMALIZE LOGICAL VALUES
# --------------------------------------------------------------------------------------------------

as_logical_strict <- function(x) {

    if (is.logical(x)) {
        return(x)
    }

    y <- toupper(
        trimws(
            as.character(x)
        )
    )

    if (any(!y %in% c("TRUE", "FALSE"))) {
        stop(
            "Unexpected logical encoding detected: ",
            paste(
                unique(
                    y[
                        !y %in% c("TRUE", "FALSE")
                    ]
                ),
                collapse = ", "
            )
        )
    }

    y == "TRUE"
}

audit$ensembl_annotation_mappable <- as_logical_strict(
    audit$ensembl_annotation_mappable
)

audit$symbol_annotation_mappable <- as_logical_strict(
    audit$symbol_annotation_mappable
)

audit$best_case_exact_annotation_mappable <- as_logical_strict(
    audit$best_case_exact_annotation_mappable
)

audit$multiple_probe_resolution_needed <- as_logical_strict(
    audit$multiple_probe_resolution_needed
)

audit$weight <- as.numeric(
    audit$weight
)

if (anyNA(audit$weight)) {
    stop("Missing/non-numeric frozen weight detected.")
}

if (abs(sum(abs(audit$weight)) - 1) > 1e-6) {
    stop(
        "Frozen normalized weights no longer sum to absolute weight 1.0."
    )
}

# --------------------------------------------------------------------------------------------------
# 09I.3E — RECOMPUTE FROZEN COVERAGE FROM STEP09I.2
# --------------------------------------------------------------------------------------------------

coverage_metrics <- function(mapped, weight) {

    mapped <- as.logical(mapped)
    weight <- as.numeric(weight)

    total_abs <- sum(abs(weight))
    total_pos <- sum(abs(weight[weight > 0]))
    total_neg <- sum(abs(weight[weight < 0]))

    mapped_abs <- sum(
        abs(
            weight[mapped]
        )
    )

    mapped_pos <- sum(
        abs(
            weight[
                mapped &
                weight > 0
            ]
        )
    )

    mapped_neg <- sum(
        abs(
            weight[
                mapped &
                weight < 0
            ]
        )
    )

    c(
        gene_count_coverage =
            sum(mapped) / length(mapped),

        absolute_weight_coverage =
            mapped_abs / total_abs,

        positive_weight_coverage =
            mapped_pos / total_pos,

        negative_weight_coverage =
            mapped_neg / total_neg
    )
}

# Primary mapping route under frozen policy:
# exact stable Ensembl IDs.
primary_metrics <- coverage_metrics(
    audit$ensembl_annotation_mappable,
    audit$weight
)

# Best-case exact deterministic union was also calculated in STEP09I.2.
# It is used ONLY as an upper-bound proof that no exact deterministic rescue can pass.
upper_bound_metrics <- coverage_metrics(
    audit$best_case_exact_annotation_mappable,
    audit$weight
)

FROZEN_THRESHOLDS <- c(
    gene_count_coverage = 0.80,
    absolute_weight_coverage = 0.80,
    positive_weight_coverage = 0.70,
    negative_weight_coverage = 0.70
)

primary_gate_component_pass <- (
    primary_metrics >=
    FROZEN_THRESHOLDS
)

upper_bound_gate_component_pass <- (
    upper_bound_metrics >=
    FROZEN_THRESHOLDS
)

PRIMARY_PASS <- all(
    primary_gate_component_pass
)

UPPER_BOUND_PASS <- all(
    upper_bound_gate_component_pass
)

# --------------------------------------------------------------------------------------------------
# 09I.3F — CONSISTENCY CHECK AGAINST STEP09I.2 REPORTED VALUES
# --------------------------------------------------------------------------------------------------

expected_step09i2 <- c(
    gene_count_coverage = 0.583333,
    absolute_weight_coverage = 0.381815,
    positive_weight_coverage = 0.261140,
    negative_weight_coverage = 0.457974
)

difference_from_reported <- abs(
    primary_metrics -
    expected_step09i2
)

if (any(difference_from_reported > 1e-5)) {
    stop(
        "Recomputed STEP09I.2 metrics do not match the recorded audit.\n",
        "Recomputed: ",
        paste(
            names(primary_metrics),
            sprintf("%.6f", primary_metrics),
            collapse = "; "
        ),
        "\nExpected: ",
        paste(
            names(expected_step09i2),
            sprintf("%.6f", expected_step09i2),
            collapse = "; "
        )
    )
}

cat("STEP09I.2 metric reproduction: PASS\n")

# --------------------------------------------------------------------------------------------------
# 09I.3G — MAPPING-POLICY CONSEQUENCES
# --------------------------------------------------------------------------------------------------

n_ensembl_mapped <- sum(
    audit$ensembl_annotation_mappable
)

n_symbol_mapped <- sum(
    audit$symbol_annotation_mappable
)

n_upper_bound_mapped <- sum(
    audit$best_case_exact_annotation_mappable
)

all_ensembl_mapped_need_multi_probe_resolution <- all(
    audit$n_exact_ensembl_probes[
        audit$ensembl_annotation_mappable
    ] > 1L
)

if (n_ensembl_mapped != 14L) {
    stop(
        "Expected 14 exact-Ensembl mapped genes from STEP09I.2; observed ",
        n_ensembl_mapped
    )
}

if (n_upper_bound_mapped != 14L) {
    stop(
        "Expected best-case exact deterministic upper bound of 14/24; observed ",
        n_upper_bound_mapped,
        "/24"
    )
}

if (UPPER_BOUND_PASS) {
    stop(
        "Unexpected result: best-case upper bound now passes the gate. ",
        "Do not finalize closure."
    )
}

# --------------------------------------------------------------------------------------------------
# 09I.3H — FINAL TECHNICAL DECISION
# --------------------------------------------------------------------------------------------------

failed_components <- names(
    FROZEN_THRESHOLDS
)[
    !upper_bound_gate_component_pass
]

FINAL_TECHNICAL_PASS <- FALSE
OUTCOME_EVALUATED <- FALSE

FINAL_STATUS <- "TECHNICALLY_NON_EVALUABLE"

FINAL_REASON <- paste0(
    "GSE208338 cannot satisfy the frozen SCZ-MSS v1 technical gate even under the ",
    "best-case exact deterministic annotation mapping. The maximum exact mapping is ",
    n_upper_bound_mapped,
    "/24 frozen genes (",
    sprintf("%.1f", 100 * upper_bound_metrics["gene_count_coverage"]),
    "%), retaining ",
    sprintf("%.1f", 100 * upper_bound_metrics["absolute_weight_coverage"]),
    "% of total absolute frozen weight, ",
    sprintf("%.1f", 100 * upper_bound_metrics["positive_weight_coverage"]),
    "% of positive-direction absolute weight, and ",
    sprintf("%.1f", 100 * upper_bound_metrics["negative_weight_coverage"]),
    "% of negative-direction absolute weight. All four coverage requirements fail. ",
    "Therefore phenotype/diagnosis-score evaluation is prohibited under the frozen protocol."
)

# --------------------------------------------------------------------------------------------------
# 09I.3I — CREATE CANONICAL DECISION TABLE
# --------------------------------------------------------------------------------------------------

decision <- data.frame(
    project_id = EXPECTED_PROJECT_ID,

    dataset = "GSE208338",

    platform = "GPL5188",

    platform_name = "Affymetrix Human Exon 1.0 ST Array",

    tissue_region = "DLPFC_BA9",

    n_samples_series_matrix = 169L,

    matrix_id_column = "ID_REF",

    matrix_identifier_type =
        "NUMERIC_AFFY_PROBESET_OR_TRANSCRIPT_CLUSTER",

    frozen_score = "SCZ_MSS_v1",

    frozen_gene_count = 24L,

    primary_mapping_rule =
        "EXACT_STABLE_ENSEMBL_ID",

    symbol_fallback_used = FALSE,

    manual_synonym_rescue_used = FALSE,

    arbitrary_probe_collapse_used = FALSE,

    exact_ensembl_mapped_genes =
        n_ensembl_mapped,

    exact_symbol_mapped_genes =
        n_symbol_mapped,

    best_case_exact_mapped_genes =
        n_upper_bound_mapped,

    gene_count_coverage =
        unname(
            upper_bound_metrics[
                "gene_count_coverage"
            ]
        ),

    gene_count_threshold =
        FROZEN_THRESHOLDS[
            "gene_count_coverage"
        ],

    gene_count_pass =
        unname(
            upper_bound_gate_component_pass[
                "gene_count_coverage"
            ]
        ),

    absolute_weight_coverage =
        unname(
            upper_bound_metrics[
                "absolute_weight_coverage"
            ]
        ),

    absolute_weight_threshold =
        FROZEN_THRESHOLDS[
            "absolute_weight_coverage"
        ],

    absolute_weight_pass =
        unname(
            upper_bound_gate_component_pass[
                "absolute_weight_coverage"
            ]
        ),

    positive_weight_coverage =
        unname(
            upper_bound_metrics[
                "positive_weight_coverage"
            ]
        ),

    positive_weight_threshold =
        FROZEN_THRESHOLDS[
            "positive_weight_coverage"
        ],

    positive_weight_pass =
        unname(
            upper_bound_gate_component_pass[
                "positive_weight_coverage"
            ]
        ),

    negative_weight_coverage =
        unname(
            upper_bound_metrics[
                "negative_weight_coverage"
            ]
        ),

    negative_weight_threshold =
        FROZEN_THRESHOLDS[
            "negative_weight_coverage"
        ],

    negative_weight_pass =
        unname(
            upper_bound_gate_component_pass[
                "negative_weight_coverage"
            ]
        ),

    all_ensembl_mapped_genes_multi_probe =
        all_ensembl_mapped_need_multi_probe_resolution,

    matrix_row_scan_required = FALSE,

    matrix_row_scan_reason =
        "Not required because annotation-level theoretical maximum already fails the frozen gate.",

    phenotype_accessed_for_gate = FALSE,

    diagnosis_accessed_for_gate = FALSE,

    frozen_score_computed = FALSE,

    outcome_evaluated =
        OUTCOME_EVALUATED,

    technical_pass =
        FINAL_TECHNICAL_PASS,

    final_status =
        FINAL_STATUS,

    failed_gate_components =
        paste(
            failed_components,
            collapse = ";"
        ),

    biological_replication_failure = FALSE,

    interpretation =
        "Technical non-evaluability due to insufficient frozen-score feature/weight coverage; not a biological replication result.",

    stringsAsFactors = FALSE
)

write.csv(
    decision,
    OUT_DECISION_CSV,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 09I.3J — HUMAN-READABLE FINAL CLOSURE RECORD
# --------------------------------------------------------------------------------------------------

decision_text <- c(

    "==============================================================================================================",
    "STEP 09I.3 — GSE208338 FINAL TECHNICAL-GATE CLOSURE",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "Dataset: GSE208338",
    "Platform: GPL5188",
    "Region: DLPFC / BA9",
    "Series-matrix samples: 169",
    "Frozen score: SCZ-MSS v1 (24 genes)",

    "",

    "PRIMARY FROZEN MAPPING POLICY",
    "Primary identifier: exact stable Ensembl gene ID",
    "Symbol fallback used: NO",
    "Manual synonym rescue: NO",
    "Arbitrary duplicate/probe collapse: NO",

    "",

    paste0(
        "Exact Ensembl-mappable genes: ",
        n_ensembl_mapped,
        "/24"
    ),

    paste0(
        "Exact symbol-mappable genes: ",
        n_symbol_mapped,
        "/24"
    ),

    paste0(
        "Best-case exact deterministic upper bound: ",
        n_upper_bound_mapped,
        "/24"
    ),

    "",

    "FROZEN TECHNICAL-GATE RESULTS",

    paste0(
        "Gene-count coverage: ",
        sprintf(
            "%.6f",
            upper_bound_metrics[
                "gene_count_coverage"
            ]
        ),
        "   threshold >= 0.80   PASS=",
        upper_bound_gate_component_pass[
            "gene_count_coverage"
        ]
    ),

    paste0(
        "Absolute-weight coverage: ",
        sprintf(
            "%.6f",
            upper_bound_metrics[
                "absolute_weight_coverage"
            ]
        ),
        "   threshold >= 0.80   PASS=",
        upper_bound_gate_component_pass[
            "absolute_weight_coverage"
        ]
    ),

    paste0(
        "Positive-direction coverage: ",
        sprintf(
            "%.6f",
            upper_bound_metrics[
                "positive_weight_coverage"
            ]
        ),
        "   threshold >= 0.70   PASS=",
        upper_bound_gate_component_pass[
            "positive_weight_coverage"
        ]
    ),

    paste0(
        "Negative-direction coverage: ",
        sprintf(
            "%.6f",
            upper_bound_metrics[
                "negative_weight_coverage"
            ]
        ),
        "   threshold >= 0.70   PASS=",
        upper_bound_gate_component_pass[
            "negative_weight_coverage"
        ]
    ),

    "",

    paste0(
        "All exact-Ensembl mapped genes require >1 platform probe resolution: ",
        all_ensembl_mapped_need_multi_probe_resolution
    ),

    "",

    "FINAL TECHNICAL PASS: FALSE",
    "FINAL STATUS: TECHNICALLY_NON_EVALUABLE",
    "OUTCOME EVALUATED: FALSE",

    "",

    "DECISION BASIS:",
    FINAL_REASON,

    "",

    "IMPORTANT INTERPRETATION:",
    "This is NOT a biological replication failure.",
    "No diagnosis-score association or AUROC is permitted for GSE208338 under the frozen rules.",
    "No outcome labels were needed to reach this decision.",
    "The 1.368-GB series matrix does not need to be rescanned for this gate decision.",
    "The partial incorrect GPL5175 download remains ignored and is not part of the analysis.",

    "",

    "PROTOCOL STATUS:",
    "GSE208338 external validation branch is CLOSED at the technical gate.",

    "=============================================================================================================="
)

writeLines(
    decision_text,
    OUT_DECISION_TXT,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 09I.3K — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    STEP09I2_GENE_AUDIT,
    STEP09I2_PROBE_AUDIT,
    STEP09I2_SUMMARY,
    TECHNICAL_GATE_PATH,
    MAPPING_POLICY_PATH,
    OUT_DECISION_CSV,
    OUT_DECISION_TXT
)

manifest <- data.frame(
    file = basename(manifest_paths),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(manifest_paths)$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 09I.3L — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nRECOMPUTED UPPER-BOUND COVERAGE\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    round(
        upper_bound_metrics,
        6
    )
)

cat(
    "\nFROZEN THRESHOLDS\n"
)

print(
    FROZEN_THRESHOLDS
)

cat(
    "\nGATE COMPONENT PASS\n"
)

print(
    upper_bound_gate_component_pass
)

cat(
    "\nExact Ensembl mapped genes: ",
    n_ensembl_mapped,
    "/24\n",
    sep = ""
)

cat(
    "Best-case exact deterministic mapped genes: ",
    n_upper_bound_mapped,
    "/24\n",
    sep = ""
)

cat(
    "All exact-Ensembl mapped genes require >1 probe resolution: ",
    all_ensembl_mapped_need_multi_probe_resolution,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFINAL GSE208338 DECISION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat("TECHNICAL PASS: FALSE\n")
cat("STATUS: TECHNICALLY NON-EVALUABLE\n")
cat("OUTCOME EVALUATED: FALSE\n")
cat("DIAGNOSIS DATA ACCESSED: FALSE\n")
cat("SCZ-MSS COMPUTED: FALSE\n")

cat(
    "\nReason:\n",
    FINAL_REASON,
    "\n",
    sep = ""
)

cat(
    "\nCanonical decision CSV:\n",
    OUT_DECISION_CSV,
    "\n",
    sep = ""
)

cat(
    "\nCanonical decision TXT:\n",
    OUT_DECISION_TXT,
    "\n",
    sep = ""
)

cat(
    "\nSHA256 manifest:\n",
    OUT_MANIFEST,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 09I.3 COMPLETE — GSE208338 BRANCH CLOSED AT TECHNICAL GATE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 09I.3 — GSE208338 FINAL TECHNICAL-GATE CLOSURE
Project sentinel: PASS
STEP09I.2 artifacts: PASS
Frozen technical-gate file: PASS
Frozen mapping-policy file: PASS
Frozen genes in audit: 24 — PASS
STEP09I.2 metric reproduction: PASS

--------------------------------------------------------------------------------------------------------------
RECOMPUTED UPPER-BOUND COVERAGE
--------------------------------------------------------------------------------------------------------------
     gene_count_coverage absolute_weight_coverage positive_weight_coverage 
                0.583333                 0.381815                 0.261140 
negative_weight_coverage 
                0.457974 

FROZEN THRESHOLDS
     gene_count_coverage absolute_weight_coverage positive_weight_coverage 
                     0.8                      0.8                      0.7 
negative_weight_coverage 
                     0.7 

GATE COMPONENT PASS
     gene_count_coverage absolute_weight_coverage positive_w

In [58]:
# ==================================================================================================
# STEP 09J.1 — MASTER EXTERNAL TECHNICAL-GATE REGISTRY FREEZE
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Consolidate all COMPLETED public external-cohort technical-gate decisions into
#   one canonical registry before screening any additional external datasets.
#
# INCLUDED DATASETS
#   GSE87194
#   GSE80655
#   GSE224683
#   GSE282552
#   GSE208338
#
# IMPORTANT
#   - This step performs NO new diagnosis analysis.
#   - This step performs NO score recalculation.
#   - This step performs NO mapping rescue.
#   - Existing completed external results are preserved exactly.
#   - Technical FAIL is NOT labeled biological replication failure.
#   - GSE224683 remains the only currently evaluated public external cohort because
#     it passed the frozen gate.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 09J.1 — MASTER EXTERNAL TECHNICAL-GATE REGISTRY FREEZE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 09J.1A — PROJECT VERIFICATION
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

# --------------------------------------------------------------------------------------------------
# 09J.1B — VERIFY GSE208338 CANONICAL CLOSURE
# --------------------------------------------------------------------------------------------------

GSE208338_DECISION_PATH <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "gse208338",
    "STEP09I3_GSE208338_FINAL_technical_gate_decision_v1.csv"
)

if (!file.exists(GSE208338_DECISION_PATH)) {
    stop(
        "Canonical GSE208338 technical-gate decision missing:\n",
        GSE208338_DECISION_PATH
    )
}

gse208338_decision <- read.csv(
    GSE208338_DECISION_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(gse208338_decision) != 1L) {
    stop(
        "Expected exactly one GSE208338 decision row; observed ",
        nrow(gse208338_decision)
    )
}

required_208338_cols <- c(
    "dataset",
    "gene_count_coverage",
    "absolute_weight_coverage",
    "positive_weight_coverage",
    "negative_weight_coverage",
    "technical_pass",
    "outcome_evaluated",
    "final_status"
)

missing_208338_cols <- setdiff(
    required_208338_cols,
    names(gse208338_decision)
)

if (length(missing_208338_cols) > 0L) {
    stop(
        "GSE208338 decision file missing expected columns: ",
        paste(missing_208338_cols, collapse = ", ")
    )
}

if (
    !identical(
        as.character(gse208338_decision$dataset[1]),
        "GSE208338"
    )
) {
    stop("Unexpected dataset ID in GSE208338 decision file.")
}

as_logical_safe <- function(x) {

    if (is.logical(x)) {
        return(x)
    }

    y <- toupper(trimws(as.character(x)))

    if (!all(y %in% c("TRUE", "FALSE"))) {
        stop(
            "Unexpected TRUE/FALSE encoding: ",
            paste(unique(y), collapse = ", ")
        )
    }

    y == "TRUE"
}

gse208338_pass <- as_logical_safe(
    gse208338_decision$technical_pass[1]
)

gse208338_outcome <- as_logical_safe(
    gse208338_decision$outcome_evaluated[1]
)

if (gse208338_pass || gse208338_outcome) {
    stop(
        "GSE208338 canonical closure is inconsistent: ",
        "technical_pass and outcome_evaluated must both be FALSE."
    )
}

cat("GSE208338 canonical closure verification: PASS\n")

# --------------------------------------------------------------------------------------------------
# 09J.1C — FROZEN GATE THRESHOLDS
# --------------------------------------------------------------------------------------------------

FROZEN_GENE_COUNT_THRESHOLD <- 0.80
FROZEN_ABS_WEIGHT_THRESHOLD <- 0.80
FROZEN_POS_WEIGHT_THRESHOLD <- 0.70
FROZEN_NEG_WEIGHT_THRESHOLD <- 0.70

# --------------------------------------------------------------------------------------------------
# 09J.1D — BUILD MASTER REGISTRY
#
# Values below are copied from completed dataset-specific audits.
# They are NOT recomputed and must NOT be outcome-modified.
# --------------------------------------------------------------------------------------------------

registry <- data.frame(

    dataset = c(
        "GSE87194",
        "GSE80655",
        "GSE224683",
        "GSE282552",
        "GSE208338"
    ),

    public_external_sequence = 1:5,

    tissue_region = c(
        "DLPFC",
        "DLPFC",
        "DLPFC",
        "DLPFC",
        "DLPFC_BA9"
    ),

    assay_platform_summary = c(
        "RNA-seq; symbol-indexed processed count matrix",
        "RNA-seq; processed gene-expression matrix",
        "RNA-seq; Salmon transcript-level counts/TPM",
        "Long-read isoform expression matrix; PB.x.x.GENE_SYMBOL identifiers",
        "Affymetrix Human Exon 1.0 ST Array; GPL5188"
    ),

    n_samples = c(
        38L,
        281L,
        30L,
        18L,
        169L
    ),

    control_n_known = c(
        19L,
        NA_integer_,
        15L,
        NA_integer_,
        NA_integer_
    ),

    schizophrenia_n_known = c(
        19L,
        NA_integer_,
        15L,
        NA_integer_,
        NA_integer_
    ),

    frozen_gene_count = rep(
        24L,
        5L
    ),

    mapped_gene_count = c(
        14L,
        20L,
        23L,
        9L,
        as.integer(
            gse208338_decision$best_case_exact_mapped_genes[1]
        )
    ),

    gene_count_coverage = c(
        0.583333,
        0.833333,
        0.958333,
        0.375000,
        as.numeric(
            gse208338_decision$gene_count_coverage[1]
        )
    ),

    absolute_weight_coverage = c(
        0.333216,
        0.719710,
        0.926600,
        0.1961925,
        as.numeric(
            gse208338_decision$absolute_weight_coverage[1]
        )
    ),

    positive_weight_coverage = c(
        0.261140,
        1.000000,
        1.000000,
        0.0304038,
        as.numeric(
            gse208338_decision$positive_weight_coverage[1]
        )
    ),

    negative_weight_coverage = c(
        0.378704,
        0.542817,
        0.880276,
        0.3008226,
        as.numeric(
            gse208338_decision$negative_weight_coverage[1]
        )
    ),

    sample_completeness = c(
        1.000000,
        1.000000,
        1.000000,
        1.000000,
        NA_real_
    ),

    mapping_basis = c(
        "Exact unique gene-symbol mapping because processed matrix was symbol-indexed",
        "Frozen deterministic mapping audit",
        "Exact stable Ensembl mapping after transcript-to-gene aggregation by explicit stable Ensembl ID",
        "Exact frozen-symbol mapping only; source provided no ENSG/ENST identifiers",
        "Exact stable-Ensembl mapping; best-case deterministic annotation upper bound"
    ),

    mapping_unambiguous_for_gate = c(
        TRUE,
        TRUE,
        TRUE,
        TRUE,
        TRUE
    ),

    gene_count_pass = c(
        FALSE,
        TRUE,
        TRUE,
        FALSE,
        FALSE
    ),

    absolute_weight_pass = c(
        FALSE,
        FALSE,
        TRUE,
        FALSE,
        FALSE
    ),

    positive_weight_pass = c(
        FALSE,
        TRUE,
        TRUE,
        FALSE,
        FALSE
    ),

    negative_weight_pass = c(
        FALSE,
        FALSE,
        TRUE,
        FALSE,
        FALSE
    ),

    technical_pass = c(
        FALSE,
        FALSE,
        TRUE,
        FALSE,
        FALSE
    ),

    outcome_evaluated = c(
        FALSE,
        FALSE,
        TRUE,
        FALSE,
        FALSE
    ),

    primary_external_auroc = c(
        NA_real_,
        NA_real_,
        0.5067,
        NA_real_,
        NA_real_
    ),

    primary_external_auroc_ci_lower = c(
        NA_real_,
        NA_real_,
        0.2889,
        NA_real_,
        NA_real_
    ),

    primary_external_auroc_ci_upper = c(
        NA_real_,
        NA_real_,
        0.7244,
        NA_real_,
        NA_real_
    ),

    primary_external_hedges_g = c(
        NA_real_,
        NA_real_,
        -0.0422,
        NA_real_,
        NA_real_
    ),

    primary_external_mean_difference = c(
        NA_real_,
        NA_real_,
        -0.000369,
        NA_real_,
        NA_real_
    ),

    measurability_note = c(
        "Not pursued after frozen technical-gate failure",
        "Not pursued after frozen technical-gate failure",
        "23/24 mapped, but only 17/23 mapped genes had nonzero TPM in >=1 sample; 6 mapped genes were all-zero",
        "Not pursued after frozen technical-gate failure",
        "Annotation-level theoretical maximum already failed; actual expression measurability not required for gate decision"
    ),

    outcome_blinding_note = c(
        "Outcome analysis not performed because technical gate failed",
        "Phenotype metadata had been inspected during preliminary exploration before final score lock; no outcome score analysis performed because technical gate failed",
        "Frozen score computed and hashed before diagnosis metadata were read in the validation cell",
        "Outcome analysis not performed because technical gate failed",
        "Diagnosis metadata never accessed because annotation-level gate failed"
    ),

    final_status = c(
        "TECHNICALLY_NON_EVALUABLE",
        "TECHNICALLY_NON_EVALUABLE",
        "TECHNICAL_PASS_OUTCOME_EVALUATED_NULL",
        "TECHNICALLY_NON_EVALUABLE",
        "TECHNICALLY_NON_EVALUABLE"
    ),

    biological_replication_failure = rep(
        FALSE,
        5L
    ),

    interpretation = c(
        "Insufficient frozen-score coverage; no biological replication conclusion permitted.",
        "Insufficient retained absolute-weight and negative-direction coverage; no biological replication conclusion permitted.",
        "Technically eligible external cohort; primary frozen score showed essentially null discrimination with wide uncertainty.",
        "Insufficient frozen-score coverage; no biological replication conclusion permitted.",
        "Insufficient exact deterministic frozen-score coverage; no biological replication conclusion permitted."
    ),

    stringsAsFactors = FALSE
)

# --------------------------------------------------------------------------------------------------
# 09J.1E — RECOMPUTE PASS FLAGS FROM NUMERIC THRESHOLDS
# --------------------------------------------------------------------------------------------------

registry$gene_count_pass_recomputed <- (
    registry$gene_count_coverage >=
    FROZEN_GENE_COUNT_THRESHOLD
)

registry$absolute_weight_pass_recomputed <- (
    registry$absolute_weight_coverage >=
    FROZEN_ABS_WEIGHT_THRESHOLD
)

registry$positive_weight_pass_recomputed <- (
    registry$positive_weight_coverage >=
    FROZEN_POS_WEIGHT_THRESHOLD
)

registry$negative_weight_pass_recomputed <- (
    registry$negative_weight_coverage >=
    FROZEN_NEG_WEIGHT_THRESHOLD
)

registry$coverage_gate_pass_recomputed <- (
    registry$gene_count_pass_recomputed &
    registry$absolute_weight_pass_recomputed &
    registry$positive_weight_pass_recomputed &
    registry$negative_weight_pass_recomputed
)

# --------------------------------------------------------------------------------------------------
# 09J.1F — INTERNAL CONSISTENCY TESTS
# --------------------------------------------------------------------------------------------------

if (
    !identical(
        registry$gene_count_pass,
        registry$gene_count_pass_recomputed
    )
) {
    stop("Gene-count PASS flag mismatch detected.")
}

if (
    !identical(
        registry$absolute_weight_pass,
        registry$absolute_weight_pass_recomputed
    )
) {
    stop("Absolute-weight PASS flag mismatch detected.")
}

if (
    !identical(
        registry$positive_weight_pass,
        registry$positive_weight_pass_recomputed
    )
) {
    stop("Positive-weight PASS flag mismatch detected.")
}

if (
    !identical(
        registry$negative_weight_pass,
        registry$negative_weight_pass_recomputed
    )
) {
    stop("Negative-weight PASS flag mismatch detected.")
}

if (
    !identical(
        registry$technical_pass,
        registry$coverage_gate_pass_recomputed
    )
) {
    stop(
        "Overall technical PASS does not agree with frozen numeric coverage rules."
    )
}

if (
    any(
        registry$outcome_evaluated &
        !registry$technical_pass
    )
) {
    stop(
        "Protocol violation detected: outcome evaluated for a technical-gate FAIL cohort."
    )
}

if (
    sum(registry$technical_pass) != 1L
) {
    stop(
        "Expected exactly one currently completed public cohort to pass the frozen gate; observed ",
        sum(registry$technical_pass)
    )
}

if (
    !identical(
        registry$dataset[
            registry$technical_pass
        ],
        "GSE224683"
    )
) {
    stop(
        "Unexpected technical-PASS cohort."
    )
}

if (
    sum(registry$outcome_evaluated) != 1L
) {
    stop(
        "Expected exactly one public external cohort with outcome evaluation."
    )
}

if (
    !identical(
        registry$dataset[
            registry$outcome_evaluated
        ],
        "GSE224683"
    )
) {
    stop(
        "Unexpected cohort marked as outcome evaluated."
    )
}

cat("Frozen-threshold consistency checks: PASS\n")
cat("Outcome-evaluation gate consistency: PASS\n")

# --------------------------------------------------------------------------------------------------
# 09J.1G — OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

MASTER_DIR <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "master_registry_v1"
)

dir.create(
    MASTER_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

MASTER_CSV <- file.path(
    MASTER_DIR,
    "SCZ_MSS_v1_external_technical_gate_registry_v1.csv"
)

MASTER_TXT <- file.path(
    MASTER_DIR,
    "SCZ_MSS_v1_external_technical_gate_registry_v1.txt"
)

MANIFEST_CSV <- file.path(
    MASTER_DIR,
    "SCZ_MSS_v1_external_technical_gate_registry_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 09J.1H — WRITE CANONICAL CSV
# --------------------------------------------------------------------------------------------------

write.csv(
    registry,
    MASTER_CSV,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 09J.1I — HUMAN-READABLE SUMMARY
# --------------------------------------------------------------------------------------------------

n_datasets <- nrow(registry)
n_pass <- sum(registry$technical_pass)
n_fail <- sum(!registry$technical_pass)
n_outcome <- sum(registry$outcome_evaluated)

summary_lines <- c(

    "==============================================================================================================",
    "SCZ-MSS v1 — PUBLIC EXTERNAL TECHNICAL-GATE MASTER REGISTRY",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    paste0(
        "Public cohorts screened: ",
        n_datasets
    ),

    paste0(
        "Technical PASS: ",
        n_pass
    ),

    paste0(
        "Technical FAIL / non-evaluable: ",
        n_fail
    ),

    paste0(
        "Outcome-evaluated cohorts: ",
        n_outcome
    ),

    "",

    "FROZEN TECHNICAL GATE:",
    "Gene-count coverage >= 0.80",
    "Absolute-weight coverage >= 0.80",
    "Positive-direction absolute-weight coverage >= 0.70",
    "Negative-direction absolute-weight coverage >= 0.70",
    "Mapping must be unambiguous",
    "Sample completeness must satisfy frozen rule",

    "",

    "DATASET STATUS:",

    paste0(
        registry$dataset,
        " | mapped=",
        registry$mapped_gene_count,
        "/24",
        " | gene=",
        sprintf("%.6f", registry$gene_count_coverage),
        " | absW=",
        sprintf("%.6f", registry$absolute_weight_coverage),
        " | posW=",
        sprintf("%.6f", registry$positive_weight_coverage),
        " | negW=",
        sprintf("%.6f", registry$negative_weight_coverage),
        " | technical_PASS=",
        registry$technical_pass,
        " | outcome_evaluated=",
        registry$outcome_evaluated,
        " | status=",
        registry$final_status
    ),

    "",

    "CURRENT EXTERNAL EVIDENCE SUMMARY:",

    "GSE87194: technically non-evaluable.",
    "GSE80655: technically non-evaluable.",
    "GSE224683: technical PASS; primary frozen-score external result was null.",
    "GSE282552: technically non-evaluable.",
    "GSE208338: technically non-evaluable.",

    "",

    "IMPORTANT INTERPRETATION:",

    paste0(
        n_fail,
        " technical-gate failures are assay/mapping transferability limitations and are NOT biological replication failures."
    ),

    "Only GSE224683 currently provides a valid public external diagnosis-score test under the frozen gate.",

    "Its null result must remain visible and must not be replaced by post-hoc measurable-only sensitivity results.",

    "External biological transferability therefore remains unresolved/challenged rather than successfully established.",

    "",

    "PROTOCOL TRANSPARENCY:",
    "The public external-validation program is an access-driven amendment because the originally planned CMC MSSM/Penn/Pitt resource was not locally available.",
    "This master registry does not retroactively convert that amendment into a prospectively locked primary validation program.",

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    MASTER_TXT,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 09J.1J — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_sources <- c(
    MASTER_CSV,
    MASTER_TXT,
    GSE208338_DECISION_PATH
)

manifest <- data.frame(
    file = basename(manifest_sources),
    path = manifest_sources,
    bytes = as.numeric(
        file.info(manifest_sources)$size
    ),
    sha256 = vapply(
        manifest_sources,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_CSV,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 09J.1K — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nMASTER EXTERNAL TECHNICAL-GATE REGISTRY\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

console_table <- registry[
    ,
    c(
        "dataset",
        "mapped_gene_count",
        "gene_count_coverage",
        "absolute_weight_coverage",
        "positive_weight_coverage",
        "negative_weight_coverage",
        "technical_pass",
        "outcome_evaluated",
        "final_status"
    )
]

print(
    console_table,
    row.names = FALSE
)

cat(
    "\nPublic cohorts screened: ",
    n_datasets,
    "\n",
    sep = ""
)

cat(
    "Technical PASS: ",
    n_pass,
    "\n",
    sep = ""
)

cat(
    "Technically non-evaluable: ",
    n_fail,
    "\n",
    sep = ""
)

cat(
    "Outcome evaluated: ",
    n_outcome,
    "\n",
    sep = ""
)

cat(
    "\nCurrent valid public external outcome cohort: GSE224683\n"
)

cat(
    "GSE224683 primary external AUROC: 0.5067 [0.2889, 0.7244]\n"
)

cat(
    "Current external biological transferability status: UNRESOLVED / CHALLENGED\n"
)

cat(
    "\nCanonical master registry:\n",
    MASTER_CSV,
    "\n",
    sep = ""
)

cat(
    "\nHuman-readable summary:\n",
    MASTER_TXT,
    "\n",
    sep = ""
)

cat(
    "\nSHA256 manifest:\n",
    MANIFEST_CSV,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 09J.1 COMPLETE — MASTER EXTERNAL TECHNICAL-GATE REGISTRY FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 09J.1 — MASTER EXTERNAL TECHNICAL-GATE REGISTRY FREEZE
Project sentinel: PASS
GSE208338 canonical closure verification: PASS
Frozen-threshold consistency checks: PASS
Outcome-evaluation gate consistency: PASS

--------------------------------------------------------------------------------------------------------------
MASTER EXTERNAL TECHNICAL-GATE REGISTRY
--------------------------------------------------------------------------------------------------------------
   dataset mapped_gene_count gene_count_coverage absolute_weight_coverage
  GSE87194                14           0.5833330                0.3332160
  GSE80655                20           0.8333330                0.7197100
 GSE224683                23           0.9583330                0.9266000
 GSE282552                 9           0.3750000                0.1961925
 GSE208338                14           0.5833333                0.3818154
 positive_weight_coverage negative_weight_coverage technical_pass
            

In [59]:
# ==================================================================================================
# STEP 09J.2 — FINAL PUBLIC EXTERNAL-VALIDATION PHASE CLOSURE
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Formally close the CURRENT public external-validation phase after five completed cohort screens.
#
# CURRENT EVIDENCE
#   Public cohorts screened:        5
#   Technical PASS:                 1
#   Technical non-evaluable:        4
#   Outcome-evaluated cohorts:      1
#   Valid evaluated cohort:         GSE224683
#   GSE224683 primary result:       NULL / chance-level with wide uncertainty
#
# IMPORTANT INTERPRETATION
#   - This does NOT claim successful external validation.
#   - This does NOT claim that no other public datasets exist.
#   - Current public screening is treated as a completed pragmatic analysis set,
#     not an exhaustive census of every potentially relevant public dataset.
#   - Additional external cohorts may be evaluated later if scientifically necessary.
#   - Current manuscript interpretation:
#         external transferability remains UNRESOLVED / CHALLENGED.
#
# NO NEW OUTCOME ANALYSIS IS PERFORMED HERE.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 09J.2 — FINAL PUBLIC EXTERNAL-VALIDATION PHASE CLOSURE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 09J.2A — PROJECT PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

MASTER_DIR <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "master_registry_v1"
)

REGISTRY_PATH <- file.path(
    MASTER_DIR,
    "SCZ_MSS_v1_external_technical_gate_registry_v1.csv"
)

OUT_TABLE <- file.path(
    MASTER_DIR,
    "SCZ_MSS_v1_FINAL_external_validation_summary_table_v1.csv"
)

OUT_CLOSURE <- file.path(
    MASTER_DIR,
    "SCZ_MSS_v1_FINAL_external_validation_phase_closure_v1.txt"
)

OUT_MANIFEST <- file.path(
    MASTER_DIR,
    "SCZ_MSS_v1_FINAL_external_validation_phase_closure_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 09J.2B — PROJECT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

if (!file.exists(REGISTRY_PATH)) {
    stop(
        "Canonical STEP09J.1 registry missing:\n",
        REGISTRY_PATH
    )
}

cat("Master external registry found: PASS\n")

# --------------------------------------------------------------------------------------------------
# 09J.2C — LOAD CANONICAL REGISTRY
# --------------------------------------------------------------------------------------------------

registry <- read.csv(
    REGISTRY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

required_columns <- c(
    "dataset",
    "assay_platform_summary",
    "n_samples",
    "mapped_gene_count",
    "gene_count_coverage",
    "absolute_weight_coverage",
    "positive_weight_coverage",
    "negative_weight_coverage",
    "technical_pass",
    "outcome_evaluated",
    "primary_external_auroc",
    "primary_external_auroc_ci_lower",
    "primary_external_auroc_ci_upper",
    "primary_external_hedges_g",
    "primary_external_mean_difference",
    "final_status",
    "interpretation"
)

missing_columns <- setdiff(
    required_columns,
    names(registry)
)

if (length(missing_columns) > 0L) {
    stop(
        "Registry missing required column(s): ",
        paste(missing_columns, collapse = ", ")
    )
}

if (nrow(registry) != 5L) {
    stop(
        "Expected exactly five completed public external cohorts; observed ",
        nrow(registry)
    )
}

expected_datasets <- c(
    "GSE87194",
    "GSE80655",
    "GSE224683",
    "GSE282552",
    "GSE208338"
)

if (!setequal(registry$dataset, expected_datasets)) {
    stop(
        "Unexpected dataset set in canonical registry.\n",
        "Observed: ",
        paste(registry$dataset, collapse = ", ")
    )
}

cat("Completed public cohort set: PASS (5 datasets)\n")

# --------------------------------------------------------------------------------------------------
# 09J.2D — STRICT LOGICAL NORMALIZATION
# --------------------------------------------------------------------------------------------------

to_logical <- function(x) {

    if (is.logical(x)) {
        return(x)
    }

    y <- toupper(
        trimws(
            as.character(x)
        )
    )

    if (any(!y %in% c("TRUE", "FALSE"))) {
        stop(
            "Unexpected logical encoding: ",
            paste(
                unique(
                    y[
                        !y %in% c("TRUE", "FALSE")
                    ]
                ),
                collapse = ", "
            )
        )
    }

    y == "TRUE"
}

registry$technical_pass <- to_logical(
    registry$technical_pass
)

registry$outcome_evaluated <- to_logical(
    registry$outcome_evaluated
)

# --------------------------------------------------------------------------------------------------
# 09J.2E — VERIFY CURRENT EXTERNAL STATE
# --------------------------------------------------------------------------------------------------

n_screened <- nrow(registry)

n_technical_pass <- sum(
    registry$technical_pass
)

n_technical_fail <- sum(
    !registry$technical_pass
)

n_outcome_evaluated <- sum(
    registry$outcome_evaluated
)

if (n_screened != 5L) {
    stop("Expected 5 screened cohorts.")
}

if (n_technical_pass != 1L) {
    stop(
        "Expected exactly one technical-PASS cohort; observed ",
        n_technical_pass
    )
}

if (n_technical_fail != 4L) {
    stop(
        "Expected four technically non-evaluable cohorts; observed ",
        n_technical_fail
    )
}

if (n_outcome_evaluated != 1L) {
    stop(
        "Expected exactly one outcome-evaluated cohort; observed ",
        n_outcome_evaluated
    )
}

pass_dataset <- registry$dataset[
    registry$technical_pass
]

evaluated_dataset <- registry$dataset[
    registry$outcome_evaluated
]

if (!identical(pass_dataset, "GSE224683")) {
    stop(
        "Unexpected technical-PASS dataset: ",
        paste(pass_dataset, collapse = ", ")
    )
}

if (!identical(evaluated_dataset, "GSE224683")) {
    stop(
        "Unexpected outcome-evaluated dataset: ",
        paste(evaluated_dataset, collapse = ", ")
    )
}

g224 <- registry[
    registry$dataset == "GSE224683",
    ,
    drop = FALSE
]

if (nrow(g224) != 1L) {
    stop("Could not uniquely identify GSE224683.")
}

if (
    is.na(g224$primary_external_auroc) ||
    abs(g224$primary_external_auroc - 0.5067) > 1e-4
) {
    stop(
        "GSE224683 AUROC does not match the frozen external result."
    )
}

if (
    is.na(g224$primary_external_auroc_ci_lower) ||
    is.na(g224$primary_external_auroc_ci_upper) ||
    abs(g224$primary_external_auroc_ci_lower - 0.2889) > 1e-4 ||
    abs(g224$primary_external_auroc_ci_upper - 0.7244) > 1e-4
) {
    stop(
        "GSE224683 AUROC CI does not match the frozen external result."
    )
}

cat("Technical-gate state verification: PASS\n")
cat("Outcome-evaluation state verification: PASS\n")
cat("GSE224683 frozen external result verification: PASS\n")

# --------------------------------------------------------------------------------------------------
# 09J.2F — CREATE FINAL MANUSCRIPT-READY EXTERNAL SUMMARY TABLE
# --------------------------------------------------------------------------------------------------

summary_table <- data.frame(

    dataset = registry$dataset,

    platform = registry$assay_platform_summary,

    n = registry$n_samples,

    mapped_genes = paste0(
        registry$mapped_gene_count,
        "/24"
    ),

    gene_coverage = registry$gene_count_coverage,

    absolute_weight_coverage =
        registry$absolute_weight_coverage,

    positive_weight_coverage =
        registry$positive_weight_coverage,

    negative_weight_coverage =
        registry$negative_weight_coverage,

    technical_gate = ifelse(
        registry$technical_pass,
        "PASS",
        "FAIL"
    ),

    outcome_evaluated = ifelse(
        registry$outcome_evaluated,
        "YES",
        "NO"
    ),

    external_AUROC = ifelse(
        registry$outcome_evaluated,
        sprintf(
            "%.4f",
            registry$primary_external_auroc
        ),
        NA_character_
    ),

    external_AUROC_95CI = ifelse(
        registry$outcome_evaluated,
        sprintf(
            "[%.4f, %.4f]",
            registry$primary_external_auroc_ci_lower,
            registry$primary_external_auroc_ci_upper
        ),
        NA_character_
    ),

    external_Hedges_g = ifelse(
        registry$outcome_evaluated,
        sprintf(
            "%.4f",
            registry$primary_external_hedges_g
        ),
        NA_character_
    ),

    external_mean_difference = ifelse(
        registry$outcome_evaluated,
        sprintf(
            "%.6f",
            registry$primary_external_mean_difference
        ),
        NA_character_
    ),

    final_interpretation = c(
        "Technically non-evaluable; no biological replication conclusion.",
        "Technically non-evaluable; no biological replication conclusion.",
        "Technically eligible; frozen-score external result was essentially null with wide uncertainty.",
        "Technically non-evaluable; no biological replication conclusion.",
        "Technically non-evaluable; no biological replication conclusion."
    ),

    stringsAsFactors = FALSE
)

write.csv(
    summary_table,
    OUT_TABLE,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 09J.2G — FINAL EXTERNAL-PHASE CLOSURE RECORD
# --------------------------------------------------------------------------------------------------

closure_lines <- c(

    "==============================================================================================================",
    "SCZ-MSS v1 — FINAL PUBLIC EXTERNAL-VALIDATION PHASE CLOSURE",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "CURRENT PUBLIC EXTERNAL SCREENING SET",
    paste0(
        "Datasets screened: ",
        n_screened
    ),
    paste0(
        "Technical PASS: ",
        n_technical_pass
    ),
    paste0(
        "Technically non-evaluable: ",
        n_technical_fail
    ),
    paste0(
        "Outcome evaluated: ",
        n_outcome_evaluated
    ),

    "",

    "DATASET STATUS",

    paste0(
        registry$dataset,
        " | mapped=",
        registry$mapped_gene_count,
        "/24",
        " | gene=",
        sprintf("%.4f", registry$gene_count_coverage),
        " | absW=",
        sprintf("%.4f", registry$absolute_weight_coverage),
        " | posW=",
        sprintf("%.4f", registry$positive_weight_coverage),
        " | negW=",
        sprintf("%.4f", registry$negative_weight_coverage),
        " | PASS=",
        registry$technical_pass,
        " | outcome=",
        registry$outcome_evaluated
    ),

    "",

    "VALID EXTERNAL OUTCOME TEST",

    "Dataset: GSE224683",

    paste0(
        "AUROC: ",
        sprintf(
            "%.4f",
            g224$primary_external_auroc
        ),
        " [",
        sprintf(
            "%.4f",
            g224$primary_external_auroc_ci_lower
        ),
        ", ",
        sprintf(
            "%.4f",
            g224$primary_external_auroc_ci_upper
        ),
        "]"
    ),

    paste0(
        "Hedges g: ",
        sprintf(
            "%.4f",
            g224$primary_external_hedges_g
        )
    ),

    paste0(
        "Mean score difference: ",
        sprintf(
            "%.6f",
            g224$primary_external_mean_difference
        )
    ),

    "",

    "PRIMARY INTERPRETATION",

    paste(
        "Among five completed public external-cohort screens,",
        "four cohorts were technically non-evaluable under the frozen",
        "SCZ-MSS v1 feature-coverage requirements."
    ),

    paste(
        "GSE224683 satisfied the frozen technical gate and therefore",
        "provided the valid public external diagnosis-score evaluation."
    ),

    paste(
        "The GSE224683 result was essentially null, with AUROC near 0.5",
        "and wide uncertainty."
    ),

    paste(
        "Therefore the current public evidence does not establish",
        "successful biological transferability of the frozen SCZ-MSS score."
    ),

    paste(
        "At the same time, the four technical-gate failures must not be",
        "described as biological replication failures."
    ),

    "",

    "CURRENT EXTERNAL CONCLUSION",

    "EXTERNAL TRANSFERABILITY STATUS: UNRESOLVED / CHALLENGED",

    "",

    "SCOPE DECISION",

    paste(
        "The current five-cohort public external-screening phase is now",
        "closed for the primary analysis workflow."
    ),

    paste(
        "This closure is pragmatic and analysis-scope based.",
        "It is NOT a claim that every potentially relevant public dataset",
        "has been exhaustively identified or screened."
    ),

    paste(
        "Additional external cohorts may be examined later if required",
        "during peer review or if a clearly compatible independent dataset",
        "becomes available."
    ),

    paste(
        "No additional cohort is required to complete the currently defined",
        "discovery, stability, score-lock, and external-transferability analysis framework."
    ),

    "",

    "MANUSCRIPT REPORTING RULES",

    "1. Do NOT write that SCZ-MSS was successfully externally validated.",

    "2. Do NOT omit the GSE224683 null result.",

    "3. Do NOT classify GSE87194, GSE80655, GSE282552, or GSE208338 as biological replication failures.",

    "4. State explicitly that assay/annotation compatibility restricted evaluability across public cohorts.",

    "5. State explicitly that only one screened public cohort passed the frozen technical gate.",

    "6. Describe additional public-dataset screening as non-exhaustive rather than claiming no other datasets exist.",

    "7. Preserve the distinction between the nested high-dimensional internal model and the portable 24-gene SCZ-MSS score.",

    "",

    "NEXT PRIMARY ANALYTICAL PHASE",

    "BIOLOGICAL / FUNCTIONAL INTERPRETATION OF THE FROZEN 24-GENE SHARED STATE",

    "Planned components:",
    "- gene-level annotation and identity verification",
    "- coding versus non-coding composition",
    "- functional/pathway context",
    "- brain/cell-type context",
    "- schizophrenia/genetic convergence where supportable",
    "- shared-state versus regional-deviation biological interpretation",

    "=============================================================================================================="
)

writeLines(
    closure_lines,
    OUT_CLOSURE,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 09J.2H — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    REGISTRY_PATH,
    OUT_TABLE,
    OUT_CLOSURE
)

manifest <- data.frame(
    file = basename(manifest_paths),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(manifest_paths)$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 09J.2I — FINAL CONSOLE OUTPUT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nFINAL EXTERNAL VALIDATION SUMMARY\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    summary_table[
        ,
        c(
            "dataset",
            "n",
            "mapped_genes",
            "gene_coverage",
            "absolute_weight_coverage",
            "positive_weight_coverage",
            "negative_weight_coverage",
            "technical_gate",
            "outcome_evaluated",
            "external_AUROC"
        )
    ],
    row.names = FALSE
)

cat(
    "\nDatasets screened: ",
    n_screened,
    "\n",
    sep = ""
)

cat(
    "Technical PASS: ",
    n_technical_pass,
    "\n",
    sep = ""
)

cat(
    "Technically non-evaluable: ",
    n_technical_fail,
    "\n",
    sep = ""
)

cat(
    "Outcome evaluated: ",
    n_outcome_evaluated,
    "\n",
    sep = ""
)

cat(
    "\nValid external outcome dataset: GSE224683\n"
)

cat(
    "GSE224683 AUROC: ",
    sprintf(
        "%.4f",
        g224$primary_external_auroc
    ),
    " [",
    sprintf(
        "%.4f",
        g224$primary_external_auroc_ci_lower
    ),
    ", ",
    sprintf(
        "%.4f",
        g224$primary_external_auroc_ci_upper
    ),
    "]\n",
    sep = ""
)

cat(
    "\nEXTERNAL TRANSFERABILITY STATUS: UNRESOLVED / CHALLENGED\n"
)

cat(
    "CURRENT FIVE-COHORT PUBLIC EXTERNAL PHASE: CLOSED\n"
)

cat(
    "ADDITIONAL PUBLIC SCREENING: OPTIONAL / FUTURE, NOT REQUIRED FOR CURRENT PRIMARY WORKFLOW\n"
)

cat(
    "\nFinal manuscript-ready summary table:\n",
    OUT_TABLE,
    "\n",
    sep = ""
)

cat(
    "\nExternal-phase closure record:\n",
    OUT_CLOSURE,
    "\n",
    sep = ""
)

cat(
    "\nClosure manifest:\n",
    OUT_MANIFEST,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 09J.2 COMPLETE — PUBLIC EXTERNAL-VALIDATION PHASE CLOSED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 09J.2 — FINAL PUBLIC EXTERNAL-VALIDATION PHASE CLOSURE
Project sentinel: PASS
Master external registry found: PASS
Completed public cohort set: PASS (5 datasets)
Technical-gate state verification: PASS
Outcome-evaluation state verification: PASS
GSE224683 frozen external result verification: PASS

--------------------------------------------------------------------------------------------------------------
FINAL EXTERNAL VALIDATION SUMMARY
--------------------------------------------------------------------------------------------------------------
   dataset   n mapped_genes gene_coverage absolute_weight_coverage
  GSE87194  38        14/24     0.5833330                0.3332160
  GSE80655 281        20/24     0.8333330                0.7197100
 GSE224683  30        23/24     0.9583330                0.9266000
 GSE282552  18         9/24     0.3750000                0.1961925
 GSE208338 169        14/24     0.5833333                0.3818154
 positive_weight_coverage negative_we

In [60]:
# ==================================================================================================
# STEP 10A — FROZEN 24-GENE BIOLOGICAL ANNOTATION + COMPOSITION AUDIT
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Prepare the frozen SCZ-MSS 24-gene state for biological interpretation WITHOUT modifying
#   the frozen signature, weights, directions, or inclusion criteria.
#
# THIS STEP WILL:
#   1. Verify the canonical 24-gene frozen score file.
#   2. Summarize coding vs non-coding composition.
#   3. Summarize positive vs negative directions.
#   4. Quantify absolute-weight distribution.
#   5. Preserve frozen Ensembl IDs / symbols / gene types as canonical annotations.
#   6. Add CURRENT org.Hs.eg.db annotation only as supplemental descriptive context.
#   7. Create enrichment-ready protein-coding and named-gene subsets.
#
# THIS STEP WILL NOT:
#   - change frozen gene identifiers
#   - rescue genes by synonyms
#   - alter weights
#   - perform pathway enrichment yet
#   - perform outcome analysis
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10A — FROZEN 24-GENE BIOLOGICAL ANNOTATION + COMPOSITION AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10A.1 — PROJECT PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

WEIGHT_PATH <- file.path(
    PROJECT_ROOT,
    "08_score_lock",
    "SCZ_MSS_v1_gene_weights.csv"
)

EXTERNAL_CLOSURE_PATH <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "master_registry_v1",
    "SCZ_MSS_v1_FINAL_external_validation_phase_closure_v1.txt"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "annotation_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_FULL <- file.path(
    OUT_DIR,
    "STEP10A_SCZ_MSS_24gene_biological_annotation_v1.csv"
)

OUT_PC <- file.path(
    OUT_DIR,
    "STEP10A_SCZ_MSS_protein_coding_subset_v1.csv"
)

OUT_NC <- file.path(
    OUT_DIR,
    "STEP10A_SCZ_MSS_non_protein_coding_subset_v1.csv"
)

OUT_NAMED <- file.path(
    OUT_DIR,
    "STEP10A_SCZ_MSS_named_gene_subset_v1.csv"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP10A_SCZ_MSS_biological_composition_summary_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP10A_SCZ_MSS_biology_annotation_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10A.2 — PROJECT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

if (!file.exists(WEIGHT_PATH)) {
    stop("Frozen score file missing: ", WEIGHT_PATH)
}

cat("Frozen SCZ-MSS weight file: PASS\n")

if (!file.exists(EXTERNAL_CLOSURE_PATH)) {
    stop(
        "External-validation phase closure missing.\n",
        "Expected: ", EXTERNAL_CLOSURE_PATH
    )
}

cat("External-validation phase closure: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10A.3 — LOAD FROZEN 24-GENE SCORE
# --------------------------------------------------------------------------------------------------

sig <- read.csv(
    WEIGHT_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

required_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "protein_coding",
    "direction",
    "discovery_median_coefficient",
    "absolute_weight",
    "final_normalized_weight",
    "normalized_absolute_weight",
    "outer_selection_frequency",
    "outer_sign_consistency",
    "half_sample_selection_probability",
    "half_sample_sign_consistency",
    "null_selection_probability"
)

missing_cols <- setdiff(
    required_cols,
    names(sig)
)

if (length(missing_cols) > 0L) {
    stop(
        "Frozen score file is missing required column(s): ",
        paste(missing_cols, collapse = ", ")
    )
}

if (nrow(sig) != 24L) {
    stop(
        "Expected exactly 24 frozen genes; observed ",
        nrow(sig)
    )
}

if (anyDuplicated(sig$ensemblID_stable) > 0L) {
    stop("Duplicate stable Ensembl IDs detected in frozen score.")
}

if (
    abs(
        sum(
            abs(sig$final_normalized_weight)
        ) - 1
    ) > 1e-6
) {
    stop(
        "Frozen normalized weights do not sum to absolute weight 1."
    )
}

cat("Frozen gene count: 24 — PASS\n")
cat("Frozen normalized absolute weight: 1.0 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10A.4 — NORMALIZE FROZEN DESCRIPTIVE FIELDS
# --------------------------------------------------------------------------------------------------

sig$ensemblID_stable <- trimws(
    as.character(sig$ensemblID_stable)
)

sig$gene_symbol <- trimws(
    as.character(sig$gene_symbol)
)

sig$gene_type <- trimws(
    as.character(sig$gene_type)
)

sig$gene_symbol[
    is.na(sig$gene_symbol) |
    sig$gene_symbol == "" |
    toupper(sig$gene_symbol) %in% c("NA", "N/A", "NULL")
] <- NA_character_

# Strict protein-coding interpretation from frozen file.
if (is.logical(sig$protein_coding)) {

    sig$protein_coding_flag <- sig$protein_coding

} else {

    pc_text <- toupper(
        trimws(
            as.character(sig$protein_coding)
        )
    )

    if (all(pc_text %in% c("TRUE", "FALSE"))) {

        sig$protein_coding_flag <- pc_text == "TRUE"

    } else {

        # Fallback to frozen gene_type only if the stored field is not logical text.
        sig$protein_coding_flag <- (
            tolower(sig$gene_type) == "protein_coding"
        )
    }
}

if (anyNA(sig$protein_coding_flag)) {
    stop("Unable to resolve frozen protein-coding flag for all 24 genes.")
}

sig$direction_standardized <- ifelse(
    sig$final_normalized_weight > 0,
    "POSITIVE",
    "NEGATIVE"
)

# --------------------------------------------------------------------------------------------------
# 10A.5 — VERIFY KNOWN FROZEN COMPOSITION
# --------------------------------------------------------------------------------------------------

n_pc <- sum(sig$protein_coding_flag)
n_nonpc <- sum(!sig$protein_coding_flag)

n_positive <- sum(
    sig$final_normalized_weight > 0
)

n_negative <- sum(
    sig$final_normalized_weight < 0
)

positive_abs_fraction <- sum(
    abs(
        sig$final_normalized_weight[
            sig$final_normalized_weight > 0
        ]
    )
)

negative_abs_fraction <- sum(
    abs(
        sig$final_normalized_weight[
            sig$final_normalized_weight < 0
        ]
    )
)

if (n_pc != 12L || n_nonpc != 12L) {
    stop(
        "Frozen coding composition mismatch.\n",
        "Expected 12 protein-coding + 12 non-protein-coding.\n",
        "Observed: ", n_pc, " + ", n_nonpc
    )
}

if (n_positive != 9L || n_negative != 15L) {
    stop(
        "Frozen direction composition mismatch.\n",
        "Expected 9 positive + 15 negative.\n",
        "Observed: ", n_positive, " + ", n_negative
    )
}

cat("Protein-coding genes: 12/24 — PASS\n")
cat("Non-protein-coding genes: 12/24 — PASS\n")
cat("Positive-direction genes: 9/24 — PASS\n")
cat("Negative-direction genes: 15/24 — PASS\n")

cat(
    "Positive absolute-weight fraction: ",
    sprintf("%.6f", positive_abs_fraction),
    "\n",
    sep = ""
)

cat(
    "Negative absolute-weight fraction: ",
    sprintf("%.6f", negative_abs_fraction),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10A.6 — LOAD SUPPLEMENTAL HUMAN ANNOTATION
#
# IMPORTANT:
# org.Hs.eg.db annotation is DESCRIPTIVE ONLY.
# Frozen identifiers remain canonical regardless of whether current annotation is available.
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("BiocManager", quietly = TRUE)) {

    install.packages(
        "BiocManager",
        repos = "https://cloud.r-project.org"
    )
}

needed <- c(
    "AnnotationDbi",
    "org.Hs.eg.db"
)

missing_packages <- needed[
    !vapply(
        needed,
        requireNamespace,
        quietly = TRUE,
        FUN.VALUE = logical(1)
    )
]

if (length(missing_packages) > 0L) {

    BiocManager::install(
        missing_packages,
        ask = FALSE,
        update = FALSE
    )
}

if (!requireNamespace("AnnotationDbi", quietly = TRUE)) {
    stop("AnnotationDbi could not be loaded.")
}

if (!requireNamespace("org.Hs.eg.db", quietly = TRUE)) {
    stop("org.Hs.eg.db could not be loaded.")
}

db <- org.Hs.eg.db::org.Hs.eg.db

current_ann <- AnnotationDbi::select(
    db,
    keys = sig$ensemblID_stable,
    keytype = "ENSEMBL",
    columns = c(
        "SYMBOL",
        "GENENAME",
        "ENTREZID"
    )
)

names(current_ann)[
    names(current_ann) == "ENSEMBL"
] <- "ensemblID_stable"

# --------------------------------------------------------------------------------------------------
# 10A.7 — REDUCE CURRENT ANNOTATION TO ONE DESCRIPTIVE RECORD PER FROZEN ID
#
# No current annotation is allowed to alter the frozen signature.
# If several rows exist, unique non-missing values are concatenated for audit visibility.
# --------------------------------------------------------------------------------------------------

collapse_unique <- function(x) {

    x <- unique(
        as.character(x)
    )

    x <- x[
        !is.na(x) &
        nzchar(x)
    ]

    if (length(x) == 0L) {
        return(NA_character_)
    }

    paste(
        sort(x),
        collapse = ";"
    )
}

if (nrow(current_ann) > 0L) {

    current_ann_collapsed <- aggregate(
        cbind(
            SYMBOL,
            GENENAME,
            ENTREZID
        ) ~ ensemblID_stable,
        data = current_ann,
        FUN = collapse_unique
    )

} else {

    current_ann_collapsed <- data.frame(
        ensemblID_stable = character(0),
        SYMBOL = character(0),
        GENENAME = character(0),
        ENTREZID = character(0),
        stringsAsFactors = FALSE
    )
}

names(current_ann_collapsed)[
    names(current_ann_collapsed) == "SYMBOL"
] <- "current_orgdb_symbol"

names(current_ann_collapsed)[
    names(current_ann_collapsed) == "GENENAME"
] <- "current_orgdb_gene_name"

names(current_ann_collapsed)[
    names(current_ann_collapsed) == "ENTREZID"
] <- "current_orgdb_entrez"

bio <- merge(
    sig,
    current_ann_collapsed,
    by = "ensemblID_stable",
    all.x = TRUE,
    sort = FALSE
)

# Restore original frozen order.
bio <- bio[
    match(
        sig$ensemblID_stable,
        bio$ensemblID_stable
    ),
    ,
    drop = FALSE
]

if (!identical(
    bio$ensemblID_stable,
    sig$ensemblID_stable
)) {
    stop("Frozen gene order could not be restored.")
}

# --------------------------------------------------------------------------------------------------
# 10A.8 — DERIVE BIOLOGICAL-INTERPRETATION FLAGS
# --------------------------------------------------------------------------------------------------

bio$has_frozen_symbol <- !is.na(
    bio$gene_symbol
)

bio$has_current_orgdb_annotation <- (
    !is.na(bio$current_orgdb_symbol) |
    !is.na(bio$current_orgdb_gene_name) |
    !is.na(bio$current_orgdb_entrez)
)

bio$symbol_agreement_current_vs_frozen <- ifelse(
    !is.na(bio$gene_symbol) &
    !is.na(bio$current_orgdb_symbol),

    vapply(
        seq_len(nrow(bio)),
        function(i) {

            frozen_symbol <- bio$gene_symbol[i]

            current_symbols <- strsplit(
                bio$current_orgdb_symbol[i],
                ";",
                fixed = TRUE
            )[[1]]

            frozen_symbol %in% current_symbols
        },
        FUN.VALUE = logical(1)
    ),

    NA
)

bio$absolute_weight_rank_recomputed <- rank(
    -abs(bio$final_normalized_weight),
    ties.method = "min"
)

bio$weight_percent <- (
    100 *
    abs(bio$final_normalized_weight)
)

bio$interpretation_class <- ifelse(
    bio$protein_coding_flag,
    "PROTEIN_CODING",
    "NON_PROTEIN_CODING"
)

bio$enrichment_eligible_conventional <- (
    bio$protein_coding_flag &
    !is.na(bio$current_orgdb_entrez)
)

# --------------------------------------------------------------------------------------------------
# 10A.9 — CREATE BIOLOGY SUBSETS
# --------------------------------------------------------------------------------------------------

protein_coding_subset <- bio[
    bio$protein_coding_flag,
    ,
    drop = FALSE
]

non_protein_coding_subset <- bio[
    !bio$protein_coding_flag,
    ,
    drop = FALSE
]

named_gene_subset <- bio[
    bio$has_frozen_symbol,
    ,
    drop = FALSE
]

n_current_annotated <- sum(
    bio$has_current_orgdb_annotation
)

n_conventional_enrichment <- sum(
    bio$enrichment_eligible_conventional
)

# --------------------------------------------------------------------------------------------------
# 10A.10 — WRITE OUTPUT TABLES
# --------------------------------------------------------------------------------------------------

write.csv(
    bio,
    OUT_FULL,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    protein_coding_subset,
    OUT_PC,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    non_protein_coding_subset,
    OUT_NC,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    named_gene_subset,
    OUT_NAMED,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10A.11 — TOP-WEIGHT GENES
# --------------------------------------------------------------------------------------------------

top_order <- order(
    abs(bio$final_normalized_weight),
    decreasing = TRUE
)

top_genes <- bio[
    top_order,
    c(
        "ensemblID_stable",
        "gene_symbol",
        "gene_type",
        "protein_coding_flag",
        "direction_standardized",
        "final_normalized_weight",
        "weight_percent"
    ),
    drop = FALSE
]

# --------------------------------------------------------------------------------------------------
# 10A.12 — HUMAN-READABLE SUMMARY
# --------------------------------------------------------------------------------------------------

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10A — SCZ-MSS 24-GENE BIOLOGICAL ANNOTATION + COMPOSITION AUDIT",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "FROZEN SIGNATURE COMPOSITION",

    paste0(
        "Total frozen genes: ",
        nrow(bio)
    ),

    paste0(
        "Protein-coding: ",
        n_pc,
        "/24"
    ),

    paste0(
        "Non-protein-coding: ",
        n_nonpc,
        "/24"
    ),

    paste0(
        "Positive-direction genes: ",
        n_positive,
        "/24"
    ),

    paste0(
        "Negative-direction genes: ",
        n_negative,
        "/24"
    ),

    paste0(
        "Positive-direction absolute-weight fraction: ",
        sprintf("%.6f", positive_abs_fraction)
    ),

    paste0(
        "Negative-direction absolute-weight fraction: ",
        sprintf("%.6f", negative_abs_fraction)
    ),

    "",

    "ANNOTATION STATUS",

    paste0(
        "Genes with frozen gene symbol: ",
        sum(bio$has_frozen_symbol),
        "/24"
    ),

    paste0(
        "Genes with supplemental current org.Hs.eg.db annotation: ",
        n_current_annotated,
        "/24"
    ),

    paste0(
        "Protein-coding genes currently suitable for conventional Entrez-based enrichment: ",
        n_conventional_enrichment,
        "/",
        n_pc
    ),

    "",

    "BIOLOGICAL INTERPRETATION PRINCIPLE",

    "The complete 24-gene frozen state remains the primary biological object.",

    paste(
        "Conventional pathway enrichment will be considered separately for genes",
        "with compatible functional annotation and must not be interpreted as",
        "representing the entire non-coding component of the frozen state."
    ),

    paste(
        "The 12 non-protein-coding genes will therefore be described and interpreted",
        "separately rather than silently discarded from the biological narrative."
    ),

    "",

    "EXTERNAL CONTEXT",

    "Five public external cohorts were screened.",
    "Only GSE224683 passed the frozen technical gate.",
    "Its frozen-score external outcome result was essentially null.",
    "Current external transferability remains unresolved / challenged.",

    "",

    "NEXT ANALYSIS",

    paste(
        "Proceed to structured functional interpretation of the frozen 24-gene state,",
        "including conventional enrichment where technically appropriate and separate",
        "gene-level interpretation of non-coding components."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10A.13 — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    WEIGHT_PATH,
    OUT_FULL,
    OUT_PC,
    OUT_NC,
    OUT_NAMED,
    OUT_SUMMARY
)

manifest <- data.frame(
    file = basename(manifest_paths),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(manifest_paths)$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10A.14 — CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nFROZEN 24-GENE COMPOSITION\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Protein-coding: ",
    n_pc,
    "/24\n",
    sep = ""
)

cat(
    "Non-protein-coding: ",
    n_nonpc,
    "/24\n",
    sep = ""
)

cat(
    "Positive direction: ",
    n_positive,
    "/24\n",
    sep = ""
)

cat(
    "Negative direction: ",
    n_negative,
    "/24\n",
    sep = ""
)

cat(
    "Positive absolute-weight fraction: ",
    sprintf("%.6f", positive_abs_fraction),
    "\n",
    sep = ""
)

cat(
    "Negative absolute-weight fraction: ",
    sprintf("%.6f", negative_abs_fraction),
    "\n",
    sep = ""
)

cat(
    "\nCurrent org.Hs.eg.db annotation available: ",
    n_current_annotated,
    "/24\n",
    sep = ""
)

cat(
    "Conventional enrichment-eligible protein-coding genes: ",
    n_conventional_enrichment,
    "/",
    n_pc,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nFROZEN GENES RANKED BY ABSOLUTE WEIGHT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    top_genes,
    row.names = FALSE
)

cat(
    "\nCanonical biological annotation table:\n",
    OUT_FULL,
    "\n",
    sep = ""
)

cat(
    "\nProtein-coding subset:\n",
    OUT_PC,
    "\n",
    sep = ""
)

cat(
    "\nNon-protein-coding subset:\n",
    OUT_NC,
    "\n",
    sep = ""
)

cat(
    "\nBiological-composition summary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10A COMPLETE — FROZEN 24-GENE BIOLOGICAL ANNOTATION AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10A — FROZEN 24-GENE BIOLOGICAL ANNOTATION + COMPOSITION AUDIT
Project sentinel: PASS
Frozen SCZ-MSS weight file: PASS
External-validation phase closure: PASS
Frozen gene count: 24 — PASS
Frozen normalized absolute weight: 1.0 — PASS
Protein-coding genes: 12/24 — PASS
Non-protein-coding genes: 12/24 — PASS
Positive-direction genes: 9/24 — PASS
Negative-direction genes: 15/24 — PASS
Positive absolute-weight fraction: 0.386919
Negative absolute-weight fraction: 0.613081


'select()' returned 1:1 mapping between keys and columns




--------------------------------------------------------------------------------------------------------------
FROZEN 24-GENE COMPOSITION
--------------------------------------------------------------------------------------------------------------
Protein-coding: 12/24
Non-protein-coding: 12/24
Positive direction: 9/24
Negative direction: 15/24
Positive absolute-weight fraction: 0.386919
Negative absolute-weight fraction: 0.613081

Current org.Hs.eg.db annotation available: 18/24
Conventional enrichment-eligible protein-coding genes: 12/12

--------------------------------------------------------------------------------------------------------------
FROZEN GENES RANKED BY ABSOLUTE WEIGHT
--------------------------------------------------------------------------------------------------------------
 ensemblID_stable gene_symbol                          gene_type
  ENSG00000222533   RNU6-705P                              snRNA
  ENSG00000273629        <NA>                              s

In [61]:
# ==================================================================================================
# STEP 10B.1 — FREEZE DISCOVERY-MATCHED FUNCTIONAL-ENRICHMENT BACKGROUND UNIVERSE
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Construct the correct enrichment background from the FINAL 23,740-gene discovery universe,
#   rather than using the whole human genome.
#
# WHY THIS MATTERS
#   The 24-gene SCZ-MSS signature arose from the filtered BrainSeq discovery universe.
#   Therefore competitive functional enrichment must use genes that had an opportunity
#   to enter the model/signature as the reference universe.
#
# THIS STEP WILL:
#   1. Locate the canonical BrainSeq primary gene-annotation table.
#   2. Verify that it corresponds to the ~23,740-gene discovery universe.
#   3. Standardize stable Ensembl IDs.
#   4. Identify protein-coding discovery genes.
#   5. Map protein-coding discovery genes to Entrez IDs using org.Hs.eg.db.
#   6. Freeze:
#        - full discovery Ensembl universe
#        - protein-coding discovery universe
#        - Entrez-mappable protein-coding enrichment background
#   7. Verify that all 12 frozen protein-coding SCZ-MSS genes are represented.
#
# THIS STEP WILL NOT:
#   - perform GO enrichment
#   - perform Reactome enrichment
#   - change any frozen signature gene
#   - use whole-genome background
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10B.1 — DISCOVERY-MATCHED ENRICHMENT BACKGROUND UNIVERSE FREEZE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10B.1A — PROJECT PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

PROCESSED_DIR <- file.path(
    PROJECT_ROOT,
    "04_processed"
)

STEP10A_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "annotation_v1",
    "STEP10A_SCZ_MSS_24gene_biological_annotation_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_FULL_ENSEMBL <- file.path(
    OUT_DIR,
    "STEP10B1_discovery_full_Ensembl_universe_v1.csv"
)

OUT_PC_ENSEMBL <- file.path(
    OUT_DIR,
    "STEP10B1_discovery_protein_coding_Ensembl_universe_v1.csv"
)

OUT_PC_ENTREZ <- file.path(
    OUT_DIR,
    "STEP10B1_discovery_protein_coding_Entrez_background_v1.csv"
)

OUT_SIGNATURE_PC <- file.path(
    OUT_DIR,
    "STEP10B1_SCZ_MSS_12_protein_coding_enrichment_input_v1.csv"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP10B1_enrichment_background_summary_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP10B1_enrichment_background_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10B.1B — PROJECT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

if (!dir.exists(PROCESSED_DIR)) {
    stop("Processed-data directory missing: ", PROCESSED_DIR)
}

if (!file.exists(STEP10A_PATH)) {
    stop("STEP10A biological annotation table missing: ", STEP10A_PATH)
}

cat("STEP10A frozen biological annotation: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10B.1C — IDENTIFY CANONICAL DISCOVERY GENE-ANNOTATION TABLE
#
# Expected canonical object is the annotation associated with the final 23,740-gene
# BrainSeq primary discovery universe.
# --------------------------------------------------------------------------------------------------

annotation_candidates <- list.files(
    PROCESSED_DIR,
    pattern = "\\.csv$",
    full.names = TRUE,
    ignore.case = TRUE
)

annotation_candidates <- annotation_candidates[
    grepl(
        "gene.*annot|annot.*gene|primary.*annot",
        basename(annotation_candidates),
        ignore.case = TRUE
    )
]

if (length(annotation_candidates) == 0L) {
    stop(
        "No plausible gene-annotation CSV found under:\n",
        PROCESSED_DIR
    )
}

inspect_candidate <- function(path) {

    dat <- tryCatch(
        read.csv(
            path,
            stringsAsFactors = FALSE,
            check.names = FALSE
        ),
        error = function(e) NULL
    )

    if (is.null(dat)) {
        return(NULL)
    }

    nm <- names(dat)

    normalized_nm <- tolower(
        gsub(
            "[^a-z0-9]+",
            "_",
            nm
        )
    )

    has_ensembl <- any(
        grepl(
            "ensembl",
            normalized_nm
        )
    )

    has_gene_type <- any(
        grepl(
            "gene_type|genetype",
            normalized_nm
        )
    )

    list(
        path = path,
        data = dat,
        nrow = nrow(dat),
        has_ensembl = has_ensembl,
        has_gene_type = has_gene_type
    )
}

candidate_info <- lapply(
    annotation_candidates,
    inspect_candidate
)

candidate_info <- candidate_info[
    !vapply(
        candidate_info,
        is.null,
        FUN.VALUE = logical(1)
    )
]

if (length(candidate_info) == 0L) {
    stop("Candidate annotation CSV files could not be read.")
}

candidate_score <- vapply(
    candidate_info,
    function(x) {

        abs(x$nrow - 23740) +
            ifelse(x$has_ensembl, 0, 100000) +
            ifelse(x$has_gene_type, 0, 100000)
    },
    FUN.VALUE = numeric(1)
)

best_index <- which.min(candidate_score)

annotation_path <- candidate_info[[best_index]]$path
annotation <- candidate_info[[best_index]]$data

cat(
    "Selected discovery annotation file:\n",
    annotation_path,
    "\n",
    sep = ""
)

cat(
    "Annotation rows: ",
    nrow(annotation),
    "\n",
    sep = ""
)

if (nrow(annotation) != 23740L) {
    stop(
        "Selected annotation table does not contain the expected 23,740 final discovery genes.\n",
        "Observed rows: ",
        nrow(annotation),
        "\nFile: ",
        annotation_path
    )
}

cat("Final discovery gene count: 23,740 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10B.1D — IDENTIFY REQUIRED ANNOTATION COLUMNS
# --------------------------------------------------------------------------------------------------

original_names <- names(annotation)

normalized_names <- tolower(
    gsub(
        "[^a-z0-9]+",
        "_",
        original_names
    )
)

find_column <- function(patterns) {

    for (p in patterns) {

        hit <- grep(
            p,
            normalized_names,
            perl = TRUE
        )

        if (length(hit) > 0L) {
            return(original_names[hit[1]])
        }
    }

    NA_character_
}

ensembl_col <- find_column(
    c(
        "^ensemblid$",
        "^ensembl_id$",
        "^ensembl_gene_id$",
        "ensembl"
    )
)

symbol_col <- find_column(
    c(
        "^symbol$",
        "^gene_symbol$",
        "symbol"
    )
)

gene_type_col <- find_column(
    c(
        "^gene_type$",
        "^genetype$",
        "gene_type"
    )
)

if (is.na(ensembl_col)) {
    stop("Could not identify Ensembl ID column.")
}

if (is.na(gene_type_col)) {
    stop("Could not identify gene_type column.")
}

cat(
    "Ensembl column: ",
    ensembl_col,
    "\n",
    sep = ""
)

cat(
    "Gene-type column: ",
    gene_type_col,
    "\n",
    sep = ""
)

if (!is.na(symbol_col)) {
    cat(
        "Symbol column: ",
        symbol_col,
        "\n",
        sep = ""
    )
} else {
    cat("Symbol column: not detected; enrichment construction can still continue.\n")
}

# --------------------------------------------------------------------------------------------------
# 10B.1E — STANDARDIZE DISCOVERY UNIVERSE
# --------------------------------------------------------------------------------------------------

discovery <- data.frame(
    ensembl_original = trimws(
        as.character(
            annotation[[ensembl_col]]
        )
    ),
    gene_type = trimws(
        as.character(
            annotation[[gene_type_col]]
        )
    ),
    stringsAsFactors = FALSE
)

if (!is.na(symbol_col)) {

    discovery$gene_symbol <- trimws(
        as.character(
            annotation[[symbol_col]]
        )
    )

} else {

    discovery$gene_symbol <- NA_character_
}

discovery$ensembl_stable <- sub(
    "\\.[0-9]+$",
    "",
    discovery$ensembl_original
)

invalid_ensembl <- (
    is.na(discovery$ensembl_stable) |
    discovery$ensembl_stable == "" |
    !grepl(
        "^ENSG[0-9]+$",
        discovery$ensembl_stable
    )
)

if (any(invalid_ensembl)) {

    stop(
        "Invalid/missing stable Ensembl IDs detected in final discovery annotation: ",
        sum(invalid_ensembl)
    )
}

if (anyDuplicated(discovery$ensembl_stable) > 0L) {

    dup <- unique(
        discovery$ensembl_stable[
            duplicated(discovery$ensembl_stable) |
            duplicated(discovery$ensembl_stable, fromLast = TRUE)
        ]
    )

    stop(
        "Final discovery annotation contains duplicate stable Ensembl IDs after version stripping.\n",
        "Number of duplicated IDs: ",
        length(dup)
    )
}

cat("Stable unique discovery Ensembl IDs: 23,740 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10B.1F — DEFINE PROTEIN-CODING DISCOVERY BACKGROUND
# --------------------------------------------------------------------------------------------------

discovery$protein_coding <- (
    tolower(discovery$gene_type) == "protein_coding"
)

n_pc_discovery <- sum(
    discovery$protein_coding
)

n_nonpc_discovery <- sum(
    !discovery$protein_coding
)

cat(
    "\nDiscovery protein-coding genes: ",
    n_pc_discovery,
    "\n",
    sep = ""
)

cat(
    "Discovery non-protein-coding genes: ",
    n_nonpc_discovery,
    "\n",
    sep = ""
)

if (n_pc_discovery < 10000L) {
    stop(
        "Unexpectedly small protein-coding discovery universe: ",
        n_pc_discovery
    )
}

# --------------------------------------------------------------------------------------------------
# 10B.1G — LOAD CURRENT HUMAN ANNOTATION
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("BiocManager", quietly = TRUE)) {

    install.packages(
        "BiocManager",
        repos = "https://cloud.r-project.org"
    )
}

needed <- c(
    "AnnotationDbi",
    "org.Hs.eg.db"
)

missing_packages <- needed[
    !vapply(
        needed,
        requireNamespace,
        quietly = TRUE,
        FUN.VALUE = logical(1)
    )
]

if (length(missing_packages) > 0L) {

    BiocManager::install(
        missing_packages,
        ask = FALSE,
        update = FALSE
    )
}

if (!requireNamespace("AnnotationDbi", quietly = TRUE)) {
    stop("AnnotationDbi unavailable.")
}

if (!requireNamespace("org.Hs.eg.db", quietly = TRUE)) {
    stop("org.Hs.eg.db unavailable.")
}

orgdb <- org.Hs.eg.db::org.Hs.eg.db

pc_discovery <- discovery[
    discovery$protein_coding,
    ,
    drop = FALSE
]

# --------------------------------------------------------------------------------------------------
# 10B.1H — MAP DISCOVERY PROTEIN-CODING GENES TO ENTREZ
# --------------------------------------------------------------------------------------------------

pc_map_raw <- AnnotationDbi::select(
    orgdb,
    keys = pc_discovery$ensembl_stable,
    keytype = "ENSEMBL",
    columns = c(
        "ENTREZID",
        "SYMBOL"
    )
)

names(pc_map_raw)[
    names(pc_map_raw) == "ENSEMBL"
] <- "ensembl_stable"

pc_map_raw <- pc_map_raw[
    !is.na(pc_map_raw$ENTREZID) &
    nzchar(as.character(pc_map_raw$ENTREZID)),
    ,
    drop = FALSE
]

# Keep only one-to-one Ensembl -> unique Entrez mappings for the formal background.
entrez_count <- aggregate(
    ENTREZID ~ ensembl_stable,
    data = pc_map_raw,
    FUN = function(x) {
        length(unique(x))
    }
)

names(entrez_count)[2] <- "n_unique_entrez"

pc_map_raw <- merge(
    pc_map_raw,
    entrez_count,
    by = "ensembl_stable",
    all.x = TRUE
)

pc_map_unique <- pc_map_raw[
    pc_map_raw$n_unique_entrez == 1L,
    ,
    drop = FALSE
]

pc_map_unique <- unique(
    pc_map_unique[
        ,
        c(
            "ensembl_stable",
            "ENTREZID",
            "SYMBOL"
        )
    ]
)

# If multiple rows remain only because SYMBOL differs, enforce unique Ensembl/Entrez pair.
pc_map_unique <- unique(
    pc_map_unique[
        ,
        c(
            "ensembl_stable",
            "ENTREZID"
        )
    ]
)

names(pc_map_unique)[
    names(pc_map_unique) == "ENTREZID"
] <- "entrez_id"

pc_background <- merge(
    pc_discovery,
    pc_map_unique,
    by = "ensembl_stable",
    all.x = FALSE,
    all.y = FALSE
)

pc_background <- pc_background[
    !duplicated(
        pc_background$ensembl_stable
    ),
    ,
    drop = FALSE
]

n_pc_entrez <- nrow(
    pc_background
)

cat(
    "Protein-coding discovery genes with unique Entrez mapping: ",
    n_pc_entrez,
    "/",
    n_pc_discovery,
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10B.1I — LOAD 12 FROZEN PROTEIN-CODING SCZ-MSS GENES
# --------------------------------------------------------------------------------------------------

sig <- read.csv(
    STEP10A_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(sig) != 24L) {
    stop("STEP10A signature table does not contain 24 genes.")
}

if (!"protein_coding_flag" %in% names(sig)) {
    stop("STEP10A table missing protein_coding_flag.")
}

if (!is.logical(sig$protein_coding_flag)) {

    tmp <- toupper(
        trimws(
            as.character(
                sig$protein_coding_flag
            )
        )
    )

    sig$protein_coding_flag <- tmp == "TRUE"
}

sig_pc <- sig[
    sig$protein_coding_flag,
    ,
    drop = FALSE
]

if (nrow(sig_pc) != 12L) {
    stop(
        "Expected 12 frozen protein-coding genes; observed ",
        nrow(sig_pc)
    )
}

signature_pc_map <- merge(
    sig_pc,
    pc_map_unique,
    by.x = "ensemblID_stable",
    by.y = "ensembl_stable",
    all.x = TRUE,
    sort = FALSE
)

signature_pc_map <- signature_pc_map[
    match(
        sig_pc$ensemblID_stable,
        signature_pc_map$ensemblID_stable
    ),
    ,
    drop = FALSE
]

n_signature_entrez <- sum(
    !is.na(
        signature_pc_map$entrez_id
    )
)

cat(
    "Frozen protein-coding genes with unique Entrez mapping: ",
    n_signature_entrez,
    "/12\n",
    sep = ""
)

if (n_signature_entrez != 12L) {

    missing_sig <- signature_pc_map[
        is.na(signature_pc_map$entrez_id),
        c(
            "ensemblID_stable",
            "gene_symbol"
        ),
        drop = FALSE
    ]

    print(missing_sig)

    stop(
        "Not all 12 protein-coding signature genes have unique Entrez mappings."
    )
}

# --------------------------------------------------------------------------------------------------
# 10B.1J — VERIFY SIGNATURE GENES BELONG TO DISCOVERY BACKGROUND
# --------------------------------------------------------------------------------------------------

in_background <- (
    signature_pc_map$ensemblID_stable %in%
    pc_background$ensembl_stable
)

if (!all(in_background)) {

    missing_background <- signature_pc_map[
        !in_background,
        c(
            "ensemblID_stable",
            "gene_symbol",
            "entrez_id"
        ),
        drop = FALSE
    ]

    print(missing_background)

    stop(
        "One or more frozen protein-coding genes are absent from the discovery-matched enrichment background."
    )
}

cat("All 12 frozen protein-coding genes represented in enrichment background: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10B.1K — WRITE FROZEN UNIVERSE FILES
# --------------------------------------------------------------------------------------------------

full_universe <- discovery[
    ,
    c(
        "ensembl_stable",
        "gene_symbol",
        "gene_type",
        "protein_coding"
    )
]

protein_coding_universe <- pc_discovery[
    ,
    c(
        "ensembl_stable",
        "gene_symbol",
        "gene_type"
    )
]

entrez_background <- pc_background[
    ,
    c(
        "ensembl_stable",
        "gene_symbol",
        "gene_type",
        "entrez_id"
    )
]

signature_enrichment_input <- signature_pc_map[
    ,
    c(
        "ensemblID_stable",
        "gene_symbol",
        "gene_type",
        "direction_standardized",
        "final_normalized_weight",
        "entrez_id"
    )
]

write.csv(
    full_universe,
    OUT_FULL_ENSEMBL,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    protein_coding_universe,
    OUT_PC_ENSEMBL,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    entrez_background,
    OUT_PC_ENTREZ,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    signature_enrichment_input,
    OUT_SIGNATURE_PC,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10B.1L — SUMMARY
# --------------------------------------------------------------------------------------------------

background_mapping_fraction <- (
    n_pc_entrez /
    n_pc_discovery
)

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10B.1 — DISCOVERY-MATCHED FUNCTIONAL-ENRICHMENT BACKGROUND",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    paste0(
        "Canonical discovery annotation: ",
        annotation_path
    ),

    "",

    paste0(
        "Final discovery genes: ",
        nrow(discovery)
    ),

    paste0(
        "Discovery protein-coding genes: ",
        n_pc_discovery
    ),

    paste0(
        "Discovery non-protein-coding genes: ",
        n_nonpc_discovery
    ),

    paste0(
        "Protein-coding genes with unique Entrez mapping: ",
        n_pc_entrez
    ),

    paste0(
        "Protein-coding Entrez mapping fraction: ",
        sprintf("%.6f", background_mapping_fraction)
    ),

    "",

    "FROZEN SIGNATURE ENRICHMENT INPUT",

    "Frozen protein-coding genes: 12",

    paste0(
        "Frozen protein-coding genes with unique Entrez mapping: ",
        n_signature_entrez
    ),

    "All 12 frozen protein-coding genes belong to the discovery-matched background: YES",

    "",

    "ENRICHMENT POLICY",

    paste(
        "Competitive pathway/ontology enrichment must use the uniquely Entrez-mapped",
        "protein-coding BrainSeq discovery universe as background."
    ),

    paste(
        "The entire human genome must NOT be used as the default background."
    ),

    paste(
        "The 12 non-protein-coding SCZ-MSS genes are outside conventional protein-coding",
        "GO/Reactome enrichment and require separate interpretation."
    ),

    "",

    "NEXT STEP",

    "Run conservative GO Biological Process and Reactome over-representation analysis for the 12 protein-coding frozen genes using this frozen discovery-matched Entrez universe.",

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10B.1M — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    annotation_path,
    STEP10A_PATH,
    OUT_FULL_ENSEMBL,
    OUT_PC_ENSEMBL,
    OUT_PC_ENTREZ,
    OUT_SIGNATURE_PC,
    OUT_SUMMARY
)

manifest <- data.frame(
    file = basename(manifest_paths),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(manifest_paths)$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10B.1N — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nENRICHMENT BACKGROUND SUMMARY\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Final discovery universe: ",
    nrow(discovery),
    "\n",
    sep = ""
)

cat(
    "Protein-coding discovery genes: ",
    n_pc_discovery,
    "\n",
    sep = ""
)

cat(
    "Non-protein-coding discovery genes: ",
    n_nonpc_discovery,
    "\n",
    sep = ""
)

cat(
    "Unique Entrez-mapped protein-coding background: ",
    n_pc_entrez,
    "\n",
    sep = ""
)

cat(
    "Background Entrez mapping fraction: ",
    sprintf("%.4f", background_mapping_fraction),
    "\n",
    sep = ""
)

cat(
    "Frozen protein-coding enrichment genes: ",
    n_signature_entrez,
    "/12\n",
    sep = ""
)

cat(
    "All frozen protein-coding genes present in background: PASS\n"
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\n12-GENE ENRICHMENT INPUT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    signature_enrichment_input[
        ,
        c(
            "ensemblID_stable",
            "gene_symbol",
            "direction_standardized",
            "final_normalized_weight",
            "entrez_id"
        )
    ],
    row.names = FALSE
)

cat(
    "\nFrozen Entrez background:\n",
    OUT_PC_ENTREZ,
    "\n",
    sep = ""
)

cat(
    "\nFrozen 12-gene enrichment input:\n",
    OUT_SIGNATURE_PC,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10B.1 COMPLETE — ENRICHMENT BACKGROUND FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10B.1 — DISCOVERY-MATCHED ENRICHMENT BACKGROUND UNIVERSE FREEZE
Project sentinel: PASS
STEP10A frozen biological annotation: PASS
Selected discovery annotation file:
E:/SCZ_CROSSREGION_MSS_Q1/04_processed/brainseq_primary_gene_annotation_v1.csv
Annotation rows: 23740
Final discovery gene count: 23,740 — PASS
Ensembl column: ensemblID_original
Gene-type column: gene_type
Symbol column: gene_symbol
Stable unique discovery Ensembl IDs: 23,740 — PASS

Discovery protein-coding genes: 14091
Discovery non-protein-coding genes: 9649


'select()' returned 1:many mapping between keys and columns



Protein-coding discovery genes with unique Entrez mapping: 13838/14091
Frozen protein-coding genes with unique Entrez mapping: 12/12
All 12 frozen protein-coding genes represented in enrichment background: PASS

--------------------------------------------------------------------------------------------------------------
ENRICHMENT BACKGROUND SUMMARY
--------------------------------------------------------------------------------------------------------------
Final discovery universe: 23740
Protein-coding discovery genes: 14091
Non-protein-coding discovery genes: 9649
Unique Entrez-mapped protein-coding background: 13838
Background Entrez mapping fraction: 0.9820
Frozen protein-coding enrichment genes: 12/12
All frozen protein-coding genes present in background: PASS

--------------------------------------------------------------------------------------------------------------
12-GENE ENRICHMENT INPUT
-------------------------------------------------------------------------------------

In [63]:
# ==================================================================================================
# STEP 10B.2 — CONSERVATIVE GO-BP + REACTOME OVER-REPRESENTATION ANALYSIS
# CORRECTED FULL CELL
# --------------------------------------------------------------------------------------------------
# CORRECTION
#   STEP10B.1 produced 13,838 protein-coding discovery Ensembl genes with a unique
#   Ensembl->Entrez mapping at the per-gene level.
#
#   However, more than one Ensembl gene can still map to the SAME Entrez ID.
#   Therefore:
#       background ROWS (Ensembl genes) may be 13,838
#       distinct Entrez universe may be smaller (observed previously: 13,837)
#
#   ORA universe must use DISTINCT Entrez IDs.
#
# PURPOSE
#   A. GO Biological Process ORA
#   B. Reactome ORA
#
# QUERY
#   Frozen 12 protein-coding SCZ-MSS genes
#
# BACKGROUND
#   Distinct Entrez IDs represented by the frozen discovery-matched
#   protein-coding background from STEP10B.1.
#
# INTERPRETATION
#   BH FDR < 0.05       = statistically supported enrichment
#   nominal p < 0.05    = exploratory only if FDR >= 0.05
#
# IMPORTANT
#   - No pathway rescue.
#   - No whole-genome background.
#   - No direction-specific ORA in this step.
#   - Null enrichment is retained if observed.
#   - These analyses cover only the 12 protein-coding genes, not the full 24-gene state.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10B.2 — CONSERVATIVE GO-BP + REACTOME OVER-REPRESENTATION ANALYSIS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10B.2A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

BACKGROUND_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1",
    "STEP10B1_discovery_protein_coding_Entrez_background_v1.csv"
)

SIGNATURE_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1",
    "STEP10B1_SCZ_MSS_12_protein_coding_enrichment_input_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1",
    "ora_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

DUP_ENTREZ_AUDIT_PATH <- file.path(
    OUT_DIR,
    "STEP10B2_background_duplicate_Entrez_audit_v1.csv"
)

GO_ALL_PATH <- file.path(
    OUT_DIR,
    "STEP10B2_GO_BP_all_tested_terms_v1.csv"
)

GO_FDR_PATH <- file.path(
    OUT_DIR,
    "STEP10B2_GO_BP_FDR05_terms_v1.csv"
)

GO_NOMINAL_PATH <- file.path(
    OUT_DIR,
    "STEP10B2_GO_BP_nominalP05_terms_v1.csv"
)

REACTOME_ALL_PATH <- file.path(
    OUT_DIR,
    "STEP10B2_Reactome_all_tested_pathways_v1.csv"
)

REACTOME_FDR_PATH <- file.path(
    OUT_DIR,
    "STEP10B2_Reactome_FDR05_pathways_v1.csv"
)

REACTOME_NOMINAL_PATH <- file.path(
    OUT_DIR,
    "STEP10B2_Reactome_nominalP05_pathways_v1.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "STEP10B2_GO_BP_Reactome_ORA_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "STEP10B2_GO_BP_Reactome_ORA_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10B.2B — PROJECT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

required_inputs <- c(
    BACKGROUND_PATH,
    SIGNATURE_PATH
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {
    stop(
        "Missing STEP10B.1 input(s):\n",
        paste(missing_inputs, collapse = "\n")
    )
}

cat("STEP10B.1 frozen background: PASS\n")
cat("STEP10B.1 frozen 12-gene input: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10B.2C — LOAD BACKGROUND + SIGNATURE
# --------------------------------------------------------------------------------------------------

background_df <- read.csv(
    BACKGROUND_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

signature_df <- read.csv(
    SIGNATURE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

required_background_cols <- c(
    "ensembl_stable",
    "entrez_id"
)

required_signature_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "direction_standardized",
    "final_normalized_weight",
    "entrez_id"
)

missing_bg_cols <- setdiff(
    required_background_cols,
    names(background_df)
)

missing_sig_cols <- setdiff(
    required_signature_cols,
    names(signature_df)
)

if (length(missing_bg_cols) > 0L) {
    stop(
        "Background file missing expected column(s): ",
        paste(missing_bg_cols, collapse = ", ")
    )
}

if (length(missing_sig_cols) > 0L) {
    stop(
        "Signature file missing expected column(s): ",
        paste(missing_sig_cols, collapse = ", ")
    )
}

# STEP10B.1 established 13,838 Ensembl rows.
if (nrow(background_df) != 13838L) {
    stop(
        "STEP10B.1 background row count changed.\n",
        "Expected 13,838 protein-coding Ensembl genes; observed ",
        nrow(background_df)
    )
}

if (nrow(signature_df) != 12L) {
    stop(
        "Frozen coding-signature row count changed.\n",
        "Expected 12; observed ",
        nrow(signature_df)
    )
}

cat(
    "Protein-coding background Ensembl rows: ",
    nrow(background_df),
    " — PASS\n",
    sep = ""
)

cat(
    "Frozen coding signature rows: ",
    nrow(signature_df),
    " — PASS\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10B.2D — CLEAN ENTREZ IDs
# --------------------------------------------------------------------------------------------------

background_df$entrez_id <- trimws(
    as.character(
        background_df$entrez_id
    )
)

signature_df$entrez_id <- trimws(
    as.character(
        signature_df$entrez_id
    )
)

bad_background_entrez <- (
    is.na(background_df$entrez_id) |
    !nzchar(background_df$entrez_id)
)

bad_signature_entrez <- (
    is.na(signature_df$entrez_id) |
    !nzchar(signature_df$entrez_id)
)

if (any(bad_background_entrez)) {
    stop(
        "Missing Entrez IDs detected in frozen STEP10B.1 background: ",
        sum(bad_background_entrez)
    )
}

if (any(bad_signature_entrez)) {
    stop(
        "Missing Entrez IDs detected in frozen 12-gene query: ",
        sum(bad_signature_entrez)
    )
}

# --------------------------------------------------------------------------------------------------
# 10B.2E — AUDIT SHARED ENTREZ IDs ACROSS MULTIPLE ENSEMBL GENES
# --------------------------------------------------------------------------------------------------

entrez_frequency <- table(
    background_df$entrez_id
)

duplicated_entrez_ids <- names(
    entrez_frequency[
        entrez_frequency > 1L
    ]
)

if (length(duplicated_entrez_ids) > 0L) {

    duplicate_entrez_audit <- background_df[
        background_df$entrez_id %in%
            duplicated_entrez_ids,
        ,
        drop = FALSE
    ]

    duplicate_entrez_audit <- duplicate_entrez_audit[
        order(
            duplicate_entrez_audit$entrez_id,
            duplicate_entrez_audit$ensembl_stable
        ),
        ,
        drop = FALSE
    ]

} else {

    duplicate_entrez_audit <- background_df[
        FALSE,
        ,
        drop = FALSE
    ]
}

write.csv(
    duplicate_entrez_audit,
    DUP_ENTREZ_AUDIT_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

cat(
    "\nBackground Entrez IDs represented by >1 Ensembl gene: ",
    length(duplicated_entrez_ids),
    "\n",
    sep = ""
)

if (length(duplicated_entrez_ids) > 0L) {

    cat(
        "Affected background Ensembl rows: ",
        nrow(duplicate_entrez_audit),
        "\n",
        sep = ""
    )

    cat("\nDUPLICATE ENTREZ AUDIT\n")

    print(
        duplicate_entrez_audit[
            ,
            intersect(
                c(
                    "ensembl_stable",
                    "gene_symbol",
                    "gene_type",
                    "entrez_id"
                ),
                names(duplicate_entrez_audit)
            ),
            drop = FALSE
        ],
        row.names = FALSE
    )
}

# --------------------------------------------------------------------------------------------------
# 10B.2F — DEFINE DISTINCT ENTREZ UNIVERSE
#
# IMPORTANT:
# enrichGO/enrichPathway operate on identifier sets.
# Duplicate Entrez IDs cannot represent independent universe elements.
# --------------------------------------------------------------------------------------------------

background_entrez <- sort(
    unique(
        background_df$entrez_id
    )
)

signature_entrez <- sort(
    unique(
        signature_df$entrez_id
    )
)

n_background_ensembl <- nrow(
    background_df
)

n_background_entrez <- length(
    background_entrez
)

n_signature_ensembl <- nrow(
    signature_df
)

n_signature_entrez <- length(
    signature_entrez
)

cat(
    "\nDiscovery protein-coding Ensembl background: ",
    n_background_ensembl,
    "\n",
    sep = ""
)

cat(
    "Distinct Entrez enrichment universe: ",
    n_background_entrez,
    "\n",
    sep = ""
)

cat(
    "Frozen coding query Ensembl genes: ",
    n_signature_ensembl,
    "\n",
    sep = ""
)

cat(
    "Distinct frozen query Entrez IDs: ",
    n_signature_entrez,
    "\n",
    sep = ""
)

if (n_signature_entrez != 12L) {
    stop(
        "The 12 frozen protein-coding genes do not correspond to 12 distinct Entrez IDs.\n",
        "Observed distinct query IDs: ",
        n_signature_entrez
    )
}

if (!all(signature_entrez %in% background_entrez)) {

    missing_query <- setdiff(
        signature_entrez,
        background_entrez
    )

    stop(
        "Frozen query Entrez IDs absent from frozen background: ",
        paste(missing_query, collapse = ", ")
    )
}

cat("All 12 frozen query Entrez IDs present in background: PASS\n")

# Check whether any background duplicate Entrez affects the frozen query.
query_intersects_duplicate_entrez <- intersect(
    signature_entrez,
    duplicated_entrez_ids
)

cat(
    "Frozen query IDs involved in background Entrez duplication: ",
    length(query_intersects_duplicate_entrez),
    "\n",
    sep = ""
)

if (length(query_intersects_duplicate_entrez) > 0L) {

    cat(
        "Affected query Entrez IDs: ",
        paste(
            query_intersects_duplicate_entrez,
            collapse = ", "
        ),
        "\n",
        sep = ""
    )
}

# --------------------------------------------------------------------------------------------------
# 10B.2G — INSTALL / LOAD PACKAGES
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("BiocManager", quietly = TRUE)) {

    install.packages(
        "BiocManager",
        repos = "https://cloud.r-project.org"
    )
}

needed_packages <- c(
    "clusterProfiler",
    "ReactomePA",
    "org.Hs.eg.db"
)

missing_packages <- needed_packages[
    !vapply(
        needed_packages,
        requireNamespace,
        quietly = TRUE,
        FUN.VALUE = logical(1)
    )
]

if (length(missing_packages) > 0L) {

    cat(
        "\nInstalling missing Bioconductor package(s): ",
        paste(
            missing_packages,
            collapse = ", "
        ),
        "\n",
        sep = ""
    )

    BiocManager::install(
        missing_packages,
        ask = FALSE,
        update = FALSE
    )
}

if (!requireNamespace("clusterProfiler", quietly = TRUE)) {
    stop("clusterProfiler unavailable.")
}

if (!requireNamespace("ReactomePA", quietly = TRUE)) {
    stop("ReactomePA unavailable.")
}

if (!requireNamespace("org.Hs.eg.db", quietly = TRUE)) {
    stop("org.Hs.eg.db unavailable.")
}

cat(
    "\nPackage versions:\n",
    "  clusterProfiler: ",
    as.character(
        utils::packageVersion(
            "clusterProfiler"
        )
    ),
    "\n",
    "  ReactomePA: ",
    as.character(
        utils::packageVersion(
            "ReactomePA"
        )
    ),
    "\n",
    "  org.Hs.eg.db: ",
    as.character(
        utils::packageVersion(
            "org.Hs.eg.db"
        )
    ),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10B.2H — GO BIOLOGICAL PROCESS ORA
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nGO BIOLOGICAL PROCESS OVER-REPRESENTATION\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

go_result <- clusterProfiler::enrichGO(
    gene = signature_entrez,
    universe = background_entrez,
    OrgDb = org.Hs.eg.db::org.Hs.eg.db,
    keyType = "ENTREZID",
    ont = "BP",
    pvalueCutoff = 1,
    pAdjustMethod = "BH",
    qvalueCutoff = 1,
    minGSSize = 10,
    maxGSSize = 500,
    readable = TRUE
)

go_df <- as.data.frame(
    go_result
)

if (nrow(go_df) > 0L) {

    go_df <- go_df[
        order(
            go_df$p.adjust,
            go_df$pvalue,
            na.last = TRUE
        ),
        ,
        drop = FALSE
    ]

    go_df$FDR05 <- (
        !is.na(go_df$p.adjust) &
        go_df$p.adjust < 0.05
    )

    go_df$nominalP05 <- (
        !is.na(go_df$pvalue) &
        go_df$pvalue < 0.05
    )

    go_df$nominal_only <- (
        go_df$nominalP05 &
        !go_df$FDR05
    )

} else {

    go_df$FDR05 <- logical(0)
    go_df$nominalP05 <- logical(0)
    go_df$nominal_only <- logical(0)
}

go_fdr <- go_df[
    go_df$FDR05,
    ,
    drop = FALSE
]

go_nominal <- go_df[
    go_df$nominalP05,
    ,
    drop = FALSE
]

write.csv(
    go_df,
    GO_ALL_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    go_fdr,
    GO_FDR_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    go_nominal,
    GO_NOMINAL_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

cat(
    "GO-BP returned terms: ",
    nrow(go_df),
    "\n",
    sep = ""
)

cat(
    "GO-BP FDR < 0.05: ",
    nrow(go_fdr),
    "\n",
    sep = ""
)

cat(
    "GO-BP nominal p < 0.05: ",
    nrow(go_nominal),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10B.2I — REACTOME ORA
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nREACTOME OVER-REPRESENTATION\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

reactome_result <- ReactomePA::enrichPathway(
    gene = signature_entrez,
    organism = "human",
    universe = background_entrez,
    pvalueCutoff = 1,
    pAdjustMethod = "BH",
    qvalueCutoff = 1,
    minGSSize = 10,
    maxGSSize = 500,
    readable = FALSE
)

reactome_df <- as.data.frame(
    reactome_result
)

if (nrow(reactome_df) > 0L) {

    reactome_df <- reactome_df[
        order(
            reactome_df$p.adjust,
            reactome_df$pvalue,
            na.last = TRUE
        ),
        ,
        drop = FALSE
    ]

    reactome_df$FDR05 <- (
        !is.na(reactome_df$p.adjust) &
        reactome_df$p.adjust < 0.05
    )

    reactome_df$nominalP05 <- (
        !is.na(reactome_df$pvalue) &
        reactome_df$pvalue < 0.05
    )

    reactome_df$nominal_only <- (
        reactome_df$nominalP05 &
        !reactome_df$FDR05
    )

} else {

    reactome_df$FDR05 <- logical(0)
    reactome_df$nominalP05 <- logical(0)
    reactome_df$nominal_only <- logical(0)
}

reactome_fdr <- reactome_df[
    reactome_df$FDR05,
    ,
    drop = FALSE
]

reactome_nominal <- reactome_df[
    reactome_df$nominalP05,
    ,
    drop = FALSE
]

write.csv(
    reactome_df,
    REACTOME_ALL_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    reactome_fdr,
    REACTOME_FDR_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    reactome_nominal,
    REACTOME_NOMINAL_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

cat(
    "Reactome returned pathways: ",
    nrow(reactome_df),
    "\n",
    sep = ""
)

cat(
    "Reactome FDR < 0.05: ",
    nrow(reactome_fdr),
    "\n",
    sep = ""
)

cat(
    "Reactome nominal p < 0.05: ",
    nrow(reactome_nominal),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10B.2J — OVERALL INTERPRETATION STATUS
# --------------------------------------------------------------------------------------------------

if (
    nrow(go_fdr) > 0L ||
    nrow(reactome_fdr) > 0L
) {

    enrichment_status <- "FDR_SUPPORTED_ENRICHMENT_PRESENT"

    enrichment_interpretation <- paste(
        "At least one conventional protein-coding functional category",
        "survived BH FDR < 0.05.",
        "Interpretation applies only to the 12 protein-coding genes."
    )

} else if (
    nrow(go_nominal) > 0L ||
    nrow(reactome_nominal) > 0L
) {

    enrichment_status <- "NO_FDR_ENRICHMENT_NOMINAL_SIGNALS_ONLY"

    enrichment_interpretation <- paste(
        "No GO-BP or Reactome term survived BH FDR < 0.05.",
        "Nominal p < 0.05 terms are exploratory only."
    )

} else {

    enrichment_status <- "NO_CONVENTIONAL_ENRICHMENT_SIGNAL"

    enrichment_interpretation <- paste(
        "No conventional GO-BP or Reactome category reached nominal p < 0.05.",
        "This is retained as a null enrichment result."
    )
}

# --------------------------------------------------------------------------------------------------
# 10B.2K — TOP-RESULT FORMATTER
# --------------------------------------------------------------------------------------------------

top_terms_text <- function(df, n = 10L) {

    if (nrow(df) == 0L) {
        return("[NONE]")
    }

    n_use <- min(
        n,
        nrow(df)
    )

    x <- df[
        seq_len(n_use),
        ,
        drop = FALSE
    ]

    paste0(
        seq_len(nrow(x)),
        ". ",
        x$Description,
        " | Count=",
        x$Count,
        " | p=",
        format(
            x$pvalue,
            scientific = TRUE,
            digits = 4
        ),
        " | BH-FDR=",
        format(
            x$p.adjust,
            scientific = TRUE,
            digits = 4
        )
    )
}

go_top_lines <- top_terms_text(
    go_df,
    10L
)

reactome_top_lines <- top_terms_text(
    reactome_df,
    10L
)

# --------------------------------------------------------------------------------------------------
# 10B.2L — SUMMARY FILE
# --------------------------------------------------------------------------------------------------

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10B.2 — SCZ-MSS GO-BP + REACTOME ORA",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "BACKGROUND AUDIT",

    paste0(
        "Protein-coding discovery Ensembl genes with unique per-gene Entrez mapping: ",
        n_background_ensembl
    ),

    paste0(
        "Distinct Entrez IDs in enrichment universe: ",
        n_background_entrez
    ),

    paste0(
        "Entrez IDs represented by >1 discovery Ensembl gene: ",
        length(duplicated_entrez_ids)
    ),

    paste0(
        "Frozen coding query genes: ",
        n_signature_ensembl
    ),

    paste0(
        "Distinct frozen query Entrez IDs: ",
        n_signature_entrez
    ),

    paste0(
        "Frozen query IDs involved in duplicated background Entrez IDs: ",
        length(query_intersects_duplicate_entrez)
    ),

    "",

    "GO BIOLOGICAL PROCESS",

    paste0(
        "Returned terms: ",
        nrow(go_df)
    ),

    paste0(
        "FDR < 0.05: ",
        nrow(go_fdr)
    ),

    paste0(
        "Nominal p < 0.05: ",
        nrow(go_nominal)
    ),

    "",

    "Top GO-BP results:",

    go_top_lines,

    "",

    "REACTOME",

    paste0(
        "Returned pathways: ",
        nrow(reactome_df)
    ),

    paste0(
        "FDR < 0.05: ",
        nrow(reactome_fdr)
    ),

    paste0(
        "Nominal p < 0.05: ",
        nrow(reactome_nominal)
    ),

    "",

    "Top Reactome results:",

    reactome_top_lines,

    "",

    paste0(
        "OVERALL STATUS: ",
        enrichment_status
    ),

    enrichment_interpretation,

    "",

    "IMPORTANT SCOPE LIMITATION",

    paste(
        "These ORA results describe only the 12 protein-coding members",
        "of the frozen 24-gene SCZ-MSS state."
    ),

    paste(
        "The 12 non-protein-coding genes require separate biological interpretation."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10B.2M — MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    BACKGROUND_PATH,
    SIGNATURE_PATH,
    DUP_ENTREZ_AUDIT_PATH,
    GO_ALL_PATH,
    GO_FDR_PATH,
    GO_NOMINAL_PATH,
    REACTOME_ALL_PATH,
    REACTOME_FDR_PATH,
    REACTOME_NOMINAL_PATH,
    SUMMARY_PATH
)

manifest <- data.frame(
    file = basename(manifest_paths),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(manifest_paths)$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10B.2N — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nBACKGROUND AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Protein-coding discovery Ensembl rows: ",
    n_background_ensembl,
    "\n",
    sep = ""
)

cat(
    "Distinct Entrez enrichment universe: ",
    n_background_entrez,
    "\n",
    sep = ""
)

cat(
    "Duplicated Entrez IDs across discovery Ensembl genes: ",
    length(duplicated_entrez_ids),
    "\n",
    sep = ""
)

cat(
    "Frozen query genes / distinct Entrez IDs: ",
    n_signature_ensembl,
    "/",
    n_signature_entrez,
    "\n",
    sep = ""
)

cat(
    "Frozen query IDs affected by duplicated background Entrez mapping: ",
    length(query_intersects_duplicate_entrez),
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nGO-BP RESULTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "GO-BP returned terms: ",
    nrow(go_df),
    "\n",
    sep = ""
)

cat(
    "GO-BP FDR < 0.05: ",
    nrow(go_fdr),
    "\n",
    sep = ""
)

cat(
    "GO-BP nominal p < 0.05: ",
    nrow(go_nominal),
    "\n",
    sep = ""
)

if (nrow(go_df) > 0L) {

    cat("\nTOP GO-BP TERMS\n")

    print(
        head(
            go_df[
                ,
                intersect(
                    c(
                        "ID",
                        "Description",
                        "GeneRatio",
                        "BgRatio",
                        "pvalue",
                        "p.adjust",
                        "qvalue",
                        "geneID",
                        "Count",
                        "FDR05",
                        "nominalP05"
                    ),
                    names(go_df)
                ),
                drop = FALSE
            ],
            15L
        ),
        row.names = FALSE
    )
}

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nREACTOME RESULTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Reactome returned pathways: ",
    nrow(reactome_df),
    "\n",
    sep = ""
)

cat(
    "Reactome FDR < 0.05: ",
    nrow(reactome_fdr),
    "\n",
    sep = ""
)

cat(
    "Reactome nominal p < 0.05: ",
    nrow(reactome_nominal),
    "\n",
    sep = ""
)

if (nrow(reactome_df) > 0L) {

    cat("\nTOP REACTOME PATHWAYS\n")

    print(
        head(
            reactome_df[
                ,
                intersect(
                    c(
                        "ID",
                        "Description",
                        "GeneRatio",
                        "BgRatio",
                        "pvalue",
                        "p.adjust",
                        "qvalue",
                        "geneID",
                        "Count",
                        "FDR05",
                        "nominalP05"
                    ),
                    names(reactome_df)
                ),
                drop = FALSE
            ],
            15L
        ),
        row.names = FALSE
    )
}

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nOVERALL FUNCTIONAL-ENRICHMENT STATUS\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    enrichment_status,
    "\n"
)

cat(
    enrichment_interpretation,
    "\n"
)

cat(
    "\nDuplicate Entrez audit:\n",
    DUP_ENTREZ_AUDIT_PATH,
    "\n",
    sep = ""
)

cat(
    "\nGO-BP full results:\n",
    GO_ALL_PATH,
    "\n",
    sep = ""
)

cat(
    "\nReactome full results:\n",
    REACTOME_ALL_PATH,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    SUMMARY_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10B.2 COMPLETE — GO-BP + REACTOME ORA\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10B.2 — CONSERVATIVE GO-BP + REACTOME OVER-REPRESENTATION ANALYSIS
Project sentinel: PASS
STEP10B.1 frozen background: PASS
STEP10B.1 frozen 12-gene input: PASS
Protein-coding background Ensembl rows: 13838 — PASS
Frozen coding signature rows: 12 — PASS

Background Entrez IDs represented by >1 Ensembl gene: 1
Affected background Ensembl rows: 2

DUPLICATE ENTREZ AUDIT
  ensembl_stable gene_symbol      gene_type entrez_id
 ENSG00000196081     ZNF724P protein_coding    440519
 ENSG00000283201             protein_coding    440519

Discovery protein-coding Ensembl background: 13838
Distinct Entrez enrichment universe: 13837
Frozen coding query Ensembl genes: 12
Distinct frozen query Entrez IDs: 12
All 12 frozen query Entrez IDs present in background: PASS
Frozen query IDs involved in background Entrez duplication: 0

Installing missing Bioconductor package(s): clusterProfiler, ReactomePA


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31 ucrt)

Installing package(s) 'clusterProfiler', 'ReactomePA'

also installing the dependencies 'sass', 'fontBitstreamVera', 'fontLiberation', 'evaluate', 'highr', 'xfun', 'bslib', 'fontawesome', 'jquerylib', 'tinytex', 'formatR', 'fontquiver', 'knitr', 'rmarkdown', 'yaml', 'lambda.r', 'futile.options', 'gridGraphics', 'lazyeval', 'gdtools', 'htmltools', 'htmlwidgets', 'base64enc', 'futile.logger', 'snow', 'BH', 'data.table', 'cowplot', 'fastmatch', 'utf8', 'ggplotify', 'patchwork', 'ape', 'tidytree', 'treeio', 'ggiraph', 'R.oo', 'R.methodsS3', 'stringi', 'farver', 'labeling', 'viridisLite', 'tweenr', 'polyclip', 'systemfonts', 'gridExtra', 'RcppArmadillo', 'filelock', 'BiocParallel', 'fgsea', 'reshape2', 'pillar', 'tibble', 't

package 'sass' successfully unpacked and MD5 sums checked
package 'fontBitstreamVera' successfully unpacked and MD5 sums checked
package 'fontLiberation' successfully unpacked and MD5 sums checked
package 'evaluate' successfully unpacked and MD5 sums checked
package 'highr' successfully unpacked and MD5 sums checked
package 'xfun' successfully unpacked and MD5 sums checked
package 'bslib' successfully unpacked and MD5 sums checked
package 'fontawesome' successfully unpacked and MD5 sums checked
package 'jquerylib' successfully unpacked and MD5 sums checked
package 'tinytex' successfully unpacked and MD5 sums checked
package 'formatR' successfully unpacked and MD5 sums checked
package 'fontquiver' successfully unpacked and MD5 sums checked
package 'knitr' successfully unpacked and MD5 sums checked
package 'rmarkdown' successfully unpacked and MD5 sums checked
package 'yaml' successfully unpacked and MD5 sums checked
package 'lambda.r' successfully unpacked and MD5 sums checked
package '

installing the source packages 'GO.db', 'reactome.db'







Package versions:
  clusterProfiler: 4.18.4
  ReactomePA: 1.54.0
  org.Hs.eg.db: 3.22.0

--------------------------------------------------------------------------------------------------------------
GO BIOLOGICAL PROCESS OVER-REPRESENTATION
--------------------------------------------------------------------------------------------------------------
GO-BP returned terms: 282
GO-BP FDR < 0.05: 0
GO-BP nominal p < 0.05: 101

--------------------------------------------------------------------------------------------------------------
REACTOME OVER-REPRESENTATION
--------------------------------------------------------------------------------------------------------------


Loading required package: org.Hs.eg.db

Loading required package: AnnotationDbi

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics

Warning message:
"package 'generics' was built under R version 4.5.3"

Attaching package: 'generics'


The following objects are masked from 'package:base':

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: 'BiocGenerics'


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


L

Reactome returned pathways: 50
Reactome FDR < 0.05: 0
Reactome nominal p < 0.05: 19

BACKGROUND AUDIT
Protein-coding discovery Ensembl rows: 13838
Distinct Entrez enrichment universe: 13837
Duplicated Entrez IDs across discovery Ensembl genes: 1
Frozen query genes / distinct Entrez IDs: 12/12
Frozen query IDs affected by duplicated background Entrez mapping: 0

GO-BP RESULTS
GO-BP returned terms: 282
GO-BP FDR < 0.05: 0
GO-BP nominal p < 0.05: 101

TOP GO-BP TERMS
         ID                                            Description GeneRatio
 GO:0071526                    semaphorin-plexin signaling pathway      2/11
 GO:0046348                          amino sugar catabolic process      1/11
 GO:0110095                    cellular detoxification of aldehyde      1/11
 GO:0007626                                    locomotory behavior      2/11
 GO:0006361  transcription initiation at RNA polymerase I promoter      1/11
 GO:0015671                                       oxygen transport   

In [64]:
# ==================================================================================================
# STEP 10C.1 — FUNCTIONAL-DATABASE COVERAGE + NOMINAL-SIGNAL DRIVER AUDIT
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   1. Determine exactly which of the 12 frozen protein-coding SCZ-MSS genes were represented
#      in the GO-BP and Reactome ORA result space used in STEP10B.2.
#   2. Explain the observed GeneRatio denominators (GO ~11; Reactome ~9).
#   3. Quantify whether nominal signals are single-gene or multi-gene driven.
#   4. Produce a gene-level ORA-driver table.
#   5. Produce a complete 24-gene biological-interpretation scaffold that keeps the
#      non-protein-coding half explicitly separate from conventional ORA.
#
# IMPORTANT
#   - NO new enrichment is performed.
#   - NO pathway rescue.
#   - NO nominal pathway is promoted to "significant".
#   - FDR < 0.05 remains the inferential threshold.
#   - Non-coding genes are not treated as "missing"; conventional GO/Reactome ORA is
#     simply not the appropriate analysis object for them.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10C.1 — FUNCTIONAL-DATABASE COVERAGE + NOMINAL-SIGNAL DRIVER AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10C.1A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

FULL24_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "annotation_v1",
    "STEP10A_SCZ_MSS_24gene_biological_annotation_v1.csv"
)

CODING12_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1",
    "STEP10B1_SCZ_MSS_12_protein_coding_enrichment_input_v1.csv"
)

GO_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1",
    "ora_v1",
    "STEP10B2_GO_BP_all_tested_terms_v1.csv"
)

REACTOME_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1",
    "ora_v1",
    "STEP10B2_Reactome_all_tested_pathways_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "gene_level_interpretation_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_CODING_AUDIT <- file.path(
    OUT_DIR,
    "STEP10C1_12coding_functional_database_driver_audit_v1.csv"
)

OUT_FULL24 <- file.path(
    OUT_DIR,
    "STEP10C1_24gene_biological_interpretation_scaffold_v1.csv"
)

OUT_GO_NOMINAL_DRIVER <- file.path(
    OUT_DIR,
    "STEP10C1_GO_BP_nominal_driver_terms_v1.csv"
)

OUT_REACTOME_NOMINAL_DRIVER <- file.path(
    OUT_DIR,
    "STEP10C1_Reactome_nominal_driver_pathways_v1.csv"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP10C1_functional_database_coverage_summary_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP10C1_functional_database_coverage_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10C.1B — VERIFY PROJECT + INPUTS
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

required_files <- c(
    FULL24_PATH,
    CODING12_PATH,
    GO_PATH,
    REACTOME_PATH
)

missing_files <- required_files[
    !file.exists(required_files)
]

if (length(missing_files) > 0L) {
    stop(
        "Required upstream file(s) missing:\n",
        paste(missing_files, collapse = "\n")
    )
}

cat("STEP10A full 24-gene table: PASS\n")
cat("STEP10B.1 coding 12-gene table: PASS\n")
cat("STEP10B.2 GO-BP results: PASS\n")
cat("STEP10B.2 Reactome results: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10C.1C — LOAD DATA
# --------------------------------------------------------------------------------------------------

full24 <- read.csv(
    FULL24_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

coding12 <- read.csv(
    CODING12_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

go <- read.csv(
    GO_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

reactome <- read.csv(
    REACTOME_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(full24) != 24L) {
    stop(
        "Expected 24 genes in STEP10A table; observed ",
        nrow(full24)
    )
}

if (nrow(coding12) != 12L) {
    stop(
        "Expected 12 protein-coding genes; observed ",
        nrow(coding12)
    )
}

required_coding_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "direction_standardized",
    "final_normalized_weight",
    "entrez_id"
)

missing_coding_cols <- setdiff(
    required_coding_cols,
    names(coding12)
)

if (length(missing_coding_cols) > 0L) {
    stop(
        "Coding input missing column(s): ",
        paste(missing_coding_cols, collapse = ", ")
    )
}

required_ora_cols <- c(
    "Description",
    "GeneRatio",
    "pvalue",
    "p.adjust",
    "geneID",
    "Count"
)

if (length(setdiff(required_ora_cols, names(go))) > 0L) {
    stop(
        "GO result missing expected columns: ",
        paste(
            setdiff(required_ora_cols, names(go)),
            collapse = ", "
        )
    )
}

if (length(setdiff(required_ora_cols, names(reactome))) > 0L) {
    stop(
        "Reactome result missing expected columns: ",
        paste(
            setdiff(required_ora_cols, names(reactome)),
            collapse = ", "
        )
    )
}

coding12$gene_symbol <- trimws(
    as.character(coding12$gene_symbol)
)

coding12$entrez_id <- trimws(
    as.character(coding12$entrez_id)
)

# --------------------------------------------------------------------------------------------------
# 10C.1D — VERIFY STEP10B.2 INFERENTIAL RESULT
# --------------------------------------------------------------------------------------------------

n_go_fdr <- sum(
    !is.na(go$p.adjust) &
    go$p.adjust < 0.05
)

n_reactome_fdr <- sum(
    !is.na(reactome$p.adjust) &
    reactome$p.adjust < 0.05
)

n_go_nominal <- sum(
    !is.na(go$pvalue) &
    go$pvalue < 0.05
)

n_reactome_nominal <- sum(
    !is.na(reactome$pvalue) &
    reactome$pvalue < 0.05
)

if (n_go_fdr != 0L) {
    stop(
        "GO-BP FDR result changed unexpectedly; expected zero FDR<0.05 terms."
    )
}

if (n_reactome_fdr != 0L) {
    stop(
        "Reactome FDR result changed unexpectedly; expected zero FDR<0.05 pathways."
    )
}

cat("GO-BP FDR < 0.05 terms: 0 — VERIFIED\n")
cat("Reactome FDR < 0.05 pathways: 0 — VERIFIED\n")

# --------------------------------------------------------------------------------------------------
# 10C.1E — PARSE GENE-RATIO DENOMINATORS
# --------------------------------------------------------------------------------------------------

parse_ratio_denominator <- function(x) {

    x <- as.character(x)

    suppressWarnings(
        as.integer(
            sub(
                "^.*/",
                "",
                x
            )
        )
    )
}

go_denominators <- sort(
    unique(
        na.omit(
            parse_ratio_denominator(
                go$GeneRatio
            )
        )
    )
)

reactome_denominators <- sort(
    unique(
        na.omit(
            parse_ratio_denominator(
                reactome$GeneRatio
            )
        )
    )
)

cat(
    "\nGO-BP GeneRatio denominator(s): ",
    paste(go_denominators, collapse = ", "),
    "\n",
    sep = ""
)

cat(
    "Reactome GeneRatio denominator(s): ",
    paste(reactome_denominators, collapse = ", "),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10C.1F — HELPER TO SPLIT geneID FIELDS
# --------------------------------------------------------------------------------------------------

split_gene_ids <- function(x) {

    if (
        length(x) == 0L ||
        is.na(x) ||
        !nzchar(trimws(x))
    ) {
        return(character(0))
    }

    z <- unlist(
        strsplit(
            as.character(x),
            "/",
            fixed = TRUE
        ),
        use.names = FALSE
    )

    unique(
        trimws(
            z[
                nzchar(trimws(z))
            ]
        )
    )
}

# --------------------------------------------------------------------------------------------------
# 10C.1G — QUERY GENES REPRESENTED IN GO RESULT SPACE
#
# enrichGO(readable=TRUE) stored geneID as gene symbols.
# --------------------------------------------------------------------------------------------------

go_all_gene_tokens <- unique(
    unlist(
        lapply(
            go$geneID,
            split_gene_ids
        ),
        use.names = FALSE
    )
)

go_all_gene_tokens <- go_all_gene_tokens[
    nzchar(go_all_gene_tokens)
]

go_query_symbols_represented <- intersect(
    coding12$gene_symbol,
    go_all_gene_tokens
)

go_query_symbols_missing <- setdiff(
    coding12$gene_symbol,
    go_query_symbols_represented
)

# --------------------------------------------------------------------------------------------------
# 10C.1H — QUERY GENES REPRESENTED IN REACTOME RESULT SPACE
#
# Reactome result remained in Entrez IDs.
# --------------------------------------------------------------------------------------------------

reactome_all_gene_tokens <- unique(
    unlist(
        lapply(
            reactome$geneID,
            split_gene_ids
        ),
        use.names = FALSE
    )
)

reactome_all_gene_tokens <- reactome_all_gene_tokens[
    nzchar(reactome_all_gene_tokens)
]

reactome_query_entrez_represented <- intersect(
    coding12$entrez_id,
    reactome_all_gene_tokens
)

reactome_query_entrez_missing <- setdiff(
    coding12$entrez_id,
    reactome_query_entrez_represented
)

# --------------------------------------------------------------------------------------------------
# 10C.1I — NOMINAL RESULT TABLES
# --------------------------------------------------------------------------------------------------

go_nominal <- go[
    !is.na(go$pvalue) &
    go$pvalue < 0.05,
    ,
    drop = FALSE
]

reactome_nominal <- reactome[
    !is.na(reactome$pvalue) &
    reactome$pvalue < 0.05,
    ,
    drop = FALSE
]

if (nrow(go_nominal) > 0L) {

    go_nominal$n_query_genes_in_term <- vapply(
        go_nominal$geneID,
        function(x) {
            length(split_gene_ids(x))
        },
        FUN.VALUE = integer(1)
    )

    go_nominal$driver_class <- ifelse(
        go_nominal$n_query_genes_in_term == 1L,
        "SINGLE_GENE",
        "MULTI_GENE"
    )

} else {

    go_nominal$n_query_genes_in_term <- integer(0)
    go_nominal$driver_class <- character(0)
}

if (nrow(reactome_nominal) > 0L) {

    reactome_nominal$n_query_genes_in_term <- vapply(
        reactome_nominal$geneID,
        function(x) {
            length(split_gene_ids(x))
        },
        FUN.VALUE = integer(1)
    )

    reactome_nominal$driver_class <- ifelse(
        reactome_nominal$n_query_genes_in_term == 1L,
        "SINGLE_GENE",
        "MULTI_GENE"
    )

} else {

    reactome_nominal$n_query_genes_in_term <- integer(0)
    reactome_nominal$driver_class <- character(0)
}

n_go_nominal_single <- sum(
    go_nominal$driver_class == "SINGLE_GENE"
)

n_go_nominal_multi <- sum(
    go_nominal$driver_class == "MULTI_GENE"
)

n_reactome_nominal_single <- sum(
    reactome_nominal$driver_class == "SINGLE_GENE"
)

n_reactome_nominal_multi <- sum(
    reactome_nominal$driver_class == "MULTI_GENE"
)

write.csv(
    go_nominal,
    OUT_GO_NOMINAL_DRIVER,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    reactome_nominal,
    OUT_REACTOME_NOMINAL_DRIVER,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10C.1J — PER-GENE NOMINAL TERM COUNTS
# --------------------------------------------------------------------------------------------------

count_symbol_terms <- function(symbol, df) {

    if (nrow(df) == 0L) {
        return(0L)
    }

    sum(
        vapply(
            df$geneID,
            function(x) {
                symbol %in% split_gene_ids(x)
            },
            FUN.VALUE = logical(1)
        )
    )
}

count_entrez_terms <- function(entrez, df) {

    if (nrow(df) == 0L) {
        return(0L)
    }

    sum(
        vapply(
            df$geneID,
            function(x) {
                entrez %in% split_gene_ids(x)
            },
            FUN.VALUE = logical(1)
        )
    )
}

coding_audit <- coding12

coding_audit$GO_BP_represented_in_result_space <- (
    coding_audit$gene_symbol %in%
    go_query_symbols_represented
)

coding_audit$Reactome_represented_in_result_space <- (
    coding_audit$entrez_id %in%
    reactome_query_entrez_represented
)

coding_audit$GO_BP_nominal_term_count <- vapply(
    coding_audit$gene_symbol,
    count_symbol_terms,
    df = go_nominal,
    FUN.VALUE = integer(1)
)

coding_audit$Reactome_nominal_pathway_count <- vapply(
    coding_audit$entrez_id,
    count_entrez_terms,
    df = reactome_nominal,
    FUN.VALUE = integer(1)
)

coding_audit$GO_BP_FDR05_term_count <- 0L
coding_audit$Reactome_FDR05_pathway_count <- 0L

coding_audit$conventional_ORA_inference <- ifelse(
    coding_audit$GO_BP_nominal_term_count > 0L |
    coding_audit$Reactome_nominal_pathway_count > 0L,
    "NOMINAL_EXPLORATORY_ONLY",
    "NO_NOMINAL_SIGNAL"
)

# --------------------------------------------------------------------------------------------------
# 10C.1K — VERIFY COVERAGE COUNTS AGAINST GeneRatio DENOMINATORS
# --------------------------------------------------------------------------------------------------

n_go_represented <- sum(
    coding_audit$GO_BP_represented_in_result_space
)

n_reactome_represented <- sum(
    coding_audit$Reactome_represented_in_result_space
)

cat(
    "\nFrozen coding genes represented in GO-BP result space: ",
    n_go_represented,
    "/12\n",
    sep = ""
)

cat(
    "Frozen coding genes represented in Reactome result space: ",
    n_reactome_represented,
    "/12\n",
    sep = ""
)

cat(
    "\nGO-BP coding genes not represented:\n"
)

if (length(go_query_symbols_missing) == 0L) {
    cat("[NONE]\n")
} else {
    cat(
        paste(
            go_query_symbols_missing,
            collapse = ", "
        ),
        "\n"
    )
}

cat(
    "\nReactome coding genes not represented:\n"
)

if (length(reactome_query_entrez_missing) == 0L) {

    cat("[NONE]\n")

} else {

    reactome_missing_rows <- coding_audit[
        coding_audit$entrez_id %in%
        reactome_query_entrez_missing,
        c(
            "ensemblID_stable",
            "gene_symbol",
            "entrez_id"
        ),
        drop = FALSE
    ]

    print(
        reactome_missing_rows,
        row.names = FALSE
    )
}

# --------------------------------------------------------------------------------------------------
# 10C.1L — CREATE COMPLETE 24-GENE INTERPRETATION SCAFFOLD
# --------------------------------------------------------------------------------------------------

if (!"protein_coding_flag" %in% names(full24)) {
    stop(
        "STEP10A table missing protein_coding_flag."
    )
}

if (!is.logical(full24$protein_coding_flag)) {

    temp <- toupper(
        trimws(
            as.character(
                full24$protein_coding_flag
            )
        )
    )

    if (!all(temp %in% c("TRUE", "FALSE"))) {
        stop(
            "Unable to interpret protein_coding_flag."
        )
    }

    full24$protein_coding_flag <- (
        temp == "TRUE"
    )
}

scaffold <- full24

scaffold$GO_BP_represented_in_result_space <- NA
scaffold$Reactome_represented_in_result_space <- NA
scaffold$GO_BP_nominal_term_count <- NA_integer_
scaffold$Reactome_nominal_pathway_count <- NA_integer_

for (i in seq_len(nrow(scaffold))) {

    if (scaffold$protein_coding_flag[i]) {

        match_index <- match(
            scaffold$ensemblID_stable[i],
            coding_audit$ensemblID_stable
        )

        if (is.na(match_index)) {
            stop(
                "Protein-coding frozen gene absent from coding audit: ",
                scaffold$ensemblID_stable[i]
            )
        }

        scaffold$GO_BP_represented_in_result_space[i] <-
            coding_audit$GO_BP_represented_in_result_space[
                match_index
            ]

        scaffold$Reactome_represented_in_result_space[i] <-
            coding_audit$Reactome_represented_in_result_space[
                match_index
            ]

        scaffold$GO_BP_nominal_term_count[i] <-
            coding_audit$GO_BP_nominal_term_count[
                match_index
            ]

        scaffold$Reactome_nominal_pathway_count[i] <-
            coding_audit$Reactome_nominal_pathway_count[
                match_index
            ]
    }
}

scaffold$biological_interpretation_track <- ifelse(
    scaffold$protein_coding_flag,
    "PROTEIN_CODING_GENE_LEVEL_PLUS_CONSERVATIVE_ORA",
    "NONCODING_GENE_LEVEL_INTERPRETATION"
)

scaffold$pathway_evidence_status <- ifelse(
    scaffold$protein_coding_flag,
    "NO_FDR_SUPPORTED_GO_BP_OR_REACTOME_ENRICHMENT",
    "NOT_APPLICABLE_TO_STANDARD_PROTEIN_CODING_ORA"
)

# --------------------------------------------------------------------------------------------------
# 10C.1M — WRITE GENE-LEVEL OUTPUTS
# --------------------------------------------------------------------------------------------------

write.csv(
    coding_audit,
    OUT_CODING_AUDIT,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    scaffold,
    OUT_FULL24,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10C.1N — OVERALL INTERPRETATION
# --------------------------------------------------------------------------------------------------

if (
    n_go_fdr == 0L &&
    n_reactome_fdr == 0L
) {

    overall_pathway_status <-
        "NO_FDR_SUPPORTED_CONVENTIONAL_PATHWAY_ENRICHMENT"

} else {

    stop(
        "Unexpected FDR-supported enrichment detected during verification."
    )
}

# --------------------------------------------------------------------------------------------------
# 10C.1O — SUMMARY FILE
# --------------------------------------------------------------------------------------------------

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10C.1 — FUNCTIONAL-DATABASE COVERAGE + NOMINAL-SIGNAL DRIVER AUDIT",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "PRIMARY INFERENTIAL RESULT",

    "GO-BP FDR < 0.05 terms: 0",
    "Reactome FDR < 0.05 pathways: 0",

    paste0(
        "Overall status: ",
        overall_pathway_status
    ),

    "",

    "NOMINAL SIGNALS",

    paste0(
        "GO-BP nominal p < 0.05 terms: ",
        n_go_nominal
    ),

    paste0(
        "GO-BP nominal single-gene terms: ",
        n_go_nominal_single
    ),

    paste0(
        "GO-BP nominal multi-gene terms: ",
        n_go_nominal_multi
    ),

    paste0(
        "Reactome nominal p < 0.05 pathways: ",
        n_reactome_nominal
    ),

    paste0(
        "Reactome nominal single-gene pathways: ",
        n_reactome_nominal_single
    ),

    paste0(
        "Reactome nominal multi-gene pathways: ",
        n_reactome_nominal_multi
    ),

    "",

    "DATABASE COVERAGE",

    paste0(
        "Frozen coding genes: 12"
    ),

    paste0(
        "GO-BP represented coding genes: ",
        n_go_represented,
        "/12"
    ),

    paste0(
        "Reactome represented coding genes: ",
        n_reactome_represented,
        "/12"
    ),

    paste0(
        "GO-BP GeneRatio denominator(s): ",
        paste(
            go_denominators,
            collapse = ", "
        )
    ),

    paste0(
        "Reactome GeneRatio denominator(s): ",
        paste(
            reactome_denominators,
            collapse = ", "
        )
    ),

    "",

    "GO-BP CODING GENES NOT REPRESENTED",

    if (
        length(go_query_symbols_missing) == 0L
    ) {
        "[NONE]"
    } else {
        paste(
            go_query_symbols_missing,
            collapse = ", "
        )
    },

    "",

    "REACTOME CODING GENES NOT REPRESENTED",

    if (
        length(reactome_query_entrez_missing) == 0L
    ) {

        "[NONE]"

    } else {

        paste(
            coding_audit$gene_symbol[
                coding_audit$entrez_id %in%
                reactome_query_entrez_missing
            ],
            collapse = ", "
        )
    },

    "",

    "INTERPRETATION",

    paste(
        "No conventional GO Biological Process or Reactome pathway survived",
        "Benjamini-Hochberg FDR < 0.05."
    ),

    paste(
        "Nominal signals are therefore exploratory and should be used only",
        "as gene-level biological context, not as statistically established pathway enrichment."
    ),

    paste(
        "Single-gene-driven nominal categories require especially cautious interpretation."
    ),

    paste(
        "The full frozen molecular state contains 12 additional non-protein-coding genes,",
        "which are outside the scope of conventional protein-coding GO/Reactome ORA and",
        "must be interpreted separately."
    ),

    "",

    "NEXT STEP",

    paste(
        "Proceed to evidence-based gene-level biological interpretation of all 24 frozen genes,",
        "keeping protein-coding and non-protein-coding components explicit and treating nominal",
        "pathway signals only as secondary contextual evidence."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10C.1P — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    FULL24_PATH,
    CODING12_PATH,
    GO_PATH,
    REACTOME_PATH,
    OUT_CODING_AUDIT,
    OUT_FULL24,
    OUT_GO_NOMINAL_DRIVER,
    OUT_REACTOME_NOMINAL_DRIVER,
    OUT_SUMMARY
)

manifest <- data.frame(
    file = basename(manifest_paths),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(manifest_paths)$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10C.1Q — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFUNCTIONAL DATABASE COVERAGE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "GO-BP represented coding genes: ",
    n_go_represented,
    "/12\n",
    sep = ""
)

cat(
    "GO-BP GeneRatio denominator(s): ",
    paste(go_denominators, collapse = ", "),
    "\n",
    sep = ""
)

cat(
    "Reactome represented coding genes: ",
    n_reactome_represented,
    "/12\n",
    sep = ""
)

cat(
    "Reactome GeneRatio denominator(s): ",
    paste(reactome_denominators, collapse = ", "),
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nNOMINAL-SIGNAL DRIVER STRUCTURE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "GO-BP nominal terms: ",
    n_go_nominal,
    "\n",
    sep = ""
)

cat(
    "  Single-gene driven: ",
    n_go_nominal_single,
    "\n",
    sep = ""
)

cat(
    "  Multi-gene driven: ",
    n_go_nominal_multi,
    "\n",
    sep = ""
)

cat(
    "Reactome nominal pathways: ",
    n_reactome_nominal,
    "\n",
    sep = ""
)

cat(
    "  Single-gene driven: ",
    n_reactome_nominal_single,
    "\n",
    sep = ""
)

cat(
    "  Multi-gene driven: ",
    n_reactome_nominal_multi,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\n12 PROTEIN-CODING GENE AUDIT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    coding_audit[
        ,
        c(
            "ensemblID_stable",
            "gene_symbol",
            "direction_standardized",
            "final_normalized_weight",
            "entrez_id",
            "GO_BP_represented_in_result_space",
            "GO_BP_nominal_term_count",
            "Reactome_represented_in_result_space",
            "Reactome_nominal_pathway_count",
            "conventional_ORA_inference"
        )
    ],
    row.names = FALSE
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nINFERENTIAL CONCLUSION\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "GO-BP FDR < 0.05: 0\n"
)

cat(
    "Reactome FDR < 0.05: 0\n"
)

cat(
    "STATUS: ",
    overall_pathway_status,
    "\n",
    sep = ""
)

cat(
    "\nDo NOT report nominal pathways as statistically established enrichment.\n"
)

cat(
    "\nCoding-gene audit:\n",
    OUT_CODING_AUDIT,
    "\n",
    sep = ""
)

cat(
    "\nFull 24-gene interpretation scaffold:\n",
    OUT_FULL24,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10C.1 COMPLETE — FUNCTIONAL DATABASE COVERAGE AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10C.1 — FUNCTIONAL-DATABASE COVERAGE + NOMINAL-SIGNAL DRIVER AUDIT
Project sentinel: PASS
STEP10A full 24-gene table: PASS
STEP10B.1 coding 12-gene table: PASS
STEP10B.2 GO-BP results: PASS
STEP10B.2 Reactome results: PASS
GO-BP FDR < 0.05 terms: 0 — VERIFIED
Reactome FDR < 0.05 pathways: 0 — VERIFIED

GO-BP GeneRatio denominator(s): 11
Reactome GeneRatio denominator(s): 9

Frozen coding genes represented in GO-BP result space: 10/12
Frozen coding genes represented in Reactome result space: 9/12

GO-BP coding genes not represented:
CD3EAP, ARHGAP11B 

Reactome coding genes not represented:
 ensemblID_stable gene_symbol entrez_id
  ENSG00000010319      SEMA3G     56920
  ENSG00000187951   ARHGAP11B 100288637
  ENSG00000244405        ETV5      2119

FUNCTIONAL DATABASE COVERAGE
GO-BP represented coding genes: 10/12
GO-BP GeneRatio denominator(s): 11
Reactome represented coding genes: 9/12
Reactome GeneRatio denominator(s): 9

NOMINAL-SIGNAL DRIVER STRUCTURE
GO-BP nominal terms: 101

In [65]:
# ==================================================================================================
# STEP 10C.1R — CORRECTED ENTREZ-LEVEL FUNCTIONAL-DATABASE COVERAGE AUDIT
# --------------------------------------------------------------------------------------------------
# REASON FOR CORRECTION
#   STEP10C.1 assessed GO coverage by comparing frozen gene symbols against enrichGO(readable=TRUE)
#   output. readble=TRUE converts Entrez IDs to CURRENT OrgDb symbols.
#
#   Example already observed:
#       frozen symbol: CD3EAP
#       Entrez ID:     10849
#       GO readable symbol observed: POLR1G
#
#   Therefore frozen-symbol comparison can falsely classify a gene as absent.
#
# CORRECTION
#   Re-run ONLY GO-BP with identical parameters but readable=FALSE,
#   then audit representation strictly by Entrez ID.
#
# IMPORTANT
#   - This does NOT alter the enrichment analysis.
#   - This does NOT change p-values/FDR interpretation.
#   - Reactome is NOT rerun.
#   - Prior STEP10C.1 coverage artifact is retained but marked superseded for GO coverage.
#   - No nominal pathway is promoted to significant.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10C.1R — CORRECTED ENTREZ-LEVEL FUNCTIONAL-DATABASE COVERAGE AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

BACKGROUND_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1",
    "STEP10B1_discovery_protein_coding_Entrez_background_v1.csv"
)

CODING12_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1",
    "STEP10B1_SCZ_MSS_12_protein_coding_enrichment_input_v1.csv"
)

OLD_GO_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1",
    "ora_v1",
    "STEP10B2_GO_BP_all_tested_terms_v1.csv"
)

REACTOME_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "enrichment_v1",
    "ora_v1",
    "STEP10B2_Reactome_all_tested_pathways_v1.csv"
)

FULL24_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "annotation_v1",
    "STEP10A_SCZ_MSS_24gene_biological_annotation_v1.csv"
)

OLD_COVERAGE_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "gene_level_interpretation_v1",
    "STEP10C1_12coding_functional_database_driver_audit_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "gene_level_interpretation_v1",
    "coverage_correction_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_GO_ENTREZ <- file.path(
    OUT_DIR,
    "STEP10C1R_GO_BP_Entrez_level_results_v1.csv"
)

OUT_CODING_AUDIT <- file.path(
    OUT_DIR,
    "STEP10C1R_12coding_corrected_database_coverage_v1.csv"
)

OUT_FULL24 <- file.path(
    OUT_DIR,
    "STEP10C1R_24gene_corrected_interpretation_scaffold_v1.csv"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP10C1R_corrected_functional_database_coverage_summary_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP10C1R_corrected_coverage_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-B — PROJECT + INPUT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

required_files <- c(
    BACKGROUND_PATH,
    CODING12_PATH,
    OLD_GO_PATH,
    REACTOME_PATH,
    FULL24_PATH
)

missing_files <- required_files[
    !file.exists(required_files)
]

if (length(missing_files) > 0L) {
    stop(
        "Required upstream file(s) missing:\n",
        paste(missing_files, collapse = "\n")
    )
}

cat("STEP10B.1 background/input: PASS\n")
cat("STEP10B.2 GO + Reactome outputs: PASS\n")
cat("STEP10A 24-gene annotation: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10C.1R-C — LOAD DATA
# --------------------------------------------------------------------------------------------------

background <- read.csv(
    BACKGROUND_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

coding12 <- read.csv(
    CODING12_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

old_go <- read.csv(
    OLD_GO_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

reactome <- read.csv(
    REACTOME_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

full24 <- read.csv(
    FULL24_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(coding12) != 12L) {
    stop(
        "Expected 12 frozen coding genes; observed ",
        nrow(coding12)
    )
}

if (nrow(full24) != 24L) {
    stop(
        "Expected 24 frozen genes; observed ",
        nrow(full24)
    )
}

background$entrez_id <- trimws(
    as.character(background$entrez_id)
)

coding12$entrez_id <- trimws(
    as.character(coding12$entrez_id)
)

background_entrez <- sort(
    unique(
        background$entrez_id[
            !is.na(background$entrez_id) &
            nzchar(background$entrez_id)
        ]
    )
)

query_entrez <- sort(
    unique(
        coding12$entrez_id[
            !is.na(coding12$entrez_id) &
            nzchar(coding12$entrez_id)
        ]
    )
)

if (length(background_entrez) != 13837L) {
    stop(
        "Expected 13,837 distinct frozen background Entrez IDs; observed ",
        length(background_entrez)
    )
}

if (length(query_entrez) != 12L) {
    stop(
        "Expected 12 distinct frozen query Entrez IDs; observed ",
        length(query_entrez)
    )
}

if (!all(query_entrez %in% background_entrez)) {
    stop("One or more frozen query IDs are absent from background.")
}

cat("Distinct background Entrez IDs: 13,837 — PASS\n")
cat("Distinct frozen coding query IDs: 12 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10C.1R-D — PACKAGE VERIFICATION
# --------------------------------------------------------------------------------------------------

required_packages <- c(
    "clusterProfiler",
    "org.Hs.eg.db",
    "AnnotationDbi"
)

missing_packages <- required_packages[
    !vapply(
        required_packages,
        requireNamespace,
        quietly = TRUE,
        FUN.VALUE = logical(1)
    )
]

if (length(missing_packages) > 0L) {
    stop(
        "Required package(s) unavailable: ",
        paste(missing_packages, collapse = ", "),
        "\nThese were installed in STEP10B.2 and should already be present."
    )
}

# --------------------------------------------------------------------------------------------------
# 10C.1R-E — RE-RUN IDENTICAL GO-BP ORA WITH readable=FALSE
#
# ONLY representation labels change.
# Statistical analysis parameters remain identical to STEP10B.2.
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nRE-RUN GO-BP WITH ENTREZ-LEVEL OUTPUT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

go_entrez_result <- clusterProfiler::enrichGO(
    gene = query_entrez,
    universe = background_entrez,
    OrgDb = org.Hs.eg.db::org.Hs.eg.db,
    keyType = "ENTREZID",
    ont = "BP",
    pvalueCutoff = 1,
    pAdjustMethod = "BH",
    qvalueCutoff = 1,
    minGSSize = 10,
    maxGSSize = 500,
    readable = FALSE
)

go_entrez <- as.data.frame(
    go_entrez_result
)

if (nrow(go_entrez) > 0L) {

    go_entrez <- go_entrez[
        order(
            go_entrez$p.adjust,
            go_entrez$pvalue,
            na.last = TRUE
        ),
        ,
        drop = FALSE
    ]
}

write.csv(
    go_entrez,
    OUT_GO_ENTREZ,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-F — VERIFY THAT STATISTICAL RESULTS ARE UNCHANGED
# --------------------------------------------------------------------------------------------------

if (nrow(go_entrez) != nrow(old_go)) {
    stop(
        "GO term count changed unexpectedly.\n",
        "Previous: ", nrow(old_go),
        "\nCurrent: ", nrow(go_entrez)
    )
}

old_go_check <- old_go[
    order(
        old_go$ID
    ),
    c(
        "ID",
        "pvalue",
        "p.adjust"
    ),
    drop = FALSE
]

new_go_check <- go_entrez[
    order(
        go_entrez$ID
    ),
    c(
        "ID",
        "pvalue",
        "p.adjust"
    ),
    drop = FALSE
]

if (!identical(
    as.character(old_go_check$ID),
    as.character(new_go_check$ID)
)) {
    stop("GO term IDs changed unexpectedly.")
}

p_diff <- max(
    abs(
        old_go_check$pvalue -
        new_go_check$pvalue
    ),
    na.rm = TRUE
)

fdr_diff <- max(
    abs(
        old_go_check$p.adjust -
        new_go_check$p.adjust
    ),
    na.rm = TRUE
)

if (
    !is.finite(p_diff) ||
    !is.finite(fdr_diff) ||
    p_diff > 1e-12 ||
    fdr_diff > 1e-12
) {
    stop(
        "GO statistical results changed unexpectedly.\n",
        "Maximum p-value difference: ", p_diff,
        "\nMaximum FDR difference: ", fdr_diff
    )
}

cat("GO statistical equivalence after readable=FALSE: PASS\n")

n_go_fdr <- sum(
    !is.na(go_entrez$p.adjust) &
    go_entrez$p.adjust < 0.05
)

n_go_nominal <- sum(
    !is.na(go_entrez$pvalue) &
    go_entrez$pvalue < 0.05
)

n_reactome_fdr <- sum(
    !is.na(reactome$p.adjust) &
    reactome$p.adjust < 0.05
)

n_reactome_nominal <- sum(
    !is.na(reactome$pvalue) &
    reactome$pvalue < 0.05
)

if (
    n_go_fdr != 0L ||
    n_reactome_fdr != 0L
) {
    stop(
        "Unexpected FDR-supported pathway detected."
    )
}

cat("GO-BP FDR < 0.05: 0 — VERIFIED\n")
cat("Reactome FDR < 0.05: 0 — VERIFIED\n")

# --------------------------------------------------------------------------------------------------
# 10C.1R-G — HELPERS
# --------------------------------------------------------------------------------------------------

split_ids <- function(x) {

    if (
        length(x) == 0L ||
        is.na(x) ||
        !nzchar(trimws(x))
    ) {
        return(character(0))
    }

    unique(
        trimws(
            unlist(
                strsplit(
                    as.character(x),
                    "/",
                    fixed = TRUE
                ),
                use.names = FALSE
            )
        )
    )
}

parse_ratio_denominator <- function(x) {

    suppressWarnings(
        as.integer(
            sub(
                "^.*/",
                "",
                as.character(x)
            )
        )
    )
}

# --------------------------------------------------------------------------------------------------
# 10C.1R-H — CORRECT GO REPRESENTATION USING ENTREZ IDs
# --------------------------------------------------------------------------------------------------

go_gene_tokens <- unique(
    unlist(
        lapply(
            go_entrez$geneID,
            split_ids
        ),
        use.names = FALSE
    )
)

go_gene_tokens <- go_gene_tokens[
    nzchar(go_gene_tokens)
]

go_represented_entrez <- intersect(
    query_entrez,
    go_gene_tokens
)

go_missing_entrez <- setdiff(
    query_entrez,
    go_represented_entrez
)

go_denominators <- sort(
    unique(
        na.omit(
            parse_ratio_denominator(
                go_entrez$GeneRatio
            )
        )
    )
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-I — REACTOME REPRESENTATION USING ENTREZ IDs
# --------------------------------------------------------------------------------------------------

reactome_gene_tokens <- unique(
    unlist(
        lapply(
            reactome$geneID,
            split_ids
        ),
        use.names = FALSE
    )
)

reactome_gene_tokens <- reactome_gene_tokens[
    nzchar(reactome_gene_tokens)
]

reactome_represented_entrez <- intersect(
    query_entrez,
    reactome_gene_tokens
)

reactome_missing_entrez <- setdiff(
    query_entrez,
    reactome_represented_entrez
)

reactome_denominators <- sort(
    unique(
        na.omit(
            parse_ratio_denominator(
                reactome$GeneRatio
            )
        )
    )
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-J — CURRENT SYMBOL MAPPING AUDIT
# --------------------------------------------------------------------------------------------------

current_symbols <- AnnotationDbi::select(
    org.Hs.eg.db::org.Hs.eg.db,
    keys = query_entrez,
    keytype = "ENTREZID",
    columns = c(
        "SYMBOL",
        "GENENAME"
    )
)

current_symbols <- current_symbols[
    !duplicated(
        current_symbols$ENTREZID
    ),
    ,
    drop = FALSE
]

coding_audit <- merge(
    coding12,
    current_symbols,
    by.x = "entrez_id",
    by.y = "ENTREZID",
    all.x = TRUE,
    sort = FALSE
)

coding_audit <- coding_audit[
    match(
        coding12$entrez_id,
        coding_audit$entrez_id
    ),
    ,
    drop = FALSE
]

names(coding_audit)[
    names(coding_audit) == "SYMBOL"
] <- "current_orgdb_symbol"

names(coding_audit)[
    names(coding_audit) == "GENENAME"
] <- "current_orgdb_gene_name"

coding_audit$symbol_changed_or_differs <- (
    !is.na(coding_audit$current_orgdb_symbol) &
    coding_audit$gene_symbol !=
        coding_audit$current_orgdb_symbol
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-K — CORRECT PER-GENE COVERAGE
# --------------------------------------------------------------------------------------------------

coding_audit$GO_BP_represented_corrected <- (
    coding_audit$entrez_id %in%
    go_represented_entrez
)

coding_audit$Reactome_represented <- (
    coding_audit$entrez_id %in%
    reactome_represented_entrez
)

go_nominal <- go_entrez[
    !is.na(go_entrez$pvalue) &
    go_entrez$pvalue < 0.05,
    ,
    drop = FALSE
]

reactome_nominal <- reactome[
    !is.na(reactome$pvalue) &
    reactome$pvalue < 0.05,
    ,
    drop = FALSE
]

count_terms <- function(entrez, df) {

    if (nrow(df) == 0L) {
        return(0L)
    }

    sum(
        vapply(
            df$geneID,
            function(x) {
                entrez %in% split_ids(x)
            },
            FUN.VALUE = logical(1)
        )
    )
}

coding_audit$GO_BP_nominal_term_count <- vapply(
    coding_audit$entrez_id,
    count_terms,
    df = go_nominal,
    FUN.VALUE = integer(1)
)

coding_audit$Reactome_nominal_pathway_count <- vapply(
    coding_audit$entrez_id,
    count_terms,
    df = reactome_nominal,
    FUN.VALUE = integer(1)
)

coding_audit$FDR_supported_GO_BP <- FALSE
coding_audit$FDR_supported_Reactome <- FALSE

coding_audit$functional_inference_status <- ifelse(
    coding_audit$GO_BP_nominal_term_count > 0L |
    coding_audit$Reactome_nominal_pathway_count > 0L,
    "NOMINAL_CONTEXT_ONLY",
    "NO_NOMINAL_ORA_CONTEXT"
)

n_go_represented <- sum(
    coding_audit$GO_BP_represented_corrected
)

n_reactome_represented <- sum(
    coding_audit$Reactome_represented
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-L — CONSISTENCY WITH GeneRatio DENOMINATORS
#
# GO denominator represents the query genes entering the tested GO universe.
# Some denominator genes may still not appear among retained result rows depending on
# ontology / gene-set-size filtering, but Entrez-level representation is the correct audit.
# --------------------------------------------------------------------------------------------------

cat(
    "\nCorrected GO-BP represented query genes: ",
    n_go_represented,
    "/12\n",
    sep = ""
)

cat(
    "GO-BP GeneRatio denominator(s): ",
    paste(go_denominators, collapse = ", "),
    "\n",
    sep = ""
)

cat(
    "Reactome represented query genes: ",
    n_reactome_represented,
    "/12\n",
    sep = ""
)

cat(
    "Reactome GeneRatio denominator(s): ",
    paste(reactome_denominators, collapse = ", "),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-M — CREATE CORRECTED FULL-24 SCAFFOLD
# --------------------------------------------------------------------------------------------------

if (!"protein_coding_flag" %in% names(full24)) {
    stop("STEP10A file missing protein_coding_flag.")
}

if (!is.logical(full24$protein_coding_flag)) {

    z <- toupper(
        trimws(
            as.character(
                full24$protein_coding_flag
            )
        )
    )

    if (!all(z %in% c("TRUE", "FALSE"))) {
        stop("Cannot interpret protein_coding_flag.")
    }

    full24$protein_coding_flag <- (
        z == "TRUE"
    )
}

scaffold <- full24

scaffold$GO_BP_represented_corrected <- NA
scaffold$Reactome_represented <- NA
scaffold$GO_BP_nominal_term_count <- NA_integer_
scaffold$Reactome_nominal_pathway_count <- NA_integer_
scaffold$current_orgdb_symbol_for_coding <- NA_character_
scaffold$current_orgdb_gene_name_for_coding <- NA_character_

for (i in seq_len(nrow(scaffold))) {

    if (scaffold$protein_coding_flag[i]) {

        j <- match(
            scaffold$ensemblID_stable[i],
            coding_audit$ensemblID_stable
        )

        if (is.na(j)) {
            stop(
                "Frozen coding gene missing from corrected audit: ",
                scaffold$ensemblID_stable[i]
            )
        }

        scaffold$GO_BP_represented_corrected[i] <-
            coding_audit$GO_BP_represented_corrected[j]

        scaffold$Reactome_represented[i] <-
            coding_audit$Reactome_represented[j]

        scaffold$GO_BP_nominal_term_count[i] <-
            coding_audit$GO_BP_nominal_term_count[j]

        scaffold$Reactome_nominal_pathway_count[i] <-
            coding_audit$Reactome_nominal_pathway_count[j]

        scaffold$current_orgdb_symbol_for_coding[i] <-
            coding_audit$current_orgdb_symbol[j]

        scaffold$current_orgdb_gene_name_for_coding[i] <-
            coding_audit$current_orgdb_gene_name[j]
    }
}

scaffold$biological_interpretation_track <- ifelse(
    scaffold$protein_coding_flag,
    "PROTEIN_CODING_GENE_LEVEL_INTERPRETATION",
    "NON_PROTEIN_CODING_GENE_LEVEL_INTERPRETATION"
)

scaffold$pathway_inference_status <- ifelse(
    scaffold$protein_coding_flag,
    "NO_FDR_SUPPORTED_GO_BP_OR_REACTOME_ENRICHMENT",
    "NOT_TESTED_BY_STANDARD_PROTEIN_CODING_ORA"
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-N — WRITE CORRECTED ARTIFACTS
# --------------------------------------------------------------------------------------------------

write.csv(
    coding_audit,
    OUT_CODING_AUDIT,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    scaffold,
    OUT_FULL24,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-O — SUMMARY
# --------------------------------------------------------------------------------------------------

symbol_difference_rows <- coding_audit[
    coding_audit$symbol_changed_or_differs,
    c(
        "entrez_id",
        "gene_symbol",
        "current_orgdb_symbol",
        "current_orgdb_gene_name"
    ),
    drop = FALSE
]

go_missing_rows <- coding_audit[
    !coding_audit$GO_BP_represented_corrected,
    c(
        "ensemblID_stable",
        "gene_symbol",
        "current_orgdb_symbol",
        "entrez_id"
    ),
    drop = FALSE
]

reactome_missing_rows <- coding_audit[
    !coding_audit$Reactome_represented,
    c(
        "ensemblID_stable",
        "gene_symbol",
        "current_orgdb_symbol",
        "entrez_id"
    ),
    drop = FALSE
]

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10C.1R — CORRECTED ENTREZ-LEVEL FUNCTIONAL-DATABASE COVERAGE AUDIT",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "REASON FOR CORRECTION",

    paste(
        "STEP10C.1 compared frozen symbols to enrichGO(readable=TRUE) output.",
        "Because readable=TRUE converts Entrez IDs to current OrgDb symbols,",
        "frozen/current symbol differences can falsely appear as missing GO coverage."
    ),

    "",

    "STATISTICAL RESULT",

    paste0(
        "GO-BP returned terms: ",
        nrow(go_entrez)
    ),

    paste0(
        "GO-BP nominal p < 0.05: ",
        n_go_nominal
    ),

    "GO-BP FDR < 0.05: 0",

    paste0(
        "Reactome nominal p < 0.05: ",
        n_reactome_nominal
    ),

    "Reactome FDR < 0.05: 0",

    "",

    "CORRECTED FUNCTIONAL-DATABASE COVERAGE",

    paste0(
        "GO-BP represented frozen coding genes: ",
        n_go_represented,
        "/12"
    ),

    paste0(
        "GO-BP GeneRatio denominator(s): ",
        paste(go_denominators, collapse = ", ")
    ),

    paste0(
        "Reactome represented frozen coding genes: ",
        n_reactome_represented,
        "/12"
    ),

    paste0(
        "Reactome GeneRatio denominator(s): ",
        paste(reactome_denominators, collapse = ", ")
    ),

    "",

    "PRIMARY INTERPRETATION",

    paste(
        "The prior GO coverage audit was potentially biased by frozen-vs-current symbol naming.",
        "This corrected audit uses Entrez IDs throughout."
    ),

    paste(
        "The inferential enrichment conclusion is unchanged:",
        "no GO-BP or Reactome category survives BH FDR < 0.05."
    ),

    paste(
        "Nominal associations remain exploratory gene-level context only."
    ),

    "",

    "SUPERSEDED ARTIFACT",

    if (file.exists(OLD_COVERAGE_PATH)) {
        paste0(
            OLD_COVERAGE_PATH,
            " [GO coverage fields superseded by STEP10C.1R]"
        )
    } else {
        "[Previous STEP10C.1 coverage artifact not found at expected path]"
    },

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-P — MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    BACKGROUND_PATH,
    CODING12_PATH,
    OLD_GO_PATH,
    REACTOME_PATH,
    FULL24_PATH,
    OUT_GO_ENTREZ,
    OUT_CODING_AUDIT,
    OUT_FULL24,
    OUT_SUMMARY
)

manifest <- data.frame(
    file = basename(manifest_paths),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(manifest_paths)$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10C.1R-Q — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nCORRECTED FUNCTIONAL-DATABASE COVERAGE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "GO-BP represented coding genes: ",
    n_go_represented,
    "/12\n",
    sep = ""
)

cat(
    "GO-BP GeneRatio denominator(s): ",
    paste(go_denominators, collapse = ", "),
    "\n",
    sep = ""
)

cat(
    "Reactome represented coding genes: ",
    n_reactome_represented,
    "/12\n",
    sep = ""
)

cat(
    "Reactome GeneRatio denominator(s): ",
    paste(reactome_denominators, collapse = ", "),
    "\n",
    sep = ""
)

cat(
    "\nGO-BP coding genes not represented in returned term space:\n"
)

if (nrow(go_missing_rows) == 0L) {
    cat("[NONE]\n")
} else {
    print(
        go_missing_rows,
        row.names = FALSE
    )
}

cat(
    "\nReactome coding genes not represented:\n"
)

if (nrow(reactome_missing_rows) == 0L) {
    cat("[NONE]\n")
} else {
    print(
        reactome_missing_rows,
        row.names = FALSE
    )
}

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nFROZEN VS CURRENT SYMBOL DIFFERENCES\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

if (nrow(symbol_difference_rows) == 0L) {

    cat("[NONE]\n")

} else {

    print(
        symbol_difference_rows,
        row.names = FALSE
    )
}

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nCORRECTED 12-GENE AUDIT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    coding_audit[
        ,
        c(
            "ensemblID_stable",
            "gene_symbol",
            "current_orgdb_symbol",
            "direction_standardized",
            "final_normalized_weight",
            "entrez_id",
            "GO_BP_represented_corrected",
            "GO_BP_nominal_term_count",
            "Reactome_represented",
            "Reactome_nominal_pathway_count",
            "functional_inference_status"
        )
    ],
    row.names = FALSE
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nINFERENTIAL CONCLUSION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat("GO-BP FDR < 0.05: 0\n")
cat("Reactome FDR < 0.05: 0\n")
cat("STATUS: NO_FDR_SUPPORTED_CONVENTIONAL_PATHWAY_ENRICHMENT\n")

cat(
    "\nPrevious STEP10C.1 GO coverage should be treated as superseded by this Entrez-level audit.\n"
)

cat(
    "\nCorrected coding audit:\n",
    OUT_CODING_AUDIT,
    "\n",
    sep = ""
)

cat(
    "\nCorrected full-24 scaffold:\n",
    OUT_FULL24,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10C.1R COMPLETE — ENTREZ-LEVEL COVERAGE CORRECTION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10C.1R — CORRECTED ENTREZ-LEVEL FUNCTIONAL-DATABASE COVERAGE AUDIT
Project sentinel: PASS
STEP10B.1 background/input: PASS
STEP10B.2 GO + Reactome outputs: PASS
STEP10A 24-gene annotation: PASS
Distinct background Entrez IDs: 13,837 — PASS
Distinct frozen coding query IDs: 12 — PASS

--------------------------------------------------------------------------------------------------------------
RE-RUN GO-BP WITH ENTREZ-LEVEL OUTPUT
--------------------------------------------------------------------------------------------------------------
GO statistical equivalence after readable=FALSE: PASS
GO-BP FDR < 0.05: 0 — VERIFIED
Reactome FDR < 0.05: 0 — VERIFIED


'select()' returned 1:1 mapping between keys and columns




Corrected GO-BP represented query genes: 11/12
GO-BP GeneRatio denominator(s): 11
Reactome represented query genes: 9/12
Reactome GeneRatio denominator(s): 9

CORRECTED FUNCTIONAL-DATABASE COVERAGE
GO-BP represented coding genes: 11/12
GO-BP GeneRatio denominator(s): 11
Reactome represented coding genes: 9/12
Reactome GeneRatio denominator(s): 9

GO-BP coding genes not represented in returned term space:
 ensemblID_stable gene_symbol current_orgdb_symbol entrez_id
  ENSG00000187951   ARHGAP11B         LOC100288637 100288637

Reactome coding genes not represented:
 ensemblID_stable gene_symbol current_orgdb_symbol entrez_id
  ENSG00000010319      SEMA3G               SEMA3G     56920
  ENSG00000187951   ARHGAP11B         LOC100288637 100288637
  ENSG00000244405        ETV5                 ETV5      2119

--------------------------------------------------------------------------------------------------------------
FROZEN VS CURRENT SYMBOL DIFFERENCES
------------------------------------

In [66]:
# ==================================================================================================
# STEP 10D.1 — FROZEN 24-GENE CURRENT IDENTITY INTEGRITY AUDIT
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Before biological/literature interpretation, independently verify the CURRENT identity
#   associated with each frozen stable Ensembl gene ID using:
#
#       1. Frozen BrainSeq annotation          [analysis-time canonical identity]
#       2. Current org.Hs.eg.db               [local current annotation snapshot]
#       3. Current Ensembl REST lookup         [stable Ensembl ID -> current display name/biotype]
#       4. Current HGNC REST lookup            [Ensembl ID -> approved HGNC nomenclature when available]
#
# CRITICAL PRINCIPLE
#   The frozen score is NOT modified.
#
#   Current annotations are used ONLY to determine how each feature should be described
#   biologically in the manuscript.
#
#   No feature is:
#       - removed
#       - renamed inside the frozen score
#       - reweighted
#       - remapped to another gene
#       - rescued by synonym
#
# IMPORTANT CURRENT ISSUE
#   STEP10C.1R showed at least two frozen/current symbol discrepancies:
#
#       frozen CD3EAP      -> current OrgDb POLR1G
#       frozen ARHGAP11B   -> current OrgDb LOC100288637
#
#   These require independent Ensembl/HGNC resolution before literature interpretation.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10D.1 — FROZEN 24-GENE CURRENT IDENTITY INTEGRITY AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10D.1A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

STEP10A_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "annotation_v1",
    "STEP10A_SCZ_MSS_24gene_biological_annotation_v1.csv"
)

STEP10C1R_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "gene_level_interpretation_v1",
    "coverage_correction_v1",
    "STEP10C1R_24gene_corrected_interpretation_scaffold_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "identity_audit_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_FULL <- file.path(
    OUT_DIR,
    "STEP10D1_SCZ_MSS_24gene_current_identity_audit_v1.csv"
)

OUT_DISCORDANT <- file.path(
    OUT_DIR,
    "STEP10D1_SCZ_MSS_identity_discordance_review_v1.csv"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP10D1_SCZ_MSS_identity_audit_summary_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP10D1_SCZ_MSS_identity_audit_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10D.1B — PROJECT / INPUT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

required_files <- c(
    STEP10A_PATH,
    STEP10C1R_PATH
)

missing_files <- required_files[
    !file.exists(required_files)
]

if (length(missing_files) > 0L) {
    stop(
        "Required upstream file(s) missing:\n",
        paste(missing_files, collapse = "\n")
    )
}

cat("STEP10A frozen annotation: PASS\n")
cat("STEP10C.1R corrected scaffold: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10D.1C — LOAD FROZEN SIGNATURE
# --------------------------------------------------------------------------------------------------

frozen <- read.csv(
    STEP10A_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

corrected <- read.csv(
    STEP10C1R_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(frozen) != 24L) {
    stop(
        "Expected 24 frozen genes; observed ",
        nrow(frozen)
    )
}

required_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "protein_coding_flag",
    "direction_standardized",
    "final_normalized_weight"
)

missing_cols <- setdiff(
    required_cols,
    names(frozen)
)

if (length(missing_cols) > 0L) {
    stop(
        "STEP10A missing required column(s): ",
        paste(missing_cols, collapse = ", ")
    )
}

frozen$ensemblID_stable <- trimws(
    as.character(frozen$ensemblID_stable)
)

frozen$gene_symbol <- trimws(
    as.character(frozen$gene_symbol)
)

frozen$gene_type <- trimws(
    as.character(frozen$gene_type)
)

frozen$gene_symbol[
    is.na(frozen$gene_symbol) |
    frozen$gene_symbol == "" |
    toupper(frozen$gene_symbol) %in% c("NA", "N/A", "NULL")
] <- NA_character_

if (anyDuplicated(frozen$ensemblID_stable) > 0L) {
    stop("Duplicate frozen stable Ensembl IDs detected.")
}

cat("Frozen 24-gene identity set: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10D.1D — PACKAGES
# --------------------------------------------------------------------------------------------------

needed_cran <- c(
    "httr",
    "jsonlite"
)

missing_cran <- needed_cran[
    !vapply(
        needed_cran,
        requireNamespace,
        quietly = TRUE,
        FUN.VALUE = logical(1)
    )
]

if (length(missing_cran) > 0L) {

    install.packages(
        missing_cran,
        repos = "https://cloud.r-project.org"
    )
}

if (!requireNamespace("AnnotationDbi", quietly = TRUE)) {
    stop("AnnotationDbi is unavailable.")
}

if (!requireNamespace("org.Hs.eg.db", quietly = TRUE)) {
    stop("org.Hs.eg.db is unavailable.")
}

if (!requireNamespace("httr", quietly = TRUE)) {
    stop("httr is unavailable.")
}

if (!requireNamespace("jsonlite", quietly = TRUE)) {
    stop("jsonlite is unavailable.")
}

# --------------------------------------------------------------------------------------------------
# 10D.1E — CURRENT LOCAL org.Hs.eg.db ANNOTATION
# --------------------------------------------------------------------------------------------------

org_raw <- AnnotationDbi::select(
    org.Hs.eg.db::org.Hs.eg.db,
    keys = frozen$ensemblID_stable,
    keytype = "ENSEMBL",
    columns = c(
        "SYMBOL",
        "GENENAME",
        "ENTREZID"
    )
)

names(org_raw)[
    names(org_raw) == "ENSEMBL"
] <- "ensemblID_stable"

collapse_unique <- function(x) {

    x <- unique(
        trimws(
            as.character(x)
        )
    )

    x <- x[
        !is.na(x) &
        nzchar(x)
    ]

    if (length(x) == 0L) {
        return(NA_character_)
    }

    paste(
        sort(x),
        collapse = ";"
    )
}

org_collapsed <- aggregate(
    cbind(
        SYMBOL,
        GENENAME,
        ENTREZID
    ) ~ ensemblID_stable,
    data = org_raw,
    FUN = collapse_unique
)

names(org_collapsed)[
    names(org_collapsed) == "SYMBOL"
] <- "orgdb_symbol"

names(org_collapsed)[
    names(org_collapsed) == "GENENAME"
] <- "orgdb_gene_name"

names(org_collapsed)[
    names(org_collapsed) == "ENTREZID"
] <- "orgdb_entrez"

# --------------------------------------------------------------------------------------------------
# 10D.1F — ENSEMBL REST LOOKUP FUNCTION
# --------------------------------------------------------------------------------------------------

ensembl_lookup <- function(ensembl_id) {

    url <- paste0(
        "https://rest.ensembl.org/lookup/id/",
        utils::URLencode(
            ensembl_id,
            reserved = TRUE
        )
    )

    result <- tryCatch(
        {

            response <- httr::GET(
                url,
                httr::add_headers(
                    Accept = "application/json"
                ),
                httr::timeout(30)
            )

            status <- httr::status_code(
                response
            )

            if (status != 200L) {

                return(
                    data.frame(
                        ensemblID_stable = ensembl_id,
                        ensembl_http_status = status,
                        ensembl_lookup_ok = FALSE,
                        ensembl_display_name = NA_character_,
                        ensembl_biotype = NA_character_,
                        ensembl_description = NA_character_,
                        ensembl_assembly = NA_character_,
                        ensembl_chromosome = NA_character_,
                        stringsAsFactors = FALSE
                    )
                )
            }

            payload <- jsonlite::fromJSON(
                httr::content(
                    response,
                    as = "text",
                    encoding = "UTF-8"
                ),
                simplifyVector = TRUE
            )

            data.frame(
                ensemblID_stable = ensembl_id,
                ensembl_http_status = status,
                ensembl_lookup_ok = TRUE,

                ensembl_display_name =
                    if (!is.null(payload$display_name)) {
                        as.character(payload$display_name)
                    } else {
                        NA_character_
                    },

                ensembl_biotype =
                    if (!is.null(payload$biotype)) {
                        as.character(payload$biotype)
                    } else {
                        NA_character_
                    },

                ensembl_description =
                    if (!is.null(payload$description)) {
                        as.character(payload$description)
                    } else {
                        NA_character_
                    },

                ensembl_assembly =
                    if (!is.null(payload$assembly_name)) {
                        as.character(payload$assembly_name)
                    } else {
                        NA_character_
                    },

                ensembl_chromosome =
                    if (!is.null(payload$seq_region_name)) {
                        as.character(payload$seq_region_name)
                    } else {
                        NA_character_
                    },

                stringsAsFactors = FALSE
            )
        },

        error = function(e) {

            data.frame(
                ensemblID_stable = ensembl_id,
                ensembl_http_status = NA_integer_,
                ensembl_lookup_ok = FALSE,
                ensembl_display_name = NA_character_,
                ensembl_biotype = NA_character_,
                ensembl_description = paste0(
                    "LOOKUP_ERROR: ",
                    conditionMessage(e)
                ),
                ensembl_assembly = NA_character_,
                ensembl_chromosome = NA_character_,
                stringsAsFactors = FALSE
            )
        }
    )

    Sys.sleep(0.15)

    result
}

# --------------------------------------------------------------------------------------------------
# 10D.1G — HGNC REST LOOKUP FUNCTION
# --------------------------------------------------------------------------------------------------

hgnc_lookup <- function(ensembl_id) {

    url <- paste0(
        "https://rest.genenames.org/search/ensembl_gene_id/",
        utils::URLencode(
            ensembl_id,
            reserved = TRUE
        )
    )

    result <- tryCatch(
        {

            response <- httr::GET(
                url,
                httr::add_headers(
                    Accept = "application/json"
                ),
                httr::timeout(30)
            )

            status <- httr::status_code(
                response
            )

            if (status != 200L) {

                return(
                    data.frame(
                        ensemblID_stable = ensembl_id,
                        hgnc_http_status = status,
                        hgnc_lookup_ok = FALSE,
                        hgnc_num_found = NA_integer_,
                        hgnc_id = NA_character_,
                        hgnc_symbol = NA_character_,
                        stringsAsFactors = FALSE
                    )
                )
            }

            payload <- jsonlite::fromJSON(
                httr::content(
                    response,
                    as = "text",
                    encoding = "UTF-8"
                ),
                simplifyVector = FALSE
            )

            response_obj <- payload$response

            num_found <- if (
                !is.null(response_obj$numFound)
            ) {
                as.integer(response_obj$numFound)
            } else {
                0L
            }

            if (
                num_found < 1L ||
                length(response_obj$docs) < 1L
            ) {

                return(
                    data.frame(
                        ensemblID_stable = ensembl_id,
                        hgnc_http_status = status,
                        hgnc_lookup_ok = TRUE,
                        hgnc_num_found = num_found,
                        hgnc_id = NA_character_,
                        hgnc_symbol = NA_character_,
                        stringsAsFactors = FALSE
                    )
                )
            }

            docs <- response_obj$docs

            hgnc_ids <- unique(
                vapply(
                    docs,
                    function(x) {
                        if (!is.null(x$hgnc_id)) {
                            as.character(x$hgnc_id)
                        } else {
                            NA_character_
                        }
                    },
                    FUN.VALUE = character(1)
                )
            )

            hgnc_symbols <- unique(
                vapply(
                    docs,
                    function(x) {
                        if (!is.null(x$symbol)) {
                            as.character(x$symbol)
                        } else {
                            NA_character_
                        }
                    },
                    FUN.VALUE = character(1)
                )
            )

            hgnc_ids <- hgnc_ids[
                !is.na(hgnc_ids)
            ]

            hgnc_symbols <- hgnc_symbols[
                !is.na(hgnc_symbols)
            ]

            data.frame(
                ensemblID_stable = ensembl_id,
                hgnc_http_status = status,
                hgnc_lookup_ok = TRUE,
                hgnc_num_found = num_found,

                hgnc_id =
                    if (length(hgnc_ids) > 0L) {
                        paste(
                            sort(hgnc_ids),
                            collapse = ";"
                        )
                    } else {
                        NA_character_
                    },

                hgnc_symbol =
                    if (length(hgnc_symbols) > 0L) {
                        paste(
                            sort(hgnc_symbols),
                            collapse = ";"
                        )
                    } else {
                        NA_character_
                    },

                stringsAsFactors = FALSE
            )
        },

        error = function(e) {

            data.frame(
                ensemblID_stable = ensembl_id,
                hgnc_http_status = NA_integer_,
                hgnc_lookup_ok = FALSE,
                hgnc_num_found = NA_integer_,
                hgnc_id = NA_character_,
                hgnc_symbol = NA_character_,
                stringsAsFactors = FALSE
            )
        }
    )

    Sys.sleep(0.15)

    result
}

# --------------------------------------------------------------------------------------------------
# 10D.1H — QUERY CURRENT ENSEMBL
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nQUERY CURRENT ENSEMBL IDENTITIES\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

ensembl_results <- do.call(
    rbind,
    lapply(
        frozen$ensemblID_stable,
        ensembl_lookup
    )
)

n_ensembl_ok <- sum(
    ensembl_results$ensembl_lookup_ok
)

cat(
    "Successful Ensembl lookups: ",
    n_ensembl_ok,
    "/24\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10D.1I — QUERY CURRENT HGNC
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nQUERY CURRENT HGNC NOMENCLATURE\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

hgnc_results <- do.call(
    rbind,
    lapply(
        frozen$ensemblID_stable,
        hgnc_lookup
    )
)

n_hgnc_http_ok <- sum(
    hgnc_results$hgnc_lookup_ok
)

n_hgnc_found <- sum(
    !is.na(
        hgnc_results$hgnc_symbol
    )
)

cat(
    "Successful HGNC HTTP lookups: ",
    n_hgnc_http_ok,
    "/24\n",
    sep = ""
)

cat(
    "Frozen Ensembl IDs with HGNC symbol record: ",
    n_hgnc_found,
    "/24\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10D.1J — MERGE ALL IDENTITY SOURCES
# --------------------------------------------------------------------------------------------------

audit <- merge(
    frozen,
    org_collapsed,
    by = "ensemblID_stable",
    all.x = TRUE,
    sort = FALSE
)

audit <- merge(
    audit,
    ensembl_results,
    by = "ensemblID_stable",
    all.x = TRUE,
    sort = FALSE
)

audit <- merge(
    audit,
    hgnc_results,
    by = "ensemblID_stable",
    all.x = TRUE,
    sort = FALSE
)

audit <- audit[
    match(
        frozen$ensemblID_stable,
        audit$ensemblID_stable
    ),
    ,
    drop = FALSE
]

if (!identical(
    audit$ensemblID_stable,
    frozen$ensemblID_stable
)) {
    stop(
        "Unable to restore original frozen 24-gene order."
    )
}

# --------------------------------------------------------------------------------------------------
# 10D.1K — NORMALIZE SYMBOL FIELDS
# --------------------------------------------------------------------------------------------------

normalize_symbol <- function(x) {

    x <- trimws(
        as.character(x)
    )

    x[
        is.na(x) |
        x == "" |
        toupper(x) %in%
            c(
                "NA",
                "N/A",
                "NULL"
            )
    ] <- NA_character_

    x
}

audit$gene_symbol <- normalize_symbol(
    audit$gene_symbol
)

audit$orgdb_symbol <- normalize_symbol(
    audit$orgdb_symbol
)

audit$ensembl_display_name <- normalize_symbol(
    audit$ensembl_display_name
)

audit$hgnc_symbol <- normalize_symbol(
    audit$hgnc_symbol
)

# --------------------------------------------------------------------------------------------------
# 10D.1L — SYMBOL AGREEMENT LOGIC
# --------------------------------------------------------------------------------------------------

exact_equal <- function(a, b) {

    if (
        is.na(a) ||
        is.na(b)
    ) {
        return(NA)
    }

    identical(
        as.character(a),
        as.character(b)
    )
}

audit$frozen_vs_orgdb_symbol_match <- mapply(
    exact_equal,
    audit$gene_symbol,
    audit$orgdb_symbol,
    USE.NAMES = FALSE
)

audit$frozen_vs_ensembl_symbol_match <- mapply(
    exact_equal,
    audit$gene_symbol,
    audit$ensembl_display_name,
    USE.NAMES = FALSE
)

audit$frozen_vs_hgnc_symbol_match <- mapply(
    exact_equal,
    audit$gene_symbol,
    audit$hgnc_symbol,
    USE.NAMES = FALSE
)

audit$ensembl_vs_hgnc_symbol_match <- mapply(
    exact_equal,
    audit$ensembl_display_name,
    audit$hgnc_symbol,
    USE.NAMES = FALSE
)

# --------------------------------------------------------------------------------------------------
# 10D.1M — BIOTYPE AGREEMENT
# --------------------------------------------------------------------------------------------------

normalize_biotype <- function(x) {

    x <- tolower(
        trimws(
            as.character(x)
        )
    )

    x[
        is.na(x) |
        x == ""
    ] <- NA_character_

    x
}

audit$frozen_gene_type_normalized <- normalize_biotype(
    audit$gene_type
)

audit$ensembl_biotype_normalized <- normalize_biotype(
    audit$ensembl_biotype
)

audit$frozen_vs_ensembl_biotype_match <- mapply(
    function(a, b) {

        if (
            is.na(a) ||
            is.na(b)
        ) {
            return(NA)
        }

        identical(a, b)
    },
    audit$frozen_gene_type_normalized,
    audit$ensembl_biotype_normalized,
    USE.NAMES = FALSE
)

# --------------------------------------------------------------------------------------------------
# 10D.1N — CURRENT CONSENSUS IDENTITY FOR INTERPRETATION
#
# This is DESCRIPTIVE ONLY.
# It never replaces the frozen feature name inside the score.
# --------------------------------------------------------------------------------------------------

audit$current_interpretation_symbol <- NA_character_
audit$current_interpretation_source <- NA_character_

for (i in seq_len(nrow(audit))) {

    ens <- audit$ensembl_display_name[i]
    hgnc <- audit$hgnc_symbol[i]
    org <- audit$orgdb_symbol[i]

    if (
        !is.na(hgnc) &&
        !is.na(ens) &&
        identical(hgnc, ens)
    ) {

        audit$current_interpretation_symbol[i] <- hgnc

        audit$current_interpretation_source[i] <-
            "HGNC_AND_ENSEMBL_AGREE"

    } else if (!is.na(hgnc)) {

        audit$current_interpretation_symbol[i] <- hgnc

        audit$current_interpretation_source[i] <-
            "HGNC_AVAILABLE_REVIEW_ENSEMBL"

    } else if (!is.na(ens)) {

        audit$current_interpretation_symbol[i] <- ens

        audit$current_interpretation_source[i] <-
            "ENSEMBL_ONLY"

    } else if (!is.na(org)) {

        audit$current_interpretation_symbol[i] <- org

        audit$current_interpretation_source[i] <-
            "ORGDB_ONLY"

    } else {

        audit$current_interpretation_symbol[i] <- NA_character_

        audit$current_interpretation_source[i] <-
            "NO_CURRENT_NAMED_IDENTITY"
    }
}

# --------------------------------------------------------------------------------------------------
# 10D.1O — REVIEW FLAGS
# --------------------------------------------------------------------------------------------------

audit$frozen_symbol_differs_from_current <- mapply(
    function(frozen_symbol, current_symbol) {

        if (
            is.na(frozen_symbol) ||
            is.na(current_symbol)
        ) {
            return(FALSE)
        }

        !identical(
            frozen_symbol,
            current_symbol
        )
    },
    audit$gene_symbol,
    audit$current_interpretation_symbol,
    USE.NAMES = FALSE
)

audit$current_biotype_differs_from_frozen <- ifelse(
    is.na(
        audit$frozen_vs_ensembl_biotype_match
    ),
    FALSE,
    !audit$frozen_vs_ensembl_biotype_match
)

audit$external_sources_disagree <- ifelse(
    !is.na(audit$ensembl_display_name) &
    !is.na(audit$hgnc_symbol),
    audit$ensembl_display_name !=
        audit$hgnc_symbol,
    FALSE
)

audit$manual_identity_review_required <- (
    audit$frozen_symbol_differs_from_current |
    audit$current_biotype_differs_from_frozen |
    audit$external_sources_disagree
)

# Unnamed frozen genes with current names are not automatically "errors";
# record them separately.
audit$frozen_unnamed_currently_named <- (
    is.na(audit$gene_symbol) &
    !is.na(
        audit$current_interpretation_symbol
    )
)

# --------------------------------------------------------------------------------------------------
# 10D.1P — INTERPRETATION-SAFETY CLASS
# --------------------------------------------------------------------------------------------------

audit$interpretation_safety_class <- ifelse(
    audit$manual_identity_review_required,
    "REVIEW_BEFORE_GENE_LEVEL_BIOLOGICAL_CLAIM",
    ifelse(
        is.na(
            audit$current_interpretation_symbol
        ),
        "ENSEMBL_ID_ONLY_INTERPRETATION",
        "CURRENT_IDENTITY_CONCORDANT_OR_NONCONFLICTING"
    )
)

# --------------------------------------------------------------------------------------------------
# 10D.1Q — WRITE OUTPUTS
# --------------------------------------------------------------------------------------------------

write.csv(
    audit,
    OUT_FULL,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

discordant <- audit[
    audit$manual_identity_review_required |
    audit$frozen_unnamed_currently_named,
    ,
    drop = FALSE
]

write.csv(
    discordant,
    OUT_DISCORDANT,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10D.1R — SUMMARY STATISTICS
# --------------------------------------------------------------------------------------------------

n_frozen_named <- sum(
    !is.na(audit$gene_symbol)
)

n_current_named <- sum(
    !is.na(
        audit$current_interpretation_symbol
    )
)

n_symbol_discordant <- sum(
    audit$frozen_symbol_differs_from_current
)

n_biotype_discordant <- sum(
    audit$current_biotype_differs_from_frozen
)

n_external_disagree <- sum(
    audit$external_sources_disagree
)

n_manual_review <- sum(
    audit$manual_identity_review_required
)

n_unnamed_now_named <- sum(
    audit$frozen_unnamed_currently_named
)

# --------------------------------------------------------------------------------------------------
# 10D.1S — HUMAN-READABLE SUMMARY
# --------------------------------------------------------------------------------------------------

review_lines <- if (nrow(discordant) == 0L) {

    "[NONE]"

} else {

    apply(
        discordant,
        1,
        function(x) {

            paste0(
                x["ensemblID_stable"],
                " | frozen=",
                ifelse(
                    is.na(x["gene_symbol"]) ||
                    x["gene_symbol"] == "",
                    "[UNNAMED]",
                    x["gene_symbol"]
                ),
                " | current=",
                ifelse(
                    is.na(x["current_interpretation_symbol"]) ||
                    x["current_interpretation_symbol"] == "",
                    "[UNNAMED]",
                    x["current_interpretation_symbol"]
                ),
                " | frozen_type=",
                x["gene_type"],
                " | Ensembl_type=",
                ifelse(
                    is.na(x["ensembl_biotype"]) ||
                    x["ensembl_biotype"] == "",
                    "[NA]",
                    x["ensembl_biotype"]
                ),
                " | source=",
                x["current_interpretation_source"],
                " | review=",
                x["manual_identity_review_required"]
            )
        }
    )
}

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10D.1 — SCZ-MSS 24-GENE CURRENT IDENTITY INTEGRITY AUDIT",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "IMPORTANT GOVERNANCE",

    paste(
        "Frozen BrainSeq stable Ensembl IDs, directions and weights remain unchanged."
    ),

    paste(
        "Current external annotations are used only to ensure accurate biological",
        "description and literature interpretation."
    ),

    "",

    "LOOKUP COMPLETENESS",

    paste0(
        "Ensembl REST successful: ",
        n_ensembl_ok,
        "/24"
    ),

    paste0(
        "HGNC REST HTTP successful: ",
        n_hgnc_http_ok,
        "/24"
    ),

    paste0(
        "HGNC named records found: ",
        n_hgnc_found,
        "/24"
    ),

    "",

    "IDENTITY SUMMARY",

    paste0(
        "Frozen features with gene symbol: ",
        n_frozen_named,
        "/24"
    ),

    paste0(
        "Features with current interpretation symbol: ",
        n_current_named,
        "/24"
    ),

    paste0(
        "Frozen/current symbol discrepancies: ",
        n_symbol_discordant
    ),

    paste0(
        "Frozen/current Ensembl-biotype discrepancies: ",
        n_biotype_discordant
    ),

    paste0(
        "Current Ensembl/HGNC symbol disagreements: ",
        n_external_disagree
    ),

    paste0(
        "Frozen unnamed features now carrying a current name: ",
        n_unnamed_now_named
    ),

    paste0(
        "Features requiring manual identity review before biological claims: ",
        n_manual_review
    ),

    "",

    "FEATURES REQUIRING REVIEW / DOCUMENTATION",

    review_lines,

    "",

    "NEXT RULE",

    paste(
        "Gene-level literature interpretation should use the stable Ensembl ID",
        "as the anchor and current approved identity only after this audit is resolved."
    ),

    paste(
        "Historical frozen symbols may still be reported in analysis tables where necessary,",
        "but manuscript biological claims must not silently attach the biology of a different",
        "current gene to an outdated or discordant identifier."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10D.1T — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    STEP10A_PATH,
    STEP10C1R_PATH,
    OUT_FULL,
    OUT_DISCORDANT,
    OUT_SUMMARY
)

manifest <- data.frame(
    file = basename(manifest_paths),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(manifest_paths)$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10D.1U — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nCURRENT IDENTITY SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Ensembl REST successful: ",
    n_ensembl_ok,
    "/24\n",
    sep = ""
)

cat(
    "HGNC named records found: ",
    n_hgnc_found,
    "/24\n",
    sep = ""
)

cat(
    "Frozen/current symbol discrepancies: ",
    n_symbol_discordant,
    "\n",
    sep = ""
)

cat(
    "Frozen/current Ensembl-biotype discrepancies: ",
    n_biotype_discordant,
    "\n",
    sep = ""
)

cat(
    "Current Ensembl/HGNC disagreements: ",
    n_external_disagree,
    "\n",
    sep = ""
)

cat(
    "Manual identity review required: ",
    n_manual_review,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\n24-GENE IDENTITY TABLE\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    audit[
        ,
        c(
            "ensemblID_stable",
            "gene_symbol",
            "gene_type",
            "orgdb_symbol",
            "ensembl_display_name",
            "ensembl_biotype",
            "hgnc_symbol",
            "current_interpretation_symbol",
            "current_interpretation_source",
            "frozen_symbol_differs_from_current",
            "current_biotype_differs_from_frozen",
            "manual_identity_review_required"
        )
    ],
    row.names = FALSE
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nIDENTITY REVIEW SET\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

if (nrow(discordant) == 0L) {

    cat("[NONE]\n")

} else {

    print(
        discordant[
            ,
            c(
                "ensemblID_stable",
                "gene_symbol",
                "gene_type",
                "orgdb_symbol",
                "ensembl_display_name",
                "ensembl_biotype",
                "hgnc_symbol",
                "current_interpretation_symbol",
                "current_interpretation_source",
                "manual_identity_review_required"
            )
        ],
        row.names = FALSE
    )
}

cat(
    "\nCanonical identity audit:\n",
    OUT_FULL,
    "\n",
    sep = ""
)

cat(
    "\nIdentity review table:\n",
    OUT_DISCORDANT,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10D.1 COMPLETE — CURRENT IDENTITY INTEGRITY AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10D.1 — FROZEN 24-GENE CURRENT IDENTITY INTEGRITY AUDIT
Project sentinel: PASS
STEP10A frozen annotation: PASS
STEP10C.1R corrected scaffold: PASS
Frozen 24-gene identity set: PASS


'select()' returned 1:1 mapping between keys and columns




--------------------------------------------------------------------------------------------------------------
QUERY CURRENT ENSEMBL IDENTITIES
--------------------------------------------------------------------------------------------------------------
Successful Ensembl lookups: 24/24

--------------------------------------------------------------------------------------------------------------
QUERY CURRENT HGNC NOMENCLATURE
--------------------------------------------------------------------------------------------------------------
Successful HGNC HTTP lookups: 24/24
Frozen Ensembl IDs with HGNC symbol record: 18/24

CURRENT IDENTITY SUMMARY
Ensembl REST successful: 24/24
HGNC named records found: 18/24
Frozen/current symbol discrepancies: 2
Frozen/current Ensembl-biotype discrepancies: 5
Current Ensembl/HGNC disagreements: 0
Manual identity review required: 6

--------------------------------------------------------------------------------------------------------------
24-GENE 

In [67]:
# ==================================================================================================
# STEP 10D.2 — INTERPRETATION-SAFE IDENTITY ADJUDICATION FREEZE
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Convert STEP10D.1 identity audit into a manuscript-safe biological interpretation table.
#
# GOVERNANCE
#   - Frozen 24-gene score remains completely unchanged.
#   - Frozen Ensembl IDs remain the analysis identifiers.
#   - Frozen weights/directions remain unchanged.
#   - Current identities are ONLY interpretation metadata.
#
# SPECIAL CASES
#
#   ENSG00000117877
#       Frozen symbol: CD3EAP
#       Current official identity: POLR1G
#       Interpretation: same gene / historical alias -> current official symbol
#
#   ENSG00000187951
#       Frozen BrainSeq identity: ARHGAP11B
#       Historical Ensembl/GRCh37 identity corresponds to ARHGAP11B.
#       Current ARHGAP11B Ensembl ID is ENSG00000285077.
#       Therefore:
#           - retain frozen feature exactly
#           - biological interpretation anchored to historical ARHGAP11B identity
#           - NEVER attach current ENSG00000187951 biology to the frozen feature
#
#   Non-HGNC Ensembl display names such as Y_RNA, Metazoa_SRP, U7:
#       retain Ensembl ID as primary interpretation anchor;
#       display names are descriptive only.
#
# OUTPUT
#   A canonical 24-gene identity adjudication table for all subsequent
#   literature / brain / cell-type / disease interpretation.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10D.2 — INTERPRETATION-SAFE IDENTITY ADJUDICATION FREEZE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10D.2A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

IDENTITY_AUDIT_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "identity_audit_v1",
    "STEP10D1_SCZ_MSS_24gene_current_identity_audit_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "identity_audit_v1",
    "adjudicated_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_TABLE <- file.path(
    OUT_DIR,
    "STEP10D2_SCZ_MSS_24gene_interpretation_identity_v1.csv"
)

OUT_REVIEW <- file.path(
    OUT_DIR,
    "STEP10D2_SCZ_MSS_special_identity_cases_v1.csv"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP10D2_SCZ_MSS_identity_adjudication_summary_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP10D2_SCZ_MSS_identity_adjudication_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10D.2B — PROJECT / INPUT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

if (!file.exists(IDENTITY_AUDIT_PATH)) {
    stop(
        "STEP10D.1 identity audit missing:\n",
        IDENTITY_AUDIT_PATH
    )
}

audit <- read.csv(
    IDENTITY_AUDIT_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(audit) != 24L) {
    stop(
        "Expected 24 frozen features; observed ",
        nrow(audit)
    )
}

if (anyDuplicated(audit$ensemblID_stable) > 0L) {
    stop("Duplicate frozen Ensembl IDs detected.")
}

cat("STEP10D.1 identity audit: PASS\n")
cat("Frozen identity rows: 24 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10D.2C — REQUIRED COLUMNS
# --------------------------------------------------------------------------------------------------

required_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "protein_coding_flag",
    "direction_standardized",
    "final_normalized_weight",
    "orgdb_symbol",
    "ensembl_display_name",
    "ensembl_biotype",
    "hgnc_symbol",
    "current_interpretation_symbol",
    "current_interpretation_source"
)

missing_cols <- setdiff(
    required_cols,
    names(audit)
)

if (length(missing_cols) > 0L) {
    stop(
        "Identity audit missing required columns: ",
        paste(missing_cols, collapse = ", ")
    )
}

clean_text <- function(x) {

    x <- trimws(
        as.character(x)
    )

    x[
        is.na(x) |
        x == "" |
        toupper(x) %in% c(
            "NA",
            "N/A",
            "NULL"
        )
    ] <- NA_character_

    x
}

for (nm in c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "orgdb_symbol",
    "ensembl_display_name",
    "ensembl_biotype",
    "hgnc_symbol",
    "current_interpretation_symbol",
    "current_interpretation_source"
)) {

    audit[[nm]] <- clean_text(
        audit[[nm]]
    )
}

# --------------------------------------------------------------------------------------------------
# 10D.2D — INITIAL INTERPRETATION IDENTITIES
# --------------------------------------------------------------------------------------------------

out <- audit

out$analysis_identity <- out$ensemblID_stable

out$interpretation_symbol <- NA_character_

out$interpretation_identity_class <- NA_character_

out$literature_search_anchor <- NA_character_

out$manuscript_gene_label <- NA_character_

out$current_ensembl_id_if_different <- NA_character_

out$identity_adjudication_note <- NA_character_

out$allow_gene_specific_biological_claim <- FALSE

out$use_current_same_ensembl_record_for_biology <- TRUE

# --------------------------------------------------------------------------------------------------
# 10D.2E — DEFAULT ADJUDICATION RULES
# --------------------------------------------------------------------------------------------------

for (i in seq_len(nrow(out))) {

    frozen_id <- out$ensemblID_stable[i]
    frozen_symbol <- out$gene_symbol[i]
    hgnc_symbol <- out$hgnc_symbol[i]
    ens_name <- out$ensembl_display_name[i]
    source <- out$current_interpretation_source[i]

    # ----------------------------------------------------------------------------------------------
    # CASE 1 — HGNC + current Ensembl agree
    # ----------------------------------------------------------------------------------------------

    if (
        !is.na(hgnc_symbol) &&
        !is.na(ens_name) &&
        identical(hgnc_symbol, ens_name)
    ) {

        out$interpretation_symbol[i] <- hgnc_symbol

        if (
            !is.na(frozen_symbol) &&
            identical(frozen_symbol, hgnc_symbol)
        ) {

            out$interpretation_identity_class[i] <-
                "CURRENT_IDENTITY_CONCORDANT"

            out$identity_adjudication_note[i] <-
                "Frozen symbol agrees with current HGNC/Ensembl identity."

        } else if (!is.na(frozen_symbol)) {

            out$interpretation_identity_class[i] <-
                "HISTORICAL_OR_ALIAS_SYMBOL_CURRENT_IDENTITY_RESOLVED"

            out$identity_adjudication_note[i] <- paste0(
                "Frozen symbol ",
                frozen_symbol,
                " differs from current HGNC/Ensembl symbol ",
                hgnc_symbol,
                "; stable feature identity otherwise retained."
            )

        } else {

            out$interpretation_identity_class[i] <-
                "FROZEN_UNNAMED_CURRENT_HGNC_IDENTITY_AVAILABLE"

            out$identity_adjudication_note[i] <- paste0(
                "Frozen feature had no symbol; current HGNC/Ensembl identity is ",
                hgnc_symbol,
                "."
            )
        }

        out$literature_search_anchor[i] <- paste(
            hgnc_symbol,
            frozen_id
        )

        out$manuscript_gene_label[i] <- paste0(
            hgnc_symbol,
            " (",
            frozen_id,
            ")"
        )

        out$allow_gene_specific_biological_claim[i] <- TRUE
    }

    # ----------------------------------------------------------------------------------------------
    # CASE 2 — Ensembl display name but no HGNC record
    # ----------------------------------------------------------------------------------------------

    else if (
        is.na(hgnc_symbol) &&
        !is.na(ens_name)
    ) {

        out$interpretation_symbol[i] <- ens_name

        out$interpretation_identity_class[i] <-
            "ENSEMBL_DISPLAY_NAME_NO_HGNC_APPROVED_SYMBOL"

        out$literature_search_anchor[i] <- frozen_id

        out$manuscript_gene_label[i] <- paste0(
            frozen_id,
            " [Ensembl display: ",
            ens_name,
            "]"
        )

        out$identity_adjudication_note[i] <- paste(
            "Current Ensembl provides a display name but no HGNC symbol was found;",
            "stable Ensembl ID remains the primary interpretation anchor."
        )

        out$allow_gene_specific_biological_claim[i] <- FALSE
    }

    # ----------------------------------------------------------------------------------------------
    # CASE 3 — no current named identity
    # ----------------------------------------------------------------------------------------------

    else if (
        is.na(hgnc_symbol) &&
        is.na(ens_name)
    ) {

        out$interpretation_symbol[i] <- NA_character_

        out$interpretation_identity_class[i] <-
            "ENSEMBL_ID_ONLY_NO_CURRENT_APPROVED_NAME"

        out$literature_search_anchor[i] <- frozen_id

        out$manuscript_gene_label[i] <- frozen_id

        out$identity_adjudication_note[i] <- paste(
            "No current HGNC-approved symbol or Ensembl display name;",
            "interpret only at stable-ID / feature-class level unless independently resolved."
        )

        out$allow_gene_specific_biological_claim[i] <- FALSE
    }
}

# --------------------------------------------------------------------------------------------------
# 10D.2F — SPECIAL ADJUDICATION: ENSG00000117877
#
# Historical frozen symbol CD3EAP; current official symbol POLR1G.
# Same underlying gene identity.
# --------------------------------------------------------------------------------------------------

idx_polr1g <- match(
    "ENSG00000117877",
    out$ensemblID_stable
)

if (is.na(idx_polr1g)) {
    stop("Expected frozen ENSG00000117877 not found.")
}

if (
    !identical(
        out$hgnc_symbol[idx_polr1g],
        "POLR1G"
    ) ||
    !identical(
        out$ensembl_display_name[idx_polr1g],
        "POLR1G"
    )
) {
    stop(
        "Current identity of ENSG00000117877 does not match expected POLR1G."
    )
}

out$interpretation_symbol[idx_polr1g] <- "POLR1G"

out$interpretation_identity_class[idx_polr1g] <-
    "HISTORICAL_ALIAS_CD3EAP_CURRENT_OFFICIAL_POLR1G"

out$literature_search_anchor[idx_polr1g] <-
    "POLR1G OR CD3EAP OR ENSG00000117877"

out$manuscript_gene_label[idx_polr1g] <-
    "POLR1G (frozen annotation: CD3EAP; ENSG00000117877)"

out$identity_adjudication_note[idx_polr1g] <- paste(
    "Frozen BrainSeq symbol CD3EAP is treated as a historical/alias designation",
    "of the same feature currently approved as POLR1G.",
    "Biological interpretation should use POLR1G while preserving CD3EAP in frozen-analysis tables."
)

out$allow_gene_specific_biological_claim[idx_polr1g] <- TRUE

out$use_current_same_ensembl_record_for_biology[idx_polr1g] <- TRUE

# --------------------------------------------------------------------------------------------------
# 10D.2G — SPECIAL ADJUDICATION: HISTORICAL ARHGAP11B FEATURE
#
# Frozen:
#     ENSG00000187951 / ARHGAP11B
#
# Current human ARHGAP11B:
#     ENSG00000285077
#
# Historical Ensembl GRCh37 records associate ENSG00000187951 with ARHGAP11B.
#
# Therefore this frozen BrainSeq feature is biologically interpreted according to its
# HISTORICAL BrainSeq/GRCh37 annotation, NOT according to the current record now associated
# with ENSG00000187951.
# --------------------------------------------------------------------------------------------------

idx_arhgap <- match(
    "ENSG00000187951",
    out$ensemblID_stable
)

if (is.na(idx_arhgap)) {
    stop("Expected frozen ENSG00000187951 not found.")
}

if (
    !identical(
        out$gene_symbol[idx_arhgap],
        "ARHGAP11B"
    )
) {
    stop(
        "Frozen ENSG00000187951 is no longer labelled ARHGAP11B in the input; ",
        "manual re-review required."
    )
}

out$interpretation_symbol[idx_arhgap] <- "ARHGAP11B"

out$interpretation_identity_class[idx_arhgap] <-
    "HISTORICAL_ENSEMBL_ID_TRANSITION_ARHGAP11B"

out$current_ensembl_id_if_different[idx_arhgap] <-
    "ENSG00000285077"

out$literature_search_anchor[idx_arhgap] <-
    "ARHGAP11B ENSG00000285077 ENSG00000187951"

out$manuscript_gene_label[idx_arhgap] <-
    "ARHGAP11B (historical BrainSeq ID ENSG00000187951; current Ensembl ID ENSG00000285077)"

out$identity_adjudication_note[idx_arhgap] <- paste(
    "Frozen BrainSeq/GRCh37 annotation identifies ENSG00000187951 as ARHGAP11B.",
    "Current ARHGAP11B uses Ensembl ID ENSG00000285077.",
    "Do not interpret the frozen feature using the current biological annotation",
    "attached to ENSG00000187951."
)

out$allow_gene_specific_biological_claim[idx_arhgap] <- TRUE

out$use_current_same_ensembl_record_for_biology[idx_arhgap] <- FALSE

# --------------------------------------------------------------------------------------------------
# 10D.2H — EXPLICIT BIOTYPE-MODERNIZATION CLASSIFICATION
#
# These do NOT alter gene identity.
# --------------------------------------------------------------------------------------------------

biotype_modernization_ids <- c(
    "ENSG00000260526",  # frozen antisense -> current lncRNA / AP1AR-DT
    "ENSG00000259363",  # frozen lincRNA -> current lncRNA
    "ENSG00000260075",  # NSFP1 pseudogene subtype update
    "ENSG00000228914"   # OR1H1P pseudogene subtype update
)

for (id in biotype_modernization_ids) {

    i <- match(
        id,
        out$ensemblID_stable
    )

    if (is.na(i)) {
        stop(
            "Expected feature missing during biotype adjudication: ",
            id
        )
    }

    old_note <- out$identity_adjudication_note[i]

    out$identity_adjudication_note[i] <- paste(
        old_note,
        paste0(
            "Frozen gene_type=",
            out$gene_type[i],
            "; current Ensembl biotype=",
            out$ensembl_biotype[i],
            ". Treated as annotation/biotype modernization, not feature replacement."
        )
    )
}

# --------------------------------------------------------------------------------------------------
# 10D.2I — ENSEMBL-ONLY NONCODING DISPLAY NAMES
#
# These are useful descriptors, but not treated as HGNC-approved gene symbols.
# --------------------------------------------------------------------------------------------------

ensembl_only_ids <- c(
    "ENSG00000277118",  # Metazoa_SRP
    "ENSG00000200737",  # Y_RNA
    "ENSG00000273629"   # U7
)

for (id in ensembl_only_ids) {

    i <- match(
        id,
        out$ensemblID_stable
    )

    if (is.na(i)) {
        stop(
            "Expected Ensembl-only noncoding feature missing: ",
            id
        )
    }

    out$interpretation_identity_class[i] <-
        "ENSEMBL_NONCODING_DISPLAY_NAME_NO_HGNC_APPROVAL"

    out$allow_gene_specific_biological_claim[i] <- FALSE

    out$literature_search_anchor[i] <-
        out$ensemblID_stable[i]

    out$identity_adjudication_note[i] <- paste(
        "Current Ensembl display name:",
        ifelse(
            is.na(out$ensembl_display_name[i]),
            "[NONE]",
            out$ensembl_display_name[i]
        ),
        ". No HGNC-approved symbol was found; use stable Ensembl ID as the primary anchor."
    )
}

# --------------------------------------------------------------------------------------------------
# 10D.2J — NO-NAME FEATURES
# --------------------------------------------------------------------------------------------------

no_current_name <- (
    is.na(out$hgnc_symbol) &
    is.na(out$ensembl_display_name)
)

out$allow_gene_specific_biological_claim[
    no_current_name
] <- FALSE

# --------------------------------------------------------------------------------------------------
# 10D.2K — LITERATURE-INTERPRETATION TIERS
# --------------------------------------------------------------------------------------------------

out$literature_interpretation_tier <- ifelse(
    out$ensemblID_stable == "ENSG00000187951",
    "TIER_B_HISTORICAL_IDENTITY_SPECIAL_HANDLING",
    ifelse(
        out$allow_gene_specific_biological_claim,
        "TIER_A_NAMED_IDENTITY_GENE_LEVEL_REVIEW",
        ifelse(
            !is.na(out$ensembl_display_name),
            "TIER_C_ID_ANCHORED_NONCODING_FEATURE_REVIEW",
            "TIER_D_ID_ONLY_LIMITED_INTERPRETATION"
        )
    )
)

# --------------------------------------------------------------------------------------------------
# 10D.2L — VERIFY FROZEN ANALYSIS FIELDS REMAIN UNCHANGED
# --------------------------------------------------------------------------------------------------

if (
    !identical(
        out$ensemblID_stable,
        audit$ensemblID_stable
    )
) {
    stop("Frozen Ensembl IDs were altered.")
}

if (
    !isTRUE(
        all.equal(
            out$final_normalized_weight,
            audit$final_normalized_weight,
            tolerance = 0
        )
    )
) {
    stop("Frozen weights were altered.")
}

if (
    !identical(
        out$direction_standardized,
        audit$direction_standardized
    )
) {
    stop("Frozen directions were altered.")
}

cat("Frozen Ensembl IDs unchanged: PASS\n")
cat("Frozen weights unchanged: PASS\n")
cat("Frozen directions unchanged: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10D.2M — SUMMARY COUNTS
# --------------------------------------------------------------------------------------------------

tier_counts <- table(
    out$literature_interpretation_tier
)

n_gene_level_safe <- sum(
    out$allow_gene_specific_biological_claim
)

n_id_anchored <- sum(
    !out$allow_gene_specific_biological_claim
)

n_historical_special <- sum(
    out$literature_interpretation_tier ==
        "TIER_B_HISTORICAL_IDENTITY_SPECIAL_HANDLING"
)

# --------------------------------------------------------------------------------------------------
# 10D.2N — SPECIAL CASE TABLE
# --------------------------------------------------------------------------------------------------

special_case <- (
    out$interpretation_identity_class !=
        "CURRENT_IDENTITY_CONCORDANT"
)

special <- out[
    special_case,
    ,
    drop = FALSE
]

# --------------------------------------------------------------------------------------------------
# 10D.2O — WRITE OUTPUTS
# --------------------------------------------------------------------------------------------------

write.csv(
    out,
    OUT_TABLE,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    special,
    OUT_REVIEW,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10D.2P — HUMAN-READABLE SUMMARY
# --------------------------------------------------------------------------------------------------

tier_lines <- paste0(
    names(tier_counts),
    ": ",
    as.integer(tier_counts)
)

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10D.2 — SCZ-MSS INTERPRETATION-SAFE IDENTITY ADJUDICATION",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "GOVERNANCE",

    "Frozen 24-gene membership: UNCHANGED",
    "Frozen Ensembl analysis IDs: UNCHANGED",
    "Frozen directions: UNCHANGED",
    "Frozen score weights: UNCHANGED",

    "",

    "IDENTITY ADJUDICATION",

    paste0(
        "Features permitted for named gene-level biological review: ",
        n_gene_level_safe,
        "/24"
    ),

    paste0(
        "Features requiring Ensembl-ID-anchored limited interpretation: ",
        n_id_anchored,
        "/24"
    ),

    paste0(
        "Historical Ensembl-ID transition requiring special handling: ",
        n_historical_special
    ),

    "",

    "LITERATURE INTERPRETATION TIERS",

    tier_lines,

    "",

    "KEY SPECIAL CASE — POLR1G / CD3EAP",

    paste(
        "Frozen ENSG00000117877 is retained exactly.",
        "Frozen symbol CD3EAP is treated as a historical/alias designation.",
        "Current biological interpretation uses official symbol POLR1G."
    ),

    "",

    "KEY SPECIAL CASE — ARHGAP11B",

    paste(
        "Frozen BrainSeq feature ENSG00000187951 was historically annotated as ARHGAP11B."
    ),

    paste(
        "Current ARHGAP11B uses Ensembl ID ENSG00000285077."
    ),

    paste(
        "The frozen feature remains unchanged for all analyses, but gene-level biology",
        "must be interpreted using the historical ARHGAP11B identity rather than the",
        "current annotation attached to ENSG00000187951."
    ),

    "",

    "NONCODING INTERPRETATION POLICY",

    paste(
        "Features without HGNC-approved symbols are interpreted primarily by their",
        "frozen stable Ensembl IDs and molecular feature classes."
    ),

    paste(
        "Ensembl-only display names may be reported descriptively but are not treated",
        "as independently validated HGNC gene identities."
    ),

    "",

    "NEXT ANALYSIS",

    paste(
        "Use this adjudicated table as the sole identity source for evidence-based",
        "gene-level literature, brain/cell-type, schizophrenia, and genetic-convergence review."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10D.2Q — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    IDENTITY_AUDIT_PATH,
    OUT_TABLE,
    OUT_REVIEW,
    OUT_SUMMARY
)

manifest <- data.frame(
    file = basename(manifest_paths),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(manifest_paths)$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10D.2R — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nINTERPRETATION IDENTITY SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Named gene-level biological review permitted: ",
    n_gene_level_safe,
    "/24\n",
    sep = ""
)

cat(
    "ID-anchored limited interpretation: ",
    n_id_anchored,
    "/24\n",
    sep = ""
)

cat(
    "Historical-ID special cases: ",
    n_historical_special,
    "\n",
    sep = ""
)

cat(
    "\nINTERPRETATION TIERS\n"
)

print(
    tier_counts
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\n24-GENE ADJUDICATED IDENTITY TABLE\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    out[
        ,
        c(
            "ensemblID_stable",
            "gene_symbol",
            "interpretation_symbol",
            "interpretation_identity_class",
            "current_ensembl_id_if_different",
            "literature_interpretation_tier",
            "allow_gene_specific_biological_claim",
            "manuscript_gene_label"
        )
    ],
    row.names = FALSE
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nSPECIAL IDENTITY CASES\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    special[
        ,
        c(
            "ensemblID_stable",
            "gene_symbol",
            "gene_type",
            "interpretation_symbol",
            "interpretation_identity_class",
            "current_ensembl_id_if_different",
            "identity_adjudication_note"
        )
    ],
    row.names = FALSE
)

cat(
    "\nCanonical interpretation identity table:\n",
    OUT_TABLE,
    "\n",
    sep = ""
)

cat(
    "\nSpecial identity cases:\n",
    OUT_REVIEW,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10D.2 COMPLETE — INTERPRETATION IDENTITIES FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10D.2 — INTERPRETATION-SAFE IDENTITY ADJUDICATION FREEZE
Project sentinel: PASS
STEP10D.1 identity audit: PASS
Frozen identity rows: 24 — PASS
Frozen Ensembl IDs unchanged: PASS
Frozen weights unchanged: PASS
Frozen directions unchanged: PASS

INTERPRETATION IDENTITY SUMMARY
Named gene-level biological review permitted: 18/24
ID-anchored limited interpretation: 6/24
Historical-ID special cases: 1

INTERPRETATION TIERS

    TIER_A_NAMED_IDENTITY_GENE_LEVEL_REVIEW 
                                         18 
TIER_B_HISTORICAL_IDENTITY_SPECIAL_HANDLING 
                                          1 
TIER_C_ID_ANCHORED_NONCODING_FEATURE_REVIEW 
                                          3 
      TIER_D_ID_ONLY_LIMITED_INTERPRETATION 
                                          2 

--------------------------------------------------------------------------------------------------------------
24-GENE ADJUDICATED IDENTITY TABLE
----------------------------------------------------------------

In [68]:
# ==================================================================================================
# STEP 10E.1 — SYSTEMATIC GENE-LEVEL LITERATURE EVIDENCE RETRIEVAL
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Build a reproducible literature evidence corpus for the adjudicated frozen 24-gene SCZ-MSS state.
#
# DATABASE
#   Europe PMC REST API
#
# SEARCH DOMAINS
#   1. SCHIZOPHRENIA_PSYCHOSIS
#   2. BRAIN_NEURAL_CELLTYPE
#   3. SCZ_GENETIC_CONVERGENCE
#
# IMPORTANT
#   - This step RETRIEVES evidence only.
#   - It does NOT assign biological conclusions.
#   - It does NOT claim schizophrenia relevance from search counts alone.
#   - It does NOT modify frozen features, weights, directions, or identities.
#   - Tier C/D features remain ID-anchored and will not be promoted to named-gene biology.
#
# QUERY SAFETY
#   - Named genes are searched using exact TITLE_ABS terms.
#   - POLR1G includes historical alias CD3EAP.
#   - ARHGAP11B is searched as ARHGAP11B, while preserving its historical-ID warning.
#   - ID-limited features use stable Ensembl ID and, where appropriate, descriptive Ensembl name.
#
# OUTPUT
#   1. Query registry
#   2. Per-gene/per-domain hit counts
#   3. Top literature records with abstracts
#   4. Literature retrieval summary
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10E.1 — SYSTEMATIC GENE-LEVEL LITERATURE EVIDENCE RETRIEVAL\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.1A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

IDENTITY_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "identity_audit_v1",
    "adjudicated_v1",
    "STEP10D2_SCZ_MSS_24gene_interpretation_identity_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

QUERY_REGISTRY_PATH <- file.path(
    OUT_DIR,
    "STEP10E1_literature_query_registry_v1.csv"
)

HIT_COUNT_PATH <- file.path(
    OUT_DIR,
    "STEP10E1_gene_domain_literature_hit_counts_v1.csv"
)

TOP_RECORDS_PATH <- file.path(
    OUT_DIR,
    "STEP10E1_top_literature_records_with_abstracts_v1.csv"
)

GENE_SUMMARY_PATH <- file.path(
    OUT_DIR,
    "STEP10E1_gene_level_literature_summary_v1.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "STEP10E1_literature_retrieval_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "STEP10E1_literature_retrieval_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10E.1B — PROJECT / INPUT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID, "\n",
        "Observed: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

if (!file.exists(IDENTITY_PATH)) {
    stop(
        "STEP10D.2 adjudicated identity table missing:\n",
        IDENTITY_PATH
    )
}

identity <- read.csv(
    IDENTITY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(identity) != 24L) {
    stop(
        "Expected 24 adjudicated frozen features; observed ",
        nrow(identity)
    )
}

if (anyDuplicated(identity$ensemblID_stable) > 0L) {
    stop("Duplicate stable Ensembl IDs detected.")
}

required_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "interpretation_symbol",
    "interpretation_identity_class",
    "current_ensembl_id_if_different",
    "literature_interpretation_tier",
    "allow_gene_specific_biological_claim",
    "direction_standardized",
    "final_normalized_weight",
    "manuscript_gene_label"
)

missing_cols <- setdiff(
    required_cols,
    names(identity)
)

if (length(missing_cols) > 0L) {
    stop(
        "STEP10D.2 table missing required column(s): ",
        paste(missing_cols, collapse = ", ")
    )
}

cat("STEP10D.2 adjudicated identities: PASS\n")
cat("Frozen feature count: 24 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10E.1C — REQUIRED PACKAGES
# --------------------------------------------------------------------------------------------------

needed <- c(
    "httr",
    "jsonlite"
)

missing_packages <- needed[
    !vapply(
        needed,
        requireNamespace,
        quietly = TRUE,
        FUN.VALUE = logical(1)
    )
]

if (length(missing_packages) > 0L) {

    install.packages(
        missing_packages,
        repos = "https://cloud.r-project.org"
    )
}

if (!requireNamespace("httr", quietly = TRUE)) {
    stop("httr unavailable.")
}

if (!requireNamespace("jsonlite", quietly = TRUE)) {
    stop("jsonlite unavailable.")
}

# --------------------------------------------------------------------------------------------------
# 10E.1D — CLEAN TEXT FIELDS
# --------------------------------------------------------------------------------------------------

clean_text <- function(x) {

    x <- trimws(
        as.character(x)
    )

    x[
        is.na(x) |
        x == "" |
        toupper(x) %in% c(
            "NA",
            "N/A",
            "NULL"
        )
    ] <- NA_character_

    x
}

for (nm in c(
    "ensemblID_stable",
    "gene_symbol",
    "interpretation_symbol",
    "current_ensembl_id_if_different",
    "literature_interpretation_tier",
    "manuscript_gene_label"
)) {

    identity[[nm]] <- clean_text(
        identity[[nm]]
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.1E — STRICT LOGICAL NORMALIZATION
# --------------------------------------------------------------------------------------------------

if (!is.logical(identity$allow_gene_specific_biological_claim)) {

    z <- toupper(
        trimws(
            as.character(
                identity$allow_gene_specific_biological_claim
            )
        )
    )

    if (!all(z %in% c("TRUE", "FALSE"))) {
        stop(
            "Cannot interpret allow_gene_specific_biological_claim."
        )
    }

    identity$allow_gene_specific_biological_claim <- (
        z == "TRUE"
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.1F — BUILD SEARCH TERMS
#
# Named Tier A/B:
#   interpretation symbol is primary.
#
# POLR1G:
#   include historical alias CD3EAP.
#
# ARHGAP11B:
#   search ARHGAP11B only as biological identity;
#   current/historical Ensembl IDs remain metadata.
#
# Tier C:
#   stable Ensembl ID + Ensembl descriptive display name if present.
#
# Tier D:
#   stable Ensembl ID only.
# --------------------------------------------------------------------------------------------------

build_search_terms <- function(i) {

    stable_id <- identity$ensemblID_stable[i]
    frozen_symbol <- identity$gene_symbol[i]
    current_symbol <- identity$interpretation_symbol[i]
    tier <- identity$literature_interpretation_tier[i]

    terms <- character(0)

    # ----------------------------------------------------------------------------------------------
    # Tier A named identity
    # ----------------------------------------------------------------------------------------------

    if (
        identical(
            tier,
            "TIER_A_NAMED_IDENTITY_GENE_LEVEL_REVIEW"
        )
    ) {

        if (!is.na(current_symbol)) {
            terms <- c(
                terms,
                current_symbol
            )
        }

        if (
            !is.na(frozen_symbol) &&
            !identical(
                frozen_symbol,
                current_symbol
            )
        ) {

            terms <- c(
                terms,
                frozen_symbol
            )
        }
    }

    # ----------------------------------------------------------------------------------------------
    # Tier B historical ARHGAP11B identity
    # ----------------------------------------------------------------------------------------------

    else if (
        identical(
            tier,
            "TIER_B_HISTORICAL_IDENTITY_SPECIAL_HANDLING"
        )
    ) {

        terms <- "ARHGAP11B"
    }

    # ----------------------------------------------------------------------------------------------
    # Tier C — descriptive noncoding identity + stable ID
    # ----------------------------------------------------------------------------------------------

    else if (
        identical(
            tier,
            "TIER_C_ID_ANCHORED_NONCODING_FEATURE_REVIEW"
        )
    ) {

        terms <- stable_id

        if (
            !is.na(current_symbol) &&
            !current_symbol %in% c(
                "Y_RNA",
                "U7",
                "Metazoa_SRP"
            )
        ) {

            terms <- c(
                terms,
                current_symbol
            )
        }
    }

    # ----------------------------------------------------------------------------------------------
    # Tier D — stable ID only
    # ----------------------------------------------------------------------------------------------

    else {

        terms <- stable_id
    }

    unique(
        terms[
            !is.na(terms) &
            nzchar(terms)
        ]
    )
}

search_terms_list <- lapply(
    seq_len(nrow(identity)),
    build_search_terms
)

names(search_terms_list) <- identity$ensemblID_stable

# --------------------------------------------------------------------------------------------------
# 10E.1G — BUILD EUROPE PMC TITLE/ABSTRACT GENE CLAUSE
# --------------------------------------------------------------------------------------------------

escape_query_phrase <- function(x) {

    x <- gsub(
        "\\\\",
        "\\\\\\\\",
        x
    )

    x <- gsub(
        "\"",
        "\\\\\"",
        x
    )

    x
}

make_gene_clause <- function(terms) {

    clauses <- paste0(
        'TITLE_ABS:"',
        vapply(
            terms,
            escape_query_phrase,
            FUN.VALUE = character(1)
        ),
        '"'
    )

    paste0(
        "(",
        paste(
            clauses,
            collapse = " OR "
        ),
        ")"
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.1H — DOMAIN CLAUSES
# --------------------------------------------------------------------------------------------------

domain_clauses <- list(

    SCHIZOPHRENIA_PSYCHOSIS = paste(
        "(",
        'TITLE_ABS:"schizophrenia"',
        "OR",
        'TITLE_ABS:"schizophrenic"',
        "OR",
        'TITLE_ABS:"psychosis"',
        "OR",
        'TITLE_ABS:"psychotic"',
        ")"
    ),

    BRAIN_NEURAL_CELLTYPE = paste(
        "(",
        'TITLE_ABS:"brain"',
        "OR",
        'TITLE_ABS:"cortex"',
        "OR",
        'TITLE_ABS:"cortical"',
        "OR",
        'TITLE_ABS:"hippocampus"',
        "OR",
        'TITLE_ABS:"hippocampal"',
        "OR",
        "TITLE_ABS:neuron*",
        "OR",
        "TITLE_ABS:astrocyt*",
        "OR",
        "TITLE_ABS:microgl*",
        "OR",
        "TITLE_ABS:oligodendrocyt*",
        "OR",
        "TITLE_ABS:dopamin*",
        ")"
    ),

    SCZ_GENETIC_CONVERGENCE = paste(
        "(",
        'TITLE_ABS:"schizophrenia"',
        "OR",
        'TITLE_ABS:"psychosis"',
        ")",
        "AND",
        "(",
        'TITLE_ABS:"genetic"',
        "OR",
        'TITLE_ABS:"genetics"',
        "OR",
        'TITLE_ABS:"variant"',
        "OR",
        'TITLE_ABS:"variants"',
        "OR",
        'TITLE_ABS:"GWAS"',
        "OR",
        'TITLE_ABS:"genome-wide"',
        "OR",
        'TITLE_ABS:"locus"',
        "OR",
        'TITLE_ABS:"loci"',
        ")"
    )
)

# --------------------------------------------------------------------------------------------------
# 10E.1I — BUILD QUERY REGISTRY
# --------------------------------------------------------------------------------------------------

query_rows <- list()
q_index <- 1L

for (i in seq_len(nrow(identity))) {

    terms <- search_terms_list[[i]]

    if (length(terms) == 0L) {
        stop(
            "No literature search term could be constructed for ",
            identity$ensemblID_stable[i]
        )
    }

    gene_clause <- make_gene_clause(
        terms
    )

    for (domain_name in names(domain_clauses)) {

        full_query <- paste(
            gene_clause,
            "AND",
            domain_clauses[[domain_name]]
        )

        query_rows[[q_index]] <- data.frame(

            gene_index = i,

            ensemblID_stable =
                identity$ensemblID_stable[i],

            frozen_symbol =
                identity$gene_symbol[i],

            interpretation_symbol =
                identity$interpretation_symbol[i],

            manuscript_gene_label =
                identity$manuscript_gene_label[i],

            literature_interpretation_tier =
                identity$literature_interpretation_tier[i],

            allow_gene_specific_biological_claim =
                identity$allow_gene_specific_biological_claim[i],

            search_terms = paste(
                terms,
                collapse = " | "
            ),

            evidence_domain =
                domain_name,

            europe_pmc_query =
                full_query,

            stringsAsFactors = FALSE
        )

        q_index <- q_index + 1L
    }
}

query_registry <- do.call(
    rbind,
    query_rows
)

if (nrow(query_registry) != 72L) {
    stop(
        "Expected 72 gene-domain literature queries (24 x 3); observed ",
        nrow(query_registry)
    )
}

write.csv(
    query_registry,
    QUERY_REGISTRY_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

cat(
    "Literature queries constructed: ",
    nrow(query_registry),
    " (24 genes x 3 domains) — PASS\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.1J — EUROPE PMC SEARCH FUNCTION
#
# resultType=core returns detailed metadata including abstracts where available.
# Top five records are retained per query.
# --------------------------------------------------------------------------------------------------

EUROPE_PMC_ENDPOINT <-
    "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

safe_scalar <- function(x) {

    if (
        is.null(x) ||
        length(x) == 0L
    ) {
        return(NA_character_)
    }

    x <- as.character(x)

    if (length(x) > 1L) {
        x <- paste(
            x,
            collapse = "; "
        )
    }

    if (
        is.na(x) ||
        !nzchar(trimws(x))
    ) {
        return(NA_character_)
    }

    trimws(x)
}

europe_pmc_search <- function(query_text, page_size = 5L) {

    last_error <- NULL

    for (attempt in 1:2) {

        result <- tryCatch(
            {

                response <- httr::GET(
                    EUROPE_PMC_ENDPOINT,
                    query = list(
                        query = query_text,
                        format = "json",
                        resultType = "core",
                        pageSize = page_size
                    ),
                    httr::timeout(45)
                )

                status <- httr::status_code(
                    response
                )

                if (status != 200L) {

                    stop(
                        "Europe PMC HTTP status ",
                        status
                    )
                }

                payload <- jsonlite::fromJSON(
                    httr::content(
                        response,
                        as = "text",
                        encoding = "UTF-8"
                    ),
                    simplifyVector = FALSE
                )

                hit_count <- if (
                    !is.null(payload$hitCount)
                ) {
                    as.numeric(payload$hitCount)
                } else {
                    NA_real_
                }

                raw_results <- NULL

                if (
                    !is.null(payload$resultList) &&
                    !is.null(payload$resultList$result)
                ) {

                    raw_results <-
                        payload$resultList$result
                }

                list(
                    success = TRUE,
                    http_status = status,
                    hit_count = hit_count,
                    results = raw_results,
                    error = NA_character_
                )
            },

            error = function(e) {

                last_error <<- conditionMessage(e)

                NULL
            }
        )

        if (!is.null(result)) {
            return(result)
        }

        if (attempt < 2L) {
            Sys.sleep(1)
        }
    }

    list(
        success = FALSE,
        http_status = NA_integer_,
        hit_count = NA_real_,
        results = NULL,
        error = last_error
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.1K — CONVERT ONE EUROPE PMC RECORD TO A FLAT ROW
# --------------------------------------------------------------------------------------------------

flatten_record <- function(record) {

    data.frame(

        source = safe_scalar(
            record$source
        ),

        external_id = safe_scalar(
            record$id
        ),

        pmid = safe_scalar(
            record$pmid
        ),

        pmcid = safe_scalar(
            record$pmcid
        ),

        doi = safe_scalar(
            record$doi
        ),

        title = safe_scalar(
            record$title
        ),

        author_string = safe_scalar(
            record$authorString
        ),

        journal_title = safe_scalar(
            record$journalTitle
        ),

        pub_year = safe_scalar(
            record$pubYear
        ),

        first_publication_date = safe_scalar(
            record$firstPublicationDate
        ),

        cited_by_count = suppressWarnings(
            as.numeric(
                safe_scalar(
                    record$citedByCount
                )
            )
        ),

        publication_type = safe_scalar(
            record$pubType
        ),

        abstract_text = safe_scalar(
            record$abstractText
        ),

        is_open_access = safe_scalar(
            record$isOpenAccess
        ),

        stringsAsFactors = FALSE
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.1L — RUN ALL 72 QUERIES
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nEUROPE PMC LITERATURE RETRIEVAL\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

hit_rows <- vector(
    "list",
    nrow(query_registry)
)

record_rows <- list()
record_index <- 1L

for (q in seq_len(nrow(query_registry))) {

    query_text <-
        query_registry$europe_pmc_query[q]

    res <- europe_pmc_search(
        query_text = query_text,
        page_size = 5L
    )

    hit_rows[[q]] <- data.frame(

        gene_index =
            query_registry$gene_index[q],

        ensemblID_stable =
            query_registry$ensemblID_stable[q],

        frozen_symbol =
            query_registry$frozen_symbol[q],

        interpretation_symbol =
            query_registry$interpretation_symbol[q],

        literature_interpretation_tier =
            query_registry$literature_interpretation_tier[q],

        evidence_domain =
            query_registry$evidence_domain[q],

        search_terms =
            query_registry$search_terms[q],

        query_success =
            res$success,

        http_status =
            res$http_status,

        hit_count =
            res$hit_count,

        retrieval_error =
            res$error,

        stringsAsFactors = FALSE
    )

    if (
        isTRUE(res$success) &&
        !is.null(res$results) &&
        length(res$results) > 0L
    ) {

        n_records <- min(
            5L,
            length(res$results)
        )

        for (r in seq_len(n_records)) {

            flat <- flatten_record(
                res$results[[r]]
            )

            record_rows[[record_index]] <- cbind(

                data.frame(
                    gene_index =
                        query_registry$gene_index[q],

                    ensemblID_stable =
                        query_registry$ensemblID_stable[q],

                    frozen_symbol =
                        query_registry$frozen_symbol[q],

                    interpretation_symbol =
                        query_registry$interpretation_symbol[q],

                    manuscript_gene_label =
                        query_registry$manuscript_gene_label[q],

                    literature_interpretation_tier =
                        query_registry$literature_interpretation_tier[q],

                    allow_gene_specific_biological_claim =
                        query_registry$allow_gene_specific_biological_claim[q],

                    evidence_domain =
                        query_registry$evidence_domain[q],

                    search_terms =
                        query_registry$search_terms[q],

                    query_rank = r,

                    stringsAsFactors = FALSE
                ),

                flat
            )

            record_index <- record_index + 1L
        }
    }

    Sys.sleep(0.12)
}

hit_counts <- do.call(
    rbind,
    hit_rows
)

if (length(record_rows) > 0L) {

    top_records <- do.call(
        rbind,
        record_rows
    )

} else {

    top_records <- data.frame(
        gene_index = integer(0),
        ensemblID_stable = character(0),
        frozen_symbol = character(0),
        interpretation_symbol = character(0),
        manuscript_gene_label = character(0),
        literature_interpretation_tier = character(0),
        allow_gene_specific_biological_claim = logical(0),
        evidence_domain = character(0),
        search_terms = character(0),
        query_rank = integer(0),
        source = character(0),
        external_id = character(0),
        pmid = character(0),
        pmcid = character(0),
        doi = character(0),
        title = character(0),
        author_string = character(0),
        journal_title = character(0),
        pub_year = character(0),
        first_publication_date = character(0),
        cited_by_count = numeric(0),
        publication_type = character(0),
        abstract_text = character(0),
        is_open_access = character(0),
        stringsAsFactors = FALSE
    )
}

write.csv(
    hit_counts,
    HIT_COUNT_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    top_records,
    TOP_RECORDS_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.1M — VERIFY RETRIEVAL SUCCESS
# --------------------------------------------------------------------------------------------------

n_queries <- nrow(
    hit_counts
)

n_success <- sum(
    hit_counts$query_success
)

n_failed <- sum(
    !hit_counts$query_success
)

cat(
    "\nEurope PMC queries successful: ",
    n_success,
    "/",
    n_queries,
    "\n",
    sep = ""
)

cat(
    "Europe PMC queries failed: ",
    n_failed,
    "\n",
    sep = ""
)

if (n_failed > 0L) {

    cat(
        "\nFAILED QUERY AUDIT\n"
    )

    print(
        hit_counts[
            !hit_counts$query_success,
            c(
                "ensemblID_stable",
                "interpretation_symbol",
                "evidence_domain",
                "http_status",
                "retrieval_error"
            ),
            drop = FALSE
        ],
        row.names = FALSE
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.1N — BUILD ONE-ROW-PER-GENE LITERATURE SUMMARY
# --------------------------------------------------------------------------------------------------

get_domain_hits <- function(id, domain) {

    x <- hit_counts$hit_count[
        hit_counts$ensemblID_stable == id &
        hit_counts$evidence_domain == domain
    ]

    if (length(x) != 1L) {
        return(NA_real_)
    }

    as.numeric(x)
}

gene_summary <- data.frame(

    gene_index = seq_len(
        nrow(identity)
    ),

    ensemblID_stable =
        identity$ensemblID_stable,

    frozen_symbol =
        identity$gene_symbol,

    interpretation_symbol =
        identity$interpretation_symbol,

    literature_interpretation_tier =
        identity$literature_interpretation_tier,

    allow_gene_specific_biological_claim =
        identity$allow_gene_specific_biological_claim,

    direction_standardized =
        identity$direction_standardized,

    final_normalized_weight =
        identity$final_normalized_weight,

    search_terms = vapply(
        search_terms_list,
        function(x) {
            paste(
                x,
                collapse = " | "
            )
        },
        FUN.VALUE = character(1)
    ),

    schizophrenia_psychosis_hits = vapply(
        identity$ensemblID_stable,
        get_domain_hits,
        domain = "SCHIZOPHRENIA_PSYCHOSIS",
        FUN.VALUE = numeric(1)
    ),

    brain_neural_celltype_hits = vapply(
        identity$ensemblID_stable,
        get_domain_hits,
        domain = "BRAIN_NEURAL_CELLTYPE",
        FUN.VALUE = numeric(1)
    ),

    scz_genetic_convergence_hits = vapply(
        identity$ensemblID_stable,
        get_domain_hits,
        domain = "SCZ_GENETIC_CONVERGENCE",
        FUN.VALUE = numeric(1)
    ),

    stringsAsFactors = FALSE
)

gene_summary$any_direct_scz_literature <- (
    !is.na(
        gene_summary$schizophrenia_psychosis_hits
    ) &
    gene_summary$schizophrenia_psychosis_hits > 0
)

gene_summary$any_brain_neural_literature <- (
    !is.na(
        gene_summary$brain_neural_celltype_hits
    ) &
    gene_summary$brain_neural_celltype_hits > 0
)

gene_summary$any_scz_genetic_literature <- (
    !is.na(
        gene_summary$scz_genetic_convergence_hits
    ) &
    gene_summary$scz_genetic_convergence_hits > 0
)

# IMPORTANT:
# These are retrieval indicators, NOT evidence-strength conclusions.

gene_summary$retrieval_status <- ifelse(
    gene_summary$any_direct_scz_literature |
    gene_summary$any_brain_neural_literature |
    gene_summary$any_scz_genetic_literature,
    "LITERATURE_HITS_RETRIEVED_REQUIRES_MANUAL_EVIDENCE_REVIEW",
    "NO_HITS_FROM_CURRENT_REPRODUCIBLE_QUERY"
)

write.csv(
    gene_summary,
    GENE_SUMMARY_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.1O — SUMMARY COUNTS
# --------------------------------------------------------------------------------------------------

named_reviewable <- (
    identity$allow_gene_specific_biological_claim
)

n_named_reviewable <- sum(
    named_reviewable
)

n_named_with_direct_scz <- sum(
    gene_summary$any_direct_scz_literature &
    named_reviewable,
    na.rm = TRUE
)

n_named_with_brain <- sum(
    gene_summary$any_brain_neural_literature &
    named_reviewable,
    na.rm = TRUE
)

n_named_with_genetic <- sum(
    gene_summary$any_scz_genetic_literature &
    named_reviewable,
    na.rm = TRUE
)

n_any_direct_scz <- sum(
    gene_summary$any_direct_scz_literature,
    na.rm = TRUE
)

n_any_brain <- sum(
    gene_summary$any_brain_neural_literature,
    na.rm = TRUE
)

n_any_genetic <- sum(
    gene_summary$any_scz_genetic_literature,
    na.rm = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10E.1P — HUMAN-READABLE SUMMARY
# --------------------------------------------------------------------------------------------------

retrieval_date <- as.character(
    Sys.Date()
)

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10E.1 — SYSTEMATIC SCZ-MSS GENE-LEVEL LITERATURE EVIDENCE RETRIEVAL",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    paste0(
        "Retrieval date: ",
        retrieval_date
    ),

    "Database: Europe PMC REST API",

    "",

    "QUERY DESIGN",

    "Frozen/adjudicated features: 24",
    "Evidence domains per feature: 3",
    paste0(
        "Total queries: ",
        n_queries
    ),

    paste0(
        "Successful queries: ",
        n_success
    ),

    paste0(
        "Failed queries: ",
        n_failed
    ),

    "",

    "EVIDENCE DOMAINS",

    "1. SCHIZOPHRENIA_PSYCHOSIS",
    "2. BRAIN_NEURAL_CELLTYPE",
    "3. SCZ_GENETIC_CONVERGENCE",

    "",

    "NAMED GENE-LEVEL REVIEW SET",

    paste0(
        "Features permitted for named gene-level review: ",
        n_named_reviewable,
        "/24"
    ),

    paste0(
        "Named reviewable features with >=1 schizophrenia/psychosis hit: ",
        n_named_with_direct_scz,
        "/",
        n_named_reviewable
    ),

    paste0(
        "Named reviewable features with >=1 brain/neural/cell-type hit: ",
        n_named_with_brain,
        "/",
        n_named_reviewable
    ),

    paste0(
        "Named reviewable features with >=1 SCZ genetic-convergence hit: ",
        n_named_with_genetic,
        "/",
        n_named_reviewable
    ),

    "",

    "ALL 24 FEATURES",

    paste0(
        "Features with >=1 schizophrenia/psychosis retrieval hit: ",
        n_any_direct_scz,
        "/24"
    ),

    paste0(
        "Features with >=1 brain/neural/cell-type retrieval hit: ",
        n_any_brain,
        "/24"
    ),

    paste0(
        "Features with >=1 SCZ genetic-convergence retrieval hit: ",
        n_any_genetic,
        "/24"
    ),

    "",

    "INTERPRETATION RULE",

    paste(
        "Search hit counts are retrieval metrics only.",
        "They are NOT measures of evidence strength, causality, replication,",
        "schizophrenia specificity, or biological importance."
    ),

    paste(
        "Each retrieved paper must be reviewed for study design, species/tissue,",
        "direction of evidence, relevance to schizophrenia, and direct versus indirect support",
        "before any manuscript biological claim is made."
    ),

    paste(
        "Tier C/D features remain stable-ID-anchored and must not be converted into",
        "named-gene claims merely because a broad literature query returns a hit."
    ),

    "",

    "NEXT STEP",

    paste(
        "Manually classify the retrieved top records into direct schizophrenia evidence,",
        "brain/cell-type functional context, genetic convergence, indirect context,",
        "or non-supportive/irrelevant evidence."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10E.1Q — MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    IDENTITY_PATH,
    QUERY_REGISTRY_PATH,
    HIT_COUNT_PATH,
    TOP_RECORDS_PATH,
    GENE_SUMMARY_PATH,
    SUMMARY_PATH
)

manifest <- data.frame(
    file = basename(
        manifest_paths
    ),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(
            manifest_paths
        )$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10E.1R — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nLITERATURE RETRIEVAL SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Europe PMC queries successful: ",
    n_success,
    "/",
    n_queries,
    "\n",
    sep = ""
)

cat(
    "Named gene-level review features: ",
    n_named_reviewable,
    "/24\n",
    sep = ""
)

cat(
    "\nNamed features with direct schizophrenia/psychosis hits: ",
    n_named_with_direct_scz,
    "/",
    n_named_reviewable,
    "\n",
    sep = ""
)

cat(
    "Named features with brain/neural/cell-type hits: ",
    n_named_with_brain,
    "/",
    n_named_reviewable,
    "\n",
    sep = ""
)

cat(
    "Named features with SCZ genetic-convergence hits: ",
    n_named_with_genetic,
    "/",
    n_named_reviewable,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nGENE-LEVEL HIT COUNTS\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    gene_summary[
        ,
        c(
            "ensemblID_stable",
            "frozen_symbol",
            "interpretation_symbol",
            "literature_interpretation_tier",
            "direction_standardized",
            "final_normalized_weight",
            "schizophrenia_psychosis_hits",
            "brain_neural_celltype_hits",
            "scz_genetic_convergence_hits",
            "retrieval_status"
        )
    ],
    row.names = FALSE
)

cat(
    "\nQuery registry:\n",
    QUERY_REGISTRY_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPer-query hit counts:\n",
    HIT_COUNT_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTop literature records + abstracts:\n",
    TOP_RECORDS_PATH,
    "\n",
    sep = ""
)

cat(
    "\nGene-level literature summary:\n",
    GENE_SUMMARY_PATH,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    SUMMARY_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10E.1 COMPLETE — LITERATURE EVIDENCE CORPUS RETRIEVED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10E.1 — SYSTEMATIC GENE-LEVEL LITERATURE EVIDENCE RETRIEVAL
Project sentinel: PASS
STEP10D.2 adjudicated identities: PASS
Frozen feature count: 24 — PASS
Literature queries constructed: 72 (24 genes x 3 domains) — PASS

--------------------------------------------------------------------------------------------------------------
EUROPE PMC LITERATURE RETRIEVAL
--------------------------------------------------------------------------------------------------------------

Europe PMC queries successful: 72/72
Europe PMC queries failed: 0

LITERATURE RETRIEVAL SUMMARY
Europe PMC queries successful: 72/72
Named gene-level review features: 18/24

Named features with direct schizophrenia/psychosis hits: 2/18
Named features with brain/neural/cell-type hits: 12/18
Named features with SCZ genetic-convergence hits: 2/18

--------------------------------------------------------------------------------------------------------------
GENE-LEVEL HIT COUNTS
---------------------------------------

In [69]:
# ==================================================================================================
# STEP 10E.2 — LITERATURE CORPUS DEDUPLICATION + EVIDENCE-REVIEW QUEUE
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Convert STEP10E.1 retrieval output into a clean, deduplicated, review-ready evidence corpus.
#
# THIS STEP:
#   1. Verifies the 72-query retrieval.
#   2. Deduplicates papers within each frozen feature.
#   3. Adds transparent keyword/context flags:
#        - schizophrenia / psychosis
#        - brain / neural / postmortem
#        - genetics / GWAS / variants
#        - human/patient context
#   4. Creates an OPERATIONAL review-priority score.
#   5. Creates a focused priority queue for papers most relevant to biological interpretation.
#   6. Prints top candidate papers for genes with direct SCZ/genetic retrieval hits.
#
# IMPORTANT
#   - Keyword flags are NOT evidence-strength judgments.
#   - Priority score is ONLY for review order.
#   - No paper is automatically classified as supportive.
#   - No gene is declared schizophrenia-associated in this step.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10E.2 — LITERATURE CORPUS DEDUPLICATION + EVIDENCE-REVIEW QUEUE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.2A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

LIT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1"
)

IDENTITY_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "identity_audit_v1",
    "adjudicated_v1",
    "STEP10D2_SCZ_MSS_24gene_interpretation_identity_v1.csv"
)

QUERY_PATH <- file.path(
    LIT_DIR,
    "STEP10E1_literature_query_registry_v1.csv"
)

HIT_COUNT_PATH <- file.path(
    LIT_DIR,
    "STEP10E1_gene_domain_literature_hit_counts_v1.csv"
)

RECORD_PATH <- file.path(
    LIT_DIR,
    "STEP10E1_top_literature_records_with_abstracts_v1.csv"
)

GENE_SUMMARY_PATH <- file.path(
    LIT_DIR,
    "STEP10E1_gene_level_literature_summary_v1.csv"
)

OUT_DIR <- file.path(
    LIT_DIR,
    "evidence_review_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

DEDUP_PATH <- file.path(
    OUT_DIR,
    "STEP10E2_deduplicated_gene_literature_corpus_v1.csv"
)

PRIORITY_PATH <- file.path(
    OUT_DIR,
    "STEP10E2_priority_evidence_review_queue_v1.csv"
)

SCZ_PRIORITY_PATH <- file.path(
    OUT_DIR,
    "STEP10E2_SCZ_genetic_priority_records_v1.csv"
)

GENE_AUDIT_PATH <- file.path(
    OUT_DIR,
    "STEP10E2_gene_level_evidence_queue_summary_v1.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "STEP10E2_evidence_review_queue_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "STEP10E2_evidence_review_queue_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10E.2B — PROJECT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop(
        "Project sentinel missing: ",
        SENTINEL_PATH
    )
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

required_files <- c(
    IDENTITY_PATH,
    QUERY_PATH,
    HIT_COUNT_PATH,
    RECORD_PATH,
    GENE_SUMMARY_PATH
)

missing_files <- required_files[
    !file.exists(
        required_files
    )
]

if (length(missing_files) > 0L) {

    stop(
        "Required STEP10E.1 file(s) missing:\n",
        paste(
            missing_files,
            collapse = "\n"
        )
    )
}

cat("STEP10D.2 identity table: PASS\n")
cat("STEP10E.1 literature artifacts: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10E.2C — LOAD DATA
# --------------------------------------------------------------------------------------------------

identity <- read.csv(
    IDENTITY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

queries <- read.csv(
    QUERY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

hits <- read.csv(
    HIT_COUNT_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

records <- read.csv(
    RECORD_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

gene_summary <- read.csv(
    GENE_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(identity) != 24L) {

    stop(
        "Expected 24 adjudicated features; observed ",
        nrow(identity)
    )
}

if (nrow(queries) != 72L) {

    stop(
        "Expected 72 STEP10E.1 queries; observed ",
        nrow(queries)
    )
}

if (nrow(hits) != 72L) {

    stop(
        "Expected 72 STEP10E.1 hit-count rows; observed ",
        nrow(hits)
    )
}

if (nrow(gene_summary) != 24L) {

    stop(
        "Expected 24 gene-summary rows; observed ",
        nrow(gene_summary)
    )
}

cat("Frozen feature count: 24 — PASS\n")
cat("Literature query count: 72 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10E.2D — VERIFY ALL QUERIES SUCCEEDED
# --------------------------------------------------------------------------------------------------

if (!is.logical(hits$query_success)) {

    z <- toupper(
        trimws(
            as.character(
                hits$query_success
            )
        )
    )

    if (!all(
        z %in% c(
            "TRUE",
            "FALSE"
        )
    )) {

        stop(
            "Unexpected query_success encoding."
        )
    }

    hits$query_success <- (
        z == "TRUE"
    )
}

n_success <- sum(
    hits$query_success
)

n_failed <- sum(
    !hits$query_success
)

if (n_success != 72L) {

    stop(
        "STEP10E.1 retrieval was not complete.\n",
        "Successful: ",
        n_success,
        "/72"
    )
}

cat("Europe PMC retrieval completeness: 72/72 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10E.2E — CLEAN RECORD TEXT
# --------------------------------------------------------------------------------------------------

clean_text <- function(x) {

    x <- as.character(x)

    x[
        is.na(x)
    ] <- ""

    trimws(x)
}

text_cols <- intersect(
    c(
        "ensemblID_stable",
        "frozen_symbol",
        "interpretation_symbol",
        "manuscript_gene_label",
        "literature_interpretation_tier",
        "evidence_domain",
        "search_terms",
        "pmid",
        "pmcid",
        "doi",
        "title",
        "author_string",
        "journal_title",
        "pub_year",
        "publication_type",
        "abstract_text"
    ),
    names(records)
)

for (nm in text_cols) {

    records[[nm]] <- clean_text(
        records[[nm]]
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.2F — VERIFY REQUIRED RECORD COLUMNS
# --------------------------------------------------------------------------------------------------

required_record_cols <- c(
    "ensemblID_stable",
    "interpretation_symbol",
    "literature_interpretation_tier",
    "evidence_domain",
    "pmid",
    "doi",
    "title",
    "abstract_text"
)

missing_record_cols <- setdiff(
    required_record_cols,
    names(records)
)

if (length(missing_record_cols) > 0L) {

    stop(
        "Literature record file missing column(s): ",
        paste(
            missing_record_cols,
            collapse = ", "
        )
    )
}

cat(
    "Retrieved top-record rows before deduplication: ",
    nrow(records),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.2G — CREATE PAPER DEDUPLICATION KEY
#
# Priority:
#     PMID
#     DOI
#     PMCID
#     normalized title
#
# Deduplication occurs WITHIN frozen feature.
# The same paper can therefore legitimately support >1 feature.
# --------------------------------------------------------------------------------------------------

normalize_title <- function(x) {

    x <- tolower(
        trimws(
            x
        )
    )

    x <- gsub(
        "[^a-z0-9]+",
        " ",
        x
    )

    gsub(
        "\\s+",
        " ",
        x
    )
}

records$normalized_title <- normalize_title(
    records$title
)

records$paper_key <- ifelse(
    nzchar(records$pmid),
    paste0(
        "PMID:",
        records$pmid
    ),
    ifelse(
        nzchar(records$doi),
        paste0(
            "DOI:",
            tolower(
                records$doi
            )
        ),
        ifelse(
            "pmcid" %in% names(records) &
                nzchar(records$pmcid),
            paste0(
                "PMCID:",
                records$pmcid
            ),
            paste0(
                "TITLE:",
                records$normalized_title
            )
        )
    )
)

records$gene_paper_key <- paste(
    records$ensemblID_stable,
    records$paper_key,
    sep = "||"
)

# --------------------------------------------------------------------------------------------------
# 10E.2H — COLLAPSE DUPLICATE QUERY-DOMAIN RECORDS
# --------------------------------------------------------------------------------------------------

split_domains <- function(x) {

    z <- unique(
        x[
            !is.na(x) &
            nzchar(x)
        ]
    )

    paste(
        sort(z),
        collapse = ";"
    )
}

dedup_keys <- unique(
    records$gene_paper_key
)

dedup_rows <- vector(
    "list",
    length(dedup_keys)
)

for (k in seq_along(dedup_keys)) {

    key <- dedup_keys[k]

    block <- records[
        records$gene_paper_key == key,
        ,
        drop = FALSE
    ]

    # Prefer the row containing the longest abstract.
    abs_lengths <- nchar(
        block$abstract_text
    )

    best_idx <- which.max(
        abs_lengths
    )

    best <- block[
        best_idx,
        ,
        drop = FALSE
    ]

    best$retrieved_evidence_domains <- split_domains(
        block$evidence_domain
    )

    best$n_query_domain_occurrences <- nrow(
        block
    )

    dedup_rows[[k]] <- best
}

dedup <- do.call(
    rbind,
    dedup_rows
)

rownames(dedup) <- NULL

cat(
    "Unique gene-paper records after deduplication: ",
    nrow(dedup),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.2I — TITLE + ABSTRACT SEARCH TEXT
# --------------------------------------------------------------------------------------------------

dedup$review_text <- paste(
    dedup$title,
    dedup$abstract_text
)

dedup$review_text_lower <- tolower(
    dedup$review_text
)

# --------------------------------------------------------------------------------------------------
# 10E.2J — TRANSPARENT CONTEXT FLAGS
#
# These flags help PRIORITIZE manual review.
# They are NOT biological conclusions.
# --------------------------------------------------------------------------------------------------

has_regex <- function(text, pattern) {

    grepl(
        pattern,
        text,
        perl = TRUE,
        ignore.case = TRUE
    )
}

SCZ_PATTERN <- paste0(
    "\\b(",
    "schizophren[a-z]*",
    "|psychosis",
    "|psychotic",
    "|first[- ]episode psychosis",
    ")\\b"
)

BRAIN_PATTERN <- paste0(
    "\\b(",
    "brain",
    "|cortex",
    "|cortical",
    "|prefrontal",
    "|dlpfc",
    "|hippocamp[a-z]*",
    "|neuron[a-z]*",
    "|astrocyt[a-z]*",
    "|microgli[a-z]*",
    "|oligodendrocyt[a-z]*",
    "|dopamin[a-z]*",
    "|postmortem",
    "|post-mortem",
    ")\\b"
)

GENETIC_PATTERN <- paste0(
    "\\b(",
    "gwas",
    "|genome[- ]wide association",
    "|genetic",
    "|genomic",
    "|variant",
    "|variants",
    "|polymorphism",
    "|polymorphisms",
    "|locus",
    "|loci",
    "|snp",
    "|snps",
    "|copy number",
    "|cnv",
    "|heritab[a-z]*",
    ")\\b"
)

HUMAN_PATTERN <- paste0(
    "\\b(",
    "human",
    "|humans",
    "|patient",
    "|patients",
    "|participant",
    "|participants",
    "|subject",
    "|subjects",
    "|case-control",
    "|postmortem",
    "|post-mortem",
    ")\\b"
)

EXPRESSION_PATTERN <- paste0(
    "\\b(",
    "expression",
    "|transcript",
    "|transcriptomic",
    "|rna[- ]seq",
    "|rna sequencing",
    "|mrna",
    "|gene expression",
    ")\\b"
)

dedup$flag_schizophrenia_psychosis <- has_regex(
    dedup$review_text_lower,
    SCZ_PATTERN
)

dedup$flag_brain_neural <- has_regex(
    dedup$review_text_lower,
    BRAIN_PATTERN
)

dedup$flag_genetic <- has_regex(
    dedup$review_text_lower,
    GENETIC_PATTERN
)

dedup$flag_human_context <- has_regex(
    dedup$review_text_lower,
    HUMAN_PATTERN
)

dedup$flag_expression_context <- has_regex(
    dedup$review_text_lower,
    EXPRESSION_PATTERN
)

# --------------------------------------------------------------------------------------------------
# 10E.2K — ABSTRACT AVAILABILITY
# --------------------------------------------------------------------------------------------------

dedup$abstract_available <- (
    nzchar(
        trimws(
            dedup$abstract_text
        )
    )
)

# --------------------------------------------------------------------------------------------------
# 10E.2L — NAMED REVIEWABILITY
# --------------------------------------------------------------------------------------------------

reviewable_map <- identity[
    ,
    c(
        "ensemblID_stable",
        "allow_gene_specific_biological_claim",
        "literature_interpretation_tier",
        "direction_standardized",
        "final_normalized_weight"
    )
]

names(reviewable_map)[
    names(reviewable_map) ==
        "literature_interpretation_tier"
] <- "adjudicated_interpretation_tier"

dedup <- merge(
    dedup,
    reviewable_map,
    by = "ensemblID_stable",
    all.x = TRUE,
    sort = FALSE
)

# --------------------------------------------------------------------------------------------------
# 10E.2M — OPERATIONAL REVIEW PRIORITY SCORE
#
# This is NOT an evidence score.
#
# It only controls which papers should be read first.
# --------------------------------------------------------------------------------------------------

dedup$review_priority_score <- (
    4L *
        as.integer(
            dedup$flag_schizophrenia_psychosis
        ) +
    3L *
        as.integer(
            dedup$flag_genetic
        ) +
    2L *
        as.integer(
            dedup$flag_brain_neural
        ) +
    1L *
        as.integer(
            dedup$flag_human_context
        ) +
    1L *
        as.integer(
            dedup$flag_expression_context
        ) +
    1L *
        as.integer(
            dedup$abstract_available
        )
)

# Historical-ID ARHGAP11B papers remain reviewable,
# but require identity caution.
dedup$historical_identity_special_case <- (
    dedup$ensemblID_stable ==
        "ENSG00000187951"
)

# --------------------------------------------------------------------------------------------------
# 10E.2N — DOMAIN-CONSISTENCY FLAGS
# --------------------------------------------------------------------------------------------------

dedup$retrieved_as_scz_query <- grepl(
    "SCHIZOPHRENIA_PSYCHOSIS",
    dedup$retrieved_evidence_domains,
    fixed = TRUE
)

dedup$retrieved_as_brain_query <- grepl(
    "BRAIN_NEURAL_CELLTYPE",
    dedup$retrieved_evidence_domains,
    fixed = TRUE
)

dedup$retrieved_as_genetic_query <- grepl(
    "SCZ_GENETIC_CONVERGENCE",
    dedup$retrieved_evidence_domains,
    fixed = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10E.2O — PRELIMINARY REVIEW CATEGORY
#
# IMPORTANT:
# These are QUEUE categories, not evidence-strength conclusions.
# --------------------------------------------------------------------------------------------------

dedup$review_queue_category <- ifelse(

    dedup$flag_schizophrenia_psychosis &
        dedup$flag_genetic,

    "PRIORITY_1_SCZ_PLUS_GENETIC_CONTEXT",

    ifelse(

        dedup$flag_schizophrenia_psychosis &
            dedup$flag_brain_neural,

        "PRIORITY_2_SCZ_PLUS_BRAIN_CONTEXT",

        ifelse(

            dedup$flag_schizophrenia_psychosis,

            "PRIORITY_3_SCZ_CONTEXT",

            ifelse(

                dedup$flag_brain_neural,

                "PRIORITY_4_BRAIN_CONTEXT",

                "PRIORITY_5_OTHER_RETRIEVED_CONTEXT"
            )
        )
    )
)

# --------------------------------------------------------------------------------------------------
# 10E.2P — SORT DEDUPLICATED CORPUS
# --------------------------------------------------------------------------------------------------

dedup <- dedup[
    order(
        dedup$gene_index,
        -dedup$review_priority_score,
        -suppressWarnings(
            as.numeric(
                dedup$cited_by_count
            )
        ),
        dedup$query_rank,
        na.last = TRUE
    ),
    ,
    drop = FALSE
]

write.csv(
    dedup,
    DEDUP_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.2Q — PRIORITY REVIEW QUEUE
#
# Keep papers with:
#   - schizophrenia flag OR
#   - genetic flag OR
#   - brain flag
#
# No claim is made from inclusion.
# --------------------------------------------------------------------------------------------------

priority <- dedup[
    dedup$flag_schizophrenia_psychosis |
    dedup$flag_genetic |
    dedup$flag_brain_neural,
    ,
    drop = FALSE
]

priority <- priority[
    order(
        priority$review_queue_category,
        -priority$review_priority_score,
        priority$gene_index,
        na.last = TRUE
    ),
    ,
    drop = FALSE
]

write.csv(
    priority,
    PRIORITY_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.2R — SCZ / GENETIC FOCUSED QUEUE
# --------------------------------------------------------------------------------------------------

scz_priority <- dedup[
    dedup$flag_schizophrenia_psychosis |
    (
        dedup$flag_genetic &
        dedup$retrieved_as_genetic_query
    ),
    ,
    drop = FALSE
]

scz_priority <- scz_priority[
    order(
        scz_priority$gene_index,
        -scz_priority$review_priority_score,
        -suppressWarnings(
            as.numeric(
                scz_priority$cited_by_count
            )
        ),
        na.last = TRUE
    ),
    ,
    drop = FALSE
]

write.csv(
    scz_priority,
    SCZ_PRIORITY_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.2S — GENE-LEVEL QUEUE SUMMARY
# --------------------------------------------------------------------------------------------------

count_gene_flag <- function(id, flag) {

    block <- dedup[
        dedup$ensemblID_stable == id,
        ,
        drop = FALSE
    ]

    if (nrow(block) == 0L) {
        return(0L)
    }

    sum(
        block[[flag]],
        na.rm = TRUE
    )
}

gene_audit <- data.frame(

    ensemblID_stable =
        identity$ensemblID_stable,

    frozen_symbol =
        identity$gene_symbol,

    interpretation_symbol =
        identity$interpretation_symbol,

    literature_interpretation_tier =
        identity$literature_interpretation_tier,

    allow_gene_specific_biological_claim =
        identity$allow_gene_specific_biological_claim,

    direction_standardized =
        identity$direction_standardized,

    final_normalized_weight =
        identity$final_normalized_weight,

    unique_retrieved_papers = vapply(
        identity$ensemblID_stable,
        function(id) {

            sum(
                dedup$ensemblID_stable == id
            )
        },
        FUN.VALUE = integer(1)
    ),

    papers_flagged_SCZ = vapply(
        identity$ensemblID_stable,
        count_gene_flag,
        flag = "flag_schizophrenia_psychosis",
        FUN.VALUE = integer(1)
    ),

    papers_flagged_brain = vapply(
        identity$ensemblID_stable,
        count_gene_flag,
        flag = "flag_brain_neural",
        FUN.VALUE = integer(1)
    ),

    papers_flagged_genetic = vapply(
        identity$ensemblID_stable,
        count_gene_flag,
        flag = "flag_genetic",
        FUN.VALUE = integer(1)
    ),

    papers_flagged_human = vapply(
        identity$ensemblID_stable,
        count_gene_flag,
        flag = "flag_human_context",
        FUN.VALUE = integer(1)
    ),

    stringsAsFactors = FALSE
)

gene_audit$manual_evidence_review_required <- (
    gene_audit$unique_retrieved_papers > 0L
)

write.csv(
    gene_audit,
    GENE_AUDIT_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.2T — VERIFY STEP10E.1 HIGH-LEVEL DIRECT-HIT PATTERN
# --------------------------------------------------------------------------------------------------

direct_named <- gene_summary[
    gene_summary$allow_gene_specific_biological_claim &
    gene_summary$schizophrenia_psychosis_hits > 0,
    ,
    drop = FALSE
]

genetic_named <- gene_summary[
    gene_summary$allow_gene_specific_biological_claim &
    gene_summary$scz_genetic_convergence_hits > 0,
    ,
    drop = FALSE
]

cat(
    "\nNamed reviewable features with STEP10E.1 direct SCZ hits: ",
    nrow(direct_named),
    "\n",
    sep = ""
)

if (nrow(direct_named) > 0L) {

    print(
        direct_named[
            ,
            c(
                "ensemblID_stable",
                "interpretation_symbol",
                "schizophrenia_psychosis_hits"
            )
        ],
        row.names = FALSE
    )
}

cat(
    "\nNamed reviewable features with STEP10E.1 genetic-convergence hits: ",
    nrow(genetic_named),
    "\n",
    sep = ""
)

if (nrow(genetic_named) > 0L) {

    print(
        genetic_named[
            ,
            c(
                "ensemblID_stable",
                "interpretation_symbol",
                "scz_genetic_convergence_hits"
            )
        ],
        row.names = FALSE
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.2U — SPECIAL HISTORICAL ARHGAP11B AUDIT
# --------------------------------------------------------------------------------------------------

arhgap_summary <- gene_summary[
    gene_summary$ensemblID_stable ==
        "ENSG00000187951",
    ,
    drop = FALSE
]

if (nrow(arhgap_summary) != 1L) {
    stop(
        "Historical ARHGAP11B feature missing from literature summary."
    )
}

cat(
    "\nHistorical ARHGAP11B retrieval:\n"
)

cat(
    "  Schizophrenia/psychosis hits: ",
    arhgap_summary$schizophrenia_psychosis_hits,
    "\n",
    sep = ""
)

cat(
    "  Brain/neural hits: ",
    arhgap_summary$brain_neural_celltype_hits,
    "\n",
    sep = ""
)

cat(
    "  Genetic-convergence hits: ",
    arhgap_summary$scz_genetic_convergence_hits,
    "\n",
    sep = ""
)

cat(
    "  Identity handling: HISTORICAL SPECIAL CASE — requires careful paper-level review\n"
)

# --------------------------------------------------------------------------------------------------
# 10E.2V — PRINT TOP SCZ/GENETIC CANDIDATE PAPERS
#
# Maximum 5 per gene to keep notebook output manageable.
# --------------------------------------------------------------------------------------------------

scz_gene_ids <- unique(
    scz_priority$ensemblID_stable
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nTOP SCZ / GENETIC REVIEW CANDIDATES\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

if (length(scz_gene_ids) == 0L) {

    cat("[NONE]\n")

} else {

    for (id in scz_gene_ids) {

        block <- scz_priority[
            scz_priority$ensemblID_stable == id,
            ,
            drop = FALSE
        ]

        block <- head(
            block,
            5L
        )

        gene_label <- unique(
            block$interpretation_symbol
        )

        gene_label <- gene_label[
            !is.na(gene_label) &
            nzchar(gene_label)
        ]

        if (length(gene_label) == 0L) {
            gene_label <- id
        } else {
            gene_label <- gene_label[1L]
        }

        cat(
            "\n",
            paste(rep("-", 100), collapse = ""),
            "\n",
            gene_label,
            " | ",
            id,
            "\n",
            paste(rep("-", 100), collapse = ""),
            "\n",
            sep = ""
        )

        for (j in seq_len(nrow(block))) {

            cat(
                "[",
                j,
                "] ",
                block$title[j],
                "\n",
                sep = ""
            )

            cat(
                "    PMID: ",
                ifelse(
                    nzchar(block$pmid[j]),
                    block$pmid[j],
                    "[NA]"
                ),
                " | Year: ",
                ifelse(
                    nzchar(block$pub_year[j]),
                    block$pub_year[j],
                    "[NA]"
                ),
                "\n",
                sep = ""
            )

            cat(
                "    Queue: ",
                block$review_queue_category[j],
                " | SCZ=",
                block$flag_schizophrenia_psychosis[j],
                " | Genetic=",
                block$flag_genetic[j],
                " | Brain=",
                block$flag_brain_neural[j],
                " | Human=",
                block$flag_human_context[j],
                "\n",
                sep = ""
            )

            abstract_preview <- block$abstract_text[j]

            if (
                is.na(abstract_preview) ||
                !nzchar(
                    trimws(
                        abstract_preview
                    )
                )
            ) {

                abstract_preview <- "[NO ABSTRACT]"

            } else {

                abstract_preview <- gsub(
                    "\\s+",
                    " ",
                    abstract_preview
                )

                if (
                    nchar(
                        abstract_preview
                    ) > 500L
                ) {

                    abstract_preview <- paste0(
                        substr(
                            abstract_preview,
                            1L,
                            500L
                        ),
                        "..."
                    )
                }
            }

            cat(
                "    Abstract preview: ",
                abstract_preview,
                "\n",
                sep = ""
            )
        }
    }
}

# --------------------------------------------------------------------------------------------------
# 10E.2W — SUMMARY COUNTS
# --------------------------------------------------------------------------------------------------

n_dedup <- nrow(
    dedup
)

n_priority <- nrow(
    priority
)

n_scz_priority <- nrow(
    scz_priority
)

n_scz_flagged <- sum(
    dedup$flag_schizophrenia_psychosis
)

n_brain_flagged <- sum(
    dedup$flag_brain_neural
)

n_genetic_flagged <- sum(
    dedup$flag_genetic
)

n_human_flagged <- sum(
    dedup$flag_human_context
)

# --------------------------------------------------------------------------------------------------
# 10E.2X — HUMAN-READABLE SUMMARY
# --------------------------------------------------------------------------------------------------

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10E.2 — LITERATURE EVIDENCE-REVIEW QUEUE",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "RETRIEVAL VERIFICATION",

    "Europe PMC STEP10E.1 queries successful: 72/72",

    paste0(
        "Top-record rows retrieved before deduplication: ",
        nrow(records)
    ),

    paste0(
        "Unique feature-paper records after deduplication: ",
        n_dedup
    ),

    "",

    "REVIEW QUEUE",

    paste0(
        "Records with SCZ/genetic/brain context flags: ",
        n_priority
    ),

    paste0(
        "SCZ/genetic focused records: ",
        n_scz_priority
    ),

    paste0(
        "Records containing schizophrenia/psychosis terminology: ",
        n_scz_flagged
    ),

    paste0(
        "Records containing brain/neural terminology: ",
        n_brain_flagged
    ),

    paste0(
        "Records containing genetic terminology: ",
        n_genetic_flagged
    ),

    paste0(
        "Records containing human/patient terminology: ",
        n_human_flagged
    ),

    "",

    "STEP10E.1 DIRECT SCHIZOPHRENIA RETRIEVAL",

    paste0(
        "Named reviewable features with direct schizophrenia/psychosis hits: ",
        nrow(direct_named),
        "/18"
    ),

    paste0(
        "Named reviewable features with genetic-convergence hits: ",
        nrow(genetic_named),
        "/18"
    ),

    "",

    "IMPORTANT",

    paste(
        "Keyword/context flags and review-priority scores are operational triage tools only."
    ),

    paste(
        "They do not establish evidence strength, causality, schizophrenia specificity,",
        "directional agreement, replication, or functional relevance."
    ),

    paste(
        "Each priority record requires paper-level review before manuscript use."
    ),

    "",

    "NEXT STEP",

    paste(
        "Review and classify the priority records into direct human schizophrenia evidence,",
        "schizophrenia genetic convergence, brain/cell-type functional context,",
        "indirect evidence, or non-supportive/irrelevant evidence."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10E.2Y — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace("digest", quietly = TRUE)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    IDENTITY_PATH,
    QUERY_PATH,
    HIT_COUNT_PATH,
    RECORD_PATH,
    GENE_SUMMARY_PATH,
    DEDUP_PATH,
    PRIORITY_PATH,
    SCZ_PRIORITY_PATH,
    GENE_AUDIT_PATH,
    SUMMARY_PATH
)

manifest <- data.frame(
    file = basename(
        manifest_paths
    ),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(
            manifest_paths
        )$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10E.2Z — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10E.2 SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Europe PMC query success: 72/72\n"
)

cat(
    "Retrieved rows before deduplication: ",
    nrow(records),
    "\n",
    sep = ""
)

cat(
    "Unique feature-paper records: ",
    n_dedup,
    "\n",
    sep = ""
)

cat(
    "Priority review records: ",
    n_priority,
    "\n",
    sep = ""
)

cat(
    "SCZ/genetic focused review records: ",
    n_scz_priority,
    "\n",
    sep = ""
)

cat(
    "\nNamed reviewable direct-SCZ features: ",
    nrow(direct_named),
    "/18\n",
    sep = ""
)

cat(
    "Named reviewable genetic-convergence features: ",
    nrow(genetic_named),
    "/18\n",
    sep = ""
)

cat(
    "\nDeduplicated literature corpus:\n",
    DEDUP_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPriority review queue:\n",
    PRIORITY_PATH,
    "\n",
    sep = ""
)

cat(
    "\nSCZ/genetic priority queue:\n",
    SCZ_PRIORITY_PATH,
    "\n",
    sep = ""
)

cat(
    "\nGene-level queue summary:\n",
    GENE_AUDIT_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10E.2 COMPLETE — EVIDENCE-REVIEW QUEUE FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10E.2 — LITERATURE CORPUS DEDUPLICATION + EVIDENCE-REVIEW QUEUE
Project sentinel: PASS
STEP10D.2 identity table: PASS
STEP10E.1 literature artifacts: PASS
Frozen feature count: 24 — PASS
Literature query count: 72 — PASS
Europe PMC retrieval completeness: 72/72 — PASS
Retrieved top-record rows before deduplication: 73
Unique gene-paper records after deduplication: 66

Named reviewable features with STEP10E.1 direct SCZ hits: 2
 ensemblID_stable interpretation_symbol schizophrenia_psychosis_hits
  ENSG00000165092               ALDH1A1                            7
  ENSG00000153234                 NR4A2                           27

Named reviewable features with STEP10E.1 genetic-convergence hits: 2
 ensemblID_stable interpretation_symbol scz_genetic_convergence_hits
  ENSG00000165092               ALDH1A1                            4
  ENSG00000153234                 NR4A2                           12

Historical ARHGAP11B retrieval:
  Schizophrenia/psychosis hits: 3
  Brain/neur

In [70]:
# ==================================================================================================
# STEP 10E.3 — PAPER-LEVEL TARGET-GENE RELEVANCE REVIEW PACKET
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Prepare the focused SCZ/genetic literature queue for MANUAL evidence adjudication.
#
# INPUT
#   STEP10E.2_SCZ_genetic_priority_records_v1.csv
#
# THIS STEP WILL:
#   1. Verify the focused review queue.
#   2. Construct interpretation-safe target-gene search terms.
#   3. Check whether the target gene appears in:
#        - title
#        - abstract
#   4. Extract sentences mentioning:
#        - target gene
#        - schizophrenia / psychosis
#        - genetics / variants / GWAS
#   5. Assign an OPERATIONAL directness candidate class based on textual proximity.
#   6. Produce a compact manual-review packet.
#
# IMPORTANT
#   - Directness candidate class is NOT an evidence conclusion.
#   - Positive/negative/supportive/non-supportive evidence is NOT assigned automatically.
#   - ARHGAP11B remains a historical-identity special case.
#   - ALDH1A2 must never be treated as ALDH1A1.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10E.3 — PAPER-LEVEL TARGET-GENE RELEVANCE REVIEW PACKET\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.3A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

IDENTITY_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "identity_audit_v1",
    "adjudicated_v1",
    "STEP10D2_SCZ_MSS_24gene_interpretation_identity_v1.csv"
)

SCZ_QUEUE_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "evidence_review_v1",
    "STEP10E2_SCZ_genetic_priority_records_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "evidence_review_v1",
    "paper_level_review_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_CSV <- file.path(
    OUT_DIR,
    "STEP10E3_SCZ_genetic_paper_level_review_packet_v1.csv"
)

OUT_TXT <- file.path(
    OUT_DIR,
    "STEP10E3_SCZ_genetic_paper_level_review_packet_v1.txt"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP10E3_paper_level_review_summary_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP10E3_paper_level_review_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10E.3B — VERIFY PROJECT + INPUTS
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID,
        "\nObserved: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

for (p in c(
    IDENTITY_PATH,
    SCZ_QUEUE_PATH
)) {

    if (!file.exists(p)) {
        stop("Required input missing: ", p)
    }
}

identity <- read.csv(
    IDENTITY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

queue <- read.csv(
    SCZ_QUEUE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(identity) != 24L) {
    stop(
        "Expected 24 adjudicated identities; observed ",
        nrow(identity)
    )
}

if (nrow(queue) != 18L) {
    stop(
        "Expected 18 focused STEP10E.2 records; observed ",
        nrow(queue)
    )
}

cat("STEP10D.2 identities: PASS\n")
cat("Focused SCZ/genetic review records: 18 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10E.3C — REQUIRED QUEUE COLUMNS
# --------------------------------------------------------------------------------------------------

required_queue_cols <- c(
    "ensemblID_stable",
    "frozen_symbol",
    "interpretation_symbol",
    "literature_interpretation_tier",
    "pmid",
    "doi",
    "title",
    "pub_year",
    "publication_type",
    "abstract_text",
    "retrieved_evidence_domains",
    "flag_schizophrenia_psychosis",
    "flag_genetic",
    "flag_brain_neural",
    "flag_human_context"
)

missing_cols <- setdiff(
    required_queue_cols,
    names(queue)
)

if (length(missing_cols) > 0L) {
    stop(
        "Focused queue missing required column(s): ",
        paste(missing_cols, collapse = ", ")
    )
}

clean_text <- function(x) {

    x <- as.character(x)
    x[is.na(x)] <- ""

    trimws(x)
}

for (nm in intersect(
    c(
        "ensemblID_stable",
        "frozen_symbol",
        "interpretation_symbol",
        "pmid",
        "doi",
        "title",
        "pub_year",
        "publication_type",
        "abstract_text",
        "retrieved_evidence_domains"
    ),
    names(queue)
)) {

    queue[[nm]] <- clean_text(
        queue[[nm]]
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.3D — BUILD INTERPRETATION-SAFE TARGET TERMS
# --------------------------------------------------------------------------------------------------

build_target_terms <- function(id) {

    row <- identity[
        identity$ensemblID_stable == id,
        ,
        drop = FALSE
    ]

    if (nrow(row) != 1L) {
        stop(
            "Could not uniquely resolve identity for ",
            id
        )
    }

    frozen_symbol <- clean_text(
        row$gene_symbol
    )

    interpretation_symbol <- clean_text(
        row$interpretation_symbol
    )

    # ----------------------------------------------------------------------------------------------
    # POLR1G / CD3EAP
    # ----------------------------------------------------------------------------------------------

    if (id == "ENSG00000117877") {

        return(
            c(
                "POLR1G",
                "CD3EAP"
            )
        )
    }

    # ----------------------------------------------------------------------------------------------
    # Historical ARHGAP11B
    # ----------------------------------------------------------------------------------------------

    if (id == "ENSG00000187951") {

        return(
            "ARHGAP11B"
        )
    }

    # ----------------------------------------------------------------------------------------------
    # Normal named identity
    # ----------------------------------------------------------------------------------------------

    terms <- c(
        interpretation_symbol,
        frozen_symbol
    )

    terms <- unique(
        terms[
            nzchar(terms)
        ]
    )

    if (length(terms) == 0L) {

        terms <- id
    }

    terms
}

unique_ids <- unique(
    queue$ensemblID_stable
)

target_term_map <- lapply(
    unique_ids,
    build_target_terms
)

names(target_term_map) <- unique_ids

# --------------------------------------------------------------------------------------------------
# 10E.3E — REGEX HELPERS
# --------------------------------------------------------------------------------------------------

escape_regex <- function(x) {

    gsub(
        "([][{}()+*^$|\\\\?.])",
        "\\\\\\1",
        x
    )
}

make_term_regex <- function(terms) {

    terms <- unique(
        terms[
            nzchar(terms)
        ]
    )

    if (length(terms) == 0L) {
        return("$a")
    }

    escaped <- vapply(
        terms,
        escape_regex,
        FUN.VALUE = character(1)
    )

    paste0(
        "(?i)(?<![A-Za-z0-9])(?:",
        paste(
            escaped,
            collapse = "|"
        ),
        ")(?![A-Za-z0-9])"
    )
}

SCZ_REGEX <- paste0(
    "(?i)\\b(",
    "schizophren[a-z]*",
    "|psychosis",
    "|psychotic",
    ")\\b"
)

GENETIC_REGEX <- paste0(
    "(?i)\\b(",
    "genetic",
    "|genetics",
    "|variant",
    "|variants",
    "|polymorphism",
    "|polymorphisms",
    "|gwas",
    "|genome[- ]wide",
    "|locus",
    "|loci",
    "|snp",
    "|snps",
    "|copy number",
    "|cnv",
    ")\\b"
)

# --------------------------------------------------------------------------------------------------
# 10E.3F — SENTENCE SPLITTER
# --------------------------------------------------------------------------------------------------

split_sentences <- function(text) {

    text <- clean_text(text)

    if (!nzchar(text)) {
        return(character(0))
    }

    text <- gsub(
        "<[^>]+>",
        " ",
        text
    )

    text <- gsub(
        "\\s+",
        " ",
        text
    )

    s <- unlist(
        strsplit(
            text,
            "(?<=[.!?])\\s+",
            perl = TRUE
        ),
        use.names = FALSE
    )

    trimws(
        s[
            nzchar(
                trimws(s)
            )
        ]
    )
}

extract_matching_sentences <- function(text, regex) {

    s <- split_sentences(
        text
    )

    if (length(s) == 0L) {
        return("")
    }

    hit <- s[
        grepl(
            regex,
            s,
            perl = TRUE
        )
    ]

    if (length(hit) == 0L) {
        return("")
    }

    paste(
        unique(hit),
        collapse = " || "
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.3G — TARGET-GENE TEXT AUDIT
# --------------------------------------------------------------------------------------------------

queue$target_search_terms <- ""

queue$target_in_title <- FALSE
queue$target_in_abstract <- FALSE

queue$scz_in_title <- FALSE
queue$scz_in_abstract <- FALSE

queue$genetic_in_title <- FALSE
queue$genetic_in_abstract <- FALSE

queue$target_sentences <- ""
queue$scz_sentences <- ""
queue$genetic_sentences <- ""
queue$target_scz_same_sentence <- FALSE
queue$target_genetic_same_sentence <- FALSE

for (i in seq_len(nrow(queue))) {

    id <- queue$ensemblID_stable[i]

    terms <- target_term_map[[id]]

    gene_regex <- make_term_regex(
        terms
    )

    queue$target_search_terms[i] <- paste(
        terms,
        collapse = " | "
    )

    queue$target_in_title[i] <- grepl(
        gene_regex,
        queue$title[i],
        perl = TRUE
    )

    queue$target_in_abstract[i] <- grepl(
        gene_regex,
        queue$abstract_text[i],
        perl = TRUE
    )

    queue$scz_in_title[i] <- grepl(
        SCZ_REGEX,
        queue$title[i],
        perl = TRUE
    )

    queue$scz_in_abstract[i] <- grepl(
        SCZ_REGEX,
        queue$abstract_text[i],
        perl = TRUE
    )

    queue$genetic_in_title[i] <- grepl(
        GENETIC_REGEX,
        queue$title[i],
        perl = TRUE
    )

    queue$genetic_in_abstract[i] <- grepl(
        GENETIC_REGEX,
        queue$abstract_text[i],
        perl = TRUE
    )

    queue$target_sentences[i] <-
        extract_matching_sentences(
            queue$abstract_text[i],
            gene_regex
        )

    queue$scz_sentences[i] <-
        extract_matching_sentences(
            queue$abstract_text[i],
            SCZ_REGEX
        )

    queue$genetic_sentences[i] <-
        extract_matching_sentences(
            queue$abstract_text[i],
            GENETIC_REGEX
        )

    sentences <- split_sentences(
        queue$abstract_text[i]
    )

    if (length(sentences) > 0L) {

        queue$target_scz_same_sentence[i] <- any(
            grepl(
                gene_regex,
                sentences,
                perl = TRUE
            ) &
            grepl(
                SCZ_REGEX,
                sentences,
                perl = TRUE
            )
        )

        queue$target_genetic_same_sentence[i] <- any(
            grepl(
                gene_regex,
                sentences,
                perl = TRUE
            ) &
            grepl(
                GENETIC_REGEX,
                sentences,
                perl = TRUE
            )
        )
    }
}

# --------------------------------------------------------------------------------------------------
# 10E.3H — DETECT OBVIOUS DIFFERENT-GENE NAME COLLISIONS
#
# Specific guard relevant to current corpus:
# ALDH1A2 must never be interpreted as ALDH1A1.
# --------------------------------------------------------------------------------------------------

queue$aldh1a2_collision_flag <- FALSE

aldh1a1_rows <- (
    queue$interpretation_symbol ==
        "ALDH1A1"
)

if (any(aldh1a1_rows)) {

    queue$aldh1a2_collision_flag[
        aldh1a1_rows
    ] <- grepl(
        "(?i)(?<![A-Za-z0-9])ALDH1A2(?![A-Za-z0-9])",
        paste(
            queue$title[
                aldh1a1_rows
            ],
            queue$abstract_text[
                aldh1a1_rows
            ]
        ),
        perl = TRUE
    )
}

# --------------------------------------------------------------------------------------------------
# 10E.3I — OPERATIONAL DIRECTNESS CANDIDATE CLASS
#
# NOT an evidence conclusion.
# --------------------------------------------------------------------------------------------------

queue$textual_directness_candidate <- ifelse(

    queue$target_in_title &
    queue$scz_in_title,

    "A_TARGET_AND_SCZ_IN_TITLE",

    ifelse(

        queue$target_scz_same_sentence,

        "B_TARGET_AND_SCZ_SAME_ABSTRACT_SENTENCE",

        ifelse(

            queue$target_in_abstract &
            queue$scz_in_abstract,

            "C_TARGET_AND_SCZ_IN_DIFFERENT_ABSTRACT_CONTEXT",

            ifelse(

                queue$target_genetic_same_sentence &
                (
                    queue$scz_in_title |
                    queue$scz_in_abstract
                ),

                "D_TARGET_GENETIC_PROXIMITY_WITH_SCZ_CONTEXT",

                "E_WEAK_TEXTUAL_PROXIMITY_REQUIRES_REVIEW"
            )
        )
    )
)

# --------------------------------------------------------------------------------------------------
# 10E.3J — HISTORICAL IDENTITY CAUTION
# --------------------------------------------------------------------------------------------------

queue$identity_caution <- ifelse(
    queue$ensemblID_stable ==
        "ENSG00000187951",
    paste(
        "HISTORICAL ARHGAP11B FEATURE:",
        "frozen BrainSeq ID ENSG00000187951;",
        "interpret only as historical ARHGAP11B identity."
    ),
    ""
)

# --------------------------------------------------------------------------------------------------
# 10E.3K — REVIEW DECISION PLACEHOLDERS
#
# These intentionally remain blank.
# They will be adjudicated after reading the paper-level evidence.
# --------------------------------------------------------------------------------------------------

queue$manual_relevance_class <- ""

queue$manual_evidence_direction <- ""

queue$manual_population_context <- ""

queue$manual_evidence_strength <- ""

queue$manual_review_note <- ""

# --------------------------------------------------------------------------------------------------
# 10E.3L — SORT FOR MANUAL REVIEW
# --------------------------------------------------------------------------------------------------

directness_order <- c(
    "A_TARGET_AND_SCZ_IN_TITLE",
    "B_TARGET_AND_SCZ_SAME_ABSTRACT_SENTENCE",
    "C_TARGET_AND_SCZ_IN_DIFFERENT_ABSTRACT_CONTEXT",
    "D_TARGET_GENETIC_PROXIMITY_WITH_SCZ_CONTEXT",
    "E_WEAK_TEXTUAL_PROXIMITY_REQUIRES_REVIEW"
)

queue$textual_directness_candidate <- factor(
    queue$textual_directness_candidate,
    levels = directness_order,
    ordered = TRUE
)

queue <- queue[
    order(
        queue$gene_index,
        queue$textual_directness_candidate,
        -suppressWarnings(
            as.numeric(
                queue$cited_by_count
            )
        ),
        na.last = TRUE
    ),
    ,
    drop = FALSE
]

queue$textual_directness_candidate <- as.character(
    queue$textual_directness_candidate
)

# --------------------------------------------------------------------------------------------------
# 10E.3M — WRITE REVIEW CSV
# --------------------------------------------------------------------------------------------------

write.csv(
    queue,
    OUT_CSV,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10E.3N — BUILD MANUAL-REVIEW TEXT PACKET
# --------------------------------------------------------------------------------------------------

packet_lines <- c(
    "==============================================================================================================",
    "STEP 10E.3 — SCZ / GENETIC PAPER-LEVEL MANUAL REVIEW PACKET",
    "==============================================================================================================",
    ""
)

for (i in seq_len(nrow(queue))) {

    packet_lines <- c(
        packet_lines,

        paste(
            rep("-", 105),
            collapse = ""
        ),

        paste0(
            "RECORD ",
            i,
            "/",
            nrow(queue)
        ),

        paste0(
            "Feature: ",
            ifelse(
                nzchar(
                    queue$interpretation_symbol[i]
                ),
                queue$interpretation_symbol[i],
                queue$ensemblID_stable[i]
            )
        ),

        paste0(
            "Frozen Ensembl ID: ",
            queue$ensemblID_stable[i]
        ),

        paste0(
            "Target search terms: ",
            queue$target_search_terms[i]
        ),

        paste0(
            "PMID: ",
            ifelse(
                nzchar(queue$pmid[i]),
                queue$pmid[i],
                "[NA]"
            )
        ),

        paste0(
            "Year: ",
            ifelse(
                nzchar(queue$pub_year[i]),
                queue$pub_year[i],
                "[NA]"
            )
        ),

        paste0(
            "Publication type: ",
            ifelse(
                nzchar(
                    queue$publication_type[i]
                ),
                queue$publication_type[i],
                "[NA]"
            )
        ),

        paste0(
            "Title: ",
            queue$title[i]
        ),

        paste0(
            "Retrieved domains: ",
            queue$retrieved_evidence_domains[i]
        ),

        paste0(
            "Textual directness candidate: ",
            queue$textual_directness_candidate[i]
        ),

        paste0(
            "Target in title: ",
            queue$target_in_title[i]
        ),

        paste0(
            "Target in abstract: ",
            queue$target_in_abstract[i]
        ),

        paste0(
            "Target + SCZ same sentence: ",
            queue$target_scz_same_sentence[i]
        ),

        paste0(
            "Target + genetic same sentence: ",
            queue$target_genetic_same_sentence[i]
        ),

        paste0(
            "Human-context flag: ",
            queue$flag_human_context[i]
        ),

        paste0(
            "Brain-context flag: ",
            queue$flag_brain_neural[i]
        ),

        if (
            queue$aldh1a2_collision_flag[i]
        ) {
            "WARNING: ALDH1A2 detected in an ALDH1A1 review record — different gene; manual review required."
        } else {
            ""
        },

        if (
            nzchar(
                queue$identity_caution[i]
            )
        ) {
            queue$identity_caution[i]
        } else {
            ""
        },

        "",

        "TARGET-GENE SENTENCE(S):",

        if (
            nzchar(
                queue$target_sentences[i]
            )
        ) {
            queue$target_sentences[i]
        } else {
            "[NO TARGET-GENE SENTENCE EXTRACTED]"
        },

        "",

        "SCHIZOPHRENIA / PSYCHOSIS SENTENCE(S):",

        if (
            nzchar(
                queue$scz_sentences[i]
            )
        ) {
            queue$scz_sentences[i]
        } else {
            "[NO SCZ SENTENCE EXTRACTED]"
        },

        "",

        "GENETIC SENTENCE(S):",

        if (
            nzchar(
                queue$genetic_sentences[i]
            )
        ) {
            queue$genetic_sentences[i]
        } else {
            "[NO GENETIC SENTENCE EXTRACTED]"
        },

        ""
    )
}

writeLines(
    packet_lines,
    OUT_TXT,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10E.3O — SUMMARY COUNTS
# --------------------------------------------------------------------------------------------------

class_counts <- table(
    queue$textual_directness_candidate
)

n_target_title <- sum(
    queue$target_in_title
)

n_target_abstract <- sum(
    queue$target_in_abstract
)

n_target_scz_same <- sum(
    queue$target_scz_same_sentence
)

n_target_genetic_same <- sum(
    queue$target_genetic_same_sentence
)

n_collision <- sum(
    queue$aldh1a2_collision_flag
)

# --------------------------------------------------------------------------------------------------
# 10E.3P — HUMAN-READABLE SUMMARY
# --------------------------------------------------------------------------------------------------

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10E.3 — PAPER-LEVEL REVIEW PACKET SUMMARY",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    paste0(
        "Focused records reviewed computationally: ",
        nrow(queue)
    ),

    paste0(
        "Target gene present in title: ",
        n_target_title,
        "/",
        nrow(queue)
    ),

    paste0(
        "Target gene present in abstract: ",
        n_target_abstract,
        "/",
        nrow(queue)
    ),

    paste0(
        "Target gene + schizophrenia/psychosis in same abstract sentence: ",
        n_target_scz_same,
        "/",
        nrow(queue)
    ),

    paste0(
        "Target gene + genetic terminology in same abstract sentence: ",
        n_target_genetic_same,
        "/",
        nrow(queue)
    ),

    paste0(
        "ALDH1A2 collision warnings within ALDH1A1 records: ",
        n_collision
    ),

    "",

    "TEXTUAL DIRECTNESS CANDIDATE COUNTS",

    paste0(
        names(class_counts),
        ": ",
        as.integer(class_counts)
    ),

    "",

    "IMPORTANT",

    paste(
        "These classes describe textual proximity only.",
        "They are NOT evidence-strength or support classifications."
    ),

    paste(
        "The next step requires manual paper-level adjudication of whether each paper",
        "provides direct supportive evidence, negative/null evidence, indirect context,",
        "different-gene evidence, or irrelevant retrieval."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10E.3Q — MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    IDENTITY_PATH,
    SCZ_QUEUE_PATH,
    OUT_CSV,
    OUT_TXT,
    OUT_SUMMARY
)

manifest <- data.frame(
    file = basename(
        manifest_paths
    ),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(
            manifest_paths
        )$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10E.3R — CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nTEXTUAL DIRECTNESS AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Focused records: ",
    nrow(queue),
    "\n",
    sep = ""
)

cat(
    "Target gene in title: ",
    n_target_title,
    "/",
    nrow(queue),
    "\n",
    sep = ""
)

cat(
    "Target gene in abstract: ",
    n_target_abstract,
    "/",
    nrow(queue),
    "\n",
    sep = ""
)

cat(
    "Target + SCZ same sentence: ",
    n_target_scz_same,
    "/",
    nrow(queue),
    "\n",
    sep = ""
)

cat(
    "Target + genetic same sentence: ",
    n_target_genetic_same,
    "/",
    nrow(queue),
    "\n",
    sep = ""
)

cat(
    "ALDH1A2 collision warnings: ",
    n_collision,
    "\n",
    sep = ""
)

cat(
    "\nDIRECTNESS CANDIDATE COUNTS\n"
)

print(
    class_counts
)

# --------------------------------------------------------------------------------------------------
# 10E.3S — PRINT COMPACT 18-RECORD REVIEW PACKET
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nMANUAL REVIEW PACKET — 18 FOCUSED RECORDS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

for (i in seq_len(nrow(queue))) {

    cat(
        "\n",
        paste(rep("-", 105), collapse = ""),
        "\n",
        "RECORD ",
        i,
        "/",
        nrow(queue),
        "\n",
        sep = ""
    )

    cat(
        "Feature: ",
        ifelse(
            nzchar(
                queue$interpretation_symbol[i]
            ),
            queue$interpretation_symbol[i],
            queue$ensemblID_stable[i]
        ),
        " | ",
        queue$ensemblID_stable[i],
        "\n",
        sep = ""
    )

    cat(
        "PMID: ",
        ifelse(
            nzchar(queue$pmid[i]),
            queue$pmid[i],
            "[NA]"
        ),
        " | Year: ",
        ifelse(
            nzchar(queue$pub_year[i]),
            queue$pub_year[i],
            "[NA]"
        ),
        "\n",
        sep = ""
    )

    cat(
        "Title: ",
        queue$title[i],
        "\n",
        sep = ""
    )

    cat(
        "Directness candidate: ",
        queue$textual_directness_candidate[i],
        "\n",
        sep = ""
    )

    if (
        queue$aldh1a2_collision_flag[i]
    ) {

        cat(
            "WARNING: ALDH1A2 detected — do not confuse with ALDH1A1.\n"
        )
    }

    if (
        nzchar(
            queue$identity_caution[i]
        )
    ) {

        cat(
            "IDENTITY CAUTION: ",
            queue$identity_caution[i],
            "\n",
            sep = ""
        )
    }

    cat(
        "TARGET SENTENCE(S): ",
        ifelse(
            nzchar(
                queue$target_sentences[i]
            ),
            queue$target_sentences[i],
            "[NONE]"
        ),
        "\n",
        sep = ""
    )

    cat(
        "SCZ SENTENCE(S): ",
        ifelse(
            nzchar(
                queue$scz_sentences[i]
            ),
            queue$scz_sentences[i],
            "[NONE]"
        ),
        "\n",
        sep = ""
    )

    cat(
        "GENETIC SENTENCE(S): ",
        ifelse(
            nzchar(
                queue$genetic_sentences[i]
            ),
            queue$genetic_sentences[i],
            "[NONE]"
        ),
        "\n",
        sep = ""
    )
}

cat(
    "\nReview CSV:\n",
    OUT_CSV,
    "\n",
    sep = ""
)

cat(
    "\nFull review packet TXT:\n",
    OUT_TXT,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10E.3 COMPLETE — PAPER-LEVEL REVIEW PACKET FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10E.3 — PAPER-LEVEL TARGET-GENE RELEVANCE REVIEW PACKET
Project sentinel: PASS
STEP10D.2 identities: PASS
Focused SCZ/genetic review records: 18 — PASS

TEXTUAL DIRECTNESS AUDIT
Focused records: 18
Target gene in title: 5/18
Target gene in abstract: 18/18
Target + SCZ same sentence: 9/18
Target + genetic same sentence: 6/18
ALDH1A2 collision warnings: 1

DIRECTNESS CANDIDATE COUNTS

                     A_TARGET_AND_SCZ_IN_TITLE 
                                             4 
       B_TARGET_AND_SCZ_SAME_ABSTRACT_SENTENCE 
                                             5 
C_TARGET_AND_SCZ_IN_DIFFERENT_ABSTRACT_CONTEXT 
                                             9 

MANUAL REVIEW PACKET — 18 FOCUSED RECORDS

---------------------------------------------------------------------------------------------------------
RECORD 1/18
Feature: ARHGAP11B | ENSG00000187951
PMID: 30886340 | Year: 2019
Title: De novo single-nucleotide and copy number variation in discordant monozygotic twins re

In [71]:
# ==================================================================================================
# STEP 10E.4 — MANUAL PAPER-LEVEL EVIDENCE ADJUDICATION + GENE-LEVEL SYNTHESIS
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Freeze manual interpretation of the 18 SCZ/genetic-focused literature records reviewed
#   in STEP10E.3.
#
# IMPORTANT
#   - Classifications below are based on the title/abstract evidence already retrieved.
#   - No hit-count based inference.
#   - No automatic promotion of indirect evidence.
#   - Duplicate reports are explicitly separated.
#   - ARHGAP11B retains historical-ID caution.
#   - ALDH1A2 is never treated as ALDH1A1.
#
# OUTPUT
#   1. 18-record adjudicated evidence table
#   2. Independent-primary-evidence subset
#   3. One-row-per-gene synthesis for ARHGAP11B, ALDH1A1, NR4A2
#   4. Manuscript-safe interpretation summary
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10E.4 — MANUAL PAPER-LEVEL EVIDENCE ADJUDICATION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

INPUT_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "evidence_review_v1",
    "paper_level_review_v1",
    "STEP10E3_SCZ_genetic_paper_level_review_packet_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "evidence_review_v1",
    "manual_adjudication_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_RECORDS <- file.path(
    OUT_DIR,
    "STEP10E4_manual_paper_evidence_adjudication_v1.csv"
)

OUT_PRIMARY <- file.path(
    OUT_DIR,
    "STEP10E4_independent_primary_evidence_subset_v1.csv"
)

OUT_GENES <- file.path(
    OUT_DIR,
    "STEP10E4_gene_level_SCZ_evidence_synthesis_v1.csv"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP10E4_manual_evidence_adjudication_summary_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP10E4_manual_evidence_adjudication_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# VERIFY PROJECT
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(project_id, EXPECTED_PROJECT_ID)) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID,
        "\nObserved: ", project_id
    )
}

cat("Project sentinel: PASS\n")

if (!file.exists(INPUT_PATH)) {
    stop(
        "STEP10E.3 review packet missing:\n",
        INPUT_PATH
    )
}

x <- read.csv(
    INPUT_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(x) != 18L) {
    stop(
        "Expected 18 focused records; observed ",
        nrow(x)
    )
}

cat("STEP10E.3 focused records: 18 — PASS\n")

# --------------------------------------------------------------------------------------------------
# CLEAN IDENTIFIERS
# --------------------------------------------------------------------------------------------------

clean <- function(z) {
    z <- as.character(z)
    z[is.na(z)] <- ""
    trimws(z)
}

for (nm in intersect(
    c(
        "ensemblID_stable",
        "interpretation_symbol",
        "pmid",
        "title",
        "pub_year"
    ),
    names(x)
)) {
    x[[nm]] <- clean(x[[nm]])
}

# --------------------------------------------------------------------------------------------------
# INITIALIZE MANUAL FIELDS
# --------------------------------------------------------------------------------------------------

x$manual_study_design <- NA_character_
x$manual_relevance_class <- NA_character_
x$manual_evidence_direction <- NA_character_
x$manual_evidence_strength <- NA_character_
x$independent_primary_evidence <- FALSE
x$duplicate_of <- NA_character_
x$manual_population_context <- NA_character_
x$manual_review_note <- NA_character_

# --------------------------------------------------------------------------------------------------
# HELPER
# --------------------------------------------------------------------------------------------------

set_record <- function(
    idx,
    design,
    relevance,
    direction,
    strength,
    primary,
    population,
    note,
    duplicate_of = NA_character_
) {

    x$manual_study_design[idx] <<- design
    x$manual_relevance_class[idx] <<- relevance
    x$manual_evidence_direction[idx] <<- direction
    x$manual_evidence_strength[idx] <<- strength
    x$independent_primary_evidence[idx] <<- primary
    x$manual_population_context[idx] <<- population
    x$manual_review_note[idx] <<- note
    x$duplicate_of[idx] <<- duplicate_of
}

find_pmid <- function(pmid) {

    which(
        x$pmid == as.character(pmid)
    )
}

find_title <- function(pattern) {

    which(
        grepl(
            pattern,
            x$title,
            ignore.case = TRUE,
            perl = TRUE
        )
    )
}

# ==================================================================================================
# ARHGAP11B
# ==================================================================================================

i <- find_pmid("30886340")

set_record(
    i,
    "Rare CNV/variant study in discordant monozygotic twins",
    "DIRECT_HUMAN_RARE_CNV_OBSERVATION",
    "WEAK_SUPPORTIVE",
    "LOW",
    TRUE,
    "Human twins; heterogeneous clinical phenotypes",
    paste(
        "Rare deletion overlapping ARHGAP11B observed in a twin pair manifesting",
        "schizotypal personality disorder or schizophrenia.",
        "Interesting direct observation but extremely limited sample/context;",
        "not evidence of general schizophrenia association."
    )
)

i <- find_pmid("33510659")

set_record(
    i,
    "Clinical CNV genotype-phenotype study",
    "CROSS_DISORDER_CNV_CONTEXT",
    "UNCERTAIN_SUPPORT",
    "LOW",
    TRUE,
    "Early-onset schizophrenia and early-onset bipolar disorder",
    paste(
        "ARHGAP11B-containing CNV was reported in the bipolar-disorder component;",
        "authors discuss schizophrenia and/or bipolar association.",
        "Not clean schizophrenia-specific gene-level evidence."
    )
)

i <- find_pmid("27244233")

set_record(
    i,
    "Pilot genome-wide CNV study",
    "SCHIZOPHRENIA_CNV_LOCUS_LEVEL_CONTEXT",
    "WEAK_SUPPORTIVE",
    "LOW",
    TRUE,
    "Human schizophrenia/bipolar case-control CNV analysis",
    paste(
        "A 15q13.2 CNV region containing ARHGAP11B and FAN1 appeared schizophrenia-specific.",
        "Evidence is locus-level rather than uniquely attributable to ARHGAP11B."
    )
)

# ==================================================================================================
# ALDH1A1
# ==================================================================================================

i <- find_pmid("27236410")

set_record(
    i,
    "Human postmortem DLPFC proteomics with independent immunoblot validation",
    "DIRECT_HUMAN_BRAIN_SCHIZOPHRENIA_EXPRESSION_TEST",
    "NULL_NEGATIVE",
    "MODERATE",
    TRUE,
    "Human postmortem DLPFC; schizophrenia versus controls",
    paste(
        "ALDH1A1 was selected for validation, but did not significantly differ",
        "in the independent schizophrenia/control cohort.",
        "This is directly relevant null evidence."
    )
)

i <- find_pmid("19703508")

set_record(
    i,
    "Human candidate-gene association study",
    "DIRECT_HUMAN_SCHIZOPHRENIA_GENETIC_TEST",
    "NULL_NEGATIVE",
    "MODERATE",
    TRUE,
    "Chinese Han schizophrenia case-control sample",
    paste(
        "ALDH1A1 was one of seven retinoid-pathway genes tested.",
        "Single-gene analyses showed no significant schizophrenia association.",
        "The positive result concerned ALDH1A2, which must not be attributed to ALDH1A1."
    )
)

i <- find_pmid("24494173")

set_record(
    i,
    "Mouse double-knockout behavioral study",
    "ANIMAL_MODEL_DIRECT_FUNCTIONAL_TEST",
    "NULL_NEGATIVE",
    "LOW",
    TRUE,
    "Aldh1a1/Gpx1 knockout mice",
    paste(
        "Deletion did not produce schizophrenia-like behavioral phenotypes.",
        "Useful negative functional context but not direct human schizophrenia evidence."
    )
)

i <- find_pmid("30678040")

set_record(
    i,
    "Mouse schizophrenia-like model proteomics",
    "INDIRECT_ANIMAL_MODEL_CONTEXT",
    "INDIRECT_SUPPORTIVE_CONTEXT",
    "LOW",
    TRUE,
    "Fgf14 knockout mouse model",
    paste(
        "ALDH1A1 decreased in a schizophrenia-like mouse model.",
        "Mechanistic/contextual evidence only; does not establish human schizophrenia association."
    )
)

i <- find_pmid("41263186")

set_record(
    i,
    "Mouse thalamic glia circuit/mechanism study",
    "INDIRECT_NEUROBIOLOGICAL_CONTEXT",
    "INDIRECT_SUPPORTIVE_CONTEXT",
    "LOW",
    TRUE,
    "Mouse thalamus; VRK2-deficient model",
    paste(
        "DAO-ALDH1A1 astrocytic GABA pathway was downregulated after VRK2 loss.",
        "VRK2 is linked to schizophrenia risk, but this does not directly implicate",
        "ALDH1A1 as a schizophrenia-risk or schizophrenia-expression gene."
    )
)

i <- find_pmid("41006064")

set_record(
    i,
    "Mouse midbrain dopamine-neuron electrophysiology study",
    "BRAIN_FUNCTION_CONTEXT_ONLY",
    "NOT_DIRECT_SCZ_EVIDENCE",
    "LOW",
    TRUE,
    "Adult mouse substantia nigra dopamine neurons",
    paste(
        "Defines Aldh1a1-positive dopamine-neuron physiology.",
        "Schizophrenia appears only as broader disease context.",
        "Useful for biological interpretation, not schizophrenia association."
    )
)

# ALDH1A1 2018 non-PMID duplicate/preliminary version
i <- which(
    x$pmid == "" &
    grepl(
        "Sex-Specific Proteomics Changes Induced",
        x$title,
        ignore.case = TRUE
    )
)

if (length(i) != 1L) {
    stop("Could not uniquely identify ALDH1A1 duplicate/preliminary record.")
}

set_record(
    i,
    "Earlier/preliminary report of FGF14 knockout proteomics study",
    "DUPLICATE_OR_PRELIMINARY_REPORT",
    "DUPLICATE",
    "EXCLUDE",
    FALSE,
    "Mouse model",
    "Same underlying ALDH1A1/Fgf14 model evidence as PMID 30678040; do not count independently.",
    duplicate_of = "PMID:30678040"
)

# ==================================================================================================
# NR4A2
# ==================================================================================================

i <- find_pmid("12627459")

set_record(
    i,
    "Human schizophrenia case-control genetic association study",
    "DIRECT_HUMAN_SCHIZOPHRENIA_GENETIC_TEST",
    "NULL_NEGATIVE",
    "MODERATE",
    TRUE,
    "Japanese schizophrenia cases and controls",
    paste(
        "Three common NR4A2 variants/haplotypes were tested.",
        "Study did not support a major schizophrenia-risk role."
    )
)

i <- find_pmid("33563249")

set_record(
    i,
    "Human schizophrenia case-control genetics plus PBMC expression/cognition study",
    "DIRECT_HUMAN_SCHIZOPHRENIA_CLINICAL_MOLECULAR_TEST",
    "MIXED",
    "MODERATE",
    TRUE,
    "Mexican schizophrenia cases and controls",
    paste(
        "Variants and overall NR4A2 expression did not differ significantly between",
        "patients and controls, arguing against simple susceptibility association.",
        "However, genotype-dependent relationships with working memory/expression were observed."
    )
)

i <- find_pmid("25982322")

set_record(
    i,
    "Systematic review and meta-analysis of schizophrenia genetic association",
    "DIRECT_HUMAN_SCHIZOPHRENIA_GENETIC_META_ANALYSIS",
    "NULL_NEGATIVE",
    "HIGH",
    FALSE,
    "Seven studies; 3027 participants",
    paste(
        "Meta-analysis found no significant association between analyzed NR4A2 variants",
        "and schizophrenia risk.",
        "Strongest retrieved evidence against a robust common-variant risk association."
    )
)

# NR4A2 2020 preprint/duplicate of 2021 paper
i <- which(
    x$pmid == "" &
    grepl(
        "Working Memory Deficits in Schizophrenia",
        x$title,
        ignore.case = TRUE
    )
)

if (length(i) != 1L) {
    stop("Could not uniquely identify NR4A2 duplicate/preprint.")
}

set_record(
    i,
    "Earlier/preprint version of later published case-control study",
    "DUPLICATE_OR_PRELIMINARY_REPORT",
    "DUPLICATE",
    "EXCLUDE",
    FALSE,
    "Human schizophrenia case-control",
    "Same study as PMID 33563249; do not count independently.",
    duplicate_of = "PMID:33563249"
)

i <- find_pmid("38326562")

set_record(
    i,
    "Human blood biomarker / precision-psychiatry study",
    "HUMAN_PSYCHOSIS_BIOMARKER_CONTEXT",
    "SUPPORTIVE_BUT_NOT_RISK_EVIDENCE",
    "LOW_TO_MODERATE",
    TRUE,
    "Human psychotic disorders; peripheral blood biomarkers",
    paste(
        "NR4A2 was reported among top biomarkers for delusions.",
        "Supports psychosis-related molecular context, but is not direct evidence",
        "of brain expression or causal schizophrenia susceptibility."
    )
)

i <- find_pmid("36993785")

set_record(
    i,
    "Secondary GEO transcriptomic machine-learning analysis",
    "COMPUTATIONAL_SCHIZOPHRENIA_TRANSCRIPTOMIC_CONTEXT",
    "SUPPORTIVE_CONTEXT",
    "LOW",
    TRUE,
    "Public schizophrenia gene-expression datasets",
    paste(
        "NR4A2 selected among candidate hub genes in a computational analysis.",
        "Useful secondary transcriptomic context but not independent causal/genetic evidence."
    )
)

i <- find_pmid("41009723")

set_record(
    i,
    "Narrative/review article",
    "REVIEW_CONTEXT",
    "CONTEXT_ONLY",
    "LOW",
    FALSE,
    "Review of NR4A2 regulation and CNS disease",
    paste(
        "Provides biological background linking NR4A2 to neuronal development and",
        "psychiatric disorders including schizophrenia.",
        "Not independent primary evidence."
    )
)

i <- find_pmid("37017099")

set_record(
    i,
    "Human family-based genome-wide DNA methylation study",
    "DIRECT_HUMAN_SCHIZOPHRENIA_EPIGENETIC_ASSOCIATION",
    "SUPPORTIVE",
    "MODERATE",
    TRUE,
    "Families with schizophrenia and intellectual disability",
    paste(
        "Altered NR4A2 methylation was reported in association with schizophrenia/ID.",
        "Direct human molecular association, but phenotype includes intellectual disability",
        "and does not establish schizophrenia-specific causality."
    )
)

# --------------------------------------------------------------------------------------------------
# VERIFY ALL 18 RECORDS WERE ADJUDICATED
# --------------------------------------------------------------------------------------------------

required_manual_fields <- c(
    "manual_study_design",
    "manual_relevance_class",
    "manual_evidence_direction",
    "manual_evidence_strength",
    "manual_population_context",
    "manual_review_note"
)

for (nm in required_manual_fields) {

    if (any(
        is.na(x[[nm]]) |
        x[[nm]] == ""
    )) {

        bad <- which(
            is.na(x[[nm]]) |
            x[[nm]] == ""
        )

        stop(
            "Unadjudicated record(s) remain in ",
            nm,
            ": ",
            paste(bad, collapse = ", ")
        )
    }
}

cat("All 18 records manually adjudicated: PASS\n")

# --------------------------------------------------------------------------------------------------
# INDEPENDENT PRIMARY EVIDENCE SUBSET
#
# Excludes:
#   - duplicates/preliminary reports
#   - reviews/meta-analysis from "primary" subset
#
# Meta-analysis remains in full evidence table.
# --------------------------------------------------------------------------------------------------

primary_subset <- x[
    x$independent_primary_evidence &
    x$manual_evidence_direction != "DUPLICATE",
    ,
    drop = FALSE
]

write.csv(
    x,
    OUT_RECORDS,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    primary_subset,
    OUT_PRIMARY,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# GENE-LEVEL SYNTHESIS
# --------------------------------------------------------------------------------------------------

gene_synthesis <- data.frame(

    ensemblID_stable = c(
        "ENSG00000187951",
        "ENSG00000165092",
        "ENSG00000153234"
    ),

    interpretation_symbol = c(
        "ARHGAP11B",
        "ALDH1A1",
        "NR4A2"
    ),

    identity_caution = c(
        paste(
            "Historical BrainSeq/GRCh37 identity.",
            "Frozen ENSG00000187951 was annotated as ARHGAP11B;",
            "current ARHGAP11B Ensembl ID is ENSG00000285077."
        ),
        "None beyond standard current identity.",
        "None beyond standard current identity."
    ),

    overall_direct_SCZ_evidence = c(
        "WEAK_AND_UNCERTAIN",
        "MOSTLY_NULL_OR_NEGATIVE",
        "MIXED_WITH_GENETIC_RISK_LARGELY_NULL"
    ),

    direct_human_evidence_summary = c(
        paste(
            "Rare/CNV observations provide limited locus-level or case-level support,",
            "but evidence is sparse, sometimes cross-disorder, and not robustly gene-specific."
        ),
        paste(
            "Human postmortem DLPFC validation showed no significant ALDH1A1 change;",
            "candidate-gene association testing also showed no significant ALDH1A1 association."
        ),
        paste(
            "Multiple human studies directly examine NR4A2.",
            "Case-control and meta-analytic genetic susceptibility results are largely null,",
            "although cognition/expression, psychosis biomarker, and epigenetic associations exist."
        )
    ),

    neurobiological_context = c(
        paste(
            "Known developmental/cortical biology may be biologically interesting,",
            "but the retrieved schizophrenia evidence remains limited."
        ),
        paste(
            "Substantial dopamine/GABA and brain-cell functional context exists,",
            "including Aldh1a1-positive dopamine neurons and astrocytic GABA pathways."
        ),
        paste(
            "Strong dopaminergic-neuron developmental and transcriptional context;",
            "NR4A2/Nurr1 is biologically relevant to dopamine-system function."
        )
    ),

    manuscript_safe_interpretation = c(
        paste(
            "ARHGAP11B has limited human CNV/locus-level schizophrenia-related evidence,",
            "which should be described cautiously because of sparse evidence and",
            "historical identifier complexity."
        ),
        paste(
            "ALDH1A1 has plausible dopamine/GABA-related neurobiological context,",
            "but the retrieved direct human schizophrenia studies are predominantly null;",
            "therefore it should not be presented as an established schizophrenia-associated gene."
        ),
        paste(
            "NR4A2 has substantial schizophrenia-relevant and dopaminergic biological literature,",
            "but direct common-variant susceptibility evidence is mixed to null.",
            "It is best described as a biologically plausible schizophrenia-related gene",
            "with context-dependent molecular and cognitive associations rather than",
            "an established schizophrenia-risk gene."
        )
    ),

    stringsAsFactors = FALSE
)

write.csv(
    gene_synthesis,
    OUT_GENES,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# SUMMARY COUNTS
# --------------------------------------------------------------------------------------------------

direction_counts <- table(
    x$manual_evidence_direction
)

strength_counts <- table(
    x$manual_evidence_strength
)

relevance_counts <- table(
    x$manual_relevance_class
)

n_duplicates <- sum(
    x$manual_evidence_direction == "DUPLICATE"
)

n_null_negative <- sum(
    x$manual_evidence_direction == "NULL_NEGATIVE"
)

n_supportive_direct <- sum(
    x$manual_evidence_direction %in%
        c(
            "SUPPORTIVE",
            "WEAK_SUPPORTIVE"
        )
)

n_mixed <- sum(
    x$manual_evidence_direction == "MIXED"
)

# --------------------------------------------------------------------------------------------------
# HUMAN-READABLE SUMMARY
# --------------------------------------------------------------------------------------------------

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10E.4 — MANUAL LITERATURE EVIDENCE ADJUDICATION",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    paste0(
        "Focused records adjudicated: ",
        nrow(x)
    ),

    paste0(
        "Duplicate/preliminary reports excluded from independent counting: ",
        n_duplicates
    ),

    paste0(
        "Independent primary evidence records retained: ",
        nrow(primary_subset)
    ),

    "",

    "EVIDENCE-DIRECTION COUNTS",

    paste0(
        names(direction_counts),
        ": ",
        as.integer(direction_counts)
    ),

    "",

    "KEY GENE-LEVEL CONCLUSIONS",

    "ARHGAP11B:",
    paste(
        "Evidence is weak and uncertain, mainly rare/CNV/locus-level;",
        "historical Ensembl-ID handling remains essential."
    ),

    "",

    "ALDH1A1:",
    paste(
        "Direct human schizophrenia evidence is predominantly null.",
        "Neurobiological dopamine/GABA context exists, but this should not be converted",
        "into a claim of established schizophrenia association."
    ),

    "",

    "NR4A2:",
    paste(
        "Schizophrenia-relevant literature is substantial, but direct susceptibility genetics",
        "is mixed to null, including a null meta-analysis.",
        "Clinical/cognitive, biomarker, transcriptomic, and epigenetic context provide",
        "additional but heterogeneous support."
    ),

    "",

    "INTERPRETATION RULE",

    paste(
        "Literature context is secondary biological interpretation of the frozen discovery signature.",
        "It does not independently validate the 24-gene score and does not override",
        "the null external score result."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    INPUT_PATH,
    OUT_RECORDS,
    OUT_PRIMARY,
    OUT_GENES,
    OUT_SUMMARY
)

manifest <- data.frame(
    file = basename(manifest_paths),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(manifest_paths)$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nMANUAL EVIDENCE ADJUDICATION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Records adjudicated: ",
    nrow(x),
    "\n",
    sep = ""
)

cat(
    "Independent primary records: ",
    nrow(primary_subset),
    "\n",
    sep = ""
)

cat(
    "Duplicate/preliminary records: ",
    n_duplicates,
    "\n",
    sep = ""
)

cat(
    "\nEVIDENCE DIRECTION\n"
)

print(
    direction_counts
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nGENE-LEVEL SYNTHESIS\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    gene_synthesis,
    row.names = FALSE
)

cat(
    "\nAdjudicated 18-record table:\n",
    OUT_RECORDS,
    "\n",
    sep = ""
)

cat(
    "\nIndependent primary evidence subset:\n",
    OUT_PRIMARY,
    "\n",
    sep = ""
)

cat(
    "\nGene-level synthesis:\n",
    OUT_GENES,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10E.4 COMPLETE — MANUAL LITERATURE EVIDENCE ADJUDICATION FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10E.4 — MANUAL PAPER-LEVEL EVIDENCE ADJUDICATION
Project sentinel: PASS
STEP10E.3 focused records: 18 — PASS
All 18 records manually adjudicated: PASS

MANUAL EVIDENCE ADJUDICATION SUMMARY
Records adjudicated: 18
Independent primary records: 14
Duplicate/preliminary records: 2

EVIDENCE DIRECTION

                    CONTEXT_ONLY                        DUPLICATE 
                               1                                2 
     INDIRECT_SUPPORTIVE_CONTEXT                            MIXED 
                               2                                1 
         NOT_DIRECT_SCZ_EVIDENCE                    NULL_NEGATIVE 
                               1                                5 
                      SUPPORTIVE SUPPORTIVE_BUT_NOT_RISK_EVIDENCE 
                               1                                1 
              SUPPORTIVE_CONTEXT                UNCERTAIN_SUPPORT 
                               1                                1 
                 WEAK_SUP

In [72]:
# ==================================================================================================
# STEP 10F.1 — EXPANDED BRAIN / NEURAL / CELL-TYPE LITERATURE REVIEW PACKET
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Build a reproducible paper-level biological-context review set for the frozen SCZ-MSS genes
#   that showed brain/neural literature hits in STEP10E.1.
#
# WHY THIS STEP IS REQUIRED
#   STEP10E.1 showed literature hit COUNTS only.
#   STEP10E.4 adjudicated schizophrenia/genetic-focused evidence only.
#
#   Before final 24-gene biological synthesis, we still need to determine:
#       - whether each gene has direct brain evidence
#       - which tissue/cell type is involved
#       - whether evidence is human or experimental/animal
#       - whether the context is cortex / hippocampus / dopamine / glia / development etc.
#
# DESIGN
#   - Re-use the EXACT frozen BRAIN_NEURAL_CELLTYPE queries from STEP10E.1.
#   - Expand retrieval to top 10 Europe PMC records per eligible feature.
#   - Deduplicate within feature.
#   - Carry forward papers already manually adjudicated in STEP10E.4.
#   - Create a NEW review queue excluding those already reviewed papers.
#   - Print at most TOP 3 new brain-context papers per feature.
#
# IMPORTANT
#   - This is evidence RETRIEVAL / TRIAGE only.
#   - No paper is automatically declared supportive.
#   - No cell-type conclusion is automatically assigned.
#   - Historical ARHGAP11B identity remains explicitly flagged.
#   - Tier C/D stable-ID-limited features are not converted into named-gene claims.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10F.1 — EXPANDED BRAIN / NEURAL / CELL-TYPE LITERATURE REVIEW PACKET\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10F.1A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

IDENTITY_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "identity_audit_v1",
    "adjudicated_v1",
    "STEP10D2_SCZ_MSS_24gene_interpretation_identity_v1.csv"
)

QUERY_REGISTRY_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "STEP10E1_literature_query_registry_v1.csv"
)

GENE_LIT_SUMMARY_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "STEP10E1_gene_level_literature_summary_v1.csv"
)

SCZ_ADJUDICATED_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "evidence_review_v1",
    "manual_adjudication_v1",
    "STEP10E4_manual_paper_evidence_adjudication_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "brain_context_review_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

EXPANDED_RECORDS_PATH <- file.path(
    OUT_DIR,
    "STEP10F1_expanded_brain_neural_literature_records_v1.csv"
)

NEW_REVIEW_QUEUE_PATH <- file.path(
    OUT_DIR,
    "STEP10F1_new_brain_neural_manual_review_queue_v1.csv"
)

TOP_REVIEW_PACKET_PATH <- file.path(
    OUT_DIR,
    "STEP10F1_top_brain_neural_review_packet_v1.csv"
)

TOP_REVIEW_TXT_PATH <- file.path(
    OUT_DIR,
    "STEP10F1_top_brain_neural_review_packet_v1.txt"
)

GENE_SUMMARY_PATH <- file.path(
    OUT_DIR,
    "STEP10F1_gene_level_brain_context_retrieval_summary_v1.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "STEP10F1_brain_context_review_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "STEP10F1_brain_context_review_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10F.1B — PROJECT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop(
        "Project sentinel missing: ",
        SENTINEL_PATH
    )
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

required_files <- c(
    IDENTITY_PATH,
    QUERY_REGISTRY_PATH,
    GENE_LIT_SUMMARY_PATH,
    SCZ_ADJUDICATED_PATH
)

missing_files <- required_files[
    !file.exists(
        required_files
    )
]

if (length(missing_files) > 0L) {

    stop(
        "Required upstream file(s) missing:\n",
        paste(
            missing_files,
            collapse = "\n"
        )
    )
}

cat("STEP10D.2 identity freeze: PASS\n")
cat("STEP10E.1 literature query registry: PASS\n")
cat("STEP10E.1 gene literature summary: PASS\n")
cat("STEP10E.4 manual SCZ adjudication: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10F.1C — LOAD INPUTS
# --------------------------------------------------------------------------------------------------

identity <- read.csv(
    IDENTITY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

query_registry <- read.csv(
    QUERY_REGISTRY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

lit_summary <- read.csv(
    GENE_LIT_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

scz_adjudicated <- read.csv(
    SCZ_ADJUDICATED_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(identity) != 24L) {
    stop(
        "Expected 24 frozen adjudicated features; observed ",
        nrow(identity)
    )
}

if (nrow(query_registry) != 72L) {
    stop(
        "Expected 72 frozen literature queries; observed ",
        nrow(query_registry)
    )
}

if (nrow(lit_summary) != 24L) {
    stop(
        "Expected 24 gene-literature summary rows; observed ",
        nrow(lit_summary)
    )
}

if (nrow(scz_adjudicated) != 18L) {
    stop(
        "Expected 18 STEP10E.4 adjudicated records; observed ",
        nrow(scz_adjudicated)
    )
}

cat("Frozen feature count: 24 — PASS\n")
cat("Frozen literature queries: 72 — PASS\n")
cat("SCZ-specific adjudicated records: 18 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10F.1D — CLEAN TEXT
# --------------------------------------------------------------------------------------------------

clean_text <- function(x) {

    x <- as.character(x)

    x[
        is.na(x)
    ] <- ""

    trimws(x)
}

for (nm in intersect(
    c(
        "ensemblID_stable",
        "gene_symbol",
        "interpretation_symbol",
        "literature_interpretation_tier",
        "manuscript_gene_label"
    ),
    names(identity)
)) {

    identity[[nm]] <- clean_text(
        identity[[nm]]
    )
}

for (nm in intersect(
    c(
        "ensemblID_stable",
        "evidence_domain",
        "europe_pmc_query"
    ),
    names(query_registry)
)) {

    query_registry[[nm]] <- clean_text(
        query_registry[[nm]]
    )
}

for (nm in intersect(
    c(
        "ensemblID_stable",
        "interpretation_symbol"
    ),
    names(lit_summary)
)) {

    lit_summary[[nm]] <- clean_text(
        lit_summary[[nm]]
    )
}

# --------------------------------------------------------------------------------------------------
# 10F.1E — NORMALIZE LOGICAL REVIEWABILITY
# --------------------------------------------------------------------------------------------------

if (!is.logical(
    identity$allow_gene_specific_biological_claim
)) {

    z <- toupper(
        trimws(
            as.character(
                identity$allow_gene_specific_biological_claim
            )
        )
    )

    if (!all(
        z %in% c(
            "TRUE",
            "FALSE"
        )
    )) {

        stop(
            "Cannot interpret allow_gene_specific_biological_claim."
        )
    }

    identity$allow_gene_specific_biological_claim <- (
        z == "TRUE"
    )
}

# --------------------------------------------------------------------------------------------------
# 10F.1F — DEFINE ELIGIBLE FEATURES
#
# Include:
#   Tier A named-gene features with >=1 brain/neural hit.
#
# Also include:
#   historical Tier B ARHGAP11B if it has >=1 brain/neural hit,
#   but keep explicit identity caution.
#
# Tier C/D remain ID-limited and are not converted to named gene biology.
# --------------------------------------------------------------------------------------------------

eligible <- merge(
    identity,
    lit_summary[
        ,
        c(
            "ensemblID_stable",
            "brain_neural_celltype_hits"
        )
    ],
    by = "ensemblID_stable",
    all.x = TRUE,
    sort = FALSE
)

eligible <- eligible[
    match(
        identity$ensemblID_stable,
        eligible$ensemblID_stable
    ),
    ,
    drop = FALSE
]

eligible$brain_neural_celltype_hits[
    is.na(
        eligible$brain_neural_celltype_hits
    )
] <- 0

eligible$brain_review_eligible <- (
    (
        eligible$allow_gene_specific_biological_claim
    ) |
    (
        eligible$literature_interpretation_tier ==
            "TIER_B_HISTORICAL_IDENTITY_SPECIAL_HANDLING"
    )
) &
    eligible$brain_neural_celltype_hits > 0

eligible_features <- eligible[
    eligible$brain_review_eligible,
    ,
    drop = FALSE
]

n_eligible <- nrow(
    eligible_features
)

cat(
    "\nFeatures eligible for brain/neural paper review: ",
    n_eligible,
    "\n",
    sep = ""
)

if (n_eligible == 0L) {
    stop(
        "No features eligible for brain/neural literature review."
    )
}

print(
    eligible_features[
        ,
        c(
            "ensemblID_stable",
            "interpretation_symbol",
            "literature_interpretation_tier",
            "brain_neural_celltype_hits"
        )
    ],
    row.names = FALSE
)

# --------------------------------------------------------------------------------------------------
# 10F.1G — EXTRACT FROZEN BRAIN QUERIES
# --------------------------------------------------------------------------------------------------

brain_queries <- query_registry[
    query_registry$evidence_domain ==
        "BRAIN_NEURAL_CELLTYPE" &
    query_registry$ensemblID_stable %in%
        eligible_features$ensemblID_stable,
    ,
    drop = FALSE
]

if (
    nrow(brain_queries) !=
    n_eligible
) {

    stop(
        "Expected one frozen brain query per eligible feature.\n",
        "Eligible features: ",
        n_eligible,
        "\nBrain queries found: ",
        nrow(brain_queries)
    )
}

cat(
    "Frozen brain/neural queries selected: ",
    nrow(brain_queries),
    " — PASS\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10F.1H — REQUIRED PACKAGES
# --------------------------------------------------------------------------------------------------

needed <- c(
    "httr",
    "jsonlite"
)

missing_packages <- needed[
    !vapply(
        needed,
        requireNamespace,
        quietly = TRUE,
        FUN.VALUE = logical(1)
    )
]

if (length(missing_packages) > 0L) {

    install.packages(
        missing_packages,
        repos = "https://cloud.r-project.org"
    )
}

if (!requireNamespace("httr", quietly = TRUE)) {
    stop("httr unavailable.")
}

if (!requireNamespace("jsonlite", quietly = TRUE)) {
    stop("jsonlite unavailable.")
}

# --------------------------------------------------------------------------------------------------
# 10F.1I — EUROPE PMC HELPERS
# --------------------------------------------------------------------------------------------------

EUROPE_PMC_ENDPOINT <-
    "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

safe_scalar <- function(x) {

    if (
        is.null(x) ||
        length(x) == 0L
    ) {
        return("")
    }

    x <- as.character(x)

    if (length(x) > 1L) {

        x <- paste(
            x,
            collapse = "; "
        )
    }

    if (
        is.na(x) ||
        !nzchar(
            trimws(x)
        )
    ) {
        return("")
    }

    trimws(x)
}

fetch_epmc <- function(
    query_text,
    page_size = 10L
) {

    last_error <- NULL

    for (attempt in 1:2) {

        result <- tryCatch(
            {

                response <- httr::GET(
                    EUROPE_PMC_ENDPOINT,
                    query = list(
                        query = query_text,
                        format = "json",
                        resultType = "core",
                        pageSize = page_size
                    ),
                    httr::timeout(45)
                )

                status <- httr::status_code(
                    response
                )

                if (status != 200L) {
                    stop(
                        "Europe PMC HTTP status ",
                        status
                    )
                }

                payload <- jsonlite::fromJSON(
                    httr::content(
                        response,
                        as = "text",
                        encoding = "UTF-8"
                    ),
                    simplifyVector = FALSE
                )

                raw_results <- NULL

                if (
                    !is.null(
                        payload$resultList
                    ) &&
                    !is.null(
                        payload$resultList$result
                    )
                ) {

                    raw_results <-
                        payload$resultList$result
                }

                list(
                    success = TRUE,
                    http_status = status,
                    hit_count =
                        if (!is.null(payload$hitCount)) {
                            as.numeric(
                                payload$hitCount
                            )
                        } else {
                            NA_real_
                        },
                    results = raw_results,
                    error = ""
                )
            },

            error = function(e) {

                last_error <<-
                    conditionMessage(e)

                NULL
            }
        )

        if (!is.null(result)) {
            return(result)
        }

        if (attempt < 2L) {
            Sys.sleep(1)
        }
    }

    list(
        success = FALSE,
        http_status = NA_integer_,
        hit_count = NA_real_,
        results = NULL,
        error = last_error
    )
}

flatten_record <- function(
    record,
    id,
    query_rank
) {

    data.frame(

        ensemblID_stable = id,

        query_rank = query_rank,

        source = safe_scalar(
            record$source
        ),

        external_id = safe_scalar(
            record$id
        ),

        pmid = safe_scalar(
            record$pmid
        ),

        pmcid = safe_scalar(
            record$pmcid
        ),

        doi = safe_scalar(
            record$doi
        ),

        title = safe_scalar(
            record$title
        ),

        author_string = safe_scalar(
            record$authorString
        ),

        journal_title = safe_scalar(
            record$journalTitle
        ),

        pub_year = safe_scalar(
            record$pubYear
        ),

        publication_type = safe_scalar(
            record$pubType
        ),

        cited_by_count = suppressWarnings(
            as.numeric(
                safe_scalar(
                    record$citedByCount
                )
            )
        ),

        abstract_text = safe_scalar(
            record$abstractText
        ),

        stringsAsFactors = FALSE
    )
}

# --------------------------------------------------------------------------------------------------
# 10F.1J — EXPANDED BRAIN RETRIEVAL
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nEXPANDED BRAIN/NEURAL EUROPE PMC RETRIEVAL\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

expanded_rows <- list()
query_audit_rows <- list()

expanded_index <- 1L

for (q in seq_len(nrow(brain_queries))) {

    id <- brain_queries$ensemblID_stable[q]

    query_text <-
        brain_queries$europe_pmc_query[q]

    res <- fetch_epmc(
        query_text,
        page_size = 10L
    )

    query_audit_rows[[q]] <- data.frame(
        ensemblID_stable = id,
        success = res$success,
        http_status = res$http_status,
        hit_count = res$hit_count,
        error = res$error,
        stringsAsFactors = FALSE
    )

    if (
        isTRUE(res$success) &&
        !is.null(res$results) &&
        length(res$results) > 0L
    ) {

        n_use <- min(
            10L,
            length(res$results)
        )

        for (r in seq_len(n_use)) {

            expanded_rows[[expanded_index]] <-
                flatten_record(
                    res$results[[r]],
                    id = id,
                    query_rank = r
                )

            expanded_index <-
                expanded_index + 1L
        }
    }

    Sys.sleep(0.12)
}

query_audit <- do.call(
    rbind,
    query_audit_rows
)

if (!all(
    query_audit$success
)) {

    failed <- query_audit[
        !query_audit$success,
        ,
        drop = FALSE
    ]

    print(failed)

    stop(
        "One or more expanded brain literature queries failed."
    )
}

cat(
    "Expanded brain queries successful: ",
    nrow(query_audit),
    "/",
    nrow(query_audit),
    "\n",
    sep = ""
)

if (length(expanded_rows) == 0L) {
    stop(
        "Expanded brain retrieval returned zero records."
    )
}

expanded <- do.call(
    rbind,
    expanded_rows
)

rownames(expanded) <- NULL

# --------------------------------------------------------------------------------------------------
# 10F.1K — ATTACH IDENTITY METADATA
# --------------------------------------------------------------------------------------------------

identity_meta <- identity[
    ,
    c(
        "ensemblID_stable",
        "gene_symbol",
        "interpretation_symbol",
        "manuscript_gene_label",
        "literature_interpretation_tier",
        "allow_gene_specific_biological_claim",
        "direction_standardized",
        "final_normalized_weight"
    )
]

expanded <- merge(
    expanded,
    identity_meta,
    by = "ensemblID_stable",
    all.x = TRUE,
    sort = FALSE
)

# --------------------------------------------------------------------------------------------------
# 10F.1L — DEDUPLICATION WITHIN FEATURE
# --------------------------------------------------------------------------------------------------

normalize_title <- function(x) {

    x <- tolower(
        trimws(
            as.character(x)
        )
    )

    x <- gsub(
        "[^a-z0-9]+",
        " ",
        x
    )

    x <- gsub(
        "\\s+",
        " ",
        x
    )

    trimws(x)
}

expanded$normalized_title <- normalize_title(
    expanded$title
)

expanded$paper_key <- ifelse(
    nzchar(expanded$pmid),
    paste0(
        "PMID:",
        expanded$pmid
    ),
    ifelse(
        nzchar(expanded$doi),
        paste0(
            "DOI:",
            tolower(
                expanded$doi
            )
        ),
        paste0(
            "TITLE:",
            expanded$normalized_title
        )
    )
)

expanded$gene_paper_key <- paste(
    expanded$ensemblID_stable,
    expanded$paper_key,
    sep = "||"
)

expanded <- expanded[
    !duplicated(
        expanded$gene_paper_key
    ),
    ,
    drop = FALSE
]

cat(
    "Expanded unique feature-paper records: ",
    nrow(expanded),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10F.1M — BUILD MATCH KEYS FOR ALREADY-ADJUDICATED SCZ PAPERS
# --------------------------------------------------------------------------------------------------

for (nm in c(
    "pmid",
    "doi",
    "title"
)) {

    if (!nm %in% names(
        scz_adjudicated
    )) {

        scz_adjudicated[[nm]] <- ""
    }

    scz_adjudicated[[nm]] <-
        clean_text(
            scz_adjudicated[[nm]]
        )
}

scz_adjudicated$normalized_title <-
    normalize_title(
        scz_adjudicated$title
    )

scz_adjudicated$paper_key <- ifelse(
    nzchar(
        scz_adjudicated$pmid
    ),
    paste0(
        "PMID:",
        scz_adjudicated$pmid
    ),
    ifelse(
        nzchar(
            scz_adjudicated$doi
        ),
        paste0(
            "DOI:",
            tolower(
                scz_adjudicated$doi
            )
        ),
        paste0(
            "TITLE:",
            scz_adjudicated$normalized_title
        )
    )
)

scz_adjudicated$gene_paper_key <-
    paste(
        scz_adjudicated$ensemblID_stable,
        scz_adjudicated$paper_key,
        sep = "||"
    )

expanded$already_reviewed_in_STEP10E4 <- (
    expanded$gene_paper_key %in%
        scz_adjudicated$gene_paper_key
)

# --------------------------------------------------------------------------------------------------
# 10F.1N — CARRY FORWARD PREVIOUS MANUAL EVIDENCE CLASS
# --------------------------------------------------------------------------------------------------

expanded$STEP10E4_relevance_class <- ""
expanded$STEP10E4_evidence_direction <- ""
expanded$STEP10E4_review_note <- ""

for (i in seq_len(nrow(expanded))) {

    j <- match(
        expanded$gene_paper_key[i],
        scz_adjudicated$gene_paper_key
    )

    if (!is.na(j)) {

        expanded$STEP10E4_relevance_class[i] <-
            clean_text(
                scz_adjudicated$manual_relevance_class[j]
            )

        expanded$STEP10E4_evidence_direction[i] <-
            clean_text(
                scz_adjudicated$manual_evidence_direction[j]
            )

        expanded$STEP10E4_review_note[i] <-
            clean_text(
                scz_adjudicated$manual_review_note[j]
            )
    }
}

# --------------------------------------------------------------------------------------------------
# 10F.1O — TARGET GENE REGEX
# --------------------------------------------------------------------------------------------------

escape_regex <- function(x) {

    gsub(
        "([][{}()+*^$|\\\\?.])",
        "\\\\\\1",
        x
    )
}

target_terms_for_id <- function(id) {

    row <- identity[
        identity$ensemblID_stable == id,
        ,
        drop = FALSE
    ]

    if (nrow(row) != 1L) {
        stop(
            "Identity lookup failed for ",
            id
        )
    }

    if (id == "ENSG00000117877") {
        return(
            c(
                "POLR1G",
                "CD3EAP"
            )
        )
    }

    if (id == "ENSG00000187951") {
        return(
            "ARHGAP11B"
        )
    }

    terms <- unique(
        c(
            clean_text(
                row$interpretation_symbol
            ),
            clean_text(
                row$gene_symbol
            )
        )
    )

    terms <- terms[
        nzchar(terms)
    ]

    if (length(terms) == 0L) {
        terms <- id
    }

    terms
}

make_target_regex <- function(id) {

    terms <- target_terms_for_id(
        id
    )

    escaped <- vapply(
        terms,
        escape_regex,
        FUN.VALUE = character(1)
    )

    paste0(
        "(?i)(?<![A-Za-z0-9])(?:",
        paste(
            escaped,
            collapse = "|"
        ),
        ")(?![A-Za-z0-9])"
    )
}

# --------------------------------------------------------------------------------------------------
# 10F.1P — BRAIN/CELL-TYPE CONTEXT PATTERNS
# --------------------------------------------------------------------------------------------------

BRAIN_REGEX <- paste0(
    "(?i)\\b(",
    "brain",
    "|cortex",
    "|cortical",
    "|prefrontal",
    "|dlpfc",
    "|hippocamp[a-z]*",
    "|striat[a-z]*",
    "|thalam[a-z]*",
    "|midbrain",
    "|substantia nigra",
    "|ventral tegment[a-z]*",
    "|postmortem",
    "|post-mortem",
    ")\\b"
)

NEURON_REGEX <- paste0(
    "(?i)\\b(",
    "neuron",
    "|neurons",
    "|neuronal",
    "|dopamin[a-z]*",
    "|gabaergic",
    "|glutamatergic",
    "|interneuron",
    "|interneurons",
    ")\\b"
)

ASTRO_REGEX <-
    "(?i)\\bastrocyt[a-z]*\\b"

MICROGLIA_REGEX <-
    "(?i)\\bmicrogli[a-z]*\\b"

OLIGO_REGEX <-
    "(?i)\\boligodendrocyt[a-z]*\\b"

HUMAN_REGEX <- paste0(
    "(?i)\\b(",
    "human",
    "|patient",
    "|patients",
    "|participant",
    "|participants",
    "|subject",
    "|subjects",
    "|postmortem",
    "|post-mortem",
    ")\\b"
)

MOUSE_REGEX <- paste0(
    "(?i)\\b(",
    "mouse",
    "|mice",
    "|murine",
    "|rat",
    "|rats",
    "|rodent",
    ")\\b"
)

DEVELOPMENT_REGEX <- paste0(
    "(?i)\\b(",
    "development",
    "|developmental",
    "|differentiation",
    "|neurogenesis",
    "|progenitor",
    "|progenitors",
    "|stem cell",
    "|stem cells",
    ")\\b"
)

# --------------------------------------------------------------------------------------------------
# 10F.1Q — SENTENCE HELPERS
# --------------------------------------------------------------------------------------------------

split_sentences <- function(text) {

    text <- clean_text(
        text
    )

    if (!nzchar(text)) {
        return(
            character(0)
        )
    }

    text <- gsub(
        "<[^>]+>",
        " ",
        text
    )

    text <- gsub(
        "\\s+",
        " ",
        text
    )

    s <- unlist(
        strsplit(
            text,
            "(?<=[.!?])\\s+",
            perl = TRUE
        ),
        use.names = FALSE
    )

    trimws(
        s[
            nzchar(
                trimws(s)
            )
        ]
    )
}

extract_sentences <- function(
    text,
    regex
) {

    s <- split_sentences(
        text
    )

    if (length(s) == 0L) {
        return("")
    }

    hit <- s[
        grepl(
            regex,
            s,
            perl = TRUE
        )
    ]

    if (length(hit) == 0L) {
        return("")
    }

    paste(
        unique(hit),
        collapse = " || "
    )
}

# --------------------------------------------------------------------------------------------------
# 10F.1R — CONTEXT FLAGS + SENTENCE EXTRACTION
# --------------------------------------------------------------------------------------------------

expanded$target_in_title <- FALSE
expanded$target_in_abstract <- FALSE

expanded$brain_context_flag <- FALSE
expanded$neuron_context_flag <- FALSE
expanded$astrocyte_context_flag <- FALSE
expanded$microglia_context_flag <- FALSE
expanded$oligodendrocyte_context_flag <- FALSE
expanded$human_context_flag <- FALSE
expanded$animal_context_flag <- FALSE
expanded$development_context_flag <- FALSE

expanded$target_sentences <- ""
expanded$brain_context_sentences <- ""

for (i in seq_len(nrow(expanded))) {

    gene_regex <- make_target_regex(
        expanded$ensemblID_stable[i]
    )

    full_text <- paste(
        expanded$title[i],
        expanded$abstract_text[i]
    )

    expanded$target_in_title[i] <- grepl(
        gene_regex,
        expanded$title[i],
        perl = TRUE
    )

    expanded$target_in_abstract[i] <- grepl(
        gene_regex,
        expanded$abstract_text[i],
        perl = TRUE
    )

    expanded$brain_context_flag[i] <- grepl(
        BRAIN_REGEX,
        full_text,
        perl = TRUE
    )

    expanded$neuron_context_flag[i] <- grepl(
        NEURON_REGEX,
        full_text,
        perl = TRUE
    )

    expanded$astrocyte_context_flag[i] <- grepl(
        ASTRO_REGEX,
        full_text,
        perl = TRUE
    )

    expanded$microglia_context_flag[i] <- grepl(
        MICROGLIA_REGEX,
        full_text,
        perl = TRUE
    )

    expanded$oligodendrocyte_context_flag[i] <- grepl(
        OLIGO_REGEX,
        full_text,
        perl = TRUE
    )

    expanded$human_context_flag[i] <- grepl(
        HUMAN_REGEX,
        full_text,
        perl = TRUE
    )

    expanded$animal_context_flag[i] <- grepl(
        MOUSE_REGEX,
        full_text,
        perl = TRUE
    )

    expanded$development_context_flag[i] <- grepl(
        DEVELOPMENT_REGEX,
        full_text,
        perl = TRUE
    )

    expanded$target_sentences[i] <-
        extract_sentences(
            expanded$abstract_text[i],
            gene_regex
        )

    expanded$brain_context_sentences[i] <-
        extract_sentences(
            expanded$abstract_text[i],
            paste0(
                "(",
                BRAIN_REGEX,
                "|",
                NEURON_REGEX,
                "|",
                ASTRO_REGEX,
                "|",
                MICROGLIA_REGEX,
                "|",
                OLIGO_REGEX,
                ")"
            )
        )
}

# --------------------------------------------------------------------------------------------------
# 10F.1S — HISTORICAL ARHGAP11B CAUTION
# --------------------------------------------------------------------------------------------------

expanded$identity_caution <- ifelse(
    expanded$ensemblID_stable ==
        "ENSG00000187951",
    paste(
        "HISTORICAL ARHGAP11B FEATURE:",
        "frozen BrainSeq ID ENSG00000187951;",
        "current ARHGAP11B Ensembl ID ENSG00000285077."
    ),
    ""
)

# --------------------------------------------------------------------------------------------------
# 10F.1T — OPERATIONAL REVIEW PRIORITY
#
# This score controls reading order ONLY.
# It is NOT evidence strength.
# --------------------------------------------------------------------------------------------------

expanded$brain_review_priority_score <- (
    3L *
        as.integer(
            expanded$target_in_title
        ) +
    2L *
        as.integer(
            expanded$target_in_abstract
        ) +
    2L *
        as.integer(
            expanded$human_context_flag
        ) +
    2L *
        as.integer(
            expanded$brain_context_flag
        ) +
    1L *
        as.integer(
            expanded$neuron_context_flag
        ) +
    1L *
        as.integer(
            expanded$astrocyte_context_flag
        ) +
    1L *
        as.integer(
            expanded$microglia_context_flag
        ) +
    1L *
        as.integer(
            expanded$oligodendrocyte_context_flag
        ) +
    1L *
        as.integer(
            expanded$development_context_flag
        )
)

# --------------------------------------------------------------------------------------------------
# 10F.1U — NEW REVIEW QUEUE
#
# Already STEP10E.4-reviewed records remain in full expanded corpus but are removed
# from the new manual-review queue.
# --------------------------------------------------------------------------------------------------

new_queue <- expanded[
    !expanded$already_reviewed_in_STEP10E4 &
    expanded$target_in_abstract,
    ,
    drop = FALSE
]

new_queue <- new_queue[
    order(
        new_queue$ensemblID_stable,
        -new_queue$brain_review_priority_score,
        -new_queue$cited_by_count,
        new_queue$query_rank,
        na.last = TRUE
    ),
    ,
    drop = FALSE
]

# --------------------------------------------------------------------------------------------------
# 10F.1V — TOP 3 NEW PAPERS PER FEATURE
# --------------------------------------------------------------------------------------------------

top_rows <- list()
top_index <- 1L

for (id in eligible_features$ensemblID_stable) {

    block <- new_queue[
        new_queue$ensemblID_stable == id,
        ,
        drop = FALSE
    ]

    if (nrow(block) == 0L) {
        next
    }

    block <- head(
        block,
        3L
    )

    top_rows[[top_index]] <- block
    top_index <- top_index + 1L
}

if (length(top_rows) > 0L) {

    top_packet <- do.call(
        rbind,
        top_rows
    )

} else {

    top_packet <- new_queue[
        FALSE,
        ,
        drop = FALSE
    ]
}

rownames(top_packet) <- NULL

# --------------------------------------------------------------------------------------------------
# 10F.1W — WRITE FULL OUTPUTS
# --------------------------------------------------------------------------------------------------

write.csv(
    expanded,
    EXPANDED_RECORDS_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    new_queue,
    NEW_REVIEW_QUEUE_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    top_packet,
    TOP_REVIEW_PACKET_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10F.1X — GENE-LEVEL RETRIEVAL SUMMARY
# --------------------------------------------------------------------------------------------------

gene_rows <- vector(
    "list",
    nrow(eligible_features)
)

for (i in seq_len(nrow(eligible_features))) {

    id <- eligible_features$ensemblID_stable[i]

    block <- expanded[
        expanded$ensemblID_stable == id,
        ,
        drop = FALSE
    ]

    new_block <- new_queue[
        new_queue$ensemblID_stable == id,
        ,
        drop = FALSE
    ]

    gene_rows[[i]] <- data.frame(

        ensemblID_stable = id,

        interpretation_symbol =
            eligible_features$interpretation_symbol[i],

        literature_interpretation_tier =
            eligible_features$literature_interpretation_tier[i],

        original_brain_hit_count =
            eligible_features$brain_neural_celltype_hits[i],

        expanded_unique_records =
            nrow(block),

        already_reviewed_in_STEP10E4 =
            sum(
                block$already_reviewed_in_STEP10E4
            ),

        new_records_for_brain_review =
            nrow(new_block),

        records_with_human_context =
            sum(
                new_block$human_context_flag
            ),

        records_with_neuronal_context =
            sum(
                new_block$neuron_context_flag
            ),

        records_with_astrocyte_context =
            sum(
                new_block$astrocyte_context_flag
            ),

        records_with_microglia_context =
            sum(
                new_block$microglia_context_flag
            ),

        records_with_oligodendrocyte_context =
            sum(
                new_block$oligodendrocyte_context_flag
            ),

        records_with_development_context =
            sum(
                new_block$development_context_flag
            ),

        stringsAsFactors = FALSE
    )
}

gene_context_summary <- do.call(
    rbind,
    gene_rows
)

write.csv(
    gene_context_summary,
    GENE_SUMMARY_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10F.1Y — BUILD HUMAN-READABLE TOP REVIEW PACKET
# --------------------------------------------------------------------------------------------------

packet_lines <- c(
    "==============================================================================================================",
    "STEP 10F.1 — TOP BRAIN / NEURAL / CELL-TYPE MANUAL REVIEW PACKET",
    "==============================================================================================================",
    ""
)

if (nrow(top_packet) == 0L) {

    packet_lines <- c(
        packet_lines,
        "[NO NEW PAPERS]"
    )

} else {

    ids <- unique(
        top_packet$ensemblID_stable
    )

    for (id in ids) {

        block <- top_packet[
            top_packet$ensemblID_stable == id,
            ,
            drop = FALSE
        ]

        gene_name <- unique(
            block$interpretation_symbol
        )

        gene_name <- gene_name[
            nzchar(
                gene_name
            )
        ]

        if (length(gene_name) == 0L) {
            gene_name <- id
        } else {
            gene_name <- gene_name[1L]
        }

        packet_lines <- c(
            packet_lines,
            paste(
                rep("-", 105),
                collapse = ""
            ),
            paste0(
                "FEATURE: ",
                gene_name,
                " | ",
                id
            )
        )

        if (
            id ==
                "ENSG00000187951"
        ) {

            packet_lines <- c(
                packet_lines,
                paste(
                    "IDENTITY CAUTION:",
                    "historical BrainSeq ARHGAP11B identity;",
                    "current ARHGAP11B Ensembl ID ENSG00000285077."
                )
            )
        }

        packet_lines <- c(
            packet_lines,
            ""
        )

        for (j in seq_len(nrow(block))) {

            packet_lines <- c(
                packet_lines,

                paste0(
                    "[",
                    j,
                    "] ",
                    block$title[j]
                ),

                paste0(
                    "PMID: ",
                    ifelse(
                        nzchar(
                            block$pmid[j]
                        ),
                        block$pmid[j],
                        "[NA]"
                    ),
                    " | Year: ",
                    ifelse(
                        nzchar(
                            block$pub_year[j]
                        ),
                        block$pub_year[j],
                        "[NA]"
                    )
                ),

                paste0(
                    "Human=",
                    block$human_context_flag[j],
                    " | Animal=",
                    block$animal_context_flag[j],
                    " | Neuron=",
                    block$neuron_context_flag[j],
                    " | Astrocyte=",
                    block$astrocyte_context_flag[j],
                    " | Microglia=",
                    block$microglia_context_flag[j],
                    " | Oligodendrocyte=",
                    block$oligodendrocyte_context_flag[j],
                    " | Development=",
                    block$development_context_flag[j]
                ),

                paste0(
                    "TARGET SENTENCE(S): ",
                    ifelse(
                        nzchar(
                            block$target_sentences[j]
                        ),
                        block$target_sentences[j],
                        "[NONE]"
                    )
                ),

                paste0(
                    "BRAIN/NEURAL SENTENCE(S): ",
                    ifelse(
                        nzchar(
                            block$brain_context_sentences[j]
                        ),
                        block$brain_context_sentences[j],
                        "[NONE]"
                    )
                ),

                ""
            )
        }
    }
}

writeLines(
    packet_lines,
    TOP_REVIEW_TXT_PATH,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10F.1Z — SUMMARY
# --------------------------------------------------------------------------------------------------

n_expanded <- nrow(
    expanded
)

n_already_reviewed <- sum(
    expanded$already_reviewed_in_STEP10E4
)

n_new <- nrow(
    new_queue
)

n_top <- nrow(
    top_packet
)

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10F.1 — BRAIN / NEURAL / CELL-TYPE LITERATURE REVIEW SUMMARY",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    paste0(
        "Features eligible for brain-context review: ",
        n_eligible
    ),

    paste0(
        "Expanded unique feature-paper records: ",
        n_expanded
    ),

    paste0(
        "Records already manually reviewed in STEP10E.4: ",
        n_already_reviewed
    ),

    paste0(
        "New records entering brain-context manual review: ",
        n_new
    ),

    paste0(
        "Top manual-review packet records: ",
        n_top
    ),

    "",

    "IMPORTANT",

    paste(
        "The top packet is a triage subset of the expanded literature retrieval."
    ),

    paste(
        "No brain-region, neuronal, glial, developmental, or disease-specific biological claim",
        "should be made until these records are manually reviewed."
    ),

    paste(
        "STEP10E.4 schizophrenia-specific adjudications remain frozen and are not overwritten."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10F.1AA — MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    IDENTITY_PATH,
    QUERY_REGISTRY_PATH,
    GENE_LIT_SUMMARY_PATH,
    SCZ_ADJUDICATED_PATH,
    EXPANDED_RECORDS_PATH,
    NEW_REVIEW_QUEUE_PATH,
    TOP_REVIEW_PACKET_PATH,
    TOP_REVIEW_TXT_PATH,
    GENE_SUMMARY_PATH,
    SUMMARY_PATH
)

manifest <- data.frame(
    file = basename(
        manifest_paths
    ),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(
            manifest_paths
        )$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10F.1AB — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10F.1 SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Features eligible for brain/neural review: ",
    n_eligible,
    "\n",
    sep = ""
)

cat(
    "Expanded unique feature-paper records: ",
    n_expanded,
    "\n",
    sep = ""
)

cat(
    "Already reviewed in STEP10E.4: ",
    n_already_reviewed,
    "\n",
    sep = ""
)

cat(
    "New brain-context review records: ",
    n_new,
    "\n",
    sep = ""
)

cat(
    "Top packet records: ",
    n_top,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nGENE-LEVEL BRAIN CONTEXT RETRIEVAL SUMMARY\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    gene_context_summary,
    row.names = FALSE
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nTOP BRAIN / NEURAL MANUAL REVIEW PACKET\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

if (nrow(top_packet) == 0L) {

    cat("[NO NEW PAPERS]\n")

} else {

    ids <- unique(
        top_packet$ensemblID_stable
    )

    for (id in ids) {

        block <- top_packet[
            top_packet$ensemblID_stable == id,
            ,
            drop = FALSE
        ]

        gene_name <- unique(
            block$interpretation_symbol
        )

        gene_name <- gene_name[
            nzchar(
                gene_name
            )
        ]

        if (length(gene_name) == 0L) {
            gene_name <- id
        } else {
            gene_name <- gene_name[1L]
        }

        cat(
            "\n",
            paste(rep("-", 105), collapse = ""),
            "\n",
            gene_name,
            " | ",
            id,
            "\n",
            sep = ""
        )

        if (
            id ==
                "ENSG00000187951"
        ) {

            cat(
                "IDENTITY CAUTION: historical ARHGAP11B feature; current ARHGAP11B ID = ENSG00000285077\n"
            )
        }

        for (j in seq_len(nrow(block))) {

            cat(
                "\n[",
                j,
                "] ",
                block$title[j],
                "\n",
                sep = ""
            )

            cat(
                "PMID: ",
                ifelse(
                    nzchar(
                        block$pmid[j]
                    ),
                    block$pmid[j],
                    "[NA]"
                ),
                " | Year: ",
                ifelse(
                    nzchar(
                        block$pub_year[j]
                    ),
                    block$pub_year[j],
                    "[NA]"
                ),
                "\n",
                sep = ""
            )

            cat(
                "Human=",
                block$human_context_flag[j],
                " | Animal=",
                block$animal_context_flag[j],
                " | Neuron=",
                block$neuron_context_flag[j],
                " | Astrocyte=",
                block$astrocyte_context_flag[j],
                " | Microglia=",
                block$microglia_context_flag[j],
                " | Oligodendrocyte=",
                block$oligodendrocyte_context_flag[j],
                " | Development=",
                block$development_context_flag[j],
                "\n",
                sep = ""
            )

            cat(
                "TARGET SENTENCE(S): ",
                ifelse(
                    nzchar(
                        block$target_sentences[j]
                    ),
                    block$target_sentences[j],
                    "[NONE]"
                ),
                "\n",
                sep = ""
            )

            cat(
                "BRAIN/NEURAL SENTENCE(S): ",
                ifelse(
                    nzchar(
                        block$brain_context_sentences[j]
                    ),
                    block$brain_context_sentences[j],
                    "[NONE]"
                ),
                "\n",
                sep = ""
            )
        }
    }
}

cat(
    "\nExpanded records:\n",
    EXPANDED_RECORDS_PATH,
    "\n",
    sep = ""
)

cat(
    "\nNew manual-review queue:\n",
    NEW_REVIEW_QUEUE_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTop review packet:\n",
    TOP_REVIEW_PACKET_PATH,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    SUMMARY_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10F.1 COMPLETE — BRAIN/NEURAL REVIEW PACKET FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10F.1 — EXPANDED BRAIN / NEURAL / CELL-TYPE LITERATURE REVIEW PACKET
Project sentinel: PASS
STEP10D.2 identity freeze: PASS
STEP10E.1 literature query registry: PASS
STEP10E.1 gene literature summary: PASS
STEP10E.4 manual SCZ adjudication: PASS
Frozen feature count: 24 — PASS
Frozen literature queries: 72 — PASS
SCZ-specific adjudicated records: 18 — PASS

Features eligible for brain/neural paper review: 13
 ensemblID_stable interpretation_symbol
  ENSG00000100092                SH3BP1
  ENSG00000117877                POLR1G
  ENSG00000196565                  HBG2
  ENSG00000260526              AP1AR-DT
  ENSG00000166428                  PLD4
  ENSG00000010319                SEMA3G
  ENSG00000124225                PMEPA1
  ENSG00000187951             ARHGAP11B
  ENSG00000260075                 NSFP1
  ENSG00000121067                  SPOP
  ENSG00000244405                  ETV5
  ENSG00000165092               ALDH1A1
  ENSG00000153234                 NR4A2
              literatu

In [73]:
# ==================================================================================================
# STEP 10F.2 — MANUAL BRAIN / NEURAL / CELL-TYPE EVIDENCE ADJUDICATION
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Manually adjudicate the 31-paper STEP10F.1 top brain/neural review packet.
#
# THIS STEP:
#   - distinguishes genuine target-gene neural evidence from incidental keyword retrieval
#   - separates direct neural biology from indirect/context-only evidence
#   - identifies duplicate/preliminary reports
#   - freezes one manuscript-safe brain-context interpretation for each of the 13 reviewed genes
#
# IMPORTANT
#   - This does NOT test schizophrenia association.
#   - STEP10E.4 schizophrenia-specific literature conclusions remain unchanged.
#   - "Strong/moderate/limited" below refers ONLY to evidence that the gene has a documented
#     brain/neural/cell-type biological context.
#   - It does not imply disease causality.
#   - ARHGAP11B retains the historical-ID caution.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10F.2 — MANUAL BRAIN / NEURAL / CELL-TYPE EVIDENCE ADJUDICATION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

INPUT_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "brain_context_review_v1",
    "STEP10F1_top_brain_neural_review_packet_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "brain_context_review_v1",
    "manual_adjudication_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_RECORDS <- file.path(
    OUT_DIR,
    "STEP10F2_manual_brain_neural_evidence_adjudication_v1.csv"
)

OUT_PRIMARY <- file.path(
    OUT_DIR,
    "STEP10F2_independent_brain_context_evidence_v1.csv"
)

OUT_GENES <- file.path(
    OUT_DIR,
    "STEP10F2_gene_level_brain_context_synthesis_v1.csv"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP10F2_brain_context_adjudication_summary_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP10F2_brain_context_adjudication_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# PROJECT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(project_id, EXPECTED_PROJECT_ID)) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID,
        "\nObserved: ", project_id
    )
}

cat("Project sentinel: PASS\n")

if (!file.exists(INPUT_PATH)) {
    stop(
        "STEP10F.1 top review packet missing:\n",
        INPUT_PATH
    )
}

x <- read.csv(
    INPUT_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(x) != 31L) {
    stop(
        "Expected 31 STEP10F.1 top-review records; observed ",
        nrow(x)
    )
}

cat("STEP10F.1 top-review records: 31 — PASS\n")

# --------------------------------------------------------------------------------------------------
# CLEAN TEXT
# --------------------------------------------------------------------------------------------------

clean_text <- function(z) {

    z <- as.character(z)
    z[is.na(z)] <- ""

    trimws(z)
}

for (nm in intersect(
    c(
        "ensemblID_stable",
        "interpretation_symbol",
        "pmid",
        "title",
        "pub_year"
    ),
    names(x)
)) {

    x[[nm]] <- clean_text(
        x[[nm]]
    )
}

# --------------------------------------------------------------------------------------------------
# INITIALIZE MANUAL FIELDS
# --------------------------------------------------------------------------------------------------

x$manual_brain_relevance_class <- NA_character_
x$manual_brain_context_strength <- NA_character_
x$manual_species_context <- NA_character_
x$manual_region_cell_context <- NA_character_
x$manual_disease_or_model_context <- NA_character_
x$independent_brain_context_evidence <- FALSE
x$duplicate_of <- NA_character_
x$manual_review_note <- NA_character_

# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

find_record <- function(
    ensembl_id,
    pmid = NULL,
    title_pattern = NULL
) {

    idx <- which(
        x$ensemblID_stable == ensembl_id
    )

    if (!is.null(pmid)) {

        idx <- idx[
            x$pmid[idx] == as.character(pmid)
        ]
    }

    if (!is.null(title_pattern)) {

        idx <- idx[
            grepl(
                title_pattern,
                x$title[idx],
                ignore.case = TRUE,
                perl = TRUE
            )
        ]
    }

    if (length(idx) != 1L) {

        stop(
            "Record lookup did not return exactly one row.\n",
            "Ensembl: ", ensembl_id,
            "\nPMID: ",
            ifelse(is.null(pmid), "[NULL]", pmid),
            "\nTitle pattern: ",
            ifelse(
                is.null(title_pattern),
                "[NULL]",
                title_pattern
            ),
            "\nMatches: ",
            length(idx)
        )
    }

    idx
}

set_record <- function(
    idx,
    relevance,
    strength,
    species,
    region_cell,
    model_context,
    independent,
    note,
    duplicate_of = NA_character_
) {

    x$manual_brain_relevance_class[idx] <<- relevance
    x$manual_brain_context_strength[idx] <<- strength
    x$manual_species_context[idx] <<- species
    x$manual_region_cell_context[idx] <<- region_cell
    x$manual_disease_or_model_context[idx] <<- model_context
    x$independent_brain_context_evidence[idx] <<- independent
    x$manual_review_note[idx] <<- note
    x$duplicate_of[idx] <<- duplicate_of
}

# ==================================================================================================
# SH3BP1
# ==================================================================================================

i <- find_record(
    "ENSG00000100092",
    pmid = "23223568"
)

set_record(
    i,
    relevance = "INDIRECT_BRAIN_SPECIFIC_SPLICE_VARIANT_CONTEXT",
    strength = "LIMITED",
    species = "Human brain / molecular study",
    region_cell = "Brain-specific BARGIN splice product; AD brain aggregates",
    model_context = "Alzheimer disease / protein-ubiquitin context",
    independent = TRUE,
    note = paste(
        "The reported brain-specific molecule BARGIN is a splice product involving adjacent",
        "SH3BP1 and CIN loci. This provides some neural context for the locus but should not",
        "be presented as direct evidence that canonical SH3BP1 itself has a defined brain-cell role."
    )
)

# ==================================================================================================
# POLR1G / historical CD3EAP
# ==================================================================================================

i <- find_record(
    "ENSG00000117877",
    pmid = "30128886"
)

set_record(
    i,
    relevance = "IRRELEVANT_BRAIN_KEYWORD_RETRIEVAL",
    strength = "NONE",
    species = "Human",
    region_cell = "None relevant to target gene",
    model_context = "Lung cancer genetics",
    independent = FALSE,
    note = paste(
        "The paper is a lung-cancer genetic association study involving historical symbol CD3EAP.",
        "The brain mention refers to other genes/tumours rather than POLR1G/CD3EAP neural biology.",
        "Exclude from brain-context interpretation."
    )
)

# ==================================================================================================
# HBG2
# ==================================================================================================

i <- find_record(
    "ENSG00000196565",
    pmid = "18183302"
)

set_record(
    i,
    relevance = "DIRECT_NEURONAL_DIFFERENTIATION_EXPRESSION_CONTEXT",
    strength = "MODERATE",
    species = "Human",
    region_cell = "hESC-derived neuronal precursors / dopaminergic differentiation",
    model_context = "Human neuronal differentiation",
    independent = TRUE,
    note = paste(
        "HBG2 was transcriptionally activated during human neuronal/dopaminergic differentiation.",
        "This is direct expression context but does not establish a functional neuronal role."
    )
)

i <- find_record(
    "ENSG00000196565",
    pmid = "33938942"
)

set_record(
    i,
    relevance = "INCIDENTAL_NON_NEURAL_CONTEXT",
    strength = "NONE",
    species = "Human",
    region_cell = "Erythroid system",
    model_context = "BCL11A haploinsufficiency / hemoglobin regulation",
    independent = FALSE,
    note = paste(
        "HBG2 was studied as an erythroid gamma-globin gene.",
        "The neuronal statement concerns BCL11A rather than HBG2.",
        "Exclude as evidence of HBG2 neural biology."
    )
)

# ==================================================================================================
# AP1AR-DT
# ==================================================================================================

i <- find_record(
    "ENSG00000260526",
    pmid = "39558356"
)

set_record(
    i,
    relevance = "DIRECT_NEUROPSYCHIATRIC_NEURONAL_FUNCTION",
    strength = "STRONG",
    species = "Human discovery + mouse functional model",
    region_cell = "Medial prefrontal cortex neurons / excitatory synapses",
    model_context = "Bipolar disorder-associated lncRNA",
    independent = TRUE,
    note = paste(
        "AP1AR-DT was linked to bipolar disorder, validated against brain RNA-seq, and experimentally",
        "manipulated in mouse mPFC where it altered dendritic spines, excitatory synaptic transmission,",
        "NEGR1 expression and behaviour. Strong neural-function context, but cross-disorder rather than SCZ-specific."
    )
)

# ==================================================================================================
# PLD4
# ==================================================================================================

i <- find_record(
    "ENSG00000166428",
    title_pattern = "^PLD4, CD247, and TREM2"
)

set_record(
    i,
    relevance = "DIRECT_NEUROIMMUNE_MICROGLIAL_CONTEXT",
    strength = "MODERATE",
    species = "Animal / transcriptomic",
    region_cell = "Microglia / traumatic brain injury",
    model_context = "Traumatic brain injury",
    independent = TRUE,
    note = paste(
        "PLD4 was identified in a phagocytosis-related TBI signature with single-cell context.",
        "Relevant neuroimmune context, but disease-specific to brain injury and not schizophrenia."
    )
)

i <- find_record(
    "ENSG00000166428",
    pmid = "39423811"
)

set_record(
    i,
    relevance = "GENERAL_LYSOSOMAL_BIOLOGY_WITH_LIMITED_BRAIN_SPECIFICITY",
    strength = "LIMITED",
    species = "Cellular and mouse",
    region_cell = "Lysosomal lipid metabolism",
    model_context = "PLD3/PLD4 lysosomal BMP synthesis",
    independent = TRUE,
    note = paste(
        "The study establishes PLD3/PLD4 biochemical function in lysosomal phospholipid metabolism.",
        "Brain enrichment in the described tissue comparison is stronger for PLD3 whereas PLD4 is prominent in spleen.",
        "Useful mechanistic context, but limited PLD4-specific neural evidence."
    )
)

i <- find_record(
    "ENSG00000166428",
    pmid = "21085684"
)

set_record(
    i,
    relevance = "DIRECT_DEVELOPING_BRAIN_MICROGLIAL_EXPRESSION",
    strength = "STRONG",
    species = "Mouse",
    region_cell = "Iba1-positive microglia / developing white matter",
    model_context = "Early postnatal brain development",
    independent = TRUE,
    note = paste(
        "Direct in situ and immunohistochemical evidence identified PLD4-expressing cells",
        "as microglia during early postnatal brain development.",
        "Strong evidence for a developmental neuroimmune/microglial context."
    )
)

# ==================================================================================================
# SEMA3G
# ==================================================================================================

i <- find_record(
    "ENSG00000010319",
    pmid = "38795838"
)

set_record(
    i,
    relevance = "NON_NEURAL_ENDOTHELIAL_FUNCTION_WITH_NEURAL_BACKGROUND_ONLY",
    strength = "LIMITED",
    species = "Human endothelial cells",
    region_cell = "Microvascular endothelial cells",
    model_context = "BMP9 / VEGF signalling",
    independent = TRUE,
    note = paste(
        "The experiment establishes endothelial/angiogenic SEMA3G function.",
        "Neuronal-development relevance is background information, not directly tested here."
    )
)

i <- find_record(
    "ENSG00000010319",
    title_pattern = "highly selective marker of cerebral amyloid angiopathy"
)

set_record(
    i,
    relevance = "DIRECT_HUMAN_BRAIN_NEUROVASCULAR_CONTEXT",
    strength = "STRONG",
    species = "Human",
    region_cell = "Cerebral blood vessels / neurovascular compartment",
    model_context = "Cerebral amyloid angiopathy",
    independent = TRUE,
    note = paste(
        "Direct human brain-tissue evidence demonstrates selective vascular accumulation of SEMA3G",
        "in cerebral amyloid angiopathy. Strong neurovascular rather than neuronal context."
    )
)

i <- find_record(
    "ENSG00000010319",
    pmid = "41504904"
)

set_record(
    i,
    relevance = "DIRECT_HIPPOCAMPAL_NEUROVASCULAR_ANIMAL_CONTEXT",
    strength = "MODERATE",
    species = "Rat",
    region_cell = "Hippocampal neurovascular unit",
    model_context = "Vascular cognitive impairment / exercise",
    independent = TRUE,
    note = paste(
        "Hippocampal Sema3G/Nrp2/PlexinA4 signalling changed alongside neurovascular,",
        "neuronal and microglial adaptations. Useful neural context but from a vascular-injury animal model."
    )
)

# ==================================================================================================
# PMEPA1
# ==================================================================================================

i <- find_record(
    "ENSG00000124225",
    pmid = "26215835"
)

set_record(
    i,
    relevance = "DIRECT_HIPPOCAMPAL_AND_HUMAN_NEURONAL_SIGNALING_CONTEXT",
    strength = "STRONG",
    species = "Mouse + human stem-cell-derived neurons",
    region_cell = "Hippocampus / human forebrain neurons",
    model_context = "Activin A / TGF-beta signalling",
    independent = TRUE,
    note = paste(
        "Pmepa1 was experimentally identified as an activin target in hippocampus and",
        "PMEPA1 induction was reproduced in human forebrain neurons.",
        "Strong direct neuronal signalling/plasticity context."
    )
)

i <- find_record(
    "ENSG00000124225",
    pmid = "33311522"
)

set_record(
    i,
    relevance = "FALSE_POSITIVE_CORTICAL_BONE_KEYWORD",
    strength = "NONE",
    species = "Mouse",
    region_cell = "Bone / osteoclast lineage",
    model_context = "Bone healing",
    independent = FALSE,
    note = paste(
        "The word 'cortical' refers to cortical bone, not cerebral cortex.",
        "This is not brain/neural evidence and must be excluded."
    )
)

i <- find_record(
    "ENSG00000124225",
    pmid = "23284715"
)

set_record(
    i,
    relevance = "INDIRECT_BRAIN_TRANSCRIPTOME_SCREEN_CONTEXT",
    strength = "LIMITED",
    species = "Human-derived molecular screen",
    region_cell = "Brain transcriptome screen",
    model_context = "HIV-1 Nef interaction partners",
    independent = TRUE,
    note = paste(
        "PMEPA1 emerged from a brain transcriptome-wide protein-interaction screen.",
        "This gives limited brain-expression/context evidence but does not establish a specific neuronal function."
    )
)

# ==================================================================================================
# ARHGAP11B — HISTORICAL IDENTITY SPECIAL CASE
# ==================================================================================================

i <- find_record(
    "ENSG00000187951",
    pmid = "36098218"
)

set_record(
    i,
    relevance = "DIRECT_HUMAN_PRIMATE_CORTICAL_NEURODEVELOPMENT",
    strength = "STRONG",
    species = "Human and chimpanzee cerebral organoids",
    region_cell = "Basal radial glia / cortical progenitors / neocortex",
    model_context = "Human neocortical evolution and development",
    independent = TRUE,
    note = paste(
        "Strong mechanistic evidence that ARHGAP11B regulates basal progenitor abundance",
        "and contributes to human neocortical expansion.",
        "Interpret using historical ARHGAP11B identity; current Ensembl ID is ENSG00000285077."
    )
)

i <- find_record(
    "ENSG00000187951",
    title_pattern = "necessary and sufficient for human-type basal progenitor"
)

set_record(
    i,
    relevance = "DUPLICATE_OR_PRELIMINARY_REPORT",
    strength = "EXCLUDE",
    species = "Human and chimpanzee cerebral organoids",
    region_cell = "Cortical progenitors",
    model_context = "Earlier/preliminary report",
    independent = FALSE,
    note = paste(
        "Earlier/preliminary report of the same organoid work later represented by PMID 36098218.",
        "Do not count independently."
    ),
    duplicate_of = "PMID:36098218"
)

i <- find_record(
    "ENSG00000187951",
    pmid = "33938018"
)

set_record(
    i,
    relevance = "DIRECT_NEOCORTICAL_DEVELOPMENT_AND_NEUROBEHAVIOR",
    strength = "STRONG",
    species = "Transgenic mouse with human-specific ARHGAP11B",
    region_cell = "Neocortex / upper-layer neurons / basal progenitors",
    model_context = "Neocortical expansion and cognition",
    independent = TRUE,
    note = paste(
        "Human-specific ARHGAP11B expression increased neocortical size and upper-layer neuron numbers",
        "and altered memory flexibility. Strong developmental/neocortical functional context."
    )
)

# ==================================================================================================
# NSFP1
# ==================================================================================================

i <- find_record(
    "ENSG00000260075",
    pmid = "40055946"
)

set_record(
    i,
    relevance = "INDIRECT_NEURONAL_GENOMIC_LOCUS_CONTEXT",
    strength = "LIMITED",
    species = "Human genomic/transcriptomic datasets",
    region_cell = "DLPFC single-nucleus RNA-seq / 17q21.31 locus",
    model_context = "Progressive supranuclear palsy",
    independent = TRUE,
    note = paste(
        "NSFP1 expression was altered within a 17q21.31 duplication context.",
        "The neuronal single-nucleus result highlighted LRRC37A/LRRC37A2 more directly.",
        "Therefore NSFP1 neural evidence is indirect and locus-dependent."
    )
)

i <- find_record(
    "ENSG00000260075",
    title_pattern = "Neuro-Genetic Mechanisms Connecting Brain Structure to Breast Cancer"
)

set_record(
    i,
    relevance = "INDIRECT_OR_IRRELEVANT_GENETIC_COLOCALIZATION_CONTEXT",
    strength = "NONE",
    species = "Human GWAS summary statistics",
    region_cell = "Brain imaging-derived traits",
    model_context = "Brain-breast cancer genetic correlation",
    independent = FALSE,
    note = paste(
        "NSFP1 was implicated through shared genetic signals in a brain-imaging/breast-cancer analysis.",
        "This is not sufficient evidence of a direct neural function."
    )
)

# ==================================================================================================
# SPOP
# ==================================================================================================

i <- find_record(
    "ENSG00000121067",
    pmid = "37059253"
)

set_record(
    i,
    relevance = "DIRECT_NEUROINFLAMMATORY_ANIMAL_CONTEXT",
    strength = "MODERATE",
    species = "Mouse",
    region_cell = "Hippocampal neurons / microglial activation",
    model_context = "Chronic stress / depressive-like behaviour",
    independent = TRUE,
    note = paste(
        "SPOP manipulation was linked to hippocampal neuronal damage, microglial activation",
        "and MyD88/NF-kB neuroinflammatory signalling.",
        "Direct brain context but depression/stress-specific rather than schizophrenia-specific."
    )
)

i <- find_record(
    "ENSG00000121067",
    title_pattern = "nuclear ubiquitin ligase adaptor SPOP is a conserved regulator of C9orf72 dipeptide toxicity$"
)

set_record(
    i,
    relevance = "DUPLICATE_OR_PRELIMINARY_REPORT",
    strength = "EXCLUDE",
    species = "C. elegans and mammalian neurons",
    region_cell = "Motor neurons",
    model_context = "C9orf72 dipeptide toxicity",
    independent = FALSE,
    note = paste(
        "Earlier/preliminary version of the published PMID 34593637 study.",
        "Do not count independently."
    ),
    duplicate_of = "PMID:34593637"
)

i <- find_record(
    "ENSG00000121067",
    pmid = "34593637"
)

set_record(
    i,
    relevance = "DIRECT_NEURONAL_PROTEOSTASIS_CONTEXT",
    strength = "MODERATE",
    species = "C. elegans + mammalian primary neurons",
    region_cell = "Primary spinal motor neurons",
    model_context = "C9orf72 dipeptide-repeat neurotoxicity",
    independent = TRUE,
    note = paste(
        "Genetic/pharmacological SPOP inhibition modified dipeptide-repeat toxicity in mammalian neurons.",
        "Direct neuronal proteostasis evidence, although in a neurodegenerative-disease context."
    )
)

# ==================================================================================================
# ETV5
# ==================================================================================================

i <- find_record(
    "ENSG00000244405",
    pmid = "42458829"
)

set_record(
    i,
    relevance = "DIRECT_HIPPOCAMPAL_NEUROGENESIS_FUNCTION",
    strength = "STRONG",
    species = "Mouse",
    region_cell = "Dentate-gyrus adult-born granule neurons",
    model_context = "Adult hippocampal neurogenesis",
    independent = TRUE,
    note = paste(
        "Conditional Etv5 ablation impaired granule-cell maturation, dendritic complexity",
        "and spine development. Strong direct hippocampal neuronal-development evidence."
    )
)

i <- find_record(
    "ENSG00000244405",
    pmid = "40562126"
)

set_record(
    i,
    relevance = "DIRECT_MICROGLIAL_NEUROPROTECTIVE_CONTEXT",
    strength = "MODERATE",
    species = "Animal and cellular models",
    region_cell = "Microglia / neurons",
    model_context = "Ischemic stroke",
    independent = TRUE,
    note = paste(
        "ETV5 altered microglial polarization and neuronal pyroptosis through HACE1/Nrf2.",
        "Direct neuroimmune evidence but in ischemic injury."
    )
)

i <- find_record(
    "ENSG00000244405",
    pmid = "41861825"
)

set_record(
    i,
    relevance = "DIRECT_HUMAN_CORTICAL_NEURODEVELOPMENT",
    strength = "STRONG",
    species = "Human/chimpanzee organoids + mouse",
    region_cell = "Fetal cortex / cortical progenitors / neurons",
    model_context = "Human-specific enhancer and neurodevelopment",
    independent = TRUE,
    note = paste(
        "Human enhancer HAR1984 interacts with ETV5 in fetal brain and experimentally alters",
        "neurogenesis/cortical development. Strong developmental brain context."
    )
)

# ==================================================================================================
# ALDH1A1
# ==================================================================================================

i <- find_record(
    "ENSG00000165092",
    title_pattern = "Spatiotemporal transcriptomics reveals distinct responses"
)

set_record(
    i,
    relevance = "DIRECT_MIDBRAIN_DOPAMINERGIC_NEURON_CONTEXT",
    strength = "STRONG",
    species = "Mouse",
    region_cell = "ALDH1A1-positive midbrain dopaminergic neurons",
    model_context = "Alpha-synuclein / Parkinson model",
    independent = TRUE,
    note = paste(
        "Spatial transcriptomics directly defines ALDH1A1-positive dopaminergic neurons",
        "as a distinct molecular subpopulation with disease-stage-specific responses.",
        "Strong dopamine-neuron biology; not schizophrenia-specific."
    )
)

i <- find_record(
    "ENSG00000165092",
    pmid = "42239820"
)

set_record(
    i,
    relevance = "DIRECT_HUMAN_DOPAMINERGIC_NEURON_CONTEXT",
    strength = "STRONG",
    species = "Human",
    region_cell = "Substantia nigra / dopaminergic-neuron marker context",
    model_context = "Parkinson disease meta-analysis",
    independent = TRUE,
    note = paste(
        "Across human substantia-nigra datasets, ALDH1A1 strongly co-expressed with dopaminergic genes",
        "and marked vulnerable dopamine-neuron subpopulations.",
        "Strong human dopaminergic context."
    )
)

i <- find_record(
    "ENSG00000165092",
    pmid = "41351369"
)

set_record(
    i,
    relevance = "BRAIN_TUMOUR_CONTEXT_NOT_NEURAL_FUNCTION",
    strength = "NONE",
    species = "Human",
    region_cell = "Glioblastoma tumour cells",
    model_context = "Glioblastoma",
    independent = FALSE,
    note = paste(
        "ALDH1A1 was studied as a cancer stem-cell marker in glioblastoma.",
        "This is brain-location evidence but not useful neural/cell-type functional context for this study."
    )
)

# ==================================================================================================
# NR4A2
# ==================================================================================================

i <- find_record(
    "ENSG00000153234",
    title_pattern = "Nr4a2 blocks"
)

set_record(
    i,
    relevance = "DIRECT_HIPPOCAMPAL_SYNAPTIC_PLASTICITY_CONTEXT",
    strength = "STRONG",
    species = "Human postmortem + mouse",
    region_cell = "Hippocampal neurons / synaptic plasticity",
    model_context = "Alzheimer disease",
    independent = TRUE,
    note = paste(
        "NR4A2/Nr4a2 was directly linked to hippocampal synaptic plasticity,",
        "BDNF/AMPA-receptor regulation and memory; human postmortem hippocampal evidence was also reported.",
        "Strong neural-function context, though AD-specific."
    )
)

i <- find_record(
    "ENSG00000153234",
    pmid = "40164579"
)

set_record(
    i,
    relevance = "DIRECT_CORTICAL_NEURON_DEVELOPMENT",
    strength = "STRONG",
    species = "Mouse",
    region_cell = "Cortical layer 6b neurons",
    model_context = "Cortical development",
    independent = TRUE,
    note = paste(
        "Conditional genetics showed Nr4a2 is required for normal differentiation of cortical layer 6b neurons.",
        "Strong direct cortical-development evidence."
    )
)

i <- find_record(
    "ENSG00000153234",
    pmid = "41390117"
)

set_record(
    i,
    relevance = "DIRECT_HIPPOCAMPAL_SUBICULAR_NEURON_FUNCTION",
    strength = "STRONG",
    species = "Mouse",
    region_cell = "Subiculum / hippocampal output neurons",
    model_context = "Development, connectivity and social novelty",
    independent = TRUE,
    note = paste(
        "Nr4a2 was required for subicular neuronal identity, connectivity and social-novelty behaviour.",
        "Strong hippocampal-development and functional context."
    )
)

# --------------------------------------------------------------------------------------------------
# VERIFY ALL 31 RECORDS ADJUDICATED
# --------------------------------------------------------------------------------------------------

required_manual_fields <- c(
    "manual_brain_relevance_class",
    "manual_brain_context_strength",
    "manual_species_context",
    "manual_region_cell_context",
    "manual_disease_or_model_context",
    "manual_review_note"
)

for (nm in required_manual_fields) {

    bad <- which(
        is.na(x[[nm]]) |
        x[[nm]] == ""
    )

    if (length(bad) > 0L) {

        stop(
            "Unadjudicated record(s) remain in ",
            nm,
            ": ",
            paste(
                bad,
                collapse = ", "
            )
        )
    }
}

cat("All 31 brain-context records manually adjudicated: PASS\n")

# --------------------------------------------------------------------------------------------------
# INDEPENDENT BRAIN-CONTEXT EVIDENCE
# --------------------------------------------------------------------------------------------------

primary <- x[
    x$independent_brain_context_evidence &
    x$manual_brain_context_strength != "EXCLUDE",
    ,
    drop = FALSE
]

write.csv(
    x,
    OUT_RECORDS,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    primary,
    OUT_PRIMARY,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# GENE-LEVEL SYNTHESIS
# --------------------------------------------------------------------------------------------------

gene_synthesis <- data.frame(

    ensemblID_stable = c(
        "ENSG00000100092",
        "ENSG00000117877",
        "ENSG00000196565",
        "ENSG00000260526",
        "ENSG00000166428",
        "ENSG00000010319",
        "ENSG00000124225",
        "ENSG00000187951",
        "ENSG00000260075",
        "ENSG00000121067",
        "ENSG00000244405",
        "ENSG00000165092",
        "ENSG00000153234"
    ),

    interpretation_symbol = c(
        "SH3BP1",
        "POLR1G",
        "HBG2",
        "AP1AR-DT",
        "PLD4",
        "SEMA3G",
        "PMEPA1",
        "ARHGAP11B",
        "NSFP1",
        "SPOP",
        "ETV5",
        "ALDH1A1",
        "NR4A2"
    ),

    overall_brain_context_strength = c(
        "LIMITED",
        "NO_CONVINCING_CONTEXT",
        "LIMITED_TO_MODERATE",
        "STRONG",
        "STRONG",
        "MODERATE_TO_STRONG",
        "STRONG",
        "STRONG",
        "LIMITED",
        "MODERATE",
        "STRONG",
        "STRONG",
        "STRONG"
    ),

    dominant_brain_context = c(
        "Brain-specific SH3BP1/CIN splice-product context",
        "No convincing target-specific neural evidence in reviewed paper",
        "Expression during human neuronal/dopaminergic differentiation",
        "mPFC excitatory synaptic regulation / neuropsychiatric lncRNA",
        "Developing-brain microglia / neuroimmune-lysosomal biology",
        "Neurovascular/endothelial brain biology",
        "Activin/TGF-beta signalling in hippocampus and human forebrain neurons",
        "Basal radial glia / cortical progenitors / neocortical development",
        "17q21.31 locus / limited neuronal-genomic context",
        "Neuronal proteostasis and neuroinflammatory signalling",
        "Hippocampal neurogenesis / cortical development / microglia",
        "Midbrain/substantia-nigra dopaminergic neuron biology",
        "Dopaminergic, cortical and hippocampal neuronal development/function"
    ),

    manuscript_safe_brain_interpretation = c(

        paste(
            "SH3BP1 has limited brain-related evidence through a brain-specific splice product",
            "derived from the SH3BP1/CIN locus; this should not be overstated as a defined",
            "canonical SH3BP1 neuronal function."
        ),

        paste(
            "The reviewed POLR1G/CD3EAP record did not provide convincing target-specific",
            "brain or neural evidence; the apparent hit arose from unrelated brain-tumour terminology."
        ),

        paste(
            "HBG2 has limited evidence of expression during human neuronal and dopaminergic",
            "differentiation, but no strong evidence for a dedicated neuronal function."
        ),

        paste(
            "AP1AR-DT has direct experimental evidence for regulation of medial prefrontal",
            "excitatory synaptic function and neuropsychiatric behaviour in a bipolar-disorder context."
        ),

        paste(
            "PLD4 has direct developmental neuroimmune evidence, particularly expression in",
            "early postnatal microglia, together with broader lysosomal/phagocytic biology."
        ),

        paste(
            "SEMA3G has substantial neurovascular rather than primarily neuronal context,",
            "including human cerebral vascular pathology and hippocampal neurovascular signalling."
        ),

        paste(
            "PMEPA1 is experimentally supported as an activin/TGF-beta-responsive target in",
            "hippocampus and human forebrain neurons, supporting a neuronal signalling/plasticity context."
        ),

        paste(
            "ARHGAP11B has strong mechanistic evidence in basal radial glia, cortical progenitors",
            "and neocortical expansion. This interpretation requires the historical-ID caveat."
        ),

        paste(
            "NSFP1 currently has only limited and largely locus-dependent neuronal evidence;",
            "a specific neural function is not established by the reviewed literature."
        ),

        paste(
            "SPOP has direct neuronal and neuroimmune functional context through protein-quality",
            "control and inflammatory signalling, although the evidence comes from non-schizophrenia models."
        ),

        paste(
            "ETV5 has strong neural-developmental evidence spanning adult hippocampal neurogenesis,",
            "cortical development and experimental microglial regulation."
        ),

        paste(
            "ALDH1A1 has strong and reproducible biological context in defined midbrain dopaminergic",
            "neuron subpopulations, including human substantia-nigra data; this neural plausibility",
            "does not override the largely null direct schizophrenia evidence."
        ),

        paste(
            "NR4A2 has strong neuronal biological support across dopaminergic systems, cortical",
            "development, hippocampal/subicular identity and synaptic plasticity; however, this",
            "biological plausibility remains distinct from its mixed-to-null schizophrenia-risk genetics."
        )
    ),

    stringsAsFactors = FALSE
)

write.csv(
    gene_synthesis,
    OUT_GENES,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# SUMMARY COUNTS
# --------------------------------------------------------------------------------------------------

strength_counts <- table(
    x$manual_brain_context_strength
)

relevance_counts <- table(
    x$manual_brain_relevance_class
)

n_independent <- nrow(primary)

n_excluded <- sum(
    x$manual_brain_context_strength %in%
        c(
            "NONE",
            "EXCLUDE"
        )
)

n_strong <- sum(
    x$manual_brain_context_strength ==
        "STRONG"
)

n_moderate <- sum(
    x$manual_brain_context_strength ==
        "MODERATE"
)

n_limited <- sum(
    x$manual_brain_context_strength ==
        "LIMITED"
)

# --------------------------------------------------------------------------------------------------
# SUMMARY FILE
# --------------------------------------------------------------------------------------------------

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10F.2 — MANUAL BRAIN / NEURAL / CELL-TYPE EVIDENCE ADJUDICATION",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    paste0(
        "Top records adjudicated: ",
        nrow(x)
    ),

    paste0(
        "Independent brain-context evidence records retained: ",
        n_independent
    ),

    paste0(
        "Strong brain-context records: ",
        n_strong
    ),

    paste0(
        "Moderate brain-context records: ",
        n_moderate
    ),

    paste0(
        "Limited brain-context records: ",
        n_limited
    ),

    paste0(
        "Records excluded as irrelevant/duplicate/non-neural: ",
        n_excluded
    ),

    "",

    "IMPORTANT INTERPRETIVE DISTINCTION",

    paste(
        "Brain/neural biological plausibility is not equivalent to schizophrenia association."
    ),

    paste(
        "In particular, ALDH1A1 and NR4A2 have strong neural biology but their direct",
        "schizophrenia literature from STEP10E.4 is predominantly null/mixed."
    ),

    paste(
        "ARHGAP11B has strong cortical developmental biology but weak/uncertain",
        "schizophrenia-specific CNV evidence and requires historical-ID handling."
    ),

    paste(
        "POLR1G has no convincing target-specific neural evidence in the current",
        "systematic top-paper review."
    ),

    "",

    "NEXT STEP",

    paste(
        "Integrate composition, identity, pathway-null results, SCZ-specific literature,",
        "brain-context literature, score direction/weight, and external-validation limitations",
        "into the final 24-feature biological synthesis."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    INPUT_PATH,
    OUT_RECORDS,
    OUT_PRIMARY,
    OUT_GENES,
    OUT_SUMMARY
)

manifest <- data.frame(
    file = basename(
        manifest_paths
    ),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(
            manifest_paths
        )$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nBRAIN / NEURAL EVIDENCE ADJUDICATION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Records adjudicated: ",
    nrow(x),
    "\n",
    sep = ""
)

cat(
    "Independent evidence records: ",
    n_independent,
    "\n",
    sep = ""
)

cat(
    "\nBRAIN-CONTEXT STRENGTH COUNTS\n"
)

print(
    strength_counts
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\n13-GENE BRAIN-CONTEXT SYNTHESIS\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    gene_synthesis,
    row.names = FALSE
)

cat(
    "\nAdjudicated 31-record table:\n",
    OUT_RECORDS,
    "\n",
    sep = ""
)

cat(
    "\nIndependent brain-context evidence:\n",
    OUT_PRIMARY,
    "\n",
    sep = ""
)

cat(
    "\nGene-level brain-context synthesis:\n",
    OUT_GENES,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10F.2 COMPLETE — BRAIN/NEURAL EVIDENCE ADJUDICATION FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10F.2 — MANUAL BRAIN / NEURAL / CELL-TYPE EVIDENCE ADJUDICATION
Project sentinel: PASS
STEP10F.1 top-review records: 31 — PASS
All 31 brain-context records manually adjudicated: PASS

BRAIN / NEURAL EVIDENCE ADJUDICATION SUMMARY
Records adjudicated: 31
Independent evidence records: 24

BRAIN-CONTEXT STRENGTH COUNTS

 EXCLUDE  LIMITED MODERATE     NONE   STRONG 
       2        5        6        5       13 

--------------------------------------------------------------------------------------------------------------
13-GENE BRAIN-CONTEXT SYNTHESIS
--------------------------------------------------------------------------------------------------------------
 ensemblID_stable interpretation_symbol overall_brain_context_strength
  ENSG00000100092                SH3BP1                        LIMITED
  ENSG00000117877                POLR1G          NO_CONVINCING_CONTEXT
  ENSG00000196565                  HBG2            LIMITED_TO_MODERATE
  ENSG00000260526              AP1AR-DT      

In [74]:
# ==================================================================================================
# STEP 10G.1 — FINAL INTEGRATED 24-FEATURE BIOLOGICAL SYNTHESIS
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Integrate ALL completed biological interpretation layers for the frozen 24-feature SCZ-MSS state:
#
#       1. Frozen feature identity / weight / direction
#       2. Coding vs non-coding composition
#       3. Interpretation-safe current identity
#       4. Corrected GO-BP / Reactome annotation coverage
#       5. Conventional pathway result: NO FDR-supported enrichment
#       6. Manually adjudicated schizophrenia-specific literature
#       7. Manually adjudicated brain/neural/cell-type literature
#
# IMPORTANT
#   - NO model is retrained.
#   - NO score weight is modified.
#   - NO feature is added or removed.
#   - NO literature evidence is used to rescue/re-rank genes.
#   - NO nominal pathway is promoted to significant.
#   - Lack of a literature hit is NOT interpreted as proof of biological irrelevance.
#
# OUTPUT
#   - Final integrated 24-feature biological evidence matrix
#   - Weight-ranked version
#   - Compact manuscript table
#   - Project-level biological synthesis
#   - SHA256 manifest
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10G.1 — FINAL INTEGRATED 24-FEATURE BIOLOGICAL SYNTHESIS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10G.1A — PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

IDENTITY_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "identity_audit_v1",
    "adjudicated_v1",
    "STEP10D2_SCZ_MSS_24gene_interpretation_identity_v1.csv"
)

PATHWAY_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "gene_level_interpretation_v1",
    "coverage_correction_v1",
    "STEP10C1R_24gene_corrected_interpretation_scaffold_v1.csv"
)

SCZ_SYNTHESIS_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "evidence_review_v1",
    "manual_adjudication_v1",
    "STEP10E4_gene_level_SCZ_evidence_synthesis_v1.csv"
)

BRAIN_SYNTHESIS_PATH <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "literature_evidence_v1",
    "brain_context_review_v1",
    "manual_adjudication_v1",
    "STEP10F2_gene_level_brain_context_synthesis_v1.csv"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "final_integrated_synthesis_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_MATRIX <- file.path(
    OUT_DIR,
    "STEP10G1_SCZ_MSS_24feature_integrated_biological_evidence_matrix_v1.csv"
)

OUT_RANKED <- file.path(
    OUT_DIR,
    "STEP10G1_SCZ_MSS_24feature_weight_ranked_biological_matrix_v1.csv"
)

OUT_MANUSCRIPT_TABLE <- file.path(
    OUT_DIR,
    "STEP10G1_SCZ_MSS_compact_manuscript_biological_table_v1.csv"
)

OUT_SYNTHESIS <- file.path(
    OUT_DIR,
    "STEP10G1_project_level_biological_synthesis_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP10G1_final_biological_synthesis_manifest_v1.csv"
)

# --------------------------------------------------------------------------------------------------
# 10G.1B — PROJECT / INPUT VERIFICATION
# --------------------------------------------------------------------------------------------------

if (!file.exists(SENTINEL_PATH)) {

    stop(
        "Project sentinel missing: ",
        SENTINEL_PATH
    )
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

required_files <- c(
    IDENTITY_PATH,
    PATHWAY_PATH,
    SCZ_SYNTHESIS_PATH,
    BRAIN_SYNTHESIS_PATH
)

missing_files <- required_files[
    !file.exists(
        required_files
    )
]

if (length(missing_files) > 0L) {

    stop(
        "Required upstream file(s) missing:\n",
        paste(
            missing_files,
            collapse = "\n"
        )
    )
}

cat("STEP10D.2 identity adjudication: PASS\n")
cat("STEP10C.1R pathway/coverage correction: PASS\n")
cat("STEP10E.4 SCZ evidence synthesis: PASS\n")
cat("STEP10F.2 brain-context synthesis: PASS\n")

# --------------------------------------------------------------------------------------------------
# 10G.1C — LOAD DATA
# --------------------------------------------------------------------------------------------------

identity <- read.csv(
    IDENTITY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

pathway <- read.csv(
    PATHWAY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

scz <- read.csv(
    SCZ_SYNTHESIS_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

brain <- read.csv(
    BRAIN_SYNTHESIS_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(identity) != 24L) {

    stop(
        "Expected 24 frozen features; observed ",
        nrow(identity)
    )
}

if (nrow(pathway) != 24L) {

    stop(
        "Expected 24 rows in corrected biological scaffold; observed ",
        nrow(pathway)
    )
}

if (nrow(scz) != 3L) {

    stop(
        "Expected 3 gene-level SCZ evidence synthesis rows; observed ",
        nrow(scz)
    )
}

if (nrow(brain) != 13L) {

    stop(
        "Expected 13 gene-level brain-context synthesis rows; observed ",
        nrow(brain)
    )
}

if (anyDuplicated(identity$ensemblID_stable) > 0L) {
    stop("Duplicate frozen feature IDs in identity table.")
}

if (anyDuplicated(pathway$ensemblID_stable) > 0L) {
    stop("Duplicate feature IDs in pathway scaffold.")
}

if (anyDuplicated(scz$ensemblID_stable) > 0L) {
    stop("Duplicate IDs in SCZ synthesis.")
}

if (anyDuplicated(brain$ensemblID_stable) > 0L) {
    stop("Duplicate IDs in brain synthesis.")
}

cat("Frozen 24-feature set: PASS\n")
cat("SCZ gene-level synthesis rows: 3 — PASS\n")
cat("Brain-context gene-level synthesis rows: 13 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10G.1D — CLEAN TEXT HELPER
# --------------------------------------------------------------------------------------------------

clean_text <- function(x) {

    x <- as.character(x)

    x[
        is.na(x)
    ] <- ""

    trimws(x)
}

# --------------------------------------------------------------------------------------------------
# 10G.1E — REQUIRED IDENTITY COLUMNS
# --------------------------------------------------------------------------------------------------

required_identity_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "protein_coding_flag",
    "direction_standardized",
    "final_normalized_weight",
    "interpretation_symbol",
    "interpretation_identity_class",
    "literature_interpretation_tier",
    "allow_gene_specific_biological_claim",
    "manuscript_gene_label"
)

missing_identity_cols <- setdiff(
    required_identity_cols,
    names(identity)
)

if (length(missing_identity_cols) > 0L) {

    stop(
        "STEP10D.2 identity table missing column(s): ",
        paste(
            missing_identity_cols,
            collapse = ", "
        )
    )
}

# --------------------------------------------------------------------------------------------------
# 10G.1F — NORMALIZE LOGICAL COLUMNS
# --------------------------------------------------------------------------------------------------

normalize_logical <- function(x, column_name) {

    if (is.logical(x)) {
        return(x)
    }

    z <- toupper(
        trimws(
            as.character(x)
        )
    )

    valid <- (
        z %in%
            c(
                "TRUE",
                "FALSE",
                ""
            )
    )

    if (!all(valid)) {

        stop(
            "Cannot normalize logical column: ",
            column_name
        )
    }

    out <- rep(
        NA,
        length(z)
    )

    out[z == "TRUE"] <- TRUE
    out[z == "FALSE"] <- FALSE

    out
}

identity$protein_coding_flag <- normalize_logical(
    identity$protein_coding_flag,
    "protein_coding_flag"
)

identity$allow_gene_specific_biological_claim <- normalize_logical(
    identity$allow_gene_specific_biological_claim,
    "allow_gene_specific_biological_claim"
)

# --------------------------------------------------------------------------------------------------
# 10G.1G — PRESERVE FROZEN ROW ORDER
# --------------------------------------------------------------------------------------------------

identity$analysis_row_order <- seq_len(
    nrow(identity)
)

base <- identity

# --------------------------------------------------------------------------------------------------
# 10G.1H — SELECT CORRECTED PATHWAY FIELDS
# --------------------------------------------------------------------------------------------------

wanted_pathway_cols <- c(
    "ensemblID_stable",
    "GO_BP_represented_corrected",
    "Reactome_represented",
    "GO_BP_nominal_term_count",
    "Reactome_nominal_pathway_count",
    "pathway_inference_status"
)

missing_pathway_cols <- setdiff(
    wanted_pathway_cols,
    names(pathway)
)

if (length(missing_pathway_cols) > 0L) {

    stop(
        "Corrected pathway scaffold missing column(s): ",
        paste(
            missing_pathway_cols,
            collapse = ", "
        )
    )
}

pathway_small <- pathway[
    ,
    wanted_pathway_cols,
    drop = FALSE
]

# --------------------------------------------------------------------------------------------------
# 10G.1I — MERGE PATHWAY INFORMATION
# --------------------------------------------------------------------------------------------------

integrated <- merge(
    base,
    pathway_small,
    by = "ensemblID_stable",
    all.x = TRUE,
    sort = FALSE
)

integrated <- integrated[
    match(
        base$ensemblID_stable,
        integrated$ensemblID_stable
    ),
    ,
    drop = FALSE
]

if (!identical(
    integrated$ensemblID_stable,
    base$ensemblID_stable
)) {

    stop(
        "Unable to restore frozen 24-feature row order after pathway merge."
    )
}

# --------------------------------------------------------------------------------------------------
# 10G.1J — VERIFY FROZEN WEIGHTS / DIRECTIONS
# --------------------------------------------------------------------------------------------------

if (any(
    is.na(
        integrated$final_normalized_weight
    )
)) {

    stop(
        "Missing frozen score weight after merge."
    )
}

if (abs(
    sum(
        abs(
            integrated$final_normalized_weight
        )
    ) - 1
) > 1e-8) {

    stop(
        "Frozen normalized absolute weights no longer sum to 1."
    )
}

cat("Frozen normalized weights sum(abs(weight)) = 1 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10G.1K — WEIGHT RANK
# --------------------------------------------------------------------------------------------------

integrated$absolute_weight <- abs(
    integrated$final_normalized_weight
)

integrated$absolute_weight_rank <- rank(
    -integrated$absolute_weight,
    ties.method = "min"
)

# --------------------------------------------------------------------------------------------------
# 10G.1L — SCZ-SPECIFIC EVIDENCE FIELDS
# --------------------------------------------------------------------------------------------------

integrated$SCZ_literature_review_status <-
    "NO_DIRECT_SCZ_EVIDENCE_RETRIEVED_IN_CURRENT_FOCUSED_QUERY"

integrated$overall_direct_SCZ_evidence <-
    "NO_DIRECT_SCZ_EVIDENCE_RETRIEVED_IN_CURRENT_FOCUSED_QUERY"

integrated$direct_human_SCZ_evidence_summary <- ""

integrated$SCZ_manuscript_safe_interpretation <- ""

for (i in seq_len(nrow(integrated))) {

    j <- match(
        integrated$ensemblID_stable[i],
        scz$ensemblID_stable
    )

    if (!is.na(j)) {

        integrated$SCZ_literature_review_status[i] <-
            "PAPER_LEVEL_SCZ_EVIDENCE_MANUALLY_ADJUDICATED"

        integrated$overall_direct_SCZ_evidence[i] <-
            clean_text(
                scz$overall_direct_SCZ_evidence[j]
            )

        integrated$direct_human_SCZ_evidence_summary[i] <-
            clean_text(
                scz$direct_human_evidence_summary[j]
            )

        integrated$SCZ_manuscript_safe_interpretation[i] <-
            clean_text(
                scz$manuscript_safe_interpretation[j]
            )
    }
}

# --------------------------------------------------------------------------------------------------
# 10G.1M — BRAIN / NEURAL EVIDENCE FIELDS
# --------------------------------------------------------------------------------------------------

integrated$brain_literature_review_status <-
    "NO_REVIEWED_BRAIN_CONTEXT_FROM_CURRENT_SYSTEMATIC_QUERY"

integrated$overall_brain_context_strength <-
    "NO_REVIEWED_BRAIN_CONTEXT_FROM_CURRENT_SYSTEMATIC_QUERY"

integrated$dominant_brain_context <- ""

integrated$brain_manuscript_safe_interpretation <- ""

for (i in seq_len(nrow(integrated))) {

    j <- match(
        integrated$ensemblID_stable[i],
        brain$ensemblID_stable
    )

    if (!is.na(j)) {

        integrated$brain_literature_review_status[i] <-
            "PAPER_LEVEL_BRAIN_CONTEXT_MANUALLY_ADJUDICATED"

        integrated$overall_brain_context_strength[i] <-
            clean_text(
                brain$overall_brain_context_strength[j]
            )

        integrated$dominant_brain_context[i] <-
            clean_text(
                brain$dominant_brain_context[j]
            )

        integrated$brain_manuscript_safe_interpretation[i] <-
            clean_text(
                brain$manuscript_safe_brain_interpretation[j]
            )
    }
}

# --------------------------------------------------------------------------------------------------
# 10G.1N — CONVENTIONAL PATHWAY INTERPRETATION
#
# Global conclusion remains fixed:
#   GO-BP FDR < 0.05      = 0
#   Reactome FDR < 0.05   = 0
#
# Nominal counts are contextual only.
# --------------------------------------------------------------------------------------------------

integrated$conventional_pathway_global_status <-
    "NO_FDR_SUPPORTED_GO_BP_OR_REACTOME_ENRICHMENT"

integrated$nominal_pathway_interpretation <- ifelse(
    integrated$protein_coding_flag %in% TRUE,
    "NOMINAL_CONTEXT_ONLY_IF_PRESENT",
    "NOT_TESTED_IN_STANDARD_PROTEIN_CODING_ORA"
)

# Historical ARHGAP11B feature requires special pathway annotation caution.
arhgap_idx <- which(
    integrated$ensemblID_stable ==
        "ENSG00000187951"
)

if (length(arhgap_idx) != 1L) {

    stop(
        "Historical ARHGAP11B feature not uniquely found."
    )
}

integrated$nominal_pathway_interpretation[
    arhgap_idx
] <- paste(
    "HISTORICAL_IDENTITY_SPECIAL_CASE;",
    "current database mapping is not suitable for gene-specific pathway interpretation;"
)

# --------------------------------------------------------------------------------------------------
# 10G.1O — INTEGRATED EVIDENCE CLASS
#
# DESCRIPTIVE only.
# This is NOT a ranking and NOT a validation score.
# --------------------------------------------------------------------------------------------------

integrated$integrated_biological_evidence_class <- ""

for (i in seq_len(nrow(integrated))) {

    scz_status <-
        integrated$overall_direct_SCZ_evidence[i]

    brain_status <-
        integrated$overall_brain_context_strength[i]

    tier <-
        integrated$literature_interpretation_tier[i]

    # ----------------------------------------------------------------------------------------------
    # Identity-limited features
    # ----------------------------------------------------------------------------------------------

    if (
        tier %in%
        c(
            "TIER_C_ID_ANCHORED_NONCODING_FEATURE_REVIEW",
            "TIER_D_ID_ONLY_LIMITED_INTERPRETATION"
        )
    ) {

        integrated$integrated_biological_evidence_class[i] <-
            "IDENTITY_LIMITED_OR_ID_ANCHORED_FEATURE"

        next
    }

    # ----------------------------------------------------------------------------------------------
    # Direct SCZ literature reviewed
    # ----------------------------------------------------------------------------------------------

    if (
        scz_status !=
            "NO_DIRECT_SCZ_EVIDENCE_RETRIEVED_IN_CURRENT_FOCUSED_QUERY"
    ) {

        if (
            grepl(
                "STRONG|MODERATE_TO_STRONG",
                brain_status
            )
        ) {

            integrated$integrated_biological_evidence_class[i] <-
                "STRONG_NEURAL_CONTEXT_WITH_ADJUDICATED_SCZ_EVIDENCE"

        } else {

            integrated$integrated_biological_evidence_class[i] <-
                "ADJUDICATED_SCZ_EVIDENCE_WITH_LIMITED_NEURAL_CONTEXT"
        }

        next
    }

    # ----------------------------------------------------------------------------------------------
    # Brain context only
    # ----------------------------------------------------------------------------------------------

    if (
        brain_status %in%
        c(
            "STRONG",
            "MODERATE_TO_STRONG"
        )
    ) {

        integrated$integrated_biological_evidence_class[i] <-
            "STRONG_OR_SUBSTANTIAL_BRAIN_CONTEXT_WITHOUT_DIRECT_SCZ_EVIDENCE_RETRIEVED"

    } else if (
        brain_status %in%
        c(
            "MODERATE",
            "LIMITED_TO_MODERATE",
            "LIMITED"
        )
    ) {

        integrated$integrated_biological_evidence_class[i] <-
            "LIMITED_OR_MODERATE_BRAIN_CONTEXT_WITHOUT_DIRECT_SCZ_EVIDENCE_RETRIEVED"

    } else if (
        brain_status ==
            "NO_CONVINCING_CONTEXT"
    ) {

        integrated$integrated_biological_evidence_class[i] <-
            "NO_CONVINCING_REVIEWED_NEURAL_CONTEXT"

    } else {

        integrated$integrated_biological_evidence_class[i] <-
            "NO_FOCUSED_SCZ_OR_REVIEWED_BRAIN_CONTEXT_RETRIEVED"
    }
}

# --------------------------------------------------------------------------------------------------
# 10G.1P — MANUSCRIPT-SAFE INTEGRATED INTERPRETATION
# --------------------------------------------------------------------------------------------------

integrated$manuscript_safe_integrated_interpretation <- ""

for (i in seq_len(nrow(integrated))) {

    scz_text <- clean_text(
        integrated$SCZ_manuscript_safe_interpretation[i]
    )

    brain_text <- clean_text(
        integrated$brain_manuscript_safe_interpretation[i]
    )

    tier <- integrated$literature_interpretation_tier[i]

    # ----------------------------------------------------------------------------------------------
    # Both reviewed
    # ----------------------------------------------------------------------------------------------

    if (
        nzchar(scz_text) &&
        nzchar(brain_text)
    ) {

        integrated$manuscript_safe_integrated_interpretation[i] <-
            paste(
                scz_text,
                brain_text
            )

        next
    }

    # ----------------------------------------------------------------------------------------------
    # SCZ evidence only
    # ----------------------------------------------------------------------------------------------

    if (
        nzchar(scz_text)
    ) {

        integrated$manuscript_safe_integrated_interpretation[i] <-
            paste(
                scz_text,
                paste(
                    "No additional strong brain/cell-type conclusion was assigned",
                    "from the current systematic brain-context review."
                )
            )

        next
    }

    # ----------------------------------------------------------------------------------------------
    # Brain evidence only
    # ----------------------------------------------------------------------------------------------

    if (
        nzchar(brain_text)
    ) {

        integrated$manuscript_safe_integrated_interpretation[i] <-
            paste(
                paste(
                    "No direct schizophrenia/psychosis evidence was retrieved for this feature",
                    "by the current predefined focused query."
                ),
                brain_text
            )

        next
    }

    # ----------------------------------------------------------------------------------------------
    # Identity-limited features
    # ----------------------------------------------------------------------------------------------

    if (
        tier %in%
        c(
            "TIER_C_ID_ANCHORED_NONCODING_FEATURE_REVIEW",
            "TIER_D_ID_ONLY_LIMITED_INTERPRETATION"
        )
    ) {

        integrated$manuscript_safe_integrated_interpretation[i] <-
            paste(
                "This feature remains primarily Ensembl-ID anchored.",
                "The current systematic literature workflow did not establish a sufficiently",
                "specific named-gene schizophrenia or brain-function interpretation.",
                "It should therefore be reported mainly as a stable molecular feature",
                "rather than assigned an unsupported biological mechanism."
            )

        next
    }

    # ----------------------------------------------------------------------------------------------
    # Named feature with no focused evidence
    # ----------------------------------------------------------------------------------------------

    integrated$manuscript_safe_integrated_interpretation[i] <-
        paste(
            "No direct schizophrenia/psychosis evidence or reviewable brain/neural evidence",
            "was retrieved by the current predefined literature queries.",
            "This retrieval result should not be interpreted as proof of biological irrelevance."
        )
}

# --------------------------------------------------------------------------------------------------
# 10G.1Q — SPECIAL ARHGAP11B IDENTITY CAVEAT
# --------------------------------------------------------------------------------------------------

integrated$manuscript_safe_integrated_interpretation[
    arhgap_idx
] <- paste(
    integrated$manuscript_safe_integrated_interpretation[
        arhgap_idx
    ],
    paste(
        "Identity caveat:",
        "the frozen BrainSeq/GRCh37 feature ENSG00000187951 was historically annotated as ARHGAP11B;",
        "current ARHGAP11B uses Ensembl ID ENSG00000285077."
    )
)

# --------------------------------------------------------------------------------------------------
# 10G.1R — BASIC COMPOSITION AUDIT
# --------------------------------------------------------------------------------------------------

n_features <- nrow(
    integrated
)

n_coding <- sum(
    integrated$protein_coding_flag %in% TRUE
)

n_noncoding <- sum(
    integrated$protein_coding_flag %in% FALSE
)

n_positive <- sum(
    integrated$final_normalized_weight > 0
)

n_negative <- sum(
    integrated$final_normalized_weight < 0
)

positive_abs_weight <- sum(
    abs(
        integrated$final_normalized_weight[
            integrated$final_normalized_weight > 0
        ]
    )
)

negative_abs_weight <- sum(
    abs(
        integrated$final_normalized_weight[
            integrated$final_normalized_weight < 0
        ]
    )
)

if (
    n_features != 24L ||
    n_coding != 12L ||
    n_noncoding != 12L ||
    n_positive != 9L ||
    n_negative != 15L
) {

    stop(
        "Frozen signature composition no longer matches expected values.\n",
        "N=", n_features,
        "; coding=", n_coding,
        "; noncoding=", n_noncoding,
        "; positive=", n_positive,
        "; negative=", n_negative
    )
}

cat("24-feature composition integrity: PASS\n")
cat("Protein-coding: 12/24 — PASS\n")
cat("Non-protein-coding: 12/24 — PASS\n")
cat("Positive weights: 9/24 — PASS\n")
cat("Negative weights: 15/24 — PASS\n")

# --------------------------------------------------------------------------------------------------
# 10G.1S — LITERATURE COVERAGE COUNTS
# --------------------------------------------------------------------------------------------------

n_scz_adjudicated <- sum(
    integrated$SCZ_literature_review_status ==
        "PAPER_LEVEL_SCZ_EVIDENCE_MANUALLY_ADJUDICATED"
)

n_brain_adjudicated <- sum(
    integrated$brain_literature_review_status ==
        "PAPER_LEVEL_BRAIN_CONTEXT_MANUALLY_ADJUDICATED"
)

brain_strength_counts <- table(
    integrated$overall_brain_context_strength[
        integrated$brain_literature_review_status ==
            "PAPER_LEVEL_BRAIN_CONTEXT_MANUALLY_ADJUDICATED"
    ]
)

integrated_class_counts <- table(
    integrated$integrated_biological_evidence_class
)

# --------------------------------------------------------------------------------------------------
# 10G.1T — RESTORE FROZEN ROW ORDER
# --------------------------------------------------------------------------------------------------

integrated <- integrated[
    order(
        integrated$analysis_row_order
    ),
    ,
    drop = FALSE
]

rownames(integrated) <- NULL

# --------------------------------------------------------------------------------------------------
# 10G.1U — WRITE FULL INTEGRATED MATRIX
# --------------------------------------------------------------------------------------------------

write.csv(
    integrated,
    OUT_MATRIX,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10G.1V — WEIGHT-RANKED MATRIX
# --------------------------------------------------------------------------------------------------

ranked <- integrated[
    order(
        integrated$absolute_weight_rank,
        integrated$analysis_row_order
    ),
    ,
    drop = FALSE
]

rownames(ranked) <- NULL

write.csv(
    ranked,
    OUT_RANKED,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10G.1W — COMPACT MANUSCRIPT TABLE
# --------------------------------------------------------------------------------------------------

manuscript_table <- integrated[
    ,
    c(
        "ensemblID_stable",
        "gene_symbol",
        "interpretation_symbol",
        "gene_type",
        "direction_standardized",
        "final_normalized_weight",
        "absolute_weight_rank",
        "literature_interpretation_tier",
        "overall_direct_SCZ_evidence",
        "overall_brain_context_strength",
        "dominant_brain_context",
        "integrated_biological_evidence_class",
        "manuscript_safe_integrated_interpretation"
    ),
    drop = FALSE
]

write.csv(
    manuscript_table,
    OUT_MANUSCRIPT_TABLE,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 10G.1X — TOP-WEIGHTED FEATURES FOR CONSOLE AUDIT
# --------------------------------------------------------------------------------------------------

top10 <- head(
    ranked[
        ,
        c(
            "absolute_weight_rank",
            "ensemblID_stable",
            "gene_symbol",
            "interpretation_symbol",
            "gene_type",
            "direction_standardized",
            "final_normalized_weight",
            "overall_direct_SCZ_evidence",
            "overall_brain_context_strength"
        ),
        drop = FALSE
    ],
    10L
)

# --------------------------------------------------------------------------------------------------
# 10G.1Y — PROJECT-LEVEL BIOLOGICAL SYNTHESIS
# --------------------------------------------------------------------------------------------------

summary_lines <- c(

    "==============================================================================================================",
    "STEP 10G.1 — FINAL INTEGRATED BIOLOGICAL SYNTHESIS OF THE FROZEN 24-FEATURE SCZ-MSS STATE",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "FROZEN SIGNATURE COMPOSITION",

    paste0(
        "Total frozen features: ",
        n_features
    ),

    paste0(
        "Protein-coding: ",
        n_coding,
        "/24"
    ),

    paste0(
        "Non-protein-coding: ",
        n_noncoding,
        "/24"
    ),

    paste0(
        "Positive-direction features: ",
        n_positive,
        "/24"
    ),

    paste0(
        "Negative-direction features: ",
        n_negative,
        "/24"
    ),

    paste0(
        "Positive absolute-weight mass: ",
        format(
            positive_abs_weight,
            digits = 8
        )
    ),

    paste0(
        "Negative absolute-weight mass: ",
        format(
            negative_abs_weight,
            digits = 8
        )
    ),

    "",

    "CONVENTIONAL PATHWAY ANALYSIS",

    "GO-BP FDR < 0.05 terms: 0",
    "Reactome FDR < 0.05 pathways: 0",

    paste(
        "Therefore the frozen state does not support a statistically established",
        "conventional pathway-enrichment claim."
    ),

    paste(
        "Nominal GO/Reactome signals are retained only as exploratory gene-level context."
    ),

    "",

    "SCHIZOPHRENIA-SPECIFIC LITERATURE",

    paste0(
        "Features receiving focused paper-level SCZ evidence adjudication: ",
        n_scz_adjudicated,
        "/24"
    ),

    paste(
        "ARHGAP11B: weak and uncertain schizophrenia-related CNV/locus-level evidence;",
        "historical identifier handling is required."
    ),

    paste(
        "ALDH1A1: direct human schizophrenia evidence is predominantly null/negative,",
        "despite substantial dopamine/GABA-related neural biology."
    ),

    paste(
        "NR4A2: schizophrenia-relevant literature is substantial, but common-variant",
        "susceptibility evidence is mixed to largely null; molecular/cognitive and",
        "epigenetic contexts remain biologically relevant."
    ),

    "",

    "BRAIN / NEURAL / CELL-TYPE LITERATURE",

    paste0(
        "Features receiving focused paper-level brain-context adjudication: ",
        n_brain_adjudicated,
        "/24"
    ),

    paste(
        "Reviewed biological contexts were heterogeneous and included neurodevelopmental,",
        "dopaminergic, neuroimmune/microglial, neurovascular, lysosomal, synaptic and",
        "TGF-beta/activin-related biology."
    ),

    paste(
        "Strong neural biological plausibility was particularly evident for AP1AR-DT, PLD4,",
        "PMEPA1, ARHGAP11B, ETV5, ALDH1A1 and NR4A2."
    ),

    paste(
        "However, brain/neural plausibility must remain distinct from evidence of",
        "schizophrenia specificity or causal disease involvement."
    ),

    "",

    "NON-CODING COMPONENT",

    paste(
        "Half of the frozen state consists of non-protein-coding features.",
        "Several lack HGNC-approved named identities or remain stable-ID anchored."
    ),

    paste(
        "These features are retained as genuine components of the frozen transcriptomic state,",
        "but they should not be assigned unsupported gene-level mechanisms."
    ),

    "",

    "OVERALL BIOLOGICAL INTERPRETATION",

    paste(
        "The frozen SCZ-MSS state is biologically heterogeneous rather than dominated by",
        "a single conventionally enriched pathway."
    ),

    paste(
        "Its interpretable coding and named non-coding members span several plausible",
        "brain-related processes, but only a small subset has direct schizophrenia-specific",
        "literature, and that evidence is itself mixed or limited."
    ),

    paste(
        "Accordingly, the 24-feature state should be presented primarily as an empirically",
        "stable cross-region schizophrenia-associated transcriptomic state, with biological",
        "interpretation used as contextual support rather than as proof of a unified",
        "mechanism or causal schizophrenia pathway."
    ),

    "",

    "INTERPRETIVE GUARDRAILS",

    "1. Do not convert nominal GO/Reactome signals into significant pathway claims.",

    "2. Do not describe ALDH1A1 or NR4A2 as established schizophrenia-risk genes based on this review.",

    "3. Do not interpret current ENSG00000187951 annotation as the frozen ARHGAP11B feature; preserve the historical-ID caveat.",

    "4. Do not assign named-gene mechanisms to Tier C/D stable-ID-anchored non-coding features without additional evidence.",

    "5. Brain/cell-type plausibility is not equivalent to external replication of the frozen molecular-state score.",

    "6. Literature interpretation does not alter frozen membership, direction or score weights.",

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SYNTHESIS,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10G.1Z — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    IDENTITY_PATH,
    PATHWAY_PATH,
    SCZ_SYNTHESIS_PATH,
    BRAIN_SYNTHESIS_PATH,
    OUT_MATRIX,
    OUT_RANKED,
    OUT_MANUSCRIPT_TABLE,
    OUT_SYNTHESIS
)

manifest <- data.frame(
    file = basename(
        manifest_paths
    ),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(
            manifest_paths
        )$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 10G.1AA — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFINAL 24-FEATURE BIOLOGICAL SYNTHESIS SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Frozen features: ",
    n_features,
    "\n",
    sep = ""
)

cat(
    "Protein-coding: ",
    n_coding,
    "/24\n",
    sep = ""
)

cat(
    "Non-protein-coding: ",
    n_noncoding,
    "/24\n",
    sep = ""
)

cat(
    "Positive / negative directions: ",
    n_positive,
    " / ",
    n_negative,
    "\n",
    sep = ""
)

cat(
    "Positive absolute-weight mass: ",
    sprintf(
        "%.6f",
        positive_abs_weight
    ),
    "\n",
    sep = ""
)

cat(
    "Negative absolute-weight mass: ",
    sprintf(
        "%.6f",
        negative_abs_weight
    ),
    "\n",
    sep = ""
)

cat(
    "\nGO-BP FDR < 0.05: 0\n"
)

cat(
    "Reactome FDR < 0.05: 0\n"
)

cat(
    "\nSCZ-specific paper-level adjudicated features: ",
    n_scz_adjudicated,
    "/24\n",
    sep = ""
)

cat(
    "Brain-context paper-level adjudicated features: ",
    n_brain_adjudicated,
    "/24\n",
    sep = ""
)

cat(
    "\nBRAIN-CONTEXT GENE-LEVEL STRENGTH COUNTS\n"
)

print(
    brain_strength_counts
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nINTEGRATED BIOLOGICAL EVIDENCE CLASSES\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    integrated_class_counts
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nTOP 10 FEATURES BY FROZEN ABSOLUTE WEIGHT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    top10,
    row.names = FALSE
)

cat(
    "\nIntegrated 24-feature matrix:\n",
    OUT_MATRIX,
    "\n",
    sep = ""
)

cat(
    "\nWeight-ranked biological matrix:\n",
    OUT_RANKED,
    "\n",
    sep = ""
)

cat(
    "\nCompact manuscript biological table:\n",
    OUT_MANUSCRIPT_TABLE,
    "\n",
    sep = ""
)

cat(
    "\nProject-level biological synthesis:\n",
    OUT_SYNTHESIS,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 10G.1 COMPLETE — FINAL 24-FEATURE BIOLOGICAL SYNTHESIS FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 10G.1 — FINAL INTEGRATED 24-FEATURE BIOLOGICAL SYNTHESIS
Project sentinel: PASS
STEP10D.2 identity adjudication: PASS
STEP10C.1R pathway/coverage correction: PASS
STEP10E.4 SCZ evidence synthesis: PASS
STEP10F.2 brain-context synthesis: PASS
Frozen 24-feature set: PASS
SCZ gene-level synthesis rows: 3 — PASS
Brain-context gene-level synthesis rows: 13 — PASS
Frozen normalized weights sum(abs(weight)) = 1 — PASS
24-feature composition integrity: PASS
Protein-coding: 12/24 — PASS
Non-protein-coding: 12/24 — PASS
Positive weights: 9/24 — PASS
Negative weights: 15/24 — PASS

FINAL 24-FEATURE BIOLOGICAL SYNTHESIS SUMMARY
Frozen features: 24
Protein-coding: 12/24
Non-protein-coding: 12/24
Positive / negative directions: 9 / 15
Positive absolute-weight mass: 0.386919
Negative absolute-weight mass: 0.613081

GO-BP FDR < 0.05: 0
Reactome FDR < 0.05: 0

SCZ-specific paper-level adjudicated features: 3/24
Brain-context paper-level adjudicated features: 13/24

BRAIN-CONTEXT GENE-LEVEL STRENG

In [75]:
# ==================================================================================================
# STEP 11A.1 — MANUSCRIPT RESULTS + CLAIM REGISTRY FREEZE
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Freeze the study-level numbers, claims, limitations, and allowed manuscript wording
#   BEFORE figure/table/manuscript generation.
#
# THIS STEP DOES NOT:
#   - retrain anything
#   - recompute DE
#   - recompute CV
#   - recompute external validation
#   - alter score weights
#   - alter biological interpretation
#
# IT ONLY:
#   - records already-verified completed results
#   - separates primary findings from post-hoc/contextual findings
#   - records transparency limitations
#   - records wording that is allowed vs wording that must be avoided
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11A.1 — MANUSCRIPT RESULTS + CLAIM REGISTRY FREEZE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 11A.1A — PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "claim_registry_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

OUT_CLAIMS <- file.path(
    OUT_DIR,
    "STEP11A1_manuscript_claim_registry_v1.csv"
)

OUT_NUMBERS <- file.path(
    OUT_DIR,
    "STEP11A1_key_numeric_results_v1.csv"
)

OUT_LIMITATIONS <- file.path(
    OUT_DIR,
    "STEP11A1_transparency_and_limitations_registry_v1.csv"
)

OUT_SUMMARY <- file.path(
    OUT_DIR,
    "STEP11A1_manuscript_claim_freeze_summary_v1.txt"
)

OUT_MANIFEST <- file.path(
    OUT_DIR,
    "STEP11A1_manuscript_claim_freeze_manifest_v1.csv"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID,
        "\nObserved: ", project_id
    )
}

cat("Project sentinel: PASS\n")

# --------------------------------------------------------------------------------------------------
# 11A.1B — KEY NUMERIC RESULTS
# --------------------------------------------------------------------------------------------------

numbers <- data.frame(

    result_group = c(

        # Cohort
        rep("DISCOVERY_COHORT", 9),

        # DE
        rep("DIFFERENTIAL_EXPRESSION", 6),

        # CV
        rep("PRIMARY_MODEL_INTERNAL_VALIDATION", 7),

        # Baselines / comparisons
        rep("MODEL_COMPARISON", 10),

        # Stability
        rep("STABILITY", 7),

        # Score
        rep("FROZEN_SCORE", 4),

        # External
        rep("EXTERNAL_VALIDATION", 14),

        # Biology
        rep("BIOLOGICAL_SYNTHESIS", 10)
    ),

    metric = c(

        # Discovery cohort
        "Discovery samples",
        "Discovery donors",
        "DLPFC samples",
        "DLPFC controls",
        "DLPFC schizophrenia",
        "HIPPO samples",
        "HIPPO controls",
        "HIPPO schizophrenia",
        "Paired donors",

        # DE
        "DLPFC FDR<0.05 genes",
        "HIPPO FDR<0.05 genes",
        "Shared-component FDR<0.05 genes",
        "Deviation-component FDR<0.05 genes",
        "Regional logFC correlation",
        "Regional sign concordance",

        # Primary model F
        "Repeated OOF donor AUROC",
        "Repeated OOF sample AUROC",
        "Repeated OOF DLPFC AUROC",
        "Repeated OOF HIPPO AUROC",
        "Mean outer-fold AUROC",
        "Mean shared nonzero features",
        "Mean deviation nonzero features",

        # Comparison
        "Shared-only pooled donor AUROC",
        "Independent-regional donor AUROC",
        "DLPFC-only donor AUROC",
        "HIPPO-only donor AUROC",
        "Covariate-only donor AUROC",
        "F vs shared-only mean fold delta",
        "F vs shared-only repeat-block CI lower",
        "F vs shared-only repeat-block CI upper",
        "F vs independent-regional mean fold delta",
        "F vs covariate-only donor AUROC delta",

        # Stability
        "Outer-CV stable shared genes",
        "Half-sample stable shared genes",
        "Final frozen genes",
        "Stable deviation genes",
        "Permutation count",
        "Observed reference donor AUROC in permutation analysis",
        "Permutation empirical p",

        # Frozen score
        "Positive-weight features",
        "Negative-weight features",
        "Positive absolute-weight mass",
        "Negative absolute-weight mass",

        # External
        "External cohorts screened",
        "Technically eligible cohorts",
        "Technically non-evaluable cohorts",
        "GSE224683 samples",
        "GSE224683 controls",
        "GSE224683 schizophrenia",
        "GSE224683 mapped genes",
        "GSE224683 gene coverage",
        "GSE224683 retained absolute weight",
        "GSE224683 AUROC",
        "GSE224683 AUROC CI lower",
        "GSE224683 AUROC CI upper",
        "GSE224683 Welch p",
        "GSE224683 Hedges g",

        # Biology
        "Frozen features",
        "Protein-coding features",
        "Non-protein-coding features",
        "GO-BP FDR<0.05 terms",
        "Reactome FDR<0.05 pathways",
        "GO-BP effective coding coverage",
        "Reactome effective coding coverage",
        "SCZ paper-level adjudicated features",
        "Brain-context paper-level adjudicated features",
        "Strong brain-context genes"
    ),

    value = c(

        # Discovery
        755,
        454,
        379,
        226,
        153,
        376,
        243,
        133,
        301,

        # DE
        2949,
        2648,
        5348,
        11,
        0.6089,
        0.735,

        # F
        0.8019743,
        0.7938,
        0.7859,
        0.7997,
        0.7868,
        1501.1,
        59.6,

        # Comparison
        0.7987926,
        0.81502784,
        0.7906762,
        0.805192,
        0.6143507,
        0.00201539,
        0.00103645,
        0.003090892,
        -0.01153163,
        0.1876237,

        # Stability
        403,
        24,
        24,
        0,
        100,
        0.7698301,
        0.00990099,

        # Score
        9,
        15,
        0.386919,
        0.613081,

        # External
        5,
        1,
        4,
        30,
        15,
        15,
        23,
        0.958333,
        0.9266,
        0.5067,
        0.2889,
        0.7244,
        0.90635,
        -0.0422,

        # Biology
        24,
        12,
        12,
        0,
        0,
        11,
        9,
        3,
        13,
        7
    ),

    stringsAsFactors = FALSE
)

write.csv(
    numbers,
    OUT_NUMBERS,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 11A.1C — CLAIM REGISTRY
# --------------------------------------------------------------------------------------------------

claims <- data.frame(

    claim_id = sprintf(
        "CLAIM_%02d",
        1:24
    ),

    manuscript_section = c(
        "Methods",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Results",
        "Discussion",
        "Discussion",
        "Discussion",
        "Discussion",
        "Discussion",
        "Discussion",
        "Abstract",
        "Conclusion"
    ),

    claim_type = c(
        "DESIGN",
        "COHORT",
        "DE",
        "DE",
        "INTERNAL_VALIDATION",
        "MODEL_STRUCTURE",
        "MODEL_COMPARISON",
        "MODEL_COMPARISON",
        "COVARIATE_SENSITIVITY",
        "STABILITY",
        "PERMUTATION",
        "FINAL_SIGNATURE",
        "SCORE",
        "EXTERNAL_SCREENING",
        "EXTERNAL_VALIDATION",
        "EXTERNAL_VALIDATION",
        "BIOLOGY",
        "BIOLOGY",
        "BIOLOGY",
        "BIOLOGY",
        "INTERPRETATION",
        "INTERPRETATION",
        "ABSTRACT_CORE",
        "CONCLUSION_CORE"
    ),

    allowed_claim = c(

        paste(
            "The study used donor-grouped nested cross-validation and explicitly decomposed",
            "shared cross-region schizophrenia signal from region-specific deviation."
        ),

        paste(
            "The discovery cohort contained 755 samples from 454 donors across DLPFC and hippocampus."
        ),

        paste(
            "Thousands of diagnosis-associated genes were identified within each region and",
            "in the shared cross-region component."
        ),

        paste(
            "Only 11 genes showed FDR-significant region-specific deviation in differential expression."
        ),

        paste(
            "The shared-plus-deviation model achieved a repeated out-of-fold donor AUROC of approximately 0.802."
        ),

        paste(
            "The fitted models were dominated by shared coefficients, with relatively few",
            "non-zero regional-deviation coefficients."
        ),

        paste(
            "The shared-plus-deviation model showed only a very small performance increment",
            "over the shared-only pooled model."
        ),

        paste(
            "Independent region-specific models had numerically higher average discrimination",
            "than the shared-plus-deviation model."
        ),

        paste(
            "The molecular model substantially outperformed the covariate-only model."
        ),

        paste(
            "A conservative conjunction of outer-CV and complementary half-sample stability",
            "analyses yielded a final 24-feature shared-state signature."
        ),

        paste(
            "Diagnosis-label permutation analysis supported that the observed predictive",
            "structure was unlikely under the tested null-label procedure."
        ),

        paste(
            "No region-specific deviation feature satisfied the final stability criteria."
        ),

        paste(
            "The frozen portable score used the 24 stable shared features with fixed",
            "direction and normalized weights."
        ),

        paste(
            "Five public external cohorts were screened, but only one met the prespecified",
            "technical evaluability gate for outcome validation."
        ),

        paste(
            "In the technically eligible public cohort GSE224683, the frozen score showed",
            "no evidence of diagnosis discrimination."
        ),

        paste(
            "The public external-validation phase therefore challenged rather than confirmed",
            "cross-dataset portability of the frozen score."
        ),

        paste(
            "The 24-feature state contained equal numbers of protein-coding and non-protein-coding features."
        ),

        paste(
            "No GO-BP or Reactome category survived FDR correction in the conventional",
            "protein-coding enrichment analysis."
        ),

        paste(
            "The interpretable features spanned heterogeneous neural contexts including",
            "dopaminergic, neurodevelopmental, neuroimmune, neurovascular, synaptic and",
            "TGF-beta/activin-related biology."
        ),

        paste(
            "Direct schizophrenia-specific literature support was limited and heterogeneous;",
            "NR4A2 and ALDH1A1 in particular had substantial neural biology but mixed/null",
            "direct schizophrenia evidence."
        ),

        paste(
            "The final signature is best interpreted as an empirically stable cross-region",
            "transcriptomic disease state rather than evidence for a single unified pathway."
        ),

        paste(
            "Biological plausibility does not constitute external replication of the frozen score."
        ),

        paste(
            "A donor-aware cross-region framework identified a compact 24-feature",
            "schizophrenia-associated shared transcriptomic state with strong internal",
            "predictive structure but limited demonstrated public-dataset portability."
        ),

        paste(
            "The study supports the existence of a reproducible internal shared cross-region",
            "schizophrenia transcriptional state while emphasizing important limitations",
            "in external technical transferability and biological specificity."
        )
    ),

    prohibited_overclaim = c(

        "Do not claim causal inference or donor-independent biological causality.",

        "Do not describe the cohort as population-representative.",

        "Do not imply every regional DE gene is shared across regions.",

        "Do not claim DLPFC and hippocampus are transcriptionally identical.",

        "Do not report the post-selection 24-gene score AUROC as unbiased internal validation.",

        "Do not claim regional deviation was completely absent.",

        "Do not claim the shared-plus-deviation model meaningfully or substantially outperformed shared-only.",

        "Do not claim Model F was the best-performing architecture.",

        "Do not imply clinical diagnostic utility.",

        "Do not describe the 24-gene conjunction as fully prospectively prespecified.",

        "Do not interpret permutation significance as proof of external generalizability.",

        "Do not claim all region-specific biology is absent.",

        "Do not retrain, reweight, or outcome-adapt the frozen external score.",

        "Do not describe technically non-evaluable cohorts as biological replication failures.",

        "Do not call GSE224683 an external replication success.",

        "Do not claim no other schizophrenia transcriptomic dataset exists.",

        "Do not reduce the signature to protein-coding genes only.",

        "Do not report nominal pathways as statistically established enrichment.",

        "Do not claim the literature demonstrates one coherent mechanism.",

        "Do not describe NR4A2 or ALDH1A1 as established schizophrenia-risk genes.",

        "Do not claim biological interpretation validates the score.",

        "Do not use literature evidence to rescue the null external score.",

        "Do not claim established clinical portability.",

        "Do not claim first-ever, definitive mechanism, or validated diagnostic biomarker."
    ),

    status = "FROZEN_ALLOWED_WORDING",

    stringsAsFactors = FALSE
)

write.csv(
    claims,
    OUT_CLAIMS,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 11A.1D — TRANSPARENCY / LIMITATION REGISTRY
# --------------------------------------------------------------------------------------------------

limitations <- data.frame(

    limitation_id = sprintf(
        "LIMIT_%02d",
        1:10
    ),

    category = c(
        "PROTOCOL",
        "OOF_GOVERNANCE",
        "STABILITY_GOVERNANCE",
        "EXTERNAL_ACCESS",
        "EXTERNAL_ACCESS",
        "OUTCOME_BLINDING",
        "EXTERNAL_GENERALIZABILITY",
        "PATHWAY_ANALYSIS",
        "BIOLOGICAL_INTERPRETATION",
        "SCORE_INTERPRETATION"
    ),

    limitation = c(

        paste(
            "The formal Step 0 analysis decision lock was not created before all modelling",
            "steps and was reconstructed retrospectively."
        ),

        paste(
            "The master out-of-fold prediction freeze was backfilled retrospectively after",
            "the nested-CV analyses had already been completed."
        ),

        paste(
            "The final outer-stable AND half-sample-stable 24-feature conjunction was",
            "conservative but was not separately prospectively locked before stability analysis."
        ),

        paste(
            "The originally intended CommonMind external cohort was not accessible under",
            "the available permissions."
        ),

        paste(
            "Public external validation therefore used an access-driven replacement strategy."
        ),

        paste(
            "GSE80655 phenotype information was inspected before final technical-gate closure;",
            "this must remain disclosed."
        ),

        paste(
            "Only one of five screened public cohorts was technically eligible for actual",
            "diagnosis-score validation, limiting conclusions about external generalizability."
        ),

        paste(
            "Conventional enrichment involved only the protein-coding component and found",
            "no FDR-significant GO-BP or Reactome enrichment."
        ),

        paste(
            "Gene-level literature evidence is heterogeneous, and several non-coding features",
            "have limited or ID-anchored biological annotation."
        ),

        paste(
            "The 24-feature post-selection discovery score separation is descriptive and",
            "must not be conflated with the unbiased nested-CV performance of the full model."
        )
    ),

    reporting_requirement = c(

        "Methods/Supplementary transparency statement",

        "Methods/Supplementary transparency statement",

        "Methods/Supplementary transparency statement",

        "External validation Methods",

        "External validation Methods",

        "External validation limitation",

        "Results + Discussion limitation",

        "Biological interpretation Results/Discussion",

        "Discussion limitation",

        "Results/Discussion"
    ),

    stringsAsFactors = FALSE
)

write.csv(
    limitations,
    OUT_LIMITATIONS,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 11A.1E — CONSISTENCY CHECKS
# --------------------------------------------------------------------------------------------------

stopifnot(
    nrow(numbers) == 67L
)

stopifnot(
    nrow(claims) == 24L
)

stopifnot(
    nrow(limitations) == 10L
)

# Expected key facts
stopifnot(
    numbers$value[
        numbers$metric == "Discovery samples"
    ] == 755
)

stopifnot(
    numbers$value[
        numbers$metric == "Discovery donors"
    ] == 454
)

stopifnot(
    numbers$value[
        numbers$metric == "Final frozen genes"
    ] == 24
)

stopifnot(
    numbers$value[
        numbers$metric == "Technically eligible cohorts"
    ] == 1
)

stopifnot(
    numbers$value[
        numbers$metric == "GSE224683 AUROC"
    ] == 0.5067
)

stopifnot(
    numbers$value[
        numbers$metric == "GO-BP FDR<0.05 terms"
    ] == 0
)

stopifnot(
    numbers$value[
        numbers$metric == "Reactome FDR<0.05 pathways"
    ] == 0
)

cat("Claim-registry internal consistency checks: PASS\n")

# --------------------------------------------------------------------------------------------------
# 11A.1F — SUMMARY
# --------------------------------------------------------------------------------------------------

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11A.1 — MANUSCRIPT RESULTS + CLAIM REGISTRY FREEZE",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    paste0(
        "Key numeric results frozen: ",
        nrow(numbers)
    ),

    paste0(
        "Allowed manuscript claims frozen: ",
        nrow(claims)
    ),

    paste0(
        "Transparency/limitation items frozen: ",
        nrow(limitations)
    ),

    "",

    "CORE STUDY MESSAGE",

    paste(
        "A donor-aware cross-region decomposition identified a compact and highly stable",
        "schizophrenia-associated shared transcriptomic state across DLPFC and hippocampus."
    ),

    paste(
        "Stable region-specific deviation was minimal, although independent regional models",
        "retained some additional predictive information."
    ),

    paste(
        "The final frozen 24-feature state showed strong internal predictive structure but",
        "did not replicate diagnostically in the one technically eligible public external cohort."
    ),

    paste(
        "Biological interpretation was heterogeneous and did not yield a single",
        "FDR-supported conventional pathway."
    ),

    "",

    "PRIMARY NUMERIC ANCHORS",

    "Discovery: 755 samples / 454 donors",

    "Primary repeated OOF donor AUROC: 0.8019743",

    "Final stable shared state: 24 features",

    "Stable deviation features: 0",

    "Permutation empirical p: 0.00990099",

    "Public external cohorts screened: 5",

    "Technically evaluable public external cohorts: 1",

    "GSE224683 AUROC: 0.5067 (95% CI 0.2889–0.7244)",

    "GO-BP FDR-significant terms: 0",

    "Reactome FDR-significant pathways: 0",

    "",

    "MANUSCRIPT FRAME",

    paste(
        "Primary contribution:",
        "donor-aware shared-vs-regional decomposition + stability-based compact state."
    ),

    paste(
        "Not the contribution:",
        "best classifier, externally validated diagnostic biomarker, or single-pathway mechanism."
    ),

    "",

    "NEXT STEP",

    paste(
        "Use this frozen registry to construct the final manuscript figures and tables",
        "without re-running completed inferential analyses."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    OUT_SUMMARY,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 11A.1G — SHA256 MANIFEST
# --------------------------------------------------------------------------------------------------

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    OUT_NUMBERS,
    OUT_CLAIMS,
    OUT_LIMITATIONS,
    OUT_SUMMARY
)

manifest <- data.frame(
    file = basename(
        manifest_paths
    ),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(
            manifest_paths
        )$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    OUT_MANIFEST,
    row.names = FALSE,
    quote = TRUE
)

# --------------------------------------------------------------------------------------------------
# 11A.1H — FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nMANUSCRIPT CLAIM FREEZE SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Key numeric results frozen: ",
    nrow(numbers),
    "\n",
    sep = ""
)

cat(
    "Allowed claims frozen: ",
    nrow(claims),
    "\n",
    sep = ""
)

cat(
    "Transparency/limitation items frozen: ",
    nrow(limitations),
    "\n",
    sep = ""
)

cat(
    "\nPRIMARY NUMERIC ANCHORS\n"
)

anchors <- numbers[
    numbers$metric %in%
        c(
            "Discovery samples",
            "Discovery donors",
            "Repeated OOF donor AUROC",
            "Final frozen genes",
            "Stable deviation genes",
            "Permutation empirical p",
            "External cohorts screened",
            "Technically eligible cohorts",
            "GSE224683 AUROC",
            "GO-BP FDR<0.05 terms",
            "Reactome FDR<0.05 pathways"
        ),
    ,
    drop = FALSE
]

print(
    anchors,
    row.names = FALSE
)

cat(
    "\nClaim registry:\n",
    OUT_CLAIMS,
    "\n",
    sep = ""
)

cat(
    "\nKey numeric results:\n",
    OUT_NUMBERS,
    "\n",
    sep = ""
)

cat(
    "\nTransparency registry:\n",
    OUT_LIMITATIONS,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    OUT_SUMMARY,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11A.1 COMPLETE — MANUSCRIPT CLAIM REGISTRY FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11A.1 — MANUSCRIPT RESULTS + CLAIM REGISTRY FREEZE
Project sentinel: PASS
Claim-registry internal consistency checks: PASS

MANUSCRIPT CLAIM FREEZE SUMMARY
Key numeric results frozen: 67
Allowed claims frozen: 24
Transparency/limitation items frozen: 10

PRIMARY NUMERIC ANCHORS
                      result_group                       metric        value
                  DISCOVERY_COHORT            Discovery samples 755.00000000
                  DISCOVERY_COHORT             Discovery donors 454.00000000
 PRIMARY_MODEL_INTERNAL_VALIDATION     Repeated OOF donor AUROC   0.80197430
                         STABILITY           Final frozen genes  24.00000000
                         STABILITY       Stable deviation genes   0.00000000
                         STABILITY      Permutation empirical p   0.00990099
               EXTERNAL_VALIDATION    External cohorts screened   5.00000000
               EXTERNAL_VALIDATION Technically eligible cohorts   1.00000000
               EXTERNA

In [77]:
# ==================================================================================================
# STEP 11B.0 — PUBLICATION ARTIFACT + REANALYSIS READINESS AUDIT
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Verify that all important saved artifacts required for:
#       1. manuscript figures,
#       2. manuscript tables,
#       3. reviewer-requested follow-up analyses,
#       4. reproducibility/environment recovery
#   are physically present on disk and usable WITHOUT rerunning completed expensive analyses.
#
# THIS STEP DOES NOT:
#   - rerun preprocessing
#   - rerun differential expression
#   - refit glmnet models
#   - rerun nested CV
#   - rerun half-sample stability
#   - rerun permutations
#   - rerun external validation
#   - alter renv.lock
#
# OUTPUT
#   1. Full project file inventory
#   2. Critical artifact readiness table
#   3. Expensive-checkpoint count audit
#   4. Figure/table source-readiness map
#   5. Notebook-source inventory
#   6. Final renv status capture
#   7. Missing/unready artifact table
#   8. Human-readable readiness summary
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11B.0 — PUBLICATION ARTIFACT + REANALYSIS READINESS AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 11B.0A — PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing: ", SENTINEL_PATH)
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID,
        "\nObserved: ", paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "artifact_readiness_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

INVENTORY_PATH <- file.path(
    OUT_DIR,
    "STEP11B0_full_project_file_inventory_v1.csv"
)

CRITICAL_PATH <- file.path(
    OUT_DIR,
    "STEP11B0_critical_artifact_readiness_v1.csv"
)

CHECKPOINT_PATH <- file.path(
    OUT_DIR,
    "STEP11B0_expensive_checkpoint_count_audit_v1.csv"
)

DISPLAY_MAP_PATH <- file.path(
    OUT_DIR,
    "STEP11B0_figure_table_source_readiness_map_v1.csv"
)

NOTEBOOK_PATH <- file.path(
    OUT_DIR,
    "STEP11B0_notebook_source_inventory_v1.csv"
)

RENV_STATUS_PATH <- file.path(
    OUT_DIR,
    "STEP11B0_final_renv_status_v1.txt"
)

FINAL_SESSION_PATH <- file.path(
    OUT_DIR,
    "STEP11B0_final_sessionInfo_v1.txt"
)

MISSING_PATH <- file.path(
    OUT_DIR,
    "STEP11B0_missing_or_unready_artifacts_v1.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "STEP11B0_publication_readiness_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "STEP11B0_publication_readiness_manifest_v1.csv"
)

# ==================================================================================================
# 11B.0B — FULL PROJECT FILE INVENTORY
# ==================================================================================================

all_paths <- list.files(
    PROJECT_ROOT,
    recursive = TRUE,
    full.names = TRUE,
    all.files = TRUE,
    no.. = TRUE
)

is_file <- file.exists(all_paths) &
    !dir.exists(all_paths)

all_files <- all_paths[
    is_file
]

if (length(all_files) == 0L) {
    stop("No project files found.")
}

fi <- file.info(
    all_files
)

relative_path <- substring(
    gsub(
        "\\\\",
        "/",
        all_files
    ),
    nchar(
        gsub(
            "\\\\",
            "/",
            PROJECT_ROOT
        )
    ) + 2L
)

top_level <- sub(
    "/.*$",
    "",
    relative_path
)

PROJECT_INVENTORY <- data.frame(
    relative_path = relative_path,
    absolute_path = all_files,
    top_level = top_level,
    bytes = as.numeric(fi$size),
    megabytes = as.numeric(fi$size) / 1024^2,
    modified_time = as.character(fi$mtime),
    zero_byte = as.numeric(fi$size) <= 0,
    extension = tolower(
        tools::file_ext(
            all_files
        )
    ),
    stringsAsFactors = FALSE
)

PROJECT_INVENTORY <- PROJECT_INVENTORY[
    order(
        PROJECT_INVENTORY$top_level,
        PROJECT_INVENTORY$relative_path
    ),
    ,
    drop = FALSE
]

write.csv(
    PROJECT_INVENTORY,
    INVENTORY_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

project_total_gb <- sum(
    PROJECT_INVENTORY$bytes,
    na.rm = TRUE
) / 1024^3

n_zero_byte <- sum(
    PROJECT_INVENTORY$zero_byte,
    na.rm = TRUE
)

cat(
    "Project files inventoried: ",
    nrow(PROJECT_INVENTORY),
    "\n",
    sep = ""
)

cat(
    "Total inventoried project size: ",
    sprintf("%.3f GB", project_total_gb),
    "\n",
    sep = ""
)

cat(
    "Zero-byte files detected: ",
    n_zero_byte,
    "\n",
    sep = ""
)

# ==================================================================================================
# 11B.0C — CRITICAL ARTIFACT REGISTRY
# ==================================================================================================

artifact <- function(
    stage,
    role,
    relative_path,
    requirement,
    expected_rows = NA_integer_,
    expected_cols = NA_integer_,
    deep_check = FALSE
) {
    data.frame(
        stage = stage,
        role = role,
        relative_path = relative_path,
        requirement = requirement,
        expected_rows = expected_rows,
        expected_cols = expected_cols,
        deep_check = deep_check,
        stringsAsFactors = FALSE
    )
}

CRITICAL <- do.call(
    rbind,
    list(

        # ------------------------------------------------------------------------------------------
        # ENVIRONMENT / SOURCE
        # ------------------------------------------------------------------------------------------

        artifact(
            "00_ENVIRONMENT",
            "Final dependency lock",
            "renv.lock",
            "REPRODUCIBILITY"
        ),

        artifact(
            "00_ENVIRONMENT",
            "Initial session information",
            "15_environment/step00a_sessionInfo.txt",
            "REPRODUCIBILITY"
        ),

        artifact(
            "00_RAW",
            "Original BrainSeq RSE source",
            "01_raw/brainseq_phase2/rse_gene_unfiltered.Rdata",
            "REANALYSIS_BACKUP"
        ),

        # ------------------------------------------------------------------------------------------
        # COHORT / PREPROCESSING
        # ------------------------------------------------------------------------------------------

        artifact(
            "01_COHORT",
            "Frozen discovery sample cohort",
            "04_processed/brainseq_discovery_cohort_samples_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 755L
        ),

        artifact(
            "01_COHORT",
            "Frozen discovery donor cohort",
            "04_processed/brainseq_discovery_cohort_donors_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 454L
        ),

        artifact(
            "02_PREPROCESSING",
            "Primary analysis count matrix",
            "04_processed/brainseq_primary_counts_v1.rds",
            "REANALYSIS_BACKUP",
            expected_rows = 23740L,
            expected_cols = 755L,
            deep_check = TRUE
        ),

        artifact(
            "02_PREPROCESSING",
            "Primary gene annotation",
            "04_processed/brainseq_primary_gene_annotation_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 23740L
        ),

        artifact(
            "02_PREPROCESSING",
            "Final model metadata",
            "04_processed/brainseq_model_metadata_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 755L
        ),

        artifact(
            "02_PREPROCESSING",
            "TMM logCPM matrix used for QC/visualization",
            "03_intermediate/brainseq_primary_logCPM_TMM_QC_v1.rds",
            "MANUSCRIPT_SOURCE",
            expected_rows = 23740L,
            expected_cols = 755L,
            deep_check = TRUE
        ),

        artifact(
            "02_PREPROCESSING",
            "Sample-level PCA/QC coordinates",
            "02_metadata/step02b_sample_PCA_QC_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 755L
        ),

        artifact(
            "02_PREPROCESSING",
            "PCA variance explained",
            "02_metadata/step02b_PCA_variance_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "02_PREPROCESSING",
            "TMM normalization summary",
            "02_metadata/step02b_TMM_normalization_summary_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "02_PREPROCESSING",
            "QC extreme-sample audit",
            "02_metadata/step02b_PCA_extreme_samples_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        # ------------------------------------------------------------------------------------------
        # DIFFERENTIAL EXPRESSION
        # ------------------------------------------------------------------------------------------

        artifact(
            "03_DE",
            "DLPFC full DE table",
            "05_differential_expression/step03a_DLPFC_SCZ_vs_Control_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 23740L
        ),

        artifact(
            "03_DE",
            "HIPPO full DE table",
            "05_differential_expression/step03a_HIPPO_SCZ_vs_Control_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 23740L
        ),

        artifact(
            "03_DE",
            "Shared-effect full DE table",
            "05_differential_expression/step03a_Shared_SCZ_effect_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 23740L
        ),

        artifact(
            "03_DE",
            "Regional-deviation full DE table",
            "05_differential_expression/step03a_Regional_Deviation_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 23740L
        ),

        artifact(
            "03_DE",
            "DE summary",
            "05_differential_expression/step03a_DE_summary_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "03_DE",
            "duplicateCorrelation/model audit",
            "05_differential_expression/step03a_model_audit_v1.txt",
            "REANALYSIS_BACKUP"
        ),

        # ------------------------------------------------------------------------------------------
        # CV SPLITS / PRIMARY MODEL
        # ------------------------------------------------------------------------------------------

        artifact(
            "04_CV",
            "Frozen donor CV split map",
            "06_models/cv_splits/outer_cv_donor_splits_5fold_5repeat_v1.csv",
            "REANALYSIS_BACKUP",
            expected_rows = 2270L
        ),

        artifact(
            "04_CV",
            "Frozen sample CV map",
            "06_models/cv_splits/outer_cv_sample_map_5fold_5repeat_v1.csv",
            "REANALYSIS_BACKUP"
        ),

        artifact(
            "04_CV",
            "All 25 primary-model outer-fold summaries",
            "06_models/nested_cv_5x5_v1/ALL_25_OUTER_FOLD_SUMMARY_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 25L
        ),

        artifact(
            "04_CV",
            "Final primary-model performance",
            "06_models/nested_cv_5x5_v1/NESTED_CV_FINAL_PERFORMANCE_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "04_CV",
            "Averaged primary-model donor OOF predictions",
            "06_models/nested_cv_5x5_v1/OOF_DONOR_PREDICTIONS_AVERAGED_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "04_CV",
            "Averaged primary-model sample OOF predictions",
            "06_models/nested_cv_5x5_v1/OOF_SAMPLE_PREDICTIONS_AVERAGED_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "04_CV",
            "Hyperparameter selection frequencies",
            "06_models/nested_cv_5x5_v1/HYPERPARAMETER_SELECTION_FREQUENCY_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        # ------------------------------------------------------------------------------------------
        # BASELINES / MODEL COMPARISON
        # ------------------------------------------------------------------------------------------

        artifact(
            "05_MODEL_COMPARISON",
            "Final predefined model performance summary",
            "06_models/model_comparison_v1/STEP05C_model_performance_summary_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "05_MODEL_COMPARISON",
            "Paired predefined model comparisons",
            "06_models/model_comparison_v1/STEP05C_paired_comparison_summary_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "05_MODEL_COMPARISON",
            "Outer-fold paired differences",
            "06_models/model_comparison_v1/STEP05C_paired_outer_fold_differences_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "05_MODEL_COMPARISON",
            "Repeat-block bootstrap distributions",
            "06_models/model_comparison_v1/STEP05C_repeat_block_bootstrap_distributions_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "05_MODEL_COMPARISON",
            "Covariate-only sensitivity summary",
            "06_models/baselines/model_covariate_only_sensitivity_v1/covariate_only_sensitivity_summary_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        # ------------------------------------------------------------------------------------------
        # CANONICAL OOF FREEZE
        # ------------------------------------------------------------------------------------------

        artifact(
            "05_OOF_FREEZE",
            "Canonical donor OOF predictions",
            "06_models/oof_freeze_v1/STEP05D_CANONICAL_donor_OOF_predictions_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "05_OOF_FREEZE",
            "Canonical sample OOF predictions",
            "06_models/oof_freeze_v1/STEP05D_CANONICAL_sample_OOF_predictions_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "05_OOF_FREEZE",
            "Canonical OOF model summary",
            "06_models/oof_freeze_v1/STEP05D_OOF_model_summary_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 5L
        ),

        artifact(
            "05_OOF_FREEZE",
            "Canonical OOF freeze manifest",
            "06_models/oof_freeze_v1/STEP05D_OOF_FREEZE_manifest_v1.csv",
            "REPRODUCIBILITY"
        ),

        # ------------------------------------------------------------------------------------------
        # STABILITY
        # ------------------------------------------------------------------------------------------

        artifact(
            "06_STABILITY",
            "Outer-CV all-gene stability table",
            "06_models/stability_v1/step05a_all_gene_stability_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "06_STABILITY",
            "Outer-CV shared selection-frequency distribution",
            "06_models/stability_v1/step05a_shared_selection_frequency_distribution_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "06_STABILITY",
            "Outer-model coefficient matrices",
            "06_models/stability_v1/step05a_outer_model_coefficient_matrices_v1.rds",
            "REANALYSIS_BACKUP"
        ),

        artifact(
            "06_STABILITY",
            "Half-sample stability summary",
            "07_stability/complementary_half_sample_v1/step06b_half_sample_stability_summary_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "06_STABILITY",
            "Half-sample coefficient matrices",
            "07_stability/complementary_half_sample_v1/step06b_half_sample_coefficient_matrices_v1.rds",
            "REANALYSIS_BACKUP"
        ),

        artifact(
            "06_STABILITY",
            "Permutation-null summary",
            "07_stability/permutation_null_v1/step06c_permutation_null_summary_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "06_STABILITY",
            "Permutation-null coefficient matrices",
            "07_stability/permutation_null_v1/step06c_null_coefficient_matrices_v1.rds",
            "REANALYSIS_BACKUP"
        ),

        artifact(
            "06_STABILITY",
            "Final frozen stable state",
            "07_stability/final_stable_state_v1/step06d_FINAL_stable_shared_state_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 24L
        ),

        artifact(
            "06_STABILITY",
            "Final stability evidence",
            "07_stability/final_stable_state_v1/step06d_final_gene_stability_evidence_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 24L
        ),

        # ------------------------------------------------------------------------------------------
        # SCORE LOCK
        # ------------------------------------------------------------------------------------------

        artifact(
            "07_SCORE",
            "Frozen 24-feature score weights",
            "08_score_lock/SCZ_MSS_v1_gene_weights.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 24L
        ),

        artifact(
            "07_SCORE",
            "Score lock manifest",
            "08_score_lock/SCZ_MSS_v1_lock_manifest.csv",
            "REPRODUCIBILITY"
        ),

        artifact(
            "07_SCORE",
            "Frozen mapping policy",
            "08_score_lock/SCZ_MSS_v1_mapping_policy.json",
            "REPRODUCIBILITY"
        ),

        artifact(
            "07_SCORE",
            "Frozen technical gate",
            "08_score_lock/SCZ_MSS_v1_technical_gate.json",
            "REPRODUCIBILITY"
        ),

        artifact(
            "07_SCORE",
            "Discovery sample-level frozen scores",
            "08_score_lock/step07b_BrainSeq_SCZ_MSS_v1_sample_scores.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 755L
        ),

        artifact(
            "07_SCORE",
            "Paired-donor scores",
            "08_score_lock/step07c_paired_donor_SCZ_MSS_v1_scores.csv",
            "MANUSCRIPT_SOURCE"
        ),

        # ------------------------------------------------------------------------------------------
        # EXTERNAL VALIDATION
        # ------------------------------------------------------------------------------------------

        artifact(
            "09_EXTERNAL",
            "Master external technical-gate registry",
            "09_external_validation/master_registry_v1/SCZ_MSS_v1_external_technical_gate_registry_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 5L
        ),

        artifact(
            "09_EXTERNAL",
            "Final external-validation summary",
            "09_external_validation/master_registry_v1/SCZ_MSS_v1_FINAL_external_validation_summary_table_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 5L
        ),

        artifact(
            "09_EXTERNAL",
            "GSE224683 frozen sample scores",
            "09_external_validation/gse224683/STEP09D_GSE224683_sample_SCZ_MSS_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 30L
        ),

        artifact(
            "09_EXTERNAL",
            "GSE224683 validation summary",
            "09_external_validation/gse224683/STEP09D_GSE224683_external_validation_summary_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "09_EXTERNAL",
            "GSE224683 bootstrap distribution",
            "09_external_validation/gse224683/STEP09D_GSE224683_bootstrap_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "09_EXTERNAL",
            "GSE224683 gene-level TPM",
            "09_external_validation/gse224683/STEP09D_GSE224683_gene_level_TPM_v1.csv",
            "REANALYSIS_BACKUP"
        ),

        artifact(
            "09_EXTERNAL",
            "GSE224683 raw TPM source",
            "01_raw/gse224683/GSE224683_salmon_uci30_sczctrl_tpms.csv.gz",
            "REANALYSIS_BACKUP"
        ),

        # ------------------------------------------------------------------------------------------
        # BIOLOGY
        # ------------------------------------------------------------------------------------------

        artifact(
            "10_BIOLOGY",
            "All GO-BP tested terms",
            "10_biology/enrichment_v1/ora_v1/STEP10B2_GO_BP_all_tested_terms_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "10_BIOLOGY",
            "All Reactome tested pathways",
            "10_biology/enrichment_v1/ora_v1/STEP10B2_Reactome_all_tested_pathways_v1.csv",
            "MANUSCRIPT_SOURCE"
        ),

        artifact(
            "10_BIOLOGY",
            "Final integrated 24-feature biological matrix",
            "10_biology/final_integrated_synthesis_v1/STEP10G1_SCZ_MSS_24feature_integrated_biological_evidence_matrix_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 24L
        ),

        artifact(
            "10_BIOLOGY",
            "Weight-ranked biological matrix",
            "10_biology/final_integrated_synthesis_v1/STEP10G1_SCZ_MSS_24feature_weight_ranked_biological_matrix_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 24L
        ),

        artifact(
            "10_BIOLOGY",
            "Compact manuscript biology table",
            "10_biology/final_integrated_synthesis_v1/STEP10G1_SCZ_MSS_compact_manuscript_biological_table_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 24L
        ),

        # ------------------------------------------------------------------------------------------
        # CLAIM REGISTRY
        # ------------------------------------------------------------------------------------------

        artifact(
            "11_MANUSCRIPT",
            "Frozen manuscript claim registry",
            "11_manuscript/claim_registry_v1/STEP11A1_manuscript_claim_registry_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 24L
        ),

        artifact(
            "11_MANUSCRIPT",
            "Frozen key numeric results",
            "11_manuscript/claim_registry_v1/STEP11A1_key_numeric_results_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 67L
        ),

        artifact(
            "11_MANUSCRIPT",
            "Frozen transparency/limitations registry",
            "11_manuscript/claim_registry_v1/STEP11A1_transparency_and_limitations_registry_v1.csv",
            "MANUSCRIPT_SOURCE",
            expected_rows = 10L
        )
    )
)

CRITICAL$absolute_path <- file.path(
    PROJECT_ROOT,
    CRITICAL$relative_path
)

CRITICAL$exists <- file.exists(
    CRITICAL$absolute_path
)

critical_info <- file.info(
    CRITICAL$absolute_path
)

CRITICAL$bytes <- as.numeric(
    critical_info$size
)

CRITICAL$megabytes <- CRITICAL$bytes / 1024^2

CRITICAL$nonzero_file <- (
    CRITICAL$exists &
    !is.na(CRITICAL$bytes) &
    CRITICAL$bytes > 0
)

# ==================================================================================================
# 11B.0D — READABILITY / STRUCTURAL CHECKS
# ==================================================================================================

CRITICAL$readable <- FALSE
CRITICAL$observed_rows <- NA_integer_
CRITICAL$observed_cols <- NA_integer_
CRITICAL$read_error <- ""

for (i in seq_len(nrow(CRITICAL))) {

    p <- CRITICAL$absolute_path[i]

    if (!CRITICAL$nonzero_file[i]) {
        CRITICAL$read_error[i] <- "MISSING_OR_ZERO_BYTE"
        next
    }

    ext <- tolower(
        tools::file_ext(p)
    )

    result <- tryCatch(
        {

            if (ext == "csv") {

                # Only fully read files where row counts are explicitly expected.
                if (!is.na(CRITICAL$expected_rows[i])) {

                    z <- read.csv(
                        p,
                        stringsAsFactors = FALSE,
                        check.names = FALSE
                    )

                    CRITICAL$observed_rows[i] <- nrow(z)
                    CRITICAL$observed_cols[i] <- ncol(z)

                    rm(z)

                } else {

                    z <- read.csv(
                        p,
                        stringsAsFactors = FALSE,
                        check.names = FALSE,
                        nrows = 5L
                    )

                    CRITICAL$observed_cols[i] <- ncol(z)

                    rm(z)
                }

                TRUE

            } else if (ext == "rds") {

                if (isTRUE(CRITICAL$deep_check[i])) {

                    z <- readRDS(p)

                    d <- dim(z)

                    if (!is.null(d) && length(d) >= 2L) {
                        CRITICAL$observed_rows[i] <- d[1L]
                        CRITICAL$observed_cols[i] <- d[2L]
                    }

                    rm(z)
                    invisible(gc())
                }

                TRUE

            } else if (
                ext %in%
                c(
                    "txt",
                    "json",
                    "lock",
                    "sha256"
                )
            ) {

                readLines(
                    p,
                    n = 3L,
                    warn = FALSE
                )

                TRUE

            } else {

                # Binary/raw archives and RData are existence/size checked only.
                TRUE
            }
        },

        error = function(e) {

            CRITICAL$read_error[i] <<-
                conditionMessage(e)

            FALSE
        }
    )

    CRITICAL$readable[i] <- isTRUE(result)
}

# Dimension/row expectations
CRITICAL$shape_match <- TRUE

for (i in seq_len(nrow(CRITICAL))) {

    if (!is.na(CRITICAL$expected_rows[i])) {

        if (
            is.na(CRITICAL$observed_rows[i]) ||
            CRITICAL$observed_rows[i] !=
                CRITICAL$expected_rows[i]
        ) {
            CRITICAL$shape_match[i] <- FALSE
        }
    }

    if (!is.na(CRITICAL$expected_cols[i])) {

        if (
            is.na(CRITICAL$observed_cols[i]) ||
            CRITICAL$observed_cols[i] !=
                CRITICAL$expected_cols[i]
        ) {
            CRITICAL$shape_match[i] <- FALSE
        }
    }
}

CRITICAL$ready <- (
    CRITICAL$exists &
    CRITICAL$nonzero_file &
    CRITICAL$readable &
    CRITICAL$shape_match
)

# --------------------------------------------------------------------------------------------------
# Optional SHA256 for manageable files only
# --------------------------------------------------------------------------------------------------

CRITICAL$sha256 <- ""

if (
    requireNamespace(
        "digest",
        quietly = TRUE
    )
) {

    hash_limit <- 50 * 1024^2

    for (i in seq_len(nrow(CRITICAL))) {

        if (
            CRITICAL$ready[i] &&
            !is.na(CRITICAL$bytes[i]) &&
            CRITICAL$bytes[i] <= hash_limit
        ) {

            CRITICAL$sha256[i] <- tryCatch(
                digest::digest(
                    file = CRITICAL$absolute_path[i],
                    algo = "sha256",
                    serialize = FALSE
                ),
                error = function(e) {
                    paste0(
                        "HASH_ERROR: ",
                        conditionMessage(e)
                    )
                }
            )

        } else if (
            CRITICAL$ready[i] &&
            !is.na(CRITICAL$bytes[i]) &&
            CRITICAL$bytes[i] > hash_limit
        ) {

            CRITICAL$sha256[i] <-
                "SKIPPED_FILE_GT_50MB"
        }
    }
}

write.csv(
    CRITICAL,
    CRITICAL_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# ==================================================================================================
# 11B.0E — EXPENSIVE CHECKPOINT COUNT AUDIT
# ==================================================================================================

count_files <- function(
    directory,
    pattern,
    recursive = FALSE
) {

    if (!dir.exists(directory)) {
        return(0L)
    }

    length(
        list.files(
            directory,
            pattern = pattern,
            recursive = recursive,
            full.names = TRUE
        )
    )
}

NESTED_DIR <- file.path(
    PROJECT_ROOT,
    "06_models",
    "nested_cv_5x5_v1"
)

HALF_FIT_DIR <- file.path(
    PROJECT_ROOT,
    "07_stability",
    "complementary_half_sample_v1",
    "fits"
)

PERM_DIR <- file.path(
    PROJECT_ROOT,
    "07_stability",
    "permutation_null_v1",
    "permutations"
)

A_FIT_DIR <- file.path(
    PROJECT_ROOT,
    "06_models",
    "baselines",
    "model_A_DLPFC_only_v1",
    "outer_fits"
)

B_FIT_DIR <- file.path(
    PROJECT_ROOT,
    "06_models",
    "baselines",
    "model_B_HIPPO_only_v1",
    "outer_fits"
)

CE_FIT_DIR <- file.path(
    PROJECT_ROOT,
    "06_models",
    "baselines",
    "model_CE_pooled_shared_only_v1",
    "outer_fits"
)

CHECKPOINT_AUDIT <- data.frame(

    checkpoint_group = c(
        rep("PRIMARY_NESTED_CV", 7),
        "BASELINE_A_FITS",
        "BASELINE_B_FITS",
        "BASELINE_CE_FITS",
        "HALF_SAMPLE_FITS",
        "PERMUTATION_FITS"
    ),

    artifact = c(
        "PASS.txt",
        "fold_summary.csv",
        "inner_grid.csv",
        "sample_predictions.csv",
        "donor_predictions.csv",
        "gene_coefficients.rds",
        "inner_donor_fold_map.csv",
        "repeat_fold_RDS",
        "repeat_fold_RDS",
        "repeat_fold_RDS",
        "fit_A_or_B_RDS",
        "permutation_RDS"
    ),

    expected_count = c(
        rep(25L, 7),
        25L,
        25L,
        25L,
        200L,
        100L
    ),

    observed_count = c(
        count_files(
            NESTED_DIR,
            "^PASS\\.txt$",
            recursive = TRUE
        ),
        count_files(
            NESTED_DIR,
            "^fold_summary\\.csv$",
            recursive = TRUE
        ),
        count_files(
            NESTED_DIR,
            "^inner_grid\\.csv$",
            recursive = TRUE
        ),
        count_files(
            NESTED_DIR,
            "^sample_predictions\\.csv$",
            recursive = TRUE
        ),
        count_files(
            NESTED_DIR,
            "^donor_predictions\\.csv$",
            recursive = TRUE
        ),
        count_files(
            NESTED_DIR,
            "^gene_coefficients\\.rds$",
            recursive = TRUE
        ),
        count_files(
            NESTED_DIR,
            "^inner_donor_fold_map\\.csv$",
            recursive = TRUE
        ),
        count_files(
            A_FIT_DIR,
            "^repeat_[0-9]+_fold_[0-9]+\\.rds$"
        ),
        count_files(
            B_FIT_DIR,
            "^repeat_[0-9]+_fold_[0-9]+\\.rds$"
        ),
        count_files(
            CE_FIT_DIR,
            "^repeat_[0-9]+_fold_[0-9]+\\.rds$"
        ),
        count_files(
            HALF_FIT_DIR,
            "^fit_[0-9]{3}_[AB]\\.rds$"
        ),
        count_files(
            PERM_DIR,
            "^perm_[0-9]{3}\\.rds$"
        )
    ),

    stringsAsFactors = FALSE
)

CHECKPOINT_AUDIT$pass <- (
    CHECKPOINT_AUDIT$observed_count ==
    CHECKPOINT_AUDIT$expected_count
)

write.csv(
    CHECKPOINT_AUDIT,
    CHECKPOINT_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 11B.0F — NOTEBOOK SOURCE INVENTORY
# ==================================================================================================

NOTEBOOK_DIR <- file.path(
    PROJECT_ROOT,
    "notebooks"
)

notebook_files <- character(0)

if (dir.exists(NOTEBOOK_DIR)) {

    notebook_files <- list.files(
        NOTEBOOK_DIR,
        recursive = TRUE,
        full.names = TRUE,
        all.files = FALSE
    )

    notebook_files <- notebook_files[
        file.exists(notebook_files) &
        !dir.exists(notebook_files)
    ]
}

NOTEBOOK_INVENTORY <- data.frame(
    path = notebook_files,
    extension = tolower(
        tools::file_ext(
            notebook_files
        )
    ),
    bytes = if (length(notebook_files) > 0L) {
        as.numeric(
            file.info(
                notebook_files
            )$size
        )
    } else {
        numeric(0)
    },
    modified_time = if (length(notebook_files) > 0L) {
        as.character(
            file.info(
                notebook_files
            )$mtime
        )
    } else {
        character(0)
    },
    stringsAsFactors = FALSE
)

write.csv(
    NOTEBOOK_INVENTORY,
    NOTEBOOK_PATH,
    row.names = FALSE,
    quote = TRUE
)

n_ipynb <- sum(
    NOTEBOOK_INVENTORY$extension ==
        "ipynb"
)

n_html_notebook <- sum(
    NOTEBOOK_INVENTORY$extension ==
        "html"
)

# ==================================================================================================
# 11B.0G — FINAL ENVIRONMENT CAPTURE
# ==================================================================================================

writeLines(
    capture.output(
        sessionInfo()
    ),
    FINAL_SESSION_PATH,
    useBytes = TRUE
)

renv_status_text <- character(0)

if (
    requireNamespace(
        "renv",
        quietly = TRUE
    )
) {

    renv_status_text <- capture.output(
        renv::status(
            project = PROJECT_ROOT
        )
    )

} else {

    renv_status_text <- c(
        "renv package is not available in the current R session.",
        "Final renv synchronization could not be audited."
    )
}

writeLines(
    renv_status_text,
    RENV_STATUS_PATH,
    useBytes = TRUE
)

renv_bad_pattern <- paste0(
    "out.of.sync|",
    "not recorded|",
    "not installed|",
    "inconsistent|",
    "issues? found|",
    "library and lockfile are out"
)

renv_has_warning <- any(
    grepl(
        renv_bad_pattern,
        renv_status_text,
        ignore.case = TRUE,
        perl = TRUE
    )
)

# ==================================================================================================
# 11B.0H — FIGURE / TABLE SOURCE READINESS MAP
# ==================================================================================================

all_ready <- function(relative_paths) {

    z <- CRITICAL$ready[
        match(
            relative_paths,
            CRITICAL$relative_path
        )
    ]

    if (
        length(z) !=
            length(relative_paths) ||
        anyNA(z)
    ) {
        return(FALSE)
    }

    all(z)
}

DISPLAY_MAP <- data.frame(

    display_domain = c(

        "Preprocessing/QC",
        "Discovery cohort table",
        "Differential-expression figure/table",
        "Internal CV performance figure/table",
        "Model-comparison figure/table",
        "Stability/frozen-state figure/table",
        "Discovery score descriptive supplement",
        "External technical-gate figure/table",
        "GSE224683 external-validation figure/table",
        "Biological interpretation figure/table",
        "Manuscript numeric/claim source"
    ),

    source_files = c(

        paste(
            "03_intermediate/brainseq_primary_logCPM_TMM_QC_v1.rds",
            "02_metadata/step02b_sample_PCA_QC_v1.csv",
            "02_metadata/step02b_PCA_variance_v1.csv",
            "02_metadata/step02b_TMM_normalization_summary_v1.csv",
            sep = ";"
        ),

        paste(
            "04_processed/brainseq_discovery_cohort_samples_v1.csv",
            "04_processed/brainseq_discovery_cohort_donors_v1.csv",
            "04_processed/brainseq_model_metadata_v1.csv",
            sep = ";"
        ),

        paste(
            "05_differential_expression/step03a_DLPFC_SCZ_vs_Control_v1.csv",
            "05_differential_expression/step03a_HIPPO_SCZ_vs_Control_v1.csv",
            "05_differential_expression/step03a_Shared_SCZ_effect_v1.csv",
            "05_differential_expression/step03a_Regional_Deviation_v1.csv",
            sep = ";"
        ),

        paste(
            "06_models/oof_freeze_v1/STEP05D_CANONICAL_donor_OOF_predictions_v1.csv",
            "06_models/oof_freeze_v1/STEP05D_CANONICAL_sample_OOF_predictions_v1.csv",
            "06_models/nested_cv_5x5_v1/ALL_25_OUTER_FOLD_SUMMARY_v1.csv",
            sep = ";"
        ),

        paste(
            "06_models/model_comparison_v1/STEP05C_model_performance_summary_v1.csv",
            "06_models/model_comparison_v1/STEP05C_paired_comparison_summary_v1.csv",
            "06_models/model_comparison_v1/STEP05C_paired_outer_fold_differences_v1.csv",
            sep = ";"
        ),

        paste(
            "06_models/stability_v1/step05a_shared_selection_frequency_distribution_v1.csv",
            "07_stability/complementary_half_sample_v1/step06b_half_sample_stability_summary_v1.csv",
            "07_stability/permutation_null_v1/step06c_permutation_null_summary_v1.csv",
            "07_stability/final_stable_state_v1/step06d_FINAL_stable_shared_state_v1.csv",
            "08_score_lock/SCZ_MSS_v1_gene_weights.csv",
            sep = ";"
        ),

        paste(
            "08_score_lock/step07b_BrainSeq_SCZ_MSS_v1_sample_scores.csv",
            "08_score_lock/step07c_paired_donor_SCZ_MSS_v1_scores.csv",
            sep = ";"
        ),

        paste(
            "09_external_validation/master_registry_v1/SCZ_MSS_v1_external_technical_gate_registry_v1.csv",
            "09_external_validation/master_registry_v1/SCZ_MSS_v1_FINAL_external_validation_summary_table_v1.csv",
            sep = ";"
        ),

        paste(
            "09_external_validation/gse224683/STEP09D_GSE224683_sample_SCZ_MSS_v1.csv",
            "09_external_validation/gse224683/STEP09D_GSE224683_external_validation_summary_v1.csv",
            "09_external_validation/gse224683/STEP09D_GSE224683_bootstrap_v1.csv",
            sep = ";"
        ),

        paste(
            "10_biology/enrichment_v1/ora_v1/STEP10B2_GO_BP_all_tested_terms_v1.csv",
            "10_biology/enrichment_v1/ora_v1/STEP10B2_Reactome_all_tested_pathways_v1.csv",
            "10_biology/final_integrated_synthesis_v1/STEP10G1_SCZ_MSS_24feature_integrated_biological_evidence_matrix_v1.csv",
            sep = ";"
        ),

        paste(
            "11_manuscript/claim_registry_v1/STEP11A1_key_numeric_results_v1.csv",
            "11_manuscript/claim_registry_v1/STEP11A1_manuscript_claim_registry_v1.csv",
            "11_manuscript/claim_registry_v1/STEP11A1_transparency_and_limitations_registry_v1.csv",
            sep = ";"
        )
    ),

    stringsAsFactors = FALSE
)

DISPLAY_MAP$ready <- vapply(
    strsplit(
        DISPLAY_MAP$source_files,
        ";",
        fixed = TRUE
    ),
    all_ready,
    FUN.VALUE = logical(1)
)

DISPLAY_MAP$status <- ifelse(
    DISPLAY_MAP$ready,
    "READY_FROM_SAVED_ARTIFACTS",
    "NOT_READY_CHECK_MISSING_SOURCE"
)

write.csv(
    DISPLAY_MAP,
    DISPLAY_MAP_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 11B.0I — MISSING / UNREADY ARTIFACTS
# ==================================================================================================

MISSING <- CRITICAL[
    !CRITICAL$ready,
    ,
    drop = FALSE
]

write.csv(
    MISSING,
    MISSING_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# ==================================================================================================
# 11B.0J — FINAL READINESS DECISIONS
# ==================================================================================================

manuscript_rows <- (
    CRITICAL$requirement ==
        "MANUSCRIPT_SOURCE"
)

reanalyze_rows <- (
    CRITICAL$requirement %in%
        c(
            "MANUSCRIPT_SOURCE",
            "REANALYSIS_BACKUP"
        )
)

repro_rows <- (
    CRITICAL$requirement ==
        "REPRODUCIBILITY"
)

MANUSCRIPT_DISPLAY_READY <- all(
    CRITICAL$ready[
        manuscript_rows
    ]
) &&
    all(
        DISPLAY_MAP$ready
    )

EXPENSIVE_CHECKPOINTS_READY <- all(
    CHECKPOINT_AUDIT$pass
)

REANALYSIS_ARTIFACTS_READY <- all(
    CRITICAL$ready[
        reanalyze_rows
    ]
) &&
    EXPENSIVE_CHECKPOINTS_READY

REPRODUCIBILITY_FILES_READY <- all(
    CRITICAL$ready[
        repro_rows
    ]
)

NOTEBOOK_SOURCE_READY <- (
    n_ipynb >= 1L
)

overall_status <- if (
    MANUSCRIPT_DISPLAY_READY &&
    EXPENSIVE_CHECKPOINTS_READY &&
    REANALYSIS_ARTIFACTS_READY &&
    REPRODUCIBILITY_FILES_READY &&
    NOTEBOOK_SOURCE_READY &&
    !renv_has_warning
) {

    "FULL_PASS_PUBLICATION_AND_REANALYSIS_READY"

} else if (
    MANUSCRIPT_DISPLAY_READY &&
    EXPENSIVE_CHECKPOINTS_READY
) {

    "MANUSCRIPT_READY_WITH_REPRODUCIBILITY_WARNINGS"

} else {

    "ACTION_REQUIRED_BEFORE_FIGURE_TABLE_GENERATION"
}

# ==================================================================================================
# 11B.0K — SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11B.0 — PUBLICATION ARTIFACT + REANALYSIS READINESS AUDIT",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "PROJECT INVENTORY",

    paste0(
        "Files inventoried: ",
        nrow(PROJECT_INVENTORY)
    ),

    paste0(
        "Total inventoried size: ",
        sprintf(
            "%.3f GB",
            project_total_gb
        )
    ),

    paste0(
        "Zero-byte files: ",
        n_zero_byte
    ),

    "",

    "CRITICAL ARTIFACTS",

    paste0(
        "Critical artifacts audited: ",
        nrow(CRITICAL)
    ),

    paste0(
        "Critical artifacts ready: ",
        sum(CRITICAL$ready),
        "/",
        nrow(CRITICAL)
    ),

    paste0(
        "Missing/unready critical artifacts: ",
        nrow(MISSING)
    ),

    "",

    "EXPENSIVE CHECKPOINTS",

    paste0(
        "Checkpoint groups passing: ",
        sum(CHECKPOINT_AUDIT$pass),
        "/",
        nrow(CHECKPOINT_AUDIT)
    ),

    paste0(
        "All primary 25-fold nested-CV checkpoint sets complete: ",
        all(
            CHECKPOINT_AUDIT$pass[
                CHECKPOINT_AUDIT$checkpoint_group ==
                    "PRIMARY_NESTED_CV"
            ]
        )
    ),

    paste0(
        "Baseline A/B/CE outer-fit checkpoints complete: ",
        all(
            CHECKPOINT_AUDIT$pass[
                CHECKPOINT_AUDIT$checkpoint_group %in%
                    c(
                        "BASELINE_A_FITS",
                        "BASELINE_B_FITS",
                        "BASELINE_CE_FITS"
                    )
            ]
        )
    ),

    paste0(
        "Half-sample 200-fit checkpoint set complete: ",
        CHECKPOINT_AUDIT$pass[
            CHECKPOINT_AUDIT$checkpoint_group ==
                "HALF_SAMPLE_FITS"
        ]
    ),

    paste0(
        "Permutation 100-fit checkpoint set complete: ",
        CHECKPOINT_AUDIT$pass[
            CHECKPOINT_AUDIT$checkpoint_group ==
                "PERMUTATION_FITS"
        ]
    ),

    "",

    "FIGURE / TABLE SOURCE READINESS",

    paste0(
        "Display domains ready directly from saved artifacts: ",
        sum(DISPLAY_MAP$ready),
        "/",
        nrow(DISPLAY_MAP)
    ),

    paste0(
        "Manuscript display ready without expensive rerun: ",
        MANUSCRIPT_DISPLAY_READY
    ),

    "",

    "REPRODUCIBILITY",

    paste0(
        "Project-local .ipynb files found: ",
        n_ipynb
    ),

    paste0(
        "Project-local notebook HTML files found: ",
        n_html_notebook
    ),

    paste0(
        "Reproducibility core files ready: ",
        REPRODUCIBILITY_FILES_READY
    ),

    paste0(
        "renv status warning detected: ",
        renv_has_warning
    ),

    "",

    "FINAL DECISIONS",

    paste0(
        "MANUSCRIPT_DISPLAY_READY = ",
        MANUSCRIPT_DISPLAY_READY
    ),

    paste0(
        "EXPENSIVE_CHECKPOINTS_READY = ",
        EXPENSIVE_CHECKPOINTS_READY
    ),

    paste0(
        "REANALYSIS_ARTIFACTS_READY = ",
        REANALYSIS_ARTIFACTS_READY
    ),

    paste0(
        "NOTEBOOK_SOURCE_READY = ",
        NOTEBOOK_SOURCE_READY
    ),

    paste0(
        "OVERALL_STATUS = ",
        overall_status
    ),

    "",

    "INTERPRETATION",

    if (MANUSCRIPT_DISPLAY_READY) {
        paste(
            "Current manuscript figures and tables can be generated from saved artifacts",
            "without rerunning completed expensive analyses."
        )
    } else {
        paste(
            "One or more manuscript display sources are missing or structurally invalid;",
            "inspect STEP11B0_missing_or_unready_artifacts_v1.csv before figure generation."
        )
    },

    if (!NOTEBOOK_SOURCE_READY) {
        paste(
            "WARNING: no project-local .ipynb notebook was detected.",
            "Save the working notebook into E:/SCZ_CROSSREGION_MSS_Q1/notebooks before submission."
        )
    } else {
        "Project-local .ipynb notebook source detected."
    },

    if (renv_has_warning) {
        paste(
            "WARNING: final renv status appears to contain synchronization issues.",
            "Do NOT change the lockfile in this audit; review the saved renv-status output first."
        )
    } else {
        paste(
            "No obvious renv synchronization warning was detected in the captured status output."
        )
    },

    "",

    "IMPORTANT",

    paste(
        "This audit verifies saved sources for future displays and many follow-up analyses.",
        "A genuinely new reviewer-requested model specification may still require refitting,",
        "but existing preprocessing, DE, CV predictions, coefficients, stability matrices,",
        "permutation results, external-validation results and biological summaries should not",
        "need to be recomputed when their saved artifacts pass this audit."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 11B.0L — MANIFEST OF AUDIT OUTPUTS
# ==================================================================================================

audit_outputs <- c(
    INVENTORY_PATH,
    CRITICAL_PATH,
    CHECKPOINT_PATH,
    DISPLAY_MAP_PATH,
    NOTEBOOK_PATH,
    RENV_STATUS_PATH,
    FINAL_SESSION_PATH,
    MISSING_PATH,
    SUMMARY_PATH
)

audit_manifest <- data.frame(
    file = basename(audit_outputs),
    path = audit_outputs,
    bytes = as.numeric(
        file.info(
            audit_outputs
        )$size
    ),
    sha256 = "",
    stringsAsFactors = FALSE
)

if (
    requireNamespace(
        "digest",
        quietly = TRUE
    )
) {

    audit_manifest$sha256 <- vapply(
        audit_outputs,
        function(p) {
            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    )
}

write.csv(
    audit_manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 11B.0M — CONSOLE REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nPUBLICATION / REANALYSIS READINESS SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Critical artifacts ready: ",
    sum(CRITICAL$ready),
    "/",
    nrow(CRITICAL),
    "\n",
    sep = ""
)

cat(
    "Expensive checkpoint groups passing: ",
    sum(CHECKPOINT_AUDIT$pass),
    "/",
    nrow(CHECKPOINT_AUDIT),
    "\n",
    sep = ""
)

cat(
    "Figure/table source domains ready: ",
    sum(DISPLAY_MAP$ready),
    "/",
    nrow(DISPLAY_MAP),
    "\n",
    sep = ""
)

cat(
    "\nMANUSCRIPT_DISPLAY_READY: ",
    MANUSCRIPT_DISPLAY_READY,
    "\n",
    sep = ""
)

cat(
    "EXPENSIVE_CHECKPOINTS_READY: ",
    EXPENSIVE_CHECKPOINTS_READY,
    "\n",
    sep = ""
)

cat(
    "REANALYSIS_ARTIFACTS_READY: ",
    REANALYSIS_ARTIFACTS_READY,
    "\n",
    sep = ""
)

cat(
    "PROJECT-LOCAL IPYNB COUNT: ",
    n_ipynb,
    "\n",
    sep = ""
)

cat(
    "RENV WARNING DETECTED: ",
    renv_has_warning,
    "\n",
    sep = ""
)

cat(
    "\nOVERALL STATUS: ",
    overall_status,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nEXPENSIVE CHECKPOINT AUDIT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    CHECKPOINT_AUDIT,
    row.names = FALSE
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nFIGURE / TABLE SOURCE READINESS\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    DISPLAY_MAP[
        ,
        c(
            "display_domain",
            "status"
        )
    ],
    row.names = FALSE
)

if (nrow(MISSING) > 0L) {

    cat(
        "\n",
        paste(rep("-", 110), collapse = ""),
        "\nMISSING / UNREADY CRITICAL ARTIFACTS\n",
        paste(rep("-", 110), collapse = ""),
        "\n",
        sep = ""
    )

    print(
        MISSING[
            ,
            c(
                "stage",
                "role",
                "relative_path",
                "requirement",
                "exists",
                "bytes",
                "readable",
                "shape_match",
                "read_error"
            )
        ],
        row.names = FALSE
    )

} else {

    cat(
        "\nMissing/unready critical artifacts: NONE\n"
    )
}

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nFINAL RENV STATUS — FIRST 30 LINES\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    paste(
        head(
            renv_status_text,
            30L
        ),
        collapse = "\n"
    ),
    "\n"
)

cat(
    "\nCritical-artifact audit:\n",
    CRITICAL_PATH,
    "\n",
    sep = ""
)

cat(
    "\nCheckpoint audit:\n",
    CHECKPOINT_PATH,
    "\n",
    sep = ""
)

cat(
    "\nFigure/table source map:\n",
    DISPLAY_MAP_PATH,
    "\n",
    sep = ""
)

cat(
    "\nMissing/unready artifacts:\n",
    MISSING_PATH,
    "\n",
    sep = ""
)

cat(
    "\nSummary:\n",
    SUMMARY_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11B.0 COMPLETE — ARTIFACT READINESS AUDIT FINISHED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11B.0 — PUBLICATION ARTIFACT + REANALYSIS READINESS AUDIT
Project sentinel: PASS
Project files inventoried: 2378
Total inventoried project size: 3.680 GB
Zero-byte files detected: 1

PUBLICATION / REANALYSIS READINESS SUMMARY
Critical artifacts ready: 65/65
Expensive checkpoint groups passing: 12/12
Figure/table source domains ready: 11/11

MANUSCRIPT_DISPLAY_READY: TRUE
EXPENSIVE_CHECKPOINTS_READY: TRUE
REANALYSIS_ARTIFACTS_READY: TRUE
PROJECT-LOCAL IPYNB COUNT: 0
RENV WARNING DETECTED: TRUE

OVERALL STATUS: MANUSCRIPT_READY_WITH_REPRODUCIBILITY_WARNINGS

--------------------------------------------------------------------------------------------------------------
EXPENSIVE CHECKPOINT AUDIT
--------------------------------------------------------------------------------------------------------------
  checkpoint_group                 artifact expected_count observed_count pass
 PRIMARY_NESTED_CV                 PASS.txt             25             25 TRUE
 PRIMARY_NESTED_CV      

In [78]:
# ==================================================================================================
# STEP 11B.1R — CORRECTED FINAL MANUSCRIPT DISPLAY ARCHITECTURE
#               + ARTIFACT-READINESS RECONCILIATION
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   1. Correct the literal-string validation bug in the previous STEP11B.1.
#   2. Reconcile STEP11B.0 readiness results.
#   3. Correct the false-positive renv warning logic.
#   4. Lock a NON-REDUNDANT manuscript figure/table architecture beginning at preprocessing.
#
# PRINCIPLE
#   FIGURE = patterns, distributions, relationships, model behaviour, workflow.
#   TABLE  = exact numerical reporting and detailed structured information.
#
# IMPORTANT
#   - No analysis is rerun.
#   - No model is refit.
#   - No DE is recomputed.
#   - No permutation or stability analysis is rerun.
#   - No external score is recomputed.
#   - Existing null/counter-evidence remains visible.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11B.1R — CORRECTED FINAL MANUSCRIPT DISPLAY ARCHITECTURE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 11B.1R-A — PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID,
        "\nObserved: ", project_id
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 11B.1R-B — INPUT PATHS
# ==================================================================================================

CLAIM_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "claim_registry_v1"
)

READINESS_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "artifact_readiness_v1"
)

CLAIM_PATH <- file.path(
    CLAIM_DIR,
    "STEP11A1_manuscript_claim_registry_v1.csv"
)

NUMBER_PATH <- file.path(
    CLAIM_DIR,
    "STEP11A1_key_numeric_results_v1.csv"
)

LIMIT_PATH <- file.path(
    CLAIM_DIR,
    "STEP11A1_transparency_and_limitations_registry_v1.csv"
)

CRITICAL_PATH <- file.path(
    READINESS_DIR,
    "STEP11B0_critical_artifact_readiness_v1.csv"
)

CHECKPOINT_PATH <- file.path(
    READINESS_DIR,
    "STEP11B0_expensive_checkpoint_count_audit_v1.csv"
)

DISPLAY_READY_PATH <- file.path(
    READINESS_DIR,
    "STEP11B0_figure_table_source_readiness_map_v1.csv"
)

NOTEBOOK_INVENTORY_PATH <- file.path(
    READINESS_DIR,
    "STEP11B0_notebook_source_inventory_v1.csv"
)

RENV_STATUS_PATH <- file.path(
    READINESS_DIR,
    "STEP11B0_final_renv_status_v1.txt"
)

PROJECT_INVENTORY_PATH <- file.path(
    READINESS_DIR,
    "STEP11B0_full_project_file_inventory_v1.csv"
)

required_inputs <- c(
    CLAIM_PATH,
    NUMBER_PATH,
    LIMIT_PATH,
    CRITICAL_PATH,
    CHECKPOINT_PATH,
    DISPLAY_READY_PATH,
    NOTEBOOK_INVENTORY_PATH,
    RENV_STATUS_PATH,
    PROJECT_INVENTORY_PATH
)

if (!all(file.exists(required_inputs))) {

    stop(
        "Required STEP11A.1 / STEP11B.0 artifact(s) missing:\n",
        paste(
            required_inputs[
                !file.exists(required_inputs)
            ],
            collapse = "\n"
        )
    )
}

# ==================================================================================================
# 11B.1R-C — LOAD GOVERNANCE + READINESS
# ==================================================================================================

claims <- read.csv(
    CLAIM_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

numbers <- read.csv(
    NUMBER_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

limitations <- read.csv(
    LIMIT_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

critical <- read.csv(
    CRITICAL_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

checkpoints <- read.csv(
    CHECKPOINT_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

display_ready <- read.csv(
    DISPLAY_READY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

notebook_inventory <- read.csv(
    NOTEBOOK_INVENTORY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

project_inventory <- read.csv(
    PROJECT_INVENTORY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (
    nrow(claims) != 24L ||
    nrow(numbers) != 67L ||
    nrow(limitations) != 10L
) {
    stop(
        "STEP11A.1 frozen artifact dimensions changed."
    )
}

cat("STEP11A.1 claim registry: PASS\n")
cat("Frozen key numbers: 67 — PASS\n")
cat("Frozen allowed claims: 24 — PASS\n")
cat("Frozen limitations: 10 — PASS\n")

# ==================================================================================================
# 11B.1R-D — ROBUST LOGICAL NORMALIZATION
# ==================================================================================================

as_bool <- function(x) {

    if (is.logical(x)) {
        return(x)
    }

    z <- toupper(
        trimws(
            as.character(x)
        )
    )

    out <- rep(
        NA,
        length(z)
    )

    out[
        z %in% c(
            "TRUE",
            "T",
            "1"
        )
    ] <- TRUE

    out[
        z %in% c(
            "FALSE",
            "F",
            "0"
        )
    ] <- FALSE

    out
}

critical$ready <- as_bool(
    critical$ready
)

checkpoints$pass <- as_bool(
    checkpoints$pass
)

display_ready$ready <- as_bool(
    display_ready$ready
)

if (
    anyNA(critical$ready) ||
    anyNA(checkpoints$pass) ||
    anyNA(display_ready$ready)
) {
    stop(
        "Could not normalize one or more STEP11B.0 readiness fields."
    )
}

# ==================================================================================================
# 11B.1R-E — RECONCILE STEP11B.0
# ==================================================================================================

CRITICAL_READY <- all(
    critical$ready
)

CHECKPOINTS_READY <- all(
    checkpoints$pass
)

DISPLAY_SOURCES_READY <- all(
    display_ready$ready
)

cat(
    "Critical artifacts: ",
    sum(critical$ready),
    "/",
    nrow(critical),
    " ready\n",
    sep = ""
)

cat(
    "Expensive checkpoint groups: ",
    sum(checkpoints$pass),
    "/",
    nrow(checkpoints),
    " PASS\n",
    sep = ""
)

cat(
    "Figure/table source domains: ",
    sum(display_ready$ready),
    "/",
    nrow(display_ready),
    " ready\n",
    sep = ""
)

if (!CRITICAL_READY) {
    stop(
        "Critical artifact audit is not fully ready."
    )
}

if (!CHECKPOINTS_READY) {
    stop(
        "Expensive checkpoint audit is not fully ready."
    )
}

if (!DISPLAY_SOURCES_READY) {
    stop(
        "Figure/table source audit is not fully ready."
    )
}

cat("STEP11B.0 substantive artifact-readiness audit: PASS\n")

# ==================================================================================================
# 11B.1R-F — CORRECT RENV STATUS LOGIC
#
# Previous false positive:
#   regex matched 'issues found' inside 'No issues found'.
# ==================================================================================================

renv_status_text <- readLines(
    RENV_STATUS_PATH,
    warn = FALSE
)

renv_clean <- any(
    grepl(
        "^\\s*No issues found\\s*--\\s*the project is in a consistent state\\.?\\s*$",
        renv_status_text,
        ignore.case = TRUE,
        perl = TRUE
    )
)

if (renv_clean) {

    RENV_STATUS <- "CONSISTENT_NO_ISSUES"

} else {

    issue_regex <- paste0(
        "out of sync|",
        "not installed|",
        "not recorded|",
        "inconsistent|",
        "dependency issues|",
        "package.*missing|",
        "lockfile.*different"
    )

    actual_issue <- any(
        grepl(
            issue_regex,
            renv_status_text,
            ignore.case = TRUE,
            perl = TRUE
        )
    )

    RENV_STATUS <- if (
        actual_issue
    ) {
        "WARNING_REVIEW_REQUIRED"
    } else {
        "STATUS_NOT_EXPLICITLY_CLEAN_REVIEW_MANUALLY"
    }
}

cat(
    "Corrected renv interpretation: ",
    RENV_STATUS,
    "\n",
    sep = ""
)

# ==================================================================================================
# 11B.1R-G — NOTEBOOK SOURCE STATUS
# ==================================================================================================

n_ipynb <- if (
    nrow(notebook_inventory) > 0L &&
    "extension" %in% names(notebook_inventory)
) {

    sum(
        tolower(
            notebook_inventory$extension
        ) == "ipynb"
    )

} else {

    0L
}

NOTEBOOK_SOURCE_STATUS <- if (
    n_ipynb >= 1L
) {
    "PASS_PROJECT_LOCAL_IPYNB_PRESENT"
} else {
    "ACTION_REQUIRED_SAVE_CURRENT_NOTEBOOK_AS_IPYNB"
}

cat(
    "Project-local .ipynb count: ",
    n_ipynb,
    "\n",
    sep = ""
)

cat(
    "Notebook source status: ",
    NOTEBOOK_SOURCE_STATUS,
    "\n",
    sep = ""
)

# ==================================================================================================
# 11B.1R-H — ZERO-BYTE NONCRITICAL FILE AUDIT
# ==================================================================================================

zero_byte_rows <- project_inventory[
    !is.na(project_inventory$zero_byte) &
    as_bool(project_inventory$zero_byte) %in% TRUE,
    ,
    drop = FALSE
]

if (nrow(zero_byte_rows) > 0L) {

    zero_critical <- zero_byte_rows$relative_path %in%
        critical$relative_path

    zero_byte_rows$critical_artifact <- zero_critical

} else {

    zero_byte_rows$critical_artifact <- logical(0)
}

if (
    nrow(zero_byte_rows) > 0L &&
    any(zero_byte_rows$critical_artifact)
) {

    stop(
        "A zero-byte file overlaps a critical manuscript/reanalysis artifact."
    )
}

cat(
    "Zero-byte project files: ",
    nrow(zero_byte_rows),
    " — none are critical\n",
    sep = ""
)

# ==================================================================================================
# 11B.1R-I — OUTPUT DIRECTORY
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "display_architecture_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

MAIN_FIG_PATH <- file.path(
    OUT_DIR,
    "STEP11B1R_main_figure_architecture_v1.csv"
)

MAIN_TABLE_PATH <- file.path(
    OUT_DIR,
    "STEP11B1R_main_table_architecture_v1.csv"
)

SUPP_FIG_PATH <- file.path(
    OUT_DIR,
    "STEP11B1R_supplementary_figure_architecture_v1.csv"
)

SUPP_TABLE_PATH <- file.path(
    OUT_DIR,
    "STEP11B1R_supplementary_table_architecture_v1.csv"
)

RESULT_SEQUENCE_PATH <- file.path(
    OUT_DIR,
    "STEP11B1R_results_section_sequence_v1.csv"
)

ZERO_BYTE_PATH <- file.path(
    OUT_DIR,
    "STEP11B1R_noncritical_zero_byte_file_audit_v1.csv"
)

NOTEBOOK_ACTION_PATH <- file.path(
    OUT_DIR,
    "STEP11B1R_notebook_source_action_v1.txt"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "STEP11B1R_display_architecture_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "STEP11B1R_display_architecture_manifest_v1.csv"
)

write.csv(
    zero_byte_rows,
    ZERO_BYTE_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# ==================================================================================================
# 11B.1R-J — MAIN FIGURES
#
# NON-REDUNDANCY:
#   Figures emphasize visual patterns/relationships.
#   Exact numerical detail belongs in tables.
# ==================================================================================================

main_figures <- data.frame(

    figure_id = paste0(
        "Figure ",
        1:5
    ),

    working_title = c(

        "Discovery workflow, preprocessing, quality control, and donor structure",

        "Shared and region-dependent schizophrenia transcriptomic architecture",

        "Leakage-free internal model performance and contribution of regional deviation",

        "Stability selection and the frozen 24-feature shared transcriptional state",

        "External technical transferability and frozen-score validation"
    ),

    scientific_role = c(

        paste(
            "Show how raw BrainSeq data became the 755-sample / 454-donor analysis cohort,",
            "demonstrate preprocessing/QC, and establish paired-donor structure."
        ),

        paste(
            "Show regional concordance, the shared diagnosis component, and the comparatively",
            "small but non-zero region-dependent component."
        ),

        paste(
            "Show unbiased donor-grouped prediction behaviour, fold-level model comparison,",
            "and the small incremental contribution of deviation terms."
        ),

        paste(
            "Show stability convergence, permutation context, and the composition/direction",
            "of the final frozen 24-feature state."
        ),

        paste(
            "Show why four public cohorts were technically non-evaluable and display",
            "the null result in the one technically eligible cohort."
        )
    ),

    panels = c(

        paste(
            "A: end-to-end discovery workflow;",
            "B: sample/gene preprocessing attrition flow;",
            "C: TMM/PCA QC visualization;",
            "D: paired vs unpaired donor architecture"
        ),

        paste(
            "A: DLPFC vs HIPPO logFC scatter;",
            "B: regional/shared/deviation FDR architecture;",
            "C: shared-effect distribution;",
            "D: deviation-effect distribution"
        ),

        paste(
            "A: donor-level repeated OOF ROC;",
            "B: outer-fold AUROC distributions across predefined models;",
            "C: paired F minus shared-only fold deltas;",
            "D: shared vs deviation sparsity across outer fits"
        ),

        paste(
            "A: outer-CV shared selection-frequency distribution;",
            "B: outer-stable to half-sample-stable convergence;",
            "C: permutation null versus observed reference;",
            "D: 24-feature frozen weight/biotype plot"
        ),

        paste(
            "A: five-cohort technical-gate flow;",
            "B: gene/absolute/positive/negative weight coverage by cohort;",
            "C: GSE224683 frozen-score distribution by diagnosis;",
            "D: GSE224683 ROC with CI"
        )
    ),

    primary_source_rule = c(

        paste(
            "Use saved preprocessing/QC artifacts only;",
            "no preprocessing rerun."
        ),

        paste(
            "Use saved full DE result tables;",
            "no limma/duplicateCorrelation rerun."
        ),

        paste(
            "Use canonical OOF predictions, fold summaries and saved paired-comparison outputs;",
            "no model refitting."
        ),

        paste(
            "Use saved stability tables/matrices and frozen weights;",
            "no half-sample/permutation rerun."
        ),

        paste(
            "Use saved technical-gate registry and GSE224683 scores/bootstrap;",
            "no external-score recomputation."
        )
    ),

    exact_numbers_reported_elsewhere = c(
        "Table 1",
        "Results text / Supplementary Table S2",
        "Table 2",
        "Supplementary Table S5",
        "Table 3"
    ),

    redundancy_rule = c(

        "Figure shows workflow/QC patterns; Table 1 carries exact cohort numbers.",

        "Figure shows cross-region relationships; full numerical DE outputs remain supplementary data.",

        "Figure shows ROC/distributions/paired deltas; Table 2 gives exact performance estimates and intervals.",

        "Figure visualizes stability/weights; full 24-feature identity/weight record is Supplementary Table S5.",

        "Figure shows gate pattern and score behaviour; Table 3 provides exact coverage/status/outcome numbers."
    ),

    status = "LOCKED",

    stringsAsFactors = FALSE
)

# ==================================================================================================
# 11B.1R-K — MAIN TABLES
#
# Only tables where exact structured numbers materially improve the manuscript.
# ==================================================================================================

main_tables <- data.frame(

    table_id = paste0(
        "Table ",
        1:3
    ),

    working_title = c(

        "Discovery cohort and donor structure",

        "Leakage-safe internal performance and predefined model comparisons",

        "Public external-cohort technical evaluability and frozen-score validation"
    ),

    primary_content = c(

        paste(
            "Exact sample and donor counts by region/diagnosis;",
            "paired/unpaired structure;",
            "key frozen demographic/QC descriptors."
        ),

        paste(
            "Exact performance for Models A, B, C/E, D and F;",
            "Model D = independent region-specific comparator;",
            "donor/sample/regional AUROCs;",
            "mean outer-fold performance;",
            "F vs shared-only paired effect and interval;",
            "F vs independent region-specific comparison;",
            "covariate-only benchmark."
        ),

        paste(
            "All five screened cohorts;",
            "mapped feature count;",
            "gene-count coverage;",
            "absolute/positive/negative weight coverage;",
            "technical PASS/FAIL;",
            "non-evaluable reason;",
            "GSE224683 outcome metrics because it passed the gate."
        )
    ),

    why_table_not_figure = c(

        "Reader needs exact cohort counts and structured descriptors.",

        paste(
            "Exact AUROCs, deltas and confidence intervals are more efficiently",
            "reported in a table than repeated as figure annotations."
        ),

        paste(
            "Exact technical-gate components and outcome metrics across cohorts",
            "require structured side-by-side reporting."
        )
    ),

    corresponding_figure = c(
        "Figure 1 — complementary, not duplicate",
        "Figure 3 — complementary fold/ROC visualization",
        "Figure 5 — complementary gate/outcome visualization"
    ),

    claims_to_support = c(

        "CLAIM_02",

        paste(
            "CLAIM_05; CLAIM_06; CLAIM_07;",
            "CLAIM_08; CLAIM_09"
        ),

        "CLAIM_14; CLAIM_15; CLAIM_16"
    ),

    status = "LOCKED",

    stringsAsFactors = FALSE
)

# ==================================================================================================
# 11B.1R-L — SUPPLEMENTARY FIGURES
# ==================================================================================================

supp_figures <- data.frame(

    figure_id = paste0(
        "Supplementary Figure S",
        1:9
    ),

    working_title = c(

        "Extended preprocessing and expression-QC diagnostics",

        "Detailed differential-expression diagnostics",

        "Outer donor-grouped split and leakage audit",

        "Nested-CV hyperparameter selection and model sparsity",

        "Extended predefined baseline-model diagnostics",

        "Extended stability-selection and permutation diagnostics",

        "Discovery frozen-score descriptive behaviour",

        "External mapping and measurability diagnostics",

        "Conventional enrichment and literature-context biological interpretation"
    ),

    planned_content = c(

        paste(
            "Library-size/TMM diagnostics;",
            "additional PCA components;",
            "robust-distance extremes;",
            "retained-sample QC checks."
        ),

        paste(
            "Regional/shared/deviation volcano or MA displays;",
            "effect concordance;",
            "sign concordance;",
            "significant deviation genes."
        ),

        paste(
            "Repeat/fold donor counts;",
            "diagnosis balance;",
            "donor overlap checks;",
            "split reproducibility."
        ),

        paste(
            "Selected alpha/deviation penalty;",
            "inner-grid performance;",
            "shared/deviation selected-feature distributions."
        ),

        paste(
            "Additional A/B/C-E/D/F OOF diagnostics;",
            "covariate-only sensitivity;",
            "bootstrap comparison context."
        ),

        paste(
            "Full outer-selection frequency;",
            "half-sample frequency/sign consistency;",
            "null selection-frequency distributions."
        ),

        paste(
            "Post-selection discovery rank-score distribution;",
            "paired DLPFC-HIPPO score relationship;",
            "clearly labeled descriptive/non-OOF."
        ),

        paste(
            "Per-cohort mapping detail;",
            "positive/negative retained weight;",
            "all-zero and measurable-feature audit;",
            "post-hoc measurable-only sensitivity."
        ),

        paste(
            "Nominal GO-BP/Reactome context clearly marked non-FDR;",
            "database coverage;",
            "SCZ-specific and brain-context evidence map."
        )
    ),

    manuscript_role = c(
        "QC depth",
        "DE depth",
        "Leakage/reproducibility audit",
        "Model-selection depth",
        "Comparator depth",
        "Stability depth",
        "Descriptive score context",
        "External technical depth",
        "Contextual biology"
    ),

    status = "LOCKED",

    stringsAsFactors = FALSE
)

# ==================================================================================================
# 11B.1R-M — SUPPLEMENTARY TABLES
# ==================================================================================================

supp_tables <- data.frame(

    table_id = paste0(
        "Supplementary Table S",
        1:8
    ),

    working_title = c(

        "Detailed preprocessing, sample accounting, and QC metrics",

        "Differential-expression summary and cross-region concordance statistics",

        "All 25 primary nested-CV outer-fold results and selected hyperparameters",

        "Complete predefined model-comparison and sensitivity statistics",

        "Frozen 24-feature state: identity, stability, direction, and score weights",

        "External mapping, technical-gate, and measurability audit",

        "Systematic literature evidence and manual biological adjudication",

        "Protocol deviations, retrospective freezes, and reporting-governance registry"
    ),

    primary_content = c(

        paste(
            "Filtering/gene-universe accounting;",
            "TMM/PCA summary;",
            "QC extreme-sample audit;",
            "sample retention."
        ),

        paste(
            "DLPFC/HIPPO/shared/deviation counts;",
            "regional effect correlation;",
            "sign concordance;",
            "key DE summaries."
        ),

        paste(
            "Repeat/fold;",
            "test size;",
            "AUROC;",
            "alpha;",
            "deviation penalty;",
            "shared/deviation nonzero counts."
        ),

        paste(
            "F vs shared-only;",
            "F vs independent region-specific Model D;",
            "repeat-block intervals;",
            "bootstrap intervals;",
            "covariate-only sensitivity."
        ),

        paste(
            "All 24 frozen features;",
            "stable Ensembl ID;",
            "frozen/current interpretation symbol;",
            "biotype;",
            "outer/half-sample stability;",
            "direction;",
            "normalized weight;",
            "identity caveats."
        ),

        paste(
            "All five public cohorts;",
            "mapping method;",
            "feature and weight coverage;",
            "technical-gate components;",
            "non-evaluable reasons;",
            "GSE224683 measurability audit."
        ),

        paste(
            "SCZ-focused paper adjudication;",
            "brain-context paper adjudication;",
            "identity tiers;",
            "manuscript-safe biological interpretation."
        ),

        paste(
            "Missed prospective Step0 lock;",
            "retrospective OOF freeze;",
            "final conjunction disclosure;",
            "external-access amendment;",
            "GSE80655 phenotype exposure;",
            "allowed/prohibited claim registry."
        )
    ),

    redundancy_rule = c(

        "Contains exact QC metrics not repeated in Figure 1.",

        "Contains exact DE summary values; Figure 2 is relationship-focused.",

        "Contains fold-level exact values; Figure 3 shows visual distributions.",

        "Contains exact intervals/sensitivity statistics not duplicated graphically.",

        "Full signature record; Figure 4 only provides visual weight/stability representation.",

        "Detailed technical audit beyond the compact main Table 3.",

        "Evidence registry underlying contextual biology; no duplicate pathway claim.",

        "Transparency/governance only; no primary scientific result duplicated."
    ),

    status = "LOCKED",

    stringsAsFactors = FALSE
)

# ==================================================================================================
# 11B.1R-N — RESULTS SECTION SEQUENCE
# ==================================================================================================

results_sequence <- data.frame(

    order = 1:9,

    subsection_id = paste0(
        "R",
        1:9
    ),

    subsection_title = c(

        "Discovery cohort construction, preprocessing, and donor-aware study design",

        "Schizophrenia expression changes show a substantial shared cross-region component",

        "Donor-grouped nested cross-validation demonstrates strong internal discrimination",

        "Regional deviation adds only limited incremental information to the pooled shared model",

        "Repeated stability analyses identify a compact 24-feature shared transcriptional state",

        "Discovery frozen-score behaviour is descriptive and distinct from unbiased OOF performance",

        "Public external datasets show substantial technical transferability constraints",

        "The technically eligible external cohort does not replicate diagnosis-score separation",

        "Biological interpretation is heterogeneous and lacks FDR-supported conventional pathway enrichment"
    ),

    primary_display = c(

        "Figure 1 + Table 1",

        "Figure 2",

        "Figure 3 + Table 2",

        "Figure 3 + Table 2",

        "Figure 4 + Supplementary Table S5",

        "Supplementary Figure S7",

        "Figure 5 + Table 3",

        "Figure 5 + Table 3",

        "Figure 4 annotation + Supplementary Figure S9 + Supplementary Table S7"
    ),

    key_message = c(

        paste(
            "A reproducibly processed 755-sample / 454-donor cross-region cohort",
            "supports donor-aware downstream analysis."
        ),

        paste(
            "Thousands of diagnosis-associated genes contribute to the shared component,",
            "whereas FDR-significant regional deviation is comparatively sparse."
        ),

        "Repeated donor-level OOF AUROC is approximately 0.802.",

        paste(
            "Deviation terms are sparse and add only a small increment over shared-only;",
            "independent region-specific Model D retains numerically higher discrimination."
        ),

        paste(
            "Outer-CV and complementary half-sample stability converge on 24 shared features;",
            "no deviation feature satisfies the final stability definition."
        ),

        paste(
            "The compact frozen-score discovery separation is post-selection/descriptive",
            "and is not substituted for nested-CV performance."
        ),

        paste(
            "Only one of five screened public cohorts satisfies all frozen technical-gate criteria."
        ),

        paste(
            "GSE224683 shows AUROC approximately 0.507 and no meaningful diagnostic score separation."
        ),

        paste(
            "The 24-feature state is half non-protein-coding, biologically heterogeneous,",
            "and yields no FDR-supported GO-BP or Reactome enrichment."
        )
    ),

    status = "LOCKED",

    stringsAsFactors = FALSE
)

# ==================================================================================================
# 11B.1R-O — CORRECT STRUCTURAL VALIDATION
# ==================================================================================================

if (nrow(main_figures) != 5L) {
    stop("Expected 5 main figures.")
}

if (nrow(main_tables) != 3L) {
    stop("Expected 3 main tables.")
}

if (nrow(supp_figures) != 9L) {
    stop("Expected 9 supplementary figures.")
}

if (nrow(supp_tables) != 8L) {
    stop("Expected 8 supplementary tables.")
}

if (nrow(results_sequence) != 9L) {
    stop("Expected 9 Results subsections.")
}

# ----------------------------------------------------------------------------------------------
# Preprocessing must be represented from the beginning.
# ----------------------------------------------------------------------------------------------

if (!any(
    grepl(
        "preprocess|quality control|QC",
        paste(
            main_figures$working_title,
            main_figures$scientific_role
        ),
        ignore.case = TRUE,
        perl = TRUE
    )
)) {

    stop(
        "Preprocessing/QC is missing from main architecture."
    )
}

# ----------------------------------------------------------------------------------------------
# Corrected comparator validation:
# Look for either Model D OR independent region-specific wording.
# ----------------------------------------------------------------------------------------------

table2_text <- paste(
    main_tables$primary_content[
        main_tables$table_id ==
            "Table 2"
    ],
    collapse = " "
)

independent_comparator_present <- grepl(
    "Model D|independent region[- ]specific",
    table2_text,
    ignore.case = TRUE,
    perl = TRUE
)

if (!independent_comparator_present) {

    stop(
        "Independent region-specific Model D comparator is missing from main Table 2."
    )
}

cat(
    "Independent region-specific Model D comparator retained in Table 2: PASS\n"
)

# ----------------------------------------------------------------------------------------------
# External null must remain MAIN.
# ----------------------------------------------------------------------------------------------

external_main_figure <- any(
    grepl(
        "GSE224683|null result|eligible cohort",
        paste(
            main_figures$working_title,
            main_figures$scientific_role,
            main_figures$panels
        ),
        ignore.case = TRUE,
        perl = TRUE
    )
)

external_main_table <- any(
    grepl(
        "GSE224683",
        main_tables$primary_content,
        ignore.case = TRUE,
        perl = TRUE
    )
)

if (
    !external_main_figure ||
    !external_main_table
) {

    stop(
        "External null validation result is not retained in both main visual and exact-number reporting."
    )
}

cat(
    "External null result retained in main manuscript: PASS\n"
)

# ----------------------------------------------------------------------------------------------
# Full 24-feature table should NOT be duplicated as a main table.
# It belongs in Supplementary Table S5 while Figure 4 gives visual representation.
# ----------------------------------------------------------------------------------------------

main_has_full24_table <- any(
    grepl(
        "all 24|24 frozen features|24-feature.*identity",
        main_tables$primary_content,
        ignore.case = TRUE,
        perl = TRUE
    )
)

if (main_has_full24_table) {

    stop(
        "Redundancy detected: full 24-feature signature should not be duplicated as a main table."
    )
}

cat(
    "Full 24-feature detail reserved for Supplementary Table S5: PASS\n"
)

# ==================================================================================================
# 11B.1R-P — WRITE ARCHITECTURE
# ==================================================================================================

write.csv(
    main_figures,
    MAIN_FIG_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    main_tables,
    MAIN_TABLE_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    supp_figures,
    SUPP_FIG_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    supp_tables,
    SUPP_TABLE_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    results_sequence,
    RESULT_SEQUENCE_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 11B.1R-Q — NOTEBOOK ACTION FILE
# ==================================================================================================

notebook_action_lines <- c(

    "SCZ-CRST NOTEBOOK SOURCE STATUS",

    paste0(
        "Project-local .ipynb count detected by STEP11B.0: ",
        n_ipynb
    ),

    "",

    if (n_ipynb == 0L) {
        paste(
            "ACTION REQUIRED:",
            "Save the current executed Jupyter notebook as an .ipynb file inside",
            "E:/SCZ_CROSSREGION_MSS_Q1/notebooks/"
        )
    } else {
        "PASS: At least one project-local .ipynb notebook source is present."
    },

    "",

    paste(
        "Recommended filename:",
        "SCZ_CRST_complete_analysis_executed_v1.ipynb"
    ),

    "",

    paste(
        "This is a source-code preservation action only.",
        "Do not rerun expensive analyses merely to create the notebook copy."
    )
)

writeLines(
    notebook_action_lines,
    NOTEBOOK_ACTION_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 11B.1R-R — SUMMARY
# ==================================================================================================

PUBLICATION_DATA_STATUS <- if (
    CRITICAL_READY &&
    CHECKPOINTS_READY &&
    DISPLAY_SOURCES_READY &&
    identical(
        RENV_STATUS,
        "CONSISTENT_NO_ISSUES"
    )
) {

    "FULL_DATA_AND_ENVIRONMENT_PASS"

} else {

    "REVIEW_REQUIRED"
}

OVERALL_REPRODUCIBILITY_STATUS <- if (
    identical(
        PUBLICATION_DATA_STATUS,
        "FULL_DATA_AND_ENVIRONMENT_PASS"
    ) &&
    n_ipynb >= 1L
) {

    "FULL_PASS"

} else if (
    identical(
        PUBLICATION_DATA_STATUS,
        "FULL_DATA_AND_ENVIRONMENT_PASS"
    ) &&
    n_ipynb == 0L
) {

    "ANALYSIS_ARTIFACTS_FULL_PASS_NOTEBOOK_COPY_REQUIRED"

} else {

    "REVIEW_REQUIRED"
}

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11B.1R — CORRECTED FINAL MANUSCRIPT DISPLAY ARCHITECTURE",
    "==============================================================================================================",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "ARTIFACT READINESS",

    paste0(
        "Critical artifacts ready: ",
        sum(critical$ready),
        "/",
        nrow(critical)
    ),

    paste0(
        "Expensive checkpoint groups passing: ",
        sum(checkpoints$pass),
        "/",
        nrow(checkpoints)
    ),

    paste0(
        "Figure/table source domains ready: ",
        sum(display_ready$ready),
        "/",
        nrow(display_ready)
    ),

    paste0(
        "Corrected renv status: ",
        RENV_STATUS
    ),

    paste0(
        "Project-local .ipynb count: ",
        n_ipynb
    ),

    paste0(
        "Zero-byte noncritical files: ",
        nrow(zero_byte_rows)
    ),

    "",

    paste0(
        "PUBLICATION_DATA_STATUS: ",
        PUBLICATION_DATA_STATUS
    ),

    paste0(
        "OVERALL_REPRODUCIBILITY_STATUS: ",
        OVERALL_REPRODUCIBILITY_STATUS
    ),

    "",

    "FINAL MAIN MANUSCRIPT ARCHITECTURE",

    "Figure 1 — Workflow, preprocessing/QC, and donor structure",
    "Table 1  — Exact discovery cohort and donor characteristics",

    "Figure 2 — Cross-region DE / shared-vs-deviation architecture",

    "Figure 3 — OOF behaviour, model comparisons, deviation contribution",
    "Table 2  — Exact internal performance and predefined comparisons",

    "Figure 4 — Stability convergence, permutation context, frozen 24-feature weights",

    "Figure 5 — External technical transferability and GSE224683 null validation",
    "Table 3  — Exact external technical-gate and validation results",

    "",

    "SUPPLEMENTARY",

    "Supplementary figures: 9",
    "Supplementary tables: 8",

    "",

    "NON-REDUNDANCY RULE",

    paste(
        "Figures carry visual patterns/distributions/relationships.",
        "Tables carry exact numerical and structured reporting.",
        "The same result should not be reproduced in two displays merely for decoration."
    ),

    "",

    "IMPORTANT MANUSCRIPT RULES",

    paste(
        "Preprocessing/QC is explicitly represented from the beginning of the display sequence."
    ),

    paste(
        "Independent region-specific Model D remains visible in the main model-comparison table."
    ),

    paste(
        "GSE224683 null validation remains a main-manuscript result."
    ),

    paste(
        "The full 24-feature detailed record is Supplementary Table S5,",
        "while Figure 4 provides the visual signature representation."
    ),

    paste(
        "Conventional pathway enrichment remains explicitly null at FDR level;",
        "biology is contextual rather than a validation claim."
    ),

    "",

    if (n_ipynb == 0L) {
        paste(
            "ONLY OUTSTANDING REPRODUCIBILITY ACTION:",
            "save the current executed notebook into",
            "E:/SCZ_CROSSREGION_MSS_Q1/notebooks/",
            "as an .ipynb file."
        )
    } else {
        "Notebook-source preservation: PASS."
    },

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 11B.1R-S — MANIFEST
# ==================================================================================================

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {

    install.packages(
        "digest",
        repos = "https://cloud.r-project.org"
    )
}

sha256_file <- function(path) {

    digest::digest(
        file = path,
        algo = "sha256",
        serialize = FALSE
    )
}

manifest_paths <- c(
    CLAIM_PATH,
    NUMBER_PATH,
    LIMIT_PATH,
    CRITICAL_PATH,
    CHECKPOINT_PATH,
    DISPLAY_READY_PATH,
    RENV_STATUS_PATH,
    MAIN_FIG_PATH,
    MAIN_TABLE_PATH,
    SUPP_FIG_PATH,
    SUPP_TABLE_PATH,
    RESULT_SEQUENCE_PATH,
    ZERO_BYTE_PATH,
    NOTEBOOK_ACTION_PATH,
    SUMMARY_PATH
)

manifest <- data.frame(
    file = basename(
        manifest_paths
    ),
    path = manifest_paths,
    bytes = as.numeric(
        file.info(
            manifest_paths
        )$size
    ),
    sha256 = vapply(
        manifest_paths,
        sha256_file,
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 11B.1R-T — CONSOLE REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nCORRECTED MANUSCRIPT ARCHITECTURE SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Critical artifacts ready: ",
    sum(critical$ready),
    "/",
    nrow(critical),
    "\n",
    sep = ""
)

cat(
    "Expensive checkpoints: ",
    sum(checkpoints$pass),
    "/",
    nrow(checkpoints),
    "\n",
    sep = ""
)

cat(
    "Saved display-source domains: ",
    sum(display_ready$ready),
    "/",
    nrow(display_ready),
    "\n",
    sep = ""
)

cat(
    "Corrected renv status: ",
    RENV_STATUS,
    "\n",
    sep = ""
)

cat(
    "Project-local .ipynb count: ",
    n_ipynb,
    "\n",
    sep = ""
)

cat(
    "Zero-byte noncritical files: ",
    nrow(zero_byte_rows),
    "\n",
    sep = ""
)

cat(
    "\nPUBLICATION_DATA_STATUS: ",
    PUBLICATION_DATA_STATUS,
    "\n",
    sep = ""
)

cat(
    "OVERALL_REPRODUCIBILITY_STATUS: ",
    OVERALL_REPRODUCIBILITY_STATUS,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nMAIN FIGURES\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    main_figures[
        ,
        c(
            "figure_id",
            "working_title",
            "redundancy_rule"
        )
    ],
    row.names = FALSE
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nMAIN TABLES\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    main_tables[
        ,
        c(
            "table_id",
            "working_title",
            "why_table_not_figure"
        )
    ],
    row.names = FALSE
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nRESULTS SECTION ORDER\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    results_sequence[
        ,
        c(
            "order",
            "subsection_title",
            "primary_display"
        )
    ],
    row.names = FALSE
)

if (nrow(zero_byte_rows) > 0L) {

    cat(
        "\n",
        paste(rep("-", 110), collapse = ""),
        "\nNONCRITICAL ZERO-BYTE FILE(S)\n",
        paste(rep("-", 110), collapse = ""),
        "\n",
        sep = ""
    )

    print(
        zero_byte_rows[
            ,
            intersect(
                c(
                    "relative_path",
                    "bytes",
                    "critical_artifact"
                ),
                names(zero_byte_rows)
            ),
            drop = FALSE
        ],
        row.names = FALSE
    )
}

cat(
    "\nArchitecture summary:\n",
    SUMMARY_PATH,
    "\n",
    sep = ""
)

cat(
    "\nNotebook-source action file:\n",
    NOTEBOOK_ACTION_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11B.1R COMPLETE — CORRECTED DISPLAY ARCHITECTURE FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11B.1R — CORRECTED FINAL MANUSCRIPT DISPLAY ARCHITECTURE
Project sentinel: PASS
STEP11A.1 claim registry: PASS
Frozen key numbers: 67 — PASS
Frozen allowed claims: 24 — PASS
Frozen limitations: 10 — PASS
Critical artifacts: 65/65 ready
Expensive checkpoint groups: 12/12 PASS
Figure/table source domains: 11/11 ready
STEP11B.0 substantive artifact-readiness audit: PASS
Corrected renv interpretation: CONSISTENT_NO_ISSUES
Project-local .ipynb count: 0
Notebook source status: ACTION_REQUIRED_SAVE_CURRENT_NOTEBOOK_AS_IPYNB
Zero-byte project files: 1 — none are critical
Independent region-specific Model D comparator retained in Table 2: PASS
External null result retained in main manuscript: PASS
Full 24-feature detail reserved for Supplementary Table S5: PASS

CORRECTED MANUSCRIPT ARCHITECTURE SUMMARY
Critical artifacts ready: 65/65
Expensive checkpoints: 12/12
Saved display-source domains: 11/11
Corrected renv status: CONSISTENT_NO_ISSUES
Project-local .ipynb count: 0
Zero-byte noncrit

In [82]:
# ==================================================================================================
# STEP 11C.1T — MAIN FIGURE 1 — FINAL TIMES NEW ROMAN MANUSCRIPT VERSION
# --------------------------------------------------------------------------------------------------
# PANELS
#   A — Sample attrition
#   B — Gene filtering attrition
#   C — TMM-normalized expression PCA
#   D — Donor pairing architecture
#
# PUBLICATION TYPOGRAPHY
#   Font                 : Times New Roman
#   Figure width         : 7.2 inches (full-width manuscript figure)
#   Figure height        : 5.7 inches
#   Panel titles         : 10.5 pt bold
#   Axis titles          : 9.5 pt
#   Axis tick labels     : 8.5 pt
#   Legend title         : 8.5 pt bold
#   Legend text          : 8.0 pt
#
# OUTPUT
#   PDF  = vector master
#   TIFF = 600 dpi
#   PNG  = 300 dpi preview
#
# IMPORTANT
#   - Uses SAVED/FROZEN artifacts only
#   - No preprocessing/PCA/model rerun
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.1T — MAIN FIGURE 1 — FINAL TIMES NEW ROMAN VERSION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID,
        "\nObserved: ", project_id
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

# Register Times New Roman explicitly on Windows.
if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat(
    "Requested manuscript font: ",
    FONT_NAME,
    "\n",
    sep = ""
)

# ==================================================================================================
# 3. INPUT PATHS
# ==================================================================================================

FILTER_PATH <- file.path(
    PROJECT_ROOT,
    "02_metadata",
    "step02a_primary_gene_filter_counts_v1.csv"
)

QC_SUMMARY_PATH <- file.path(
    PROJECT_ROOT,
    "02_metadata",
    "step01c_QC_summary.csv"
)

PCA_PATH <- file.path(
    PROJECT_ROOT,
    "02_metadata",
    "step02b_sample_PCA_QC_v1.csv"
)

PCA_VAR_PATH <- file.path(
    PROJECT_ROOT,
    "02_metadata",
    "step02b_PCA_variance_v1.csv"
)

MODEL_META_PATH <- file.path(
    PROJECT_ROOT,
    "04_processed",
    "brainseq_model_metadata_v1.csv"
)

CLAIM_NUMBERS_PATH <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "claim_registry_v1",
    "STEP11A1_key_numeric_results_v1.csv"
)

required_paths <- c(
    FILTER_PATH,
    QC_SUMMARY_PATH,
    PCA_PATH,
    PCA_VAR_PATH,
    MODEL_META_PATH,
    CLAIM_NUMBERS_PATH
)

missing_paths <- required_paths[
    !file.exists(required_paths)
]

if (length(missing_paths) > 0L) {

    stop(
        "Required source artifact(s) missing:\n",
        paste(
            missing_paths,
            collapse = "\n"
        )
    )
}

cat("All Figure 1 frozen source artifacts: PASS\n")

# ==================================================================================================
# 4. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

# ==================================================================================================
# 5. OUTPUT
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "main",
    "Figure1_final_TNR_v3"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "Figure1_Final_TimesNewRoman_v3.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "Figure1_Final_TimesNewRoman_v3.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "Figure1_Final_TimesNewRoman_v3.png"
)

PANEL_A_PATH <- file.path(
    SOURCE_DIR,
    "Figure1_PanelA_sample_attrition_v3.csv"
)

PANEL_B_PATH <- file.path(
    SOURCE_DIR,
    "Figure1_PanelB_gene_attrition_v3.csv"
)

PANEL_C_PATH <- file.path(
    SOURCE_DIR,
    "Figure1_PanelC_PCA_v3.csv"
)

PANEL_D_PATH <- file.path(
    SOURCE_DIR,
    "Figure1_PanelD_donor_architecture_v3.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "Figure1_Final_generation_summary_v3.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Figure1_Final_manifest_v3.csv"
)

# ==================================================================================================
# 6. LOAD
# ==================================================================================================

filter_counts <- read.csv(
    FILTER_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

qc_summary <- read.csv(
    QC_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

pca <- read.csv(
    PCA_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

pca_var <- read.csv(
    PCA_VAR_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

meta <- read.csv(
    MODEL_META_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

claim_numbers <- read.csv(
    CLAIM_NUMBERS_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 7. VERIFY
# ==================================================================================================

get_claim_value <- function(metric_name) {

    z <- claim_numbers$value[
        claim_numbers$metric ==
            metric_name
    ]

    if (length(z) != 1L) {
        stop(
            "Cannot uniquely retrieve metric: ",
            metric_name
        )
    }

    as.numeric(z)
}

N_DISCOVERY <- get_claim_value(
    "Discovery samples"
)

N_DONORS <- get_claim_value(
    "Discovery donors"
)

if (
    N_DISCOVERY != 755L ||
    N_DONORS != 454L
) {
    stop(
        "Frozen cohort anchors changed."
    )
}

if (nrow(pca) != 755L) {
    stop(
        "Expected 755 PCA rows."
    )
}

if (nrow(meta) != 755L) {
    stop(
        "Expected 755 model-metadata rows."
    )
}

required_meta_cols <- c(
    "BrNum",
    "Region",
    "Dx",
    "paired_donor"
)

missing_meta <- setdiff(
    required_meta_cols,
    names(meta)
)

if (length(missing_meta) > 0L) {
    stop(
        "Missing model-metadata column(s): ",
        paste(
            missing_meta,
            collapse = ", "
        )
    )
}

# ==================================================================================================
# 8. PAIRING
# ==================================================================================================

if (is.logical(
    meta$paired_donor
)) {

    meta$paired_clean <-
        meta$paired_donor

} else {

    z <- toupper(
        trimws(
            as.character(
                meta$paired_donor
            )
        )
    )

    if (!all(
        z %in%
            c(
                "TRUE",
                "FALSE",
                "T",
                "F",
                "1",
                "0"
            )
    )) {
        stop(
            "Unexpected paired_donor encoding."
        )
    }

    meta$paired_clean <- z %in%
        c(
            "TRUE",
            "T",
            "1"
        )
}

donor_level <- meta[
    !duplicated(
        meta$BrNum
    ),
    c(
        "BrNum",
        "Dx",
        "paired_clean"
    ),
    drop = FALSE
]

N_PAIRED <- sum(
    donor_level$paired_clean
)

N_SINGLE <- sum(
    !donor_level$paired_clean
)

if (
    nrow(donor_level) != 454L ||
    N_PAIRED != 301L ||
    N_SINGLE != 153L
) {
    stop(
        "Donor architecture mismatch."
    )
}

cat(
    "Donor architecture: 454 total / 301 paired / 153 single-region — PASS\n"
)

# ==================================================================================================
# 9. SAMPLE ATTRITION
# ==================================================================================================

raw_n <- suppressWarnings(
    as.numeric(
        qc_summary$n
    )
)

raw_n <- raw_n[
    is.finite(raw_n)
]

RAW_SAMPLE_N <- max(
    raw_n
)

if (RAW_SAMPLE_N != 900L) {
    stop(
        "Expected raw sample count = 900."
    )
}

sample_df <- data.frame(

    stage = factor(
        c(
            "BrainSeq source",
            "Analysis cohort"
        ),
        levels = c(
            "BrainSeq source",
            "Analysis cohort"
        )
    ),

    n = c(
        RAW_SAMPLE_N,
        N_DISCOVERY
    ),

    stringsAsFactors = FALSE
)

write.csv(
    sample_df,
    PANEL_A_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 10. GENE ATTRITION
# ==================================================================================================

if (!all(
    c(
        "stage",
        "n_genes"
    ) %in%
        names(filter_counts)
)) {
    stop(
        "Gene filter source missing stage/n_genes."
    )
}

gene_df <- data.frame(
    stage = filter_counts$stage,
    n = filter_counts$n_genes,
    stringsAsFactors = FALSE
)

if (
    tail(
        gene_df$n,
        1L
    ) != 23740L
) {
    stop(
        "Final gene universe is not 23,740."
    )
}

gene_labels <- c(
    "All genes",
    "Expression\nfilter",
    "Valid\nEnsembl",
    "Unique\nEnsembl",
    "Autosomal\nfinal"
)

if (
    length(gene_labels) !=
        nrow(gene_df)
) {
    stop(
        "Unexpected number of gene-filter stages."
    )
}

gene_df$plot_id <- seq_len(
    nrow(gene_df)
)

gene_df$plot_label <- factor(
    gene_labels,
    levels = gene_labels
)

write.csv(
    gene_df,
    PANEL_B_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 11. PCA
# ==================================================================================================

if (!all(
    c(
        "PC1",
        "PC2",
        "Region",
        "Dx"
    ) %in%
        names(pca)
)) {
    stop(
        "PCA source missing required fields."
    )
}

pca$Region <- factor(
    pca$Region,
    levels = c(
        "DLPFC",
        "HIPPO"
    )
)

pca$Dx <- factor(
    pca$Dx,
    levels = c(
        "Control",
        "Schizo"
    )
)

pc1_var <- pca_var$variance_explained[
    pca_var$PC ==
        "PC1"
]

pc2_var <- pca_var$variance_explained[
    pca_var$PC ==
        "PC2"
]

if (
    length(pc1_var) != 1L ||
    length(pc2_var) != 1L
) {
    stop(
        "PC variance lookup failed."
    )
}

PC1_LABEL <- paste0(
    "PC1 (",
    sprintf(
        "%.1f%%",
        100 * pc1_var
    ),
    ")"
)

PC2_LABEL <- paste0(
    "PC2 (",
    sprintf(
        "%.1f%%",
        100 * pc2_var
    ),
    ")"
)

write.csv(
    pca,
    PANEL_C_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 12. DONOR COUNTS
# ==================================================================================================

donor_df <- stats::aggregate(

    BrNum ~ Dx + paired_clean,

    data = donor_level,

    FUN = length
)

names(
    donor_df
)[
    names(donor_df) ==
        "BrNum"
] <- "n_donors"

donor_df$availability <- ifelse(
    donor_df$paired_clean,
    "Paired DLPFC + HIPPO",
    "Single-region"
)

donor_df$Dx <- factor(
    donor_df$Dx,
    levels = c(
        "Control",
        "Schizo"
    )
)

donor_df$availability <- factor(
    donor_df$availability,
    levels = c(
        "Paired DLPFC + HIPPO",
        "Single-region"
    )
)

write.csv(
    donor_df,
    PANEL_D_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 13. PUBLICATION TYPOGRAPHY
# ==================================================================================================

BASE_SIZE <- 9

theme_publication <- ggplot2::theme_classic(
    base_size = BASE_SIZE,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.5,
            hjust = 0,
            margin = ggplot2::margin(
                b = 5
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.5
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.5,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 8.5
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.0
        ),

        legend.key.height = grid::unit(
            0.35,
            "cm"
        ),

        legend.key.width = grid::unit(
            0.45,
            "cm"
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.45,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.40,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            6,
            7,
            6,
            7
        )
    )

# ==================================================================================================
# 14. PANEL A — SAMPLE ATTRITION
# ==================================================================================================

pA <- ggplot2::ggplot(
    sample_df,
    ggplot2::aes(
        x = stage,
        y = n,
        fill = stage
    )
) +
    ggplot2::geom_col(
        width = 0.58,
        colour = "black",
        linewidth = 0.35
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = format(
                n,
                big.mark = ","
            )
        ),
        family = FONT_FAMILY,
        face = "bold",
        size = 3.0,
        vjust = -0.55
    ) +
    ggplot2::scale_fill_manual(
        values = c(
            "BrainSeq source" = "#4E79A7",
            "Analysis cohort" = "#76B7B2"
        ),
        guide = "none"
    ) +
    ggplot2::expand_limits(
        y = 1030
    ) +
    ggplot2::labs(
        title = "A  Sample attrition",
        x = NULL,
        y = "Number of samples"
    ) +
    theme_publication

# ==================================================================================================
# 15. PANEL B — GENE FILTERING
# ==================================================================================================

gene_cols <- c(
    "#E15759",
    "#F28E2B",
    "#EDC948",
    "#B07AA1",
    "#59A14F"
)

names(
    gene_cols
) <- levels(
    gene_df$plot_label
)

pB <- ggplot2::ggplot(
    gene_df,
    ggplot2::aes(
        x = plot_id,
        y = n
    )
) +
    ggplot2::geom_line(
        colour = "#555555",
        linewidth = 0.65
    ) +
    ggplot2::geom_point(
        ggplot2::aes(
            fill = plot_label
        ),
        shape = 21,
        size = 3.0,
        stroke = 0.35,
        colour = "black"
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = format(
                n,
                big.mark = ","
            )
        ),
        family = FONT_FAMILY,
        face = "bold",
        size = 2.65,
        vjust = -0.70
    ) +
    ggplot2::scale_fill_manual(
        values = gene_cols,
        guide = "none"
    ) +
    ggplot2::scale_x_continuous(
        breaks = gene_df$plot_id,
        labels = levels(
            gene_df$plot_label
        )
    ) +
    ggplot2::expand_limits(
        y = 62500
    ) +
    ggplot2::labs(
        title = "B  Gene filtering attrition",
        x = NULL,
        y = "Number of genes"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.8,
            lineheight = 0.88
        )
    )

# ==================================================================================================
# 16. PANEL C — PCA
# ==================================================================================================

pC <- ggplot2::ggplot(
    pca,
    ggplot2::aes(
        x = PC1,
        y = PC2,
        colour = Region,
        shape = Dx
    )
) +
    ggplot2::geom_point(
        size = 1.35,
        alpha = 0.70
    ) +
    ggplot2::scale_colour_manual(
        values = c(
            "DLPFC" = "#2F85B8",
            "HIPPO" = "#E8893A"
        ),
        drop = FALSE
    ) +
    ggplot2::scale_shape_manual(
        values = c(
            "Control" = 16,
            "Schizo" = 17
        ),
        drop = FALSE
    ) +
    ggplot2::labs(
        title = "C  TMM-normalized expression PCA",
        x = PC1_LABEL,
        y = PC2_LABEL,
        colour = "Region",
        shape = "Diagnosis"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.box = "horizontal",
        legend.margin = ggplot2::margin(
            t = 1
        )
    )

# ==================================================================================================
# 17. PANEL D — DONOR ARCHITECTURE
# ==================================================================================================

pD <- ggplot2::ggplot(
    donor_df,
    ggplot2::aes(
        x = Dx,
        y = n_donors,
        fill = availability
    )
) +
    ggplot2::geom_col(
        width = 0.64,
        position = ggplot2::position_dodge(
            width = 0.74
        ),
        colour = "black",
        linewidth = 0.32
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = n_donors
        ),
        family = FONT_FAMILY,
        face = "bold",
        size = 2.9,
        vjust = -0.50,
        position = ggplot2::position_dodge(
            width = 0.74
        )
    ) +
    ggplot2::scale_fill_manual(
        values = c(
            "Paired DLPFC + HIPPO" = "#1B9E77",
            "Single-region" = "#A7A7A7"
        ),
        drop = FALSE
    ) +
    ggplot2::expand_limits(
        y = 210
    ) +
    ggplot2::labs(
        title = "D  Donor pairing architecture",
        x = "Diagnosis",
        y = "Number of donors",
        fill = "Regional availability"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.margin = ggplot2::margin(
            t = 1
        )
    )

# ==================================================================================================
# 18. DRAW 2 × 2 FIGURE
# ==================================================================================================

draw_final_figure1 <- function() {

    grid::grid.newpage()

    grid::pushViewport(
        grid::viewport(
            layout = grid::grid.layout(
                nrow = 2,
                ncol = 2,
                widths = grid::unit(
                    c(
                        1,
                        1
                    ),
                    "null"
                ),
                heights = grid::unit(
                    c(
                        1,
                        1
                    ),
                    "null"
                )
            )
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pD,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 19. SAVE VECTOR PDF
# ==================================================================================================

# cairo_pdf gives substantially better font handling for Times New Roman.
grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 5.7,
    family = FONT_NAME,
    pointsize = 9
)

draw_final_figure1()

grDevices::dev.off()

# ==================================================================================================
# 20. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 5.7,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_final_figure1()

grDevices::dev.off()

# ==================================================================================================
# 21. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 5.7,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_final_figure1()

grDevices::dev.off()

# ==================================================================================================
# 22. VALIDATE OUTPUT
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    PANEL_A_PATH,
    PANEL_B_PATH,
    PANEL_C_PATH,
    PANEL_D_PATH
)

if (!all(
    file.exists(outputs)
)) {
    stop(
        "One or more final Figure 1 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {
    stop(
        "One or more final Figure 1 outputs are zero-byte."
    )
}

cat(
    "Figure 1 final files created and non-zero: PASS\n"
)

# ==================================================================================================
# 23. SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11C.1T — FINAL FIGURE 1",
    "==============================================================================================================",

    paste0(
        "Font: ",
        FONT_NAME
    ),

    "Figure dimensions: 7.2 x 5.7 inches",

    "Panel-title font size: 10.5 pt",

    "Axis-title font size: 9.5 pt",

    "Axis tick font size: 8.5 pt",

    "Legend title/text: 8.5 / 8.0 pt",

    "",

    paste0(
        "Raw BrainSeq samples: ",
        RAW_SAMPLE_N
    ),

    paste0(
        "Discovery samples: ",
        N_DISCOVERY
    ),

    paste0(
        "Discovery donors: ",
        N_DONORS
    ),

    paste0(
        "Paired donors: ",
        N_PAIRED
    ),

    paste0(
        "Single-region donors: ",
        N_SINGLE
    ),

    paste0(
        "Final gene universe: ",
        format(
            tail(
                gene_df$n,
                1L
            ),
            big.mark = ","
        )
    ),

    "",

    "All panels generated from previously saved frozen artifacts.",

    "No expensive analysis was rerun.",

    "",

    paste0(
        "PDF: ",
        PDF_PATH
    ),

    paste0(
        "TIFF: ",
        TIFF_PATH
    ),

    paste0(
        "PNG: ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 24. MANIFEST
# ==================================================================================================

manifest_files <- c(
    FILTER_PATH,
    QC_SUMMARY_PATH,
    PCA_PATH,
    PCA_VAR_PATH,
    MODEL_META_PATH,
    CLAIM_NUMBERS_PATH,
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {
            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 25. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFINAL FIGURE 1 SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Font: Times New Roman\n"
)

cat(
    "Figure size: 7.2 x 5.7 inches\n"
)

cat(
    "Panel title: 10.5 pt\n"
)

cat(
    "Axis title: 9.5 pt\n"
)

cat(
    "Axis tick: 8.5 pt\n"
)

cat(
    "Legend: 8.0–8.5 pt\n"
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.1T COMPLETE — FINAL TIMES NEW ROMAN FIGURE 1 GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.1T — MAIN FIGURE 1 — FINAL TIMES NEW ROMAN VERSION
Project sentinel: PASS
Requested manuscript font: Times New Roman
All Figure 1 frozen source artifacts: PASS
Donor architecture: 454 total / 301 paired / 153 single-region — PASS


Warning message in ggplot2::geom_text(ggplot2::aes(label = format(n, big.mark = ",")), :
"Ignoring unknown parameters: `face`"
Warning message in ggplot2::geom_text(ggplot2::aes(label = format(n, big.mark = ",")), :
"Ignoring unknown parameters: `face`"
Warning message in ggplot2::geom_text(ggplot2::aes(label = n_donors), family = FONT_FAMILY, :
"Ignoring unknown parameters: `face`"


agg_record_211c33ff7d87 
                      2

agg_record_211c33ff7d87 
                      2

agg_record_211c33ff7d87 
                      2

Figure 1 final files created and non-zero: PASS

FINAL FIGURE 1 SUMMARY
Font: Times New Roman
Figure size: 7.2 x 5.7 inches
Panel title: 10.5 pt
Axis title: 9.5 pt
Axis tick: 8.5 pt
Legend: 8.0–8.5 pt

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure1_final_TNR_v3/Figure1_Final_TimesNewRoman_v3.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure1_final_TNR_v3/Figure1_Final_TimesNewRoman_v3.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure1_final_TNR_v3/Figure1_Final_TimesNewRoman_v3.png

STEP 11C.1T COMPLETE — FINAL TIMES NEW ROMAN FIGURE 1 GENERATED


In [86]:
# ==================================================================================================
# STEP 11C.2S — MAIN TABLE 1 — FULLY CORRECTED
# DISCOVERY COHORT CHARACTERISTICS BY REGION AND DIAGNOSIS
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Create manuscript-ready descriptive Table 1 from frozen BrainSeq model metadata.
#
# COLUMNS
#   DLPFC Control
#   DLPFC Schizophrenia
#   HIPPO Control
#   HIPPO Schizophrenia
#
# ROWS
#   Samples
#   Age
#   RIN
#   Sex
#   Race
#
# IMPORTANT
#   - Uses frozen saved metadata only
#   - No statistical testing
#   - No analysis rerun
#   - No duplication of Figure 1 donor-pairing visualization
#   - Donor structure summarized only in table footnote
#   - Times New Roman HTML manuscript preview
#
# CORRECTION
#   Avoids all [[...]] assignment syntax that caused parser failure.
#   Uses pre-created columns + safe numeric column assignment.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.2S — MAIN TABLE 1 — FULLY CORRECTED\n",
    "DISCOVERY COHORT CHARACTERISTICS BY REGION AND DIAGNOSIS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop(
        "Project sentinel missing: ",
        SENTINEL_PATH
    )
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. INPUT PATHS
# ==================================================================================================

MODEL_META_PATH <- file.path(
    PROJECT_ROOT,
    "04_processed",
    "brainseq_model_metadata_v1.csv"
)

CLAIM_NUMBERS_PATH <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "claim_registry_v1",
    "STEP11A1_key_numeric_results_v1.csv"
)

required_inputs <- c(
    MODEL_META_PATH,
    CLAIM_NUMBERS_PATH
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {
    stop(
        "Required input file(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

# ==================================================================================================
# 3. LOAD FROZEN DATA
# ==================================================================================================

meta <- read.csv(
    MODEL_META_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

claim_numbers <- read.csv(
    CLAIM_NUMBERS_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(meta) != 755L) {
    stop(
        "Expected 755 model-metadata rows; observed ",
        nrow(meta)
    )
}

required_cols <- c(
    "BrNum",
    "Region",
    "Dx",
    "donor_age",
    "Sex",
    "Race",
    "RIN",
    "paired_donor"
)

missing_cols <- setdiff(
    required_cols,
    names(meta)
)

if (length(missing_cols) > 0L) {
    stop(
        "Model metadata missing required Table 1 column(s): ",
        paste(
            missing_cols,
            collapse = ", "
        ),
        "\nAvailable columns:\n",
        paste(
            names(meta),
            collapse = ", "
        )
    )
}

cat("Frozen Table 1 metadata source: PASS\n")

# ==================================================================================================
# 4. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "main",
    "Table1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

CSV_PATH <- file.path(
    OUT_DIR,
    "Table1_Discovery_Cohort_Characteristics_v2.csv"
)

HTML_PATH <- file.path(
    OUT_DIR,
    "Table1_Discovery_Cohort_Characteristics_v2.html"
)

SOURCE_PATH <- file.path(
    OUT_DIR,
    "Table1_source_numeric_summary_v2.csv"
)

NOTE_PATH <- file.path(
    OUT_DIR,
    "Table1_notes_v2.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Table1_generation_manifest_v2.csv"
)

# ==================================================================================================
# 5. CLAIM REGISTRY ANCHORS
# ==================================================================================================

get_claim_value <- function(metric_name) {

    idx <- which(
        claim_numbers$metric ==
            metric_name
    )

    if (length(idx) != 1L) {
        stop(
            "Could not uniquely retrieve claim metric: ",
            metric_name
        )
    }

    as.numeric(
        claim_numbers$value[idx]
    )
}

EXPECTED_SAMPLES <- get_claim_value(
    "Discovery samples"
)

EXPECTED_DONORS <- get_claim_value(
    "Discovery donors"
)

if (
    EXPECTED_SAMPLES != 755L ||
    EXPECTED_DONORS != 454L
) {
    stop(
        "Frozen cohort claim anchors changed."
    )
}

cat(
    "Frozen claim anchors: 755 samples / 454 donors — PASS\n"
)

# ==================================================================================================
# 6. NORMALIZE VARIABLES
# ==================================================================================================

meta$Region <- factor(
    meta$Region,
    levels = c(
        "DLPFC",
        "HIPPO"
    )
)

meta$Dx <- factor(
    meta$Dx,
    levels = c(
        "Control",
        "Schizo"
    )
)

# --------------------------------------------------------------------------------------------------
# Sex
# --------------------------------------------------------------------------------------------------

sex_values <- unique(
    as.character(
        meta$Sex
    )
)

cat(
    "Observed Sex categories: ",
    paste(
        sex_values,
        collapse = ", "
    ),
    "\n",
    sep = ""
)

if (!all(
    sex_values %in%
        c(
            "F",
            "M"
        )
)) {
    stop(
        "Unexpected Sex encoding detected."
    )
}

meta$Sex <- factor(
    meta$Sex,
    levels = c(
        "F",
        "M"
    )
)

# --------------------------------------------------------------------------------------------------
# Race
# --------------------------------------------------------------------------------------------------

race_order <- c(
    "AA",
    "AS",
    "CAUC",
    "HISP"
)

race_values <- unique(
    as.character(
        meta$Race
    )
)

unexpected_race <- setdiff(
    race_values,
    race_order
)

if (length(unexpected_race) > 0L) {
    stop(
        "Unexpected Race category detected: ",
        paste(
            unexpected_race,
            collapse = ", "
        )
    )
}

meta$Race <- factor(
    meta$Race,
    levels = race_order
)

# --------------------------------------------------------------------------------------------------
# paired_donor
# --------------------------------------------------------------------------------------------------

if (is.logical(
    meta$paired_donor
)) {

    meta$paired_clean <- meta$paired_donor

} else {

    pair_text <- toupper(
        trimws(
            as.character(
                meta$paired_donor
            )
        )
    )

    valid_pair_values <- c(
        "TRUE",
        "FALSE",
        "T",
        "F",
        "1",
        "0"
    )

    if (!all(
        pair_text %in%
            valid_pair_values
    )) {
        stop(
            "Unexpected paired_donor encoding: ",
            paste(
                unique(
                    pair_text[
                        !pair_text %in%
                            valid_pair_values
                    ]
                ),
                collapse = ", "
            )
        )
    }

    meta$paired_clean <- pair_text %in%
        c(
            "TRUE",
            "T",
            "1"
        )
}

# ==================================================================================================
# 7. COMPLETENESS CHECK
# ==================================================================================================

table1_vars <- c(
    "donor_age",
    "Sex",
    "Race",
    "RIN"
)

missingness <- vapply(
    meta[
        ,
        table1_vars,
        drop = FALSE
    ],
    function(x) {
        sum(
            is.na(x)
        )
    },
    FUN.VALUE = integer(1)
)

cat(
    "\nTable 1 missingness audit:\n"
)

print(
    missingness
)

if (any(
    missingness > 0L
)) {
    stop(
        "Unexpected Table 1 covariate missingness."
    )
}

cat(
    "Table 1 covariate completeness: PASS\n"
)

# ==================================================================================================
# 8. DEFINE GROUPS
# ==================================================================================================

group_specs <- data.frame(

    key = c(
        "DLPFC_Control",
        "DLPFC_Schizo",
        "HIPPO_Control",
        "HIPPO_Schizo"
    ),

    region = c(
        "DLPFC",
        "DLPFC",
        "HIPPO",
        "HIPPO"
    ),

    diagnosis = c(
        "Control",
        "Schizo",
        "Control",
        "Schizo"
    ),

    display = c(
        "DLPFC Control",
        "DLPFC Schizophrenia",
        "HIPPO Control",
        "HIPPO Schizophrenia"
    ),

    stringsAsFactors = FALSE
)

expected_group_n <- c(
    226L,
    153L,
    243L,
    133L
)

observed_group_n <- integer(
    nrow(
        group_specs
    )
)

for (i in seq_len(
    nrow(group_specs)
)) {

    observed_group_n[i] <- sum(
        as.character(
            meta$Region
        ) ==
            group_specs$region[i] &
        as.character(
            meta$Dx
        ) ==
            group_specs$diagnosis[i]
    )
}

if (!identical(
    observed_group_n,
    expected_group_n
)) {
    stop(
        "Region × diagnosis sample counts do not match frozen values.\n",
        "Observed: ",
        paste(
            observed_group_n,
            collapse = ", "
        ),
        "\nExpected: ",
        paste(
            expected_group_n,
            collapse = ", "
        )
    )
}

cat(
    "Region × diagnosis sample counts: ",
    paste(
        observed_group_n,
        collapse = " / "
    ),
    " — PASS\n",
    sep = ""
)

# ==================================================================================================
# 9. FORMAT FUNCTIONS
# ==================================================================================================

fmt_mean_sd <- function(
    x,
    digits = 1L
) {

    m <- mean(
        x,
        na.rm = TRUE
    )

    s <- stats::sd(
        x,
        na.rm = TRUE
    )

    paste0(
        formatC(
            m,
            format = "f",
            digits = digits
        ),
        " ± ",
        formatC(
            s,
            format = "f",
            digits = digits
        )
    )
}

fmt_n_pct <- function(
    n,
    denominator
) {

    pct <- 100 * n / denominator

    paste0(
        n,
        " (",
        formatC(
            pct,
            format = "f",
            digits = 1L
        ),
        "%)"
    )
}

# ==================================================================================================
# 10. BUILD NUMERIC SOURCE SUMMARY
# ==================================================================================================

source_rows <- list()
k <- 1L

for (i in seq_len(
    nrow(group_specs)
)) {

    keep <- (
        as.character(meta$Region) ==
            group_specs$region[i] &
        as.character(meta$Dx) ==
            group_specs$diagnosis[i]
    )

    d <- meta[
        keep,
        ,
        drop = FALSE
    ]

    N <- nrow(d)

    # Samples
    source_rows[[k]] <- data.frame(
        group = group_specs$key[i],
        variable = "Samples",
        level = "",
        n = N,
        mean = NA_real_,
        sd = NA_real_,
        percent = NA_real_,
        stringsAsFactors = FALSE
    )

    k <- k + 1L

    # Age
    source_rows[[k]] <- data.frame(
        group = group_specs$key[i],
        variable = "Age",
        level = "",
        n = N,
        mean = mean(d$donor_age),
        sd = stats::sd(d$donor_age),
        percent = NA_real_,
        stringsAsFactors = FALSE
    )

    k <- k + 1L

    # RIN
    source_rows[[k]] <- data.frame(
        group = group_specs$key[i],
        variable = "RIN",
        level = "",
        n = N,
        mean = mean(d$RIN),
        sd = stats::sd(d$RIN),
        percent = NA_real_,
        stringsAsFactors = FALSE
    )

    k <- k + 1L

    # Sex
    for (sx in c(
        "F",
        "M"
    )) {

        nn <- sum(
            as.character(
                d$Sex
            ) ==
                sx
        )

        source_rows[[k]] <- data.frame(
            group = group_specs$key[i],
            variable = "Sex",
            level = sx,
            n = nn,
            mean = NA_real_,
            sd = NA_real_,
            percent = 100 * nn / N,
            stringsAsFactors = FALSE
        )

        k <- k + 1L
    }

    # Race
    for (rr in race_order) {

        nn <- sum(
            as.character(
                d$Race
            ) ==
                rr
        )

        source_rows[[k]] <- data.frame(
            group = group_specs$key[i],
            variable = "Race",
            level = rr,
            n = nn,
            mean = NA_real_,
            sd = NA_real_,
            percent = 100 * nn / N,
            stringsAsFactors = FALSE
        )

        k <- k + 1L
    }
}

source_summary <- do.call(
    rbind,
    source_rows
)

rownames(
    source_summary
) <- NULL

write.csv(
    source_summary,
    SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# ==================================================================================================
# 11. CREATE FINAL TABLE STRUCTURE
#
# IMPORTANT:
# All columns are created HERE first.
# No [[...]] syntax is used anywhere.
# ==================================================================================================

table1 <- data.frame(

    Characteristic = c(
        "Samples, n",
        "Age, years, mean ± SD",
        "RNA integrity number, mean ± SD",
        "Sex, n (%)",
        "  Female",
        "  Male",
        "Race, n (%)",
        "  African American",
        "  Asian",
        "  Caucasian",
        "  Hispanic"
    ),

    DLPFC_Control = rep(
        "",
        11L
    ),

    DLPFC_Schizophrenia = rep(
        "",
        11L
    ),

    HIPPO_Control = rep(
        "",
        11L
    ),

    HIPPO_Schizophrenia = rep(
        "",
        11L
    ),

    stringsAsFactors = FALSE,

    check.names = FALSE
)

# ==================================================================================================
# 12. FILL TABLE COLUMNS
#
# SAFE ASSIGNMENT:
# table1[, i + 1L] <- values
#
# No double bracket syntax.
# ==================================================================================================

for (i in seq_len(
    nrow(group_specs)
)) {

    keep <- (
        as.character(meta$Region) ==
            group_specs$region[i] &
        as.character(meta$Dx) ==
            group_specs$diagnosis[i]
    )

    d <- meta[
        keep,
        ,
        drop = FALSE
    ]

    N <- nrow(d)

    female_n <- sum(
        as.character(
            d$Sex
        ) ==
            "F"
    )

    male_n <- sum(
        as.character(
            d$Sex
        ) ==
            "M"
    )

    aa_n <- sum(
        as.character(
            d$Race
        ) ==
            "AA"
    )

    asian_n <- sum(
        as.character(
            d$Race
        ) ==
            "AS"
    )

    cauc_n <- sum(
        as.character(
            d$Race
        ) ==
            "CAUC"
    )

    hisp_n <- sum(
        as.character(
            d$Race
        ) ==
            "HISP"
    )

    values <- c(
        as.character(N),
        fmt_mean_sd(
            d$donor_age,
            digits = 1L
        ),
        fmt_mean_sd(
            d$RIN,
            digits = 2L
        ),
        "",
        fmt_n_pct(
            female_n,
            N
        ),
        fmt_n_pct(
            male_n,
            N
        ),
        "",
        fmt_n_pct(
            aa_n,
            N
        ),
        fmt_n_pct(
            asian_n,
            N
        ),
        fmt_n_pct(
            cauc_n,
            N
        ),
        fmt_n_pct(
            hisp_n,
            N
        )
    )

    # SAFE numeric-column assignment
    table1[
        ,
        i + 1L
    ] <- values
}

# ==================================================================================================
# 13. FINAL DISPLAY NAMES
# ==================================================================================================

names(table1) <- c(
    "Characteristic",
    "DLPFC Control",
    "DLPFC Schizophrenia",
    "HIPPO Control",
    "HIPPO Schizophrenia"
)

# ==================================================================================================
# 14. VERIFY TABLE COUNTS
# ==================================================================================================

if (!identical(
    as.character(
        table1[
            1L,
            2:5
        ]
    ),
    as.character(
        expected_group_n
    )
)) {
    stop(
        "Table 1 first-row sample counts failed verification."
    )
}

cat(
    "Table 1 sample-count row verification: PASS\n"
)

# ==================================================================================================
# 15. DONOR-LEVEL FOOTNOTE
# ==================================================================================================

donor_level <- meta[
    !duplicated(
        meta$BrNum
    ),
    c(
        "BrNum",
        "Dx",
        "paired_clean"
    ),
    drop = FALSE
]

if (nrow(donor_level) != 454L) {
    stop(
        "Expected 454 unique donors; observed ",
        nrow(donor_level)
    )
}

control_donors <- sum(
    as.character(
        donor_level$Dx
    ) ==
        "Control"
)

schizo_donors <- sum(
    as.character(
        donor_level$Dx
    ) ==
        "Schizo"
)

paired_control <- sum(
    as.character(
        donor_level$Dx
    ) ==
        "Control" &
    donor_level$paired_clean
)

paired_schizo <- sum(
    as.character(
        donor_level$Dx
    ) ==
        "Schizo" &
    donor_level$paired_clean
)

single_control <- sum(
    as.character(
        donor_level$Dx
    ) ==
        "Control" &
    !donor_level$paired_clean
)

single_schizo <- sum(
    as.character(
        donor_level$Dx
    ) ==
        "Schizo" &
    !donor_level$paired_clean
)

observed_donor_counts <- as.integer(
    c(
        control_donors,
        schizo_donors,
        paired_control,
        paired_schizo,
        single_control,
        single_schizo
    )
)

expected_donor_counts <- as.integer(
    c(
        277,
        177,
        192,
        109,
        85,
        68
    )
)

if (!identical(
    observed_donor_counts,
    expected_donor_counts
)) {
    stop(
        "Donor counts changed.\n",
        "Observed: ",
        paste(
            observed_donor_counts,
            collapse = ", "
        ),
        "\nExpected: ",
        paste(
            expected_donor_counts,
            collapse = ", "
        )
    )
}

cat(
    "Donor-level footnote counts: PASS\n"
)

# ==================================================================================================
# 16. WRITE CANONICAL CSV
# ==================================================================================================

write.csv(
    table1,
    CSV_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 17. TABLE NOTES
# ==================================================================================================

notes <- c(

    "Table 1. Discovery cohort characteristics by region and diagnosis.",

    "",

    paste(
        "Values are n (%) unless otherwise indicated.",
        "Continuous variables are reported as mean ± SD."
    ),

    "",

    paste(
        "The 755 tissue samples were obtained from 454 unique donors:",
        "277 control donors and 177 donors with schizophrenia."
    ),

    paste(
        "Among the 454 donors, 301 contributed both DLPFC and hippocampal samples",
        "(192 controls and 109 schizophrenia), whereas 153 contributed a single region",
        "(85 controls and 68 schizophrenia)."
    ),

    "",

    paste(
        "Race coding follows the frozen BrainSeq metadata:",
        "AA = African American;",
        "AS = Asian;",
        "CAUC = Caucasian;",
        "HISP = Hispanic."
    ),

    "",

    paste(
        "DLPFC = dorsolateral prefrontal cortex;",
        "RIN = RNA integrity number;",
        "SD = standard deviation."
    ),

    "",

    paste(
        "No baseline inferential p values are reported because this table is descriptive.",
        "Covariates were handled within the prespecified donor-aware modelling framework."
    )
)

writeLines(
    notes,
    NOTE_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 18. HTML ESCAPE
# ==================================================================================================

html_escape <- function(x) {

    x <- as.character(x)

    x <- gsub(
        "&",
        "&amp;",
        x,
        fixed = TRUE
    )

    x <- gsub(
        "<",
        "&lt;",
        x,
        fixed = TRUE
    )

    x <- gsub(
        ">",
        "&gt;",
        x,
        fixed = TRUE
    )

    x
}

# ==================================================================================================
# 19. HTML MANUSCRIPT TABLE — TIMES NEW ROMAN
# ==================================================================================================

header_labels <- c(
    "Characteristic",
    "DLPFC Control<br>(n = 226)",
    "DLPFC Schizophrenia<br>(n = 153)",
    "HIPPO Control<br>(n = 243)",
    "HIPPO Schizophrenia<br>(n = 133)"
)

html_lines <- c(
    "<!DOCTYPE html>",
    "<html>",
    "<head>",
    "<meta charset='UTF-8'>",
    "<title>Table 1</title>",
    "<style>",

    "body {",
    "font-family: 'Times New Roman', Times, serif;",
    "margin: 28px;",
    "color: #000000;",
    "background-color: #ffffff;",
    "}",

    ".title {",
    "font-size: 11pt;",
    "font-weight: bold;",
    "margin-bottom: 8px;",
    "}",

    "table {",
    "border-collapse: collapse;",
    "width: 100%;",
    "table-layout: fixed;",
    "font-family: 'Times New Roman', Times, serif;",
    "font-size: 9pt;",
    "}",

    "thead th {",
    "padding: 6px 5px;",
    "text-align: center;",
    "font-weight: bold;",
    "vertical-align: bottom;",
    "border-top: 1.5px solid black;",
    "border-bottom: 1px solid black;",
    "}",

    "thead th:first-child {",
    "text-align: left;",
    "width: 27%;",
    "}",

    "tbody td {",
    "padding: 4px 5px;",
    "vertical-align: middle;",
    "}",

    "tbody td:first-child {",
    "text-align: left;",
    "}",

    "tbody td:not(:first-child) {",
    "text-align: center;",
    "}",

    "tbody tr.section td {",
    "font-weight: bold;",
    "padding-top: 7px;",
    "}",

    "tbody tr:last-child td {",
    "border-bottom: 1.5px solid black;",
    "}",

    ".notes {",
    "font-family: 'Times New Roman', Times, serif;",
    "font-size: 8pt;",
    "line-height: 1.25;",
    "margin-top: 8px;",
    "}",

    "</style>",
    "</head>",
    "<body>",

    "<div class='title'>Table 1. Discovery cohort characteristics by region and diagnosis</div>",

    "<table>",
    "<thead>",
    "<tr>"
)

for (h in header_labels) {

    html_lines <- c(
        html_lines,
        paste0(
            "<th>",
            h,
            "</th>"
        )
    )
}

html_lines <- c(
    html_lines,
    "</tr>",
    "</thead>",
    "<tbody>"
)

section_rows <- c(
    "Sex, n (%)",
    "Race, n (%)"
)

for (i in seq_len(
    nrow(table1)
)) {

    current_label <- as.character(
        table1$Characteristic[i]
    )

    row_tag <- if (
        current_label %in%
            section_rows
    ) {
        "<tr class='section'>"
    } else {
        "<tr>"
    }

    html_lines <- c(
        html_lines,
        row_tag
    )

    for (j in seq_len(
        ncol(table1)
    )) {

        cell_value <- as.character(
            table1[
                i,
                j
            ]
        )

        html_lines <- c(
            html_lines,
            paste0(
                "<td>",
                html_escape(
                    cell_value
                ),
                "</td>"
            )
        )
    }

    html_lines <- c(
        html_lines,
        "</tr>"
    )
}

html_lines <- c(
    html_lines,

    "</tbody>",
    "</table>",

    "<div class='notes'>",

    paste0(
        "<b>Notes.</b> Values are n (%) unless otherwise indicated; ",
        "continuous variables are mean ± SD. "
    ),

    paste0(
        "The 755 samples were obtained from 454 unique donors ",
        "(277 controls; 177 schizophrenia). "
    ),

    paste0(
        "A total of 301 donors contributed both regions ",
        "(192 controls; 109 schizophrenia), while 153 contributed a single region ",
        "(85 controls; 68 schizophrenia). "
    ),

    paste0(
        "AA, African American; AS, Asian; CAUC, Caucasian; HISP, Hispanic; ",
        "DLPFC, dorsolateral prefrontal cortex; RIN, RNA integrity number; ",
        "SD, standard deviation."
    ),

    "</div>",
    "</body>",
    "</html>"
)

writeLines(
    html_lines,
    HTML_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 20. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    CSV_PATH,
    HTML_PATH,
    SOURCE_PATH,
    NOTE_PATH
)

if (!all(
    file.exists(outputs)
)) {
    stop(
        "One or more Table 1 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {
    stop(
        "One or more Table 1 outputs are zero-byte."
    )
}

cat(
    "Table 1 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 21. MANIFEST
# ==================================================================================================

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop(
        "digest unavailable in frozen environment."
    )
}

manifest_files <- c(
    MODEL_META_PATH,
    CLAIM_NUMBERS_PATH,
    outputs
)

manifest <- data.frame(
    file = basename(
        manifest_files
    ),
    path = manifest_files,
    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),
    sha256 = vapply(
        manifest_files,
        function(p) {
            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 22. CONSOLE PREVIEW
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nMAIN TABLE 1 — MANUSCRIPT PREVIEW\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    table1,
    row.names = FALSE,
    right = FALSE
)

cat(
    "\nDONOR-LEVEL SUMMARY\n"
)

cat(
    "Unique donors: ",
    nrow(donor_level),
    "\n",
    sep = ""
)

cat(
    "Control donors: ",
    control_donors,
    "\n",
    sep = ""
)

cat(
    "Schizophrenia donors: ",
    schizo_donors,
    "\n",
    sep = ""
)

cat(
    "Paired donors: ",
    sum(
        donor_level$paired_clean
    ),
    " (Control ",
    paired_control,
    "; Schizophrenia ",
    paired_schizo,
    ")\n",
    sep = ""
)

cat(
    "Single-region donors: ",
    sum(
        !donor_level$paired_clean
    ),
    " (Control ",
    single_control,
    "; Schizophrenia ",
    single_schizo,
    ")\n",
    sep = ""
)

cat(
    "\nCSV:\n",
    CSV_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTimes New Roman HTML preview:\n",
    HTML_PATH,
    "\n",
    sep = ""
)

cat(
    "\nNotes:\n",
    NOTE_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.2S COMPLETE — MAIN TABLE 1 GENERATED CORRECTLY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.2S — MAIN TABLE 1 — FULLY CORRECTED
DISCOVERY COHORT CHARACTERISTICS BY REGION AND DIAGNOSIS
Project sentinel: PASS
Frozen Table 1 metadata source: PASS
Frozen claim anchors: 755 samples / 454 donors — PASS
Observed Sex categories: M, F

Table 1 missingness audit:
donor_age       Sex      Race       RIN 
        0         0         0         0 
Table 1 covariate completeness: PASS
Region × diagnosis sample counts: 226 / 153 / 243 / 133 — PASS
Table 1 sample-count row verification: PASS
Donor-level footnote counts: PASS
Table 1 outputs created and non-zero: PASS

MAIN TABLE 1 — MANUSCRIPT PREVIEW
 Characteristic                  DLPFC Control DLPFC Schizophrenia
 Samples, n                      226           153                
 Age, years, mean ± SD           45.0 ± 15.0   50.6 ± 15.7        
 RNA integrity number, mean ± SD 7.72 ± 0.87   7.62 ± 0.97        
 Sex, n (%)                                                       
   Female                        68 (30.1%)    49 (3

In [88]:
# ==================================================================================================
# STEP 11C.3R — MAIN FIGURE 2 — SOURCE-RECONCILED CORRECTED VERSION
# SHARED AND REGION-DEPENDENT SCHIZOPHRENIA TRANSCRIPTOMIC ARCHITECTURE
# --------------------------------------------------------------------------------------------------
# WHY THIS REVISION EXISTS
#
# Previous Figure 2 cell incorrectly imposed the historical manuscript anchor
#     DLPFC-HIPPO logFC correlation = 0.6089
# as a hard stop.
#
# The CURRENT saved DLPFC/HIPPO DE CSV files produce a different correlation.
#
# This cell therefore:
#   1. treats the CURRENT saved DE result files as the plotting source,
#   2. verifies DLPFC/HIPPO effects against shared ± deviation algebra,
#   3. compares current saved-source values against STEP11A.1 historical claim anchors,
#   4. saves a reconciliation audit,
#   5. generates Figure 2 WITHOUT rerunning limma/voom/duplicateCorrelation.
#
# PANELS
#   A — DLPFC vs HIPPO logFC concordance
#   B — FDR-significant gene burden across contrasts
#   C — Shared schizophrenia-effect volcano
#   D — Regional-deviation volcano
#
# TYPOGRAPHY
#   Times New Roman
#   Full-width manuscript figure
#   PDF vector master + 600-dpi TIFF + 300-dpi PNG
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3R — MAIN FIGURE 2 — SOURCE-RECONCILED CORRECTED VERSION\n",
    "SHARED AND REGION-DEPENDENT SCHIZOPHRENIA TRANSCRIPTOMIC ARCHITECTURE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat(
    "Figure font: ",
    FONT_NAME,
    "\n",
    sep = ""
)

# ==================================================================================================
# 3. INPUT PATHS
# ==================================================================================================

DE_DIR <- file.path(
    PROJECT_ROOT,
    "05_differential_expression"
)

DLPFC_PATH <- file.path(
    DE_DIR,
    "step03a_DLPFC_SCZ_vs_Control_v1.csv"
)

HIPPO_PATH <- file.path(
    DE_DIR,
    "step03a_HIPPO_SCZ_vs_Control_v1.csv"
)

SHARED_PATH <- file.path(
    DE_DIR,
    "step03a_Shared_SCZ_effect_v1.csv"
)

DEVIATION_PATH <- file.path(
    DE_DIR,
    "step03a_Regional_Deviation_v1.csv"
)

DE_SUMMARY_PATH <- file.path(
    DE_DIR,
    "step03a_DE_summary_v1.csv"
)

MODEL_AUDIT_PATH <- file.path(
    DE_DIR,
    "step03a_model_audit_v1.txt"
)

CLAIM_PATH <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "claim_registry_v1",
    "STEP11A1_key_numeric_results_v1.csv"
)

required_inputs <- c(
    DLPFC_PATH,
    HIPPO_PATH,
    SHARED_PATH,
    DEVIATION_PATH,
    DE_SUMMARY_PATH,
    CLAIM_PATH
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required frozen source artifact(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat("Required saved DE artifacts: PASS\n")

# ==================================================================================================
# 4. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

USE_GGREPEL <- requireNamespace(
    "ggrepel",
    quietly = TRUE
)

cat(
    "ggrepel available: ",
    USE_GGREPEL,
    "\n",
    sep = ""
)

# ==================================================================================================
# 5. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "main",
    "Figure2_reconciled_v2"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "Figure2_Shared_Regional_DE_Architecture_reconciled_v2.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "Figure2_Shared_Regional_DE_Architecture_reconciled_v2.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "Figure2_Shared_Regional_DE_Architecture_reconciled_v2.png"
)

PANEL_A_PATH <- file.path(
    SOURCE_DIR,
    "Figure2_PanelA_regional_logFC_source_v2.csv"
)

PANEL_B_PATH <- file.path(
    SOURCE_DIR,
    "Figure2_PanelB_FDR_counts_source_v2.csv"
)

PANEL_C_PATH <- file.path(
    SOURCE_DIR,
    "Figure2_PanelC_shared_volcano_source_v2.csv"
)

PANEL_D_PATH <- file.path(
    SOURCE_DIR,
    "Figure2_PanelD_deviation_volcano_source_v2.csv"
)

RECON_CSV_PATH <- file.path(
    SOURCE_DIR,
    "Figure2_DE_source_reconciliation_v2.csv"
)

RECON_TXT_PATH <- file.path(
    OUT_DIR,
    "Figure2_DE_source_reconciliation_v2.txt"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "Figure2_generation_summary_v2.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Figure2_generation_manifest_v2.csv"
)

# ==================================================================================================
# 6. LOAD SAVED DATA
# ==================================================================================================

dlpfc <- read.csv(
    DLPFC_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

hippo <- read.csv(
    HIPPO_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

shared <- read.csv(
    SHARED_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

deviation <- read.csv(
    DEVIATION_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

de_summary <- read.csv(
    DE_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

claim_numbers <- read.csv(
    CLAIM_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 7. BASIC TABLE INTEGRITY
# ==================================================================================================

de_tables <- list(
    dlpfc,
    hippo,
    shared,
    deviation
)

de_names <- c(
    "DLPFC",
    "HIPPO",
    "Shared",
    "Deviation"
)

required_de_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "logFC",
    "P.Value",
    "adj.P.Val"
)

for (i in seq_along(de_tables)) {

    x <- de_tables[[i]]

    if (nrow(x) != 23740L) {

        stop(
            de_names[i],
            " table should contain 23,740 rows; observed ",
            nrow(x)
        )
    }

    missing_cols <- setdiff(
        required_de_cols,
        names(x)
    )

    if (length(missing_cols) > 0L) {

        stop(
            de_names[i],
            " table missing required column(s): ",
            paste(
                missing_cols,
                collapse = ", "
            )
        )
    }

    if (anyDuplicated(x$ensemblID_stable) > 0L) {

        stop(
            de_names[i],
            " table contains duplicated stable Ensembl IDs."
        )
    }
}

cat(
    "Four DE tables: 23,740 unique genes each — PASS\n"
)

# ==================================================================================================
# 8. ALIGN ALL FOUR TABLES TO DLPFC STABLE-ID ORDER
# ==================================================================================================

MASTER_ID <- dlpfc$ensemblID_stable

idx_hippo <- match(
    MASTER_ID,
    hippo$ensemblID_stable
)

idx_shared <- match(
    MASTER_ID,
    shared$ensemblID_stable
)

idx_deviation <- match(
    MASTER_ID,
    deviation$ensemblID_stable
)

if (
    anyNA(idx_hippo) ||
    anyNA(idx_shared) ||
    anyNA(idx_deviation)
) {

    stop(
        "Stable-Ensembl alignment failed across saved DE tables."
    )
}

cat(
    "Stable-Ensembl alignment across four DE tables: PASS\n"
)

# ==================================================================================================
# 9. CURRENT SAVED-SOURCE REGIONAL EFFECT TABLE
# ==================================================================================================

regional <- data.frame(

    ensemblID_stable = MASTER_ID,

    gene_symbol = dlpfc$gene_symbol,

    gene_type = dlpfc$gene_type,

    DLPFC_logFC = as.numeric(
        dlpfc$logFC
    ),

    HIPPO_logFC = as.numeric(
        hippo$logFC[idx_hippo]
    ),

    Shared_logFC = as.numeric(
        shared$logFC[idx_shared]
    ),

    Deviation_logFC = as.numeric(
        deviation$logFC[idx_deviation]
    ),

    DLPFC_FDR = as.numeric(
        dlpfc$adj.P.Val
    ),

    HIPPO_FDR = as.numeric(
        hippo$adj.P.Val[idx_hippo]
    ),

    stringsAsFactors = FALSE
)

# ==================================================================================================
# 10. ALGEBRAIC RECONSTRUCTION AUDIT
#
# Frozen DE definition:
#   shared    = (DLPFC + HIPPO) / 2
#   deviation = (DLPFC - HIPPO) / 2
#
# Therefore:
#   DLPFC = shared + deviation
#   HIPPO = shared - deviation
# ==================================================================================================

regional$DLPFC_reconstructed <- (
    regional$Shared_logFC +
    regional$Deviation_logFC
)

regional$HIPPO_reconstructed <- (
    regional$Shared_logFC -
    regional$Deviation_logFC
)

regional$DLPFC_reconstruction_error <- (
    regional$DLPFC_logFC -
    regional$DLPFC_reconstructed
)

regional$HIPPO_reconstruction_error <- (
    regional$HIPPO_logFC -
    regional$HIPPO_reconstructed
)

MAX_DLPFC_RECON_ERROR <- max(
    abs(
        regional$DLPFC_reconstruction_error
    ),
    na.rm = TRUE
)

MAX_HIPPO_RECON_ERROR <- max(
    abs(
        regional$HIPPO_reconstruction_error
    ),
    na.rm = TRUE
)

RECON_TOL <- 1e-8

DE_TABLES_ALGEBRAICALLY_CONSISTENT <- (
    MAX_DLPFC_RECON_ERROR <= RECON_TOL &&
    MAX_HIPPO_RECON_ERROR <= RECON_TOL
)

cat(
    "Max DLPFC shared+deviation reconstruction error: ",
    format(
        MAX_DLPFC_RECON_ERROR,
        scientific = TRUE,
        digits = 5
    ),
    "\n",
    sep = ""
)

cat(
    "Max HIPPO shared-deviation reconstruction error: ",
    format(
        MAX_HIPPO_RECON_ERROR,
        scientific = TRUE,
        digits = 5
    ),
    "\n",
    sep = ""
)

if (!DE_TABLES_ALGEBRAICALLY_CONSISTENT) {

    stop(
        "Current saved DLPFC/HIPPO/shared/deviation files are not algebraically consistent."
    )
}

cat(
    "Saved DE contrast algebra: PASS\n"
)

# ==================================================================================================
# 11. CURRENT SAVED-SOURCE CORRELATION + SIGN CONCORDANCE
# ==================================================================================================

CURRENT_COR <- stats::cor(
    regional$DLPFC_logFC,
    regional$HIPPO_logFC,
    use = "complete.obs",
    method = "pearson"
)

CURRENT_SIGN <- mean(
    sign(
        regional$DLPFC_logFC
    ) ==
    sign(
        regional$HIPPO_logFC
    ),
    na.rm = TRUE
)

cat(
    "CURRENT SAVED-TABLE DLPFC-HIPPO correlation: ",
    sprintf(
        "%.10f",
        CURRENT_COR
    ),
    "\n",
    sep = ""
)

cat(
    "CURRENT SAVED-TABLE sign concordance: ",
    sprintf(
        "%.10f",
        CURRENT_SIGN
    ),
    "\n",
    sep = ""
)

# ==================================================================================================
# 12. HISTORICAL STEP11A.1 CLAIM ANCHORS
# ==================================================================================================

get_claim_value <- function(metric_name) {

    idx <- which(
        claim_numbers$metric ==
            metric_name
    )

    if (length(idx) != 1L) {

        return(
            NA_real_
        )
    }

    as.numeric(
        claim_numbers$value[idx]
    )
}

CLAIM_COR <- get_claim_value(
    "Regional logFC correlation"
)

CLAIM_SIGN <- get_claim_value(
    "Regional sign concordance"
)

COR_DIFFERENCE <- (
    CURRENT_COR -
    CLAIM_COR
)

SIGN_DIFFERENCE <- (
    CURRENT_SIGN -
    CLAIM_SIGN
)

CLAIM_COR_MATCH <- (
    is.finite(CLAIM_COR) &&
    abs(COR_DIFFERENCE) < 1e-10
)

CLAIM_SIGN_MATCH <- (
    is.finite(CLAIM_SIGN) &&
    abs(SIGN_DIFFERENCE) < 1e-10
)

CLAIM_REGISTRY_RECONCILIATION_REQUIRED <- !(
    CLAIM_COR_MATCH &&
    CLAIM_SIGN_MATCH
)

cat(
    "\nHistorical STEP11A.1 correlation: ",
    ifelse(
        is.finite(CLAIM_COR),
        sprintf("%.10f", CLAIM_COR),
        "NA"
    ),
    "\n",
    sep = ""
)

cat(
    "Historical STEP11A.1 sign concordance: ",
    ifelse(
        is.finite(CLAIM_SIGN),
        sprintf("%.10f", CLAIM_SIGN),
        "NA"
    ),
    "\n",
    sep = ""
)

cat(
    "CLAIM_REGISTRY_RECONCILIATION_REQUIRED: ",
    CLAIM_REGISTRY_RECONCILIATION_REQUIRED,
    "\n",
    sep = ""
)

# ==================================================================================================
# 13. SAVE RECONCILIATION AUDIT
# ==================================================================================================

reconciliation <- data.frame(

    metric = c(
        "DLPFC_HIPPO_logFC_correlation",
        "DLPFC_HIPPO_sign_concordance",
        "max_DLPFC_reconstruction_error",
        "max_HIPPO_reconstruction_error"
    ),

    current_saved_source_value = c(
        CURRENT_COR,
        CURRENT_SIGN,
        MAX_DLPFC_RECON_ERROR,
        MAX_HIPPO_RECON_ERROR
    ),

    historical_STEP11A1_value = c(
        CLAIM_COR,
        CLAIM_SIGN,
        NA_real_,
        NA_real_
    ),

    difference_current_minus_historical = c(
        COR_DIFFERENCE,
        SIGN_DIFFERENCE,
        NA_real_,
        NA_real_
    ),

    stringsAsFactors = FALSE
)

write.csv(
    reconciliation,
    RECON_CSV_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

recon_txt <- c(

    "==============================================================================================================",
    "FIGURE 2 — SAVED DE SOURCE RECONCILIATION",
    "==============================================================================================================",

    paste0(
        "Current saved-table DLPFC-HIPPO correlation: ",
        format(
            CURRENT_COR,
            digits = 12
        )
    ),

    paste0(
        "Historical STEP11A.1 correlation: ",
        format(
            CLAIM_COR,
            digits = 12
        )
    ),

    paste0(
        "Difference: ",
        format(
            COR_DIFFERENCE,
            digits = 12
        )
    ),

    "",

    paste0(
        "Current saved-table sign concordance: ",
        format(
            CURRENT_SIGN,
            digits = 12
        )
    ),

    paste0(
        "Historical STEP11A.1 sign concordance: ",
        format(
            CLAIM_SIGN,
            digits = 12
        )
    ),

    paste0(
        "Difference: ",
        format(
            SIGN_DIFFERENCE,
            digits = 12
        )
    ),

    "",

    paste0(
        "Maximum DLPFC shared+deviation reconstruction error: ",
        format(
            MAX_DLPFC_RECON_ERROR,
            scientific = TRUE
        )
    ),

    paste0(
        "Maximum HIPPO shared-deviation reconstruction error: ",
        format(
            MAX_HIPPO_RECON_ERROR,
            scientific = TRUE
        )
    ),

    paste0(
        "Saved DE tables algebraically consistent: ",
        DE_TABLES_ALGEBRAICALLY_CONSISTENT
    ),

    "",

    paste0(
        "CLAIM_REGISTRY_RECONCILIATION_REQUIRED: ",
        CLAIM_REGISTRY_RECONCILIATION_REQUIRED
    ),

    "",

    paste(
        "No differential-expression model was rerun.",
        "This audit compares already-saved result artifacts only."
    ),

    "=============================================================================================================="
)

writeLines(
    recon_txt,
    RECON_TXT_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 14. FDR COUNT AUDIT
# ==================================================================================================

CURRENT_FDR_COUNTS <- c(
    sum(
        dlpfc$adj.P.Val < 0.05,
        na.rm = TRUE
    ),
    sum(
        hippo$adj.P.Val < 0.05,
        na.rm = TRUE
    ),
    sum(
        shared$adj.P.Val < 0.05,
        na.rm = TRUE
    ),
    sum(
        deviation$adj.P.Val < 0.05,
        na.rm = TRUE
    )
)

names(
    CURRENT_FDR_COUNTS
) <- c(
    "DLPFC_SCZ",
    "HIPPO_SCZ",
    "Shared_SCZ",
    "Regional_Deviation"
)

if (!all(
    c(
        "contrast",
        "FDR_lt_0.05"
    ) %in%
        names(de_summary)
)) {

    stop(
        "Saved DE summary lacks contrast/FDR_lt_0.05 columns."
    )
}

SUMMARY_FDR_COUNTS <- numeric(
    length(
        CURRENT_FDR_COUNTS
    )
)

names(
    SUMMARY_FDR_COUNTS
) <- names(
    CURRENT_FDR_COUNTS
)

for (nm in names(
    CURRENT_FDR_COUNTS
)) {

    idx <- which(
        de_summary$contrast ==
            nm
    )

    if (length(idx) != 1L) {

        stop(
            "Cannot uniquely locate contrast in saved DE summary: ",
            nm
        )
    }

    SUMMARY_FDR_COUNTS[nm] <- as.numeric(
        de_summary$FDR_lt_0.05[idx]
    )
}

if (!identical(
    as.integer(
        CURRENT_FDR_COUNTS
    ),
    as.integer(
        SUMMARY_FDR_COUNTS
    )
)) {

    stop(
        "Current full DE tables disagree with saved DE summary FDR counts."
    )
}

cat(
    "Full DE tables vs saved DE summary FDR counts: PASS\n"
)

cat(
    "FDR<0.05 counts — DLPFC / HIPPO / Shared / Deviation: ",
    paste(
        CURRENT_FDR_COUNTS,
        collapse = " / "
    ),
    "\n",
    sep = ""
)

# ==================================================================================================
# 15. PANEL A CLASSIFICATION
# ==================================================================================================

dlpfc_sig <- (
    regional$DLPFC_FDR < 0.05
)

hippo_sig <- (
    regional$HIPPO_FDR < 0.05
)

regional$significance_class <- "Neither"

regional$significance_class[
    dlpfc_sig &
    !hippo_sig
] <- "DLPFC only"

regional$significance_class[
    !dlpfc_sig &
    hippo_sig
] <- "HIPPO only"

regional$significance_class[
    dlpfc_sig &
    hippo_sig
] <- "Both regions"

regional$significance_class <- factor(
    regional$significance_class,
    levels = c(
        "Neither",
        "DLPFC only",
        "HIPPO only",
        "Both regions"
    )
)

write.csv(
    regional,
    PANEL_A_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# ==================================================================================================
# 16. PANEL B DATA
# ==================================================================================================

count_df <- data.frame(

    contrast = factor(
        c(
            "DLPFC",
            "HIPPO",
            "Shared",
            "Regional\ndeviation"
        ),
        levels = c(
            "DLPFC",
            "HIPPO",
            "Shared",
            "Regional\ndeviation"
        )
    ),

    n_FDR_0_05 = as.numeric(
        CURRENT_FDR_COUNTS
    ),

    stringsAsFactors = FALSE
)

write.csv(
    count_df,
    PANEL_B_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 17. VOLCANO HELPERS
# ==================================================================================================

safe_gene_label <- function(
    symbol,
    ensembl
) {

    symbol <- as.character(
        symbol
    )

    ensembl <- as.character(
        ensembl
    )

    missing_symbol <- (
        is.na(symbol) |
        trimws(symbol) == ""
    )

    out <- symbol

    out[
        missing_symbol
    ] <- ensembl[
        missing_symbol
    ]

    out
}

prepare_volcano <- function(
    x,
    positive_name,
    negative_name
) {

    out <- x

    out$display_label <- safe_gene_label(
        out$gene_symbol,
        out$ensemblID_stable
    )

    out$FDR_plot <- pmax(
        as.numeric(
            out$adj.P.Val
        ),
        1e-300
    )

    out$minus_log10_FDR <- -log10(
        out$FDR_plot
    )

    out$DE_class <- "Not FDR-significant"

    positive_idx <- (
        out$adj.P.Val < 0.05 &
        out$logFC > 0
    )

    negative_idx <- (
        out$adj.P.Val < 0.05 &
        out$logFC < 0
    )

    out$DE_class[
        positive_idx
    ] <- positive_name

    out$DE_class[
        negative_idx
    ] <- negative_name

    out
}

shared_plot <- prepare_volcano(
    shared,
    "Positive shared effect",
    "Negative shared effect"
)

deviation_plot <- prepare_volcano(
    deviation,
    "Positive deviation",
    "Negative deviation"
)

write.csv(
    shared_plot,
    PANEL_C_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    deviation_plot,
    PANEL_D_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# ==================================================================================================
# 18. SHARED LABELS
# ==================================================================================================

shared_named <- shared_plot[
    shared_plot$adj.P.Val < 0.05 &
    !is.na(
        shared_plot$gene_symbol
    ) &
    trimws(
        shared_plot$gene_symbol
    ) != "",
    ,
    drop = FALSE
]

shared_positive <- shared_named[
    shared_named$logFC > 0,
    ,
    drop = FALSE
]

shared_negative <- shared_named[
    shared_named$logFC < 0,
    ,
    drop = FALSE
]

shared_positive <- shared_positive[
    order(
        shared_positive$adj.P.Val,
        -abs(
            shared_positive$logFC
        )
    ),
    ,
    drop = FALSE
]

shared_negative <- shared_negative[
    order(
        shared_negative$adj.P.Val,
        -abs(
            shared_negative$logFC
        )
    ),
    ,
    drop = FALSE
]

shared_labels <- rbind(
    head(
        shared_positive,
        4L
    ),
    head(
        shared_negative,
        4L
    )
)

# ==================================================================================================
# 19. DEVIATION LABELS
# ==================================================================================================

deviation_labels <- deviation_plot[
    deviation_plot$adj.P.Val < 0.05,
    ,
    drop = FALSE
]

deviation_labels <- deviation_labels[
    order(
        deviation_labels$adj.P.Val
    ),
    ,
    drop = FALSE
]

if (
    nrow(
        deviation_labels
    ) !=
        CURRENT_FDR_COUNTS[
            "Regional_Deviation"
        ]
) {

    stop(
        "Deviation-label count disagrees with current saved FDR count."
    )
}

# ==================================================================================================
# 20. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.5,
            hjust = 0,
            margin = ggplot2::margin(
                b = 5
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.3
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.1,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 8.1
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.6
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            6,
            7,
            6,
            7
        )
    )

# ==================================================================================================
# 21. PANEL A — REGIONAL CONCORDANCE
# ==================================================================================================

regional_cols <- c(
    "Neither" = "#C8C8C8",
    "DLPFC only" = "#4E79A7",
    "HIPPO only" = "#F28E2B",
    "Both regions" = "#8E63B0"
)

pA <- ggplot2::ggplot(
    regional,
    ggplot2::aes(
        x = DLPFC_logFC,
        y = HIPPO_logFC,
        colour = significance_class
    )
) +
    ggplot2::geom_hline(
        yintercept = 0,
        linewidth = 0.28,
        colour = "#AFAFAF"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.28,
        colour = "#AFAFAF"
    ) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.42,
        colour = "#555555"
    ) +
    ggplot2::geom_point(
        size = 0.78,
        alpha = 0.48
    ) +
    ggplot2::scale_colour_manual(
        values = regional_cols,
        drop = FALSE
    ) +
    ggplot2::annotate(
        "text",
        x = -Inf,
        y = Inf,
        hjust = -0.05,
        vjust = 1.15,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.6,
        label = paste0(
            "r = ",
            sprintf(
                "%.3f",
                CURRENT_COR
            ),
            "\nSign concordance = ",
            sprintf(
                "%.1f%%",
                100 * CURRENT_SIGN
            )
        )
    ) +
    ggplot2::labs(
        title = "A  Cross-region diagnosis-effect concordance",
        x = "DLPFC logFC (schizophrenia vs control)",
        y = "HIPPO logFC (schizophrenia vs control)",
        colour = "FDR < 0.05"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom"
    )

# ==================================================================================================
# 22. PANEL B — SIGNIFICANT-GENE BURDEN
# ==================================================================================================

count_cols <- c(
    "DLPFC" = "#59A14F",
    "HIPPO" = "#E15759",
    "Shared" = "#B07AA1",
    "Regional\ndeviation" = "#76B7B2"
)

pB <- ggplot2::ggplot(
    count_df,
    ggplot2::aes(
        x = contrast,
        y = n_FDR_0_05,
        colour = contrast
    )
) +
    ggplot2::geom_segment(
        ggplot2::aes(
            xend = contrast,
            y = 8,
            yend = n_FDR_0_05
        ),
        linewidth = 1.0
    ) +
    ggplot2::geom_point(
        size = 4.0
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = format(
                n_FDR_0_05,
                big.mark = ","
            )
        ),
        family = FONT_FAMILY,
        face = "bold",
        size = 2.8,
        vjust = -0.8
    ) +
    ggplot2::scale_colour_manual(
        values = count_cols,
        guide = "none"
    ) +
    ggplot2::scale_y_log10(
        limits = c(
            8,
            8000
        ),
        breaks = c(
            10,
            100,
            1000,
            5000
        ),
        labels = c(
            "10",
            "100",
            "1,000",
            "5,000"
        )
    ) +
    ggplot2::labs(
        title = "B  FDR-significant transcriptomic burden",
        x = NULL,
        y = "FDR < 0.05 genes (log scale)"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.9,
            lineheight = 0.9
        )
    )

# ==================================================================================================
# 23. PANEL C — SHARED VOLCANO
# ==================================================================================================

shared_cols <- c(
    "Not FDR-significant" = "#D2D2D2",
    "Positive shared effect" = "#D95F5F",
    "Negative shared effect" = "#3C78B5"
)

pC <- ggplot2::ggplot(
    shared_plot,
    ggplot2::aes(
        x = logFC,
        y = minus_log10_FDR,
        colour = DE_class
    )
) +
    ggplot2::geom_hline(
        yintercept = -log10(0.05),
        linetype = "dashed",
        linewidth = 0.42,
        colour = "#555555"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.28,
        colour = "#999999"
    ) +
    ggplot2::geom_point(
        size = 0.70,
        alpha = 0.52
    ) +
    ggplot2::scale_colour_manual(
        values = shared_cols,
        drop = FALSE
    ) +
    ggplot2::labs(
        title = "C  Shared schizophrenia effect",
        x = "Shared-component logFC",
        y = expression(-log[10](FDR)),
        colour = NULL
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom"
    )

if (
    nrow(shared_labels) > 0L &&
    USE_GGREPEL
) {

    pC <- pC +
        ggrepel::geom_text_repel(
            data = shared_labels,
            ggplot2::aes(
                label = display_label
            ),
            family = FONT_FAMILY,
            size = 2.25,
            box.padding = 0.25,
            point.padding = 0.12,
            segment.size = 0.22,
            max.overlaps = Inf,
            seed = 2026,
            show.legend = FALSE
        )

} else if (nrow(shared_labels) > 0L) {

    pC <- pC +
        ggplot2::geom_text(
            data = shared_labels,
            ggplot2::aes(
                label = display_label
            ),
            family = FONT_FAMILY,
            size = 2.2,
            vjust = -0.55,
            check_overlap = TRUE,
            show.legend = FALSE
        )
}

# ==================================================================================================
# 24. PANEL D — REGIONAL-DEVIATION VOLCANO
# ==================================================================================================

deviation_cols <- c(
    "Not FDR-significant" = "#D4D4D4",
    "Positive deviation" = "#009E73",
    "Negative deviation" = "#CC79A7"
)

pD <- ggplot2::ggplot(
    deviation_plot,
    ggplot2::aes(
        x = logFC,
        y = minus_log10_FDR,
        colour = DE_class
    )
) +
    ggplot2::geom_hline(
        yintercept = -log10(0.05),
        linetype = "dashed",
        linewidth = 0.42,
        colour = "#555555"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.28,
        colour = "#999999"
    ) +
    ggplot2::geom_point(
        size = 0.72,
        alpha = 0.55
    ) +
    ggplot2::scale_colour_manual(
        values = deviation_cols,
        drop = FALSE
    ) +
    ggplot2::labs(
        title = "D  Regional-deviation contrast",
        x = "Regional-deviation logFC",
        y = expression(-log[10](FDR)),
        colour = NULL
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom"
    )

if (
    nrow(deviation_labels) > 0L &&
    USE_GGREPEL
) {

    pD <- pD +
        ggrepel::geom_text_repel(
            data = deviation_labels,
            ggplot2::aes(
                label = display_label
            ),
            family = FONT_FAMILY,
            size = 2.15,
            box.padding = 0.26,
            point.padding = 0.12,
            segment.size = 0.22,
            max.overlaps = Inf,
            seed = 2026,
            show.legend = FALSE
        )

} else if (nrow(deviation_labels) > 0L) {

    pD <- pD +
        ggplot2::geom_text(
            data = deviation_labels,
            ggplot2::aes(
                label = display_label
            ),
            family = FONT_FAMILY,
            size = 2.1,
            vjust = -0.55,
            check_overlap = TRUE,
            show.legend = FALSE
        )
}

# ==================================================================================================
# 25. DRAW FIGURE
# ==================================================================================================

draw_figure2 <- function() {

    grid::grid.newpage()

    grid::pushViewport(
        grid::viewport(
            layout = grid::grid.layout(
                nrow = 2,
                ncol = 2,
                widths = grid::unit(
                    c(1, 1),
                    "null"
                ),
                heights = grid::unit(
                    c(1, 1),
                    "null"
                )
            )
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pD,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 26. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 6.2,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure2()

grDevices::dev.off()

# ==================================================================================================
# 27. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 6.2,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure2()

grDevices::dev.off()

# ==================================================================================================
# 28. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 6.2,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure2()

grDevices::dev.off()

# ==================================================================================================
# 29. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    PANEL_A_PATH,
    PANEL_B_PATH,
    PANEL_C_PATH,
    PANEL_D_PATH,
    RECON_CSV_PATH,
    RECON_TXT_PATH
)

if (!all(
    file.exists(outputs)
)) {

    stop(
        "One or more Figure 2 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(output_info$size) |
    output_info$size <= 0
)) {

    stop(
        "One or more Figure 2 outputs are zero-byte."
    )
}

cat(
    "Figure 2 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 30. SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11C.3R — MAIN FIGURE 2 SUMMARY",
    "==============================================================================================================",

    paste0(
        "Current saved-table DLPFC-HIPPO correlation: ",
        sprintf(
            "%.10f",
            CURRENT_COR
        )
    ),

    paste0(
        "Historical STEP11A.1 correlation: ",
        sprintf(
            "%.10f",
            CLAIM_COR
        )
    ),

    paste0(
        "Current saved-table sign concordance: ",
        sprintf(
            "%.10f",
            CURRENT_SIGN
        )
    ),

    paste0(
        "Historical STEP11A.1 sign concordance: ",
        sprintf(
            "%.10f",
            CLAIM_SIGN
        )
    ),

    paste0(
        "CLAIM_REGISTRY_RECONCILIATION_REQUIRED: ",
        CLAIM_REGISTRY_RECONCILIATION_REQUIRED
    ),

    "",

    paste0(
        "FDR counts — DLPFC / HIPPO / Shared / Deviation: ",
        paste(
            CURRENT_FDR_COUNTS,
            collapse = " / "
        )
    ),

    "",

    paste0(
        "Max DLPFC reconstruction error: ",
        format(
            MAX_DLPFC_RECON_ERROR,
            scientific = TRUE
        )
    ),

    paste0(
        "Max HIPPO reconstruction error: ",
        format(
            MAX_HIPPO_RECON_ERROR,
            scientific = TRUE
        )
    ),

    "",

    "Figure generated exclusively from current saved DE tables.",

    "No limma, voom, duplicateCorrelation, normalization, or DE fitting was rerun.",

    "",

    paste0(
        "PDF: ",
        PDF_PATH
    ),

    paste0(
        "TIFF: ",
        TIFF_PATH
    ),

    paste0(
        "PNG: ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 31. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    if (file.exists(MODEL_AUDIT_PATH)) MODEL_AUDIT_PATH else character(0),
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 32. FINAL CONSOLE REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFIGURE 2 — SOURCE RECONCILIATION + GENERATION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Saved DE tables algebraically consistent: ",
    DE_TABLES_ALGEBRAICALLY_CONSISTENT,
    "\n",
    sep = ""
)

cat(
    "Current saved-table correlation: ",
    sprintf(
        "%.10f",
        CURRENT_COR
    ),
    "\n",
    sep = ""
)

cat(
    "Historical STEP11A.1 correlation: ",
    sprintf(
        "%.10f",
        CLAIM_COR
    ),
    "\n",
    sep = ""
)

cat(
    "Current saved-table sign concordance: ",
    sprintf(
        "%.10f",
        CURRENT_SIGN
    ),
    "\n",
    sep = ""
)

cat(
    "Historical STEP11A.1 sign concordance: ",
    sprintf(
        "%.10f",
        CLAIM_SIGN
    ),
    "\n",
    sep = ""
)

cat(
    "\nCLAIM_REGISTRY_RECONCILIATION_REQUIRED: ",
    CLAIM_REGISTRY_RECONCILIATION_REQUIRED,
    "\n",
    sep = ""
)

cat(
    "\nFDR counts — DLPFC / HIPPO / Shared / Deviation: ",
    paste(
        CURRENT_FDR_COUNTS,
        collapse = " / "
    ),
    "\n",
    sep = ""
)

cat(
    "Shared genes labelled: ",
    nrow(shared_labels),
    "\n",
    sep = ""
)

cat(
    "Deviation genes labelled: ",
    nrow(deviation_labels),
    "\n",
    sep = ""
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\nReconciliation audit:\n",
    RECON_TXT_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3R COMPLETE — FIGURE 2 GENERATED FROM RECONCILED SAVED DE SOURCES\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.3R — MAIN FIGURE 2 — SOURCE-RECONCILED CORRECTED VERSION
SHARED AND REGION-DEPENDENT SCHIZOPHRENIA TRANSCRIPTOMIC ARCHITECTURE
Project sentinel: PASS
Figure font: Times New Roman
Required saved DE artifacts: PASS
ggrepel available: TRUE
Four DE tables: 23,740 unique genes each — PASS
Stable-Ensembl alignment across four DE tables: PASS
Max DLPFC shared+deviation reconstruction error: 3.9968e-15
Max HIPPO shared-deviation reconstruction error: 2.2204e-15
Saved DE contrast algebra: PASS
CURRENT SAVED-TABLE DLPFC-HIPPO correlation: 0.6183141559
CURRENT SAVED-TABLE sign concordance: 0.7380791912

Historical STEP11A.1 correlation: 0.6089000000
Historical STEP11A.1 sign concordance: 0.7350000000
CLAIM_REGISTRY_RECONCILIATION_REQUIRED: TRUE
Full DE tables vs saved DE summary FDR counts: PASS
FDR<0.05 counts — DLPFC / HIPPO / Shared / Deviation: 3014 / 2643 / 4970 / 28


Warning message in ggplot2::geom_text(ggplot2::aes(label = format(n_FDR_0_05, big.mark = ",")), :
"Ignoring unknown parameters: `face`"


agg_record_211ca321ca1 
                     2

agg_record_211ca321ca1 
                     2

agg_record_211ca321ca1 
                     2

Figure 2 outputs created and non-zero: PASS

FIGURE 2 — SOURCE RECONCILIATION + GENERATION SUMMARY
Saved DE tables algebraically consistent: TRUE
Current saved-table correlation: 0.6183141559
Historical STEP11A.1 correlation: 0.6089000000
Current saved-table sign concordance: 0.7380791912
Historical STEP11A.1 sign concordance: 0.7350000000

CLAIM_REGISTRY_RECONCILIATION_REQUIRED: TRUE

FDR counts — DLPFC / HIPPO / Shared / Deviation: 3014 / 2643 / 4970 / 28
Shared genes labelled: 8
Deviation genes labelled: 28

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure2_reconciled_v2/Figure2_Shared_Regional_DE_Architecture_reconciled_v2.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure2_reconciled_v2/Figure2_Shared_Regional_DE_Architecture_reconciled_v2.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure2_reconciled_v2/Figure2_Shared_Regional_DE_Architecture_reconciled_v2.png

Reconciliation audit:
E:/SCZ_CROSSREGION_MSS_Q1/

In [89]:
# ==================================================================================================
# STEP 11C.3P — STEP03A DE PROVENANCE / OVERWRITE AUDIT
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Resolve the conflict between:
#
#   CANONICAL EXECUTED NOTEBOOK:
#       correlation        = 0.6089
#       sign concordance   = 0.735
#       FDR counts         = 2949 / 2648 / 5348 / 11
#
#   CURRENT DISK FILES:
#       correlation        ~= 0.618314
#       sign concordance   ~= 0.738079
#       FDR counts         = 3014 / 2643 / 4970 / 28
#
# THIS CELL:
#   - DOES NOT rerun limma/voom/duplicateCorrelation
#   - DOES NOT modify any existing DE file
#   - DOES NOT modify the claim registry
#   - DOES NOT replace Figure 2
#
# IT WILL:
#   1. hash and timestamp current STEP03A files
#   2. inspect current step03a_model_audit_v1.txt
#   3. scan the project for alternate/copy STEP03A artifacts
#   4. identify candidate DE sets
#   5. compare candidate sets against the canonical executed-notebook anchors
#   6. scan manifest-like files for STEP03A hash/path references
#   7. save a provenance report
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3P — STEP03A DE PROVENANCE / OVERWRITE AUDIT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ", EXPECTED_PROJECT_ID,
        "\nObserved: ", project_id
    )
}

cat("Project sentinel: PASS\n")

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest package unavailable.")
}

# ==================================================================================================
# 2. CANONICAL EXECUTED-NOTEBOOK ANCHORS
# ==================================================================================================

CANONICAL_COR <- 0.6089
CANONICAL_SIGN <- 0.735

CANONICAL_COUNTS <- c(
    DLPFC_SCZ = 2949L,
    HIPPO_SCZ = 2648L,
    Shared_SCZ = 5348L,
    Regional_Deviation = 11L
)

cat(
    "Canonical executed-notebook anchors loaded: PASS\n"
)

# ==================================================================================================
# 3. CURRENT STEP03A FILES
# ==================================================================================================

DE_DIR <- file.path(
    PROJECT_ROOT,
    "05_differential_expression"
)

CURRENT_PATHS <- c(
    DLPFC = file.path(
        DE_DIR,
        "step03a_DLPFC_SCZ_vs_Control_v1.csv"
    ),
    HIPPO = file.path(
        DE_DIR,
        "step03a_HIPPO_SCZ_vs_Control_v1.csv"
    ),
    Shared = file.path(
        DE_DIR,
        "step03a_Shared_SCZ_effect_v1.csv"
    ),
    Deviation = file.path(
        DE_DIR,
        "step03a_Regional_Deviation_v1.csv"
    ),
    Summary = file.path(
        DE_DIR,
        "step03a_DE_summary_v1.csv"
    ),
    ModelAudit = file.path(
        DE_DIR,
        "step03a_model_audit_v1.txt"
    )
)

missing_current <- CURRENT_PATHS[
    !file.exists(CURRENT_PATHS)
]

if (length(missing_current) > 0L) {
    stop(
        "Current STEP03A artifact(s) missing:\n",
        paste(
            missing_current,
            collapse = "\n"
        )
    )
}

cat(
    "Current STEP03A file set present: PASS\n"
)

# ==================================================================================================
# 4. OUTPUT DIRECTORY
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "de_provenance_audit_v1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

CURRENT_INVENTORY_PATH <- file.path(
    OUT_DIR,
    "STEP11C3P_current_STEP03A_file_inventory_v1.csv"
)

CANDIDATE_FILES_PATH <- file.path(
    OUT_DIR,
    "STEP11C3P_all_STEP03A_candidate_files_v1.csv"
)

CANDIDATE_SETS_PATH <- file.path(
    OUT_DIR,
    "STEP11C3P_candidate_DE_sets_v1.csv"
)

MANIFEST_HITS_PATH <- file.path(
    OUT_DIR,
    "STEP11C3P_manifest_STEP03A_references_v1.csv"
)

MODEL_AUDIT_COPY_PATH <- file.path(
    OUT_DIR,
    "STEP11C3P_current_model_audit_snapshot_v1.txt"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "STEP11C3P_DE_provenance_audit_summary_v1.txt"
)

# ==================================================================================================
# 5. CURRENT FILE INVENTORY / HASHES
# ==================================================================================================

current_info <- file.info(
    CURRENT_PATHS
)

current_inventory <- data.frame(
    artifact = names(CURRENT_PATHS),
    path = unname(CURRENT_PATHS),
    bytes = as.numeric(current_info$size),
    modified_time = as.character(current_info$mtime),
    sha256 = vapply(
        unname(CURRENT_PATHS),
        function(p) {
            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    current_inventory,
    CURRENT_INVENTORY_PATH,
    row.names = FALSE,
    quote = TRUE
)

cat(
    "\nCURRENT STEP03A FILE INVENTORY\n"
)

print(
    current_inventory,
    row.names = FALSE
)

# ==================================================================================================
# 6. CURRENT MODEL AUDIT CONTENT
# ==================================================================================================

model_audit_lines <- readLines(
    CURRENT_PATHS["ModelAudit"],
    warn = FALSE
)

writeLines(
    model_audit_lines,
    MODEL_AUDIT_COPY_PATH,
    useBytes = TRUE
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nCURRENT step03a_model_audit_v1.txt — RELEVANT LINES\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

audit_keep <- grepl(
    paste(
        c(
            "correlation",
            "concordance",
            "donor",
            "duplicate",
            "500",
            "23740",
            "DLPFC",
            "HIPPO"
        ),
        collapse = "|"
    ),
    model_audit_lines,
    ignore.case = TRUE
)

if (any(audit_keep)) {
    cat(
        paste(
            model_audit_lines[audit_keep],
            collapse = "\n"
        ),
        "\n"
    )
} else {
    cat(
        "No matching audit lines found.\n"
    )
}

# ==================================================================================================
# 7. FUNCTION TO CHARACTERIZE A COMPLETE DE SET
# ==================================================================================================

characterize_DE_set <- function(
    dlpfc_path,
    hippo_path,
    shared_path,
    deviation_path,
    summary_path = NA_character_
) {

    out <- tryCatch(
        {

            a <- read.csv(
                dlpfc_path,
                stringsAsFactors = FALSE,
                check.names = FALSE
            )

            b <- read.csv(
                hippo_path,
                stringsAsFactors = FALSE,
                check.names = FALSE
            )

            s <- read.csv(
                shared_path,
                stringsAsFactors = FALSE,
                check.names = FALSE
            )

            d <- read.csv(
                deviation_path,
                stringsAsFactors = FALSE,
                check.names = FALSE
            )

            required_cols <- c(
                "ensemblID_stable",
                "logFC",
                "adj.P.Val"
            )

            if (
                !all(required_cols %in% names(a)) ||
                !all(required_cols %in% names(b)) ||
                !all(required_cols %in% names(s)) ||
                !all(required_cols %in% names(d))
            ) {
                stop(
                    "Required DE columns absent."
                )
            }

            if (
                nrow(a) != 23740L ||
                nrow(b) != 23740L ||
                nrow(s) != 23740L ||
                nrow(d) != 23740L
            ) {
                stop(
                    "Unexpected row count."
                )
            }

            ib <- match(
                a$ensemblID_stable,
                b$ensemblID_stable
            )

            is <- match(
                a$ensemblID_stable,
                s$ensemblID_stable
            )

            id <- match(
                a$ensemblID_stable,
                d$ensemblID_stable
            )

            if (
                anyNA(ib) ||
                anyNA(is) ||
                anyNA(id)
            ) {
                stop(
                    "Stable-ID alignment failure."
                )
            }

            a_logfc <- as.numeric(
                a$logFC
            )

            b_logfc <- as.numeric(
                b$logFC[ib]
            )

            s_logfc <- as.numeric(
                s$logFC[is]
            )

            d_logfc <- as.numeric(
                d$logFC[id]
            )

            current_cor <- cor(
                a_logfc,
                b_logfc,
                use = "complete.obs",
                method = "pearson"
            )

            current_sign <- mean(
                sign(a_logfc) ==
                    sign(b_logfc),
                na.rm = TRUE
            )

            counts <- c(
                DLPFC_SCZ = sum(
                    a$adj.P.Val < 0.05,
                    na.rm = TRUE
                ),
                HIPPO_SCZ = sum(
                    b$adj.P.Val < 0.05,
                    na.rm = TRUE
                ),
                Shared_SCZ = sum(
                    s$adj.P.Val < 0.05,
                    na.rm = TRUE
                ),
                Regional_Deviation = sum(
                    d$adj.P.Val < 0.05,
                    na.rm = TRUE
                )
            )

            max_a_recon <- max(
                abs(
                    a_logfc -
                    (
                        s_logfc +
                        d_logfc
                    )
                ),
                na.rm = TRUE
            )

            max_b_recon <- max(
                abs(
                    b_logfc -
                    (
                        s_logfc -
                        d_logfc
                    )
                ),
                na.rm = TRUE
            )

            canonical_count_match <- identical(
                as.integer(counts),
                as.integer(CANONICAL_COUNTS)
            )

            canonical_cor_match <- (
                abs(
                    current_cor -
                    CANONICAL_COR
                ) <
                    0.00005
            )

            canonical_sign_match <- (
                abs(
                    current_sign -
                    CANONICAL_SIGN
                ) <
                    0.00005
            )

            data.frame(
                valid_set = TRUE,
                correlation = current_cor,
                sign_concordance = current_sign,
                DLPFC_FDR = counts["DLPFC_SCZ"],
                HIPPO_FDR = counts["HIPPO_SCZ"],
                Shared_FDR = counts["Shared_SCZ"],
                Deviation_FDR = counts["Regional_Deviation"],
                max_DLPFC_reconstruction_error = max_a_recon,
                max_HIPPO_reconstruction_error = max_b_recon,
                canonical_count_match = canonical_count_match,
                canonical_correlation_match = canonical_cor_match,
                canonical_sign_match = canonical_sign_match,
                full_canonical_match = (
                    canonical_count_match &&
                    canonical_cor_match &&
                    canonical_sign_match
                ),
                error = "",
                stringsAsFactors = FALSE
            )
        },

        error = function(e) {

            data.frame(
                valid_set = FALSE,
                correlation = NA_real_,
                sign_concordance = NA_real_,
                DLPFC_FDR = NA_integer_,
                HIPPO_FDR = NA_integer_,
                Shared_FDR = NA_integer_,
                Deviation_FDR = NA_integer_,
                max_DLPFC_reconstruction_error = NA_real_,
                max_HIPPO_reconstruction_error = NA_real_,
                canonical_count_match = FALSE,
                canonical_correlation_match = FALSE,
                canonical_sign_match = FALSE,
                full_canonical_match = FALSE,
                error = conditionMessage(e),
                stringsAsFactors = FALSE
            )
        }
    )

    out
}

# ==================================================================================================
# 8. CHARACTERIZE CURRENT SET
# ==================================================================================================

CURRENT_SET <- characterize_DE_set(
    CURRENT_PATHS["DLPFC"],
    CURRENT_PATHS["HIPPO"],
    CURRENT_PATHS["Shared"],
    CURRENT_PATHS["Deviation"],
    CURRENT_PATHS["Summary"]
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nCURRENT DISK DE SET\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    CURRENT_SET,
    row.names = FALSE
)

# ==================================================================================================
# 9. SCAN PROJECT FOR STEP03A-LIKE FILES
# ==================================================================================================

all_project_files <- list.files(
    PROJECT_ROOT,
    recursive = TRUE,
    full.names = TRUE,
    all.files = TRUE,
    no.. = TRUE
)

all_project_files <- all_project_files[
    file.exists(all_project_files) &
    !dir.exists(all_project_files)
]

basenames <- basename(
    all_project_files
)

step03a_like <- grepl(
    "step03a|STEP03A",
    basenames,
    ignore.case = TRUE
)

candidate_files <- all_project_files[
    step03a_like
]

candidate_info <- if (
    length(candidate_files) > 0L
) {

    fi <- file.info(
        candidate_files
    )

    data.frame(
        path = candidate_files,
        basename = basename(candidate_files),
        directory = dirname(candidate_files),
        bytes = as.numeric(fi$size),
        modified_time = as.character(fi$mtime),
        extension = tolower(
            tools::file_ext(
                candidate_files
            )
        ),
        stringsAsFactors = FALSE
    )

} else {

    data.frame(
        path = character(0),
        basename = character(0),
        directory = character(0),
        bytes = numeric(0),
        modified_time = character(0),
        extension = character(0),
        stringsAsFactors = FALSE
    )
}

if (nrow(candidate_info) > 0L) {

    candidate_info$sha256 <- vapply(
        candidate_info$path,
        function(p) {

            if (
                file.info(p)$size >
                    100 * 1024^2
            ) {
                return(
                    "SKIPPED_GT_100MB"
                )
            }

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    )
}

write.csv(
    candidate_info,
    CANDIDATE_FILES_PATH,
    row.names = FALSE,
    quote = TRUE
)

cat(
    "\nSTEP03A-like project files found: ",
    nrow(candidate_info),
    "\n",
    sep = ""
)

# ==================================================================================================
# 10. IDENTIFY DIRECTORIES CONTAINING POSSIBLE COMPLETE DE SETS
# ==================================================================================================

candidate_dirs <- unique(
    candidate_info$directory
)

set_results <- list()
set_index <- 1L

for (dir_i in candidate_dirs) {

    files_here <- list.files(
        dir_i,
        full.names = TRUE
    )

    get_one <- function(pattern) {

        z <- files_here[
            grepl(
                pattern,
                basename(files_here),
                ignore.case = TRUE,
                perl = TRUE
            )
        ]

        if (length(z) == 1L) {
            return(z)
        }

        NA_character_
    }

    a <- get_one(
        "^step03a_DLPFC_SCZ_vs_Control.*\\.csv$"
    )

    b <- get_one(
        "^step03a_HIPPO_SCZ_vs_Control.*\\.csv$"
    )

    s <- get_one(
        "^step03a_Shared_SCZ_effect.*\\.csv$"
    )

    d <- get_one(
        "^step03a_Regional_Deviation.*\\.csv$"
    )

    summ <- get_one(
        "^step03a_DE_summary.*\\.csv$"
    )

    if (
        all(
            !is.na(
                c(
                    a,
                    b,
                    s,
                    d
                )
            )
        )
    ) {

        result <- characterize_DE_set(
            a,
            b,
            s,
            d,
            summ
        )

        result$directory <- dir_i
        result$DLPFC_path <- a
        result$HIPPO_path <- b
        result$Shared_path <- s
        result$Deviation_path <- d
        result$Summary_path <- summ

        set_results[[set_index]] <- result

        set_index <- set_index + 1L
    }
}

candidate_sets <- if (
    length(set_results) > 0L
) {

    do.call(
        rbind,
        set_results
    )

} else {

    data.frame(
        valid_set = logical(0),
        correlation = numeric(0),
        sign_concordance = numeric(0),
        DLPFC_FDR = integer(0),
        HIPPO_FDR = integer(0),
        Shared_FDR = integer(0),
        Deviation_FDR = integer(0),
        canonical_count_match = logical(0),
        canonical_correlation_match = logical(0),
        canonical_sign_match = logical(0),
        full_canonical_match = logical(0),
        directory = character(0),
        stringsAsFactors = FALSE
    )
}

write.csv(
    candidate_sets,
    CANDIDATE_SETS_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

cat(
    "Complete candidate DE sets found: ",
    nrow(candidate_sets),
    "\n",
    sep = ""
)

if (nrow(candidate_sets) > 0L) {

    cat(
        "\nCANDIDATE SET SUMMARY\n"
    )

    print(
        candidate_sets[
            ,
            intersect(
                c(
                    "directory",
                    "correlation",
                    "sign_concordance",
                    "DLPFC_FDR",
                    "HIPPO_FDR",
                    "Shared_FDR",
                    "Deviation_FDR",
                    "full_canonical_match"
                ),
                names(candidate_sets)
            ),
            drop = FALSE
        ],
        row.names = FALSE
    )
}

# ==================================================================================================
# 11. SEARCH MANIFEST-LIKE FILES FOR STEP03A REFERENCES
# ==================================================================================================

manifest_candidates <- all_project_files[
    grepl(
        "manifest|registry|hash|sha256|inventory",
        basename(all_project_files),
        ignore.case = TRUE
    ) &
    tolower(
        tools::file_ext(
            all_project_files
        )
    ) %in%
        c(
            "csv",
            "txt",
            "json",
            "tsv"
        )
]

manifest_hits <- list()
hit_index <- 1L

for (p in manifest_candidates) {

    file_size <- file.info(p)$size

    if (
        is.na(file_size) ||
        file_size <= 0 ||
        file_size > 20 * 1024^2
    ) {
        next
    }

    lines <- tryCatch(
        readLines(
            p,
            warn = FALSE,
            encoding = "UTF-8"
        ),
        error = function(e) {
            character(0)
        }
    )

    if (length(lines) == 0L) {
        next
    }

    hit <- grep(
        "step03a_(DLPFC|HIPPO|Shared|Regional|DE_summary|model_audit)",
        lines,
        ignore.case = TRUE,
        perl = TRUE
    )

    if (length(hit) > 0L) {

        for (h in hit) {

            manifest_hits[[hit_index]] <- data.frame(
                manifest_path = p,
                line_number = h,
                text = lines[h],
                stringsAsFactors = FALSE
            )

            hit_index <- hit_index + 1L
        }
    }
}

manifest_hit_table <- if (
    length(manifest_hits) > 0L
) {

    do.call(
        rbind,
        manifest_hits
    )

} else {

    data.frame(
        manifest_path = character(0),
        line_number = integer(0),
        text = character(0),
        stringsAsFactors = FALSE
    )
}

write.csv(
    manifest_hit_table,
    MANIFEST_HITS_PATH,
    row.names = FALSE,
    quote = TRUE
)

cat(
    "\nManifest/registry STEP03A references found: ",
    nrow(manifest_hit_table),
    "\n",
    sep = ""
)

# ==================================================================================================
# 12. DETERMINE PROVENANCE STATUS
# ==================================================================================================

CANONICAL_BACKUP_FOUND <- (
    nrow(candidate_sets) > 0L &&
    any(
        candidate_sets$full_canonical_match,
        na.rm = TRUE
    )
)

CURRENT_MATCHES_CANONICAL <- isTRUE(
    CURRENT_SET$full_canonical_match[1]
)

CURRENT_MODEL_AUDIT_TEXT <- paste(
    model_audit_lines,
    collapse = "\n"
)

MODEL_AUDIT_HAS_CANONICAL_COR <- grepl(
    "0\\.6089",
    CURRENT_MODEL_AUDIT_TEXT
)

MODEL_AUDIT_HAS_CURRENT_COR <- grepl(
    "0\\.6183",
    CURRENT_MODEL_AUDIT_TEXT
)

if (
    !CURRENT_MATCHES_CANONICAL &&
    MODEL_AUDIT_HAS_CANONICAL_COR
) {

    PROVENANCE_STATUS <-
        "CURRENT_DE_CSVS_CONFLICT_WITH_CANONICAL_MODEL_AUDIT_LIKELY_PARTIAL_OVERWRITE"

} else if (
    !CURRENT_MATCHES_CANONICAL &&
    MODEL_AUDIT_HAS_CURRENT_COR
) {

    PROVENANCE_STATUS <-
        "CURRENT_DE_SET_IS_COHERENT_ALTERNATIVE_RUN_FULL_OVERWRITE_POSSIBLE"

} else if (
    CURRENT_MATCHES_CANONICAL
) {

    PROVENANCE_STATUS <-
        "CURRENT_DE_SET_MATCHES_CANONICAL"

} else {

    PROVENANCE_STATUS <-
        "DE_PROVENANCE_CONFLICT_REQUIRES_MANUAL_REVIEW"
}

# ==================================================================================================
# 13. SUMMARY REPORT
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11C.3P — STEP03A DE PROVENANCE / OVERWRITE AUDIT",
    "==============================================================================================================",

    "",

    "CANONICAL EXECUTED-NOTEBOOK ANCHORS",

    paste0(
        "Correlation: ",
        CANONICAL_COR
    ),

    paste0(
        "Sign concordance: ",
        CANONICAL_SIGN
    ),

    paste0(
        "FDR counts DLPFC/HIPPO/Shared/Deviation: ",
        paste(
            CANONICAL_COUNTS,
            collapse = " / "
        )
    ),

    "",

    "CURRENT DISK DE SET",

    paste0(
        "Correlation: ",
        format(
            CURRENT_SET$correlation[1],
            digits = 12
        )
    ),

    paste0(
        "Sign concordance: ",
        format(
            CURRENT_SET$sign_concordance[1],
            digits = 12
        )
    ),

    paste0(
        "FDR counts DLPFC/HIPPO/Shared/Deviation: ",
        paste(
            c(
                CURRENT_SET$DLPFC_FDR[1],
                CURRENT_SET$HIPPO_FDR[1],
                CURRENT_SET$Shared_FDR[1],
                CURRENT_SET$Deviation_FDR[1]
            ),
            collapse = " / "
        )
    ),

    paste0(
        "Current disk set matches canonical: ",
        CURRENT_MATCHES_CANONICAL
    ),

    "",

    paste0(
        "Current model audit contains canonical 0.6089: ",
        MODEL_AUDIT_HAS_CANONICAL_COR
    ),

    paste0(
        "Current model audit contains current 0.6183: ",
        MODEL_AUDIT_HAS_CURRENT_COR
    ),

    "",

    paste0(
        "Complete alternate DE sets located: ",
        nrow(candidate_sets)
    ),

    paste0(
        "Canonical-matching backup set found: ",
        CANONICAL_BACKUP_FOUND
    ),

    paste0(
        "Manifest/registry references found: ",
        nrow(manifest_hit_table)
    ),

    "",

    paste0(
        "PROVENANCE_STATUS: ",
        PROVENANCE_STATUS
    ),

    "",

    "DECISION RULE",

    paste(
        "Do NOT amend STEP11A.1 and do NOT treat Figure 2 v2 as final",
        "until the provenance conflict is resolved."
    ),

    paste(
        "No DE model was rerun in this audit."
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 14. CONSOLE REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP03A PROVENANCE DECISION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "CURRENT_MATCHES_CANONICAL: ",
    CURRENT_MATCHES_CANONICAL,
    "\n",
    sep = ""
)

cat(
    "MODEL_AUDIT_HAS_CANONICAL_COR: ",
    MODEL_AUDIT_HAS_CANONICAL_COR,
    "\n",
    sep = ""
)

cat(
    "MODEL_AUDIT_HAS_CURRENT_COR: ",
    MODEL_AUDIT_HAS_CURRENT_COR,
    "\n",
    sep = ""
)

cat(
    "CANONICAL_BACKUP_FOUND: ",
    CANONICAL_BACKUP_FOUND,
    "\n",
    sep = ""
)

cat(
    "COMPLETE_CANDIDATE_SETS_FOUND: ",
    nrow(candidate_sets),
    "\n",
    sep = ""
)

cat(
    "MANIFEST_REFERENCES_FOUND: ",
    nrow(manifest_hit_table),
    "\n",
    sep = ""
)

cat(
    "\nPROVENANCE_STATUS: ",
    PROVENANCE_STATUS,
    "\n",
    sep = ""
)

if (
    CANONICAL_BACKUP_FOUND
) {

    cat(
        "\nCANONICAL-MATCHING SET(S):\n"
    )

    print(
        candidate_sets[
            candidate_sets$full_canonical_match,
            ,
            drop = FALSE
        ],
        row.names = FALSE
    )
}

cat(
    "\nCurrent-file inventory:\n",
    CURRENT_INVENTORY_PATH,
    "\n",
    sep = ""
)

cat(
    "\nCandidate DE sets:\n",
    CANDIDATE_SETS_PATH,
    "\n",
    sep = ""
)

cat(
    "\nManifest references:\n",
    MANIFEST_HITS_PATH,
    "\n",
    sep = ""
)

cat(
    "\nAudit summary:\n",
    SUMMARY_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3P COMPLETE — NO DE ANALYSIS RERUN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.3P — STEP03A DE PROVENANCE / OVERWRITE AUDIT
Project sentinel: PASS
Canonical executed-notebook anchors loaded: PASS
Current STEP03A file set present: PASS

CURRENT STEP03A FILE INVENTORY
   artifact
      DLPFC
      HIPPO
     Shared
  Deviation
    Summary
 ModelAudit
                                                                                     path
 E:/SCZ_CROSSREGION_MSS_Q1/05_differential_expression/step03a_DLPFC_SCZ_vs_Control_v1.csv
 E:/SCZ_CROSSREGION_MSS_Q1/05_differential_expression/step03a_HIPPO_SCZ_vs_Control_v1.csv
    E:/SCZ_CROSSREGION_MSS_Q1/05_differential_expression/step03a_Shared_SCZ_effect_v1.csv
   E:/SCZ_CROSSREGION_MSS_Q1/05_differential_expression/step03a_Regional_Deviation_v1.csv
           E:/SCZ_CROSSREGION_MSS_Q1/05_differential_expression/step03a_DE_summary_v1.csv
          E:/SCZ_CROSSREGION_MSS_Q1/05_differential_expression/step03a_model_audit_v1.txt
   bytes              modified_time
 4393052 2026-09-30 06:49:20.013121
 4391957 2026-09

In [90]:
# ==================================================================================================
# STEP 11C.3Q — CANONICAL STEP03A DE RECOVERY
# RECOVER THE FROZEN 500-GENE CORRELATION VERSION WITHOUT TOUCHING CURRENT OVERWRITTEN FILES
# --------------------------------------------------------------------------------------------------
# PURPOSE
#
# Recover the canonical STEP03A result documented in the executed notebook:
#
#   Correlation estimation:
#       500 outcome-blind variance-stratified genes
#       paired donors only
#       seed = 20260929
#       trim = 0.15
#
#   Expected canonical donor correlation:
#       ~0.188518
#
#   Expected canonical DE:
#       DLPFC FDR<0.05            = 2949
#       HIPPO FDR<0.05            = 2648
#       Shared FDR<0.05           = 5348
#       Regional deviation        = 11
#
#   Expected cross-region:
#       correlation               = 0.6089
#       sign concordance          = 0.7350
#
# SAFETY
#   - DOES NOT run duplicateCorrelation on all 23,740 genes.
#   - DOES NOT overwrite current 05_differential_expression/step03a_* files.
#   - DOES NOT modify claim registry.
#   - DOES NOT modify any model/stability/external-validation artifact.
#   - Writes only to:
#         05_differential_expression/canonical_recovery_v1/
#
# NOTE
#   Final voom + full limma fit on 23,740 genes is required to reconstruct the lost canonical
#   result tables. The expensive full-gene duplicateCorrelation is explicitly prohibited here.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3Q — CANONICAL STEP03A DE RECOVERY\n",
    "500-GENE CORRELATION VERSION — CURRENT FILES WILL NOT BE OVERWRITTEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

TOTAL_START <- proc.time()[3]

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. REQUIRED PACKAGES
# ==================================================================================================

required_packages <- c(
    "edgeR",
    "limma",
    "digest"
)

for (pkg in required_packages) {

    if (!requireNamespace(
        pkg,
        quietly = TRUE
    )) {
        stop(
            "Required frozen-environment package unavailable: ",
            pkg
        )
    }
}

cat(
    "edgeR version: ",
    as.character(
        utils::packageVersion("edgeR")
    ),
    "\n",
    sep = ""
)

cat(
    "limma version: ",
    as.character(
        utils::packageVersion("limma")
    ),
    "\n",
    sep = ""
)

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. CANONICAL INPUT PATHS
# ==================================================================================================

COUNTS_PATH <- file.path(
    PROJECT_ROOT,
    "04_processed",
    "brainseq_primary_counts_v1.rds"
)

ANNOTATION_PATH <- file.path(
    PROJECT_ROOT,
    "04_processed",
    "brainseq_primary_gene_annotation_v1.csv"
)

META_PATH <- file.path(
    PROJECT_ROOT,
    "04_processed",
    "brainseq_model_metadata_v1.csv"
)

OLD_CORR_GENE_PATH <- file.path(
    PROJECT_ROOT,
    "05_differential_expression",
    "step03a_correlation_estimation_genes_v1.csv"
)

CURRENT_MODEL_AUDIT_PATH <- file.path(
    PROJECT_ROOT,
    "05_differential_expression",
    "step03a_model_audit_v1.txt"
)

required_inputs <- c(
    COUNTS_PATH,
    ANNOTATION_PATH,
    META_PATH
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {

    stop(
        "Canonical input file(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat("Canonical input files: PASS\n")

# ==================================================================================================
# 4. RECOVERY OUTPUT DIRECTORY
# ==================================================================================================

RECOVERY_DIR <- file.path(
    PROJECT_ROOT,
    "05_differential_expression",
    "canonical_recovery_v1"
)

dir.create(
    RECOVERY_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

REC_DLPFC_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_DLPFC_SCZ_vs_Control_v1.csv"
)

REC_HIPPO_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_HIPPO_SCZ_vs_Control_v1.csv"
)

REC_SHARED_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_Shared_SCZ_effect_v1.csv"
)

REC_DEVIATION_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_Regional_Deviation_v1.csv"
)

REC_SUMMARY_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_DE_summary_v1.csv"
)

REC_CORR_GENE_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_correlation_estimation_genes_v1.csv"
)

REC_DESIGN_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_design_matrix_v1.csv"
)

REC_CONTRAST_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_contrast_matrix_v1.csv"
)

REC_MODEL_AUDIT_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_model_audit_v1.txt"
)

REC_VALIDATION_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_canonical_validation_v1.csv"
)

REC_MANIFEST_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_manifest_v1.csv"
)

# ==================================================================================================
# 5. LOAD CANONICAL INPUTS
# ==================================================================================================

cat("Loading canonical counts / annotation / metadata...\n")
flush.console()

COUNTS_PRIMARY <- readRDS(
    COUNTS_PATH
)

ANNOTATION_PRIMARY <- read.csv(
    ANNOTATION_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

MODEL_META <- read.csv(
    META_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 6. DIMENSION / GENE ALIGNMENT
# ==================================================================================================

if (
    nrow(COUNTS_PRIMARY) != 23740L ||
    ncol(COUNTS_PRIMARY) != 755L
) {
    stop(
        "Primary count matrix dimension mismatch: ",
        nrow(COUNTS_PRIMARY),
        " x ",
        ncol(COUNTS_PRIMARY)
    )
}

if (nrow(ANNOTATION_PRIMARY) != 23740L) {
    stop(
        "Primary annotation should contain 23,740 rows."
    )
}

if (nrow(MODEL_META) != 755L) {
    stop(
        "Model metadata should contain 755 rows."
    )
}

if (!"ensemblID_stable" %in% names(ANNOTATION_PRIMARY)) {
    stop(
        "Annotation lacks ensemblID_stable."
    )
}

if (!identical(
    rownames(COUNTS_PRIMARY),
    ANNOTATION_PRIMARY$ensemblID_stable
)) {
    stop(
        "Count-row / annotation stable-Ensembl alignment failed."
    )
}

if (anyDuplicated(
    rownames(COUNTS_PRIMARY)
) > 0L) {
    stop(
        "Duplicated gene IDs detected in primary count matrix."
    )
}

cat(
    "Canonical gene universe: 23,740 — PASS\n"
)

# ==================================================================================================
# 7. SAMPLE ALIGNMENT
# ==================================================================================================

required_meta_cols <- c(
    "RNum",
    "BrNum",
    "Region",
    "Dx",
    "donor_age",
    "Sex",
    "Race",
    "RIN",
    "paired_donor",
    "donor_balance_weight",
    paste0(
        "snpPC",
        1:10
    )
)

missing_meta_cols <- setdiff(
    required_meta_cols,
    names(MODEL_META)
)

if (length(missing_meta_cols) > 0L) {

    stop(
        "Model metadata missing canonical field(s): ",
        paste(
            missing_meta_cols,
            collapse = ", "
        )
    )
}

count_sample_ids <- colnames(
    COUNTS_PRIMARY
)

if (is.null(count_sample_ids)) {
    stop(
        "Primary count matrix has no sample column names."
    )
}

if (!identical(
    count_sample_ids,
    MODEL_META$RNum
)) {

    if (
        length(count_sample_ids) ==
            length(MODEL_META$RNum) &&
        setequal(
            count_sample_ids,
            MODEL_META$RNum
        )
    ) {

        meta_match <- match(
            count_sample_ids,
            MODEL_META$RNum
        )

        if (anyNA(meta_match)) {
            stop(
                "Sample matching unexpectedly produced NA."
            )
        }

        MODEL_META <- MODEL_META[
            meta_match,
            ,
            drop = FALSE
        ]

        rownames(MODEL_META) <- NULL

        cat(
            "Metadata reordered to existing canonical count-column order.\n"
        )

    } else {

        stop(
            "Count-column sample IDs and MODEL_META$RNum do not contain the same 755 samples."
        )
    }
}

if (!identical(
    colnames(COUNTS_PRIMARY),
    MODEL_META$RNum
)) {
    stop(
        "Final count / metadata sample alignment failed."
    )
}

if (
    length(
        unique(
            MODEL_META$BrNum
        )
    ) != 454L
) {
    stop(
        "Expected 454 unique donors."
    )
}

cat(
    "Sample alignment: 755 samples / 454 donors — PASS\n"
)

# ==================================================================================================
# 8. NORMALIZE paired_donor
# ==================================================================================================

if (is.logical(
    MODEL_META$paired_donor
)) {

    paired_clean <- MODEL_META$paired_donor

} else {

    paired_text <- toupper(
        trimws(
            as.character(
                MODEL_META$paired_donor
            )
        )
    )

    valid_pair_values <- c(
        "TRUE",
        "FALSE",
        "T",
        "F",
        "1",
        "0"
    )

    if (!all(
        paired_text %in%
            valid_pair_values
    )) {
        stop(
            "Unexpected paired_donor encoding."
        )
    }

    paired_clean <- paired_text %in%
        c(
            "TRUE",
            "T",
            "1"
        )
}

MODEL_META$paired_donor <- paired_clean

PAIRED_INDEX <- which(
    MODEL_META$paired_donor
)

if (length(PAIRED_INDEX) != 602L) {
    stop(
        "Expected 602 paired samples; observed ",
        length(PAIRED_INDEX)
    )
}

if (
    length(
        unique(
            MODEL_META$BrNum[
                PAIRED_INDEX
            ]
        )
    ) != 301L
) {
    stop(
        "Expected 301 paired donors."
    )
}

if (
    abs(
        sum(
            MODEL_META$donor_balance_weight
        ) -
        454
    ) >
        1e-8
) {
    stop(
        "Donor-balanced total weight should equal 454."
    )
}

cat(
    "Paired architecture: 602 samples / 301 donors — PASS\n"
)

# ==================================================================================================
# 9. FACTORS
# ==================================================================================================

MODEL_META$Region <- factor(
    MODEL_META$Region,
    levels = c(
        "DLPFC",
        "HIPPO"
    )
)

MODEL_META$Dx <- factor(
    MODEL_META$Dx,
    levels = c(
        "Control",
        "Schizo"
    )
)

MODEL_META$Sex <- factor(
    MODEL_META$Sex
)

MODEL_META$Race <- factor(
    MODEL_META$Race
)

MODEL_META$group <- factor(
    paste(
        MODEL_META$Region,
        MODEL_META$Dx,
        sep = "_"
    ),
    levels = c(
        "DLPFC_Control",
        "DLPFC_Schizo",
        "HIPPO_Control",
        "HIPPO_Schizo"
    )
)

if (anyNA(
    MODEL_META$group
)) {
    stop(
        "Unexpected Region x Diagnosis combination."
    )
}

# ==================================================================================================
# 10. STANDARDIZE CONTINUOUS COVARIATES
# ==================================================================================================

MODEL_META$age_z <- as.numeric(
    scale(
        MODEL_META$donor_age
    )
)

MODEL_META$RIN_z <- as.numeric(
    scale(
        MODEL_META$RIN
    )
)

for (pc_i in 1:10) {

    source_name <- paste0(
        "snpPC",
        pc_i
    )

    target_name <- paste0(
        "snpPC",
        pc_i,
        "_z"
    )

    MODEL_META[
        ,
        target_name
    ] <- as.numeric(
        scale(
            MODEL_META[
                ,
                source_name
            ]
        )
    )
}

standardized_cols <- c(
    "age_z",
    "RIN_z",
    paste0(
        "snpPC",
        1:10,
        "_z"
    )
)

if (any(
    !is.finite(
        as.matrix(
            MODEL_META[
                ,
                standardized_cols,
                drop = FALSE
            ]
        )
    )
)) {
    stop(
        "Non-finite standardized covariate detected."
    )
}

# ==================================================================================================
# 11. REBUILD CANONICAL FULL DESIGN
# ==================================================================================================

DESIGN <- model.matrix(
    ~ 0 +
        group +
        age_z +
        Sex +
        Race +
        RIN_z +
        snpPC1_z +
        snpPC2_z +
        snpPC3_z +
        snpPC4_z +
        snpPC5_z +
        snpPC6_z +
        snpPC7_z +
        snpPC8_z +
        snpPC9_z +
        snpPC10_z,
    data = MODEL_META
)

colnames(DESIGN) <- sub(
    "^group",
    "",
    colnames(DESIGN)
)

if (nrow(DESIGN) != 755L) {
    stop(
        "Full design row count mismatch."
    )
}

if (ncol(DESIGN) != 20L) {
    stop(
        "Canonical design should contain 20 columns; observed ",
        ncol(DESIGN)
    )
}

if (
    qr(DESIGN)$rank !=
        ncol(DESIGN)
) {

    cat(
        "\nNon-estimable full-design columns:\n"
    )

    print(
        limma::nonEstimable(
            DESIGN
        )
    )

    stop(
        "Canonical full design is rank deficient."
    )
}

cat(
    "Canonical full design: 755 x 20; full rank — PASS\n"
)

# ==================================================================================================
# 12. CANONICAL CONTRASTS
# ==================================================================================================

CONTRASTS <- limma::makeContrasts(

    DLPFC_SCZ =
        DLPFC_Schizo -
        DLPFC_Control,

    HIPPO_SCZ =
        HIPPO_Schizo -
        HIPPO_Control,

    Shared_SCZ =
        0.5 * (
            DLPFC_Schizo -
            DLPFC_Control +
            HIPPO_Schizo -
            HIPPO_Control
        ),

    Regional_Deviation =
        0.5 * (
            DLPFC_Schizo -
            DLPFC_Control -
            HIPPO_Schizo +
            HIPPO_Control
        ),

    levels = DESIGN
)

cat(
    "Canonical four-contrast matrix: PASS\n"
)

# ==================================================================================================
# 13. TMM + INITIAL VOOM
# ==================================================================================================

cat(
    "\nRunning TMM normalization + initial voom...\n"
)
flush.console()

DGE <- edgeR::DGEList(
    counts = COUNTS_PRIMARY
)

DGE <- edgeR::calcNormFactors(
    DGE,
    method = "TMM"
)

VOOM_PRE <- limma::voom(
    DGE,
    design = DESIGN,
    plot = FALSE
)

cat(
    "Initial voom: PASS\n"
)

# ==================================================================================================
# 14. PAIRED-SAMPLE CORRELATION DESIGN
# ==================================================================================================

META_CORR <- droplevels(
    MODEL_META[
        PAIRED_INDEX,
        ,
        drop = FALSE
    ]
)

DESIGN_CORR <- model.matrix(
    ~ 0 +
        group +
        age_z +
        Sex +
        Race +
        RIN_z +
        snpPC1_z +
        snpPC2_z +
        snpPC3_z +
        snpPC4_z +
        snpPC5_z +
        snpPC6_z +
        snpPC7_z +
        snpPC8_z +
        snpPC9_z +
        snpPC10_z,
    data = META_CORR
)

colnames(DESIGN_CORR) <- sub(
    "^group",
    "",
    colnames(DESIGN_CORR)
)

if (
    qr(DESIGN_CORR)$rank !=
        ncol(DESIGN_CORR)
) {

    cat(
        "\nNon-estimable paired-design columns:\n"
    )

    print(
        limma::nonEstimable(
            DESIGN_CORR
        )
    )

    stop(
        "Canonical paired-sample correlation design is rank deficient."
    )
}

cat(
    "Paired-sample correlation design: full rank — PASS\n"
)

# ==================================================================================================
# 15. REGENERATE EXACT CANONICAL 500-GENE OUTCOME-BLIND SUBSET
#
# We regenerate from the documented algorithm rather than trusting the currently overwritten
# STEP03A directory.
# ==================================================================================================

cat(
    "Regenerating canonical 500-gene variance-stratified subset...\n"
)
flush.console()

GENE_VAR <- apply(
    VOOM_PRE$E,
    1,
    var
)

if (any(
    !is.finite(
        GENE_VAR
    )
)) {
    stop(
        "Non-finite gene variance detected."
    )
}

VAR_RANK <- rank(
    GENE_VAR,
    ties.method = "first"
)

VAR_BIN <- cut(
    VAR_RANK,
    breaks = 10,
    labels = FALSE
)

set.seed(
    20260929
)

CORR_GENE_INDEX <- unlist(
    lapply(
        1:10,
        function(bin_i) {

            candidates <- which(
                VAR_BIN ==
                    bin_i
            )

            sample(
                candidates,
                size = min(
                    50L,
                    length(candidates)
                ),
                replace = FALSE
            )
        }
    ),
    use.names = FALSE
)

CORR_GENE_INDEX <- sort(
    unique(
        CORR_GENE_INDEX
    )
)

CORR_GENE_N <- length(
    CORR_GENE_INDEX
)

if (CORR_GENE_N != 500L) {
    stop(
        "Canonical algorithm should return exactly 500 genes; obtained ",
        CORR_GENE_N
    )
}

CORR_GENE_IDS <- rownames(
    VOOM_PRE$E
)[
    CORR_GENE_INDEX
]

if (
    length(
        unique(
            CORR_GENE_IDS
        )
    ) != 500L
) {
    stop(
        "Canonical correlation-gene set contains duplicate IDs."
    )
}

cat(
    "Canonical correlation-gene subset: 500 genes — PASS\n"
)

# ==================================================================================================
# 16. COMPARE WITH ANY EXISTING SAVED CORRELATION-GENE LIST
# ==================================================================================================

SAVED_CORR_LIST_PRESENT <- file.exists(
    OLD_CORR_GENE_PATH
)

SAVED_CORR_LIST_EXACT_MATCH <- NA
SAVED_CORR_LIST_SET_MATCH <- NA
SAVED_CORR_LIST_N <- NA_integer_

if (SAVED_CORR_LIST_PRESENT) {

    old_corr <- read.csv(
        OLD_CORR_GENE_PATH,
        stringsAsFactors = FALSE,
        check.names = FALSE
    )

    if (
        "ensemblID_stable" %in%
            names(old_corr)
    ) {

        old_ids <- as.character(
            old_corr$ensemblID_stable
        )

        SAVED_CORR_LIST_N <- length(
            old_ids
        )

        SAVED_CORR_LIST_EXACT_MATCH <- identical(
            old_ids,
            CORR_GENE_IDS
        )

        SAVED_CORR_LIST_SET_MATCH <- (
            length(old_ids) ==
                500L &&
            setequal(
                old_ids,
                CORR_GENE_IDS
            )
        )
    }
}

cat(
    "Existing correlation-gene list rows: ",
    ifelse(
        is.na(SAVED_CORR_LIST_N),
        "NA",
        SAVED_CORR_LIST_N
    ),
    "\n",
    sep = ""
)

cat(
    "Existing list exact canonical match: ",
    SAVED_CORR_LIST_EXACT_MATCH,
    "\n",
    sep = ""
)

cat(
    "Existing list canonical set match: ",
    SAVED_CORR_LIST_SET_MATCH,
    "\n",
    sep = ""
)

# ==================================================================================================
# 17. CANONICAL 500-GENE duplicateCorrelation
#
# CRITICAL:
# This is the ONLY duplicateCorrelation call in this recovery.
# It runs on 500 genes x 602 paired samples — NEVER all 23,740 genes.
# ==================================================================================================

cat(
    "\nEstimating canonical donor correlation from ONLY 500 genes x 602 paired samples...\n"
)
flush.console()

CORR_START <- proc.time()[3]

CORFIT <- limma::duplicateCorrelation(

    VOOM_PRE$E[
        CORR_GENE_INDEX,
        PAIRED_INDEX,
        drop = FALSE
    ],

    design = DESIGN_CORR,

    block = META_CORR$BrNum,

    weights = VOOM_PRE$weights[
        CORR_GENE_INDEX,
        PAIRED_INDEX,
        drop = FALSE
    ],

    trim = 0.15
)

CORR_SECONDS <- proc.time()[3] -
    CORR_START

DONOR_CORRELATION <- CORFIT$consensus.correlation

if (!is.finite(
    DONOR_CORRELATION
)) {
    stop(
        "Recovered donor correlation is non-finite."
    )
}

cat(
    "Recovered donor correlation: ",
    format(
        DONOR_CORRELATION,
        digits = 12
    ),
    "\n",
    sep = ""
)

cat(
    "500-gene duplicateCorrelation seconds: ",
    round(
        CORR_SECONDS,
        1
    ),
    "\n",
    sep = ""
)

CANONICAL_CORRELATION_MATCH <- (
    abs(
        DONOR_CORRELATION -
        0.188518
    ) <
        5e-7
)

cat(
    "Matches historical 0.188518 to printed precision: ",
    CANONICAL_CORRELATION_MATCH,
    "\n",
    sep = ""
)

if (!CANONICAL_CORRELATION_MATCH) {

    stop(
        paste0(
            "RECOVERY STOP: 500-gene donor correlation did not reproduce canonical 0.188518. ",
            "No full-gene DE fit will be run."
        )
    )
}

# ==================================================================================================
# 18. FINAL VOOM USING RECOVERED CANONICAL CORRELATION
# ==================================================================================================

cat(
    "\nRunning final voom on 23,740 genes using recovered fixed donor correlation...\n"
)
flush.console()

VOOM_FINAL <- limma::voom(
    DGE,
    design = DESIGN,
    block = MODEL_META$BrNum,
    correlation = DONOR_CORRELATION,
    plot = FALSE
)

VOOM_FINAL$weights <- sweep(
    VOOM_FINAL$weights,
    2,
    MODEL_META$donor_balance_weight,
    "*"
)

cat(
    "Final voom: PASS\n"
)
flush.console()

# ==================================================================================================
# 19. FULL-GENE LIMMA FIT
#
# NOTE:
# This is lmFit only.
# duplicateCorrelation is NOT rerun here.
# ==================================================================================================

cat(
    "Fitting canonical donor-blocked limma model on 23,740 genes...\n"
)
flush.console()

FIT <- limma::lmFit(
    VOOM_FINAL,
    design = DESIGN,
    block = MODEL_META$BrNum,
    correlation = DONOR_CORRELATION
)

FIT <- limma::contrasts.fit(
    FIT,
    CONTRASTS
)

FIT <- limma::eBayes(
    FIT,
    robust = TRUE
)

cat(
    "Canonical full-gene limma fit: PASS\n"
)
flush.console()

# ==================================================================================================
# 20. RESULT EXTRACTOR
# ==================================================================================================

required_annotation_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "chromosome",
    "entrez_id"
)

missing_ann_cols <- setdiff(
    required_annotation_cols,
    names(ANNOTATION_PRIMARY)
)

if (length(missing_ann_cols) > 0L) {
    stop(
        "Primary annotation missing field(s): ",
        paste(
            missing_ann_cols,
            collapse = ", "
        )
    )
}

extract_result <- function(
    fit_object,
    coefficient_name,
    annotation
) {

    tt <- limma::topTable(
        fit_object,
        coef = coefficient_name,
        number = Inf,
        adjust.method = "BH",
        sort.by = "P"
    )

    tt$ensemblID_stable <- rownames(
        tt
    )

    rownames(tt) <- NULL

    ann_keep <- annotation[
        ,
        required_annotation_cols,
        drop = FALSE
    ]

    out <- merge(
        tt,
        ann_keep,
        by = "ensemblID_stable",
        all.x = TRUE,
        sort = FALSE
    )

    restore_index <- match(
        tt$ensemblID_stable,
        out$ensemblID_stable
    )

    if (anyNA(
        restore_index
    )) {
        stop(
            "Annotation merge/order restoration failed for ",
            coefficient_name
        )
    }

    out <- out[
        restore_index,
        ,
        drop = FALSE
    ]

    rownames(out) <- NULL

    out$FDR_lt_0.05 <- (
        out$adj.P.Val <
            0.05
    )

    out$direction <- ifelse(
        out$logFC > 0,
        "Higher_in_SCZ",
        ifelse(
            out$logFC < 0,
            "Lower_in_SCZ",
            "Zero"
        )
    )

    # Put identity columns first for readability.
    final_columns <- c(
        "ensemblID_stable",
        "gene_symbol",
        "gene_type",
        "chromosome",
        "entrez_id",
        "logFC",
        "AveExpr",
        "t",
        "P.Value",
        "adj.P.Val",
        "B",
        "FDR_lt_0.05",
        "direction"
    )

    remaining_columns <- setdiff(
        names(out),
        final_columns
    )

    out[
        ,
        c(
            final_columns[
                final_columns %in%
                    names(out)
            ],
            remaining_columns
        ),
        drop = FALSE
    ]
}

# ==================================================================================================
# 21. EXTRACT FOUR CANONICAL RESULTS
# ==================================================================================================

cat(
    "Extracting four canonical contrasts...\n"
)
flush.console()

RES_DLPFC <- extract_result(
    FIT,
    "DLPFC_SCZ",
    ANNOTATION_PRIMARY
)

RES_HIPPO <- extract_result(
    FIT,
    "HIPPO_SCZ",
    ANNOTATION_PRIMARY
)

RES_SHARED <- extract_result(
    FIT,
    "Shared_SCZ",
    ANNOTATION_PRIMARY
)

RES_DEVIATION <- extract_result(
    FIT,
    "Regional_Deviation",
    ANNOTATION_PRIMARY
)

if (
    nrow(RES_DLPFC) != 23740L ||
    nrow(RES_HIPPO) != 23740L ||
    nrow(RES_SHARED) != 23740L ||
    nrow(RES_DEVIATION) != 23740L
) {
    stop(
        "Recovered result-table row count failure."
    )
}

cat(
    "Four recovered result tables: 23,740 genes each — PASS\n"
)

# ==================================================================================================
# 22. CANONICAL DE SUMMARY
# ==================================================================================================

make_summary_row <- function(
    result_table,
    contrast_name
) {

    data.frame(
        contrast = contrast_name,
        n_genes = nrow(result_table),
        FDR_lt_0.05 = sum(
            result_table$adj.P.Val <
                0.05,
            na.rm = TRUE
        ),
        FDR_lt_0.05_positive = sum(
            result_table$adj.P.Val <
                0.05 &
            result_table$logFC >
                0,
            na.rm = TRUE
        ),
        FDR_lt_0.05_negative = sum(
            result_table$adj.P.Val <
                0.05 &
            result_table$logFC <
                0,
            na.rm = TRUE
        ),
        P_lt_0.001 = sum(
            result_table$P.Value <
                0.001,
            na.rm = TRUE
        ),
        stringsAsFactors = FALSE
    )
}

DE_SUMMARY <- rbind(
    make_summary_row(
        RES_DLPFC,
        "DLPFC_SCZ"
    ),
    make_summary_row(
        RES_HIPPO,
        "HIPPO_SCZ"
    ),
    make_summary_row(
        RES_SHARED,
        "Shared_SCZ"
    ),
    make_summary_row(
        RES_DEVIATION,
        "Regional_Deviation"
    )
)

# ==================================================================================================
# 23. CROSS-REGION CORRELATION
# ==================================================================================================

hippo_index <- match(
    RES_DLPFC$ensemblID_stable,
    RES_HIPPO$ensemblID_stable
)

if (anyNA(
    hippo_index
)) {
    stop(
        "Recovered DLPFC/HIPPO stable-ID alignment failed."
    )
}

DLPFC_LOGFC <- RES_DLPFC$logFC

HIPPO_LOGFC <- RES_HIPPO$logFC[
    hippo_index
]

REGION_LOGFC_COR <- cor(
    DLPFC_LOGFC,
    HIPPO_LOGFC,
    use = "complete.obs",
    method = "pearson"
)

SIGN_CONCORDANCE_RATE <- mean(
    sign(
        DLPFC_LOGFC
    ) ==
    sign(
        HIPPO_LOGFC
    ),
    na.rm = TRUE
)

# ==================================================================================================
# 24. STRICT CANONICAL VALIDATION
# ==================================================================================================

EXPECTED_FDR_COUNTS <- c(
    2949L,
    2648L,
    5348L,
    11L
)

OBSERVED_FDR_COUNTS <- as.integer(
    DE_SUMMARY$FDR_lt_0.05
)

FDR_COUNTS_PASS <- identical(
    OBSERVED_FDR_COUNTS,
    EXPECTED_FDR_COUNTS
)

REGION_COR_PASS <- (
    abs(
        REGION_LOGFC_COR -
        0.6089
    ) <
        0.00005
)

SIGN_CONCORDANCE_PASS <- (
    abs(
        SIGN_CONCORDANCE_RATE -
        0.7350
    ) <
        0.00005
)

pld4_index <- which(
    RES_SHARED$gene_symbol ==
        "PLD4"
)

PLD4_LOGFC <- if (
    length(pld4_index) >= 1L
) {
    RES_SHARED$logFC[
        pld4_index[1L]
    ]
} else {
    NA_real_
}

PLD4_P <- if (
    length(pld4_index) >= 1L
) {
    RES_SHARED$P.Value[
        pld4_index[1L]
    ]
} else {
    NA_real_
}

PLD4_FDR <- if (
    length(pld4_index) >= 1L
) {
    RES_SHARED$adj.P.Val[
        pld4_index[1L]
    ]
} else {
    NA_real_
}

PLD4_PASS <- (
    is.finite(PLD4_LOGFC) &&
    abs(
        PLD4_LOGFC -
        (-0.5732572)
    ) <
        5e-6
)

ALL_CANONICAL_CHECKS_PASS <- (
    CANONICAL_CORRELATION_MATCH &&
    FDR_COUNTS_PASS &&
    REGION_COR_PASS &&
    SIGN_CONCORDANCE_PASS &&
    PLD4_PASS
)

VALIDATION <- data.frame(
    check = c(
        "donor_correlation_0.188518",
        "FDR_counts_2949_2648_5348_11",
        "regional_correlation_0.6089",
        "sign_concordance_0.7350",
        "shared_PLD4_logFC_minus0.5732572"
    ),
    observed = c(
        format(
            DONOR_CORRELATION,
            digits = 12
        ),
        paste(
            OBSERVED_FDR_COUNTS,
            collapse = "/"
        ),
        format(
            REGION_LOGFC_COR,
            digits = 12
        ),
        format(
            SIGN_CONCORDANCE_RATE,
            digits = 12
        ),
        format(
            PLD4_LOGFC,
            digits = 12
        )
    ),
    expected = c(
        "0.188518",
        "2949/2648/5348/11",
        "0.6089",
        "0.7350",
        "-0.5732572"
    ),
    pass = c(
        CANONICAL_CORRELATION_MATCH,
        FDR_COUNTS_PASS,
        REGION_COR_PASS,
        SIGN_CONCORDANCE_PASS,
        PLD4_PASS
    ),
    stringsAsFactors = FALSE
)

write.csv(
    VALIDATION,
    REC_VALIDATION_PATH,
    row.names = FALSE,
    quote = TRUE
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nCANONICAL RECOVERY VALIDATION\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    VALIDATION,
    row.names = FALSE
)

cat(
    "\nRecovered DE summary:\n"
)

print(
    DE_SUMMARY,
    row.names = FALSE
)

cat(
    "\nRecovered DLPFC-HIPPO logFC correlation: ",
    format(
        REGION_LOGFC_COR,
        digits = 12
    ),
    "\n",
    sep = ""
)

cat(
    "Recovered DLPFC-HIPPO sign concordance: ",
    format(
        SIGN_CONCORDANCE_RATE,
        digits = 12
    ),
    "\n",
    sep = ""
)

cat(
    "Recovered PLD4 shared logFC: ",
    format(
        PLD4_LOGFC,
        digits = 12
    ),
    "\n",
    sep = ""
)

# ==================================================================================================
# 25. FAIL CLOSED IF CANONICAL OUTPUT WAS NOT REPRODUCED
# ==================================================================================================

if (!ALL_CANONICAL_CHECKS_PASS) {

    failure_lines <- c(
        "STEP 11C.3Q CANONICAL RECOVERY: FAIL",
        "",
        paste0(
            "Recovered donor correlation = ",
            format(
                DONOR_CORRELATION,
                digits = 12
            )
        ),
        paste0(
            "Recovered FDR counts = ",
            paste(
                OBSERVED_FDR_COUNTS,
                collapse = " / "
            )
        ),
        paste0(
            "Recovered regional correlation = ",
            format(
                REGION_LOGFC_COR,
                digits = 12
            )
        ),
        paste0(
            "Recovered sign concordance = ",
            format(
                SIGN_CONCORDANCE_RATE,
                digits = 12
            )
        ),
        paste0(
            "Recovered PLD4 shared logFC = ",
            format(
                PLD4_LOGFC,
                digits = 12
            )
        ),
        "",
        "No canonical DE result CSVs were written.",
        "Current overwritten STEP03A files were not modified."
    )

    writeLines(
        failure_lines,
        file.path(
            RECOVERY_DIR,
            "RECOVERY_FAILED_DO_NOT_USE.txt"
        ),
        useBytes = TRUE
    )

    stop(
        paste0(
            "STEP 11C.3Q FAIL: recovered results did not reproduce all canonical anchors. ",
            "Current STEP03A files remain untouched."
        )
    )
}

cat(
    "\nALL CANONICAL RECOVERY CHECKS: PASS\n"
)

# ==================================================================================================
# 26. SAVE RECOVERED CANONICAL ARTIFACTS
#
# Only reached after ALL validation checks pass.
# ==================================================================================================

write.csv(
    RES_DLPFC,
    REC_DLPFC_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    RES_HIPPO,
    REC_HIPPO_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    RES_SHARED,
    REC_SHARED_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    RES_DEVIATION,
    REC_DEVIATION_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    DE_SUMMARY,
    REC_SUMMARY_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    data.frame(
        ensemblID_stable = CORR_GENE_IDS,
        variance = GENE_VAR[
            CORR_GENE_INDEX
        ],
        stringsAsFactors = FALSE
    ),
    REC_CORR_GENE_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    DESIGN,
    REC_DESIGN_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    CONTRASTS,
    REC_CONTRAST_PATH,
    row.names = TRUE,
    quote = TRUE
)

# ==================================================================================================
# 27. RECOVERED MODEL AUDIT
# ==================================================================================================

TOTAL_SECONDS <- proc.time()[3] -
    TOTAL_START

MODEL_AUDIT <- c(

    "recovery_status=CANONICAL_RECOVERY_PASS",

    "method=limma_voom_donor_blocked",

    "recovery_reason=current_STEP03A_files_were_overwritten_by_coherent_alternative_run",

    "correlation_estimation=500_variance_stratified_outcome_blind_genes",

    "correlation_gene_seed=20260929",

    "correlation_trim=0.15",

    "full_gene_duplicateCorrelation=NOT_RUN",

    paste0(
        "correlation_genes=",
        CORR_GENE_N
    ),

    "paired_donors=301",

    "paired_samples=602",

    "samples=755",

    "donors=454",

    "genes=23740",

    "design_columns=20",

    paste0(
        "duplicateCorrelation=",
        format(
            DONOR_CORRELATION,
            digits = 12
        )
    ),

    paste0(
        "DLPFC_HIPPO_logFC_correlation=",
        format(
            REGION_LOGFC_COR,
            digits = 12
        )
    ),

    paste0(
        "DLPFC_HIPPO_sign_concordance=",
        format(
            SIGN_CONCORDANCE_RATE,
            digits = 12
        )
    ),

    paste0(
        "DLPFC_FDR_lt_0.05=",
        OBSERVED_FDR_COUNTS[1]
    ),

    paste0(
        "HIPPO_FDR_lt_0.05=",
        OBSERVED_FDR_COUNTS[2]
    ),

    paste0(
        "Shared_FDR_lt_0.05=",
        OBSERVED_FDR_COUNTS[3]
    ),

    paste0(
        "Regional_Deviation_FDR_lt_0.05=",
        OBSERVED_FDR_COUNTS[4]
    ),

    paste0(
        "PLD4_shared_logFC=",
        format(
            PLD4_LOGFC,
            digits = 12
        )
    ),

    paste0(
        "PLD4_shared_P.Value=",
        format(
            PLD4_P,
            scientific = TRUE,
            digits = 12
        )
    ),

    paste0(
        "PLD4_shared_adj.P.Val=",
        format(
            PLD4_FDR,
            scientific = TRUE,
            digits = 12
        )
    ),

    paste0(
        "saved_old_corr_gene_list_present=",
        SAVED_CORR_LIST_PRESENT
    ),

    paste0(
        "saved_old_corr_gene_list_exact_match=",
        SAVED_CORR_LIST_EXACT_MATCH
    ),

    paste0(
        "saved_old_corr_gene_list_set_match=",
        SAVED_CORR_LIST_SET_MATCH
    ),

    paste0(
        "500_gene_duplicateCorrelation_seconds=",
        round(
            CORR_SECONDS,
            1
        )
    ),

    paste0(
        "total_recovery_seconds=",
        round(
            TOTAL_SECONDS,
            1
        )
    ),

    "current_overwritten_STEP03A_files_modified=FALSE"
)

writeLines(
    MODEL_AUDIT,
    REC_MODEL_AUDIT_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 28. OUTPUT READ-BACK VALIDATION
# ==================================================================================================

saved_outputs <- c(
    REC_DLPFC_PATH,
    REC_HIPPO_PATH,
    REC_SHARED_PATH,
    REC_DEVIATION_PATH,
    REC_SUMMARY_PATH,
    REC_CORR_GENE_PATH,
    REC_DESIGN_PATH,
    REC_CONTRAST_PATH,
    REC_MODEL_AUDIT_PATH,
    REC_VALIDATION_PATH
)

if (!all(
    file.exists(
        saved_outputs
    )
)) {
    stop(
        "One or more recovered canonical artifacts were not written."
    )
}

saved_sizes <- file.info(
    saved_outputs
)$size

if (any(
    is.na(saved_sizes) |
    saved_sizes <= 0
)) {
    stop(
        "One or more recovered canonical artifacts are zero-byte."
    )
}

check_summary <- read.csv(
    REC_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (!identical(
    as.integer(
        check_summary$FDR_lt_0.05
    ),
    EXPECTED_FDR_COUNTS
)) {
    stop(
        "Recovered summary read-back verification failed."
    )
}

cat(
    "Recovered artifact read-back: PASS\n"
)

# ==================================================================================================
# 29. MANIFEST / HASHES
# ==================================================================================================

manifest_files <- c(
    COUNTS_PATH,
    ANNOTATION_PATH,
    META_PATH,
    saved_outputs
)

manifest_info <- file.info(
    manifest_files
)

MANIFEST <- data.frame(
    file = basename(
        manifest_files
    ),
    path = manifest_files,
    bytes = as.numeric(
        manifest_info$size
    ),
    modified_time = as.character(
        manifest_info$mtime
    ),
    sha256 = vapply(
        manifest_files,
        function(p) {
            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    MANIFEST,
    REC_MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 30. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nCANONICAL STEP03A RECOVERY SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Correlation genes:                 ",
    CORR_GENE_N,
    "\n",
    sep = ""
)

cat(
    "Recovered donor correlation:       ",
    format(
        DONOR_CORRELATION,
        digits = 12
    ),
    "\n",
    sep = ""
)

cat(
    "DLPFC-HIPPO logFC correlation:     ",
    format(
        REGION_LOGFC_COR,
        digits = 12
    ),
    "\n",
    sep = ""
)

cat(
    "DLPFC-HIPPO sign concordance:      ",
    format(
        SIGN_CONCORDANCE_RATE,
        digits = 12
    ),
    "\n",
    sep = ""
)

cat(
    "FDR counts DLPFC/HIPPO/Shared/Dev: ",
    paste(
        OBSERVED_FDR_COUNTS,
        collapse = " / "
    ),
    "\n",
    sep = ""
)

cat(
    "PLD4 shared logFC:                 ",
    format(
        PLD4_LOGFC,
        digits = 12
    ),
    "\n",
    sep = ""
)

cat(
    "Full 23,740-gene duplicateCorrelation run: NO\n"
)

cat(
    "Current overwritten STEP03A files touched: NO\n"
)

cat(
    "Canonical recovery directory:\n",
    RECOVERY_DIR,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3Q COMPLETE — CANONICAL STEP03A RECOVERED AND VALIDATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.3Q — CANONICAL STEP03A DE RECOVERY
500-GENE CORRELATION VERSION — CURRENT FILES WILL NOT BE OVERWRITTEN
Project sentinel: PASS
edgeR version: 4.8.2
limma version: 3.66.0
Required packages: PASS
Canonical input files: PASS
Loading canonical counts / annotation / metadata...
Canonical gene universe: 23,740 — PASS
Sample alignment: 755 samples / 454 donors — PASS
Paired architecture: 602 samples / 301 donors — PASS
Canonical full design: 755 x 20; full rank — PASS
Canonical four-contrast matrix: PASS

Running TMM normalization + initial voom...
Initial voom: PASS
Paired-sample correlation design: full rank — PASS
Regenerating canonical 500-gene variance-stratified subset...
Canonical correlation-gene subset: 500 genes — PASS
Existing correlation-gene list rows: 500
Existing list exact canonical match: TRUE
Existing list canonical set match: TRUE

Estimating canonical donor correlation from ONLY 500 genes x 602 paired samples...
Recovered donor correlation: 0.188518383584
500-gen

In [92]:
# ==================================================================================================
# STEP 11C.3S — SAFE CANONICAL STEP03A RESTORE — HASH-COMPARISON CORRECTED
# --------------------------------------------------------------------------------------------------
# PURPOSE
#
# Correct the previous restore-cell bug:
#
#   hash_file(temp_path)
#
# returned an UNNAMED character scalar, while:
#
#   recovered_hashes[nm]
#
# returned a NAMED character scalar.
#
# identical() therefore returned FALSE even when the SHA256 text itself was identical.
#
# THIS CELL:
#   - DOES NOT rerun any analysis
#   - DOES NOT rerun voom
#   - DOES NOT rerun limma
#   - DOES NOT rerun duplicateCorrelation
#   - preserves the archived alternative STEP03A run
#   - safely restores the already validated canonical recovery set
#   - performs byte-level SHA256 validation after restore
#
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3S — SAFE CANONICAL STEP03A RESTORE — HASH FIX\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        project_id
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGE
# ==================================================================================================

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop(
        "digest package unavailable."
    )
}

# ==================================================================================================
# 3. HASH HELPERS
# ==================================================================================================

hash_file <- function(path) {

    digest::digest(
        file = unname(path),
        algo = "sha256",
        serialize = FALSE
    )
}

hash_equal <- function(
    path,
    expected_hash
) {

    observed <- hash_file(
        path
    )

    expected <- unname(
        as.character(
            expected_hash
        )
    )

    identical(
        observed,
        expected
    )
}

# ==================================================================================================
# 4. DIRECTORIES
# ==================================================================================================

DE_DIR <- file.path(
    PROJECT_ROOT,
    "05_differential_expression"
)

RECOVERY_DIR <- file.path(
    DE_DIR,
    "canonical_recovery_v1"
)

ARCHIVE_DIR <- file.path(
    DE_DIR,
    "step03a_overwritten_alternative_archive_v1"
)

if (!dir.exists(
    RECOVERY_DIR
)) {
    stop(
        "Canonical recovery directory missing:\n",
        RECOVERY_DIR
    )
}

if (!dir.exists(
    ARCHIVE_DIR
)) {
    stop(
        "Alternative-run archive directory missing.\n",
        "The previous restore attempt reported the archive as successfully created."
    )
}

cat(
    "Recovery directory: PASS\n"
)

cat(
    "Alternative archive directory: PASS\n"
)

# ==================================================================================================
# 5. ACTIVE CANONICAL-NAME PATHS
# ==================================================================================================

ACTIVE_PATHS <- c(

    DLPFC = file.path(
        DE_DIR,
        "step03a_DLPFC_SCZ_vs_Control_v1.csv"
    ),

    HIPPO = file.path(
        DE_DIR,
        "step03a_HIPPO_SCZ_vs_Control_v1.csv"
    ),

    Shared = file.path(
        DE_DIR,
        "step03a_Shared_SCZ_effect_v1.csv"
    ),

    Deviation = file.path(
        DE_DIR,
        "step03a_Regional_Deviation_v1.csv"
    ),

    Summary = file.path(
        DE_DIR,
        "step03a_DE_summary_v1.csv"
    ),

    CorrelationGenes = file.path(
        DE_DIR,
        "step03a_correlation_estimation_genes_v1.csv"
    ),

    ModelAudit = file.path(
        DE_DIR,
        "step03a_model_audit_v1.txt"
    )
)

missing_active <- ACTIVE_PATHS[
    !file.exists(
        ACTIVE_PATHS
    )
]

if (length(
    missing_active
) > 0L) {

    stop(
        "Active STEP03A artifact(s) missing:\n",
        paste(
            missing_active,
            collapse = "\n"
        )
    )
}

cat(
    "Active STEP03A paths present: PASS\n"
)

# ==================================================================================================
# 6. VALIDATED RECOVERY PATHS
# ==================================================================================================

REC_PATHS <- c(

    DLPFC = file.path(
        RECOVERY_DIR,
        "RECOVERED_step03a_DLPFC_SCZ_vs_Control_v1.csv"
    ),

    HIPPO = file.path(
        RECOVERY_DIR,
        "RECOVERED_step03a_HIPPO_SCZ_vs_Control_v1.csv"
    ),

    Shared = file.path(
        RECOVERY_DIR,
        "RECOVERED_step03a_Shared_SCZ_effect_v1.csv"
    ),

    Deviation = file.path(
        RECOVERY_DIR,
        "RECOVERED_step03a_Regional_Deviation_v1.csv"
    ),

    Summary = file.path(
        RECOVERY_DIR,
        "RECOVERED_step03a_DE_summary_v1.csv"
    ),

    CorrelationGenes = file.path(
        RECOVERY_DIR,
        "RECOVERED_step03a_correlation_estimation_genes_v1.csv"
    ),

    ModelAudit = file.path(
        RECOVERY_DIR,
        "RECOVERED_step03a_model_audit_v1.txt"
    )
)

REC_VALIDATION_PATH <- file.path(
    RECOVERY_DIR,
    "RECOVERED_step03a_canonical_validation_v1.csv"
)

missing_recovered <- REC_PATHS[
    !file.exists(
        REC_PATHS
    )
]

if (length(
    missing_recovered
) > 0L) {

    stop(
        "Recovered canonical artifact(s) missing:\n",
        paste(
            missing_recovered,
            collapse = "\n"
        )
    )
}

if (!file.exists(
    REC_VALIDATION_PATH
)) {

    stop(
        "Recovered validation table missing."
    )
}

cat(
    "Recovered canonical files present: PASS\n"
)

# ==================================================================================================
# 7. VERIFY RECOVERY VALIDATION
# ==================================================================================================

validation <- read.csv(
    REC_VALIDATION_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (!all(
    c(
        "check",
        "pass"
    ) %in%
        names(validation)
)) {

    stop(
        "Recovered validation table structure unexpected."
    )
}

pass_text <- toupper(
    trimws(
        as.character(
            validation$pass
        )
    )
)

validation_pass <- pass_text %in%
    c(
        "TRUE",
        "T",
        "1"
    )

if (
    nrow(validation) < 5L ||
    !all(validation_pass)
) {

    stop(
        "Canonical recovery validation is not fully PASS."
    )
}

cat(
    "Recovered canonical validation: ALL PASS\n"
)

# ==================================================================================================
# 8. EXPECTED ALTERNATIVE-RUN HASHES FROM PROVENANCE AUDIT
# ==================================================================================================

EXPECTED_ALTERNATIVE_HASHES <- c(

    DLPFC =
        "55b6b85db45dc778b6268c75868aa531d7121e0b16c371ece40cb503fce638b1",

    HIPPO =
        "993b850e70c653b360e5d5b4647e2767f85e7473435ceb69973536706d1227ed",

    Shared =
        "b24448a729d462efba65891a320b8ea53463b70872f352b1efd635745a1f3ae0",

    Deviation =
        "4ef87e25832a0a77482c0986efd15239600edf70c877067060cef018af8ee47b",

    Summary =
        "079dbccec87b8d92da3a7625be6c1ef9f66a78da46ab6b0117fb074a1e118248",

    ModelAudit =
        "8609a126d7d84cb89ed42653aba74f4914a1a2ed28a8946e57b370a5ae4f589a"
)

# ==================================================================================================
# 9. VERIFY ARCHIVED ALTERNATIVE RUN
# ==================================================================================================

ARCHIVE_PATHS <- file.path(
    ARCHIVE_DIR,
    basename(
        ACTIVE_PATHS
    )
)

names(
    ARCHIVE_PATHS
) <- names(
    ACTIVE_PATHS
)

missing_archive <- ARCHIVE_PATHS[
    !file.exists(
        ARCHIVE_PATHS
    )
]

if (length(
    missing_archive
) > 0L) {

    stop(
        "Expected archived alternative file(s) missing:\n",
        paste(
            missing_archive,
            collapse = "\n"
        )
    )
}

for (nm in names(
    EXPECTED_ALTERNATIVE_HASHES
)) {

    if (!hash_equal(
        ARCHIVE_PATHS[nm],
        EXPECTED_ALTERNATIVE_HASHES[nm]
    )) {

        stop(
            "Archived alternative-run SHA256 mismatch for: ",
            nm
        )
    }
}

cat(
    "Archived alternative run matches provenance-audit hashes: PASS\n"
)

# ==================================================================================================
# 10. HASH VALIDATED RECOVERY SOURCES
# ==================================================================================================

RECOVERED_HASHES <- vapply(
    REC_PATHS,
    hash_file,
    FUN.VALUE = character(1)
)

RECOVERED_INFO <- file.info(
    REC_PATHS
)

if (any(
    is.na(
        RECOVERED_INFO$size
    ) |
    RECOVERED_INFO$size <= 0
)) {

    stop(
        "One or more recovered canonical sources are zero-byte."
    )
}

cat(
    "Recovered canonical SHA256 values calculated: PASS\n"
)

# ==================================================================================================
# 11. DETERMINE CURRENT ACTIVE STATE
#
# This allows safe rerunning.
#
# Each active artifact may currently be:
#   ALTERNATIVE
#   CANONICAL
#   UNKNOWN
# ==================================================================================================

ACTIVE_CURRENT_HASHES <- vapply(
    ACTIVE_PATHS,
    hash_file,
    FUN.VALUE = character(1)
)

active_state <- rep(
    "UNKNOWN",
    length(
        ACTIVE_PATHS
    )
)

names(
    active_state
) <- names(
    ACTIVE_PATHS
)

for (nm in names(
    ACTIVE_PATHS
)) {

    current_hash <- unname(
        ACTIVE_CURRENT_HASHES[nm]
    )

    canonical_hash <- unname(
        RECOVERED_HASHES[nm]
    )

    if (identical(
        current_hash,
        canonical_hash
    )) {

        active_state[nm] <- "CANONICAL"

    } else if (
        nm %in%
            names(
                EXPECTED_ALTERNATIVE_HASHES
            ) &&
        identical(
            current_hash,
            unname(
                EXPECTED_ALTERNATIVE_HASHES[nm]
            )
        )
    ) {

        active_state[nm] <- "ALTERNATIVE"

    } else if (
        nm ==
            "CorrelationGenes"
    ) {

        # The 500-gene file was independently shown during recovery to be
        # the exact canonical gene list. Compare directly to recovery hash.

        active_state[nm] <- "OTHER"
    }
}

cat(
    "\nACTIVE FILE STATE BEFORE RESTORE\n"
)

print(
    data.frame(
        artifact = names(
            active_state
        ),
        state = unname(
            active_state
        ),
        stringsAsFactors = FALSE
    ),
    row.names = FALSE
)

# For all provenance-critical overwritten files, only ALTERNATIVE or already-CANONICAL are acceptable.
critical_names <- c(
    "DLPFC",
    "HIPPO",
    "Shared",
    "Deviation",
    "Summary",
    "ModelAudit"
)

bad_critical <- critical_names[
    !active_state[
        critical_names
    ] %in%
        c(
            "ALTERNATIVE",
            "CANONICAL"
        )
]

if (length(
    bad_critical
) > 0L) {

    stop(
        "Unexpected active state for provenance-critical artifact(s): ",
        paste(
            bad_critical,
            collapse = ", "
        ),
        "\nRestore aborted."
    )
}

cat(
    "Pre-restore active-state safety gate: PASS\n"
)

# ==================================================================================================
# 12. COPY + VERIFY + RESTORE EACH ARTIFACT
#
# IMPORTANT HASH FIX:
#
#     hash_equal(temp_path, RECOVERED_HASHES[nm])
#
# compares only the actual SHA256 strings and ignores vector names.
# ==================================================================================================

cat(
    "\nInstalling validated canonical files...\n"
)

for (nm in names(
    REC_PATHS
)) {

    source_path <- unname(
        REC_PATHS[nm]
    )

    target_path <- unname(
        ACTIVE_PATHS[nm]
    )

    expected_hash <- unname(
        RECOVERED_HASHES[nm]
    )

    # If already canonical, no need to recopy.
    if (hash_equal(
        target_path,
        expected_hash
    )) {

        cat(
            "  ",
            nm,
            ": already canonical — verified\n",
            sep = ""
        )

        next
    }

    temp_path <- paste0(
        target_path,
        ".canonical_restore_tmp"
    )

    if (file.exists(
        temp_path
    )) {

        unlink(
            temp_path
        )
    }

    # ----------------------------------------------------------------------------------------------
    # Step A: copy recovered source to temporary file
    # ----------------------------------------------------------------------------------------------

    copy_to_temp <- file.copy(
        from = source_path,
        to = temp_path,
        overwrite = TRUE,
        copy.mode = TRUE
    )

    if (!isTRUE(
        copy_to_temp
    )) {

        stop(
            "Failed to create temporary restore file for: ",
            nm
        )
    }

    # ----------------------------------------------------------------------------------------------
    # Step B: verify temporary file SHA256
    # ----------------------------------------------------------------------------------------------

    if (!hash_equal(
        temp_path,
        expected_hash
    )) {

        observed_temp_hash <- hash_file(
            temp_path
        )

        unlink(
            temp_path
        )

        stop(
            "Temporary canonical copy SHA256 mismatch for: ",
            nm,
            "\nExpected: ",
            expected_hash,
            "\nObserved: ",
            observed_temp_hash
        )
    }

    # ----------------------------------------------------------------------------------------------
    # Step C: replace active file
    # ----------------------------------------------------------------------------------------------

    copy_to_active <- file.copy(
        from = temp_path,
        to = target_path,
        overwrite = TRUE,
        copy.mode = TRUE
    )

    unlink(
        temp_path
    )

    if (!isTRUE(
        copy_to_active
    )) {

        stop(
            "Failed to restore canonical active file for: ",
            nm
        )
    }

    # ----------------------------------------------------------------------------------------------
    # Step D: verify active SHA256
    # ----------------------------------------------------------------------------------------------

    if (!hash_equal(
        target_path,
        expected_hash
    )) {

        stop(
            "Post-restore active SHA256 mismatch for: ",
            nm,
            "\nExpected: ",
            expected_hash,
            "\nObserved: ",
            hash_file(
                target_path
            )
        )
    }

    cat(
        "  ",
        nm,
        ": restored + SHA256 verified\n",
        sep = ""
    )
}

cat(
    "Validated canonical artifacts installed: PASS\n"
)

# ==================================================================================================
# 13. VERIFY ACTIVE FILES ARE BYTE-IDENTICAL TO RECOVERY SOURCES
# ==================================================================================================

ACTIVE_FINAL_HASHES <- vapply(
    ACTIVE_PATHS,
    hash_file,
    FUN.VALUE = character(1)
)

if (!identical(
    unname(
        ACTIVE_FINAL_HASHES
    ),
    unname(
        RECOVERED_HASHES
    )
)) {

    comparison <- data.frame(
        artifact = names(
            ACTIVE_PATHS
        ),
        active_sha256 = unname(
            ACTIVE_FINAL_HASHES
        ),
        recovered_sha256 = unname(
            RECOVERED_HASHES
        ),
        identical = (
            unname(
                ACTIVE_FINAL_HASHES
            ) ==
            unname(
                RECOVERED_HASHES
            )
        ),
        stringsAsFactors = FALSE
    )

    print(
        comparison,
        row.names = FALSE
    )

    stop(
        "Final active/recovery SHA256 identity check failed."
    )
}

cat(
    "Active STEP03A files are byte-identical to validated recovery files: PASS\n"
)

# ==================================================================================================
# 14. READ RESTORED CANONICAL RESULTS
# ==================================================================================================

dlpfc <- read.csv(
    ACTIVE_PATHS["DLPFC"],
    stringsAsFactors = FALSE,
    check.names = FALSE
)

hippo <- read.csv(
    ACTIVE_PATHS["HIPPO"],
    stringsAsFactors = FALSE,
    check.names = FALSE
)

shared <- read.csv(
    ACTIVE_PATHS["Shared"],
    stringsAsFactors = FALSE,
    check.names = FALSE
)

deviation <- read.csv(
    ACTIVE_PATHS["Deviation"],
    stringsAsFactors = FALSE,
    check.names = FALSE
)

de_summary <- read.csv(
    ACTIVE_PATHS["Summary"],
    stringsAsFactors = FALSE,
    check.names = FALSE
)

corr_genes <- read.csv(
    ACTIVE_PATHS["CorrelationGenes"],
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 15. ROW-COUNT VALIDATION
# ==================================================================================================

if (
    nrow(dlpfc) != 23740L ||
    nrow(hippo) != 23740L ||
    nrow(shared) != 23740L ||
    nrow(deviation) != 23740L
) {

    stop(
        "Restored active DE result dimensions are incorrect."
    )
}

if (nrow(
    corr_genes
) != 500L) {

    stop(
        "Restored correlation-gene file should contain exactly 500 rows."
    )
}

cat(
    "Canonical dimensions: 23,740 genes + 500 correlation genes — PASS\n"
)

# ==================================================================================================
# 16. FDR COUNT VALIDATION
# ==================================================================================================

RESTORED_COUNTS <- c(

    DLPFC = sum(
        dlpfc$adj.P.Val <
            0.05,
        na.rm = TRUE
    ),

    HIPPO = sum(
        hippo$adj.P.Val <
            0.05,
        na.rm = TRUE
    ),

    Shared = sum(
        shared$adj.P.Val <
            0.05,
        na.rm = TRUE
    ),

    Deviation = sum(
        deviation$adj.P.Val <
            0.05,
        na.rm = TRUE
    )
)

EXPECTED_COUNTS <- c(
    DLPFC = 2949L,
    HIPPO = 2648L,
    Shared = 5348L,
    Deviation = 11L
)

if (!identical(
    as.integer(
        RESTORED_COUNTS
    ),
    as.integer(
        EXPECTED_COUNTS
    )
)) {

    stop(
        "Canonical FDR count validation failed.\n",
        "Observed: ",
        paste(
            RESTORED_COUNTS,
            collapse = " / "
        )
    )
}

if (!identical(
    as.integer(
        de_summary$FDR_lt_0.05
    ),
    as.integer(
        EXPECTED_COUNTS
    )
)) {

    stop(
        "Restored DE summary disagrees with canonical FDR counts."
    )
}

cat(
    "Canonical FDR counts: 2949 / 2648 / 5348 / 11 — PASS\n"
)

# ==================================================================================================
# 17. REGIONAL CORRELATION VALIDATION
# ==================================================================================================

hippo_index <- match(
    dlpfc$ensemblID_stable,
    hippo$ensemblID_stable
)

if (anyNA(
    hippo_index
)) {

    stop(
        "DLPFC/HIPPO stable-Ensembl alignment failed after restore."
    )
}

REGION_COR <- stats::cor(
    dlpfc$logFC,
    hippo$logFC[
        hippo_index
    ],
    use = "complete.obs",
    method = "pearson"
)

SIGN_CONCORDANCE <- mean(
    sign(
        dlpfc$logFC
    ) ==
    sign(
        hippo$logFC[
            hippo_index
        ]
    ),
    na.rm = TRUE
)

if (
    abs(
        REGION_COR -
        0.608869223845
    ) >
        5e-10
) {

    stop(
        "Canonical regional correlation validation failed: ",
        REGION_COR
    )
}

if (
    abs(
        SIGN_CONCORDANCE -
        0.734962089301
    ) >
        5e-10
) {

    stop(
        "Canonical sign concordance validation failed: ",
        SIGN_CONCORDANCE
    )
}

cat(
    "Canonical regional logFC correlation: ",
    sprintf(
        "%.12f",
        REGION_COR
    ),
    " — PASS\n",
    sep = ""
)

cat(
    "Canonical sign concordance: ",
    sprintf(
        "%.12f",
        SIGN_CONCORDANCE
    ),
    " — PASS\n",
    sep = ""
)

# ==================================================================================================
# 18. PLD4 ANCHOR
# ==================================================================================================

pld4_index <- which(
    shared$gene_symbol ==
        "PLD4"
)

if (length(
    pld4_index
) < 1L) {

    stop(
        "PLD4 absent from restored shared result."
    )
}

PLD4_LOGFC <- shared$logFC[
    pld4_index[1L]
]

if (
    abs(
        PLD4_LOGFC -
        (-0.573257224039)
    ) >
        5e-10
) {

    stop(
        "PLD4 canonical anchor failed."
    )
}

cat(
    "Canonical PLD4 shared logFC: ",
    sprintf(
        "%.12f",
        PLD4_LOGFC
    ),
    " — PASS\n",
    sep = ""
)

# ==================================================================================================
# 19. MODEL AUDIT VALIDATION
# ==================================================================================================

model_audit_lines <- readLines(
    ACTIVE_PATHS["ModelAudit"],
    warn = FALSE
)

model_audit_text <- paste(
    model_audit_lines,
    collapse = "\n"
)

if (!grepl(
    "0\\.188518",
    model_audit_text
)) {

    stop(
        "Restored model audit lacks canonical donor correlation."
    )
}

if (!grepl(
    "full_gene_duplicateCorrelation=NOT_RUN",
    model_audit_text,
    fixed = TRUE
)) {

    stop(
        "Restored model audit lacks explicit full-gene duplicateCorrelation prohibition record."
    )
}

cat(
    "Canonical model audit: PASS\n"
)

# ==================================================================================================
# 20. FINAL RESTORE MANIFEST
# ==================================================================================================

RESTORE_MANIFEST_PATH <- file.path(
    DE_DIR,
    "STEP03A_CANONICAL_RESTORE_MANIFEST_v2.csv"
)

restore_manifest <- data.frame(

    artifact = names(
        ACTIVE_PATHS
    ),

    archived_alternative_path = unname(
        ARCHIVE_PATHS
    ),

    archived_alternative_sha256 = vapply(
        ARCHIVE_PATHS,
        hash_file,
        FUN.VALUE = character(1)
    ),

    validated_recovery_path = unname(
        REC_PATHS
    ),

    validated_recovery_sha256 = unname(
        RECOVERED_HASHES
    ),

    active_restored_path = unname(
        ACTIVE_PATHS
    ),

    active_restored_sha256 = unname(
        ACTIVE_FINAL_HASHES
    ),

    active_equals_recovery = (
        unname(
            ACTIVE_FINAL_HASHES
        ) ==
        unname(
            RECOVERED_HASHES
        )
    ),

    stringsAsFactors = FALSE
)

write.csv(
    restore_manifest,
    RESTORE_MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 21. RESTORE PROVENANCE NOTE
# ==================================================================================================

PROVENANCE_NOTE_PATH <- file.path(
    DE_DIR,
    "STEP03A_CANONICAL_RESTORE_PROVENANCE_v2.txt"
)

provenance_note <- c(

    "==============================================================================================================",
    "STEP03A CANONICAL RESTORE — FINAL VERIFIED STATE",
    "==============================================================================================================",

    "",

    "Canonical analysis recovered and restored after detection of an overwritten alternative STEP03A run.",

    "",

    "Canonical recovery anchors:",

    "Consensus donor correlation = 0.188518383584",

    "DLPFC FDR<0.05 = 2949",

    "HIPPO FDR<0.05 = 2648",

    "Shared FDR<0.05 = 5348",

    "Regional deviation FDR<0.05 = 11",

    "DLPFC-HIPPO logFC correlation = 0.608869223845",

    "DLPFC-HIPPO sign concordance = 0.734962089301",

    "PLD4 shared logFC = -0.573257224039",

    "",

    paste0(
        "Alternative run archive: ",
        ARCHIVE_DIR
    ),

    paste0(
        "Validated recovery directory: ",
        RECOVERY_DIR
    ),

    "",

    "Alternative run preserved before restore.",

    "Active STEP03A files are byte-identical to validated recovered files.",

    "No DE model was rerun during this restore step.",

    "=============================================================================================================="
)

writeLines(
    provenance_note,
    PROVENANCE_NOTE_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 22. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP03A FINAL RESTORE SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Alternative run safely archived: YES\n"
)

cat(
    "Canonical active STEP03A restored: YES\n"
)

cat(
    "Active files byte-identical to validated recovery: YES\n"
)

cat(
    "Consensus donor correlation: 0.188518383584\n"
)

cat(
    "FDR counts DLPFC / HIPPO / Shared / Deviation: ",
    paste(
        RESTORED_COUNTS,
        collapse = " / "
    ),
    "\n",
    sep = ""
)

cat(
    "DLPFC-HIPPO logFC correlation: ",
    sprintf(
        "%.12f",
        REGION_COR
    ),
    "\n",
    sep = ""
)

cat(
    "DLPFC-HIPPO sign concordance: ",
    sprintf(
        "%.12f",
        SIGN_CONCORDANCE
    ),
    "\n",
    sep = ""
)

cat(
    "PLD4 shared logFC: ",
    sprintf(
        "%.12f",
        PLD4_LOGFC
    ),
    "\n",
    sep = ""
)

cat(
    "\nAlternative archive:\n",
    ARCHIVE_DIR,
    "\n",
    sep = ""
)

cat(
    "\nRestore manifest:\n",
    RESTORE_MANIFEST_PATH,
    "\n",
    sep = ""
)

cat(
    "\nProvenance note:\n",
    PROVENANCE_NOTE_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3S COMPLETE — CANONICAL STEP03A SAFELY RESTORED AND VERIFIED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.3S — SAFE CANONICAL STEP03A RESTORE — HASH FIX
Project sentinel: PASS
Recovery directory: PASS
Alternative archive directory: PASS
Active STEP03A paths present: PASS
Recovered canonical files present: PASS
Recovered canonical validation: ALL PASS
Archived alternative run matches provenance-audit hashes: PASS
Recovered canonical SHA256 values calculated: PASS

ACTIVE FILE STATE BEFORE RESTORE
         artifact       state
            DLPFC ALTERNATIVE
            HIPPO ALTERNATIVE
           Shared ALTERNATIVE
        Deviation ALTERNATIVE
          Summary ALTERNATIVE
 CorrelationGenes   CANONICAL
       ModelAudit ALTERNATIVE
Pre-restore active-state safety gate: PASS

Installing validated canonical files...
  DLPFC: restored + SHA256 verified
  HIPPO: restored + SHA256 verified
  Shared: restored + SHA256 verified
  Deviation: restored + SHA256 verified
  Summary: restored + SHA256 verified
  CorrelationGenes: already canonical — verified
  ModelAudit: restored + SHA256 ver

In [93]:
# ==================================================================================================
# STEP 11C.3T — FINAL MAIN FIGURE 2
# SHARED AND REGION-DEPENDENT SCHIZOPHRENIA TRANSCRIPTOMIC ARCHITECTURE
# --------------------------------------------------------------------------------------------------
# CANONICAL ACTIVE STEP03A INPUTS ONLY
#
# PANELS
#   A — DLPFC vs HIPPO schizophrenia-effect concordance
#   B — Number of FDR-significant genes across the four contrasts
#   C — Shared schizophrenia-effect volcano
#   D — Regional-deviation volcano
#
# CANONICAL ANCHORS
#   donor correlation                = 0.188518383584
#   DLPFC FDR<0.05                  = 2949
#   HIPPO FDR<0.05                  = 2648
#   Shared FDR<0.05                 = 5348
#   Regional deviation FDR<0.05     = 11
#   DLPFC-HIPPO logFC correlation   = 0.608869223845
#   sign concordance                = 0.734962089301
#
# IMPORTANT
#   - NO DE rerun
#   - NO limma
#   - NO voom
#   - NO duplicateCorrelation
#   - NO model fitting
#   - Times New Roman
#   - PDF vector master
#   - TIFF 600 dpi
#   - PNG 300 dpi preview
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3T — FINAL MAIN FIGURE 2\n",
    "SHARED AND REGION-DEPENDENT SCHIZOPHRENIA TRANSCRIPTOMIC ARCHITECTURE\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        project_id
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

USE_GGREPEL <- requireNamespace(
    "ggrepel",
    quietly = TRUE
)

cat(
    "ggrepel available: ",
    USE_GGREPEL,
    "\n",
    sep = ""
)

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat(
    "Figure font: Times New Roman\n"
)

# ==================================================================================================
# 4. CANONICAL ACTIVE INPUT PATHS
# ==================================================================================================

DE_DIR <- file.path(
    PROJECT_ROOT,
    "05_differential_expression"
)

DLPFC_PATH <- file.path(
    DE_DIR,
    "step03a_DLPFC_SCZ_vs_Control_v1.csv"
)

HIPPO_PATH <- file.path(
    DE_DIR,
    "step03a_HIPPO_SCZ_vs_Control_v1.csv"
)

SHARED_PATH <- file.path(
    DE_DIR,
    "step03a_Shared_SCZ_effect_v1.csv"
)

DEVIATION_PATH <- file.path(
    DE_DIR,
    "step03a_Regional_Deviation_v1.csv"
)

SUMMARY_PATH <- file.path(
    DE_DIR,
    "step03a_DE_summary_v1.csv"
)

MODEL_AUDIT_PATH <- file.path(
    DE_DIR,
    "step03a_model_audit_v1.txt"
)

RESTORE_MANIFEST_PATH <- file.path(
    DE_DIR,
    "STEP03A_CANONICAL_RESTORE_MANIFEST_v2.csv"
)

required_inputs <- c(
    DLPFC_PATH,
    HIPPO_PATH,
    SHARED_PATH,
    DEVIATION_PATH,
    SUMMARY_PATH,
    MODEL_AUDIT_PATH,
    RESTORE_MANIFEST_PATH
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required canonical input(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat(
    "Canonical active STEP03A source files: PASS\n"
)

# ==================================================================================================
# 5. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "main",
    "Figure2_final_TNR_v3"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "Figure2_Final_TimesNewRoman_v3.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "Figure2_Final_TimesNewRoman_v3.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "Figure2_Final_TimesNewRoman_v3.png"
)

PANEL_A_PATH <- file.path(
    SOURCE_DIR,
    "Figure2_PanelA_cross_region_source_v3.csv"
)

PANEL_B_PATH <- file.path(
    SOURCE_DIR,
    "Figure2_PanelB_FDR_counts_source_v3.csv"
)

PANEL_C_PATH <- file.path(
    SOURCE_DIR,
    "Figure2_PanelC_shared_volcano_source_v3.csv"
)

PANEL_D_PATH <- file.path(
    SOURCE_DIR,
    "Figure2_PanelD_deviation_volcano_source_v3.csv"
)

FIGURE_SUMMARY_PATH <- file.path(
    OUT_DIR,
    "Figure2_final_summary_v3.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Figure2_final_manifest_v3.csv"
)

# ==================================================================================================
# 6. LOAD CANONICAL ACTIVE RESULTS
# ==================================================================================================

dlpfc <- read.csv(
    DLPFC_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

hippo <- read.csv(
    HIPPO_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

shared <- read.csv(
    SHARED_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

deviation <- read.csv(
    DEVIATION_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

de_summary <- read.csv(
    SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 7. BASIC INTEGRITY
# ==================================================================================================

de_tables <- list(
    dlpfc,
    hippo,
    shared,
    deviation
)

de_names <- c(
    "DLPFC",
    "HIPPO",
    "Shared",
    "Deviation"
)

required_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "logFC",
    "P.Value",
    "adj.P.Val"
)

for (i in seq_along(
    de_tables
)) {

    x <- de_tables[[i]]

    if (nrow(x) != 23740L) {

        stop(
            de_names[i],
            " table should contain 23,740 rows; observed ",
            nrow(x)
        )
    }

    missing_cols <- setdiff(
        required_cols,
        names(x)
    )

    if (length(missing_cols) > 0L) {

        stop(
            de_names[i],
            " table missing required column(s): ",
            paste(
                missing_cols,
                collapse = ", "
            )
        )
    }

    if (anyDuplicated(
        x$ensemblID_stable
    ) > 0L) {

        stop(
            de_names[i],
            " table contains duplicated stable Ensembl IDs."
        )
    }
}

cat(
    "Four canonical 23,740-gene DE tables: PASS\n"
)

# ==================================================================================================
# 8. ALIGN DLPFC / HIPPO
# ==================================================================================================

hippo_index <- match(
    dlpfc$ensemblID_stable,
    hippo$ensemblID_stable
)

if (anyNA(
    hippo_index
)) {
    stop(
        "DLPFC/HIPPO stable-Ensembl alignment failed."
    )
}

regional <- data.frame(

    ensemblID_stable =
        dlpfc$ensemblID_stable,

    gene_symbol =
        dlpfc$gene_symbol,

    gene_type =
        dlpfc$gene_type,

    DLPFC_logFC =
        as.numeric(
            dlpfc$logFC
        ),

    HIPPO_logFC =
        as.numeric(
            hippo$logFC[
                hippo_index
            ]
        ),

    DLPFC_FDR =
        as.numeric(
            dlpfc$adj.P.Val
        ),

    HIPPO_FDR =
        as.numeric(
            hippo$adj.P.Val[
                hippo_index
            ]
        ),

    stringsAsFactors = FALSE
)

# ==================================================================================================
# 9. STRICT CANONICAL ANCHORS
# ==================================================================================================

REGION_COR <- stats::cor(
    regional$DLPFC_logFC,
    regional$HIPPO_logFC,
    use = "complete.obs",
    method = "pearson"
)

SIGN_CONCORDANCE <- mean(
    sign(
        regional$DLPFC_logFC
    ) ==
    sign(
        regional$HIPPO_logFC
    ),
    na.rm = TRUE
)

FDR_COUNTS <- c(

    DLPFC = sum(
        dlpfc$adj.P.Val < 0.05,
        na.rm = TRUE
    ),

    HIPPO = sum(
        hippo$adj.P.Val < 0.05,
        na.rm = TRUE
    ),

    Shared = sum(
        shared$adj.P.Val < 0.05,
        na.rm = TRUE
    ),

    Deviation = sum(
        deviation$adj.P.Val < 0.05,
        na.rm = TRUE
    )
)

EXPECTED_COUNTS <- c(
    DLPFC = 2949L,
    HIPPO = 2648L,
    Shared = 5348L,
    Deviation = 11L
)

if (!identical(
    as.integer(
        FDR_COUNTS
    ),
    as.integer(
        EXPECTED_COUNTS
    )
)) {

    stop(
        "Canonical FDR counts failed before plotting."
    )
}

if (
    abs(
        REGION_COR -
        0.608869223845
    ) >
        5e-10
) {

    stop(
        "Canonical regional correlation failed before plotting."
    )
}

if (
    abs(
        SIGN_CONCORDANCE -
        0.734962089301
    ) >
        5e-10
) {

    stop(
        "Canonical sign concordance failed before plotting."
    )
}

cat(
    "Canonical FDR counts: 2949 / 2648 / 5348 / 11 — PASS\n"
)

cat(
    "Canonical regional correlation: ",
    sprintf(
        "%.12f",
        REGION_COR
    ),
    " — PASS\n",
    sep = ""
)

cat(
    "Canonical sign concordance: ",
    sprintf(
        "%.12f",
        SIGN_CONCORDANCE
    ),
    " — PASS\n",
    sep = ""
)

# ==================================================================================================
# 10. PANEL A CLASSIFICATION
# ==================================================================================================

regional$significance_class <- "Neither"

regional$significance_class[
    regional$DLPFC_FDR < 0.05 &
    regional$HIPPO_FDR >= 0.05
] <- "DLPFC only"

regional$significance_class[
    regional$DLPFC_FDR >= 0.05 &
    regional$HIPPO_FDR < 0.05
] <- "HIPPO only"

regional$significance_class[
    regional$DLPFC_FDR < 0.05 &
    regional$HIPPO_FDR < 0.05
] <- "Both regions"

regional$significance_class <- factor(
    regional$significance_class,
    levels = c(
        "Neither",
        "DLPFC only",
        "HIPPO only",
        "Both regions"
    )
)

write.csv(
    regional,
    PANEL_A_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# ==================================================================================================
# 11. PANEL B DATA
# ==================================================================================================

count_df <- data.frame(

    contrast = factor(
        c(
            "DLPFC",
            "HIPPO",
            "Shared",
            "Regional\ndeviation"
        ),
        levels = c(
            "DLPFC",
            "HIPPO",
            "Shared",
            "Regional\ndeviation"
        )
    ),

    n_FDR_0_05 = as.numeric(
        FDR_COUNTS
    ),

    stringsAsFactors = FALSE
)

write.csv(
    count_df,
    PANEL_B_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 12. VOLCANO HELPER
# ==================================================================================================

safe_label <- function(
    symbol,
    ensembl
) {

    symbol <- as.character(
        symbol
    )

    ensembl <- as.character(
        ensembl
    )

    bad <- (
        is.na(symbol) |
        trimws(symbol) == ""
    )

    out <- symbol

    out[bad] <- ensembl[bad]

    out
}

prepare_volcano <- function(
    x,
    positive_name,
    negative_name
) {

    out <- x

    out$display_label <- safe_label(
        out$gene_symbol,
        out$ensemblID_stable
    )

    out$FDR_plot <- pmax(
        as.numeric(
            out$adj.P.Val
        ),
        1e-300
    )

    out$minus_log10_FDR <- -log10(
        out$FDR_plot
    )

    out$DE_class <- "Not FDR-significant"

    out$DE_class[
        out$adj.P.Val < 0.05 &
        out$logFC > 0
    ] <- positive_name

    out$DE_class[
        out$adj.P.Val < 0.05 &
        out$logFC < 0
    ] <- negative_name

    out
}

shared_plot <- prepare_volcano(
    shared,
    "Positive shared effect",
    "Negative shared effect"
)

deviation_plot <- prepare_volcano(
    deviation,
    "Positive deviation",
    "Negative deviation"
)

write.csv(
    shared_plot,
    PANEL_C_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    deviation_plot,
    PANEL_D_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# ==================================================================================================
# 13. SHARED LABELS
#
# Four strongest named positive + four strongest named negative genes.
# ==================================================================================================

shared_named <- shared_plot[
    shared_plot$adj.P.Val < 0.05 &
    !is.na(
        shared_plot$gene_symbol
    ) &
    trimws(
        shared_plot$gene_symbol
    ) != "",
    ,
    drop = FALSE
]

shared_positive <- shared_named[
    shared_named$logFC > 0,
    ,
    drop = FALSE
]

shared_negative <- shared_named[
    shared_named$logFC < 0,
    ,
    drop = FALSE
]

shared_positive <- shared_positive[
    order(
        shared_positive$adj.P.Val,
        -abs(
            shared_positive$logFC
        )
    ),
    ,
    drop = FALSE
]

shared_negative <- shared_negative[
    order(
        shared_negative$adj.P.Val,
        -abs(
            shared_negative$logFC
        )
    ),
    ,
    drop = FALSE
]

shared_labels <- rbind(
    head(
        shared_positive,
        4L
    ),
    head(
        shared_negative,
        4L
    )
)

# ==================================================================================================
# 14. DEVIATION LABELS
#
# All 11 canonical FDR-significant regional-deviation genes.
# ==================================================================================================

deviation_labels <- deviation_plot[
    deviation_plot$adj.P.Val < 0.05,
    ,
    drop = FALSE
]

deviation_labels <- deviation_labels[
    order(
        deviation_labels$adj.P.Val
    ),
    ,
    drop = FALSE
]

if (nrow(
    deviation_labels
) != 11L) {

    stop(
        "Expected exactly 11 canonical regional-deviation genes."
    )
}

# ==================================================================================================
# 15. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.5,
            hjust = 0,
            margin = ggplot2::margin(
                b = 5
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.5
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.3,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 8.3
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.8
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            7,
            8,
            7,
            8
        )
    )

# ==================================================================================================
# 16. PANEL A — CROSS-REGION EFFECT CONCORDANCE
# ==================================================================================================

regional_colours <- c(
    "Neither" = "#C7C7C7",
    "DLPFC only" = "#4E79A7",
    "HIPPO only" = "#F28E2B",
    "Both regions" = "#8E63B0"
)

pA <- ggplot2::ggplot(
    regional,
    ggplot2::aes(
        x = DLPFC_logFC,
        y = HIPPO_logFC,
        colour = significance_class
    )
) +
    ggplot2::geom_hline(
        yintercept = 0,
        linewidth = 0.28,
        colour = "#B0B0B0"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.28,
        colour = "#B0B0B0"
    ) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.42,
        colour = "#555555"
    ) +
    ggplot2::geom_point(
        size = 0.78,
        alpha = 0.48
    ) +
    ggplot2::scale_colour_manual(
        values = regional_colours,
        drop = FALSE
    ) +
    ggplot2::annotate(
        "text",
        x = -Inf,
        y = Inf,
        hjust = -0.05,
        vjust = 1.15,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.6,
        label = paste0(
            "r = ",
            sprintf(
                "%.3f",
                REGION_COR
            ),
            "\nSign concordance = ",
            sprintf(
                "%.1f%%",
                100 * SIGN_CONCORDANCE
            )
        )
    ) +
    ggplot2::labs(
        title = "A  Cross-region diagnosis-effect concordance",
        x = "DLPFC logFC (schizophrenia vs control)",
        y = "HIPPO logFC (schizophrenia vs control)",
        colour = "FDR < 0.05"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom"
    )

# ==================================================================================================
# 17. PANEL B — FDR-SIGNIFICANT GENE COUNTS
#
# Log scale is necessary because deviation = 11 whereas shared = 5,348.
# ==================================================================================================

count_colours <- c(
    "DLPFC" = "#59A14F",
    "HIPPO" = "#E15759",
    "Shared" = "#B07AA1",
    "Regional\ndeviation" = "#76B7B2"
)

pB <- ggplot2::ggplot(
    count_df,
    ggplot2::aes(
        x = contrast,
        y = n_FDR_0_05,
        colour = contrast
    )
) +
    ggplot2::geom_segment(
        ggplot2::aes(
            xend = contrast,
            y = 8,
            yend = n_FDR_0_05
        ),
        linewidth = 1.0
    ) +
    ggplot2::geom_point(
        size = 4.0
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = format(
                n_FDR_0_05,
                big.mark = ","
            )
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.8,
        vjust = -0.8
    ) +
    ggplot2::scale_colour_manual(
        values = count_colours,
        guide = "none"
    ) +
    ggplot2::scale_y_log10(
        limits = c(
            8,
            8000
        ),
        breaks = c(
            10,
            100,
            1000,
            5000
        ),
        labels = c(
            "10",
            "100",
            "1,000",
            "5,000"
        )
    ) +
    ggplot2::labs(
        title = "B  FDR-significant transcriptomic burden",
        x = NULL,
        y = "FDR < 0.05 genes (log scale)"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.0,
            lineheight = 0.9
        )
    )

# ==================================================================================================
# 18. PANEL C — SHARED-EFFECT VOLCANO
# ==================================================================================================

shared_colours <- c(
    "Not FDR-significant" = "#D0D0D0",
    "Positive shared effect" = "#D95F5F",
    "Negative shared effect" = "#3C78B5"
)

pC <- ggplot2::ggplot(
    shared_plot,
    ggplot2::aes(
        x = logFC,
        y = minus_log10_FDR,
        colour = DE_class
    )
) +
    ggplot2::geom_hline(
        yintercept = -log10(
            0.05
        ),
        linetype = "dashed",
        linewidth = 0.42,
        colour = "#555555"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.28,
        colour = "#999999"
    ) +
    ggplot2::geom_point(
        size = 0.70,
        alpha = 0.53
    ) +
    ggplot2::scale_colour_manual(
        values = shared_colours,
        drop = FALSE
    ) +
    ggplot2::labs(
        title = "C  Shared schizophrenia effect",
        x = "Shared-component logFC",
        y = expression(
            -log[10](FDR)
        ),
        colour = NULL
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom"
    )

if (
    nrow(shared_labels) > 0L &&
    USE_GGREPEL
) {

    pC <- pC +
        ggrepel::geom_text_repel(
            data = shared_labels,
            ggplot2::aes(
                label = display_label
            ),
            family = FONT_FAMILY,
            size = 2.3,
            box.padding = 0.25,
            point.padding = 0.13,
            segment.size = 0.22,
            max.overlaps = Inf,
            seed = 2026,
            show.legend = FALSE
        )

} else if (
    nrow(shared_labels) > 0L
) {

    pC <- pC +
        ggplot2::geom_text(
            data = shared_labels,
            ggplot2::aes(
                label = display_label
            ),
            family = FONT_FAMILY,
            size = 2.25,
            vjust = -0.55,
            check_overlap = TRUE,
            show.legend = FALSE
        )
}

# ==================================================================================================
# 19. PANEL D — REGIONAL-DEVIATION VOLCANO
# ==================================================================================================

deviation_colours <- c(
    "Not FDR-significant" = "#D5D5D5",
    "Positive deviation" = "#009E73",
    "Negative deviation" = "#CC79A7"
)

pD <- ggplot2::ggplot(
    deviation_plot,
    ggplot2::aes(
        x = logFC,
        y = minus_log10_FDR,
        colour = DE_class
    )
) +
    ggplot2::geom_hline(
        yintercept = -log10(
            0.05
        ),
        linetype = "dashed",
        linewidth = 0.42,
        colour = "#555555"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.28,
        colour = "#999999"
    ) +
    ggplot2::geom_point(
        size = 0.72,
        alpha = 0.55
    ) +
    ggplot2::scale_colour_manual(
        values = deviation_colours,
        drop = FALSE
    ) +
    ggplot2::labs(
        title = "D  Regional-deviation contrast",
        x = "Regional-deviation logFC",
        y = expression(
            -log[10](FDR)
        ),
        colour = NULL
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom"
    )

if (
    nrow(deviation_labels) > 0L &&
    USE_GGREPEL
) {

    pD <- pD +
        ggrepel::geom_text_repel(
            data = deviation_labels,
            ggplot2::aes(
                label = display_label
            ),
            family = FONT_FAMILY,
            size = 2.2,
            box.padding = 0.27,
            point.padding = 0.13,
            segment.size = 0.22,
            max.overlaps = Inf,
            seed = 2026,
            show.legend = FALSE
        )

} else if (
    nrow(deviation_labels) > 0L
) {

    pD <- pD +
        ggplot2::geom_text(
            data = deviation_labels,
            ggplot2::aes(
                label = display_label
            ),
            family = FONT_FAMILY,
            size = 2.15,
            vjust = -0.55,
            check_overlap = TRUE,
            show.legend = FALSE
        )
}

# ==================================================================================================
# 20. DRAW FINAL 2 x 2 FIGURE
# ==================================================================================================

draw_figure2 <- function() {

    grid::grid.newpage()

    grid::pushViewport(
        grid::viewport(
            layout = grid::grid.layout(
                nrow = 2,
                ncol = 2,
                widths = grid::unit(
                    c(
                        1,
                        1
                    ),
                    "null"
                ),
                heights = grid::unit(
                    c(
                        1,
                        1
                    ),
                    "null"
                )
            )
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pD,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 21. SAVE VECTOR PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 6.2,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure2()

grDevices::dev.off()

# ==================================================================================================
# 22. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 6.2,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure2()

grDevices::dev.off()

# ==================================================================================================
# 23. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 6.2,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure2()

grDevices::dev.off()

# ==================================================================================================
# 24. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    PANEL_A_PATH,
    PANEL_B_PATH,
    PANEL_C_PATH,
    PANEL_D_PATH
)

if (!all(
    file.exists(
        outputs
    )
)) {

    stop(
        "One or more final Figure 2 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {

    stop(
        "One or more final Figure 2 outputs are zero-byte."
    )
}

cat(
    "Final Figure 2 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 25. FIGURE SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11C.3T — FINAL MAIN FIGURE 2",
    "==============================================================================================================",

    "",

    "Canonical active STEP03A sources used.",

    "No differential-expression analysis was rerun.",

    "",

    paste0(
        "DLPFC FDR<0.05 genes = ",
        FDR_COUNTS["DLPFC"]
    ),

    paste0(
        "HIPPO FDR<0.05 genes = ",
        FDR_COUNTS["HIPPO"]
    ),

    paste0(
        "Shared FDR<0.05 genes = ",
        FDR_COUNTS["Shared"]
    ),

    paste0(
        "Regional-deviation FDR<0.05 genes = ",
        FDR_COUNTS["Deviation"]
    ),

    "",

    paste0(
        "DLPFC-HIPPO logFC correlation = ",
        format(
            REGION_COR,
            digits = 12
        )
    ),

    paste0(
        "DLPFC-HIPPO sign concordance = ",
        format(
            SIGN_CONCORDANCE,
            digits = 12
        )
    ),

    "",

    paste0(
        "Shared volcano labelled genes = ",
        nrow(
            shared_labels
        )
    ),

    paste0(
        "Regional-deviation labelled genes = ",
        nrow(
            deviation_labels
        )
    ),

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    FIGURE_SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 26. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    FIGURE_SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 27. FINAL CONSOLE REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFINAL FIGURE 2 GENERATION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Canonical source state: PASS\n"
)

cat(
    "DLPFC / HIPPO / Shared / Deviation FDR counts: ",
    paste(
        FDR_COUNTS,
        collapse = " / "
    ),
    "\n",
    sep = ""
)

cat(
    "Regional logFC correlation: ",
    sprintf(
        "%.12f",
        REGION_COR
    ),
    "\n",
    sep = ""
)

cat(
    "Regional sign concordance: ",
    sprintf(
        "%.12f",
        SIGN_CONCORDANCE
    ),
    "\n",
    sep = ""
)

cat(
    "Shared labelled genes: ",
    nrow(
        shared_labels
    ),
    "\n",
    sep = ""
)

cat(
    "Deviation labelled genes: ",
    nrow(
        deviation_labels
    ),
    "\n",
    sep = ""
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3T COMPLETE — FINAL FIGURE 2 GENERATED FROM CANONICAL RESTORED STEP03A RESULTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.3T — FINAL MAIN FIGURE 2
SHARED AND REGION-DEPENDENT SCHIZOPHRENIA TRANSCRIPTOMIC ARCHITECTURE
Project sentinel: PASS
ggrepel available: TRUE
Figure font: Times New Roman
Canonical active STEP03A source files: PASS
Four canonical 23,740-gene DE tables: PASS
Canonical FDR counts: 2949 / 2648 / 5348 / 11 — PASS
Canonical regional correlation: 0.608869223845 — PASS
Canonical sign concordance: 0.734962089301 — PASS


agg_record_211c1a4d6fff 
                      2

agg_record_211c1a4d6fff 
                      2

agg_record_211c1a4d6fff 
                      2

Final Figure 2 outputs created and non-zero: PASS

FINAL FIGURE 2 GENERATION SUMMARY
Canonical source state: PASS
DLPFC / HIPPO / Shared / Deviation FDR counts: 2949 / 2648 / 5348 / 11
Regional logFC correlation: 0.608869223845
Regional sign concordance: 0.734962089301
Shared labelled genes: 8
Deviation labelled genes: 11

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure2_final_TNR_v3/Figure2_Final_TimesNewRoman_v3.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure2_final_TNR_v3/Figure2_Final_TimesNewRoman_v3.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure2_final_TNR_v3/Figure2_Final_TimesNewRoman_v3.png

STEP 11C.3T COMPLETE — FINAL FIGURE 2 GENERATED FROM CANONICAL RESTORED STEP03A RESULTS


In [94]:
# ==================================================================================================
# STEP 11C.3U — FINAL FIGURE 2 — VISUAL QC CORRECTION
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   Correct ONLY the visual issues seen in Figure 2 v3:
#
#   - prevent legend clipping in Panels A/C/D
#   - shorten legend labels without changing scientific meaning
#   - improve regional-deviation gene-label spacing
#   - retain canonical numerical results unchanged
#
# NO ANALYSIS IS RERUN:
#   - no voom
#   - no limma
#   - no duplicateCorrelation
#   - no model fitting
#
# INPUT
#   Restored canonical STEP03A CSV files
#
# OUTPUT
#   Figure2_Final_TimesNewRoman_v4
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3U — FINAL FIGURE 2 — VISUAL QC CORRECTION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        project_id
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "ggrepel",
    quietly = TRUE
)) {
    stop("ggrepel unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required plotting packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. CANONICAL INPUT PATHS
# ==================================================================================================

DE_DIR <- file.path(
    PROJECT_ROOT,
    "05_differential_expression"
)

DLPFC_PATH <- file.path(
    DE_DIR,
    "step03a_DLPFC_SCZ_vs_Control_v1.csv"
)

HIPPO_PATH <- file.path(
    DE_DIR,
    "step03a_HIPPO_SCZ_vs_Control_v1.csv"
)

SHARED_PATH <- file.path(
    DE_DIR,
    "step03a_Shared_SCZ_effect_v1.csv"
)

DEVIATION_PATH <- file.path(
    DE_DIR,
    "step03a_Regional_Deviation_v1.csv"
)

SUMMARY_PATH <- file.path(
    DE_DIR,
    "step03a_DE_summary_v1.csv"
)

RESTORE_MANIFEST_PATH <- file.path(
    DE_DIR,
    "STEP03A_CANONICAL_RESTORE_MANIFEST_v2.csv"
)

required_inputs <- c(
    DLPFC_PATH,
    HIPPO_PATH,
    SHARED_PATH,
    DEVIATION_PATH,
    SUMMARY_PATH,
    RESTORE_MANIFEST_PATH
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {
    stop(
        "Required canonical input(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat("Canonical STEP03A inputs: PASS\n")

# ==================================================================================================
# 5. LOAD CANONICAL RESULTS
# ==================================================================================================

dlpfc <- read.csv(
    DLPFC_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

hippo <- read.csv(
    HIPPO_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

shared <- read.csv(
    SHARED_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

deviation <- read.csv(
    DEVIATION_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

de_summary <- read.csv(
    SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 6. STRICT CANONICAL VALIDATION
# ==================================================================================================

if (
    nrow(dlpfc) != 23740L ||
    nrow(hippo) != 23740L ||
    nrow(shared) != 23740L ||
    nrow(deviation) != 23740L
) {
    stop("Canonical DE table dimension failure.")
}

hippo_index <- match(
    dlpfc$ensemblID_stable,
    hippo$ensemblID_stable
)

if (anyNA(hippo_index)) {
    stop("DLPFC/HIPPO stable-Ensembl alignment failure.")
}

REGION_COR <- stats::cor(
    dlpfc$logFC,
    hippo$logFC[hippo_index],
    use = "complete.obs",
    method = "pearson"
)

SIGN_CONCORDANCE <- mean(
    sign(dlpfc$logFC) ==
        sign(hippo$logFC[hippo_index]),
    na.rm = TRUE
)

FDR_COUNTS <- c(
    DLPFC = sum(
        dlpfc$adj.P.Val < 0.05,
        na.rm = TRUE
    ),
    HIPPO = sum(
        hippo$adj.P.Val < 0.05,
        na.rm = TRUE
    ),
    Shared = sum(
        shared$adj.P.Val < 0.05,
        na.rm = TRUE
    ),
    Deviation = sum(
        deviation$adj.P.Val < 0.05,
        na.rm = TRUE
    )
)

EXPECTED_COUNTS <- c(
    DLPFC = 2949L,
    HIPPO = 2648L,
    Shared = 5348L,
    Deviation = 11L
)

if (!identical(
    as.integer(FDR_COUNTS),
    as.integer(EXPECTED_COUNTS)
)) {
    stop("Canonical FDR counts changed.")
}

if (
    abs(
        REGION_COR -
        0.608869223845
    ) > 5e-10
) {
    stop("Canonical regional correlation changed.")
}

if (
    abs(
        SIGN_CONCORDANCE -
        0.734962089301
    ) > 5e-10
) {
    stop("Canonical sign concordance changed.")
}

cat(
    "Canonical FDR counts: 2949 / 2648 / 5348 / 11 — PASS\n"
)

cat(
    "Canonical regional correlation: ",
    sprintf("%.12f", REGION_COR),
    " — PASS\n",
    sep = ""
)

cat(
    "Canonical sign concordance: ",
    sprintf("%.12f", SIGN_CONCORDANCE),
    " — PASS\n",
    sep = ""
)

# ==================================================================================================
# 7. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "main",
    "Figure2_final_TNR_v4"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "Figure2_Final_TimesNewRoman_v4.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "Figure2_Final_TimesNewRoman_v4.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "Figure2_Final_TimesNewRoman_v4.png"
)

SUMMARY_OUT_PATH <- file.path(
    OUT_DIR,
    "Figure2_final_summary_v4.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Figure2_final_manifest_v4.csv"
)

# ==================================================================================================
# 8. PANEL A SOURCE
# ==================================================================================================

regional <- data.frame(
    ensemblID_stable = dlpfc$ensemblID_stable,
    gene_symbol = dlpfc$gene_symbol,
    DLPFC_logFC = dlpfc$logFC,
    HIPPO_logFC = hippo$logFC[hippo_index],
    DLPFC_FDR = dlpfc$adj.P.Val,
    HIPPO_FDR = hippo$adj.P.Val[hippo_index],
    stringsAsFactors = FALSE
)

regional$significance_class <- "Neither"

regional$significance_class[
    regional$DLPFC_FDR < 0.05 &
    regional$HIPPO_FDR >= 0.05
] <- "DLPFC only"

regional$significance_class[
    regional$DLPFC_FDR >= 0.05 &
    regional$HIPPO_FDR < 0.05
] <- "HIPPO only"

regional$significance_class[
    regional$DLPFC_FDR < 0.05 &
    regional$HIPPO_FDR < 0.05
] <- "Both"

regional$significance_class <- factor(
    regional$significance_class,
    levels = c(
        "Neither",
        "DLPFC only",
        "HIPPO only",
        "Both"
    )
)

# ==================================================================================================
# 9. PANEL B SOURCE
# ==================================================================================================

count_df <- data.frame(
    contrast = factor(
        c(
            "DLPFC",
            "HIPPO",
            "Shared",
            "Regional\ndeviation"
        ),
        levels = c(
            "DLPFC",
            "HIPPO",
            "Shared",
            "Regional\ndeviation"
        )
    ),
    count = as.numeric(FDR_COUNTS),
    stringsAsFactors = FALSE
)

# ==================================================================================================
# 10. VOLCANO SOURCES
# ==================================================================================================

safe_label <- function(
    symbol,
    ensembl
) {

    symbol <- as.character(symbol)
    ensembl <- as.character(ensembl)

    bad <- (
        is.na(symbol) |
        trimws(symbol) == ""
    )

    symbol[bad] <- ensembl[bad]

    symbol
}

prepare_volcano <- function(
    x,
    positive_class,
    negative_class
) {

    out <- x

    out$display_label <- safe_label(
        out$gene_symbol,
        out$ensemblID_stable
    )

    out$FDR_plot <- pmax(
        out$adj.P.Val,
        1e-300
    )

    out$minus_log10_FDR <- -log10(
        out$FDR_plot
    )

    out$effect_class <- "Not significant"

    out$effect_class[
        out$adj.P.Val < 0.05 &
        out$logFC > 0
    ] <- positive_class

    out$effect_class[
        out$adj.P.Val < 0.05 &
        out$logFC < 0
    ] <- negative_class

    out
}

shared_plot <- prepare_volcano(
    shared,
    "Positive",
    "Negative"
)

deviation_plot <- prepare_volcano(
    deviation,
    "Positive",
    "Negative"
)

# ==================================================================================================
# 11. LABELS
# ==================================================================================================

shared_named <- shared_plot[
    shared_plot$adj.P.Val < 0.05 &
    !is.na(shared_plot$gene_symbol) &
    trimws(shared_plot$gene_symbol) != "",
    ,
    drop = FALSE
]

shared_positive <- shared_named[
    shared_named$logFC > 0,
    ,
    drop = FALSE
]

shared_negative <- shared_named[
    shared_named$logFC < 0,
    ,
    drop = FALSE
]

shared_positive <- shared_positive[
    order(
        shared_positive$adj.P.Val,
        -abs(shared_positive$logFC)
    ),
    ,
    drop = FALSE
]

shared_negative <- shared_negative[
    order(
        shared_negative$adj.P.Val,
        -abs(shared_negative$logFC)
    ),
    ,
    drop = FALSE
]

shared_labels <- rbind(
    head(shared_positive, 4L),
    head(shared_negative, 4L)
)

deviation_labels <- deviation_plot[
    deviation_plot$adj.P.Val < 0.05,
    ,
    drop = FALSE
]

deviation_labels <- deviation_labels[
    order(
        deviation_labels$adj.P.Val
    ),
    ,
    drop = FALSE
]

if (nrow(deviation_labels) != 11L) {
    stop(
        "Expected exactly 11 regional-deviation labels."
    )
}

# ==================================================================================================
# 12. SAVE FIGURE SOURCE DATA
# ==================================================================================================

write.csv(
    regional,
    file.path(
        SOURCE_DIR,
        "Figure2_PanelA_source_v4.csv"
    ),
    row.names = FALSE
)

write.csv(
    count_df,
    file.path(
        SOURCE_DIR,
        "Figure2_PanelB_source_v4.csv"
    ),
    row.names = FALSE
)

write.csv(
    shared_plot,
    file.path(
        SOURCE_DIR,
        "Figure2_PanelC_source_v4.csv"
    ),
    row.names = FALSE
)

write.csv(
    deviation_plot,
    file.path(
        SOURCE_DIR,
        "Figure2_PanelD_source_v4.csv"
    ),
    row.names = FALSE
)

# ==================================================================================================
# 13. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.5,
            hjust = 0,
            margin = ggplot2::margin(
                b = 5
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.5
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.3,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.7
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.4
        ),

        legend.key.width = grid::unit(
            0.9,
            "lines"
        ),

        legend.key.height = grid::unit(
            0.8,
            "lines"
        ),

        legend.spacing.x = grid::unit(
            0.18,
            "cm"
        ),

        legend.box.spacing = grid::unit(
            0.08,
            "cm"
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            7,
            8,
            7,
            8
        )
    )

# ==================================================================================================
# 14. PANEL A
# ==================================================================================================

panel_A_colours <- c(
    "Neither" = "#C7C7C7",
    "DLPFC only" = "#4E79A7",
    "HIPPO only" = "#F28E2B",
    "Both" = "#8E63B0"
)

pA <- ggplot2::ggplot(
    regional,
    ggplot2::aes(
        x = DLPFC_logFC,
        y = HIPPO_logFC,
        colour = significance_class
    )
) +
    ggplot2::geom_hline(
        yintercept = 0,
        linewidth = 0.28,
        colour = "#B0B0B0"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.28,
        colour = "#B0B0B0"
    ) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.42,
        colour = "#555555"
    ) +
    ggplot2::geom_point(
        size = 0.78,
        alpha = 0.48
    ) +
    ggplot2::scale_colour_manual(
        values = panel_A_colours,
        drop = FALSE
    ) +
    ggplot2::annotate(
        "text",
        x = -Inf,
        y = Inf,
        hjust = -0.05,
        vjust = 1.18,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.55,
        label = paste0(
            "r = ",
            sprintf("%.3f", REGION_COR),
            "\nSign concordance = ",
            sprintf(
                "%.1f%%",
                100 * SIGN_CONCORDANCE
            )
        )
    ) +
    ggplot2::labs(
        title = "A  Cross-region diagnosis-effect concordance",
        x = "DLPFC logFC (schizophrenia vs control)",
        y = "HIPPO logFC (schizophrenia vs control)",
        colour = "FDR < 0.05"
    ) +
    ggplot2::guides(
        colour = ggplot2::guide_legend(
            nrow = 2,
            byrow = TRUE,
            override.aes = list(
                size = 2.2,
                alpha = 1
            )
        )
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom"
    )

# ==================================================================================================
# 15. PANEL B
# ==================================================================================================

panel_B_colours <- c(
    "DLPFC" = "#59A14F",
    "HIPPO" = "#E15759",
    "Shared" = "#B07AA1",
    "Regional\ndeviation" = "#76B7B2"
)

pB <- ggplot2::ggplot(
    count_df,
    ggplot2::aes(
        x = contrast,
        y = count,
        colour = contrast
    )
) +
    ggplot2::geom_segment(
        ggplot2::aes(
            xend = contrast,
            y = 8,
            yend = count
        ),
        linewidth = 1.0
    ) +
    ggplot2::geom_point(
        size = 4.0
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = format(
                count,
                big.mark = ","
            )
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.8,
        vjust = -0.75
    ) +
    ggplot2::scale_colour_manual(
        values = panel_B_colours,
        guide = "none"
    ) +
    ggplot2::scale_y_log10(
        limits = c(
            8,
            8000
        ),
        breaks = c(
            10,
            100,
            1000,
            5000
        ),
        labels = c(
            "10",
            "100",
            "1,000",
            "5,000"
        )
    ) +
    ggplot2::labs(
        title = "B  FDR-significant transcriptomic burden",
        x = NULL,
        y = "FDR < 0.05 genes (log scale)"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.0,
            lineheight = 0.9
        )
    )

# ==================================================================================================
# 16. PANEL C
# ==================================================================================================

panel_C_colours <- c(
    "Negative" = "#3C78B5",
    "Not significant" = "#D0D0D0",
    "Positive" = "#D95F5F"
)

shared_plot$effect_class <- factor(
    shared_plot$effect_class,
    levels = c(
        "Negative",
        "Not significant",
        "Positive"
    )
)

pC <- ggplot2::ggplot(
    shared_plot,
    ggplot2::aes(
        x = logFC,
        y = minus_log10_FDR,
        colour = effect_class
    )
) +
    ggplot2::geom_hline(
        yintercept = -log10(0.05),
        linetype = "dashed",
        linewidth = 0.42,
        colour = "#555555"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.28,
        colour = "#999999"
    ) +
    ggplot2::geom_point(
        size = 0.70,
        alpha = 0.53
    ) +
    ggrepel::geom_text_repel(
        data = shared_labels,
        ggplot2::aes(
            label = display_label
        ),
        family = FONT_FAMILY,
        size = 2.25,
        box.padding = 0.28,
        point.padding = 0.14,
        segment.size = 0.22,
        min.segment.length = 0,
        force = 1.1,
        max.overlaps = Inf,
        seed = 2026,
        show.legend = FALSE
    ) +
    ggplot2::scale_colour_manual(
        values = panel_C_colours,
        drop = FALSE
    ) +
    ggplot2::scale_y_continuous(
        expand = ggplot2::expansion(
            mult = c(
                0.03,
                0.10
            )
        )
    ) +
    ggplot2::labs(
        title = "C  Shared schizophrenia effect",
        x = "Shared-component logFC",
        y = expression(
            -log[10](FDR)
        ),
        colour = "Shared effect"
    ) +
    ggplot2::guides(
        colour = ggplot2::guide_legend(
            nrow = 1,
            byrow = TRUE,
            override.aes = list(
                size = 2.1,
                alpha = 1
            )
        )
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom"
    )

# ==================================================================================================
# 17. PANEL D
# ==================================================================================================

panel_D_colours <- c(
    "Negative" = "#CC79A7",
    "Not significant" = "#D5D5D5",
    "Positive" = "#009E73"
)

deviation_plot$effect_class <- factor(
    deviation_plot$effect_class,
    levels = c(
        "Negative",
        "Not significant",
        "Positive"
    )
)

pD <- ggplot2::ggplot(
    deviation_plot,
    ggplot2::aes(
        x = logFC,
        y = minus_log10_FDR,
        colour = effect_class
    )
) +
    ggplot2::geom_hline(
        yintercept = -log10(0.05),
        linetype = "dashed",
        linewidth = 0.42,
        colour = "#555555"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.28,
        colour = "#999999"
    ) +
    ggplot2::geom_point(
        size = 0.72,
        alpha = 0.55
    ) +
    ggrepel::geom_text_repel(
        data = deviation_labels,
        ggplot2::aes(
            label = display_label
        ),
        family = FONT_FAMILY,
        size = 2.15,
        box.padding = 0.32,
        point.padding = 0.16,
        segment.size = 0.22,
        min.segment.length = 0,
        force = 1.5,
        force_pull = 0.8,
        max.time = 3,
        max.iter = 20000,
        max.overlaps = Inf,
        seed = 2026,
        show.legend = FALSE
    ) +
    ggplot2::scale_colour_manual(
        values = panel_D_colours,
        drop = FALSE
    ) +
    ggplot2::scale_x_continuous(
        expand = ggplot2::expansion(
            mult = c(
                0.10,
                0.12
            )
        )
    ) +
    ggplot2::scale_y_continuous(
        expand = ggplot2::expansion(
            mult = c(
                0.03,
                0.16
            )
        )
    ) +
    ggplot2::labs(
        title = "D  Regional-deviation contrast",
        x = "Regional-deviation logFC",
        y = expression(
            -log[10](FDR)
        ),
        colour = "Regional deviation"
    ) +
    ggplot2::guides(
        colour = ggplot2::guide_legend(
            nrow = 1,
            byrow = TRUE,
            override.aes = list(
                size = 2.1,
                alpha = 1
            )
        )
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom"
    )

# ==================================================================================================
# 18. DRAW FIGURE
# ==================================================================================================

draw_figure2 <- function() {

    grid::grid.newpage()

    grid::pushViewport(
        grid::viewport(
            layout = grid::grid.layout(
                nrow = 2,
                ncol = 2,
                widths = grid::unit(
                    c(
                        1,
                        1
                    ),
                    "null"
                ),
                heights = grid::unit(
                    c(
                        1,
                        1
                    ),
                    "null"
                )
            )
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pD,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 19. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 6.35,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure2()

grDevices::dev.off()

# ==================================================================================================
# 20. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 6.35,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure2()

grDevices::dev.off()

# ==================================================================================================
# 21. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 6.35,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure2()

grDevices::dev.off()

# ==================================================================================================
# 22. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH
)

if (!all(
    file.exists(outputs)
)) {
    stop(
        "One or more Figure 2 v4 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(output_info$size) |
    output_info$size <= 0
)) {
    stop(
        "One or more Figure 2 v4 outputs are zero-byte."
    )
}

cat(
    "Figure 2 v4 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 23. SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11C.3U — FIGURE 2 FINAL VISUAL QC",
    "==============================================================================================================",

    "Canonical restored STEP03A sources used.",

    "No analysis rerun.",

    "",

    paste0(
        "DLPFC FDR<0.05 = ",
        FDR_COUNTS["DLPFC"]
    ),

    paste0(
        "HIPPO FDR<0.05 = ",
        FDR_COUNTS["HIPPO"]
    ),

    paste0(
        "Shared FDR<0.05 = ",
        FDR_COUNTS["Shared"]
    ),

    paste0(
        "Regional deviation FDR<0.05 = ",
        FDR_COUNTS["Deviation"]
    ),

    paste0(
        "Regional logFC correlation = ",
        format(
            REGION_COR,
            digits = 12
        )
    ),

    paste0(
        "Regional sign concordance = ",
        format(
            SIGN_CONCORDANCE,
            digits = 12
        )
    ),

    "",

    "Visual changes only:",

    "- legend labels shortened",

    "- Panel A legend wrapped safely",

    "- Panels C/D legends constrained within panel width",

    "- Panel D label spacing increased",

    "- no numerical or analytical changes",

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_OUT_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 24. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_OUT_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 25. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFIGURE 2 v4 — FINAL VISUAL QC SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Canonical source validation: PASS\n"
)

cat(
    "Numerical results changed: NO\n"
)

cat(
    "Legend clipping correction applied: YES\n"
)

cat(
    "Deviation-label spacing correction applied: YES\n"
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.3U COMPLETE — FIGURE 2 FINAL VISUAL QC VERSION GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.3U — FINAL FIGURE 2 — VISUAL QC CORRECTION
Project sentinel: PASS
Required plotting packages: PASS
Figure font: Times New Roman
Canonical STEP03A inputs: PASS
Canonical FDR counts: 2949 / 2648 / 5348 / 11 — PASS
Canonical regional correlation: 0.608869223845 — PASS
Canonical sign concordance: 0.734962089301 — PASS


agg_record_211c23e02030 
                      2

agg_record_211c23e02030 
                      2

agg_record_211c23e02030 
                      2

Figure 2 v4 outputs created and non-zero: PASS

FIGURE 2 v4 — FINAL VISUAL QC SUMMARY
Canonical source validation: PASS
Numerical results changed: NO
Legend clipping correction applied: YES
Deviation-label spacing correction applied: YES

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure2_final_TNR_v4/Figure2_Final_TimesNewRoman_v4.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure2_final_TNR_v4/Figure2_Final_TimesNewRoman_v4.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure2_final_TNR_v4/Figure2_Final_TimesNewRoman_v4.png

STEP 11C.3U COMPLETE — FIGURE 2 FINAL VISUAL QC VERSION GENERATED


In [96]:
# ==================================================================================================
# STEP 11C.4R — MAIN FIGURE 3 — FULLY CORRECTED
# LEAKAGE-FREE NESTED-CV PERFORMANCE AND MODEL COMPARISON
# --------------------------------------------------------------------------------------------------
# PANELS
#
#   A — Model F donor-level outer-fold AUROC across 5 repeats x 5 folds
#
#   B — Repeated-OOF donor AUROC in the SAME 454-donor population:
#         Covariate-only
#         Model C/E pooled/shared-only
#         Model F shared + deviation
#         Model D independent regional models
#
#   C — Paired donor-bootstrap delta AUROC for Model F:
#         F - Covariate-only
#         F - C/E pooled/shared-only
#         F - D independent regions
#
# IMPORTANT
#   - SAVED outputs only
#   - NO glmnet fitting
#   - NO nested CV rerun
#   - NO bootstrap rerun
#   - NO AUROC optimization
#   - Model A/B regional-only baselines excluded from Panel B because populations differ
#   - Model D retained as required comparator
#   - Times New Roman
#   - PDF vector + 600-dpi TIFF + 300-dpi PNG
#
# CORRECTIONS
#   1. R reserved word `repeat` is NOT used as a data-frame column name.
#      Column is now `outer_repeat`.
#   2. Panel C text annotation now explicitly includes y = comparison.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.4R — MAIN FIGURE 3 — FULLY CORRECTED\n",
    "LEAKAGE-FREE NESTED-CV PERFORMANCE AND MODEL COMPARISON\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. REQUIRED PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. SAVED INPUT PATHS
# ==================================================================================================

NESTED_DIR <- file.path(
    PROJECT_ROOT,
    "06_models",
    "nested_cv_5x5_v1"
)

MODEL_COMP_DIR <- file.path(
    PROJECT_ROOT,
    "06_models",
    "model_comparison_v1"
)

COV_DIR <- file.path(
    PROJECT_ROOT,
    "06_models",
    "baselines",
    "model_covariate_only_sensitivity_v1"
)

FOLD_PATH <- file.path(
    NESTED_DIR,
    "ALL_25_OUTER_FOLD_SUMMARY_v1.csv"
)

FINAL_PERF_PATH <- file.path(
    NESTED_DIR,
    "NESTED_CV_FINAL_PERFORMANCE_v1.csv"
)

MODEL_PERF_PATH <- file.path(
    MODEL_COMP_DIR,
    "STEP05C_model_performance_summary_v1.csv"
)

COMPARISON_PATH <- file.path(
    MODEL_COMP_DIR,
    "STEP05C_paired_comparison_summary_v1.csv"
)

MODEL_COMPARISON_NOTES_PATH <- file.path(
    MODEL_COMP_DIR,
    "STEP05C_MODEL_COMPARISON_NOTES_v1.txt"
)

COV_SUMMARY_PATH <- file.path(
    COV_DIR,
    "covariate_only_sensitivity_summary_v1.csv"
)

required_inputs <- c(
    FOLD_PATH,
    FINAL_PERF_PATH,
    MODEL_PERF_PATH,
    COMPARISON_PATH,
    MODEL_COMPARISON_NOTES_PATH,
    COV_SUMMARY_PATH
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required frozen model artifact(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat(
    "Required frozen model-performance artifacts: PASS\n"
)

# ==================================================================================================
# 5. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "main",
    "Figure3_final_TNR_v1"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "Figure3_NestedCV_ModelComparison_TimesNewRoman_v1.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "Figure3_NestedCV_ModelComparison_TimesNewRoman_v1.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "Figure3_NestedCV_ModelComparison_TimesNewRoman_v1.png"
)

PANEL_A_PATH <- file.path(
    SOURCE_DIR,
    "Figure3_PanelA_outer_fold_AUROC_source_v1.csv"
)

PANEL_B_PATH <- file.path(
    SOURCE_DIR,
    "Figure3_PanelB_repeated_OOF_AUROC_source_v1.csv"
)

PANEL_C_PATH <- file.path(
    SOURCE_DIR,
    "Figure3_PanelC_delta_AUROC_source_v1.csv"
)

SUMMARY_OUT_PATH <- file.path(
    OUT_DIR,
    "Figure3_generation_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Figure3_generation_manifest_v1.csv"
)

# ==================================================================================================
# 6. LOAD SAVED OUTPUTS
# ==================================================================================================

folds <- read.csv(
    FOLD_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

final_perf <- read.csv(
    FINAL_PERF_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

model_perf <- read.csv(
    MODEL_PERF_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

comparisons <- read.csv(
    COMPARISON_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

cov_summary <- read.csv(
    COV_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 7. SCHEMA VALIDATION
# ==================================================================================================

required_fold_cols <- c(
    "outer_repeat",
    "outer_fold",
    "outer_donor_auc",
    "outer_sample_auc",
    "outer_DLPFC_auc",
    "outer_HIPPO_auc",
    "n_shared_nonzero",
    "n_deviation_nonzero"
)

missing_fold_cols <- setdiff(
    required_fold_cols,
    names(folds)
)

if (length(missing_fold_cols) > 0L) {

    stop(
        "ALL_25_OUTER_FOLD_SUMMARY missing required field(s): ",
        paste(
            missing_fold_cols,
            collapse = ", "
        )
    )
}

if (!all(
    c(
        "metric",
        "value"
    ) %in%
        names(final_perf)
)) {

    stop(
        "NESTED_CV_FINAL_PERFORMANCE schema changed."
    )
}

if (!all(
    c(
        "model",
        "evaluation_population",
        "n_donors",
        "repeated_OOF_AUROC"
    ) %in%
        names(model_perf)
)) {

    stop(
        "STEP05C model-performance schema changed."
    )
}

required_comparison_cols <- c(
    "comparison",
    "population",
    "n_donors",
    "mean_paired_fold_delta",
    "repeat_block_CI_low",
    "repeat_block_CI_high",
    "averaged_OOF_reference_AUROC",
    "averaged_OOF_comparator_AUROC",
    "averaged_OOF_delta_AUROC",
    "paired_donor_bootstrap_CI_low",
    "paired_donor_bootstrap_CI_high"
)

missing_comparison_cols <- setdiff(
    required_comparison_cols,
    names(comparisons)
)

if (length(missing_comparison_cols) > 0L) {

    stop(
        "STEP05C paired-comparison table missing required field(s): ",
        paste(
            missing_comparison_cols,
            collapse = ", "
        )
    )
}

if (!all(
    c(
        "metric",
        "value"
    ) %in%
        names(cov_summary)
)) {

    stop(
        "Covariate-only summary schema changed."
    )
}

cat("Input schemas: PASS\n")

# ==================================================================================================
# 8. EXACT METRIC HELPER
# ==================================================================================================

get_metric <- function(
    x,
    metric_name
) {

    idx <- which(
        as.character(
            x$metric
        ) ==
            metric_name
    )

    if (length(idx) != 1L) {

        stop(
            "Could not uniquely retrieve metric: ",
            metric_name
        )
    }

    value <- as.numeric(
        x$value[idx]
    )

    if (
        length(value) != 1L ||
        !is.finite(value)
    ) {

        stop(
            "Metric is non-finite: ",
            metric_name
        )
    }

    value
}

# ==================================================================================================
# 9. MODEL F CANONICAL PERFORMANCE
# ==================================================================================================

F_REPEATED_OOF <- get_metric(
    final_perf,
    "Repeated_OOF_donor_AUROC"
)

F_SAMPLE_OOF <- get_metric(
    final_perf,
    "Repeated_OOF_sample_AUROC"
)

F_DLPFC_OOF <- get_metric(
    final_perf,
    "Repeated_OOF_DLPFC_AUROC"
)

F_HIPPO_OOF <- get_metric(
    final_perf,
    "Repeated_OOF_HIPPO_AUROC"
)

F_MEAN_FOLD <- get_metric(
    final_perf,
    "Mean_outer_fold_donor_AUROC"
)

F_SD_FOLD <- get_metric(
    final_perf,
    "SD_outer_fold_donor_AUROC"
)

if (
    abs(
        F_REPEATED_OOF -
        0.8019743
    ) >
        5e-7
) {

    stop(
        "Model F repeated-OOF donor AUROC changed."
    )
}

if (
    abs(
        F_MEAN_FOLD -
        0.7868384
    ) >
        5e-7
) {

    stop(
        "Model F mean fold donor AUROC changed."
    )
}

if (
    abs(
        F_SD_FOLD -
        0.04047966
    ) >
        5e-7
) {

    stop(
        "Model F fold-AUROC SD changed."
    )
}

cat(
    "Model F repeated-OOF donor AUROC: ",
    sprintf(
        "%.7f",
        F_REPEATED_OOF
    ),
    " — PASS\n",
    sep = ""
)

# ==================================================================================================
# 10. PANEL A — 25 OUTER FOLDS
#
# IMPORTANT:
# `repeat` is an R reserved keyword.
# Therefore the plotting data column is named `outer_repeat`.
# ==================================================================================================

if (nrow(folds) != 25L) {

    stop(
        "Expected exactly 25 Model F outer-fold summaries; observed ",
        nrow(folds)
    )
}

if (
    !setequal(
        unique(
            as.integer(
                folds$outer_repeat
            )
        ),
        1:5
    )
) {

    stop(
        "Expected outer repeats 1-5."
    )
}

if (
    !setequal(
        unique(
            as.integer(
                folds$outer_fold
            )
        ),
        1:5
    )
) {

    stop(
        "Expected outer folds 1-5."
    )
}

if (any(
    !is.finite(
        folds$outer_donor_auc
    )
)) {

    stop(
        "Non-finite outer donor AUROC detected."
    )
}

panel_A <- data.frame(

    outer_repeat = as.integer(
        folds$outer_repeat
    ),

    outer_fold = as.integer(
        folds$outer_fold
    ),

    donor_AUROC = as.numeric(
        folds$outer_donor_auc
    ),

    shared_nonzero = as.integer(
        folds$n_shared_nonzero
    ),

    deviation_nonzero = as.integer(
        folds$n_deviation_nonzero
    ),

    stringsAsFactors = FALSE
)

panel_A <- panel_A[
    order(
        panel_A$outer_repeat,
        panel_A$outer_fold
    ),
    ,
    drop = FALSE
]

panel_A$repeat_factor <- factor(
    panel_A$outer_repeat,
    levels = 1:5,
    labels = paste0(
        "Repeat ",
        1:5
    )
)

if (
    abs(
        mean(
            panel_A$donor_AUROC
        ) -
        F_MEAN_FOLD
    ) >
        1e-10
) {

    stop(
        "Outer-fold mean does not match frozen final-performance table."
    )
}

if (
    abs(
        stats::sd(
            panel_A$donor_AUROC
        ) -
        F_SD_FOLD
    ) >
        1e-10
) {

    stop(
        "Outer-fold SD does not match frozen final-performance table."
    )
}

write.csv(
    panel_A,
    PANEL_A_PATH,
    row.names = FALSE,
    quote = TRUE
)

cat(
    "Model F 25-fold performance integrity: PASS\n"
)

# ==================================================================================================
# 11. PANEL B — ALL-DONOR REPEATED-OOF MODEL COMPARISON
# ==================================================================================================

required_models <- c(
    "C_E_pooled_shared_only",
    "D_independent_regions",
    "F_shared_plus_deviation"
)

for (model_name in required_models) {

    if (
        sum(
            model_perf$model ==
                model_name
        ) != 1L
    ) {

        stop(
            "Could not uniquely locate model in STEP05C performance table: ",
            model_name
        )
    }
}

get_model_auc <- function(
    model_name
) {

    idx <- which(
        model_perf$model ==
            model_name
    )

    value <- as.numeric(
        model_perf$repeated_OOF_AUROC[idx]
    )

    if (
        length(value) != 1L ||
        !is.finite(value)
    ) {

        stop(
            "Non-finite model AUROC for: ",
            model_name
        )
    }

    value
}

get_model_n <- function(
    model_name
) {

    idx <- which(
        model_perf$model ==
            model_name
    )

    value <- as.integer(
        model_perf$n_donors[idx]
    )

    if (
        length(value) != 1L ||
        is.na(value)
    ) {

        stop(
            "Invalid model donor count for: ",
            model_name
        )
    }

    value
}

CE_AUC <- get_model_auc(
    "C_E_pooled_shared_only"
)

D_AUC <- get_model_auc(
    "D_independent_regions"
)

F_AUC_FROM_COMPARISON <- get_model_auc(
    "F_shared_plus_deviation"
)

CE_N <- get_model_n(
    "C_E_pooled_shared_only"
)

D_N <- get_model_n(
    "D_independent_regions"
)

F_N <- get_model_n(
    "F_shared_plus_deviation"
)

COV_AUC <- get_metric(
    cov_summary,
    "Covariate_only_repeated_OOF_AUROC"
)

COV_F_AUC <- get_metric(
    cov_summary,
    "Model_F_repeated_OOF_AUROC"
)

if (
    CE_N != 454L ||
    D_N != 454L ||
    F_N != 454L
) {

    stop(
        "All-donor comparator population changed."
    )
}

if (
    abs(
        CE_AUC -
        0.7987926
    ) >
        5e-7
) {

    stop(
        "Model C/E AUROC changed."
    )
}

if (
    abs(
        D_AUC -
        0.81502784
    ) >
        5e-7
) {

    stop(
        "Model D AUROC changed."
    )
}

if (
    abs(
        F_AUC_FROM_COMPARISON -
        F_REPEATED_OOF
    ) >
        5e-7
) {

    stop(
        "Model F AUROC disagrees across frozen sources."
    )
}

if (
    abs(
        COV_AUC -
        0.6143507
    ) >
        5e-7
) {

    stop(
        "Covariate-only AUROC changed."
    )
}

if (
    abs(
        COV_F_AUC -
        F_REPEATED_OOF
    ) >
        5e-7
) {

    stop(
        "Covariate-only sensitivity source uses a different Model F AUROC."
    )
}

panel_B <- data.frame(

    model = factor(
        c(
            "Covariates only",
            "C/E shared-only",
            "F shared + deviation",
            "D independent regions"
        ),
        levels = c(
            "Covariates only",
            "C/E shared-only",
            "F shared + deviation",
            "D independent regions"
        )
    ),

    AUROC = c(
        COV_AUC,
        CE_AUC,
        F_REPEATED_OOF,
        D_AUC
    ),

    n_donors = c(
        454L,
        CE_N,
        F_N,
        D_N
    ),

    stringsAsFactors = FALSE
)

write.csv(
    panel_B,
    PANEL_B_PATH,
    row.names = FALSE,
    quote = TRUE
)

cat(
    "All-donor model comparison: PASS\n"
)

# ==================================================================================================
# 12. PANEL C — PAIRED DONOR-BOOTSTRAP DELTA AUROC
# ==================================================================================================

comparison_CE_name <-
    "F_minus_C_E_pooled_shared_only"

comparison_D_name <-
    "F_minus_D_independent_regions"

ce_index <- which(
    comparisons$comparison ==
        comparison_CE_name
)

d_index <- which(
    comparisons$comparison ==
        comparison_D_name
)

if (length(ce_index) != 1L) {

    stop(
        "F vs C/E comparison not uniquely found."
    )
}

if (length(d_index) != 1L) {

    stop(
        "F vs D comparison not uniquely found."
    )
}

CE_DELTA <- as.numeric(
    comparisons$averaged_OOF_delta_AUROC[
        ce_index
    ]
)

CE_CI_LOW <- as.numeric(
    comparisons$paired_donor_bootstrap_CI_low[
        ce_index
    ]
)

CE_CI_HIGH <- as.numeric(
    comparisons$paired_donor_bootstrap_CI_high[
        ce_index
    ]
)

D_DELTA <- as.numeric(
    comparisons$averaged_OOF_delta_AUROC[
        d_index
    ]
)

D_CI_LOW <- as.numeric(
    comparisons$paired_donor_bootstrap_CI_low[
        d_index
    ]
)

D_CI_HIGH <- as.numeric(
    comparisons$paired_donor_bootstrap_CI_high[
        d_index
    ]
)

COV_DELTA <- get_metric(
    cov_summary,
    "Delta_F_minus_covariates"
)

COV_CI_LOW <- get_metric(
    cov_summary,
    "Delta_95CI_low"
)

COV_CI_HIGH <- get_metric(
    cov_summary,
    "Delta_95CI_high"
)

if (
    abs(
        CE_DELTA -
        0.00318179
    ) >
        5e-7
) {

    stop(
        "F vs C/E averaged-OOF delta changed."
    )
}

if (
    abs(
        CE_CI_LOW -
        (-0.00108099)
    ) >
        5e-7 ||
    abs(
        CE_CI_HIGH -
        0.00766893
    ) >
        5e-7
) {

    stop(
        "F vs C/E donor-bootstrap CI changed."
    )
}

if (
    abs(
        D_DELTA -
        (-0.01305350)
    ) >
        5e-7
) {

    stop(
        "F vs D averaged-OOF delta changed."
    )
}

if (
    abs(
        D_CI_LOW -
        (-0.03085980)
    ) >
        5e-7 ||
    abs(
        D_CI_HIGH -
        0.00528259
    ) >
        5e-7
) {

    stop(
        "F vs D donor-bootstrap CI changed."
    )
}

if (
    abs(
        COV_DELTA -
        0.1876237
    ) >
        5e-7
) {

    stop(
        "F vs covariate-only delta changed."
    )
}

if (
    abs(
        COV_CI_LOW -
        0.1406101
    ) >
        5e-7 ||
    abs(
        COV_CI_HIGH -
        0.2374156
    ) >
        5e-7
) {

    stop(
        "F vs covariate-only donor-bootstrap CI changed."
    )
}

panel_C <- data.frame(

    comparison = factor(
        c(
            "F vs covariates only",
            "F vs C/E shared-only",
            "F vs D independent regions"
        ),
        levels = rev(
            c(
                "F vs covariates only",
                "F vs C/E shared-only",
                "F vs D independent regions"
            )
        )
    ),

    delta_AUROC = c(
        COV_DELTA,
        CE_DELTA,
        D_DELTA
    ),

    CI_low = c(
        COV_CI_LOW,
        CE_CI_LOW,
        D_CI_LOW
    ),

    CI_high = c(
        COV_CI_HIGH,
        CE_CI_HIGH,
        D_CI_HIGH
    ),

    n_donors = c(
        454L,
        454L,
        454L
    ),

    stringsAsFactors = FALSE
)

panel_C$display_text <- sprintf(
    "%+.3f [%.3f, %.3f]",
    panel_C$delta_AUROC,
    panel_C$CI_low,
    panel_C$CI_high
)

write.csv(
    panel_C,
    PANEL_C_PATH,
    row.names = FALSE,
    quote = TRUE
)

cat(
    "Paired donor-bootstrap comparison anchors: PASS\n"
)

# ==================================================================================================
# 13. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.5,
            hjust = 0,
            margin = ggplot2::margin(
                b = 6
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.5
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.3,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 8.2
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.8
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            8,
            9,
            8,
            9
        )
    )

# ==================================================================================================
# 14. PANEL A — OUTER-FOLD PERFORMANCE
# ==================================================================================================

repeat_colours <- c(
    "Repeat 1" = "#4E79A7",
    "Repeat 2" = "#F28E2B",
    "Repeat 3" = "#59A14F",
    "Repeat 4" = "#E15759",
    "Repeat 5" = "#B07AA1"
)

pA <- ggplot2::ggplot(
    panel_A,
    ggplot2::aes(
        x = outer_fold,
        y = donor_AUROC,
        group = repeat_factor,
        colour = repeat_factor
    )
) +
    ggplot2::geom_hline(
        yintercept = F_REPEATED_OOF,
        linetype = "dashed",
        linewidth = 0.65,
        colour = "#303030"
    ) +
    ggplot2::geom_line(
        linewidth = 0.65,
        alpha = 0.82
    ) +
    ggplot2::geom_point(
        size = 2.2,
        alpha = 0.96
    ) +
    ggplot2::annotate(
        "text",
        x = 5.15,
        y = F_REPEATED_OOF + 0.006,
        label = paste0(
            "Repeated OOF = ",
            sprintf(
                "%.3f",
                F_REPEATED_OOF
            )
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.55,
        hjust = 1
    ) +
    ggplot2::scale_colour_manual(
        values = repeat_colours
    ) +
    ggplot2::scale_x_continuous(
        breaks = 1:5,
        limits = c(
            0.8,
            5.2
        )
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            0.64,
            0.89
        ),
        breaks = seq(
            0.65,
            0.85,
            by = 0.05
        )
    ) +
    ggplot2::labs(
        title = "A  Nested-CV donor discrimination across outer folds",
        x = "Outer fold",
        y = "Donor-level AUROC",
        colour = "Outer repeat"
    ) +
    ggplot2::guides(
        colour = ggplot2::guide_legend(
            nrow = 2,
            byrow = TRUE,
            override.aes = list(
                linewidth = 0.8,
                size = 2.1
            )
        )
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom"
    )

# ==================================================================================================
# 15. PANEL B — SAME-POPULATION REPEATED-OOF MODEL COMPARISON
# ==================================================================================================

model_colours <- c(
    "Covariates only" = "#9E9E9E",
    "C/E shared-only" = "#76B7B2",
    "F shared + deviation" = "#4E79A7",
    "D independent regions" = "#B07AA1"
)

pB <- ggplot2::ggplot(
    panel_B,
    ggplot2::aes(
        x = model,
        y = AUROC,
        colour = model
    )
) +
    ggplot2::geom_hline(
        yintercept = 0.5,
        linetype = "dotted",
        linewidth = 0.4,
        colour = "#8A8A8A"
    ) +
    ggplot2::geom_segment(
        ggplot2::aes(
            xend = model,
            y = 0.5,
            yend = AUROC
        ),
        linewidth = 1.0
    ) +
    ggplot2::geom_point(
        size = 4.2
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = sprintf(
                "%.3f",
                AUROC
            )
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.8,
        vjust = -0.8
    ) +
    ggplot2::scale_colour_manual(
        values = model_colours,
        guide = "none"
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            0.50,
            0.85
        ),
        breaks = seq(
            0.50,
            0.85,
            by = 0.05
        )
    ) +
    ggplot2::labs(
        title = "B  Repeated-OOF comparison in all 454 donors",
        x = NULL,
        y = "Donor-level AUROC"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.6,
            angle = 18,
            hjust = 1,
            vjust = 1
        )
    )

# ==================================================================================================
# 16. PANEL C — PAIRED DONOR-BOOTSTRAP DELTA FOREST
# ==================================================================================================

delta_colours <- c(
    "F vs covariates only" = "#59A14F",
    "F vs C/E shared-only" = "#4E79A7",
    "F vs D independent regions" = "#B07AA1"
)

pC <- ggplot2::ggplot(
    panel_C,
    ggplot2::aes(
        x = delta_AUROC,
        y = comparison,
        colour = comparison
    )
) +
    ggplot2::geom_vline(
        xintercept = 0,
        linetype = "dashed",
        linewidth = 0.55,
        colour = "#555555"
    ) +
    ggplot2::geom_segment(
        ggplot2::aes(
            x = CI_low,
            xend = CI_high,
            y = comparison,
            yend = comparison
        ),
        linewidth = 1.25
    ) +
    ggplot2::geom_point(
        size = 3.8
    ) +
    ggplot2::geom_text(
        data = panel_C,
        ggplot2::aes(
            x = 0.255,
            y = comparison,
            label = display_text
        ),
        inherit.aes = FALSE,
        family = FONT_FAMILY,
        size = 2.70,
        hjust = 1,
        colour = "black"
    ) +
    ggplot2::scale_colour_manual(
        values = delta_colours,
        guide = "none"
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            -0.05,
            0.26
        ),
        breaks = c(
            -0.05,
            0,
            0.05,
            0.10,
            0.15,
            0.20,
            0.25
        ),
        labels = function(x) {

            sprintf(
                "%+.2f",
                x
            )
        }
    ) +
    ggplot2::labs(
        title = "C  Paired donor-bootstrap differences in repeated-OOF AUROC",
        x = expression(
            Delta * "AUROC (Model F - comparator)"
        ),
        y = NULL
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.y = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.5
        )
    )

# ==================================================================================================
# 17. DRAW 3-PANEL FIGURE
# ==================================================================================================

draw_figure3 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 2,
        ncol = 2,
        heights = grid::unit(
            c(
                1.03,
                0.82
            ),
            "null"
        ),
        widths = grid::unit(
            c(
                1,
                1
            ),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1:2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 18. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 5.7,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure3()

grDevices::dev.off()

# ==================================================================================================
# 19. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 5.7,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure3()

grDevices::dev.off()

# ==================================================================================================
# 20. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 5.7,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure3()

grDevices::dev.off()

# ==================================================================================================
# 21. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    PANEL_A_PATH,
    PANEL_B_PATH,
    PANEL_C_PATH
)

if (!all(
    file.exists(
        outputs
    )
)) {

    stop(
        "One or more Figure 3 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {

    stop(
        "One or more Figure 3 outputs are zero-byte."
    )
}

cat(
    "Figure 3 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 22. SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11C.4R — MAIN FIGURE 3",
    "==============================================================================================================",

    "",

    "All panels generated from previously saved leakage-free OOF/CV/model-comparison artifacts.",

    "No model fitting or bootstrap resampling was rerun.",

    "",

    paste0(
        "Model F repeated-OOF donor AUROC = ",
        format(
            F_REPEATED_OOF,
            digits = 9
        )
    ),

    paste0(
        "Model F mean outer-fold donor AUROC = ",
        format(
            F_MEAN_FOLD,
            digits = 9
        )
    ),

    paste0(
        "Model F outer-fold AUROC SD = ",
        format(
            F_SD_FOLD,
            digits = 9
        )
    ),

    "",

    paste0(
        "Covariate-only repeated-OOF donor AUROC = ",
        format(
            COV_AUC,
            digits = 9
        )
    ),

    paste0(
        "C/E repeated-OOF donor AUROC = ",
        format(
            CE_AUC,
            digits = 9
        )
    ),

    paste0(
        "Model F repeated-OOF donor AUROC = ",
        format(
            F_REPEATED_OOF,
            digits = 9
        )
    ),

    paste0(
        "Model D repeated-OOF donor AUROC = ",
        format(
            D_AUC,
            digits = 9
        )
    ),

    "",

    paste0(
        "F - C/E delta = ",
        format(
            CE_DELTA,
            digits = 9
        ),
        " [",
        format(
            CE_CI_LOW,
            digits = 9
        ),
        ", ",
        format(
            CE_CI_HIGH,
            digits = 9
        ),
        "]"
    ),

    paste0(
        "F - D delta = ",
        format(
            D_DELTA,
            digits = 9
        ),
        " [",
        format(
            D_CI_LOW,
            digits = 9
        ),
        ", ",
        format(
            D_CI_HIGH,
            digits = 9
        ),
        "]"
    ),

    paste0(
        "F - covariates delta = ",
        format(
            COV_DELTA,
            digits = 9
        ),
        " [",
        format(
            COV_CI_LOW,
            digits = 9
        ),
        ", ",
        format(
            COV_CI_HIGH,
            digits = 9
        ),
        "]"
    ),

    "",

    "Model A/B regional-only baselines intentionally omitted from Panel B because their eligible donor populations differ.",

    "They remain available for exact reporting in Main Table 2.",

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_OUT_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 23. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_OUT_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 24. FINAL CONSOLE REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFINAL FIGURE 3 GENERATION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Frozen model-performance sources: PASS\n"
)

cat(
    "Outer-fold evaluations: ",
    nrow(panel_A),
    "\n",
    sep = ""
)

cat(
    "Model F repeated-OOF donor AUROC: ",
    sprintf(
        "%.7f",
        F_REPEATED_OOF
    ),
    "\n",
    sep = ""
)

cat(
    "Mean fold donor AUROC: ",
    sprintf(
        "%.7f",
        F_MEAN_FOLD
    ),
    " +/- ",
    sprintf(
        "%.7f",
        F_SD_FOLD
    ),
    "\n",
    sep = ""
)

cat(
    "\nAll-donor repeated-OOF AUROCs:\n"
)

print(
    panel_B,
    row.names = FALSE
)

cat(
    "\nPaired donor-bootstrap delta AUROC:\n"
)

print(
    panel_C[
        ,
        c(
            "comparison",
            "delta_AUROC",
            "CI_low",
            "CI_high"
        ),
        drop = FALSE
    ],
    row.names = FALSE
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.4R COMPLETE — MAIN FIGURE 3 GENERATED FROM FROZEN LEAKAGE-FREE MODEL OUTPUTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.4R — MAIN FIGURE 3 — FULLY CORRECTED
LEAKAGE-FREE NESTED-CV PERFORMANCE AND MODEL COMPARISON
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Required frozen model-performance artifacts: PASS
Input schemas: PASS
Model F repeated-OOF donor AUROC: 0.8019743 — PASS
Model F 25-fold performance integrity: PASS
All-donor model comparison: PASS
Paired donor-bootstrap comparison anchors: PASS


agg_record_211ca71d9c 
                    2

agg_record_211ca71d9c 
                    2

agg_record_211ca71d9c 
                    2

Figure 3 outputs created and non-zero: PASS

FINAL FIGURE 3 GENERATION SUMMARY
Frozen model-performance sources: PASS
Outer-fold evaluations: 25
Model F repeated-OOF donor AUROC: 0.8019743
Mean fold donor AUROC: 0.7868384 +/- 0.0404797

All-donor repeated-OOF AUROCs:
                 model     AUROC n_donors
       Covariates only 0.6143507      454
       C/E shared-only 0.7987926      454
  F shared + deviation 0.8019743      454
 D independent regions 0.8150278      454

Paired donor-bootstrap delta AUROC:
                 comparison delta_AUROC       CI_low     CI_high
       F vs covariates only  0.18762365  0.140610149 0.237415611
       F vs C/E shared-only  0.00318179 -0.001080993 0.007668931
 F vs D independent regions -0.01305350 -0.030859797 0.005282588

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure3_final_TNR_v1/Figure3_NestedCV_ModelComparison_TimesNewRoman_v1.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure3_final_TNR_v1/Fi

In [97]:
# ==================================================================================================
# STEP 11C.5 — MAIN FIGURE 4
# STABILITY EVIDENCE AND DONOR-LEVEL PERMUTATION NULL
# --------------------------------------------------------------------------------------------------
# PANELS
#
#   A — Final 24-feature stability evidence:
#         Outer-CV selection frequency
#         Complementary half-sample selection probability
#         Permutation-null selection probability
#
#   B — Donor-level permutation-null AUROC distribution:
#         100 diagnosis permutations
#         observed matched-reference donor AUROC
#         empirical permutation P value
#
# IMPORTANT
#   - SAVED artifacts only
#   - NO glmnet fitting
#   - NO half-sample fitting
#   - NO permutation rerun
#   - NO bootstrap
#   - NO score recalculation
#   - Times New Roman
#   - PDF vector + TIFF 600 dpi + PNG 300 dpi
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.5 — MAIN FIGURE 4\n",
    "STABILITY EVIDENCE AND DONOR-LEVEL PERMUTATION NULL\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. INPUT PATHS
# ==================================================================================================

FINAL_DIR <- file.path(
    PROJECT_ROOT,
    "07_stability",
    "final_stable_state_v1"
)

PERM_DIR <- file.path(
    PROJECT_ROOT,
    "07_stability",
    "permutation_null_v1"
)

FINAL_STATE_PATH <- file.path(
    FINAL_DIR,
    "step06d_FINAL_stable_shared_state_v1.csv"
)

FINAL_SUMMARY_PATH <- file.path(
    FINAL_DIR,
    "step06d_final_stable_state_summary_v1.csv"
)

EVIDENCE_PATH <- file.path(
    FINAL_DIR,
    "step06d_final_gene_stability_evidence_v1.csv"
)

PERM_RUN_PATH <- file.path(
    PERM_DIR,
    "step06c_permutation_run_summary_v1.csv"
)

PERM_SUMMARY_PATH <- file.path(
    PERM_DIR,
    "step06c_permutation_null_summary_v1.csv"
)

required_inputs <- c(
    FINAL_STATE_PATH,
    FINAL_SUMMARY_PATH,
    EVIDENCE_PATH,
    PERM_RUN_PATH,
    PERM_SUMMARY_PATH
)

missing_inputs <- required_inputs[
    !file.exists(
        required_inputs
    )
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required frozen stability artifact(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat(
    "Required frozen stability/permutation artifacts: PASS\n"
)

# ==================================================================================================
# 5. FINAL-STATE SHA256 CHECK
# ==================================================================================================

EXPECTED_FINAL_SHA <-
    "a2bd0c8b26ee3380d2d0fb3ab1f39365b124f0ce550a753a9b59e164f3142330"

OBSERVED_FINAL_SHA <- digest::digest(
    file = FINAL_STATE_PATH,
    algo = "sha256",
    serialize = FALSE
)

if (!identical(
    OBSERVED_FINAL_SHA,
    EXPECTED_FINAL_SHA
)) {

    stop(
        "Final 24-feature stable-state SHA256 mismatch.\n",
        "Expected: ",
        EXPECTED_FINAL_SHA,
        "\nObserved: ",
        OBSERVED_FINAL_SHA
    )
}

cat(
    "Canonical final 24-feature SHA256: PASS\n"
)

# ==================================================================================================
# 6. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "main",
    "Figure4_final_TNR_v1"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "Figure4_Stability_Permutation_TimesNewRoman_v1.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "Figure4_Stability_Permutation_TimesNewRoman_v1.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "Figure4_Stability_Permutation_TimesNewRoman_v1.png"
)

PANEL_A_PATH <- file.path(
    SOURCE_DIR,
    "Figure4_PanelA_final24_stability_source_v1.csv"
)

PANEL_B_PATH <- file.path(
    SOURCE_DIR,
    "Figure4_PanelB_permutation_AUROC_source_v1.csv"
)

SUMMARY_OUT_PATH <- file.path(
    OUT_DIR,
    "Figure4_generation_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Figure4_generation_manifest_v1.csv"
)

# ==================================================================================================
# 7. LOAD FROZEN DATA
# ==================================================================================================

final_state <- read.csv(
    FINAL_STATE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

final_summary <- read.csv(
    FINAL_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

evidence <- read.csv(
    EVIDENCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

perm_run <- read.csv(
    PERM_RUN_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

perm_summary <- read.csv(
    PERM_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 8. SCHEMA VALIDATION
# ==================================================================================================

required_evidence_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "direction",
    "outer_selection_frequency",
    "outer_sign_consistency",
    "half_sample_selection_probability",
    "half_sample_sign_consistency",
    "null_selection_probability",
    "half_sample_median_nonzero_beta"
)

missing_evidence_cols <- setdiff(
    required_evidence_cols,
    names(evidence)
)

if (length(missing_evidence_cols) > 0L) {

    stop(
        "Final stability-evidence file missing field(s): ",
        paste(
            missing_evidence_cols,
            collapse = ", "
        )
    )
}

if (!all(
    c(
        "metric",
        "value"
    ) %in%
        names(final_summary)
)) {

    stop(
        "Final stable-state summary schema changed."
    )
}

if (!all(
    c(
        "permutation_id",
        "donor_auc"
    ) %in%
        names(perm_run)
)) {

    stop(
        "Permutation run-summary schema changed."
    )
}

if (!all(
    c(
        "metric",
        "value"
    ) %in%
        names(perm_summary)
)) {

    stop(
        "Permutation-null summary schema changed."
    )
}

cat("Input schemas: PASS\n")

# ==================================================================================================
# 9. METRIC HELPER
# ==================================================================================================

get_metric <- function(
    x,
    metric_name
) {

    idx <- which(
        as.character(
            x$metric
        ) ==
            metric_name
    )

    if (length(idx) != 1L) {

        stop(
            "Could not uniquely retrieve metric: ",
            metric_name
        )
    }

    value <- as.numeric(
        x$value[idx]
    )

    if (
        length(value) != 1L ||
        !is.finite(value)
    ) {

        stop(
            "Metric is non-finite: ",
            metric_name
        )
    }

    value
}

# ==================================================================================================
# 10. FINAL-STATE ANCHORS
# ==================================================================================================

OUTER_CANDIDATES <- get_metric(
    final_summary,
    "Outer_CV_candidate_shared_genes"
)

HALF_CANDIDATES <- get_metric(
    final_summary,
    "Half_sample_candidate_shared_genes"
)

FINAL_N <- get_metric(
    final_summary,
    "Final_stable_shared_genes"
)

FINAL_POSITIVE <- get_metric(
    final_summary,
    "Final_positive_direction"
)

FINAL_NEGATIVE <- get_metric(
    final_summary,
    "Final_negative_direction"
)

DIRECTION_AGREEMENT <- get_metric(
    final_summary,
    "Outer_half_direction_agreement"
)

MIN_HALF_SELECTION <- get_metric(
    final_summary,
    "Minimum_half_sample_selection_probability"
)

MIN_HALF_SIGN <- get_metric(
    final_summary,
    "Minimum_half_sample_sign_consistency"
)

MAX_CANDIDATE_NULL <- get_metric(
    final_summary,
    "Maximum_candidate_null_selection_probability"
)

OBSERVED_AUC <- get_metric(
    final_summary,
    "Observed_reference_donor_AUROC"
)

NULL_MEAN_ANCHOR <- get_metric(
    final_summary,
    "Null_donor_AUROC_mean"
)

NULL_MAX_ANCHOR <- get_metric(
    final_summary,
    "Null_donor_AUROC_max"
)

EMPIRICAL_P_ANCHOR <- get_metric(
    final_summary,
    "Permutation_empirical_P"
)

NULL_STABLE_SHARED <- get_metric(
    final_summary,
    "Null_stable_like_shared_genes"
)

STABLE_DEVIATION <- get_metric(
    final_summary,
    "Stable_deviation_genes"
)

# ==================================================================================================
# 11. STRICT FROZEN-ANCHOR VALIDATION
# ==================================================================================================

if (
    OUTER_CANDIDATES != 403 ||
    HALF_CANDIDATES != 24 ||
    FINAL_N != 24
) {

    stop(
        "Final-state candidate counts changed."
    )
}

if (
    FINAL_POSITIVE != 9 ||
    FINAL_NEGATIVE != 15
) {

    stop(
        "Final direction counts changed."
    )
}

if (
    DIRECTION_AGREEMENT != 24
) {

    stop(
        "Outer/half direction agreement changed."
    )
}

if (
    abs(
        MIN_HALF_SELECTION -
        0.80
    ) >
        1e-12
) {

    stop(
        "Minimum half-sample selection probability changed."
    )
}

if (
    abs(
        MIN_HALF_SIGN -
        1.0
    ) >
        1e-12
) {

    stop(
        "Minimum half-sample sign consistency changed."
    )
}

if (
    abs(
        MAX_CANDIDATE_NULL -
        0.22
    ) >
        1e-12
) {

    stop(
        "Maximum candidate null-selection probability changed."
    )
}

if (
    STABLE_DEVIATION != 0
) {

    stop(
        "Stable deviation-gene count changed."
    )
}

if (
    abs(
        OBSERVED_AUC -
        0.76983010
    ) >
        5e-8
) {

    stop(
        "Observed permutation-reference donor AUROC changed."
    )
}

if (
    abs(
        NULL_MEAN_ANCHOR -
        0.50645353
    ) >
        5e-8
) {

    stop(
        "Null donor-AUROC mean changed."
    )
}

if (
    abs(
        NULL_MAX_ANCHOR -
        0.59950234
    ) >
        5e-8
) {

    stop(
        "Null donor-AUROC maximum changed."
    )
}

if (
    abs(
        EMPIRICAL_P_ANCHOR -
        0.00990099
    ) >
        5e-8
) {

    stop(
        "Permutation empirical P changed."
    )
}

if (
    NULL_STABLE_SHARED != 0
) {

    stop(
        "Null stable-like shared-gene count changed."
    )
}

cat(
    "Frozen final-state anchors: PASS\n"
)

# ==================================================================================================
# 12. GENE-LEVEL EVIDENCE VALIDATION
# ==================================================================================================

if (nrow(evidence) != 24L) {

    stop(
        "Expected exactly 24 final stability-evidence rows; observed ",
        nrow(evidence)
    )
}

if (nrow(final_state) != 24L) {

    stop(
        "Expected exactly 24 rows in canonical final-state file."
    )
}

if (anyDuplicated(
    evidence$ensemblID_stable
) > 0L) {

    stop(
        "Duplicated Ensembl IDs detected in final stability evidence."
    )
}

if (
    !setequal(
        evidence$ensemblID_stable,
        final_state$ensemblID_stable
    )
) {

    stop(
        "Final-state and stability-evidence gene sets disagree."
    )
}

if (any(
    !evidence$direction %in%
        c(
            "Positive",
            "Negative"
        )
)) {

    stop(
        "Unexpected final-state direction label."
    )
}

if (
    sum(
        evidence$direction ==
            "Positive"
    ) != 9L ||
    sum(
        evidence$direction ==
            "Negative"
    ) != 15L
) {

    stop(
        "Gene-level direction counts disagree with final summary."
    )
}

if (
    min(
        evidence$half_sample_selection_probability
    ) <
        0.80 - 1e-12
) {

    stop(
        "A final feature has half-sample selection probability below 0.80."
    )
}

if (
    min(
        evidence$half_sample_sign_consistency
    ) <
        1.0 - 1e-12
) {

    stop(
        "A final feature has half-sample sign consistency below 1.0."
    )
}

if (
    max(
        evidence$null_selection_probability
    ) >
        0.22 + 1e-12
) {

    stop(
        "A final feature exceeds the frozen maximum null-selection probability."
    )
}

cat(
    "Final 24-feature gene-level evidence: PASS\n"
)

# ==================================================================================================
# 13. DISPLAY LABELS
# ==================================================================================================

gene_symbol_clean <- trimws(
    as.character(
        evidence$gene_symbol
    )
)

missing_symbol <- (
    is.na(gene_symbol_clean) |
    gene_symbol_clean == ""
)

gene_symbol_clean[
    missing_symbol
] <- evidence$ensemblID_stable[
    missing_symbol
]

direction_suffix <- ifelse(
    evidence$direction ==
        "Positive",
    " (+)",
    " (-)"
)

evidence$display_gene <- paste0(
    gene_symbol_clean,
    direction_suffix
)

# ==================================================================================================
# 14. ORDER FINAL 24 FEATURES
#
# Highest complementary half-sample stability at the top.
# Ties resolved by lower null-selection probability and then gene label.
# ==================================================================================================

gene_order_index <- order(
    -evidence$half_sample_selection_probability,
    evidence$null_selection_probability,
    evidence$display_gene
)

evidence <- evidence[
    gene_order_index,
    ,
    drop = FALSE
]

evidence$display_gene <- factor(
    evidence$display_gene,
    levels = rev(
        evidence$display_gene
    )
)

# ==================================================================================================
# 15. PANEL A — LONG-FORM HEATMAP DATA
# ==================================================================================================

panel_A_outer <- data.frame(
    display_gene = evidence$display_gene,
    metric = "Outer-CV\nselection",
    probability = evidence$outer_selection_frequency,
    stringsAsFactors = FALSE
)

panel_A_half <- data.frame(
    display_gene = evidence$display_gene,
    metric = "Half-sample\nselection",
    probability = evidence$half_sample_selection_probability,
    stringsAsFactors = FALSE
)

panel_A_null <- data.frame(
    display_gene = evidence$display_gene,
    metric = "Permutation-null\nselection",
    probability = evidence$null_selection_probability,
    stringsAsFactors = FALSE
)

panel_A <- rbind(
    panel_A_outer,
    panel_A_half,
    panel_A_null
)

panel_A$metric <- factor(
    panel_A$metric,
    levels = c(
        "Outer-CV\nselection",
        "Half-sample\nselection",
        "Permutation-null\nselection"
    )
)

panel_A$display_gene <- factor(
    as.character(
        panel_A$display_gene
    ),
    levels = levels(
        evidence$display_gene
    )
)

panel_A$text_colour <- ifelse(
    panel_A$probability >= 0.65,
    "white",
    "black"
)

write.csv(
    panel_A,
    PANEL_A_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 16. PANEL B — PERMUTATION DISTRIBUTION VALIDATION
# ==================================================================================================

if (nrow(perm_run) != 100L) {

    stop(
        "Expected 100 donor-level permutation runs; observed ",
        nrow(perm_run)
    )
}

if (any(
    !is.finite(
        perm_run$donor_auc
    )
)) {

    stop(
        "Non-finite permutation donor AUROC detected."
    )
}

NULL_MEAN <- mean(
    perm_run$donor_auc
)

NULL_SD <- stats::sd(
    perm_run$donor_auc
)

NULL_MIN <- min(
    perm_run$donor_auc
)

NULL_MAX <- max(
    perm_run$donor_auc
)

EMPIRICAL_P <- (
    1 +
    sum(
        perm_run$donor_auc >=
            OBSERVED_AUC
    )
) / (
    nrow(perm_run) +
    1
)

if (
    abs(
        NULL_MEAN -
        NULL_MEAN_ANCHOR
    ) >
        5e-8
) {

    stop(
        "Permutation-run null mean disagrees with frozen summary."
    )
}

if (
    abs(
        NULL_MAX -
        NULL_MAX_ANCHOR
    ) >
        5e-8
) {

    stop(
        "Permutation-run null maximum disagrees with frozen summary."
    )
}

if (
    abs(
        EMPIRICAL_P -
        EMPIRICAL_P_ANCHOR
    ) >
        5e-8
) {

    stop(
        "Recomputed empirical P disagrees with frozen summary."
    )
}

panel_B <- data.frame(
    permutation_id = as.integer(
        perm_run$permutation_id
    ),
    donor_AUROC = as.numeric(
        perm_run$donor_auc
    ),
    stringsAsFactors = FALSE
)

write.csv(
    panel_B,
    PANEL_B_PATH,
    row.names = FALSE,
    quote = TRUE
)

cat(
    "Permutation-null distribution integrity: PASS\n"
)

# ==================================================================================================
# 17. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.5,
            hjust = 0,
            margin = ggplot2::margin(
                b = 6
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.5
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.2,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 8.2
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.8
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            8,
            9,
            8,
            9
        )
    )

# ==================================================================================================
# 18. PANEL A — FINAL 24 STABILITY HEATMAP
# ==================================================================================================

pA <- ggplot2::ggplot(
    panel_A,
    ggplot2::aes(
        x = metric,
        y = display_gene,
        fill = probability
    )
) +
    ggplot2::geom_tile(
        colour = "white",
        linewidth = 0.55,
        width = 0.97,
        height = 0.96
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = sprintf(
                "%.2f",
                probability
            ),
            colour = text_colour
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.25,
        show.legend = FALSE
    ) +
    ggplot2::scale_colour_identity() +
    ggplot2::scale_fill_gradientn(
        colours = c(
            "#F1F1F1",
            "#E9C46A",
            "#2A9D8F",
            "#264653"
        ),
        values = c(
            0,
            0.35,
            0.75,
            1
        ),
        limits = c(
            0,
            1
        ),
        breaks = c(
            0,
            0.25,
            0.50,
            0.75,
            1.00
        )
    ) +
    ggplot2::labs(
        title = "A  Stability evidence for the final 24-feature shared state",
        x = NULL,
        y = NULL,
        fill = "Selection\nprobability"
    ) +
    theme_publication +
    ggplot2::theme(

        axis.line = ggplot2::element_blank(),

        axis.ticks = ggplot2::element_blank(),

        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.2,
            face = "bold",
            lineheight = 0.9
        ),

        axis.text.y = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.4
        ),

        legend.position = "bottom",

        legend.key.width = grid::unit(
            2.0,
            "cm"
        )
    )

# ==================================================================================================
# 19. PANEL B — DONOR-LEVEL PERMUTATION NULL
# ==================================================================================================

annotation_text <- paste0(
    "Observed AUROC = ",
    sprintf(
        "%.3f",
        OBSERVED_AUC
    ),
    "\nNull mean = ",
    sprintf(
        "%.3f",
        NULL_MEAN
    ),
    " (SD ",
    sprintf(
        "%.3f",
        NULL_SD
    ),
    ")",
    "\nNull max = ",
    sprintf(
        "%.3f",
        NULL_MAX
    ),
    "\nEmpirical p = ",
    sprintf(
        "%.4f",
        EMPIRICAL_P
    )
)

pB <- ggplot2::ggplot(
    panel_B,
    ggplot2::aes(
        x = donor_AUROC
    )
) +
    ggplot2::geom_histogram(
        binwidth = 0.015,
        boundary = 0.40,
        closed = "left",
        fill = "#76B7B2",
        colour = "white",
        linewidth = 0.45
    ) +
    ggplot2::geom_rug(
        sides = "b",
        alpha = 0.45,
        linewidth = 0.35,
        colour = "#3D6F6C"
    ) +
    ggplot2::geom_vline(
        xintercept = NULL_MEAN,
        linetype = "dashed",
        linewidth = 0.70,
        colour = "#333333"
    ) +
    ggplot2::geom_vline(
        xintercept = OBSERVED_AUC,
        linewidth = 1.05,
        colour = "#E15759"
    ) +
    ggplot2::annotate(
        "text",
        x = 0.795,
        y = Inf,
        label = annotation_text,
        hjust = 1,
        vjust = 1.12,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.65,
        lineheight = 1.05
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            0.42,
            0.80
        ),
        breaks = seq(
            0.45,
            0.80,
            by = 0.05
        )
    ) +
    ggplot2::scale_y_continuous(
        expand = ggplot2::expansion(
            mult = c(
                0,
                0.10
            )
        )
    ) +
    ggplot2::labs(
        title = "B  Donor-level diagnosis-permutation null",
        x = "Donor-level AUROC",
        y = "Permutation count"
    ) +
    theme_publication

# ==================================================================================================
# 20. DRAW 2-PANEL FIGURE
# ==================================================================================================

draw_figure4 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 1,
        ncol = 2,
        widths = grid::unit(
            c(
                1.35,
                1.00
            ),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 21. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 5.9,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure4()

grDevices::dev.off()

# ==================================================================================================
# 22. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 5.9,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure4()

grDevices::dev.off()

# ==================================================================================================
# 23. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 5.9,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure4()

grDevices::dev.off()

# ==================================================================================================
# 24. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    PANEL_A_PATH,
    PANEL_B_PATH
)

if (!all(
    file.exists(
        outputs
    )
)) {

    stop(
        "One or more Figure 4 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {

    stop(
        "One or more Figure 4 outputs are zero-byte."
    )
}

cat(
    "Figure 4 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 25. SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11C.5 — MAIN FIGURE 4",
    "==============================================================================================================",

    "",

    "All panels generated from frozen stability and permutation artifacts.",

    "No model fitting, half-sample resampling, or permutation analysis was rerun.",

    "",

    paste0(
        "Outer-CV stable shared candidates = ",
        OUTER_CANDIDATES
    ),

    paste0(
        "Complementary half-sample stable candidates = ",
        HALF_CANDIDATES
    ),

    paste0(
        "Final shared-state features = ",
        FINAL_N
    ),

    paste0(
        "Final positive / negative = ",
        FINAL_POSITIVE,
        " / ",
        FINAL_NEGATIVE
    ),

    paste0(
        "Outer-half direction agreement = ",
        DIRECTION_AGREEMENT,
        " / ",
        FINAL_N
    ),

    paste0(
        "Minimum half-sample selection probability = ",
        format(
            MIN_HALF_SELECTION,
            digits = 5
        )
    ),

    paste0(
        "Minimum half-sample sign consistency = ",
        format(
            MIN_HALF_SIGN,
            digits = 5
        )
    ),

    paste0(
        "Maximum final-feature null-selection probability = ",
        format(
            MAX_CANDIDATE_NULL,
            digits = 5
        )
    ),

    paste0(
        "Stable deviation features = ",
        STABLE_DEVIATION
    ),

    "",

    paste0(
        "Permutation runs = ",
        nrow(
            perm_run
        )
    ),

    paste0(
        "Observed matched-reference donor AUROC = ",
        format(
            OBSERVED_AUC,
            digits = 9
        )
    ),

    paste0(
        "Null donor AUROC mean = ",
        format(
            NULL_MEAN,
            digits = 9
        )
    ),

    paste0(
        "Null donor AUROC SD = ",
        format(
            NULL_SD,
            digits = 9
        )
    ),

    paste0(
        "Null donor AUROC range = ",
        format(
            NULL_MIN,
            digits = 9
        ),
        " to ",
        format(
            NULL_MAX,
            digits = 9
        )
    ),

    paste0(
        "Empirical permutation P = ",
        format(
            EMPIRICAL_P,
            digits = 9
        )
    ),

    paste0(
        "Null stable-like shared features = ",
        NULL_STABLE_SHARED
    ),

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_OUT_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 26. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_OUT_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 27. FINAL CONSOLE REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFINAL FIGURE 4 GENERATION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Canonical final-state SHA256: PASS\n"
)

cat(
    "Final stable shared features: ",
    FINAL_N,
    "\n",
    sep = ""
)

cat(
    "Positive / Negative: ",
    FINAL_POSITIVE,
    " / ",
    FINAL_NEGATIVE,
    "\n",
    sep = ""
)

cat(
    "Stable deviation features: ",
    STABLE_DEVIATION,
    "\n",
    sep = ""
)

cat(
    "Minimum half-sample selection probability: ",
    sprintf(
        "%.3f",
        MIN_HALF_SELECTION
    ),
    "\n",
    sep = ""
)

cat(
    "Maximum final-feature null-selection probability: ",
    sprintf(
        "%.3f",
        MAX_CANDIDATE_NULL
    ),
    "\n",
    sep = ""
)

cat(
    "\nPermutation null:\n"
)

cat(
    "Runs: ",
    nrow(
        perm_run
    ),
    "\n",
    sep = ""
)

cat(
    "Observed donor AUROC: ",
    sprintf(
        "%.7f",
        OBSERVED_AUC
    ),
    "\n",
    sep = ""
)

cat(
    "Null mean +/- SD: ",
    sprintf(
        "%.7f",
        NULL_MEAN
    ),
    " +/- ",
    sprintf(
        "%.7f",
        NULL_SD
    ),
    "\n",
    sep = ""
)

cat(
    "Null maximum: ",
    sprintf(
        "%.7f",
        NULL_MAX
    ),
    "\n",
    sep = ""
)

cat(
    "Empirical P: ",
    sprintf(
        "%.8f",
        EMPIRICAL_P
    ),
    "\n",
    sep = ""
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.5 COMPLETE — MAIN FIGURE 4 GENERATED FROM FROZEN STABILITY/PERMUTATION OUTPUTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.5 — MAIN FIGURE 4
STABILITY EVIDENCE AND DONOR-LEVEL PERMUTATION NULL
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Required frozen stability/permutation artifacts: PASS
Canonical final 24-feature SHA256: PASS
Input schemas: PASS
Frozen final-state anchors: PASS
Final 24-feature gene-level evidence: PASS
Permutation-null distribution integrity: PASS


Warning message:
"Removed 2 rows containing missing values or values outside the scale range (`geom_bar()`)."


agg_record_211c7efe765a 
                      2

Warning message:
"Removed 2 rows containing missing values or values outside the scale range (`geom_bar()`)."


agg_record_211c7efe765a 
                      2

Warning message:
"Removed 2 rows containing missing values or values outside the scale range (`geom_bar()`)."


agg_record_211c7efe765a 
                      2

Figure 4 outputs created and non-zero: PASS

FINAL FIGURE 4 GENERATION SUMMARY
Canonical final-state SHA256: PASS
Final stable shared features: 24
Positive / Negative: 9 / 15
Stable deviation features: 0
Minimum half-sample selection probability: 0.800
Maximum final-feature null-selection probability: 0.220

Permutation null:
Runs: 100
Observed donor AUROC: 0.7698301
Null mean +/- SD: 0.5064535 +/- 0.0325831
Null maximum: 0.5995023
Empirical P: 0.00990099

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure4_final_TNR_v1/Figure4_Stability_Permutation_TimesNewRoman_v1.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure4_final_TNR_v1/Figure4_Stability_Permutation_TimesNewRoman_v1.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure4_final_TNR_v1/Figure4_Stability_Permutation_TimesNewRoman_v1.png

STEP 11C.5 COMPLETE — MAIN FIGURE 4 GENERATED FROM FROZEN STABILITY/PERMUTATION OUTPUTS


In [99]:
# ==================================================================================================
# STEP 11C.5S — MAIN FIGURE 4 — FINAL IDENTITY-AWARE VISUAL VERSION
# STABILITY EVIDENCE AND DONOR-LEVEL PERMUTATION NULL
# --------------------------------------------------------------------------------------------------
# PURPOSE
#
#   Produce the final manuscript Figure 4 with:
#
#   A — Stability evidence for the final 24-feature shared state
#       * Outer-CV selection frequency
#       * Complementary half-sample selection probability
#       * Permutation-null selection probability
#
#   B — Donor-level diagnosis-permutation null distribution
#
# IMPORTANT IDENTITY POLICY
#
#   - Frozen Ensembl IDs remain the ANALYSIS identifiers.
#   - Frozen feature inclusion / direction / weights are unchanged.
#   - STEP10D.2 adjudicated identities are used ONLY as manuscript display labels.
#   - If STEP10D.2 provides an interpretation_symbol, that readable label is shown.
#   - Otherwise frozen gene_symbol is used.
#   - If neither exists, stable Ensembl ID is retained.
#
#   Therefore genuinely unresolved features remain Ensembl IDs.
#
# NO ANALYSIS RERUN
#   - NO glmnet
#   - NO half-sample fitting
#   - NO permutation rerun
#   - NO score recalculation
#   - NO bootstrap
#
# VISUAL CORRECTIONS
#   - shorter Panel A title to prevent clipping
#   - identity-aware readable feature labels
#   - compact complete 0.00–1.00 colorbar
#   - warning-free histogram using coord_cartesian()
#   - Times New Roman
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.5S — MAIN FIGURE 4 — FINAL IDENTITY-AWARE VISUAL VERSION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. INPUT PATHS
# ==================================================================================================

FINAL_DIR <- file.path(
    PROJECT_ROOT,
    "07_stability",
    "final_stable_state_v1"
)

PERM_DIR <- file.path(
    PROJECT_ROOT,
    "07_stability",
    "permutation_null_v1"
)

IDENTITY_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology",
    "identity_audit_v1",
    "adjudicated_v1"
)

FINAL_STATE_PATH <- file.path(
    FINAL_DIR,
    "step06d_FINAL_stable_shared_state_v1.csv"
)

FINAL_SUMMARY_PATH <- file.path(
    FINAL_DIR,
    "step06d_final_stable_state_summary_v1.csv"
)

EVIDENCE_PATH <- file.path(
    FINAL_DIR,
    "step06d_final_gene_stability_evidence_v1.csv"
)

PERM_RUN_PATH <- file.path(
    PERM_DIR,
    "step06c_permutation_run_summary_v1.csv"
)

IDENTITY_PATH <- file.path(
    IDENTITY_DIR,
    "STEP10D2_SCZ_MSS_24gene_interpretation_identity_v1.csv"
)

required_inputs <- c(
    FINAL_STATE_PATH,
    FINAL_SUMMARY_PATH,
    EVIDENCE_PATH,
    PERM_RUN_PATH,
    IDENTITY_PATH
)

missing_inputs <- required_inputs[
    !file.exists(
        required_inputs
    )
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required frozen input(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat(
    "Frozen stability, permutation, and identity artifacts: PASS\n"
)

# ==================================================================================================
# 5. CANONICAL FINAL-STATE SHA256
# ==================================================================================================

EXPECTED_FINAL_SHA <-
    "a2bd0c8b26ee3380d2d0fb3ab1f39365b124f0ce550a753a9b59e164f3142330"

OBSERVED_FINAL_SHA <- digest::digest(
    file = FINAL_STATE_PATH,
    algo = "sha256",
    serialize = FALSE
)

if (!identical(
    OBSERVED_FINAL_SHA,
    EXPECTED_FINAL_SHA
)) {

    stop(
        "Canonical final 24-feature SHA256 mismatch.\n",
        "Expected: ",
        EXPECTED_FINAL_SHA,
        "\nObserved: ",
        OBSERVED_FINAL_SHA
    )
}

cat(
    "Canonical final 24-feature SHA256: PASS\n"
)

# ==================================================================================================
# 6. LOAD SAVED DATA
# ==================================================================================================

final_state <- read.csv(
    FINAL_STATE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

final_summary <- read.csv(
    FINAL_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

evidence <- read.csv(
    EVIDENCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

perm_run <- read.csv(
    PERM_RUN_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

identity <- read.csv(
    IDENTITY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 7. SCHEMA VALIDATION
# ==================================================================================================

required_evidence_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "direction",
    "outer_selection_frequency",
    "outer_sign_consistency",
    "half_sample_selection_probability",
    "half_sample_sign_consistency",
    "null_selection_probability",
    "half_sample_median_nonzero_beta"
)

missing_evidence_cols <- setdiff(
    required_evidence_cols,
    names(evidence)
)

if (length(missing_evidence_cols) > 0L) {

    stop(
        "Final stability-evidence table missing field(s): ",
        paste(
            missing_evidence_cols,
            collapse = ", "
        )
    )
}

if (!all(
    c(
        "metric",
        "value"
    ) %in%
        names(final_summary)
)) {

    stop(
        "Final stable-state summary schema changed."
    )
}

if (!all(
    c(
        "permutation_id",
        "donor_auc"
    ) %in%
        names(perm_run)
)) {

    stop(
        "Permutation-run summary schema changed."
    )
}

if (!all(
    c(
        "ensemblID_stable",
        "interpretation_symbol"
    ) %in%
        names(identity)
)) {

    stop(
        "STEP10D.2 identity table lacks required identity fields."
    )
}

cat("Input schemas: PASS\n")

# ==================================================================================================
# 8. METRIC HELPER
# ==================================================================================================

get_metric <- function(
    x,
    metric_name
) {

    idx <- which(
        as.character(
            x$metric
        ) ==
            metric_name
    )

    if (length(idx) != 1L) {

        stop(
            "Could not uniquely retrieve frozen metric: ",
            metric_name
        )
    }

    value <- as.numeric(
        x$value[idx]
    )

    if (
        length(value) != 1L ||
        !is.finite(value)
    ) {

        stop(
            "Frozen metric is non-finite: ",
            metric_name
        )
    }

    value
}

# ==================================================================================================
# 9. FROZEN SUMMARY ANCHORS
# ==================================================================================================

FINAL_N <- get_metric(
    final_summary,
    "Final_stable_shared_genes"
)

FINAL_POSITIVE <- get_metric(
    final_summary,
    "Final_positive_direction"
)

FINAL_NEGATIVE <- get_metric(
    final_summary,
    "Final_negative_direction"
)

STABLE_DEVIATION <- get_metric(
    final_summary,
    "Stable_deviation_genes"
)

MIN_HALF_SELECTION <- get_metric(
    final_summary,
    "Minimum_half_sample_selection_probability"
)

MIN_HALF_SIGN <- get_metric(
    final_summary,
    "Minimum_half_sample_sign_consistency"
)

MAX_NULL_SELECTION <- get_metric(
    final_summary,
    "Maximum_candidate_null_selection_probability"
)

OBSERVED_AUC <- get_metric(
    final_summary,
    "Observed_reference_donor_AUROC"
)

NULL_MEAN_ANCHOR <- get_metric(
    final_summary,
    "Null_donor_AUROC_mean"
)

NULL_MAX_ANCHOR <- get_metric(
    final_summary,
    "Null_donor_AUROC_max"
)

EMPIRICAL_P_ANCHOR <- get_metric(
    final_summary,
    "Permutation_empirical_P"
)

# ==================================================================================================
# 10. STRICT FROZEN VALIDATION
# ==================================================================================================

if (FINAL_N != 24) {
    stop("Final stable feature count changed.")
}

if (
    FINAL_POSITIVE != 9 ||
    FINAL_NEGATIVE != 15
) {
    stop(
        "Final positive/negative direction counts changed."
    )
}

if (STABLE_DEVIATION != 0) {
    stop(
        "Stable deviation-feature count changed."
    )
}

if (
    abs(
        MIN_HALF_SELECTION -
        0.80
    ) >
        1e-12
) {
    stop(
        "Minimum half-sample selection probability changed."
    )
}

if (
    abs(
        MIN_HALF_SIGN -
        1.00
    ) >
        1e-12
) {
    stop(
        "Minimum half-sample sign consistency changed."
    )
}

if (
    abs(
        MAX_NULL_SELECTION -
        0.22
    ) >
        1e-12
) {
    stop(
        "Maximum final-feature null-selection probability changed."
    )
}

if (
    abs(
        OBSERVED_AUC -
        0.76983010
    ) >
        5e-8
) {
    stop(
        "Observed matched-reference donor AUROC changed."
    )
}

cat(
    "Frozen final-state numerical anchors: PASS\n"
)

# ==================================================================================================
# 11. FINAL 24 SET INTEGRITY
# ==================================================================================================

if (nrow(final_state) != 24L) {
    stop(
        "Canonical final-state table must contain 24 rows."
    )
}

if (nrow(evidence) != 24L) {
    stop(
        "Final stability-evidence table must contain 24 rows."
    )
}

if (nrow(identity) != 24L) {
    stop(
        "STEP10D.2 identity table must contain 24 rows."
    )
}

if (anyDuplicated(
    evidence$ensemblID_stable
) > 0L) {
    stop(
        "Duplicated feature ID in stability evidence."
    )
}

if (anyDuplicated(
    identity$ensemblID_stable
) > 0L) {
    stop(
        "Duplicated feature ID in STEP10D.2 identity table."
    )
}

if (
    !setequal(
        final_state$ensemblID_stable,
        evidence$ensemblID_stable
    )
) {
    stop(
        "Final-state and stability-evidence feature sets disagree."
    )
}

if (
    !setequal(
        final_state$ensemblID_stable,
        identity$ensemblID_stable
    )
) {
    stop(
        "Final-state and STEP10D.2 identity feature sets disagree."
    )
}

cat(
    "Final 24-feature identity alignment: PASS\n"
)

# ==================================================================================================
# 12. ADD STEP10D.2 INTERPRETATION IDENTITIES
#
# DISPLAY ONLY.
# Frozen IDs remain untouched.
# ==================================================================================================

identity_index <- match(
    evidence$ensemblID_stable,
    identity$ensemblID_stable
)

if (anyNA(identity_index)) {
    stop(
        "STEP10D.2 identity matching failed."
    )
}

interpretation_symbol <- as.character(
    identity$interpretation_symbol[
        identity_index
    ]
)

interpretation_symbol <- trimws(
    interpretation_symbol
)

interpretation_missing <- (
    is.na(interpretation_symbol) |
    interpretation_symbol == "" |
    toupper(interpretation_symbol) == "NA"
)

frozen_symbol <- trimws(
    as.character(
        evidence$gene_symbol
    )
)

frozen_symbol_missing <- (
    is.na(frozen_symbol) |
    frozen_symbol == "" |
    toupper(frozen_symbol) == "NA"
)

display_symbol <- interpretation_symbol

# If no adjudicated interpretation symbol, retain frozen symbol.
display_symbol[
    interpretation_missing &
    !frozen_symbol_missing
] <- frozen_symbol[
    interpretation_missing &
    !frozen_symbol_missing
]

# If neither identity source has a name, stable Ensembl ID is the correct label.
still_missing <- (
    is.na(display_symbol) |
    trimws(display_symbol) == "" |
    toupper(trimws(display_symbol)) == "NA"
)

display_symbol[
    still_missing
] <- evidence$ensemblID_stable[
    still_missing
]

evidence$interpretation_symbol <- interpretation_symbol
evidence$display_symbol <- display_symbol

# ==================================================================================================
# 13. REPORT IDENTITY-LABEL CHANGES
# ==================================================================================================

identity_report <- data.frame(

    ensemblID_stable =
        evidence$ensemblID_stable,

    frozen_symbol =
        evidence$gene_symbol,

    interpretation_symbol =
        evidence$interpretation_symbol,

    figure_display_symbol =
        evidence$display_symbol,

    stringsAsFactors = FALSE
)

cat(
    "\n",
    paste(rep("-", 110), collapse = ""),
    "\nFIGURE 4 IDENTITY DISPLAY AUDIT\n",
    paste(rep("-", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    identity_report,
    row.names = FALSE
)

N_ENSEMBL_ONLY <- sum(
    grepl(
        "^ENSG[0-9]+$",
        evidence$display_symbol
    )
)

cat(
    "\nFeatures remaining Ensembl-ID-only after STEP10D.2 adjudication: ",
    N_ENSEMBL_ONLY,
    "\n",
    sep = ""
)

# Frozen identity audit established exactly two without a current named identity.
if (N_ENSEMBL_ONLY != 2L) {

    stop(
        "Expected exactly two genuinely Ensembl-ID-only final features after identity adjudication; observed ",
        N_ENSEMBL_ONLY
    )
}

cat(
    "Expected unresolved identity count = 2 — PASS\n"
)

# ==================================================================================================
# 14. DIRECTION LABELS
# ==================================================================================================

if (any(
    !evidence$direction %in%
        c(
            "Positive",
            "Negative"
        )
)) {
    stop(
        "Unexpected direction label in final stability evidence."
    )
}

direction_suffix <- ifelse(
    evidence$direction ==
        "Positive",
    " (+)",
    " (-)"
)

evidence$display_gene <- paste0(
    evidence$display_symbol,
    direction_suffix
)

# ==================================================================================================
# 15. ORDER FEATURES
#
# Highest complementary half-sample selection probability first.
# Tie-break:
#   lower permutation-null selection probability
#   then display label
# ==================================================================================================

order_index <- order(
    -evidence$half_sample_selection_probability,
    evidence$null_selection_probability,
    evidence$display_gene
)

evidence <- evidence[
    order_index,
    ,
    drop = FALSE
]

gene_levels <- rev(
    evidence$display_gene
)

# ==================================================================================================
# 16. PANEL A DATA
# ==================================================================================================

panel_A_outer <- data.frame(

    display_gene =
        evidence$display_gene,

    metric =
        "Outer-CV\nselection",

    probability =
        evidence$outer_selection_frequency,

    stringsAsFactors = FALSE
)

panel_A_half <- data.frame(

    display_gene =
        evidence$display_gene,

    metric =
        "Half-sample\nselection",

    probability =
        evidence$half_sample_selection_probability,

    stringsAsFactors = FALSE
)

panel_A_null <- data.frame(

    display_gene =
        evidence$display_gene,

    metric =
        "Permutation-null\nselection",

    probability =
        evidence$null_selection_probability,

    stringsAsFactors = FALSE
)

panel_A <- rbind(
    panel_A_outer,
    panel_A_half,
    panel_A_null
)

panel_A$display_gene <- factor(
    panel_A$display_gene,
    levels = gene_levels
)

panel_A$metric <- factor(
    panel_A$metric,
    levels = c(
        "Outer-CV\nselection",
        "Half-sample\nselection",
        "Permutation-null\nselection"
    )
)

panel_A$text_colour <- ifelse(
    panel_A$probability >= 0.65,
    "white",
    "black"
)

# ==================================================================================================
# 17. PERMUTATION DATA VALIDATION
# ==================================================================================================

if (nrow(perm_run) != 100L) {

    stop(
        "Expected exactly 100 saved permutation runs; observed ",
        nrow(perm_run)
    )
}

if (any(
    !is.finite(
        perm_run$donor_auc
    )
)) {

    stop(
        "Non-finite permutation donor AUROC detected."
    )
}

NULL_MEAN <- mean(
    perm_run$donor_auc
)

NULL_SD <- stats::sd(
    perm_run$donor_auc
)

NULL_MIN <- min(
    perm_run$donor_auc
)

NULL_MAX <- max(
    perm_run$donor_auc
)

EMPIRICAL_P <- (
    1 +
    sum(
        perm_run$donor_auc >=
            OBSERVED_AUC
    )
) / (
    nrow(perm_run) +
    1
)

if (
    abs(
        NULL_MEAN -
        NULL_MEAN_ANCHOR
    ) >
        5e-8
) {

    stop(
        "Saved permutation-null mean disagrees with frozen summary."
    )
}

if (
    abs(
        NULL_MAX -
        NULL_MAX_ANCHOR
    ) >
        5e-8
) {

    stop(
        "Saved permutation-null maximum disagrees with frozen summary."
    )
}

if (
    abs(
        EMPIRICAL_P -
        EMPIRICAL_P_ANCHOR
    ) >
        5e-8
) {

    stop(
        "Empirical permutation P disagrees with frozen summary."
    )
}

if (
    NULL_MIN <= 0.42 ||
    NULL_MAX >= 0.80
) {

    stop(
        "Permutation data fall outside intended display window."
    )
}

panel_B <- data.frame(

    permutation_id =
        as.integer(
            perm_run$permutation_id
        ),

    donor_AUROC =
        as.numeric(
            perm_run$donor_auc
        ),

    stringsAsFactors = FALSE
)

cat(
    "Permutation-null saved distribution: PASS\n"
)

# ==================================================================================================
# 18. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "main",
    "Figure4_final_TNR_v3"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "Figure4_Stability_Permutation_TimesNewRoman_v3.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "Figure4_Stability_Permutation_TimesNewRoman_v3.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "Figure4_Stability_Permutation_TimesNewRoman_v3.png"
)

PANEL_A_PATH <- file.path(
    SOURCE_DIR,
    "Figure4_PanelA_final24_stability_identityaware_source_v3.csv"
)

PANEL_B_PATH <- file.path(
    SOURCE_DIR,
    "Figure4_PanelB_permutation_AUROC_source_v3.csv"
)

IDENTITY_SOURCE_PATH <- file.path(
    SOURCE_DIR,
    "Figure4_feature_display_identity_map_v3.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "Figure4_generation_summary_v3.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Figure4_generation_manifest_v3.csv"
)

write.csv(
    panel_A,
    PANEL_A_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    panel_B,
    PANEL_B_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    identity_report,
    IDENTITY_SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# ==================================================================================================
# 19. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.5,
            hjust = 0,
            margin = ggplot2::margin(
                b = 6
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.5
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.2,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 8.0
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.5
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            8,
            9,
            8,
            9
        )
    )

# ==================================================================================================
# 20. PANEL A — IDENTITY-AWARE STABILITY HEATMAP
# ==================================================================================================

pA <- ggplot2::ggplot(
    panel_A,
    ggplot2::aes(
        x = metric,
        y = display_gene,
        fill = probability
    )
) +
    ggplot2::geom_tile(
        colour = "white",
        linewidth = 0.55,
        width = 0.97,
        height = 0.96
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = sprintf(
                "%.2f",
                probability
            ),
            colour = text_colour
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.20,
        show.legend = FALSE
    ) +
    ggplot2::scale_colour_identity() +
    ggplot2::scale_fill_gradientn(
        colours = c(
            "#F1F1F1",
            "#E9C46A",
            "#2A9D8F",
            "#264653"
        ),
        values = c(
            0.00,
            0.35,
            0.75,
            1.00
        ),
        limits = c(
            0,
            1
        ),
        breaks = c(
            0.00,
            0.25,
            0.50,
            0.75,
            1.00
        ),
        labels = c(
            "0.00",
            "0.25",
            "0.50",
            "0.75",
            "1.00"
        ),
        guide = ggplot2::guide_colourbar(
            title.position = "top",
            title.hjust = 0.5,
            label.position = "bottom",
            barwidth = grid::unit(
                4.0,
                "cm"
            ),
            barheight = grid::unit(
                0.24,
                "cm"
            ),
            ticks = TRUE,
            frame.colour = NA
        )
    ) +
    ggplot2::labs(
        title = "A  Stability of the final 24-feature shared state",
        x = NULL,
        y = NULL,
        fill = "Selection probability"
    ) +
    theme_publication +
    ggplot2::theme(

        axis.line =
            ggplot2::element_blank(),

        axis.ticks =
            ggplot2::element_blank(),

        axis.text.x =
            ggplot2::element_text(
                family = FONT_FAMILY,
                size = 8.0,
                face = "bold",
                lineheight = 0.90
            ),

        axis.text.y =
            ggplot2::element_text(
                family = FONT_FAMILY,
                size = 7.25
            ),

        legend.position = "bottom",

        legend.justification = "center",

        legend.box.just = "center",

        legend.margin = ggplot2::margin(
            t = 2,
            r = 2,
            b = 0,
            l = 2
        )
    )

# ==================================================================================================
# 21. PANEL B — WARNING-FREE PERMUTATION HISTOGRAM
# ==================================================================================================

annotation_text <- paste0(

    "Observed AUROC = ",
    sprintf(
        "%.3f",
        OBSERVED_AUC
    ),

    "\nNull mean = ",
    sprintf(
        "%.3f",
        NULL_MEAN
    ),

    " (SD ",
    sprintf(
        "%.3f",
        NULL_SD
    ),
    ")",

    "\nNull max = ",
    sprintf(
        "%.3f",
        NULL_MAX
    ),

    "\nEmpirical p = ",
    sprintf(
        "%.4f",
        EMPIRICAL_P
    )
)

pB <- ggplot2::ggplot(
    panel_B,
    ggplot2::aes(
        x = donor_AUROC
    )
) +
    ggplot2::geom_histogram(
        binwidth = 0.015,
        boundary = 0.40,
        closed = "left",
        fill = "#76B7B2",
        colour = "white",
        linewidth = 0.45
    ) +
    ggplot2::geom_rug(
        sides = "b",
        alpha = 0.45,
        linewidth = 0.35,
        colour = "#3D6F6C"
    ) +
    ggplot2::geom_vline(
        xintercept = NULL_MEAN,
        linetype = "dashed",
        linewidth = 0.70,
        colour = "#333333"
    ) +
    ggplot2::geom_vline(
        xintercept = OBSERVED_AUC,
        linewidth = 1.05,
        colour = "#E15759"
    ) +
    ggplot2::annotate(
        "text",
        x = 0.792,
        y = Inf,
        label = annotation_text,
        hjust = 1,
        vjust = 1.10,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.60,
        lineheight = 1.05
    ) +
    ggplot2::scale_x_continuous(
        breaks = seq(
            0.45,
            0.80,
            by = 0.05
        ),
        expand = ggplot2::expansion(
            mult = c(
                0.01,
                0.01
            )
        )
    ) +
    ggplot2::scale_y_continuous(
        expand = ggplot2::expansion(
            mult = c(
                0,
                0.10
            )
        )
    ) +
    ggplot2::coord_cartesian(
        xlim = c(
            0.42,
            0.80
        ),
        clip = "on"
    ) +
    ggplot2::labs(
        title = "B  Diagnosis-permutation null",
        x = "Donor-level AUROC",
        y = "Permutation count"
    ) +
    theme_publication

# ==================================================================================================
# 22. DRAW FINAL FIGURE
#
# Panel A receives slightly more horizontal space because it carries:
#   - 24 feature labels
#   - three heatmap columns
#   - continuous legend
# ==================================================================================================

draw_figure4 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 1,
        ncol = 2,
        widths = grid::unit(
            c(
                1.48,
                0.92
            ),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 23. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 5.9,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure4()

grDevices::dev.off()

# ==================================================================================================
# 24. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 5.9,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure4()

grDevices::dev.off()

# ==================================================================================================
# 25. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 5.9,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure4()

grDevices::dev.off()

# ==================================================================================================
# 26. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    PANEL_A_PATH,
    PANEL_B_PATH,
    IDENTITY_SOURCE_PATH
)

if (!all(
    file.exists(
        outputs
    )
)) {

    stop(
        "One or more Figure 4 v3 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {

    stop(
        "One or more Figure 4 v3 outputs are zero-byte."
    )
}

cat(
    "Figure 4 v3 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 27. SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11C.5S — MAIN FIGURE 4 — FINAL IDENTITY-AWARE VERSION",
    "==============================================================================================================",

    "",

    "Frozen analysis identifiers remain unchanged.",

    "STEP10D.2 adjudicated identities used for figure display labels only.",

    "",

    paste0(
        "Final stable shared features = ",
        FINAL_N
    ),

    paste0(
        "Positive / Negative = ",
        FINAL_POSITIVE,
        " / ",
        FINAL_NEGATIVE
    ),

    paste0(
        "Stable deviation features = ",
        STABLE_DEVIATION
    ),

    paste0(
        "Minimum half-sample selection probability = ",
        MIN_HALF_SELECTION
    ),

    paste0(
        "Maximum final-feature null-selection probability = ",
        MAX_NULL_SELECTION
    ),

    paste0(
        "Features remaining Ensembl-ID-only after identity adjudication = ",
        N_ENSEMBL_ONLY
    ),

    "",

    paste0(
        "Permutation runs = ",
        nrow(
            panel_B
        )
    ),

    paste0(
        "Observed donor AUROC = ",
        format(
            OBSERVED_AUC,
            digits = 9
        )
    ),

    paste0(
        "Null mean = ",
        format(
            NULL_MEAN,
            digits = 9
        )
    ),

    paste0(
        "Null SD = ",
        format(
            NULL_SD,
            digits = 9
        )
    ),

    paste0(
        "Null maximum = ",
        format(
            NULL_MAX,
            digits = 9
        )
    ),

    paste0(
        "Empirical permutation P = ",
        format(
            EMPIRICAL_P,
            digits = 9
        )
    ),

    "",

    "No stability analysis or permutation analysis rerun.",

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 28. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 29. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFINAL FIGURE 4 v3 GENERATION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Frozen source validation: PASS\n"
)

cat(
    "STEP10D.2 identity alignment: PASS\n"
)

cat(
    "Frozen analysis identities changed: NO\n"
)

cat(
    "Interpretation identities used for display only: YES\n"
)

cat(
    "Features remaining Ensembl-ID-only: ",
    N_ENSEMBL_ONLY,
    "\n",
    sep = ""
)

cat(
    "\nFinal stable shared features: 24\n"
)

cat(
    "Positive / Negative: 9 / 15\n"
)

cat(
    "Stable deviation features: 0\n"
)

cat(
    "Minimum half-sample selection probability: 0.800\n"
)

cat(
    "Maximum final-feature null-selection probability: 0.220\n"
)

cat(
    "\nPermutation null:\n"
)

cat(
    "Runs: ",
    nrow(
        panel_B
    ),
    "\n",
    sep = ""
)

cat(
    "Observed donor AUROC: ",
    sprintf(
        "%.7f",
        OBSERVED_AUC
    ),
    "\n",
    sep = ""
)

cat(
    "Null mean +/- SD: ",
    sprintf(
        "%.7f",
        NULL_MEAN
    ),
    " +/- ",
    sprintf(
        "%.7f",
        NULL_SD
    ),
    "\n",
    sep = ""
)

cat(
    "Null maximum: ",
    sprintf(
        "%.7f",
        NULL_MAX
    ),
    "\n",
    sep = ""
)

cat(
    "Empirical P: ",
    sprintf(
        "%.8f",
        EMPIRICAL_P
    ),
    "\n",
    sep = ""
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\nIdentity display map:\n",
    IDENTITY_SOURCE_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.5S COMPLETE — FIGURE 4 FINAL IDENTITY-AWARE VERSION GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.5S — MAIN FIGURE 4 — FINAL IDENTITY-AWARE VISUAL VERSION
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Frozen stability, permutation, and identity artifacts: PASS
Canonical final 24-feature SHA256: PASS
Input schemas: PASS
Frozen final-state numerical anchors: PASS
Final 24-feature identity alignment: PASS

--------------------------------------------------------------------------------------------------------------
FIGURE 4 IDENTITY DISPLAY AUDIT
--------------------------------------------------------------------------------------------------------------
 ensemblID_stable frozen_symbol interpretation_symbol figure_display_symbol
  ENSG00000100092        SH3BP1                SH3BP1                SH3BP1
  ENSG00000117877        CD3EAP                POLR1G                POLR1G
  ENSG00000196565          HBG2                  HBG2                  HBG2
  ENSG00000251521       IMPA1P1               IMPA1P1               IMPA1P1
  ENSG00000260526

agg_record_211c288d3341 
                      2

agg_record_211c288d3341 
                      2

agg_record_211c288d3341 
                      2

Figure 4 v3 outputs created and non-zero: PASS

FINAL FIGURE 4 v3 GENERATION SUMMARY
Frozen source validation: PASS
STEP10D.2 identity alignment: PASS
Frozen analysis identities changed: NO
Interpretation identities used for display only: YES
Features remaining Ensembl-ID-only: 2

Final stable shared features: 24
Positive / Negative: 9 / 15
Stable deviation features: 0
Minimum half-sample selection probability: 0.800
Maximum final-feature null-selection probability: 0.220

Permutation null:
Runs: 100
Observed donor AUROC: 0.7698301
Null mean +/- SD: 0.5064535 +/- 0.0325831
Null maximum: 0.5995023
Empirical P: 0.00990099

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure4_final_TNR_v3/Figure4_Stability_Permutation_TimesNewRoman_v3.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure4_final_TNR_v3/Figure4_Stability_Permutation_TimesNewRoman_v3.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure4_final_TNR_v3/Figure4_Stab

In [102]:
# ==================================================================================================
# STEP 11C.6S — MAIN FIGURE 5 — FINAL VISUAL/TEXT CORRECTION
# EXTERNAL TECHNICAL ELIGIBILITY AND PRIMARY EXTERNAL VALIDATION
# --------------------------------------------------------------------------------------------------
# PURPOSE
#
# This cell performs VISUAL/TEXT correction only, using the already validated Figure 5 v2
# source-data files.
#
# CHANGES FROM v2
#
#   1. Panel B title shortened:
#          "B  GSE224683 external validation"
#
#   2. Overall-gate labels changed from:
#          PASS / FAIL
#      to:
#          ELIGIBLE / NON-EVALUABLE
#
#      This prevents technical-gate failure from being misread as biological replication failure.
#
#   3. Panel A overall-gate column widened.
#
#   4. Panel B receives slightly more horizontal space.
#
#   5. Statistical annotation compacted slightly.
#
# NO ANALYSIS IS RERUN
#   - no score calculation
#   - no mapping
#   - no phenotype processing
#   - no bootstrap
#   - no model fitting
#   - no external tuning
#
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.6S — MAIN FIGURE 5 — FINAL VISUAL/TEXT CORRECTION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. VERIFIED FIGURE 5 v2 SOURCE FILES
# ==================================================================================================

V2_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "main",
    "Figure5_final_TNR_v2"
)

V2_SOURCE_DIR <- file.path(
    V2_DIR,
    "source_data"
)

PANEL_A_SOURCE_V2 <- file.path(
    V2_SOURCE_DIR,
    "Figure5_PanelA_technical_eligibility_source_v2.csv"
)

PANEL_B_SOURCE_V2 <- file.path(
    V2_SOURCE_DIR,
    "Figure5_PanelB_GSE224683_frozen_scores_source_v2.csv"
)

STAT_SOURCE_V2 <- file.path(
    V2_SOURCE_DIR,
    "Figure5_GSE224683_frozen_statistics_source_v2.csv"
)

required_inputs <- c(
    PANEL_A_SOURCE_V2,
    PANEL_B_SOURCE_V2,
    STAT_SOURCE_V2
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required validated Figure 5 v2 source-data file(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat(
    "Validated Figure 5 v2 source-data files: PASS\n"
)

# ==================================================================================================
# 5. LOAD VERIFIED SOURCE DATA
# ==================================================================================================

panel_A <- read.csv(
    PANEL_A_SOURCE_V2,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

panel_B <- read.csv(
    PANEL_B_SOURCE_V2,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

stats_source <- read.csv(
    STAT_SOURCE_V2,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 6. SOURCE SCHEMA CHECKS
# ==================================================================================================

required_A_cols <- c(
    "dataset",
    "metric",
    "coverage",
    "threshold",
    "component_pass",
    "value_label",
    "x_position"
)

missing_A_cols <- setdiff(
    required_A_cols,
    names(panel_A)
)

if (length(missing_A_cols) > 0L) {

    stop(
        "Panel A v2 source missing field(s): ",
        paste(
            missing_A_cols,
            collapse = ", "
        )
    )
}

required_B_cols <- c(
    "sample_name",
    "diagnosis",
    "SCZ_MSS"
)

missing_B_cols <- setdiff(
    required_B_cols,
    names(panel_B)
)

if (length(missing_B_cols) > 0L) {

    stop(
        "Panel B v2 source missing field(s): ",
        paste(
            missing_B_cols,
            collapse = ", "
        )
    )
}

if (!all(
    c(
        "metric",
        "value",
        "source"
    ) %in%
        names(stats_source)
)) {

    stop(
        "Figure 5 v2 frozen-statistics source schema changed."
    )
}

cat(
    "Figure 5 v2 source schemas: PASS\n"
)

# ==================================================================================================
# 7. NORMALIZE component_pass
# ==================================================================================================

normalize_logical <- function(x) {

    if (is.logical(x)) {
        return(x)
    }

    z <- toupper(
        trimws(
            as.character(x)
        )
    )

    out <- rep(
        NA,
        length(z)
    )

    out[
        z %in%
            c(
                "TRUE",
                "T",
                "1",
                "YES",
                "PASS"
            )
    ] <- TRUE

    out[
        z %in%
            c(
                "FALSE",
                "F",
                "0",
                "NO",
                "FAIL"
            )
    ] <- FALSE

    if (anyNA(out)) {

        stop(
            "Unexpected component_pass encoding: ",
            paste(
                unique(
                    z[
                        is.na(out)
                    ]
                ),
                collapse = ", "
            )
        )
    }

    out
}

panel_A$component_pass <- normalize_logical(
    panel_A$component_pass
)

# ==================================================================================================
# 8. STRICT FIVE-COHORT STRUCTURE CHECK
# ==================================================================================================

EXPECTED_DATASETS <- c(
    "GSE87194",
    "GSE80655",
    "GSE224683",
    "GSE282552",
    "GSE208338"
)

EXPECTED_METRICS <- c(
    "Gene coverage",
    "Absolute-weight coverage",
    "Positive-weight coverage",
    "Negative-weight coverage"
)

if (
    length(
        unique(
            panel_A$dataset
        )
    ) != 5L ||
    !setequal(
        unique(
            panel_A$dataset
        ),
        EXPECTED_DATASETS
    )
) {

    stop(
        "Panel A does not contain the frozen five-cohort set."
    )
}

if (
    length(
        unique(
            panel_A$metric
        )
    ) != 4L ||
    !setequal(
        unique(
            panel_A$metric
        ),
        EXPECTED_METRICS
    )
) {

    stop(
        "Panel A does not contain the frozen four technical-gate components."
    )
}

if (nrow(panel_A) != 20L) {

    stop(
        "Expected 20 Panel A coverage cells; observed ",
        nrow(panel_A)
    )
}

cat(
    "Five-cohort x four-component technical-gate structure: PASS\n"
)

# ==================================================================================================
# 9. STRICT COVERAGE ANCHORS
# ==================================================================================================

anchor_table <- data.frame(

    dataset = c(
        "GSE87194",
        "GSE80655",
        "GSE224683",
        "GSE282552",
        "GSE208338"
    ),

    gene = c(
        0.5833330,
        0.8333330,
        0.9583330,
        0.3750000,
        0.5833333
    ),

    absolute = c(
        0.3332160,
        0.7197100,
        0.9266000,
        0.1961925,
        0.3818154
    ),

    positive = c(
        0.2611400,
        1.0000000,
        1.0000000,
        0.0304038,
        0.2611397
    ),

    negative = c(
        0.3787040,
        0.5428170,
        0.8802760,
        0.3008226,
        0.4579744
    ),

    stringsAsFactors = FALSE
)

check_metric_values <- function(
    metric_name,
    anchor_column
) {

    z <- panel_A[
        panel_A$metric ==
            metric_name,
        ,
        drop = FALSE
    ]

    idx <- match(
        z$dataset,
        anchor_table$dataset
    )

    expected <- anchor_table[
        idx,
        anchor_column
    ]

    if (any(
        abs(
            z$coverage -
            expected
        ) >
            1e-6
    )) {

        stop(
            "Frozen coverage anchor changed for: ",
            metric_name
        )
    }
}

check_metric_values(
    "Gene coverage",
    "gene"
)

check_metric_values(
    "Absolute-weight coverage",
    "absolute"
)

check_metric_values(
    "Positive-weight coverage",
    "positive"
)

check_metric_values(
    "Negative-weight coverage",
    "negative"
)

cat(
    "Frozen five-cohort coverage anchors: PASS\n"
)

# ==================================================================================================
# 10. OVERALL TECHNICAL ELIGIBILITY
#
# A cohort is eligible only when ALL FOUR component gates pass.
# ==================================================================================================

overall_gate_raw <- aggregate(
    component_pass ~ dataset,
    data = panel_A,
    FUN = all
)

colnames(
    overall_gate_raw
)[
    colnames(
        overall_gate_raw
    ) ==
        "component_pass"
] <- "technical_eligible"

if (
    sum(
        overall_gate_raw$technical_eligible
    ) != 1L
) {

    stop(
        "Expected exactly one technically eligible public cohort."
    )
}

eligible_dataset <- overall_gate_raw$dataset[
    overall_gate_raw$technical_eligible
]

if (!identical(
    eligible_dataset,
    "GSE224683"
)) {

    stop(
        "Technically eligible cohort is not GSE224683."
    )
}

cat(
    "Overall technical eligibility: 1 eligible / 4 non-evaluable — PASS\n"
)

# ==================================================================================================
# 11. PANEL B SAMPLE VALIDATION
# ==================================================================================================

panel_B$diagnosis <- factor(
    panel_B$diagnosis,
    levels = c(
        "Control",
        "Schizophrenia"
    )
)

if (nrow(panel_B) != 30L) {

    stop(
        "Expected 30 frozen GSE224683 samples."
    )
}

if (
    sum(
        panel_B$diagnosis ==
            "Control"
    ) != 15L ||
    sum(
        panel_B$diagnosis ==
            "Schizophrenia"
    ) != 15L
) {

    stop(
        "GSE224683 diagnosis counts are not 15 / 15."
    )
}

if (
    anyNA(
        panel_B$SCZ_MSS
    ) ||
    any(
        !is.finite(
            panel_B$SCZ_MSS
        )
    )
) {

    stop(
        "Invalid frozen GSE224683 SCZ-MSS value."
    )
}

cat(
    "GSE224683 frozen samples: 30 total / 15 Control / 15 Schizophrenia — PASS\n"
)

# ==================================================================================================
# 12. STATISTIC HELPER
# ==================================================================================================

get_stat <- function(
    metric_name
) {

    idx <- which(
        trimws(
            as.character(
                stats_source$metric
            )
        ) ==
            metric_name
    )

    if (length(idx) != 1L) {

        stop(
            "Could not uniquely retrieve frozen Figure 5 statistic: ",
            metric_name
        )
    }

    value <- as.numeric(
        stats_source$value[idx]
    )

    if (
        length(value) != 1L ||
        !is.finite(value)
    ) {

        stop(
            "Frozen Figure 5 statistic is non-finite: ",
            metric_name
        )
    }

    value
}

# ==================================================================================================
# 13. FROZEN EXTERNAL STATISTICS
# ==================================================================================================

CONTROL_MEAN <- get_stat(
    "Control_mean"
)

CONTROL_SD <- get_stat(
    "Control_SD"
)

SCZ_MEAN <- get_stat(
    "Schizophrenia_mean"
)

SCZ_SD <- get_stat(
    "Schizophrenia_SD"
)

MEAN_DIFF <- get_stat(
    "Mean_difference"
)

MEAN_DIFF_LOW <- get_stat(
    "Mean_difference_CI_low"
)

MEAN_DIFF_HIGH <- get_stat(
    "Mean_difference_CI_high"
)

HEDGES_G <- get_stat(
    "Hedges_g"
)

HEDGES_G_LOW <- get_stat(
    "Hedges_g_CI_low"
)

HEDGES_G_HIGH <- get_stat(
    "Hedges_g_CI_high"
)

AUROC <- get_stat(
    "AUROC"
)

AUROC_LOW <- get_stat(
    "AUROC_CI_low"
)

AUROC_HIGH <- get_stat(
    "AUROC_CI_high"
)

WELCH_P <- get_stat(
    "Welch_p"
)

# ==================================================================================================
# 14. STRICT FROZEN NUMERIC CHECKS
# ==================================================================================================

if (
    abs(
        AUROC -
        0.5067
    ) >
        5e-4
) {

    stop(
        "Frozen AUROC changed."
    )
}

if (
    abs(
        AUROC_LOW -
        0.2889
    ) >
        5e-4 ||
    abs(
        AUROC_HIGH -
        0.7244
    ) >
        5e-4
) {

    stop(
        "Frozen AUROC CI changed."
    )
}

if (
    abs(
        MEAN_DIFF -
        (-0.000369)
    ) >
        5e-6
) {

    stop(
        "Frozen mean difference changed."
    )
}

if (
    abs(
        MEAN_DIFF_LOW -
        (-0.006485)
    ) >
        5e-6 ||
    abs(
        MEAN_DIFF_HIGH -
        0.005459
    ) >
        5e-6
) {

    stop(
        "Frozen mean-difference CI changed."
    )
}

if (
    abs(
        HEDGES_G -
        (-0.0422)
    ) >
        5e-4
) {

    stop(
        "Frozen Hedges g changed."
    )
}

if (
    abs(
        HEDGES_G_LOW -
        (-0.7574)
    ) >
        5e-4 ||
    abs(
        HEDGES_G_HIGH -
        0.7262
    ) >
        5e-4
) {

    stop(
        "Frozen Hedges g CI changed."
    )
}

if (
    abs(
        WELCH_P -
        0.90635
    ) >
        5e-5
) {

    stop(
        "Frozen Welch p changed."
    )
}

cat(
    "Frozen GSE224683 external outcome statistics: PASS\n"
)

# ==================================================================================================
# 15. SCORE DESCRIPTIVE CROSS-CHECK
# ==================================================================================================

control_scores <- panel_B$SCZ_MSS[
    panel_B$diagnosis ==
        "Control"
]

scz_scores <- panel_B$SCZ_MSS[
    panel_B$diagnosis ==
        "Schizophrenia"
]

if (
    abs(
        mean(
            control_scores
        ) -
        CONTROL_MEAN
    ) >
        1e-10
) {

    stop(
        "Frozen Control score mean disagrees with saved statistic."
    )
}

if (
    abs(
        mean(
            scz_scores
        ) -
        SCZ_MEAN
    ) >
        1e-10
) {

    stop(
        "Frozen Schizophrenia score mean disagrees with saved statistic."
    )
}

cat(
    "Frozen sample scores vs saved means: PASS\n"
)

# ==================================================================================================
# 16. PANEL A FACTOR ORDER
# ==================================================================================================

cohort_order <- c(
    "GSE87194",
    "GSE80655",
    "GSE224683",
    "GSE282552",
    "GSE208338"
)

panel_A$dataset <- factor(
    as.character(
        panel_A$dataset
    ),
    levels = rev(
        cohort_order
    )
)

panel_A$metric <- factor(
    as.character(
        panel_A$metric
    ),
    levels = c(
        "Gene coverage",
        "Absolute-weight coverage",
        "Positive-weight coverage",
        "Negative-weight coverage"
    )
)

metric_positions <- c(
    "Gene coverage" = 1,
    "Absolute-weight coverage" = 2,
    "Positive-weight coverage" = 3,
    "Negative-weight coverage" = 4
)

panel_A$x_position <- unname(
    metric_positions[
        as.character(
            panel_A$metric
        )
    ]
)

panel_A$value_label <- sprintf(
    "%.2f",
    panel_A$coverage
)

# ==================================================================================================
# 17. OVERALL GATE DISPLAY LABELS
# ==================================================================================================

overall_gate <- overall_gate_raw

overall_gate$dataset <- factor(
    overall_gate$dataset,
    levels = rev(
        cohort_order
    )
)

overall_gate$x_position <- 5.00

overall_gate$status_label <- ifelse(
    overall_gate$technical_eligible,
    "ELIGIBLE",
    "NON-\nEVALUABLE"
)

# ==================================================================================================
# 18. OUTPUT DIRECTORY
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "main",
    "Figure5_final_TNR_v3"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "Figure5_ExternalValidation_TimesNewRoman_v3.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "Figure5_ExternalValidation_TimesNewRoman_v3.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "Figure5_ExternalValidation_TimesNewRoman_v3.png"
)

PANEL_A_OUT_PATH <- file.path(
    SOURCE_DIR,
    "Figure5_PanelA_technical_eligibility_source_v3.csv"
)

PANEL_B_OUT_PATH <- file.path(
    SOURCE_DIR,
    "Figure5_PanelB_GSE224683_frozen_scores_source_v3.csv"
)

STAT_OUT_PATH <- file.path(
    SOURCE_DIR,
    "Figure5_GSE224683_frozen_statistics_source_v3.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "Figure5_generation_summary_v3.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Figure5_generation_manifest_v3.csv"
)

write.csv(
    panel_A,
    PANEL_A_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    panel_B,
    PANEL_B_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    stats_source,
    STAT_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 19. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.3,
            hjust = 0,
            margin = ggplot2::margin(
                b = 6
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.5
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.2,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 8.0
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.6
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            8,
            10,
            8,
            10
        )
    )

# ==================================================================================================
# 20. PANEL A — TECHNICAL ELIGIBILITY
# ==================================================================================================

eligibility_colours <- c(
    "FALSE" = "#C44E52",
    "TRUE" = "#2E8B57"
)

pA <- ggplot2::ggplot(
    panel_A,
    ggplot2::aes(
        x = x_position,
        y = dataset,
        fill = coverage,
        colour = component_pass
    )
) +
    ggplot2::geom_tile(
        width = 0.92,
        height = 0.86,
        linewidth = 1.00
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = value_label
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.70,
        colour = "black",
        show.legend = FALSE
    ) +
    ggplot2::geom_text(
        data = overall_gate,
        ggplot2::aes(
            x = x_position,
            y = dataset,
            label = status_label,
            colour = technical_eligible
        ),
        inherit.aes = FALSE,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35,
        lineheight = 0.86
    ) +
    ggplot2::scale_colour_manual(
        values = eligibility_colours,
        guide = "none"
    ) +
    ggplot2::scale_fill_gradientn(
        colours = c(
            "#F2F2F2",
            "#F3D58A",
            "#7BC8A4",
            "#246B64"
        ),
        values = c(
            0.00,
            0.35,
            0.75,
            1.00
        ),
        limits = c(
            0,
            1
        ),
        breaks = c(
            0.00,
            0.25,
            0.50,
            0.75,
            1.00
        ),
        labels = c(
            "0.00",
            "0.25",
            "0.50",
            "0.75",
            "1.00"
        ),
        guide = ggplot2::guide_colourbar(
            title.position = "top",
            title.hjust = 0.5,
            label.position = "bottom",
            barwidth = grid::unit(
                3.7,
                "cm"
            ),
            barheight = grid::unit(
                0.24,
                "cm"
            )
        )
    ) +
    ggplot2::scale_x_continuous(
        breaks = c(
            1,
            2,
            3,
            4,
            5.00
        ),
        labels = c(
            "Gene\n>=0.80",
            "Absolute weight\n>=0.80",
            "Positive weight\n>=0.70",
            "Negative weight\n>=0.70",
            "Technical\nstatus"
        ),
        limits = c(
            0.48,
            5.50
        ),
        expand = c(
            0,
            0
        )
    ) +
    ggplot2::labs(
        title = "A  Technical eligibility across five public cohorts",
        x = NULL,
        y = NULL,
        fill = "Frozen coverage"
    ) +
    theme_publication +
    ggplot2::theme(

        axis.line =
            ggplot2::element_blank(),

        axis.ticks =
            ggplot2::element_blank(),

        axis.text.x =
            ggplot2::element_text(
                family = FONT_FAMILY,
                size = 7.15,
                face = "bold",
                lineheight = 0.86
            ),

        axis.text.y =
            ggplot2::element_text(
                family = FONT_FAMILY,
                size = 8.25,
                face = "bold"
            ),

        legend.position =
            "bottom",

        legend.justification =
            "center",

        legend.box.just =
            "center"
    )

# ==================================================================================================
# 21. PANEL B — GSE224683 EXTERNAL VALIDATION
# ==================================================================================================

diagnosis_colours <- c(
    "Control" = "#4E79A7",
    "Schizophrenia" = "#E15759"
)

annotation_text <- paste0(

    "AUROC = ",
    sprintf(
        "%.3f",
        AUROC
    ),
    " [",
    sprintf(
        "%.3f",
        AUROC_LOW
    ),
    ", ",
    sprintf(
        "%.3f",
        AUROC_HIGH
    ),
    "]",

    "\nMean difference = ",
    sprintf(
        "%+.4f",
        MEAN_DIFF
    ),
    " [",
    sprintf(
        "%+.4f",
        MEAN_DIFF_LOW
    ),
    ", ",
    sprintf(
        "%+.4f",
        MEAN_DIFF_HIGH
    ),
    "]",

    "\nHedges' g = ",
    sprintf(
        "%+.3f",
        HEDGES_G
    ),
    " [",
    sprintf(
        "%+.3f",
        HEDGES_G_LOW
    ),
    ", ",
    sprintf(
        "%+.3f",
        HEDGES_G_HIGH
    ),
    "]",

    "\nWelch p = ",
    sprintf(
        "%.3f",
        WELCH_P
    )
)

score_min <- min(
    panel_B$SCZ_MSS
)

score_max <- max(
    panel_B$SCZ_MSS
)

score_range <- (
    score_max -
    score_min
)

if (
    !is.finite(
        score_range
    ) ||
    score_range <= 0
) {

    stop(
        "Unexpected GSE224683 frozen-score range."
    )
}

annotation_y <- (
    score_max +
    0.25 *
        score_range
)

pB <- ggplot2::ggplot(
    panel_B,
    ggplot2::aes(
        x = diagnosis,
        y = SCZ_MSS,
        colour = diagnosis
    )
) +
    ggplot2::geom_boxplot(
        ggplot2::aes(
            fill = diagnosis
        ),
        width = 0.52,
        alpha = 0.17,
        outlier.shape = NA,
        linewidth = 0.65,
        colour = "black",
        show.legend = FALSE
    ) +
    ggplot2::geom_point(
        position = ggplot2::position_jitter(
            width = 0.105,
            height = 0,
            seed = 20260930
        ),
        size = 2.0,
        alpha = 0.80,
        show.legend = FALSE
    ) +
    ggplot2::stat_summary(
        fun = mean,
        geom = "point",
        shape = 23,
        size = 3.5,
        stroke = 0.75,
        fill = "white",
        colour = "black"
    ) +
    ggplot2::annotate(
        "text",
        x = 1.5,
        y = annotation_y,
        label = annotation_text,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35,
        lineheight = 1.04,
        hjust = 0.5
    ) +
    ggplot2::scale_colour_manual(
        values = diagnosis_colours,
        guide = "none"
    ) +
    ggplot2::scale_fill_manual(
        values = diagnosis_colours,
        guide = "none"
    ) +
    ggplot2::scale_y_continuous(
        expand = ggplot2::expansion(
            mult = c(
                0.07,
                0.34
            )
        )
    ) +
    ggplot2::labs(
        title = "B  GSE224683 external validation",
        x = NULL,
        y = "Frozen SCZ-MSS v1"
    ) +
    theme_publication +
    ggplot2::theme(

        axis.text.x =
            ggplot2::element_text(
                family = FONT_FAMILY,
                size = 8.5,
                face = "bold"
            )
    )

# ==================================================================================================
# 22. DRAW FINAL FIGURE
# ==================================================================================================

draw_figure5 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 1,
        ncol = 2,
        widths = grid::unit(
            c(
                1.28,
                1.02
            ),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 23. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 4.65,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure5()

grDevices::dev.off()

# ==================================================================================================
# 24. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 4.65,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure5()

grDevices::dev.off()

# ==================================================================================================
# 25. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 4.65,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure5()

grDevices::dev.off()

# ==================================================================================================
# 26. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    PANEL_A_OUT_PATH,
    PANEL_B_OUT_PATH,
    STAT_OUT_PATH
)

if (!all(
    file.exists(
        outputs
    )
)) {

    stop(
        "One or more Figure 5 v3 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {

    stop(
        "One or more Figure 5 v3 outputs are zero-byte."
    )
}

cat(
    "Figure 5 v3 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 27. SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11C.6S — MAIN FIGURE 5 — FINAL VISUAL/TEXT VERSION",
    "==============================================================================================================",

    "",

    "Figure generated from already validated Figure 5 v2 frozen source-data files.",

    "No analytical operation was rerun.",

    "",

    "Cohorts screened = 5",

    "Technically eligible = 1",

    "Technically non-evaluable = 4",

    "Eligible cohort = GSE224683",

    "",

    "Terminology correction:",

    "Technical gate failures are labelled NON-EVALUABLE, not FAIL.",

    "This avoids implying biological non-replication from technical ineligibility.",

    "",

    paste0(
        "GSE224683 AUROC = ",
        sprintf(
            "%.4f",
            AUROC
        ),
        " [",
        sprintf(
            "%.4f",
            AUROC_LOW
        ),
        ", ",
        sprintf(
            "%.4f",
            AUROC_HIGH
        ),
        "]"
    ),

    paste0(
        "Mean difference = ",
        sprintf(
            "%+.6f",
            MEAN_DIFF
        ),
        " [",
        sprintf(
            "%+.6f",
            MEAN_DIFF_LOW
        ),
        ", ",
        sprintf(
            "%+.6f",
            MEAN_DIFF_HIGH
        ),
        "]"
    ),

    paste0(
        "Hedges g = ",
        sprintf(
            "%+.4f",
            HEDGES_G
        ),
        " [",
        sprintf(
            "%+.4f",
            HEDGES_G_LOW
        ),
        ", ",
        sprintf(
            "%+.4f",
            HEDGES_G_HIGH
        ),
        "]"
    ),

    paste0(
        "Welch p = ",
        sprintf(
            "%.5f",
            WELCH_P
        )
    ),

    "",

    "External biological transferability remains unresolved / challenged.",

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 28. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 29. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFINAL FIGURE 5 v3 GENERATION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Validated Figure 5 v2 numerical source data: PASS\n"
)

cat(
    "Numerical results changed: NO\n"
)

cat(
    "Technical-status wording corrected: YES\n"
)

cat(
    "Panel B title clipping correction applied: YES\n"
)

cat(
    "\nCohorts screened: 5\n"
)

cat(
    "Technically eligible: 1\n"
)

cat(
    "Technically non-evaluable: 4\n"
)

cat(
    "Eligible cohort: GSE224683\n"
)

cat(
    "\nGSE224683 primary external result:\n"
)

cat(
    "AUROC: ",
    sprintf(
        "%.4f",
        AUROC
    ),
    " [",
    sprintf(
        "%.4f",
        AUROC_LOW
    ),
    ", ",
    sprintf(
        "%.4f",
        AUROC_HIGH
    ),
    "]\n",
    sep = ""
)

cat(
    "Mean difference: ",
    sprintf(
        "%+.6f",
        MEAN_DIFF
    ),
    " [",
    sprintf(
        "%+.6f",
        MEAN_DIFF_LOW
    ),
    ", ",
    sprintf(
        "%+.6f",
        MEAN_DIFF_HIGH
    ),
    "]\n",
    sep = ""
)

cat(
    "Hedges g: ",
    sprintf(
        "%+.4f",
        HEDGES_G
    ),
    " [",
    sprintf(
        "%+.4f",
        HEDGES_G_LOW
    ),
    ", ",
    sprintf(
        "%+.4f",
        HEDGES_G_HIGH
    ),
    "]\n",
    sep = ""
)

cat(
    "Welch p: ",
    sprintf(
        "%.5f",
        WELCH_P
    ),
    "\n",
    sep = ""
)

cat(
    "\nExternal transferability status: UNRESOLVED / CHALLENGED\n"
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.6S COMPLETE — FIGURE 5 FINAL VISUAL/TEXT VERSION GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.6S — MAIN FIGURE 5 — FINAL VISUAL/TEXT CORRECTION
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Validated Figure 5 v2 source-data files: PASS
Figure 5 v2 source schemas: PASS
Five-cohort x four-component technical-gate structure: PASS
Frozen five-cohort coverage anchors: PASS
Overall technical eligibility: 1 eligible / 4 non-evaluable — PASS
GSE224683 frozen samples: 30 total / 15 Control / 15 Schizophrenia — PASS
Frozen GSE224683 external outcome statistics: PASS
Frozen sample scores vs saved means: PASS


agg_record_211c131e5c76 
                      2

agg_record_211c131e5c76 
                      2

agg_record_211c131e5c76 
                      2

Figure 5 v3 outputs created and non-zero: PASS

FINAL FIGURE 5 v3 GENERATION SUMMARY
Validated Figure 5 v2 numerical source data: PASS
Numerical results changed: NO
Technical-status wording corrected: YES
Panel B title clipping correction applied: YES

Cohorts screened: 5
Technically eligible: 1
Technically non-evaluable: 4
Eligible cohort: GSE224683

GSE224683 primary external result:
AUROC: 0.5067 [0.2889, 0.7244]
Mean difference: -0.000369 [-0.006485, +0.005459]
Hedges g: -0.0422 [-0.7574, +0.7262]
Welch p: 0.90635

External transferability status: UNRESOLVED / CHALLENGED

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure5_final_TNR_v3/Figure5_ExternalValidation_TimesNewRoman_v3.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure5_final_TNR_v3/Figure5_ExternalValidation_TimesNewRoman_v3.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/main/Figure5_final_TNR_v3/Figure5_ExternalValidation_TimesNewRoman_v3.png

STEP 11C.6S COM

In [103]:
# ==================================================================================================
# STEP 11C.7 — MAIN TABLE 2
# LEAKAGE-FREE MODEL PERFORMANCE AND PAIRED COMPARISONS
# --------------------------------------------------------------------------------------------------
# PURPOSE
#
# Create manuscript Main Table 2 from frozen saved model-comparison artifacts only.
#
# PART A
#   Repeated-OOF donor-level performance:
#     Model A — DLPFC-only
#     Model B — HIPPO-only
#     Model C/E — pooled/shared-only
#     Model D — independent regional models
#     Model F — shared + regional deviation
#     Covariate-only sensitivity
#
# PART B
#   Paired Model F comparisons:
#     F vs C/E
#     F vs D
#     F (DLPFC subset) vs A
#     F (HIPPO subset) vs B
#     F vs covariate-only
#
# IMPORTANT
#   - NO model fitting
#   - NO nested CV rerun
#   - NO bootstrap rerun
#   - NO score recalculation
#   - NO naive t-test of folds
#   - Model D retained explicitly
#   - Times New Roman HTML preview
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.7 — MAIN TABLE 2\n",
    "LEAKAGE-FREE MODEL PERFORMANCE AND PAIRED COMPARISONS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGE
# ==================================================================================================

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest package unavailable.")
}

cat("Required package: PASS\n")

# ==================================================================================================
# 3. FROZEN INPUT PATHS
# ==================================================================================================

MODEL_COMP_DIR <- file.path(
    PROJECT_ROOT,
    "06_models",
    "model_comparison_v1"
)

COV_DIR <- file.path(
    PROJECT_ROOT,
    "06_models",
    "baselines",
    "model_covariate_only_sensitivity_v1"
)

MODEL_PERF_PATH <- file.path(
    MODEL_COMP_DIR,
    "STEP05C_model_performance_summary_v1.csv"
)

COMPARISON_PATH <- file.path(
    MODEL_COMP_DIR,
    "STEP05C_paired_comparison_summary_v1.csv"
)

NOTES_PATH <- file.path(
    MODEL_COMP_DIR,
    "STEP05C_MODEL_COMPARISON_NOTES_v1.txt"
)

COV_SUMMARY_PATH <- file.path(
    COV_DIR,
    "covariate_only_sensitivity_summary_v1.csv"
)

required_inputs <- c(
    MODEL_PERF_PATH,
    COMPARISON_PATH,
    NOTES_PATH,
    COV_SUMMARY_PATH
)

missing_inputs <- required_inputs[
    !file.exists(
        required_inputs
    )
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required frozen model-comparison artifact(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat(
    "Required frozen model-comparison artifacts: PASS\n"
)

# ==================================================================================================
# 4. LOAD SAVED ARTIFACTS
# ==================================================================================================

model_perf <- read.csv(
    MODEL_PERF_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

comparisons <- read.csv(
    COMPARISON_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

cov_summary <- read.csv(
    COV_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 5. SCHEMA VALIDATION
# ==================================================================================================

required_perf_cols <- c(
    "model",
    "evaluation_population",
    "n_donors",
    "repeated_OOF_AUROC"
)

missing_perf_cols <- setdiff(
    required_perf_cols,
    names(model_perf)
)

if (length(missing_perf_cols) > 0L) {

    stop(
        "STEP05C model-performance file missing field(s): ",
        paste(
            missing_perf_cols,
            collapse = ", "
        )
    )
}

required_comparison_cols <- c(
    "comparison",
    "population",
    "n_donors",
    "mean_paired_fold_delta",
    "repeat_block_CI_low",
    "repeat_block_CI_high",
    "proportion_outer_folds_F_higher",
    "averaged_OOF_reference_AUROC",
    "averaged_OOF_comparator_AUROC",
    "averaged_OOF_delta_AUROC",
    "paired_donor_bootstrap_CI_low",
    "paired_donor_bootstrap_CI_high"
)

missing_comparison_cols <- setdiff(
    required_comparison_cols,
    names(comparisons)
)

if (length(missing_comparison_cols) > 0L) {

    stop(
        "STEP05C paired-comparison file missing field(s): ",
        paste(
            missing_comparison_cols,
            collapse = ", "
        )
    )
}

if (!all(
    c(
        "metric",
        "value"
    ) %in%
        names(cov_summary)
)) {

    stop(
        "Covariate-only sensitivity summary schema changed."
    )
}

cat("Input schemas: PASS\n")

# ==================================================================================================
# 6. EXPECTED MODEL SET
# ==================================================================================================

EXPECTED_MODELS <- c(
    "A_DLPFC_only",
    "B_HIPPO_only",
    "C_E_pooled_shared_only",
    "D_independent_regions",
    "F_shared_plus_deviation"
)

if (
    nrow(model_perf) != 5L ||
    !setequal(
        model_perf$model,
        EXPECTED_MODELS
    )
) {

    stop(
        "Frozen STEP05C model set changed."
    )
}

EXPECTED_COMPARISONS <- c(
    "F_minus_C_E_pooled_shared_only",
    "F_minus_D_independent_regions",
    "F_DLPFC_minus_A_DLPFC_only",
    "F_HIPPO_minus_B_HIPPO_only"
)

if (
    nrow(comparisons) != 4L ||
    !setequal(
        comparisons$comparison,
        EXPECTED_COMPARISONS
    )
) {

    stop(
        "Frozen STEP05C paired-comparison set changed."
    )
}

cat(
    "Frozen five-model + four-comparison structure: PASS\n"
)

# ==================================================================================================
# 7. ROBUST METRIC HELPER FOR COVARIATE-ONLY SUMMARY
# ==================================================================================================

get_unique_metric <- function(
    x,
    metric_name
) {

    idx <- which(
        trimws(
            as.character(
                x$metric
            )
        ) ==
            metric_name
    )

    if (length(idx) < 1L) {

        stop(
            "Metric not found: ",
            metric_name
        )
    }

    values <- suppressWarnings(
        as.numeric(
            x$value[idx]
        )
    )

    values <- values[
        is.finite(values)
    ]

    if (length(values) < 1L) {

        stop(
            "No finite value for metric: ",
            metric_name
        )
    }

    unique_values <- unique(
        values
    )

    if (length(unique_values) != 1L) {

        stop(
            "Conflicting values found for metric: ",
            metric_name,
            "\nObserved: ",
            paste(
                unique_values,
                collapse = ", "
            )
        )
    }

    unique_values[1L]
}

# ==================================================================================================
# 8. COVARIATE-ONLY FROZEN VALUES
# ==================================================================================================

COV_AUC <- get_unique_metric(
    cov_summary,
    "Covariate_only_repeated_OOF_AUROC"
)

COV_F_AUC <- get_unique_metric(
    cov_summary,
    "Model_F_repeated_OOF_AUROC"
)

COV_DELTA <- get_unique_metric(
    cov_summary,
    "Delta_F_minus_covariates"
)

COV_CI_LOW <- get_unique_metric(
    cov_summary,
    "Delta_95CI_low"
)

COV_CI_HIGH <- get_unique_metric(
    cov_summary,
    "Delta_95CI_high"
)

if (
    abs(
        COV_AUC -
        0.6143507
    ) >
        5e-7
) {

    stop(
        "Covariate-only repeated-OOF AUROC changed."
    )
}

if (
    abs(
        COV_F_AUC -
        0.8019743
    ) >
        5e-7
) {

    stop(
        "Model F AUROC in covariate sensitivity changed."
    )
}

if (
    abs(
        COV_DELTA -
        0.18762365
    ) >
        5e-7
) {

    stop(
        "F-minus-covariate delta changed."
    )
}

if (
    abs(
        COV_CI_LOW -
        0.140610149
    ) >
        5e-7 ||
    abs(
        COV_CI_HIGH -
        0.237415611
    ) >
        5e-7
) {

    stop(
        "F-minus-covariate paired-donor bootstrap CI changed."
    )
}

cat(
    "Covariate-only sensitivity anchors: PASS\n"
)

# ==================================================================================================
# 9. STRICT REPEATED-OOF MODEL ANCHORS
# ==================================================================================================

get_model_row <- function(
    model_code
) {

    idx <- which(
        model_perf$model ==
            model_code
    )

    if (length(idx) != 1L) {

        stop(
            "Could not uniquely locate model: ",
            model_code
        )
    }

    model_perf[
        idx,
        ,
        drop = FALSE
    ]
}

A_ROW <- get_model_row(
    "A_DLPFC_only"
)

B_ROW <- get_model_row(
    "B_HIPPO_only"
)

CE_ROW <- get_model_row(
    "C_E_pooled_shared_only"
)

D_ROW <- get_model_row(
    "D_independent_regions"
)

F_ROW <- get_model_row(
    "F_shared_plus_deviation"
)

anchor_check <- function(
    observed,
    expected,
    label,
    tolerance = 5e-7
) {

    if (
        !is.finite(observed) ||
        abs(
            observed -
            expected
        ) >
            tolerance
    ) {

        stop(
            label,
            " changed.\nExpected: ",
            expected,
            "\nObserved: ",
            observed
        )
    }
}

anchor_check(
    A_ROW$repeated_OOF_AUROC,
    0.7906762,
    "Model A repeated-OOF AUROC"
)

anchor_check(
    B_ROW$repeated_OOF_AUROC,
    0.8051920,
    "Model B repeated-OOF AUROC"
)

anchor_check(
    CE_ROW$repeated_OOF_AUROC,
    0.7987926,
    "Model C/E repeated-OOF AUROC"
)

anchor_check(
    D_ROW$repeated_OOF_AUROC,
    0.81502784,
    "Model D repeated-OOF AUROC"
)

anchor_check(
    F_ROW$repeated_OOF_AUROC,
    0.8019743,
    "Model F repeated-OOF AUROC"
)

if (
    A_ROW$n_donors != 379L ||
    B_ROW$n_donors != 376L ||
    CE_ROW$n_donors != 454L ||
    D_ROW$n_donors != 454L ||
    F_ROW$n_donors != 454L
) {

    stop(
        "Frozen model evaluation-population sizes changed."
    )
}

cat(
    "Five-model repeated-OOF anchors: PASS\n"
)

# ==================================================================================================
# 10. PART A — MODEL PERFORMANCE TABLE
# ==================================================================================================

table2A <- data.frame(

    Model = c(
        "A",
        "B",
        "C/E",
        "D",
        "F",
        "Sensitivity"
    ),

    Specification = c(
        "DLPFC-only elastic net",
        "HIPPO-only elastic net",
        "Pooled/shared-only elastic net",
        "Independent region-specific models",
        "Shared + regional-deviation model",
        "Covariates only"
    ),

    Evaluation_population = c(
        as.character(
            A_ROW$evaluation_population
        ),
        as.character(
            B_ROW$evaluation_population
        ),
        as.character(
            CE_ROW$evaluation_population
        ),
        as.character(
            D_ROW$evaluation_population
        ),
        as.character(
            F_ROW$evaluation_population
        ),
        "All discovery donors"
    ),

    N_donors = c(
        as.integer(
            A_ROW$n_donors
        ),
        as.integer(
            B_ROW$n_donors
        ),
        as.integer(
            CE_ROW$n_donors
        ),
        as.integer(
            D_ROW$n_donors
        ),
        as.integer(
            F_ROW$n_donors
        ),
        454L
    ),

    Repeated_OOF_donor_AUROC = c(
        as.numeric(
            A_ROW$repeated_OOF_AUROC
        ),
        as.numeric(
            B_ROW$repeated_OOF_AUROC
        ),
        as.numeric(
            CE_ROW$repeated_OOF_AUROC
        ),
        as.numeric(
            D_ROW$repeated_OOF_AUROC
        ),
        as.numeric(
            F_ROW$repeated_OOF_AUROC
        ),
        COV_AUC
    ),

    stringsAsFactors = FALSE
)

table2A$Repeated_OOF_donor_AUROC_display <- sprintf(
    "%.3f",
    table2A$Repeated_OOF_donor_AUROC
)

# ==================================================================================================
# 11. STRICT PAIRED-COMPARISON ANCHORS
# ==================================================================================================

get_comparison_row <- function(
    comparison_code
) {

    idx <- which(
        comparisons$comparison ==
            comparison_code
    )

    if (length(idx) != 1L) {

        stop(
            "Could not uniquely locate paired comparison: ",
            comparison_code
        )
    }

    comparisons[
        idx,
        ,
        drop = FALSE
    ]
}

CE_COMP <- get_comparison_row(
    "F_minus_C_E_pooled_shared_only"
)

D_COMP <- get_comparison_row(
    "F_minus_D_independent_regions"
)

A_COMP <- get_comparison_row(
    "F_DLPFC_minus_A_DLPFC_only"
)

B_COMP <- get_comparison_row(
    "F_HIPPO_minus_B_HIPPO_only"
)

# --------------------------------------------------------------------------------------------------
# Main all-donor anchors
# --------------------------------------------------------------------------------------------------

anchor_check(
    CE_COMP$mean_paired_fold_delta,
    0.00201539,
    "F vs C/E mean paired-fold delta"
)

anchor_check(
    CE_COMP$repeat_block_CI_low,
    0.00103645,
    "F vs C/E repeat-block CI lower"
)

anchor_check(
    CE_COMP$repeat_block_CI_high,
    0.003090892,
    "F vs C/E repeat-block CI upper"
)

anchor_check(
    CE_COMP$averaged_OOF_delta_AUROC,
    0.00318179,
    "F vs C/E averaged-OOF delta"
)

anchor_check(
    CE_COMP$paired_donor_bootstrap_CI_low,
    -0.00108099,
    "F vs C/E donor-bootstrap CI lower"
)

anchor_check(
    CE_COMP$paired_donor_bootstrap_CI_high,
    0.00766893,
    "F vs C/E donor-bootstrap CI upper"
)

anchor_check(
    D_COMP$mean_paired_fold_delta,
    -0.01153163,
    "F vs D mean paired-fold delta"
)

anchor_check(
    D_COMP$repeat_block_CI_low,
    -0.01764658,
    "F vs D repeat-block CI lower"
)

anchor_check(
    D_COMP$repeat_block_CI_high,
    -0.004924369,
    "F vs D repeat-block CI upper"
)

anchor_check(
    D_COMP$averaged_OOF_delta_AUROC,
    -0.01305350,
    "F vs D averaged-OOF delta"
)

anchor_check(
    D_COMP$paired_donor_bootstrap_CI_low,
    -0.03085980,
    "F vs D donor-bootstrap CI lower"
)

anchor_check(
    D_COMP$paired_donor_bootstrap_CI_high,
    0.00528259,
    "F vs D donor-bootstrap CI upper"
)

# --------------------------------------------------------------------------------------------------
# Regional subset anchors
# --------------------------------------------------------------------------------------------------

anchor_check(
    A_COMP$mean_paired_fold_delta,
    -0.00751799,
    "F-DLPFC vs A mean paired-fold delta"
)

anchor_check(
    A_COMP$averaged_OOF_delta_AUROC,
    -0.00480074,
    "F-DLPFC vs A averaged-OOF delta"
)

anchor_check(
    A_COMP$paired_donor_bootstrap_CI_low,
    -0.02984701,
    "F-DLPFC vs A donor-bootstrap CI lower"
)

anchor_check(
    A_COMP$paired_donor_bootstrap_CI_high,
    0.02108494,
    "F-DLPFC vs A donor-bootstrap CI upper"
)

anchor_check(
    B_COMP$mean_paired_fold_delta,
    -0.00480501,
    "F-HIPPO vs B mean paired-fold delta"
)

anchor_check(
    B_COMP$averaged_OOF_delta_AUROC,
    -0.00544571,
    "F-HIPPO vs B averaged-OOF delta"
)

anchor_check(
    B_COMP$paired_donor_bootstrap_CI_low,
    -0.02846777,
    "F-HIPPO vs B donor-bootstrap CI lower"
)

anchor_check(
    B_COMP$paired_donor_bootstrap_CI_high,
    0.01729710,
    "F-HIPPO vs B donor-bootstrap CI upper"
)

cat(
    "All paired-comparison anchors: PASS\n"
)

# ==================================================================================================
# 12. PART B — PAIRED COMPARISON TABLE
# ==================================================================================================

format_delta_ci <- function(
    estimate,
    lower,
    upper,
    digits = 4
) {

    paste0(
        sprintf(
            paste0(
                "%+.",
                digits,
                "f"
            ),
            estimate
        ),
        " [",
        sprintf(
            paste0(
                "%+.",
                digits,
                "f"
            ),
            lower
        ),
        ", ",
        sprintf(
            paste0(
                "%+.",
                digits,
                "f"
            ),
            upper
        ),
        "]"
    )
}

table2B_step05c <- data.frame(

    Comparison = c(
        "F vs C/E shared-only",
        "F vs D independent regions",
        "F-DLPFC vs A DLPFC-only",
        "F-HIPPO vs B HIPPO-only"
    ),

    Population = c(
        as.character(
            CE_COMP$population
        ),
        as.character(
            D_COMP$population
        ),
        as.character(
            A_COMP$population
        ),
        as.character(
            B_COMP$population
        )
    ),

    N_donors = c(
        as.integer(
            CE_COMP$n_donors
        ),
        as.integer(
            D_COMP$n_donors
        ),
        as.integer(
            A_COMP$n_donors
        ),
        as.integer(
            B_COMP$n_donors
        )
    ),

    Reference_AUROC = c(
        as.numeric(
            CE_COMP$averaged_OOF_reference_AUROC
        ),
        as.numeric(
            D_COMP$averaged_OOF_reference_AUROC
        ),
        as.numeric(
            A_COMP$averaged_OOF_reference_AUROC
        ),
        as.numeric(
            B_COMP$averaged_OOF_reference_AUROC
        )
    ),

    Comparator_AUROC = c(
        as.numeric(
            CE_COMP$averaged_OOF_comparator_AUROC
        ),
        as.numeric(
            D_COMP$averaged_OOF_comparator_AUROC
        ),
        as.numeric(
            A_COMP$averaged_OOF_comparator_AUROC
        ),
        as.numeric(
            B_COMP$averaged_OOF_comparator_AUROC
        )
    ),

    Mean_paired_fold_delta = c(
        as.numeric(
            CE_COMP$mean_paired_fold_delta
        ),
        as.numeric(
            D_COMP$mean_paired_fold_delta
        ),
        as.numeric(
            A_COMP$mean_paired_fold_delta
        ),
        as.numeric(
            B_COMP$mean_paired_fold_delta
        )
    ),

    Repeat_block_CI_low = c(
        as.numeric(
            CE_COMP$repeat_block_CI_low
        ),
        as.numeric(
            D_COMP$repeat_block_CI_low
        ),
        as.numeric(
            A_COMP$repeat_block_CI_low
        ),
        as.numeric(
            B_COMP$repeat_block_CI_low
        )
    ),

    Repeat_block_CI_high = c(
        as.numeric(
            CE_COMP$repeat_block_CI_high
        ),
        as.numeric(
            D_COMP$repeat_block_CI_high
        ),
        as.numeric(
            A_COMP$repeat_block_CI_high
        ),
        as.numeric(
            B_COMP$repeat_block_CI_high
        )
    ),

    Averaged_OOF_delta_AUROC = c(
        as.numeric(
            CE_COMP$averaged_OOF_delta_AUROC
        ),
        as.numeric(
            D_COMP$averaged_OOF_delta_AUROC
        ),
        as.numeric(
            A_COMP$averaged_OOF_delta_AUROC
        ),
        as.numeric(
            B_COMP$averaged_OOF_delta_AUROC
        )
    ),

    Donor_bootstrap_CI_low = c(
        as.numeric(
            CE_COMP$paired_donor_bootstrap_CI_low
        ),
        as.numeric(
            D_COMP$paired_donor_bootstrap_CI_low
        ),
        as.numeric(
            A_COMP$paired_donor_bootstrap_CI_low
        ),
        as.numeric(
            B_COMP$paired_donor_bootstrap_CI_low
        )
    ),

    Donor_bootstrap_CI_high = c(
        as.numeric(
            CE_COMP$paired_donor_bootstrap_CI_high
        ),
        as.numeric(
            D_COMP$paired_donor_bootstrap_CI_high
        ),
        as.numeric(
            A_COMP$paired_donor_bootstrap_CI_high
        ),
        as.numeric(
            B_COMP$paired_donor_bootstrap_CI_high
        )
    ),

    stringsAsFactors = FALSE
)

# --------------------------------------------------------------------------------------------------
# Add covariate-only sensitivity comparison
# --------------------------------------------------------------------------------------------------

table2B_covariate <- data.frame(

    Comparison =
        "F vs covariates only",

    Population =
        "All discovery donors",

    N_donors =
        454L,

    Reference_AUROC =
        COV_F_AUC,

    Comparator_AUROC =
        COV_AUC,

    Mean_paired_fold_delta =
        NA_real_,

    Repeat_block_CI_low =
        NA_real_,

    Repeat_block_CI_high =
        NA_real_,

    Averaged_OOF_delta_AUROC =
        COV_DELTA,

    Donor_bootstrap_CI_low =
        COV_CI_LOW,

    Donor_bootstrap_CI_high =
        COV_CI_HIGH,

    stringsAsFactors = FALSE
)

table2B <- rbind(
    table2B_step05c,
    table2B_covariate
)

# ==================================================================================================
# 13. DISPLAY COLUMNS
# ==================================================================================================

table2B$Reference_AUROC_display <- sprintf(
    "%.3f",
    table2B$Reference_AUROC
)

table2B$Comparator_AUROC_display <- sprintf(
    "%.3f",
    table2B$Comparator_AUROC
)

table2B$Paired_fold_delta_display <- ifelse(
    is.finite(
        table2B$Mean_paired_fold_delta
    ),
    format_delta_ci(
        table2B$Mean_paired_fold_delta,
        table2B$Repeat_block_CI_low,
        table2B$Repeat_block_CI_high,
        digits = 4
    ),
    "—"
)

table2B$OOF_delta_display <- format_delta_ci(
    table2B$Averaged_OOF_delta_AUROC,
    table2B$Donor_bootstrap_CI_low,
    table2B$Donor_bootstrap_CI_high,
    digits = 4
)

# ==================================================================================================
# 14. OUTPUT DIRECTORY
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "main",
    "Table2"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

TABLE2A_PATH <- file.path(
    OUT_DIR,
    "Table2A_Model_Performance_v1.csv"
)

TABLE2B_PATH <- file.path(
    OUT_DIR,
    "Table2B_Paired_Model_Comparisons_v1.csv"
)

HTML_PATH <- file.path(
    OUT_DIR,
    "Table2_Leakage_Free_Model_Performance_and_Comparisons_v1.html"
)

NOTES_OUT_PATH <- file.path(
    OUT_DIR,
    "Table2_notes_v1.txt"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "Table2_generation_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Table2_generation_manifest_v1.csv"
)

# ==================================================================================================
# 15. SAVE SOURCE CSVs
# ==================================================================================================

write.csv(
    table2A,
    TABLE2A_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    table2B,
    TABLE2B_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 16. TABLE NOTES
# ==================================================================================================

table_notes <- c(

    "Table 2. Leakage-free model performance and paired comparisons.",

    "",

    "Model A: DLPFC-only elastic-net model.",

    "Model B: hippocampus-only elastic-net model.",

    paste(
        "Models C and E are reported jointly because the implemented",
        "pooled-region and shared-only formulations are mathematically equivalent",
        "under the same covariates, donor-balanced weights, and penalty structure."
    ),

    paste(
        "Model D uses independent leakage-free regional models;",
        "paired donors use the mean of the DLPFC and hippocampal model probabilities,",
        "whereas single-region donors use the available regional prediction."
    ),

    "Model F is the shared-plus-regional-deviation model.",

    "",

    paste(
        "Repeated-OOF AUROC is calculated from averaged leakage-free",
        "out-of-fold donor predictions."
    ),

    paste(
        "DLPFC-only and hippocampus-only models have different eligible donor populations;",
        "therefore their paired comparisons with Model F are restricted to the corresponding",
        "regional donor populations."
    ),

    paste(
        "All-donor comparisons of Model F with C/E, Model D, and the covariate-only",
        "sensitivity use all 454 discovery donors."
    ),

    paste(
        "Mean paired-fold delta AUROC is summarized with a repeat-block bootstrap 95% CI;",
        "the 25 outer folds were not treated as independent observations."
    ),

    paste(
        "Averaged-OOF delta AUROC is summarized with a diagnosis-stratified",
        "paired-donor bootstrap 95% CI."
    ),

    paste(
        "Positive delta AUROC indicates higher discrimination for Model F;",
        "negative delta indicates higher discrimination for the comparator."
    ),

    paste(
        "The covariate-only sensitivity comparison did not use the repeat-block",
        "fold-delta summary, so that cell is shown as an em dash."
    ),

    "No external data were used in these model comparisons.",

    "Frozen SCZ-MSS v1 was not modified."
)

writeLines(
    table_notes,
    NOTES_OUT_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 17. HTML ESCAPE HELPER
# ==================================================================================================

html_escape <- function(x) {

    x <- as.character(x)

    x <- gsub(
        "&",
        "&amp;",
        x,
        fixed = TRUE
    )

    x <- gsub(
        "<",
        "&lt;",
        x,
        fixed = TRUE
    )

    x <- gsub(
        ">",
        "&gt;",
        x,
        fixed = TRUE
    )

    x <- gsub(
        "\"",
        "&quot;",
        x,
        fixed = TRUE
    )

    x
}

# ==================================================================================================
# 18. HTML PART A ROWS
# ==================================================================================================

rows_A <- character(
    nrow(table2A)
)

for (i in seq_len(
    nrow(table2A)
)) {

    row_class <- ""

    if (
        table2A$Model[i] ==
            "F"
    ) {
        row_class <- " class='main-model'"
    }

    if (
        table2A$Model[i] ==
            "D"
    ) {
        row_class <- " class='comparator-model'"
    }

    rows_A[i] <- paste0(

        "<tr",
        row_class,
        ">",

        "<td>",
        html_escape(
            table2A$Model[i]
        ),
        "</td>",

        "<td>",
        html_escape(
            table2A$Specification[i]
        ),
        "</td>",

        "<td>",
        html_escape(
            table2A$Evaluation_population[i]
        ),
        "</td>",

        "<td class='num'>",
        table2A$N_donors[i],
        "</td>",

        "<td class='num'>",
        table2A$Repeated_OOF_donor_AUROC_display[i],
        "</td>",

        "</tr>"
    )
}

# ==================================================================================================
# 19. HTML PART B ROWS
# ==================================================================================================

rows_B <- character(
    nrow(table2B)
)

for (i in seq_len(
    nrow(table2B)
)) {

    rows_B[i] <- paste0(

        "<tr>",

        "<td>",
        html_escape(
            table2B$Comparison[i]
        ),
        "</td>",

        "<td>",
        html_escape(
            table2B$Population[i]
        ),
        "</td>",

        "<td class='num'>",
        table2B$N_donors[i],
        "</td>",

        "<td class='num'>",
        table2B$Reference_AUROC_display[i],
        "</td>",

        "<td class='num'>",
        table2B$Comparator_AUROC_display[i],
        "</td>",

        "<td class='num nowrap'>",
        html_escape(
            table2B$Paired_fold_delta_display[i]
        ),
        "</td>",

        "<td class='num nowrap'>",
        html_escape(
            table2B$OOF_delta_display[i]
        ),
        "</td>",

        "</tr>"
    )
}

# ==================================================================================================
# 20. HTML NOTES
# ==================================================================================================

html_notes <- paste0(
    "<p>",
    html_escape(
        paste(
            table_notes[
                -c(
                    1,
                    2
                )
            ],
            collapse = " "
        )
    ),
    "</p>"
)

# ==================================================================================================
# 21. WRITE TIMES NEW ROMAN HTML PREVIEW
# ==================================================================================================

html <- paste0(

"<!DOCTYPE html>
<html>
<head>
<meta charset='UTF-8'>

<style>

body {
    font-family: 'Times New Roman', Times, serif;
    background: white;
    color: black;
    margin: 28px;
}

.table-title {
    font-size: 11pt;
    font-weight: bold;
    margin-bottom: 10px;
}

.section-title {
    font-size: 9.5pt;
    font-weight: bold;
    margin-top: 12px;
    margin-bottom: 5px;
}

table {
    border-collapse: collapse;
    width: 100%;
    font-size: 9pt;
    margin-bottom: 12px;
}

th {
    font-weight: bold;
    text-align: center;
    vertical-align: bottom;
    border-top: 1.3px solid black;
    border-bottom: 0.9px solid black;
    padding: 5px 6px;
}

td {
    padding: 5px 6px;
    vertical-align: top;
    border-bottom: 0.35px solid #bfbfbf;
}

.num {
    text-align: center;
}

.nowrap {
    white-space: nowrap;
}

.main-model td {
    font-weight: bold;
}

.comparator-model td {
    background: #f7f7f7;
}

.notes {
    font-size: 8pt;
    line-height: 1.25;
    margin-top: 5px;
}

.final-rule {
    border-top: 1.3px solid black;
    margin-top: 4px;
}

</style>
</head>

<body>

<div class='table-title'>
Table 2. Leakage-free model performance and paired comparisons
</div>

<div class='section-title'>
A. Repeated out-of-fold donor-level performance
</div>

<table>

<thead>
<tr>
<th>Model</th>
<th>Specification</th>
<th>Evaluation population</th>
<th>Donors, n</th>
<th>Repeated-OOF donor AUROC</th>
</tr>
</thead>

<tbody>",

paste(
    rows_A,
    collapse = "\n"
),

"</tbody>
</table>

<div class='section-title'>
B. Paired comparisons with Model F
</div>

<table>

<thead>
<tr>
<th>Comparison</th>
<th>Population</th>
<th>Donors, n</th>
<th>Model F AUROC</th>
<th>Comparator AUROC</th>
<th>Mean paired-fold &Delta;AUROC<br>(95% repeat-block CI)</th>
<th>Averaged-OOF &Delta;AUROC<br>(95% paired-donor bootstrap CI)</th>
</tr>
</thead>

<tbody>",

paste(
    rows_B,
    collapse = "\n"
),

"</tbody>
</table>

<div class='final-rule'></div>

<div class='notes'>",
html_notes,
"</div>

</body>
</html>"
)

writeLines(
    html,
    HTML_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 22. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    TABLE2A_PATH,
    TABLE2B_PATH,
    HTML_PATH,
    NOTES_OUT_PATH
)

if (!all(
    file.exists(
        outputs
    )
)) {

    stop(
        "One or more Table 2 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {

    stop(
        "One or more Table 2 outputs are zero-byte."
    )
}

cat(
    "Table 2 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 23. GENERATION SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11C.7 — MAIN TABLE 2",
    "==============================================================================================================",

    "",

    "Frozen saved model-comparison artifacts only.",

    "No model fitting, CV rerun, bootstrap rerun, or score recalculation.",

    "",

    "PART A — REPEATED OOF DONOR AUROC",

    paste0(
        "Model A = ",
        sprintf(
            "%.7f",
            A_ROW$repeated_OOF_AUROC
        ),
        " | n=",
        A_ROW$n_donors
    ),

    paste0(
        "Model B = ",
        sprintf(
            "%.7f",
            B_ROW$repeated_OOF_AUROC
        ),
        " | n=",
        B_ROW$n_donors
    ),

    paste0(
        "Model C/E = ",
        sprintf(
            "%.7f",
            CE_ROW$repeated_OOF_AUROC
        ),
        " | n=",
        CE_ROW$n_donors
    ),

    paste0(
        "Model D = ",
        sprintf(
            "%.8f",
            D_ROW$repeated_OOF_AUROC
        ),
        " | n=",
        D_ROW$n_donors
    ),

    paste0(
        "Model F = ",
        sprintf(
            "%.7f",
            F_ROW$repeated_OOF_AUROC
        ),
        " | n=",
        F_ROW$n_donors
    ),

    paste0(
        "Covariate-only = ",
        sprintf(
            "%.7f",
            COV_AUC
        ),
        " | n=454"
    ),

    "",

    "PART B — PAIRED COMPARISONS",

    paste0(
        "F vs C/E averaged-OOF delta = ",
        sprintf(
            "%+.8f",
            CE_COMP$averaged_OOF_delta_AUROC
        ),
        " [",
        sprintf(
            "%+.8f",
            CE_COMP$paired_donor_bootstrap_CI_low
        ),
        ", ",
        sprintf(
            "%+.8f",
            CE_COMP$paired_donor_bootstrap_CI_high
        ),
        "]"
    ),

    paste0(
        "F vs D averaged-OOF delta = ",
        sprintf(
            "%+.8f",
            D_COMP$averaged_OOF_delta_AUROC
        ),
        " [",
        sprintf(
            "%+.8f",
            D_COMP$paired_donor_bootstrap_CI_low
        ),
        ", ",
        sprintf(
            "%+.8f",
            D_COMP$paired_donor_bootstrap_CI_high
        ),
        "]"
    ),

    paste0(
        "F-DLPFC vs A averaged-OOF delta = ",
        sprintf(
            "%+.8f",
            A_COMP$averaged_OOF_delta_AUROC
        ),
        " [",
        sprintf(
            "%+.8f",
            A_COMP$paired_donor_bootstrap_CI_low
        ),
        ", ",
        sprintf(
            "%+.8f",
            A_COMP$paired_donor_bootstrap_CI_high
        ),
        "]"
    ),

    paste0(
        "F-HIPPO vs B averaged-OOF delta = ",
        sprintf(
            "%+.8f",
            B_COMP$averaged_OOF_delta_AUROC
        ),
        " [",
        sprintf(
            "%+.8f",
            B_COMP$paired_donor_bootstrap_CI_low
        ),
        ", ",
        sprintf(
            "%+.8f",
            B_COMP$paired_donor_bootstrap_CI_high
        ),
        "]"
    ),

    paste0(
        "F vs covariates averaged-OOF delta = ",
        sprintf(
            "%+.8f",
            COV_DELTA
        ),
        " [",
        sprintf(
            "%+.8f",
            COV_CI_LOW
        ),
        ", ",
        sprintf(
            "%+.8f",
            COV_CI_HIGH
        ),
        "]"
    ),

    "",

    paste0(
        "Table 2A CSV = ",
        TABLE2A_PATH
    ),

    paste0(
        "Table 2B CSV = ",
        TABLE2B_PATH
    ),

    paste0(
        "Times New Roman HTML = ",
        HTML_PATH
    ),

    paste0(
        "Notes = ",
        NOTES_OUT_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 24. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 25. CONSOLE PREVIEW
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nMAIN TABLE 2 — PART A PREVIEW\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    table2A[
        ,
        c(
            "Model",
            "Specification",
            "Evaluation_population",
            "N_donors",
            "Repeated_OOF_donor_AUROC_display"
        ),
        drop = FALSE
    ],
    row.names = FALSE
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nMAIN TABLE 2 — PART B PREVIEW\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

print(
    data.frame(

        Comparison =
            table2B$Comparison,

        Population =
            table2B$Population,

        N_donors =
            table2B$N_donors,

        Model_F_AUROC =
            table2B$Reference_AUROC_display,

        Comparator_AUROC =
            table2B$Comparator_AUROC_display,

        Paired_fold_delta_95CI =
            table2B$Paired_fold_delta_display,

        Averaged_OOF_delta_95CI =
            table2B$OOF_delta_display,

        stringsAsFactors = FALSE
    ),

    row.names = FALSE
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nTABLE 2 OUTPUTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Table 2A CSV:\n",
    TABLE2A_PATH,
    "\n\n",
    sep = ""
)

cat(
    "Table 2B CSV:\n",
    TABLE2B_PATH,
    "\n\n",
    sep = ""
)

cat(
    "Times New Roman HTML preview:\n",
    HTML_PATH,
    "\n\n",
    sep = ""
)

cat(
    "Notes:\n",
    NOTES_OUT_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11C.7 COMPLETE — MAIN TABLE 2 GENERATED FROM FROZEN LEAKAGE-FREE MODEL OUTPUTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11C.7 — MAIN TABLE 2
LEAKAGE-FREE MODEL PERFORMANCE AND PAIRED COMPARISONS
Project sentinel: PASS
Required package: PASS
Required frozen model-comparison artifacts: PASS
Input schemas: PASS
Frozen five-model + four-comparison structure: PASS
Covariate-only sensitivity anchors: PASS
Five-model repeated-OOF anchors: PASS
All paired-comparison anchors: PASS
Table 2 outputs created and non-zero: PASS

MAIN TABLE 2 — PART A PREVIEW
       Model                      Specification Evaluation_population N_donors
           A             DLPFC-only elastic net          DLPFC donors      379
           B             HIPPO-only elastic net          HIPPO donors      376
         C/E     Pooled/shared-only elastic net  All discovery donors      454
           D Independent region-specific models  All discovery donors      454
           F  Shared + regional-deviation model  All discovery donors      454
 Sensitivity                    Covariates only  All discovery donors      454
 Repeated_

In [104]:
# ==================================================================================================
# STEP 11C.8 — MAIN TABLE 3
# PUBLIC EXTERNAL-COHORT TECHNICAL EVALUABILITY AND FROZEN VALIDATION SUMMARY
# ==================================================================================================

cat(
    "\n",
    paste(rep("=",110),collapse=""),
    "\nSTEP 11C.8 — MAIN TABLE 3\n",
    "EXTERNAL TECHNICAL EVALUABILITY AND FROZEN VALIDATION SUMMARY\n",
    paste(rep("=",110),collapse=""),
    "\n",
    sep=""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

sentinel <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(sentinel)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        sentinel,
        n=1,
        warn=FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {
    stop("Project sentinel mismatch.")
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "digest",
    quietly=TRUE
)) {
    stop("digest unavailable.")
}

cat("Required package: PASS\n")

# ==================================================================================================
# 3. INPUT PATHS
# ==================================================================================================

MASTER_DIR <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "master_registry_v1"
)

GSE_DIR <- file.path(
    PROJECT_ROOT,
    "09_external_validation",
    "gse224683"
)

CLAIM_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "claim_registry_v1"
)

REGISTRY_PATH <- file.path(
    MASTER_DIR,
    "SCZ_MSS_v1_external_technical_gate_registry_v1.csv"
)

FINAL_TABLE_PATH <- file.path(
    MASTER_DIR,
    "SCZ_MSS_v1_FINAL_external_validation_summary_table_v1.csv"
)

SCORE_PATH <- file.path(
    GSE_DIR,
    "STEP09D_GSE224683_sample_SCZ_MSS_v1.csv"
)

STAT_PATH <- file.path(
    "E:/SCZ_CROSSREGION_MSS_Q1",
    "11_manuscript",
    "figures",
    "main",
    "Figure5_final_TNR_v3",
    "source_data",
    "Figure5_GSE224683_frozen_statistics_source_v3.csv"
)

required <- c(
    REGISTRY_PATH,
    FINAL_TABLE_PATH,
    SCORE_PATH,
    STAT_PATH
)

missing <- required[
    !file.exists(required)
]

if (length(missing)>0) {
    stop(
        "Missing frozen external artifact:\n",
        paste(missing,collapse="\n")
    )
}

cat("Frozen external artifacts: PASS\n")

# ==================================================================================================
# 4. LOAD
# ==================================================================================================

registry <- read.csv(
    REGISTRY_PATH,
    stringsAsFactors=FALSE,
    check.names=FALSE
)

final_table <- read.csv(
    FINAL_TABLE_PATH,
    stringsAsFactors=FALSE,
    check.names=FALSE
)

scores <- read.csv(
    SCORE_PATH,
    stringsAsFactors=FALSE,
    check.names=FALSE
)

stats5 <- read.csv(
    STAT_PATH,
    stringsAsFactors=FALSE,
    check.names=FALSE
)

# ==================================================================================================
# 5. VALIDATE FIVE COHORTS
# ==================================================================================================

expected <- c(
    "GSE87194",
    "GSE80655",
    "GSE224683",
    "GSE282552",
    "GSE208338"
)

if (
    nrow(registry)!=5 ||
    !setequal(
        registry$dataset,
        expected
    )
) {
    stop("External cohort set changed.")
}

cat(
    "Five public cohorts: PASS\n"
)

# ==================================================================================================
# 6. TABLE 3A
# ==================================================================================================

table3A <- data.frame(

    Dataset =
        registry$dataset,

    Samples_n =
        registry$n_samples,

    Mapped_features =
        registry$mapped_gene_count,

    Gene_coverage =
        registry$gene_count_coverage,

    Absolute_weight_coverage =
        registry$absolute_weight_coverage,

    Positive_weight_coverage =
        registry$positive_weight_coverage,

    Negative_weight_coverage =
        registry$negative_weight_coverage,

    Technical_status =
        ifelse(
            registry$technical_pass,
            "ELIGIBLE",
            "NON-EVALUABLE"
        ),

    Outcome_evaluated =
        ifelse(
            registry$outcome_evaluated,
            "YES",
            "NO"
        ),

    stringsAsFactors=FALSE
)

# ==================================================================================================
# 7. GSE224683 STAT HELPER
# ==================================================================================================

get_stat <- function(
    name
){

    idx <- which(
        stats5$metric==name
    )

    if(length(idx)!=1){
        stop(
            "Metric missing or duplicated: ",
            name
        )
    }

    as.numeric(
        stats5$value[idx]
    )
}

# ==================================================================================================
# 8. GSE224683 TABLE 3B
# ==================================================================================================

gse_score <- scores$SCZ_MSS

control <- scores$SCZ_MSS[
    scores$y==0
]

scz <- scores$SCZ_MSS[
    scores$y==1
]


table3B <- data.frame(

    Cohort =
        "GSE224683",

    Samples =
        nrow(scores),

    Control_n =
        sum(scores$y==0),

    Schizophrenia_n =
        sum(scores$y==1),

    Frozen_features_mapped =
        23,

    Retained_absolute_weight =
        0.9266,

    Control_mean_SD =
        sprintf(
            "%.6f ± %.6f",
            mean(control),
            sd(control)
        ),

    Schizophrenia_mean_SD =
        sprintf(
            "%.6f ± %.6f",
            mean(scz),
            sd(scz)
        ),

    Mean_difference_CI =
        sprintf(
            "%+.6f [%+.6f, %+.6f]",
            get_stat("Mean_difference"),
            get_stat("Mean_difference_CI_low"),
            get_stat("Mean_difference_CI_high")
        ),

    Hedges_g_CI =
        sprintf(
            "%+.4f [%+.4f, %+.4f]",
            get_stat("Hedges_g"),
            get_stat("Hedges_g_CI_low"),
            get_stat("Hedges_g_CI_high")
        ),

    AUROC_CI =
        sprintf(
            "%.4f [%.4f, %.4f]",
            get_stat("AUROC"),
            get_stat("AUROC_CI_low"),
            get_stat("AUROC_CI_high")
        ),

    Welch_p =
        get_stat("Welch_p"),

    stringsAsFactors=FALSE
)

# ==================================================================================================
# 9. OUTPUT
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "main",
    "Table3"
)

dir.create(
    OUT_DIR,
    recursive=TRUE,
    showWarnings=FALSE
)

TABLE3A_PATH <- file.path(
    OUT_DIR,
    "Table3A_External_Technical_Evaluability_v1.csv"
)

TABLE3B_PATH <- file.path(
    OUT_DIR,
    "Table3B_GSE224683_External_Validation_v1.csv"
)

NOTES_PATH <- file.path(
    OUT_DIR,
    "Table3_notes_v1.txt"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "Table3_generation_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "Table3_generation_manifest_v1.csv"
)

write.csv(
    table3A,
    TABLE3A_PATH,
    row.names=FALSE
)

write.csv(
    table3B,
    TABLE3B_PATH,
    row.names=FALSE
)

notes <- c(

"Table 3 summarizes public external validation.",
"",
"Technical non-evaluable cohorts failed predefined frozen technical coverage requirements.",
"They were not interpreted as biological replication failures.",
"",
"GSE224683 was the only technically eligible cohort.",
"",
"The external SCZ-MSS outcome was evaluated without model retraining or recalibration.",
"",
"The null external result is retained as a prespecified validation outcome."
)

writeLines(
    notes,
    NOTES_PATH
)

# ==================================================================================================
# 10. SUMMARY
# ==================================================================================================

summary <- c(

"==============================================================================================================",
"STEP 11C.8 COMPLETE — MAIN TABLE 3",
"==============================================================================================================",

paste0(
"Public cohorts screened = ",
nrow(table3A)
),

paste0(
"Technically eligible = ",
sum(
    table3A$Technical_status=="ELIGIBLE"
)
),

paste0(
"Outcome evaluated = ",
sum(
    table3A$Outcome_evaluated=="YES"
)
),

paste0(
"Valid external cohort = GSE224683"
),

"",
"Table3A:",
TABLE3A_PATH,

"Table3B:",
TABLE3B_PATH,

"Notes:",
NOTES_PATH,

"=============================================================================================================="
)

writeLines(
    summary,
    SUMMARY_PATH
)

# ==================================================================================================
# 11. MANIFEST
# ==================================================================================================

files <- c(
    required,
    TABLE3A_PATH,
    TABLE3B_PATH,
    NOTES_PATH,
    SUMMARY_PATH
)

manifest <- data.frame(

file=basename(files),

path=files,

bytes=file.info(files)$size,

sha256=vapply(
    files,
    function(x)
        digest::digest(
            file=x,
            algo="sha256",
            serialize=FALSE
        ),
    character(1)
),

stringsAsFactors=FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names=FALSE
)

cat(
"\n",
paste(rep("=",110),collapse=""),
"\nMAIN TABLE 3 PREVIEW\n",
paste(rep("=",110),collapse=""),
"\n",
sep=""
)

print(
    table3A,
    row.names=FALSE
)

print(
    table3B,
    row.names=FALSE
)

cat(
"\nSTEP 11C.8 COMPLETE — MAIN TABLE 3 GENERATED\n"
)


STEP 11C.8 — MAIN TABLE 3
EXTERNAL TECHNICAL EVALUABILITY AND FROZEN VALIDATION SUMMARY
Project sentinel: PASS
Required package: PASS
Frozen external artifacts: PASS
Five public cohorts: PASS

MAIN TABLE 3 PREVIEW
   Dataset Samples_n Mapped_features Gene_coverage Absolute_weight_coverage
  GSE87194        38              14     0.5833330                0.3332160
  GSE80655       281              20     0.8333330                0.7197100
 GSE224683        30              23     0.9583330                0.9266000
 GSE282552        18               9     0.3750000                0.1961925
 GSE208338       169              14     0.5833333                0.3818154
 Positive_weight_coverage Negative_weight_coverage Technical_status
                0.2611400                0.3787040    NON-EVALUABLE
                1.0000000                0.5428170    NON-EVALUABLE
                1.0000000                0.8802760         ELIGIBLE
                0.0304038                0.3008226    NON

In [4]:
# ==================================================================================================
# STEP 11D.1S — SUPPLEMENTARY FIGURE S1 — FINAL VISUAL VERSION
# EXTENDED NORMALIZATION AND PCA-QC DIAGNOSTICS
# --------------------------------------------------------------------------------------------------
# VISUAL-ONLY REVISION OF VALIDATED S1 v2
#
# FIXES
#   1. Shorter Panel A title to eliminate clipping.
#   2. Panel A legends stacked vertically so "Schizophrenia" is fully visible.
#   3. Panel D threshold annotation moved into a compact two-line label
#      away from R2809 and the other flagged samples.
#   4. Retain the four manually separated PCA-audit sample labels.
#
# NO ANALYTICAL RERUN:
#   - no TMM normalization
#   - no PCA
#   - no QC refitting
#   - no sample exclusion
#
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.1S — SUPPLEMENTARY FIGURE S1 — FINAL VISUAL VERSION\n",
    "EXTENDED NORMALIZATION AND PCA-QC DIAGNOSTICS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        project_id
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. VALIDATED S1 v2 SOURCE DATA
# ==================================================================================================

V2_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS1_QC_TNR_v2"
)

V2_SOURCE_DIR <- file.path(
    V2_DIR,
    "source_data"
)

SAMPLE_SOURCE_PATH <- file.path(
    V2_SOURCE_DIR,
    "FigureS1_sample_QC_source_v2.csv"
)

PCA_SOURCE_PATH <- file.path(
    V2_SOURCE_DIR,
    "FigureS1_PC1_PC10_variance_source_v2.csv"
)

EXTREME_SOURCE_PATH <- file.path(
    V2_SOURCE_DIR,
    "FigureS1_PCA_extreme_audit_source_v2.csv"
)

required_inputs <- c(
    SAMPLE_SOURCE_PATH,
    PCA_SOURCE_PATH,
    EXTREME_SOURCE_PATH
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required validated S1 v2 source-data file(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat(
    "Validated S1 v2 source-data files: PASS\n"
)

# ==================================================================================================
# 5. LOAD
# ==================================================================================================

sample_qc <- read.csv(
    SAMPLE_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

pca10 <- read.csv(
    PCA_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

extreme_saved <- read.csv(
    EXTREME_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 6. FLAG NORMALIZATION
# ==================================================================================================

normalize_logical <- function(x) {

    if (is.logical(x)) {
        return(x)
    }

    z <- toupper(
        trimws(
            as.character(x)
        )
    )

    out <- rep(
        NA,
        length(z)
    )

    out[
        z %in%
            c(
                "TRUE",
                "T",
                "1",
                "YES",
                "Y"
            )
    ] <- TRUE

    out[
        z %in%
            c(
                "FALSE",
                "F",
                "0",
                "NO",
                "N"
            )
    ] <- FALSE

    if (anyNA(out)) {

        stop(
            "Unexpected PCA flag encoding: ",
            paste(
                unique(
                    z[
                        is.na(out)
                    ]
                ),
                collapse = ", "
            )
        )
    }

    out
}

sample_qc$pca_extreme_flag <- normalize_logical(
    sample_qc$pca_extreme_flag
)

# ==================================================================================================
# 7. BASIC FROZEN VALIDATION
# ==================================================================================================

if (nrow(sample_qc) != 755L) {
    stop("Expected 755 frozen discovery samples.")
}

if (
    length(
        unique(
            sample_qc$BrNum
        )
    ) != 454L
) {
    stop("Expected 454 frozen discovery donors.")
}

if (
    sum(
        sample_qc$pca_extreme_flag
    ) != 4L
) {
    stop("Expected exactly four frozen PCA audit flags.")
}

EXPECTED_EXTREME_IDS <- c(
    "R10424",
    "R2809",
    "R5027",
    "R2538"
)

if (
    !setequal(
        as.character(
            sample_qc$RNum[
                sample_qc$pca_extreme_flag
            ]
        ),
        EXPECTED_EXTREME_IDS
    )
) {
    stop(
        "Frozen PCA-audit sample IDs changed."
    )
}

cat(
    "Frozen samples / donors / audit flags: 755 / 454 / 4 — PASS\n"
)

# ==================================================================================================
# 8. DIAGNOSIS DISPLAY
# ==================================================================================================

dx_raw <- trimws(
    as.character(
        sample_qc$Dx
    )
)

sample_qc$Dx_display <- ifelse(
    grepl(
        "^sch",
        dx_raw,
        ignore.case = TRUE
    ),
    "Schizophrenia",
    "Control"
)

sample_qc$Dx_display <- factor(
    sample_qc$Dx_display,
    levels = c(
        "Control",
        "Schizophrenia"
    )
)

sample_qc$Region <- factor(
    as.character(
        sample_qc$Region
    ),
    levels = c(
        "DLPFC",
        "HIPPO"
    )
)

sample_qc$group4 <- factor(
    paste(
        sample_qc$Region,
        ifelse(
            sample_qc$Dx_display ==
                "Schizophrenia",
            "SCZ",
            "Control"
        ),
        sep = " "
    ),
    levels = c(
        "DLPFC Control",
        "DLPFC SCZ",
        "HIPPO Control",
        "HIPPO SCZ"
    )
)

sample_qc$x_group <- match(
    as.character(
        sample_qc$group4
    ),
    levels(
        sample_qc$group4
    )
)

# ==================================================================================================
# 9. PLOTTING NUMBERS
# ==================================================================================================

sample_qc$raw_library_millions <- (
    sample_qc$raw_library_size /
        1e6
)

sample_qc$effective_library_millions <- (
    sample_qc$effective_library_size /
        1e6
)

if (!"PC_number" %in% names(pca10)) {

    pca10$PC_number <- suppressWarnings(
        as.integer(
            sub(
                "^PC",
                "",
                pca10$PC
            )
        )
    )
}

pca10 <- pca10[
    order(
        pca10$PC_number
    ),
    ,
    drop = FALSE
]

if (
    nrow(pca10) != 10L ||
    !identical(
        as.integer(
            pca10$PC_number
        ),
        1:10
    )
) {
    stop("Frozen PC1-PC10 structure changed.")
}

if (
    abs(
        pca10$variance_explained[1] -
        0.20087937
    ) >
        5e-8 ||
    abs(
        pca10$variance_explained[2] -
        0.06114427
    ) >
        5e-8 ||
    abs(
        pca10$cumulative_variance[10] -
        0.4312099
    ) >
        5e-7
) {
    stop("Frozen PCA variance anchors changed.")
}

ROBUST_THRESHOLD <- unname(
    stats::quantile(
        sample_qc$pca10_robust_distance,
        probs = 0.995,
        na.rm = TRUE,
        type = 7
    )
)

if (!identical(
    as.logical(
        sample_qc$pca10_robust_distance >
            ROBUST_THRESHOLD
    ),
    as.logical(
        sample_qc$pca_extreme_flag
    )
)) {
    stop(
        "Frozen PCA audit flags disagree with 99.5th-percentile rule."
    )
}

cat(
    "Frozen PCA numerical anchors: PASS\n"
)

# ==================================================================================================
# 10. FOUR FLAGGED SAMPLES + CLEAN LABEL POSITIONS
# ==================================================================================================

extreme_plot <- sample_qc[
    sample_qc$pca_extreme_flag,
    ,
    drop = FALSE
]

extreme_plot$label_x <- extreme_plot$x_group
extreme_plot$label_y <- extreme_plot$pca10_robust_distance

# R10424
idx <- which(
    extreme_plot$RNum ==
        "R10424"
)

extreme_plot$label_x[idx] <- 0.88
extreme_plot$label_y[idx] <- (
    extreme_plot$pca10_robust_distance[idx] -
        0.43
)

# R2809
idx <- which(
    extreme_plot$RNum ==
        "R2809"
)

extreme_plot$label_x[idx] <- 2.14
extreme_plot$label_y[idx] <- (
    extreme_plot$pca10_robust_distance[idx] +
        0.40
)

# R5027
idx <- which(
    extreme_plot$RNum ==
        "R5027"
)

extreme_plot$label_x[idx] <- 3.79
extreme_plot$label_y[idx] <- (
    extreme_plot$pca10_robust_distance[idx] +
        0.38
)

# R2538
idx <- which(
    extreme_plot$RNum ==
        "R2538"
)

extreme_plot$label_x[idx] <- 4.18
extreme_plot$label_y[idx] <- (
    extreme_plot$pca10_robust_distance[idx] -
        0.48
)

# ==================================================================================================
# 11. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS1_QC_TNR_v3"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "FigureS1_Extended_Normalization_PCA_QC_TimesNewRoman_v3.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "FigureS1_Extended_Normalization_PCA_QC_TimesNewRoman_v3.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "FigureS1_Extended_Normalization_PCA_QC_TimesNewRoman_v3.png"
)

SAMPLE_OUT_PATH <- file.path(
    SOURCE_DIR,
    "FigureS1_sample_QC_source_v3.csv"
)

PCA_OUT_PATH <- file.path(
    SOURCE_DIR,
    "FigureS1_PC1_PC10_variance_source_v3.csv"
)

EXTREME_OUT_PATH <- file.path(
    SOURCE_DIR,
    "FigureS1_PCA_extreme_audit_source_v3.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "FigureS1_generation_summary_v3.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "FigureS1_generation_manifest_v3.csv"
)

write.csv(
    sample_qc,
    SAMPLE_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    pca10,
    PCA_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    extreme_plot,
    EXTREME_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 12. THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.2,
            hjust = 0,
            margin = ggplot2::margin(
                b = 5
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.5
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.2,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.8
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.4
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            8,
            9,
            8,
            9
        )
    )

# ==================================================================================================
# 13. PANEL A — SHORT TITLE + STACKED LEGENDS
# ==================================================================================================

region_colours <- c(
    "DLPFC" = "#4E79A7",
    "HIPPO" = "#F28E2B"
)

diagnosis_shapes <- c(
    "Control" = 16,
    "Schizophrenia" = 17
)

pA <- ggplot2::ggplot(
    sample_qc,
    ggplot2::aes(
        x = raw_library_millions,
        y = effective_library_millions,
        colour = Region,
        shape = Dx_display
    )
) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#555555"
    ) +
    ggplot2::geom_point(
        size = 1.20,
        alpha = 0.58
    ) +
    ggplot2::scale_colour_manual(
        values = region_colours
    ) +
    ggplot2::scale_shape_manual(
        values = diagnosis_shapes
    ) +
    ggplot2::coord_equal() +
    ggplot2::labs(
        title = "A  Raw versus effective library size",
        x = "Raw library size (millions)",
        y = "TMM-effective library size (millions)",
        colour = "Region",
        shape = "Diagnosis"
    ) +
    ggplot2::guides(

        colour = ggplot2::guide_legend(
            order = 1,
            nrow = 1
        ),

        shape = ggplot2::guide_legend(
            order = 2,
            nrow = 1
        )
    ) +
    theme_publication +
    ggplot2::theme(

        legend.position = "bottom",

        legend.box = "vertical",

        legend.box.just = "left",

        legend.justification = "left",

        legend.spacing.y = grid::unit(
            -0.02,
            "cm"
        ),

        legend.margin = ggplot2::margin(
            t = 0,
            r = 0,
            b = 0,
            l = 0
        )
    )

# ==================================================================================================
# 14. PANEL B
# ==================================================================================================

group_colours <- c(
    "DLPFC Control" = "#4E79A7",
    "DLPFC SCZ" = "#A0CBE8",
    "HIPPO Control" = "#F28E2B",
    "HIPPO SCZ" = "#FFBE7D"
)

pB <- ggplot2::ggplot(
    sample_qc,
    ggplot2::aes(
        x = group4,
        y = TMM_norm_factor,
        fill = group4
    )
) +
    ggplot2::geom_hline(
        yintercept = 1,
        linetype = "dashed",
        linewidth = 0.45,
        colour = "#555555"
    ) +
    ggplot2::geom_violin(
        width = 0.88,
        scale = "width",
        trim = TRUE,
        alpha = 0.55,
        colour = NA
    ) +
    ggplot2::geom_boxplot(
        width = 0.24,
        outlier.shape = NA,
        linewidth = 0.52,
        fill = "white",
        colour = "black"
    ) +
    ggplot2::scale_fill_manual(
        values = group_colours,
        guide = "none"
    ) +
    ggplot2::scale_x_discrete(
        labels = c(
            "DLPFC Control" = "DLPFC\nControl",
            "DLPFC SCZ" = "DLPFC\nSchizophrenia",
            "HIPPO Control" = "HIPPO\nControl",
            "HIPPO SCZ" = "HIPPO\nSchizophrenia"
        )
    ) +
    ggplot2::labs(
        title = "B  TMM normalization-factor distribution",
        x = NULL,
        y = "TMM normalization factor"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.35,
            lineheight = 0.88,
            hjust = 0.5
        )
    )

# ==================================================================================================
# 15. PANEL C
# ==================================================================================================

pC <- ggplot2::ggplot(
    pca10,
    ggplot2::aes(
        x = PC_number
    )
) +
    ggplot2::geom_col(
        ggplot2::aes(
            y = variance_explained
        ),
        width = 0.72,
        fill = "#8F63A8"
    ) +
    ggplot2::geom_line(
        ggplot2::aes(
            y = cumulative_variance
        ),
        linewidth = 0.75,
        colour = "#303030"
    ) +
    ggplot2::geom_point(
        ggplot2::aes(
            y = cumulative_variance
        ),
        size = 1.8,
        colour = "#303030"
    ) +
    ggplot2::annotate(
        "text",
        x = 9.95,
        y = pca10$cumulative_variance[10] + 0.022,
        label = paste0(
            "Cumulative PC1-PC10 = ",
            sprintf(
                "%.1f%%",
                100 *
                    pca10$cumulative_variance[10]
            )
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35,
        hjust = 1
    ) +
    ggplot2::scale_x_continuous(
        breaks = 1:10
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            0,
            0.475
        ),
        breaks = seq(
            0,
            0.4,
            by = 0.1
        ),
        labels = function(x) {
            paste0(
                round(
                    100 * x
                ),
                "%"
            )
        }
    ) +
    ggplot2::labs(
        title = "C  Variance explained by the first 10 PCs",
        x = "Principal component",
        y = "Variance proportion"
    ) +
    theme_publication

# ==================================================================================================
# 16. PANEL D — THRESHOLD LABEL MOVED AWAY FROM FLAGGED SAMPLES
# ==================================================================================================

pD <- ggplot2::ggplot(
    sample_qc,
    ggplot2::aes(
        x = x_group,
        y = pca10_robust_distance
    )
) +
    ggplot2::geom_hline(
        yintercept = ROBUST_THRESHOLD,
        linetype = "dashed",
        linewidth = 0.65,
        colour = "#C44E52"
    ) +
    ggplot2::geom_point(
        data = sample_qc[
            !sample_qc$pca_extreme_flag,
            ,
            drop = FALSE
        ],
        position = ggplot2::position_jitter(
            width = 0.15,
            height = 0,
            seed = 20261001
        ),
        size = 1.02,
        alpha = 0.38,
        colour = "#777777"
    ) +
    ggplot2::geom_point(
        data = extreme_plot,
        ggplot2::aes(
            x = x_group,
            y = pca10_robust_distance
        ),
        inherit.aes = FALSE,
        size = 2.6,
        shape = 21,
        stroke = 0.70,
        fill = "#E15759",
        colour = "black"
    ) +
    ggplot2::geom_segment(
        data = extreme_plot,
        ggplot2::aes(
            x = x_group,
            y = pca10_robust_distance,
            xend = label_x,
            yend = label_y
        ),
        inherit.aes = FALSE,
        linewidth = 0.25,
        colour = "#555555"
    ) +
    ggplot2::geom_text(
        data = extreme_plot,
        ggplot2::aes(
            x = label_x,
            y = label_y,
            label = RNum
        ),
        inherit.aes = FALSE,
        family = FONT_FAMILY,
        size = 2.20,
        colour = "black"
    ) +
    ggplot2::annotate(
        "label",
        x = 0.62,
        y = ROBUST_THRESHOLD + 0.05,
        label = "99.5th-percentile\nthreshold",
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.05,
        hjust = 0,
        vjust = 0.5,
        colour = "#B23B3F",
        fill = "white",
        label.size = 0,
        label.padding = grid::unit(
            0.08,
            "lines"
        )
    ) +
    ggplot2::annotate(
        "text",
        x = 4.43,
        y = max(
            sample_qc$pca10_robust_distance
        ) +
            0.25,
        label = "4 audit flags; all retained",
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35,
        hjust = 1
    ) +
    ggplot2::scale_x_continuous(
        breaks = 1:4,
        labels = c(
            "DLPFC\nControl",
            "DLPFC\nSchizophrenia",
            "HIPPO\nControl",
            "HIPPO\nSchizophrenia"
        ),
        limits = c(
            0.55,
            4.48
        )
    ) +
    ggplot2::scale_y_continuous(
        expand = ggplot2::expansion(
            mult = c(
                0.04,
                0.10
            )
        )
    ) +
    ggplot2::labs(
        title = "D  Robust PCA-distance audit",
        x = NULL,
        y = "10-PC robust distance"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.25,
            lineheight = 0.88,
            hjust = 0.5
        )
    )

# ==================================================================================================
# 17. DRAW FIGURE
# ==================================================================================================

draw_figure_s1 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 2,
        ncol = 2,
        widths = grid::unit(
            c(
                1,
                1
            ),
            "null"
        ),
        heights = grid::unit(
            c(
                1,
                1
            ),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pD,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 18. SAVE
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 5.9,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure_s1()

grDevices::dev.off()

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 5.9,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure_s1()

grDevices::dev.off()

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 5.9,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure_s1()

grDevices::dev.off()

# ==================================================================================================
# 19. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    SAMPLE_OUT_PATH,
    PCA_OUT_PATH,
    EXTREME_OUT_PATH
)

if (!all(
    file.exists(outputs)
)) {
    stop(
        "One or more S1 v3 outputs were not created."
    )
}

if (any(
    file.info(outputs)$size <= 0
)) {
    stop(
        "One or more S1 v3 outputs are zero-byte."
    )
}

cat(
    "Supplementary Figure S1 v3 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 20. SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11D.1S — SUPPLEMENTARY FIGURE S1 — FINAL VISUAL VERSION",
    "==============================================================================================================",

    "",

    "Generated from validated S1 v2 frozen source data.",

    "No analytical operation rerun.",

    "",

    "Samples = 755",
    "Donors = 454",
    "PCA audit flags = 4",
    "PCA audit exclusions = 0",

    paste0(
        "PC1 variance = ",
        sprintf(
            "%.2f%%",
            100 *
                pca10$variance_explained[1]
        )
    ),

    paste0(
        "PC2 variance = ",
        sprintf(
            "%.2f%%",
            100 *
                pca10$variance_explained[2]
        )
    ),

    paste0(
        "PC1-PC10 cumulative variance = ",
        sprintf(
            "%.2f%%",
            100 *
                pca10$cumulative_variance[10]
        )
    ),

    "",

    "Visual corrections:",
    "Panel A title shortened.",
    "Region and diagnosis legends stacked vertically.",
    "Full Schizophrenia legend label retained.",
    "Panel D threshold annotation moved away from flagged sample labels.",

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 21. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {
            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 22. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSUPPLEMENTARY FIGURE S1 v3 SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat("Validated S1 v2 source data: PASS\n")
cat("Analytical results changed: NO\n")
cat("Panel A title clipping corrected: YES\n")
cat("Full Schizophrenia legend label retained: YES\n")
cat("Panel D threshold-label collision corrected: YES\n")
cat("Samples / donors: 755 / 454\n")
cat("PCA audit flags / exclusions: 4 / 0\n")

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.1S COMPLETE — SUPPLEMENTARY FIGURE S1 v3 GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11D.1S — SUPPLEMENTARY FIGURE S1 — FINAL VISUAL VERSION
EXTENDED NORMALIZATION AND PCA-QC DIAGNOSTICS
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Validated S1 v2 source-data files: PASS
Frozen samples / donors / audit flags: 755 / 454 / 4 — PASS
Frozen PCA numerical anchors: PASS


Warning message in ggplot2::annotate("label", x = 0.62, y = ROBUST_THRESHOLD + 0.05, :
"Ignoring unknown parameters: `label.size`"


agg_record_f7453084a36 
                     2

agg_record_f7453084a36 
                     2

agg_record_f7453084a36 
                     2

Supplementary Figure S1 v3 outputs created and non-zero: PASS

SUPPLEMENTARY FIGURE S1 v3 SUMMARY
Validated S1 v2 source data: PASS
Analytical results changed: NO
Panel A title clipping corrected: YES
Full Schizophrenia legend label retained: YES
Panel D threshold-label collision corrected: YES
Samples / donors: 755 / 454
PCA audit flags / exclusions: 4 / 0

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS1_QC_TNR_v3/FigureS1_Extended_Normalization_PCA_QC_TimesNewRoman_v3.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS1_QC_TNR_v3/FigureS1_Extended_Normalization_PCA_QC_TimesNewRoman_v3.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS1_QC_TNR_v3/FigureS1_Extended_Normalization_PCA_QC_TimesNewRoman_v3.png

STEP 11D.1S COMPLETE — SUPPLEMENTARY FIGURE S1 v3 GENERATED


In [6]:
# ==================================================================================================
# STEP 11D.2R — SUPPLEMENTARY FIGURE S2 — FINAL VISUAL CORRECTION
# REGION-STRATIFIED SCHIZOPHRENIA DIFFERENTIAL-EXPRESSION LANDSCAPES
# --------------------------------------------------------------------------------------------------
# PURPOSE
#
# Visual-only correction of validated Figure S2 v1.
#
# FIX
#   - Remove duplicated/clipped per-panel legends.
#   - Add ONE shared full-width legend below both volcano panels.
#
# UNCHANGED
#   - canonical DE results
#   - 23,740-gene universe
#   - DLPFC FDR counts
#   - HIPPO FDR counts
#   - gene labels
#   - effect sizes
#   - FDR values
#
# NO:
#   - limma rerun
#   - voom rerun
#   - duplicateCorrelation
#   - DE recalculation
#   - gene filtering
#   - model fitting
#
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.2R — SUPPLEMENTARY FIGURE S2 — FINAL VISUAL CORRECTION\n",
    "REGION-STRATIFIED SCHIZOPHRENIA DIFFERENTIAL-EXPRESSION LANDSCAPES\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "ggrepel",
    quietly = TRUE
)) {
    stop("ggrepel unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. VALIDATED S2 v1 SOURCE FILES
# ==================================================================================================

V1_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS2_Regional_DE_TNR_v1"
)

V1_SOURCE_DIR <- file.path(
    V1_DIR,
    "source_data"
)

DLPFC_SOURCE_PATH <- file.path(
    V1_SOURCE_DIR,
    "FigureS2_DLPFC_DE_source_v1.csv"
)

HIPPO_SOURCE_PATH <- file.path(
    V1_SOURCE_DIR,
    "FigureS2_HIPPO_DE_source_v1.csv"
)

LABEL_SOURCE_PATH <- file.path(
    V1_SOURCE_DIR,
    "FigureS2_labelled_genes_v1.csv"
)

required_inputs <- c(
    DLPFC_SOURCE_PATH,
    HIPPO_SOURCE_PATH,
    LABEL_SOURCE_PATH
)

missing_inputs <- required_inputs[
    !file.exists(
        required_inputs
    )
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required validated S2 v1 source-data file(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat("Validated Figure S2 v1 source data: PASS\n")

# ==================================================================================================
# 5. LOAD VALIDATED SOURCE DATA
# ==================================================================================================

dlpfc_plot <- read.csv(
    DLPFC_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

hippo_plot <- read.csv(
    HIPPO_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

label_source <- read.csv(
    LABEL_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 6. SCHEMA CHECK
# ==================================================================================================

required_de_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "display_symbol",
    "logFC",
    "P.Value",
    "adj.P.Val",
    "minus_log10_FDR",
    "significance",
    "labelled_in_figure"
)

for (nm in c(
    "dlpfc_plot",
    "hippo_plot"
)) {

    x <- get(nm)

    missing_cols <- setdiff(
        required_de_cols,
        names(x)
    )

    if (length(missing_cols) > 0L) {

        stop(
            nm,
            " source missing required field(s): ",
            paste(
                missing_cols,
                collapse = ", "
            )
        )
    }
}

required_label_cols <- c(
    "region",
    "ensemblID_stable",
    "gene_symbol",
    "display_symbol",
    "logFC",
    "P.Value",
    "adj.P.Val",
    "significance"
)

missing_label_cols <- setdiff(
    required_label_cols,
    names(label_source)
)

if (length(missing_label_cols) > 0L) {

    stop(
        "S2 label source missing required field(s): ",
        paste(
            missing_label_cols,
            collapse = ", "
        )
    )
}

cat("Validated S2 source schemas: PASS\n")

# ==================================================================================================
# 7. CANONICAL DIMENSION CHECK
# ==================================================================================================

if (
    nrow(dlpfc_plot) != 23740L ||
    nrow(hippo_plot) != 23740L
) {

    stop(
        "Expected 23,740 genes in each regional source table."
    )
}

if (
    anyDuplicated(
        dlpfc_plot$ensemblID_stable
    ) > 0L ||
    anyDuplicated(
        hippo_plot$ensemblID_stable
    ) > 0L
) {

    stop(
        "Duplicated stable Ensembl ID detected."
    )
}

if (!setequal(
    dlpfc_plot$ensemblID_stable,
    hippo_plot$ensemblID_stable
)) {

    stop(
        "DLPFC and HIPPO frozen gene universes disagree."
    )
}

cat(
    "Regional 23,740-gene source tables: PASS\n"
)

# ==================================================================================================
# 8. REBUILD SIGNIFICANCE FACTOR FOR DISPLAY ONLY
# ==================================================================================================

SIG_LEVELS <- c(
    "Lower in schizophrenia",
    "Not FDR-significant",
    "Higher in schizophrenia"
)

dlpfc_plot$significance <- factor(
    as.character(
        dlpfc_plot$significance
    ),
    levels = SIG_LEVELS
)

hippo_plot$significance <- factor(
    as.character(
        hippo_plot$significance
    ),
    levels = SIG_LEVELS
)

if (
    anyNA(
        dlpfc_plot$significance
    ) ||
    anyNA(
        hippo_plot$significance
    )
) {

    stop(
        "Unexpected significance category in validated S2 source."
    )
}

# ==================================================================================================
# 9. STRICT NUMERICAL QC
# ==================================================================================================

DLPFC_TOTAL <- sum(
    dlpfc_plot$adj.P.Val < 0.05,
    na.rm = TRUE
)

DLPFC_POS <- sum(
    dlpfc_plot$adj.P.Val < 0.05 &
        dlpfc_plot$logFC > 0,
    na.rm = TRUE
)

DLPFC_NEG <- sum(
    dlpfc_plot$adj.P.Val < 0.05 &
        dlpfc_plot$logFC < 0,
    na.rm = TRUE
)

HIPPO_TOTAL <- sum(
    hippo_plot$adj.P.Val < 0.05,
    na.rm = TRUE
)

HIPPO_POS <- sum(
    hippo_plot$adj.P.Val < 0.05 &
        hippo_plot$logFC > 0,
    na.rm = TRUE
)

HIPPO_NEG <- sum(
    hippo_plot$adj.P.Val < 0.05 &
        hippo_plot$logFC < 0,
    na.rm = TRUE
)

if (
    DLPFC_TOTAL != 2949L ||
    DLPFC_POS != 1312L ||
    DLPFC_NEG != 1637L
) {

    stop(
        "Validated DLPFC canonical counts changed."
    )
}

if (
    HIPPO_TOTAL != 2648L ||
    HIPPO_POS != 1349L ||
    HIPPO_NEG != 1299L
) {

    stop(
        "Validated HIPPO canonical counts changed."
    )
}

cat(
    "Canonical DLPFC / HIPPO FDR counts: PASS\n"
)

# ==================================================================================================
# 10. LABEL SET CHECK
# ==================================================================================================

dlpfc_labels <- label_source[
    label_source$region ==
        "DLPFC",
    ,
    drop = FALSE
]

hippo_labels <- label_source[
    label_source$region ==
        "HIPPO",
    ,
    drop = FALSE
]

if (
    nrow(dlpfc_labels) != 8L ||
    nrow(hippo_labels) != 8L
) {

    stop(
        "Expected 8 frozen visualization labels per regional volcano."
    )
}

cat(
    "Frozen visualization labels: 8 DLPFC / 8 HIPPO — PASS\n"
)

# ==================================================================================================
# 11. COMMON AXIS LIMITS
# ==================================================================================================

combined_logFC <- c(
    dlpfc_plot$logFC,
    hippo_plot$logFC
)

combined_y <- c(
    dlpfc_plot$minus_log10_FDR,
    hippo_plot$minus_log10_FDR
)

X_MAX <- ceiling(
    max(
        abs(
            combined_logFC
        ),
        na.rm = TRUE
    ) *
        10
) / 10

Y_MAX <- ceiling(
    max(
        combined_y,
        na.rm = TRUE
    ) +
        0.8
)

if (
    !is.finite(X_MAX) ||
    X_MAX <= 0 ||
    !is.finite(Y_MAX) ||
    Y_MAX <= 0
) {

    stop(
        "Invalid common volcano axis limits."
    )
}

FDR_LINE <- -log10(
    0.05
)

# ==================================================================================================
# 12. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS2_Regional_DE_TNR_v2"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "FigureS2_Regional_DE_Landscapes_TimesNewRoman_v2.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "FigureS2_Regional_DE_Landscapes_TimesNewRoman_v2.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "FigureS2_Regional_DE_Landscapes_TimesNewRoman_v2.png"
)

DLPFC_OUT_PATH <- file.path(
    SOURCE_DIR,
    "FigureS2_DLPFC_DE_source_v2.csv"
)

HIPPO_OUT_PATH <- file.path(
    SOURCE_DIR,
    "FigureS2_HIPPO_DE_source_v2.csv"
)

LABEL_OUT_PATH <- file.path(
    SOURCE_DIR,
    "FigureS2_labelled_genes_v2.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "FigureS2_generation_summary_v2.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "FigureS2_generation_manifest_v2.csv"
)

write.csv(
    dlpfc_plot,
    DLPFC_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    hippo_plot,
    HIPPO_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    label_source,
    LABEL_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 13. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.5,
            hjust = 0,
            margin = ggplot2::margin(
                b = 6
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.5
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.3,
            colour = "black"
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            8,
            10,
            5,
            10
        )
    )

# ==================================================================================================
# 14. COLOURS
# ==================================================================================================

volcano_colours <- c(
    "Lower in schizophrenia" = "#4E79A7",
    "Not FDR-significant" = "#CFCFCF",
    "Higher in schizophrenia" = "#E15759"
)

# ==================================================================================================
# 15. PANEL ANNOTATIONS
# ==================================================================================================

DLPFC_ANNOTATION <- paste0(
    "FDR < 0.05: ",
    format(
        DLPFC_TOTAL,
        big.mark = ","
    ),
    "\nHigher: ",
    format(
        DLPFC_POS,
        big.mark = ","
    ),
    "   Lower: ",
    format(
        DLPFC_NEG,
        big.mark = ","
    )
)

HIPPO_ANNOTATION <- paste0(
    "FDR < 0.05: ",
    format(
        HIPPO_TOTAL,
        big.mark = ","
    ),
    "\nHigher: ",
    format(
        HIPPO_POS,
        big.mark = ","
    ),
    "   Lower: ",
    format(
        HIPPO_NEG,
        big.mark = ","
    )
)

# ==================================================================================================
# 16. PANEL A — DLPFC
# ==================================================================================================

pA <- ggplot2::ggplot(
    dlpfc_plot,
    ggplot2::aes(
        x = logFC,
        y = minus_log10_FDR,
        colour = significance
    )
) +
    ggplot2::geom_hline(
        yintercept = FDR_LINE,
        linetype = "dashed",
        linewidth = 0.52,
        colour = "#555555"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.38,
        colour = "#B5B5B5"
    ) +
    ggplot2::geom_point(
        size = 1.05,
        alpha = 0.68
    ) +
    ggrepel::geom_text_repel(
        data = dlpfc_labels,
        ggplot2::aes(
            x = logFC,
            y = -log10(
                pmax(
                    adj.P.Val,
                    1e-300
                )
            ),
            label = display_symbol
        ),
        inherit.aes = FALSE,
        family = FONT_FAMILY,
        size = 2.35,
        colour = ifelse(
            dlpfc_labels$logFC < 0,
            "#4E79A7",
            "#E15759"
        ),
        box.padding = 0.28,
        point.padding = 0.16,
        segment.size = 0.24,
        segment.colour = "#777777",
        max.overlaps = Inf,
        min.segment.length = 0,
        seed = 20261001
    ) +
    ggplot2::annotate(
        "text",
        x = X_MAX * 0.96,
        y = Y_MAX * 0.96,
        label = DLPFC_ANNOTATION,
        hjust = 1,
        vjust = 1,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.55,
        lineheight = 1.05
    ) +
    ggplot2::scale_colour_manual(
        values = volcano_colours,
        drop = FALSE
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            -X_MAX,
            X_MAX
        ),
        expand = ggplot2::expansion(
            mult = c(
                0.02,
                0.02
            )
        )
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            0,
            Y_MAX
        ),
        expand = ggplot2::expansion(
            mult = c(
                0,
                0.02
            )
        )
    ) +
    ggplot2::labs(
        title = "A  DLPFC schizophrenia effect",
        x = "log2 fold change (schizophrenia vs control)",
        y = expression(-log[10]("FDR"))
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "none"
    )

# ==================================================================================================
# 17. PANEL B — HIPPO
# ==================================================================================================

pB <- ggplot2::ggplot(
    hippo_plot,
    ggplot2::aes(
        x = logFC,
        y = minus_log10_FDR,
        colour = significance
    )
) +
    ggplot2::geom_hline(
        yintercept = FDR_LINE,
        linetype = "dashed",
        linewidth = 0.52,
        colour = "#555555"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.38,
        colour = "#B5B5B5"
    ) +
    ggplot2::geom_point(
        size = 1.05,
        alpha = 0.68
    ) +
    ggrepel::geom_text_repel(
        data = hippo_labels,
        ggplot2::aes(
            x = logFC,
            y = -log10(
                pmax(
                    adj.P.Val,
                    1e-300
                )
            ),
            label = display_symbol
        ),
        inherit.aes = FALSE,
        family = FONT_FAMILY,
        size = 2.35,
        colour = ifelse(
            hippo_labels$logFC < 0,
            "#4E79A7",
            "#E15759"
        ),
        box.padding = 0.28,
        point.padding = 0.16,
        segment.size = 0.24,
        segment.colour = "#777777",
        max.overlaps = Inf,
        min.segment.length = 0,
        seed = 20261001
    ) +
    ggplot2::annotate(
        "text",
        x = X_MAX * 0.96,
        y = Y_MAX * 0.96,
        label = HIPPO_ANNOTATION,
        hjust = 1,
        vjust = 1,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.55,
        lineheight = 1.05
    ) +
    ggplot2::scale_colour_manual(
        values = volcano_colours,
        drop = FALSE
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            -X_MAX,
            X_MAX
        ),
        expand = ggplot2::expansion(
            mult = c(
                0.02,
                0.02
            )
        )
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            0,
            Y_MAX
        ),
        expand = ggplot2::expansion(
            mult = c(
                0,
                0.02
            )
        )
    ) +
    ggplot2::labs(
        title = "B  Hippocampal schizophrenia effect",
        x = "log2 fold change (schizophrenia vs control)",
        y = expression(-log[10]("FDR"))
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "none"
    )

# ==================================================================================================
# 18. SHARED FULL-WIDTH LEGEND GROBS
# ==================================================================================================

legend_title <- grid::textGrob(
    "Differential expression",
    x = grid::unit(
        0.075,
        "npc"
    ),
    y = grid::unit(
        0.50,
        "npc"
    ),
    just = "left",
    gp = grid::gpar(
        fontfamily = FONT_FAMILY,
        fontface = "bold",
        fontsize = 8
    )
)

legend_point_1 <- grid::pointsGrob(
    x = grid::unit(
        0.285,
        "npc"
    ),
    y = grid::unit(
        0.50,
        "npc"
    ),
    pch = 16,
    size = grid::unit(
        1.5,
        "mm"
    ),
    gp = grid::gpar(
        col = "#4E79A7"
    )
)

legend_text_1 <- grid::textGrob(
    "Lower in schizophrenia",
    x = grid::unit(
        0.300,
        "npc"
    ),
    y = grid::unit(
        0.50,
        "npc"
    ),
    just = "left",
    gp = grid::gpar(
        fontfamily = FONT_FAMILY,
        fontsize = 7.8
    )
)

legend_point_2 <- grid::pointsGrob(
    x = grid::unit(
        0.515,
        "npc"
    ),
    y = grid::unit(
        0.50,
        "npc"
    ),
    pch = 16,
    size = grid::unit(
        1.5,
        "mm"
    ),
    gp = grid::gpar(
        col = "#CFCFCF"
    )
)

legend_text_2 <- grid::textGrob(
    "Not FDR-significant",
    x = grid::unit(
        0.530,
        "npc"
    ),
    y = grid::unit(
        0.50,
        "npc"
    ),
    just = "left",
    gp = grid::gpar(
        fontfamily = FONT_FAMILY,
        fontsize = 7.8
    )
)

legend_point_3 <- grid::pointsGrob(
    x = grid::unit(
        0.735,
        "npc"
    ),
    y = grid::unit(
        0.50,
        "npc"
    ),
    pch = 16,
    size = grid::unit(
        1.5,
        "mm"
    ),
    gp = grid::gpar(
        col = "#E15759"
    )
)

legend_text_3 <- grid::textGrob(
    "Higher in schizophrenia",
    x = grid::unit(
        0.750,
        "npc"
    ),
    y = grid::unit(
        0.50,
        "npc"
    ),
    just = "left",
    gp = grid::gpar(
        fontfamily = FONT_FAMILY,
        fontsize = 7.8
    )
)

shared_legend <- grid::grobTree(
    legend_title,
    legend_point_1,
    legend_text_1,
    legend_point_2,
    legend_text_2,
    legend_point_3,
    legend_text_3
)

# ==================================================================================================
# 19. DRAW FINAL FIGURE
# ==================================================================================================

draw_figure_s2 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 2,
        ncol = 2,
        widths = grid::unit(
            c(
                1,
                1
            ),
            "null"
        ),
        heights = grid::unit(
            c(
                1,
                0.105
            ),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::pushViewport(
        grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1:2
        )
    )

    grid::grid.draw(
        shared_legend
    )

    grid::popViewport()
    grid::popViewport()
}

# ==================================================================================================
# 20. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 3.85,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure_s2()

grDevices::dev.off()

# ==================================================================================================
# 21. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 3.85,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure_s2()

grDevices::dev.off()

# ==================================================================================================
# 22. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 3.85,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure_s2()

grDevices::dev.off()

# ==================================================================================================
# 23. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    DLPFC_OUT_PATH,
    HIPPO_OUT_PATH,
    LABEL_OUT_PATH
)

if (!all(
    file.exists(
        outputs
    )
)) {

    stop(
        "One or more Figure S2 v2 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {

    stop(
        "One or more Figure S2 v2 outputs are zero-byte."
    )
}

cat(
    "Supplementary Figure S2 v2 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 24. GENERATION SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11D.2R — SUPPLEMENTARY FIGURE S2 — FINAL VISUAL VERSION",
    "==============================================================================================================",

    "",

    "Generated from validated Figure S2 v1 frozen source data.",

    "No DE analysis was rerun.",

    "",

    paste0(
        "DLPFC FDR < 0.05 = ",
        DLPFC_TOTAL,
        " (higher ",
        DLPFC_POS,
        "; lower ",
        DLPFC_NEG,
        ")"
    ),

    paste0(
        "HIPPO FDR < 0.05 = ",
        HIPPO_TOTAL,
        " (higher ",
        HIPPO_POS,
        "; lower ",
        HIPPO_NEG,
        ")"
    ),

    "",

    "Visual correction:",

    "Duplicated per-panel legends removed.",

    "Single shared full-width legend added below both volcano plots.",

    "No analytical or numerical result changed.",

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 25. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 26. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSUPPLEMENTARY FIGURE S2 v2 SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Validated Figure S2 v1 sources: PASS\n"
)

cat(
    "Analytical results changed: NO\n"
)

cat(
    "DLPFC FDR < 0.05: ",
    DLPFC_TOTAL,
    " | Higher / Lower: ",
    DLPFC_POS,
    " / ",
    DLPFC_NEG,
    "\n",
    sep = ""
)

cat(
    "HIPPO FDR < 0.05: ",
    HIPPO_TOTAL,
    " | Higher / Lower: ",
    HIPPO_POS,
    " / ",
    HIPPO_NEG,
    "\n",
    sep = ""
)

cat(
    "Shared full-width legend: YES\n"
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.2R COMPLETE — SUPPLEMENTARY FIGURE S2 v2 GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11D.2R — SUPPLEMENTARY FIGURE S2 — FINAL VISUAL CORRECTION
REGION-STRATIFIED SCHIZOPHRENIA DIFFERENTIAL-EXPRESSION LANDSCAPES
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Validated Figure S2 v1 source data: PASS
Validated S2 source schemas: PASS
Regional 23,740-gene source tables: PASS
Canonical DLPFC / HIPPO FDR counts: PASS
Frozen visualization labels: 8 DLPFC / 8 HIPPO — PASS


agg_record_f743c72649e 
                     2

agg_record_f743c72649e 
                     2

agg_record_f743c72649e 
                     2

Supplementary Figure S2 v2 outputs created and non-zero: PASS

SUPPLEMENTARY FIGURE S2 v2 SUMMARY
Validated Figure S2 v1 sources: PASS
Analytical results changed: NO
DLPFC FDR < 0.05: 2949 | Higher / Lower: 1312 / 1637
HIPPO FDR < 0.05: 2648 | Higher / Lower: 1349 / 1299
Shared full-width legend: YES

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS2_Regional_DE_TNR_v2/FigureS2_Regional_DE_Landscapes_TimesNewRoman_v2.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS2_Regional_DE_TNR_v2/FigureS2_Regional_DE_Landscapes_TimesNewRoman_v2.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS2_Regional_DE_TNR_v2/FigureS2_Regional_DE_Landscapes_TimesNewRoman_v2.png

STEP 11D.2R COMPLETE — SUPPLEMENTARY FIGURE S2 v2 GENERATED


In [8]:
# ==================================================================================================
# STEP 11D.3R — SUPPLEMENTARY FIGURE S3 — FINAL VISUAL / INTERPRETATION-SAFE VERSION
# NESTED-CV TUNING AND MODEL-COMPLEXITY DIAGNOSTICS
# --------------------------------------------------------------------------------------------------
# VISUAL-ONLY REVISION OF VALIDATED S3 v1
#
# FIXES
#
#   1. Shorten Panel D title to prevent clipping.
#   2. Panel C explicitly retains deviation-penalty levels 1, 2, 4, 8;
#      penalty 4 remains visibly empty because it was selected in 0/25 outer fits.
#   3. Panel D legend shows ONLY penalties actually selected (1, 2, 8),
#      eliminating the visually blank penalty-4 legend key.
#   4. Remove descriptive Pearson-r annotation from the plotted panel.
#      The correlation is preserved in the generation summary/source audit,
#      but is not elevated to a manuscript visual result.
#   5. Preserve all frozen numerical values.
#
# NO ANALYTICAL RERUN:
#   - no glmnet fitting
#   - no inner CV
#   - no outer CV
#   - no tuning
#   - no predictions
#   - no bootstrap
#
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.3R — SUPPLEMENTARY FIGURE S3 — FINAL VISUAL VERSION\n",
    "NESTED-CV TUNING AND MODEL-COMPLEXITY DIAGNOSTICS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. VALIDATED S3 v1 SOURCE FILES
# ==================================================================================================

V1_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS3_NestedCV_Diagnostics_TNR_v1"
)

V1_SOURCE_DIR <- file.path(
    V1_DIR,
    "source_data"
)

FOLD_SOURCE_PATH <- file.path(
    V1_SOURCE_DIR,
    "FigureS3_25_outer_fold_diagnostics_source_v1.csv"
)

HYPER_SOURCE_PATH <- file.path(
    V1_SOURCE_DIR,
    "FigureS3_hyperparameter_frequency_source_v1.csv"
)

required_inputs <- c(
    FOLD_SOURCE_PATH,
    HYPER_SOURCE_PATH
)

missing_inputs <- required_inputs[
    !file.exists(
        required_inputs
    )
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required validated S3 v1 source-data file(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat(
    "Validated Figure S3 v1 source data: PASS\n"
)

# ==================================================================================================
# 5. LOAD VALIDATED SOURCES
# ==================================================================================================

folds <- read.csv(
    FOLD_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

hyper_source <- read.csv(
    HYPER_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 6. SCHEMA VALIDATION
# ==================================================================================================

required_fold_cols <- c(
    "outer_repeat",
    "outer_fold",
    "best_alpha",
    "best_deviation_penalty",
    "best_lambda",
    "best_inner_auc",
    "outer_donor_auc",
    "n_shared_nonzero",
    "n_deviation_nonzero"
)

missing_fold_cols <- setdiff(
    required_fold_cols,
    names(folds)
)

if (length(missing_fold_cols) > 0L) {

    stop(
        "Validated S3 fold source missing field(s): ",
        paste(
            missing_fold_cols,
            collapse = ", "
        )
    )
}

required_hyper_cols <- c(
    "deviation_penalty",
    "n_outer_folds"
)

missing_hyper_cols <- setdiff(
    required_hyper_cols,
    names(hyper_source)
)

if (length(missing_hyper_cols) > 0L) {

    stop(
        "Validated S3 hyperparameter source missing field(s): ",
        paste(
            missing_hyper_cols,
            collapse = ", "
        )
    )
}

cat(
    "Validated S3 source schemas: PASS\n"
)

# ==================================================================================================
# 7. OUTER-CV STRUCTURE
# ==================================================================================================

if (nrow(folds) != 25L) {

    stop(
        "Expected 25 frozen outer evaluations; observed ",
        nrow(folds)
    )
}

expected_pairs <- expand.grid(
    outer_repeat = 1:5,
    outer_fold = 1:5,
    KEEP.OUT.ATTRS = FALSE,
    stringsAsFactors = FALSE
)

observed_key <- paste(
    folds$outer_repeat,
    folds$outer_fold,
    sep = "_"
)

expected_key <- paste(
    expected_pairs$outer_repeat,
    expected_pairs$outer_fold,
    sep = "_"
)

if (
    anyDuplicated(observed_key) > 0L ||
    !setequal(
        observed_key,
        expected_key
    )
) {

    stop(
        "Frozen 5-repeat x 5-fold structure changed."
    )
}

folds <- folds[
    order(
        folds$outer_repeat,
        folds$outer_fold
    ),
    ,
    drop = FALSE
]

folds$fold_index <- seq_len(
    nrow(folds)
)

folds$fold_label <- paste0(
    "R",
    folds$outer_repeat,
    "/F",
    folds$outer_fold
)

cat(
    "Outer structure: 25 held-out evaluations — PASS\n"
)

# ==================================================================================================
# 8. STRICT HYPERPARAMETER ANCHORS
# ==================================================================================================

if (
    !all(
        abs(
            folds$best_alpha -
            0.10
        ) <
            1e-12
    )
) {

    stop(
        "Frozen alpha-selection pattern changed."
    )
}

penalty_levels_numeric <- c(
    1,
    2,
    4,
    8
)

penalty_counts <- vapply(
    penalty_levels_numeric,
    function(p) {

        sum(
            folds$best_deviation_penalty ==
                p
        )
    },
    FUN.VALUE = integer(1)
)

if (!identical(
    as.integer(
        penalty_counts
    ),
    c(
        18L,
        1L,
        0L,
        6L
    )
)) {

    stop(
        "Frozen deviation-penalty frequencies changed."
    )
}

cat(
    "Selected alpha = 0.10 in all 25 fits — PASS\n"
)

cat(
    "Deviation penalty 1 / 2 / 4 / 8 = 18 / 1 / 0 / 6 — PASS\n"
)

# ==================================================================================================
# 9. MODEL-COMPLEXITY ANCHORS
# ==================================================================================================

MEAN_SHARED <- mean(
    folds$n_shared_nonzero
)

MEDIAN_SHARED <- median(
    folds$n_shared_nonzero
)

MIN_SHARED <- min(
    folds$n_shared_nonzero
)

MAX_SHARED <- max(
    folds$n_shared_nonzero
)

MEAN_DEVIATION <- mean(
    folds$n_deviation_nonzero
)

MEDIAN_DEVIATION <- median(
    folds$n_deviation_nonzero
)

MIN_DEVIATION <- min(
    folds$n_deviation_nonzero
)

MAX_DEVIATION <- max(
    folds$n_deviation_nonzero
)

if (
    abs(
        MEAN_SHARED -
        1501.12
    ) >
        5e-8 ||
    MEDIAN_SHARED != 1534 ||
    MIN_SHARED != 759 ||
    MAX_SHARED != 1677
) {

    stop(
        "Frozen shared-complexity anchors changed."
    )
}

if (
    abs(
        MEAN_DEVIATION -
        59.6
    ) >
        5e-8 ||
    MEDIAN_DEVIATION != 78 ||
    MIN_DEVIATION != 0 ||
    MAX_DEVIATION != 100
) {

    stop(
        "Frozen deviation-complexity anchors changed."
    )
}

cat(
    "Frozen shared/deviation model-complexity anchors: PASS\n"
)

# ==================================================================================================
# 10. PERFORMANCE ANCHORS
# ==================================================================================================

MEAN_OUTER_AUC <- mean(
    folds$outer_donor_auc
)

SD_OUTER_AUC <- stats::sd(
    folds$outer_donor_auc
)

MIN_OUTER_AUC <- min(
    folds$outer_donor_auc
)

MAX_OUTER_AUC <- max(
    folds$outer_donor_auc
)

if (
    abs(
        MEAN_OUTER_AUC -
        0.7868384
    ) >
        5e-7
) {

    stop(
        "Mean outer donor AUROC changed."
    )
}

if (
    abs(
        SD_OUTER_AUC -
        0.04047966
    ) >
        5e-7
) {

    stop(
        "SD outer donor AUROC changed."
    )
}

if (
    abs(
        MIN_OUTER_AUC -
        0.6782828
    ) >
        5e-7 ||
    abs(
        MAX_OUTER_AUC -
        0.8647587
    ) >
        5e-7
) {

    stop(
        "Outer donor AUROC range changed."
    )
}

cat(
    "Frozen outer donor AUROC anchors: PASS\n"
)

# ==================================================================================================
# 11. DESCRIPTIVE INNER-vs-OUTER CORRELATION
#
# Preserved for audit/summary only.
# NOT displayed in the final figure.
# ==================================================================================================

DESCRIPTIVE_COR <- stats::cor(
    folds$best_inner_auc,
    folds$outer_donor_auc,
    method = "pearson",
    use = "complete.obs"
)

if (!is.finite(
    DESCRIPTIVE_COR
)) {

    stop(
        "Could not calculate descriptive inner-vs-outer correlation."
    )
}

cat(
    "Descriptive inner-vs-outer Pearson r = ",
    sprintf(
        "%.4f",
        DESCRIPTIVE_COR
    ),
    " — audit only\n",
    sep = ""
)

# ==================================================================================================
# 12. PANEL A DATA
# ==================================================================================================

panel_A <- data.frame(

    deviation_penalty = factor(
        penalty_levels_numeric,
        levels = penalty_levels_numeric
    ),

    n_outer_folds = as.integer(
        penalty_counts
    ),

    stringsAsFactors = FALSE
)

# ==================================================================================================
# 13. PANEL B DATA
# ==================================================================================================

panel_B <- rbind(

    data.frame(
        fold_index =
            folds$fold_index,

        component =
            "Shared",

        n_nonzero =
            folds$n_shared_nonzero,

        stringsAsFactors = FALSE
    ),

    data.frame(
        fold_index =
            folds$fold_index,

        component =
            "Regional deviation",

        n_nonzero =
            folds$n_deviation_nonzero,

        stringsAsFactors = FALSE
    )
)

panel_B$component <- factor(
    panel_B$component,
    levels = c(
        "Shared",
        "Regional deviation"
    )
)

# +1 only to permit zero counts on logarithmic display.
panel_B$count_plus_one <- (
    panel_B$n_nonzero +
        1
)

# ==================================================================================================
# 14. PANEL C DATA
# ==================================================================================================

panel_C <- data.frame(

    fold_index =
        folds$fold_index,

    deviation_penalty = factor(
        folds$best_deviation_penalty,
        levels = penalty_levels_numeric
    ),

    best_lambda =
        folds$best_lambda,

    stringsAsFactors = FALSE
)

# ==================================================================================================
# 15. PANEL D DATA
# ==================================================================================================

selected_penalty_levels <- c(
    "1",
    "2",
    "8"
)

panel_D <- data.frame(

    fold_index =
        folds$fold_index,

    deviation_penalty = factor(
        as.character(
            folds$best_deviation_penalty
        ),
        levels = selected_penalty_levels
    ),

    best_inner_auc =
        folds$best_inner_auc,

    outer_donor_auc =
        folds$outer_donor_auc,

    stringsAsFactors = FALSE
)

if (anyNA(
    panel_D$deviation_penalty
)) {

    stop(
        "Unexpected selected deviation penalty in Panel D."
    )
}

# ==================================================================================================
# 16. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS3_NestedCV_Diagnostics_TNR_v2"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "FigureS3_NestedCV_Tuning_Complexity_TimesNewRoman_v2.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "FigureS3_NestedCV_Tuning_Complexity_TimesNewRoman_v2.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "FigureS3_NestedCV_Tuning_Complexity_TimesNewRoman_v2.png"
)

FOLD_SOURCE_OUT <- file.path(
    SOURCE_DIR,
    "FigureS3_25_outer_fold_diagnostics_source_v2.csv"
)

HYPER_SOURCE_OUT <- file.path(
    SOURCE_DIR,
    "FigureS3_hyperparameter_frequency_source_v2.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "FigureS3_generation_summary_v2.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "FigureS3_generation_manifest_v2.csv"
)

write.csv(
    folds,
    FOLD_SOURCE_OUT,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    panel_A,
    HYPER_SOURCE_OUT,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 17. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.1,
            hjust = 0,
            margin = ggplot2::margin(
                b = 5
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.2
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.0,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.8
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.5
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            8,
            9,
            8,
            9
        )
    )

# ==================================================================================================
# 18. COLOURS
# ==================================================================================================

penalty_colours <- c(
    "1" = "#4E79A7",
    "2" = "#F28E2B",
    "4" = "#59A14F",
    "8" = "#B07AA1"
)

component_colours <- c(
    "Shared" = "#4E79A7",
    "Regional deviation" = "#E15759"
)

# ==================================================================================================
# 19. PANEL A — PENALTY SELECTION FREQUENCY
# ==================================================================================================

pA <- ggplot2::ggplot(
    panel_A,
    ggplot2::aes(
        x = deviation_penalty,
        y = n_outer_folds,
        fill = deviation_penalty
    )
) +
    ggplot2::geom_col(
        width = 0.66,
        colour = "black",
        linewidth = 0.45
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = n_outer_folds
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.75,
        vjust = -0.50
    ) +
    ggplot2::annotate(
        "text",
        x = 4,
        y = 20.2,
        label = expression(alpha == 0.10~"in all 25 outer fits"),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.45,
        hjust = 1
    ) +
    ggplot2::scale_fill_manual(
        values = penalty_colours,
        drop = FALSE,
        guide = "none"
    ) +
    ggplot2::scale_x_discrete(
        drop = FALSE
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            0,
            21
        ),
        breaks = c(
            0,
            5,
            10,
            15,
            20
        ),
        expand = c(
            0,
            0
        )
    ) +
    ggplot2::labs(
        title = "A  Selected deviation-penalty frequency",
        x = "Regional-deviation penalty multiplier",
        y = "Outer fits, n"
    ) +
    theme_publication

# ==================================================================================================
# 20. PANEL B — MODEL COMPLEXITY
# ==================================================================================================

pB <- ggplot2::ggplot(
    panel_B,
    ggplot2::aes(
        x = component,
        y = count_plus_one,
        colour = component
    )
) +
    ggplot2::geom_boxplot(
        ggplot2::aes(
            fill = component
        ),
        width = 0.44,
        alpha = 0.16,
        outlier.shape = NA,
        colour = "black",
        linewidth = 0.58,
        show.legend = FALSE
    ) +
    ggplot2::geom_point(
        position = ggplot2::position_jitter(
            width = 0.105,
            height = 0,
            seed = 20261001
        ),
        size = 1.70,
        alpha = 0.78,
        show.legend = FALSE
    ) +
    ggplot2::scale_colour_manual(
        values = component_colours,
        guide = "none"
    ) +
    ggplot2::scale_fill_manual(
        values = component_colours,
        guide = "none"
    ) +
    ggplot2::scale_y_log10(
        breaks = c(
            1,
            2,
            11,
            101,
            1001,
            2001
        ),
        labels = c(
            "0",
            "1",
            "10",
            "100",
            "1,000",
            "2,000"
        )
    ) +
    ggplot2::annotate(
        "text",
        x = 1,
        y = 2000,
        label = paste0(
            "Mean = ",
            sprintf(
                "%.1f",
                MEAN_SHARED
            )
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.30
    ) +
    ggplot2::annotate(
        "text",
        x = 2,
        y = 135,
        label = paste0(
            "Mean = ",
            sprintf(
                "%.1f",
                MEAN_DEVIATION
            )
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.30
    ) +
    ggplot2::labs(
        title = "B  Selected transcriptomic complexity",
        x = NULL,
        y = "Nonzero coefficients, n\n(log10[n + 1] display)"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 8.0
        )
    )

# ==================================================================================================
# 21. PANEL C — SELECTED LAMBDA
#
# Keep all four candidate deviation penalties visible on x-axis.
# Penalty 4 therefore appears as an intentionally empty category.
# ==================================================================================================

pC <- ggplot2::ggplot(
    panel_C,
    ggplot2::aes(
        x = deviation_penalty,
        y = best_lambda,
        colour = deviation_penalty
    )
) +
    ggplot2::geom_boxplot(
        width = 0.42,
        outlier.shape = NA,
        colour = "black",
        fill = "white",
        linewidth = 0.55
    ) +
    ggplot2::geom_point(
        position = ggplot2::position_jitter(
            width = 0.10,
            height = 0,
            seed = 20261001
        ),
        size = 1.70,
        alpha = 0.82
    ) +
    ggplot2::scale_colour_manual(
        values = penalty_colours,
        limits = c(
            "1",
            "2",
            "4",
            "8"
        ),
        drop = FALSE,
        guide = "none"
    ) +
    ggplot2::scale_x_discrete(
        limits = c(
            "1",
            "2",
            "4",
            "8"
        ),
        drop = FALSE
    ) +
    ggplot2::scale_y_log10() +
    ggplot2::labs(
        title = "C  Selected regularization strength",
        x = "Regional-deviation penalty multiplier",
        y = expression("Selected "*lambda*" (log scale)")
    ) +
    theme_publication

# ==================================================================================================
# 22. PANEL D — INNER-CV vs OUTER DONOR AUROC
# ==================================================================================================

auc_values <- c(
    panel_D$best_inner_auc,
    panel_D$outer_donor_auc
)

AUC_MIN <- floor(
    min(
        auc_values,
        na.rm = TRUE
    ) *
        20
) / 20

AUC_MAX <- ceiling(
    max(
        auc_values,
        na.rm = TRUE
    ) *
        20
) / 20

if (
    !is.finite(AUC_MIN) ||
    !is.finite(AUC_MAX) ||
    AUC_MIN >= AUC_MAX
) {

    stop(
        "Invalid AUROC plotting range."
    )
}

pD <- ggplot2::ggplot(
    panel_D,
    ggplot2::aes(
        x = best_inner_auc,
        y = outer_donor_auc,
        colour = deviation_penalty
    )
) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#777777"
    ) +
    ggplot2::geom_point(
        size = 2.10,
        alpha = 0.88
    ) +
    ggplot2::annotate(
        "text",
        x = AUC_MIN + 0.010,
        y = AUC_MAX - 0.012,
        label = "25 held-out outer tests",
        hjust = 0,
        vjust = 1,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35
    ) +
    ggplot2::scale_colour_manual(
        values = penalty_colours[
            selected_penalty_levels
        ],
        breaks = selected_penalty_levels,
        drop = TRUE
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            AUC_MIN,
            AUC_MAX
        )
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            AUC_MIN,
            AUC_MAX
        )
    ) +
    ggplot2::coord_equal() +
    ggplot2::labs(
        title = "D  Inner-CV vs outer-donor AUROC",
        x = "Best inner-CV AUROC",
        y = "Outer donor AUROC",
        colour = "Selected deviation\npenalty"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.direction = "horizontal",
        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.4
        ),
        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.4
        ),
        legend.key.width = grid::unit(
            0.45,
            "cm"
        ),
        legend.spacing.x = grid::unit(
            0.06,
            "cm"
        )
    )

# ==================================================================================================
# 23. DRAW FINAL 2 x 2 FIGURE
# ==================================================================================================

draw_figure_s3 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 2,
        ncol = 2,
        widths = grid::unit(
            c(
                1,
                1
            ),
            "null"
        ),
        heights = grid::unit(
            c(
                1,
                1
            ),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pD,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 24. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 5.85,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure_s3()

grDevices::dev.off()

# ==================================================================================================
# 25. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 5.85,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure_s3()

grDevices::dev.off()

# ==================================================================================================
# 26. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 5.85,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure_s3()

grDevices::dev.off()

# ==================================================================================================
# 27. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    FOLD_SOURCE_OUT,
    HYPER_SOURCE_OUT
)

if (!all(
    file.exists(
        outputs
    )
)) {

    stop(
        "One or more Figure S3 v2 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {

    stop(
        "One or more Figure S3 v2 outputs are zero-byte."
    )
}

cat(
    "Supplementary Figure S3 v2 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 28. GENERATION SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11D.3R — SUPPLEMENTARY FIGURE S3 — FINAL VISUAL VERSION",
    "==============================================================================================================",

    "",

    "Generated from validated Figure S3 v1 frozen source data.",

    "No model fitting, nested CV, tuning, prediction, or bootstrap was rerun.",

    "",

    "Outer evaluations = 25",

    "Alpha = 0.10 selected in all 25 fits",

    paste0(
        "Deviation penalty frequency: 1=",
        penalty_counts[1],
        ", 2=",
        penalty_counts[2],
        ", 4=",
        penalty_counts[3],
        ", 8=",
        penalty_counts[4]
    ),

    "",

    paste0(
        "Shared nonzero: mean=",
        sprintf(
            "%.2f",
            MEAN_SHARED
        ),
        ", median=",
        MEDIAN_SHARED,
        ", range=",
        MIN_SHARED,
        "-",
        MAX_SHARED
    ),

    paste0(
        "Deviation nonzero: mean=",
        sprintf(
            "%.2f",
            MEAN_DEVIATION
        ),
        ", median=",
        MEDIAN_DEVIATION,
        ", range=",
        MIN_DEVIATION,
        "-",
        MAX_DEVIATION
    ),

    "",

    paste0(
        "Mean outer donor AUROC = ",
        sprintf(
            "%.7f",
            MEAN_OUTER_AUC
        )
    ),

    paste0(
        "SD outer donor AUROC = ",
        sprintf(
            "%.7f",
            SD_OUTER_AUC
        )
    ),

    paste0(
        "Outer donor AUROC range = ",
        sprintf(
            "%.7f",
            MIN_OUTER_AUC
        ),
        " to ",
        sprintf(
            "%.7f",
            MAX_OUTER_AUC
        )
    ),

    paste0(
        "Descriptive Pearson r(inner CV, outer donor AUROC) = ",
        sprintf(
            "%.4f",
            DESCRIPTIVE_COR
        ),
        " [audit only; intentionally omitted from plotted annotation]"
    ),

    "",

    "Visual corrections:",

    "Panel D title shortened to avoid clipping.",

    "Penalty 4 retained as an empty candidate level in Panel C.",

    "Panel D legend restricted to penalties actually selected: 1, 2, 8.",

    "Descriptive inner-vs-outer correlation removed from plotted annotation.",

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 29. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 30. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSUPPLEMENTARY FIGURE S3 v2 SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Validated S3 v1 source data: PASS\n"
)

cat(
    "Analytical results changed: NO\n"
)

cat(
    "Outer evaluations: 25\n"
)

cat(
    "Alpha = 0.10 selected in all 25 fits\n"
)

cat(
    "Deviation penalty 1 / 2 / 4 / 8: ",
    paste(
        penalty_counts,
        collapse = " / "
    ),
    "\n",
    sep = ""
)

cat(
    "Shared nonzero mean / median / range: ",
    sprintf(
        "%.2f",
        MEAN_SHARED
    ),
    " / ",
    MEDIAN_SHARED,
    " / ",
    MIN_SHARED,
    "-",
    MAX_SHARED,
    "\n",
    sep = ""
)

cat(
    "Deviation nonzero mean / median / range: ",
    sprintf(
        "%.2f",
        MEAN_DEVIATION
    ),
    " / ",
    MEDIAN_DEVIATION,
    " / ",
    MIN_DEVIATION,
    "-",
    MAX_DEVIATION,
    "\n",
    sep = ""
)

cat(
    "Mean outer donor AUROC: ",
    sprintf(
        "%.7f",
        MEAN_OUTER_AUC
    ),
    "\n",
    sep = ""
)

cat(
    "Descriptive inner-vs-outer r retained in audit only: ",
    sprintf(
        "%.4f",
        DESCRIPTIVE_COR
    ),
    "\n",
    sep = ""
)

cat(
    "Panel D title clipping corrected: YES\n"
)

cat(
    "Penalty-4 empty candidate level preserved in Panel C: YES\n"
)

cat(
    "Blank penalty-4 legend key removed from Panel D: YES\n"
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.3R COMPLETE — SUPPLEMENTARY FIGURE S3 v2 GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11D.3R — SUPPLEMENTARY FIGURE S3 — FINAL VISUAL VERSION
NESTED-CV TUNING AND MODEL-COMPLEXITY DIAGNOSTICS
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Validated Figure S3 v1 source data: PASS
Validated S3 source schemas: PASS
Outer structure: 25 held-out evaluations — PASS
Selected alpha = 0.10 in all 25 fits — PASS
Deviation penalty 1 / 2 / 4 / 8 = 18 / 1 / 0 / 6 — PASS
Frozen shared/deviation model-complexity anchors: PASS
Frozen outer donor AUROC anchors: PASS
Descriptive inner-vs-outer Pearson r = -0.6255 — audit only


agg_record_f745a112a42 
                     2

agg_record_f745a112a42 
                     2

agg_record_f745a112a42 
                     2

Supplementary Figure S3 v2 outputs created and non-zero: PASS

SUPPLEMENTARY FIGURE S3 v2 SUMMARY
Validated S3 v1 source data: PASS
Analytical results changed: NO
Outer evaluations: 25
Alpha = 0.10 selected in all 25 fits
Deviation penalty 1 / 2 / 4 / 8: 18 / 1 / 0 / 6
Shared nonzero mean / median / range: 1501.12 / 1534 / 759-1677
Deviation nonzero mean / median / range: 59.60 / 78 / 0-100
Mean outer donor AUROC: 0.7868384
Descriptive inner-vs-outer r retained in audit only: -0.6255
Panel D title clipping corrected: YES
Penalty-4 empty candidate level preserved in Panel C: YES
Blank penalty-4 legend key removed from Panel D: YES

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS3_NestedCV_Diagnostics_TNR_v2/FigureS3_NestedCV_Tuning_Complexity_TimesNewRoman_v2.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS3_NestedCV_Diagnostics_TNR_v2/FigureS3_NestedCV_Tuning_Complexity_TimesNewRoman_v2.tiff

PNG preview:
E:/SCZ_CROSSRE

In [9]:
# ==================================================================================================
# STEP 11D.4 — SUPPLEMENTARY FIGURE S4
# EXTENDED STABILITY AND RESAMPLING-REFINEMENT DIAGNOSTICS
# --------------------------------------------------------------------------------------------------
# PURPOSE
#
# Complement Main Figure 4 WITHOUT repeating:
#   - the final 24-feature stability heatmap
#   - the diagnosis-permutation AUROC histogram
#
# PANELS
#
#   A — Stability-survival curves across all 23,740 genes
#       Number of genes retained at or above each selection-probability threshold
#       Outer-CV stability vs complementary half-sample stability
#
#   B — Outer-CV vs half-sample shared-selection stability
#       All 23,740 genes shown
#       403 outer-stable genes identified
#       final 24-feature state highlighted
#
#   C — Coefficient-direction reproducibility among the 403 outer-stable genes
#       Outer-model median nonzero shared coefficient
#       vs half-sample median nonzero shared coefficient
#       final 24-feature state highlighted
#
# IMPORTANT
#
#   - SAVED STABILITY TABLES ONLY
#   - NO model fitting
#   - NO glmnet
#   - NO resampling
#   - NO half-sample rerun
#   - NO coefficient recalculation from raw expression
#   - NO permutation rerun
#   - Final 24 SHA verified
#
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.4 — SUPPLEMENTARY FIGURE S4\n",
    "EXTENDED STABILITY AND RESAMPLING-REFINEMENT DIAGNOSTICS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. FROZEN INPUT PATHS
# ==================================================================================================

OUTER_DIR <- file.path(
    PROJECT_ROOT,
    "06_models",
    "stability_v1"
)

HALF_DIR <- file.path(
    PROJECT_ROOT,
    "07_stability",
    "complementary_half_sample_v1"
)

FINAL_DIR <- file.path(
    PROJECT_ROOT,
    "07_stability",
    "final_stable_state_v1"
)

OUTER_ALL_PATH <- file.path(
    OUTER_DIR,
    "step05a_all_gene_stability_v1.csv"
)

OUTER_STABLE_PATH <- file.path(
    OUTER_DIR,
    "step05a_stable_shared_genes_v1.csv"
)

OUTER_SUMMARY_PATH <- file.path(
    OUTER_DIR,
    "step05a_stability_summary_v1.csv"
)

HALF_ALL_PATH <- file.path(
    HALF_DIR,
    "step06b_half_sample_all_gene_stability_v1.csv"
)

HALF_STABLE_PATH <- file.path(
    HALF_DIR,
    "step06b_half_sample_candidate_shared_v1.csv"
)

HALF_SUMMARY_PATH <- file.path(
    HALF_DIR,
    "step06b_half_sample_stability_summary_v1.csv"
)

FINAL_PATH <- file.path(
    FINAL_DIR,
    "step06d_FINAL_stable_shared_state_v1.csv"
)

FINAL_EVIDENCE_PATH <- file.path(
    FINAL_DIR,
    "step06d_final_gene_stability_evidence_v1.csv"
)

required_inputs <- c(
    OUTER_ALL_PATH,
    OUTER_STABLE_PATH,
    OUTER_SUMMARY_PATH,
    HALF_ALL_PATH,
    HALF_STABLE_PATH,
    HALF_SUMMARY_PATH,
    FINAL_PATH,
    FINAL_EVIDENCE_PATH
)

missing_inputs <- required_inputs[
    !file.exists(
        required_inputs
    )
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required frozen stability artifact(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat(
    "Required frozen stability artifacts: PASS\n"
)

# ==================================================================================================
# 5. FINAL 24 SHA256
# ==================================================================================================

EXPECTED_FINAL_SHA256 <-
    "a2bd0c8b26ee3380d2d0fb3ab1f39365b124f0ce550a753a9b59e164f3142330"

observed_final_sha <- digest::digest(
    file = FINAL_PATH,
    algo = "sha256",
    serialize = FALSE
)

if (!identical(
    observed_final_sha,
    EXPECTED_FINAL_SHA256
)) {

    stop(
        "Canonical final 24-feature SHA256 mismatch.\n",
        "Expected: ",
        EXPECTED_FINAL_SHA256,
        "\nObserved: ",
        observed_final_sha
    )
}

cat(
    "Canonical final 24-feature SHA256: PASS\n"
)

# ==================================================================================================
# 6. LOAD
# ==================================================================================================

outer_all <- read.csv(
    OUTER_ALL_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

outer_stable <- read.csv(
    OUTER_STABLE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

outer_summary <- read.csv(
    OUTER_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

half_all <- read.csv(
    HALF_ALL_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

half_stable <- read.csv(
    HALF_STABLE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

half_summary <- read.csv(
    HALF_SUMMARY_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

final24 <- read.csv(
    FINAL_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

final_evidence <- read.csv(
    FINAL_EVIDENCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 7. REQUIRED SCHEMAS
# ==================================================================================================

required_outer_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "shared_selection_count",
    "shared_selection_frequency",
    "shared_dominant_sign",
    "shared_sign_consistency",
    "shared_median_nonzero_beta",
    "stable_shared"
)

missing_outer_cols <- setdiff(
    required_outer_cols,
    names(outer_all)
)

if (length(missing_outer_cols) > 0L) {

    stop(
        "Outer all-gene stability table missing field(s): ",
        paste(
            missing_outer_cols,
            collapse = ", "
        )
    )
}

required_half_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "shared_selection_count",
    "shared_selection_probability",
    "shared_sign_consistency",
    "shared_median_nonzero_beta",
    "candidate_stable_shared",
    "stable_in_25_outer_models"
)

missing_half_cols <- setdiff(
    required_half_cols,
    names(half_all)
)

if (length(missing_half_cols) > 0L) {

    stop(
        "Half-sample all-gene stability table missing field(s): ",
        paste(
            missing_half_cols,
            collapse = ", "
        )
    )
}

if (!"ensemblID_stable" %in% names(final24)) {

    stop(
        "Final 24-feature table lacks ensemblID_stable."
    )
}

if (!all(
    c(
        "metric",
        "value"
    ) %in%
        names(outer_summary)
)) {

    stop(
        "Outer stability summary schema changed."
    )
}

if (!all(
    c(
        "metric",
        "value"
    ) %in%
        names(half_summary)
)) {

    stop(
        "Half-sample stability summary schema changed."
    )
}

cat(
    "Frozen stability schemas: PASS\n"
)

# ==================================================================================================
# 8. LOGICAL NORMALIZATION
# ==================================================================================================

normalize_logical <- function(x) {

    if (is.logical(x)) {
        return(x)
    }

    z <- toupper(
        trimws(
            as.character(x)
        )
    )

    out <- rep(
        NA,
        length(z)
    )

    out[
        z %in%
            c(
                "TRUE",
                "T",
                "1",
                "YES",
                "Y"
            )
    ] <- TRUE

    out[
        z %in%
            c(
                "FALSE",
                "F",
                "0",
                "NO",
                "N"
            )
    ] <- FALSE

    if (anyNA(out)) {

        stop(
            "Unexpected logical encoding: ",
            paste(
                unique(
                    z[
                        is.na(out)
                    ]
                ),
                collapse = ", "
            )
        )
    }

    out
}

outer_all$stable_shared <- normalize_logical(
    outer_all$stable_shared
)

half_all$candidate_stable_shared <- normalize_logical(
    half_all$candidate_stable_shared
)

half_all$stable_in_25_outer_models <- normalize_logical(
    half_all$stable_in_25_outer_models
)

# ==================================================================================================
# 9. BASIC 23,740-GENE INTEGRITY
# ==================================================================================================

if (
    nrow(outer_all) != 23740L ||
    nrow(half_all) != 23740L
) {

    stop(
        "Expected 23,740 genes in both stability universes."
    )
}

if (
    anyDuplicated(
        outer_all$ensemblID_stable
    ) > 0L ||
    anyDuplicated(
        half_all$ensemblID_stable
    ) > 0L
) {

    stop(
        "Duplicated stable Ensembl ID detected."
    )
}

if (!setequal(
    outer_all$ensemblID_stable,
    half_all$ensemblID_stable
)) {

    stop(
        "Outer and half-sample gene universes disagree."
    )
}

if (
    nrow(outer_stable) != 403L ||
    nrow(half_stable) != 24L ||
    nrow(final24) != 24L
) {

    stop(
        "Frozen stable-set sizes changed."
    )
}

cat(
    "Gene universe / candidate sizes: 23,740 / 403 / 24 / 24 — PASS\n"
)

# ==================================================================================================
# 10. SUMMARY METRIC HELPER
# ==================================================================================================

get_metric <- function(
    x,
    metric_name
) {

    idx <- which(
        trimws(
            as.character(
                x$metric
            )
        ) ==
            metric_name
    )

    if (length(idx) != 1L) {

        stop(
            "Could not uniquely retrieve metric: ",
            metric_name
        )
    }

    value <- as.numeric(
        x$value[idx]
    )

    if (
        length(value) != 1L ||
        !is.finite(value)
    ) {

        stop(
            "Non-finite metric: ",
            metric_name
        )
    }

    value
}

# ==================================================================================================
# 11. OUTER-STABILITY ANCHORS
# ==================================================================================================

OUTER_MODELS <- get_metric(
    outer_summary,
    "Number_outer_models"
)

OUTER_EVER <- get_metric(
    outer_summary,
    "Genes_selected_shared_at_least_once"
)

OUTER_20PLUS <- get_metric(
    outer_summary,
    "Genes_selected_shared_20_of_25_or_more"
)

OUTER_STABLE_N <- get_metric(
    outer_summary,
    "Stable_shared_genes"
)

OUTER_STABLE_DEV <- get_metric(
    outer_summary,
    "Stable_deviation_genes"
)

if (
    OUTER_MODELS != 25 ||
    OUTER_EVER != 6131 ||
    OUTER_20PLUS != 403 ||
    OUTER_STABLE_N != 403 ||
    OUTER_STABLE_DEV != 0
) {

    stop(
        "Frozen Step05A stability anchors changed."
    )
}

if (
    sum(
        outer_all$shared_selection_frequency > 0
    ) != 6131L ||
    sum(
        outer_all$stable_shared
    ) != 403L
) {

    stop(
        "Outer all-gene stability table disagrees with frozen summary."
    )
}

cat(
    "Outer-model stability anchors: PASS\n"
)

# ==================================================================================================
# 12. HALF-SAMPLE ANCHORS
# ==================================================================================================

HALF_FITS <- get_metric(
    half_summary,
    "Half_sample_fits"
)

HALF_EVER <- get_metric(
    half_summary,
    "Genes_selected_shared_at_least_once"
)

HALF_STABLE_N <- get_metric(
    half_summary,
    "Candidate_stable_shared_genes"
)

HALF_STABLE_DEV <- get_metric(
    half_summary,
    "Candidate_stable_deviation_genes"
)

HALF_OUTER_N <- get_metric(
    half_summary,
    "Outer_25fit_candidate_genes"
)

HALF_OVERLAP <- get_metric(
    half_summary,
    "Overlap_half_sample_vs_outer"
)

HALF_UNION <- get_metric(
    half_summary,
    "Union_half_sample_vs_outer"
)

HALF_JACCARD <- get_metric(
    half_summary,
    "Jaccard_half_sample_vs_outer"
)

if (
    HALF_FITS != 200 ||
    HALF_EVER != 13592 ||
    HALF_STABLE_N != 24 ||
    HALF_STABLE_DEV != 0 ||
    HALF_OUTER_N != 403 ||
    HALF_OVERLAP != 24 ||
    HALF_UNION != 403
) {

    stop(
        "Frozen Step06B stability anchors changed."
    )
}

if (
    abs(
        HALF_JACCARD -
        0.05955335
    ) >
        5e-7
) {

    stop(
        "Frozen outer-vs-half Jaccard value changed."
    )
}

if (
    sum(
        half_all$shared_selection_probability > 0
    ) != 13592L ||
    sum(
        half_all$candidate_stable_shared
    ) != 24L
) {

    stop(
        "Half-sample all-gene table disagrees with frozen summary."
    )
}

cat(
    "Complementary half-sample stability anchors: PASS\n"
)

# ==================================================================================================
# 13. FINAL-24 SET INTEGRITY
# ==================================================================================================

OUTER_IDS <- as.character(
    outer_stable$ensemblID_stable
)

HALF_IDS <- as.character(
    half_stable$ensemblID_stable
)

FINAL_IDS <- as.character(
    final24$ensemblID_stable
)

if (
    length(
        unique(FINAL_IDS)
    ) != 24L
) {

    stop(
        "Final state does not contain 24 unique stable Ensembl IDs."
    )
}

if (!setequal(
    FINAL_IDS,
    intersect(
        OUTER_IDS,
        HALF_IDS
    )
)) {

    stop(
        "Final 24 is not the frozen outer-stable x half-sample-stable intersection."
    )
}

if (
    !all(
        FINAL_IDS %in%
            OUTER_IDS
    ) ||
    !all(
        FINAL_IDS %in%
            HALF_IDS
    )
) {

    stop(
        "Final 24 stability-set membership failed."
    )
}

cat(
    "Final 24 = outer-stable ∩ half-sample-stable — PASS\n"
)

# ==================================================================================================
# 14. MERGE OUTER + HALF STABILITY TABLES
# ==================================================================================================

outer_keep <- outer_all[
    ,
    c(
        "ensemblID_stable",
        "gene_symbol",
        "gene_type",
        "shared_selection_count",
        "shared_selection_frequency",
        "shared_dominant_sign",
        "shared_sign_consistency",
        "shared_median_nonzero_beta",
        "stable_shared"
    ),
    drop = FALSE
]

names(outer_keep)[
    names(outer_keep) !=
        "ensemblID_stable" &
    names(outer_keep) !=
        "gene_symbol" &
    names(outer_keep) !=
        "gene_type"
] <- paste0(
    "outer_",
    names(outer_keep)[
        names(outer_keep) !=
            "ensemblID_stable" &
        names(outer_keep) !=
            "gene_symbol" &
        names(outer_keep) !=
            "gene_type"
    ]
)

half_keep <- half_all[
    ,
    c(
        "ensemblID_stable",
        "shared_selection_count",
        "shared_selection_probability",
        "shared_sign_consistency",
        "shared_median_nonzero_beta",
        "candidate_stable_shared",
        "stable_in_25_outer_models"
    ),
    drop = FALSE
]

names(half_keep)[
    names(half_keep) !=
        "ensemblID_stable"
] <- paste0(
    "half_",
    names(half_keep)[
        names(half_keep) !=
            "ensemblID_stable"
    ]
)

stability <- merge(
    outer_keep,
    half_keep,
    by = "ensemblID_stable",
    all = FALSE,
    sort = FALSE
)

if (nrow(stability) != 23740L) {

    stop(
        "Outer/half stability merge did not return 23,740 genes."
    )
}

# ==================================================================================================
# 15. FINAL-STATE FLAGS
# ==================================================================================================

stability$final24 <- (
    stability$ensemblID_stable %in%
        FINAL_IDS
)

stability$outer_stable_only <- (
    stability$outer_stable_shared &
    !stability$final24
)

stability$stability_class <- "Other genes"

stability$stability_class[
    stability$outer_stable_only
] <- "Outer-stable only"

stability$stability_class[
    stability$final24
] <- "Final 24"

stability$stability_class <- factor(
    stability$stability_class,
    levels = c(
        "Other genes",
        "Outer-stable only",
        "Final 24"
    )
)

if (
    sum(
        stability$stability_class ==
            "Outer-stable only"
    ) != 379L ||
    sum(
        stability$stability_class ==
            "Final 24"
    ) != 24L
) {

    stop(
        "403 outer-stable genes did not partition into 379 nonfinal + 24 final."
    )
}

cat(
    "Outer-stable refinement: 403 = 379 nonfinal + 24 final — PASS\n"
)

# ==================================================================================================
# 16. FINAL-24 SELECTION / SIGN CHECKS
# ==================================================================================================

final_rows <- stability[
    stability$final24,
    ,
    drop = FALSE
]

if (
    min(
        final_rows$outer_shared_selection_frequency
    ) <
        0.80
) {

    stop(
        "A final gene failed the outer selection-frequency threshold."
    )
}

if (
    min(
        final_rows$half_shared_selection_probability
    ) <
        0.80
) {

    stop(
        "A final gene failed the half-sample selection-probability threshold."
    )
}

if (
    min(
        final_rows$half_shared_sign_consistency,
        na.rm = TRUE
    ) <
        0.90
) {

    stop(
        "A final gene failed the half-sample sign-consistency threshold."
    )
}

# ==================================================================================================
# 17. DIRECTION AGREEMENT CHECK
# ==================================================================================================

outer_beta_final <- final_rows$outer_shared_median_nonzero_beta
half_beta_final <- final_rows$half_shared_median_nonzero_beta

if (
    any(
        !is.finite(
            outer_beta_final
        )
    ) ||
    any(
        !is.finite(
            half_beta_final
        )
    )
) {

    stop(
        "Non-finite final-24 median coefficient detected."
    )
}

direction_agreement <- (
    sign(
        outer_beta_final
    ) ==
        sign(
            half_beta_final
        )
)

if (
    sum(
        direction_agreement
    ) != 24L
) {

    stop(
        "Final 24 direction agreement is not 24/24."
    )
}

cat(
    "Final 24 outer-vs-half coefficient direction agreement: 24/24 — PASS\n"
)

# ==================================================================================================
# 18. PANEL A — STABILITY SURVIVAL CURVES
#
# This is a visualization of the already-saved selection frequencies/probabilities.
# It does NOT rerun resampling.
# ==================================================================================================

thresholds <- seq(
    0,
    1,
    by = 0.005
)

outer_survival <- vapply(
    thresholds,
    function(t) {
        sum(
            stability$outer_shared_selection_frequency >=
                t
        )
    },
    FUN.VALUE = integer(1)
)

half_survival <- vapply(
    thresholds,
    function(t) {
        sum(
            stability$half_shared_selection_probability >=
                t
        )
    },
    FUN.VALUE = integer(1)
)

panel_A <- rbind(

    data.frame(
        threshold = thresholds,
        n_genes = outer_survival,
        resampling = "Outer-CV models",
        stringsAsFactors = FALSE
    ),

    data.frame(
        threshold = thresholds,
        n_genes = half_survival,
        resampling = "Complementary half-samples",
        stringsAsFactors = FALSE
    )
)

panel_A$resampling <- factor(
    panel_A$resampling,
    levels = c(
        "Outer-CV models",
        "Complementary half-samples"
    )
)

# ==================================================================================================
# 19. PANEL B — SELECTION-STABILITY SCATTER
# ==================================================================================================

panel_B <- stability[
    ,
    c(
        "ensemblID_stable",
        "gene_symbol",
        "outer_shared_selection_frequency",
        "half_shared_selection_probability",
        "stability_class"
    ),
    drop = FALSE
]

# ==================================================================================================
# 20. PANEL C — COEFFICIENT CONCORDANCE AMONG OUTER-STABLE GENES
# ==================================================================================================

panel_C <- stability[
    stability$outer_stable_shared,
    c(
        "ensemblID_stable",
        "gene_symbol",
        "outer_shared_median_nonzero_beta",
        "half_shared_median_nonzero_beta",
        "final24"
    ),
    drop = FALSE
]

panel_C <- panel_C[
    is.finite(
        panel_C$outer_shared_median_nonzero_beta
    ) &
    is.finite(
        panel_C$half_shared_median_nonzero_beta
    ),
    ,
    drop = FALSE
]

if (
    sum(
        panel_C$final24
    ) != 24L
) {

    stop(
        "All final 24 genes must be present in coefficient-concordance panel."
    )
}

panel_C$class <- ifelse(
    panel_C$final24,
    "Final 24",
    "Other outer-stable genes"
)

panel_C$class <- factor(
    panel_C$class,
    levels = c(
        "Other outer-stable genes",
        "Final 24"
    )
)

# Descriptive coefficient correlation among available outer-stable genes.
COEF_COR <- stats::cor(
    panel_C$outer_shared_median_nonzero_beta,
    panel_C$half_shared_median_nonzero_beta,
    method = "pearson",
    use = "complete.obs"
)

if (!is.finite(COEF_COR)) {

    stop(
        "Could not calculate coefficient-concordance correlation."
    )
}

# ==================================================================================================
# 21. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS4_Stability_Refinement_TNR_v1"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "FigureS4_Extended_Stability_Refinement_TimesNewRoman_v1.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "FigureS4_Extended_Stability_Refinement_TimesNewRoman_v1.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "FigureS4_Extended_Stability_Refinement_TimesNewRoman_v1.png"
)

MERGED_SOURCE_PATH <- file.path(
    SOURCE_DIR,
    "FigureS4_outer_half_merged_stability_source_v1.csv"
)

SURVIVAL_SOURCE_PATH <- file.path(
    SOURCE_DIR,
    "FigureS4_stability_survival_source_v1.csv"
)

COEF_SOURCE_PATH <- file.path(
    SOURCE_DIR,
    "FigureS4_coefficient_concordance_source_v1.csv"
)

SUMMARY_OUT_PATH <- file.path(
    OUT_DIR,
    "FigureS4_generation_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "FigureS4_generation_manifest_v1.csv"
)

write.csv(
    stability,
    MERGED_SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    panel_A,
    SURVIVAL_SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    panel_C,
    COEF_SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 22. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.2,
            hjust = 0,
            margin = ggplot2::margin(
                b = 5
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.3
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.1,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.8
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.5
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            8,
            9,
            8,
            9
        )
    )

# ==================================================================================================
# 23. PANEL A — STABILITY SURVIVAL
# ==================================================================================================

resampling_colours <- c(
    "Outer-CV models" = "#4E79A7",
    "Complementary half-samples" = "#E15759"
)

pA <- ggplot2::ggplot(
    panel_A,
    ggplot2::aes(
        x = threshold,
        y = n_genes,
        colour = resampling
    )
) +
    ggplot2::geom_line(
        linewidth = 0.88
    ) +
    ggplot2::geom_vline(
        xintercept = 0.80,
        linetype = "dashed",
        linewidth = 0.55,
        colour = "#555555"
    ) +
    ggplot2::annotate(
        "point",
        x = 0.80,
        y = 403,
        colour = "#4E79A7",
        size = 2.6
    ) +
    ggplot2::annotate(
        "point",
        x = 0.80,
        y = 24,
        colour = "#E15759",
        size = 2.6
    ) +
    ggplot2::annotate(
        "text",
        x = 0.825,
        y = 403,
        label = "403 outer-stable",
        hjust = 0,
        vjust = -0.45,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35,
        colour = "#4E79A7"
    ) +
    ggplot2::annotate(
        "text",
        x = 0.825,
        y = 24,
        label = "24 half-sample stable",
        hjust = 0,
        vjust = -0.45,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35,
        colour = "#B74347"
    ) +
    ggplot2::scale_colour_manual(
        values = resampling_colours
    ) +
    ggplot2::scale_y_log10(
        breaks = c(
            1,
            10,
            100,
            1000,
            10000
        ),
        labels = c(
            "1",
            "10",
            "100",
            "1,000",
            "10,000"
        )
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            0,
            1
        ),
        breaks = c(
            0,
            0.2,
            0.4,
            0.6,
            0.8,
            1.0
        )
    ) +
    ggplot2::labs(
        title = "A  Stability-survival across the gene universe",
        x = "Minimum shared-selection probability",
        y = "Genes retained, n (log scale)",
        colour = "Resampling framework"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.direction = "horizontal"
    )

# ==================================================================================================
# 24. PANEL B — OUTER vs HALF-SAMPLE SELECTION
# ==================================================================================================

stability_colours <- c(
    "Other genes" = "#CFCFCF",
    "Outer-stable only" = "#7F8C8D",
    "Final 24" = "#D1495B"
)

stability_sizes <- c(
    "Other genes" = 0.75,
    "Outer-stable only" = 1.25,
    "Final 24" = 2.25
)

pB <- ggplot2::ggplot(
    panel_B,
    ggplot2::aes(
        x = outer_shared_selection_frequency,
        y = half_shared_selection_probability,
        colour = stability_class,
        size = stability_class
    )
) +
    ggplot2::geom_vline(
        xintercept = 0.80,
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#666666"
    ) +
    ggplot2::geom_hline(
        yintercept = 0.80,
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#666666"
    ) +
    ggplot2::geom_point(
        alpha = 0.62
    ) +
    ggplot2::annotate(
        "text",
        x = 0.98,
        y = 0.82,
        label = "Final intersection = 24",
        hjust = 1,
        vjust = 0,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.40,
        colour = "#A92F40"
    ) +
    ggplot2::scale_colour_manual(
        values = stability_colours,
        drop = FALSE
    ) +
    ggplot2::scale_size_manual(
        values = stability_sizes,
        drop = FALSE,
        guide = "none"
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            0,
            1
        ),
        breaks = seq(
            0,
            1,
            by = 0.2
        )
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            0,
            1
        ),
        breaks = seq(
            0,
            1,
            by = 0.2
        )
    ) +
    ggplot2::coord_equal() +
    ggplot2::labs(
        title = "B  Outer-CV vs half-sample stability",
        x = "Outer-CV shared-selection frequency",
        y = "Half-sample shared-selection probability",
        colour = "Stability class"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.direction = "horizontal"
    )

# ==================================================================================================
# 25. PANEL C — COEFFICIENT DIRECTION / MAGNITUDE CONCORDANCE
# ==================================================================================================

coef_colours <- c(
    "Other outer-stable genes" = "#9D9D9D",
    "Final 24" = "#2A9D8F"
)

coef_limits <- range(
    c(
        panel_C$outer_shared_median_nonzero_beta,
        panel_C$half_shared_median_nonzero_beta
    ),
    finite = TRUE
)

coef_abs_max <- max(
    abs(
        coef_limits
    )
)

coef_abs_max <- ceiling(
    coef_abs_max *
        10
) / 10

pC <- ggplot2::ggplot(
    panel_C,
    ggplot2::aes(
        x = outer_shared_median_nonzero_beta,
        y = half_shared_median_nonzero_beta,
        colour = class
    )
) +
    ggplot2::geom_hline(
        yintercept = 0,
        linewidth = 0.35,
        colour = "#B8B8B8"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.35,
        colour = "#B8B8B8"
    ) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.52,
        colour = "#555555"
    ) +
    ggplot2::geom_point(
        ggplot2::aes(
            size = class
        ),
        alpha = 0.78
    ) +
    ggplot2::annotate(
        "text",
        x = -coef_abs_max * 0.95,
        y = coef_abs_max * 0.92,
        label = paste0(
            "Final 24 direction agreement = 24/24",
            "\n403 outer-stable genes"
        ),
        hjust = 0,
        vjust = 1,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35,
        lineheight = 1.05
    ) +
    ggplot2::scale_colour_manual(
        values = coef_colours
    ) +
    ggplot2::scale_size_manual(
        values = c(
            "Other outer-stable genes" = 1.25,
            "Final 24" = 2.45
        ),
        guide = "none"
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            -coef_abs_max,
            coef_abs_max
        )
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            -coef_abs_max,
            coef_abs_max
        )
    ) +
    ggplot2::coord_equal() +
    ggplot2::labs(
        title = "C  Coefficient reproducibility in outer-stable genes",
        x = "Outer-CV median nonzero shared coefficient",
        y = "Half-sample median nonzero shared coefficient",
        colour = "Gene set"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.direction = "horizontal"
    )

# ==================================================================================================
# 26. DRAW FIGURE
#
# Top panel spans full width.
# Bottom row contains Panels B and C.
# ==================================================================================================

draw_figure_s4 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 2,
        ncol = 2,
        widths = grid::unit(
            c(
                1,
                1
            ),
            "null"
        ),
        heights = grid::unit(
            c(
                0.90,
                1.10
            ),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1:2
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 27. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 6.0,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure_s4()

grDevices::dev.off()

# ==================================================================================================
# 28. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 6.0,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure_s4()

grDevices::dev.off()

# ==================================================================================================
# 29. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 6.0,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure_s4()

grDevices::dev.off()

# ==================================================================================================
# 30. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    MERGED_SOURCE_PATH,
    SURVIVAL_SOURCE_PATH,
    COEF_SOURCE_PATH
)

if (!all(
    file.exists(
        outputs
    )
)) {

    stop(
        "One or more Supplementary Figure S4 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (any(
    is.na(
        output_info$size
    ) |
    output_info$size <= 0
)) {

    stop(
        "One or more Supplementary Figure S4 outputs are zero-byte."
    )
}

cat(
    "Supplementary Figure S4 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 31. GENERATION SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11D.4 — SUPPLEMENTARY FIGURE S4",
    "EXTENDED STABILITY AND RESAMPLING-REFINEMENT DIAGNOSTICS",
    "==============================================================================================================",

    "",

    "Frozen saved stability outputs only.",

    "No model fitting, resampling, half-sample analysis, or permutation rerun.",

    "",

    "Gene universe = 23,740",

    "Outer-CV models = 25",

    "Outer shared genes selected at least once = 6,131",

    "Outer stable shared genes = 403",

    "Outer stable deviation genes = 0",

    "",

    "Complementary half-sample fits = 200",

    "Half-sample shared genes selected at least once = 13,592",

    "Half-sample stable shared genes = 24",

    "Half-sample stable deviation genes = 0",

    "",

    "Outer stable / half-sample stable overlap = 24",

    "Outer/half union = 403",

    paste0(
        "Jaccard = ",
        sprintf(
            "%.6f",
            HALF_JACCARD
        )
    ),

    "Final intersection = 24",

    "Final 24 direction agreement = 24/24",

    paste0(
        "Descriptive coefficient Pearson r among plotted outer-stable genes = ",
        sprintf(
            "%.4f",
            COEF_COR
        )
    ),

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_OUT_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 32. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_OUT_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 33. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSUPPLEMENTARY FIGURE S4 GENERATION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Frozen stability sources: PASS\n"
)

cat(
    "Stability analysis rerun: NO\n"
)

cat(
    "Final 24 SHA256: PASS\n"
)

cat(
    "\nOuter stability:\n"
)

cat(
    "Selected at least once: 6131\n"
)

cat(
    "Stable shared: 403\n"
)

cat(
    "Stable deviation: 0\n"
)

cat(
    "\nComplementary half-sample stability:\n"
)

cat(
    "Fits: 200\n"
)

cat(
    "Selected at least once: 13592\n"
)

cat(
    "Stable shared: 24\n"
)

cat(
    "Stable deviation: 0\n"
)

cat(
    "\nRefinement:\n"
)

cat(
    "Outer stable -> final: 403 -> 24\n"
)

cat(
    "Half-sample stable / final intersection: 24 / 24\n"
)

cat(
    "Direction agreement: 24 / 24\n"
)

cat(
    "Outer-only stable after refinement: 379\n"
)

cat(
    "Jaccard outer vs half-sample stable sets: ",
    sprintf(
        "%.6f",
        HALF_JACCARD
    ),
    "\n",
    sep = ""
)

cat(
    "Descriptive coefficient correlation among plotted outer-stable genes: ",
    sprintf(
        "%.4f",
        COEF_COR
    ),
    "\n",
    sep = ""
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.4 COMPLETE — SUPPLEMENTARY FIGURE S4 GENERATED FROM FROZEN STABILITY OUTPUTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11D.4 — SUPPLEMENTARY FIGURE S4
EXTENDED STABILITY AND RESAMPLING-REFINEMENT DIAGNOSTICS
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Required frozen stability artifacts: PASS
Canonical final 24-feature SHA256: PASS
Frozen stability schemas: PASS
Gene universe / candidate sizes: 23,740 / 403 / 24 / 24 — PASS
Outer-model stability anchors: PASS
Complementary half-sample stability anchors: PASS
Final 24 = outer-stable ∩ half-sample-stable — PASS
Outer-stable refinement: 403 = 379 nonfinal + 24 final — PASS
Final 24 outer-vs-half coefficient direction agreement: 24/24 — PASS


Warning message in ggplot2::scale_y_log10(breaks = c(1, 10, 100, 1000, 10000), labels = c("1", :
"log-10 transformation introduced infinite values."


agg_record_f7453e341ee 
                     2

Warning message in ggplot2::scale_y_log10(breaks = c(1, 10, 100, 1000, 10000), labels = c("1", :
"log-10 transformation introduced infinite values."


agg_record_f7453e341ee 
                     2

Warning message in ggplot2::scale_y_log10(breaks = c(1, 10, 100, 1000, 10000), labels = c("1", :
"log-10 transformation introduced infinite values."


agg_record_f7453e341ee 
                     2

Supplementary Figure S4 outputs created and non-zero: PASS

SUPPLEMENTARY FIGURE S4 GENERATION SUMMARY
Frozen stability sources: PASS
Stability analysis rerun: NO
Final 24 SHA256: PASS

Outer stability:
Selected at least once: 6131
Stable shared: 403
Stable deviation: 0

Complementary half-sample stability:
Fits: 200
Selected at least once: 13592
Stable shared: 24
Stable deviation: 0

Refinement:
Outer stable -> final: 403 -> 24
Half-sample stable / final intersection: 24 / 24
Direction agreement: 24 / 24
Outer-only stable after refinement: 379
Jaccard outer vs half-sample stable sets: 0.059553
Descriptive coefficient correlation among plotted outer-stable genes: 0.9726

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS4_Stability_Refinement_TNR_v1/FigureS4_Extended_Stability_Refinement_TimesNewRoman_v1.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS4_Stability_Refinement_TNR_v1/FigureS4_Extended_Stability_Refinement_Tim

In [11]:
# ==================================================================================================
# STEP 11D.4S — SUPPLEMENTARY FIGURE S4 — FINAL TITLE-CORRECTED VERSION
# EXTENDED STABILITY AND RESAMPLING-REFINEMENT DIAGNOSTICS
# --------------------------------------------------------------------------------------------------
# VISUAL-ONLY REVISION OF VALIDATED S4 v2
#
# ONLY CHANGE
#   Panel C title shortened to eliminate right-edge clipping.
#
# UNCHANGED
#   - 23,740-gene stability universe
#   - 403 outer-stable genes
#   - 24 half-sample-stable genes
#   - 24 final intersection
#   - 379 outer-stable nonfinal genes
#   - 24/24 coefficient-direction agreement
#   - coefficient concordance r = 0.9726
#   - all plotted source values
#
# NO:
#   - model fitting
#   - outer CV rerun
#   - half-sample resampling
#   - coefficient reconstruction
#   - permutation rerun
#
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.4S — SUPPLEMENTARY FIGURE S4 — FINAL TITLE-CORRECTED VERSION\n",
    "EXTENDED STABILITY AND RESAMPLING-REFINEMENT DIAGNOSTICS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        project_id
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. VALIDATED S4 v2 SOURCE FILES
# ==================================================================================================

V2_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS4_Stability_Refinement_TNR_v2"
)

V2_SOURCE_DIR <- file.path(
    V2_DIR,
    "source_data"
)

MERGED_SOURCE_PATH <- file.path(
    V2_SOURCE_DIR,
    "FigureS4_outer_half_merged_stability_source_v2.csv"
)

SURVIVAL_SOURCE_PATH <- file.path(
    V2_SOURCE_DIR,
    "FigureS4_stability_survival_source_v2.csv"
)

COEF_SOURCE_PATH <- file.path(
    V2_SOURCE_DIR,
    "FigureS4_coefficient_concordance_source_v2.csv"
)

FINAL_PATH <- file.path(
    PROJECT_ROOT,
    "07_stability",
    "final_stable_state_v1",
    "step06d_FINAL_stable_shared_state_v1.csv"
)

required_inputs <- c(
    MERGED_SOURCE_PATH,
    SURVIVAL_SOURCE_PATH,
    COEF_SOURCE_PATH,
    FINAL_PATH
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required validated S4 v2 source file(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat(
    "Validated Figure S4 v2 source data: PASS\n"
)

# ==================================================================================================
# 5. FINAL 24 SHA256
# ==================================================================================================

EXPECTED_FINAL_SHA256 <-
    "a2bd0c8b26ee3380d2d0fb3ab1f39365b124f0ce550a753a9b59e164f3142330"

observed_final_sha <- digest::digest(
    file = FINAL_PATH,
    algo = "sha256",
    serialize = FALSE
)

if (!identical(
    observed_final_sha,
    EXPECTED_FINAL_SHA256
)) {

    stop(
        "Canonical final 24-feature SHA256 mismatch.\n",
        "Expected: ",
        EXPECTED_FINAL_SHA256,
        "\nObserved: ",
        observed_final_sha
    )
}

cat(
    "Canonical final 24-feature SHA256: PASS\n"
)

# ==================================================================================================
# 6. LOAD
# ==================================================================================================

stability <- read.csv(
    MERGED_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

panel_A <- read.csv(
    SURVIVAL_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

panel_C <- read.csv(
    COEF_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 7. LOGICAL NORMALIZATION
# ==================================================================================================

normalize_logical <- function(x) {

    if (is.logical(x)) {
        return(x)
    }

    z <- toupper(
        trimws(
            as.character(x)
        )
    )

    out <- rep(
        NA,
        length(z)
    )

    out[
        z %in% c(
            "TRUE",
            "T",
            "1",
            "YES",
            "Y"
        )
    ] <- TRUE

    out[
        z %in% c(
            "FALSE",
            "F",
            "0",
            "NO",
            "N"
        )
    ] <- FALSE

    if (anyNA(out)) {
        stop(
            "Unexpected logical encoding: ",
            paste(
                unique(
                    z[
                        is.na(out)
                    ]
                ),
                collapse = ", "
            )
        )
    }

    out
}

stability$outer_stable_shared <- normalize_logical(
    stability$outer_stable_shared
)

stability$half_candidate_stable_shared <- normalize_logical(
    stability$half_candidate_stable_shared
)

stability$final24 <- normalize_logical(
    stability$final24
)

stability$outer_stable_only <- normalize_logical(
    stability$outer_stable_only
)

panel_C$final24 <- normalize_logical(
    panel_C$final24
)

# ==================================================================================================
# 8. STRICT FROZEN VALIDATION
# ==================================================================================================

if (nrow(stability) != 23740L) {
    stop("Expected 23,740 genes.")
}

if (
    sum(stability$outer_stable_shared) != 403L ||
    sum(stability$half_candidate_stable_shared) != 24L ||
    sum(stability$final24) != 24L ||
    sum(stability$outer_stable_only) != 379L
) {

    stop(
        "Frozen stability-set counts changed."
    )
}

final_rows <- stability[
    stability$final24,
    ,
    drop = FALSE
]

if (
    nrow(final_rows) != 24L ||
    sum(
        sign(
            final_rows$outer_shared_median_nonzero_beta
        ) ==
            sign(
                final_rows$half_shared_median_nonzero_beta
            )
    ) != 24L
) {

    stop(
        "Final 24 coefficient-direction agreement changed."
    )
}

if (
    nrow(panel_C) != 403L ||
    sum(panel_C$final24) != 24L
) {

    stop(
        "Coefficient-concordance source changed."
    )
}

COEF_COR <- stats::cor(
    panel_C$outer_shared_median_nonzero_beta,
    panel_C$half_shared_median_nonzero_beta,
    method = "pearson",
    use = "complete.obs"
)

if (
    abs(
        COEF_COR -
        0.9726
    ) >
        0.0001
) {

    stop(
        "Coefficient concordance changed: ",
        COEF_COR
    )
}

cat(
    "Frozen stability refinement: 403 -> 24; direction agreement 24/24 — PASS\n"
)

cat(
    "Coefficient concordance r = ",
    sprintf(
        "%.4f",
        COEF_COR
    ),
    " — PASS\n",
    sep = ""
)

# ==================================================================================================
# 9. THRESHOLD-0.80 CHECK
# ==================================================================================================

get_survival_count <- function(
    framework
) {

    idx <- which(
        abs(
            panel_A$threshold -
            0.80
        ) <
            1e-12 &
        panel_A$resampling ==
            framework
    )

    if (length(idx) != 1L) {

        stop(
            "Could not retrieve threshold 0.80 for ",
            framework
        )
    }

    as.integer(
        panel_A$n_genes[idx]
    )
}

OUTER_AT_080 <- get_survival_count(
    "Outer-CV models"
)

HALF_AT_080 <- get_survival_count(
    "Complementary half-samples"
)

if (
    OUTER_AT_080 != 403L ||
    HALF_AT_080 != 24L
) {

    stop(
        "Threshold-0.80 stability counts changed."
    )
}

cat(
    "Threshold 0.80: 403 outer / 24 half-sample — PASS\n"
)

# ==================================================================================================
# 10. PANEL A DRAWING DATA
# ==================================================================================================

panel_A_plot <- panel_A[
    is.finite(
        panel_A$n_genes
    ) &
    panel_A$n_genes > 0,
    ,
    drop = FALSE
]

panel_A_plot$resampling <- factor(
    panel_A_plot$resampling,
    levels = c(
        "Outer-CV models",
        "Complementary half-samples"
    )
)

N_ZERO_ROWS <- (
    nrow(panel_A) -
        nrow(panel_A_plot)
)

if (N_ZERO_ROWS != 2L) {

    stop(
        "Expected exactly two zero-count survival rows."
    )
}

cat(
    "Zero-count survival rows excluded from log drawing only: 2 — PASS\n"
)

# ==================================================================================================
# 11. PANEL B DATA
# ==================================================================================================

panel_B <- stability[
    ,
    c(
        "ensemblID_stable",
        "gene_symbol",
        "outer_shared_selection_frequency",
        "half_shared_selection_probability",
        "stability_class"
    ),
    drop = FALSE
]

panel_B$stability_class <- factor(
    as.character(
        panel_B$stability_class
    ),
    levels = c(
        "Other genes",
        "Outer-stable only",
        "Final 24"
    )
)

if (anyNA(
    panel_B$stability_class
)) {
    stop("Unexpected Panel B stability class.")
}

# ==================================================================================================
# 12. PANEL C CLASS
# ==================================================================================================

panel_C$class <- factor(
    as.character(
        panel_C$class
    ),
    levels = c(
        "Other outer-stable genes",
        "Final 24"
    )
)

if (anyNA(
    panel_C$class
)) {
    stop("Unexpected Panel C class.")
}

# ==================================================================================================
# 13. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS4_Stability_Refinement_TNR_v3"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "FigureS4_Extended_Stability_Refinement_TimesNewRoman_v3.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "FigureS4_Extended_Stability_Refinement_TimesNewRoman_v3.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "FigureS4_Extended_Stability_Refinement_TimesNewRoman_v3.png"
)

MERGED_OUT_PATH <- file.path(
    SOURCE_DIR,
    "FigureS4_outer_half_merged_stability_source_v3.csv"
)

SURVIVAL_OUT_PATH <- file.path(
    SOURCE_DIR,
    "FigureS4_stability_survival_source_v3.csv"
)

COEF_OUT_PATH <- file.path(
    SOURCE_DIR,
    "FigureS4_coefficient_concordance_source_v3.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "FigureS4_generation_summary_v3.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "FigureS4_generation_manifest_v3.csv"
)

write.csv(
    stability,
    MERGED_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    panel_A,
    SURVIVAL_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    panel_C,
    COEF_OUT_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 14. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.1,
            hjust = 0,
            margin = ggplot2::margin(
                b = 5
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.2
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 8.0,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.8
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.5
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            8,
            9,
            8,
            9
        )
    )

# ==================================================================================================
# 15. PANEL A
# ==================================================================================================

resampling_colours <- c(
    "Outer-CV models" = "#4E79A7",
    "Complementary half-samples" = "#E15759"
)

pA <- ggplot2::ggplot(
    panel_A_plot,
    ggplot2::aes(
        x = threshold,
        y = n_genes,
        colour = resampling
    )
) +
    ggplot2::geom_line(
        linewidth = 0.88
    ) +
    ggplot2::geom_vline(
        xintercept = 0.80,
        linetype = "dashed",
        linewidth = 0.55,
        colour = "#555555"
    ) +
    ggplot2::annotate(
        "point",
        x = 0.80,
        y = 403,
        colour = "#4E79A7",
        size = 2.6
    ) +
    ggplot2::annotate(
        "point",
        x = 0.80,
        y = 24,
        colour = "#E15759",
        size = 2.6
    ) +
    ggplot2::annotate(
        "text",
        x = 0.825,
        y = 403,
        label = "403 outer-stable",
        hjust = 0,
        vjust = -0.45,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35,
        colour = "#4E79A7"
    ) +
    ggplot2::annotate(
        "text",
        x = 0.825,
        y = 24,
        label = "24 half-sample stable",
        hjust = 0,
        vjust = -0.45,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35,
        colour = "#B74347"
    ) +
    ggplot2::scale_colour_manual(
        values = resampling_colours
    ) +
    ggplot2::scale_y_log10(
        breaks = c(
            1,
            10,
            100,
            1000,
            10000
        ),
        labels = c(
            "1",
            "10",
            "100",
            "1,000",
            "10,000"
        )
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            0,
            1
        ),
        breaks = c(
            0,
            0.2,
            0.4,
            0.6,
            0.8,
            1.0
        )
    ) +
    ggplot2::labs(
        title = "A  Stability survival across the gene universe",
        x = "Minimum shared-selection probability",
        y = "Genes retained, n (log10 scale)",
        colour = "Resampling framework"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.direction = "horizontal"
    )

# ==================================================================================================
# 16. PANEL B
# ==================================================================================================

stability_colours <- c(
    "Other genes" = "#CFCFCF",
    "Outer-stable only" = "#7F8C8D",
    "Final 24" = "#D1495B"
)

stability_sizes <- c(
    "Other genes" = 0.72,
    "Outer-stable only" = 1.25,
    "Final 24" = 2.30
)

pB <- ggplot2::ggplot(
    panel_B,
    ggplot2::aes(
        x = outer_shared_selection_frequency,
        y = half_shared_selection_probability,
        colour = stability_class,
        size = stability_class
    )
) +
    ggplot2::geom_vline(
        xintercept = 0.80,
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#666666"
    ) +
    ggplot2::geom_hline(
        yintercept = 0.80,
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#666666"
    ) +
    ggplot2::geom_point(
        alpha = 0.62
    ) +
    ggplot2::annotate(
        "text",
        x = 0.98,
        y = 0.82,
        label = "Final intersection = 24",
        hjust = 1,
        vjust = 0,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.35,
        colour = "#A92F40"
    ) +
    ggplot2::scale_colour_manual(
        values = stability_colours,
        drop = FALSE
    ) +
    ggplot2::scale_size_manual(
        values = stability_sizes,
        drop = FALSE,
        guide = "none"
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            0,
            1
        ),
        breaks = seq(
            0,
            1,
            by = 0.2
        )
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            0,
            1
        ),
        breaks = seq(
            0,
            1,
            by = 0.2
        )
    ) +
    ggplot2::coord_equal() +
    ggplot2::labs(
        title = "B  Outer-CV vs half-sample stability",
        x = "Outer-CV shared-selection frequency",
        y = "Half-sample shared-selection probability",
        colour = "Stability class"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.direction = "horizontal"
    )

# ==================================================================================================
# 17. PANEL C — SHORT FINAL TITLE
# ==================================================================================================

coef_colours <- c(
    "Other outer-stable genes" = "#9D9D9D",
    "Final 24" = "#2A9D8F"
)

coef_abs_max <- max(
    abs(
        c(
            panel_C$outer_shared_median_nonzero_beta,
            panel_C$half_shared_median_nonzero_beta
        )
    ),
    na.rm = TRUE
)

coef_abs_max <- ceiling(
    coef_abs_max *
        10
) / 10

pC <- ggplot2::ggplot(
    panel_C,
    ggplot2::aes(
        x = outer_shared_median_nonzero_beta,
        y = half_shared_median_nonzero_beta,
        colour = class
    )
) +
    ggplot2::geom_hline(
        yintercept = 0,
        linewidth = 0.35,
        colour = "#B8B8B8"
    ) +
    ggplot2::geom_vline(
        xintercept = 0,
        linewidth = 0.35,
        colour = "#B8B8B8"
    ) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.52,
        colour = "#555555"
    ) +
    ggplot2::geom_point(
        ggplot2::aes(
            size = class
        ),
        alpha = 0.78
    ) +
    ggplot2::annotate(
        "text",
        x = -coef_abs_max * 0.94,
        y = coef_abs_max * 0.90,
        label = paste0(
            "Final 24 direction agreement = 24/24",
            "\nOuter-stable set = 403 genes"
        ),
        hjust = 0,
        vjust = 1,
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.28,
        lineheight = 1.04
    ) +
    ggplot2::scale_colour_manual(
        values = coef_colours
    ) +
    ggplot2::scale_size_manual(
        values = c(
            "Other outer-stable genes" = 1.25,
            "Final 24" = 2.45
        ),
        guide = "none"
    ) +
    ggplot2::scale_x_continuous(
        limits = c(
            -coef_abs_max,
            coef_abs_max
        )
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            -coef_abs_max,
            coef_abs_max
        )
    ) +
    ggplot2::coord_equal() +
    ggplot2::labs(
        title = "C  Coefficient agreement across resampling",
        x = "Outer-CV median shared coefficient",
        y = "Half-sample median shared coefficient",
        colour = "Gene set"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.direction = "horizontal"
    )

# ==================================================================================================
# 18. DRAW
# ==================================================================================================

draw_figure_s4 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 2,
        ncol = 2,
        widths = grid::unit(
            c(
                1,
                1.06
            ),
            "null"
        ),
        heights = grid::unit(
            c(
                0.90,
                1.10
            ),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1:2
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 19. SAVE
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 6.0,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure_s4()
grDevices::dev.off()

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 6.0,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure_s4()
grDevices::dev.off()

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 6.0,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure_s4()
grDevices::dev.off()

# ==================================================================================================
# 20. OUTPUT CHECK
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    MERGED_OUT_PATH,
    SURVIVAL_OUT_PATH,
    COEF_OUT_PATH
)

if (!all(
    file.exists(outputs)
)) {
    stop("One or more S4 v3 outputs were not created.")
}

if (any(
    is.na(file.info(outputs)$size) |
    file.info(outputs)$size <= 0
)) {
    stop("One or more S4 v3 outputs are zero-byte.")
}

cat(
    "Supplementary Figure S4 v3 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 21. SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11D.4S — SUPPLEMENTARY FIGURE S4 — FINAL TITLE-CORRECTED VERSION",
    "==============================================================================================================",

    "",

    "Generated from validated S4 v2 source data.",

    "No analytical operation was rerun.",

    "",

    "Gene universe = 23,740",
    "Outer stable shared = 403",
    "Half-sample stable shared = 24",
    "Final intersection = 24",
    "Outer-stable nonfinal = 379",
    "Final direction agreement = 24/24",

    paste0(
        "Coefficient concordance r = ",
        sprintf(
            "%.4f",
            COEF_COR
        )
    ),

    paste0(
        "Threshold 0.80: outer=",
        OUTER_AT_080,
        "; half-sample=",
        HALF_AT_080
    ),

    "Zero-count survival rows excluded from log10 plotting only = 2",

    "",

    "Visual change only:",
    "Panel C title shortened to 'Coefficient agreement across resampling'.",

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 22. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 23. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSUPPLEMENTARY FIGURE S4 v3 SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat("Validated S4 v2 sources: PASS\n")
cat("Analytical results changed: NO\n")
cat("Stability/resampling rerun: NO\n")
cat("Final 24 SHA256: PASS\n")

cat(
    "Outer / half / final / outer-only: ",
    "403 / 24 / 24 / 379\n",
    sep = ""
)

cat("Direction agreement: 24 / 24\n")

cat(
    "Coefficient concordance r: ",
    sprintf(
        "%.4f",
        COEF_COR
    ),
    "\n",
    sep = ""
)

cat(
    "Threshold 0.80 outer / half: ",
    OUTER_AT_080,
    " / ",
    HALF_AT_080,
    "\n",
    sep = ""
)

cat("Panel C title clipping corrected: YES\n")

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.4S COMPLETE — SUPPLEMENTARY FIGURE S4 v3 GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11D.4S — SUPPLEMENTARY FIGURE S4 — FINAL TITLE-CORRECTED VERSION
EXTENDED STABILITY AND RESAMPLING-REFINEMENT DIAGNOSTICS
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Validated Figure S4 v2 source data: PASS
Canonical final 24-feature SHA256: PASS
Frozen stability refinement: 403 -> 24; direction agreement 24/24 — PASS
Coefficient concordance r = 0.9726 — PASS
Threshold 0.80: 403 outer / 24 half-sample — PASS
Zero-count survival rows excluded from log drawing only: 2 — PASS


agg_record_f74e832f4c 
                    2

agg_record_f74e832f4c 
                    2

agg_record_f74e832f4c 
                    2

Supplementary Figure S4 v3 outputs created and non-zero: PASS

SUPPLEMENTARY FIGURE S4 v3 SUMMARY
Validated S4 v2 sources: PASS
Analytical results changed: NO
Stability/resampling rerun: NO
Final 24 SHA256: PASS
Outer / half / final / outer-only: 403 / 24 / 24 / 379
Direction agreement: 24 / 24
Coefficient concordance r: 0.9726
Threshold 0.80 outer / half: 403 / 24
Panel C title clipping corrected: YES

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS4_Stability_Refinement_TNR_v3/FigureS4_Extended_Stability_Refinement_TimesNewRoman_v3.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS4_Stability_Refinement_TNR_v3/FigureS4_Extended_Stability_Refinement_TimesNewRoman_v3.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS4_Stability_Refinement_TNR_v3/FigureS4_Extended_Stability_Refinement_TimesNewRoman_v3.png

STEP 11D.4S COMPLETE — SUPPLEMENTARY FIGURE S4 v3 GENERATED


In [15]:
# ==================================================================================================
# STEP 11D.5T — SUPPLEMENTARY FIGURE S5 — FINAL VISUAL LOCK
# FROZEN SCZ-MSS SCORE ARCHITECTURE AND DISCOVERY-ONLY DESCRIPTIVE BEHAVIOUR
# --------------------------------------------------------------------------------------------------
# VISUAL-ONLY REVISION OF VALIDATED S5 v3
#
# ONLY CHANGE:
#   Panel C title shortened so it is fully visible within the right-hand panel.
#
# NO:
#   - score recalculation
#   - feature selection
#   - model fitting
#   - nested CV
#   - external reanalysis
#   - coefficient re-estimation
#
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.5T — SUPPLEMENTARY FIGURE S5 — FINAL VISUAL LOCK\n",
    "FROZEN SCZ-MSS SCORE ARCHITECTURE AND DISCOVERY-ONLY DESCRIPTIVE BEHAVIOUR\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(project_id, EXPECTED_PROJECT_ID)
) {
    stop(
        "Project ID mismatch. Expected: ",
        EXPECTED_PROJECT_ID,
        " | Observed: ",
        paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace("digest", quietly = TRUE)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont("Times New Roman")
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. VALIDATED S5 v3 SOURCE DATA
# ==================================================================================================

V3_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS5_Score_Behaviour_TNR_v3"
)

SOURCE_DIR <- file.path(
    V3_DIR,
    "source_data"
)

WEIGHT_SOURCE <- file.path(
    SOURCE_DIR,
    "FigureS5_frozen_weight_architecture_source_v3.csv"
)

DONOR_SOURCE <- file.path(
    SOURCE_DIR,
    "FigureS5_discovery_donor_score_source_v3.csv"
)

PAIRED_SOURCE <- file.path(
    SOURCE_DIR,
    "FigureS5_paired_region_score_source_v3.csv"
)

required_sources <- c(
    WEIGHT_SOURCE,
    DONOR_SOURCE,
    PAIRED_SOURCE
)

missing_sources <- required_sources[
    !file.exists(required_sources)
]

if (length(missing_sources) > 0L) {
    stop(
        "Validated S5 v3 source file(s) missing:\n",
        paste(missing_sources, collapse = "\n")
    )
}

cat("Validated Figure S5 v3 source data: PASS\n")

# ==================================================================================================
# 5. LOAD
# ==================================================================================================

weight_plot <- read.csv(
    WEIGHT_SOURCE,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

donor_scores <- read.csv(
    DONOR_SOURCE,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

paired <- read.csv(
    PAIRED_SOURCE,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 6. SCHEMA
# ==================================================================================================

required_weight_cols <- c(
    "ensemblID_stable",
    "display_symbol",
    "final_normalized_weight",
    "direction_display"
)

required_donor_cols <- c(
    "donor_id",
    "diagnosis_display",
    "SCZ_MSS_v1"
)

required_paired_cols <- c(
    "donor_id",
    "diagnosis_display",
    "DLPFC_score",
    "HIPPO_score"
)

if (length(setdiff(required_weight_cols, names(weight_plot))) > 0L) {
    stop("Weight-source schema mismatch.")
}

if (length(setdiff(required_donor_cols, names(donor_scores))) > 0L) {
    stop("Donor-score source schema mismatch.")
}

if (length(setdiff(required_paired_cols, names(paired))) > 0L) {
    stop("Paired-score source schema mismatch.")
}

cat("Validated S5 source schemas: PASS\n")

# ==================================================================================================
# 7. FROZEN STRUCTURE
# ==================================================================================================

if (nrow(weight_plot) != 24L) {
    stop("Expected 24 frozen features.")
}

if (nrow(donor_scores) != 454L) {
    stop("Expected 454 discovery donors.")
}

if (nrow(paired) != 301L) {
    stop("Expected 301 paired donors.")
}

if (anyDuplicated(weight_plot$ensemblID_stable) > 0L) {
    stop("Duplicated frozen feature.")
}

if (anyDuplicated(donor_scores$donor_id) > 0L) {
    stop("Duplicated discovery donor.")
}

if (anyDuplicated(paired$donor_id) > 0L) {
    stop("Duplicated paired donor.")
}

weight_mass <- sum(
    abs(weight_plot$final_normalized_weight)
)

if (abs(weight_mass - 1) > 1e-10) {
    stop("Absolute normalized frozen weight does not sum to 1.")
}

POS_N <- sum(
    weight_plot$final_normalized_weight > 0
)

NEG_N <- sum(
    weight_plot$final_normalized_weight < 0
)

if (POS_N != 9L || NEG_N != 15L) {
    stop("Frozen positive/negative feature counts changed.")
}

cat(
    "Frozen 24-feature architecture: 9 positive / 15 negative — PASS\n"
)

# ==================================================================================================
# 8. DIAGNOSIS
# ==================================================================================================

donor_scores$diagnosis_display <- factor(
    as.character(donor_scores$diagnosis_display),
    levels = c(
        "Control",
        "Schizophrenia"
    )
)

paired$diagnosis_display <- factor(
    as.character(paired$diagnosis_display),
    levels = c(
        "Control",
        "Schizophrenia"
    )
)

if (
    anyNA(donor_scores$diagnosis_display) ||
    anyNA(paired$diagnosis_display)
) {
    stop("Unexpected diagnosis label.")
}

CONTROL_N <- sum(
    donor_scores$diagnosis_display == "Control"
)

SCZ_N <- sum(
    donor_scores$diagnosis_display == "Schizophrenia"
)

PAIRED_CONTROL_N <- sum(
    paired$diagnosis_display == "Control"
)

PAIRED_SCZ_N <- sum(
    paired$diagnosis_display == "Schizophrenia"
)

if (
    CONTROL_N != 277L ||
    SCZ_N != 177L ||
    PAIRED_CONTROL_N != 192L ||
    PAIRED_SCZ_N != 109L
) {
    stop("Frozen diagnosis counts changed.")
}

cat("Frozen donor counts: PASS\n")

# ==================================================================================================
# 9. DESCRIPTIVE ANCHORS
# ==================================================================================================

auc_rank <- function(y, score) {

    y <- as.integer(y)

    n1 <- sum(y == 1L)
    n0 <- sum(y == 0L)

    if (n1 == 0L || n0 == 0L) {
        stop("Both outcome classes required for AUROC.")
    }

    score_rank <- rank(
        score,
        ties.method = "average"
    )

    (
        sum(score_rank[y == 1L]) -
            n1 * (n1 + 1) / 2
    ) /
        (n1 * n0)
}

hedges_g_two_group <- function(control, schizophrenia) {

    n0 <- length(control)
    n1 <- length(schizophrenia)

    s0 <- stats::sd(control)
    s1 <- stats::sd(schizophrenia)

    pooled_sd <- sqrt(
        (
            (n0 - 1) * s0^2 +
            (n1 - 1) * s1^2
        ) /
            (n0 + n1 - 2)
    )

    d <- (
        mean(schizophrenia) -
        mean(control)
    ) /
        pooled_sd

    df <- n0 + n1 - 2

    correction <- 1 - 3 / (4 * df - 1)

    correction * d
}

donor_y <- ifelse(
    donor_scores$diagnosis_display == "Schizophrenia",
    1L,
    0L
)

DONOR_AUC <- auc_rank(
    donor_y,
    donor_scores$SCZ_MSS_v1
)

DONOR_G <- hedges_g_two_group(
    donor_scores$SCZ_MSS_v1[
        donor_scores$diagnosis_display == "Control"
    ],
    donor_scores$SCZ_MSS_v1[
        donor_scores$diagnosis_display == "Schizophrenia"
    ]
)

PEARSON_ALL <- stats::cor(
    paired$DLPFC_score,
    paired$HIPPO_score,
    method = "pearson"
)

SPEARMAN_ALL <- stats::cor(
    paired$DLPFC_score,
    paired$HIPPO_score,
    method = "spearman"
)

PEARSON_CONTROL <- stats::cor(
    paired$DLPFC_score[
        paired$diagnosis_display == "Control"
    ],
    paired$HIPPO_score[
        paired$diagnosis_display == "Control"
    ],
    method = "pearson"
)

PEARSON_SCZ <- stats::cor(
    paired$DLPFC_score[
        paired$diagnosis_display == "Schizophrenia"
    ],
    paired$HIPPO_score[
        paired$diagnosis_display == "Schizophrenia"
    ],
    method = "pearson"
)

if (abs(DONOR_AUC - 0.75228946) > 5e-8) {
    stop("Frozen donor AUROC changed.")
}

if (abs(DONOR_G - 0.94122939) > 5e-7) {
    stop("Frozen donor Hedges g changed.")
}

if (
    abs(PEARSON_ALL - 0.2074) > 5e-4 ||
    abs(PEARSON_CONTROL - 0.1003) > 5e-4 ||
    abs(PEARSON_SCZ - 0.1473) > 5e-4 ||
    abs(SPEARMAN_ALL - 0.2341) > 5e-4
) {
    stop("Frozen paired-region correlation anchor changed.")
}

cat("Frozen descriptive score anchors: PASS\n")

# ==================================================================================================
# 10. WEIGHT ORDER
# ==================================================================================================

weight_plot$direction_display <- factor(
    as.character(weight_plot$direction_display),
    levels = c(
        "Negative weight",
        "Positive weight"
    )
)

weight_plot <- weight_plot[
    order(weight_plot$final_normalized_weight),
    ,
    drop = FALSE
]

weight_plot$display_symbol <- factor(
    as.character(weight_plot$display_symbol),
    levels = as.character(weight_plot$display_symbol)
)

# ==================================================================================================
# 11. OUTPUT
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS5_Score_Behaviour_TNR_v4_FINAL"
)

SOURCE_OUT_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "FigureS5_Frozen_SCZ_MSS_Score_Behaviour_TimesNewRoman_v4_FINAL.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "FigureS5_Frozen_SCZ_MSS_Score_Behaviour_TimesNewRoman_v4_FINAL.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "FigureS5_Frozen_SCZ_MSS_Score_Behaviour_TimesNewRoman_v4_FINAL.png"
)

WEIGHT_SOURCE_V4 <- file.path(
    SOURCE_OUT_DIR,
    "FigureS5_frozen_weight_architecture_source_v4.csv"
)

DONOR_SOURCE_V4 <- file.path(
    SOURCE_OUT_DIR,
    "FigureS5_discovery_donor_score_source_v4.csv"
)

PAIRED_SOURCE_V4 <- file.path(
    SOURCE_OUT_DIR,
    "FigureS5_paired_region_score_source_v4.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "FigureS5_FINAL_generation_summary_v4.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "FigureS5_FINAL_manifest_v4.csv"
)

write.csv(
    weight_plot,
    WEIGHT_SOURCE_V4,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    donor_scores,
    DONOR_SOURCE_V4,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    paired,
    PAIRED_SOURCE_V4,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 12. THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.2,
            hjust = 0,
            margin = ggplot2::margin(b = 2)
        ),

        plot.subtitle = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.35,
            lineheight = 1.02,
            colour = "black",
            margin = ggplot2::margin(b = 5)
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.2
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.9,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.8
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.5
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            7,
            8,
            7,
            8
        )
    )

weight_colours <- c(
    "Negative weight" = "#4E79A7",
    "Positive weight" = "#E15759"
)

diagnosis_colours <- c(
    "Control" = "#4E79A7",
    "Schizophrenia" = "#E15759"
)

# ==================================================================================================
# 13. PANEL A
# ==================================================================================================

pA <- ggplot2::ggplot(
    weight_plot,
    ggplot2::aes(
        x = display_symbol,
        y = final_normalized_weight,
        fill = direction_display
    )
) +
    ggplot2::geom_hline(
        yintercept = 0,
        linewidth = 0.42,
        colour = "#555555"
    ) +
    ggplot2::geom_col(
        width = 0.72
    ) +
    ggplot2::scale_fill_manual(
        values = weight_colours
    ) +
    ggplot2::coord_flip() +
    ggplot2::labs(
        title = "A  Frozen 24-feature SCZ-MSS weights",
        x = NULL,
        y = "Frozen normalized weight",
        fill = "Direction"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.y = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.3,
            colour = "black"
        ),
        legend.position = "bottom",
        legend.direction = "horizontal",
        plot.subtitle = ggplot2::element_blank()
    )

# ==================================================================================================
# 14. PANEL B
# ==================================================================================================

B_SUBTITLE <- paste0(
    "Post-selection descriptive only; donor AUROC = ",
    sprintf("%.3f", DONOR_AUC),
    "; Hedges' g = ",
    sprintf("%.3f", DONOR_G)
)

pB <- ggplot2::ggplot(
    donor_scores,
    ggplot2::aes(
        x = diagnosis_display,
        y = SCZ_MSS_v1,
        fill = diagnosis_display,
        colour = diagnosis_display
    )
) +
    ggplot2::geom_violin(
        width = 0.86,
        trim = TRUE,
        alpha = 0.18,
        colour = NA
    ) +
    ggplot2::geom_boxplot(
        width = 0.28,
        outlier.shape = NA,
        fill = "white",
        colour = "black",
        linewidth = 0.56
    ) +
    ggplot2::geom_point(
        position = ggplot2::position_jitter(
            width = 0.11,
            height = 0,
            seed = 20261001
        ),
        size = 0.82,
        alpha = 0.42,
        show.legend = FALSE
    ) +
    ggplot2::scale_fill_manual(
        values = diagnosis_colours,
        guide = "none"
    ) +
    ggplot2::scale_colour_manual(
        values = diagnosis_colours,
        guide = "none"
    ) +
    ggplot2::scale_y_continuous(
        expand = ggplot2::expansion(
            mult = c(
                0.04,
                0.05
            )
        )
    ) +
    ggplot2::labs(
        title = "B  Discovery donor scores",
        subtitle = B_SUBTITLE,
        x = NULL,
        y = "Frozen SCZ-MSS v1"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 8.0
        )
    )

# ==================================================================================================
# 15. PANEL C
# ==================================================================================================

C_SUBTITLE <- paste0(
    "Paired donors = 301; overall Pearson r = ",
    sprintf("%.3f", PEARSON_ALL),
    "\nControl r = ",
    sprintf("%.3f", PEARSON_CONTROL),
    "; Schizophrenia r = ",
    sprintf("%.3f", PEARSON_SCZ)
)

paired_limits <- range(
    c(
        paired$DLPFC_score,
        paired$HIPPO_score
    ),
    finite = TRUE
)

paired_span <- diff(
    paired_limits
)

paired_limits <- c(
    paired_limits[1] - 0.04 * paired_span,
    paired_limits[2] + 0.04 * paired_span
)

pC <- ggplot2::ggplot(
    paired,
    ggplot2::aes(
        x = DLPFC_score,
        y = HIPPO_score,
        colour = diagnosis_display
    )
) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#777777"
    ) +
    ggplot2::geom_point(
        size = 1.45,
        alpha = 0.62
    ) +
    ggplot2::scale_colour_manual(
        values = diagnosis_colours
    ) +
    ggplot2::scale_x_continuous(
        limits = paired_limits,
        expand = ggplot2::expansion(mult = 0)
    ) +
    ggplot2::scale_y_continuous(
        limits = paired_limits,
        expand = ggplot2::expansion(mult = 0)
    ) +
    ggplot2::coord_equal() +
    ggplot2::labs(
        title = "C  Paired cross-region scores",
        subtitle = C_SUBTITLE,
        x = "DLPFC frozen SCZ-MSS v1",
        y = "HIPPO frozen SCZ-MSS v1",
        colour = "Diagnosis"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.direction = "horizontal",
        plot.subtitle = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.15,
            lineheight = 0.98,
            colour = "black",
            margin = ggplot2::margin(b = 5)
        )
    )

# ==================================================================================================
# 16. COMBINE
# ==================================================================================================

draw_figure_s5 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 2,
        ncol = 2,
        widths = grid::unit(
            c(1, 1),
            "null"
        ),
        heights = grid::unit(
            c(1.26, 1.00),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1:2
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 17. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 7.0,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure_s5()

grDevices::dev.off()

# ==================================================================================================
# 18. SAVE TIFF 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 7.0,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure_s5()

grDevices::dev.off()

# ==================================================================================================
# 19. SAVE PNG 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 7.0,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure_s5()

grDevices::dev.off()

# ==================================================================================================
# 20. OUTPUT VERIFICATION
# ==================================================================================================

figure_outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    WEIGHT_SOURCE_V4,
    DONOR_SOURCE_V4,
    PAIRED_SOURCE_V4
)

if (!all(file.exists(figure_outputs))) {
    stop("One or more S5 FINAL outputs were not created.")
}

output_info <- file.info(
    figure_outputs
)

if (
    any(
        is.na(output_info$size) |
        output_info$size <= 0
    )
) {
    stop("One or more S5 FINAL outputs are zero-byte.")
}

cat("Supplementary Figure S5 FINAL outputs created and non-zero: PASS\n")

# ==================================================================================================
# 21. SUMMARY
# ==================================================================================================

summary_lines <- c(
    "==============================================================================================================",
    "STEP 11D.5T — SUPPLEMENTARY FIGURE S5 — FINAL VISUAL LOCK",
    "==============================================================================================================",
    "",
    "Validated Figure S5 v3 source data reused.",
    "Analytical results changed: NO",
    "Score recalculated: NO",
    "Model/CV analysis rerun: NO",
    "",
    "Frozen score architecture:",
    "Features = 24",
    paste0(
        "Positive / negative = ",
        POS_N,
        " / ",
        NEG_N
    ),
    "",
    "Discovery score — POST-SELECTION DESCRIPTIVE:",
    paste0(
        "Donors = 454 (",
        CONTROL_N,
        " Control / ",
        SCZ_N,
        " Schizophrenia)"
    ),
    paste0(
        "Donor AUROC = ",
        sprintf("%.8f", DONOR_AUC)
    ),
    paste0(
        "Hedges g = ",
        sprintf("%.8f", DONOR_G)
    ),
    "",
    "Paired-region descriptive behaviour:",
    paste0(
        "Paired donors = 301 (",
        PAIRED_CONTROL_N,
        " Control / ",
        PAIRED_SCZ_N,
        " Schizophrenia)"
    ),
    paste0(
        "Pearson overall / Control / Schizophrenia = ",
        sprintf("%.4f", PEARSON_ALL),
        " / ",
        sprintf("%.4f", PEARSON_CONTROL),
        " / ",
        sprintf("%.4f", PEARSON_SCZ)
    ),
    paste0(
        "Spearman overall = ",
        sprintf("%.4f", SPEARMAN_ALL)
    ),
    "",
    "Final visual correction:",
    "Panel C title shortened to prevent clipping.",
    "No analytical value changed.",
    "",
    paste0("PDF = ", PDF_PATH),
    paste0("TIFF = ", TIFF_PATH),
    paste0("PNG = ", PNG_PATH),
    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 22. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_sources,
    figure_outputs,
    SUMMARY_PATH
)

manifest <- data.frame(
    file = basename(manifest_files),
    path = manifest_files,
    bytes = as.numeric(
        file.info(manifest_files)$size
    ),
    sha256 = vapply(
        manifest_files,
        function(p) {
            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 23. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSUPPLEMENTARY FIGURE S5 FINAL SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat("Validated S5 v3 source data: PASS\n")
cat("Analytical results changed: NO\n")
cat("Score/model/CV analysis rerun: NO\n")

cat(
    "\nFrozen score architecture:\n",
    "Features: 24\n",
    "Positive / negative: ",
    POS_N,
    " / ",
    NEG_N,
    "\n",
    sep = ""
)

cat(
    "\nDiscovery donor scores — POST-SELECTION DESCRIPTIVE:\n",
    "Control / Schizophrenia donors: ",
    CONTROL_N,
    " / ",
    SCZ_N,
    "\n",
    "Donor AUROC: ",
    sprintf("%.8f", DONOR_AUC),
    "\n",
    "Hedges g: ",
    sprintf("%.8f", DONOR_G),
    "\n",
    sep = ""
)

cat(
    "\nPaired-region behaviour:\n",
    "Paired donors: 301\n",
    "Pearson overall / Control / Schizophrenia: ",
    sprintf("%.4f", PEARSON_ALL),
    " / ",
    sprintf("%.4f", PEARSON_CONTROL),
    " / ",
    sprintf("%.4f", PEARSON_SCZ),
    "\n",
    "Spearman overall: ",
    sprintf("%.4f", SPEARMAN_ALL),
    "\n",
    sep = ""
)

cat(
    "\nPanel C title clipping corrected: YES\n"
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.5T COMPLETE — SUPPLEMENTARY FIGURE S5 FINAL LOCK GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11D.5T — SUPPLEMENTARY FIGURE S5 — FINAL VISUAL LOCK
FROZEN SCZ-MSS SCORE ARCHITECTURE AND DISCOVERY-ONLY DESCRIPTIVE BEHAVIOUR
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Validated Figure S5 v3 source data: PASS
Validated S5 source schemas: PASS
Frozen 24-feature architecture: 9 positive / 15 negative — PASS
Frozen donor counts: PASS
Frozen descriptive score anchors: PASS


agg_record_f743bb04ecc 
                     2

agg_record_f743bb04ecc 
                     2

agg_record_f743bb04ecc 
                     2

Supplementary Figure S5 FINAL outputs created and non-zero: PASS

SUPPLEMENTARY FIGURE S5 FINAL SUMMARY
Validated S5 v3 source data: PASS
Analytical results changed: NO
Score/model/CV analysis rerun: NO

Frozen score architecture:
Features: 24
Positive / negative: 9 / 15

Discovery donor scores — POST-SELECTION DESCRIPTIVE:
Control / Schizophrenia donors: 277 / 177
Donor AUROC: 0.75228946
Hedges g: 0.94122939

Paired-region behaviour:
Paired donors: 301
Pearson overall / Control / Schizophrenia: 0.2074 / 0.1003 / 0.1473
Spearman overall: 0.2341

Panel C title clipping corrected: YES

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS5_Score_Behaviour_TNR_v4_FINAL/FigureS5_Frozen_SCZ_MSS_Score_Behaviour_TimesNewRoman_v4_FINAL.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS5_Score_Behaviour_TNR_v4_FINAL/FigureS5_Frozen_SCZ_MSS_Score_Behaviour_TimesNewRoman_v4_FINAL.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/

In [16]:
# ==================================================================================================
# STEP 11D.6 — SUPPLEMENTARY FIGURE S6
# BIOLOGICAL ANNOTATION AND CONSERVATIVE ENRICHMENT CONTEXT
# --------------------------------------------------------------------------------------------------
# PURPOSE
#
# Complement the manuscript with biological interpretation of the frozen
# 24-feature SCZ-MSS state WITHOUT repeating model-performance, stability,
# score-distribution, or external-validation figures.
#
# PANELS
#
#   A — Molecular annotation composition of the final 24 features
#       Protein-coding vs non-protein-coding.
#
#   B — Literature-context coverage of the final 24 features
#       Direct SCZ paper-level evidence and reviewed brain/neural context.
#       These categories are NOT mutually exclusive.
#
#   C — Top nominal GO Biological Process signals
#       Exploratory visualization only.
#       No GO-BP term reached FDR < 0.05.
#
#   D — Top nominal Reactome signals
#       Exploratory visualization only.
#       No Reactome pathway reached FDR < 0.05.
#
# IMPORTANT
#
#   - SAVED STEP10 BIOLOGY OUTPUTS ONLY.
#   - NO ORA rerun.
#   - NO annotation rerun.
#   - NO literature search rerun.
#   - NO pathway-selection optimization.
#   - NO outcome-driven pathway rescue.
#   - Nominal signals are shown only for biological context.
#
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.6 — SUPPLEMENTARY FIGURE S6\n",
    "BIOLOGICAL ANNOTATION AND CONSERVATIVE ENRICHMENT CONTEXT\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {

    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(
            project_id,
            collapse = ", "
        )
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace(
    "ggplot2",
    quietly = TRUE
)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace(
    "digest",
    quietly = TRUE
)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. FROZEN INPUT PATHS
# ==================================================================================================

BIOLOGY_DIR <- file.path(
    PROJECT_ROOT,
    "10_biology"
)

INTEGRATED_PATH <- file.path(
    BIOLOGY_DIR,
    "final_integrated_synthesis_v1",
    "STEP10G1_SCZ_MSS_24feature_integrated_biological_evidence_matrix_v1.csv"
)

ORA_DIR <- file.path(
    BIOLOGY_DIR,
    "enrichment_v1",
    "ora_v1"
)

GO_ALL_PATH <- file.path(
    ORA_DIR,
    "STEP10B2_GO_BP_all_tested_terms_v1.csv"
)

GO_NOMINAL_PATH <- file.path(
    ORA_DIR,
    "STEP10B2_GO_BP_nominalP05_terms_v1.csv"
)

GO_FDR_PATH <- file.path(
    ORA_DIR,
    "STEP10B2_GO_BP_FDR05_terms_v1.csv"
)

REACTOME_ALL_PATH <- file.path(
    ORA_DIR,
    "STEP10B2_Reactome_all_tested_pathways_v1.csv"
)

REACTOME_NOMINAL_PATH <- file.path(
    ORA_DIR,
    "STEP10B2_Reactome_nominalP05_pathways_v1.csv"
)

REACTOME_FDR_PATH <- file.path(
    ORA_DIR,
    "STEP10B2_Reactome_FDR05_pathways_v1.csv"
)

required_inputs <- c(
    INTEGRATED_PATH,
    GO_ALL_PATH,
    GO_NOMINAL_PATH,
    GO_FDR_PATH,
    REACTOME_ALL_PATH,
    REACTOME_NOMINAL_PATH,
    REACTOME_FDR_PATH
)

missing_inputs <- required_inputs[
    !file.exists(
        required_inputs
    )
]

if (length(missing_inputs) > 0L) {

    stop(
        "Required frozen biology artifact(s) missing:\n",
        paste(
            missing_inputs,
            collapse = "\n"
        )
    )
}

cat("Required frozen STEP10 biology artifacts: PASS\n")

# ==================================================================================================
# 5. LOAD SAVED RESULTS
# ==================================================================================================

integrated <- read.csv(
    INTEGRATED_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

go_all <- read.csv(
    GO_ALL_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

go_nominal_saved <- read.csv(
    GO_NOMINAL_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

go_fdr_saved <- read.csv(
    GO_FDR_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

reactome_all <- read.csv(
    REACTOME_ALL_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

reactome_nominal_saved <- read.csv(
    REACTOME_NOMINAL_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

reactome_fdr_saved <- read.csv(
    REACTOME_FDR_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 6. INTEGRATED BIOLOGY SCHEMA
# ==================================================================================================

required_integrated_cols <- c(
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "direction_standardized",
    "final_normalized_weight",
    "SCZ_literature_review_status",
    "brain_literature_review_status",
    "overall_direct_SCZ_evidence",
    "overall_brain_context_strength"
)

missing_integrated_cols <- setdiff(
    required_integrated_cols,
    names(integrated)
)

if (length(missing_integrated_cols) > 0L) {

    stop(
        "Integrated biological matrix missing field(s): ",
        paste(
            missing_integrated_cols,
            collapse = ", "
        )
    )
}

if (
    nrow(integrated) != 24L ||
    anyDuplicated(
        integrated$ensemblID_stable
    ) > 0L
) {

    stop(
        "Integrated biology matrix must contain 24 unique frozen features."
    )
}

if (
    abs(
        sum(
            abs(
                integrated$final_normalized_weight
            )
        ) -
        1
    ) >
        1e-10
) {

    stop(
        "Frozen normalized weights in integrated biology matrix do not sum to 1."
    )
}

cat(
    "Frozen integrated 24-feature biological matrix: PASS\n"
)

# ==================================================================================================
# 7. FEATURE COMPOSITION
# ==================================================================================================

N_PROTEIN_CODING <- sum(
    integrated$gene_type ==
        "protein_coding"
)

N_NON_PROTEIN_CODING <- nrow(integrated) -
    N_PROTEIN_CODING

if (
    N_PROTEIN_CODING != 12L ||
    N_NON_PROTEIN_CODING != 12L
) {

    stop(
        "Frozen protein-coding/non-protein-coding composition changed."
    )
}

cat(
    "Protein-coding / non-protein-coding: 12 / 12 — PASS\n"
)

# ==================================================================================================
# 8. LITERATURE-CONTEXT COVERAGE
# ==================================================================================================

SCZ_REVIEW_STATUS <-
    "PAPER_LEVEL_SCZ_EVIDENCE_MANUALLY_ADJUDICATED"

BRAIN_REVIEW_STATUS <-
    "PAPER_LEVEL_BRAIN_CONTEXT_MANUALLY_ADJUDICATED"

N_DIRECT_SCZ <- sum(
    integrated$SCZ_literature_review_status ==
        SCZ_REVIEW_STATUS,
    na.rm = TRUE
)

N_BRAIN_CONTEXT <- sum(
    integrated$brain_literature_review_status ==
        BRAIN_REVIEW_STATUS,
    na.rm = TRUE
)

if (
    N_DIRECT_SCZ != 3L ||
    N_BRAIN_CONTEXT != 13L
) {

    stop(
        "Frozen paper-level biological-context counts changed."
    )
}

cat(
    "Paper-level direct SCZ / brain-context features: ",
    N_DIRECT_SCZ,
    " / ",
    N_BRAIN_CONTEXT,
    " — PASS\n",
    sep = ""
)

# ==================================================================================================
# 9. ORA SCHEMA
# ==================================================================================================

required_ora_cols <- c(
    "ID",
    "Description",
    "pvalue",
    "p.adjust",
    "Count"
)

missing_go_cols <- setdiff(
    required_ora_cols,
    names(go_all)
)

missing_reactome_cols <- setdiff(
    required_ora_cols,
    names(reactome_all)
)

if (length(missing_go_cols) > 0L) {

    stop(
        "GO-BP saved table missing field(s): ",
        paste(
            missing_go_cols,
            collapse = ", "
        )
    )
}

if (length(missing_reactome_cols) > 0L) {

    stop(
        "Reactome saved table missing field(s): ",
        paste(
            missing_reactome_cols,
            collapse = ", "
        )
    )
}

go_all$pvalue <- as.numeric(
    go_all$pvalue
)

go_all$p.adjust <- as.numeric(
    go_all$p.adjust
)

go_all$Count <- as.numeric(
    go_all$Count
)

reactome_all$pvalue <- as.numeric(
    reactome_all$pvalue
)

reactome_all$p.adjust <- as.numeric(
    reactome_all$p.adjust
)

reactome_all$Count <- as.numeric(
    reactome_all$Count
)

if (
    any(
        !is.finite(
            go_all$pvalue
        )
    ) ||
    any(
        !is.finite(
            go_all$p.adjust
        )
    ) ||
    any(
        !is.finite(
            reactome_all$pvalue
        )
    ) ||
    any(
        !is.finite(
            reactome_all$p.adjust
        )
    )
) {

    stop(
        "Non-finite saved ORA p-value detected."
    )
}

cat("Saved GO-BP / Reactome ORA schemas: PASS\n")

# ==================================================================================================
# 10. FROZEN ORA COUNTS
# ==================================================================================================

GO_TESTED_N <- nrow(
    go_all
)

GO_NOMINAL_N <- sum(
    go_all$pvalue < 0.05,
    na.rm = TRUE
)

GO_FDR_N <- sum(
    go_all$p.adjust < 0.05,
    na.rm = TRUE
)

REACTOME_TESTED_N <- nrow(
    reactome_all
)

REACTOME_NOMINAL_N <- sum(
    reactome_all$pvalue < 0.05,
    na.rm = TRUE
)

REACTOME_FDR_N <- sum(
    reactome_all$p.adjust < 0.05,
    na.rm = TRUE
)

if (
    GO_TESTED_N != 282L ||
    GO_NOMINAL_N != 101L ||
    GO_FDR_N != 0L
) {

    stop(
        "Frozen GO-BP ORA anchors changed.\n",
        "Observed tested / nominal / FDR: ",
        GO_TESTED_N,
        " / ",
        GO_NOMINAL_N,
        " / ",
        GO_FDR_N
    )
}

if (
    REACTOME_TESTED_N != 50L ||
    REACTOME_NOMINAL_N != 19L ||
    REACTOME_FDR_N != 0L
) {

    stop(
        "Frozen Reactome ORA anchors changed.\n",
        "Observed tested / nominal / FDR: ",
        REACTOME_TESTED_N,
        " / ",
        REACTOME_NOMINAL_N,
        " / ",
        REACTOME_FDR_N
    )
}

# Cross-check saved subset files.

if (nrow(go_nominal_saved) != GO_NOMINAL_N) {

    stop(
        "Saved GO nominal subset disagrees with full GO result table."
    )
}

if (nrow(reactome_nominal_saved) != REACTOME_NOMINAL_N) {

    stop(
        "Saved Reactome nominal subset disagrees with full Reactome result table."
    )
}

if (
    nrow(go_fdr_saved) != 0L ||
    nrow(reactome_fdr_saved) != 0L
) {

    stop(
        "Expected zero FDR-significant GO-BP and Reactome results."
    )
}

cat(
    "GO-BP tested / nominal p<0.05 / FDR<0.05: ",
    GO_TESTED_N,
    " / ",
    GO_NOMINAL_N,
    " / ",
    GO_FDR_N,
    " — PASS\n",
    sep = ""
)

cat(
    "Reactome tested / nominal p<0.05 / FDR<0.05: ",
    REACTOME_TESTED_N,
    " / ",
    REACTOME_NOMINAL_N,
    " / ",
    REACTOME_FDR_N,
    " — PASS\n",
    sep = ""
)

# ==================================================================================================
# 11. PANEL A DATA — MOLECULAR COMPOSITION
# ==================================================================================================

panel_A <- data.frame(

    annotation = factor(
        c(
            "Protein-coding",
            "Non-protein-coding"
        ),
        levels = c(
            "Protein-coding",
            "Non-protein-coding"
        )
    ),

    n_features = c(
        N_PROTEIN_CODING,
        N_NON_PROTEIN_CODING
    ),

    stringsAsFactors = FALSE
)

panel_A$percent <- (
    panel_A$n_features /
        24
) *
    100

panel_A$label <- paste0(
    panel_A$n_features,
    "/24\n(",
    sprintf(
        "%.0f",
        panel_A$percent
    ),
    "%)"
)

# ==================================================================================================
# 12. PANEL B DATA — EVIDENCE COVERAGE
# ==================================================================================================

panel_B <- data.frame(

    evidence = factor(
        c(
            "Direct SCZ\npaper-level evidence",
            "Reviewed brain/neural\ncontext"
        ),
        levels = c(
            "Direct SCZ\npaper-level evidence",
            "Reviewed brain/neural\ncontext"
        )
    ),

    n_features = c(
        N_DIRECT_SCZ,
        N_BRAIN_CONTEXT
    ),

    stringsAsFactors = FALSE
)

panel_B$percent <- (
    panel_B$n_features /
        24
) *
    100

panel_B$label <- paste0(
    panel_B$n_features,
    "/24\n(",
    sprintf(
        "%.1f",
        panel_B$percent
    ),
    "%)"
)

# ==================================================================================================
# 13. TERM-WRAPPING HELPER
# ==================================================================================================

wrap_term <- function(
    x,
    width = 58
) {

    vapply(
        as.character(x),
        function(z) {

            paste(
                strwrap(
                    z,
                    width = width
                ),
                collapse = "\n"
            )
        },
        FUN.VALUE = character(1)
    )
}

# ==================================================================================================
# 14. TOP NOMINAL GO-BP TERMS
#
# Visualization only.
# Selection rule is purely:
#   saved nominal p < 0.05,
#   then smallest saved p-values.
#
# No new enrichment inference is performed.
# ==================================================================================================

go_nominal <- go_all[
    go_all$pvalue < 0.05,
    ,
    drop = FALSE
]

go_nominal <- go_nominal[
    order(
        go_nominal$pvalue,
        go_nominal$p.adjust,
        go_nominal$Description
    ),
    ,
    drop = FALSE
]

N_GO_DISPLAY <- min(
    8L,
    nrow(go_nominal)
)

top_go <- head(
    go_nominal,
    N_GO_DISPLAY
)

top_go$minus_log10_p <- -log10(
    top_go$pvalue
)

top_go$wrapped_description <- wrap_term(
    top_go$Description,
    width = 55
)

top_go$wrapped_description <- factor(
    top_go$wrapped_description,
    levels = rev(
        top_go$wrapped_description
    )
)

# ==================================================================================================
# 15. TOP NOMINAL REACTOME TERMS
# ==================================================================================================

reactome_nominal <- reactome_all[
    reactome_all$pvalue < 0.05,
    ,
    drop = FALSE
]

reactome_nominal <- reactome_nominal[
    order(
        reactome_nominal$pvalue,
        reactome_nominal$p.adjust,
        reactome_nominal$Description
    ),
    ,
    drop = FALSE
]

N_REACTOME_DISPLAY <- min(
    8L,
    nrow(reactome_nominal)
)

top_reactome <- head(
    reactome_nominal,
    N_REACTOME_DISPLAY
)

top_reactome$minus_log10_p <- -log10(
    top_reactome$pvalue
)

top_reactome$wrapped_description <- wrap_term(
    top_reactome$Description,
    width = 55
)

top_reactome$wrapped_description <- factor(
    top_reactome$wrapped_description,
    levels = rev(
        top_reactome$wrapped_description
    )
)

cat(
    "Visualization terms selected from saved nominal results: ",
    N_GO_DISPLAY,
    " GO-BP / ",
    N_REACTOME_DISPLAY,
    " Reactome — PASS\n",
    sep = ""
)

# ==================================================================================================
# 16. OUTPUT PATHS
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS6_Biology_Enrichment_TNR_v1"
)

SOURCE_DIR <- file.path(
    OUT_DIR,
    "source_data"
)

dir.create(
    SOURCE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "FigureS6_Biological_Annotation_Enrichment_TimesNewRoman_v1.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "FigureS6_Biological_Annotation_Enrichment_TimesNewRoman_v1.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "FigureS6_Biological_Annotation_Enrichment_TimesNewRoman_v1.png"
)

COMPOSITION_SOURCE_PATH <- file.path(
    SOURCE_DIR,
    "FigureS6_annotation_composition_source_v1.csv"
)

EVIDENCE_SOURCE_PATH <- file.path(
    SOURCE_DIR,
    "FigureS6_literature_context_source_v1.csv"
)

GO_SOURCE_PATH <- file.path(
    SOURCE_DIR,
    "FigureS6_GO_BP_nominal_display_source_v1.csv"
)

REACTOME_SOURCE_PATH <- file.path(
    SOURCE_DIR,
    "FigureS6_Reactome_nominal_display_source_v1.csv"
)

FULL_GO_SOURCE_PATH <- file.path(
    SOURCE_DIR,
    "FigureS6_GO_BP_all_saved_results_v1.csv"
)

FULL_REACTOME_SOURCE_PATH <- file.path(
    SOURCE_DIR,
    "FigureS6_Reactome_all_saved_results_v1.csv"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "FigureS6_generation_summary_v1.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "FigureS6_generation_manifest_v1.csv"
)

write.csv(
    panel_A,
    COMPOSITION_SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    panel_B,
    EVIDENCE_SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    top_go,
    GO_SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    top_reactome,
    REACTOME_SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    go_all,
    FULL_GO_SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE
)

write.csv(
    reactome_all,
    FULL_REACTOME_SOURCE_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 17. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.1,
            hjust = 0,
            margin = ggplot2::margin(
                b = 3
            )
        ),

        plot.subtitle = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.4,
            lineheight = 1.04,
            colour = "black",
            margin = ggplot2::margin(
                b = 5
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.1
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.8,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.7
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.4
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            7,
            9,
            7,
            9
        )
    )

# ==================================================================================================
# 18. PANEL A — MOLECULAR ANNOTATION
# ==================================================================================================

composition_colours <- c(
    "Protein-coding" = "#4E79A7",
    "Non-protein-coding" = "#B07AA1"
)

pA <- ggplot2::ggplot(
    panel_A,
    ggplot2::aes(
        x = annotation,
        y = n_features,
        fill = annotation
    )
) +
    ggplot2::geom_col(
        width = 0.58,
        colour = "black",
        linewidth = 0.45
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = label
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.75,
        vjust = -0.45,
        lineheight = 0.98
    ) +
    ggplot2::scale_fill_manual(
        values = composition_colours,
        guide = "none"
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            0,
            15
        ),
        breaks = c(
            0,
            3,
            6,
            9,
            12,
            15
        ),
        expand = c(
            0,
            0
        )
    ) +
    ggplot2::labs(
        title = "A  Molecular annotation of the final state",
        x = NULL,
        y = "Frozen features, n"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.8
        )
    )

# ==================================================================================================
# 19. PANEL B — LITERATURE-CONTEXT COVERAGE
# ==================================================================================================

evidence_colours <- c(
    "Direct SCZ\npaper-level evidence" = "#E15759",
    "Reviewed brain/neural\ncontext" = "#2A9D8F"
)

pB <- ggplot2::ggplot(
    panel_B,
    ggplot2::aes(
        x = evidence,
        y = n_features,
        fill = evidence
    )
) +
    ggplot2::geom_col(
        width = 0.58,
        colour = "black",
        linewidth = 0.45
    ) +
    ggplot2::geom_text(
        ggplot2::aes(
            label = label
        ),
        family = FONT_FAMILY,
        fontface = "bold",
        size = 2.75,
        vjust = -0.45,
        lineheight = 0.98
    ) +
    ggplot2::scale_fill_manual(
        values = evidence_colours,
        guide = "none"
    ) +
    ggplot2::scale_y_continuous(
        limits = c(
            0,
            16
        ),
        breaks = c(
            0,
            4,
            8,
            12,
            16
        ),
        expand = c(
            0,
            0
        )
    ) +
    ggplot2::labs(
        title = "B  Reviewed biological-context coverage",
        subtitle = "Evidence categories are coverage indicators and are not mutually exclusive",
        x = NULL,
        y = "Frozen features, n"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.5
        )
    )

# ==================================================================================================
# 20. PANEL C — GO-BP NOMINAL SIGNALS
# ==================================================================================================

pC <- ggplot2::ggplot(
    top_go,
    ggplot2::aes(
        x = minus_log10_p,
        y = wrapped_description,
        size = Count
    )
) +
    ggplot2::geom_vline(
        xintercept = -log10(0.05),
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#777777"
    ) +
    ggplot2::geom_point(
        shape = 21,
        fill = "#4E79A7",
        colour = "black",
        stroke = 0.32,
        alpha = 0.88
    ) +
    ggplot2::scale_size_continuous(
        range = c(
            2.2,
            5.2
        )
    ) +
    ggplot2::labs(
        title = "C  GO Biological Process: strongest nominal signals",
        subtitle = paste0(
            "Exploratory only: ",
            GO_NOMINAL_N,
            " nominal p<0.05 among ",
            GO_TESTED_N,
            " tested terms; 0 reached FDR<0.05"
        ),
        x = expression(-log[10]("nominal p-value")),
        y = NULL,
        size = "Genes"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.y = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.0,
            colour = "black",
            lineheight = 0.95
        ),
        legend.position = "right"
    )

# ==================================================================================================
# 21. PANEL D — REACTOME NOMINAL SIGNALS
# ==================================================================================================

pD <- ggplot2::ggplot(
    top_reactome,
    ggplot2::aes(
        x = minus_log10_p,
        y = wrapped_description,
        size = Count
    )
) +
    ggplot2::geom_vline(
        xintercept = -log10(0.05),
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#777777"
    ) +
    ggplot2::geom_point(
        shape = 21,
        fill = "#F28E2B",
        colour = "black",
        stroke = 0.32,
        alpha = 0.88
    ) +
    ggplot2::scale_size_continuous(
        range = c(
            2.2,
            5.2
        )
    ) +
    ggplot2::labs(
        title = "D  Reactome: strongest nominal signals",
        subtitle = paste0(
            "Exploratory only: ",
            REACTOME_NOMINAL_N,
            " nominal p<0.05 among ",
            REACTOME_TESTED_N,
            " tested pathways; 0 reached FDR<0.05"
        ),
        x = expression(-log[10]("nominal p-value")),
        y = NULL,
        size = "Genes"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.y = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.0,
            colour = "black",
            lineheight = 0.95
        ),
        legend.position = "right"
    )

# ==================================================================================================
# 22. FIGURE LAYOUT
#
# Top row:
#   A + B
#
# Middle:
#   C full width
#
# Bottom:
#   D full width
#
# This gives long pathway labels enough horizontal space.
# ==================================================================================================

draw_figure_s6 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 3,
        ncol = 2,
        widths = grid::unit(
            c(
                1,
                1
            ),
            "null"
        ),
        heights = grid::unit(
            c(
                0.80,
                1.15,
                1.15
            ),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1:2
        ),
        newpage = FALSE
    )

    print(
        pD,
        vp = grid::viewport(
            layout.pos.row = 3,
            layout.pos.col = 1:2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 23. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 7.3,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure_s6()

grDevices::dev.off()

# ==================================================================================================
# 24. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 7.3,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure_s6()

grDevices::dev.off()

# ==================================================================================================
# 25. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 7.3,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure_s6()

grDevices::dev.off()

# ==================================================================================================
# 26. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH,
    COMPOSITION_SOURCE_PATH,
    EVIDENCE_SOURCE_PATH,
    GO_SOURCE_PATH,
    REACTOME_SOURCE_PATH,
    FULL_GO_SOURCE_PATH,
    FULL_REACTOME_SOURCE_PATH
)

if (!all(
    file.exists(
        outputs
    )
)) {

    stop(
        "One or more Supplementary Figure S6 outputs were not created."
    )
}

output_info <- file.info(
    outputs
)

if (
    any(
        is.na(
            output_info$size
        ) |
        output_info$size <= 0
    )
) {

    stop(
        "One or more Supplementary Figure S6 outputs are zero-byte."
    )
}

cat(
    "Supplementary Figure S6 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 27. GENERATION SUMMARY
# ==================================================================================================

summary_lines <- c(

    "==============================================================================================================",
    "STEP 11D.6 — SUPPLEMENTARY FIGURE S6",
    "BIOLOGICAL ANNOTATION AND CONSERVATIVE ENRICHMENT CONTEXT",
    "==============================================================================================================",

    "",

    "Frozen saved STEP10 biology outputs only.",

    "No annotation, literature retrieval, GO-BP ORA, or Reactome ORA was rerun.",

    "",

    "FINAL 24-FEATURE BIOLOGICAL COMPOSITION",

    paste0(
        "Protein-coding = ",
        N_PROTEIN_CODING,
        "/24"
    ),

    paste0(
        "Non-protein-coding = ",
        N_NON_PROTEIN_CODING,
        "/24"
    ),

    "",

    "PAPER-LEVEL BIOLOGICAL CONTEXT",

    paste0(
        "Direct SCZ evidence = ",
        N_DIRECT_SCZ,
        "/24"
    ),

    paste0(
        "Reviewed brain/neural context = ",
        N_BRAIN_CONTEXT,
        "/24"
    ),

    "These literature-context categories are not mutually exclusive.",

    "",

    "GO BIOLOGICAL PROCESS",

    paste0(
        "Tested terms = ",
        GO_TESTED_N
    ),

    paste0(
        "Nominal p < 0.05 = ",
        GO_NOMINAL_N
    ),

    paste0(
        "FDR < 0.05 = ",
        GO_FDR_N
    ),

    paste0(
        "Displayed strongest nominal terms = ",
        N_GO_DISPLAY
    ),

    "",

    "REACTOME",

    paste0(
        "Tested pathways = ",
        REACTOME_TESTED_N
    ),

    paste0(
        "Nominal p < 0.05 = ",
        REACTOME_NOMINAL_N
    ),

    paste0(
        "FDR < 0.05 = ",
        REACTOME_FDR_N
    ),

    paste0(
        "Displayed strongest nominal pathways = ",
        N_REACTOME_DISPLAY
    ),

    "",

    "INTERPRETATION",

    "No GO-BP or Reactome term reached FDR < 0.05.",

    "Nominal pathway signals are exploratory biological context only.",

    "They must not be presented as statistically confirmed enrichment.",

    "",

    paste0(
        "PDF = ",
        PDF_PATH
    ),

    paste0(
        "TIFF = ",
        TIFF_PATH
    ),

    paste0(
        "PNG = ",
        PNG_PATH
    ),

    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 28. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(

    file = basename(
        manifest_files
    ),

    path = manifest_files,

    bytes = as.numeric(
        file.info(
            manifest_files
        )$size
    ),

    sha256 = vapply(
        manifest_files,
        function(p) {

            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 29. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSUPPLEMENTARY FIGURE S6 GENERATION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "Frozen STEP10 biology sources: PASS\n"
)

cat(
    "Biology/enrichment analysis rerun: NO\n"
)

cat(
    "\nFinal 24-feature composition:\n"
)

cat(
    "Protein-coding / non-protein-coding: ",
    N_PROTEIN_CODING,
    " / ",
    N_NON_PROTEIN_CODING,
    "\n",
    sep = ""
)

cat(
    "Direct SCZ paper-level evidence: ",
    N_DIRECT_SCZ,
    "/24\n",
    sep = ""
)

cat(
    "Reviewed brain/neural context: ",
    N_BRAIN_CONTEXT,
    "/24\n",
    sep = ""
)

cat(
    "\nGO-BP tested / nominal / FDR-significant: ",
    GO_TESTED_N,
    " / ",
    GO_NOMINAL_N,
    " / ",
    GO_FDR_N,
    "\n",
    sep = ""
)

cat(
    "Reactome tested / nominal / FDR-significant: ",
    REACTOME_TESTED_N,
    " / ",
    REACTOME_NOMINAL_N,
    " / ",
    REACTOME_FDR_N,
    "\n",
    sep = ""
)

cat(
    "\nIMPORTANT: nominal enrichment signals are exploratory only.\n"
)

cat(
    "No GO-BP or Reactome result reached FDR < 0.05.\n"
)

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.6 COMPLETE — SUPPLEMENTARY FIGURE S6 GENERATED FROM FROZEN BIOLOGY OUTPUTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11D.6 — SUPPLEMENTARY FIGURE S6
BIOLOGICAL ANNOTATION AND CONSERVATIVE ENRICHMENT CONTEXT
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Required frozen STEP10 biology artifacts: PASS
Frozen integrated 24-feature biological matrix: PASS
Protein-coding / non-protein-coding: 12 / 12 — PASS
Paper-level direct SCZ / brain-context features: 3 / 13 — PASS
Saved GO-BP / Reactome ORA schemas: PASS
GO-BP tested / nominal p<0.05 / FDR<0.05: 282 / 101 / 0 — PASS
Reactome tested / nominal p<0.05 / FDR<0.05: 50 / 19 / 0 — PASS
Visualization terms selected from saved nominal results: 8 GO-BP / 8 Reactome — PASS


agg_record_f742e45471d 
                     2

agg_record_f742e45471d 
                     2

agg_record_f742e45471d 
                     2

Supplementary Figure S6 outputs created and non-zero: PASS

SUPPLEMENTARY FIGURE S6 GENERATION SUMMARY
Frozen STEP10 biology sources: PASS
Biology/enrichment analysis rerun: NO

Final 24-feature composition:
Protein-coding / non-protein-coding: 12 / 12
Direct SCZ paper-level evidence: 3/24
Reviewed brain/neural context: 13/24

GO-BP tested / nominal / FDR-significant: 282 / 101 / 0
Reactome tested / nominal / FDR-significant: 50 / 19 / 0

IMPORTANT: nominal enrichment signals are exploratory only.
No GO-BP or Reactome result reached FDR < 0.05.

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS6_Biology_Enrichment_TNR_v1/FigureS6_Biological_Annotation_Enrichment_TimesNewRoman_v1.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS6_Biology_Enrichment_TNR_v1/FigureS6_Biological_Annotation_Enrichment_TimesNewRoman_v1.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS6_Biology_Enrichment_TNR

In [19]:
# ==================================================================================================
# STEP 11D.7S — SUPPLEMENTARY FIGURE S7 — FINAL VISUAL CORRECTION
# LEAKAGE-FREE OOF DISCRIMINATION AND DESCRIPTIVE CALIBRATION
# --------------------------------------------------------------------------------------------------
# PURPOSE
#
# Redraw the already validated Figure S7 v2 using its saved source data.
#
# ANALYTICAL CHANGES: NONE
#
# ONLY VISUAL CORRECTION:
#   - shorten Panel D title/subtitle so no text is clipped
#   - retain all validated numerical results exactly
#
# NO:
#   - model rerun
#   - CV rerun
#   - averaging rerun from raw fold files
#   - feature selection
#   - recalibration fitting
#
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.7S — SUPPLEMENTARY FIGURE S7 — FINAL VISUAL CORRECTION\n",
    "LEAKAGE-FREE OOF DISCRIMINATION AND DESCRIPTIVE CALIBRATION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (
    length(project_id) != 1L ||
    !identical(
        project_id,
        EXPECTED_PROJECT_ID
    )
) {
    stop(
        "PROJECT SENTINEL MISMATCH.\n",
        "Expected: ",
        EXPECTED_PROJECT_ID,
        "\nObserved: ",
        paste(project_id, collapse = ", ")
    )
}

cat("Project sentinel: PASS\n")

# ==================================================================================================
# 2. PACKAGES
# ==================================================================================================

if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("ggplot2 unavailable.")
}

if (!requireNamespace("digest", quietly = TRUE)) {
    stop("digest unavailable.")
}

cat("Required packages: PASS\n")

# ==================================================================================================
# 3. FONT
# ==================================================================================================

FONT_NAME <- "Times New Roman"

if (.Platform$OS.type == "windows") {

    try(
        grDevices::windowsFonts(
            TNR = grDevices::windowsFont(
                "Times New Roman"
            )
        ),
        silent = TRUE
    )

    FONT_FAMILY <- "TNR"

} else {

    FONT_FAMILY <- FONT_NAME
}

cat("Figure font: Times New Roman\n")

# ==================================================================================================
# 4. VALIDATED S7 v2 SOURCE DATA
# ==================================================================================================

V2_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS7_OOF_Diagnostics_TNR_v2"
)

SOURCE_DIR_V2 <- file.path(
    V2_DIR,
    "source_data"
)

DONOR_SOURCE_PATH <- file.path(
    SOURCE_DIR_V2,
    "FigureS7_ModelF_averaged_donor_OOF_source_v2.csv"
)

SAMPLE_SOURCE_PATH <- file.path(
    SOURCE_DIR_V2,
    "FigureS7_ModelF_averaged_sample_OOF_source_v2.csv"
)

DONOR_ROC_PATH <- file.path(
    SOURCE_DIR_V2,
    "FigureS7_donor_ROC_source_v2.csv"
)

REGION_ROC_PATH <- file.path(
    SOURCE_DIR_V2,
    "FigureS7_region_ROC_source_v2.csv"
)

CALIBRATION_PATH <- file.path(
    SOURCE_DIR_V2,
    "FigureS7_donor_calibration_groups_source_v2.csv"
)

required_inputs <- c(
    DONOR_SOURCE_PATH,
    SAMPLE_SOURCE_PATH,
    DONOR_ROC_PATH,
    REGION_ROC_PATH,
    CALIBRATION_PATH
)

missing_inputs <- required_inputs[
    !file.exists(required_inputs)
]

if (length(missing_inputs) > 0L) {
    stop(
        "Validated S7 v2 source file(s) missing:\n",
        paste(missing_inputs, collapse = "\n")
    )
}

cat("Validated Figure S7 v2 source data: PASS\n")

# ==================================================================================================
# 5. LOAD
# ==================================================================================================

donor_oof <- read.csv(
    DONOR_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

sample_oof <- read.csv(
    SAMPLE_SOURCE_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

roc_donor <- read.csv(
    DONOR_ROC_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

roc_region <- read.csv(
    REGION_ROC_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

calibration_df <- read.csv(
    CALIBRATION_PATH,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# ==================================================================================================
# 6. VALIDATED SOURCE SCHEMA
# ==================================================================================================

required_donor_cols <- c(
    "donor_id",
    "diagnosis",
    "y",
    "prediction"
)

required_sample_cols <- c(
    "sample_id",
    "donor_id",
    "region",
    "diagnosis",
    "y",
    "prediction"
)

required_roc_cols <- c(
    "curve",
    "threshold",
    "FPR",
    "TPR"
)

required_calibration_cols <- c(
    "decile",
    "n_donors",
    "mean_prediction",
    "observed_SCZ_fraction"
)

if (
    length(setdiff(required_donor_cols, names(donor_oof))) > 0L ||
    length(setdiff(required_sample_cols, names(sample_oof))) > 0L ||
    length(setdiff(required_roc_cols, names(roc_donor))) > 0L ||
    length(setdiff(required_roc_cols, names(roc_region))) > 0L ||
    length(setdiff(required_calibration_cols, names(calibration_df))) > 0L
) {
    stop("Validated S7 v2 source schema changed.")
}

cat("Validated S7 source schemas: PASS\n")

# ==================================================================================================
# 7. BASIC FROZEN DIMENSION CHECKS
# ==================================================================================================

if (
    nrow(donor_oof) != 454L ||
    nrow(sample_oof) != 755L ||
    nrow(calibration_df) != 10L
) {
    stop("Validated S7 dimensions changed.")
}

if (
    sum(donor_oof$y == 0L) != 277L ||
    sum(donor_oof$y == 1L) != 177L
) {
    stop("Validated donor diagnosis counts changed.")
}

if (
    sum(sample_oof$region == "DLPFC") != 379L ||
    sum(sample_oof$region == "HIPPO") != 376L
) {
    stop("Validated regional sample counts changed.")
}

cat(
    "Validated dimensions: 454 donors / 755 samples / 10 calibration groups — PASS\n"
)

# ==================================================================================================
# 8. AUROC FUNCTION — VALIDATION ONLY
# ==================================================================================================

auc_rank <- function(y, score) {

    y <- as.integer(y)
    score <- as.numeric(score)

    keep <- (
        !is.na(y) &
        is.finite(score) &
        y %in% c(0L, 1L)
    )

    y <- y[keep]
    score <- score[keep]

    n1 <- sum(y == 1L)
    n0 <- sum(y == 0L)

    if (n1 == 0L || n0 == 0L) {
        stop("AUROC requires both classes.")
    }

    r <- rank(
        score,
        ties.method = "average"
    )

    (
        sum(r[y == 1L]) -
        n1 * (n1 + 1) / 2
    ) /
        (n1 * n0)
}

DONOR_AUC <- auc_rank(
    donor_oof$y,
    donor_oof$prediction
)

SAMPLE_AUC <- auc_rank(
    sample_oof$y,
    sample_oof$prediction
)

DLPFC_AUC <- auc_rank(
    sample_oof$y[
        sample_oof$region == "DLPFC"
    ],
    sample_oof$prediction[
        sample_oof$region == "DLPFC"
    ]
)

HIPPO_AUC <- auc_rank(
    sample_oof$y[
        sample_oof$region == "HIPPO"
    ],
    sample_oof$prediction[
        sample_oof$region == "HIPPO"
    ]
)

if (
    abs(DONOR_AUC - 0.8019743) > 5e-7 ||
    abs(SAMPLE_AUC - 0.7938032) > 5e-7 ||
    abs(DLPFC_AUC - 0.785875) > 5e-6 ||
    abs(HIPPO_AUC - 0.799746) > 5e-6
) {
    stop("Validated Figure S7 AUROC anchor changed.")
}

cat(
    "Frozen AUROC anchors: donor ",
    sprintf("%.7f", DONOR_AUC),
    "; sample ",
    sprintf("%.7f", SAMPLE_AUC),
    "; DLPFC ",
    sprintf("%.6f", DLPFC_AUC),
    "; HIPPO ",
    sprintf("%.6f", HIPPO_AUC),
    " — PASS\n",
    sep = ""
)

# ==================================================================================================
# 9. FACTORS
# ==================================================================================================

donor_oof$diagnosis_display <- ifelse(
    donor_oof$y == 1L,
    "Schizophrenia",
    "Control"
)

donor_oof$diagnosis_display <- factor(
    donor_oof$diagnosis_display,
    levels = c(
        "Control",
        "Schizophrenia"
    )
)

roc_region$curve <- factor(
    roc_region$curve,
    levels = c(
        "DLPFC",
        "HIPPO"
    )
)

# ==================================================================================================
# 10. OUTPUT DIRECTORY
# ==================================================================================================

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "figures",
    "supplementary",
    "FigureS7_OOF_Diagnostics_TNR_v3"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PDF_PATH <- file.path(
    OUT_DIR,
    "FigureS7_OOF_Discrimination_Calibration_TimesNewRoman_v3.pdf"
)

TIFF_PATH <- file.path(
    OUT_DIR,
    "FigureS7_OOF_Discrimination_Calibration_TimesNewRoman_v3.tiff"
)

PNG_PATH <- file.path(
    OUT_DIR,
    "FigureS7_OOF_Discrimination_Calibration_TimesNewRoman_v3.png"
)

SUMMARY_PATH <- file.path(
    OUT_DIR,
    "FigureS7_visual_correction_summary_v3.txt"
)

MANIFEST_PATH <- file.path(
    OUT_DIR,
    "FigureS7_visual_correction_manifest_v3.csv"
)

# ==================================================================================================
# 11. PUBLICATION THEME
# ==================================================================================================

theme_publication <- ggplot2::theme_classic(
    base_size = 9,
    base_family = FONT_FAMILY
) +
    ggplot2::theme(

        text = ggplot2::element_text(
            family = FONT_FAMILY,
            colour = "black"
        ),

        plot.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 10.0,
            hjust = 0,
            margin = ggplot2::margin(
                b = 3
            )
        ),

        plot.subtitle = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.35,
            lineheight = 1.02,
            colour = "black",
            margin = ggplot2::margin(
                b = 5
            )
        ),

        axis.title = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 9.0
        ),

        axis.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.9,
            colour = "black"
        ),

        legend.title = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 7.7
        ),

        legend.text = ggplot2::element_text(
            family = FONT_FAMILY,
            size = 7.4
        ),

        axis.line = ggplot2::element_line(
            linewidth = 0.42,
            colour = "black"
        ),

        axis.ticks = ggplot2::element_line(
            linewidth = 0.38,
            colour = "black"
        ),

        plot.margin = ggplot2::margin(
            8,
            10,
            8,
            10
        )
    )

# ==================================================================================================
# 12. COLOURS
# ==================================================================================================

diagnosis_colours <- c(
    "Control" = "#4E79A7",
    "Schizophrenia" = "#E15759"
)

region_colours <- c(
    "DLPFC" = "#4E79A7",
    "HIPPO" = "#F28E2B"
)

# ==================================================================================================
# 13. PANEL A
# ==================================================================================================

pA <- ggplot2::ggplot(
    roc_donor,
    ggplot2::aes(
        x = FPR,
        y = TPR
    )
) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#777777"
    ) +
    ggplot2::geom_path(
        linewidth = 1.02,
        colour = "#2A9D8F"
    ) +
    ggplot2::coord_equal() +
    ggplot2::scale_x_continuous(
        limits = c(0, 1),
        breaks = seq(0, 1, 0.2),
        expand = c(0, 0)
    ) +
    ggplot2::scale_y_continuous(
        limits = c(0, 1),
        breaks = seq(0, 1, 0.2),
        expand = c(0, 0)
    ) +
    ggplot2::labs(
        title = "A  Donor-level repeated-OOF ROC",
        subtitle = paste0(
            "454 donors; five held-out repeats averaged; AUROC = ",
            sprintf("%.3f", DONOR_AUC)
        ),
        x = "False-positive rate",
        y = "True-positive rate"
    ) +
    theme_publication

# ==================================================================================================
# 14. PANEL B
# ==================================================================================================

pB <- ggplot2::ggplot(
    roc_region,
    ggplot2::aes(
        x = FPR,
        y = TPR,
        colour = curve
    )
) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.50,
        colour = "#777777"
    ) +
    ggplot2::geom_path(
        linewidth = 0.92
    ) +
    ggplot2::scale_colour_manual(
        values = region_colours,
        breaks = c(
            "DLPFC",
            "HIPPO"
        ),
        labels = c(
            paste0(
                "DLPFC (",
                sprintf("%.3f", DLPFC_AUC),
                ")"
            ),
            paste0(
                "HIPPO (",
                sprintf("%.3f", HIPPO_AUC),
                ")"
            )
        )
    ) +
    ggplot2::coord_equal() +
    ggplot2::scale_x_continuous(
        limits = c(0, 1),
        breaks = seq(0, 1, 0.2),
        expand = c(0, 0)
    ) +
    ggplot2::scale_y_continuous(
        limits = c(0, 1),
        breaks = seq(0, 1, 0.2),
        expand = c(0, 0)
    ) +
    ggplot2::labs(
        title = "B  Region-specific repeated-OOF ROC",
        subtitle = "Averaged held-out sample predictions",
        x = "False-positive rate",
        y = "True-positive rate",
        colour = "Region"
    ) +
    theme_publication +
    ggplot2::theme(
        legend.position = "bottom",
        legend.direction = "horizontal"
    )

# ==================================================================================================
# 15. PANEL C
# ==================================================================================================

pC <- ggplot2::ggplot(
    donor_oof,
    ggplot2::aes(
        x = diagnosis_display,
        y = prediction,
        fill = diagnosis_display,
        colour = diagnosis_display
    )
) +
    ggplot2::geom_violin(
        width = 0.86,
        trim = TRUE,
        alpha = 0.18,
        colour = NA
    ) +
    ggplot2::geom_boxplot(
        width = 0.27,
        outlier.shape = NA,
        fill = "white",
        colour = "black",
        linewidth = 0.55
    ) +
    ggplot2::geom_point(
        position = ggplot2::position_jitter(
            width = 0.10,
            height = 0,
            seed = 20261001
        ),
        size = 0.72,
        alpha = 0.34,
        show.legend = FALSE
    ) +
    ggplot2::scale_fill_manual(
        values = diagnosis_colours,
        guide = "none"
    ) +
    ggplot2::scale_colour_manual(
        values = diagnosis_colours,
        guide = "none"
    ) +
    ggplot2::scale_y_continuous(
        limits = c(0, 1),
        breaks = seq(0, 1, 0.2),
        expand = ggplot2::expansion(
            mult = c(0.01, 0.02)
        )
    ) +
    ggplot2::labs(
        title = "C  Donor-level held-out predictions",
        subtitle = "Five held-out probabilities averaged per donor",
        x = NULL,
        y = "Repeated-OOF schizophrenia probability"
    ) +
    theme_publication +
    ggplot2::theme(
        axis.text.x = ggplot2::element_text(
            family = FONT_FAMILY,
            face = "bold",
            size = 8.0
        )
    )

# ==================================================================================================
# 16. PANEL D — SHORTENED TEXT TO REMOVE CLIPPING
# ==================================================================================================

pD <- ggplot2::ggplot(
    calibration_df,
    ggplot2::aes(
        x = mean_prediction,
        y = observed_SCZ_fraction
    )
) +
    ggplot2::geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed",
        linewidth = 0.52,
        colour = "#777777"
    ) +
    ggplot2::geom_line(
        linewidth = 0.75,
        colour = "#B07AA1"
    ) +
    ggplot2::geom_point(
        ggplot2::aes(
            size = n_donors
        ),
        shape = 21,
        fill = "#B07AA1",
        colour = "black",
        stroke = 0.34,
        alpha = 0.90
    ) +
    ggplot2::scale_size_continuous(
        range = c(2.6, 4.0),
        guide = "none"
    ) +
    ggplot2::scale_x_continuous(
        limits = c(0, 1),
        breaks = seq(0, 1, 0.2)
    ) +
    ggplot2::scale_y_continuous(
        limits = c(0, 1),
        breaks = seq(0, 1, 0.2)
    ) +
    ggplot2::coord_equal() +
    ggplot2::labs(
        title = "D  Donor-level OOF calibration",
        subtitle = "10 equal-rank donor groups; descriptive only",
        x = "Mean repeated-OOF predicted probability",
        y = "Observed schizophrenia fraction"
    ) +
    theme_publication

# ==================================================================================================
# 17. DRAW
# ==================================================================================================

draw_figure_s7 <- function() {

    grid::grid.newpage()

    figure_layout <- grid::grid.layout(
        nrow = 2,
        ncol = 2,
        widths = grid::unit(
            c(1, 1),
            "null"
        ),
        heights = grid::unit(
            c(1, 1),
            "null"
        )
    )

    grid::pushViewport(
        grid::viewport(
            layout = figure_layout
        )
    )

    print(
        pA,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pB,
        vp = grid::viewport(
            layout.pos.row = 1,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    print(
        pC,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 1
        ),
        newpage = FALSE
    )

    print(
        pD,
        vp = grid::viewport(
            layout.pos.row = 2,
            layout.pos.col = 2
        ),
        newpage = FALSE
    )

    grid::popViewport()
}

# ==================================================================================================
# 18. SAVE PDF
# ==================================================================================================

grDevices::cairo_pdf(
    filename = PDF_PATH,
    width = 7.2,
    height = 5.9,
    family = FONT_NAME,
    pointsize = 9
)

draw_figure_s7()

grDevices::dev.off()

# ==================================================================================================
# 19. SAVE TIFF — 600 DPI
# ==================================================================================================

grDevices::tiff(
    filename = TIFF_PATH,
    width = 7.2,
    height = 5.9,
    units = "in",
    res = 600,
    compression = "lzw",
    pointsize = 9
)

draw_figure_s7()

grDevices::dev.off()

# ==================================================================================================
# 20. SAVE PNG — 300 DPI
# ==================================================================================================

grDevices::png(
    filename = PNG_PATH,
    width = 7.2,
    height = 5.9,
    units = "in",
    res = 300,
    pointsize = 9
)

draw_figure_s7()

grDevices::dev.off()

# ==================================================================================================
# 21. OUTPUT VALIDATION
# ==================================================================================================

outputs <- c(
    PDF_PATH,
    TIFF_PATH,
    PNG_PATH
)

if (!all(file.exists(outputs))) {
    stop("One or more S7 v3 figure outputs were not created.")
}

output_info <- file.info(
    outputs
)

if (
    any(
        is.na(output_info$size) |
        output_info$size <= 0
    )
) {
    stop("One or more S7 v3 figure outputs are zero-byte.")
}

cat(
    "Supplementary Figure S7 v3 outputs created and non-zero: PASS\n"
)

# ==================================================================================================
# 22. SUMMARY
# ==================================================================================================

summary_lines <- c(
    "==============================================================================================================",
    "STEP 11D.7S — SUPPLEMENTARY FIGURE S7 v3",
    "FINAL VISUAL CORRECTION",
    "==============================================================================================================",
    "",
    "Validated Figure S7 v2 source data reused.",
    "Analytical results changed: NO",
    "Model/CV analysis rerun: NO",
    "Calibration model fitted: NO",
    "",
    paste0(
        "Donor AUROC = ",
        sprintf("%.7f", DONOR_AUC)
    ),
    paste0(
        "Sample AUROC = ",
        sprintf("%.7f", SAMPLE_AUC)
    ),
    paste0(
        "DLPFC AUROC = ",
        sprintf("%.6f", DLPFC_AUC)
    ),
    paste0(
        "HIPPO AUROC = ",
        sprintf("%.6f", HIPPO_AUC)
    ),
    "",
    "Visual correction:",
    "Panel D title/subtitle shortened to prevent right-edge clipping.",
    "",
    paste0("PDF = ", PDF_PATH),
    paste0("TIFF = ", TIFF_PATH),
    paste0("PNG = ", PNG_PATH),
    "=============================================================================================================="
)

writeLines(
    summary_lines,
    SUMMARY_PATH,
    useBytes = TRUE
)

# ==================================================================================================
# 23. MANIFEST
# ==================================================================================================

manifest_files <- c(
    required_inputs,
    outputs,
    SUMMARY_PATH
)

manifest <- data.frame(
    file = basename(manifest_files),
    path = manifest_files,
    bytes = as.numeric(
        file.info(manifest_files)$size
    ),
    sha256 = vapply(
        manifest_files,
        function(p) {
            digest::digest(
                file = p,
                algo = "sha256",
                serialize = FALSE
            )
        },
        FUN.VALUE = character(1)
    ),
    stringsAsFactors = FALSE
)

write.csv(
    manifest,
    MANIFEST_PATH,
    row.names = FALSE,
    quote = TRUE
)

# ==================================================================================================
# 24. FINAL REPORT
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSUPPLEMENTARY FIGURE S7 v3 SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat("Validated S7 v2 source data: PASS\n")
cat("Analytical results changed: NO\n")
cat("Model/CV analysis rerun: NO\n")

cat(
    "Donor / sample AUROC: ",
    sprintf("%.7f", DONOR_AUC),
    " / ",
    sprintf("%.7f", SAMPLE_AUC),
    "\n",
    sep = ""
)

cat(
    "DLPFC / HIPPO AUROC: ",
    sprintf("%.6f", DLPFC_AUC),
    " / ",
    sprintf("%.6f", HIPPO_AUC),
    "\n",
    sep = ""
)

cat("Panel D clipping corrected: YES\n")

cat(
    "\nPDF:\n",
    PDF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nTIFF 600 dpi:\n",
    TIFF_PATH,
    "\n",
    sep = ""
)

cat(
    "\nPNG preview:\n",
    PNG_PATH,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11D.7S COMPLETE — SUPPLEMENTARY FIGURE S7 v3 GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11D.7S — SUPPLEMENTARY FIGURE S7 — FINAL VISUAL CORRECTION
LEAKAGE-FREE OOF DISCRIMINATION AND DESCRIPTIVE CALIBRATION
Project sentinel: PASS
Required packages: PASS
Figure font: Times New Roman
Validated Figure S7 v2 source data: PASS
Validated S7 source schemas: PASS
Validated dimensions: 454 donors / 755 samples / 10 calibration groups — PASS
Frozen AUROC anchors: donor 0.8019743; sample 0.7938032; DLPFC 0.785875; HIPPO 0.799746 — PASS


agg_record_f7433446be9 
                     2

agg_record_f7433446be9 
                     2

agg_record_f7433446be9 
                     2

Supplementary Figure S7 v3 outputs created and non-zero: PASS

SUPPLEMENTARY FIGURE S7 v3 SUMMARY
Validated S7 v2 source data: PASS
Analytical results changed: NO
Model/CV analysis rerun: NO
Donor / sample AUROC: 0.8019743 / 0.7938032
DLPFC / HIPPO AUROC: 0.785875 / 0.799746
Panel D clipping corrected: YES

PDF:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS7_OOF_Diagnostics_TNR_v3/FigureS7_OOF_Discrimination_Calibration_TimesNewRoman_v3.pdf

TIFF 600 dpi:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS7_OOF_Diagnostics_TNR_v3/FigureS7_OOF_Discrimination_Calibration_TimesNewRoman_v3.tiff

PNG preview:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/figures/supplementary/FigureS7_OOF_Diagnostics_TNR_v3/FigureS7_OOF_Discrimination_Calibration_TimesNewRoman_v3.png

STEP 11D.7S COMPLETE — SUPPLEMENTARY FIGURE S7 v3 GENERATED


In [21]:
# ==================================================================================================
# STEP 11E.1R — SUPPLEMENTARY TABLE S1
# R-KERNEL WRAPPER -> PYTHON-DOCX GENERATOR
# DISCOVERY COHORT, DONOR ARCHITECTURE, GENE UNIVERSE, AND QC SUMMARY
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.1R — SUPPLEMENTARY TABLE S1\n",
    "R-KERNEL WRAPPER -> PYTHON-DOCX GENERATOR\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 1. PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

sentinel_path <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(sentinel_path)) {
    stop("Project sentinel missing: ", sentinel_path)
}

observed_project_id <- trimws(
    readLines(
        sentinel_path,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    observed_project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "Project ID mismatch. Expected ",
        EXPECTED_PROJECT_ID,
        " but observed ",
        observed_project_id
    )
}

cat("Project sentinel: PASS\n")

# --------------------------------------------------------------------------------------------------
# 2. PYTHON EXECUTABLE
# --------------------------------------------------------------------------------------------------

PYTHON_EXE <- "E:/Python/python.exe"

if (!file.exists(PYTHON_EXE)) {
    stop(
        "Python executable not found: ",
        PYTHON_EXE
    )
}

cat("Python executable: PASS\n")

# --------------------------------------------------------------------------------------------------
# 3. OUTPUT DIRECTORY
# --------------------------------------------------------------------------------------------------

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "supplementary",
    "TableS1"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PY_SCRIPT_PATH <- file.path(
    OUT_DIR,
    "generate_TableS1_v1.py"
)

# --------------------------------------------------------------------------------------------------
# 4. COMPLETE PYTHON SCRIPT
# --------------------------------------------------------------------------------------------------

py_code <- r"(from pathlib import Path
import csv
import hashlib
import sys
from collections import defaultdict

try:
    from docx import Document
    from docx.shared import Inches, Pt
    from docx.enum.section import WD_ORIENT
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
except ImportError as exc:
    raise RuntimeError(
        "python-docx is unavailable. Install once with: "
        "E:/Python/python.exe -m pip install python-docx"
    ) from exc


print("=" * 110)
print("STEP 11E.1 — SUPPLEMENTARY TABLE S1")
print("DISCOVERY COHORT, DONOR ARCHITECTURE, GENE UNIVERSE, AND QC SUMMARY")
print("=" * 110)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT = Path("E:/SCZ_CROSSREGION_MSS_Q1")
EXPECTED_PROJECT_ID = "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL = PROJECT_ROOT / ".project_id"

if not SENTINEL.exists():
    raise FileNotFoundError(
        f"Project sentinel missing: {SENTINEL}"
    )

observed_project_id = SENTINEL.read_text(
    encoding="utf-8"
).strip()

if observed_project_id != EXPECTED_PROJECT_ID:
    raise RuntimeError(
        "Project sentinel mismatch.\n"
        f"Expected: {EXPECTED_PROJECT_ID}\n"
        f"Observed: {observed_project_id}"
    )

print("Project sentinel: PASS")

# ==================================================================================================
# 2. FROZEN PROJECT SOURCES USED FOR CONSISTENCY CHECKING
# ==================================================================================================

META_PATH = (
    PROJECT_ROOT
    / "04_processed"
    / "brainseq_model_metadata_v1.csv"
)

GENE_ANNOT_PATH = (
    PROJECT_ROOT
    / "04_processed"
    / "brainseq_primary_gene_annotation_v1.csv"
)

PCA_EXTREME_PATH = (
    PROJECT_ROOT
    / "02_metadata"
    / "step02b_PCA_extreme_samples_v1.csv"
)

required_project_sources = [
    META_PATH,
    GENE_ANNOT_PATH,
    PCA_EXTREME_PATH,
]

missing_sources = [
    str(p)
    for p in required_project_sources
    if not p.exists()
]

if missing_sources:
    raise FileNotFoundError(
        "Required frozen source artifact(s) missing:\n"
        + "\n".join(missing_sources)
    )

print("Frozen Table S1 project sources: PASS")

# ==================================================================================================
# 3. READ METADATA
# ==================================================================================================

with META_PATH.open(
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    reader = csv.DictReader(f)
    metadata = list(reader)
    meta_columns = reader.fieldnames or []

required_meta_columns = [
    "BrNum",
    "Region",
    "Dx",
    "donor_age",
    "Sex",
    "Race",
    "RIN",
]

missing_meta_columns = [
    c
    for c in required_meta_columns
    if c not in meta_columns
]

if missing_meta_columns:
    raise RuntimeError(
        "Metadata schema mismatch. Missing: "
        + ", ".join(missing_meta_columns)
    )

if len(metadata) != 755:
    raise RuntimeError(
        f"Expected 755 discovery samples; observed {len(metadata)}"
    )

# ==================================================================================================
# 4. NORMALIZE METADATA LABELS
# ==================================================================================================

def normalize_region(x):
    z = str(x).strip().upper()

    if z == "DLPFC":
        return "DLPFC"

    if z in {
        "HIPPO",
        "HIPPOCAMPUS",
        "HIPPOCAMPAL"
    }:
        return "HIPPO"

    raise RuntimeError(
        f"Unexpected region label: {x}"
    )


def normalize_dx(x):
    z = str(x).strip().lower()

    if z.startswith("control"):
        return "Control"

    if z.startswith("schiz"):
        return "Schizophrenia"

    raise RuntimeError(
        f"Unexpected diagnosis label: {x}"
    )


for row in metadata:
    row["_region"] = normalize_region(
        row["Region"]
    )

    row["_dx"] = normalize_dx(
        row["Dx"]
    )

    row["_donor"] = str(
        row["BrNum"]
    ).strip()

    if not row["_donor"]:
        raise RuntimeError(
            "Blank donor ID encountered."
        )

# ==================================================================================================
# 5. DISCOVERY COHORT COUNTS FROM FROZEN METADATA
# ==================================================================================================

donor_rows = defaultdict(list)

for row in metadata:
    donor_rows[
        row["_donor"]
    ].append(row)

N_SAMPLES = len(metadata)
N_DONORS = len(donor_rows)

N_DLPFC = sum(
    row["_region"] == "DLPFC"
    for row in metadata
)

N_HIPPO = sum(
    row["_region"] == "HIPPO"
    for row in metadata
)

N_DLPFC_CONTROL = sum(
    row["_region"] == "DLPFC"
    and row["_dx"] == "Control"
    for row in metadata
)

N_DLPFC_SCZ = sum(
    row["_region"] == "DLPFC"
    and row["_dx"] == "Schizophrenia"
    for row in metadata
)

N_HIPPO_CONTROL = sum(
    row["_region"] == "HIPPO"
    and row["_dx"] == "Control"
    for row in metadata
)

N_HIPPO_SCZ = sum(
    row["_region"] == "HIPPO"
    and row["_dx"] == "Schizophrenia"
    for row in metadata
)

paired_donors = []
dlpfc_only_donors = []
hippo_only_donors = []

donor_dx = {}

for donor_id, donor_samples in donor_rows.items():

    donor_regions = {
        x["_region"]
        for x in donor_samples
    }

    donor_diagnoses = {
        x["_dx"]
        for x in donor_samples
    }

    if len(donor_diagnoses) != 1:
        raise RuntimeError(
            f"Diagnosis inconsistency for donor {donor_id}"
        )

    donor_dx[donor_id] = next(
        iter(donor_diagnoses)
    )

    if donor_regions == {
        "DLPFC",
        "HIPPO"
    }:
        paired_donors.append(
            donor_id
        )

    elif donor_regions == {
        "DLPFC"
    }:
        dlpfc_only_donors.append(
            donor_id
        )

    elif donor_regions == {
        "HIPPO"
    }:
        hippo_only_donors.append(
            donor_id
        )

    else:
        raise RuntimeError(
            f"Unexpected region architecture for donor {donor_id}: "
            f"{sorted(donor_regions)}"
        )

single_region_donors = (
    dlpfc_only_donors
    + hippo_only_donors
)

N_PAIRED = len(
    paired_donors
)

N_SINGLE = len(
    single_region_donors
)

N_DLPFC_ONLY = len(
    dlpfc_only_donors
)

N_HIPPO_ONLY = len(
    hippo_only_donors
)

N_CONTROL_DONORS = sum(
    dx == "Control"
    for dx in donor_dx.values()
)

N_SCZ_DONORS = sum(
    dx == "Schizophrenia"
    for dx in donor_dx.values()
)

N_PAIRED_CONTROL = sum(
    donor_dx[d] == "Control"
    for d in paired_donors
)

N_PAIRED_SCZ = sum(
    donor_dx[d] == "Schizophrenia"
    for d in paired_donors
)

N_SINGLE_CONTROL = sum(
    donor_dx[d] == "Control"
    for d in single_region_donors
)

N_SINGLE_SCZ = sum(
    donor_dx[d] == "Schizophrenia"
    for d in single_region_donors
)

# ==================================================================================================
# 6. EXACT FROZEN COHORT GATES
# ==================================================================================================

expected_counts = {
    "samples": 755,
    "donors": 454,
    "DLPFC": 379,
    "HIPPO": 376,
    "DLPFC_Control": 226,
    "DLPFC_Schizophrenia": 153,
    "HIPPO_Control": 243,
    "HIPPO_Schizophrenia": 133,
    "paired": 301,
    "single": 153,
    "DLPFC_only": 78,
    "HIPPO_only": 75,
    "Control_donors": 277,
    "Schizophrenia_donors": 177,
    "paired_Control": 192,
    "paired_Schizophrenia": 109,
    "single_Control": 85,
    "single_Schizophrenia": 68,
}

observed_counts = {
    "samples": N_SAMPLES,
    "donors": N_DONORS,
    "DLPFC": N_DLPFC,
    "HIPPO": N_HIPPO,
    "DLPFC_Control": N_DLPFC_CONTROL,
    "DLPFC_Schizophrenia": N_DLPFC_SCZ,
    "HIPPO_Control": N_HIPPO_CONTROL,
    "HIPPO_Schizophrenia": N_HIPPO_SCZ,
    "paired": N_PAIRED,
    "single": N_SINGLE,
    "DLPFC_only": N_DLPFC_ONLY,
    "HIPPO_only": N_HIPPO_ONLY,
    "Control_donors": N_CONTROL_DONORS,
    "Schizophrenia_donors": N_SCZ_DONORS,
    "paired_Control": N_PAIRED_CONTROL,
    "paired_Schizophrenia": N_PAIRED_SCZ,
    "single_Control": N_SINGLE_CONTROL,
    "single_Schizophrenia": N_SINGLE_SCZ,
}

for key, expected in expected_counts.items():

    observed = observed_counts[key]

    if observed != expected:
        raise RuntimeError(
            f"Frozen cohort inconsistency for {key}: "
            f"observed {observed}, expected {expected}"
        )

print("Frozen cohort architecture: PASS")

# ==================================================================================================
# 7. DONOR-BALANCED WEIGHT CHECK
# ==================================================================================================

DONOR_BALANCED_TOTAL_WEIGHT = (
    N_PAIRED
    * 2
    * 0.5
    +
    N_SINGLE
    * 1.0
)

if abs(
    DONOR_BALANCED_TOTAL_WEIGHT
    - 454
) > 1e-12:
    raise RuntimeError(
        "Donor-balanced total weight no longer equals 454."
    )

print("Donor-balanced total weight = 454: PASS")

# ==================================================================================================
# 8. METADATA COMPLETENESS
# ==================================================================================================

def is_missing(value):
    z = str(value).strip()

    return (
        z == ""
        or z.upper() in {
            "NA",
            "NAN",
            "NULL",
            "NONE"
        }
    )


missing_age = sum(
    is_missing(
        row["donor_age"]
    )
    for row in metadata
)

missing_sex = sum(
    is_missing(
        row["Sex"]
    )
    for row in metadata
)

missing_race = sum(
    is_missing(
        row["Race"]
    )
    for row in metadata
)

missing_rin = sum(
    is_missing(
        row["RIN"]
    )
    for row in metadata
)

if any(
    x != 0
    for x in [
        missing_age,
        missing_sex,
        missing_race,
        missing_rin,
    ]
):
    raise RuntimeError(
        "Unexpected missing core covariate data."
    )

print("Core metadata completeness: PASS")

# ==================================================================================================
# 9. FINAL AUTOSOMAL GENE-UNIVERSE CHECK
# ==================================================================================================

with GENE_ANNOT_PATH.open(
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    gene_reader = csv.DictReader(f)
    gene_rows = list(gene_reader)

N_FINAL_GENES = len(
    gene_rows
)

if N_FINAL_GENES != 23740:
    raise RuntimeError(
        f"Expected 23,740 final autosomal genes; observed {N_FINAL_GENES}"
    )

print("Final autosomal gene universe = 23,740: PASS")

# ==================================================================================================
# 10. ROBUST PCA AUDIT-FLAG CHECK
# ==================================================================================================

with PCA_EXTREME_PATH.open(
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    pca_reader = csv.DictReader(f)
    pca_extreme_rows = list(
        pca_reader
    )

N_PCA_FLAGS = len(
    pca_extreme_rows
)

if N_PCA_FLAGS != 4:
    raise RuntimeError(
        f"Expected 4 robust PCA-distance audit flags; observed {N_PCA_FLAGS}"
    )

print("Robust PCA-distance audit flags = 4: PASS")

# ==================================================================================================
# 11. OUTPUT DIRECTORY
# ==================================================================================================

OUT_DIR = (
    PROJECT_ROOT
    / "11_manuscript"
    / "tables"
    / "supplementary"
    / "TableS1"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOCX_PATH = (
    OUT_DIR
    / "TableS1_Discovery_Cohort_Donor_Gene_QC_Summary_v1.docx"
)

CSV_PATH = (
    OUT_DIR
    / "TableS1_source_data_v1.csv"
)

NOTES_PATH = (
    OUT_DIR
    / "TableS1_notes_v1.txt"
)

SUMMARY_PATH = (
    OUT_DIR
    / "TableS1_generation_summary_v1.txt"
)

MANIFEST_PATH = (
    OUT_DIR
    / "TableS1_generation_manifest_v1.csv"
)

# ==================================================================================================
# 12. TABLE CONTENT
# ==================================================================================================

rows = [

    {
        "type": "section",
        "domain": "Discovery cohort",
        "item": "",
        "result": "",
        "note": "",
    },

    {
        "type": "data",
        "domain": "Cohort",
        "item": "Raw BrainSeq Phase II samples",
        "result": "900",
        "note": (
            "Starting sample pool before the frozen discovery-cohort restrictions."
        ),
    },

    {
        "type": "data",
        "domain": "Cohort",
        "item": "Final discovery samples",
        "result": f"{N_SAMPLES}",
        "note": (
            "Adult (>17 years) Control or schizophrenia samples from "
            "DLPFC or hippocampus; no descriptive-QC-driven sample exclusion."
        ),
    },

    {
        "type": "data",
        "domain": "Cohort",
        "item": "Unique donors",
        "result": f"{N_DONORS}",
        "note": (
            f"{N_CONTROL_DONORS} Control and "
            f"{N_SCZ_DONORS} schizophrenia donors."
        ),
    },

    {
        "type": "data",
        "domain": "Cohort",
        "item": "DLPFC samples",
        "result": f"{N_DLPFC}",
        "note": (
            f"{N_DLPFC_CONTROL} Control and "
            f"{N_DLPFC_SCZ} schizophrenia samples."
        ),
    },

    {
        "type": "data",
        "domain": "Cohort",
        "item": "Hippocampal samples",
        "result": f"{N_HIPPO}",
        "note": (
            f"{N_HIPPO_CONTROL} Control and "
            f"{N_HIPPO_SCZ} schizophrenia samples."
        ),
    },

    {
        "type": "data",
        "domain": "Cohort",
        "item": "Paired donors",
        "result": f"{N_PAIRED}",
        "note": (
            f"{N_PAIRED_CONTROL} Control and "
            f"{N_PAIRED_SCZ} schizophrenia donors; "
            "both DLPFC and hippocampus available."
        ),
    },

    {
        "type": "data",
        "domain": "Cohort",
        "item": "Single-region donors",
        "result": f"{N_SINGLE}",
        "note": (
            f"{N_SINGLE_CONTROL} Control and "
            f"{N_SINGLE_SCZ} schizophrenia donors; "
            f"{N_DLPFC_ONLY} DLPFC-only and "
            f"{N_HIPPO_ONLY} hippocampus-only."
        ),
    },

    {
        "type": "section",
        "domain": "Donor-aware weighting",
        "item": "",
        "result": "",
        "note": "",
    },

    {
        "type": "data",
        "domain": "Weighting",
        "item": "Paired-donor regional row weight",
        "result": "0.5",
        "note": (
            "Each of the two regional samples from a paired donor receives "
            "weight 0.5, for total donor contribution 1.0."
        ),
    },

    {
        "type": "data",
        "domain": "Weighting",
        "item": "Single-region row weight",
        "result": "1.0",
        "note": (
            "A donor represented by only one region contributes total weight 1.0."
        ),
    },

    {
        "type": "data",
        "domain": "Weighting",
        "item": "Total donor-balanced weight",
        "result": f"{DONOR_BALANCED_TOTAL_WEIGHT:.0f}",
        "note": (
            "Exactly equals the 454 unique discovery donors."
        ),
    },

    {
        "type": "section",
        "domain": "Gene-universe construction",
        "item": "",
        "result": "",
        "note": "",
    },

    {
        "type": "data",
        "domain": "Genes",
        "item": "Raw gene features",
        "result": "58,037",
        "note": (
            "Initial unfiltered BrainSeq gene-level feature space."
        ),
    },

    {
        "type": "data",
        "domain": "Genes",
        "item": "Expression-filtered genes",
        "result": "24,652",
        "note": (
            "Genes retained by the frozen expression-filtering procedure."
        ),
    },

    {
        "type": "data",
        "domain": "Genes",
        "item": "Stable unique Ensembl genes",
        "result": "24,646",
        "note": (
            "Six entries removed during stable, unique Ensembl-ID resolution."
        ),
    },

    {
        "type": "data",
        "domain": "Genes",
        "item": "Final autosomal analysis universe",
        "result": f"{N_FINAL_GENES:,}",
        "note": (
            "Frozen autosomal gene universe used for DE and predictive modelling."
        ),
    },

    {
        "type": "section",
        "domain": "Normalization and descriptive QC",
        "item": "",
        "result": "",
        "note": "",
    },

    {
        "type": "data",
        "domain": "QC",
        "item": "Normalization",
        "result": "TMM",
        "note": (
            "Trimmed mean of M-values normalization; normalized logCPM values "
            "were used for descriptive QC."
        ),
    },

    {
        "type": "data",
        "domain": "QC",
        "item": "PCA variance explained",
        "result": "PC1 20.1%; PC2 6.1%",
        "note": (
            "PCA was descriptive only and was not used as an exclusion rule."
        ),
    },

    {
        "type": "data",
        "domain": "QC",
        "item": "PC1-PC10 cumulative variance",
        "result": "43.1%",
        "note": (
            "Cumulative variance reported from the frozen TMM-logCPM PCA audit."
        ),
    },

    {
        "type": "data",
        "domain": "QC",
        "item": "Robust PCA-distance audit flags",
        "result": f"{N_PCA_FLAGS}",
        "note": (
            "Four robust extremes were flagged for audit; all were retained."
        ),
    },

    {
        "type": "data",
        "domain": "QC",
        "item": "QC-driven sample exclusions",
        "result": "0",
        "note": (
            "No sample was removed because of PCA or robust-distance extremeness."
        ),
    },

    {
        "type": "section",
        "domain": "Metadata completeness",
        "item": "",
        "result": "",
        "note": "",
    },

    {
        "type": "data",
        "domain": "Metadata",
        "item": "Missing donor age",
        "result": f"{missing_age} / {N_SAMPLES}",
        "note": "Complete across the discovery cohort.",
    },

    {
        "type": "data",
        "domain": "Metadata",
        "item": "Missing sex",
        "result": f"{missing_sex} / {N_SAMPLES}",
        "note": "Complete across the discovery cohort.",
    },

    {
        "type": "data",
        "domain": "Metadata",
        "item": "Missing race",
        "result": f"{missing_race} / {N_SAMPLES}",
        "note": "Complete across the discovery cohort.",
    },

    {
        "type": "data",
        "domain": "Metadata",
        "item": "Missing RIN",
        "result": f"{missing_rin} / {N_SAMPLES}",
        "note": "Complete across the discovery cohort.",
    },

    {
        "type": "data",
        "domain": "Metadata",
        "item": "Missing SNP PCs 1-10",
        "result": "0 / 755",
        "note": (
            "All ten genotype principal-component covariates were complete "
            "in the frozen model metadata."
        ),
    },
]

# ==================================================================================================
# 13. WORD / OOXML HELPERS
# ==================================================================================================

def set_run_font(
    run,
    size_pt=9,
    bold=False,
    italic=False
):

    run.font.name = "Times New Roman"
    run.font.size = Pt(
        size_pt
    )

    run.bold = bold
    run.italic = italic

    rPr = run._element.get_or_add_rPr()

    rFonts = rPr.rFonts

    if rFonts is None:
        rFonts = OxmlElement(
            "w:rFonts"
        )

        rPr.insert(
            0,
            rFonts
        )

    for attr in [
        "ascii",
        "hAnsi",
        "eastAsia",
        "cs",
    ]:
        rFonts.set(
            qn(
                f"w:{attr}"
            ),
            "Times New Roman"
        )


def set_cell_margins(
    cell,
    top=85,
    start=100,
    bottom=85,
    end=100
):

    tcPr = cell._tc.get_or_add_tcPr()

    tcMar = tcPr.first_child_found_in(
        "w:tcMar"
    )

    if tcMar is None:

        tcMar = OxmlElement(
            "w:tcMar"
        )

        tcPr.append(
            tcMar
        )

    for name, value in {
        "top": top,
        "start": start,
        "bottom": bottom,
        "end": end,
    }.items():

        node = tcMar.find(
            qn(
                f"w:{name}"
            )
        )

        if node is None:

            node = OxmlElement(
                f"w:{name}"
            )

            tcMar.append(
                node
            )

        node.set(
            qn("w:w"),
            str(value)
        )

        node.set(
            qn("w:type"),
            "dxa"
        )


def remove_cell_shading(cell):

    tcPr = cell._tc.get_or_add_tcPr()

    for node in list(
        tcPr.findall(
            qn("w:shd")
        )
    ):
        tcPr.remove(
            node
        )


def set_cell_width(
    cell,
    width_inches
):

    tcPr = cell._tc.get_or_add_tcPr()

    tcW = tcPr.first_child_found_in(
        "w:tcW"
    )

    if tcW is None:

        tcW = OxmlElement(
            "w:tcW"
        )

        tcPr.append(
            tcW
        )

    tcW.set(
        qn("w:w"),
        str(
            int(
                width_inches
                * 1440
            )
        )
    )

    tcW.set(
        qn("w:type"),
        "dxa"
    )


def set_cell_border(
    cell,
    size=8
):

    tcPr = cell._tc.get_or_add_tcPr()

    borders = tcPr.first_child_found_in(
        "w:tcBorders"
    )

    if borders is None:

        borders = OxmlElement(
            "w:tcBorders"
        )

        tcPr.append(
            borders
        )

    for edge_name in [
        "top",
        "bottom",
        "left",
        "right",
    ]:

        edge = borders.find(
            qn(
                f"w:{edge_name}"
            )
        )

        if edge is None:

            edge = OxmlElement(
                f"w:{edge_name}"
            )

            borders.append(
                edge
            )

        edge.set(
            qn("w:val"),
            "single"
        )

        edge.set(
            qn("w:sz"),
            str(size)
        )

        edge.set(
            qn("w:space"),
            "0"
        )

        edge.set(
            qn("w:color"),
            "000000"
        )


def set_table_borders(table):

    tblPr = table._tbl.tblPr

    borders = tblPr.first_child_found_in(
        "w:tblBorders"
    )

    if borders is None:

        borders = OxmlElement(
            "w:tblBorders"
        )

        tblPr.append(
            borders
        )

    border_sizes = {
        "top": 10,
        "bottom": 10,
        "left": 10,
        "right": 10,
        "insideH": 8,
        "insideV": 8,
    }

    for name, size in border_sizes.items():

        edge = borders.find(
            qn(
                f"w:{name}"
            )
        )

        if edge is None:

            edge = OxmlElement(
                f"w:{name}"
            )

            borders.append(
                edge
            )

        edge.set(
            qn("w:val"),
            "single"
        )

        edge.set(
            qn("w:sz"),
            str(size)
        )

        edge.set(
            qn("w:space"),
            "0"
        )

        edge.set(
            qn("w:color"),
            "000000"
        )


def set_fixed_table_layout(table):

    tblPr = table._tbl.tblPr

    layout = tblPr.first_child_found_in(
        "w:tblLayout"
    )

    if layout is None:

        layout = OxmlElement(
            "w:tblLayout"
        )

        tblPr.append(
            layout
        )

    layout.set(
        qn("w:type"),
        "fixed"
    )


def set_repeat_header(row):

    trPr = row._tr.get_or_add_trPr()

    tblHeader = OxmlElement(
        "w:tblHeader"
    )

    tblHeader.set(
        qn("w:val"),
        "true"
    )

    trPr.append(
        tblHeader
    )


def prevent_row_split(row):

    trPr = row._tr.get_or_add_trPr()

    cantSplit = OxmlElement(
        "w:cantSplit"
    )

    cantSplit.set(
        qn("w:val"),
        "true"
    )

    trPr.append(
        cantSplit
    )


def format_paragraph(
    paragraph,
    alignment,
    space_before=0,
    space_after=0
):

    paragraph.alignment = alignment

    pf = paragraph.paragraph_format

    pf.space_before = Pt(
        space_before
    )

    pf.space_after = Pt(
        space_after
    )

    pf.line_spacing = 1.0


# ==================================================================================================
# 14. CREATE DOCUMENT
# ==================================================================================================

doc = Document()

section = doc.sections[0]

section.orientation = WD_ORIENT.LANDSCAPE

section.page_width = Inches(
    11
)

section.page_height = Inches(
    8.5
)

section.top_margin = Inches(
    0.50
)

section.bottom_margin = Inches(
    0.50
)

section.left_margin = Inches(
    0.50
)

section.right_margin = Inches(
    0.50
)

# ==================================================================================================
# 15. DEFAULT FONT
# ==================================================================================================

normal = doc.styles[
    "Normal"
]

normal.font.name = (
    "Times New Roman"
)

normal.font.size = Pt(
    9
)

normal_rPr = (
    normal.element
    .get_or_add_rPr()
)

normal_fonts = (
    normal_rPr.rFonts
)

if normal_fonts is None:

    normal_fonts = OxmlElement(
        "w:rFonts"
    )

    normal_rPr.insert(
        0,
        normal_fonts
    )

for attr in [
    "ascii",
    "hAnsi",
    "eastAsia",
    "cs",
]:

    normal_fonts.set(
        qn(
            f"w:{attr}"
        ),
        "Times New Roman"
    )

# ==================================================================================================
# 16. TITLE
# ==================================================================================================

title = doc.add_paragraph()

format_paragraph(
    title,
    WD_ALIGN_PARAGRAPH.LEFT,
    space_before=0,
    space_after=5
)

title_run = title.add_run(
    "Supplementary Table S1. "
    "Discovery cohort, donor architecture, gene-universe construction, "
    "normalization, and descriptive quality-control summary."
)

set_run_font(
    title_run,
    size_pt=10.5,
    bold=True
)

# ==================================================================================================
# 17. TABLE
# ==================================================================================================

table = doc.add_table(
    rows=1,
    cols=4
)

table.alignment = (
    WD_TABLE_ALIGNMENT.CENTER
)

table.autofit = False

set_fixed_table_layout(
    table
)

set_table_borders(
    table
)

COLUMN_WIDTHS = [
    1.20,
    2.40,
    1.65,
    4.65,
]

headers = [
    "Domain",
    "Item",
    "Frozen result",
    "Analysis / QC note",
]

header_row = table.rows[0]

for col_idx, header_text in enumerate(
    headers
):

    cell = header_row.cells[
        col_idx
    ]

    set_cell_width(
        cell,
        COLUMN_WIDTHS[
            col_idx
        ]
    )

    set_cell_margins(
        cell
    )

    remove_cell_shading(
        cell
    )

    set_cell_border(
        cell,
        size=10
    )

    cell.vertical_alignment = (
        WD_CELL_VERTICAL_ALIGNMENT.CENTER
    )

    p = cell.paragraphs[0]

    if col_idx in [
        0,
        2,
    ]:
        alignment = (
            WD_ALIGN_PARAGRAPH.CENTER
        )
    else:
        alignment = (
            WD_ALIGN_PARAGRAPH.LEFT
        )

    format_paragraph(
        p,
        alignment
    )

    run = p.add_run(
        header_text
    )

    set_run_font(
        run,
        size_pt=9,
        bold=True
    )

set_repeat_header(
    header_row
)

prevent_row_split(
    header_row
)

# ==================================================================================================
# 18. BODY
# ==================================================================================================

for record in rows:

    row = table.add_row()

    prevent_row_split(
        row
    )

    values = [
        record["domain"],
        record["item"],
        record["result"],
        record["note"],
    ]

    for col_idx, value in enumerate(
        values
    ):

        cell = row.cells[
            col_idx
        ]

        set_cell_width(
            cell,
            COLUMN_WIDTHS[
                col_idx
            ]
        )

        set_cell_margins(
            cell
        )

        remove_cell_shading(
            cell
        )

        # Explicit border on EVERY cell:
        # visible horizontal + vertical grid.
        set_cell_border(
            cell,
            size=8
        )

        cell.vertical_alignment = (
            WD_CELL_VERTICAL_ALIGNMENT.CENTER
        )

        p = cell.paragraphs[0]

        if (
            record["type"]
            == "section"
        ):
            alignment = (
                WD_ALIGN_PARAGRAPH.LEFT
            )

        elif col_idx == 2:
            alignment = (
                WD_ALIGN_PARAGRAPH.CENTER
            )

        else:
            alignment = (
                WD_ALIGN_PARAGRAPH.LEFT
            )

        format_paragraph(
            p,
            alignment
        )

        run = p.add_run(
            str(value)
        )

        if (
            record["type"]
            == "section"
        ):

            set_run_font(
                run,
                size_pt=9,
                bold=True
            )

        else:

            set_run_font(
                run,
                size_pt=8.7,
                bold=False
            )

# ==================================================================================================
# 19. NOTES
# ==================================================================================================

notes_heading = doc.add_paragraph()

format_paragraph(
    notes_heading,
    WD_ALIGN_PARAGRAPH.LEFT,
    space_before=5,
    space_after=1
)

notes_heading_run = (
    notes_heading.add_run(
        "Notes:"
    )
)

set_run_font(
    notes_heading_run,
    size_pt=8.5,
    bold=True
)

notes_text = (
    "DLPFC, dorsolateral prefrontal cortex; HIPPO, hippocampus; "
    "RIN, RNA integrity number; TMM, trimmed mean of M-values; "
    "PCA, principal component analysis; PC, principal component. "
    "PCA and robust-distance screening were descriptive QC audits only; "
    "all four robust PCA-distance extremes were retained. "
    "For paired donors, each regional sample received weight 0.5, "
    "whereas single-region donors received weight 1.0, yielding "
    "a total donor-balanced weight of 454."
)

notes_p = doc.add_paragraph()

format_paragraph(
    notes_p,
    WD_ALIGN_PARAGRAPH.LEFT,
    space_before=0,
    space_after=0
)

notes_run = notes_p.add_run(
    notes_text
)

set_run_font(
    notes_run,
    size_pt=8.2
)

# ==================================================================================================
# 20. SAVE DOCX
# ==================================================================================================

doc.save(
    DOCX_PATH
)

if (
    not DOCX_PATH.exists()
    or DOCX_PATH.stat().st_size <= 0
):
    raise RuntimeError(
        "DOCX was not created correctly."
    )

print("Microsoft Word DOCX created: PASS")

# ==================================================================================================
# 21. STRUCTURAL RE-OPEN CHECK
# ==================================================================================================

check_doc = Document(
    DOCX_PATH
)

if len(
    check_doc.tables
) != 1:
    raise RuntimeError(
        "DOCX structural validation failed: expected exactly one table."
    )

check_table = check_doc.tables[0]

if len(
    check_table.columns
) != 4:
    raise RuntimeError(
        "DOCX structural validation failed: expected four columns."
    )

expected_table_rows = (
    1
    + len(rows)
)

if len(
    check_table.rows
) != expected_table_rows:
    raise RuntimeError(
        "DOCX structural validation failed: "
        f"expected {expected_table_rows} rows, "
        f"observed {len(check_table.rows)}."
    )

print(
    "DOCX structural read-back: PASS"
)

# ==================================================================================================
# 22. SOURCE CSV
# ==================================================================================================

with CSV_PATH.open(
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow(
        [
            "row_type",
            "Domain",
            "Item",
            "Frozen result",
            "Analysis / QC note",
        ]
    )

    for record in rows:

        writer.writerow(
            [
                record["type"],
                record["domain"],
                record["item"],
                record["result"],
                record["note"],
            ]
        )

# ==================================================================================================
# 23. NOTES TXT
# ==================================================================================================

NOTES_PATH.write_text(
    (
        "Supplementary Table S1\n"
        "========================================\n\n"
        + notes_text
        + "\n\n"
        + "Formatting specification:\n"
        + "- Microsoft Word DOCX\n"
        + "- Times New Roman\n"
        + "- Pure black-and-white formatting\n"
        + "- No cell shading\n"
        + "- Full horizontal row borders\n"
        + "- Full vertical column borders\n"
        + "- Landscape Letter page\n"
        + "- Repeating header row\n"
    ),
    encoding="utf-8"
)

# ==================================================================================================
# 24. GENERATION SUMMARY
# ==================================================================================================

summary_lines = [

    "=" * 110,

    "STEP 11E.1 — SUPPLEMENTARY TABLE S1",

    "=" * 110,

    "",

    "Project sentinel: PASS",

    "Frozen project-source consistency audit: PASS",

    "Analytical rerun: NO",

    "",

    "DISCOVERY COHORT",

    f"Final samples = {N_SAMPLES}",

    f"Unique donors = {N_DONORS}",

    f"DLPFC / HIPPO = {N_DLPFC} / {N_HIPPO}",

    f"Paired / single-region donors = {N_PAIRED} / {N_SINGLE}",

    (
        f"Control / schizophrenia donors = "
        f"{N_CONTROL_DONORS} / {N_SCZ_DONORS}"
    ),

    "",

    "DONOR ARCHITECTURE",

    (
        f"Paired Control / schizophrenia = "
        f"{N_PAIRED_CONTROL} / {N_PAIRED_SCZ}"
    ),

    (
        f"Single-region Control / schizophrenia = "
        f"{N_SINGLE_CONTROL} / {N_SINGLE_SCZ}"
    ),

    (
        f"DLPFC-only / HIPPO-only = "
        f"{N_DLPFC_ONLY} / {N_HIPPO_ONLY}"
    ),

    (
        f"Total donor-balanced weight = "
        f"{DONOR_BALANCED_TOTAL_WEIGHT:.0f}"
    ),

    "",

    "GENE UNIVERSE",

    "Raw genes = 58,037",

    "Expression-filtered = 24,652",

    "Stable unique Ensembl = 24,646",

    f"Final autosomal = {N_FINAL_GENES:,}",

    "",

    "QC",

    "PC1 variance = 20.1%",

    "PC2 variance = 6.1%",

    "PC1-PC10 cumulative variance = 43.1%",

    f"Robust PCA-distance flags = {N_PCA_FLAGS}",

    "QC-driven exclusions = 0",

    "",

    "FORMATTING",

    "Times New Roman = YES",

    "Color/shading = NONE",

    "Horizontal borders = YES",

    "Vertical column borders = YES",

    "Repeating table header = YES",

    "DOCX structural read-back = PASS",

    "",

    f"DOCX = {DOCX_PATH}",

    f"Source CSV = {CSV_PATH}",

    f"Notes = {NOTES_PATH}",

    "=" * 110,
]

SUMMARY_PATH.write_text(
    "\n".join(
        summary_lines
    ),
    encoding="utf-8"
)

# ==================================================================================================
# 25. SHA256 MANIFEST
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


manifest_inputs = [
    META_PATH,
    GENE_ANNOT_PATH,
    PCA_EXTREME_PATH,
]

manifest_outputs = [
    DOCX_PATH,
    CSV_PATH,
    NOTES_PATH,
    SUMMARY_PATH,
]

with MANIFEST_PATH.open(
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow(
        [
            "role",
            "file",
            "path",
            "bytes",
            "sha256",
        ]
    )

    for path in manifest_inputs:

        writer.writerow(
            [
                "input",
                path.name,
                str(path),
                path.stat().st_size,
                sha256_file(path),
            ]
        )

    for path in manifest_outputs:

        writer.writerow(
            [
                "output",
                path.name,
                str(path),
                path.stat().st_size,
                sha256_file(path),
            ]
        )

# ==================================================================================================
# 26. FINAL OUTPUT CHECK
# ==================================================================================================

final_outputs = [
    DOCX_PATH,
    CSV_PATH,
    NOTES_PATH,
    SUMMARY_PATH,
    MANIFEST_PATH,
]

for path in final_outputs:

    if (
        not path.exists()
        or path.stat().st_size <= 0
    ):
        raise RuntimeError(
            f"Missing or zero-byte output: {path}"
        )

print(
    "All Supplementary Table S1 outputs created and non-zero: PASS"
)

# ==================================================================================================
# 27. FINAL REPORT
# ==================================================================================================

print()
print("=" * 110)
print("SUPPLEMENTARY TABLE S1 GENERATION SUMMARY")
print("=" * 110)

print(
    "Frozen project-source consistency: PASS"
)

print(
    "Analytical results changed: NO"
)

print(
    "Times New Roman: PASS"
)

print(
    "Black-and-white only: PASS"
)

print(
    "Horizontal borders: PASS"
)

print(
    "Vertical column borders: PASS"
)

print(
    "DOCX structural read-back: PASS"
)

print()

print(
    "DOCX:"
)

print(
    DOCX_PATH
)

print()

print(
    "Source CSV:"
)

print(
    CSV_PATH
)

print()

print(
    "Notes:"
)

print(
    NOTES_PATH
)

print()

print(
    "Manifest:"
)

print(
    MANIFEST_PATH
)

print()

print("=" * 110)
print(
    "STEP 11E.1 COMPLETE — SUPPLEMENTARY TABLE S1 SAVED PROJECT-LOCALLY"
)
print("=" * 110)
)"

# --------------------------------------------------------------------------------------------------
# 5. SAVE PYTHON GENERATOR SCRIPT
# --------------------------------------------------------------------------------------------------

writeLines(
    py_code,
    PY_SCRIPT_PATH,
    useBytes = TRUE
)

if (
    !file.exists(PY_SCRIPT_PATH) ||
    file.info(PY_SCRIPT_PATH)$size <= 0
) {
    stop(
        "Python generator script was not created."
    )
}

cat(
    "Python generator script saved: ",
    PY_SCRIPT_PATH,
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 6. EXECUTE PYTHON FROM R KERNEL
# --------------------------------------------------------------------------------------------------

cmd_output <- system2(
    command = PYTHON_EXE,
    args = shQuote(
        PY_SCRIPT_PATH
    ),
    stdout = TRUE,
    stderr = TRUE
)

exit_status <- attr(
    cmd_output,
    "status"
)

cat(
    paste(
        cmd_output,
        collapse = "\n"
    ),
    "\n"
)

if (
    !is.null(exit_status) &&
    exit_status != 0
) {
    stop(
        "Python Table S1 generator failed with exit status ",
        exit_status,
        ". See console output above."
    )
}

# --------------------------------------------------------------------------------------------------
# 7. EXPECTED OUTPUT CHECK FROM R SIDE
# --------------------------------------------------------------------------------------------------

EXPECTED_DOCX <- file.path(
    OUT_DIR,
    "TableS1_Discovery_Cohort_Donor_Gene_QC_Summary_v1.docx"
)

EXPECTED_CSV <- file.path(
    OUT_DIR,
    "TableS1_source_data_v1.csv"
)

EXPECTED_NOTES <- file.path(
    OUT_DIR,
    "TableS1_notes_v1.txt"
)

EXPECTED_SUMMARY <- file.path(
    OUT_DIR,
    "TableS1_generation_summary_v1.txt"
)

EXPECTED_MANIFEST <- file.path(
    OUT_DIR,
    "TableS1_generation_manifest_v1.csv"
)

expected_outputs <- c(
    EXPECTED_DOCX,
    EXPECTED_CSV,
    EXPECTED_NOTES,
    EXPECTED_SUMMARY,
    EXPECTED_MANIFEST,
    PY_SCRIPT_PATH
)

if (!all(
    file.exists(
        expected_outputs
    )
)) {
    stop(
        "One or more expected Table S1 artifacts are missing."
    )
}

if (any(
    file.info(
        expected_outputs
    )$size <= 0
)) {
    stop(
        "One or more expected Table S1 artifacts are zero-byte."
    )
}

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.1R COMPLETE — TABLE S1 GENERATED THROUGH PYTHON FROM THE R KERNEL\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "\nMicrosoft Word DOCX:\n",
    EXPECTED_DOCX,
    "\n",
    sep = ""
)

cat(
    "\nSaved reproducible Python generator:\n",
    PY_SCRIPT_PATH,
    "\n",
    sep = ""
)

cat(
    "\nSource CSV:\n",
    EXPECTED_CSV,
    "\n",
    sep = ""
)

cat(
    "\nManifest:\n",
    EXPECTED_MANIFEST,
    "\n",
    sep = ""
)


STEP 11E.1R — SUPPLEMENTARY TABLE S1
R-KERNEL WRAPPER -> PYTHON-DOCX GENERATOR
Project sentinel: PASS
Python executable: PASS
Python generator script saved: E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/tables/supplementary/TableS1/generate_TableS1_v1.py
STEP 11E.1 � SUPPLEMENTARY TABLE S1
DISCOVERY COHORT, DONOR ARCHITECTURE, GENE UNIVERSE, AND QC SUMMARY
Project sentinel: PASS
Frozen Table S1 project sources: PASS
Frozen cohort architecture: PASS
Donor-balanced total weight = 454: PASS
Core metadata completeness: PASS
Final autosomal gene universe = 23,740: PASS
Robust PCA-distance audit flags = 4: PASS
Microsoft Word DOCX created: PASS
DOCX structural read-back: PASS
All Supplementary Table S1 outputs created and non-zero: PASS

SUPPLEMENTARY TABLE S1 GENERATION SUMMARY
Frozen project-source consistency: PASS
Analytical results changed: NO
Times New Roman: PASS
Black-and-white only: PASS
Horizontal borders: PASS
Vertical column borders: PASS
DOCX structural read-back: PASS

DOCX:
E:\SCZ_

In [23]:
# ==================================================================================================
# STEP 11E.2S - SUPPLEMENTARY TABLE S2 - FULLY CORRECTED
# R-KERNEL WRAPPER -> PYTHON-DOCX GENERATOR
# DONOR-BLOCKED DIFFERENTIAL-EXPRESSION MODEL AUDIT AND RESULTS
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.2S - SUPPLEMENTARY TABLE S2 - FULLY CORRECTED\n",
    "DONOR-BLOCKED DIFFERENTIAL-EXPRESSION MODEL AUDIT AND RESULTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 1. PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

observed_project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    observed_project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "Project ID mismatch. Expected ",
        EXPECTED_PROJECT_ID,
        " but observed ",
        observed_project_id
    )
}

cat("Project sentinel: PASS\n")

# --------------------------------------------------------------------------------------------------
# 2. PYTHON
# --------------------------------------------------------------------------------------------------

PYTHON_EXE <- "E:/Python/python.exe"

if (!file.exists(PYTHON_EXE)) {
    stop(
        "Python executable not found: ",
        PYTHON_EXE
    )
}

cat("Python executable: PASS\n")

# --------------------------------------------------------------------------------------------------
# 3. OUTPUT DIRECTORY
# --------------------------------------------------------------------------------------------------

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "supplementary",
    "TableS2"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PY_SCRIPT_PATH <- file.path(
    OUT_DIR,
    "generate_TableS2_v2.py"
)

# --------------------------------------------------------------------------------------------------
# 4. COMPLETE PYTHON SCRIPT
# --------------------------------------------------------------------------------------------------

py_code <- r"(from pathlib import Path
import csv
import hashlib
import math
from collections import defaultdict

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


print("=" * 110)
print("STEP 11E.2S - SUPPLEMENTARY TABLE S2 - FULLY CORRECTED")
print("DONOR-BLOCKED DIFFERENTIAL-EXPRESSION MODEL AUDIT AND RESULTS")
print("=" * 110)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT = Path("E:/SCZ_CROSSREGION_MSS_Q1")
EXPECTED_PROJECT_ID = "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL = PROJECT_ROOT / ".project_id"

if not SENTINEL.exists():
    raise FileNotFoundError(
        f"Project sentinel missing: {SENTINEL}"
    )

observed_project_id = SENTINEL.read_text(
    encoding="utf-8"
).strip()

if observed_project_id != EXPECTED_PROJECT_ID:
    raise RuntimeError(
        f"Project sentinel mismatch: {observed_project_id}"
    )

print("Project sentinel: PASS")

# ==================================================================================================
# 2. CANONICAL RESTORED STEP03A SOURCES
# ==================================================================================================

DE_DIR = (
    PROJECT_ROOT
    / "05_differential_expression"
)

DLPFC_PATH = (
    DE_DIR
    / "step03a_DLPFC_SCZ_vs_Control_v1.csv"
)

HIPPO_PATH = (
    DE_DIR
    / "step03a_HIPPO_SCZ_vs_Control_v1.csv"
)

SHARED_PATH = (
    DE_DIR
    / "step03a_Shared_SCZ_effect_v1.csv"
)

DEVIATION_PATH = (
    DE_DIR
    / "step03a_Regional_Deviation_v1.csv"
)

SUMMARY_INPUT_PATH = (
    DE_DIR
    / "step03a_DE_summary_v1.csv"
)

MODEL_AUDIT_PATH = (
    DE_DIR
    / "step03a_model_audit_v1.txt"
)

META_PATH = (
    PROJECT_ROOT
    / "04_processed"
    / "brainseq_model_metadata_v1.csv"
)

required_inputs = [
    DLPFC_PATH,
    HIPPO_PATH,
    SHARED_PATH,
    DEVIATION_PATH,
    SUMMARY_INPUT_PATH,
    MODEL_AUDIT_PATH,
    META_PATH,
]

missing_inputs = [
    str(p)
    for p in required_inputs
    if not p.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing canonical source artifact(s):\n"
        + "\n".join(missing_inputs)
    )

print("Canonical restored STEP03A files: PASS")

# ==================================================================================================
# 3. CSV READER
# ==================================================================================================

def read_csv_dict(path):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        reader = csv.DictReader(f)

        rows = list(
            reader
        )

        columns = reader.fieldnames or []

    return rows, columns


dlpfc, dlpfc_cols = read_csv_dict(
    DLPFC_PATH
)

hippo, hippo_cols = read_csv_dict(
    HIPPO_PATH
)

shared, shared_cols = read_csv_dict(
    SHARED_PATH
)

deviation, deviation_cols = read_csv_dict(
    DEVIATION_PATH
)

summary_rows, summary_cols = read_csv_dict(
    SUMMARY_INPUT_PATH
)

metadata, metadata_cols = read_csv_dict(
    META_PATH
)

# ==================================================================================================
# 4. NUMERIC HELPERS
# ==================================================================================================

def num(x):

    try:
        value = float(x)

    except Exception as exc:
        raise RuntimeError(
            f"Cannot parse numeric value: {x}"
        ) from exc

    if not math.isfinite(
        value
    ):
        raise RuntimeError(
            f"Non-finite numeric value: {x}"
        )

    return value


def integer_value(x):

    value = num(
        x
    )

    if abs(
        value
        - round(value)
    ) > 1e-8:
        raise RuntimeError(
            f"Expected integer-like value: {x}"
        )

    return int(
        round(value)
    )

# ==================================================================================================
# 5. DE SCHEMA
# ==================================================================================================

required_de_cols = [
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "logFC",
    "P.Value",
    "adj.P.Val",
]

for label, columns in [
    ("DLPFC", dlpfc_cols),
    ("HIPPO", hippo_cols),
    ("Shared", shared_cols),
    ("Deviation", deviation_cols),
]:

    missing = [
        x
        for x in required_de_cols
        if x not in columns
    ]

    if missing:
        raise RuntimeError(
            f"{label} DE schema missing fields: "
            + ", ".join(
                missing
            )
        )

for label, rows in [
    ("DLPFC", dlpfc),
    ("HIPPO", hippo),
    ("Shared", shared),
    ("Deviation", deviation),
]:

    if len(
        rows
    ) != 23740:
        raise RuntimeError(
            f"{label} table has {len(rows)} rows; expected 23740"
        )

    ids = [
        str(
            r["ensemblID_stable"]
        ).strip()
        for r in rows
    ]

    if len(
        set(ids)
    ) != 23740:
        raise RuntimeError(
            f"{label} table does not contain 23740 unique stable Ensembl IDs"
        )

print("Four canonical 23,740-gene DE tables: PASS")

# ==================================================================================================
# 6. ALIGNMENT
# ==================================================================================================

id_sets = []

for rows in [
    dlpfc,
    hippo,
    shared,
    deviation,
]:

    id_sets.append(
        {
            str(
                r["ensemblID_stable"]
            ).strip()
            for r in rows
        }
    )

if not (
    id_sets[0]
    == id_sets[1]
    == id_sets[2]
    == id_sets[3]
):
    raise RuntimeError(
        "Stable-Ensembl alignment across four DE tables failed."
    )

print("Stable-Ensembl alignment across four DE tables: PASS")

# ==================================================================================================
# 7. DIRECT SUMMARY RECOUNT
# ==================================================================================================

def summarize_de(rows):

    significant = [
        r
        for r in rows
        if num(
            r["adj.P.Val"]
        ) < 0.05
    ]

    return {
        "n_genes":
            len(rows),

        "FDR_lt_0.05":
            len(
                significant
            ),

        "FDR_lt_0.05_positive":
            sum(
                num(
                    r["logFC"]
                ) > 0
                for r in significant
            ),

        "FDR_lt_0.05_negative":
            sum(
                num(
                    r["logFC"]
                ) < 0
                for r in significant
            ),

        "P_lt_0.001":
            sum(
                num(
                    r["P.Value"]
                ) < 0.001
                for r in rows
            ),
    }


direct_summaries = {
    "DLPFC_SCZ":
        summarize_de(
            dlpfc
        ),

    "HIPPO_SCZ":
        summarize_de(
            hippo
        ),

    "Shared_SCZ":
        summarize_de(
            shared
        ),

    "Regional_Deviation":
        summarize_de(
            deviation
        ),
}

# ==================================================================================================
# 8. SAVED SUMMARY CHECK
# ==================================================================================================

required_summary_cols = [
    "contrast",
    "n_genes",
    "FDR_lt_0.05",
    "FDR_lt_0.05_positive",
    "FDR_lt_0.05_negative",
    "P_lt_0.001",
]

missing_summary_cols = [
    x
    for x in required_summary_cols
    if x not in summary_cols
]

if missing_summary_cols:
    raise RuntimeError(
        "STEP03A summary schema missing fields: "
        + ", ".join(
            missing_summary_cols
        )
    )

summary_by_contrast = {
    str(
        r["contrast"]
    ).strip():
        r
    for r in summary_rows
}

expected_contrasts = [
    "DLPFC_SCZ",
    "HIPPO_SCZ",
    "Shared_SCZ",
    "Regional_Deviation",
]

if set(
    summary_by_contrast.keys()
) != set(
    expected_contrasts
):
    raise RuntimeError(
        "Unexpected STEP03A contrast set."
    )

for contrast in expected_contrasts:

    saved = summary_by_contrast[
        contrast
    ]

    direct = direct_summaries[
        contrast
    ]

    for field in [
        "n_genes",
        "FDR_lt_0.05",
        "FDR_lt_0.05_positive",
        "FDR_lt_0.05_negative",
        "P_lt_0.001",
    ]:

        saved_value = integer_value(
            saved[
                field
            ]
        )

        direct_value = int(
            direct[
                field
            ]
        )

        if saved_value != direct_value:
            raise RuntimeError(
                f"DE summary mismatch for {contrast} / {field}: "
                f"saved={saved_value}, direct={direct_value}"
            )

print("Full DE tables vs saved DE summary: PASS")

# ==================================================================================================
# 9. CANONICAL DE COUNT ANCHORS
# ==================================================================================================

canonical_anchors = {
    "DLPFC_SCZ": {
        "FDR_lt_0.05": 2949,
        "FDR_lt_0.05_positive": 1312,
        "FDR_lt_0.05_negative": 1637,
        "P_lt_0.001": 1470,
    },

    "HIPPO_SCZ": {
        "FDR_lt_0.05": 2648,
        "FDR_lt_0.05_positive": 1349,
        "FDR_lt_0.05_negative": 1299,
        "P_lt_0.001": 1418,
    },

    "Shared_SCZ": {
        "FDR_lt_0.05": 5348,
        "FDR_lt_0.05_positive": 2539,
        "FDR_lt_0.05_negative": 2809,
        "P_lt_0.001": 2707,
    },

    "Regional_Deviation": {
        "FDR_lt_0.05": 11,
        "FDR_lt_0.05_positive": 4,
        "FDR_lt_0.05_negative": 7,
        "P_lt_0.001": 125,
    },
}

for contrast, anchors in canonical_anchors.items():

    observed = direct_summaries[
        contrast
    ]

    for field, expected in anchors.items():

        if int(
            observed[
                field
            ]
        ) != expected:

            raise RuntimeError(
                f"Canonical count changed for {contrast} / {field}"
            )

print("Canonical DE count anchors: PASS")

# ==================================================================================================
# 10. PARSE RESTORED MODEL AUDIT
# ==================================================================================================

audit_lines = MODEL_AUDIT_PATH.read_text(
    encoding="utf-8-sig"
).splitlines()

audit = {}

for line in audit_lines:

    line = line.strip()

    if (
        not line
        or "=" not in line
    ):
        continue

    key, value = line.split(
        "=",
        1
    )

    audit[
        key.strip()
    ] = value.strip()


# IMPORTANT:
# Restored canonical audit is intentionally allowed to be leaner than
# the originally executed notebook audit.
#
# We require ONLY fields actually present in the restored canonical file.

required_restored_audit_keys = [
    "samples",
    "donors",
    "design_columns",
    "duplicateCorrelation",
    "DLPFC_HIPPO_logFC_correlation",
    "DLPFC_HIPPO_sign_concordance",
]

missing_restored_keys = [
    x
    for x in required_restored_audit_keys
    if x not in audit
]

if missing_restored_keys:
    raise RuntimeError(
        "Restored canonical model audit missing required key(s): "
        + ", ".join(
            missing_restored_keys
        )
    )

AUDIT_SAMPLES = integer_value(
    audit[
        "samples"
    ]
)

AUDIT_DONORS = integer_value(
    audit[
        "donors"
    ]
)

DESIGN_COLUMNS = integer_value(
    audit[
        "design_columns"
    ]
)

DONOR_CORRELATION = num(
    audit[
        "duplicateCorrelation"
    ]
)

AUDIT_REGION_COR = num(
    audit[
        "DLPFC_HIPPO_logFC_correlation"
    ]
)

AUDIT_SIGN_CONCORDANCE = num(
    audit[
        "DLPFC_HIPPO_sign_concordance"
    ]
)

if AUDIT_SAMPLES != 755:
    raise RuntimeError(
        f"Canonical model-audit sample count changed: {AUDIT_SAMPLES}"
    )

if AUDIT_DONORS != 454:
    raise RuntimeError(
        f"Canonical model-audit donor count changed: {AUDIT_DONORS}"
    )

if DESIGN_COLUMNS != 20:
    raise RuntimeError(
        f"Canonical design-column count changed: {DESIGN_COLUMNS}"
    )

if abs(
    DONOR_CORRELATION
    - 0.188518383584
) > 1e-8:
    raise RuntimeError(
        f"Canonical donor correlation changed: {DONOR_CORRELATION}"
    )

print(
    "Restored canonical model-audit schema reconciled: PASS"
)

print(
    "design_rank absent from restored audit: accepted and not inserted into manuscript table"
)

print(
    "donor_balanced_total_weight absent from restored audit: will be recomputed from frozen metadata"
)

# ==================================================================================================
# 11. METADATA SCHEMA
# ==================================================================================================

required_meta_cols = [
    "BrNum",
    "Region",
    "donor_balance_weight",
]

missing_meta_cols = [
    x
    for x in required_meta_cols
    if x not in metadata_cols
]

if missing_meta_cols:
    raise RuntimeError(
        "Frozen metadata missing required field(s): "
        + ", ".join(
            missing_meta_cols
        )
    )

if len(
    metadata
) != 755:
    raise RuntimeError(
        f"Frozen metadata must contain 755 rows; observed {len(metadata)}"
    )

# ==================================================================================================
# 12. DONOR-BALANCED TOTAL WEIGHT FROM FROZEN METADATA
# ==================================================================================================

DONOR_BALANCED_WEIGHT = sum(
    num(
        r[
            "donor_balance_weight"
        ]
    )
    for r in metadata
)

if abs(
    DONOR_BALANCED_WEIGHT
    - 454.0
) > 1e-8:
    raise RuntimeError(
        f"Frozen metadata donor-balanced weight changed: {DONOR_BALANCED_WEIGHT}"
    )

print(
    "Donor-balanced total weight recomputed from frozen metadata = 454: PASS"
)

# ==================================================================================================
# 13. PAIRED DONOR ARCHITECTURE
# ==================================================================================================

donor_regions = defaultdict(
    set
)

for row in metadata:

    donor_id = str(
        row[
            "BrNum"
        ]
    ).strip()

    region_raw = str(
        row[
            "Region"
        ]
    ).strip().upper()

    if region_raw == "DLPFC":

        region = "DLPFC"

    elif region_raw in {
        "HIPPO",
        "HIPPOCAMPUS",
        "HIPPOCAMPAL",
    }:

        region = "HIPPO"

    else:

        raise RuntimeError(
            f"Unexpected region label: {region_raw}"
        )

    donor_regions[
        donor_id
    ].add(
        region
    )


PAIRED_DONORS = sum(
    regions
    == {
        "DLPFC",
        "HIPPO"
    }
    for regions in donor_regions.values()
)

PAIRED_SAMPLES = (
    PAIRED_DONORS
    * 2
)

if PAIRED_DONORS != 301:
    raise RuntimeError(
        f"Paired donor count changed: {PAIRED_DONORS}"
    )

if PAIRED_SAMPLES != 602:
    raise RuntimeError(
        f"Paired sample count changed: {PAIRED_SAMPLES}"
    )

print(
    "Paired donor architecture 301 donors / 602 samples: PASS"
)

# ==================================================================================================
# 14. CROSS-REGION EFFECT CORRELATION
# ==================================================================================================

dlpfc_map = {
    str(
        r[
            "ensemblID_stable"
        ]
    ).strip():
        num(
            r[
                "logFC"
            ]
        )
    for r in dlpfc
}

hippo_map = {
    str(
        r[
            "ensemblID_stable"
        ]
    ).strip():
        num(
            r[
                "logFC"
            ]
        )
    for r in hippo
}

ordered_ids = sorted(
    dlpfc_map.keys()
)

x = [
    dlpfc_map[
        gene_id
    ]
    for gene_id in ordered_ids
]

y = [
    hippo_map[
        gene_id
    ]
    for gene_id in ordered_ids
]


def pearson(a, b):

    mean_a = sum(
        a
    ) / len(a)

    mean_b = sum(
        b
    ) / len(b)

    numerator = sum(
        (
            ai - mean_a
        )
        *
        (
            bi - mean_b
        )
        for ai, bi in zip(
            a,
            b
        )
    )

    denominator = math.sqrt(
        sum(
            (
                ai - mean_a
            )
            ** 2
            for ai in a
        )
        *
        sum(
            (
                bi - mean_b
            )
            ** 2
            for bi in b
        )
    )

    return (
        numerator
        / denominator
    )


REGION_COR_DIRECT = pearson(
    x,
    y
)

SIGN_CONCORDANCE_DIRECT = sum(
    (
        ai > 0
        and bi > 0
    )
    or
    (
        ai < 0
        and bi < 0
    )
    for ai, bi in zip(
        x,
        y
    )
) / len(
    x
)

if abs(
    REGION_COR_DIRECT
    - 0.608869223845
) > 1e-10:
    raise RuntimeError(
        f"Canonical regional correlation changed: {REGION_COR_DIRECT}"
    )

if abs(
    SIGN_CONCORDANCE_DIRECT
    - 0.734962089301
) > 1e-10:
    raise RuntimeError(
        f"Canonical sign concordance changed: {SIGN_CONCORDANCE_DIRECT}"
    )

if abs(
    REGION_COR_DIRECT
    - AUDIT_REGION_COR
) > 1e-8:
    raise RuntimeError(
        "Restored model audit correlation disagrees with full canonical DE tables."
    )

if abs(
    SIGN_CONCORDANCE_DIRECT
    - AUDIT_SIGN_CONCORDANCE
) > 1e-8:
    raise RuntimeError(
        "Restored model audit sign concordance disagrees with full canonical DE tables."
    )

print(
    "Cross-region correlation and sign concordance: PASS"
)

# ==================================================================================================
# 15. PLD4 CANONICAL ANCHOR
# ==================================================================================================

pld4 = [
    r
    for r in shared
    if str(
        r[
            "gene_symbol"
        ]
    ).strip() == "PLD4"
]

if len(
    pld4
) != 1:
    raise RuntimeError(
        "Could not uniquely identify PLD4."
    )

PLD4_SHARED_LOGFC = num(
    pld4[
        0
    ][
        "logFC"
    ]
)

if abs(
    PLD4_SHARED_LOGFC
    - (-0.573257224039)
) > 1e-10:
    raise RuntimeError(
        f"Canonical PLD4 shared logFC changed: {PLD4_SHARED_LOGFC}"
    )

print(
    "PLD4 canonical shared logFC anchor: PASS"
)

# ==================================================================================================
# 16. ALL 11 FDR-SIGNIFICANT REGIONAL-DEVIATION GENES
# ==================================================================================================

deviation_sig = [
    dict(r)
    for r in deviation
    if num(
        r[
            "adj.P.Val"
        ]
    ) < 0.05
]

if len(
    deviation_sig
) != 11:
    raise RuntimeError(
        f"Expected 11 regional-deviation genes; observed {len(deviation_sig)}"
    )

deviation_sig.sort(
    key=lambda r: (
        num(
            r[
                "adj.P.Val"
            ]
        ),
        num(
            r[
                "P.Value"
            ]
        ),
        str(
            r[
                "ensemblID_stable"
            ]
        ),
    )
)

for r in deviation_sig:

    symbol = str(
        r[
            "gene_symbol"
        ]
    ).strip()

    if (
        not symbol
        or symbol.upper()
        in {
            "NA",
            "NAN",
            "NONE",
        }
    ):
        symbol = str(
            r[
                "ensemblID_stable"
            ]
        ).strip()

    r[
        "display_gene"
    ] = symbol

print(
    "All 11 FDR-significant regional-deviation genes: PASS"
)

# ==================================================================================================
# 17. TABLE S2A
# ==================================================================================================

table_s2a = [
    [
        "Discovery samples",
        "755",
        "Final donor-aware discovery cohort used for canonical STEP03A.",
    ],

    [
        "Unique donors",
        "454",
        "Donor was the repeated-measure blocking unit.",
    ],

    [
        "Autosomal genes tested",
        "23,740",
        "Same frozen analysis universe used for all four contrasts.",
    ],

    [
        "Design-matrix columns",
        str(
            DESIGN_COLUMNS
        ),
        "Canonical restored model audit reports 20 design columns.",
    ],

    [
        "Paired donors",
        str(
            PAIRED_DONORS
        ),
        "Both DLPFC and hippocampus available.",
    ],

    [
        "Paired regional samples",
        str(
            PAIRED_SAMPLES
        ),
        "Two regional samples from each of 301 paired donors.",
    ],

    [
        "Genes used to estimate donor correlation",
        "500",
        "Outcome-blind variance-stratified subset used for correlation estimation.",
    ],

    [
        "Consensus within-donor correlation",
        f"{DONOR_CORRELATION:.12f}",
        "Frozen correlation subsequently fixed for the full 23,740-gene limma fit.",
    ],

    [
        "Donor-balanced total weight",
        f"{DONOR_BALANCED_WEIGHT:.0f}",
        "Recomputed directly from the frozen donor_balance_weight column in model metadata.",
    ],

    [
        "DLPFC-HIPPO logFC correlation",
        f"{REGION_COR_DIRECT:.6f}",
        "Pearson correlation across all 23,740 aligned gene-level diagnosis effects.",
    ],

    [
        "DLPFC-HIPPO sign concordance",
        f"{SIGN_CONCORDANCE_DIRECT:.1%}",
        "Fraction of genes with effects having the same sign in both regions.",
    ],
]

# ==================================================================================================
# 18. TABLE S2B
# ==================================================================================================

display_names = {
    "DLPFC_SCZ":
        "DLPFC schizophrenia vs control",

    "HIPPO_SCZ":
        "Hippocampus schizophrenia vs control",

    "Shared_SCZ":
        "Shared schizophrenia component",

    "Regional_Deviation":
        "Regional-deviation component",
}

table_s2b = []

for contrast in expected_contrasts:

    s = direct_summaries[
        contrast
    ]

    table_s2b.append(
        [
            display_names[
                contrast
            ],

            f"{s['n_genes']:,}",

            f"{s['FDR_lt_0.05']:,}",

            f"{s['FDR_lt_0.05_positive']:,}",

            f"{s['FDR_lt_0.05_negative']:,}",

            f"{s['P_lt_0.001']:,}",
        ]
    )

# ==================================================================================================
# 19. TABLE S2C
# ==================================================================================================

table_s2c = []

for r in deviation_sig:

    table_s2c.append(
        [
            r[
                "display_gene"
            ],

            str(
                r[
                    "ensemblID_stable"
                ]
            ).strip(),

            str(
                r[
                    "gene_type"
                ]
            ).strip(),

            f"{num(r['logFC']):.6f}",

            f"{num(r['P.Value']):.3e}",

            f"{num(r['adj.P.Val']):.3e}",
        ]
    )

# ==================================================================================================
# 20. OUTPUT PATHS
# ==================================================================================================

OUT_DIR = (
    PROJECT_ROOT
    / "11_manuscript"
    / "tables"
    / "supplementary"
    / "TableS2"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOCX_PATH = (
    OUT_DIR
    / "TableS2_Differential_Expression_Model_Audit_v2.docx"
)

S2A_CSV_PATH = (
    OUT_DIR
    / "TableS2A_DE_model_audit_source_v2.csv"
)

S2B_CSV_PATH = (
    OUT_DIR
    / "TableS2B_DE_contrast_summary_source_v2.csv"
)

S2C_CSV_PATH = (
    OUT_DIR
    / "TableS2C_regional_deviation_FDR05_source_v2.csv"
)

NOTES_PATH = (
    OUT_DIR
    / "TableS2_notes_v2.txt"
)

SUMMARY_PATH = (
    OUT_DIR
    / "TableS2_generation_summary_v2.txt"
)

MANIFEST_PATH = (
    OUT_DIR
    / "TableS2_generation_manifest_v2.csv"
)

# ==================================================================================================
# 21. WRITE SOURCE CSVs
# ==================================================================================================

with S2A_CSV_PATH.open(
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow(
        [
            "Model / audit item",
            "Frozen result",
            "Implementation / interpretation note",
        ]
    )

    writer.writerows(
        table_s2a
    )


with S2B_CSV_PATH.open(
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow(
        [
            "Contrast",
            "Genes tested",
            "FDR < 0.05",
            "Positive FDR < 0.05",
            "Negative FDR < 0.05",
            "Nominal P < 0.001",
        ]
    )

    writer.writerows(
        table_s2b
    )


with S2C_CSV_PATH.open(
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow(
        [
            "Gene",
            "Stable Ensembl ID",
            "Gene type",
            "Regional-deviation logFC",
            "Nominal P",
            "FDR",
        ]
    )

    writer.writerows(
        table_s2c
    )

# ==================================================================================================
# 22. WORD HELPERS
# ==================================================================================================

def set_run_font(
    run,
    size_pt=9,
    bold=False,
    italic=False
):

    run.font.name = (
        "Times New Roman"
    )

    run.font.size = Pt(
        size_pt
    )

    run.bold = bold
    run.italic = italic

    rPr = run._element.get_or_add_rPr()

    rFonts = rPr.rFonts

    if rFonts is None:

        rFonts = OxmlElement(
            "w:rFonts"
        )

        rPr.insert(
            0,
            rFonts
        )

    for attr in [
        "ascii",
        "hAnsi",
        "eastAsia",
        "cs",
    ]:

        rFonts.set(
            qn(
                f"w:{attr}"
            ),
            "Times New Roman"
        )


def set_cell_margins(
    cell,
    top=85,
    start=95,
    bottom=85,
    end=95
):

    tcPr = cell._tc.get_or_add_tcPr()

    tcMar = tcPr.first_child_found_in(
        "w:tcMar"
    )

    if tcMar is None:

        tcMar = OxmlElement(
            "w:tcMar"
        )

        tcPr.append(
            tcMar
        )

    for name, value in {
        "top": top,
        "start": start,
        "bottom": bottom,
        "end": end,
    }.items():

        node = tcMar.find(
            qn(
                f"w:{name}"
            )
        )

        if node is None:

            node = OxmlElement(
                f"w:{name}"
            )

            tcMar.append(
                node
            )

        node.set(
            qn(
                "w:w"
            ),
            str(
                value
            )
        )

        node.set(
            qn(
                "w:type"
            ),
            "dxa"
        )


def remove_shading(
    cell
):

    tcPr = cell._tc.get_or_add_tcPr()

    for node in list(
        tcPr.findall(
            qn(
                "w:shd"
            )
        )
    ):

        tcPr.remove(
            node
        )


def set_cell_width(
    cell,
    width_inches
):

    tcPr = cell._tc.get_or_add_tcPr()

    tcW = tcPr.first_child_found_in(
        "w:tcW"
    )

    if tcW is None:

        tcW = OxmlElement(
            "w:tcW"
        )

        tcPr.append(
            tcW
        )

    tcW.set(
        qn(
            "w:w"
        ),
        str(
            int(
                width_inches
                * 1440
            )
        )
    )

    tcW.set(
        qn(
            "w:type"
        ),
        "dxa"
    )


def set_cell_border(
    cell,
    size=8
):

    tcPr = cell._tc.get_or_add_tcPr()

    borders = tcPr.first_child_found_in(
        "w:tcBorders"
    )

    if borders is None:

        borders = OxmlElement(
            "w:tcBorders"
        )

        tcPr.append(
            borders
        )

    for edge_name in [
        "top",
        "bottom",
        "left",
        "right",
    ]:

        edge = borders.find(
            qn(
                f"w:{edge_name}"
            )
        )

        if edge is None:

            edge = OxmlElement(
                f"w:{edge_name}"
            )

            borders.append(
                edge
            )

        edge.set(
            qn(
                "w:val"
            ),
            "single"
        )

        edge.set(
            qn(
                "w:sz"
            ),
            str(
                size
            )
        )

        edge.set(
            qn(
                "w:space"
            ),
            "0"
        )

        edge.set(
            qn(
                "w:color"
            ),
            "000000"
        )


def set_table_borders(
    table
):

    tblPr = table._tbl.tblPr

    borders = tblPr.first_child_found_in(
        "w:tblBorders"
    )

    if borders is None:

        borders = OxmlElement(
            "w:tblBorders"
        )

        tblPr.append(
            borders
        )

    specs = {
        "top": 10,
        "bottom": 10,
        "left": 10,
        "right": 10,
        "insideH": 8,
        "insideV": 8,
    }

    for name, size in specs.items():

        edge = borders.find(
            qn(
                f"w:{name}"
            )
        )

        if edge is None:

            edge = OxmlElement(
                f"w:{name}"
            )

            borders.append(
                edge
            )

        edge.set(
            qn(
                "w:val"
            ),
            "single"
        )

        edge.set(
            qn(
                "w:sz"
            ),
            str(
                size
            )
        )

        edge.set(
            qn(
                "w:space"
            ),
            "0"
        )

        edge.set(
            qn(
                "w:color"
            ),
            "000000"
        )


def fixed_layout(
    table
):

    tblPr = table._tbl.tblPr

    layout = tblPr.first_child_found_in(
        "w:tblLayout"
    )

    if layout is None:

        layout = OxmlElement(
            "w:tblLayout"
        )

        tblPr.append(
            layout
        )

    layout.set(
        qn(
            "w:type"
        ),
        "fixed"
    )


def repeat_header(
    row
):

    trPr = row._tr.get_or_add_trPr()

    node = OxmlElement(
        "w:tblHeader"
    )

    node.set(
        qn(
            "w:val"
        ),
        "true"
    )

    trPr.append(
        node
    )


def prevent_split(
    row
):

    trPr = row._tr.get_or_add_trPr()

    node = OxmlElement(
        "w:cantSplit"
    )

    node.set(
        qn(
            "w:val"
        ),
        "true"
    )

    trPr.append(
        node
    )


def format_paragraph(
    paragraph,
    alignment=WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=0
):

    paragraph.alignment = (
        alignment
    )

    pf = paragraph.paragraph_format

    pf.space_before = Pt(
        before
    )

    pf.space_after = Pt(
        after
    )

    pf.line_spacing = 1.0


def add_heading(
    document,
    text
):

    p = document.add_paragraph()

    format_paragraph(
        p,
        WD_ALIGN_PARAGRAPH.LEFT,
        before=5,
        after=3
    )

    run = p.add_run(
        text
    )

    set_run_font(
        run,
        size_pt=9.5,
        bold=True
    )


def make_table(
    document,
    headers,
    body_rows,
    widths,
    center_columns=None,
    font_size=8.4
):

    if center_columns is None:
        center_columns = set()

    table = document.add_table(
        rows=1,
        cols=len(
            headers
        )
    )

    table.alignment = (
        WD_TABLE_ALIGNMENT.CENTER
    )

    table.autofit = False

    fixed_layout(
        table
    )

    set_table_borders(
        table
    )

    header_row = table.rows[
        0
    ]

    for j, header_text in enumerate(
        headers
    ):

        cell = header_row.cells[
            j
        ]

        set_cell_width(
            cell,
            widths[
                j
            ]
        )

        set_cell_margins(
            cell
        )

        remove_shading(
            cell
        )

        set_cell_border(
            cell,
            size=10
        )

        cell.vertical_alignment = (
            WD_CELL_VERTICAL_ALIGNMENT.CENTER
        )

        p = cell.paragraphs[
            0
        ]

        alignment = (
            WD_ALIGN_PARAGRAPH.CENTER
            if j in center_columns
            else WD_ALIGN_PARAGRAPH.LEFT
        )

        format_paragraph(
            p,
            alignment
        )

        run = p.add_run(
            str(
                header_text
            )
        )

        set_run_font(
            run,
            size_pt=8.7,
            bold=True
        )

    repeat_header(
        header_row
    )

    prevent_split(
        header_row
    )

    for values in body_rows:

        row = table.add_row()

        prevent_split(
            row
        )

        for j, value in enumerate(
            values
        ):

            cell = row.cells[
                j
            ]

            set_cell_width(
                cell,
                widths[
                    j
                ]
            )

            set_cell_margins(
                cell
            )

            remove_shading(
                cell
            )

            set_cell_border(
                cell,
                size=8
            )

            cell.vertical_alignment = (
                WD_CELL_VERTICAL_ALIGNMENT.CENTER
            )

            p = cell.paragraphs[
                0
            ]

            alignment = (
                WD_ALIGN_PARAGRAPH.CENTER
                if j in center_columns
                else WD_ALIGN_PARAGRAPH.LEFT
            )

            format_paragraph(
                p,
                alignment
            )

            run = p.add_run(
                str(
                    value
                )
            )

            set_run_font(
                run,
                size_pt=font_size
            )

    return table

# ==================================================================================================
# 23. CREATE DOCUMENT
# ==================================================================================================

doc = Document()

section = doc.sections[
    0
]

section.orientation = (
    WD_ORIENT.LANDSCAPE
)

section.page_width = Inches(
    11
)

section.page_height = Inches(
    8.5
)

section.top_margin = Inches(
    0.48
)

section.bottom_margin = Inches(
    0.48
)

section.left_margin = Inches(
    0.48
)

section.right_margin = Inches(
    0.48
)

# ==================================================================================================
# 24. DEFAULT FONT
# ==================================================================================================

normal = doc.styles[
    "Normal"
]

normal.font.name = (
    "Times New Roman"
)

normal.font.size = Pt(
    9
)

normal_rPr = (
    normal.element.get_or_add_rPr()
)

normal_fonts = normal_rPr.rFonts

if normal_fonts is None:

    normal_fonts = OxmlElement(
        "w:rFonts"
    )

    normal_rPr.insert(
        0,
        normal_fonts
    )

for attr in [
    "ascii",
    "hAnsi",
    "eastAsia",
    "cs",
]:

    normal_fonts.set(
        qn(
            f"w:{attr}"
        ),
        "Times New Roman"
    )

# ==================================================================================================
# 25. TITLE
# ==================================================================================================

title = doc.add_paragraph()

format_paragraph(
    title,
    WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=4
)

title_run = title.add_run(
    "Supplementary Table S2. "
    "Donor-blocked differential-expression model audit, "
    "contrast-level results, and FDR-significant regional-deviation genes."
)

set_run_font(
    title_run,
    size_pt=10.5,
    bold=True
)

# ==================================================================================================
# 26. PART A
# ==================================================================================================

add_heading(
    doc,
    "A. Differential-expression model and audit summary"
)

make_table(
    doc,
    headers=[
        "Model / audit item",
        "Frozen result",
        "Implementation / interpretation note",
    ],
    body_rows=table_s2a,
    widths=[
        2.65,
        1.85,
        5.65,
    ],
    center_columns={
        1
    },
    font_size=8.4
)

# ==================================================================================================
# 27. PART B
# ==================================================================================================

add_heading(
    doc,
    "B. Differential-expression results across the four frozen contrasts"
)

make_table(
    doc,
    headers=[
        "Contrast",
        "Genes tested",
        "FDR < 0.05",
        "Positive FDR < 0.05",
        "Negative FDR < 0.05",
        "Nominal P < 0.001",
    ],
    body_rows=table_s2b,
    widths=[
        2.80,
        1.15,
        1.20,
        1.55,
        1.55,
        1.55,
    ],
    center_columns={
        1,
        2,
        3,
        4,
        5,
    },
    font_size=8.4
)

# ==================================================================================================
# 28. PART C
# ==================================================================================================

add_heading(
    doc,
    "C. FDR-significant regional-deviation genes"
)

make_table(
    doc,
    headers=[
        "Gene",
        "Stable Ensembl ID",
        "Gene type",
        "Regional-deviation logFC",
        "Nominal P",
        "FDR",
    ],
    body_rows=table_s2c,
    widths=[
        1.45,
        2.25,
        1.60,
        1.55,
        1.50,
        1.50,
    ],
    center_columns={
        3,
        4,
        5,
    },
    font_size=8.25
)

# ==================================================================================================
# 29. NOTES
# ==================================================================================================

notes_heading = doc.add_paragraph()

format_paragraph(
    notes_heading,
    WD_ALIGN_PARAGRAPH.LEFT,
    before=5,
    after=1
)

run = notes_heading.add_run(
    "Notes:"
)

set_run_font(
    run,
    size_pt=8.4,
    bold=True
)

notes_text = (
    "DLPFC, dorsolateral prefrontal cortex; FDR, false-discovery rate. "
    "Positive and negative counts refer to the sign of the saved coefficient within each contrast. "
    "For the regional-deviation contrast, coefficient sign refers to the frozen regional-deviation "
    "parameterization and is not a standalone diagnosis effect in either region. "
    "Within-donor correlation was estimated using 500 outcome-blind variance-stratified genes and "
    "then fixed for the full 23,740-gene donor-blocked limma analysis. "
    "The restored canonical model-audit file does not contain a design-rank field; therefore no "
    "design-rank value is inserted into this supplementary table. "
    "Donor-balanced total weight was independently verified from the frozen model metadata."
)

notes_p = doc.add_paragraph()

format_paragraph(
    notes_p,
    WD_ALIGN_PARAGRAPH.LEFT
)

notes_run = notes_p.add_run(
    notes_text
)

set_run_font(
    notes_run,
    size_pt=8.1
)

# ==================================================================================================
# 30. SAVE DOCX
# ==================================================================================================

doc.save(
    DOCX_PATH
)

if (
    not DOCX_PATH.exists()
    or DOCX_PATH.stat().st_size <= 0
):
    raise RuntimeError(
        "Table S2 DOCX was not created."
    )

print(
    "Microsoft Word DOCX created: PASS"
)

# ==================================================================================================
# 31. STRUCTURAL READ-BACK
# ==================================================================================================

check_doc = Document(
    DOCX_PATH
)

if len(
    check_doc.tables
) != 3:
    raise RuntimeError(
        f"Expected 3 Word tables; observed {len(check_doc.tables)}"
    )

expected_columns = [
    3,
    6,
    6,
]

for idx, expected_ncol in enumerate(
    expected_columns
):

    observed_ncol = len(
        check_doc.tables[
            idx
        ].columns
    )

    if observed_ncol != expected_ncol:
        raise RuntimeError(
            f"Word table {idx + 1} column-count failure."
        )

if len(
    check_doc.tables[
        2
    ].rows
) != 12:
    raise RuntimeError(
        "Part C must contain 1 header + 11 regional-deviation genes."
    )

print(
    "DOCX structural read-back: PASS"
)

# ==================================================================================================
# 32. NOTES TXT
# ==================================================================================================

NOTES_PATH.write_text(
    (
        "Supplementary Table S2\n"
        "========================================\n\n"
        + notes_text
        + "\n\n"
        + "Formatting:\n"
        + "- Microsoft Word DOCX\n"
        + "- Times New Roman\n"
        + "- Black and white only\n"
        + "- No cell shading\n"
        + "- Full horizontal borders\n"
        + "- Full vertical borders\n"
        + "- Landscape Letter layout\n"
        + "- Repeating table headers\n"
    ),
    encoding="utf-8"
)

# ==================================================================================================
# 33. SUMMARY
# ==================================================================================================

summary_lines = [
    "=" * 110,
    "STEP 11E.2S - SUPPLEMENTARY TABLE S2",
    "=" * 110,
    "",
    "Canonical restored STEP03A sources: PASS",
    "DE analysis rerun: NO",
    "duplicateCorrelation rerun: NO",
    "",
    "RESTORED AUDIT RECONCILIATION",
    "design_rank key absent: accepted; not inserted into table",
    "donor_balanced_total_weight key absent: recomputed from frozen metadata",
    "",
    f"Samples = {AUDIT_SAMPLES}",
    f"Donors = {AUDIT_DONORS}",
    f"Design columns = {DESIGN_COLUMNS}",
    f"Paired donors / samples = {PAIRED_DONORS} / {PAIRED_SAMPLES}",
    "Correlation-estimation genes = 500",
    f"Consensus donor correlation = {DONOR_CORRELATION:.12f}",
    f"Donor-balanced total weight = {DONOR_BALANCED_WEIGHT:.0f}",
    "",
    "DE RESULTS",
    "DLPFC FDR < 0.05 = 2949",
    "HIPPO FDR < 0.05 = 2648",
    "Shared FDR < 0.05 = 5348",
    "Regional deviation FDR < 0.05 = 11",
    f"Regional logFC correlation = {REGION_COR_DIRECT:.12f}",
    f"Regional sign concordance = {SIGN_CONCORDANCE_DIRECT:.12f}",
    f"PLD4 shared logFC = {PLD4_SHARED_LOGFC:.12f}",
    "",
    "FORMATTING",
    "Times New Roman = YES",
    "Color/shading = NONE",
    "Horizontal borders = YES",
    "Vertical borders = YES",
    "DOCX structural read-back = PASS",
    "",
    f"DOCX = {DOCX_PATH}",
    f"S2A CSV = {S2A_CSV_PATH}",
    f"S2B CSV = {S2B_CSV_PATH}",
    f"S2C CSV = {S2C_CSV_PATH}",
    "=" * 110,
]

SUMMARY_PATH.write_text(
    "\n".join(
        summary_lines
    ),
    encoding="utf-8"
)

# ==================================================================================================
# 34. SHA256 MANIFEST
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024
                * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


manifest_outputs = [
    DOCX_PATH,
    S2A_CSV_PATH,
    S2B_CSV_PATH,
    S2C_CSV_PATH,
    NOTES_PATH,
    SUMMARY_PATH,
]

with MANIFEST_PATH.open(
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow(
        [
            "role",
            "file",
            "path",
            "bytes",
            "sha256",
        ]
    )

    for path in required_inputs:

        writer.writerow(
            [
                "input",
                path.name,
                str(
                    path
                ),
                path.stat().st_size,
                sha256_file(
                    path
                ),
            ]
        )

    for path in manifest_outputs:

        writer.writerow(
            [
                "output",
                path.name,
                str(
                    path
                ),
                path.stat().st_size,
                sha256_file(
                    path
                ),
            ]
        )

# ==================================================================================================
# 35. FINAL OUTPUT CHECK
# ==================================================================================================

final_outputs = [
    DOCX_PATH,
    S2A_CSV_PATH,
    S2B_CSV_PATH,
    S2C_CSV_PATH,
    NOTES_PATH,
    SUMMARY_PATH,
    MANIFEST_PATH,
]

for path in final_outputs:

    if (
        not path.exists()
        or path.stat().st_size <= 0
    ):
        raise RuntimeError(
            f"Missing or zero-byte output: {path}"
        )

print(
    "All Supplementary Table S2 outputs created and non-zero: PASS"
)

# ==================================================================================================
# 36. FINAL REPORT
# ==================================================================================================

print()
print("=" * 110)
print("SUPPLEMENTARY TABLE S2 v2 GENERATION SUMMARY")
print("=" * 110)

print(
    "Canonical restored STEP03A state: PASS"
)

print(
    "DE analysis rerun: NO"
)

print(
    "duplicateCorrelation rerun: NO"
)

print()

print(
    "Restored audit schema reconciliation: PASS"
)

print(
    "design_rank omitted because absent from restored canonical audit"
)

print(
    "donor-balanced total weight recomputed from frozen metadata: "
    f"{DONOR_BALANCED_WEIGHT:.0f}"
)

print()

print(
    "FDR counts DLPFC / HIPPO / Shared / Deviation: "
    "2949 / 2648 / 5348 / 11"
)

print(
    "Positive FDR counts: "
    "1312 / 1349 / 2539 / 4"
)

print(
    "Negative FDR counts: "
    "1637 / 1299 / 2809 / 7"
)

print(
    "P < 0.001 counts: "
    "1470 / 1418 / 2707 / 125"
)

print()

print(
    "Consensus donor correlation: "
    f"{DONOR_CORRELATION:.12f}"
)

print(
    "Regional logFC correlation: "
    f"{REGION_COR_DIRECT:.12f}"
)

print(
    "Regional sign concordance: "
    f"{SIGN_CONCORDANCE_DIRECT:.12f}"
)

print(
    "Regional-deviation genes listed: "
    f"{len(deviation_sig)}"
)

print()

print("Times New Roman: PASS")
print("Black-and-white only: PASS")
print("Full row borders: PASS")
print("Full column borders: PASS")
print("DOCX structural read-back: PASS")

print()

print("DOCX:")
print(DOCX_PATH)

print()

print("Source CSVs:")
print(S2A_CSV_PATH)
print(S2B_CSV_PATH)
print(S2C_CSV_PATH)

print()

print("Manifest:")
print(MANIFEST_PATH)

print()
print("=" * 110)
print("STEP 11E.2S COMPLETE - SUPPLEMENTARY TABLE S2 v2 SAVED PROJECT-LOCALLY")
print("=" * 110)
)"

# --------------------------------------------------------------------------------------------------
# 5. SAVE PYTHON GENERATOR
# --------------------------------------------------------------------------------------------------

writeLines(
    py_code,
    PY_SCRIPT_PATH,
    useBytes = TRUE
)

if (
    !file.exists(PY_SCRIPT_PATH) ||
    file.info(PY_SCRIPT_PATH)$size <= 0
) {
    stop(
        "Corrected Table S2 Python generator was not created."
    )
}

cat(
    "Corrected Python generator saved: ",
    PY_SCRIPT_PATH,
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 6. EXECUTE PYTHON
# --------------------------------------------------------------------------------------------------

cmd_output <- system2(
    command = PYTHON_EXE,
    args = shQuote(
        PY_SCRIPT_PATH
    ),
    stdout = TRUE,
    stderr = TRUE
)

exit_status <- attr(
    cmd_output,
    "status"
)

cat(
    paste(
        cmd_output,
        collapse = "\n"
    ),
    "\n"
)

if (
    !is.null(exit_status) &&
    exit_status != 0
) {
    stop(
        "Corrected Python Table S2 generator failed with exit status ",
        exit_status,
        ". See console output above."
    )
}

# --------------------------------------------------------------------------------------------------
# 7. R-SIDE OUTPUT VERIFICATION
# --------------------------------------------------------------------------------------------------

EXPECTED_DOCX <- file.path(
    OUT_DIR,
    "TableS2_Differential_Expression_Model_Audit_v2.docx"
)

EXPECTED_S2A <- file.path(
    OUT_DIR,
    "TableS2A_DE_model_audit_source_v2.csv"
)

EXPECTED_S2B <- file.path(
    OUT_DIR,
    "TableS2B_DE_contrast_summary_source_v2.csv"
)

EXPECTED_S2C <- file.path(
    OUT_DIR,
    "TableS2C_regional_deviation_FDR05_source_v2.csv"
)

EXPECTED_NOTES <- file.path(
    OUT_DIR,
    "TableS2_notes_v2.txt"
)

EXPECTED_SUMMARY <- file.path(
    OUT_DIR,
    "TableS2_generation_summary_v2.txt"
)

EXPECTED_MANIFEST <- file.path(
    OUT_DIR,
    "TableS2_generation_manifest_v2.csv"
)

expected_outputs <- c(
    EXPECTED_DOCX,
    EXPECTED_S2A,
    EXPECTED_S2B,
    EXPECTED_S2C,
    EXPECTED_NOTES,
    EXPECTED_SUMMARY,
    EXPECTED_MANIFEST,
    PY_SCRIPT_PATH
)

if (!all(
    file.exists(
        expected_outputs
    )
)) {
    stop(
        "One or more expected Table S2 v2 artifacts are missing."
    )
}

if (any(
    file.info(
        expected_outputs
    )$size <= 0
)) {
    stop(
        "One or more expected Table S2 v2 artifacts are zero-byte."
    )
}

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.2S COMPLETE - TABLE S2 v2 GENERATED THROUGH PYTHON FROM R KERNEL\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "\nMicrosoft Word DOCX:\n",
    EXPECTED_DOCX,
    "\n",
    sep = ""
)

cat(
    "\nSaved Python generator:\n",
    PY_SCRIPT_PATH,
    "\n",
    sep = ""
)

cat(
    "\nSource CSVs:\n",
    EXPECTED_S2A,
    "\n",
    EXPECTED_S2B,
    "\n",
    EXPECTED_S2C,
    "\n",
    sep = ""
)

cat(
    "\nManifest:\n",
    EXPECTED_MANIFEST,
    "\n",
    sep = ""
)


STEP 11E.2S - SUPPLEMENTARY TABLE S2 - FULLY CORRECTED
DONOR-BLOCKED DIFFERENTIAL-EXPRESSION MODEL AUDIT AND RESULTS
Project sentinel: PASS
Python executable: PASS
Corrected Python generator saved: E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/tables/supplementary/TableS2/generate_TableS2_v2.py
STEP 11E.2S - SUPPLEMENTARY TABLE S2 - FULLY CORRECTED
DONOR-BLOCKED DIFFERENTIAL-EXPRESSION MODEL AUDIT AND RESULTS
Project sentinel: PASS
Canonical restored STEP03A files: PASS
Four canonical 23,740-gene DE tables: PASS
Stable-Ensembl alignment across four DE tables: PASS
Full DE tables vs saved DE summary: PASS
Canonical DE count anchors: PASS
Restored canonical model-audit schema reconciled: PASS
design_rank absent from restored audit: accepted and not inserted into manuscript table
donor_balanced_total_weight absent from restored audit: will be recomputed from frozen metadata
Donor-balanced total weight recomputed from frozen metadata = 454: PASS
Paired donor architecture 301 donors / 602 sample

In [24]:
# ==================================================================================================
# STEP 11E.3R - SUPPLEMENTARY TABLE S3
# R-KERNEL WRAPPER -> PYTHON-DOCX GENERATOR
# FULL 5 x 5 NESTED-CV RESULTS AND HYPERPARAMETER SELECTION
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.3R - SUPPLEMENTARY TABLE S3\n",
    "FULL 5 x 5 NESTED-CV RESULTS AND HYPERPARAMETER SELECTION\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 1. PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

observed_project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    observed_project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "Project ID mismatch. Expected ",
        EXPECTED_PROJECT_ID,
        " but observed ",
        observed_project_id
    )
}

cat("Project sentinel: PASS\n")

# --------------------------------------------------------------------------------------------------
# 2. PYTHON
# --------------------------------------------------------------------------------------------------

PYTHON_EXE <- "E:/Python/python.exe"

if (!file.exists(PYTHON_EXE)) {
    stop(
        "Python executable not found: ",
        PYTHON_EXE
    )
}

cat("Python executable: PASS\n")

# --------------------------------------------------------------------------------------------------
# 3. OUTPUT DIRECTORY
# --------------------------------------------------------------------------------------------------

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "supplementary",
    "TableS3"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PY_SCRIPT_PATH <- file.path(
    OUT_DIR,
    "generate_TableS3_v1.py"
)

# --------------------------------------------------------------------------------------------------
# 4. COMPLETE PYTHON GENERATOR
# --------------------------------------------------------------------------------------------------

py_code <- r"(from pathlib import Path
import csv
import hashlib
import math
import statistics

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


print("=" * 110)
print("STEP 11E.3 - SUPPLEMENTARY TABLE S3")
print("FULL 5 x 5 NESTED-CV RESULTS AND HYPERPARAMETER SELECTION")
print("=" * 110)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT = Path(
    "E:/SCZ_CROSSREGION_MSS_Q1"
)

EXPECTED_PROJECT_ID = (
    "SCZ_CROSSREGION_MSS_Q1_2026"
)

SENTINEL = (
    PROJECT_ROOT
    / ".project_id"
)

if not SENTINEL.exists():
    raise FileNotFoundError(
        f"Project sentinel missing: {SENTINEL}"
    )

project_id = SENTINEL.read_text(
    encoding="utf-8"
).strip()

if project_id != EXPECTED_PROJECT_ID:
    raise RuntimeError(
        f"Project sentinel mismatch: {project_id}"
    )

print("Project sentinel: PASS")

# ==================================================================================================
# 2. FROZEN NESTED-CV INPUTS
# ==================================================================================================

NESTED_DIR = (
    PROJECT_ROOT
    / "06_models"
    / "nested_cv_5x5_v1"
)

FOLD_PATH = (
    NESTED_DIR
    / "ALL_25_OUTER_FOLD_SUMMARY_v1.csv"
)

PERF_PATH = (
    NESTED_DIR
    / "NESTED_CV_FINAL_PERFORMANCE_v1.csv"
)

HYPER_PATH = (
    NESTED_DIR
    / "HYPERPARAMETER_SELECTION_FREQUENCY_v1.csv"
)

OOF_DONOR_PATH = (
    NESTED_DIR
    / "OOF_DONOR_PREDICTIONS_AVERAGED_v1.csv"
)

OOF_SAMPLE_PATH = (
    NESTED_DIR
    / "OOF_SAMPLE_PREDICTIONS_AVERAGED_v1.csv"
)

required_inputs = [
    FOLD_PATH,
    PERF_PATH,
    HYPER_PATH,
    OOF_DONOR_PATH,
    OOF_SAMPLE_PATH,
]

missing_inputs = [
    str(p)
    for p in required_inputs
    if not p.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required frozen nested-CV artifact(s) missing:\n"
        + "\n".join(missing_inputs)
    )

print("Frozen nested-CV artifacts: PASS")

# ==================================================================================================
# 3. CSV READER
# ==================================================================================================

def read_csv_dict(path):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        reader = csv.DictReader(f)

        rows = list(reader)

        columns = reader.fieldnames or []

    return rows, columns


fold_rows, fold_cols = read_csv_dict(
    FOLD_PATH
)

perf_rows, perf_cols = read_csv_dict(
    PERF_PATH
)

hyper_rows, hyper_cols = read_csv_dict(
    HYPER_PATH
)

oof_donor, donor_cols = read_csv_dict(
    OOF_DONOR_PATH
)

oof_sample, sample_cols = read_csv_dict(
    OOF_SAMPLE_PATH
)

# ==================================================================================================
# 4. NUMERIC HELPERS
# ==================================================================================================

def num(x):

    try:
        value = float(x)

    except Exception as exc:
        raise RuntimeError(
            f"Cannot parse numeric value: {x}"
        ) from exc

    if not math.isfinite(value):
        raise RuntimeError(
            f"Non-finite numeric value: {x}"
        )

    return value


def int_value(x):

    value = num(x)

    if abs(
        value - round(value)
    ) > 1e-8:
        raise RuntimeError(
            f"Expected integer-like value: {x}"
        )

    return int(
        round(value)
    )


def close_enough(
    observed,
    expected,
    tolerance
):

    return abs(
        observed - expected
    ) <= tolerance

# ==================================================================================================
# 5. EXACT SCHEMAS
# ==================================================================================================

required_fold_cols = [
    "outer_repeat",
    "outer_fold",
    "train_donors",
    "test_donors",
    "train_samples",
    "test_samples",
    "best_alpha",
    "best_deviation_penalty",
    "best_lambda",
    "best_inner_auc",
    "best_inner_auc_sd",
    "outer_sample_auc",
    "outer_donor_auc",
    "outer_DLPFC_auc",
    "outer_HIPPO_auc",
    "n_shared_nonzero",
    "n_deviation_nonzero",
    "grid_seconds",
    "final_fit_seconds",
    "total_fold_seconds",
]

required_perf_cols = [
    "metric",
    "value",
]

required_hyper_cols = [
    "alpha",
    "deviation_penalty",
    "n_outer_folds",
]

required_donor_cols = [
    "BrNum",
    "y",
    "prediction",
]

required_sample_cols = [
    "discovery_row_index",
    "RNum",
    "BrNum",
    "Region",
    "Dx",
    "y",
    "prediction",
]

for label, required, observed in [
    (
        "Outer-fold summary",
        required_fold_cols,
        fold_cols,
    ),
    (
        "Final performance",
        required_perf_cols,
        perf_cols,
    ),
    (
        "Hyperparameter frequency",
        required_hyper_cols,
        hyper_cols,
    ),
    (
        "OOF donor",
        required_donor_cols,
        donor_cols,
    ),
    (
        "OOF sample",
        required_sample_cols,
        sample_cols,
    ),
]:

    missing = [
        x
        for x in required
        if x not in observed
    ]

    if missing:
        raise RuntimeError(
            f"{label} schema missing field(s): "
            + ", ".join(missing)
        )

print("Nested-CV input schemas: PASS")

# ==================================================================================================
# 6. OUTER-FOLD STRUCTURE
# ==================================================================================================

if len(
    fold_rows
) != 25:
    raise RuntimeError(
        f"Expected 25 outer-fold rows; observed {len(fold_rows)}"
    )

fold_pairs = [
    (
        int_value(
            r["outer_repeat"]
        ),
        int_value(
            r["outer_fold"]
        ),
    )
    for r in fold_rows
]

if len(
    set(fold_pairs)
) != 25:
    raise RuntimeError(
        "Outer repeat/fold combinations are not unique."
    )

expected_pairs = {
    (repeat_id, fold_id)
    for repeat_id in range(
        1,
        6
    )
    for fold_id in range(
        1,
        6
    )
}

if set(
    fold_pairs
) != expected_pairs:
    raise RuntimeError(
        "Outer-fold structure is not exactly 5 repeats x 5 folds."
    )

fold_rows.sort(
    key=lambda r: (
        int_value(
            r["outer_repeat"]
        ),
        int_value(
            r["outer_fold"]
        ),
    )
)

print("Outer CV structure: 5 repeats x 5 folds = 25 tests - PASS")

# ==================================================================================================
# 7. TRAIN / TEST DONOR INTEGRITY
# ==================================================================================================

for row in fold_rows:

    train_donors = int_value(
        row[
            "train_donors"
        ]
    )

    test_donors = int_value(
        row[
            "test_donors"
        ]
    )

    if (
        train_donors
        + test_donors
        != 454
    ):
        raise RuntimeError(
            "Train + test donor count does not equal 454 "
            f"for repeat {row['outer_repeat']} fold {row['outer_fold']}."
        )

    train_samples = int_value(
        row[
            "train_samples"
        ]
    )

    test_samples = int_value(
        row[
            "test_samples"
        ]
    )

    if (
        train_samples
        + test_samples
        != 755
    ):
        raise RuntimeError(
            "Train + test sample count does not equal 755 "
            f"for repeat {row['outer_repeat']} fold {row['outer_fold']}."
        )

print("Outer-fold train/test population accounting: PASS")

# ==================================================================================================
# 8. FINAL PERFORMANCE METRICS
# ==================================================================================================

perf = {}

for row in perf_rows:

    metric = str(
        row[
            "metric"
        ]
    ).strip()

    if metric in perf:
        raise RuntimeError(
            f"Duplicate final-performance metric: {metric}"
        )

    perf[
        metric
    ] = num(
        row[
            "value"
        ]
    )


required_perf_metrics = [
    "Repeated_OOF_donor_AUROC",
    "Repeated_OOF_sample_AUROC",
    "Repeated_OOF_DLPFC_AUROC",
    "Repeated_OOF_HIPPO_AUROC",
    "Mean_outer_fold_donor_AUROC",
    "SD_outer_fold_donor_AUROC",
    "Min_outer_fold_donor_AUROC",
    "Max_outer_fold_donor_AUROC",
    "Mean_outer_fold_sample_AUROC",
    "SD_outer_fold_sample_AUROC",
    "Mean_outer_fold_DLPFC_AUROC",
    "Mean_outer_fold_HIPPO_AUROC",
    "Mean_selected_shared_genes",
    "Median_selected_shared_genes",
    "Min_selected_shared_genes",
    "Max_selected_shared_genes",
    "Mean_selected_deviation_genes",
    "Median_selected_deviation_genes",
    "Min_selected_deviation_genes",
    "Max_selected_deviation_genes",
]

missing_metrics = [
    x
    for x in required_perf_metrics
    if x not in perf
]

if missing_metrics:
    raise RuntimeError(
        "Final-performance file missing metric(s): "
        + ", ".join(
            missing_metrics
        )
    )

# ==================================================================================================
# 9. RECOMPUTE FOLD SUMMARY STATISTICS FROM SAVED 25 ROWS
# ==================================================================================================

fold_donor_auc = [
    num(
        r[
            "outer_donor_auc"
        ]
    )
    for r in fold_rows
]

fold_sample_auc = [
    num(
        r[
            "outer_sample_auc"
        ]
    )
    for r in fold_rows
]

fold_dlpfc_auc = [
    num(
        r[
            "outer_DLPFC_auc"
        ]
    )
    for r in fold_rows
]

fold_hippo_auc = [
    num(
        r[
            "outer_HIPPO_auc"
        ]
    )
    for r in fold_rows
]

shared_nonzero = [
    int_value(
        r[
            "n_shared_nonzero"
        ]
    )
    for r in fold_rows
]

deviation_nonzero = [
    int_value(
        r[
            "n_deviation_nonzero"
        ]
    )
    for r in fold_rows
]


recomputed_metrics = {
    "Mean_outer_fold_donor_AUROC":
        statistics.mean(
            fold_donor_auc
        ),

    "SD_outer_fold_donor_AUROC":
        statistics.stdev(
            fold_donor_auc
        ),

    "Min_outer_fold_donor_AUROC":
        min(
            fold_donor_auc
        ),

    "Max_outer_fold_donor_AUROC":
        max(
            fold_donor_auc
        ),

    "Mean_outer_fold_sample_AUROC":
        statistics.mean(
            fold_sample_auc
        ),

    "SD_outer_fold_sample_AUROC":
        statistics.stdev(
            fold_sample_auc
        ),

    "Mean_outer_fold_DLPFC_AUROC":
        statistics.mean(
            fold_dlpfc_auc
        ),

    "Mean_outer_fold_HIPPO_AUROC":
        statistics.mean(
            fold_hippo_auc
        ),

    "Mean_selected_shared_genes":
        statistics.mean(
            shared_nonzero
        ),

    "Median_selected_shared_genes":
        statistics.median(
            shared_nonzero
        ),

    "Min_selected_shared_genes":
        min(
            shared_nonzero
        ),

    "Max_selected_shared_genes":
        max(
            shared_nonzero
        ),

    "Mean_selected_deviation_genes":
        statistics.mean(
            deviation_nonzero
        ),

    "Median_selected_deviation_genes":
        statistics.median(
            deviation_nonzero
        ),

    "Min_selected_deviation_genes":
        min(
            deviation_nonzero
        ),

    "Max_selected_deviation_genes":
        max(
            deviation_nonzero
        ),
}

for metric_name, recomputed in recomputed_metrics.items():

    saved = perf[
        metric_name
    ]

    if not close_enough(
        recomputed,
        saved,
        5e-7
    ):
        raise RuntimeError(
            f"Final-performance mismatch for {metric_name}: "
            f"saved={saved}, recomputed={recomputed}"
        )

print("25-fold summary vs final-performance file: PASS")

# ==================================================================================================
# 10. CANONICAL PERFORMANCE ANCHORS
# ==================================================================================================

canonical_performance = {
    "Repeated_OOF_donor_AUROC":
        0.8019743,

    "Repeated_OOF_sample_AUROC":
        0.7938032,

    "Repeated_OOF_DLPFC_AUROC":
        0.7858754,

    "Repeated_OOF_HIPPO_AUROC":
        0.7997463,

    "Mean_outer_fold_donor_AUROC":
        0.7868384,

    "SD_outer_fold_donor_AUROC":
        0.04047966,

    "Min_outer_fold_donor_AUROC":
        0.6782828,

    "Max_outer_fold_donor_AUROC":
        0.8647587,

    "Mean_selected_shared_genes":
        1501.12,

    "Median_selected_shared_genes":
        1534.0,

    "Mean_selected_deviation_genes":
        59.6,

    "Median_selected_deviation_genes":
        78.0,
}

for metric_name, expected in canonical_performance.items():

    observed = perf[
        metric_name
    ]

    if not close_enough(
        observed,
        expected,
        5e-6
    ):
        raise RuntimeError(
            f"Canonical performance anchor changed for {metric_name}: "
            f"observed={observed}, expected={expected}"
        )

print("Canonical nested-CV performance anchors: PASS")

# ==================================================================================================
# 11. AUROC FUNCTION FOR OOF CROSS-CHECK
# ==================================================================================================

def auc_rank(
    y,
    score
):

    pairs = [
        (
            int_value(
                yi
            ),
            num(
                si
            ),
        )
        for yi, si
        in zip(
            y,
            score
        )
    ]

    y_values = [
        p[0]
        for p in pairs
    ]

    score_values = [
        p[1]
        for p in pairs
    ]

    if not all(
        yi in {
            0,
            1,
        }
        for yi in y_values
    ):
        raise RuntimeError(
            "Unexpected binary outcome coding."
        )

    n1 = sum(
        yi == 1
        for yi in y_values
    )

    n0 = sum(
        yi == 0
        for yi in y_values
    )

    if (
        n1 == 0
        or n0 == 0
    ):
        raise RuntimeError(
            "Both classes required for AUROC."
        )

    indexed = sorted(
        enumerate(
            score_values
        ),
        key=lambda z: z[1]
    )

    ranks = [
        0.0
        for _ in score_values
    ]

    i = 0

    while i < len(
        indexed
    ):

        j = i + 1

        while (
            j < len(
                indexed
            )
            and indexed[j][1]
            == indexed[i][1]
        ):
            j += 1

        average_rank = (
            (
                i + 1
            )
            +
            j
        ) / 2.0

        for k in range(
            i,
            j
        ):
            original_index = indexed[
                k
            ][0]

            ranks[
                original_index
            ] = average_rank

        i = j

    rank_sum_positive = sum(
        rank
        for rank, yi
        in zip(
            ranks,
            y_values
        )
        if yi == 1
    )

    return (
        rank_sum_positive
        - n1
        * (
            n1 + 1
        )
        / 2.0
    ) / (
        n1
        * n0
    )

# ==================================================================================================
# 12. OOF DONOR FILE CROSS-CHECK
# ==================================================================================================

if len(
    oof_donor
) != 454:
    raise RuntimeError(
        f"Expected 454 averaged OOF donors; observed {len(oof_donor)}"
    )

if len(
    {
        str(
            r[
                "BrNum"
            ]
        ).strip()
        for r in oof_donor
    }
) != 454:
    raise RuntimeError(
        "OOF donor IDs are not unique."
    )

DONOR_OOF_AUC = auc_rank(
    [
        r[
            "y"
        ]
        for r in oof_donor
    ],
    [
        r[
            "prediction"
        ]
        for r in oof_donor
    ],
)

if not close_enough(
    DONOR_OOF_AUC,
    perf[
        "Repeated_OOF_donor_AUROC"
    ],
    5e-7
):
    raise RuntimeError(
        "OOF donor prediction file disagrees with final performance."
    )

# ==================================================================================================
# 13. OOF SAMPLE FILE CROSS-CHECK
# ==================================================================================================

if len(
    oof_sample
) != 755:
    raise RuntimeError(
        f"Expected 755 averaged OOF samples; observed {len(oof_sample)}"
    )

if len(
    {
        str(
            r[
                "discovery_row_index"
            ]
        ).strip()
        for r in oof_sample
    }
) != 755:
    raise RuntimeError(
        "OOF sample discovery-row IDs are not unique."
    )

SAMPLE_OOF_AUC = auc_rank(
    [
        r[
            "y"
        ]
        for r in oof_sample
    ],
    [
        r[
            "prediction"
        ]
        for r in oof_sample
    ],
)

dlpfc_rows = [
    r
    for r in oof_sample
    if str(
        r[
            "Region"
        ]
    ).strip().upper()
    == "DLPFC"
]

hippo_rows = [
    r
    for r in oof_sample
    if str(
        r[
            "Region"
        ]
    ).strip().upper()
    in {
        "HIPPO",
        "HIPPOCAMPUS",
        "HIPPOCAMPAL",
    }
]

if (
    len(
        dlpfc_rows
    ) != 379
    or len(
        hippo_rows
    ) != 376
):
    raise RuntimeError(
        "OOF regional sample counts changed."
    )

DLPFC_OOF_AUC = auc_rank(
    [
        r[
            "y"
        ]
        for r in dlpfc_rows
    ],
    [
        r[
            "prediction"
        ]
        for r in dlpfc_rows
    ],
)

HIPPO_OOF_AUC = auc_rank(
    [
        r[
            "y"
        ]
        for r in hippo_rows
    ],
    [
        r[
            "prediction"
        ]
        for r in hippo_rows
    ],
)

for metric_name, observed in [
    (
        "Repeated_OOF_sample_AUROC",
        SAMPLE_OOF_AUC,
    ),
    (
        "Repeated_OOF_DLPFC_AUROC",
        DLPFC_OOF_AUC,
    ),
    (
        "Repeated_OOF_HIPPO_AUROC",
        HIPPO_OOF_AUC,
    ),
]:

    if not close_enough(
        observed,
        perf[
            metric_name
        ],
        5e-7
    ):
        raise RuntimeError(
            f"OOF prediction file disagreement for {metric_name}."
        )

print("Averaged OOF donor/sample files vs final performance: PASS")

# ==================================================================================================
# 14. HYPERPARAMETER FREQUENCY
# ==================================================================================================

if len(
    hyper_rows
) != 3:
    raise RuntimeError(
        f"Expected three selected hyperparameter combinations; observed {len(hyper_rows)}"
    )

hyper_clean = []

for row in hyper_rows:

    alpha = num(
        row[
            "alpha"
        ]
    )

    penalty = num(
        row[
            "deviation_penalty"
        ]
    )

    count = int_value(
        row[
            "n_outer_folds"
        ]
    )

    hyper_clean.append(
        {
            "alpha":
                alpha,

            "deviation_penalty":
                penalty,

            "n_outer_folds":
                count,
        }
    )

if sum(
    r[
        "n_outer_folds"
    ]
    for r in hyper_clean
) != 25:
    raise RuntimeError(
        "Hyperparameter selection frequencies do not sum to 25."
    )

if any(
    abs(
        r[
            "alpha"
        ]
        - 0.1
    ) > 1e-12
    for r in hyper_clean
):
    raise RuntimeError(
        "Selected alpha was not 0.1 in all 25 folds."
    )

penalty_counts = {
    int(
        round(
            r[
                "deviation_penalty"
            ]
        )
    ):
        r[
            "n_outer_folds"
        ]
    for r in hyper_clean
}

if penalty_counts != {
    1: 18,
    2: 1,
    8: 6,
}:
    raise RuntimeError(
        f"Deviation-penalty frequency changed: {penalty_counts}"
    )

hyper_clean.sort(
    key=lambda r: (
        r[
            "deviation_penalty"
        ]
    )
)

print(
    "Hyperparameter selections: alpha 0.1 in 25/25; "
    "deviation penalty 1/2/8 = 18/1/6 - PASS"
)

# ==================================================================================================
# 15. DIRECT FOLD-LEVEL HYPERPARAMETER CROSS-CHECK
# ==================================================================================================

fold_alpha = [
    num(
        r[
            "best_alpha"
        ]
    )
    for r in fold_rows
]

fold_penalty = [
    int_value(
        r[
            "best_deviation_penalty"
        ]
    )
    for r in fold_rows
]

if any(
    abs(
        x - 0.1
    ) > 1e-12
    for x in fold_alpha
):
    raise RuntimeError(
        "At least one outer fold selected alpha != 0.1."
    )

direct_penalty_counts = {
    penalty:
        fold_penalty.count(
            penalty
        )
    for penalty in sorted(
        set(
            fold_penalty
        )
    )
}

if direct_penalty_counts != penalty_counts:
    raise RuntimeError(
        "Hyperparameter frequency file disagrees with the 25 fold summary."
    )

print("Hyperparameter frequency file vs 25-fold summary: PASS")

# ==================================================================================================
# 16. BUILD TABLE S3A - PERFORMANCE SUMMARY
# ==================================================================================================

table_s3a = [
    [
        "Outer CV design",
        "5-fold x 5 repeats",
        "25 leakage-free outer test evaluations; donors were grouped within folds.",
    ],

    [
        "Inner CV design",
        "4-fold donor-grouped",
        "Used only within each outer-training set for hyperparameter selection.",
    ],

    [
        "Repeated-OOF donor AUROC",
        f"{perf['Repeated_OOF_donor_AUROC']:.4f}",
        "Five held-out predictions were averaged for each of 454 donors.",
    ],

    [
        "Repeated-OOF sample AUROC",
        f"{perf['Repeated_OOF_sample_AUROC']:.4f}",
        "Five held-out predictions were averaged for each of 755 tissue samples.",
    ],

    [
        "Repeated-OOF DLPFC AUROC",
        f"{perf['Repeated_OOF_DLPFC_AUROC']:.4f}",
        "379 DLPFC samples.",
    ],

    [
        "Repeated-OOF hippocampus AUROC",
        f"{perf['Repeated_OOF_HIPPO_AUROC']:.4f}",
        "376 hippocampal samples.",
    ],

    [
        "Outer-fold donor AUROC",
        (
            f"{perf['Mean_outer_fold_donor_AUROC']:.4f} "
            f"+/- {perf['SD_outer_fold_donor_AUROC']:.4f}"
        ),
        (
            f"Range "
            f"{perf['Min_outer_fold_donor_AUROC']:.4f}-"
            f"{perf['Max_outer_fold_donor_AUROC']:.4f} across 25 folds."
        ),
    ],

    [
        "Outer-fold sample AUROC",
        (
            f"{perf['Mean_outer_fold_sample_AUROC']:.4f} "
            f"+/- {perf['SD_outer_fold_sample_AUROC']:.4f}"
        ),
        "Mean +/- SD across 25 outer tests.",
    ],

    [
        "Mean outer-fold DLPFC AUROC",
        f"{perf['Mean_outer_fold_DLPFC_AUROC']:.4f}",
        "Mean across the 25 outer folds.",
    ],

    [
        "Mean outer-fold hippocampus AUROC",
        f"{perf['Mean_outer_fold_HIPPO_AUROC']:.4f}",
        "Mean across the 25 outer folds.",
    ],

    [
        "Selected shared features",
        (
            f"mean {perf['Mean_selected_shared_genes']:.1f}; "
            f"median {perf['Median_selected_shared_genes']:.0f}"
        ),
        (
            f"Range "
            f"{perf['Min_selected_shared_genes']:.0f}-"
            f"{perf['Max_selected_shared_genes']:.0f} across outer models."
        ),
    ],

    [
        "Selected deviation features",
        (
            f"mean {perf['Mean_selected_deviation_genes']:.1f}; "
            f"median {perf['Median_selected_deviation_genes']:.0f}"
        ),
        (
            f"Range "
            f"{perf['Min_selected_deviation_genes']:.0f}-"
            f"{perf['Max_selected_deviation_genes']:.0f} across outer models."
        ),
    ],
]

# ==================================================================================================
# 17. BUILD TABLE S3B - HYPERPARAMETER FREQUENCY
# ==================================================================================================

table_s3b = []

for row in hyper_clean:

    count = row[
        "n_outer_folds"
    ]

    table_s3b.append(
        [
            f"{row['alpha']:.1f}",

            f"{row['deviation_penalty']:.0f}",

            str(
                count
            ),

            f"{100.0 * count / 25.0:.1f}%",
        ]
    )

# ==================================================================================================
# 18. BUILD TABLE S3C - TUNING + DONOR PERFORMANCE
# ==================================================================================================

table_s3c = []

for row in fold_rows:

    table_s3c.append(
        [
            str(
                int_value(
                    row[
                        "outer_repeat"
                    ]
                )
            ),

            str(
                int_value(
                    row[
                        "outer_fold"
                    ]
                )
            ),

            str(
                int_value(
                    row[
                        "train_donors"
                    ]
                )
            ),

            str(
                int_value(
                    row[
                        "test_donors"
                    ]
                )
            ),

            f"{num(row['best_alpha']):.1f}",

            f"{num(row['best_deviation_penalty']):.0f}",

            f"{num(row['best_lambda']):.6g}",

            f"{num(row['best_inner_auc']):.4f}",

            f"{num(row['best_inner_auc_sd']):.4f}",

            f"{num(row['outer_donor_auc']):.4f}",
        ]
    )

# ==================================================================================================
# 19. BUILD TABLE S3D - SAMPLE/REGION PERFORMANCE + COMPLEXITY
# ==================================================================================================

table_s3d = []

for row in fold_rows:

    table_s3d.append(
        [
            str(
                int_value(
                    row[
                        "outer_repeat"
                    ]
                )
            ),

            str(
                int_value(
                    row[
                        "outer_fold"
                    ]
                )
            ),

            str(
                int_value(
                    row[
                        "test_samples"
                    ]
                )
            ),

            f"{num(row['outer_sample_auc']):.4f}",

            f"{num(row['outer_DLPFC_auc']):.4f}",

            f"{num(row['outer_HIPPO_auc']):.4f}",

            f"{int_value(row['n_shared_nonzero']):,}",

            f"{int_value(row['n_deviation_nonzero']):,}",
        ]
    )

# ==================================================================================================
# 20. OUTPUT PATHS
# ==================================================================================================

OUT_DIR = (
    PROJECT_ROOT
    / "11_manuscript"
    / "tables"
    / "supplementary"
    / "TableS3"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOCX_PATH = (
    OUT_DIR
    / "TableS3_NestedCV_Full_Results_Hyperparameters_v1.docx"
)

S3A_PATH = (
    OUT_DIR
    / "TableS3A_nestedCV_summary_source_v1.csv"
)

S3B_PATH = (
    OUT_DIR
    / "TableS3B_hyperparameter_frequency_source_v1.csv"
)

S3C_PATH = (
    OUT_DIR
    / "TableS3C_outer_fold_tuning_donor_performance_source_v1.csv"
)

S3D_PATH = (
    OUT_DIR
    / "TableS3D_outer_fold_region_complexity_source_v1.csv"
)

NOTES_PATH = (
    OUT_DIR
    / "TableS3_notes_v1.txt"
)

SUMMARY_PATH = (
    OUT_DIR
    / "TableS3_generation_summary_v1.txt"
)

MANIFEST_PATH = (
    OUT_DIR
    / "TableS3_generation_manifest_v1.csv"
)

# ==================================================================================================
# 21. SOURCE CSVs
# ==================================================================================================

def write_rows(
    path,
    header,
    rows
):

    with path.open(
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        writer = csv.writer(
            f
        )

        writer.writerow(
            header
        )

        writer.writerows(
            rows
        )


write_rows(
    S3A_PATH,
    [
        "Metric",
        "Frozen result",
        "Interpretation / evaluation note",
    ],
    table_s3a
)

write_rows(
    S3B_PATH,
    [
        "Alpha",
        "Deviation penalty",
        "Outer folds selected",
        "Selection frequency",
    ],
    table_s3b
)

write_rows(
    S3C_PATH,
    [
        "Repeat",
        "Fold",
        "Train donors",
        "Test donors",
        "Alpha",
        "Deviation penalty",
        "Lambda",
        "Best inner AUROC",
        "Inner AUROC SD",
        "Outer donor AUROC",
    ],
    table_s3c
)

write_rows(
    S3D_PATH,
    [
        "Repeat",
        "Fold",
        "Test samples",
        "Outer sample AUROC",
        "DLPFC AUROC",
        "HIPPO AUROC",
        "Shared nonzero",
        "Deviation nonzero",
    ],
    table_s3d
)

# ==================================================================================================
# 22. WORD HELPERS
# ==================================================================================================

def set_run_font(
    run,
    size_pt=9,
    bold=False,
    italic=False
):

    run.font.name = (
        "Times New Roman"
    )

    run.font.size = Pt(
        size_pt
    )

    run.bold = bold
    run.italic = italic

    rPr = (
        run._element
        .get_or_add_rPr()
    )

    rFonts = rPr.rFonts

    if rFonts is None:

        rFonts = OxmlElement(
            "w:rFonts"
        )

        rPr.insert(
            0,
            rFonts
        )

    for attr in [
        "ascii",
        "hAnsi",
        "eastAsia",
        "cs",
    ]:

        rFonts.set(
            qn(
                f"w:{attr}"
            ),
            "Times New Roman"
        )


def set_cell_margins(
    cell,
    top=70,
    start=75,
    bottom=70,
    end=75
):

    tcPr = (
        cell._tc
        .get_or_add_tcPr()
    )

    tcMar = (
        tcPr
        .first_child_found_in(
            "w:tcMar"
        )
    )

    if tcMar is None:

        tcMar = OxmlElement(
            "w:tcMar"
        )

        tcPr.append(
            tcMar
        )

    for name, value in {
        "top": top,
        "start": start,
        "bottom": bottom,
        "end": end,
    }.items():

        node = tcMar.find(
            qn(
                f"w:{name}"
            )
        )

        if node is None:

            node = OxmlElement(
                f"w:{name}"
            )

            tcMar.append(
                node
            )

        node.set(
            qn(
                "w:w"
            ),
            str(
                value
            )
        )

        node.set(
            qn(
                "w:type"
            ),
            "dxa"
        )


def remove_shading(
    cell
):

    tcPr = (
        cell._tc
        .get_or_add_tcPr()
    )

    for node in list(
        tcPr.findall(
            qn(
                "w:shd"
            )
        )
    ):

        tcPr.remove(
            node
        )


def set_cell_width(
    cell,
    width_inches
):

    tcPr = (
        cell._tc
        .get_or_add_tcPr()
    )

    tcW = (
        tcPr
        .first_child_found_in(
            "w:tcW"
        )
    )

    if tcW is None:

        tcW = OxmlElement(
            "w:tcW"
        )

        tcPr.append(
            tcW
        )

    tcW.set(
        qn(
            "w:w"
        ),
        str(
            int(
                width_inches
                * 1440
            )
        )
    )

    tcW.set(
        qn(
            "w:type"
        ),
        "dxa"
    )


def set_cell_border(
    cell,
    size=8
):

    tcPr = (
        cell._tc
        .get_or_add_tcPr()
    )

    borders = (
        tcPr
        .first_child_found_in(
            "w:tcBorders"
        )
    )

    if borders is None:

        borders = OxmlElement(
            "w:tcBorders"
        )

        tcPr.append(
            borders
        )

    for edge_name in [
        "top",
        "bottom",
        "left",
        "right",
    ]:

        edge = borders.find(
            qn(
                f"w:{edge_name}"
            )
        )

        if edge is None:

            edge = OxmlElement(
                f"w:{edge_name}"
            )

            borders.append(
                edge
            )

        edge.set(
            qn(
                "w:val"
            ),
            "single"
        )

        edge.set(
            qn(
                "w:sz"
            ),
            str(
                size
            )
        )

        edge.set(
            qn(
                "w:space"
            ),
            "0"
        )

        edge.set(
            qn(
                "w:color"
            ),
            "000000"
        )


def set_table_borders(
    table
):

    tblPr = (
        table._tbl.tblPr
    )

    borders = (
        tblPr
        .first_child_found_in(
            "w:tblBorders"
        )
    )

    if borders is None:

        borders = OxmlElement(
            "w:tblBorders"
        )

        tblPr.append(
            borders
        )

    specs = {
        "top": 10,
        "bottom": 10,
        "left": 10,
        "right": 10,
        "insideH": 8,
        "insideV": 8,
    }

    for name, size in specs.items():

        edge = borders.find(
            qn(
                f"w:{name}"
            )
        )

        if edge is None:

            edge = OxmlElement(
                f"w:{name}"
            )

            borders.append(
                edge
            )

        edge.set(
            qn(
                "w:val"
            ),
            "single"
        )

        edge.set(
            qn(
                "w:sz"
            ),
            str(
                size
            )
        )

        edge.set(
            qn(
                "w:space"
            ),
            "0"
        )

        edge.set(
            qn(
                "w:color"
            ),
            "000000"
        )


def fixed_layout(
    table
):

    tblPr = (
        table._tbl.tblPr
    )

    layout = (
        tblPr
        .first_child_found_in(
            "w:tblLayout"
        )
    )

    if layout is None:

        layout = OxmlElement(
            "w:tblLayout"
        )

        tblPr.append(
            layout
        )

    layout.set(
        qn(
            "w:type"
        ),
        "fixed"
    )


def repeat_header(
    row
):

    trPr = (
        row._tr
        .get_or_add_trPr()
    )

    node = OxmlElement(
        "w:tblHeader"
    )

    node.set(
        qn(
            "w:val"
        ),
        "true"
    )

    trPr.append(
        node
    )


def prevent_split(
    row
):

    trPr = (
        row._tr
        .get_or_add_trPr()
    )

    node = OxmlElement(
        "w:cantSplit"
    )

    node.set(
        qn(
            "w:val"
        ),
        "true"
    )

    trPr.append(
        node
    )


def format_paragraph(
    paragraph,
    alignment=WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=0
):

    paragraph.alignment = (
        alignment
    )

    fmt = (
        paragraph
        .paragraph_format
    )

    fmt.space_before = Pt(
        before
    )

    fmt.space_after = Pt(
        after
    )

    fmt.line_spacing = 1.0


def add_heading(
    document,
    text
):

    p = (
        document
        .add_paragraph()
    )

    format_paragraph(
        p,
        WD_ALIGN_PARAGRAPH.LEFT,
        before=5,
        after=3
    )

    run = p.add_run(
        text
    )

    set_run_font(
        run,
        size_pt=9.4,
        bold=True
    )


def make_table(
    document,
    headers,
    body_rows,
    widths,
    center_columns=None,
    font_size=8.2
):

    if center_columns is None:
        center_columns = set()

    table = document.add_table(
        rows=1,
        cols=len(
            headers
        )
    )

    table.alignment = (
        WD_TABLE_ALIGNMENT.CENTER
    )

    table.autofit = False

    fixed_layout(
        table
    )

    set_table_borders(
        table
    )

    header_row = table.rows[
        0
    ]

    for j, text in enumerate(
        headers
    ):

        cell = header_row.cells[
            j
        ]

        set_cell_width(
            cell,
            widths[
                j
            ]
        )

        set_cell_margins(
            cell
        )

        remove_shading(
            cell
        )

        set_cell_border(
            cell,
            size=10
        )

        cell.vertical_alignment = (
            WD_CELL_VERTICAL_ALIGNMENT.CENTER
        )

        p = cell.paragraphs[
            0
        ]

        alignment = (
            WD_ALIGN_PARAGRAPH.CENTER
            if j in center_columns
            else WD_ALIGN_PARAGRAPH.LEFT
        )

        format_paragraph(
            p,
            alignment
        )

        run = p.add_run(
            str(
                text
            )
        )

        set_run_font(
            run,
            size_pt=8.3,
            bold=True
        )

    repeat_header(
        header_row
    )

    prevent_split(
        header_row
    )

    for values in body_rows:

        row = table.add_row()

        prevent_split(
            row
        )

        for j, value in enumerate(
            values
        ):

            cell = row.cells[
                j
            ]

            set_cell_width(
                cell,
                widths[
                    j
                ]
            )

            set_cell_margins(
                cell
            )

            remove_shading(
                cell
            )

            set_cell_border(
                cell,
                size=8
            )

            cell.vertical_alignment = (
                WD_CELL_VERTICAL_ALIGNMENT.CENTER
            )

            p = cell.paragraphs[
                0
            ]

            alignment = (
                WD_ALIGN_PARAGRAPH.CENTER
                if j in center_columns
                else WD_ALIGN_PARAGRAPH.LEFT
            )

            format_paragraph(
                p,
                alignment
            )

            run = p.add_run(
                str(
                    value
                )
            )

            set_run_font(
                run,
                size_pt=font_size
            )

    return table

# ==================================================================================================
# 23. CREATE DOCUMENT
# ==================================================================================================

doc = Document()

section = doc.sections[
    0
]

section.orientation = (
    WD_ORIENT.LANDSCAPE
)

section.page_width = Inches(
    11
)

section.page_height = Inches(
    8.5
)

section.top_margin = Inches(
    0.42
)

section.bottom_margin = Inches(
    0.42
)

section.left_margin = Inches(
    0.42
)

section.right_margin = Inches(
    0.42
)

# ==================================================================================================
# 24. DEFAULT FONT
# ==================================================================================================

normal = doc.styles[
    "Normal"
]

normal.font.name = (
    "Times New Roman"
)

normal.font.size = Pt(
    8.5
)

normal_rPr = (
    normal.element
    .get_or_add_rPr()
)

normal_fonts = (
    normal_rPr.rFonts
)

if normal_fonts is None:

    normal_fonts = OxmlElement(
        "w:rFonts"
    )

    normal_rPr.insert(
        0,
        normal_fonts
    )

for attr in [
    "ascii",
    "hAnsi",
    "eastAsia",
    "cs",
]:

    normal_fonts.set(
        qn(
            f"w:{attr}"
        ),
        "Times New Roman"
    )

# ==================================================================================================
# 25. TITLE
# ==================================================================================================

title = doc.add_paragraph()

format_paragraph(
    title,
    WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=4
)

run = title.add_run(
    "Supplementary Table S3. "
    "Full donor-grouped 5-fold x 5-repeat nested-cross-validation results "
    "and selected elastic-net hyperparameters for the shared plus regional-deviation model."
)

set_run_font(
    run,
    size_pt=10.3,
    bold=True
)

# ==================================================================================================
# 26. PART A
# ==================================================================================================

add_heading(
    doc,
    "A. Overall nested-CV performance and model-complexity summary"
)

make_table(
    doc,
    headers=[
        "Metric",
        "Frozen result",
        "Evaluation / interpretation note",
    ],
    body_rows=table_s3a,
    widths=[
        2.80,
        2.10,
        5.15,
    ],
    center_columns={
        1
    },
    font_size=8.25
)

# ==================================================================================================
# 27. PART B
# ==================================================================================================

add_heading(
    doc,
    "B. Hyperparameter-selection frequency across the 25 outer folds"
)

make_table(
    doc,
    headers=[
        "Alpha",
        "Deviation penalty",
        "Outer folds selected",
        "Selection frequency",
    ],
    body_rows=table_s3b,
    widths=[
        1.50,
        2.25,
        2.25,
        2.25,
    ],
    center_columns={
        0,
        1,
        2,
        3,
    },
    font_size=8.3
)

# ==================================================================================================
# 28. PART C
# ==================================================================================================

add_heading(
    doc,
    "C. Outer-fold tuning selections and donor-level performance"
)

make_table(
    doc,
    headers=[
        "Repeat",
        "Fold",
        "Train donors",
        "Test donors",
        "Alpha",
        "Dev. penalty",
        "Lambda",
        "Best inner AUROC",
        "Inner AUROC SD",
        "Outer donor AUROC",
    ],
    body_rows=table_s3c,
    widths=[
        0.65,
        0.55,
        0.95,
        0.90,
        0.65,
        0.90,
        1.15,
        1.20,
        1.10,
        1.20,
    ],
    center_columns=set(
        range(
            10
        )
    ),
    font_size=7.65
)

# ==================================================================================================
# 29. PART D
# ==================================================================================================

add_heading(
    doc,
    "D. Outer-fold sample/region performance and selected model complexity"
)

make_table(
    doc,
    headers=[
        "Repeat",
        "Fold",
        "Test samples",
        "Outer sample AUROC",
        "DLPFC AUROC",
        "HIPPO AUROC",
        "Shared nonzero",
        "Deviation nonzero",
    ],
    body_rows=table_s3d,
    widths=[
        0.70,
        0.60,
        1.20,
        1.55,
        1.35,
        1.35,
        1.40,
        1.55,
    ],
    center_columns=set(
        range(
            8
        )
    ),
    font_size=7.8
)

# ==================================================================================================
# 30. NOTES
# ==================================================================================================

notes_heading = doc.add_paragraph()

format_paragraph(
    notes_heading,
    WD_ALIGN_PARAGRAPH.LEFT,
    before=5,
    after=1
)

run = notes_heading.add_run(
    "Notes:"
)

set_run_font(
    run,
    size_pt=8.3,
    bold=True
)

notes_text = (
    "AUROC, area under the receiver-operating-characteristic curve; "
    "DLPFC, dorsolateral prefrontal cortex; HIPPO, hippocampus. "
    "Outer folds were donor-grouped and leakage-free. Within each outer-training set, "
    "hyperparameters were selected using four-fold donor-grouped inner cross-validation. "
    "The primary selection rule was the highest inner AUROC; prespecified tie-breaks were "
    "lower inner-CV uncertainty, a stronger regional-deviation penalty, and then lower alpha. "
    "All 25 outer fits selected alpha = 0.1. Regional-deviation penalty 1 was selected in "
    "18 folds, penalty 2 in one fold, and penalty 8 in six folds. "
    "Repeated-OOF AUROCs were calculated after averaging each donor/sample's five independently "
    "held-out predictions across the five outer-CV repeats. No model fitting or tuning was "
    "repeated for generation of this supplementary table."
)

notes_p = doc.add_paragraph()

format_paragraph(
    notes_p,
    WD_ALIGN_PARAGRAPH.LEFT
)

notes_run = notes_p.add_run(
    notes_text
)

set_run_font(
    notes_run,
    size_pt=8.0
)

# ==================================================================================================
# 31. SAVE DOCX
# ==================================================================================================

doc.save(
    DOCX_PATH
)

if (
    not DOCX_PATH.exists()
    or DOCX_PATH.stat().st_size <= 0
):
    raise RuntimeError(
        "Table S3 DOCX was not created."
    )

print("Microsoft Word DOCX created: PASS")

# ==================================================================================================
# 32. STRUCTURAL READ-BACK
# ==================================================================================================

check_doc = Document(
    DOCX_PATH
)

if len(
    check_doc.tables
) != 4:
    raise RuntimeError(
        f"Expected four Word tables; observed {len(check_doc.tables)}"
    )

expected_columns = [
    3,
    4,
    10,
    8,
]

for idx, expected_ncol in enumerate(
    expected_columns
):

    observed_ncol = len(
        check_doc.tables[
            idx
        ].columns
    )

    if observed_ncol != expected_ncol:
        raise RuntimeError(
            f"Table S3 part {idx + 1} column-count failure: "
            f"observed {observed_ncol}, expected {expected_ncol}"
        )

if len(
    check_doc.tables[
        2
    ].rows
) != 26:
    raise RuntimeError(
        "Table S3C must contain one header plus 25 outer-fold rows."
    )

if len(
    check_doc.tables[
        3
    ].rows
) != 26:
    raise RuntimeError(
        "Table S3D must contain one header plus 25 outer-fold rows."
    )

print("DOCX structural read-back: PASS")

# ==================================================================================================
# 33. NOTES TXT
# ==================================================================================================

NOTES_PATH.write_text(
    (
        "Supplementary Table S3\n"
        "========================================\n\n"
        + notes_text
        + "\n\n"
        + "Formatting:\n"
        + "- Microsoft Word DOCX\n"
        + "- Times New Roman\n"
        + "- Black and white only\n"
        + "- No shading or colored fills\n"
        + "- Full horizontal and vertical borders\n"
        + "- Landscape Letter layout\n"
        + "- Repeating table headers\n"
    ),
    encoding="utf-8"
)

# ==================================================================================================
# 34. SUMMARY TXT
# ==================================================================================================

summary_lines = [
    "=" * 110,
    "STEP 11E.3 - SUPPLEMENTARY TABLE S3",
    "=" * 110,
    "",
    "Frozen nested-CV sources: PASS",
    "Model fitting rerun: NO",
    "Hyperparameter tuning rerun: NO",
    "",
    "CV ARCHITECTURE",
    "Outer CV = 5 folds x 5 repeats",
    "Outer tests = 25",
    "Inner CV = 4-fold donor-grouped",
    "",
    "REPEATED-OOF PERFORMANCE",
    f"Donor AUROC = {perf['Repeated_OOF_donor_AUROC']:.7f}",
    f"Sample AUROC = {perf['Repeated_OOF_sample_AUROC']:.7f}",
    f"DLPFC AUROC = {perf['Repeated_OOF_DLPFC_AUROC']:.7f}",
    f"HIPPO AUROC = {perf['Repeated_OOF_HIPPO_AUROC']:.7f}",
    "",
    "OUTER-FOLD DONOR PERFORMANCE",
    (
        f"Mean +/- SD = "
        f"{perf['Mean_outer_fold_donor_AUROC']:.7f} +/- "
        f"{perf['SD_outer_fold_donor_AUROC']:.7f}"
    ),
    (
        f"Range = "
        f"{perf['Min_outer_fold_donor_AUROC']:.7f} - "
        f"{perf['Max_outer_fold_donor_AUROC']:.7f}"
    ),
    "",
    "MODEL COMPLEXITY",
    (
        f"Shared selected mean / median / range = "
        f"{perf['Mean_selected_shared_genes']:.2f} / "
        f"{perf['Median_selected_shared_genes']:.0f} / "
        f"{perf['Min_selected_shared_genes']:.0f}-"
        f"{perf['Max_selected_shared_genes']:.0f}"
    ),
    (
        f"Deviation selected mean / median / range = "
        f"{perf['Mean_selected_deviation_genes']:.2f} / "
        f"{perf['Median_selected_deviation_genes']:.0f} / "
        f"{perf['Min_selected_deviation_genes']:.0f}-"
        f"{perf['Max_selected_deviation_genes']:.0f}"
    ),
    "",
    "HYPERPARAMETER SELECTION",
    "Alpha 0.1 = 25/25 folds",
    "Deviation penalty 1 = 18/25 folds",
    "Deviation penalty 2 = 1/25 folds",
    "Deviation penalty 8 = 6/25 folds",
    "",
    "WORD CONTENT",
    "S3A = overall nested-CV performance and complexity",
    "S3B = hyperparameter selection frequency",
    "S3C = all 25 folds, tuning and donor performance",
    "S3D = all 25 folds, sample/region performance and complexity",
    "",
    "FORMATTING",
    "Times New Roman = YES",
    "Color/shading = NONE",
    "Full horizontal borders = YES",
    "Full vertical borders = YES",
    "DOCX structural read-back = PASS",
    "",
    f"DOCX = {DOCX_PATH}",
    f"S3A CSV = {S3A_PATH}",
    f"S3B CSV = {S3B_PATH}",
    f"S3C CSV = {S3C_PATH}",
    f"S3D CSV = {S3D_PATH}",
    "=" * 110,
]

SUMMARY_PATH.write_text(
    "\n".join(
        summary_lines
    ),
    encoding="utf-8"
)

# ==================================================================================================
# 35. MANIFEST
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024
                * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


manifest_outputs = [
    DOCX_PATH,
    S3A_PATH,
    S3B_PATH,
    S3C_PATH,
    S3D_PATH,
    NOTES_PATH,
    SUMMARY_PATH,
]

with MANIFEST_PATH.open(
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow(
        [
            "role",
            "file",
            "path",
            "bytes",
            "sha256",
        ]
    )

    for path in required_inputs:

        writer.writerow(
            [
                "input",
                path.name,
                str(
                    path
                ),
                path.stat().st_size,
                sha256_file(
                    path
                ),
            ]
        )

    for path in manifest_outputs:

        writer.writerow(
            [
                "output",
                path.name,
                str(
                    path
                ),
                path.stat().st_size,
                sha256_file(
                    path
                ),
            ]
        )

# ==================================================================================================
# 36. FINAL OUTPUT CHECK
# ==================================================================================================

final_outputs = [
    DOCX_PATH,
    S3A_PATH,
    S3B_PATH,
    S3C_PATH,
    S3D_PATH,
    NOTES_PATH,
    SUMMARY_PATH,
    MANIFEST_PATH,
]

for path in final_outputs:

    if (
        not path.exists()
        or path.stat().st_size <= 0
    ):
        raise RuntimeError(
            f"Missing or zero-byte Table S3 output: {path}"
        )

print("All Supplementary Table S3 outputs created and non-zero: PASS")

# ==================================================================================================
# 37. FINAL REPORT
# ==================================================================================================

print()
print("=" * 110)
print("SUPPLEMENTARY TABLE S3 GENERATION SUMMARY")
print("=" * 110)

print("Frozen nested-CV sources: PASS")
print("Model/CV rerun: NO")
print("Outer folds: 25 - PASS")
print("OOF donor/sample files cross-check: PASS")

print()

print(
    "Repeated-OOF donor / sample AUROC: "
    f"{perf['Repeated_OOF_donor_AUROC']:.7f} / "
    f"{perf['Repeated_OOF_sample_AUROC']:.7f}"
)

print(
    "Repeated-OOF DLPFC / HIPPO AUROC: "
    f"{perf['Repeated_OOF_DLPFC_AUROC']:.7f} / "
    f"{perf['Repeated_OOF_HIPPO_AUROC']:.7f}"
)

print(
    "Mean outer-fold donor AUROC +/- SD: "
    f"{perf['Mean_outer_fold_donor_AUROC']:.7f} +/- "
    f"{perf['SD_outer_fold_donor_AUROC']:.7f}"
)

print()

print(
    "Selected alpha 0.1: 25/25"
)

print(
    "Deviation penalty 1 / 2 / 8: 18 / 1 / 6"
)

print()

print(
    "Shared selected mean / median: "
    f"{perf['Mean_selected_shared_genes']:.2f} / "
    f"{perf['Median_selected_shared_genes']:.0f}"
)

print(
    "Deviation selected mean / median: "
    f"{perf['Mean_selected_deviation_genes']:.2f} / "
    f"{perf['Median_selected_deviation_genes']:.0f}"
)

print()

print("Times New Roman: PASS")
print("Black-and-white only: PASS")
print("Full row borders: PASS")
print("Full column borders: PASS")
print("DOCX structural read-back: PASS")

print()

print("DOCX:")
print(DOCX_PATH)

print()

print("Source CSVs:")
print(S3A_PATH)
print(S3B_PATH)
print(S3C_PATH)
print(S3D_PATH)

print()

print("Manifest:")
print(MANIFEST_PATH)

print()
print("=" * 110)
print("STEP 11E.3 COMPLETE - SUPPLEMENTARY TABLE S3 SAVED PROJECT-LOCALLY")
print("=" * 110)
)"

# --------------------------------------------------------------------------------------------------
# 5. SAVE PYTHON SCRIPT
# --------------------------------------------------------------------------------------------------

writeLines(
    py_code,
    PY_SCRIPT_PATH,
    useBytes = TRUE
)

if (
    !file.exists(PY_SCRIPT_PATH) ||
    file.info(PY_SCRIPT_PATH)$size <= 0
) {
    stop(
        "Table S3 Python generator was not created."
    )
}

cat(
    "Python generator saved: ",
    PY_SCRIPT_PATH,
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 6. RUN PYTHON
# --------------------------------------------------------------------------------------------------

cmd_output <- system2(
    command = PYTHON_EXE,
    args = shQuote(
        PY_SCRIPT_PATH
    ),
    stdout = TRUE,
    stderr = TRUE
)

exit_status <- attr(
    cmd_output,
    "status"
)

cat(
    paste(
        cmd_output,
        collapse = "\n"
    ),
    "\n"
)

if (
    !is.null(exit_status) &&
    exit_status != 0
) {
    stop(
        "Python Table S3 generator failed with exit status ",
        exit_status,
        ". See console output above."
    )
}

# --------------------------------------------------------------------------------------------------
# 7. R-SIDE OUTPUT CHECK
# --------------------------------------------------------------------------------------------------

EXPECTED_DOCX <- file.path(
    OUT_DIR,
    "TableS3_NestedCV_Full_Results_Hyperparameters_v1.docx"
)

EXPECTED_S3A <- file.path(
    OUT_DIR,
    "TableS3A_nestedCV_summary_source_v1.csv"
)

EXPECTED_S3B <- file.path(
    OUT_DIR,
    "TableS3B_hyperparameter_frequency_source_v1.csv"
)

EXPECTED_S3C <- file.path(
    OUT_DIR,
    "TableS3C_outer_fold_tuning_donor_performance_source_v1.csv"
)

EXPECTED_S3D <- file.path(
    OUT_DIR,
    "TableS3D_outer_fold_region_complexity_source_v1.csv"
)

EXPECTED_NOTES <- file.path(
    OUT_DIR,
    "TableS3_notes_v1.txt"
)

EXPECTED_SUMMARY <- file.path(
    OUT_DIR,
    "TableS3_generation_summary_v1.txt"
)

EXPECTED_MANIFEST <- file.path(
    OUT_DIR,
    "TableS3_generation_manifest_v1.csv"
)

expected_outputs <- c(
    EXPECTED_DOCX,
    EXPECTED_S3A,
    EXPECTED_S3B,
    EXPECTED_S3C,
    EXPECTED_S3D,
    EXPECTED_NOTES,
    EXPECTED_SUMMARY,
    EXPECTED_MANIFEST,
    PY_SCRIPT_PATH
)

if (!all(
    file.exists(
        expected_outputs
    )
)) {
    stop(
        "One or more expected Table S3 artifacts are missing."
    )
}

if (any(
    file.info(
        expected_outputs
    )$size <= 0
)) {
    stop(
        "One or more expected Table S3 artifacts are zero-byte."
    )
}

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.3R COMPLETE - TABLE S3 GENERATED THROUGH PYTHON FROM R KERNEL\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "\nMicrosoft Word DOCX:\n",
    EXPECTED_DOCX,
    "\n",
    sep = ""
)

cat(
    "\nSaved Python generator:\n",
    PY_SCRIPT_PATH,
    "\n",
    sep = ""
)

cat(
    "\nSource CSVs:\n",
    EXPECTED_S3A,
    "\n",
    EXPECTED_S3B,
    "\n",
    EXPECTED_S3C,
    "\n",
    EXPECTED_S3D,
    "\n",
    sep = ""
)

cat(
    "\nManifest:\n",
    EXPECTED_MANIFEST,
    "\n",
    sep = ""
)


STEP 11E.3R - SUPPLEMENTARY TABLE S3
FULL 5 x 5 NESTED-CV RESULTS AND HYPERPARAMETER SELECTION
Project sentinel: PASS
Python executable: PASS
Python generator saved: E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/tables/supplementary/TableS3/generate_TableS3_v1.py
STEP 11E.3 - SUPPLEMENTARY TABLE S3
FULL 5 x 5 NESTED-CV RESULTS AND HYPERPARAMETER SELECTION
Project sentinel: PASS
Frozen nested-CV artifacts: PASS
Nested-CV input schemas: PASS
Outer CV structure: 5 repeats x 5 folds = 25 tests - PASS
Outer-fold train/test population accounting: PASS
25-fold summary vs final-performance file: PASS
Canonical nested-CV performance anchors: PASS
Averaged OOF donor/sample files vs final performance: PASS
Hyperparameter selections: alpha 0.1 in 25/25; deviation penalty 1/2/8 = 18/1/6 - PASS
Hyperparameter frequency file vs 25-fold summary: PASS
Microsoft Word DOCX created: PASS
DOCX structural read-back: PASS
All Supplementary Table S3 outputs created and non-zero: PASS

SUPPLEMENTARY TABLE S3 GENERA

In [25]:
# ==================================================================================================
# STEP 11E.4R - SUPPLEMENTARY TABLE S4
# R-KERNEL WRAPPER -> PYTHON-DOCX GENERATOR
# STABILITY REFINEMENT, FINAL-STATE EVIDENCE, AND PERMUTATION NULL
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.4R - SUPPLEMENTARY TABLE S4\n",
    "STABILITY REFINEMENT, FINAL-STATE EVIDENCE, AND PERMUTATION NULL\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 1. PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

observed_project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    observed_project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "Project ID mismatch. Expected ",
        EXPECTED_PROJECT_ID,
        " but observed ",
        observed_project_id
    )
}

cat("Project sentinel: PASS\n")

# --------------------------------------------------------------------------------------------------
# 2. PYTHON
# --------------------------------------------------------------------------------------------------

PYTHON_EXE <- "E:/Python/python.exe"

if (!file.exists(PYTHON_EXE)) {
    stop(
        "Python executable not found: ",
        PYTHON_EXE
    )
}

cat("Python executable: PASS\n")

# --------------------------------------------------------------------------------------------------
# 3. OUTPUT DIRECTORY
# --------------------------------------------------------------------------------------------------

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "supplementary",
    "TableS4"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PY_SCRIPT_PATH <- file.path(
    OUT_DIR,
    "generate_TableS4_v1.py"
)

# --------------------------------------------------------------------------------------------------
# 4. COMPLETE PYTHON GENERATOR
# --------------------------------------------------------------------------------------------------

py_code <- r"(from pathlib import Path
import csv
import hashlib
import math

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


print("=" * 110)
print("STEP 11E.4 - SUPPLEMENTARY TABLE S4")
print("STABILITY REFINEMENT, FINAL-STATE EVIDENCE, AND PERMUTATION NULL")
print("=" * 110)

# ==================================================================================================
# 1. PROJECT
# ==================================================================================================

PROJECT_ROOT = Path(
    "E:/SCZ_CROSSREGION_MSS_Q1"
)

EXPECTED_PROJECT_ID = (
    "SCZ_CROSSREGION_MSS_Q1_2026"
)

SENTINEL = (
    PROJECT_ROOT
    / ".project_id"
)

if not SENTINEL.exists():
    raise FileNotFoundError(
        f"Project sentinel missing: {SENTINEL}"
    )

project_id = SENTINEL.read_text(
    encoding="utf-8"
).strip()

if project_id != EXPECTED_PROJECT_ID:
    raise RuntimeError(
        f"Project sentinel mismatch: {project_id}"
    )

print("Project sentinel: PASS")

# ==================================================================================================
# 2. INPUT PATHS
# ==================================================================================================

OUTER_DIR = (
    PROJECT_ROOT
    / "06_models"
    / "stability_v1"
)

HALF_DIR = (
    PROJECT_ROOT
    / "07_stability"
    / "complementary_half_sample_v1"
)

NULL_DIR = (
    PROJECT_ROOT
    / "07_stability"
    / "permutation_null_v1"
)

FINAL_DIR = (
    PROJECT_ROOT
    / "07_stability"
    / "final_stable_state_v1"
)

OUTER_STABLE_PATH = (
    OUTER_DIR
    / "step05a_stable_shared_genes_v1.csv"
)

OUTER_SUMMARY_PATH = (
    OUTER_DIR
    / "step05a_stability_summary_v1.csv"
)

HALF_STABLE_PATH = (
    HALF_DIR
    / "step06b_half_sample_candidate_shared_v1.csv"
)

HALF_SUMMARY_PATH = (
    HALF_DIR
    / "step06b_half_sample_stability_summary_v1.csv"
)

NULL24_PATH = (
    NULL_DIR
    / "step06c_candidate24_null_behavior_v1.csv"
)

NULL_SUMMARY_PATH = (
    NULL_DIR
    / "step06c_permutation_null_summary_v1.csv"
)

FINAL_PATH = (
    FINAL_DIR
    / "step06d_FINAL_stable_shared_state_v1.csv"
)

FINAL_SUMMARY_PATH = (
    FINAL_DIR
    / "step06d_final_stable_state_summary_v1.csv"
)

FINAL_SHA_PATH = (
    FINAL_DIR
    / "step06d_FINAL_stable_shared_state_v1.sha256"
)

FINAL_LOCK_PATH = (
    FINAL_DIR
    / "step06d_FINAL_STABLE_STATE_LOCK_v1.txt"
)

required_inputs = [
    OUTER_STABLE_PATH,
    OUTER_SUMMARY_PATH,
    HALF_STABLE_PATH,
    HALF_SUMMARY_PATH,
    NULL24_PATH,
    NULL_SUMMARY_PATH,
    FINAL_PATH,
    FINAL_SUMMARY_PATH,
    FINAL_SHA_PATH,
    FINAL_LOCK_PATH,
]

missing_inputs = [
    str(p)
    for p in required_inputs
    if not p.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required stability artifact(s) missing:\n"
        + "\n".join(
            missing_inputs
        )
    )

print("Frozen stability/permutation artifacts: PASS")

# ==================================================================================================
# 3. CSV READER
# ==================================================================================================

def read_csv_dict(path):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        reader = csv.DictReader(f)

        rows = list(
            reader
        )

        columns = reader.fieldnames or []

    return rows, columns


outer_stable, outer_cols = read_csv_dict(
    OUTER_STABLE_PATH
)

outer_summary_rows, outer_summary_cols = read_csv_dict(
    OUTER_SUMMARY_PATH
)

half_stable, half_cols = read_csv_dict(
    HALF_STABLE_PATH
)

half_summary_rows, half_summary_cols = read_csv_dict(
    HALF_SUMMARY_PATH
)

null24, null24_cols = read_csv_dict(
    NULL24_PATH
)

null_summary_rows, null_summary_cols = read_csv_dict(
    NULL_SUMMARY_PATH
)

final_rows, final_cols = read_csv_dict(
    FINAL_PATH
)

final_summary_rows, final_summary_cols = read_csv_dict(
    FINAL_SUMMARY_PATH
)

# ==================================================================================================
# 4. HELPERS
# ==================================================================================================

def num(x):

    try:
        value = float(x)

    except Exception as exc:
        raise RuntimeError(
            f"Cannot parse numeric value: {x}"
        ) from exc

    if not math.isfinite(
        value
    ):
        raise RuntimeError(
            f"Non-finite numeric value: {x}"
        )

    return value


def int_value(x):

    value = num(
        x
    )

    if abs(
        value - round(value)
    ) > 1e-8:
        raise RuntimeError(
            f"Expected integer-like value: {x}"
        )

    return int(
        round(
            value
        )
    )


def metric_dict(
    rows,
    columns,
    label
):

    if not all(
        x in columns
        for x in [
            "metric",
            "value",
        ]
    ):
        raise RuntimeError(
            f"{label} is not a metric/value table."
        )

    result = {}

    for row in rows:

        metric = str(
            row[
                "metric"
            ]
        ).strip()

        if metric in result:
            raise RuntimeError(
                f"Duplicate metric in {label}: {metric}"
            )

        result[
            metric
        ] = num(
            row[
                "value"
            ]
        )

    return result


def require_metric(
    source,
    key,
    label
):

    if key not in source:
        raise RuntimeError(
            f"Missing {label} metric: {key}"
        )

    return source[
        key
    ]


def sha256_file(
    path
):

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


# ==================================================================================================
# 5. SUMMARY TABLES
# ==================================================================================================

outer_summary = metric_dict(
    outer_summary_rows,
    outer_summary_cols,
    "outer stability summary"
)

half_summary = metric_dict(
    half_summary_rows,
    half_summary_cols,
    "half-sample stability summary"
)

null_summary = metric_dict(
    null_summary_rows,
    null_summary_cols,
    "permutation-null summary"
)

final_summary = metric_dict(
    final_summary_rows,
    final_summary_cols,
    "final stable-state summary"
)

print("Metric/value summary schemas: PASS")

# ==================================================================================================
# 6. OUTER-STABILITY ANCHORS
# ==================================================================================================

outer_anchors = {
    "Number_outer_models":
        25,

    "Primary_genes":
        23740,

    "Shared_selection_threshold":
        0.80,

    "Shared_minimum_models_required":
        20,

    "Sign_consistency_threshold":
        0.90,

    "Genes_selected_shared_at_least_once":
        6131,

    "Genes_selected_shared_20_of_25_or_more":
        403,

    "Stable_shared_genes":
        403,

    "Genes_selected_deviation_at_least_once":
        390,

    "Genes_selected_deviation_20_of_25_or_more":
        0,

    "Stable_deviation_genes":
        0,

    "Stable_shared_positive":
        191,

    "Stable_shared_negative":
        212,
}

for key, expected in outer_anchors.items():

    observed = require_metric(
        outer_summary,
        key,
        "outer stability"
    )

    if abs(
        observed - expected
    ) > 1e-8:

        raise RuntimeError(
            f"Outer stability anchor changed for {key}: "
            f"observed={observed}, expected={expected}"
        )

if len(
    outer_stable
) != 403:
    raise RuntimeError(
        f"Expected 403 outer-stable shared genes; observed {len(outer_stable)}"
    )

if "ensemblID_stable" not in outer_cols:
    raise RuntimeError(
        "Outer stable-shared file missing ensemblID_stable."
    )

outer_ids = {
    str(
        r[
            "ensemblID_stable"
        ]
    ).strip()
    for r in outer_stable
}

if len(
    outer_ids
) != 403:
    raise RuntimeError(
        "Outer stable-shared set contains duplicate stable Ensembl IDs."
    )

print("Outer-model stability anchors: PASS")

# ==================================================================================================
# 7. HALF-SAMPLE ANCHORS
# ==================================================================================================

half_required_anchors = {
    "Half_sample_fits":
        200,

    "Alpha_fixed":
        0.1,

    "Deviation_penalty_fixed":
        1.0,

    "Shared_selection_threshold":
        0.80,

    "Shared_sign_threshold":
        0.90,

    "Genes_selected_shared_at_least_once":
        13592,

    "Candidate_stable_shared_genes":
        24,

    "Candidate_stable_deviation_genes":
        0,

    "Outer_25fit_candidate_genes":
        403,

    "Overlap_half_sample_vs_outer":
        24,

    "Union_half_sample_vs_outer":
        403,

    "Half_sample_positive_stable":
        9,

    "Half_sample_negative_stable":
        15,
}

for key, expected in half_required_anchors.items():

    observed = require_metric(
        half_summary,
        key,
        "half-sample stability"
    )

    if abs(
        observed - expected
    ) > 1e-8:

        raise RuntimeError(
            f"Half-sample anchor changed for {key}: "
            f"observed={observed}, expected={expected}"
        )

HALF_JACCARD = require_metric(
    half_summary,
    "Jaccard_half_sample_vs_outer",
    "half-sample stability"
)

expected_jaccard = (
    24.0
    / 403.0
)

if abs(
    HALF_JACCARD
    - expected_jaccard
) > 1e-8:
    raise RuntimeError(
        f"Half-sample Jaccard changed: {HALF_JACCARD}"
    )

if len(
    half_stable
) != 24:
    raise RuntimeError(
        f"Expected 24 half-sample stable genes; observed {len(half_stable)}"
    )

if "ensemblID_stable" not in half_cols:
    raise RuntimeError(
        "Half-sample candidate file missing ensemblID_stable."
    )

half_ids = {
    str(
        r[
            "ensemblID_stable"
        ]
    ).strip()
    for r in half_stable
}

if len(
    half_ids
) != 24:
    raise RuntimeError(
        "Half-sample stable set contains duplicate stable Ensembl IDs."
    )

if not half_ids.issubset(
    outer_ids
):
    raise RuntimeError(
        "Half-sample stable set is not a subset of the 403 outer-stable genes."
    )

print("Complementary half-sample stability anchors: PASS")

# ==================================================================================================
# 8. PERMUTATION-NULL ANCHORS
# ==================================================================================================

null_anchors = {
    "Permutations":
        100,

    "Observed_reference_donor_AUROC":
        0.76983010,

    "Null_donor_AUROC_mean":
        0.50645353,

    "Null_donor_AUROC_SD":
        0.03258309,

    "Null_donor_AUROC_min":
        0.43800608,

    "Null_donor_AUROC_max":
        0.59950234,

    "Empirical_P_observed_donor_AUROC":
        0.00990099,

    "Max_null_shared_selection_probability":
        0.22,

    "Max_null_deviation_selection_probability":
        0.12,

    "Null_stable_like_shared_genes":
        0,

    "Null_stable_like_deviation_genes":
        0,

    "Candidate24_max_null_selection_probability":
        0.22,

    "Candidate24_median_null_selection_probability":
        0.065,
}

for key, expected in null_anchors.items():

    observed = require_metric(
        null_summary,
        key,
        "permutation null"
    )

    tolerance = (
        5e-8
        if abs(
            expected
        ) < 1
        else 1e-8
    )

    if abs(
        observed - expected
    ) > tolerance:

        raise RuntimeError(
            f"Permutation-null anchor changed for {key}: "
            f"observed={observed}, expected={expected}"
        )

if len(
    null24
) != 24:
    raise RuntimeError(
        f"Expected 24 candidate-null rows; observed {len(null24)}"
    )

required_null24_cols = [
    "ensemblID_stable",
    "null_shared_selection_probability",
]

missing_null24_cols = [
    x
    for x in required_null24_cols
    if x not in null24_cols
]

if missing_null24_cols:
    raise RuntimeError(
        "Candidate24 null-behavior schema missing: "
        + ", ".join(
            missing_null24_cols
        )
    )

null24_ids = {
    str(
        r[
            "ensemblID_stable"
        ]
    ).strip()
    for r in null24
}

if len(
    null24_ids
) != 24:
    raise RuntimeError(
        "Candidate24 null-behavior table contains duplicate IDs."
    )

null24_probabilities = [
    num(
        r[
            "null_shared_selection_probability"
        ]
    )
    for r in null24
]

if abs(
    max(
        null24_probabilities
    )
    - 0.22
) > 1e-8:
    raise RuntimeError(
        "Candidate24 maximum null selection probability changed."
    )

sorted_null_probs = sorted(
    null24_probabilities
)

candidate24_median = (
    sorted_null_probs[
        11
    ]
    + sorted_null_probs[
        12
    ]
) / 2.0

if abs(
    candidate24_median
    - 0.065
) > 1e-8:
    raise RuntimeError(
        f"Candidate24 median null selection probability changed: {candidate24_median}"
    )

print("Permutation-null anchors: PASS")

# ==================================================================================================
# 9. FINAL 24 SCHEMA
# ==================================================================================================

required_final_cols = [
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "direction",
    "outer_selection_frequency",
    "outer_sign_consistency",
    "half_sample_selection_probability",
    "half_sample_sign_consistency",
    "null_selection_probability",
    "half_sample_median_nonzero_beta",
]

missing_final_cols = [
    x
    for x in required_final_cols
    if x not in final_cols
]

if missing_final_cols:
    raise RuntimeError(
        "Final 24 stability table missing field(s): "
        + ", ".join(
            missing_final_cols
        )
    )

if len(
    final_rows
) != 24:
    raise RuntimeError(
        f"Expected final 24 rows; observed {len(final_rows)}"
    )

final_ids = {
    str(
        r[
            "ensemblID_stable"
        ]
    ).strip()
    for r in final_rows
}

if len(
    final_ids
) != 24:
    raise RuntimeError(
        "Final stable-state table contains duplicate stable Ensembl IDs."
    )

if final_ids != half_ids:
    raise RuntimeError(
        "Final 24 set is not exactly equal to the half-sample stable set."
    )

if not final_ids.issubset(
    outer_ids
):
    raise RuntimeError(
        "Final 24 set is not a subset of the outer stable set."
    )

if final_ids != null24_ids:
    raise RuntimeError(
        "Final 24 set and candidate24 null-behavior set do not match."
    )

print("Final 24 set identity across outer/half/null sources: PASS")

# ==================================================================================================
# 10. FINAL 24 HARD CHECKS
# ==================================================================================================

outer_freq = [
    num(
        r[
            "outer_selection_frequency"
        ]
    )
    for r in final_rows
]

outer_sign = [
    num(
        r[
            "outer_sign_consistency"
        ]
    )
    for r in final_rows
]

half_prob = [
    num(
        r[
            "half_sample_selection_probability"
        ]
    )
    for r in final_rows
]

half_sign = [
    num(
        r[
            "half_sample_sign_consistency"
        ]
    )
    for r in final_rows
]

final_null_prob = [
    num(
        r[
            "null_selection_probability"
        ]
    )
    for r in final_rows
]

if min(
    outer_freq
) < 0.80:
    raise RuntimeError(
        "Final 24 contains a gene below outer stability threshold."
    )

if min(
    outer_sign
) < 0.90:
    raise RuntimeError(
        "Final 24 contains a gene below outer sign-consistency threshold."
    )

if abs(
    min(
        half_prob
    )
    - 0.80
) > 1e-8:
    raise RuntimeError(
        f"Minimum half-sample selection probability changed: {min(half_prob)}"
    )

if abs(
    min(
        half_sign
    )
    - 1.0
) > 1e-8:
    raise RuntimeError(
        f"Minimum half-sample sign consistency changed: {min(half_sign)}"
    )

if abs(
    max(
        final_null_prob
    )
    - 0.22
) > 1e-8:
    raise RuntimeError(
        "Final 24 maximum null selection probability changed."
    )

positive_n = sum(
    str(
        r[
            "direction"
        ]
    ).strip().lower()
    == "positive"
    for r in final_rows
)

negative_n = sum(
    str(
        r[
            "direction"
        ]
    ).strip().lower()
    == "negative"
    for r in final_rows
)

protein_coding_n = sum(
    str(
        r[
            "gene_type"
        ]
    ).strip()
    == "protein_coding"
    for r in final_rows
)

non_protein_coding_n = (
    len(
        final_rows
    )
    - protein_coding_n
)

if (
    positive_n != 9
    or negative_n != 15
):
    raise RuntimeError(
        f"Final direction composition changed: {positive_n}/{negative_n}"
    )

if (
    protein_coding_n != 12
    or non_protein_coding_n != 12
):
    raise RuntimeError(
        "Final coding/noncoding composition changed."
    )

print("Final 24 gene-level stability evidence: PASS")

# ==================================================================================================
# 11. FINAL SUMMARY ANCHORS
# ==================================================================================================

final_anchors = {
    "Outer_CV_candidate_shared_genes":
        403,

    "Half_sample_candidate_shared_genes":
        24,

    "Final_stable_shared_genes":
        24,

    "Final_positive_direction":
        9,

    "Final_negative_direction":
        15,

    "Final_protein_coding":
        12,

    "Final_non_protein_coding":
        12,

    "Outer_half_direction_agreement":
        24,

    "Minimum_half_sample_selection_probability":
        0.80,

    "Minimum_half_sample_sign_consistency":
        1.0,

    "Maximum_candidate_null_selection_probability":
        0.22,

    "Observed_reference_donor_AUROC":
        0.76983010,

    "Null_donor_AUROC_mean":
        0.50645353,

    "Null_donor_AUROC_max":
        0.59950234,

    "Permutation_empirical_P":
        0.00990099,

    "Null_stable_like_shared_genes":
        0,

    "Maximum_null_shared_selection_probability":
        0.22,

    "Stable_deviation_genes":
        0,
}

for key, expected in final_anchors.items():

    observed = require_metric(
        final_summary,
        key,
        "final stable-state summary"
    )

    if abs(
        observed - expected
    ) > 5e-8:

        raise RuntimeError(
            f"Final-state summary anchor changed for {key}: "
            f"observed={observed}, expected={expected}"
        )

print("Final stable-state summary anchors: PASS")

# ==================================================================================================
# 12. FINAL SHA256
# ==================================================================================================

stored_sha = FINAL_SHA_PATH.read_text(
    encoding="utf-8"
).strip().lower()

computed_sha = sha256_file(
    FINAL_PATH
).lower()

if stored_sha != computed_sha:
    raise RuntimeError(
        "Final 24 SHA256 file does not match current final artifact."
    )

print("Canonical final 24 SHA256: PASS")

# ==================================================================================================
# 13. PREPARE TABLE S4A - STABILITY REFINEMENT SUMMARY
# ==================================================================================================

table_s4a = [
    [
        "Outer-model stability",
        "Outer fitted models",
        "25",
        "Observed repeated outer-training models.",
    ],

    [
        "Outer-model stability",
        "Primary gene universe",
        "23,740",
        "Shared and regional-deviation coefficients assessed for every autosomal gene.",
    ],

    [
        "Outer-model stability",
        "Shared genes selected at least once",
        "6,131",
        "Nonzero shared coefficient in at least one of 25 outer models.",
    ],

    [
        "Outer-model stability",
        "Stable shared genes",
        "403",
        "Selection frequency >=0.80 and sign consistency >=0.90.",
    ],

    [
        "Outer-model stability",
        "Stable regional-deviation genes",
        "0",
        "No regional-deviation feature satisfied both stability thresholds.",
    ],

    [
        "Complementary half-sample",
        "Half-sample fits",
        "200",
        "100 complementary donor pairs, yielding 200 fitted half-samples.",
    ],

    [
        "Complementary half-sample",
        "Shared genes selected at least once",
        "13,592",
        "Nonzero shared coefficient in at least one half-sample fit.",
    ],

    [
        "Complementary half-sample",
        "Stable shared genes",
        "24",
        "Selection probability >=0.80 and sign consistency >=0.90.",
    ],

    [
        "Complementary half-sample",
        "Stable regional-deviation genes",
        "0",
        "No deviation feature satisfied the half-sample stability rule.",
    ],

    [
        "Refinement",
        "Outer-stable / half-sample intersection",
        "24",
        "All 24 half-sample-stable features were contained within the 403 outer-stable set.",
    ],

    [
        "Refinement",
        "Outer vs half-sample Jaccard index",
        f"{HALF_JACCARD:.6f}",
        "24 / 403 = 0.059553.",
    ],

    [
        "Final state",
        "Final frozen shared features",
        "24",
        "Intersection with coefficient-direction agreement.",
    ],

    [
        "Final state",
        "Direction agreement",
        "24 / 24",
        "Outer-model and half-sample dominant directions agreed for every final feature.",
    ],

    [
        "Final state",
        "Positive / negative directions",
        "9 / 15",
        "Direction of the frozen shared predictive coefficient.",
    ],

    [
        "Final state",
        "Protein-coding / non-protein-coding",
        "12 / 12",
        "Frozen annotation composition.",
    ],
]

# ==================================================================================================
# 14. PREPARE TABLE S4B - FINAL 24 GENE-LEVEL STABILITY EVIDENCE
# ==================================================================================================

def display_gene(
    row
):

    symbol = str(
        row[
            "gene_symbol"
        ]
    ).strip()

    if (
        not symbol
        or symbol.upper()
        in {
            "NA",
            "NAN",
            "NONE",
        }
    ):
        return str(
            row[
                "ensemblID_stable"
            ]
        ).strip()

    return symbol


final_rows_sorted = sorted(
    final_rows,
    key=lambda r: (
        -num(
            r[
                "half_sample_selection_probability"
            ]
        ),
        str(
            r[
                "ensemblID_stable"
            ]
        ),
    )
)

table_s4b = []

for row in final_rows_sorted:

    table_s4b.append(
        [
            display_gene(
                row
            ),

            str(
                row[
                    "ensemblID_stable"
                ]
            ).strip(),

            str(
                row[
                    "direction"
                ]
            ).strip(),

            f"{num(row['outer_selection_frequency']):.2f}",

            f"{num(row['outer_sign_consistency']):.2f}",

            f"{num(row['half_sample_selection_probability']):.3f}",

            f"{num(row['half_sample_sign_consistency']):.2f}",

            f"{num(row['null_selection_probability']):.2f}",

            f"{num(row['half_sample_median_nonzero_beta']):.6f}",
        ]
    )

# ==================================================================================================
# 15. PREPARE TABLE S4C - PERMUTATION NULL
# ==================================================================================================

table_s4c = [
    [
        "Diagnosis permutations",
        "100",
        "Diagnosis permuted at donor level.",
    ],

    [
        "Observed matched-reference donor AUROC",
        f"{null_summary['Observed_reference_donor_AUROC']:.4f}",
        "Observed reference from the fixed permutation-analysis configuration.",
    ],

    [
        "Null donor AUROC, mean +/- SD",
        (
            f"{null_summary['Null_donor_AUROC_mean']:.4f} "
            f"+/- {null_summary['Null_donor_AUROC_SD']:.4f}"
        ),
        "Across 100 donor-level diagnosis permutations.",
    ],

    [
        "Null donor AUROC range",
        (
            f"{null_summary['Null_donor_AUROC_min']:.4f} - "
            f"{null_summary['Null_donor_AUROC_max']:.4f}"
        ),
        "Minimum and maximum null donor AUROC.",
    ],

    [
        "Empirical P for observed donor AUROC",
        f"{null_summary['Empirical_P_observed_donor_AUROC']:.6f}",
        "(1 + number of null AUROCs >= observed) / (100 + 1).",
    ],

    [
        "Maximum null shared-selection probability",
        f"{null_summary['Max_null_shared_selection_probability']:.2f}",
        "Maximum across the complete gene universe.",
    ],

    [
        "Maximum null deviation-selection probability",
        f"{null_summary['Max_null_deviation_selection_probability']:.2f}",
        "Maximum across the complete gene universe.",
    ],

    [
        "Null stable-like shared genes",
        "0",
        "No null gene satisfied the observed shared stability thresholds.",
    ],

    [
        "Null stable-like deviation genes",
        "0",
        "No null gene satisfied the observed deviation stability thresholds.",
    ],

    [
        "Final-24 maximum null selection probability",
        f"{null_summary['Candidate24_max_null_selection_probability']:.2f}",
        "Maximum null shared-selection probability among the final 24 candidates.",
    ],

    [
        "Final-24 median null selection probability",
        f"{null_summary['Candidate24_median_null_selection_probability']:.3f}",
        "Median null shared-selection probability among the final 24 candidates.",
    ],
]

# ==================================================================================================
# 16. OUTPUT PATHS
# ==================================================================================================

OUT_DIR = (
    PROJECT_ROOT
    / "11_manuscript"
    / "tables"
    / "supplementary"
    / "TableS4"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOCX_PATH = (
    OUT_DIR
    / "TableS4_Stability_Refinement_Permutation_Evidence_v1.docx"
)

S4A_PATH = (
    OUT_DIR
    / "TableS4A_stability_refinement_summary_source_v1.csv"
)

S4B_PATH = (
    OUT_DIR
    / "TableS4B_final24_stability_evidence_source_v1.csv"
)

S4C_PATH = (
    OUT_DIR
    / "TableS4C_permutation_null_summary_source_v1.csv"
)

NOTES_PATH = (
    OUT_DIR
    / "TableS4_notes_v1.txt"
)

SUMMARY_PATH = (
    OUT_DIR
    / "TableS4_generation_summary_v1.txt"
)

MANIFEST_PATH = (
    OUT_DIR
    / "TableS4_generation_manifest_v1.csv"
)

# ==================================================================================================
# 17. SOURCE CSV OUTPUTS
# ==================================================================================================

def write_csv(
    path,
    header,
    rows
):

    with path.open(
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        writer = csv.writer(
            f
        )

        writer.writerow(
            header
        )

        writer.writerows(
            rows
        )


write_csv(
    S4A_PATH,
    [
        "Analysis stage",
        "Metric",
        "Frozen result",
        "Interpretation / rule",
    ],
    table_s4a
)

write_csv(
    S4B_PATH,
    [
        "Gene",
        "Stable Ensembl ID",
        "Direction",
        "Outer selection frequency",
        "Outer sign consistency",
        "Half-sample selection probability",
        "Half-sample sign consistency",
        "Null selection probability",
        "Half-sample median nonzero beta",
    ],
    table_s4b
)

write_csv(
    S4C_PATH,
    [
        "Permutation-null metric",
        "Frozen result",
        "Interpretation",
    ],
    table_s4c
)

# ==================================================================================================
# 18. WORD HELPERS
# ==================================================================================================

def set_run_font(
    run,
    size_pt=9,
    bold=False,
    italic=False
):

    run.font.name = (
        "Times New Roman"
    )

    run.font.size = Pt(
        size_pt
    )

    run.bold = bold
    run.italic = italic

    rPr = (
        run._element
        .get_or_add_rPr()
    )

    rFonts = rPr.rFonts

    if rFonts is None:

        rFonts = OxmlElement(
            "w:rFonts"
        )

        rPr.insert(
            0,
            rFonts
        )

    for attr in [
        "ascii",
        "hAnsi",
        "eastAsia",
        "cs",
    ]:

        rFonts.set(
            qn(
                f"w:{attr}"
            ),
            "Times New Roman"
        )


def set_cell_margins(
    cell,
    top=70,
    start=75,
    bottom=70,
    end=75
):

    tcPr = (
        cell._tc
        .get_or_add_tcPr()
    )

    tcMar = (
        tcPr
        .first_child_found_in(
            "w:tcMar"
        )
    )

    if tcMar is None:

        tcMar = OxmlElement(
            "w:tcMar"
        )

        tcPr.append(
            tcMar
        )

    for name, value in {
        "top": top,
        "start": start,
        "bottom": bottom,
        "end": end,
    }.items():

        node = tcMar.find(
            qn(
                f"w:{name}"
            )
        )

        if node is None:

            node = OxmlElement(
                f"w:{name}"
            )

            tcMar.append(
                node
            )

        node.set(
            qn(
                "w:w"
            ),
            str(
                value
            )
        )

        node.set(
            qn(
                "w:type"
            ),
            "dxa"
        )


def remove_shading(
    cell
):

    tcPr = (
        cell._tc
        .get_or_add_tcPr()
    )

    for node in list(
        tcPr.findall(
            qn(
                "w:shd"
            )
        )
    ):

        tcPr.remove(
            node
        )


def set_cell_width(
    cell,
    width_inches
):

    tcPr = (
        cell._tc
        .get_or_add_tcPr()
    )

    tcW = (
        tcPr
        .first_child_found_in(
            "w:tcW"
        )
    )

    if tcW is None:

        tcW = OxmlElement(
            "w:tcW"
        )

        tcPr.append(
            tcW
        )

    tcW.set(
        qn(
            "w:w"
        ),
        str(
            int(
                width_inches
                * 1440
            )
        )
    )

    tcW.set(
        qn(
            "w:type"
        ),
        "dxa"
    )


def set_cell_border(
    cell,
    size=8
):

    tcPr = (
        cell._tc
        .get_or_add_tcPr()
    )

    borders = (
        tcPr
        .first_child_found_in(
            "w:tcBorders"
        )
    )

    if borders is None:

        borders = OxmlElement(
            "w:tcBorders"
        )

        tcPr.append(
            borders
        )

    for edge_name in [
        "top",
        "bottom",
        "left",
        "right",
    ]:

        edge = borders.find(
            qn(
                f"w:{edge_name}"
            )
        )

        if edge is None:

            edge = OxmlElement(
                f"w:{edge_name}"
            )

            borders.append(
                edge
            )

        edge.set(
            qn(
                "w:val"
            ),
            "single"
        )

        edge.set(
            qn(
                "w:sz"
            ),
            str(
                size
            )
        )

        edge.set(
            qn(
                "w:space"
            ),
            "0"
        )

        edge.set(
            qn(
                "w:color"
            ),
            "000000"
        )


def set_table_borders(
    table
):

    tblPr = (
        table._tbl.tblPr
    )

    borders = (
        tblPr
        .first_child_found_in(
            "w:tblBorders"
        )
    )

    if borders is None:

        borders = OxmlElement(
            "w:tblBorders"
        )

        tblPr.append(
            borders
        )

    specs = {
        "top": 10,
        "bottom": 10,
        "left": 10,
        "right": 10,
        "insideH": 8,
        "insideV": 8,
    }

    for name, size in specs.items():

        edge = borders.find(
            qn(
                f"w:{name}"
            )
        )

        if edge is None:

            edge = OxmlElement(
                f"w:{name}"
            )

            borders.append(
                edge
            )

        edge.set(
            qn(
                "w:val"
            ),
            "single"
        )

        edge.set(
            qn(
                "w:sz"
            ),
            str(
                size
            )
        )

        edge.set(
            qn(
                "w:space"
            ),
            "0"
        )

        edge.set(
            qn(
                "w:color"
            ),
            "000000"
        )


def fixed_layout(
    table
):

    tblPr = (
        table._tbl.tblPr
    )

    layout = (
        tblPr
        .first_child_found_in(
            "w:tblLayout"
        )
    )

    if layout is None:

        layout = OxmlElement(
            "w:tblLayout"
        )

        tblPr.append(
            layout
        )

    layout.set(
        qn(
            "w:type"
        ),
        "fixed"
    )


def repeat_header(
    row
):

    trPr = (
        row._tr
        .get_or_add_trPr()
    )

    node = OxmlElement(
        "w:tblHeader"
    )

    node.set(
        qn(
            "w:val"
        ),
        "true"
    )

    trPr.append(
        node
    )


def prevent_split(
    row
):

    trPr = (
        row._tr
        .get_or_add_trPr()
    )

    node = OxmlElement(
        "w:cantSplit"
    )

    node.set(
        qn(
            "w:val"
        ),
        "true"
    )

    trPr.append(
        node
    )


def format_paragraph(
    paragraph,
    alignment=WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=0
):

    paragraph.alignment = (
        alignment
    )

    fmt = (
        paragraph
        .paragraph_format
    )

    fmt.space_before = Pt(
        before
    )

    fmt.space_after = Pt(
        after
    )

    fmt.line_spacing = 1.0


def add_heading(
    document,
    text
):

    p = (
        document
        .add_paragraph()
    )

    format_paragraph(
        p,
        WD_ALIGN_PARAGRAPH.LEFT,
        before=5,
        after=3
    )

    run = p.add_run(
        text
    )

    set_run_font(
        run,
        size_pt=9.4,
        bold=True
    )


def make_table(
    document,
    headers,
    body_rows,
    widths,
    center_columns=None,
    font_size=8.1
):

    if center_columns is None:
        center_columns = set()

    table = document.add_table(
        rows=1,
        cols=len(
            headers
        )
    )

    table.alignment = (
        WD_TABLE_ALIGNMENT.CENTER
    )

    table.autofit = False

    fixed_layout(
        table
    )

    set_table_borders(
        table
    )

    header_row = table.rows[
        0
    ]

    for j, text in enumerate(
        headers
    ):

        cell = header_row.cells[
            j
        ]

        set_cell_width(
            cell,
            widths[
                j
            ]
        )

        set_cell_margins(
            cell
        )

        remove_shading(
            cell
        )

        set_cell_border(
            cell,
            size=10
        )

        cell.vertical_alignment = (
            WD_CELL_VERTICAL_ALIGNMENT.CENTER
        )

        p = cell.paragraphs[
            0
        ]

        alignment = (
            WD_ALIGN_PARAGRAPH.CENTER
            if j in center_columns
            else WD_ALIGN_PARAGRAPH.LEFT
        )

        format_paragraph(
            p,
            alignment
        )

        run = p.add_run(
            str(
                text
            )
        )

        set_run_font(
            run,
            size_pt=8.25,
            bold=True
        )

    repeat_header(
        header_row
    )

    prevent_split(
        header_row
    )

    for values in body_rows:

        row = table.add_row()

        prevent_split(
            row
        )

        for j, value in enumerate(
            values
        ):

            cell = row.cells[
                j
            ]

            set_cell_width(
                cell,
                widths[
                    j
                ]
            )

            set_cell_margins(
                cell
            )

            remove_shading(
                cell
            )

            set_cell_border(
                cell,
                size=8
            )

            cell.vertical_alignment = (
                WD_CELL_VERTICAL_ALIGNMENT.CENTER
            )

            p = cell.paragraphs[
                0
            ]

            alignment = (
                WD_ALIGN_PARAGRAPH.CENTER
                if j in center_columns
                else WD_ALIGN_PARAGRAPH.LEFT
            )

            format_paragraph(
                p,
                alignment
            )

            run = p.add_run(
                str(
                    value
                )
            )

            set_run_font(
                run,
                size_pt=font_size
            )

    return table

# ==================================================================================================
# 19. CREATE WORD DOCUMENT
# ==================================================================================================

doc = Document()

section = doc.sections[
    0
]

section.orientation = (
    WD_ORIENT.LANDSCAPE
)

section.page_width = Inches(
    11
)

section.page_height = Inches(
    8.5
)

section.top_margin = Inches(
    0.42
)

section.bottom_margin = Inches(
    0.42
)

section.left_margin = Inches(
    0.42
)

section.right_margin = Inches(
    0.42
)

# ==================================================================================================
# 20. DEFAULT FONT
# ==================================================================================================

normal = doc.styles[
    "Normal"
]

normal.font.name = (
    "Times New Roman"
)

normal.font.size = Pt(
    8.5
)

normal_rPr = (
    normal.element
    .get_or_add_rPr()
)

normal_fonts = (
    normal_rPr.rFonts
)

if normal_fonts is None:

    normal_fonts = OxmlElement(
        "w:rFonts"
    )

    normal_rPr.insert(
        0,
        normal_fonts
    )

for attr in [
    "ascii",
    "hAnsi",
    "eastAsia",
    "cs",
]:

    normal_fonts.set(
        qn(
            f"w:{attr}"
        ),
        "Times New Roman"
    )

# ==================================================================================================
# 21. TITLE
# ==================================================================================================

title = doc.add_paragraph()

format_paragraph(
    title,
    WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=4
)

run = title.add_run(
    "Supplementary Table S4. "
    "Stability refinement, gene-level evidence for the final 24-feature shared state, "
    "and donor-level permutation-null analysis."
)

set_run_font(
    run,
    size_pt=10.3,
    bold=True
)

# ==================================================================================================
# 22. PART A
# ==================================================================================================

add_heading(
    doc,
    "A. Stability-selection and refinement summary"
)

make_table(
    doc,
    headers=[
        "Analysis stage",
        "Metric",
        "Frozen result",
        "Interpretation / rule",
    ],
    body_rows=table_s4a,
    widths=[
        1.80,
        2.65,
        1.50,
        4.20,
    ],
    center_columns={
        2
    },
    font_size=8.15
)

# ==================================================================================================
# 23. PART B
# ==================================================================================================

add_heading(
    doc,
    "B. Gene-level stability evidence for the final 24-feature shared state"
)

make_table(
    doc,
    headers=[
        "Gene",
        "Stable Ensembl ID",
        "Direction",
        "Outer sel. freq.",
        "Outer sign cons.",
        "Half-sample sel. prob.",
        "Half-sample sign cons.",
        "Null sel. prob.",
        "Half-sample median beta",
    ],
    body_rows=table_s4b,
    widths=[
        1.15,
        1.80,
        0.85,
        0.95,
        0.95,
        1.10,
        1.10,
        0.90,
        1.15,
    ],
    center_columns={
        2,
        3,
        4,
        5,
        6,
        7,
        8,
    },
    font_size=7.45
)

# ==================================================================================================
# 24. PART C
# ==================================================================================================

add_heading(
    doc,
    "C. Donor-level permutation-null summary"
)

make_table(
    doc,
    headers=[
        "Permutation-null metric",
        "Frozen result",
        "Interpretation",
    ],
    body_rows=table_s4c,
    widths=[
        3.15,
        2.05,
        4.95,
    ],
    center_columns={
        1
    },
    font_size=8.1
)

# ==================================================================================================
# 25. NOTES
# ==================================================================================================

notes_heading = doc.add_paragraph()

format_paragraph(
    notes_heading,
    WD_ALIGN_PARAGRAPH.LEFT,
    before=5,
    after=1
)

run = notes_heading.add_run(
    "Notes:"
)

set_run_font(
    run,
    size_pt=8.3,
    bold=True
)

notes_text = (
    "Selection frequency/probability threshold was 0.80 and sign-consistency threshold was 0.90. "
    "The final 24-feature state was defined as the intersection of the 403 outer-model stable "
    "shared features and the 24 complementary half-sample stable shared features, with coefficient-"
    "direction agreement. This conjunction was a conservative refinement and was not separately "
    "prospectively prelocked before the component stability analyses; this should remain explicit "
    "in manuscript transparency reporting. Permutation results were used only as null validation "
    "and did not add or remove genes from the final state. The observed reference AUROC in the "
    "permutation analysis (0.7698) is from the fixed matched permutation configuration and is not "
    "the same estimand as the repeated nested-CV OOF donor AUROC (0.8020). "
    "No stability, half-sample, permutation, or model fitting was rerun for this table."
)

notes_p = doc.add_paragraph()

format_paragraph(
    notes_p,
    WD_ALIGN_PARAGRAPH.LEFT
)

notes_run = notes_p.add_run(
    notes_text
)

set_run_font(
    notes_run,
    size_pt=8.0
)

# ==================================================================================================
# 26. SAVE DOCX
# ==================================================================================================

doc.save(
    DOCX_PATH
)

if (
    not DOCX_PATH.exists()
    or DOCX_PATH.stat().st_size <= 0
):
    raise RuntimeError(
        "Table S4 DOCX was not created."
    )

print("Microsoft Word DOCX created: PASS")

# ==================================================================================================
# 27. STRUCTURAL READ-BACK
# ==================================================================================================

check_doc = Document(
    DOCX_PATH
)

if len(
    check_doc.tables
) != 3:
    raise RuntimeError(
        f"Expected 3 Word tables; observed {len(check_doc.tables)}"
    )

expected_columns = [
    4,
    9,
    3,
]

for idx, expected_ncol in enumerate(
    expected_columns
):

    observed_ncol = len(
        check_doc.tables[
            idx
        ].columns
    )

    if observed_ncol != expected_ncol:
        raise RuntimeError(
            f"Table S4 part {idx + 1} column-count failure: "
            f"observed={observed_ncol}, expected={expected_ncol}"
        )

if len(
    check_doc.tables[
        1
    ].rows
) != 25:
    raise RuntimeError(
        "Table S4B must contain one header plus 24 final features."
    )

print("DOCX structural read-back: PASS")

# ==================================================================================================
# 28. NOTES TXT
# ==================================================================================================

NOTES_PATH.write_text(
    (
        "Supplementary Table S4\n"
        "========================================\n\n"
        + notes_text
        + "\n\n"
        + "Formatting:\n"
        + "- Microsoft Word DOCX\n"
        + "- Times New Roman\n"
        + "- Black and white only\n"
        + "- No shading or colored fills\n"
        + "- Full horizontal and vertical borders\n"
        + "- Landscape Letter layout\n"
        + "- Repeating table headers\n"
    ),
    encoding="utf-8"
)

# ==================================================================================================
# 29. SUMMARY TXT
# ==================================================================================================

summary_lines = [
    "=" * 110,
    "STEP 11E.4 - SUPPLEMENTARY TABLE S4",
    "=" * 110,
    "",
    "Frozen stability/permutation sources: PASS",
    "Stability analysis rerun: NO",
    "Half-sample fitting rerun: NO",
    "Permutation analysis rerun: NO",
    "",
    "OUTER MODEL STABILITY",
    "Outer models = 25",
    "Genes ever selected shared = 6,131",
    "Stable shared genes = 403",
    "Genes ever selected deviation = 390",
    "Stable deviation genes = 0",
    "",
    "HALF-SAMPLE STABILITY",
    "Half-sample fits = 200",
    "Genes ever selected shared = 13,592",
    "Stable shared genes = 24",
    "Stable deviation genes = 0",
    "Overlap with outer stable set = 24",
    f"Jaccard outer vs half-sample = {HALF_JACCARD:.8f}",
    "",
    "FINAL STATE",
    "Final features = 24",
    "Positive / negative = 9 / 15",
    "Protein-coding / non-protein-coding = 12 / 12",
    "Outer-half direction agreement = 24 / 24",
    f"Minimum half-sample selection probability = {min(half_prob):.3f}",
    f"Minimum half-sample sign consistency = {min(half_sign):.3f}",
    f"Maximum final-feature null selection probability = {max(final_null_prob):.2f}",
    "",
    "PERMUTATION NULL",
    "Permutations = 100",
    f"Observed reference donor AUROC = {null_summary['Observed_reference_donor_AUROC']:.8f}",
    (
        f"Null donor AUROC mean +/- SD = "
        f"{null_summary['Null_donor_AUROC_mean']:.8f} +/- "
        f"{null_summary['Null_donor_AUROC_SD']:.8f}"
    ),
    (
        f"Null donor AUROC range = "
        f"{null_summary['Null_donor_AUROC_min']:.8f} - "
        f"{null_summary['Null_donor_AUROC_max']:.8f}"
    ),
    f"Empirical P = {null_summary['Empirical_P_observed_donor_AUROC']:.8f}",
    "Null stable-like shared genes = 0",
    "Null stable-like deviation genes = 0",
    "",
    f"Final 24 SHA256 = {computed_sha}",
    "",
    "FORMATTING",
    "Times New Roman = YES",
    "Color/shading = NONE",
    "Horizontal borders = YES",
    "Vertical borders = YES",
    "DOCX structural read-back = PASS",
    "",
    f"DOCX = {DOCX_PATH}",
    f"S4A CSV = {S4A_PATH}",
    f"S4B CSV = {S4B_PATH}",
    f"S4C CSV = {S4C_PATH}",
    "=" * 110,
]

SUMMARY_PATH.write_text(
    "\n".join(
        summary_lines
    ),
    encoding="utf-8"
)

# ==================================================================================================
# 30. MANIFEST
# ==================================================================================================

manifest_outputs = [
    DOCX_PATH,
    S4A_PATH,
    S4B_PATH,
    S4C_PATH,
    NOTES_PATH,
    SUMMARY_PATH,
]

with MANIFEST_PATH.open(
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow(
        [
            "role",
            "file",
            "path",
            "bytes",
            "sha256",
        ]
    )

    for path in required_inputs:

        writer.writerow(
            [
                "input",
                path.name,
                str(
                    path
                ),
                path.stat().st_size,
                sha256_file(
                    path
                ),
            ]
        )

    for path in manifest_outputs:

        writer.writerow(
            [
                "output",
                path.name,
                str(
                    path
                ),
                path.stat().st_size,
                sha256_file(
                    path
                ),
            ]
        )

# ==================================================================================================
# 31. FINAL OUTPUT CHECK
# ==================================================================================================

final_outputs = [
    DOCX_PATH,
    S4A_PATH,
    S4B_PATH,
    S4C_PATH,
    NOTES_PATH,
    SUMMARY_PATH,
    MANIFEST_PATH,
]

for path in final_outputs:

    if (
        not path.exists()
        or path.stat().st_size <= 0
    ):
        raise RuntimeError(
            f"Missing or zero-byte Table S4 output: {path}"
        )

print("All Supplementary Table S4 outputs created and non-zero: PASS")

# ==================================================================================================
# 32. FINAL REPORT
# ==================================================================================================

print()
print("=" * 110)
print("SUPPLEMENTARY TABLE S4 GENERATION SUMMARY")
print("=" * 110)

print("Frozen stability/permutation sources: PASS")
print("Stability/permutation rerun: NO")
print("Canonical final 24 SHA256: PASS")

print()

print(
    "Outer stability: shared ever / stable shared / stable deviation = "
    "6131 / 403 / 0"
)

print(
    "Half-sample stability: fits / shared ever / stable shared / stable deviation = "
    "200 / 13592 / 24 / 0"
)

print(
    "Final intersection / direction agreement = "
    "24 / 24"
)

print(
    "Positive / negative final directions = "
    "9 / 15"
)

print(
    "Protein-coding / non-protein-coding = "
    "12 / 12"
)

print()

print(
    "Minimum final half-sample selection probability: "
    f"{min(half_prob):.3f}"
)

print(
    "Minimum final half-sample sign consistency: "
    f"{min(half_sign):.3f}"
)

print(
    "Maximum final null selection probability: "
    f"{max(final_null_prob):.2f}"
)

print()

print(
    "Permutation observed donor AUROC: "
    f"{null_summary['Observed_reference_donor_AUROC']:.7f}"
)

print(
    "Null donor AUROC mean +/- SD: "
    f"{null_summary['Null_donor_AUROC_mean']:.7f} +/- "
    f"{null_summary['Null_donor_AUROC_SD']:.7f}"
)

print(
    "Null donor AUROC maximum: "
    f"{null_summary['Null_donor_AUROC_max']:.7f}"
)

print(
    "Empirical P: "
    f"{null_summary['Empirical_P_observed_donor_AUROC']:.8f}"
)

print()

print("Times New Roman: PASS")
print("Black-and-white only: PASS")
print("Full row borders: PASS")
print("Full column borders: PASS")
print("DOCX structural read-back: PASS")

print()

print("DOCX:")
print(DOCX_PATH)

print()

print("Source CSVs:")
print(S4A_PATH)
print(S4B_PATH)
print(S4C_PATH)

print()

print("Manifest:")
print(MANIFEST_PATH)

print()
print("=" * 110)
print("STEP 11E.4 COMPLETE - SUPPLEMENTARY TABLE S4 SAVED PROJECT-LOCALLY")
print("=" * 110)
)"

# --------------------------------------------------------------------------------------------------
# 5. SAVE PYTHON SCRIPT
# --------------------------------------------------------------------------------------------------

writeLines(
    py_code,
    PY_SCRIPT_PATH,
    useBytes = TRUE
)

if (
    !file.exists(PY_SCRIPT_PATH) ||
    file.info(PY_SCRIPT_PATH)$size <= 0
) {
    stop(
        "Table S4 Python generator was not created."
    )
}

cat(
    "Python generator saved: ",
    PY_SCRIPT_PATH,
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 6. RUN PYTHON
# --------------------------------------------------------------------------------------------------

cmd_output <- system2(
    command = PYTHON_EXE,
    args = shQuote(
        PY_SCRIPT_PATH
    ),
    stdout = TRUE,
    stderr = TRUE
)

exit_status <- attr(
    cmd_output,
    "status"
)

cat(
    paste(
        cmd_output,
        collapse = "\n"
    ),
    "\n"
)

if (
    !is.null(exit_status) &&
    exit_status != 0
) {
    stop(
        "Python Table S4 generator failed with exit status ",
        exit_status,
        ". See console output above."
    )
}

# --------------------------------------------------------------------------------------------------
# 7. R-SIDE OUTPUT CHECK
# --------------------------------------------------------------------------------------------------

EXPECTED_DOCX <- file.path(
    OUT_DIR,
    "TableS4_Stability_Refinement_Permutation_Evidence_v1.docx"
)

EXPECTED_S4A <- file.path(
    OUT_DIR,
    "TableS4A_stability_refinement_summary_source_v1.csv"
)

EXPECTED_S4B <- file.path(
    OUT_DIR,
    "TableS4B_final24_stability_evidence_source_v1.csv"
)

EXPECTED_S4C <- file.path(
    OUT_DIR,
    "TableS4C_permutation_null_summary_source_v1.csv"
)

EXPECTED_NOTES <- file.path(
    OUT_DIR,
    "TableS4_notes_v1.txt"
)

EXPECTED_SUMMARY <- file.path(
    OUT_DIR,
    "TableS4_generation_summary_v1.txt"
)

EXPECTED_MANIFEST <- file.path(
    OUT_DIR,
    "TableS4_generation_manifest_v1.csv"
)

expected_outputs <- c(
    EXPECTED_DOCX,
    EXPECTED_S4A,
    EXPECTED_S4B,
    EXPECTED_S4C,
    EXPECTED_NOTES,
    EXPECTED_SUMMARY,
    EXPECTED_MANIFEST,
    PY_SCRIPT_PATH
)

if (!all(
    file.exists(
        expected_outputs
    )
)) {
    stop(
        "One or more expected Table S4 artifacts are missing."
    )
}

if (any(
    file.info(
        expected_outputs
    )$size <= 0
)) {
    stop(
        "One or more expected Table S4 artifacts are zero-byte."
    )
}

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.4R COMPLETE - TABLE S4 GENERATED THROUGH PYTHON FROM R KERNEL\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "\nMicrosoft Word DOCX:\n",
    EXPECTED_DOCX,
    "\n",
    sep = ""
)

cat(
    "\nSaved Python generator:\n",
    PY_SCRIPT_PATH,
    "\n",
    sep = ""
)

cat(
    "\nSource CSVs:\n",
    EXPECTED_S4A,
    "\n",
    EXPECTED_S4B,
    "\n",
    EXPECTED_S4C,
    "\n",
    sep = ""
)

cat(
    "\nManifest:\n",
    EXPECTED_MANIFEST,
    "\n",
    sep = ""
)


STEP 11E.4R - SUPPLEMENTARY TABLE S4
STABILITY REFINEMENT, FINAL-STATE EVIDENCE, AND PERMUTATION NULL
Project sentinel: PASS
Python executable: PASS
Python generator saved: E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/tables/supplementary/TableS4/generate_TableS4_v1.py
STEP 11E.4 - SUPPLEMENTARY TABLE S4
STABILITY REFINEMENT, FINAL-STATE EVIDENCE, AND PERMUTATION NULL
Project sentinel: PASS
Frozen stability/permutation artifacts: PASS
Metric/value summary schemas: PASS
Outer-model stability anchors: PASS
Complementary half-sample stability anchors: PASS
Permutation-null anchors: PASS
Final 24 set identity across outer/half/null sources: PASS
Final 24 gene-level stability evidence: PASS
Final stable-state summary anchors: PASS
Canonical final 24 SHA256: PASS
Microsoft Word DOCX created: PASS
DOCX structural read-back: PASS
All Supplementary Table S4 outputs created and non-zero: PASS

SUPPLEMENTARY TABLE S4 GENERATION SUMMARY
Frozen stability/permutation sources: PASS
Stability/permutation 

In [28]:
# ==================================================================================================
# STEP 11E.5T - SUPPLEMENTARY TABLE S5 - CANONICAL FORMULA VALIDATION FIX
# R KERNEL -> CORRECTED PYTHON GENERATOR v3
#
# PURPOSE:
#   - Reuse the already generated Table S5 v2 Python generator.
#   - Remove ONLY the redundant/brittle literal formula-fragment check.
#   - Retain the canonical formula SHA256 gate as the authoritative integrity check.
#   - Save a new independent v3 Python generator.
#   - Generate v3 DOCX + source CSVs + notes + summary + manifest.
#
# ANALYSIS CHANGES: NONE
# MODEL RERUN: NO
# FEATURE SELECTION RERUN: NO
# WEIGHT ESTIMATION RERUN: NO
# SCORE FORMULA CHANGE: NO
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.5T - SUPPLEMENTARY TABLE S5 - CANONICAL FORMULA VALIDATION FIX\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 1. PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

observed_project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    observed_project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "Project ID mismatch. Expected ",
        EXPECTED_PROJECT_ID,
        " but observed ",
        observed_project_id
    )
}

cat("Project sentinel: PASS\n")

# --------------------------------------------------------------------------------------------------
# 2. PYTHON
# --------------------------------------------------------------------------------------------------

PYTHON_EXE <- "E:/Python/python.exe"

if (!file.exists(PYTHON_EXE)) {
    stop(
        "Python executable not found: ",
        PYTHON_EXE
    )
}

cat("Python executable: PASS\n")

# --------------------------------------------------------------------------------------------------
# 3. TABLE S5 DIRECTORY
# --------------------------------------------------------------------------------------------------

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "supplementary",
    "TableS5"
)

if (!dir.exists(OUT_DIR)) {
    stop(
        "Table S5 directory missing: ",
        OUT_DIR
    )
}

SOURCE_SCRIPT <- file.path(
    OUT_DIR,
    "generate_TableS5_v2.py"
)

TARGET_SCRIPT <- file.path(
    OUT_DIR,
    "generate_TableS5_v3.py"
)

if (!file.exists(SOURCE_SCRIPT)) {
    stop(
        "Previous complete v2 Python generator missing: ",
        SOURCE_SCRIPT
    )
}

if (file.info(SOURCE_SCRIPT)$size <= 0) {
    stop(
        "Previous v2 Python generator is zero-byte."
    )
}

cat("Previously generated complete v2 Python source: PASS\n")

# --------------------------------------------------------------------------------------------------
# 4. READ COMPLETE v2 GENERATOR
# --------------------------------------------------------------------------------------------------

script_lines <- readLines(
    SOURCE_SCRIPT,
    warn = FALSE,
    encoding = "UTF-8"
)

if (length(script_lines) < 100L) {
    stop(
        "Unexpectedly short v2 Python generator."
    )
}

cat(
    "Python source lines loaded: ",
    length(script_lines),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 5. LOCATE EXACT REDUNDANT FORMULA-FRAGMENT VALIDATION BLOCK
# --------------------------------------------------------------------------------------------------

start_hits <- which(
    trimws(script_lines) ==
        "formula_required_fragments = ["
)

if (length(start_hits) != 1L) {
    stop(
        "Could not uniquely locate formula_required_fragments block. Hits = ",
        length(start_hits)
    )
}

START <- start_hits[[1]]

end_candidates <- which(
    seq_along(script_lines) >= START &
    grepl(
        'print("Canonical frozen score formula SHA256/content: PASS")',
        script_lines,
        fixed = TRUE
    )
)

if (length(end_candidates) != 1L) {
    stop(
        "Could not uniquely locate the end of the redundant formula-content validation block. Hits = ",
        length(end_candidates)
    )
}

END <- end_candidates[[1]]

if (END <= START) {
    stop(
        "Invalid formula-validation block boundaries."
    )
}

cat(
    "Redundant literal formula-content block located: lines ",
    START,
    "-",
    END,
    " - PASS\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 6. REPLACE ONLY THAT BLOCK
#
# Canonical integrity is already enforced immediately above this block by:
#
#   computed_formula_sha ==
#   615b3934804d22c1264b2ef80784c01a13b4212a6bff778475970727493b45d5
#
# Therefore another markdown-format-dependent literal substring test is neither needed nor safer.
# --------------------------------------------------------------------------------------------------

replacement_lines <- c(
    "# Canonical formula integrity is governed by the frozen SHA256 above.",
    "# No additional literal Markdown substring test is applied because",
    "# line breaks / Markdown presentation are not part of the mathematical validation logic.",
    'print("Canonical frozen score formula SHA256: PASS")'
)

before_block <- if (START > 1L) {
    script_lines[
        seq_len(
            START - 1L
        )
    ]
} else {
    character(0)
}

after_block <- if (END < length(script_lines)) {
    script_lines[
        seq.int(
            END + 1L,
            length(script_lines)
        )
    ]
} else {
    character(0)
}

script_v3 <- c(
    before_block,
    replacement_lines,
    after_block
)

# --------------------------------------------------------------------------------------------------
# 7. VERSION OUTPUT ARTIFACTS FROM v2 -> v3
# --------------------------------------------------------------------------------------------------

script_v3 <- gsub(
    "_v2",
    "_v3",
    script_v3,
    fixed = TRUE
)

script_v3 <- gsub(
    "S5 v2",
    "S5 v3",
    script_v3,
    fixed = TRUE
)

# --------------------------------------------------------------------------------------------------
# 8. CORRECTION INTEGRITY CHECKS
# --------------------------------------------------------------------------------------------------

if (any(
    trimws(script_v3) ==
        "formula_required_fragments = ["
)) {
    stop(
        "Redundant formula-fragment validation block still present."
    )
}

if (!any(
    grepl(
        'Canonical frozen score formula SHA256: PASS',
        script_v3,
        fixed = TRUE
    )
)) {
    stop(
        "Corrected canonical formula SHA message missing."
    )
}

EXPECTED_FORMULA_SHA <- paste0(
    "615b3934804d22c1264b2ef80784c01",
    "a13b4212a6bff778475970727493b45d5"
)

if (!any(
    grepl(
        EXPECTED_FORMULA_SHA,
        script_v3,
        fixed = TRUE
    )
)) {
    stop(
        "Canonical formula SHA256 gate missing from corrected script."
    )
}

EXPECTED_WEIGHT_SHA <- paste0(
    "53e394ccd41b272b9ab31f4ab85728c7",
    "b970ca0383c70ec0537228eec795aef3"
)

if (!any(
    grepl(
        EXPECTED_WEIGHT_SHA,
        script_v3,
        fixed = TRUE
    )
)) {
    stop(
        "Canonical weight SHA256 gate missing from corrected script."
    )
}

cat("Canonical formula SHA256 gate retained: PASS\n")
cat("Canonical weight SHA256 gate retained: PASS\n")
cat("Brittle literal formula-content gate removed: PASS\n")

# --------------------------------------------------------------------------------------------------
# 9. SAVE COMPLETE CORRECTED v3 PYTHON GENERATOR
# --------------------------------------------------------------------------------------------------

writeLines(
    script_v3,
    TARGET_SCRIPT,
    useBytes = TRUE
)

if (
    !file.exists(TARGET_SCRIPT) ||
    file.info(TARGET_SCRIPT)$size <= 0
) {
    stop(
        "Corrected v3 Python generator was not created."
    )
}

cat(
    "Corrected complete Python generator saved:\n",
    TARGET_SCRIPT,
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 10. PYTHON SYNTAX PREFLIGHT
# --------------------------------------------------------------------------------------------------

compile_output <- system2(
    command = PYTHON_EXE,
    args = c(
        "-m",
        "py_compile",
        shQuote(
            TARGET_SCRIPT
        )
    ),
    stdout = TRUE,
    stderr = TRUE
)

compile_status <- attr(
    compile_output,
    "status"
)

if (
    !is.null(compile_status) &&
    compile_status != 0
) {

    cat(
        paste(
            compile_output,
            collapse = "\n"
        ),
        "\n"
    )

    stop(
        "Corrected v3 Python generator failed Python syntax compilation."
    )
}

cat("Python syntax preflight: PASS\n")

# --------------------------------------------------------------------------------------------------
# 11. EXECUTE COMPLETE v3 GENERATOR
# --------------------------------------------------------------------------------------------------

cmd_output <- system2(
    command = PYTHON_EXE,
    args = shQuote(
        TARGET_SCRIPT
    ),
    stdout = TRUE,
    stderr = TRUE
)

exit_status <- attr(
    cmd_output,
    "status"
)

cat(
    paste(
        cmd_output,
        collapse = "\n"
    ),
    "\n"
)

if (
    !is.null(exit_status) &&
    exit_status != 0
) {
    stop(
        "Python Table S5 v3 generator failed with exit status ",
        exit_status,
        ". See complete console output above."
    )
}

# --------------------------------------------------------------------------------------------------
# 12. EXPECTED v3 OUTPUTS
# --------------------------------------------------------------------------------------------------

EXPECTED_DOCX <- file.path(
    OUT_DIR,
    "TableS5_SCZ_MSS_v1_Frozen_Feature_Weights_and_Score_Definition_v3.docx"
)

EXPECTED_S5A <- file.path(
    OUT_DIR,
    "TableS5A_score_definition_source_v3.csv"
)

EXPECTED_S5B <- file.path(
    OUT_DIR,
    "TableS5B_24feature_weights_source_v3.csv"
)

EXPECTED_NOTES <- file.path(
    OUT_DIR,
    "TableS5_notes_v3.txt"
)

EXPECTED_SUMMARY <- file.path(
    OUT_DIR,
    "TableS5_generation_summary_v3.txt"
)

EXPECTED_MANIFEST <- file.path(
    OUT_DIR,
    "TableS5_generation_manifest_v3.csv"
)

expected_outputs <- c(
    EXPECTED_DOCX,
    EXPECTED_S5A,
    EXPECTED_S5B,
    EXPECTED_NOTES,
    EXPECTED_SUMMARY,
    EXPECTED_MANIFEST,
    TARGET_SCRIPT
)

missing_outputs <- expected_outputs[
    !file.exists(
        expected_outputs
    )
]

if (length(missing_outputs) > 0L) {
    stop(
        "Expected Table S5 v3 artifact(s) missing:\n",
        paste(
            missing_outputs,
            collapse = "\n"
        )
    )
}

output_sizes <- file.info(
    expected_outputs
)$size

if (any(
    is.na(output_sizes) |
    output_sizes <= 0
)) {
    stop(
        "One or more Table S5 v3 artifacts are zero-byte."
    )
}

cat("All Table S5 v3 outputs created and non-zero: PASS\n")

# --------------------------------------------------------------------------------------------------
# 13. FINAL REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nTABLE S5 v3 CORRECTION SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat("Source v2 generator retained unchanged: YES\n")
cat("Complete v3 generator created: YES\n")
cat("Canonical weight SHA256 validation retained: YES\n")
cat("Canonical formula SHA256 validation retained: YES\n")
cat("Fragile literal 'sum(abs' content test removed: YES\n")
cat("Score formula changed: NO\n")
cat("Feature weights changed: NO\n")
cat("Feature identities changed: NO\n")
cat("Model/CV rerun: NO\n")
cat("Stability rerun: NO\n")
cat("Python syntax preflight: PASS\n")
cat("DOCX/output generation: PASS\n")

cat(
    "\nMicrosoft Word DOCX:\n",
    EXPECTED_DOCX,
    "\n",
    sep = ""
)

cat(
    "\nCorrected reproducible Python generator:\n",
    TARGET_SCRIPT,
    "\n",
    sep = ""
)

cat(
    "\nSource CSVs:\n",
    EXPECTED_S5A,
    "\n",
    EXPECTED_S5B,
    "\n",
    sep = ""
)

cat(
    "\nManifest:\n",
    EXPECTED_MANIFEST,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.5T COMPLETE - SUPPLEMENTARY TABLE S5 v3 GENERATED\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11E.5T - SUPPLEMENTARY TABLE S5 - CANONICAL FORMULA VALIDATION FIX
Project sentinel: PASS
Python executable: PASS
Previously generated complete v2 Python source: PASS
Python source lines loaded: 2385
Redundant literal formula-content block located: lines 697-713 - PASS
Canonical formula SHA256 gate retained: PASS
Canonical weight SHA256 gate retained: PASS
Brittle literal formula-content gate removed: PASS
Corrected complete Python generator saved:
E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/tables/supplementary/TableS5/generate_TableS5_v3.py
Python syntax preflight: PASS
STEP 11E.5S - SUPPLEMENTARY TABLE S5 - FULLY CORRECTED
FROZEN 24-FEATURE IDENTITIES, WEIGHTS, AND SCZ-MSS v1 SCORE DEFINITION
Project sentinel: PASS
Frozen score/identity artifacts: PASS
Frozen CSV inputs loaded: PASS
Frozen 24-feature weight schema: PASS
Frozen Step07A summary anchors: PASS
Frozen raw-to-normalized weight reconstruction: PASS
Absolute-weight rank 1-24: PASS
Canonical frozen weight SHA256: PASS
Cano

In [29]:
# ==================================================================================================
# STEP 11E.6R - SUPPLEMENTARY TABLE S6
# FROZEN EXTERNAL TECHNICAL-GATE AUDIT ACROSS FIVE PUBLIC COHORTS
#
# R KERNEL -> SAVED PYTHON-DOCX GENERATOR
#
# PURPOSE
#   1. Read ONLY the frozen STEP09J.1 external technical-gate master registry.
#   2. Verify all five completed public cohorts and all frozen gate anchors.
#   3. Create a detailed, non-redundant Supplementary Table S6.
#   4. Preserve technical FAIL as technically non-evaluable, not biological failure.
#   5. Preserve outcome-access / blinding transparency.
#   6. Save DOCX + source CSVs + notes + summary + manifest + reproducible Python generator.
#
# IMPORTANT
#   - NO external analysis rerun.
#   - NO score recalculation.
#   - NO phenotype analysis.
#   - NO mapping rescue.
#   - NO threshold change.
#   - NO outcome-dependent modification.
#   - Black-and-white Word manuscript table.
#   - Times New Roman.
#   - Full horizontal + vertical borders.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.6R - SUPPLEMENTARY TABLE S6\n",
    "FROZEN EXTERNAL TECHNICAL-GATE AUDIT ACROSS FIVE PUBLIC COHORTS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 1. PROJECT VERIFICATION
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

observed_project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    observed_project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "Project ID mismatch. Expected ",
        EXPECTED_PROJECT_ID,
        " but observed ",
        observed_project_id
    )
}

cat("Project sentinel: PASS\n")

# --------------------------------------------------------------------------------------------------
# 2. PYTHON
# --------------------------------------------------------------------------------------------------

PYTHON_EXE <- "E:/Python/python.exe"

if (!file.exists(PYTHON_EXE)) {
    stop(
        "Python executable not found: ",
        PYTHON_EXE
    )
}

cat("Python executable: PASS\n")

# --------------------------------------------------------------------------------------------------
# 3. OUTPUT DIRECTORY
# --------------------------------------------------------------------------------------------------

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "supplementary",
    "TableS6"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

if (!dir.exists(OUT_DIR)) {
    stop(
        "Could not create Table S6 output directory."
    )
}

PY_SCRIPT <- file.path(
    OUT_DIR,
    "generate_TableS6_v1.py"
)

# --------------------------------------------------------------------------------------------------
# 4. COMPLETE PYTHON GENERATOR
#
# Raw R string is intentional:
# Python code remains fully quoted inside the R kernel and is written to disk before execution.
# --------------------------------------------------------------------------------------------------

PY_SOURCE <- r"---(
from pathlib import Path
import csv
import hashlib
import math
import sys

from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT
from docx.shared import Inches, Pt
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


# ==================================================================================================
# STEP 11E.6 - SUPPLEMENTARY TABLE S6
# FROZEN EXTERNAL TECHNICAL-GATE AUDIT ACROSS FIVE PUBLIC COHORTS
# ==================================================================================================

print("=" * 110)
print("STEP 11E.6 - SUPPLEMENTARY TABLE S6")
print("FROZEN EXTERNAL TECHNICAL-GATE AUDIT ACROSS FIVE PUBLIC COHORTS")
print("=" * 110)


# --------------------------------------------------------------------------------------------------
# 1. PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path("E:/SCZ_CROSSREGION_MSS_Q1")
EXPECTED_PROJECT_ID = "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL = PROJECT_ROOT / ".project_id"

REGISTRY_PATH = (
    PROJECT_ROOT
    / "09_external_validation"
    / "master_registry_v1"
    / "SCZ_MSS_v1_external_technical_gate_registry_v1.csv"
)

OUT_DIR = (
    PROJECT_ROOT
    / "11_manuscript"
    / "tables"
    / "supplementary"
    / "TableS6"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PY_SCRIPT_PATH = OUT_DIR / "generate_TableS6_v1.py"

DOCX_OUT = (
    OUT_DIR
    / "TableS6_External_Technical_Gate_Audit_v1.docx"
)

S6A_OUT = (
    OUT_DIR
    / "TableS6A_frozen_gate_definition_source_v1.csv"
)

S6B_OUT = (
    OUT_DIR
    / "TableS6B_external_cohort_gate_audit_source_v1.csv"
)

S6C_OUT = (
    OUT_DIR
    / "TableS6C_mapping_provenance_source_v1.csv"
)

NOTES_OUT = (
    OUT_DIR
    / "TableS6_notes_v1.txt"
)

SUMMARY_OUT = (
    OUT_DIR
    / "TableS6_generation_summary_v1.txt"
)

MANIFEST_OUT = (
    OUT_DIR
    / "TableS6_generation_manifest_v1.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            block = handle.read(1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def parse_float(value):
    if value is None:
        return None

    text = str(value).strip()

    if text == "" or text.upper() in {
        "NA",
        "NAN",
        "NONE",
        "<NA>"
    }:
        return None

    return float(text)


def parse_int(value):
    x = parse_float(value)

    if x is None:
        return None

    return int(round(x))


def parse_bool(value):
    if isinstance(value, bool):
        return value

    text = str(value).strip().upper()

    if text in {
        "TRUE",
        "T",
        "1",
        "YES"
    }:
        return True

    if text in {
        "FALSE",
        "F",
        "0",
        "NO"
    }:
        return False

    raise RuntimeError(
        f"Cannot parse logical value: {value!r}"
    )


def approx_equal(a, b, tol=1e-6):
    if a is None or b is None:
        return False

    return abs(float(a) - float(b)) <= tol


def pct(x):
    if x is None:
        return "N/A"

    return f"{100.0 * x:.1f}%"


def yes_no(x):
    return "Yes" if x else "No"


def pass_fail(x):
    return "PASS" if x else "FAIL"


def display_sample_completeness(x):
    if x is None:
        return "Not required after earlier gate failure"

    return pct(x)


def write_csv(path, fieldnames, rows):
    with open(
        path,
        "w",
        newline="",
        encoding="utf-8"
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
            extrasaction="ignore"
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(row)


def set_run_font(run, size_pt=9.0, bold=False, italic=False):
    run.font.name = "Times New Roman"
    run.font.size = Pt(size_pt)
    run.font.bold = bold
    run.font.italic = italic

    rpr = run._element.get_or_add_rPr()
    rfonts = rpr.find(qn("w:rFonts"))

    if rfonts is None:
        rfonts = OxmlElement("w:rFonts")
        rpr.insert(0, rfonts)

    rfonts.set(
        qn("w:ascii"),
        "Times New Roman"
    )

    rfonts.set(
        qn("w:hAnsi"),
        "Times New Roman"
    )

    rfonts.set(
        qn("w:eastAsia"),
        "Times New Roman"
    )

    rfonts.set(
        qn("w:cs"),
        "Times New Roman"
    )


def set_paragraph_format(
    paragraph,
    align=None,
    space_before=0,
    space_after=0,
    line_spacing=1.0
):
    if align is not None:
        paragraph.alignment = align

    paragraph.paragraph_format.space_before = Pt(space_before)
    paragraph.paragraph_format.space_after = Pt(space_after)
    paragraph.paragraph_format.line_spacing = line_spacing


def set_cell_margins(
    cell,
    top=65,
    start=75,
    bottom=65,
    end=75
):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()

    tcMar = tcPr.first_child_found_in(
        "w:tcMar"
    )

    if tcMar is None:
        tcMar = OxmlElement("w:tcMar")
        tcPr.append(tcMar)

    for margin_name, margin_value in (
        ("top", top),
        ("start", start),
        ("bottom", bottom),
        ("end", end)
    ):
        node = tcMar.find(
            qn(f"w:{margin_name}")
        )

        if node is None:
            node = OxmlElement(
                f"w:{margin_name}"
            )
            tcMar.append(node)

        node.set(
            qn("w:w"),
            str(margin_value)
        )

        node.set(
            qn("w:type"),
            "dxa"
        )


def set_cell_width(cell, width_inches):
    cell.width = Inches(width_inches)

    tcPr = cell._tc.get_or_add_tcPr()

    tcW = tcPr.find(
        qn("w:tcW")
    )

    if tcW is None:
        tcW = OxmlElement("w:tcW")
        tcPr.append(tcW)

    tcW.set(
        qn("w:w"),
        str(int(width_inches * 1440))
    )

    tcW.set(
        qn("w:type"),
        "dxa"
    )


def set_cell_borders(cell, size=4):
    tcPr = cell._tc.get_or_add_tcPr()

    borders = tcPr.find(
        qn("w:tcBorders")
    )

    if borders is None:
        borders = OxmlElement(
            "w:tcBorders"
        )
        tcPr.append(borders)

    for edge in (
        "top",
        "left",
        "bottom",
        "right",
        "insideH",
        "insideV"
    ):
        tag = qn(f"w:{edge}")
        edge_node = borders.find(tag)

        if edge_node is None:
            edge_node = OxmlElement(
                f"w:{edge}"
            )
            borders.append(edge_node)

        edge_node.set(
            qn("w:val"),
            "single"
        )

        edge_node.set(
            qn("w:sz"),
            str(size)
        )

        edge_node.set(
            qn("w:space"),
            "0"
        )

        edge_node.set(
            qn("w:color"),
            "000000"
        )


def remove_cell_shading(cell):
    tcPr = cell._tc.get_or_add_tcPr()

    for shd in list(
        tcPr.findall(qn("w:shd"))
    ):
        tcPr.remove(shd)


def repeat_table_header(row):
    trPr = row._tr.get_or_add_trPr()

    tblHeader = OxmlElement(
        "w:tblHeader"
    )

    tblHeader.set(
        qn("w:val"),
        "true"
    )

    trPr.append(tblHeader)


def prevent_row_split(row):
    trPr = row._tr.get_or_add_trPr()

    cantSplit = OxmlElement(
        "w:cantSplit"
    )

    trPr.append(cantSplit)


def format_cell(
    cell,
    text,
    width,
    align=WD_ALIGN_PARAGRAPH.LEFT,
    bold=False,
    size_pt=8.6
):
    cell.text = str(text)
    cell.vertical_alignment = (
        WD_CELL_VERTICAL_ALIGNMENT.CENTER
    )

    set_cell_width(
        cell,
        width
    )

    set_cell_margins(cell)
    set_cell_borders(cell)
    remove_cell_shading(cell)

    for p in cell.paragraphs:
        set_paragraph_format(
            p,
            align=align,
            space_before=0,
            space_after=0,
            line_spacing=1.0
        )

        for run in p.runs:
            set_run_font(
                run,
                size_pt=size_pt,
                bold=bold
            )


def add_plain_table(
    document,
    headers,
    rows,
    widths,
    center_columns=None,
    body_font_size=8.6
):
    if center_columns is None:
        center_columns = set()

    table = document.add_table(
        rows=1,
        cols=len(headers)
    )

    table.autofit = False

    try:
        table.style = "Table Grid"
    except Exception:
        pass

    header_row = table.rows[0]

    for j, header in enumerate(headers):
        format_cell(
            header_row.cells[j],
            header,
            widths[j],
            align=WD_ALIGN_PARAGRAPH.CENTER,
            bold=True,
            size_pt=8.8
        )

    repeat_table_header(
        header_row
    )

    prevent_row_split(
        header_row
    )

    for data_row in rows:
        row = table.add_row()

        prevent_row_split(row)

        for j, value in enumerate(data_row):
            align = (
                WD_ALIGN_PARAGRAPH.CENTER
                if j in center_columns
                else WD_ALIGN_PARAGRAPH.LEFT
            )

            format_cell(
                row.cells[j],
                value,
                widths[j],
                align=align,
                bold=False,
                size_pt=body_font_size
            )

    return table


def add_section_label(document, text):
    p = document.add_paragraph()

    set_paragraph_format(
        p,
        align=WD_ALIGN_PARAGRAPH.LEFT,
        space_before=6,
        space_after=3,
        line_spacing=1.0
    )

    run = p.add_run(text)

    set_run_font(
        run,
        size_pt=9.5,
        bold=True
    )

    return p


def add_note_paragraph(document, text):
    p = document.add_paragraph()

    set_paragraph_format(
        p,
        align=WD_ALIGN_PARAGRAPH.LEFT,
        space_before=4,
        space_after=0,
        line_spacing=1.05
    )

    run = p.add_run(text)

    set_run_font(
        run,
        size_pt=8.5,
        bold=False
    )

    return p


# --------------------------------------------------------------------------------------------------
# 3. PROJECT / INPUT VERIFICATION
# --------------------------------------------------------------------------------------------------

if not SENTINEL.exists():
    raise RuntimeError(
        f"Project sentinel missing: {SENTINEL}"
    )

observed_project_id = (
    SENTINEL
    .read_text(
        encoding="utf-8"
    )
    .splitlines()[0]
    .strip()
)

if observed_project_id != EXPECTED_PROJECT_ID:
    raise RuntimeError(
        "Project sentinel mismatch. "
        f"Expected {EXPECTED_PROJECT_ID}; "
        f"observed {observed_project_id}"
    )

print("Project sentinel: PASS")

if not REGISTRY_PATH.exists():
    raise RuntimeError(
        f"Frozen master registry missing: {REGISTRY_PATH}"
    )

if REGISTRY_PATH.stat().st_size <= 0:
    raise RuntimeError(
        "Frozen master registry is zero-byte."
    )

print("Frozen STEP09J.1 external master registry: PASS")


# --------------------------------------------------------------------------------------------------
# 4. LOAD MASTER REGISTRY
# --------------------------------------------------------------------------------------------------

with open(
    REGISTRY_PATH,
    "r",
    newline="",
    encoding="utf-8-sig"
) as handle:

    reader = csv.DictReader(handle)

    registry = list(reader)

if len(registry) != 5:
    raise RuntimeError(
        "Expected exactly 5 completed public external cohorts; "
        f"found {len(registry)}."
    )

required_columns = {
    "dataset",
    "public_external_sequence",
    "tissue_region",
    "assay_platform_summary",
    "n_samples",
    "control_n_known",
    "schizophrenia_n_known",
    "frozen_gene_count",
    "mapped_gene_count",
    "gene_count_coverage",
    "absolute_weight_coverage",
    "positive_weight_coverage",
    "negative_weight_coverage",
    "sample_completeness",
    "mapping_basis",
    "mapping_unambiguous_for_gate",
    "gene_count_pass",
    "absolute_weight_pass",
    "positive_weight_pass",
    "negative_weight_pass",
    "technical_pass",
    "outcome_evaluated",
    "measurability_note",
    "outcome_blinding_note",
    "final_status"
}

missing_columns = sorted(
    required_columns
    - set(registry[0].keys())
)

if missing_columns:
    raise RuntimeError(
        "Master external registry missing required column(s): "
        + ", ".join(missing_columns)
    )

registry = sorted(
    registry,
    key=lambda x: int(
        float(
            x["public_external_sequence"]
        )
    )
)

observed_datasets = [
    row["dataset"]
    for row in registry
]

expected_datasets = [
    "GSE87194",
    "GSE80655",
    "GSE224683",
    "GSE282552",
    "GSE208338"
]

if observed_datasets != expected_datasets:
    raise RuntimeError(
        "Completed public-cohort order mismatch. "
        f"Observed: {observed_datasets}"
    )

print(
    "Completed public cohort set = "
    "GSE87194 / GSE80655 / GSE224683 / "
    "GSE282552 / GSE208338: PASS"
)


# --------------------------------------------------------------------------------------------------
# 5. FROZEN TECHNICAL-GATE THRESHOLDS
# --------------------------------------------------------------------------------------------------

GENE_THRESHOLD = 0.80
ABS_THRESHOLD = 0.80
POS_THRESHOLD = 0.70
NEG_THRESHOLD = 0.70
SAMPLE_COMPLETENESS_REQUIRED = 1.00

print("Frozen technical-gate thresholds loaded: PASS")


# --------------------------------------------------------------------------------------------------
# 6. CANONICAL NUMERIC ANCHORS
# --------------------------------------------------------------------------------------------------

expected = {
    "GSE87194": {
        "n": 38,
        "mapped": 14,
        "gene": 0.5833330,
        "abs": 0.3332160,
        "pos": 0.2611400,
        "neg": 0.3787040,
        "technical": False,
        "outcome": False
    },
    "GSE80655": {
        "n": 281,
        "mapped": 20,
        "gene": 0.8333330,
        "abs": 0.7197100,
        "pos": 1.0000000,
        "neg": 0.5428170,
        "technical": False,
        "outcome": False
    },
    "GSE224683": {
        "n": 30,
        "mapped": 23,
        "gene": 0.9583330,
        "abs": 0.9266000,
        "pos": 1.0000000,
        "neg": 0.8802760,
        "technical": True,
        "outcome": True
    },
    "GSE282552": {
        "n": 18,
        "mapped": 9,
        "gene": 0.3750000,
        "abs": 0.1961925,
        "pos": 0.0304038,
        "neg": 0.3008226,
        "technical": False,
        "outcome": False
    },
    "GSE208338": {
        "n": 169,
        "mapped": 14,
        "gene": 0.5833333,
        "abs": 0.3818154,
        "pos": 0.2611397,
        "neg": 0.4579744,
        "technical": False,
        "outcome": False
    }
}

for row in registry:
    ds = row["dataset"]
    exp = expected[ds]

    if parse_int(row["n_samples"]) != exp["n"]:
        raise RuntimeError(
            f"{ds}: sample-size anchor mismatch."
        )

    if parse_int(row["frozen_gene_count"]) != 24:
        raise RuntimeError(
            f"{ds}: frozen feature count is not 24."
        )

    if parse_int(row["mapped_gene_count"]) != exp["mapped"]:
        raise RuntimeError(
            f"{ds}: mapped-gene count anchor mismatch."
        )

    numeric_checks = [
        (
            "gene_count_coverage",
            exp["gene"]
        ),
        (
            "absolute_weight_coverage",
            exp["abs"]
        ),
        (
            "positive_weight_coverage",
            exp["pos"]
        ),
        (
            "negative_weight_coverage",
            exp["neg"]
        )
    ]

    for column, anchor in numeric_checks:
        observed = parse_float(
            row[column]
        )

        if not approx_equal(
            observed,
            anchor,
            tol=1e-6
        ):
            raise RuntimeError(
                f"{ds}: {column} mismatch. "
                f"Observed {observed}; expected {anchor}."
            )

    if parse_bool(
        row["technical_pass"]
    ) != exp["technical"]:
        raise RuntimeError(
            f"{ds}: technical-pass anchor mismatch."
        )

    if parse_bool(
        row["outcome_evaluated"]
    ) != exp["outcome"]:
        raise RuntimeError(
            f"{ds}: outcome-evaluated anchor mismatch."
        )

print("Five-cohort frozen numeric anchors: PASS")


# --------------------------------------------------------------------------------------------------
# 7. REPRODUCE COMPONENT GATE LOGIC WITHOUT REANALYSIS
# --------------------------------------------------------------------------------------------------

for row in registry:
    ds = row["dataset"]

    gc = parse_float(
        row["gene_count_coverage"]
    )

    aw = parse_float(
        row["absolute_weight_coverage"]
    )

    pw = parse_float(
        row["positive_weight_coverage"]
    )

    nw = parse_float(
        row["negative_weight_coverage"]
    )

    sc = parse_float(
        row["sample_completeness"]
    )

    map_ok = parse_bool(
        row["mapping_unambiguous_for_gate"]
    )

    computed_gene_pass = (
        gc >= GENE_THRESHOLD
    )

    computed_abs_pass = (
        aw >= ABS_THRESHOLD
    )

    computed_pos_pass = (
        pw >= POS_THRESHOLD
    )

    computed_neg_pass = (
        nw >= NEG_THRESHOLD
    )

    if parse_bool(
        row["gene_count_pass"]
    ) != computed_gene_pass:
        raise RuntimeError(
            f"{ds}: gene-count gate flag mismatch."
        )

    if parse_bool(
        row["absolute_weight_pass"]
    ) != computed_abs_pass:
        raise RuntimeError(
            f"{ds}: absolute-weight gate flag mismatch."
        )

    if parse_bool(
        row["positive_weight_pass"]
    ) != computed_pos_pass:
        raise RuntimeError(
            f"{ds}: positive-weight gate flag mismatch."
        )

    if parse_bool(
        row["negative_weight_pass"]
    ) != computed_neg_pass:
        raise RuntimeError(
            f"{ds}: negative-weight gate flag mismatch."
        )

    sample_ok = (
        sc is not None
        and sc >= SAMPLE_COMPLETENESS_REQUIRED
    )

    computed_technical_pass = (
        computed_gene_pass
        and computed_abs_pass
        and computed_pos_pass
        and computed_neg_pass
        and map_ok
        and sample_ok
    )

    observed_technical_pass = parse_bool(
        row["technical_pass"]
    )

    if observed_technical_pass != computed_technical_pass:
        raise RuntimeError(
            f"{ds}: reproduced overall technical-gate decision "
            "does not match frozen registry."
        )

    outcome_evaluated = parse_bool(
        row["outcome_evaluated"]
    )

    if outcome_evaluated != observed_technical_pass:
        raise RuntimeError(
            f"{ds}: outcome-evaluation state is inconsistent "
            "with the frozen technical gate."
        )

print("Frozen component-gate reproduction: PASS")
print("Outcome-evaluation gate consistency: PASS")


# --------------------------------------------------------------------------------------------------
# 8. MASTER SUMMARY CHECKS
# --------------------------------------------------------------------------------------------------

n_screened = len(registry)

n_pass = sum(
    parse_bool(
        row["technical_pass"]
    )
    for row in registry
)

n_fail = n_screened - n_pass

n_outcome = sum(
    parse_bool(
        row["outcome_evaluated"]
    )
    for row in registry
)

if (
    n_screened != 5
    or n_pass != 1
    or n_fail != 4
    or n_outcome != 1
):
    raise RuntimeError(
        "Master external-gate count anchors failed."
    )

pass_dataset = [
    row["dataset"]
    for row in registry
    if parse_bool(
        row["technical_pass"]
    )
]

if pass_dataset != ["GSE224683"]:
    raise RuntimeError(
        "GSE224683 is not the unique technical-pass cohort."
    )

print("External cohorts screened = 5: PASS")
print("Technical PASS = 1: PASS")
print("Technically non-evaluable = 4: PASS")
print("Outcome evaluated = 1: PASS")
print("Unique technical-pass cohort = GSE224683: PASS")


# --------------------------------------------------------------------------------------------------
# 9. SOURCE TABLE S6A - FROZEN GATE DEFINITION
# --------------------------------------------------------------------------------------------------

s6a_rows = [
    {
        "criterion": "Frozen feature-count coverage",
        "frozen_requirement": ">= 0.80",
        "decision_role": (
            "At least 80% of the 24 frozen features must map "
            "unambiguously."
        )
    },
    {
        "criterion": "Retained total absolute weight",
        "frozen_requirement": ">= 0.80",
        "decision_role": (
            "Mapped features must retain at least 80% of the total "
            "absolute frozen score weight."
        )
    },
    {
        "criterion": "Retained positive-direction absolute weight",
        "frozen_requirement": ">= 0.70",
        "decision_role": (
            "Mapped features must retain at least 70% of the positive "
            "absolute weight mass."
        )
    },
    {
        "criterion": "Retained negative-direction absolute weight",
        "frozen_requirement": ">= 0.70",
        "decision_role": (
            "Mapped features must retain at least 70% of the negative "
            "absolute weight mass."
        )
    },
    {
        "criterion": "Mapping determinism / ambiguity",
        "frozen_requirement": "Unambiguous deterministic mapping required",
        "decision_role": (
            "No arbitrary duplicate collapse or ambiguous feature rescue "
            "is allowed."
        )
    },
    {
        "criterion": "Sample completeness",
        "frozen_requirement": "Complete score inputs required",
        "decision_role": (
            "Outcome-score evaluation requires complete usable mapped "
            "inputs for the evaluated samples."
        )
    },
    {
        "criterion": "Overall technical decision",
        "frozen_requirement": "All applicable requirements must pass",
        "decision_role": (
            "A failed gate is classified as technically non-evaluable; "
            "diagnosis-score evaluation is not performed."
        )
    }
]

write_csv(
    S6A_OUT,
    [
        "criterion",
        "frozen_requirement",
        "decision_role"
    ],
    s6a_rows
)


# --------------------------------------------------------------------------------------------------
# 10. SOURCE TABLE S6B - DETAILED COHORT GATE AUDIT
# --------------------------------------------------------------------------------------------------

s6b_rows = []

for row in registry:
    s6b_rows.append(
        {
            "public_external_sequence": parse_int(
                row["public_external_sequence"]
            ),
            "dataset": row["dataset"],
            "n_samples": parse_int(
                row["n_samples"]
            ),
            "tissue_region": row["tissue_region"],
            "frozen_gene_count": parse_int(
                row["frozen_gene_count"]
            ),
            "mapped_gene_count": parse_int(
                row["mapped_gene_count"]
            ),
            "gene_count_coverage": parse_float(
                row["gene_count_coverage"]
            ),
            "gene_count_pass": parse_bool(
                row["gene_count_pass"]
            ),
            "absolute_weight_coverage": parse_float(
                row["absolute_weight_coverage"]
            ),
            "absolute_weight_pass": parse_bool(
                row["absolute_weight_pass"]
            ),
            "positive_weight_coverage": parse_float(
                row["positive_weight_coverage"]
            ),
            "positive_weight_pass": parse_bool(
                row["positive_weight_pass"]
            ),
            "negative_weight_coverage": parse_float(
                row["negative_weight_coverage"]
            ),
            "negative_weight_pass": parse_bool(
                row["negative_weight_pass"]
            ),
            "mapping_unambiguous_for_gate": parse_bool(
                row["mapping_unambiguous_for_gate"]
            ),
            "sample_completeness": parse_float(
                row["sample_completeness"]
            ),
            "technical_pass": parse_bool(
                row["technical_pass"]
            ),
            "outcome_evaluated": parse_bool(
                row["outcome_evaluated"]
            ),
            "final_status": row["final_status"]
        }
    )

write_csv(
    S6B_OUT,
    [
        "public_external_sequence",
        "dataset",
        "n_samples",
        "tissue_region",
        "frozen_gene_count",
        "mapped_gene_count",
        "gene_count_coverage",
        "gene_count_pass",
        "absolute_weight_coverage",
        "absolute_weight_pass",
        "positive_weight_coverage",
        "positive_weight_pass",
        "negative_weight_coverage",
        "negative_weight_pass",
        "mapping_unambiguous_for_gate",
        "sample_completeness",
        "technical_pass",
        "outcome_evaluated",
        "final_status"
    ],
    s6b_rows
)


# --------------------------------------------------------------------------------------------------
# 11. SOURCE TABLE S6C - ASSAY / MAPPING / TRANSPARENCY AUDIT
# --------------------------------------------------------------------------------------------------

s6c_rows = []

for row in registry:
    s6c_rows.append(
        {
            "dataset": row["dataset"],
            "assay_platform_summary": row[
                "assay_platform_summary"
            ],
            "mapping_basis": row[
                "mapping_basis"
            ],
            "mapping_unambiguous_for_gate": parse_bool(
                row["mapping_unambiguous_for_gate"]
            ),
            "sample_completeness": parse_float(
                row["sample_completeness"]
            ),
            "outcome_evaluated": parse_bool(
                row["outcome_evaluated"]
            ),
            "outcome_blinding_note": row[
                "outcome_blinding_note"
            ],
            "measurability_note": row[
                "measurability_note"
            ]
        }
    )

write_csv(
    S6C_OUT,
    [
        "dataset",
        "assay_platform_summary",
        "mapping_basis",
        "mapping_unambiguous_for_gate",
        "sample_completeness",
        "outcome_evaluated",
        "outcome_blinding_note",
        "measurability_note"
    ],
    s6c_rows
)

print("Supplementary Table S6 source CSVs: PASS")


# --------------------------------------------------------------------------------------------------
# 12. WORD DOCUMENT
# --------------------------------------------------------------------------------------------------

doc = Document()

section = doc.sections[0]

section.orientation = (
    WD_ORIENT.LANDSCAPE
)

section.page_width = Inches(11.0)
section.page_height = Inches(8.5)

section.top_margin = Inches(0.45)
section.bottom_margin = Inches(0.45)
section.left_margin = Inches(0.45)
section.right_margin = Inches(0.45)

section.header_distance = Inches(0.2)
section.footer_distance = Inches(0.2)

styles = doc.styles

normal_style = styles["Normal"]
normal_style.font.name = "Times New Roman"
normal_style.font.size = Pt(9.0)

normal_style._element.rPr.rFonts.set(
    qn("w:ascii"),
    "Times New Roman"
)

normal_style._element.rPr.rFonts.set(
    qn("w:hAnsi"),
    "Times New Roman"
)

normal_style._element.rPr.rFonts.set(
    qn("w:eastAsia"),
    "Times New Roman"
)


# --------------------------------------------------------------------------------------------------
# 12A. TITLE
# --------------------------------------------------------------------------------------------------

title = doc.add_paragraph()

set_paragraph_format(
    title,
    align=WD_ALIGN_PARAGRAPH.LEFT,
    space_before=0,
    space_after=3,
    line_spacing=1.0
)

title_run = title.add_run(
    "Supplementary Table S6. "
    "Frozen external technical-gate audit across five public cohorts"
)

set_run_font(
    title_run,
    size_pt=11.0,
    bold=True
)

intro = doc.add_paragraph()

set_paragraph_format(
    intro,
    align=WD_ALIGN_PARAGRAPH.LEFT,
    space_before=0,
    space_after=4,
    line_spacing=1.05
)

intro_run = intro.add_run(
    "The frozen SCZ-MSS v1 technical gate was applied before "
    "diagnosis-score evaluation. Cohorts failing the gate were "
    "classified as technically non-evaluable and were not interpreted "
    "as biological replication failures."
)

set_run_font(
    intro_run,
    size_pt=8.6,
    bold=False
)


# --------------------------------------------------------------------------------------------------
# 12B. TABLE S6A
# --------------------------------------------------------------------------------------------------

add_section_label(
    doc,
    "A. Frozen technical-gate criteria"
)

s6a_word_rows = [
    [
        row["criterion"],
        row["frozen_requirement"],
        row["decision_role"]
    ]
    for row in s6a_rows
]

add_plain_table(
    doc,
    headers=[
        "Criterion",
        "Frozen requirement",
        "Decision role"
    ],
    rows=s6a_word_rows,
    widths=[
        2.35,
        2.25,
        5.50
    ],
    center_columns={
        1
    },
    body_font_size=8.6
)


# --------------------------------------------------------------------------------------------------
# 12C. TABLE S6B
# --------------------------------------------------------------------------------------------------

add_section_label(
    doc,
    "B. Cohort-level technical-gate results"
)

s6b_word_rows = []

for row in registry:
    mapped = parse_int(
        row["mapped_gene_count"]
    )

    frozen = parse_int(
        row["frozen_gene_count"]
    )

    tech = parse_bool(
        row["technical_pass"]
    )

    s6b_word_rows.append(
        [
            row["dataset"],
            str(
                parse_int(
                    row["n_samples"]
                )
            ),
            row["tissue_region"].replace(
                "_",
                " "
            ),
            f"{mapped}/{frozen}",
            (
                f"{pct(parse_float(row['gene_count_coverage']))} "
                f"({pass_fail(parse_bool(row['gene_count_pass']))})"
            ),
            (
                f"{pct(parse_float(row['absolute_weight_coverage']))} "
                f"({pass_fail(parse_bool(row['absolute_weight_pass']))})"
            ),
            (
                f"{pct(parse_float(row['positive_weight_coverage']))} "
                f"({pass_fail(parse_bool(row['positive_weight_pass']))})"
            ),
            (
                f"{pct(parse_float(row['negative_weight_coverage']))} "
                f"({pass_fail(parse_bool(row['negative_weight_pass']))})"
            ),
            (
                "PASS - outcome evaluation eligible"
                if tech
                else "FAIL - technically non-evaluable"
            )
        ]
    )

add_plain_table(
    doc,
    headers=[
        "Cohort",
        "n",
        "Region",
        "Mapped\nfeatures",
        "Feature-count\ncoverage",
        "Absolute-weight\ncoverage",
        "Positive-weight\ncoverage",
        "Negative-weight\ncoverage",
        "Technical decision"
    ],
    rows=s6b_word_rows,
    widths=[
        1.00,
        0.45,
        0.80,
        0.75,
        1.10,
        1.15,
        1.15,
        1.15,
        1.55
    ],
    center_columns={
        0,
        1,
        2,
        3,
        4,
        5,
        6,
        7
    },
    body_font_size=8.3
)


# --------------------------------------------------------------------------------------------------
# 12D. TABLE S6C
# --------------------------------------------------------------------------------------------------

add_section_label(
    doc,
    "C. Assay, mapping, measurability, and outcome-access audit"
)

s6c_word_rows = []

for row in registry:
    sc = parse_float(
        row["sample_completeness"]
    )

    s6c_word_rows.append(
        [
            row["dataset"],
            row["assay_platform_summary"],
            row["mapping_basis"],
            (
                "Yes"
                if parse_bool(
                    row["mapping_unambiguous_for_gate"]
                )
                else "No"
            ),
            display_sample_completeness(sc),
            (
                yes_no(
                    parse_bool(
                        row["outcome_evaluated"]
                    )
                )
                + ". "
                + row["outcome_blinding_note"]
            ),
            row["measurability_note"]
        ]
    )

add_plain_table(
    doc,
    headers=[
        "Cohort",
        "Assay / platform",
        "Mapping basis",
        "Unambiguous\nmapping",
        "Sample\ncompleteness",
        "Outcome evaluation / access note",
        "Measurability note"
    ],
    rows=s6c_word_rows,
    widths=[
        0.85,
        1.50,
        1.85,
        0.75,
        0.90,
        2.20,
        2.05
    ],
    center_columns={
        0,
        3,
        4
    },
    body_font_size=7.9
)


# --------------------------------------------------------------------------------------------------
# 12E. NOTES
# --------------------------------------------------------------------------------------------------

add_note_paragraph(
    doc,
    (
        "Notes. Coverage values are fractions of the frozen 24-feature "
        "SCZ-MSS v1 state. DLPFC, dorsolateral prefrontal cortex; "
        "BA9, Brodmann area 9; TPM, transcripts per million. "
        "Technical failure indicates insufficient assay/annotation "
        "transferability under the frozen gate and does not establish "
        "biological non-replication."
    )
)

add_note_paragraph(
    doc,
    (
        "Only GSE224683 satisfied all frozen technical requirements and "
        "therefore proceeded to diagnosis-score evaluation. "
        "The primary external performance result itself is reported "
        "separately and is not duplicated in this technical-audit table."
    )
)


# --------------------------------------------------------------------------------------------------
# 13. FORCE TIMES NEW ROMAN THROUGHOUT
# --------------------------------------------------------------------------------------------------

for paragraph in doc.paragraphs:
    for run in paragraph.runs:
        current_size = (
            run.font.size.pt
            if run.font.size is not None
            else 9.0
        )

        set_run_font(
            run,
            size_pt=current_size,
            bold=bool(run.bold),
            italic=bool(run.italic)
        )

for table in doc.tables:
    for row in table.rows:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    current_size = (
                        run.font.size.pt
                        if run.font.size is not None
                        else 8.6
                    )

                    set_run_font(
                        run,
                        size_pt=current_size,
                        bold=bool(run.bold),
                        italic=bool(run.italic)
                    )


# --------------------------------------------------------------------------------------------------
# 14. SAVE DOCX
# --------------------------------------------------------------------------------------------------

doc.save(
    DOCX_OUT
)

if (
    not DOCX_OUT.exists()
    or DOCX_OUT.stat().st_size <= 0
):
    raise RuntimeError(
        "Microsoft Word DOCX was not created."
    )

print("Microsoft Word DOCX created: PASS")


# --------------------------------------------------------------------------------------------------
# 15. DOCX STRUCTURAL READ-BACK
# --------------------------------------------------------------------------------------------------

check_doc = Document(
    DOCX_OUT
)

if len(check_doc.tables) != 3:
    raise RuntimeError(
        "DOCX structural check failed: expected 3 tables."
    )

expected_row_counts = [
    1 + len(s6a_rows),
    1 + len(registry),
    1 + len(registry)
]

for i, expected_rows in enumerate(
    expected_row_counts
):
    observed_rows = len(
        check_doc.tables[i].rows
    )

    if observed_rows != expected_rows:
        raise RuntimeError(
            f"DOCX table {i + 1}: expected "
            f"{expected_rows} rows, found {observed_rows}."
        )

required_title_fragment = (
    "Supplementary Table S6."
)

all_doc_text = "\n".join(
    p.text
    for p in check_doc.paragraphs
)

if required_title_fragment not in all_doc_text:
    raise RuntimeError(
        "DOCX title read-back failed."
    )

for table_index, table in enumerate(
    check_doc.tables,
    start=1
):
    for row_index, row in enumerate(
        table.rows,
        start=1
    ):
        for cell_index, cell in enumerate(
            row.cells,
            start=1
        ):
            tcPr = cell._tc.get_or_add_tcPr()

            borders = tcPr.find(
                qn("w:tcBorders")
            )

            if borders is None:
                raise RuntimeError(
                    f"Missing cell borders in table "
                    f"{table_index}, row {row_index}, "
                    f"cell {cell_index}."
                )

            for edge in (
                "top",
                "left",
                "bottom",
                "right"
            ):
                edge_node = borders.find(
                    qn(f"w:{edge}")
                )

                if edge_node is None:
                    raise RuntimeError(
                        f"Missing {edge} border in table "
                        f"{table_index}, row {row_index}, "
                        f"cell {cell_index}."
                    )

            shading = tcPr.find(
                qn("w:shd")
            )

            if shading is not None:
                fill = shading.get(
                    qn("w:fill")
                )

                if (
                    fill is not None
                    and fill.upper()
                    not in {
                        "AUTO",
                        "FFFFFF",
                        "NIL"
                    }
                ):
                    raise RuntimeError(
                        "Unexpected table-cell shading/color "
                        "detected."
                    )

print("DOCX structural read-back: PASS")
print("Full row borders: PASS")
print("Full column borders: PASS")
print("Black-and-white table structure: PASS")


# --------------------------------------------------------------------------------------------------
# 16. NOTES FILE
# --------------------------------------------------------------------------------------------------

notes_text = """SUPPLEMENTARY TABLE S6 NOTES
===============================================

Title:
Frozen external technical-gate audit across five public cohorts

Source:
Frozen STEP09J.1 external technical-gate master registry only.

Frozen gate:
- Frozen feature-count coverage >= 0.80
- Retained total absolute weight >= 0.80
- Retained positive-direction absolute weight >= 0.70
- Retained negative-direction absolute weight >= 0.70
- Deterministic unambiguous mapping required
- Complete score inputs required for outcome evaluation

Interpretation rule:
Technical FAIL means technically non-evaluable under the frozen
cross-platform transferability requirements. It must not be described
as a biological replication failure.

Outcome rule:
Diagnosis-score evaluation is performed only after technical PASS.

Five-cohort state:
- Public cohorts screened: 5
- Technical PASS: 1
- Technically non-evaluable: 4
- Outcome evaluated: 1
- Unique technical-pass cohort: GSE224683

No model fitting, feature selection, mapping rescue, score recalculation,
or phenotype reanalysis was performed in Table S6 generation.
"""

NOTES_OUT.write_text(
    notes_text,
    encoding="utf-8"
)


# --------------------------------------------------------------------------------------------------
# 17. GENERATION SUMMARY
# --------------------------------------------------------------------------------------------------

summary_text = f"""SUPPLEMENTARY TABLE S6 GENERATION SUMMARY
============================================================

Frozen master registry: PASS
External analysis rerun: NO
Score recalculation: NO
Mapping rescue: NO
Phenotype reanalysis: NO
Frozen thresholds changed: NO

Public cohorts screened: {n_screened}
Technical PASS: {n_pass}
Technically non-evaluable: {n_fail}
Outcome evaluated: {n_outcome}
Unique technical-pass cohort: GSE224683

Cohort anchors:
GSE87194  : 14/24; gene 0.5833330; abs 0.3332160; pos 0.2611400; neg 0.3787040; FAIL
GSE80655  : 20/24; gene 0.8333330; abs 0.7197100; pos 1.0000000; neg 0.5428170; FAIL
GSE224683 : 23/24; gene 0.9583330; abs 0.9266000; pos 1.0000000; neg 0.8802760; PASS
GSE282552 :  9/24; gene 0.3750000; abs 0.1961925; pos 0.0304038; neg 0.3008226; FAIL
GSE208338 : 14/24; gene 0.5833333; abs 0.3818154; pos 0.2611397; neg 0.4579744; FAIL

Times New Roman: PASS
Black-and-white only: PASS
Full row borders: PASS
Full column borders: PASS
DOCX structural read-back: PASS

DOCX:
{DOCX_OUT}

Source CSVs:
{S6A_OUT}
{S6B_OUT}
{S6C_OUT}
"""

SUMMARY_OUT.write_text(
    summary_text,
    encoding="utf-8"
)


# --------------------------------------------------------------------------------------------------
# 18. MANIFEST
# --------------------------------------------------------------------------------------------------

manifest_targets = [
    (
        "frozen_input_registry",
        REGISTRY_PATH
    ),
    (
        "python_generator",
        PY_SCRIPT_PATH
    ),
    (
        "word_docx",
        DOCX_OUT
    ),
    (
        "source_table_S6A",
        S6A_OUT
    ),
    (
        "source_table_S6B",
        S6B_OUT
    ),
    (
        "source_table_S6C",
        S6C_OUT
    ),
    (
        "notes",
        NOTES_OUT
    ),
    (
        "generation_summary",
        SUMMARY_OUT
    )
]

manifest_rows = []

for artifact_role, path in manifest_targets:
    if not path.exists():
        raise RuntimeError(
            f"Manifest target missing: {path}"
        )

    manifest_rows.append(
        {
            "artifact_role": artifact_role,
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path)
        }
    )

write_csv(
    MANIFEST_OUT,
    [
        "artifact_role",
        "path",
        "size_bytes",
        "sha256"
    ],
    manifest_rows
)


# --------------------------------------------------------------------------------------------------
# 19. FINAL OUTPUT CHECK
# --------------------------------------------------------------------------------------------------

expected_outputs = [
    DOCX_OUT,
    S6A_OUT,
    S6B_OUT,
    S6C_OUT,
    NOTES_OUT,
    SUMMARY_OUT,
    MANIFEST_OUT,
    PY_SCRIPT_PATH
]

for path in expected_outputs:
    if not path.exists():
        raise RuntimeError(
            f"Expected Table S6 artifact missing: {path}"
        )

    if path.stat().st_size <= 0:
        raise RuntimeError(
            f"Expected Table S6 artifact is zero-byte: {path}"
        )

print("All Supplementary Table S6 outputs created and non-zero: PASS")


# --------------------------------------------------------------------------------------------------
# 20. CONSOLE SUMMARY
# --------------------------------------------------------------------------------------------------

print()
print("=" * 110)
print("SUPPLEMENTARY TABLE S6 GENERATION SUMMARY")
print("=" * 110)

print("Frozen STEP09J.1 master registry: PASS")
print("External analysis rerun: NO")
print("Score recalculation: NO")
print("Mapping rescue: NO")
print("Phenotype reanalysis: NO")
print()

print(f"Public cohorts screened: {n_screened}")
print(f"Technical PASS: {n_pass}")
print(f"Technically non-evaluable: {n_fail}")
print(f"Outcome evaluated: {n_outcome}")
print("Unique technical-pass cohort: GSE224683")
print()

for row in registry:
    print(
        f"{row['dataset']}: "
        f"{parse_int(row['mapped_gene_count'])}/24 mapped; "
        f"gene={parse_float(row['gene_count_coverage']):.7f}; "
        f"abs={parse_float(row['absolute_weight_coverage']):.7f}; "
        f"pos={parse_float(row['positive_weight_coverage']):.7f}; "
        f"neg={parse_float(row['negative_weight_coverage']):.7f}; "
        f"technical={pass_fail(parse_bool(row['technical_pass']))}"
    )

print()
print("Times New Roman: PASS")
print("Black-and-white only: PASS")
print("Full row borders: PASS")
print("Full column borders: PASS")
print("DOCX structural read-back: PASS")
print()

print("DOCX:")
print(DOCX_OUT)
print()

print("Source CSVs:")
print(S6A_OUT)
print(S6B_OUT)
print(S6C_OUT)
print()

print("Manifest:")
print(MANIFEST_OUT)

print()
print("=" * 110)
print("STEP 11E.6 COMPLETE - SUPPLEMENTARY TABLE S6 SAVED PROJECT-LOCALLY")
print("=" * 110)
)---"

# --------------------------------------------------------------------------------------------------
# 5. SAVE PYTHON GENERATOR
# --------------------------------------------------------------------------------------------------

writeLines(
    PY_SOURCE,
    PY_SCRIPT,
    useBytes = TRUE
)

if (
    !file.exists(PY_SCRIPT) ||
    file.info(PY_SCRIPT)$size <= 0
) {
    stop(
        "Python Table S6 generator was not created."
    )
}

cat(
    "Python generator saved: ",
    PY_SCRIPT,
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 6. PYTHON SYNTAX PREFLIGHT
# --------------------------------------------------------------------------------------------------

compile_output <- system2(
    command = PYTHON_EXE,
    args = c(
        "-m",
        "py_compile",
        shQuote(PY_SCRIPT)
    ),
    stdout = TRUE,
    stderr = TRUE
)

compile_status <- attr(
    compile_output,
    "status"
)

if (
    !is.null(compile_status) &&
    compile_status != 0
) {
    cat(
        paste(
            compile_output,
            collapse = "\n"
        ),
        "\n"
    )

    stop(
        "Table S6 Python generator failed syntax preflight."
    )
}

cat("Python syntax preflight: PASS\n")

# --------------------------------------------------------------------------------------------------
# 7. EXECUTE GENERATOR
# --------------------------------------------------------------------------------------------------

cmd_output <- system2(
    command = PYTHON_EXE,
    args = shQuote(PY_SCRIPT),
    stdout = TRUE,
    stderr = TRUE
)

exit_status <- attr(
    cmd_output,
    "status"
)

cat(
    paste(
        cmd_output,
        collapse = "\n"
    ),
    "\n"
)

if (
    !is.null(exit_status) &&
    exit_status != 0
) {
    stop(
        "Python Table S6 generator failed with exit status ",
        exit_status,
        ". See complete console output above."
    )
}

# --------------------------------------------------------------------------------------------------
# 8. R-SIDE FINAL ARTIFACT CHECK
# --------------------------------------------------------------------------------------------------

EXPECTED_DOCX <- file.path(
    OUT_DIR,
    "TableS6_External_Technical_Gate_Audit_v1.docx"
)

EXPECTED_S6A <- file.path(
    OUT_DIR,
    "TableS6A_frozen_gate_definition_source_v1.csv"
)

EXPECTED_S6B <- file.path(
    OUT_DIR,
    "TableS6B_external_cohort_gate_audit_source_v1.csv"
)

EXPECTED_S6C <- file.path(
    OUT_DIR,
    "TableS6C_mapping_provenance_source_v1.csv"
)

EXPECTED_NOTES <- file.path(
    OUT_DIR,
    "TableS6_notes_v1.txt"
)

EXPECTED_SUMMARY <- file.path(
    OUT_DIR,
    "TableS6_generation_summary_v1.txt"
)

EXPECTED_MANIFEST <- file.path(
    OUT_DIR,
    "TableS6_generation_manifest_v1.csv"
)

expected_outputs <- c(
    EXPECTED_DOCX,
    EXPECTED_S6A,
    EXPECTED_S6B,
    EXPECTED_S6C,
    EXPECTED_NOTES,
    EXPECTED_SUMMARY,
    EXPECTED_MANIFEST,
    PY_SCRIPT
)

missing_outputs <- expected_outputs[
    !file.exists(
        expected_outputs
    )
]

if (length(missing_outputs) > 0L) {
    stop(
        "Expected Table S6 artifact(s) missing:\n",
        paste(
            missing_outputs,
            collapse = "\n"
        )
    )
}

output_sizes <- file.info(
    expected_outputs
)$size

if (any(
    is.na(output_sizes) |
    output_sizes <= 0
)) {
    stop(
        "One or more Table S6 artifacts are zero-byte."
    )
}

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.6R COMPLETE - TABLE S6 GENERATED THROUGH PYTHON FROM R KERNEL\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "\nMicrosoft Word DOCX:\n",
    EXPECTED_DOCX,
    "\n",
    sep = ""
)

cat(
    "\nSaved reproducible Python generator:\n",
    PY_SCRIPT,
    "\n",
    sep = ""
)

cat(
    "\nSource CSVs:\n",
    EXPECTED_S6A,
    "\n",
    EXPECTED_S6B,
    "\n",
    EXPECTED_S6C,
    "\n",
    sep = ""
)

cat(
    "\nManifest:\n",
    EXPECTED_MANIFEST,
    "\n",
    sep = ""
)


STEP 11E.6R - SUPPLEMENTARY TABLE S6
FROZEN EXTERNAL TECHNICAL-GATE AUDIT ACROSS FIVE PUBLIC COHORTS
Project sentinel: PASS
Python executable: PASS
Python generator saved: E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/tables/supplementary/TableS6/generate_TableS6_v1.py
Python syntax preflight: PASS
STEP 11E.6 - SUPPLEMENTARY TABLE S6
FROZEN EXTERNAL TECHNICAL-GATE AUDIT ACROSS FIVE PUBLIC COHORTS
Project sentinel: PASS
Frozen STEP09J.1 external master registry: PASS
Completed public cohort set = GSE87194 / GSE80655 / GSE224683 / GSE282552 / GSE208338: PASS
Frozen technical-gate thresholds loaded: PASS
Five-cohort frozen numeric anchors: PASS
Frozen component-gate reproduction: PASS
Outcome-evaluation gate consistency: PASS
External cohorts screened = 5: PASS
Technical PASS = 1: PASS
Technically non-evaluable = 4: PASS
Outcome evaluated = 1: PASS
Unique technical-pass cohort = GSE224683: PASS
Supplementary Table S6 source CSVs: PASS
Microsoft Word DOCX created: PASS
DOCX structural read-back

In [30]:
# ==================================================================================================
# STEP 11E.7R - SUPPLEMENTARY TABLE S7
# GSE224683 PRIMARY EXTERNAL OUTCOME EVALUATION,
# OUTCOME-INDEPENDENT MEASURABILITY AUDIT,
# AND POST-HOC MEASURABLE-ONLY SENSITIVITY
#
# R KERNEL -> SAVED PYTHON-DOCX GENERATOR
#
# IMPORTANT
#   - PRIMARY STEP09D result remains primary.
#   - STEP09E is outcome-independent descriptive measurability audit.
#   - STEP09F is explicitly POST-HOC / EXPLORATORY.
#   - No score recalculation.
#   - No retraining.
#   - No external tuning.
#   - No feature rescue.
#   - No new outcome analysis.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.7R - SUPPLEMENTARY TABLE S7\n",
    "GSE224683 PRIMARY EXTERNAL OUTCOME EVALUATION + MEASURABILITY + POST-HOC SENSITIVITY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 1. PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

observed_project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    observed_project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "Project ID mismatch. Expected ",
        EXPECTED_PROJECT_ID,
        " but observed ",
        observed_project_id
    )
}

cat("Project sentinel: PASS\n")

# --------------------------------------------------------------------------------------------------
# 2. PYTHON
# --------------------------------------------------------------------------------------------------

PYTHON_EXE <- "E:/Python/python.exe"

if (!file.exists(PYTHON_EXE)) {
    stop(
        "Python executable not found: ",
        PYTHON_EXE
    )
}

cat("Python executable: PASS\n")

# --------------------------------------------------------------------------------------------------
# 3. OUTPUT DIRECTORY
# --------------------------------------------------------------------------------------------------

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "supplementary",
    "TableS7"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PY_SCRIPT <- file.path(
    OUT_DIR,
    "generate_TableS7_v1.py"
)

# --------------------------------------------------------------------------------------------------
# 4. COMPLETE PYTHON GENERATOR
# --------------------------------------------------------------------------------------------------

PY_SOURCE <- r"------(
from pathlib import Path
import csv
import hashlib
import math

from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT
from docx.shared import Inches, Pt
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


print("=" * 110)
print("STEP 11E.7 - SUPPLEMENTARY TABLE S7")
print("GSE224683 PRIMARY EXTERNAL EVALUATION + MEASURABILITY + POST-HOC SENSITIVITY")
print("=" * 110)

# ==================================================================================================
# 1. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "E:/SCZ_CROSSREGION_MSS_Q1"
)

EXPECTED_PROJECT_ID = (
    "SCZ_CROSSREGION_MSS_Q1_2026"
)

SENTINEL = (
    PROJECT_ROOT
    / ".project_id"
)

GSE_DIR = (
    PROJECT_ROOT
    / "09_external_validation"
    / "gse224683"
)

PRIMARY_SUMMARY = (
    GSE_DIR
    / "STEP09D_GSE224683_external_validation_summary_v1.csv"
)

PRIMARY_SAMPLE = (
    GSE_DIR
    / "STEP09D_GSE224683_sample_SCZ_MSS_v1.csv"
)

MEAS_SUMMARY = (
    GSE_DIR
    / "STEP09E_GSE224683_measurability_summary_v1.csv"
)

MEAS_AUDIT = (
    GSE_DIR
    / "STEP09E_GSE224683_frozen_gene_measurability_v1.csv"
)

SENS_SUMMARY = (
    GSE_DIR
    / "STEP09F_GSE224683_measurable_only_summary_v1.csv"
)

SENS_GENES = (
    GSE_DIR
    / "STEP09F_GSE224683_measurable_only_genes_v1.csv"
)

MASTER_REGISTRY = (
    PROJECT_ROOT
    / "09_external_validation"
    / "master_registry_v1"
    / "SCZ_MSS_v1_external_technical_gate_registry_v1.csv"
)

OUT_DIR = (
    PROJECT_ROOT
    / "11_manuscript"
    / "tables"
    / "supplementary"
    / "TableS7"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PY_SCRIPT_PATH = (
    OUT_DIR
    / "generate_TableS7_v1.py"
)

DOCX_OUT = (
    OUT_DIR
    / "TableS7_GSE224683_Primary_and_Posthoc_External_Evaluation_v1.docx"
)

S7A_OUT = (
    OUT_DIR
    / "TableS7A_primary_external_results_source_v1.csv"
)

S7B_OUT = (
    OUT_DIR
    / "TableS7B_measurability_summary_source_v1.csv"
)

S7C_OUT = (
    OUT_DIR
    / "TableS7C_unmapped_allzero_features_source_v1.csv"
)

S7D_OUT = (
    OUT_DIR
    / "TableS7D_posthoc_measurable_only_source_v1.csv"
)

NOTES_OUT = (
    OUT_DIR
    / "TableS7_notes_v1.txt"
)

SUMMARY_OUT = (
    OUT_DIR
    / "TableS7_generation_summary_v1.txt"
)

MANIFEST_OUT = (
    OUT_DIR
    / "TableS7_generation_manifest_v1.csv"
)

required_inputs = [
    PRIMARY_SUMMARY,
    PRIMARY_SAMPLE,
    MEAS_SUMMARY,
    MEAS_AUDIT,
    SENS_SUMMARY,
    SENS_GENES,
    MASTER_REGISTRY,
]

for path in required_inputs:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen input missing: {path}"
        )

    if path.stat().st_size <= 0:

        raise RuntimeError(
            f"Required frozen input is zero-byte: {path}"
        )

print("Frozen STEP09D/E/F inputs: PASS")


# ==================================================================================================
# 2. HELPERS
# ==================================================================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:

        while True:

            block = handle.read(
                1024 * 1024
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def read_csv(path):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as handle:

        reader = csv.DictReader(
            handle
        )

        rows = list(
            reader
        )

        fields = reader.fieldnames or []

    return rows, fields


def clean_text(value):

    if value is None:
        return ""

    text = str(
        value
    ).strip()

    if text.upper() in {
        "",
        "NA",
        "NAN",
        "NONE",
        "<NA>"
    }:
        return ""

    return text


def num(value):

    text = clean_text(
        value
    )

    if text == "":
        return None

    result = float(
        text
    )

    if not math.isfinite(
        result
    ):

        raise RuntimeError(
            f"Non-finite numeric value: {value}"
        )

    return result


def integer(value):

    result = num(
        value
    )

    if result is None:

        raise RuntimeError(
            f"Expected integer-like value, observed missing: {value}"
        )

    if abs(
        result - round(result)
    ) > 1e-8:

        raise RuntimeError(
            f"Expected integer-like value: {value}"
        )

    return int(
        round(
            result
        )
    )


def logical(value):

    if isinstance(
        value,
        bool
    ):
        return value

    text = clean_text(
        value
    ).upper()

    if text in {
        "TRUE",
        "T",
        "1",
        "YES"
    }:
        return True

    if text in {
        "FALSE",
        "F",
        "0",
        "NO"
    }:
        return False

    raise RuntimeError(
        f"Cannot parse logical value: {value}"
    )


def metric_table(
    path,
    label
):

    rows, fields = read_csv(
        path
    )

    if (
        "metric" not in fields
        or "value" not in fields
    ):

        raise RuntimeError(
            f"{label} is not a metric/value table."
        )

    output = {}

    for row in rows:

        key = clean_text(
            row[
                "metric"
            ]
        )

        if key in output:

            raise RuntimeError(
                f"Duplicate metric in {label}: {key}"
            )

        output[
            key
        ] = num(
            row[
                "value"
            ]
        )

    return output


def require_metric(
    source,
    metric,
    source_name
):

    if metric not in source:

        raise RuntimeError(
            f"{source_name} missing metric: {metric}"
        )

    return source[
        metric
    ]


def approx(
    observed,
    expected,
    tolerance
):

    if observed is None:
        return False

    return abs(
        observed - expected
    ) <= tolerance


def fmt_number(
    value,
    digits=6
):

    return f"{value:.{digits}f}"


def fmt_p(value):

    if value < 0.0001:

        return f"{value:.2e}"

    return f"{value:.5f}"


def write_csv(
    path,
    header,
    rows
):

    with path.open(
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as handle:

        writer = csv.writer(
            handle
        )

        writer.writerow(
            header
        )

        writer.writerows(
            rows
        )


# ==================================================================================================
# 3. PROJECT SENTINEL
# ==================================================================================================

if not SENTINEL.exists():

    raise RuntimeError(
        "Project sentinel missing."
    )

project_id = SENTINEL.read_text(
    encoding="utf-8"
).strip()

if project_id != EXPECTED_PROJECT_ID:

    raise RuntimeError(
        f"Project sentinel mismatch: {project_id}"
    )

print("Project sentinel: PASS")


# ==================================================================================================
# 4. VERIFY GSE224683 IS THE FROZEN ELIGIBLE COHORT
# ==================================================================================================

registry, registry_fields = read_csv(
    MASTER_REGISTRY
)

if "dataset" not in registry_fields:

    raise RuntimeError(
        "Master external registry missing dataset field."
    )

g224_registry = [
    row
    for row in registry
    if clean_text(
        row[
            "dataset"
        ]
    ) == "GSE224683"
]

if len(
    g224_registry
) != 1:

    raise RuntimeError(
        "Could not uniquely identify GSE224683 in frozen master registry."
    )

g224_registry = g224_registry[
    0
]

if not logical(
    g224_registry[
        "technical_pass"
    ]
):

    raise RuntimeError(
        "GSE224683 is no longer technical PASS."
    )

if not logical(
    g224_registry[
        "outcome_evaluated"
    ]
):

    raise RuntimeError(
        "GSE224683 frozen outcome-evaluated flag is no longer TRUE."
    )

if clean_text(
    g224_registry[
        "final_status"
    ]
) != "TECHNICAL_PASS_OUTCOME_EVALUATED_NULL":

    raise RuntimeError(
        "GSE224683 frozen final status changed."
    )

print("GSE224683 frozen technical-pass / outcome-evaluated state: PASS")


# ==================================================================================================
# 5. PRIMARY STEP09D SUMMARY
# ==================================================================================================

primary = metric_table(
    PRIMARY_SUMMARY,
    "STEP09D primary summary"
)

primary_required = [
    "N_control",
    "N_schizophrenia",
    "Frozen_genes_available",
    "Retained_absolute_weight",
    "Control_mean_SCZ_MSS",
    "Control_SD_SCZ_MSS",
    "Schizophrenia_mean_SCZ_MSS",
    "Schizophrenia_SD_SCZ_MSS",
    "Mean_difference_SCZ_minus_Control",
    "Mean_difference_bootstrap_CI_low",
    "Mean_difference_bootstrap_CI_high",
    "Hedges_g",
    "Hedges_g_bootstrap_CI_low",
    "Hedges_g_bootstrap_CI_high",
    "AUROC",
    "AUROC_bootstrap_CI_low",
    "AUROC_bootstrap_CI_high",
    "Welch_t_p",
    "Wilcoxon_p",
]

for metric in primary_required:

    require_metric(
        primary,
        metric,
        "STEP09D"
    )

if integer(
    primary[
        "N_control"
    ]
) != 15:

    raise RuntimeError(
        "Primary control count changed."
    )

if integer(
    primary[
        "N_schizophrenia"
    ]
) != 15:

    raise RuntimeError(
        "Primary schizophrenia count changed."
    )

if integer(
    primary[
        "Frozen_genes_available"
    ]
) != 23:

    raise RuntimeError(
        "Primary frozen-gene count changed."
    )

primary_anchor_checks = [
    (
        "Retained_absolute_weight",
        0.92659973,
        5e-6
    ),
    (
        "Control_mean_SCZ_MSS",
        0.005851,
        5e-6
    ),
    (
        "Control_SD_SCZ_MSS",
        0.009426,
        5e-6
    ),
    (
        "Schizophrenia_mean_SCZ_MSS",
        0.005482,
        5e-6
    ),
    (
        "Schizophrenia_SD_SCZ_MSS",
        0.007473,
        5e-6
    ),
    (
        "Mean_difference_SCZ_minus_Control",
        -0.000369,
        5e-6
    ),
    (
        "Mean_difference_bootstrap_CI_low",
        -0.006485,
        5e-6
    ),
    (
        "Mean_difference_bootstrap_CI_high",
        0.005459,
        5e-6
    ),
    (
        "Hedges_g",
        -0.0422,
        5e-4
    ),
    (
        "Hedges_g_bootstrap_CI_low",
        -0.7574,
        5e-4
    ),
    (
        "Hedges_g_bootstrap_CI_high",
        0.7262,
        5e-4
    ),
    (
        "AUROC",
        0.5067,
        5e-4
    ),
    (
        "AUROC_bootstrap_CI_low",
        0.2889,
        5e-4
    ),
    (
        "AUROC_bootstrap_CI_high",
        0.7244,
        5e-4
    ),
    (
        "Welch_t_p",
        0.90635,
        5e-5
    ),
    (
        "Wilcoxon_p",
        0.96691,
        5e-5
    ),
]

for metric, expected, tolerance in primary_anchor_checks:

    observed = primary[
        metric
    ]

    if not approx(
        observed,
        expected,
        tolerance
    ):

        raise RuntimeError(
            f"Primary STEP09D anchor changed for {metric}: "
            f"observed={observed}, expected~={expected}"
        )

print("STEP09D primary external outcome anchors: PASS")


# ==================================================================================================
# 6. PRIMARY SAMPLE FILE CHECK
# ==================================================================================================

primary_samples, primary_sample_fields = read_csv(
    PRIMARY_SAMPLE
)

if len(
    primary_samples
) != 30:

    raise RuntimeError(
        f"Primary score file must contain 30 samples; observed {len(primary_samples)}."
    )

print("STEP09D primary 30-sample score file: PASS")


# ==================================================================================================
# 7. STEP09E OUTCOME-INDEPENDENT MEASURABILITY SUMMARY
# ==================================================================================================

meas = metric_table(
    MEAS_SUMMARY,
    "STEP09E measurability summary"
)

meas_expected = {
    "Frozen_gene_count":
        24.0,

    "Mapped_gene_count":
        23.0,

    "Unmapped_gene_count":
        1.0,

    "Mapped_absolute_weight":
        0.92659973,

    "Unmapped_absolute_weight":
        0.07340027,

    "Genes_nonzero_in_any_sample":
        17.0,

    "Genes_all_zero_TPM":
        6.0,

    "Genes_all_zero_counts":
        6.0,

    "All_zero_TPM_absolute_weight":
        0.44462073,

    "All_zero_counts_absolute_weight":
        0.44462073,

    "Absolute_weight_nonzero_in_any_sample":
        0.48197900,

    "All_zero_TPM_protein_coding_count":
        0.0,

    "All_zero_TPM_nonprotein_coding_count":
        6.0,
}

for metric, expected in meas_expected.items():

    observed = require_metric(
        meas,
        metric,
        "STEP09E"
    )

    tolerance = (
        5e-6
        if not float(expected).is_integer()
        else 1e-8
    )

    if not approx(
        observed,
        expected,
        tolerance
    ):

        raise RuntimeError(
            f"STEP09E measurability anchor changed for {metric}: "
            f"observed={observed}, expected={expected}"
        )

print("STEP09E outcome-independent measurability anchors: PASS")


# ==================================================================================================
# 8. STEP09E COMPLETE 24-FEATURE AUDIT
# ==================================================================================================

audit_rows, audit_fields = read_csv(
    MEAS_AUDIT
)

required_audit_fields = [
    "frozen_gene_id",
    "gene_symbol",
    "gene_type",
    "protein_coding",
    "frozen_weight",
    "absolute_weight",
    "mapped_in_external",
    "tpm_nonzero_samples",
    "count_nonzero_samples",
    "all_zero_TPM",
    "all_zero_counts",
    "nonzero_in_any_sample",
]

missing_audit_fields = [
    field
    for field in required_audit_fields
    if field not in audit_fields
]

if missing_audit_fields:

    raise RuntimeError(
        "STEP09E audit missing field(s): "
        + ", ".join(
            missing_audit_fields
        )
    )

if len(
    audit_rows
) != 24:

    raise RuntimeError(
        f"STEP09E audit must contain 24 rows; observed {len(audit_rows)}."
    )

mapped_rows = [
    row
    for row in audit_rows
    if logical(
        row[
            "mapped_in_external"
        ]
    )
]

unmapped_rows = [
    row
    for row in audit_rows
    if not logical(
        row[
            "mapped_in_external"
        ]
    )
]

zero_tpm_rows = [
    row
    for row in audit_rows
    if logical(
        row[
            "all_zero_TPM"
        ]
    )
]

measurable_rows = [
    row
    for row in audit_rows
    if logical(
        row[
            "nonzero_in_any_sample"
        ]
    )
]

if len(
    mapped_rows
) != 23:

    raise RuntimeError(
        "STEP09E direct mapped-row count != 23."
    )

if len(
    unmapped_rows
) != 1:

    raise RuntimeError(
        "STEP09E direct unmapped-row count != 1."
    )

if len(
    zero_tpm_rows
) != 6:

    raise RuntimeError(
        "STEP09E direct all-zero TPM count != 6."
    )

if len(
    measurable_rows
) != 17:

    raise RuntimeError(
        "STEP09E direct measurable feature count != 17."
    )

allzero_abs_weight = sum(
    num(
        row[
            "absolute_weight"
        ]
    )
    for row in zero_tpm_rows
)

measurable_abs_weight = sum(
    num(
        row[
            "absolute_weight"
        ]
    )
    for row in measurable_rows
)

unmapped_abs_weight = sum(
    num(
        row[
            "absolute_weight"
        ]
    )
    for row in unmapped_rows
)

if not approx(
    allzero_abs_weight,
    0.44462073,
    5e-6
):

    raise RuntimeError(
        "Direct all-zero absolute-weight sum changed."
    )

if not approx(
    measurable_abs_weight,
    0.48197900,
    5e-6
):

    raise RuntimeError(
        "Direct measurable absolute-weight sum changed."
    )

if not approx(
    unmapped_abs_weight,
    0.07340027,
    5e-6
):

    raise RuntimeError(
        "Direct unmapped absolute-weight sum changed."
    )

allzero_pc = sum(
    logical(
        row[
            "protein_coding"
        ]
    )
    for row in zero_tpm_rows
)

if allzero_pc != 0:

    raise RuntimeError(
        "At least one STEP09E all-zero feature is now protein-coding."
    )

unmapped_symbols = [
    clean_text(
        row[
            "gene_symbol"
        ]
    )
    for row in unmapped_rows
]

if unmapped_symbols != [
    "MIR6859-4"
]:

    raise RuntimeError(
        f"Unexpected unmapped frozen feature: {unmapped_symbols}"
    )

print("STEP09E 24-feature gene-level audit: PASS")
print("Unmapped feature = MIR6859-4: PASS")
print("All six all-zero TPM features are non-protein-coding: PASS")


# ==================================================================================================
# 9. STEP09F POST-HOC / EXPLORATORY SENSITIVITY
# ==================================================================================================

sens = metric_table(
    SENS_SUMMARY,
    "STEP09F measurable-only summary"
)

sens_required = [
    "Gene_count",
    "Retained_absolute_weight",
    "Control_mean",
    "Control_SD",
    "Schizophrenia_mean",
    "Schizophrenia_SD",
    "Mean_difference",
    "Mean_difference_CI_low",
    "Mean_difference_CI_high",
    "Hedges_g",
    "Hedges_g_CI_low",
    "Hedges_g_CI_high",
    "AUROC",
    "AUROC_CI_low",
    "AUROC_CI_high",
    "Welch_p",
    "Wilcoxon_p",
    "Pearson_primary_vs_sensitivity",
    "Spearman_primary_vs_sensitivity",
]

for metric in sens_required:

    require_metric(
        sens,
        metric,
        "STEP09F"
    )

sens_anchor_checks = [
    (
        "Gene_count",
        17.0,
        1e-8
    ),
    (
        "Retained_absolute_weight",
        0.481979,
        5e-6
    ),
    (
        "Control_mean",
        -0.136396,
        5e-6
    ),
    (
        "Control_SD",
        0.019663,
        5e-6
    ),
    (
        "Schizophrenia_mean",
        -0.136173,
        5e-6
    ),
    (
        "Schizophrenia_SD",
        0.017022,
        5e-6
    ),
    (
        "Mean_difference",
        0.000223,
        5e-6
    ),
    (
        "Mean_difference_CI_low",
        -0.013325,
        5e-6
    ),
    (
        "Mean_difference_CI_high",
        0.012535,
        5e-6
    ),
    (
        "Hedges_g",
        0.0118,
        5e-4
    ),
    (
        "Hedges_g_CI_low",
        -0.6458,
        5e-4
    ),
    (
        "Hedges_g_CI_high",
        0.8632,
        5e-4
    ),
    (
        "AUROC",
        0.5467,
        5e-4
    ),
    (
        "AUROC_CI_low",
        0.3378,
        5e-4
    ),
    (
        "AUROC_CI_high",
        0.7556,
        5e-4
    ),
    (
        "Welch_p",
        0.9737,
        5e-4
    ),
    (
        "Wilcoxon_p",
        0.6783,
        5e-4
    ),
    (
        "Pearson_primary_vs_sensitivity",
        0.7407,
        5e-4
    ),
    (
        "Spearman_primary_vs_sensitivity",
        0.7046,
        5e-4
    ),
]

for metric, expected, tolerance in sens_anchor_checks:

    observed = sens[
        metric
    ]

    if not approx(
        observed,
        expected,
        tolerance
    ):

        raise RuntimeError(
            f"STEP09F anchor changed for {metric}: "
            f"observed={observed}, expected~={expected}"
        )

print("STEP09F measurable-only post-hoc sensitivity anchors: PASS")


# ==================================================================================================
# 10. STEP09F MEASURABLE-GENE FILE
# ==================================================================================================

sens_genes, sens_gene_fields = read_csv(
    SENS_GENES
)

required_sens_gene_fields = [
    "frozen_gene_id",
    "frozen_symbol",
    "frozen_weight",
    "nonzero_sample_count",
]

missing_sens_gene_fields = [
    field
    for field in required_sens_gene_fields
    if field not in sens_gene_fields
]

if missing_sens_gene_fields:

    raise RuntimeError(
        "STEP09F measurable-gene file missing field(s): "
        + ", ".join(
            missing_sens_gene_fields
        )
    )

if len(
    sens_genes
) != 17:

    raise RuntimeError(
        f"STEP09F gene list must contain 17 rows; observed {len(sens_genes)}."
    )

sens_gene_ids = {
    clean_text(
        row[
            "frozen_gene_id"
        ]
    )
    for row in sens_genes
}

measurable_ids = {
    clean_text(
        row[
            "frozen_gene_id"
        ]
    )
    for row in measurable_rows
}

if sens_gene_ids != measurable_ids:

    raise RuntimeError(
        "STEP09F 17-gene set is not exactly the STEP09E outcome-independent measurable set."
    )

sens_abs_weight = sum(
    abs(
        num(
            row[
                "frozen_weight"
            ]
        )
    )
    for row in sens_genes
)

if not approx(
    sens_abs_weight,
    0.481979,
    5e-6
):

    raise RuntimeError(
        "STEP09F 17-gene retained absolute weight changed."
    )

if any(
    integer(
        row[
            "nonzero_sample_count"
        ]
    ) <= 0
    for row in sens_genes
):

    raise RuntimeError(
        "STEP09F contains a gene not measurable in any sample."
    )

print("STEP09E measurable set == STEP09F sensitivity set: PASS")
print("STEP09F retained 17-gene absolute weight: PASS")


# ==================================================================================================
# 11. PREPARE TABLE S7A - PRIMARY RESULT
# ==================================================================================================

s7a_rows = [
    [
        "Analysis status",
        "PRIMARY EXTERNAL OUTCOME EVALUATION",
        "Technically eligible GSE224683 cohort; this remains the primary external result."
    ],

    [
        "Participants",
        "30 total: 15 Control; 15 schizophrenia",
        "Frozen diagnosis groups."
    ],

    [
        "Frozen features used",
        "23 / 24",
        "One frozen feature was unavailable under exact stable-Ensembl mapping."
    ],

    [
        "Retained absolute frozen weight",
        f"{primary['Retained_absolute_weight']:.7f}",
        "No external weight tuning or re-estimation."
    ],

    [
        "Control SCZ-MSS",
        (
            f"{primary['Control_mean_SCZ_MSS']:.6f} "
            f"+/- {primary['Control_SD_SCZ_MSS']:.6f}"
        ),
        "Mean +/- SD."
    ],

    [
        "Schizophrenia SCZ-MSS",
        (
            f"{primary['Schizophrenia_mean_SCZ_MSS']:.6f} "
            f"+/- {primary['Schizophrenia_SD_SCZ_MSS']:.6f}"
        ),
        "Mean +/- SD."
    ],

    [
        "Mean difference: schizophrenia - Control",
        (
            f"{primary['Mean_difference_SCZ_minus_Control']:.6f} "
            f"[{primary['Mean_difference_bootstrap_CI_low']:.6f}, "
            f"{primary['Mean_difference_bootstrap_CI_high']:.6f}]"
        ),
        "Bootstrap 95% CI."
    ],

    [
        "Hedges' g",
        (
            f"{primary['Hedges_g']:.4f} "
            f"[{primary['Hedges_g_bootstrap_CI_low']:.4f}, "
            f"{primary['Hedges_g_bootstrap_CI_high']:.4f}]"
        ),
        "Bootstrap 95% CI."
    ],

    [
        "AUROC",
        (
            f"{primary['AUROC']:.4f} "
            f"[{primary['AUROC_bootstrap_CI_low']:.4f}, "
            f"{primary['AUROC_bootstrap_CI_high']:.4f}]"
        ),
        "Bootstrap 95% CI."
    ],

    [
        "Welch t-test P",
        fmt_p(
            primary[
                "Welch_t_p"
            ]
        ),
        "Two-group score comparison."
    ],

    [
        "Wilcoxon P",
        fmt_p(
            primary[
                "Wilcoxon_p"
            ]
        ),
        "Nonparametric two-group comparison."
    ],
]


# ==================================================================================================
# 12. PREPARE TABLE S7B - OUTCOME-INDEPENDENT MEASURABILITY
# ==================================================================================================

s7b_rows = [
    [
        "Frozen features",
        "24",
        "Complete frozen score definition."
    ],

    [
        "Mapped features",
        "23",
        "Exact stable-Ensembl mapping."
    ],

    [
        "Unmapped features",
        "1",
        "MIR6859-4."
    ],

    [
        "Mapped absolute weight",
        f"{meas['Mapped_absolute_weight']:.8f}",
        "Absolute frozen weight represented by the 23 mapped features."
    ],

    [
        "Unmapped absolute weight",
        f"{meas['Unmapped_absolute_weight']:.8f}",
        "Absolute frozen weight unavailable because of the single unmapped feature."
    ],

    [
        "Features with TPM > 0 in at least one sample",
        "17",
        "Outcome-independent measurability definition."
    ],

    [
        "Mapped features with TPM = 0 in all samples",
        "6",
        "All six were non-protein-coding."
    ],

    [
        "Mapped features with counts = 0 in all samples",
        "6",
        "Same six features were all-zero in estimated counts."
    ],

    [
        "Absolute weight carried by all-zero TPM features",
        f"{meas['All_zero_TPM_absolute_weight']:.8f}",
        "Descriptive assay-measurability limitation; no new gate was introduced."
    ],

    [
        "Absolute weight represented by measurable features",
        f"{meas['Absolute_weight_nonzero_in_any_sample']:.8f}",
        "Absolute frozen weight carried by the 17 genes with nonzero TPM in at least one sample."
    ],
]


# ==================================================================================================
# 13. PREPARE TABLE S7C - UNMAPPED / ALL-ZERO FEATURE DETAIL
# ==================================================================================================

issue_rows = []

for row in audit_rows:

    mapped = logical(
        row[
            "mapped_in_external"
        ]
    )

    all_zero = logical(
        row[
            "all_zero_TPM"
        ]
    )

    if not mapped:

        status = "Unmapped"

    elif all_zero:

        status = "Mapped; all-zero TPM"

    else:

        continue

    symbol = clean_text(
        row[
            "gene_symbol"
        ]
    )

    if not symbol:

        symbol = clean_text(
            row[
                "frozen_gene_id"
            ]
        )

    issue_rows.append(
        [
            symbol,
            clean_text(
                row[
                    "frozen_gene_id"
                ]
            ),
            clean_text(
                row[
                    "gene_type"
                ]
            ),
            f"{num(row['frozen_weight']):+.8f}",
            status,
            str(
                integer(
                    row[
                        "tpm_nonzero_samples"
                    ]
                )
            ),
            str(
                integer(
                    row[
                        "count_nonzero_samples"
                    ]
                )
            ),
        ]
    )

issue_rows.sort(
    key=lambda row: (
        0
        if row[
            4
        ] == "Unmapped"
        else 1,
        row[
            0
        ]
    )
)

if len(
    issue_rows
) != 7:

    raise RuntimeError(
        f"Expected 1 unmapped + 6 all-zero feature rows; observed {len(issue_rows)}."
    )


# ==================================================================================================
# 14. PREPARE TABLE S7D - POST-HOC / EXPLORATORY SENSITIVITY
# ==================================================================================================

s7d_rows = [
    [
        "Analysis status",
        "POST-HOC / EXPLORATORY",
        "Does not replace or redefine the primary STEP09D external result."
    ],

    [
        "Subset definition",
        "17 measurable features",
        "Defined solely by the outcome-independent STEP09E measurability audit."
    ],

    [
        "Retained absolute frozen weight",
        f"{sens['Retained_absolute_weight']:.6f}",
        "Frozen directions and relative weights unchanged; no diagnosis-based feature selection."
    ],

    [
        "Control SCZ-MSS",
        (
            f"{sens['Control_mean']:.6f} "
            f"+/- {sens['Control_SD']:.6f}"
        ),
        "Mean +/- SD."
    ],

    [
        "Schizophrenia SCZ-MSS",
        (
            f"{sens['Schizophrenia_mean']:.6f} "
            f"+/- {sens['Schizophrenia_SD']:.6f}"
        ),
        "Mean +/- SD."
    ],

    [
        "Mean difference: schizophrenia - Control",
        (
            f"{sens['Mean_difference']:.6f} "
            f"[{sens['Mean_difference_CI_low']:.6f}, "
            f"{sens['Mean_difference_CI_high']:.6f}]"
        ),
        "Bootstrap 95% CI."
    ],

    [
        "Hedges' g",
        (
            f"{sens['Hedges_g']:.4f} "
            f"[{sens['Hedges_g_CI_low']:.4f}, "
            f"{sens['Hedges_g_CI_high']:.4f}]"
        ),
        "Bootstrap 95% CI."
    ],

    [
        "AUROC",
        (
            f"{sens['AUROC']:.4f} "
            f"[{sens['AUROC_CI_low']:.4f}, "
            f"{sens['AUROC_CI_high']:.4f}]"
        ),
        "Bootstrap 95% CI."
    ],

    [
        "Welch t-test P",
        f"{sens['Welch_p']:.4f}",
        "Post-hoc 17-feature score comparison."
    ],

    [
        "Wilcoxon P",
        f"{sens['Wilcoxon_p']:.4f}",
        "Post-hoc 17-feature score comparison."
    ],

    [
        "Pearson correlation: primary vs sensitivity score",
        f"{sens['Pearson_primary_vs_sensitivity']:.4f}",
        "Across the same 30 external samples."
    ],

    [
        "Spearman correlation: primary vs sensitivity score",
        f"{sens['Spearman_primary_vs_sensitivity']:.4f}",
        "Across the same 30 external samples."
    ],
]


# ==================================================================================================
# 15. SAVE SOURCE CSVs
# ==================================================================================================

write_csv(
    S7A_OUT,
    [
        "Metric",
        "Primary frozen result",
        "Interpretation / analysis note",
    ],
    s7a_rows
)

write_csv(
    S7B_OUT,
    [
        "Measurability metric",
        "Frozen result",
        "Interpretation / audit note",
    ],
    s7b_rows
)

write_csv(
    S7C_OUT,
    [
        "Gene",
        "Stable Ensembl ID",
        "Gene type",
        "Frozen weight",
        "External measurability status",
        "TPM nonzero samples",
        "Count nonzero samples",
    ],
    issue_rows
)

write_csv(
    S7D_OUT,
    [
        "Metric",
        "Post-hoc / exploratory result",
        "Interpretation / analysis note",
    ],
    s7d_rows
)

print("Supplementary Table S7 source CSVs: PASS")


# ==================================================================================================
# 16. WORD HELPERS
# ==================================================================================================

def set_run_font(
    run,
    size_pt=8.5,
    bold=False,
    italic=False
):

    run.font.name = (
        "Times New Roman"
    )

    run.font.size = Pt(
        size_pt
    )

    run.font.bold = bold
    run.font.italic = italic

    rpr = run._element.get_or_add_rPr()

    rfonts = rpr.find(
        qn(
            "w:rFonts"
        )
    )

    if rfonts is None:

        rfonts = OxmlElement(
            "w:rFonts"
        )

        rpr.insert(
            0,
            rfonts
        )

    for key in [
        "ascii",
        "hAnsi",
        "eastAsia",
        "cs"
    ]:

        rfonts.set(
            qn(
                f"w:{key}"
            ),
            "Times New Roman"
        )


def format_paragraph(
    paragraph,
    alignment=WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=0
):

    paragraph.alignment = alignment

    paragraph.paragraph_format.space_before = Pt(
        before
    )

    paragraph.paragraph_format.space_after = Pt(
        after
    )

    paragraph.paragraph_format.line_spacing = 1.0


def set_cell_margins(
    cell,
    top=65,
    start=70,
    bottom=65,
    end=70
):

    tcPr = cell._tc.get_or_add_tcPr()

    tcMar = tcPr.first_child_found_in(
        "w:tcMar"
    )

    if tcMar is None:

        tcMar = OxmlElement(
            "w:tcMar"
        )

        tcPr.append(
            tcMar
        )

    for name, value in {
        "top": top,
        "start": start,
        "bottom": bottom,
        "end": end
    }.items():

        node = tcMar.find(
            qn(
                f"w:{name}"
            )
        )

        if node is None:

            node = OxmlElement(
                f"w:{name}"
            )

            tcMar.append(
                node
            )

        node.set(
            qn(
                "w:w"
            ),
            str(
                value
            )
        )

        node.set(
            qn(
                "w:type"
            ),
            "dxa"
        )


def set_cell_width(
    cell,
    width_inches
):

    cell.width = Inches(
        width_inches
    )

    tcPr = cell._tc.get_or_add_tcPr()

    tcW = tcPr.find(
        qn(
            "w:tcW"
        )
    )

    if tcW is None:

        tcW = OxmlElement(
            "w:tcW"
        )

        tcPr.append(
            tcW
        )

    tcW.set(
        qn(
            "w:w"
        ),
        str(
            int(
                width_inches * 1440
            )
        )
    )

    tcW.set(
        qn(
            "w:type"
        ),
        "dxa"
    )


def remove_shading(
    cell
):

    tcPr = cell._tc.get_or_add_tcPr()

    for node in list(
        tcPr.findall(
            qn(
                "w:shd"
            )
        )
    ):

        tcPr.remove(
            node
        )


def set_cell_borders(
    cell,
    size=4
):

    tcPr = cell._tc.get_or_add_tcPr()

    borders = tcPr.find(
        qn(
            "w:tcBorders"
        )
    )

    if borders is None:

        borders = OxmlElement(
            "w:tcBorders"
        )

        tcPr.append(
            borders
        )

    for edge in [
        "top",
        "left",
        "bottom",
        "right"
    ]:

        node = borders.find(
            qn(
                f"w:{edge}"
            )
        )

        if node is None:

            node = OxmlElement(
                f"w:{edge}"
            )

            borders.append(
                node
            )

        node.set(
            qn(
                "w:val"
            ),
            "single"
        )

        node.set(
            qn(
                "w:sz"
            ),
            str(
                size
            )
        )

        node.set(
            qn(
                "w:space"
            ),
            "0"
        )

        node.set(
            qn(
                "w:color"
            ),
            "000000"
        )


def repeat_header(
    row
):

    trPr = row._tr.get_or_add_trPr()

    node = OxmlElement(
        "w:tblHeader"
    )

    node.set(
        qn(
            "w:val"
        ),
        "true"
    )

    trPr.append(
        node
    )


def prevent_split(
    row
):

    trPr = row._tr.get_or_add_trPr()

    node = OxmlElement(
        "w:cantSplit"
    )

    trPr.append(
        node
    )


def format_cell(
    cell,
    value,
    width,
    alignment,
    bold=False,
    size=8.3
):

    cell.text = str(
        value
    )

    cell.vertical_alignment = (
        WD_CELL_VERTICAL_ALIGNMENT.CENTER
    )

    set_cell_width(
        cell,
        width
    )

    set_cell_margins(
        cell
    )

    remove_shading(
        cell
    )

    set_cell_borders(
        cell
    )

    for paragraph in cell.paragraphs:

        format_paragraph(
            paragraph,
            alignment
        )

        for run in paragraph.runs:

            set_run_font(
                run,
                size_pt=size,
                bold=bold
            )


def add_table(
    document,
    headers,
    rows,
    widths,
    center_columns=None,
    font_size=8.3
):

    if center_columns is None:
        center_columns = set()

    table = document.add_table(
        rows=1,
        cols=len(
            headers
        )
    )

    table.autofit = False

    header = table.rows[
        0
    ]

    for index, value in enumerate(
        headers
    ):

        format_cell(
            header.cells[
                index
            ],
            value,
            widths[
                index
            ],
            WD_ALIGN_PARAGRAPH.CENTER,
            bold=True,
            size=8.5
        )

    repeat_header(
        header
    )

    prevent_split(
        header
    )

    for values in rows:

        row = table.add_row()

        prevent_split(
            row
        )

        for index, value in enumerate(
            values
        ):

            alignment = (
                WD_ALIGN_PARAGRAPH.CENTER
                if index in center_columns
                else WD_ALIGN_PARAGRAPH.LEFT
            )

            format_cell(
                row.cells[
                    index
                ],
                value,
                widths[
                    index
                ],
                alignment,
                bold=False,
                size=font_size
            )

    return table


def add_heading(
    document,
    text
):

    paragraph = document.add_paragraph()

    format_paragraph(
        paragraph,
        WD_ALIGN_PARAGRAPH.LEFT,
        before=5,
        after=3
    )

    run = paragraph.add_run(
        text
    )

    set_run_font(
        run,
        size_pt=9.4,
        bold=True
    )


# ==================================================================================================
# 17. CREATE WORD DOCUMENT
# ==================================================================================================

doc = Document()

section = doc.sections[
    0
]

section.orientation = (
    WD_ORIENT.LANDSCAPE
)

section.page_width = Inches(
    11
)

section.page_height = Inches(
    8.5
)

section.top_margin = Inches(
    0.42
)

section.bottom_margin = Inches(
    0.42
)

section.left_margin = Inches(
    0.42
)

section.right_margin = Inches(
    0.42
)

normal = doc.styles[
    "Normal"
]

normal.font.name = (
    "Times New Roman"
)

normal.font.size = Pt(
    8.5
)

normal_rpr = normal.element.get_or_add_rPr()

normal_fonts = normal_rpr.rFonts

if normal_fonts is None:

    normal_fonts = OxmlElement(
        "w:rFonts"
    )

    normal_rpr.insert(
        0,
        normal_fonts
    )

for key in [
    "ascii",
    "hAnsi",
    "eastAsia",
    "cs"
]:

    normal_fonts.set(
        qn(
            f"w:{key}"
        ),
        "Times New Roman"
    )


# ==================================================================================================
# 18. TITLE
# ==================================================================================================

title = doc.add_paragraph()

format_paragraph(
    title,
    WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=3
)

run = title.add_run(
    "Supplementary Table S7. "
    "GSE224683 primary external outcome evaluation, "
    "outcome-independent measurability audit, and post-hoc measurable-only sensitivity."
)

set_run_font(
    run,
    size_pt=10.5,
    bold=True
)

intro = doc.add_paragraph()

format_paragraph(
    intro,
    WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=3
)

run = intro.add_run(
    "The STEP09D 23-feature result is the primary external outcome analysis. "
    "STEP09E was an outcome-independent descriptive assay-measurability audit. "
    "STEP09F is post-hoc/exploratory and does not replace the primary result."
)

set_run_font(
    run,
    size_pt=8.3
)


# ==================================================================================================
# 19. PART A
# ==================================================================================================

add_heading(
    doc,
    "A. Primary GSE224683 external outcome evaluation (STEP09D)"
)

add_table(
    doc,
    headers=[
        "Metric",
        "Primary frozen result",
        "Analysis / interpretation note"
    ],
    rows=s7a_rows,
    widths=[
        2.80,
        2.70,
        4.60
    ],
    center_columns={
        1
    },
    font_size=8.3
)


# ==================================================================================================
# 20. PART B
# ==================================================================================================

add_heading(
    doc,
    "B. Outcome-independent GSE224683 measurability audit (STEP09E)"
)

add_table(
    doc,
    headers=[
        "Measurability metric",
        "Frozen result",
        "Audit interpretation"
    ],
    rows=s7b_rows,
    widths=[
        3.00,
        1.80,
        5.30
    ],
    center_columns={
        1
    },
    font_size=8.3
)


# ==================================================================================================
# 21. PART C
# ==================================================================================================

add_heading(
    doc,
    "C. Frozen features that were unmapped or all-zero in GSE224683"
)

add_table(
    doc,
    headers=[
        "Gene",
        "Stable Ensembl ID",
        "Gene type",
        "Frozen weight",
        "Status",
        "TPM nonzero\nsamples",
        "Count nonzero\nsamples"
    ],
    rows=issue_rows,
    widths=[
        1.15,
        1.65,
        1.55,
        1.15,
        1.55,
        1.25,
        1.25
    ],
    center_columns={
        3,
        4,
        5,
        6
    },
    font_size=7.9
)


# ==================================================================================================
# 22. PART D
# ==================================================================================================

add_heading(
    doc,
    "D. Post-hoc / exploratory measurable-only sensitivity (STEP09F)"
)

add_table(
    doc,
    headers=[
        "Metric",
        "Post-hoc / exploratory result",
        "Analysis / interpretation note"
    ],
    rows=s7d_rows,
    widths=[
        3.00,
        2.50,
        4.60
    ],
    center_columns={
        1
    },
    font_size=8.25
)


# ==================================================================================================
# 23. NOTES
# ==================================================================================================

notes = doc.add_paragraph()

format_paragraph(
    notes,
    WD_ALIGN_PARAGRAPH.LEFT,
    before=5,
    after=0
)

run = notes.add_run(
    "Notes. "
)

set_run_font(
    run,
    size_pt=8.2,
    bold=True
)

run = notes.add_run(
    "SCZ-MSS, schizophrenia molecular-state score; "
    "AUROC, area under the receiver-operating-characteristic curve; "
    "TPM, transcripts per million. "
    "The primary external result is a null outcome evaluation in the one public cohort that "
    "passed the frozen technical gate; technical eligibility does not imply biological replication. "
    "The wide confidence intervals reflect the small external sample (n=30). "
    "The outcome-independent measurability audit showed that six mapped non-protein-coding "
    "features were all-zero, but the post-hoc 17-feature sensitivity also remained null-like; "
    "therefore the primary null result cannot be attributed solely to those six all-zero features. "
    "No retraining, external tuning, or diagnosis-based feature selection was performed."
)

set_run_font(
    run,
    size_pt=8.2
)


# ==================================================================================================
# 24. SAVE + READ-BACK
# ==================================================================================================

doc.save(
    DOCX_OUT
)

if (
    not DOCX_OUT.exists()
    or DOCX_OUT.stat().st_size <= 0
):

    raise RuntimeError(
        "Table S7 DOCX was not created."
    )

print("Microsoft Word DOCX created: PASS")

check_doc = Document(
    DOCX_OUT
)

if len(
    check_doc.tables
) != 4:

    raise RuntimeError(
        f"Expected 4 Word tables; observed {len(check_doc.tables)}."
    )

expected_row_counts = [
    1 + len(
        s7a_rows
    ),
    1 + len(
        s7b_rows
    ),
    1 + len(
        issue_rows
    ),
    1 + len(
        s7d_rows
    )
]

for index, expected_rows in enumerate(
    expected_row_counts
):

    observed_rows = len(
        check_doc.tables[
            index
        ].rows
    )

    if observed_rows != expected_rows:

        raise RuntimeError(
            f"Table S7 part {index + 1}: "
            f"expected {expected_rows} rows; observed {observed_rows}."
        )

for table_number, table in enumerate(
    check_doc.tables,
    start=1
):

    for row_number, row in enumerate(
        table.rows,
        start=1
    ):

        for cell_number, cell in enumerate(
            row.cells,
            start=1
        ):

            tcPr = cell._tc.get_or_add_tcPr()

            borders = tcPr.find(
                qn(
                    "w:tcBorders"
                )
            )

            if borders is None:

                raise RuntimeError(
                    f"Missing borders: table {table_number}, "
                    f"row {row_number}, cell {cell_number}."
                )

            for edge in [
                "top",
                "left",
                "bottom",
                "right"
            ]:

                if borders.find(
                    qn(
                        f"w:{edge}"
                    )
                ) is None:

                    raise RuntimeError(
                        f"Missing {edge} border: table {table_number}, "
                        f"row {row_number}, cell {cell_number}."
                    )

            shading = tcPr.find(
                qn(
                    "w:shd"
                )
            )

            if shading is not None:

                fill = shading.get(
                    qn(
                        "w:fill"
                    )
                )

                if (
                    fill is not None
                    and fill.upper() not in {
                        "AUTO",
                        "FFFFFF",
                        "NIL"
                    }
                ):

                    raise RuntimeError(
                        "Unexpected colored table shading detected."
                    )

print("DOCX structural read-back: PASS")
print("Full row borders: PASS")
print("Full column borders: PASS")
print("Black-and-white table structure: PASS")


# ==================================================================================================
# 25. NOTES FILE
# ==================================================================================================

notes_text = """SUPPLEMENTARY TABLE S7 NOTES
============================================================

PRIMARY:
STEP09D is the primary external outcome evaluation.

POST-PRIMARY DIAGNOSTIC AUDIT:
STEP09E is outcome-independent and descriptive.
It did not use diagnosis, AUROC, p-values, or outcome-driven tuning.

POST-HOC SENSITIVITY:
STEP09F is POST-HOC / EXPLORATORY.
It uses exactly the 17 features identified as measurable in STEP09E.
Frozen directions and relative weights were unchanged.
No diagnosis-based feature selection or retraining was performed.

Primary external result:
n = 30
Control = 15
Schizophrenia = 15
Features = 23/24
Retained absolute weight = 0.92659973
AUROC = 0.5067 [0.2889, 0.7244]
Hedges g = -0.0422 [-0.7574, 0.7262]
Welch p = 0.90635
Wilcoxon p = 0.96691

Outcome-independent measurability:
Mapped = 23/24
Measurable = 17/24
All-zero mapped features = 6
All-zero absolute weight = 0.44462073
Measurable absolute weight = 0.48197900
All six all-zero features were non-protein-coding.
Unmapped feature = MIR6859-4

Post-hoc measurable-only:
Features = 17/24
Retained absolute weight = 0.481979
AUROC = 0.5467 [0.3378, 0.7556]
Hedges g = 0.0118 [-0.6458, 0.8632]
Pearson primary-vs-sensitivity = 0.7407
Spearman primary-vs-sensitivity = 0.7046

Interpretation:
The measurable-only sensitivity did not convert the primary null result
into clear external replication. Therefore the null cannot be explained
solely by the six mapped all-zero non-protein-coding features.
"""

NOTES_OUT.write_text(
    notes_text,
    encoding="utf-8"
)


# ==================================================================================================
# 26. GENERATION SUMMARY
# ==================================================================================================

summary_text = f"""SUPPLEMENTARY TABLE S7 GENERATION SUMMARY
============================================================

Frozen STEP09D/E/F artifacts: PASS
External outcome analysis rerun: NO
Score recalculation: NO
Feature selection rerun: NO
External tuning: NO
Retraining: NO

PRIMARY STEP09D:
Control / schizophrenia = 15 / 15
Frozen features = 23 / 24
Retained absolute weight = {primary['Retained_absolute_weight']:.8f}
Control mean +/- SD = {primary['Control_mean_SCZ_MSS']:.6f} +/- {primary['Control_SD_SCZ_MSS']:.6f}
Schizophrenia mean +/- SD = {primary['Schizophrenia_mean_SCZ_MSS']:.6f} +/- {primary['Schizophrenia_SD_SCZ_MSS']:.6f}
Mean difference = {primary['Mean_difference_SCZ_minus_Control']:.6f}
Hedges g = {primary['Hedges_g']:.4f}
AUROC = {primary['AUROC']:.4f}
Welch p = {primary['Welch_t_p']:.5f}
Wilcoxon p = {primary['Wilcoxon_p']:.5f}

STEP09E MEASURABILITY:
Mapped = 23 / 24
Unmapped = 1
Measurable = 17
All-zero TPM = 6
All-zero counts = 6
All-zero absolute weight = {meas['All_zero_TPM_absolute_weight']:.8f}
Measurable absolute weight = {meas['Absolute_weight_nonzero_in_any_sample']:.8f}
All-zero protein-coding = 0
All-zero non-protein-coding = 6
Unmapped = MIR6859-4

STEP09F POST-HOC / EXPLORATORY:
Features = {int(round(sens['Gene_count']))}
Retained absolute weight = {sens['Retained_absolute_weight']:.6f}
AUROC = {sens['AUROC']:.4f}
Hedges g = {sens['Hedges_g']:.4f}
Pearson primary-vs-sensitivity = {sens['Pearson_primary_vs_sensitivity']:.4f}
Spearman primary-vs-sensitivity = {sens['Spearman_primary_vs_sensitivity']:.4f}

Primary result replaced: NO

Times New Roman: PASS
Black-and-white only: PASS
Full row borders: PASS
Full column borders: PASS
DOCX structural read-back: PASS

DOCX:
{DOCX_OUT}
"""

SUMMARY_OUT.write_text(
    summary_text,
    encoding="utf-8"
)


# ==================================================================================================
# 27. MANIFEST
# ==================================================================================================

manifest_targets = [
    (
        "input_primary_summary",
        PRIMARY_SUMMARY
    ),
    (
        "input_primary_sample_scores",
        PRIMARY_SAMPLE
    ),
    (
        "input_measurability_summary",
        MEAS_SUMMARY
    ),
    (
        "input_measurability_feature_audit",
        MEAS_AUDIT
    ),
    (
        "input_posthoc_summary",
        SENS_SUMMARY
    ),
    (
        "input_posthoc_genes",
        SENS_GENES
    ),
    (
        "input_master_external_registry",
        MASTER_REGISTRY
    ),
    (
        "python_generator",
        PY_SCRIPT_PATH
    ),
    (
        "word_docx",
        DOCX_OUT
    ),
    (
        "source_S7A",
        S7A_OUT
    ),
    (
        "source_S7B",
        S7B_OUT
    ),
    (
        "source_S7C",
        S7C_OUT
    ),
    (
        "source_S7D",
        S7D_OUT
    ),
    (
        "notes",
        NOTES_OUT
    ),
    (
        "summary",
        SUMMARY_OUT
    ),
]

manifest_rows = []

for role, path in manifest_targets:

    if not path.exists():

        raise RuntimeError(
            f"Manifest artifact missing: {path}"
        )

    manifest_rows.append(
        [
            role,
            str(
                path
            ),
            path.stat().st_size,
            sha256_file(
                path
            ),
        ]
    )

write_csv(
    MANIFEST_OUT,
    [
        "artifact_role",
        "path",
        "size_bytes",
        "sha256",
    ],
    manifest_rows
)


# ==================================================================================================
# 28. FINAL CHECK
# ==================================================================================================

outputs = [
    DOCX_OUT,
    S7A_OUT,
    S7B_OUT,
    S7C_OUT,
    S7D_OUT,
    NOTES_OUT,
    SUMMARY_OUT,
    MANIFEST_OUT,
    PY_SCRIPT_PATH,
]

for path in outputs:

    if (
        not path.exists()
        or path.stat().st_size <= 0
    ):

        raise RuntimeError(
            f"Missing or zero-byte Table S7 artifact: {path}"
        )

print("All Supplementary Table S7 outputs created and non-zero: PASS")


# ==================================================================================================
# 29. FINAL CONSOLE REPORT
# ==================================================================================================

print()
print("=" * 110)
print("SUPPLEMENTARY TABLE S7 GENERATION SUMMARY")
print("=" * 110)

print("Frozen STEP09D/E/F sources: PASS")
print("External analysis rerun: NO")
print("Primary STEP09D replaced: NO")
print()

print("PRIMARY STEP09D")
print(
    "Control / schizophrenia: "
    "15 / 15"
)

print(
    "Frozen features used: "
    "23 / 24"
)

print(
    "Retained absolute weight: "
    f"{primary['Retained_absolute_weight']:.8f}"
)

print(
    "AUROC: "
    f"{primary['AUROC']:.4f} "
    f"[{primary['AUROC_bootstrap_CI_low']:.4f}, "
    f"{primary['AUROC_bootstrap_CI_high']:.4f}]"
)

print(
    "Hedges g: "
    f"{primary['Hedges_g']:.4f} "
    f"[{primary['Hedges_g_bootstrap_CI_low']:.4f}, "
    f"{primary['Hedges_g_bootstrap_CI_high']:.4f}]"
)

print()

print("STEP09E OUTCOME-INDEPENDENT MEASURABILITY")
print("Mapped / frozen: 23 / 24")
print("Measurable / frozen: 17 / 24")
print("Mapped all-zero TPM features: 6")
print(
    "All-zero absolute weight: "
    f"{meas['All_zero_TPM_absolute_weight']:.8f}"
)

print(
    "Measurable absolute weight: "
    f"{meas['Absolute_weight_nonzero_in_any_sample']:.8f}"
)

print("All-zero protein-coding / non-protein-coding: 0 / 6")
print("Unmapped feature: MIR6859-4")
print()

print("STEP09F POST-HOC / EXPLORATORY")
print("Features: 17 / 24")
print(
    "Retained absolute weight: "
    f"{sens['Retained_absolute_weight']:.6f}"
)

print(
    "AUROC: "
    f"{sens['AUROC']:.4f} "
    f"[{sens['AUROC_CI_low']:.4f}, "
    f"{sens['AUROC_CI_high']:.4f}]"
)

print(
    "Hedges g: "
    f"{sens['Hedges_g']:.4f} "
    f"[{sens['Hedges_g_CI_low']:.4f}, "
    f"{sens['Hedges_g_CI_high']:.4f}]"
)

print(
    "Primary-vs-sensitivity Pearson / Spearman: "
    f"{sens['Pearson_primary_vs_sensitivity']:.4f} / "
    f"{sens['Spearman_primary_vs_sensitivity']:.4f}"
)

print()

print("Times New Roman: PASS")
print("Black-and-white only: PASS")
print("Full row borders: PASS")
print("Full column borders: PASS")
print("DOCX structural read-back: PASS")

print()

print("DOCX:")
print(DOCX_OUT)

print()

print("Source CSVs:")
print(S7A_OUT)
print(S7B_OUT)
print(S7C_OUT)
print(S7D_OUT)

print()

print("Manifest:")
print(MANIFEST_OUT)

print()
print("=" * 110)
print("STEP 11E.7 COMPLETE - SUPPLEMENTARY TABLE S7 SAVED PROJECT-LOCALLY")
print("=" * 110)
)------"

# --------------------------------------------------------------------------------------------------
# 5. SAVE PYTHON GENERATOR
# --------------------------------------------------------------------------------------------------

writeLines(
    PY_SOURCE,
    PY_SCRIPT,
    useBytes = TRUE
)

if (
    !file.exists(PY_SCRIPT) ||
    file.info(PY_SCRIPT)$size <= 0
) {
    stop(
        "Table S7 Python generator was not created."
    )
}

cat(
    "Python generator saved: ",
    PY_SCRIPT,
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 6. PYTHON SYNTAX PREFLIGHT
# --------------------------------------------------------------------------------------------------

compile_output <- system2(
    command = PYTHON_EXE,
    args = c(
        "-m",
        "py_compile",
        shQuote(
            PY_SCRIPT
        )
    ),
    stdout = TRUE,
    stderr = TRUE
)

compile_status <- attr(
    compile_output,
    "status"
)

if (
    !is.null(compile_status) &&
    compile_status != 0
) {

    cat(
        paste(
            compile_output,
            collapse = "\n"
        ),
        "\n"
    )

    stop(
        "Table S7 Python generator failed syntax preflight."
    )
}

cat("Python syntax preflight: PASS\n")

# --------------------------------------------------------------------------------------------------
# 7. RUN PYTHON
# --------------------------------------------------------------------------------------------------

cmd_output <- system2(
    command = PYTHON_EXE,
    args = shQuote(
        PY_SCRIPT
    ),
    stdout = TRUE,
    stderr = TRUE
)

exit_status <- attr(
    cmd_output,
    "status"
)

cat(
    paste(
        cmd_output,
        collapse = "\n"
    ),
    "\n"
)

if (
    !is.null(exit_status) &&
    exit_status != 0
) {
    stop(
        "Python Table S7 generator failed with exit status ",
        exit_status,
        ". See complete console output above."
    )
}

# --------------------------------------------------------------------------------------------------
# 8. R-SIDE FINAL CHECK
# --------------------------------------------------------------------------------------------------

EXPECTED_DOCX <- file.path(
    OUT_DIR,
    "TableS7_GSE224683_Primary_and_Posthoc_External_Evaluation_v1.docx"
)

EXPECTED_S7A <- file.path(
    OUT_DIR,
    "TableS7A_primary_external_results_source_v1.csv"
)

EXPECTED_S7B <- file.path(
    OUT_DIR,
    "TableS7B_measurability_summary_source_v1.csv"
)

EXPECTED_S7C <- file.path(
    OUT_DIR,
    "TableS7C_unmapped_allzero_features_source_v1.csv"
)

EXPECTED_S7D <- file.path(
    OUT_DIR,
    "TableS7D_posthoc_measurable_only_source_v1.csv"
)

EXPECTED_NOTES <- file.path(
    OUT_DIR,
    "TableS7_notes_v1.txt"
)

EXPECTED_SUMMARY <- file.path(
    OUT_DIR,
    "TableS7_generation_summary_v1.txt"
)

EXPECTED_MANIFEST <- file.path(
    OUT_DIR,
    "TableS7_generation_manifest_v1.csv"
)

expected_outputs <- c(
    EXPECTED_DOCX,
    EXPECTED_S7A,
    EXPECTED_S7B,
    EXPECTED_S7C,
    EXPECTED_S7D,
    EXPECTED_NOTES,
    EXPECTED_SUMMARY,
    EXPECTED_MANIFEST,
    PY_SCRIPT
)

missing_outputs <- expected_outputs[
    !file.exists(
        expected_outputs
    )
]

if (length(
    missing_outputs
) > 0L) {

    stop(
        "Expected Table S7 artifact(s) missing:\n",
        paste(
            missing_outputs,
            collapse = "\n"
        )
    )
}

sizes <- file.info(
    expected_outputs
)$size

if (any(
    is.na(
        sizes
    ) |
    sizes <= 0
)) {

    stop(
        "One or more Table S7 artifacts are zero-byte."
    )
}

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.7R COMPLETE - TABLE S7 GENERATED THROUGH PYTHON FROM R KERNEL\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "\nMicrosoft Word DOCX:\n",
    EXPECTED_DOCX,
    "\n",
    sep = ""
)

cat(
    "\nSaved reproducible Python generator:\n",
    PY_SCRIPT,
    "\n",
    sep = ""
)

cat(
    "\nSource CSVs:\n",
    EXPECTED_S7A,
    "\n",
    EXPECTED_S7B,
    "\n",
    EXPECTED_S7C,
    "\n",
    EXPECTED_S7D,
    "\n",
    sep = ""
)

cat(
    "\nManifest:\n",
    EXPECTED_MANIFEST,
    "\n",
    sep = ""
)


STEP 11E.7R - SUPPLEMENTARY TABLE S7
GSE224683 PRIMARY EXTERNAL OUTCOME EVALUATION + MEASURABILITY + POST-HOC SENSITIVITY
Project sentinel: PASS
Python executable: PASS
Python generator saved: E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/tables/supplementary/TableS7/generate_TableS7_v1.py
Python syntax preflight: PASS
STEP 11E.7 - SUPPLEMENTARY TABLE S7
GSE224683 PRIMARY EXTERNAL EVALUATION + MEASURABILITY + POST-HOC SENSITIVITY
Frozen STEP09D/E/F inputs: PASS
Project sentinel: PASS
GSE224683 frozen technical-pass / outcome-evaluated state: PASS
STEP09D primary external outcome anchors: PASS
STEP09D primary 30-sample score file: PASS
STEP09E outcome-independent measurability anchors: PASS
STEP09E 24-feature gene-level audit: PASS
Unmapped feature = MIR6859-4: PASS
All six all-zero TPM features are non-protein-coding: PASS
STEP09F measurable-only post-hoc sensitivity anchors: PASS
STEP09E measurable set == STEP09F sensitivity set: PASS
STEP09F retained 17-gene absolute weight: PASS
Suppleme

In [31]:
# ==================================================================================================
# STEP 11E.8R - SUPPLEMENTARY TABLE S8
# FINAL 24-FEATURE BIOLOGICAL INTERPRETATION,
# IDENTITY CAUTIONS, LITERATURE CONTEXT, AND PATHWAY SUMMARY
#
# R KERNEL -> SAVED PYTHON-DOCX GENERATOR
#
# IMPORTANT
#   - Uses only frozen STEP10 biological-synthesis artifacts.
#   - No enrichment rerun.
#   - No literature search rerun.
#   - No feature re-ranking.
#   - No gene rescue.
#   - No weight modification.
#   - No nominal pathway is promoted to significant.
#   - Black-and-white DOCX with full row + column borders.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.8R - SUPPLEMENTARY TABLE S8\n",
    "FINAL BIOLOGICAL INTERPRETATION, IDENTITY, LITERATURE CONTEXT, AND PATHWAY SUMMARY\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 1. PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

observed_project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    observed_project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "Project ID mismatch. Expected ",
        EXPECTED_PROJECT_ID,
        " but observed ",
        observed_project_id
    )
}

cat("Project sentinel: PASS\n")

# --------------------------------------------------------------------------------------------------
# 2. PYTHON
# --------------------------------------------------------------------------------------------------

PYTHON_EXE <- "E:/Python/python.exe"

if (!file.exists(PYTHON_EXE)) {
    stop(
        "Python executable not found: ",
        PYTHON_EXE
    )
}

cat("Python executable: PASS\n")

# --------------------------------------------------------------------------------------------------
# 3. OUTPUT DIRECTORY
# --------------------------------------------------------------------------------------------------

OUT_DIR <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "supplementary",
    "TableS8"
)

dir.create(
    OUT_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

PY_SCRIPT <- file.path(
    OUT_DIR,
    "generate_TableS8_v1.py"
)

# --------------------------------------------------------------------------------------------------
# 4. COMPLETE PYTHON GENERATOR
# --------------------------------------------------------------------------------------------------

PY_SOURCE <- r"-------(
from pathlib import Path
import csv
import hashlib
import math
from collections import Counter

from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT
from docx.shared import Inches, Pt
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


print("=" * 110)
print("STEP 11E.8 - SUPPLEMENTARY TABLE S8")
print("FINAL BIOLOGICAL INTERPRETATION, IDENTITY, LITERATURE CONTEXT, AND PATHWAY SUMMARY")
print("=" * 110)


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "E:/SCZ_CROSSREGION_MSS_Q1"
)

EXPECTED_PROJECT_ID = (
    "SCZ_CROSSREGION_MSS_Q1_2026"
)

SENTINEL = (
    PROJECT_ROOT
    / ".project_id"
)

BIO_DIR = (
    PROJECT_ROOT
    / "10_biology"
)

FINAL_SYNTHESIS_DIR = (
    BIO_DIR
    / "final_integrated_synthesis_v1"
)

COMPACT_BIO = (
    FINAL_SYNTHESIS_DIR
    / "STEP10G1_SCZ_MSS_compact_manuscript_biological_table_v1.csv"
)

FULL_BIO = (
    FINAL_SYNTHESIS_DIR
    / "STEP10G1_SCZ_MSS_24feature_integrated_biological_evidence_matrix_v1.csv"
)

BIO_SYNTHESIS_TXT = (
    FINAL_SYNTHESIS_DIR
    / "STEP10G1_project_level_biological_synthesis_v1.txt"
)

IDENTITY_PATH = (
    BIO_DIR
    / "identity_audit_v1"
    / "adjudicated_v1"
    / "STEP10D2_SCZ_MSS_24gene_interpretation_identity_v1.csv"
)

ORA_DIR = (
    BIO_DIR
    / "enrichment_v1"
    / "ora_v1"
)

GO_ALL = (
    ORA_DIR
    / "STEP10B2_GO_BP_all_tested_terms_v1.csv"
)

GO_FDR = (
    ORA_DIR
    / "STEP10B2_GO_BP_FDR05_terms_v1.csv"
)

GO_NOMINAL = (
    ORA_DIR
    / "STEP10B2_GO_BP_nominalP05_terms_v1.csv"
)

REACTOME_ALL = (
    ORA_DIR
    / "STEP10B2_Reactome_all_tested_pathways_v1.csv"
)

REACTOME_FDR = (
    ORA_DIR
    / "STEP10B2_Reactome_FDR05_pathways_v1.csv"
)

REACTOME_NOMINAL = (
    ORA_DIR
    / "STEP10B2_Reactome_nominalP05_pathways_v1.csv"
)

OUT_DIR = (
    PROJECT_ROOT
    / "11_manuscript"
    / "tables"
    / "supplementary"
    / "TableS8"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PY_SCRIPT_PATH = (
    OUT_DIR
    / "generate_TableS8_v1.py"
)

DOCX_OUT = (
    OUT_DIR
    / "TableS8_Biological_Interpretation_and_Enrichment_Summary_v1.docx"
)

S8A_OUT = (
    OUT_DIR
    / "TableS8A_biological_synthesis_summary_source_v1.csv"
)

S8B_OUT = (
    OUT_DIR
    / "TableS8B_24feature_biological_context_source_v1.csv"
)

S8C_OUT = (
    OUT_DIR
    / "TableS8C_special_identity_cases_source_v1.csv"
)

S8D_OUT = (
    OUT_DIR
    / "TableS8D_pathway_enrichment_summary_source_v1.csv"
)

NOTES_OUT = (
    OUT_DIR
    / "TableS8_notes_v1.txt"
)

SUMMARY_OUT = (
    OUT_DIR
    / "TableS8_generation_summary_v1.txt"
)

MANIFEST_OUT = (
    OUT_DIR
    / "TableS8_generation_manifest_v1.csv"
)

required_inputs = [
    COMPACT_BIO,
    FULL_BIO,
    BIO_SYNTHESIS_TXT,
    IDENTITY_PATH,
    GO_ALL,
    GO_FDR,
    GO_NOMINAL,
    REACTOME_ALL,
    REACTOME_FDR,
    REACTOME_NOMINAL,
]

for path in required_inputs:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen biological artifact missing: {path}"
        )

    if path.stat().st_size <= 0:

        raise RuntimeError(
            f"Required biological artifact is zero-byte: {path}"
        )

print("Frozen STEP10 biological synthesis artifacts: PASS")


# ==================================================================================================
# 2. HELPERS
# ==================================================================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:

        while True:

            block = handle.read(
                1024 * 1024
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def read_csv(path):

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as handle:

        reader = csv.DictReader(
            handle
        )

        rows = list(
            reader
        )

        fields = reader.fieldnames or []

    return rows, fields


def clean(value):

    if value is None:
        return ""

    text = str(
        value
    ).strip()

    if text.upper() in {
        "",
        "NA",
        "NAN",
        "<NA>",
        "NONE"
    }:
        return ""

    return text


def number(value):

    text = clean(
        value
    )

    if text == "":
        return None

    result = float(
        text
    )

    if not math.isfinite(
        result
    ):

        raise RuntimeError(
            f"Non-finite numeric value: {value}"
        )

    return result


def integer(value):

    result = number(
        value
    )

    if result is None:

        raise RuntimeError(
            f"Missing integer-like value: {value}"
        )

    if abs(
        result
        - round(
            result
        )
    ) > 1e-8:

        raise RuntimeError(
            f"Expected integer-like value: {value}"
        )

    return int(
        round(
            result
        )
    )


def humanize(value):

    text = clean(
        value
    )

    if text == "":
        return "Not specifically reviewed / not available"

    return text.replace(
        "_",
        " "
    )


def write_csv(
    path,
    headers,
    rows
):

    with path.open(
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as handle:

        writer = csv.writer(
            handle
        )

        writer.writerow(
            headers
        )

        writer.writerows(
            rows
        )


# ==================================================================================================
# 3. PROJECT SENTINEL
# ==================================================================================================

if not SENTINEL.exists():

    raise RuntimeError(
        "Project sentinel missing."
    )

project_id = SENTINEL.read_text(
    encoding="utf-8"
).strip()

if project_id != EXPECTED_PROJECT_ID:

    raise RuntimeError(
        f"Project sentinel mismatch: {project_id}"
    )

print("Project sentinel: PASS")


# ==================================================================================================
# 4. LOAD FINAL COMPACT BIOLOGICAL TABLE
# ==================================================================================================

compact, compact_fields = read_csv(
    COMPACT_BIO
)

required_compact_fields = [
    "ensemblID_stable",
    "gene_symbol",
    "interpretation_symbol",
    "gene_type",
    "direction_standardized",
    "final_normalized_weight",
    "absolute_weight_rank",
    "literature_interpretation_tier",
    "overall_direct_SCZ_evidence",
    "overall_brain_context_strength",
    "dominant_brain_context",
    "integrated_biological_evidence_class",
    "manuscript_safe_integrated_interpretation",
]

missing_compact_fields = [
    field
    for field in required_compact_fields
    if field not in compact_fields
]

if missing_compact_fields:

    raise RuntimeError(
        "Final compact biological table missing field(s): "
        + ", ".join(
            missing_compact_fields
        )
    )

if len(
    compact
) != 24:

    raise RuntimeError(
        f"Expected 24 final biological rows; observed {len(compact)}."
    )

stable_ids = [
    clean(
        row[
            "ensemblID_stable"
        ]
    )
    for row in compact
]

if len(
    set(
        stable_ids
    )
) != 24:

    raise RuntimeError(
        "Final biological table has duplicate stable Ensembl IDs."
    )

print("Final 24-feature compact biological table: PASS")


# ==================================================================================================
# 5. LOAD IDENTITY ADJUDICATION
# ==================================================================================================

identity, identity_fields = read_csv(
    IDENTITY_PATH
)

required_identity_fields = [
    "ensemblID_stable",
    "gene_symbol",
    "gene_type",
    "interpretation_symbol",
    "interpretation_identity_class",
    "current_ensembl_id_if_different",
    "identity_adjudication_note",
    "literature_interpretation_tier",
    "manuscript_gene_label",
]

missing_identity_fields = [
    field
    for field in required_identity_fields
    if field not in identity_fields
]

if missing_identity_fields:

    raise RuntimeError(
        "Identity adjudication file missing field(s): "
        + ", ".join(
            missing_identity_fields
        )
    )

if len(
    identity
) != 24:

    raise RuntimeError(
        f"Expected 24 adjudicated identity rows; observed {len(identity)}."
    )

identity_by_id = {}

for row in identity:

    stable_id = clean(
        row[
            "ensemblID_stable"
        ]
    )

    if stable_id in identity_by_id:

        raise RuntimeError(
            f"Duplicate identity row: {stable_id}"
        )

    identity_by_id[
        stable_id
    ] = row

if set(
    identity_by_id.keys()
) != set(
    stable_ids
):

    raise RuntimeError(
        "Identity-adjudication and final biological feature sets differ."
    )

print("STEP10D.2 identity adjudication vs final 24 set: PASS")


# ==================================================================================================
# 6. COMPOSITION CHECKS
# ==================================================================================================

protein_coding_count = sum(
    clean(
        row[
            "gene_type"
        ]
    ) == "protein_coding"
    for row in compact
)

nonprotein_count = (
    len(
        compact
    )
    - protein_coding_count
)

positive_count = sum(
    clean(
        row[
            "direction_standardized"
        ]
    ) == "POSITIVE"
    for row in compact
)

negative_count = sum(
    clean(
        row[
            "direction_standardized"
        ]
    ) == "NEGATIVE"
    for row in compact
)

abs_weight_sum = sum(
    abs(
        number(
            row[
                "final_normalized_weight"
            ]
        )
    )
    for row in compact
)

positive_abs_weight = sum(
    abs(
        number(
            row[
                "final_normalized_weight"
            ]
        )
    )
    for row in compact
    if clean(
        row[
            "direction_standardized"
        ]
    ) == "POSITIVE"
)

negative_abs_weight = sum(
    abs(
        number(
            row[
                "final_normalized_weight"
            ]
        )
    )
    for row in compact
    if clean(
        row[
            "direction_standardized"
        ]
    ) == "NEGATIVE"
)

if protein_coding_count != 12:

    raise RuntimeError(
        f"Protein-coding count changed: {protein_coding_count}"
    )

if nonprotein_count != 12:

    raise RuntimeError(
        f"Non-protein-coding count changed: {nonprotein_count}"
    )

if positive_count != 9:

    raise RuntimeError(
        f"Positive-direction feature count changed: {positive_count}"
    )

if negative_count != 15:

    raise RuntimeError(
        f"Negative-direction feature count changed: {negative_count}"
    )

if abs(
    abs_weight_sum - 1.0
) > 1e-8:

    raise RuntimeError(
        "Frozen normalized absolute weights no longer sum to 1."
    )

if abs(
    positive_abs_weight - 0.3869186
) > 5e-6:

    raise RuntimeError(
        "Positive absolute-weight mass changed."
    )

if abs(
    negative_abs_weight - 0.6130814
) > 5e-6:

    raise RuntimeError(
        "Negative absolute-weight mass changed."
    )

print("24-feature composition integrity: PASS")


# ==================================================================================================
# 7. SCZ / BRAIN LITERATURE COVERAGE
# ==================================================================================================

NO_SCZ = (
    "NO_DIRECT_SCZ_EVIDENCE_RETRIEVED_IN_CURRENT_FOCUSED_QUERY"
)

NO_BRAIN = (
    "NO_REVIEWED_BRAIN_CONTEXT_FROM_CURRENT_SYSTEMATIC_QUERY"
)

scz_reviewed = [
    row
    for row in compact
    if clean(
        row[
            "overall_direct_SCZ_evidence"
        ]
    ) != NO_SCZ
]

brain_reviewed = [
    row
    for row in compact
    if clean(
        row[
            "overall_brain_context_strength"
        ]
    ) != NO_BRAIN
]

if len(
    scz_reviewed
) != 3:

    raise RuntimeError(
        f"Expected 3 SCZ paper-level adjudicated features; observed {len(scz_reviewed)}."
    )

if len(
    brain_reviewed
) != 13:

    raise RuntimeError(
        f"Expected 13 brain-context adjudicated features; observed {len(brain_reviewed)}."
    )

brain_strength_counts = Counter(
    clean(
        row[
            "overall_brain_context_strength"
        ]
    )
    for row in brain_reviewed
)

expected_brain_strength_counts = {
    "LIMITED": 2,
    "LIMITED_TO_MODERATE": 1,
    "MODERATE": 1,
    "MODERATE_TO_STRONG": 1,
    "NO_CONVINCING_CONTEXT": 1,
    "STRONG": 7,
}

if dict(
    brain_strength_counts
) != expected_brain_strength_counts:

    raise RuntimeError(
        "Brain-context strength composition changed: "
        + str(
            dict(
                brain_strength_counts
            )
        )
    )

integrated_class_counts = Counter(
    clean(
        row[
            "integrated_biological_evidence_class"
        ]
    )
    for row in compact
)

expected_integrated_classes = {
    "IDENTITY_LIMITED_OR_ID_ANCHORED_FEATURE":
        5,

    "LIMITED_OR_MODERATE_BRAIN_CONTEXT_WITHOUT_DIRECT_SCZ_EVIDENCE_RETRIEVED":
        4,

    "NO_CONVINCING_REVIEWED_NEURAL_CONTEXT":
        1,

    "NO_FOCUSED_SCZ_OR_REVIEWED_BRAIN_CONTEXT_RETRIEVED":
        6,

    "STRONG_NEURAL_CONTEXT_WITH_ADJUDICATED_SCZ_EVIDENCE":
        3,

    "STRONG_OR_SUBSTANTIAL_BRAIN_CONTEXT_WITHOUT_DIRECT_SCZ_EVIDENCE_RETRIEVED":
        5,
}

if dict(
    integrated_class_counts
) != expected_integrated_classes:

    raise RuntimeError(
        "Integrated biological evidence-class composition changed: "
        + str(
            dict(
                integrated_class_counts
            )
        )
    )

print("SCZ-specific paper-level adjudicated features = 3/24: PASS")
print("Brain-context paper-level adjudicated features = 13/24: PASS")
print("Brain-context strength composition: PASS")
print("Integrated biological evidence-class composition: PASS")


# ==================================================================================================
# 8. ENRICHMENT FILE CHECKS
# ==================================================================================================

go_all, go_all_fields = read_csv(
    GO_ALL
)

go_fdr, go_fdr_fields = read_csv(
    GO_FDR
)

go_nominal, go_nominal_fields = read_csv(
    GO_NOMINAL
)

reactome_all, reactome_all_fields = read_csv(
    REACTOME_ALL
)

reactome_fdr, reactome_fdr_fields = read_csv(
    REACTOME_FDR
)

reactome_nominal, reactome_nominal_fields = read_csv(
    REACTOME_NOMINAL
)

if len(
    go_all
) != 282:

    raise RuntimeError(
        f"GO-BP returned-term count changed: {len(go_all)}"
    )

if len(
    go_fdr
) != 0:

    raise RuntimeError(
        f"GO-BP FDR<0.05 count changed: {len(go_fdr)}"
    )

if len(
    go_nominal
) != 101:

    raise RuntimeError(
        f"GO-BP nominal P<0.05 count changed: {len(go_nominal)}"
    )

if len(
    reactome_all
) != 50:

    raise RuntimeError(
        f"Reactome returned-pathway count changed: {len(reactome_all)}"
    )

if len(
    reactome_fdr
) != 0:

    raise RuntimeError(
        f"Reactome FDR<0.05 count changed: {len(reactome_fdr)}"
    )

if len(
    reactome_nominal
) != 19:

    raise RuntimeError(
        f"Reactome nominal P<0.05 count changed: {len(reactome_nominal)}"
    )

print("GO-BP 282 tested / 0 FDR<0.05 / 101 nominal P<0.05: PASS")
print("Reactome 50 tested / 0 FDR<0.05 / 19 nominal P<0.05: PASS")


# ==================================================================================================
# 9. BUILD PART A - PROJECT-LEVEL BIOLOGICAL SYNTHESIS
# ==================================================================================================

s8a_rows = [
    [
        "Frozen molecular-state features",
        "24",
        "Final locked feature set."
    ],

    [
        "Protein-coding / non-protein-coding",
        "12 / 12",
        "Balanced coding-class composition."
    ],

    [
        "Positive / negative directions",
        "9 / 15",
        "Frozen score directions."
    ],

    [
        "Positive / negative absolute-weight mass",
        (
            f"{positive_abs_weight:.6f} / "
            f"{negative_abs_weight:.6f}"
        ),
        "Absolute normalized-weight mass; total = 1."
    ],

    [
        "SCZ-specific paper-level adjudicated features",
        "3 / 24",
        "Focused schizophrenia-specific evidence review."
    ],

    [
        "Brain/neural-context paper-level adjudicated features",
        "13 / 24",
        "Focused independent brain/neural/cell-type evidence review."
    ],

    [
        "Brain-context strength: STRONG",
        "7",
        "Among the 13 paper-level brain-context adjudicated features."
    ],

    [
        "Brain-context strength: MODERATE_TO_STRONG",
        "1",
        "Among the 13 paper-level brain-context adjudicated features."
    ],

    [
        "Brain-context strength: MODERATE",
        "1",
        "Among the 13 paper-level brain-context adjudicated features."
    ],

    [
        "Brain-context strength: LIMITED_TO_MODERATE",
        "1",
        "Among the 13 paper-level brain-context adjudicated features."
    ],

    [
        "Brain-context strength: LIMITED",
        "2",
        "Among the 13 paper-level brain-context adjudicated features."
    ],

    [
        "Brain-context strength: NO_CONVINCING_CONTEXT",
        "1",
        "A reviewed feature for which convincing neural context was not established."
    ],
]

for class_name, count in expected_integrated_classes.items():

    s8a_rows.append(
        [
            "Integrated evidence class",
            f"{humanize(class_name)}: {count}",
            "Descriptive evidence class only; not a ranking or validation score."
        ]
    )


# ==================================================================================================
# 10. BUILD PART B - COMPLETE 24-FEATURE BIOLOGICAL CONTEXT
# ==================================================================================================

compact_sorted = sorted(
    compact,
    key=lambda row:
        integer(
            row[
                "absolute_weight_rank"
            ]
        )
)

s8b_rows = []

for row in compact_sorted:

    stable_id = clean(
        row[
            "ensemblID_stable"
        ]
    )

    identity_row = identity_by_id[
        stable_id
    ]

    manuscript_label = clean(
        identity_row[
            "manuscript_gene_label"
        ]
    )

    if manuscript_label == "":

        manuscript_label = (
            clean(
                row[
                    "interpretation_symbol"
                ]
            )
            or clean(
                row[
                    "gene_symbol"
                ]
            )
            or stable_id
        )

    s8b_rows.append(
        [
            manuscript_label,

            stable_id,

            clean(
                row[
                    "gene_type"
                ]
            ),

            humanize(
                row[
                    "overall_direct_SCZ_evidence"
                ]
            ),

            humanize(
                row[
                    "overall_brain_context_strength"
                ]
            ),

            clean(
                row[
                    "dominant_brain_context"
                ]
            )
            or "No focused reviewed context",

            humanize(
                row[
                    "integrated_biological_evidence_class"
                ]
            ),

            clean(
                row[
                    "manuscript_safe_integrated_interpretation"
                ]
            ),
        ]
    )

if len(
    s8b_rows
) != 24:

    raise RuntimeError(
        "Biological manuscript table does not contain exactly 24 rows."
    )


# ==================================================================================================
# 11. BUILD PART C - SPECIAL IDENTITY CASES
# ==================================================================================================

special_ids = [
    "ENSG00000117877",  # frozen CD3EAP -> current POLR1G
    "ENSG00000260526",  # AP1AR-DT
    "ENSG00000277869",  # unresolved stable-ID-only feature
    "ENSG00000259363",  # unresolved stable-ID-only feature
    "ENSG00000277118",  # Metazoa_SRP
    "ENSG00000200737",  # Y_RNA
    "ENSG00000187951",  # historical ARHGAP11B identity
    "ENSG00000273629",  # U7
]

if not set(
    special_ids
).issubset(
    set(
        identity_by_id.keys()
    )
):

    raise RuntimeError(
        "One or more expected special identity cases are missing."
    )

s8c_rows = []

for stable_id in special_ids:

    row = identity_by_id[
        stable_id
    ]

    s8c_rows.append(
        [
            stable_id,

            clean(
                row[
                    "gene_symbol"
                ]
            )
            or "Unassigned in frozen annotation",

            clean(
                row[
                    "interpretation_symbol"
                ]
            )
            or "Stable-ID / feature-class interpretation only",

            humanize(
                row[
                    "interpretation_identity_class"
                ]
            ),

            clean(
                row[
                    "current_ensembl_id_if_different"
                ]
            )
            or "No different current Ensembl ID recorded",

            clean(
                row[
                    "identity_adjudication_note"
                ]
            ),
        ]
    )

if len(
    s8c_rows
) != 8:

    raise RuntimeError(
        "Special identity table does not contain eight expected rows."
    )

arhgap_row = identity_by_id[
    "ENSG00000187951"
]

if clean(
    arhgap_row[
        "current_ensembl_id_if_different"
    ]
) != "ENSG00000285077":

    raise RuntimeError(
        "Historical ARHGAP11B current-ID caution changed."
    )

print("Special identity cases: PASS")
print("Historical ARHGAP11B current ID ENSG00000285077: PASS")


# ==================================================================================================
# 12. BUILD PART D - CONVENTIONAL ENRICHMENT SUMMARY
# ==================================================================================================

s8d_rows = [
    [
        "GO Biological Process",
        "282",
        "101",
        "0",
        (
            "No FDR-supported GO-BP enrichment. "
            "Nominal terms are exploratory only."
        ),
    ],

    [
        "Reactome",
        "50",
        "19",
        "0",
        (
            "No FDR-supported Reactome enrichment. "
            "Nominal pathways are exploratory only."
        ),
    ],
]


# ==================================================================================================
# 13. SAVE SOURCE CSVs
# ==================================================================================================

write_csv(
    S8A_OUT,
    [
        "Biological synthesis metric",
        "Frozen result",
        "Interpretation note",
    ],
    s8a_rows
)

write_csv(
    S8B_OUT,
    [
        "Feature",
        "Stable Ensembl ID",
        "Gene type",
        "Direct SCZ evidence",
        "Brain/neural context strength",
        "Dominant brain/neural context",
        "Integrated evidence class",
        "Manuscript-safe integrated interpretation",
    ],
    s8b_rows
)

write_csv(
    S8C_OUT,
    [
        "Stable Ensembl ID",
        "Frozen symbol",
        "Interpretation-safe identity",
        "Identity class",
        "Current Ensembl ID if different",
        "Identity adjudication note",
    ],
    s8c_rows
)

write_csv(
    S8D_OUT,
    [
        "Database",
        "Returned/tested terms or pathways",
        "Nominal P < 0.05",
        "FDR < 0.05",
        "Inferential interpretation",
    ],
    s8d_rows
)

print("Supplementary Table S8 source CSVs: PASS")


# ==================================================================================================
# 14. WORD HELPERS
# ==================================================================================================

def set_run_font(
    run,
    size_pt=8.5,
    bold=False,
    italic=False
):

    run.font.name = (
        "Times New Roman"
    )

    run.font.size = Pt(
        size_pt
    )

    run.font.bold = bold
    run.font.italic = italic

    rpr = run._element.get_or_add_rPr()

    rfonts = rpr.find(
        qn(
            "w:rFonts"
        )
    )

    if rfonts is None:

        rfonts = OxmlElement(
            "w:rFonts"
        )

        rpr.insert(
            0,
            rfonts
        )

    for key in [
        "ascii",
        "hAnsi",
        "eastAsia",
        "cs"
    ]:

        rfonts.set(
            qn(
                f"w:{key}"
            ),
            "Times New Roman"
        )


def format_paragraph(
    paragraph,
    alignment=WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=0,
    line_spacing=1.0
):

    paragraph.alignment = alignment

    paragraph.paragraph_format.space_before = Pt(
        before
    )

    paragraph.paragraph_format.space_after = Pt(
        after
    )

    paragraph.paragraph_format.line_spacing = line_spacing


def set_cell_margins(
    cell,
    top=70,
    start=70,
    bottom=70,
    end=70
):

    tcPr = cell._tc.get_or_add_tcPr()

    tcMar = tcPr.first_child_found_in(
        "w:tcMar"
    )

    if tcMar is None:

        tcMar = OxmlElement(
            "w:tcMar"
        )

        tcPr.append(
            tcMar
        )

    for name, value in {
        "top": top,
        "start": start,
        "bottom": bottom,
        "end": end
    }.items():

        node = tcMar.find(
            qn(
                f"w:{name}"
            )
        )

        if node is None:

            node = OxmlElement(
                f"w:{name}"
            )

            tcMar.append(
                node
            )

        node.set(
            qn(
                "w:w"
            ),
            str(
                value
            )
        )

        node.set(
            qn(
                "w:type"
            ),
            "dxa"
        )


def set_cell_width(
    cell,
    width_inches
):

    cell.width = Inches(
        width_inches
    )

    tcPr = cell._tc.get_or_add_tcPr()

    tcW = tcPr.find(
        qn(
            "w:tcW"
        )
    )

    if tcW is None:

        tcW = OxmlElement(
            "w:tcW"
        )

        tcPr.append(
            tcW
        )

    tcW.set(
        qn(
            "w:w"
        ),
        str(
            int(
                width_inches * 1440
            )
        )
    )

    tcW.set(
        qn(
            "w:type"
        ),
        "dxa"
    )


def remove_shading(
    cell
):

    tcPr = cell._tc.get_or_add_tcPr()

    for node in list(
        tcPr.findall(
            qn(
                "w:shd"
            )
        )
    ):

        tcPr.remove(
            node
        )


def set_cell_borders(
    cell,
    size=4
):

    tcPr = cell._tc.get_or_add_tcPr()

    borders = tcPr.find(
        qn(
            "w:tcBorders"
        )
    )

    if borders is None:

        borders = OxmlElement(
            "w:tcBorders"
        )

        tcPr.append(
            borders
        )

    for edge in [
        "top",
        "left",
        "bottom",
        "right"
    ]:

        node = borders.find(
            qn(
                f"w:{edge}"
            )
        )

        if node is None:

            node = OxmlElement(
                f"w:{edge}"
            )

            borders.append(
                node
            )

        node.set(
            qn(
                "w:val"
            ),
            "single"
        )

        node.set(
            qn(
                "w:sz"
            ),
            str(
                size
            )
        )

        node.set(
            qn(
                "w:space"
            ),
            "0"
        )

        node.set(
            qn(
                "w:color"
            ),
            "000000"
        )


def repeat_header(
    row
):

    trPr = row._tr.get_or_add_trPr()

    node = OxmlElement(
        "w:tblHeader"
    )

    node.set(
        qn(
            "w:val"
        ),
        "true"
    )

    trPr.append(
        node
    )


def prevent_split(
    row
):

    trPr = row._tr.get_or_add_trPr()

    node = OxmlElement(
        "w:cantSplit"
    )

    trPr.append(
        node
    )


def format_cell(
    cell,
    value,
    width,
    alignment,
    bold=False,
    size=8.0
):

    cell.text = str(
        value
    )

    cell.vertical_alignment = (
        WD_CELL_VERTICAL_ALIGNMENT.CENTER
    )

    set_cell_width(
        cell,
        width
    )

    set_cell_margins(
        cell
    )

    remove_shading(
        cell
    )

    set_cell_borders(
        cell
    )

    for paragraph in cell.paragraphs:

        format_paragraph(
            paragraph,
            alignment=alignment,
            line_spacing=1.0
        )

        for run in paragraph.runs:

            set_run_font(
                run,
                size_pt=size,
                bold=bold
            )


def add_table(
    document,
    headers,
    rows,
    widths,
    center_columns=None,
    font_size=8.0
):

    if center_columns is None:
        center_columns = set()

    table = document.add_table(
        rows=1,
        cols=len(
            headers
        )
    )

    table.autofit = False

    header = table.rows[
        0
    ]

    for index, value in enumerate(
        headers
    ):

        format_cell(
            header.cells[
                index
            ],
            value,
            widths[
                index
            ],
            WD_ALIGN_PARAGRAPH.CENTER,
            bold=True,
            size=8.2
        )

    repeat_header(
        header
    )

    prevent_split(
        header
    )

    for values in rows:

        row = table.add_row()

        prevent_split(
            row
        )

        for index, value in enumerate(
            values
        ):

            alignment = (
                WD_ALIGN_PARAGRAPH.CENTER
                if index in center_columns
                else WD_ALIGN_PARAGRAPH.LEFT
            )

            format_cell(
                row.cells[
                    index
                ],
                value,
                widths[
                    index
                ],
                alignment,
                bold=False,
                size=font_size
            )

    return table


def add_heading(
    document,
    text
):

    paragraph = document.add_paragraph()

    format_paragraph(
        paragraph,
        alignment=WD_ALIGN_PARAGRAPH.LEFT,
        before=6,
        after=3
    )

    run = paragraph.add_run(
        text
    )

    set_run_font(
        run,
        size_pt=9.4,
        bold=True
    )


# ==================================================================================================
# 15. CREATE DOCX
# ==================================================================================================

doc = Document()

section = doc.sections[
    0
]

section.orientation = (
    WD_ORIENT.LANDSCAPE
)

section.page_width = Inches(
    11
)

section.page_height = Inches(
    8.5
)

section.top_margin = Inches(
    0.40
)

section.bottom_margin = Inches(
    0.40
)

section.left_margin = Inches(
    0.40
)

section.right_margin = Inches(
    0.40
)

normal = doc.styles[
    "Normal"
]

normal.font.name = (
    "Times New Roman"
)

normal.font.size = Pt(
    8.5
)

normal_rpr = normal.element.get_or_add_rPr()

normal_fonts = normal_rpr.rFonts

if normal_fonts is None:

    normal_fonts = OxmlElement(
        "w:rFonts"
    )

    normal_rpr.insert(
        0,
        normal_fonts
    )

for key in [
    "ascii",
    "hAnsi",
    "eastAsia",
    "cs"
]:

    normal_fonts.set(
        qn(
            f"w:{key}"
        ),
        "Times New Roman"
    )


# ==================================================================================================
# 16. TITLE
# ==================================================================================================

title = doc.add_paragraph()

format_paragraph(
    title,
    alignment=WD_ALIGN_PARAGRAPH.LEFT,
    before=0,
    after=4
)

run = title.add_run(
    "Supplementary Table S8. "
    "Integrated biological interpretation, identity adjudication, "
    "literature context, and conventional pathway analysis of the frozen 24-feature SCZ-MSS state."
)

set_run_font(
    run,
    size_pt=10.3,
    bold=True
)


# ==================================================================================================
# 17. PART A
# ==================================================================================================

add_heading(
    doc,
    "A. Project-level biological synthesis"
)

add_table(
    doc,
    headers=[
        "Biological synthesis metric",
        "Frozen result",
        "Interpretation note"
    ],
    rows=s8a_rows,
    widths=[
        3.25,
        3.10,
        3.85
    ],
    center_columns={
        1
    },
    font_size=8.15
)


# ==================================================================================================
# 18. PART B
# ==================================================================================================

add_heading(
    doc,
    "B. Complete 24-feature biological evidence matrix"
)

add_table(
    doc,
    headers=[
        "Feature",
        "Stable Ensembl ID",
        "Gene type",
        "Direct SCZ evidence",
        "Brain/neural context",
        "Dominant context",
        "Integrated evidence class",
        "Manuscript-safe interpretation"
    ],
    rows=s8b_rows,
    widths=[
        1.65,
        1.28,
        1.05,
        1.35,
        1.20,
        1.35,
        1.55,
        1.77
    ],
    center_columns=set(),
    font_size=6.9
)


# ==================================================================================================
# 19. PART C
# ==================================================================================================

add_heading(
    doc,
    "C. Special identity and annotation cases requiring manuscript caution"
)

add_table(
    doc,
    headers=[
        "Stable Ensembl ID",
        "Frozen symbol",
        "Interpretation-safe identity",
        "Identity class",
        "Current Ensembl ID if different",
        "Adjudication note"
    ],
    rows=s8c_rows,
    widths=[
        1.35,
        1.15,
        1.55,
        1.65,
        1.55,
        2.95
    ],
    center_columns={
        0,
        1,
        2,
        4
    },
    font_size=7.45
)


# ==================================================================================================
# 20. PART D
# ==================================================================================================

add_heading(
    doc,
    "D. Conventional GO-BP and Reactome over-representation analysis"
)

add_table(
    doc,
    headers=[
        "Database",
        "Returned / tested terms or pathways",
        "Nominal P < 0.05",
        "FDR < 0.05",
        "Inferential interpretation"
    ],
    rows=s8d_rows,
    widths=[
        1.55,
        2.25,
        1.40,
        1.30,
        3.70
    ],
    center_columns={
        0,
        1,
        2,
        3
    },
    font_size=8.2
)


# ==================================================================================================
# 21. NOTES
# ==================================================================================================

notes = doc.add_paragraph()

format_paragraph(
    notes,
    alignment=WD_ALIGN_PARAGRAPH.LEFT,
    before=5,
    after=0,
    line_spacing=1.03
)

run = notes.add_run(
    "Notes. "
)

set_run_font(
    run,
    size_pt=8.0,
    bold=True
)

run = notes.add_run(
    "The biological synthesis is descriptive and was performed only after the frozen feature set "
    "and score definition were established. Literature evidence was not used to select, rescue, "
    "remove, re-rank, or reweight features. Lack of a retrieved literature hit is not evidence of "
    "biological irrelevance. Conventional GO-BP and Reactome analyses yielded no FDR-supported "
    "enrichment; nominal signals are therefore retained only as exploratory biological context. "
    "For the historical BrainSeq/GRCh37 ARHGAP11B feature, frozen ENSG00000187951 must not be "
    "interpreted using the current biological annotation attached to that Ensembl identifier; "
    "current ARHGAP11B is ENSG00000285077. POLR1G is the interpretation-safe current identity for "
    "the frozen CD3EAP annotation."
)

set_run_font(
    run,
    size_pt=8.0
)


# ==================================================================================================
# 22. SAVE DOCX
# ==================================================================================================

doc.save(
    DOCX_OUT
)

if (
    not DOCX_OUT.exists()
    or DOCX_OUT.stat().st_size <= 0
):

    raise RuntimeError(
        "Table S8 DOCX was not created."
    )

print("Microsoft Word DOCX created: PASS")


# ==================================================================================================
# 23. STRUCTURAL READ-BACK
# ==================================================================================================

check_doc = Document(
    DOCX_OUT
)

if len(
    check_doc.tables
) != 4:

    raise RuntimeError(
        f"Expected 4 Word tables; observed {len(check_doc.tables)}."
    )

expected_rows = [
    1 + len(
        s8a_rows
    ),
    1 + len(
        s8b_rows
    ),
    1 + len(
        s8c_rows
    ),
    1 + len(
        s8d_rows
    )
]

for index, expected in enumerate(
    expected_rows
):

    observed = len(
        check_doc.tables[
            index
        ].rows
    )

    if observed != expected:

        raise RuntimeError(
            f"Table S8 part {index + 1}: "
            f"expected {expected} rows; observed {observed}."
        )

for table_number, table in enumerate(
    check_doc.tables,
    start=1
):

    for row_number, row in enumerate(
        table.rows,
        start=1
    ):

        for cell_number, cell in enumerate(
            row.cells,
            start=1
        ):

            tcPr = cell._tc.get_or_add_tcPr()

            borders = tcPr.find(
                qn(
                    "w:tcBorders"
                )
            )

            if borders is None:

                raise RuntimeError(
                    f"Missing borders in table {table_number}, "
                    f"row {row_number}, cell {cell_number}."
                )

            for edge in [
                "top",
                "left",
                "bottom",
                "right"
            ]:

                if borders.find(
                    qn(
                        f"w:{edge}"
                    )
                ) is None:

                    raise RuntimeError(
                        f"Missing {edge} border in table {table_number}, "
                        f"row {row_number}, cell {cell_number}."
                    )

            shading = tcPr.find(
                qn(
                    "w:shd"
                )
            )

            if shading is not None:

                fill = shading.get(
                    qn(
                        "w:fill"
                    )
                )

                if (
                    fill is not None
                    and fill.upper() not in {
                        "AUTO",
                        "FFFFFF",
                        "NIL"
                    }
                ):

                    raise RuntimeError(
                        "Unexpected colored table-cell shading detected."
                    )

print("DOCX structural read-back: PASS")
print("Full row borders: PASS")
print("Full column borders: PASS")
print("Black-and-white table structure: PASS")


# ==================================================================================================
# 24. NOTES FILE
# ==================================================================================================

notes_text = """SUPPLEMENTARY TABLE S8 NOTES
============================================================

Final frozen feature set:
24 features

Composition:
Protein-coding = 12
Non-protein-coding = 12
Positive direction = 9
Negative direction = 15
Positive absolute-weight mass = 0.3869186
Negative absolute-weight mass = 0.6130814

Literature evidence:
SCZ-specific paper-level adjudicated features = 3/24
Brain/neural-context paper-level adjudicated features = 13/24

Brain-context strengths among the 13 reviewed features:
STRONG = 7
MODERATE_TO_STRONG = 1
MODERATE = 1
LIMITED_TO_MODERATE = 1
LIMITED = 2
NO_CONVINCING_CONTEXT = 1

Conventional enrichment:
GO-BP returned terms = 282
GO-BP nominal P < 0.05 = 101
GO-BP FDR < 0.05 = 0

Reactome returned pathways = 50
Reactome nominal P < 0.05 = 19
Reactome FDR < 0.05 = 0

Inferential conclusion:
No FDR-supported conventional pathway-enrichment claim is permitted.
Nominal pathway signals remain exploratory only.

Identity caution:
Frozen BrainSeq/GRCh37 ENSG00000187951 was annotated as ARHGAP11B.
Current ARHGAP11B Ensembl ID is ENSG00000285077.
Do not attach current ENSG00000187951 biology to the frozen feature.

Frozen CD3EAP is interpreted using current POLR1G identity while preserving
the original CD3EAP label in frozen-analysis contexts.

No model, score, feature selection, enrichment, or literature retrieval was rerun.
"""

NOTES_OUT.write_text(
    notes_text,
    encoding="utf-8"
)


# ==================================================================================================
# 25. GENERATION SUMMARY
# ==================================================================================================

summary_lines = [
    "=" * 110,
    "SUPPLEMENTARY TABLE S8 GENERATION SUMMARY",
    "=" * 110,
    "",
    "Frozen STEP10 biological synthesis sources: PASS",
    "Biological analysis rerun: NO",
    "Literature search rerun: NO",
    "Enrichment rerun: NO",
    "Feature re-ranking: NO",
    "Weight modification: NO",
    "",
    "COMPOSITION",
    "Frozen features = 24",
    "Protein-coding / non-protein-coding = 12 / 12",
    "Positive / negative = 9 / 15",
    f"Positive absolute-weight mass = {positive_abs_weight:.7f}",
    f"Negative absolute-weight mass = {negative_abs_weight:.7f}",
    "",
    "LITERATURE",
    "SCZ-specific paper-level adjudicated = 3 / 24",
    "Brain-context paper-level adjudicated = 13 / 24",
    "",
    "PATHWAY",
    f"GO-BP returned terms = {len(go_all)}",
    f"GO-BP nominal P<0.05 = {len(go_nominal)}",
    f"GO-BP FDR<0.05 = {len(go_fdr)}",
    f"Reactome returned pathways = {len(reactome_all)}",
    f"Reactome nominal P<0.05 = {len(reactome_nominal)}",
    f"Reactome FDR<0.05 = {len(reactome_fdr)}",
    "",
    "IDENTITY",
    "Special identity cases = 8",
    "Historical ARHGAP11B current Ensembl ID = ENSG00000285077",
    "",
    "FORMATTING",
    "Times New Roman = PASS",
    "Black-and-white only = PASS",
    "Full row borders = PASS",
    "Full column borders = PASS",
    "DOCX structural read-back = PASS",
    "",
    f"DOCX = {DOCX_OUT}",
    "=" * 110,
]

SUMMARY_OUT.write_text(
    "\n".join(
        summary_lines
    ),
    encoding="utf-8"
)


# ==================================================================================================
# 26. MANIFEST
# ==================================================================================================

manifest_targets = [
    (
        "input_compact_biological_table",
        COMPACT_BIO
    ),
    (
        "input_full_biological_matrix",
        FULL_BIO
    ),
    (
        "input_project_biological_synthesis",
        BIO_SYNTHESIS_TXT
    ),
    (
        "input_identity_adjudication",
        IDENTITY_PATH
    ),
    (
        "input_GO_all",
        GO_ALL
    ),
    (
        "input_GO_FDR",
        GO_FDR
    ),
    (
        "input_GO_nominal",
        GO_NOMINAL
    ),
    (
        "input_Reactome_all",
        REACTOME_ALL
    ),
    (
        "input_Reactome_FDR",
        REACTOME_FDR
    ),
    (
        "input_Reactome_nominal",
        REACTOME_NOMINAL
    ),
    (
        "python_generator",
        PY_SCRIPT_PATH
    ),
    (
        "word_docx",
        DOCX_OUT
    ),
    (
        "source_S8A",
        S8A_OUT
    ),
    (
        "source_S8B",
        S8B_OUT
    ),
    (
        "source_S8C",
        S8C_OUT
    ),
    (
        "source_S8D",
        S8D_OUT
    ),
    (
        "notes",
        NOTES_OUT
    ),
    (
        "summary",
        SUMMARY_OUT
    ),
]

manifest_rows = []

for role, path in manifest_targets:

    if not path.exists():

        raise RuntimeError(
            f"Manifest artifact missing: {path}"
        )

    manifest_rows.append(
        [
            role,
            str(
                path
            ),
            path.stat().st_size,
            sha256_file(
                path
            ),
        ]
    )

write_csv(
    MANIFEST_OUT,
    [
        "artifact_role",
        "path",
        "size_bytes",
        "sha256",
    ],
    manifest_rows
)


# ==================================================================================================
# 27. FINAL OUTPUT CHECK
# ==================================================================================================

expected_outputs = [
    DOCX_OUT,
    S8A_OUT,
    S8B_OUT,
    S8C_OUT,
    S8D_OUT,
    NOTES_OUT,
    SUMMARY_OUT,
    MANIFEST_OUT,
    PY_SCRIPT_PATH,
]

for path in expected_outputs:

    if (
        not path.exists()
        or path.stat().st_size <= 0
    ):

        raise RuntimeError(
            f"Missing or zero-byte Table S8 artifact: {path}"
        )

print("All Supplementary Table S8 outputs created and non-zero: PASS")


# ==================================================================================================
# 28. FINAL REPORT
# ==================================================================================================

print()
print("=" * 110)
print("SUPPLEMENTARY TABLE S8 GENERATION SUMMARY")
print("=" * 110)

print("Frozen final biological synthesis sources: PASS")
print("Biological analysis rerun: NO")
print("Literature search rerun: NO")
print("Enrichment rerun: NO")
print()

print("Frozen features: 24")
print("Protein-coding / non-protein-coding: 12 / 12")
print("Positive / negative: 9 / 15")
print(
    "Positive / negative absolute-weight mass: "
    f"{positive_abs_weight:.7f} / "
    f"{negative_abs_weight:.7f}"
)

print()

print("SCZ-specific paper-level adjudicated: 3 / 24")
print("Brain-context paper-level adjudicated: 13 / 24")
print()

print(
    "GO-BP returned / nominal P<0.05 / FDR<0.05: "
    f"{len(go_all)} / {len(go_nominal)} / {len(go_fdr)}"
)

print(
    "Reactome returned / nominal P<0.05 / FDR<0.05: "
    f"{len(reactome_all)} / "
    f"{len(reactome_nominal)} / "
    f"{len(reactome_fdr)}"
)

print()

print("Special identity cases: 8")
print("Historical ARHGAP11B current Ensembl ID: ENSG00000285077")
print()

print("Times New Roman: PASS")
print("Black-and-white only: PASS")
print("Full row borders: PASS")
print("Full column borders: PASS")
print("DOCX structural read-back: PASS")

print()

print("DOCX:")
print(DOCX_OUT)

print()

print("Source CSVs:")
print(S8A_OUT)
print(S8B_OUT)
print(S8C_OUT)
print(S8D_OUT)

print()

print("Manifest:")
print(MANIFEST_OUT)

print()
print("=" * 110)
print("STEP 11E.8 COMPLETE - SUPPLEMENTARY TABLE S8 SAVED PROJECT-LOCALLY")
print("=" * 110)
)-------"

# --------------------------------------------------------------------------------------------------
# 5. SAVE PYTHON GENERATOR
# --------------------------------------------------------------------------------------------------

writeLines(
    PY_SOURCE,
    PY_SCRIPT,
    useBytes = TRUE
)

if (
    !file.exists(PY_SCRIPT) ||
    file.info(PY_SCRIPT)$size <= 0
) {
    stop(
        "Table S8 Python generator was not created."
    )
}

cat(
    "Python generator saved: ",
    PY_SCRIPT,
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 6. PYTHON SYNTAX PREFLIGHT
# --------------------------------------------------------------------------------------------------

compile_output <- system2(
    command = PYTHON_EXE,
    args = c(
        "-m",
        "py_compile",
        shQuote(
            PY_SCRIPT
        )
    ),
    stdout = TRUE,
    stderr = TRUE
)

compile_status <- attr(
    compile_output,
    "status"
)

if (
    !is.null(compile_status) &&
    compile_status != 0
) {

    cat(
        paste(
            compile_output,
            collapse = "\n"
        ),
        "\n"
    )

    stop(
        "Table S8 Python generator failed syntax preflight."
    )
}

cat("Python syntax preflight: PASS\n")

# --------------------------------------------------------------------------------------------------
# 7. RUN PYTHON
# --------------------------------------------------------------------------------------------------

cmd_output <- system2(
    command = PYTHON_EXE,
    args = shQuote(
        PY_SCRIPT
    ),
    stdout = TRUE,
    stderr = TRUE
)

exit_status <- attr(
    cmd_output,
    "status"
)

cat(
    paste(
        cmd_output,
        collapse = "\n"
    ),
    "\n"
)

if (
    !is.null(exit_status) &&
    exit_status != 0
) {
    stop(
        "Python Table S8 generator failed with exit status ",
        exit_status,
        ". See complete console output above."
    )
}

# --------------------------------------------------------------------------------------------------
# 8. R-SIDE FINAL CHECK
# --------------------------------------------------------------------------------------------------

EXPECTED_DOCX <- file.path(
    OUT_DIR,
    "TableS8_Biological_Interpretation_and_Enrichment_Summary_v1.docx"
)

EXPECTED_S8A <- file.path(
    OUT_DIR,
    "TableS8A_biological_synthesis_summary_source_v1.csv"
)

EXPECTED_S8B <- file.path(
    OUT_DIR,
    "TableS8B_24feature_biological_context_source_v1.csv"
)

EXPECTED_S8C <- file.path(
    OUT_DIR,
    "TableS8C_special_identity_cases_source_v1.csv"
)

EXPECTED_S8D <- file.path(
    OUT_DIR,
    "TableS8D_pathway_enrichment_summary_source_v1.csv"
)

EXPECTED_NOTES <- file.path(
    OUT_DIR,
    "TableS8_notes_v1.txt"
)

EXPECTED_SUMMARY <- file.path(
    OUT_DIR,
    "TableS8_generation_summary_v1.txt"
)

EXPECTED_MANIFEST <- file.path(
    OUT_DIR,
    "TableS8_generation_manifest_v1.csv"
)

expected_outputs <- c(
    EXPECTED_DOCX,
    EXPECTED_S8A,
    EXPECTED_S8B,
    EXPECTED_S8C,
    EXPECTED_S8D,
    EXPECTED_NOTES,
    EXPECTED_SUMMARY,
    EXPECTED_MANIFEST,
    PY_SCRIPT
)

missing_outputs <- expected_outputs[
    !file.exists(
        expected_outputs
    )
]

if (length(
    missing_outputs
) > 0L) {

    stop(
        "Expected Table S8 artifact(s) missing:\n",
        paste(
            missing_outputs,
            collapse = "\n"
        )
    )
}

sizes <- file.info(
    expected_outputs
)$size

if (any(
    is.na(
        sizes
    ) |
    sizes <= 0
)) {

    stop(
        "One or more Table S8 artifacts are zero-byte."
    )
}

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.8R COMPLETE - TABLE S8 GENERATED THROUGH PYTHON FROM R KERNEL\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

cat(
    "\nMicrosoft Word DOCX:\n",
    EXPECTED_DOCX,
    "\n",
    sep = ""
)

cat(
    "\nSaved reproducible Python generator:\n",
    PY_SCRIPT,
    "\n",
    sep = ""
)

cat(
    "\nSource CSVs:\n",
    EXPECTED_S8A,
    "\n",
    EXPECTED_S8B,
    "\n",
    EXPECTED_S8C,
    "\n",
    EXPECTED_S8D,
    "\n",
    sep = ""
)

cat(
    "\nManifest:\n",
    EXPECTED_MANIFEST,
    "\n",
    sep = ""
)


STEP 11E.8R - SUPPLEMENTARY TABLE S8
FINAL BIOLOGICAL INTERPRETATION, IDENTITY, LITERATURE CONTEXT, AND PATHWAY SUMMARY
Project sentinel: PASS
Python executable: PASS
Python generator saved: E:/SCZ_CROSSREGION_MSS_Q1/11_manuscript/tables/supplementary/TableS8/generate_TableS8_v1.py
Python syntax preflight: PASS
STEP 11E.8 - SUPPLEMENTARY TABLE S8
FINAL BIOLOGICAL INTERPRETATION, IDENTITY, LITERATURE CONTEXT, AND PATHWAY SUMMARY
Frozen STEP10 biological synthesis artifacts: PASS
Project sentinel: PASS
Final 24-feature compact biological table: PASS
STEP10D.2 identity adjudication vs final 24 set: PASS
24-feature composition integrity: PASS
SCZ-specific paper-level adjudicated features = 3/24: PASS
Brain-context paper-level adjudicated features = 13/24: PASS
Brain-context strength composition: PASS
Integrated biological evidence-class composition: PASS
GO-BP 282 tested / 0 FDR<0.05 / 101 nominal P<0.05: PASS
Reactome 50 tested / 0 FDR<0.05 / 19 nominal P<0.05: PASS
Special identity case

In [32]:
# ==================================================================================================
# STEP 11E.9R - FINAL SUPPLEMENTARY TABLE SERIES FREEZE
# MASTER INVENTORY + SHA256 + MANUSCRIPT CROSS-REFERENCE MAP
#
# PURPOSE
#   1. Verify the FINAL locked version of Supplementary Tables S1-S8.
#   2. Verify each final DOCX, reproducible Python generator, source CSV(s), and generation manifest.
#   3. Compute SHA256 hashes for final DOCX files.
#   4. Record older/non-final DOCX versions without treating their presence as an error.
#   5. Freeze a single master inventory for manuscript/submission use.
#   6. Save a nonredundant manuscript cross-reference map.
#
# IMPORTANT
#   - NO analysis rerun.
#   - NO model rerun.
#   - NO DE rerun.
#   - NO stability/permutation rerun.
#   - NO external-validation rerun.
#   - NO biology/enrichment rerun.
#   - NO Word table regeneration.
# ==================================================================================================

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.9R - FINAL SUPPLEMENTARY TABLE SERIES FREEZE\n",
    "MASTER INVENTORY + SHA256 + MANUSCRIPT CROSS-REFERENCE MAP\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

# --------------------------------------------------------------------------------------------------
# 1. PROJECT VERIFICATION
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT <- "E:/SCZ_CROSSREGION_MSS_Q1"
EXPECTED_PROJECT_ID <- "SCZ_CROSSREGION_MSS_Q1_2026"
PYTHON_EXE <- "E:/Python/python.exe"

SENTINEL_PATH <- file.path(
    PROJECT_ROOT,
    ".project_id"
)

if (!file.exists(SENTINEL_PATH)) {
    stop("Project sentinel missing.")
}

project_id <- trimws(
    readLines(
        SENTINEL_PATH,
        n = 1L,
        warn = FALSE
    )
)

if (!identical(
    project_id,
    EXPECTED_PROJECT_ID
)) {
    stop(
        "Project sentinel mismatch. Expected ",
        EXPECTED_PROJECT_ID,
        "; observed ",
        project_id
    )
}

cat("Project sentinel: PASS\n")

if (!file.exists(PYTHON_EXE)) {
    stop(
        "Python executable missing: ",
        PYTHON_EXE
    )
}

cat("Python executable: PASS\n")

# --------------------------------------------------------------------------------------------------
# 2. MASTER SUPPLEMENTARY DIRECTORY
# --------------------------------------------------------------------------------------------------

SUPP_ROOT <- file.path(
    PROJECT_ROOT,
    "11_manuscript",
    "tables",
    "supplementary"
)

if (!dir.exists(SUPP_ROOT)) {
    stop(
        "Supplementary-table root missing: ",
        SUPP_ROOT
    )
}

FREEZE_DIR <- file.path(
    SUPP_ROOT,
    "00_master_inventory"
)

dir.create(
    FREEZE_DIR,
    recursive = TRUE,
    showWarnings = FALSE
)

if (!dir.exists(FREEZE_DIR)) {
    stop(
        "Could not create master inventory directory."
    )
}

# --------------------------------------------------------------------------------------------------
# 3. FINAL LOCKED VERSION REGISTRY
#
# These are the manuscript-use versions.
# Older versions may remain for provenance, but they are NOT submission versions.
# --------------------------------------------------------------------------------------------------

registry <- data.frame(
    table_id = paste0(
        "S",
        1:8
    ),

    final_version = c(
        "v1",
        "v2",
        "v1",
        "v1",
        "v3",
        "v1",
        "v1",
        "v1"
    ),

    folder = c(
        "TableS1",
        "TableS2",
        "TableS3",
        "TableS4",
        "TableS5",
        "TableS6",
        "TableS7",
        "TableS8"
    ),

    final_docx = c(
        "TableS1_Discovery_Cohort_Donor_Gene_QC_Summary_v1.docx",
        "TableS2_Differential_Expression_Model_Audit_v2.docx",
        "TableS3_NestedCV_Full_Results_Hyperparameters_v1.docx",
        "TableS4_Stability_Refinement_Permutation_Evidence_v1.docx",
        "TableS5_SCZ_MSS_v1_Frozen_Feature_Weights_and_Score_Definition_v3.docx",
        "TableS6_External_Technical_Gate_Audit_v1.docx",
        "TableS7_GSE224683_Primary_and_Posthoc_External_Evaluation_v1.docx",
        "TableS8_Biological_Interpretation_and_Enrichment_Summary_v1.docx"
    ),

    generator = c(
        "generate_TableS1_v1.py",
        "generate_TableS2_v2.py",
        "generate_TableS3_v1.py",
        "generate_TableS4_v1.py",
        "generate_TableS5_v3.py",
        "generate_TableS6_v1.py",
        "generate_TableS7_v1.py",
        "generate_TableS8_v1.py"
    ),

    generation_manifest = c(
        "TableS1_generation_manifest_v1.csv",
        "TableS2_generation_manifest_v2.csv",
        "TableS3_generation_manifest_v1.csv",
        "TableS4_generation_manifest_v1.csv",
        "TableS5_generation_manifest_v3.csv",
        "TableS6_generation_manifest_v1.csv",
        "TableS7_generation_manifest_v1.csv",
        "TableS8_generation_manifest_v1.csv"
    ),

    title_short = c(
        "Discovery cohort architecture, preprocessing, gene universe, and QC",
        "Donor-blocked differential-expression model audit and results",
        "Full 5 x 5 nested-CV results and hyperparameter selection",
        "Stability refinement, final-state evidence, and permutation null",
        "Frozen 24-feature identities, weights, and SCZ-MSS v1 score definition",
        "External technical-gate audit across five public cohorts",
        "GSE224683 primary external evaluation and post-hoc sensitivity",
        "Biological interpretation, identity, literature context, and enrichment"
    ),

    stringsAsFactors = FALSE
)

if (!identical(
    registry$table_id,
    paste0(
        "S",
        1:8
    )
)) {
    stop(
        "Supplementary table numbering is not continuous S1-S8."
    )
}

cat("Final S1-S8 version registry: PASS\n")

# --------------------------------------------------------------------------------------------------
# 4. SHA256 HELPER
#
# Prefer R digest if available.
# Otherwise use the frozen project Python installation.
# No package installation is performed.
# --------------------------------------------------------------------------------------------------

sha256_file <- function(path) {

    if (!file.exists(path)) {
        stop(
            "Cannot hash missing file: ",
            path
        )
    }

    if (requireNamespace(
        "digest",
        quietly = TRUE
    )) {

        return(
            digest::digest(
                file = path,
                algo = "sha256",
                serialize = FALSE
            )
        )
    }

    py_code <- paste0(
        "import hashlib,sys;",
        "h=hashlib.sha256();",
        "f=open(sys.argv[1],'rb');",
        "b=f.read(1048576);",
        "exec(\"while b:\\n h.update(b)\\n b=f.read(1048576)\");",
        "f.close();",
        "print(h.hexdigest())"
    )

    out <- system2(
        command = PYTHON_EXE,
        args = c(
            "-c",
            shQuote(py_code),
            shQuote(path)
        ),
        stdout = TRUE,
        stderr = TRUE
    )

    status <- attr(
        out,
        "status"
    )

    if (
        !is.null(status) &&
        status != 0
    ) {
        stop(
            "Python SHA256 failed for:\n",
            path,
            "\n",
            paste(
                out,
                collapse = "\n"
            )
        )
    }

    hash_candidates <- trimws(
        out[
            grepl(
                "^[0-9a-fA-F]{64}$",
                trimws(out)
            )
        ]
    )

    if (length(
        hash_candidates
    ) != 1L) {
        stop(
            "Could not uniquely parse SHA256 for:\n",
            path,
            "\nPython output:\n",
            paste(
                out,
                collapse = "\n"
            )
        )
    }

    tolower(
        hash_candidates[[1]]
    )
}

# --------------------------------------------------------------------------------------------------
# 5. VERIFY EACH FINAL SUPPLEMENTARY TABLE PACKAGE
# --------------------------------------------------------------------------------------------------

inventory_rows <- vector(
    "list",
    nrow(registry)
)

for (i in seq_len(
    nrow(registry)
)) {

    table_dir <- file.path(
        SUPP_ROOT,
        registry$folder[i]
    )

    if (!dir.exists(table_dir)) {
        stop(
            registry$table_id[i],
            ": folder missing: ",
            table_dir
        )
    }

    docx_path <- file.path(
        table_dir,
        registry$final_docx[i]
    )

    generator_path <- file.path(
        table_dir,
        registry$generator[i]
    )

    manifest_path <- file.path(
        table_dir,
        registry$generation_manifest[i]
    )

    required_paths <- c(
        docx_path,
        generator_path,
        manifest_path
    )

    missing_required <- required_paths[
        !file.exists(
            required_paths
        )
    ]

    if (length(
        missing_required
    ) > 0L) {
        stop(
            registry$table_id[i],
            ": required final artifact(s) missing:\n",
            paste(
                missing_required,
                collapse = "\n"
            )
        )
    }

    required_sizes <- file.info(
        required_paths
    )$size

    if (any(
        is.na(required_sizes) |
        required_sizes <= 0
    )) {
        stop(
            registry$table_id[i],
            ": one or more required final artifacts are zero-byte."
        )
    }

    source_csvs <- list.files(
        table_dir,
        pattern = "source.*\\.csv$",
        full.names = TRUE,
        ignore.case = TRUE
    )

    source_csvs <- source_csvs[
        file.info(
            source_csvs
        )$size > 0
    ]

    if (length(
        source_csvs
    ) < 1L) {
        stop(
            registry$table_id[i],
            ": no non-zero source CSV found."
        )
    }

    all_docx <- list.files(
        table_dir,
        pattern = "\\.docx$",
        full.names = FALSE,
        ignore.case = TRUE
    )

    nonfinal_docx <- setdiff(
        all_docx,
        registry$final_docx[i]
    )

    docx_sha256 <- sha256_file(
        docx_path
    )

    generator_sha256 <- sha256_file(
        generator_path
    )

    manifest_sha256 <- sha256_file(
        manifest_path
    )

    inventory_rows[[i]] <- data.frame(
        table_id = registry$table_id[i],

        final_version = registry$final_version[i],

        title_short = registry$title_short[i],

        status = "FINAL_LOCKED",

        final_docx = normalizePath(
            docx_path,
            winslash = "/",
            mustWork = TRUE
        ),

        final_docx_size_bytes = file.info(
            docx_path
        )$size,

        final_docx_sha256 = docx_sha256,

        python_generator = normalizePath(
            generator_path,
            winslash = "/",
            mustWork = TRUE
        ),

        python_generator_sha256 = generator_sha256,

        source_csv_count = length(
            source_csvs
        ),

        source_csv_files = paste(
            basename(
                source_csvs
            ),
            collapse = " | "
        ),

        generation_manifest = normalizePath(
            manifest_path,
            winslash = "/",
            mustWork = TRUE
        ),

        generation_manifest_sha256 = manifest_sha256,

        nonfinal_docx_count = length(
            nonfinal_docx
        ),

        nonfinal_docx_files = if (
            length(
                nonfinal_docx
            ) == 0L
        ) {
            ""
        } else {
            paste(
                nonfinal_docx,
                collapse = " | "
            )
        },

        stringsAsFactors = FALSE
    )

    cat(
        registry$table_id[i],
        " ",
        registry$final_version[i],
        ": FINAL PACKAGE PASS",
        " | source CSVs=",
        length(source_csvs),
        " | legacy DOCX=",
        length(nonfinal_docx),
        "\n",
        sep = ""
    )
}

inventory <- do.call(
    rbind,
    inventory_rows
)

rownames(
    inventory
) <- NULL

if (nrow(
    inventory
) != 8L) {
    stop(
        "Master inventory does not contain exactly eight final tables."
    )
}

if (!all(
    inventory$status == "FINAL_LOCKED"
)) {
    stop(
        "One or more supplementary tables are not marked FINAL_LOCKED."
    )
}

if (anyDuplicated(
    inventory$final_docx_sha256
)) {
    stop(
        "Unexpected duplicate SHA256 among final S1-S8 DOCX files."
    )
}

cat("All final S1-S8 DOCX packages: PASS\n")
cat("Final DOCX SHA256 uniqueness: PASS\n")

# --------------------------------------------------------------------------------------------------
# 6. CROSS-REFERENCE MAP
#
# This is the manuscript-use map.
# It does NOT claim that the current manuscript DOCX already contains these citations.
# That actual manuscript audit must use the current manuscript file.
# --------------------------------------------------------------------------------------------------

xref <- data.frame(
    table_id = paste0(
        "Supplementary Table S",
        1:8
    ),

    recommended_manuscript_location = c(
        "Methods: discovery cohort / preprocessing; Results: cohort description and QC",
        "Methods: differential-expression model; Results: cross-region differential expression",
        "Methods: donor-grouped nested CV; Results: predictive modeling and hyperparameter selection",
        "Methods/Results: stability selection and diagnosis-permutation analysis",
        "Methods: frozen score construction; Results: final stable molecular-state definition",
        "Methods: external technical eligibility; Results: public-cohort screening",
        "Results: technically eligible GSE224683 external outcome evaluation",
        "Results/Discussion: biological interpretation, literature context, and pathway analysis"
    ),

    main_item_complement = c(
        "Main Table 1; Figure 1; Supplementary Figure S1",
        "Figure 2; Supplementary Figure S2",
        "Figure 3; Main Table 2; Supplementary Figure S3",
        "Figure 4; Supplementary Figure S4",
        "Supplementary Figure S5",
        "Figure 5; Main Table 3",
        "Figure 5; Main Table 3",
        "Supplementary Figure S6"
    ),

    unique_role = c(
        "Exact donor architecture, weighting, preprocessing, gene-universe and QC audit",
        "Exact DE model specification, canonical contrast counts, and 11 deviation genes",
        "All 25 outer tests, tuning selections, regional performance, and model complexity",
        "Outer-model stability, half-sample refinement, final-24 evidence, and permutation null",
        "Exact frozen feature identities, score weights, directions, and score definition",
        "Detailed technical-gate thresholds and five-cohort evaluability audit",
        "Detailed primary null validation, measurability audit, and explicitly post-hoc sensitivity",
        "Complete feature-level biological evidence, identity cautions, and null FDR enrichment result"
    ),

    redundancy_control = c(
        "Do not repeat Main Table 1 demographic detail verbatim in text.",
        "Do not duplicate volcano plots or present the same DE result as another decorative table.",
        "Do not duplicate the model-comparison summary already reported in Main Table 2.",
        "Do not describe permutation null as external validation.",
        "Do not present internal post-selection score separation as unbiased predictive performance.",
        "Technical FAIL means non-evaluable, not biological replication failure.",
        "STEP09F must remain labeled post-hoc/exploratory and must not replace STEP09D.",
        "Nominal GO/Reactome signals must not be described as statistically established enrichment."
    ),

    stringsAsFactors = FALSE
)

if (nrow(
    xref
) != 8L) {
    stop(
        "Cross-reference map must contain exactly eight rows."
    )
}

cat("Nonredundant manuscript cross-reference map: PASS\n")

# --------------------------------------------------------------------------------------------------
# 7. WRITE MASTER INVENTORY
# --------------------------------------------------------------------------------------------------

MASTER_INVENTORY <- file.path(
    FREEZE_DIR,
    "Supplementary_Table_Series_Master_Inventory_v1.csv"
)

CROSSREF_MAP <- file.path(
    FREEZE_DIR,
    "Supplementary_Table_Manuscript_CrossReference_Map_v1.csv"
)

FREEZE_NOTE <- file.path(
    FREEZE_DIR,
    "Supplementary_Table_Series_FINAL_FREEZE_v1.txt"
)

MASTER_MANIFEST <- file.path(
    FREEZE_DIR,
    "Supplementary_Table_Series_Master_Manifest_v1.csv"
)

write.csv(
    inventory,
    MASTER_INVENTORY,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

write.csv(
    xref,
    CROSSREF_MAP,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

# --------------------------------------------------------------------------------------------------
# 8. FREEZE NOTE
# --------------------------------------------------------------------------------------------------

freeze_lines <- c(
    paste(
        rep(
            "=",
            110
        ),
        collapse = ""
    ),

    "SCZ-CRST FINAL SUPPLEMENTARY TABLE SERIES FREEZE",

    paste(
        rep(
            "=",
            110
        ),
        collapse = ""
    ),

    "",

    paste0(
        "Project ID: ",
        EXPECTED_PROJECT_ID
    ),

    "",

    "FINAL SUBMISSION TABLE VERSIONS",

    paste0(
        "Supplementary Table S1: ",
        registry$final_version[1],
        " - ",
        registry$final_docx[1]
    ),

    paste0(
        "Supplementary Table S2: ",
        registry$final_version[2],
        " - ",
        registry$final_docx[2]
    ),

    paste0(
        "Supplementary Table S3: ",
        registry$final_version[3],
        " - ",
        registry$final_docx[3]
    ),

    paste0(
        "Supplementary Table S4: ",
        registry$final_version[4],
        " - ",
        registry$final_docx[4]
    ),

    paste0(
        "Supplementary Table S5: ",
        registry$final_version[5],
        " - ",
        registry$final_docx[5]
    ),

    paste0(
        "Supplementary Table S6: ",
        registry$final_version[6],
        " - ",
        registry$final_docx[6]
    ),

    paste0(
        "Supplementary Table S7: ",
        registry$final_version[7],
        " - ",
        registry$final_docx[7]
    ),

    paste0(
        "Supplementary Table S8: ",
        registry$final_version[8],
        " - ",
        registry$final_docx[8]
    ),

    "",

    "SERIES STATUS",

    "Supplementary Tables S1-S8 complete: YES",
    "All eight final DOCX files present and non-zero: YES",
    "All eight reproducible Python generators present and non-zero: YES",
    "All eight generation manifests present and non-zero: YES",
    "Every table has at least one non-zero source CSV: YES",
    "Final DOCX SHA256 recorded: YES",

    "",

    "ANALYSIS STATUS",

    "Discovery preprocessing rerun: NO",
    "Differential-expression analysis rerun: NO",
    "duplicateCorrelation rerun: NO",
    "Nested CV/model fitting rerun: NO",
    "Stability/permutation analysis rerun: NO",
    "Frozen score re-estimated: NO",
    "External validation rerun: NO",
    "Biological enrichment rerun: NO",
    "Literature search rerun: NO",

    "",

    "INTERPRETATION GUARDS",

    paste(
        "Technical external-cohort failure means technically non-evaluable;",
        "it must not be described as biological replication failure."
    ),

    paste(
        "GSE224683 STEP09D is the primary external outcome evaluation;",
        "STEP09F remains post-hoc/exploratory."
    ),

    paste(
        "No FDR-supported GO-BP or Reactome enrichment was observed;",
        "nominal results remain exploratory only."
    ),

    paste(
        "Frozen BrainSeq ENSG00000187951 retains the historical ARHGAP11B identity;",
        "current ARHGAP11B uses ENSG00000285077."
    ),

    paste(
        "Older DOCX versions may remain in project folders for provenance.",
        "Only versions listed in this freeze are manuscript/submission versions."
    ),

    "",

    "NEXT MANUSCRIPT STEP",

    paste(
        "Perform an actual manuscript citation/cross-reference audit against the current manuscript DOCX.",
        "Do not infer that citations are already correct from this inventory alone."
    ),

    "",

    paste(
        rep(
            "=",
            110
        ),
        collapse = ""
    )
)

writeLines(
    freeze_lines,
    FREEZE_NOTE,
    useBytes = TRUE
)

# --------------------------------------------------------------------------------------------------
# 9. MASTER FREEZE ARTIFACT MANIFEST
# --------------------------------------------------------------------------------------------------

freeze_artifacts <- c(
    MASTER_INVENTORY,
    CROSSREF_MAP,
    FREEZE_NOTE
)

if (any(
    !file.exists(
        freeze_artifacts
    )
)) {
    stop(
        "One or more master freeze artifacts were not created."
    )
}

if (any(
    file.info(
        freeze_artifacts
    )$size <= 0
)) {
    stop(
        "One or more master freeze artifacts are zero-byte."
    )
}

master_manifest <- data.frame(
    artifact_role = c(
        "master_inventory",
        "manuscript_cross_reference_map",
        "final_freeze_note"
    ),

    path = normalizePath(
        freeze_artifacts,
        winslash = "/",
        mustWork = TRUE
    ),

    size_bytes = file.info(
        freeze_artifacts
    )$size,

    sha256 = vapply(
        freeze_artifacts,
        sha256_file,
        FUN.VALUE = character(1)
    ),

    stringsAsFactors = FALSE
)

write.csv(
    master_manifest,
    MASTER_MANIFEST,
    row.names = FALSE,
    quote = TRUE,
    na = ""
)

if (
    !file.exists(
        MASTER_MANIFEST
    ) ||
    file.info(
        MASTER_MANIFEST
    )$size <= 0
) {
    stop(
        "Master freeze manifest was not created."
    )
}

# --------------------------------------------------------------------------------------------------
# 10. READ-BACK VALIDATION
# --------------------------------------------------------------------------------------------------

inventory_check <- read.csv(
    MASTER_INVENTORY,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

xref_check <- read.csv(
    CROSSREF_MAP,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

manifest_check <- read.csv(
    MASTER_MANIFEST,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (nrow(
    inventory_check
) != 8L) {
    stop(
        "Master inventory read-back != 8 tables."
    )
}

if (!identical(
    inventory_check$table_id,
    paste0(
        "S",
        1:8
    )
)) {
    stop(
        "Master inventory S1-S8 order changed."
    )
}

if (nrow(
    xref_check
) != 8L) {
    stop(
        "Cross-reference map read-back != 8 rows."
    )
}

if (nrow(
    manifest_check
) != 3L) {
    stop(
        "Master freeze manifest read-back != 3 artifacts."
    )
}

cat("Master inventory read-back: PASS\n")
cat("Cross-reference map read-back: PASS\n")
cat("Master freeze manifest read-back: PASS\n")

# --------------------------------------------------------------------------------------------------
# 11. FINAL CONSOLE REPORT
# --------------------------------------------------------------------------------------------------

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nFINAL SUPPLEMENTARY TABLE SERIES STATUS\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)

for (i in seq_len(
    nrow(inventory)
)) {

    cat(
        inventory$table_id[i],
        " ",
        inventory$final_version[i],
        " | FINAL_LOCKED",
        " | DOCX SHA256: ",
        inventory$final_docx_sha256[i],
        "\n",
        sep = ""
    )
}

cat(
    "\nSupplementary Tables complete: 8 / 8\n"
)

cat(
    "Final DOCX files verified: 8 / 8\n"
)

cat(
    "Reproducible Python generators verified: 8 / 8\n"
)

cat(
    "Generation manifests verified: 8 / 8\n"
)

cat(
    "Source-data packages verified: 8 / 8\n"
)

cat(
    "Final DOCX SHA256 hashes frozen: 8 / 8\n"
)

cat(
    "\nFINAL SUBMISSION VERSIONS:\n"
)

cat(
    "S1 v1\n",
    "S2 v2\n",
    "S3 v1\n",
    "S4 v1\n",
    "S5 v3\n",
    "S6 v1\n",
    "S7 v1\n",
    "S8 v1\n",
    sep = ""
)

cat(
    "\nMaster inventory:\n",
    MASTER_INVENTORY,
    "\n",
    sep = ""
)

cat(
    "\nManuscript cross-reference map:\n",
    CROSSREF_MAP,
    "\n",
    sep = ""
)

cat(
    "\nFinal freeze note:\n",
    FREEZE_NOTE,
    "\n",
    sep = ""
)

cat(
    "\nMaster freeze manifest:\n",
    MASTER_MANIFEST,
    "\n",
    sep = ""
)

cat(
    "\n",
    paste(rep("=", 110), collapse = ""),
    "\nSTEP 11E.9R COMPLETE - SUPPLEMENTARY TABLES S1-S8 FINAL SERIES FROZEN\n",
    paste(rep("=", 110), collapse = ""),
    "\n",
    sep = ""
)


STEP 11E.9R - FINAL SUPPLEMENTARY TABLE SERIES FREEZE
MASTER INVENTORY + SHA256 + MANUSCRIPT CROSS-REFERENCE MAP
Project sentinel: PASS
Python executable: PASS
Final S1-S8 version registry: PASS
S1 v1: FINAL PACKAGE PASS | source CSVs=1 | legacy DOCX=0
S2 v2: FINAL PACKAGE PASS | source CSVs=3 | legacy DOCX=0
S3 v1: FINAL PACKAGE PASS | source CSVs=4 | legacy DOCX=0
S4 v1: FINAL PACKAGE PASS | source CSVs=3 | legacy DOCX=0
S5 v3: FINAL PACKAGE PASS | source CSVs=2 | legacy DOCX=0
S6 v1: FINAL PACKAGE PASS | source CSVs=3 | legacy DOCX=0
S7 v1: FINAL PACKAGE PASS | source CSVs=4 | legacy DOCX=0
S8 v1: FINAL PACKAGE PASS | source CSVs=4 | legacy DOCX=0
All final S1-S8 DOCX packages: PASS
Final DOCX SHA256 uniqueness: PASS
Nonredundant manuscript cross-reference map: PASS
Master inventory read-back: PASS
Cross-reference map read-back: PASS
Master freeze manifest read-back: PASS

FINAL SUPPLEMENTARY TABLE SERIES STATUS
S1 v1 | FINAL_LOCKED | DOCX SHA256: 62ffb16a277f70d7a71c4e1c7b4dbeb84a